# Table Transformer v1.1-all — DIMER E2E supervised adaptation tutorial: table structure on public-domain scientific tables, zero-shot vs restricted heads vs bounded decoder unfreeze (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/table-transformer-structure-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/table-transformer-structure-pipeline/blob/main/tutorials/table_transformer_structure_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Ftable--transformer--structure--recognition--v1.1--all-ffcc4d?style=flat)](https://huggingface.co/microsoft/table-transformer-structure-recognition-v1.1-all) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Ftable--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/table-transformer) [![arXiv](https://img.shields.io/badge/arXiv-2303.00716-b31b1b.svg)](https://arxiv.org/abs/2303.00716)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table structure recognition on a table-crop image (boxes labelled `table`, `table column`, `table row`, `table column header`, `table projected row header`, `table spanning cell`) and bounded supervised adaptation of four of those labels to a new table corpus and box convention — restricted heads copied from the checkpoint, trained on the frozen DETR decoder features, with an optional unfreeze of the last decoder layers — measured by held-out per-label AP@0.5 / AP@0.75 / AP and grid agreement, using the pinned `microsoft/table-transformer-structure-recognition-v1.1-all` weights

**This notebook is standalone.** It carries the repository's package (4 modules under `src/table_transformer_structure_pipeline/`, at revision `0a0bd0c97f81`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `7587a7ef111d9dcbf8ac695f1376ab7014340a0c` (~116 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Table Transformer structure snapshot (safetensors, 115 MB), decodes the 94 public-domain SciTSR-PD tables embedded in the carried `sample_data` module (0.9 MB of PNG, no download, no credential), validates them and draws 50 / 23 / 21 training, validation and test tables by a seeded split of whole papers, runs one test table through the inference contract with an input manifest, a rejection probe and a per-label `sample-sanity` evaluation report against its reference boxes, scores a grid prior and the untouched checkpoint on the test split (the **zero-shot row**), copies the checkpoint's own rows for the four scored labels into restricted heads and scores them untouched (the **zero-shot policy**, epoch 0), trains them on the frozen DETR decoder features (the **frozen policy**) and then the last two decoder layers with them (the **unfrozen policy**), selects among the three by validation DETR loss, scores the held-out split by per-label AP@0.5 / AP@0.75 / AP, their class means and grid agreement with the selected model, renders structure before and after, exports the trained tensors as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about 3 minutes of model time after the install and the checkpoint download; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own structure-labelled table crops as a `.zip` holding `structure.csv` (columns `id`, `file`, `label`, `x_min`, `y_min`, `x_max`, `y_max`, optional `group`; one row per structure box, pixel coordinates, labels among `table`, `table column`, `table row`, `table spanning cell`) beside the image files — images are decoded from the archive, never extracted to disk. They pass through the same validation, seeded group-disjoint split, prior, zero-shot scoring, three-policy ladder and selection, held-out evaluation, structure rendering, artifact export and reload-parity cells as the SciTSR-PD sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the DETR-style model reads one table image resized so its longest edge is 800 px, runs an in-library ResNet-18 backbone and a 6-layer encoder–decoder, and emits exactly 125 query proposals, each a box and a softmax over the six structure classes and *no object*; the processor keeps the queries whose class score reaches the threshold and maps their boxes back to input pixels. The carried pipeline module adds snapshot verification, the input contract, a fixed output contract and the `box_iou`, `structure_summary`, `validate_inputs` and `evaluation_report` helpers; `evaluation_report` becomes `sample-sanity` only when a caller supplies reference boxes per label — which this notebook, unlike its inference-only predecessor, does, on a real table.

What this notebook adds to inference is **supervised adaptation of the structure vocabulary to a new table corpus and box convention, under an explicit zero-shot / frozen / unfrozen policy ladder**. The dataset is real and public domain: 94 scientific tables from SciTSR-PD (arXiv LaTeX tables rendered at 150 DPI whose source papers carry a CC0 or public-domain dedication), embedded in the carried `sample_data` module with structure boxes **derived once** from the dataset's text chunks and logical cells — rows and columns tiling an ink-bounded table box at the mid-gaps, spanning cells over their grid area — so the boxes are a stated convention, not hand annotation, and SciTSR annotates no column or row headers, so the contract scores four of the six labels. Several tables come from the same paper, so the sample is split by **paper**, never by table. The carried `metrics.py` scores a prediction set by **per-label AP@0.5 / AP@0.75 / AP** (the COCO convention, every (query, class) pair of every image ranked by score), their class means, the recall and precision per label at the pipeline's operating threshold and **grid agreement** (how often the surviving row and column counts both match the reference); a **grid prior** (the training split's mean row and column counts laid out uniformly) and the **untouched checkpoint** frame the numbers. The **zero-shot policy** copies the checkpoint's own rows for the four labels into restricted heads and scores them untrained; the **frozen policy** trains those heads on the frozen decoder features under the DETR set loss (exact Hungarian matching, cross-entropy with a 0.1 no-object weight, L1 and GIoU — implemented in the carried module, no external matcher); the **unfrozen policy** continues by training the last decoder layers with them end to end, and the epoch with the lowest validation loss — which may be the untouched checkpoint — is kept. The checkpoint's own heads and `recognize` are never trained or exported. This corpus is close to the PubTables-1M renders the model was trained on, so the adaptation question is not whether the model can learn the task but whether adapting to a new corpus and a new box convention buys anything the checkpoint does not already give — and what it costs. Nothing here is a quality claim about your tables: it is one seeded split of one small corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics, sample-data and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; decode an embedded public-domain structure-labelled corpus, understand how its boxes were derived, and validate and split it by paper without leakage; run a real table through the public recognition API and read a per-label `sample-sanity` report against reference boxes; read per-label AP@0.5 / AP@0.75 / AP, class means and grid agreement beside a grid prior and the zero-shot checkpoint and understand why a score threshold is an operating point, not part of AP; train restricted heads on frozen features and a bounded decoder unfreeze with explicit hyperparameters and validation-based selection among three policies, one of which is the checkpoint itself; evaluate on an independent paper-disjoint test split; compare structure before and after; and export a safetensors adapter (heads plus any trained decoder layers) that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** table *detection* on a full page (the sibling `table-transformer-detection-pipeline` finds the crop this notebook expects), OCR or cell text extraction, assembling the final cell grid or HTML/CSV export, GriTS or any cell-level metric, column-header or projected-row-header adaptation (SciTSR does not annotate them), backbone or encoder training, data augmentation, any training of the checkpoint's own heads, any PubTables-1M or FinTabNet accuracy claim, and any claim that 94 arXiv tables stand in for your documents. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; float32 on both. The build record measured about 0.1 s per table to run the decoder on CPU (2 s for the 21-table zero-shot pass), about 45 s for the restricted heads including feature extraction and two validation passes, and about 15 s per unfreeze epoch over 50 tables plus a 23-table validation pass. The pinned `torch==2.14.0` install and the 115 MB checkpoint are the large downloads of the run; the tables travel inside the notebook.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union and average precision measure and why AP does not depend on a score threshold; what a Hungarian (one-to-one) matching between predictions and references is; what validation-based selection among policies means; that a table's cell grid is the intersection of its row and column boxes.
- **Data contract:** records are `{{id, image, objects}}` — a PIL image (or a path to one) with sides 16..4,096 px and a list of 1..125 `{{label, box}}` structure objects with `box` = `[x_min, y_min, x_max, y_max]` pixels inside the image (sides of at least 2 px), `label` among `table` (at most one), `table column`, `table row` (at least one each) and `table spanning cell`, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a training set needs 8..2,000 records; tables are de-duplicated by decoded-pixel digest and split by `group` / `paper_id` so one paper never straddles splits. BYOD accepts a `.zip` (or a directory) holding `structure.csv` and the image files.
- **Validation is structural, not semantic:** nothing checks that a box is a row or a column — a mislabelled set is trained on without complaint; the four labels are the checkpoint's own, so a BYOD set must use exactly those strings.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing and downloads nothing beyond the Hub snapshot.
- **External access (data):** none beyond the Hub. The 94 tables are embedded in the carried `sample_data` module as base64 PNGs (each verified against its recorded byte size and SHA-256 before it is decoded); they come from `bevaya/SciTSR-pd` on the Hugging Face Hub (commit `dae336ef`, two parquet files pinned by SHA-256 in `SOURCE_FILES`), whose source papers carry a CC0 or public-domain dedication, credited in the References.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/table-transformer-structure-recognition-v1.1-all` snapshot (~116 MB in total) at revision `7587a7ef111d…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'table-transformer-structure-pipeline',
    'repository_revision': '0a0bd0c97f813128c1d6dee5af2d08665574832e',
    'embedded_module': 'src/table_transformer_structure_pipeline/pipeline.py',
    'embedded_modules': ['src/table_transformer_structure_pipeline/pipeline.py', 'src/table_transformer_structure_pipeline/sample_data.py', 'src/table_transformer_structure_pipeline/metrics.py', 'src/table_transformer_structure_pipeline/samples.py'],
    'module_sha256': '8156370b725ddabcf9a8d0eade19a5b38f555bb355e5bbabaf3f1f948496cee0',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/table_transformer_structure_pipeline/` @ `0a0bd0c97f81`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/table_transformer_structure_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "microsoft/table-transformer-structure-recognition-v1.1-all"
MODEL_REVISION = "7587a7ef111d9dcbf8ac695f1376ab7014340a0c"
MODEL_LICENSE = "mit"
MODEL_KEY = "table-transformer-structure-v1.1-all"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The six structure classes the checkpoint was fine-tuned on (config.json id2label, in id order).
LABELS = (
    "table",
    "table column",
    "table row",
    "table column header",
    "table projected row header",
    "table spanning cell",
)
# Recognition threshold: the value the upstream repository's inference script applies to every
# structure class (microsoft/table-transformer src/inference.py `structure_class_thresholds`, main @
# 16d124f, 2023-09-07). It gates a softmax class score over 125 DETR queries that was not calibrated
# for any document domain; the deployment owns tuning it on labelled tables.
RECOGNITION_THRESHOLD = 0.5
# The upstream inference script crops each detected table with this many pixels of padding before
# structure recognition; the tutorial reproduces that convention. It is documentation, not enforced.
UPSTREAM_CROP_PADDING = 10
# The checkpoint's DETR decoder emits exactly num_queries proposals per image (config.json), so no
# image can yield more than this many structure objects.
MAX_DETECTIONS = 125
# Input ceilings. The processor resizes so the longest edge is 800 px (preprocessor_config.json
# `size.longest_edge`), so image cost is bounded whatever the caller sends; the side ceiling only
# guards memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# preprocessor_config.json says `size: {"longest_edge": 800}`; the pinned transformers image processor
# only accepts the two-key form, and capping both edges at 800 resizes every image so its longest edge
# is 800 px — the same transform (see from_pretrained).
PROCESSOR_SIZE = {"shortest_edge": 800, "longest_edge": 800}

# ---- the adaptation contract ----
WEIGHTS_FILE = "model.safetensors"
WEIGHT_SHA256 = "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"  # dimer-base-manifest.json
PARAMETER_COUNT = 28_828_619
D_MODEL = 256  # the decoder's hidden size (config.json d_model)
NUM_QUERIES = MAX_DETECTIONS  # the decoder emits exactly this many (query, class, box) proposals per image
DECODER_LAYERS = 6  # config.json decoder_layers; only these are ever unfrozen
DEFAULT_TRAINABLE_LAYERS = 2
# The four checkpoint labels the adaptation contract scores and trains; SciTSR annotates no headers, so the
# restricted head has one logit per entry here plus no-object, initialised from the checkpoint's own rows.
ADAPT_LABELS: tuple[str, ...] = ("table", "table column", "table row", "table spanning cell")
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this `evaluate` says measured-small-sample
CLASS_COST, BBOX_COST, GIOU_COST = 1.0, 5.0, 2.0  # DETR matching costs / loss weights
NO_OBJECT_WEIGHT = 0.1  # DETR's eos_coef
ARTIFACT_FORMAT = "org.valcorza.table-transformer-structure.adapter.v1"
ARTIFACT_FORMAT_VERSION = 1
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
POLICY_ZERO_SHOT = "zero-shot checkpoint (restricted heads copied from it, untrained)"
POLICY_FROZEN = "frozen backbone, encoder and decoder + restricted heads"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"threshold must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one table image as PIL.Image.Image (any mode, converted to RGB): a crop of a single table, "
        f"ideally with about {UPSTREAM_CROP_PADDING} px of page around it, as the upstream inference "
        "script crops tables"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes so the longest edge is 800 px, normalises with "
        "ImageNet mean/std, and returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``recognize`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = RECOGNITION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``recognize`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (recognize takes one table image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def structure_summary(result: Mapping[str, Any]) -> dict[str, Any]:
    """Count the structure objects per label and derive the grid size the rows and columns imply.

    ``n_rows`` x ``n_columns`` is the cell grid a caller would build by intersecting row and column
    boxes (the upstream ``objects_to_structures`` step); this helper only counts, it does not build cells.
    """
    counts = {label: 0 for label in LABELS}
    for det in result["detections"]:
        counts[det["label"]] += 1
    return {
        "counts": counts,
        "n_rows": counts["table row"],
        "n_columns": counts["table column"],
        "n_cells_implied": counts["table row"] * counts["table column"],
        "has_column_header": counts["table column header"] > 0,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (label -> xyxy reference boxes for that structure class) the report
    carries one ``box_iou`` entry per reference — the best-overlapping detection **of the same label**
    — plus per-label reference/detection counts, as sample-sanity geometry evidence; without them the
    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    summary = structure_summary(result)
    base = {
        "task": "table structure recognition on a table-crop image",
        "decision_rule": (
            "a DETR query survives when its softmax score for one of the six structure classes reaches "
            "the threshold; the score is a class probability under the model's own softmax, not a "
            "calibrated estimate for the deployment's tables"
        ),
        "threshold": result.get("threshold", RECOGNITION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "structure_summary": summary,
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth structure boxes were supplied for the evaluated table",
            "needs": (
                "labelled row/column/header/spanning-cell boxes on your own tables, scored per object with "
                "box_iou and aggregated into per-class precision/recall or the cell-level metrics (GriTS) "
                "the upstream paper uses; no such labelled set ships with this repository"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in LABELS:
            raise ValueError(f"unknown reference label {label!r}; expected one of {LABELS}")
        same = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "label": label,
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "n_reference": len(boxes),
                    "n_detected": len(same),
                    "estimation": (
                        "one reference box per structure object on a single table, no dispersion estimate"
                    ),
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial table; geometry sanity evidence, "
            "not a structure-recognition benchmark"
        ),
        "needs": (
            "a labelled table set from the deployment domain (publishers, scans, layouts) for any "
            "precision/recall or GriTS claim"
        ),
    }


@dataclass
class TableTransformerStructurePipeline:
    """Table structure recognition (rows, columns, headers, spanning cells) over the pinned checkpoint."""

    _runner: Callable[[Image.Image, float], list[dict[str, Any]]]
    device: str
    source: str = "injected-runner"
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    _head: Any = field(default=None, repr=False)  # restricted class head (len(classes) + 1 logits)
    _bbox_head: Any = field(default=None, repr=False)  # a trained copy of the checkpoint's box head
    classes: list[str] = field(default_factory=list)
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerStructurePipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, TableTransformerForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        # The pinned preprocessor_config.json declares `size: {"longest_edge": 800}`, a shape the pinned
        # transformers release's DetrImageProcessor refuses ("Size must contain ... 'shortest_edge' and
        # 'longest_edge'"). PROCESSOR_SIZE is the same resize expressed in the accepted form: with both
        # edges capped at 800 the longest edge always lands on 800 px, exactly as the upstream file says.
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, size=dict(PROCESSOR_SIZE), **kwargs
        )
        # This checkpoint's config carries an in-library ResNet backbone_config (use_timm_backbone=False),
        # so nothing is fetched at construction; use_pretrained_backbone=False is passed anyway so the
        # loader can never reach for ImageNet weights the checkpoint already contains.
        model = TableTransformerForObjectDetection.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            use_pretrained_backbone=False,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        id2label = {int(k): v for k, v in model.config.id2label.items()}

        def runner(image: Image.Image, threshold: float) -> list[dict]:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            result = processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": id2label[int(label)],
                    "score": float(score),
                }
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
            ]

        return cls(
            runner,
            resolved_device,
            source="local-snapshot" if kwargs else "hub",
            _model=model,
            _processor=processor,
        )

    def recognize(self, image: Image.Image, *, threshold: float = RECOGNITION_THRESHOLD) -> dict[str, Any]:
        """Recognise the structure of one table-crop image; boxes are xyxy pixel coordinates in the input."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._runner(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in LABELS:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation ----
    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def _require_heads(self) -> tuple[Any, Any, list[str]]:
        if self._head is None or self._bbox_head is None or not self.classes:
            raise ValueError("no adapted heads: call adapt() or load an artifact first")
        return self._head, self._bbox_head, list(self.classes)

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if isinstance(trainable_layers, bool) or not isinstance(trainable_layers, int):
            raise ValueError(f"trainable_layers must be an int in 0..{DECODER_LAYERS}")
        if not 0 <= trainable_layers <= DECODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 0..{DECODER_LAYERS}")
        if trainable_layers == 0:
            return []
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_layers
        prefixes = tuple(f"model.decoder.layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _decoder_features(self, image: Image.Image, *, grad: bool = False) -> Any:
        """The (NUM_QUERIES, D_MODEL) decoder output for one validated image through the processor."""
        import torch

        model, processor = self._require_model()
        inputs = processor(images=image, return_tensors="pt").to(self.device)
        if grad:
            outputs = model.model(pixel_values=inputs["pixel_values"], pixel_mask=inputs.get("pixel_mask"))
        else:
            with torch.no_grad():
                outputs = model.model(
                    pixel_values=inputs["pixel_values"], pixel_mask=inputs.get("pixel_mask")
                )
        hidden = outputs.last_hidden_state[0]
        if tuple(hidden.shape) != (NUM_QUERIES, D_MODEL):
            raise RuntimeError(f"decoder returned {tuple(hidden.shape)}, expected {(NUM_QUERIES, D_MODEL)}")
        return hidden

    @staticmethod
    def _xyxy_pixels(boxes_cxcywh: Any, width: int, height: int) -> list[list[float]]:
        out = []
        for cx, cy, w, h in boxes_cxcywh.tolist():
            out.append(
                [(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height]
            )
        return out

    @staticmethod
    def _expand(xyxy: list[list[float]], probabilities: Any, classes: Sequence[str]) -> list[dict[str, Any]]:
        """One prediction per (query, class): the ranking average precision consumes."""
        probs = probabilities.tolist()
        out = []
        for box, row in zip(xyxy, probs, strict=True):
            for c, label in enumerate(classes):
                out.append({"box": box, "label": label, "score": float(row[c])})
        return out

    def _checkpoint_class_index(self) -> list[int]:
        model, _ = self._require_model()
        id2label = {int(k): v for k, v in model.config.id2label.items()}
        label2id = {v: k for k, v in id2label.items()}
        missing = [label for label in ADAPT_LABELS if label not in label2id]
        if missing:
            raise RuntimeError(f"checkpoint lacks the adaptation labels {missing}")
        return [label2id[label] for label in ADAPT_LABELS]

    def evaluate_zero_shot(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score the untouched checkpoint on validated `{id, image, objects}` records with its own heads:
        every query's softmax probability for each of the four scored labels (headers are neither scored nor
        penalised); per-label AP over all 125 queries per image, the operating point at RECOGNITION_THRESHOLD.
        """
        model, _ = self._require_model()
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .metrics import structure_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        index = self._checkpoint_class_index()
        started = time.perf_counter()
        predictions = []
        for record in checked:
            hidden = self._decoder_features(record["image"])
            with torch.no_grad():
                probabilities = torch.softmax(model.class_labels_classifier(hidden), dim=-1)[:, index]
                boxes = model.bbox_predictor(hidden).sigmoid()
            xyxy = self._xyxy_pixels(boxes, *record["image"].size)
            predictions.append(self._expand(xyxy, probabilities, ADAPT_LABELS))
        metrics = structure_metrics(
            predictions, [r["objects"] for r in checked], labels=ADAPT_LABELS, threshold=RECOGNITION_THRESHOLD
        )
        metrics["policy"] = "zero-shot checkpoint (its own heads, softmax over all seven classes)"
        metrics["adapted"] = False
        metrics["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample"
        metrics["seconds"] = round(time.perf_counter() - started, 3)
        metrics["model_id"] = MODEL_ID
        metrics["model_revision"] = MODEL_REVISION
        return metrics

    def _query_objects(self, image: Image.Image) -> list[dict[str, Any]]:
        import torch

        head, bbox_head, classes = self._require_heads()
        hidden = self._decoder_features(image)
        with torch.no_grad():
            probabilities = torch.softmax(head(hidden), dim=-1)[:, : len(classes)]
            boxes = bbox_head(hidden).sigmoid()
        return self._expand(self._xyxy_pixels(boxes, *image.size), probabilities, classes)

    def predict_objects(self, records: Sequence[Mapping[str, Any]]) -> list[list[dict[str, Any]]]:
        """Every (query, class) score and pixel box of the adapted heads, per validated record."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        self._require_heads()
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        return [self._query_objects(record["image"]) for record in checked]

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score the adapted heads on validated labelled tables: per-label AP@0.5 / AP@0.75 / AP and their
        class means, grid agreement and the operating point at RECOGNITION_THRESHOLD, plus the DETR loss the
        selection used."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import structure_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = self.predict_objects(checked)
        metrics = structure_metrics(
            predictions, [r["objects"] for r in checked], labels=self.classes, threshold=RECOGNITION_THRESHOLD
        )
        metrics["loss"] = self._dataset_loss(checked)
        metrics["policy"] = self.adapter["policy"] if self.adapter else "unknown"
        metrics["adapted"] = True
        metrics["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample"
        metrics["seconds"] = round(time.perf_counter() - started, 3)
        metrics["model_id"] = MODEL_ID
        metrics["model_revision"] = MODEL_REVISION
        return metrics

    def recognize_adapted(
        self, image: Image.Image, *, threshold: float = RECOGNITION_THRESHOLD
    ) -> dict[str, Any]:
        """`recognize` with the adapted heads: each query's arg-max label among the adapted classes when its
        probability reaches `threshold`; boxes in pixels."""
        rgb, checked = _check_inputs(image, threshold)
        head, bbox_head, classes = self._require_heads()
        queries = self._query_objects(rgb)
        detections = []
        for start in range(0, len(queries), len(classes)):
            best = max(queries[start : start + len(classes)], key=lambda q: q["score"])
            if best["score"] >= checked:
                detections.append({"box": best["box"], "label": best["label"], "score": best["score"]})
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "classes": classes,
            "policy": self.adapter["policy"] if self.adapter else "unknown",
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- the DETR set loss, implemented here (no scipy) ----
    @staticmethod
    def _targets(record: Mapping[str, Any], classes: Sequence[str]) -> tuple[Any, Any]:
        """(class indices, cxcywh boxes normalised to the image) for one validated record."""
        import torch

        width, height = record["image"].size
        labels, rows = [], []
        for obj in record["objects"]:
            x0, y0, x1, y1 = obj["box"]
            labels.append(list(classes).index(obj["label"]))
            rows.append(
                [(x0 + x1) / 2 / width, (y0 + y1) / 2 / height, (x1 - x0) / width, (y1 - y0) / height]
            )
        return torch.tensor(labels, dtype=torch.long), torch.tensor(rows, dtype=torch.float32)

    @staticmethod
    def _giou(a: Any, b: Any) -> Any:
        """Generalised IoU between (N, 4) and (M, 4) cxcywh boxes -> (N, M)."""
        import torch

        def xyxy(box):
            return torch.stack(
                [
                    box[:, 0] - box[:, 2] / 2,
                    box[:, 1] - box[:, 3] / 2,
                    box[:, 0] + box[:, 2] / 2,
                    box[:, 1] + box[:, 3] / 2,
                ],
                dim=-1,
            )

        a, b = xyxy(a), xyxy(b)
        area_a = (a[:, 2] - a[:, 0]).clamp_min(0) * (a[:, 3] - a[:, 1]).clamp_min(0)
        area_b = (b[:, 2] - b[:, 0]).clamp_min(0) * (b[:, 3] - b[:, 1]).clamp_min(0)
        lt = torch.max(a[:, None, :2], b[None, :, :2])
        rb = torch.min(a[:, None, 2:], b[None, :, 2:])
        inter = (rb - lt).clamp_min(0).prod(dim=-1)
        union = area_a[:, None] + area_b[None, :] - inter
        iou = inter / union.clamp_min(1e-9)
        lt_c = torch.min(a[:, None, :2], b[None, :, :2])
        rb_c = torch.max(a[:, None, 2:], b[None, :, 2:])
        enclosing = (rb_c - lt_c).clamp_min(0).prod(dim=-1)
        return iou - (enclosing - union) / enclosing.clamp_min(1e-9)

    @staticmethod
    def hungarian(cost: Sequence[Sequence[float]]) -> list[tuple[int, int]]:
        """Exact minimum-cost assignment of every row of a rectangular cost matrix (rows <= columns) to a
        distinct column: the O(n^2 m) shortest-augmenting-path algorithm with dual potentials (Jonker–
        Volgenant style), in plain Python. Returns (row, column) pairs."""
        n = len(cost)
        m = len(cost[0]) if n else 0
        if n == 0:
            return []
        if n > m:
            raise ValueError(f"hungarian needs rows <= columns, got {n} x {m}")
        inf = math.inf
        u = [0.0] * (n + 1)
        v = [0.0] * (m + 1)
        way = [0] * (m + 1)  # for each column, the column it was reached from on the augmenting path
        assigned = [0] * (m + 1)  # column -> row (1-based), 0 = free
        for i in range(1, n + 1):
            assigned[0] = i
            j0 = 0
            minv = [inf] * (m + 1)
            used = [False] * (m + 1)
            while True:
                used[j0] = True
                i0 = assigned[j0]
                delta, j1 = inf, 0
                row = cost[i0 - 1]
                for j in range(1, m + 1):
                    if used[j]:
                        continue
                    cur = row[j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur
                        way[j] = j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
                for j in range(m + 1):
                    if used[j]:
                        u[assigned[j]] += delta
                        v[j] -= delta
                    else:
                        minv[j] -= delta
                j0 = j1
                if assigned[j0] == 0:
                    break
            while j0:
                j1 = way[j0]
                assigned[j0] = assigned[j1]
                j0 = j1
        return sorted((assigned[j] - 1, j - 1) for j in range(1, m + 1) if assigned[j])

    def _match(
        self, probabilities: Any, boxes: Any, target_classes: Any, target_boxes: Any
    ) -> list[tuple[int, int]]:
        """Minimum-cost one-to-one assignment of reference objects to queries (exact Hungarian)."""
        import torch

        with torch.no_grad():
            cost = -CLASS_COST * probabilities[:, target_classes]
            cost = (
                cost
                + BBOX_COST * torch.cdist(boxes, target_boxes, p=1)
                - GIOU_COST * self._giou(boxes, target_boxes)
            )
            matrix = cost.t().cpu().tolist()  # targets x queries
        return [(q, g) for g, q in self.hungarian(matrix)]

    def _loss(self, logits: Any, boxes: Any, target_classes: Any, target_boxes: Any) -> Any:
        """DETR set loss for one image: weighted cross-entropy over queries (no-object weight 0.1), L1 and
        GIoU on the matched boxes, normalised by the number of reference objects."""
        import torch

        device = logits.device
        target_classes = target_classes.to(device)
        target_boxes = target_boxes.to(device)
        probabilities = torch.softmax(logits, dim=-1)
        pairs = self._match(probabilities.detach(), boxes.detach(), target_classes, target_boxes)
        n_classes = logits.shape[-1] - 1
        assigned = torch.full((NUM_QUERIES,), n_classes, dtype=torch.long, device=device)
        query_index = torch.tensor([q for q, _g in pairs], dtype=torch.long, device=device)
        target_index = torch.tensor([g for _q, g in pairs], dtype=torch.long, device=device)
        assigned[query_index] = target_classes[target_index]
        weights = torch.ones(n_classes + 1, device=device)
        weights[n_classes] = NO_OBJECT_WEIGHT
        loss_ce = torch.nn.functional.cross_entropy(logits, assigned, weight=weights)
        matched_boxes = boxes[query_index]
        matched_targets = target_boxes[target_index]
        loss_bbox = torch.nn.functional.l1_loss(matched_boxes, matched_targets, reduction="sum") / max(
            len(pairs), 1
        )
        loss_giou = (1.0 - torch.diagonal(self._giou(matched_boxes, matched_targets))).sum() / max(
            len(pairs), 1
        )
        return loss_ce + BBOX_COST * loss_bbox + GIOU_COST * loss_giou

    def _dataset_loss(self, checked: Sequence[Mapping[str, Any]]) -> float:
        import torch

        head, bbox_head, classes = self._require_heads()
        total = 0.0
        for record in checked:
            hidden = self._decoder_features(record["image"])
            with torch.no_grad():
                total += float(
                    self._loss(head(hidden), bbox_head(hidden).sigmoid(), *self._targets(record, classes))
                )
        return total / max(len(checked), 1)

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        head_steps: int = 300,
        head_lr: float = 1e-3,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        epochs: int = 3,
        lr: float = 1e-4,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Three-policy bounded adaptation to the four scored labels, selected on validation.

        Epoch 0 (the **zero-shot policy**): a restricted `Linear(D_MODEL, 5)` class head whose rows are
        copied from the checkpoint's own `table` / `table column` / `table row` / `table spanning cell` /
        no-object rows, and a copy of the checkpoint's box head, scored on `val` untouched. Stage A (the
        **frozen policy**, epoch 1): both trained on the cached decoder features of the training tables with
        AdamW (`head_lr`, weight decay 1e-4) for `head_steps` full-batch steps under the DETR set loss (exact
        Hungarian matching). Stage B (the **unfrozen policy**, epochs 2.., when `trainable_layers` > 0 and
        `epochs` > 0): the last `trainable_layers` decoder layers trained with both heads end to end, one
        table per step, for `epochs` epochs (AdamW at `lr`, weight decay 0.01, gradient clipping 0.1, seeded
        order, no augmentation); the backbone, the input projection, the encoder, the query embeddings, the
        earlier decoder layers and the checkpoint's own heads stay untouched. Every epoch is scored on `val`
        and the epoch with the **lowest validation DETR loss** is kept (epoch 0 competes, so the selected
        policy can be the checkpoint itself) and its tensors restored; without `val` the final epoch is kept.
        """
        if isinstance(head_steps, bool) or not isinstance(head_steps, int) or not 1 <= head_steps <= 5_000:
            raise ValueError("head_steps must be an int in 1..5000")
        if not isinstance(head_lr, int | float) or not 0.0 < float(head_lr) <= 1e-1:
            raise ValueError("head_lr must be in (0, 0.1]")
        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 0 <= epochs <= 20:
            raise ValueError("epochs must be an int in 0..20")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        names = self._trainable_names(trainable_layers)
        model, _ = self._require_model()
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_records = validate_dataset(train)["records"]
        val_records = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
            if val is not None
            else None
        )
        classes = list(ADAPT_LABELS)
        index = self._checkpoint_class_index()
        started = time.perf_counter()
        torch.manual_seed(seed)
        rng = random.Random(seed)
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)

        # Epoch 0: the restricted heads copied from the checkpoint's rows, scored untouched.
        cached = [self._decoder_features(r["image"]) for r in train_records]
        targets = [self._targets(r, classes) for r in train_records]
        head = torch.nn.Linear(D_MODEL, len(classes) + 1).to(self.device)
        with torch.no_grad():
            source = model.class_labels_classifier
            rows = [*index, source.weight.shape[0] - 1]  # the checkpoint's no-object row is its last
            head.weight.copy_(source.weight[rows])
            head.bias.copy_(source.bias[rows])
        bbox_head = copy.deepcopy(model.bbox_predictor).to(self.device)
        for p in bbox_head.parameters():
            p.requires_grad_(True)
        self._head, self._bbox_head, self.classes = head, bbox_head, classes
        self.adapter = {"policy": POLICY_ZERO_SHOT}

        def brief(metrics: Mapping[str, Any] | None) -> dict[str, float] | None:
            if metrics is None:
                return None
            return {
                "n": metrics["n_images"],
                "loss": round(metrics["loss"], 6),
                "map50": round(metrics["map50"], 6),
                "map": round(metrics["map"], 6),
                "grid_exact": round(metrics["grid_exact_at_threshold"], 6),
            }

        def score() -> dict[str, float] | None:
            head.eval()
            bbox_head.eval()
            model.eval()
            return brief(self.evaluate(val_records)) if val_records is not None else None

        def snapshot() -> dict[str, Any]:
            return {
                "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
                "bbox_head": {k: v.detach().clone() for k, v in bbox_head.state_dict().items()},
                "layers": {n: params[n].detach().clone() for n in names},
            }

        params = dict(model.named_parameters())
        history: list[dict[str, Any]] = [
            {"epoch": 0, "stage": POLICY_ZERO_SHOT, "train_loss": None, "val": score()}
        ]
        best_epoch, policy = 0, POLICY_ZERO_SHOT
        best_score = history[0]["val"]["loss"] if history[0]["val"] else math.inf
        best_state = snapshot()

        # Stage A: the restricted heads trained on the cached features (epoch 1).
        head_opt = torch.optim.AdamW(
            [*head.parameters(), *bbox_head.parameters()], lr=float(head_lr), weight_decay=1e-4
        )
        head_loss = math.nan
        for _ in range(head_steps):
            head_opt.zero_grad(set_to_none=True)
            loss = sum(
                self._loss(head(h), bbox_head(h).sigmoid(), *tg)
                for h, tg in zip(cached, targets, strict=True)
            ) / len(cached)
            loss.backward()
            head_opt.step()
            head_loss = float(loss.detach())
        self.adapter = {"policy": POLICY_FROZEN}
        val_metrics = score()
        history.append({"epoch": 1, "stage": POLICY_FROZEN, "train_loss": head_loss, "val": val_metrics})
        if progress is not None:
            progress(history[-1])
        current = val_metrics["loss"] if val_metrics else -1
        if current < best_score:
            best_epoch, best_score, policy = 1, current, POLICY_FROZEN
            best_state = snapshot()
        if names and epochs > 0:
            policy_b = f"unfrozen last {trainable_layers} decoder layers + restricted heads"
            for n in names:
                params[n].requires_grad_(True)
            optimiser = torch.optim.AdamW(
                [*head.parameters(), *bbox_head.parameters(), *(params[n] for n in names)],
                lr=float(lr),
                weight_decay=0.01,
            )
            for epoch in range(2, epochs + 2):
                model.train()
                head.train()
                bbox_head.train()
                order = list(range(len(train_records)))
                rng.shuffle(order)
                losses = []
                for i in order:
                    hidden = self._decoder_features(train_records[i]["image"], grad=True)
                    loss = self._loss(head(hidden), bbox_head(hidden).sigmoid(), *targets[i])
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        [*head.parameters(), *bbox_head.parameters(), *(params[n] for n in names)], 0.1
                    )
                    optimiser.step()
                    losses.append(float(loss.detach()))
                self.adapter = {"policy": policy_b}
                val_metrics = score()
                entry = {
                    "epoch": epoch,
                    "stage": policy_b,
                    "train_loss": float(sum(losses) / len(losses)),
                    "val": val_metrics,
                }
                history.append(entry)
                if progress is not None:
                    progress(entry)
                current = val_metrics["loss"] if val_metrics else -epoch  # no val: the last epoch wins
                if current < best_score:
                    best_epoch, best_score, policy = epoch, current, policy_b
                    best_state = snapshot()
        with torch.no_grad():
            head.load_state_dict(best_state["head"])
            bbox_head.load_state_dict(best_state["bbox_head"])
            for n, value in best_state["layers"].items():
                params[n].copy_(value)
        for p in model.parameters():
            p.requires_grad_(False)
        for p in [*head.parameters(), *bbox_head.parameters()]:
            p.requires_grad_(False)
        model.eval()
        head.eval()
        bbox_head.eval()
        self.adapter = {
            "policy": policy,
            "classes": classes,
            "head_steps": head_steps,
            "head_lr": float(head_lr),
            "head_final_loss": head_loss,
            "trainable_layers": trainable_layers,
            "n_trainable_head": sum(p.numel() for p in head.parameters())
            + sum(p.numel() for p in bbox_head.parameters()),
            "n_trainable_layers": sum(params[n].numel() for n in names),
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": (
                "lowest validation DETR loss (epoch 0 = the checkpoint's own rows untrained, epoch 1 = the "
                "restricted heads trained on frozen features, later epochs = the unfreeze)"
                if val is not None
                else "final epoch (no validation split)"
            ),
            "lr": float(lr),
            "n_train": len(train_records),
            "n_val": len(val_records) if val_records is not None else 0,
            "seed": seed,
            "history": history,
            "trainable_names": names if policy not in (POLICY_ZERO_SHOT, POLICY_FROZEN) else [],
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the restricted heads (and any trained decoder-layer tensors) as safetensors with a base
        manifest."""
        if self.adapter is None or self._head is None or self._bbox_head is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter.get("trainable_names", []))
        tensors = {f"head.{k}": v.detach().cpu().contiguous() for k, v in self._head.state_dict().items()}
        tensors.update(
            {f"bbox_head.{k}": v.detach().cpu().contiguous() for k, v in self._bbox_head.state_dict().items()}
        )
        tensors.update(
            {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        )
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHTS_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter.get("history", []),
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest **before** deserialising, rebuild the heads and overlay any
        decoder-layer tensors onto the base."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        classes = list(manifest.get("adapter", {}).get("classes") or [])
        if tuple(classes) != ADAPT_LABELS:
            raise ValueError(f"artifact classes {classes} are not the adaptation labels {list(ADAPT_LABELS)}")
        model, _ = self._require_model()
        import torch
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        if (
            tuple(tensors.get("head.weight", torch.empty(0)).shape) != (len(classes) + 1, D_MODEL)
            or "head.bias" not in tensors
        ):
            raise ValueError("artifact class head does not match D_MODEL and the manifest's classes")
        bbox_head = copy.deepcopy(model.bbox_predictor)
        bbox_state = {k[len("bbox_head.") :]: v for k, v in tensors.items() if k.startswith("bbox_head.")}
        if set(bbox_state) != set(bbox_head.state_dict()):
            raise ValueError("artifact box head tensors do not match the checkpoint's box head")
        state = model.state_dict()
        layer_tensors = {k: v for k, v in tensors.items() if not k.startswith(("head.", "bbox_head."))}
        for key, value in layer_tensors.items():
            if key not in state or not key.startswith("model.decoder.layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        head = torch.nn.Linear(D_MODEL, len(classes) + 1)
        head.load_state_dict({"weight": tensors["head.weight"].float(), "bias": tensors["head.bias"].float()})
        bbox_head.load_state_dict({k: v.float() for k, v in bbox_state.items()})
        head = head.to(self.device).eval()
        bbox_head = bbox_head.to(self.device).eval()
        for p in [*head.parameters(), *bbox_head.parameters()]:
            p.requires_grad_(False)
        if layer_tensors:
            with torch.no_grad():
                params = dict(model.named_parameters())
                for key, value in layer_tensors.items():
                    params[key].copy_(value.to(params[key].dtype))
            model.eval()
        self._head, self._bbox_head, self.classes = head, bbox_head, classes
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": sorted(layer_tensors),
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerStructurePipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/4:** `src/table_transformer_structure_pipeline/sample_data.py` (carried verbatim; see the note above)

In [ ]:
"""Embedded sample tables for the Table Transformer structure tutorial: 94 public-domain SciTSR tables.

Source: the SciTSR-PD subset of SciTSR (Chi et al. 2019, arXiv LaTeX tables rendered at 150 DPI), the
tables whose source papers carry a CC0 or the older Creative Commons public-domain dedication, as
published on the Hugging Face Hub as `bevaya/SciTSR-pd` (licence tag cc0-1.0) at commit
`dae336efa7af07d69194a510ff5568980e8ef253` — two parquet files pinned by size and SHA-256 in
`SOURCE_FILES`. Each PNG is the dataset's own render (mode L), re-encoded losslessly (pixels asserted
identical) and embedded here so the standalone notebook needs no parquet reader or download.

Structure boxes were **derived once** by the build script recorded in docs/WEIGHTS.md from the dataset's
text chunks (PDF points) and logical cells (row / column spans), never edited by hand:
1. the chunk boxes are placed on the trimmed image by a whole-pixel offset search (scale 150/72) that
   maximises ink coverage inside them (`alignment` records the offset and the coverage score);
2. chunks are matched to logical cells by normalised text (exact, then repeated texts in reading order,
   then multi-chunk continuations; the conversion's doubled final character of each table's last chunk
   is undone); a table whose cells cannot all be matched, whose rows or columns overlap after matching,
   or whose grid has a row or column without a single-span text cell is dropped (14 of 108);
3. the `table` box is the ink bounding box around the cell union (expanded by 1.5 row / column pitches,
   so rules are included and captions are not); `table row` and `table column` boxes tile it with
   boundaries at the mid-gaps between adjacent rows / columns (the PubTables-1M convention); each cell
   spanning more than one row or column is a `table spanning cell` box over its grid area.
Column headers and projected row headers are not annotated in SciTSR and are therefore absent from the
sample; the adaptation contract restricts its vocabulary to the four labels above.

Do not edit by hand: the dataset digest in the tutorial depends on these bytes.
"""

from __future__ import annotations

SAMPLE_SOURCE = "SciTSR-PD (bevaya/SciTSR-pd @ dae336ef, CC0 / public-domain source papers), structure boxes derived 2026-09-19"
SAMPLE_LICENSE = "CC0-1.0"
SAMPLE_DPI = 150
SAMPLE_LABELS = ("table", "table column", "table row", "table spanning cell")
SOURCE_FILES: tuple[dict[str, str | int], ...] = (
    {'split': 'train', 'path': 'data/train-00000-of-00001.parquet', 'bytes': 5185141, 'sha256': 'd295b918777425a26fa13a5c3ba076517429682ede8c71b671a8b6115968e465'},
    {'split': 'test', 'path': 'data/test-00000-of-00001.parquet', 'bytes': 1066042, 'sha256': '73af6317219349a08acdf1c8fee7cff21b463cdded64111b643123f7df4f8bed'},
)

# One entry per table: the embedded file, its SciTSR identity and provenance, and the derived structure boxes
# (xyxy pixels in the embedded image).
SAMPLE_RECORDS: tuple[dict, ...] = (
    {'file': '1003.3684v1.1.png', 'table_id': '1003.3684v1.1', 'paper_id': '1003.3684', 'paper_title': 'Parallel Generation of Massive Scale-Free Graphs', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 566, 'height': 87, 'png_bytes': 4648, 'png_sha256': 'fc1b2a3f4b28245819ac734109fc9345b626bcbb85859daf60f3f0a6bb9a2ff2', 'n_rows': 3, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 20, 'dy': 7, 'ink_score': 0.265}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 563.0, 84.0]}, {'label': 'table row', 'box': [3.0, 3.0, 563.0, 30.9]}, {'label': 'table row', 'box': [3.0, 30.9, 563.0, 58.3]}, {'label': 'table row', 'box': [3.0, 58.3, 563.0, 84.0]}, {'label': 'table column', 'box': [3.0, 3.0, 113.1, 84.0]}, {'label': 'table column', 'box': [113.1, 3.0, 255.3, 84.0]}, {'label': 'table column', 'box': [255.3, 3.0, 391.0, 84.0]}, {'label': 'table column', 'box': [391.0, 3.0, 563.0, 84.0]}]},
    {'file': '1003.3684v1.2.png', 'table_id': '1003.3684v1.2', 'paper_id': '1003.3684', 'paper_title': 'Parallel Generation of Massive Scale-Free Graphs', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 580, 'height': 145, 'png_bytes': 6653, 'png_sha256': 'ebebcc6caad18b8214482767f9f6ac3a70e79df99b5a3da844f9ac8ea7e04f52', 'n_rows': 5, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 20, 'dy': 15, 'ink_score': 0.285}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 577.0, 142.0]}, {'label': 'table row', 'box': [3.0, 3.0, 577.0, 35.1]}, {'label': 'table row', 'box': [3.0, 35.1, 577.0, 62.5]}, {'label': 'table row', 'box': [3.0, 62.5, 577.0, 87.4]}, {'label': 'table row', 'box': [3.0, 87.4, 577.0, 112.3]}, {'label': 'table row', 'box': [3.0, 112.3, 577.0, 142.0]}, {'label': 'table column', 'box': [3.0, 3.0, 161.2, 142.0]}, {'label': 'table column', 'box': [161.2, 3.0, 352.2, 142.0]}, {'label': 'table column', 'box': [352.2, 3.0, 577.0, 142.0]}]},
    {'file': '1004.5186v1.1.png', 'table_id': '1004.5186v1.1', 'paper_id': '1004.5186', 'paper_title': 'Multiscale approach for the network compression-friendly ordering', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 429, 'height': 158, 'png_bytes': 7806, 'png_sha256': '52691d132f9210df9d083c568dad824d4f18f1c42864f7fef78826984bbc2729', 'n_rows': 6, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.273}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 426.0, 155.0]}, {'label': 'table row', 'box': [3.0, 3.0, 426.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 426.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 426.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 426.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 426.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 426.0, 155.0]}, {'label': 'table column', 'box': [3.0, 3.0, 184.2, 155.0]}, {'label': 'table column', 'box': [184.2, 3.0, 283.6, 155.0]}, {'label': 'table column', 'box': [283.6, 3.0, 426.0, 155.0]}]},
    {'file': '1007.0920v1.1.png', 'table_id': '1007.0920v1.1', 'paper_id': '1007.0920', 'paper_title': 'End-Host Distribution in Application-Layer Multicast: Main Issues and Solutions', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 546, 'height': 216, 'png_bytes': 8362, 'png_sha256': '046e7dc23337686d987ff71680ee14243ffc589161cd8880456327ffb2df5305', 'n_rows': 8, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.318}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 543.0, 213.0]}, {'label': 'table row', 'box': [3.0, 3.0, 543.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 543.0, 58.5]}, {'label': 'table row', 'box': [3.0, 58.5, 543.0, 83.4]}, {'label': 'table row', 'box': [3.0, 83.4, 543.0, 108.3]}, {'label': 'table row', 'box': [3.0, 108.3, 543.0, 133.3]}, {'label': 'table row', 'box': [3.0, 133.3, 543.0, 158.2]}, {'label': 'table row', 'box': [3.0, 158.2, 543.0, 183.1]}, {'label': 'table row', 'box': [3.0, 183.1, 543.0, 213.0]}, {'label': 'table column', 'box': [3.0, 3.0, 115.4, 213.0]}, {'label': 'table column', 'box': [115.4, 3.0, 263.6, 213.0]}, {'label': 'table column', 'box': [263.6, 3.0, 396.7, 213.0]}, {'label': 'table column', 'box': [396.7, 3.0, 543.0, 213.0]}]},
    {'file': '1007.0920v1.2.png', 'table_id': '1007.0920v1.2', 'paper_id': '1007.0920', 'paper_title': 'End-Host Distribution in Application-Layer Multicast: Main Issues and Solutions', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 552, 'height': 341, 'png_bytes': 12236, 'png_sha256': 'fdb1160c26ca6c50067d46f9c234fc3b09a6d6c5a4a59a793da120c4a1862727', 'n_rows': 13, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.313}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 549.0, 338.0]}, {'label': 'table row', 'box': [3.0, 3.0, 549.0, 32.1]}, {'label': 'table row', 'box': [3.0, 32.1, 549.0, 59.5]}, {'label': 'table row', 'box': [3.0, 59.5, 549.0, 84.4]}, {'label': 'table row', 'box': [3.0, 84.4, 549.0, 109.3]}, {'label': 'table row', 'box': [3.0, 109.3, 549.0, 134.3]}, {'label': 'table row', 'box': [3.0, 134.3, 549.0, 159.2]}, {'label': 'table row', 'box': [3.0, 159.2, 549.0, 184.1]}, {'label': 'table row', 'box': [3.0, 184.1, 549.0, 209.0]}, {'label': 'table row', 'box': [3.0, 209.0, 549.0, 233.9]}, {'label': 'table row', 'box': [3.0, 233.9, 549.0, 258.8]}, {'label': 'table row', 'box': [3.0, 258.8, 549.0, 283.7]}, {'label': 'table row', 'box': [3.0, 283.7, 549.0, 308.6]}, {'label': 'table row', 'box': [3.0, 308.6, 549.0, 338.0]}, {'label': 'table column', 'box': [3.0, 3.0, 118.4, 338.0]}, {'label': 'table column', 'box': [118.4, 3.0, 266.5, 338.0]}, {'label': 'table column', 'box': [266.5, 3.0, 402.7, 338.0]}, {'label': 'table column', 'box': [402.7, 3.0, 549.0, 338.0]}]},
    {'file': '1108.4723v1.1.png', 'table_id': '1108.4723v1.1', 'paper_id': '1108.4723', 'paper_title': 'Self-Optimized OFDMA via Multiple Stackelberg Leader Equilibrium', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 644, 'height': 140, 'png_bytes': 6749, 'png_sha256': 'a08266f63d11d6a44ab1a34f21bda8df4a2a064f7fa6769250fe4709fe20258c', 'n_rows': 5, 'n_cols': 7, 'n_spanning': 3, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.272}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 641.0, 137.0]}, {'label': 'table row', 'box': [3.0, 3.0, 641.0, 32.1]}, {'label': 'table row', 'box': [3.0, 32.1, 641.0, 59.9]}, {'label': 'table row', 'box': [3.0, 59.9, 641.0, 87.2]}, {'label': 'table row', 'box': [3.0, 87.2, 641.0, 113.0]}, {'label': 'table row', 'box': [3.0, 113.0, 641.0, 137.0]}, {'label': 'table column', 'box': [3.0, 3.0, 138.7, 137.0]}, {'label': 'table column', 'box': [138.7, 3.0, 224.1, 137.0]}, {'label': 'table column', 'box': [224.1, 3.0, 307.4, 137.0]}, {'label': 'table column', 'box': [307.4, 3.0, 390.7, 137.0]}, {'label': 'table column', 'box': [390.7, 3.0, 474.0, 137.0]}, {'label': 'table column', 'box': [474.0, 3.0, 557.3, 137.0]}, {'label': 'table column', 'box': [557.3, 3.0, 641.0, 137.0]}, {'label': 'table spanning cell', 'box': [474.0, 3.0, 641.0, 32.1]}, {'label': 'table spanning cell', 'box': [138.7, 3.0, 307.4, 32.1]}, {'label': 'table spanning cell', 'box': [307.4, 3.0, 474.0, 32.1]}]},
    {'file': '1108.4723v1.2.png', 'table_id': '1108.4723v1.2', 'paper_id': '1108.4723', 'paper_title': 'Self-Optimized OFDMA via Multiple Stackelberg Leader Equilibrium', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 709, 'height': 165, 'png_bytes': 7831, 'png_sha256': 'e3673313fcef16441cde144a504fa91c63952c2849a02c4e048edd271330b7c9', 'n_rows': 6, 'n_cols': 10, 'n_spanning': 3, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.272}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 706.0, 162.0]}, {'label': 'table row', 'box': [3.0, 3.0, 706.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 706.0, 56.9]}, {'label': 'table row', 'box': [3.0, 56.9, 706.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 706.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 706.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 706.0, 162.0]}, {'label': 'table column', 'box': [3.0, 3.0, 138.7, 162.0]}, {'label': 'table column', 'box': [138.7, 3.0, 195.4, 162.0]}, {'label': 'table column', 'box': [195.4, 3.0, 262.1, 162.0]}, {'label': 'table column', 'box': [262.1, 3.0, 328.2, 162.0]}, {'label': 'table column', 'box': [328.2, 3.0, 384.2, 162.0]}, {'label': 'table column', 'box': [384.2, 3.0, 450.9, 162.0]}, {'label': 'table column', 'box': [450.9, 3.0, 517.1, 162.0]}, {'label': 'table column', 'box': [517.1, 3.0, 573.1, 162.0]}, {'label': 'table column', 'box': [573.1, 3.0, 639.8, 162.0]}, {'label': 'table column', 'box': [639.8, 3.0, 706.0, 162.0]}, {'label': 'table spanning cell', 'box': [328.2, 3.0, 517.1, 29.1]}, {'label': 'table spanning cell', 'box': [138.7, 3.0, 328.2, 29.1]}, {'label': 'table spanning cell', 'box': [517.1, 3.0, 706.0, 29.1]}]},
    {'file': '1108.4723v1.3.png', 'table_id': '1108.4723v1.3', 'paper_id': '1108.4723', 'paper_title': 'Self-Optimized OFDMA via Multiple Stackelberg Leader Equilibrium', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 398, 'height': 114, 'png_bytes': 5150, 'png_sha256': '56f7c9d52031ecd53220df6634497a019a2d9e0ef00c2a7cbef22d6f9dc0084e', 'n_rows': 4, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.27}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 395.0, 111.0]}, {'label': 'table row', 'box': [3.0, 3.0, 395.0, 32.1]}, {'label': 'table row', 'box': [3.0, 32.1, 395.0, 59.9]}, {'label': 'table row', 'box': [3.0, 59.9, 395.0, 85.7]}, {'label': 'table row', 'box': [3.0, 85.7, 395.0, 111.0]}, {'label': 'table column', 'box': [3.0, 3.0, 113.6, 111.0]}, {'label': 'table column', 'box': [113.6, 3.0, 208.6, 111.0]}, {'label': 'table column', 'box': [208.6, 3.0, 301.6, 111.0]}, {'label': 'table column', 'box': [301.6, 3.0, 395.0, 111.0]}]},
    {'file': '1108.4723v1.4.png', 'table_id': '1108.4723v1.4', 'paper_id': '1108.4723', 'paper_title': 'Self-Optimized OFDMA via Multiple Stackelberg Leader Equilibrium', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'test', 'width': 377, 'height': 89, 'png_bytes': 4456, 'png_sha256': '78303ce9391e253419d4a334a3ebaebc47fd83729f96acbe911f640cf0a3fc84', 'n_rows': 3, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.257}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 374.0, 86.0]}, {'label': 'table row', 'box': [3.0, 3.0, 374.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 374.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 374.0, 86.0]}, {'label': 'table column', 'box': [3.0, 3.0, 92.8, 86.0]}, {'label': 'table column', 'box': [92.8, 3.0, 187.8, 86.0]}, {'label': 'table column', 'box': [187.8, 3.0, 280.7, 86.0]}, {'label': 'table column', 'box': [280.7, 3.0, 374.0, 86.0]}]},
    {'file': '1109.4653v2.11.png', 'table_id': '1109.4653v2.11', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 404, 'height': 187, 'png_bytes': 8722, 'png_sha256': '97bb011b03f0391293487c02aad01c764949daa364ea95b75d5ec4e895223f62', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.236}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 401.0, 184.0]}, {'label': 'table row', 'box': [3.0, 3.0, 401.0, 30.8]}, {'label': 'table row', 'box': [3.0, 30.8, 401.0, 56.6]}, {'label': 'table row', 'box': [3.0, 56.6, 401.0, 81.5]}, {'label': 'table row', 'box': [3.0, 81.5, 401.0, 106.4]}, {'label': 'table row', 'box': [3.0, 106.4, 401.0, 131.4]}, {'label': 'table row', 'box': [3.0, 131.4, 401.0, 156.3]}, {'label': 'table row', 'box': [3.0, 156.3, 401.0, 184.0]}, {'label': 'table column', 'box': [3.0, 3.0, 232.7, 184.0]}, {'label': 'table column', 'box': [232.7, 3.0, 317.4, 184.0]}, {'label': 'table column', 'box': [317.4, 3.0, 401.0, 184.0]}]},
    {'file': '1109.4653v2.12.png', 'table_id': '1109.4653v2.12', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 435, 'height': 162, 'png_bytes': 7432, 'png_sha256': '15198d07eca02b76f57e0ecd7903ad2ed5418b0eb4110c952c62e5b243203c60', 'n_rows': 6, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.222}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 432.0, 159.0]}, {'label': 'table row', 'box': [3.0, 3.0, 432.0, 30.8]}, {'label': 'table row', 'box': [3.0, 30.8, 432.0, 56.6]}, {'label': 'table row', 'box': [3.0, 56.6, 432.0, 81.5]}, {'label': 'table row', 'box': [3.0, 81.5, 432.0, 106.4]}, {'label': 'table row', 'box': [3.0, 106.4, 432.0, 131.4]}, {'label': 'table row', 'box': [3.0, 131.4, 432.0, 159.0]}, {'label': 'table column', 'box': [3.0, 3.0, 346.5, 159.0]}, {'label': 'table column', 'box': [346.5, 3.0, 432.0, 159.0]}]},
    {'file': '1109.4653v2.2.png', 'table_id': '1109.4653v2.2', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'test', 'width': 374, 'height': 208, 'png_bytes': 7044, 'png_sha256': 'bd1e508e0397517e33158e4a0378c681a75cba6e9f0dfe3c6240c1fe08a1d9ee', 'n_rows': 8, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.289}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 371.0, 205.0]}, {'label': 'table row', 'box': [3.0, 3.0, 371.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 371.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 371.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 371.0, 102.3]}, {'label': 'table row', 'box': [3.0, 102.3, 371.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 371.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 371.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 371.0, 205.0]}, {'label': 'table column', 'box': [3.0, 3.0, 143.1, 205.0]}, {'label': 'table column', 'box': [143.1, 3.0, 371.0, 205.0]}]},
    {'file': '1109.4653v2.6.png', 'table_id': '1109.4653v2.6', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'test', 'width': 321, 'height': 438, 'png_bytes': 13187, 'png_sha256': '45c0382cbad0a8f18b25a0208d77045be0414a538d3109fcf86986316be7b393', 'n_rows': 17, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.278}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 318.0, 435.0]}, {'label': 'table row', 'box': [3.0, 3.0, 318.0, 30.6]}, {'label': 'table row', 'box': [3.0, 30.6, 318.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 318.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 318.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 318.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 318.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 318.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 318.0, 206.3]}, {'label': 'table row', 'box': [3.0, 206.3, 318.0, 235.7]}, {'label': 'table row', 'box': [3.0, 235.7, 318.0, 261.0]}, {'label': 'table row', 'box': [3.0, 261.0, 318.0, 285.9]}, {'label': 'table row', 'box': [3.0, 285.9, 318.0, 310.8]}, {'label': 'table row', 'box': [3.0, 310.8, 318.0, 335.7]}, {'label': 'table row', 'box': [3.0, 335.7, 318.0, 360.6]}, {'label': 'table row', 'box': [3.0, 360.6, 318.0, 385.5]}, {'label': 'table row', 'box': [3.0, 385.5, 318.0, 410.4]}, {'label': 'table row', 'box': [3.0, 410.4, 318.0, 435.0]}, {'label': 'table column', 'box': [3.0, 3.0, 143.1, 435.0]}, {'label': 'table column', 'box': [143.1, 3.0, 234.6, 435.0]}, {'label': 'table column', 'box': [234.6, 3.0, 318.0, 435.0]}]},
    {'file': '1109.4653v2.8.png', 'table_id': '1109.4653v2.8', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 444, 'height': 187, 'png_bytes': 8557, 'png_sha256': '315dabf6322d2337c86ee0eec06834e55e344d72069ac9a03fc03f2723c504ba', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.242}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 441.0, 184.0]}, {'label': 'table row', 'box': [3.0, 3.0, 441.0, 30.8]}, {'label': 'table row', 'box': [3.0, 30.8, 441.0, 56.6]}, {'label': 'table row', 'box': [3.0, 56.6, 441.0, 81.5]}, {'label': 'table row', 'box': [3.0, 81.5, 441.0, 106.4]}, {'label': 'table row', 'box': [3.0, 106.4, 441.0, 131.4]}, {'label': 'table row', 'box': [3.0, 131.4, 441.0, 156.3]}, {'label': 'table row', 'box': [3.0, 156.3, 441.0, 184.0]}, {'label': 'table column', 'box': [3.0, 3.0, 273.5, 184.0]}, {'label': 'table column', 'box': [273.5, 3.0, 358.2, 184.0]}, {'label': 'table column', 'box': [358.2, 3.0, 441.0, 184.0]}]},
    {'file': '1109.4653v2.9.png', 'table_id': '1109.4653v2.9', 'paper_id': '1109.4653', 'paper_title': 'Can the evolution of music be analyzed in a quantitative manner?', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 467, 'height': 162, 'png_bytes': 7286, 'png_sha256': '2d441926d0418fac040fdae13072c27f099e2dcf973abee47079ca0a37de6a07', 'n_rows': 6, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 17, 'dy': 11, 'ink_score': 0.228}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 464.0, 159.0]}, {'label': 'table row', 'box': [3.0, 3.0, 464.0, 30.8]}, {'label': 'table row', 'box': [3.0, 30.8, 464.0, 56.6]}, {'label': 'table row', 'box': [3.0, 56.6, 464.0, 81.5]}, {'label': 'table row', 'box': [3.0, 81.5, 464.0, 106.4]}, {'label': 'table row', 'box': [3.0, 106.4, 464.0, 131.4]}, {'label': 'table row', 'box': [3.0, 131.4, 464.0, 159.0]}, {'label': 'table column', 'box': [3.0, 3.0, 379.4, 159.0]}, {'label': 'table column', 'box': [379.4, 3.0, 464.0, 159.0]}]},
    {'file': '1301.0302v2.1.png', 'table_id': '1301.0302v2.1', 'paper_id': '1301.0302', 'paper_title': 'MANCaLog: A Logic for Multi-Attribute Network Cascades (Technical Report)', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 961, 'height': 217, 'png_bytes': 13708, 'png_sha256': 'ff2f56ba3717e920216831b322a8f67ed7a97e2b1ce8fef46322f149c2bbcb10', 'n_rows': 8, 'n_cols': 6, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 17, 'dy': 11, 'ink_score': 0.272}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 958.0, 214.0]}, {'label': 'table row', 'box': [3.0, 3.0, 958.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 958.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 958.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 958.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 958.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 958.0, 161.9]}, {'label': 'table row', 'box': [3.0, 161.9, 958.0, 187.6]}, {'label': 'table row', 'box': [3.0, 187.6, 958.0, 214.0]}, {'label': 'table column', 'box': [3.0, 3.0, 415.2, 214.0]}, {'label': 'table column', 'box': [415.2, 3.0, 549.4, 214.0]}, {'label': 'table column', 'box': [549.4, 3.0, 654.2, 214.0]}, {'label': 'table column', 'box': [654.2, 3.0, 748.9, 214.0]}, {'label': 'table column', 'box': [748.9, 3.0, 843.5, 214.0]}, {'label': 'table column', 'box': [843.5, 3.0, 958.0, 214.0]}]},
    {'file': '1305.1199v4.1.png', 'table_id': '1305.1199v4.1', 'paper_id': '1305.1199', 'paper_title': 'How to find real-world applications for compressive sensing', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 965, 'height': 231, 'png_bytes': 12960, 'png_sha256': 'e62172085e0277964b58df4e5654b5d7bb03c18ce509f9b600d168244fa30889', 'n_rows': 7, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 23, 'dy': 16, 'ink_score': 0.28}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 962.0, 228.0]}, {'label': 'table row', 'box': [3.0, 3.0, 962.0, 37.2]}, {'label': 'table row', 'box': [3.0, 37.2, 962.0, 69.4]}, {'label': 'table row', 'box': [3.0, 69.4, 962.0, 101.5]}, {'label': 'table row', 'box': [3.0, 101.5, 962.0, 133.6]}, {'label': 'table row', 'box': [3.0, 133.6, 962.0, 165.7]}, {'label': 'table row', 'box': [3.0, 165.7, 962.0, 197.8]}, {'label': 'table row', 'box': [3.0, 197.8, 962.0, 228.0]}, {'label': 'table column', 'box': [3.0, 3.0, 553.1, 228.0]}, {'label': 'table column', 'box': [553.1, 3.0, 962.0, 228.0]}]},
    {'file': '1401.1475v1.1.png', 'table_id': '1401.1475v1.1', 'paper_id': '1401.1475', 'paper_title': 'Belief Revision in Structured Probabilistic Argumentation', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 702, 'height': 152, 'png_bytes': 10301, 'png_sha256': '77b423ffe3e79f503d6bef23a902595c49f4a96699a107d97f2d3cc1807708a0', 'n_rows': 7, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 9, 'ink_score': 0.263}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 699.0, 149.0]}, {'label': 'table row', 'box': [3.0, 3.0, 699.0, 25.5]}, {'label': 'table row', 'box': [3.0, 25.5, 699.0, 47.7]}, {'label': 'table row', 'box': [3.0, 47.7, 699.0, 67.8]}, {'label': 'table row', 'box': [3.0, 67.8, 699.0, 88.0]}, {'label': 'table row', 'box': [3.0, 88.0, 699.0, 108.1]}, {'label': 'table row', 'box': [3.0, 108.1, 699.0, 128.2]}, {'label': 'table row', 'box': [3.0, 128.2, 699.0, 149.0]}, {'label': 'table column', 'box': [3.0, 3.0, 352.9, 149.0]}, {'label': 'table column', 'box': [352.9, 3.0, 699.0, 149.0]}]},
    {'file': '1404.5002v1.1.png', 'table_id': '1404.5002v1.1', 'paper_id': '1404.5002', 'paper_title': 'A Geometric Distance Oracle for Large Real-World Graphs', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 491, 'height': 266, 'png_bytes': 15235, 'png_sha256': 'c49d8434c5a7383e3658ad98778e08e3c2c8bdd77e79be885cf47709f98b7b22', 'n_rows': 13, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.277}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 488.0, 263.0]}, {'label': 'table row', 'box': [3.0, 3.0, 488.0, 24.0]}, {'label': 'table row', 'box': [3.0, 24.0, 488.0, 44.1]}, {'label': 'table row', 'box': [3.0, 44.1, 488.0, 64.3]}, {'label': 'table row', 'box': [3.0, 64.3, 488.0, 84.0]}, {'label': 'table row', 'box': [3.0, 84.0, 488.0, 103.7]}, {'label': 'table row', 'box': [3.0, 103.7, 488.0, 123.4]}, {'label': 'table row', 'box': [3.0, 123.4, 488.0, 143.1]}, {'label': 'table row', 'box': [3.0, 143.1, 488.0, 162.9]}, {'label': 'table row', 'box': [3.0, 162.9, 488.0, 182.6]}, {'label': 'table row', 'box': [3.0, 182.6, 488.0, 202.3]}, {'label': 'table row', 'box': [3.0, 202.3, 488.0, 222.4]}, {'label': 'table row', 'box': [3.0, 222.4, 488.0, 242.6]}, {'label': 'table row', 'box': [3.0, 242.6, 488.0, 263.0]}, {'label': 'table column', 'box': [3.0, 3.0, 133.9, 263.0]}, {'label': 'table column', 'box': [133.9, 3.0, 220.7, 263.0]}, {'label': 'table column', 'box': [220.7, 3.0, 305.7, 263.0]}, {'label': 'table column', 'box': [305.7, 3.0, 366.8, 263.0]}, {'label': 'table column', 'box': [366.8, 3.0, 488.0, 263.0]}]},
    {'file': '1404.5002v1.2.png', 'table_id': '1404.5002v1.2', 'paper_id': '1404.5002', 'paper_title': 'A Geometric Distance Oracle for Large Real-World Graphs', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 456, 'height': 246, 'png_bytes': 11965, 'png_sha256': 'd7d3ea7c1114b23507c67d35c7df9bfecbaccbf00ec2090dbc5e82fa17f8882a', 'n_rows': 12, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 9, 'ink_score': 0.255}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 453.0, 243.0]}, {'label': 'table row', 'box': [3.0, 3.0, 453.0, 23.4]}, {'label': 'table row', 'box': [3.0, 23.4, 453.0, 43.6]}, {'label': 'table row', 'box': [3.0, 43.6, 453.0, 63.3]}, {'label': 'table row', 'box': [3.0, 63.3, 453.0, 83.0]}, {'label': 'table row', 'box': [3.0, 83.0, 453.0, 102.7]}, {'label': 'table row', 'box': [3.0, 102.7, 453.0, 122.4]}, {'label': 'table row', 'box': [3.0, 122.4, 453.0, 142.1]}, {'label': 'table row', 'box': [3.0, 142.1, 453.0, 161.9]}, {'label': 'table row', 'box': [3.0, 161.9, 453.0, 181.6]}, {'label': 'table row', 'box': [3.0, 181.6, 453.0, 201.7]}, {'label': 'table row', 'box': [3.0, 201.7, 453.0, 221.8]}, {'label': 'table row', 'box': [3.0, 221.8, 453.0, 243.0]}, {'label': 'table column', 'box': [3.0, 3.0, 133.9, 243.0]}, {'label': 'table column', 'box': [133.9, 3.0, 251.1, 243.0]}, {'label': 'table column', 'box': [251.1, 3.0, 319.9, 243.0]}, {'label': 'table column', 'box': [319.9, 3.0, 385.5, 243.0]}, {'label': 'table column', 'box': [385.5, 3.0, 453.0, 243.0]}]},
    {'file': '1407.3745v1.1.png', 'table_id': '1407.3745v1.1', 'paper_id': '1407.3745', 'paper_title': 'Query Optimization for Dynamic Graphs', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 720, 'height': 114, 'png_bytes': 9295, 'png_sha256': '1f91b6409c4a002c2737fb03772e44f5e87b91c42db1be049fa11b52f120aeda', 'n_rows': 4, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 15, 'ink_score': 0.282}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 717.0, 111.0]}, {'label': 'table row', 'box': [3.0, 3.0, 717.0, 33.1]}, {'label': 'table row', 'box': [3.0, 33.1, 717.0, 58.8]}, {'label': 'table row', 'box': [3.0, 58.8, 717.0, 84.5]}, {'label': 'table row', 'box': [3.0, 84.5, 717.0, 111.0]}, {'label': 'table column', 'box': [3.0, 3.0, 326.6, 111.0]}, {'label': 'table column', 'box': [326.6, 3.0, 487.7, 111.0]}, {'label': 'table column', 'box': [487.7, 3.0, 596.8, 111.0]}, {'label': 'table column', 'box': [596.8, 3.0, 717.0, 111.0]}]},
    {'file': '1409.5313v2.2.png', 'table_id': '1409.5313v2.2', 'paper_id': '1409.5313', 'paper_title': 'Sandboxing for Software Transactional Memory with Deferred Updates', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'test', 'width': 590, 'height': 151, 'png_bytes': 8030, 'png_sha256': '72750de77e2771f90e57464ba3ef81c7f91753e5eef3e96ccdf87319207a5d19', 'n_rows': 7, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.316}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 587.0, 148.0]}, {'label': 'table row', 'box': [3.0, 3.0, 587.0, 24.4]}, {'label': 'table row', 'box': [3.0, 24.4, 587.0, 45.0]}, {'label': 'table row', 'box': [3.0, 45.0, 587.0, 65.5]}, {'label': 'table row', 'box': [3.0, 65.5, 587.0, 86.1]}, {'label': 'table row', 'box': [3.0, 86.1, 587.0, 106.6]}, {'label': 'table row', 'box': [3.0, 106.6, 587.0, 127.2]}, {'label': 'table row', 'box': [3.0, 127.2, 587.0, 148.0]}, {'label': 'table column', 'box': [3.0, 3.0, 129.3, 148.0]}, {'label': 'table column', 'box': [129.3, 3.0, 248.0, 148.0]}, {'label': 'table column', 'box': [248.0, 3.0, 354.6, 148.0]}, {'label': 'table column', 'box': [354.6, 3.0, 457.2, 148.0]}, {'label': 'table column', 'box': [457.2, 3.0, 587.0, 148.0]}]},
    {'file': '1410.1237v2.1.png', 'table_id': '1410.1237v2.1', 'paper_id': '1410.1237', 'paper_title': 'Parallel Heuristics for Scalable Community Detection', 'paper_license': 'other:http://creativecommons.org/licenses/publicdomain/', 'scitsr_split': 'train', 'width': 746, 'height': 349, 'png_bytes': 25792, 'png_sha256': '05b16d565337616d1684bb41a43eb60fb47917014272916544f7ff02927874f4', 'n_rows': 13, 'n_cols': 6, 'n_spanning': 1, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.267}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 743.0, 346.0]}, {'label': 'table row', 'box': [3.0, 3.0, 743.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 743.0, 56.0]}, {'label': 'table row', 'box': [3.0, 56.0, 743.0, 83.9]}, {'label': 'table row', 'box': [3.0, 83.9, 743.0, 109.6]}, {'label': 'table row', 'box': [3.0, 109.6, 743.0, 135.3]}, {'label': 'table row', 'box': [3.0, 135.3, 743.0, 161.1]}, {'label': 'table row', 'box': [3.0, 161.1, 743.0, 186.8]}, {'label': 'table row', 'box': [3.0, 186.8, 743.0, 212.5]}, {'label': 'table row', 'box': [3.0, 212.5, 743.0, 238.3]}, {'label': 'table row', 'box': [3.0, 238.3, 743.0, 264.0]}, {'label': 'table row', 'box': [3.0, 264.0, 743.0, 289.7]}, {'label': 'table row', 'box': [3.0, 289.7, 743.0, 315.5]}, {'label': 'table row', 'box': [3.0, 315.5, 743.0, 346.0]}, {'label': 'table column', 'box': [3.0, 3.0, 184.2, 346.0]}, {'label': 'table column', 'box': [184.2, 3.0, 311.9, 346.0]}, {'label': 'table column', 'box': [311.9, 3.0, 457.9, 346.0]}, {'label': 'table column', 'box': [457.9, 3.0, 566.9, 346.0]}, {'label': 'table column', 'box': [566.9, 3.0, 659.9, 346.0]}, {'label': 'table column', 'box': [659.9, 3.0, 743.0, 346.0]}, {'label': 'table spanning cell', 'box': [457.9, 3.0, 743.0, 28.6]}]},
    {'file': '1510.01234v1.1.png', 'table_id': '1510.01234v1.1', 'paper_id': '1510.01234', 'paper_title': 'Hardware Random number Generator for cryptography', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 368, 'height': 332, 'png_bytes': 10545, 'png_sha256': '9ddfade5eb0d7259c5f3b0555e3093054024f6c62bfdc68a87b5005a8f45fe39', 'n_rows': 13, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.314}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 365.0, 329.0]}, {'label': 'table row', 'box': [3.0, 3.0, 365.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 365.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 365.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 365.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 365.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 365.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 365.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 365.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 365.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 365.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 365.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 365.0, 303.4]}, {'label': 'table row', 'box': [3.0, 303.4, 365.0, 329.0]}, {'label': 'table column', 'box': [3.0, 3.0, 261.4, 329.0]}, {'label': 'table column', 'box': [261.4, 3.0, 365.0, 329.0]}]},
    {'file': '1510.01234v1.2.png', 'table_id': '1510.01234v1.2', 'paper_id': '1510.01234', 'paper_title': 'Hardware Random number Generator for cryptography', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 720, 'height': 477, 'png_bytes': 18534, 'png_sha256': '47503b6b7fd3ead3e71dcc63c35d5a6eda3d821409548ed5ee84284c4a2c5db3', 'n_rows': 17, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 3, 'alignment': {'dx': 2, 'dy': 11, 'ink_score': 0.298}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 390.0, 443.0]}, {'label': 'table row', 'box': [3.0, 3.0, 390.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 390.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 390.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 390.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 390.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 390.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 390.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 390.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 390.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 390.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 390.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 390.0, 303.4]}, {'label': 'table row', 'box': [3.0, 303.4, 390.0, 328.4]}, {'label': 'table row', 'box': [3.0, 328.4, 390.0, 353.3]}, {'label': 'table row', 'box': [3.0, 353.3, 390.0, 378.2]}, {'label': 'table row', 'box': [3.0, 378.2, 390.0, 403.1]}, {'label': 'table row', 'box': [3.0, 403.1, 390.0, 443.0]}, {'label': 'table column', 'box': [3.0, 3.0, 261.4, 443.0]}, {'label': 'table column', 'box': [261.4, 3.0, 390.0, 443.0]}]},
    {'file': '1510.04780v1.6.png', 'table_id': '1510.04780v1.6', 'paper_id': '1510.04780', 'paper_title': 'A Graph Traversal Based Approach to Answer Non-Aggregation Questions Over DBpedia', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 706, 'height': 781, 'png_bytes': 51751, 'png_sha256': '052fba12b4a84de7149daf1b77b6edbe7aaadeca2bc22f99b8c855e826a3f9ef', 'n_rows': 31, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.268}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 703.0, 778.0]}, {'label': 'table row', 'box': [3.0, 3.0, 703.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 703.0, 55.4]}, {'label': 'table row', 'box': [3.0, 55.4, 703.0, 80.3]}, {'label': 'table row', 'box': [3.0, 80.3, 703.0, 105.2]}, {'label': 'table row', 'box': [3.0, 105.2, 703.0, 130.1]}, {'label': 'table row', 'box': [3.0, 130.1, 703.0, 155.0]}, {'label': 'table row', 'box': [3.0, 155.0, 703.0, 179.9]}, {'label': 'table row', 'box': [3.0, 179.9, 703.0, 204.8]}, {'label': 'table row', 'box': [3.0, 204.8, 703.0, 229.7]}, {'label': 'table row', 'box': [3.0, 229.7, 703.0, 254.6]}, {'label': 'table row', 'box': [3.0, 254.6, 703.0, 279.5]}, {'label': 'table row', 'box': [3.0, 279.5, 703.0, 304.4]}, {'label': 'table row', 'box': [3.0, 304.4, 703.0, 329.4]}, {'label': 'table row', 'box': [3.0, 329.4, 703.0, 354.3]}, {'label': 'table row', 'box': [3.0, 354.3, 703.0, 379.2]}, {'label': 'table row', 'box': [3.0, 379.2, 703.0, 404.1]}, {'label': 'table row', 'box': [3.0, 404.1, 703.0, 429.0]}, {'label': 'table row', 'box': [3.0, 429.0, 703.0, 453.9]}, {'label': 'table row', 'box': [3.0, 453.9, 703.0, 478.8]}, {'label': 'table row', 'box': [3.0, 478.8, 703.0, 503.7]}, {'label': 'table row', 'box': [3.0, 503.7, 703.0, 528.6]}, {'label': 'table row', 'box': [3.0, 528.6, 703.0, 553.5]}, {'label': 'table row', 'box': [3.0, 553.5, 703.0, 578.4]}, {'label': 'table row', 'box': [3.0, 578.4, 703.0, 603.3]}, {'label': 'table row', 'box': [3.0, 603.3, 703.0, 628.2]}, {'label': 'table row', 'box': [3.0, 628.2, 703.0, 653.1]}, {'label': 'table row', 'box': [3.0, 653.1, 703.0, 678.0]}, {'label': 'table row', 'box': [3.0, 678.0, 703.0, 703.0]}, {'label': 'table row', 'box': [3.0, 703.0, 703.0, 727.9]}, {'label': 'table row', 'box': [3.0, 727.9, 703.0, 752.8]}, {'label': 'table row', 'box': [3.0, 752.8, 703.0, 778.0]}, {'label': 'table column', 'box': [3.0, 3.0, 75.7, 778.0]}, {'label': 'table column', 'box': [75.7, 3.0, 703.0, 778.0]}]},
    {'file': '1510.04780v1.7.png', 'table_id': '1510.04780v1.7', 'paper_id': '1510.04780', 'paper_title': 'A Graph Traversal Based Approach to Answer Non-Aggregation Questions Over DBpedia', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 817, 'height': 357, 'png_bytes': 24929, 'png_sha256': 'a8d238cf9e1e12e4d42dd285957e5f93e6920efb760885cb1f619451659f96e8', 'n_rows': 14, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.271}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 814.0, 354.0]}, {'label': 'table row', 'box': [3.0, 3.0, 814.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 814.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 814.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 814.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 814.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 814.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 814.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 814.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 814.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 814.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 814.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 814.0, 303.4]}, {'label': 'table row', 'box': [3.0, 303.4, 814.0, 328.4]}, {'label': 'table row', 'box': [3.0, 328.4, 814.0, 354.0]}, {'label': 'table column', 'box': [3.0, 3.0, 65.4, 354.0]}, {'label': 'table column', 'box': [65.4, 3.0, 814.0, 354.0]}]},
    {'file': '1511.06278v1.1.png', 'table_id': '1511.06278v1.1', 'paper_id': '1511.06278', 'paper_title': 'Quantum Walks with Gremlin', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 722, 'height': 232, 'png_bytes': 8215, 'png_sha256': '6ba12accf7e13faf797877d84625ef966cae1cc8ada33af27f6804cf4255dcdf', 'n_rows': 6, 'n_cols': 12, 'n_spanning': 0, 'caption_chunks': 2, 'alignment': {'dx': 4, 'dy': 11, 'ink_score': 0.247}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 585.0, 162.0]}, {'label': 'table row', 'box': [3.0, 3.0, 585.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 585.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 585.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 585.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 585.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 585.0, 162.0]}, {'label': 'table column', 'box': [3.0, 3.0, 66.2, 162.0]}, {'label': 'table column', 'box': [66.2, 3.0, 119.3, 162.0]}, {'label': 'table column', 'box': [119.3, 3.0, 166.7, 162.0]}, {'label': 'table column', 'box': [166.7, 3.0, 212.4, 162.0]}, {'label': 'table column', 'box': [212.4, 3.0, 258.0, 162.0]}, {'label': 'table column', 'box': [258.0, 3.0, 303.7, 162.0]}, {'label': 'table column', 'box': [303.7, 3.0, 349.4, 162.0]}, {'label': 'table column', 'box': [349.4, 3.0, 395.0, 162.0]}, {'label': 'table column', 'box': [395.0, 3.0, 440.7, 162.0]}, {'label': 'table column', 'box': [440.7, 3.0, 486.3, 162.0]}, {'label': 'table column', 'box': [486.3, 3.0, 532.0, 162.0]}, {'label': 'table column', 'box': [532.0, 3.0, 585.0, 162.0]}]},
    {'file': '1602.02332v1.3.png', 'table_id': '1602.02332v1.3', 'paper_id': '1602.02332', 'paper_title': 'Scalable Text Mining with Sparse Generative Models', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 587, 'height': 308, 'png_bytes': 11444, 'png_sha256': '1d0486a66f6b77e1e022603dab8a9f077f0d97a79770ac6c31ad7bfe4cca2503', 'n_rows': 12, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.302}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 584.0, 305.0]}, {'label': 'table row', 'box': [3.0, 3.0, 584.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 584.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 584.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 584.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 584.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 584.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 584.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 584.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 584.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 584.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 584.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 584.0, 305.0]}, {'label': 'table column', 'box': [3.0, 3.0, 210.7, 305.0]}, {'label': 'table column', 'box': [210.7, 3.0, 454.9, 305.0]}, {'label': 'table column', 'box': [454.9, 3.0, 584.0, 305.0]}]},
    {'file': '1603.01595v1.1.png', 'table_id': '1603.01595v1.1', 'paper_id': '1603.01595', 'paper_title': 'Sentiment Analysis in Scholarly Book Reviews', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 511, 'height': 208, 'png_bytes': 7209, 'png_sha256': '93ecb2d57b1f4299c855cbe778234d560695f5c0a113231f64ef82ddd9bc59e4', 'n_rows': 8, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 17, 'dy': 12, 'ink_score': 0.28}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 508.0, 205.0]}, {'label': 'table row', 'box': [3.0, 3.0, 508.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 508.0, 55.4]}, {'label': 'table row', 'box': [3.0, 55.4, 508.0, 80.3]}, {'label': 'table row', 'box': [3.0, 80.3, 508.0, 105.2]}, {'label': 'table row', 'box': [3.0, 105.2, 508.0, 130.1]}, {'label': 'table row', 'box': [3.0, 130.1, 508.0, 155.0]}, {'label': 'table row', 'box': [3.0, 155.0, 508.0, 179.9]}, {'label': 'table row', 'box': [3.0, 179.9, 508.0, 205.0]}, {'label': 'table column', 'box': [3.0, 3.0, 185.7, 205.0]}, {'label': 'table column', 'box': [185.7, 3.0, 278.3, 205.0]}, {'label': 'table column', 'box': [278.3, 3.0, 397.9, 205.0]}, {'label': 'table column', 'box': [397.9, 3.0, 508.0, 205.0]}]},
    {'file': '1603.01595v1.2.png', 'table_id': '1603.01595v1.2', 'paper_id': '1603.01595', 'paper_title': 'Sentiment Analysis in Scholarly Book Reviews', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 599, 'height': 158, 'png_bytes': 8339, 'png_sha256': '8e0b1250fb2de42bdee703e97832c0e60250bde35796ab0e6b119b063bcc6f28', 'n_rows': 6, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.276}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 596.0, 155.0]}, {'label': 'table row', 'box': [3.0, 3.0, 596.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 596.0, 55.4]}, {'label': 'table row', 'box': [3.0, 55.4, 596.0, 80.3]}, {'label': 'table row', 'box': [3.0, 80.3, 596.0, 105.2]}, {'label': 'table row', 'box': [3.0, 105.2, 596.0, 130.1]}, {'label': 'table row', 'box': [3.0, 130.1, 596.0, 155.0]}, {'label': 'table column', 'box': [3.0, 3.0, 301.0, 155.0]}, {'label': 'table column', 'box': [301.0, 3.0, 383.5, 155.0]}, {'label': 'table column', 'box': [383.5, 3.0, 490.8, 155.0]}, {'label': 'table column', 'box': [490.8, 3.0, 596.0, 155.0]}]},
    {'file': '1603.01595v1.3.png', 'table_id': '1603.01595v1.3', 'paper_id': '1603.01595', 'paper_title': 'Sentiment Analysis in Scholarly Book Reviews', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 320, 'height': 307, 'png_bytes': 8891, 'png_sha256': '6bc268e156faa5c167d653ec6e0b49bb8f9c685ab9dfc432b2a5c80144b28774', 'n_rows': 12, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.259}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 317.0, 304.0]}, {'label': 'table row', 'box': [3.0, 3.0, 317.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 317.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 317.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 317.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 317.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 317.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 317.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 317.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 317.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 317.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 317.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 317.0, 304.0]}, {'label': 'table column', 'box': [3.0, 3.0, 151.7, 304.0]}, {'label': 'table column', 'box': [151.7, 3.0, 317.0, 304.0]}]},
    {'file': '1603.02514v3.1.png', 'table_id': '1603.02514v3.1', 'paper_id': '1603.02514', 'paper_title': 'Variational Autoencoders for Semi-supervised Text Classification', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 586, 'height': 77, 'png_bytes': 4318, 'png_sha256': '80c314489f07d617eb2dc539cc08cd17f8a21f9c1b4717ab858c107acf74c2a7', 'n_rows': 3, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.355}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 583.0, 74.0]}, {'label': 'table row', 'box': [3.0, 3.0, 583.0, 27.5]}, {'label': 'table row', 'box': [3.0, 27.5, 583.0, 50.7]}, {'label': 'table row', 'box': [3.0, 50.7, 583.0, 74.0]}, {'label': 'table column', 'box': [3.0, 3.0, 121.1, 74.0]}, {'label': 'table column', 'box': [121.1, 3.0, 232.9, 74.0]}, {'label': 'table column', 'box': [232.9, 3.0, 369.2, 74.0]}, {'label': 'table column', 'box': [369.2, 3.0, 476.1, 74.0]}, {'label': 'table column', 'box': [476.1, 3.0, 583.0, 74.0]}]},
    {'file': '1604.06285v1.2.png', 'table_id': '1604.06285v1.2', 'paper_id': '1604.06285', 'paper_title': 'A Novel Approach to Dropped Pronoun Translation', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 501, 'height': 362, 'png_bytes': 15405, 'png_sha256': '09635fb8fb4afcb15be91ca86ab93d9078ffd98ac37b6279bcbc92a8f9bfb52f', 'n_rows': 14, 'n_cols': 2, 'n_spanning': 3, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.28}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 498.0, 359.0]}, {'label': 'table row', 'box': [3.0, 3.0, 498.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 498.0, 54.8]}, {'label': 'table row', 'box': [3.0, 54.8, 498.0, 80.1]}, {'label': 'table row', 'box': [3.0, 80.1, 498.0, 105.0]}, {'label': 'table row', 'box': [3.0, 105.0, 498.0, 129.9]}, {'label': 'table row', 'box': [3.0, 129.9, 498.0, 155.3]}, {'label': 'table row', 'box': [3.0, 155.3, 498.0, 181.0]}, {'label': 'table row', 'box': [3.0, 181.0, 498.0, 206.3]}, {'label': 'table row', 'box': [3.0, 206.3, 498.0, 231.2]}, {'label': 'table row', 'box': [3.0, 231.2, 498.0, 256.1]}, {'label': 'table row', 'box': [3.0, 256.1, 498.0, 281.4]}, {'label': 'table row', 'box': [3.0, 281.4, 498.0, 307.2]}, {'label': 'table row', 'box': [3.0, 307.2, 498.0, 330.6]}, {'label': 'table row', 'box': [3.0, 330.6, 498.0, 359.0]}, {'label': 'table column', 'box': [3.0, 3.0, 62.4, 359.0]}, {'label': 'table column', 'box': [62.4, 3.0, 498.0, 359.0]}, {'label': 'table spanning cell', 'box': [3.0, 29.1, 498.0, 54.8]}, {'label': 'table spanning cell', 'box': [3.0, 155.3, 498.0, 181.0]}, {'label': 'table spanning cell', 'box': [3.0, 281.4, 498.0, 307.2]}]},
    {'file': '1604.06285v1.4.png', 'table_id': '1604.06285v1.4', 'paper_id': '1604.06285', 'paper_title': 'A Novel Approach to Dropped Pronoun Translation', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 412, 'height': 134, 'png_bytes': 4842, 'png_sha256': '47acd693195292c380f3098c5c4c0b156fb3339947300c0e312e52e96fd70596', 'n_rows': 5, 'n_cols': 5, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.286}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 409.0, 131.0]}, {'label': 'table row', 'box': [3.0, 3.0, 409.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 409.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 409.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 409.0, 105.0]}, {'label': 'table row', 'box': [3.0, 105.0, 409.0, 131.0]}, {'label': 'table column', 'box': [3.0, 3.0, 159.1, 131.0]}, {'label': 'table column', 'box': [159.1, 3.0, 222.8, 131.0]}, {'label': 'table column', 'box': [222.8, 3.0, 284.6, 131.0]}, {'label': 'table column', 'box': [284.6, 3.0, 346.4, 131.0]}, {'label': 'table column', 'box': [346.4, 3.0, 409.0, 131.0]}, {'label': 'table spanning cell', 'box': [3.0, 79.7, 159.1, 131.0]}, {'label': 'table spanning cell', 'box': [3.0, 29.1, 159.1, 79.7]}]},
    {'file': '1604.06979v1.1.png', 'table_id': '1604.06979v1.1', 'paper_id': '1604.06979', 'paper_title': 'Cardiac Motion Analysis by Temporal Flow Graphs', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 583, 'height': 191, 'png_bytes': 8013, 'png_sha256': 'b87ede8883cb874aa5fb13836e9c20f6939dc3d9191585f226dcae59375988fc', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.278}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 580.0, 188.0]}, {'label': 'table row', 'box': [3.0, 3.0, 580.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 580.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 580.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 580.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 580.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 580.0, 161.9]}, {'label': 'table row', 'box': [3.0, 161.9, 580.0, 188.0]}, {'label': 'table column', 'box': [3.0, 3.0, 287.5, 188.0]}, {'label': 'table column', 'box': [287.5, 3.0, 444.8, 188.0]}, {'label': 'table column', 'box': [444.8, 3.0, 580.0, 188.0]}]},
    {'file': '1604.06979v1.2.png', 'table_id': '1604.06979v1.2', 'paper_id': '1604.06979', 'paper_title': 'Cardiac Motion Analysis by Temporal Flow Graphs', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 479, 'height': 191, 'png_bytes': 7300, 'png_sha256': '39f5f707227cc8faa1686b0df6beaa2cb2647a235f29144d636fd953333fa738', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.263}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 476.0, 188.0]}, {'label': 'table row', 'box': [3.0, 3.0, 476.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 476.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 476.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 476.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 476.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 476.0, 161.9]}, {'label': 'table row', 'box': [3.0, 161.9, 476.0, 188.0]}, {'label': 'table column', 'box': [3.0, 3.0, 190.9, 188.0]}, {'label': 'table column', 'box': [190.9, 3.0, 341.2, 188.0]}, {'label': 'table column', 'box': [341.2, 3.0, 476.0, 188.0]}]},
    {'file': '1605.04635v2.1.png', 'table_id': '1605.04635v2.1', 'paper_id': '1605.04635', 'paper_title': 'Cumulative Activation in Social Networks', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 473, 'height': 110, 'png_bytes': 6612, 'png_sha256': '89ee8823d41bd6bfbcb0a8550dc77c4bbb3725ce6f394289f2d2455e5c3b47cd', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.287}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 470.0, 107.0]}, {'label': 'table row', 'box': [3.0, 3.0, 470.0, 29.0]}, {'label': 'table row', 'box': [3.0, 29.0, 470.0, 54.8]}, {'label': 'table row', 'box': [3.0, 54.8, 470.0, 80.5]}, {'label': 'table row', 'box': [3.0, 80.5, 470.0, 107.0]}, {'label': 'table column', 'box': [3.0, 3.0, 106.6, 107.0]}, {'label': 'table column', 'box': [106.6, 3.0, 197.6, 107.0]}, {'label': 'table column', 'box': [197.6, 3.0, 280.4, 107.0]}, {'label': 'table column', 'box': [280.4, 3.0, 401.1, 107.0]}, {'label': 'table column', 'box': [401.1, 3.0, 470.0, 107.0]}]},
    {'file': '1605.06770v1.1.png', 'table_id': '1605.06770v1.1', 'paper_id': '1605.06770', 'paper_title': 'Automatic Construction of Discourse Corpora for Dialogue Translation', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 516, 'height': 212, 'png_bytes': 8359, 'png_sha256': '22775ef0647770d26cbd7a338476f64fff407435d0b0d4229a3d00cfde853f0b', 'n_rows': 8, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.282}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 513.0, 209.0]}, {'label': 'table row', 'box': [3.0, 3.0, 513.0, 32.1]}, {'label': 'table row', 'box': [3.0, 32.1, 513.0, 59.5]}, {'label': 'table row', 'box': [3.0, 59.5, 513.0, 84.4]}, {'label': 'table row', 'box': [3.0, 84.4, 513.0, 109.3]}, {'label': 'table row', 'box': [3.0, 109.3, 513.0, 134.3]}, {'label': 'table row', 'box': [3.0, 134.3, 513.0, 159.2]}, {'label': 'table row', 'box': [3.0, 159.2, 513.0, 184.1]}, {'label': 'table row', 'box': [3.0, 184.1, 513.0, 209.0]}, {'label': 'table column', 'box': [3.0, 3.0, 419.6, 209.0]}, {'label': 'table column', 'box': [419.6, 3.0, 513.0, 209.0]}]},
    {'file': '1605.06770v1.2.png', 'table_id': '1605.06770v1.2', 'paper_id': '1605.06770', 'paper_title': 'Automatic Construction of Discourse Corpora for Dialogue Translation', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 476, 'height': 143, 'png_bytes': 7121, 'png_sha256': '1b4b605c6f8ae30e48fe359c07736b64d5ccc6b944215aeac4f8be21e2d37be3', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.258}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 473.0, 140.0]}, {'label': 'table row', 'box': [3.0, 3.0, 473.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 473.0, 58.5]}, {'label': 'table row', 'box': [3.0, 58.5, 473.0, 87.5]}, {'label': 'table row', 'box': [3.0, 87.5, 473.0, 113.3]}, {'label': 'table row', 'box': [3.0, 113.3, 473.0, 140.0]}, {'label': 'table column', 'box': [3.0, 3.0, 101.9, 140.0]}, {'label': 'table column', 'box': [101.9, 3.0, 226.3, 140.0]}, {'label': 'table column', 'box': [226.3, 3.0, 324.3, 140.0]}, {'label': 'table column', 'box': [324.3, 3.0, 473.0, 140.0]}, {'label': 'table spanning cell', 'box': [3.0, 87.5, 101.9, 140.0]}, {'label': 'table spanning cell', 'box': [3.0, 31.1, 101.9, 87.5]}]},
    {'file': '1606.09370v1.1.png', 'table_id': '1606.09370v1.1', 'paper_id': '1606.09370', 'paper_title': 'Relation extraction from clinical texts using domain invariant convolutional neural network', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 354, 'height': 187, 'png_bytes': 5839, 'png_sha256': '585847ca9e0ebe44fef24430279c9d6b2c9f8378f68268493db08194f56a7c4a', 'n_rows': 7, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.299}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 351.0, 184.0]}, {'label': 'table row', 'box': [3.0, 3.0, 351.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 351.0, 55.8]}, {'label': 'table row', 'box': [3.0, 55.8, 351.0, 81.5]}, {'label': 'table row', 'box': [3.0, 81.5, 351.0, 107.3]}, {'label': 'table row', 'box': [3.0, 107.3, 351.0, 133.0]}, {'label': 'table row', 'box': [3.0, 133.0, 351.0, 158.7]}, {'label': 'table row', 'box': [3.0, 158.7, 351.0, 184.0]}, {'label': 'table column', 'box': [3.0, 3.0, 137.7, 184.0]}, {'label': 'table column', 'box': [137.7, 3.0, 351.0, 184.0]}]},
    {'file': '1606.09370v1.4.png', 'table_id': '1606.09370v1.4', 'paper_id': '1606.09370', 'paper_title': 'Relation extraction from clinical texts using domain invariant convolutional neural network', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 420, 'height': 196, 'png_bytes': 10464, 'png_sha256': 'dd93e18bd2719edcbb3a93c4395b12db690c825ecb5cb3f8825c72e7a4404543', 'n_rows': 7, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.242}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 417.0, 193.0]}, {'label': 'table row', 'box': [3.0, 3.0, 417.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 417.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 417.0, 86.2]}, {'label': 'table row', 'box': [3.0, 86.2, 417.0, 112.5]}, {'label': 'table row', 'box': [3.0, 112.5, 417.0, 140.3]}, {'label': 'table row', 'box': [3.0, 140.3, 417.0, 167.6]}, {'label': 'table row', 'box': [3.0, 167.6, 417.0, 193.0]}, {'label': 'table column', 'box': [3.0, 3.0, 177.0, 193.0]}, {'label': 'table column', 'box': [177.0, 3.0, 258.1, 193.0]}, {'label': 'table column', 'box': [258.1, 3.0, 337.4, 193.0]}, {'label': 'table column', 'box': [337.4, 3.0, 417.0, 193.0]}]},
    {'file': '1606.09370v1.5.png', 'table_id': '1606.09370v1.5', 'paper_id': '1606.09370', 'paper_title': 'Relation extraction from clinical texts using domain invariant convolutional neural network', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 478, 'height': 140, 'png_bytes': 7210, 'png_sha256': '1ae83934ef1adea67d8756676506195ded93cd57cfce014724e7234a4c02a250', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.253}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 475.0, 137.0]}, {'label': 'table row', 'box': [3.0, 3.0, 475.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 475.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 475.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 475.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 475.0, 137.0]}, {'label': 'table column', 'box': [3.0, 3.0, 236.4, 137.0]}, {'label': 'table column', 'box': [236.4, 3.0, 315.6, 137.0]}, {'label': 'table column', 'box': [315.6, 3.0, 394.9, 137.0]}, {'label': 'table column', 'box': [394.9, 3.0, 475.0, 137.0]}]},
    {'file': '1606.09371v1.2.png', 'table_id': '1606.09371v1.2', 'paper_id': '1606.09371', 'paper_title': 'Recurrent neural network models for disease name recognition using domain invariant features', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 478, 'height': 187, 'png_bytes': 8234, 'png_sha256': '2010c45ac092628a4e23a52b3a1378ffe1e9194b7d7841ed71d7ab217915534f', 'n_rows': 7, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 17, 'dy': 12, 'ink_score': 0.296}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 475.0, 184.0]}, {'label': 'table row', 'box': [3.0, 3.0, 475.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 475.0, 55.8]}, {'label': 'table row', 'box': [3.0, 55.8, 475.0, 81.6]}, {'label': 'table row', 'box': [3.0, 81.6, 475.0, 107.3]}, {'label': 'table row', 'box': [3.0, 107.3, 475.0, 133.0]}, {'label': 'table row', 'box': [3.0, 133.0, 475.0, 158.8]}, {'label': 'table row', 'box': [3.0, 158.8, 475.0, 184.0]}, {'label': 'table column', 'box': [3.0, 3.0, 140.6, 184.0]}, {'label': 'table column', 'box': [140.6, 3.0, 261.3, 184.0]}, {'label': 'table column', 'box': [261.3, 3.0, 366.2, 184.0]}, {'label': 'table column', 'box': [366.2, 3.0, 475.0, 184.0]}]},
    {'file': '1606.09371v1.5.png', 'table_id': '1606.09371v1.5', 'paper_id': '1606.09371', 'paper_title': 'Recurrent neural network models for disease name recognition using domain invariant features', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 386, 'height': 136, 'png_bytes': 6476, 'png_sha256': '115c87256d4383981038f432e40aeaa736166cf741dd840ca81175cc688cc5f6', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 7, 'ink_score': 0.263}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 383.0, 133.0]}, {'label': 'table row', 'box': [3.0, 3.0, 383.0, 25.1]}, {'label': 'table row', 'box': [3.0, 25.1, 383.0, 50.8]}, {'label': 'table row', 'box': [3.0, 50.8, 383.0, 76.6]}, {'label': 'table row', 'box': [3.0, 76.6, 383.0, 102.3]}, {'label': 'table row', 'box': [3.0, 102.3, 383.0, 133.0]}, {'label': 'table column', 'box': [3.0, 3.0, 166.0, 133.0]}, {'label': 'table column', 'box': [166.0, 3.0, 238.1, 133.0]}, {'label': 'table column', 'box': [238.1, 3.0, 310.3, 133.0]}, {'label': 'table column', 'box': [310.3, 3.0, 383.0, 133.0]}]},
    {'file': '1609.01344v1.1.png', 'table_id': '1609.01344v1.1', 'paper_id': '1609.01344', 'paper_title': 'Vision-based Engagement Detection in Virtual Reality', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 570, 'height': 145, 'png_bytes': 8926, 'png_sha256': 'd22e918a810d4afe79be56af7527a113c10db945f5219b7d0a4614eca1167161', 'n_rows': 8, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 8, 'ink_score': 0.284}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 567.0, 142.0]}, {'label': 'table row', 'box': [3.0, 3.0, 567.0, 22.4]}, {'label': 'table row', 'box': [3.0, 22.4, 567.0, 41.5]}, {'label': 'table row', 'box': [3.0, 41.5, 567.0, 58.1]}, {'label': 'table row', 'box': [3.0, 58.1, 567.0, 74.7]}, {'label': 'table row', 'box': [3.0, 74.7, 567.0, 91.3]}, {'label': 'table row', 'box': [3.0, 91.3, 567.0, 107.9]}, {'label': 'table row', 'box': [3.0, 107.9, 567.0, 124.5]}, {'label': 'table row', 'box': [3.0, 124.5, 567.0, 142.0]}, {'label': 'table column', 'box': [3.0, 3.0, 147.4, 142.0]}, {'label': 'table column', 'box': [147.4, 3.0, 567.0, 142.0]}]},
    {'file': '1609.01344v1.2.png', 'table_id': '1609.01344v1.2', 'paper_id': '1609.01344', 'paper_title': 'Vision-based Engagement Detection in Virtual Reality', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 309, 'height': 137, 'png_bytes': 3040, 'png_sha256': '67f751b75543e7276717ad89b0e239557f38cc6309dfc2d98d58e72473c5af8d', 'n_rows': 5, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.189}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 306.0, 134.0]}, {'label': 'table row', 'box': [3.0, 3.0, 306.0, 32.7]}, {'label': 'table row', 'box': [3.0, 32.7, 306.0, 60.1]}, {'label': 'table row', 'box': [3.0, 60.1, 306.0, 85.0]}, {'label': 'table row', 'box': [3.0, 85.0, 306.0, 109.9]}, {'label': 'table row', 'box': [3.0, 109.9, 306.0, 134.0]}, {'label': 'table column', 'box': [3.0, 3.0, 75.7, 134.0]}, {'label': 'table column', 'box': [75.7, 3.0, 132.5, 134.0]}, {'label': 'table column', 'box': [132.5, 3.0, 189.9, 134.0]}, {'label': 'table column', 'box': [189.9, 3.0, 247.2, 134.0]}, {'label': 'table column', 'box': [247.2, 3.0, 306.0, 134.0]}]},
    {'file': '1610.06272v2.1.png', 'table_id': '1610.06272v2.1', 'paper_id': '1610.06272', 'paper_title': 'Lexicon Integrated CNN Models with Attention for Sentiment Analysis', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 363, 'height': 103, 'png_bytes': 4267, 'png_sha256': '593ca6f4af19b69a0c09da976e6ec45100183aa4d3f9350d223e7aae6682de22', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.294}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 360.0, 100.0]}, {'label': 'table row', 'box': [3.0, 3.0, 360.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 360.0, 53.9]}, {'label': 'table row', 'box': [3.0, 53.9, 360.0, 76.7]}, {'label': 'table row', 'box': [3.0, 76.7, 360.0, 100.0]}, {'label': 'table column', 'box': [3.0, 3.0, 60.0, 100.0]}, {'label': 'table column', 'box': [60.0, 3.0, 130.7, 100.0]}, {'label': 'table column', 'box': [130.7, 3.0, 208.9, 100.0]}, {'label': 'table column', 'box': [208.9, 3.0, 279.6, 100.0]}, {'label': 'table column', 'box': [279.6, 3.0, 360.0, 100.0]}]},
    {'file': '1610.06272v2.2.png', 'table_id': '1610.06272v2.2', 'paper_id': '1610.06272', 'paper_title': 'Lexicon Integrated CNN Models with Attention for Sentiment Analysis', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 455, 'height': 99, 'png_bytes': 4282, 'png_sha256': 'bdce88e721ba40517c2cc30dbe841554c851f9b7a4dad2f48033be8a58e329c2', 'n_rows': 4, 'n_cols': 7, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.294}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 452.0, 96.0]}, {'label': 'table row', 'box': [3.0, 3.0, 452.0, 26.5]}, {'label': 'table row', 'box': [3.0, 26.5, 452.0, 49.7]}, {'label': 'table row', 'box': [3.0, 49.7, 452.0, 72.6]}, {'label': 'table row', 'box': [3.0, 72.6, 452.0, 96.0]}, {'label': 'table column', 'box': [3.0, 3.0, 59.9, 96.0]}, {'label': 'table column', 'box': [59.9, 3.0, 125.3, 96.0]}, {'label': 'table column', 'box': [125.3, 3.0, 188.6, 96.0]}, {'label': 'table column', 'box': [188.6, 3.0, 251.9, 96.0]}, {'label': 'table column', 'box': [251.9, 3.0, 315.2, 96.0]}, {'label': 'table column', 'box': [315.2, 3.0, 380.6, 96.0]}, {'label': 'table column', 'box': [380.6, 3.0, 452.0, 96.0]}]},
    {'file': '1610.06272v2.3.png', 'table_id': '1610.06272v2.3', 'paper_id': '1610.06272', 'paper_title': 'Lexicon Integrated CNN Models with Attention for Sentiment Analysis', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 358, 'height': 126, 'png_bytes': 5491, 'png_sha256': 'e50981996435def92c6e326dc83672ab8f5e88c8b948d4be96f6e37dc32086aa', 'n_rows': 5, 'n_cols': 5, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.321}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 355.0, 123.0]}, {'label': 'table row', 'box': [3.0, 3.0, 355.0, 26.1]}, {'label': 'table row', 'box': [3.0, 26.1, 355.0, 51.4]}, {'label': 'table row', 'box': [3.0, 51.4, 355.0, 76.7]}, {'label': 'table row', 'box': [3.0, 76.7, 355.0, 99.6]}, {'label': 'table row', 'box': [3.0, 99.6, 355.0, 123.0]}, {'label': 'table column', 'box': [3.0, 3.0, 59.9, 123.0]}, {'label': 'table column', 'box': [59.9, 3.0, 130.7, 123.0]}, {'label': 'table column', 'box': [130.7, 3.0, 201.4, 123.0]}, {'label': 'table column', 'box': [201.4, 3.0, 276.5, 123.0]}, {'label': 'table column', 'box': [276.5, 3.0, 355.0, 123.0]}, {'label': 'table spanning cell', 'box': [201.4, 3.0, 355.0, 26.1]}, {'label': 'table spanning cell', 'box': [59.9, 3.0, 201.4, 26.1]}]},
    {'file': '1611.04741v2.1.png', 'table_id': '1611.04741v2.1', 'paper_id': '1611.04741', 'paper_title': 'A Neural Architecture Mimicking Humans End-to-End for Natural Language Inference', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 456, 'height': 217, 'png_bytes': 9001, 'png_sha256': '88e74c13c53e0fdc131d71f0b266b51bef8306f40deca92bb4458546ea31115e', 'n_rows': 8, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.292}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 453.0, 214.0]}, {'label': 'table row', 'box': [3.0, 3.0, 453.0, 32.1]}, {'label': 'table row', 'box': [3.0, 32.1, 453.0, 59.9]}, {'label': 'table row', 'box': [3.0, 59.9, 453.0, 85.7]}, {'label': 'table row', 'box': [3.0, 85.7, 453.0, 111.4]}, {'label': 'table row', 'box': [3.0, 111.4, 453.0, 137.2]}, {'label': 'table row', 'box': [3.0, 137.2, 453.0, 162.9]}, {'label': 'table row', 'box': [3.0, 162.9, 453.0, 188.6]}, {'label': 'table row', 'box': [3.0, 188.6, 453.0, 214.0]}, {'label': 'table column', 'box': [3.0, 3.0, 368.8, 214.0]}, {'label': 'table column', 'box': [368.8, 3.0, 453.0, 214.0]}]},
    {'file': '1611.04741v2.2.png', 'table_id': '1611.04741v2.2', 'paper_id': '1611.04741', 'paper_title': 'A Neural Architecture Mimicking Humans End-to-End for Natural Language Inference', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 905, 'height': 324, 'png_bytes': 21004, 'png_sha256': '80be622daa745d8a11eb2c453891767224289d9130c8e824e26330ef4423ecb1', 'n_rows': 12, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.294}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 902.0, 321.0]}, {'label': 'table row', 'box': [3.0, 3.0, 902.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 902.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 902.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 902.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 902.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 902.0, 161.9]}, {'label': 'table row', 'box': [3.0, 161.9, 902.0, 187.6]}, {'label': 'table row', 'box': [3.0, 187.6, 902.0, 213.4]}, {'label': 'table row', 'box': [3.0, 213.4, 902.0, 239.1]}, {'label': 'table row', 'box': [3.0, 239.1, 902.0, 266.9]}, {'label': 'table row', 'box': [3.0, 266.9, 902.0, 294.7]}, {'label': 'table row', 'box': [3.0, 294.7, 902.0, 321.0]}, {'label': 'table column', 'box': [3.0, 3.0, 376.7, 321.0]}, {'label': 'table column', 'box': [376.7, 3.0, 562.8, 321.0]}, {'label': 'table column', 'box': [562.8, 3.0, 737.4, 321.0]}, {'label': 'table column', 'box': [737.4, 3.0, 902.0, 321.0]}]},
    {'file': '1611.04741v2.3.png', 'table_id': '1611.04741v2.3', 'paper_id': '1611.04741', 'paper_title': 'A Neural Architecture Mimicking Humans End-to-End for Natural Language Inference', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 583, 'height': 170, 'png_bytes': 10081, 'png_sha256': '309a4da6be53284762a41af34de96aca58ad4e4bfe69ef6721d8b2e21e73f788', 'n_rows': 6, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.288}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 580.0, 167.0]}, {'label': 'table row', 'box': [3.0, 3.0, 580.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 580.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 580.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 580.0, 112.5]}, {'label': 'table row', 'box': [3.0, 112.5, 580.0, 140.3]}, {'label': 'table row', 'box': [3.0, 140.3, 580.0, 167.0]}, {'label': 'table column', 'box': [3.0, 3.0, 376.7, 167.0]}, {'label': 'table column', 'box': [376.7, 3.0, 444.0, 167.0]}, {'label': 'table column', 'box': [444.0, 3.0, 511.4, 167.0]}, {'label': 'table column', 'box': [511.4, 3.0, 580.0, 167.0]}]},
    {'file': '1611.09235v1.1.png', 'table_id': '1611.09235v1.1', 'paper_id': '1611.09235', 'paper_title': 'Joint Copying and Restricted Generation for Paraphrase', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 445, 'height': 168, 'png_bytes': 7866, 'png_sha256': 'b05f9b840911d8918ec66232f8478bb976dc00ff043178526c3a3a52ddf2e081', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.302}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 442.0, 165.0]}, {'label': 'table row', 'box': [3.0, 3.0, 442.0, 26.5]}, {'label': 'table row', 'box': [3.0, 26.5, 442.0, 49.7]}, {'label': 'table row', 'box': [3.0, 49.7, 442.0, 72.6]}, {'label': 'table row', 'box': [3.0, 72.6, 442.0, 95.4]}, {'label': 'table row', 'box': [3.0, 95.4, 442.0, 118.2]}, {'label': 'table row', 'box': [3.0, 118.2, 442.0, 141.1]}, {'label': 'table row', 'box': [3.0, 141.1, 442.0, 165.0]}, {'label': 'table column', 'box': [3.0, 3.0, 149.3, 165.0]}, {'label': 'table column', 'box': [149.3, 3.0, 301.2, 165.0]}, {'label': 'table column', 'box': [301.2, 3.0, 442.0, 165.0]}]},
    {'file': '1611.09238v1.1.png', 'table_id': '1611.09238v1.1', 'paper_id': '1611.09238', 'paper_title': 'Improving Multi-Document Summarization via Text Classification', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 494, 'height': 102, 'png_bytes': 5071, 'png_sha256': '941333d7de3c6ab144478dd6f82818da9b0cebcc0137db70449006a48a2ae8fe', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.281}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 491.0, 99.0]}, {'label': 'table row', 'box': [3.0, 3.0, 491.0, 27.5]}, {'label': 'table row', 'box': [3.0, 27.5, 491.0, 51.2]}, {'label': 'table row', 'box': [3.0, 51.2, 491.0, 74.8]}, {'label': 'table row', 'box': [3.0, 74.8, 491.0, 99.0]}, {'label': 'table column', 'box': [3.0, 3.0, 97.0, 99.0]}, {'label': 'table column', 'box': [97.0, 3.0, 205.2, 99.0]}, {'label': 'table column', 'box': [205.2, 3.0, 293.3, 99.0]}, {'label': 'table column', 'box': [293.3, 3.0, 376.6, 99.0]}, {'label': 'table column', 'box': [376.6, 3.0, 491.0, 99.0]}]},
    {'file': '1702.02925v1.1.png', 'table_id': '1702.02925v1.1', 'paper_id': '1702.02925', 'paper_title': 'EAC-Net: A Region-based Deep Enhancing and Cropping Approach for Facial Action Unit Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 616, 'height': 333, 'png_bytes': 15549, 'png_sha256': '1eb6b35e406d8cd078f0ff3a3e6081f2642d37bd814453f63f010b3b3e677607', 'n_rows': 13, 'n_cols': 3, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.268}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 613.0, 330.0]}, {'label': 'table row', 'box': [3.0, 3.0, 613.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 613.0, 55.4]}, {'label': 'table row', 'box': [3.0, 55.4, 613.0, 80.3]}, {'label': 'table row', 'box': [3.0, 80.3, 613.0, 105.2]}, {'label': 'table row', 'box': [3.0, 105.2, 613.0, 130.1]}, {'label': 'table row', 'box': [3.0, 130.1, 613.0, 155.0]}, {'label': 'table row', 'box': [3.0, 155.0, 613.0, 179.9]}, {'label': 'table row', 'box': [3.0, 179.9, 613.0, 204.8]}, {'label': 'table row', 'box': [3.0, 204.8, 613.0, 229.7]}, {'label': 'table row', 'box': [3.0, 229.7, 613.0, 254.6]}, {'label': 'table row', 'box': [3.0, 254.6, 613.0, 279.5]}, {'label': 'table row', 'box': [3.0, 279.5, 613.0, 304.4]}, {'label': 'table row', 'box': [3.0, 304.4, 613.0, 330.0]}, {'label': 'table column', 'box': [3.0, 3.0, 114.9, 330.0]}, {'label': 'table column', 'box': [114.9, 3.0, 335.2, 330.0]}, {'label': 'table column', 'box': [335.2, 3.0, 613.0, 330.0]}]},
    {'file': '1702.02925v1.2.png', 'table_id': '1702.02925v1.2', 'paper_id': '1702.02925', 'paper_title': 'EAC-Net: A Region-based Deep Enhancing and Cropping Approach for Facial Action Unit Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 951, 'height': 109, 'png_bytes': 7581, 'png_sha256': '71ca005e056cc8da10f6f1128adf0ca2875cfb60db07eb2d2e8e105a6946038c', 'n_rows': 4, 'n_cols': 13, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.279}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 948.0, 106.0]}, {'label': 'table row', 'box': [3.0, 3.0, 948.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 948.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 948.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 948.0, 106.0]}, {'label': 'table column', 'box': [3.0, 3.0, 205.6, 106.0]}, {'label': 'table column', 'box': [205.6, 3.0, 267.4, 106.0]}, {'label': 'table column', 'box': [267.4, 3.0, 329.2, 106.0]}, {'label': 'table column', 'box': [329.2, 3.0, 391.0, 106.0]}, {'label': 'table column', 'box': [391.0, 3.0, 452.8, 106.0]}, {'label': 'table column', 'box': [452.8, 3.0, 514.7, 106.0]}, {'label': 'table column', 'box': [514.7, 3.0, 576.5, 106.0]}, {'label': 'table column', 'box': [576.5, 3.0, 638.3, 106.0]}, {'label': 'table column', 'box': [638.3, 3.0, 700.1, 106.0]}, {'label': 'table column', 'box': [700.1, 3.0, 761.9, 106.0]}, {'label': 'table column', 'box': [761.9, 3.0, 823.7, 106.0]}, {'label': 'table column', 'box': [823.7, 3.0, 885.5, 106.0]}, {'label': 'table column', 'box': [885.5, 3.0, 948.0, 106.0]}]},
    {'file': '1702.02925v1.3.png', 'table_id': '1702.02925v1.3', 'paper_id': '1702.02925', 'paper_title': 'EAC-Net: A Region-based Deep Enhancing and Cropping Approach for Facial Action Unit Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 556, 'height': 357, 'png_bytes': 17392, 'png_sha256': '8f3e2a4b9433b9def5654a34c5261029e087b8192e76eb50c91451d9e27854a9', 'n_rows': 14, 'n_cols': 7, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 11, 'ink_score': 0.269}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 553.0, 354.0]}, {'label': 'table row', 'box': [3.0, 3.0, 553.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 553.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 553.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 553.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 553.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 553.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 553.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 553.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 553.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 553.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 553.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 553.0, 303.4]}, {'label': 'table row', 'box': [3.0, 303.4, 553.0, 328.4]}, {'label': 'table row', 'box': [3.0, 328.4, 553.0, 354.0]}, {'label': 'table column', 'box': [3.0, 3.0, 64.4, 354.0]}, {'label': 'table column', 'box': [64.4, 3.0, 148.4, 354.0]}, {'label': 'table column', 'box': [148.4, 3.0, 230.0, 354.0]}, {'label': 'table column', 'box': [230.0, 3.0, 318.1, 354.0]}, {'label': 'table column', 'box': [318.1, 3.0, 404.1, 354.0]}, {'label': 'table column', 'box': [404.1, 3.0, 482.9, 354.0]}, {'label': 'table column', 'box': [482.9, 3.0, 553.0, 354.0]}]},
    {'file': '1702.02925v1.6.png', 'table_id': '1702.02925v1.6', 'paper_id': '1702.02925', 'paper_title': 'EAC-Net: A Region-based Deep Enhancing and Cropping Approach for Facial Action Unit Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 525, 'height': 257, 'png_bytes': 11179, 'png_sha256': 'd050d33a2edfa2e8a6487ccc4ea122f6d41d370b21e0fb59b898254c9d29c98c', 'n_rows': 10, 'n_cols': 7, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.25}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 522.0, 254.0]}, {'label': 'table row', 'box': [3.0, 3.0, 522.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 522.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 522.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 522.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 522.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 522.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 522.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 522.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 522.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 522.0, 254.0]}, {'label': 'table column', 'box': [3.0, 3.0, 65.4, 254.0]}, {'label': 'table column', 'box': [65.4, 3.0, 149.4, 254.0]}, {'label': 'table column', 'box': [149.4, 3.0, 216.9, 254.0]}, {'label': 'table column', 'box': [216.9, 3.0, 305.0, 254.0]}, {'label': 'table column', 'box': [305.0, 3.0, 391.0, 254.0]}, {'label': 'table column', 'box': [391.0, 3.0, 452.8, 254.0]}, {'label': 'table column', 'box': [452.8, 3.0, 522.0, 254.0]}]},
    {'file': '1702.02925v1.7.png', 'table_id': '1702.02925v1.7', 'paper_id': '1702.02925', 'paper_title': 'EAC-Net: A Region-based Deep Enhancing and Cropping Approach for Facial Action Unit Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 525, 'height': 257, 'png_bytes': 11533, 'png_sha256': '75d5f49a7956928490f4c39e9bd8c5711145cfebd12a9319f2e2e9904d9856f7', 'n_rows': 10, 'n_cols': 7, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.254}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 522.0, 254.0]}, {'label': 'table row', 'box': [3.0, 3.0, 522.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 522.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 522.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 522.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 522.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 522.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 522.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 522.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 522.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 522.0, 254.0]}, {'label': 'table column', 'box': [3.0, 3.0, 65.4, 254.0]}, {'label': 'table column', 'box': [65.4, 3.0, 149.4, 254.0]}, {'label': 'table column', 'box': [149.4, 3.0, 216.9, 254.0]}, {'label': 'table column', 'box': [216.9, 3.0, 305.0, 254.0]}, {'label': 'table column', 'box': [305.0, 3.0, 391.0, 254.0]}, {'label': 'table column', 'box': [391.0, 3.0, 452.8, 254.0]}, {'label': 'table column', 'box': [452.8, 3.0, 522.0, 254.0]}]},
    {'file': '1705.02407v2.1.png', 'table_id': '1705.02407v2.1', 'paper_id': '1705.02407', 'paper_title': 'Knowledge-Guided Deep Fractal Neural Networks for Human Pose Estimation', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 732, 'height': 197, 'png_bytes': 14235, 'png_sha256': '1aa7cdb0c6e0d21e5981b3a4a4b0bdbf536a54671a174d1bda8773ee762d1696', 'n_rows': 8, 'n_cols': 9, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.288}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 729.0, 194.0]}, {'label': 'table row', 'box': [3.0, 3.0, 729.0, 29.6]}, {'label': 'table row', 'box': [3.0, 29.6, 729.0, 54.9]}, {'label': 'table row', 'box': [3.0, 54.9, 729.0, 77.7]}, {'label': 'table row', 'box': [3.0, 77.7, 729.0, 101.0]}, {'label': 'table row', 'box': [3.0, 101.0, 729.0, 124.2]}, {'label': 'table row', 'box': [3.0, 124.2, 729.0, 147.1]}, {'label': 'table row', 'box': [3.0, 147.1, 729.0, 169.9]}, {'label': 'table row', 'box': [3.0, 169.9, 729.0, 194.0]}, {'label': 'table column', 'box': [3.0, 3.0, 208.3, 194.0]}, {'label': 'table column', 'box': [208.3, 3.0, 277.5, 194.0]}, {'label': 'table column', 'box': [277.5, 3.0, 339.8, 194.0]}, {'label': 'table column', 'box': [339.8, 3.0, 400.2, 194.0]}, {'label': 'table column', 'box': [400.2, 3.0, 462.4, 194.0]}, {'label': 'table column', 'box': [462.4, 3.0, 522.6, 194.0]}, {'label': 'table column', 'box': [522.6, 3.0, 591.2, 194.0]}, {'label': 'table column', 'box': [591.2, 3.0, 657.7, 194.0]}, {'label': 'table column', 'box': [657.7, 3.0, 729.0, 194.0]}]},
    {'file': '1705.04915v1.2.png', 'table_id': '1705.04915v1.2', 'paper_id': '1705.04915', 'paper_title': 'Discovering Multiple Truths with a Hybrid Model', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 424, 'height': 108, 'png_bytes': 4817, 'png_sha256': '771b8b433a470c8163f2caa7588d741db041bbef5fae2e4ff963fea5bd177f40', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 3, 'dy': 11, 'ink_score': 0.306}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 421.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 421.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 421.0, 53.5]}, {'label': 'table row', 'box': [3.0, 53.5, 421.0, 78.5]}, {'label': 'table row', 'box': [3.0, 78.5, 421.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 127.4, 105.0]}, {'label': 'table column', 'box': [127.4, 3.0, 216.8, 105.0]}, {'label': 'table column', 'box': [216.8, 3.0, 283.4, 105.0]}, {'label': 'table column', 'box': [283.4, 3.0, 358.0, 105.0]}, {'label': 'table column', 'box': [358.0, 3.0, 421.0, 105.0]}]},
    {'file': '1706.08653v2.10.png', 'table_id': '1706.08653v2.10', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 511, 'height': 108, 'png_bytes': 6444, 'png_sha256': '3f052a88d454c1122241d448bb7281eb9625ae494d571a196f5ec73a6240fd96', 'n_rows': 4, 'n_cols': 7, 'n_spanning': 4, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.258}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 508.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 508.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 508.0, 55.5]}, {'label': 'table row', 'box': [3.0, 55.5, 508.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 508.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 114.6, 105.0]}, {'label': 'table column', 'box': [114.6, 3.0, 180.6, 105.0]}, {'label': 'table column', 'box': [180.6, 3.0, 248.4, 105.0]}, {'label': 'table column', 'box': [248.4, 3.0, 316.9, 105.0]}, {'label': 'table column', 'box': [316.9, 3.0, 379.9, 105.0]}, {'label': 'table column', 'box': [379.9, 3.0, 442.3, 105.0]}, {'label': 'table column', 'box': [442.3, 3.0, 508.0, 105.0]}, {'label': 'table spanning cell', 'box': [114.6, 3.0, 180.6, 55.5]}, {'label': 'table spanning cell', 'box': [180.6, 3.0, 248.4, 55.5]}, {'label': 'table spanning cell', 'box': [316.9, 3.0, 508.0, 28.6]}, {'label': 'table spanning cell', 'box': [248.4, 3.0, 316.9, 55.5]}]},
    {'file': '1706.08653v2.11.png', 'table_id': '1706.08653v2.11', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 521, 'height': 210, 'png_bytes': 11232, 'png_sha256': '4ff0cb5aa88482a39589f46bff1c4dbff1ee415468d19281ba43d895d459d574', 'n_rows': 8, 'n_cols': 7, 'n_spanning': 4, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.26}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 518.0, 207.0]}, {'label': 'table row', 'box': [3.0, 3.0, 518.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 518.0, 55.5]}, {'label': 'table row', 'box': [3.0, 55.5, 518.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 518.0, 104.6]}, {'label': 'table row', 'box': [3.0, 104.6, 518.0, 129.9]}, {'label': 'table row', 'box': [3.0, 129.9, 518.0, 155.3]}, {'label': 'table row', 'box': [3.0, 155.3, 518.0, 180.6]}, {'label': 'table row', 'box': [3.0, 180.6, 518.0, 207.0]}, {'label': 'table column', 'box': [3.0, 3.0, 124.4, 207.0]}, {'label': 'table column', 'box': [124.4, 3.0, 190.4, 207.0]}, {'label': 'table column', 'box': [190.4, 3.0, 258.2, 207.0]}, {'label': 'table column', 'box': [258.2, 3.0, 326.7, 207.0]}, {'label': 'table column', 'box': [326.7, 3.0, 389.7, 207.0]}, {'label': 'table column', 'box': [389.7, 3.0, 452.1, 207.0]}, {'label': 'table column', 'box': [452.1, 3.0, 518.0, 207.0]}, {'label': 'table spanning cell', 'box': [326.7, 3.0, 518.0, 28.6]}, {'label': 'table spanning cell', 'box': [258.2, 3.0, 326.7, 55.5]}, {'label': 'table spanning cell', 'box': [190.4, 3.0, 258.2, 55.5]}, {'label': 'table spanning cell', 'box': [124.4, 3.0, 190.4, 55.5]}]},
    {'file': '1706.08653v2.12.png', 'table_id': '1706.08653v2.12', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 521, 'height': 210, 'png_bytes': 11128, 'png_sha256': 'c1280c1d2f296b638a85dca00e1c16e0962cea91a40a162950af73d1f08ddaab', 'n_rows': 8, 'n_cols': 7, 'n_spanning': 4, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.26}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 518.0, 207.0]}, {'label': 'table row', 'box': [3.0, 3.0, 518.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 518.0, 55.5]}, {'label': 'table row', 'box': [3.0, 55.5, 518.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 518.0, 104.6]}, {'label': 'table row', 'box': [3.0, 104.6, 518.0, 129.9]}, {'label': 'table row', 'box': [3.0, 129.9, 518.0, 155.3]}, {'label': 'table row', 'box': [3.0, 155.3, 518.0, 180.6]}, {'label': 'table row', 'box': [3.0, 180.6, 518.0, 207.0]}, {'label': 'table column', 'box': [3.0, 3.0, 124.4, 207.0]}, {'label': 'table column', 'box': [124.4, 3.0, 190.4, 207.0]}, {'label': 'table column', 'box': [190.4, 3.0, 258.2, 207.0]}, {'label': 'table column', 'box': [258.2, 3.0, 326.7, 207.0]}, {'label': 'table column', 'box': [326.7, 3.0, 389.7, 207.0]}, {'label': 'table column', 'box': [389.7, 3.0, 452.1, 207.0]}, {'label': 'table column', 'box': [452.1, 3.0, 518.0, 207.0]}, {'label': 'table spanning cell', 'box': [124.4, 3.0, 190.4, 55.5]}, {'label': 'table spanning cell', 'box': [258.2, 3.0, 326.7, 55.5]}, {'label': 'table spanning cell', 'box': [190.4, 3.0, 258.2, 55.5]}, {'label': 'table spanning cell', 'box': [326.7, 3.0, 518.0, 28.6]}]},
    {'file': '1706.08653v2.4.png', 'table_id': '1706.08653v2.4', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 473, 'height': 188, 'png_bytes': 11300, 'png_sha256': 'a0a7c916581e67679c80d6d0b5d708df2484439d29ee376231154613a6afa40f', 'n_rows': 7, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.284}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 470.0, 185.0]}, {'label': 'table row', 'box': [3.0, 3.0, 470.0, 31.1]}, {'label': 'table row', 'box': [3.0, 31.1, 470.0, 58.5]}, {'label': 'table row', 'box': [3.0, 58.5, 470.0, 83.4]}, {'label': 'table row', 'box': [3.0, 83.4, 470.0, 108.3]}, {'label': 'table row', 'box': [3.0, 108.3, 470.0, 133.2]}, {'label': 'table row', 'box': [3.0, 133.2, 470.0, 158.6]}, {'label': 'table row', 'box': [3.0, 158.6, 470.0, 185.0]}, {'label': 'table column', 'box': [3.0, 3.0, 152.1, 185.0]}, {'label': 'table column', 'box': [152.1, 3.0, 231.4, 185.0]}, {'label': 'table column', 'box': [231.4, 3.0, 310.7, 185.0]}, {'label': 'table column', 'box': [310.7, 3.0, 389.9, 185.0]}, {'label': 'table column', 'box': [389.9, 3.0, 470.0, 185.0]}]},
    {'file': '1706.08653v2.5.png', 'table_id': '1706.08653v2.5', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 515, 'height': 388, 'png_bytes': 17975, 'png_sha256': 'dc588363c049198abcb20187a624d20bc7afd60bfe2fbab24678fb792f0c0235', 'n_rows': 15, 'n_cols': 5, 'n_spanning': 6, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.273}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 512.0, 385.0]}, {'label': 'table row', 'box': [3.0, 3.0, 512.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 512.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 512.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 512.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 512.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 512.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 512.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 512.0, 204.2]}, {'label': 'table row', 'box': [3.0, 204.2, 512.0, 231.4]}, {'label': 'table row', 'box': [3.0, 231.4, 512.0, 258.2]}, {'label': 'table row', 'box': [3.0, 258.2, 512.0, 283.5]}, {'label': 'table row', 'box': [3.0, 283.5, 512.0, 308.8]}, {'label': 'table row', 'box': [3.0, 308.8, 512.0, 333.7]}, {'label': 'table row', 'box': [3.0, 333.7, 512.0, 359.1]}, {'label': 'table row', 'box': [3.0, 359.1, 512.0, 385.0]}, {'label': 'table column', 'box': [3.0, 3.0, 197.4, 385.0]}, {'label': 'table column', 'box': [197.4, 3.0, 273.8, 385.0]}, {'label': 'table column', 'box': [273.8, 3.0, 353.0, 385.0]}, {'label': 'table column', 'box': [353.0, 3.0, 432.3, 385.0]}, {'label': 'table column', 'box': [432.3, 3.0, 512.0, 385.0]}, {'label': 'table spanning cell', 'box': [353.0, 231.4, 512.0, 258.2]}, {'label': 'table spanning cell', 'box': [353.0, 3.0, 512.0, 28.6]}, {'label': 'table spanning cell', 'box': [197.4, 231.4, 353.0, 258.2]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 197.4, 54.0]}, {'label': 'table spanning cell', 'box': [197.4, 3.0, 353.0, 28.6]}, {'label': 'table spanning cell', 'box': [3.0, 231.4, 197.4, 283.5]}]},
    {'file': '1706.08653v2.6.png', 'table_id': '1706.08653v2.6', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 936, 'height': 358, 'png_bytes': 27072, 'png_sha256': '0abee7dd6ce2d94848fa735446042d94a291a6a783b64e79ee1e11faa4994b60', 'n_rows': 14, 'n_cols': 13, 'n_spanning': 8, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.261}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 933.0, 355.0]}, {'label': 'table row', 'box': [3.0, 3.0, 933.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 933.0, 55.5]}, {'label': 'table row', 'box': [3.0, 55.5, 933.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 933.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 933.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 933.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 933.0, 178.9]}, {'label': 'table row', 'box': [3.0, 178.9, 933.0, 203.8]}, {'label': 'table row', 'box': [3.0, 203.8, 933.0, 228.7]}, {'label': 'table row', 'box': [3.0, 228.7, 933.0, 253.6]}, {'label': 'table row', 'box': [3.0, 253.6, 933.0, 278.5]}, {'label': 'table row', 'box': [3.0, 278.5, 933.0, 303.9]}, {'label': 'table row', 'box': [3.0, 303.9, 933.0, 329.2]}, {'label': 'table row', 'box': [3.0, 329.2, 933.0, 355.0]}, {'label': 'table column', 'box': [3.0, 3.0, 124.6, 355.0]}, {'label': 'table column', 'box': [124.6, 3.0, 192.0, 355.0]}, {'label': 'table column', 'box': [192.0, 3.0, 259.3, 355.0]}, {'label': 'table column', 'box': [259.3, 3.0, 326.7, 355.0]}, {'label': 'table column', 'box': [326.7, 3.0, 394.0, 355.0]}, {'label': 'table column', 'box': [394.0, 3.0, 461.3, 355.0]}, {'label': 'table column', 'box': [461.3, 3.0, 528.7, 355.0]}, {'label': 'table column', 'box': [528.7, 3.0, 596.0, 355.0]}, {'label': 'table column', 'box': [596.0, 3.0, 663.4, 355.0]}, {'label': 'table column', 'box': [663.4, 3.0, 730.7, 355.0]}, {'label': 'table column', 'box': [730.7, 3.0, 798.0, 355.0]}, {'label': 'table column', 'box': [798.0, 3.0, 865.4, 355.0]}, {'label': 'table column', 'box': [865.4, 3.0, 933.0, 355.0]}, {'label': 'table spanning cell', 'box': [730.7, 3.0, 933.0, 28.6]}, {'label': 'table spanning cell', 'box': [124.6, 329.2, 326.7, 355.0]}, {'label': 'table spanning cell', 'box': [326.7, 329.2, 528.7, 355.0]}, {'label': 'table spanning cell', 'box': [528.7, 329.2, 730.7, 355.0]}, {'label': 'table spanning cell', 'box': [730.7, 329.2, 933.0, 355.0]}, {'label': 'table spanning cell', 'box': [326.7, 3.0, 528.7, 28.6]}, {'label': 'table spanning cell', 'box': [124.6, 3.0, 326.7, 28.6]}, {'label': 'table spanning cell', 'box': [528.7, 3.0, 730.7, 28.6]}]},
    {'file': '1706.08653v2.7.png', 'table_id': '1706.08653v2.7', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 541, 'height': 234, 'png_bytes': 12578, 'png_sha256': 'c3203a8ae04b52e10777872766add112cc27bbda2fd2539b71f69f0852efefd3', 'n_rows': 9, 'n_cols': 7, 'n_spanning': 5, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.265}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 538.0, 231.0]}, {'label': 'table row', 'box': [3.0, 3.0, 538.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 538.0, 55.5]}, {'label': 'table row', 'box': [3.0, 55.5, 538.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 538.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 538.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 538.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 538.0, 179.3]}, {'label': 'table row', 'box': [3.0, 179.3, 538.0, 205.1]}, {'label': 'table row', 'box': [3.0, 205.1, 538.0, 231.0]}, {'label': 'table column', 'box': [3.0, 3.0, 124.6, 231.0]}, {'label': 'table column', 'box': [124.6, 3.0, 196.8, 231.0]}, {'label': 'table column', 'box': [196.8, 3.0, 264.2, 231.0]}, {'label': 'table column', 'box': [264.2, 3.0, 333.6, 231.0]}, {'label': 'table column', 'box': [333.6, 3.0, 403.0, 231.0]}, {'label': 'table column', 'box': [403.0, 3.0, 470.3, 231.0]}, {'label': 'table column', 'box': [470.3, 3.0, 538.0, 231.0]}, {'label': 'table spanning cell', 'box': [124.6, 3.0, 333.6, 28.6]}, {'label': 'table spanning cell', 'box': [333.6, 3.0, 538.0, 28.6]}, {'label': 'table spanning cell', 'box': [124.6, 205.1, 333.6, 231.0]}, {'label': 'table spanning cell', 'box': [333.6, 205.1, 538.0, 231.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 124.6, 55.5]}]},
    {'file': '1706.08653v2.8.png', 'table_id': '1706.08653v2.8', 'paper_id': '1706.08653', 'paper_title': 'A Unified approach for Conventional Zero-shot, Generalized Zero-shot and Few-shot Learning', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 445, 'height': 108, 'png_bytes': 6352, 'png_sha256': '0f3cc5c2604cab2a9eeb6b6b0ae018a379a000a07ee488e26e85bcaf9083eabd', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 3, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.29}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 442.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 442.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 442.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 442.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 442.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 127.3, 105.0]}, {'label': 'table column', 'box': [127.3, 3.0, 203.9, 105.0]}, {'label': 'table column', 'box': [203.9, 3.0, 283.2, 105.0]}, {'label': 'table column', 'box': [283.2, 3.0, 362.5, 105.0]}, {'label': 'table column', 'box': [362.5, 3.0, 442.0, 105.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 127.3, 54.0]}, {'label': 'table spanning cell', 'box': [127.3, 3.0, 283.2, 28.6]}, {'label': 'table spanning cell', 'box': [283.2, 3.0, 442.0, 28.6]}]},
    {'file': '1706.10239v2.1.png', 'table_id': '1706.10239v2.1', 'paper_id': '1706.10239', 'paper_title': 'Towards Understanding Generalization of Deep Learning: Perspective of Loss Landscapes', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 587, 'height': 83, 'png_bytes': 4970, 'png_sha256': '6efc2ae57224addd72ff6bd857efb8564b47cc9d6b5e6df89ad5f3809fcac662', 'n_rows': 3, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 11, 'ink_score': 0.245}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 584.0, 80.0]}, {'label': 'table row', 'box': [3.0, 3.0, 584.0, 26.8]}, {'label': 'table row', 'box': [3.0, 26.8, 584.0, 52.5]}, {'label': 'table row', 'box': [3.0, 52.5, 584.0, 80.0]}, {'label': 'table column', 'box': [3.0, 3.0, 179.1, 80.0]}, {'label': 'table column', 'box': [179.1, 3.0, 313.5, 80.0]}, {'label': 'table column', 'box': [313.5, 3.0, 448.0, 80.0]}, {'label': 'table column', 'box': [448.0, 3.0, 584.0, 80.0]}]},
    {'file': '1708.06828v1.5.png', 'table_id': '1708.06828v1.5', 'paper_id': '1708.06828', 'paper_title': 'Classification of Radiology Reports Using Neural Attention Models', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 611, 'height': 186, 'png_bytes': 8597, 'png_sha256': '1afd78d0855d4f7b223be3f1c8232d590d6dba43b63b157c085b69ec48e75e3c', 'n_rows': 7, 'n_cols': 5, 'n_spanning': 3, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 10, 'ink_score': 0.295}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 608.0, 183.0]}, {'label': 'table row', 'box': [3.0, 3.0, 608.0, 27.6]}, {'label': 'table row', 'box': [3.0, 27.6, 608.0, 55.0]}, {'label': 'table row', 'box': [3.0, 55.0, 608.0, 82.4]}, {'label': 'table row', 'box': [3.0, 82.4, 608.0, 107.3]}, {'label': 'table row', 'box': [3.0, 107.3, 608.0, 132.3]}, {'label': 'table row', 'box': [3.0, 132.3, 608.0, 157.2]}, {'label': 'table row', 'box': [3.0, 157.2, 608.0, 183.0]}, {'label': 'table column', 'box': [3.0, 3.0, 90.6, 183.0]}, {'label': 'table column', 'box': [90.6, 3.0, 291.0, 183.0]}, {'label': 'table column', 'box': [291.0, 3.0, 461.9, 183.0]}, {'label': 'table column', 'box': [461.9, 3.0, 532.9, 183.0]}, {'label': 'table column', 'box': [532.9, 3.0, 608.0, 183.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 90.6, 55.0]}, {'label': 'table spanning cell', 'box': [90.6, 3.0, 291.0, 55.0]}, {'label': 'table spanning cell', 'box': [291.0, 3.0, 608.0, 27.6]}]},
    {'file': '1709.04959v2.2.png', 'table_id': '1709.04959v2.2', 'paper_id': '1709.04959', 'paper_title': 'Towards Baselines for Shoulder Surfing on Mobile Authentication', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 609, 'height': 260, 'png_bytes': 14478, 'png_sha256': '4c33de0102c867ec14bd35e37361c13b0ef0e2321b669c3926778939361671cb', 'n_rows': 11, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.291}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 606.0, 257.0]}, {'label': 'table row', 'box': [3.0, 3.0, 606.0, 27.5]}, {'label': 'table row', 'box': [3.0, 27.5, 606.0, 50.7]}, {'label': 'table row', 'box': [3.0, 50.7, 606.0, 73.6]}, {'label': 'table row', 'box': [3.0, 73.6, 606.0, 96.4]}, {'label': 'table row', 'box': [3.0, 96.4, 606.0, 119.2]}, {'label': 'table row', 'box': [3.0, 119.2, 606.0, 142.5]}, {'label': 'table row', 'box': [3.0, 142.5, 606.0, 165.7]}, {'label': 'table row', 'box': [3.0, 165.7, 606.0, 188.6]}, {'label': 'table row', 'box': [3.0, 188.6, 606.0, 211.4]}, {'label': 'table row', 'box': [3.0, 211.4, 606.0, 234.2]}, {'label': 'table row', 'box': [3.0, 234.2, 606.0, 257.0]}, {'label': 'table column', 'box': [3.0, 3.0, 86.1, 257.0]}, {'label': 'table column', 'box': [86.1, 3.0, 328.9, 257.0]}, {'label': 'table column', 'box': [328.9, 3.0, 427.2, 257.0]}, {'label': 'table column', 'box': [427.2, 3.0, 606.0, 257.0]}]},
    {'file': '1709.08718v1.4.png', 'table_id': '1709.08718v1.4', 'paper_id': '1709.08718', 'paper_title': 'Detecting Censor Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 347, 'height': 173, 'png_bytes': 7875, 'png_sha256': 'c5b9599f72aa09976da9808efacdc8c49c9b8ce3b715784ef227072b1207b200', 'n_rows': 7, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 8, 'ink_score': 0.301}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 344.0, 170.0]}, {'label': 'table row', 'box': [3.0, 3.0, 344.0, 26.1]}, {'label': 'table row', 'box': [3.0, 26.1, 344.0, 47.8]}, {'label': 'table row', 'box': [3.0, 47.8, 344.0, 76.3]}, {'label': 'table row', 'box': [3.0, 76.3, 344.0, 101.2]}, {'label': 'table row', 'box': [3.0, 101.2, 344.0, 126.1]}, {'label': 'table row', 'box': [3.0, 126.1, 344.0, 151.0]}, {'label': 'table row', 'box': [3.0, 151.0, 344.0, 170.0]}, {'label': 'table column', 'box': [3.0, 3.0, 214.6, 170.0]}, {'label': 'table column', 'box': [214.6, 3.0, 344.0, 170.0]}]},
    {'file': '1709.08718v1.5.png', 'table_id': '1709.08718v1.5', 'paper_id': '1709.08718', 'paper_title': 'Detecting Censor Detection', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 466, 'height': 146, 'png_bytes': 8002, 'png_sha256': '9e4ac39ad4e5f96613f1d479bdc44ed06625984b8e37ce4af13c7ace2f7f9edd', 'n_rows': 6, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 6, 'ink_score': 0.297}, 'objects': [{'label': 'table', 'box': [3.0, 2.0, 463.0, 143.0]}, {'label': 'table row', 'box': [3.0, 2.0, 463.0, 24.1]}, {'label': 'table row', 'box': [3.0, 24.1, 463.0, 49.4]}, {'label': 'table row', 'box': [3.0, 49.4, 463.0, 74.3]}, {'label': 'table row', 'box': [3.0, 74.3, 463.0, 99.2]}, {'label': 'table row', 'box': [3.0, 99.2, 463.0, 124.1]}, {'label': 'table row', 'box': [3.0, 124.1, 463.0, 143.0]}, {'label': 'table column', 'box': [3.0, 2.0, 167.3, 143.0]}, {'label': 'table column', 'box': [167.3, 2.0, 246.3, 143.0]}, {'label': 'table column', 'box': [246.3, 2.0, 325.0, 143.0]}, {'label': 'table column', 'box': [325.0, 2.0, 463.0, 143.0]}]},
    {'file': '1711.04434v1.4.png', 'table_id': '1711.04434v1.4', 'paper_id': '1711.04434', 'paper_title': 'Faithful to the Original: Fact Aware Neural Abstractive Summarization', 'paper_license': 'CC0', 'scitsr_split': 'test', 'width': 370, 'height': 108, 'png_bytes': 4812, 'png_sha256': 'ecfecbc94e67f55cbafb96b25478411a68c48f175a1fd0c82d5471f8e3b8e666', 'n_rows': 4, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.29}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 367.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 367.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 367.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 367.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 367.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 159.1, 105.0]}, {'label': 'table column', 'box': [159.1, 3.0, 233.1, 105.0]}, {'label': 'table column', 'box': [233.1, 3.0, 300.1, 105.0]}, {'label': 'table column', 'box': [300.1, 3.0, 367.0, 105.0]}]},
    {'file': '1711.04434v1.5.png', 'table_id': '1711.04434v1.5', 'paper_id': '1711.04434', 'paper_title': 'Faithful to the Original: Fact Aware Neural Abstractive Summarization', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 267, 'height': 158, 'png_bytes': 5244, 'png_sha256': 'cf7a216a2a8d0c9b008554e8e173685135367d262213dd2b7ffaf6408efc244f', 'n_rows': 6, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.251}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 264.0, 155.0]}, {'label': 'table row', 'box': [3.0, 3.0, 264.0, 25.6]}, {'label': 'table row', 'box': [3.0, 25.6, 264.0, 51.1]}, {'label': 'table row', 'box': [3.0, 51.1, 264.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 264.0, 104.6]}, {'label': 'table row', 'box': [3.0, 104.6, 264.0, 131.0]}, {'label': 'table row', 'box': [3.0, 131.0, 264.0, 155.0]}, {'label': 'table column', 'box': [3.0, 3.0, 145.0, 155.0]}, {'label': 'table column', 'box': [145.0, 3.0, 264.0, 155.0]}]},
    {'file': '1711.04434v1.7.png', 'table_id': '1711.04434v1.7', 'paper_id': '1711.04434', 'paper_title': 'Faithful to the Original: Fact Aware Neural Abstractive Summarization', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 323, 'height': 184, 'png_bytes': 5223, 'png_sha256': '86cc072f82931ad07b140b688320a701ca548ed28f1ca149319da5bd3d846ff6', 'n_rows': 7, 'n_cols': 3, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.262}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 320.0, 181.0]}, {'label': 'table row', 'box': [3.0, 3.0, 320.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 320.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 320.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 320.0, 104.6]}, {'label': 'table row', 'box': [3.0, 104.6, 320.0, 129.9]}, {'label': 'table row', 'box': [3.0, 129.9, 320.0, 154.8]}, {'label': 'table row', 'box': [3.0, 154.8, 320.0, 181.0]}, {'label': 'table column', 'box': [3.0, 3.0, 106.2, 181.0]}, {'label': 'table column', 'box': [106.2, 3.0, 238.9, 181.0]}, {'label': 'table column', 'box': [238.9, 3.0, 320.0, 181.0]}, {'label': 'table spanning cell', 'box': [3.0, 104.6, 106.2, 181.0]}, {'label': 'table spanning cell', 'box': [3.0, 29.1, 106.2, 104.6]}]},
    {'file': '1801.00005v1.1.png', 'table_id': '1801.00005v1.1', 'paper_id': '1801.00005', 'paper_title': "Analytical Inverter Delay Modeling Using Matlab's Curve Fitting Toolbox", 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 499, 'height': 166, 'png_bytes': 5660, 'png_sha256': 'cfd1a9ef487f9b5a4473e67e1cea246899dc8a60a83300257059a85664c7e15c', 'n_rows': 6, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 33, 'dy': 11, 'ink_score': 0.259}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 496.0, 163.0]}, {'label': 'table row', 'box': [3.0, 3.0, 496.0, 32.7]}, {'label': 'table row', 'box': [3.0, 32.7, 496.0, 58.9]}, {'label': 'table row', 'box': [3.0, 58.9, 496.0, 84.7]}, {'label': 'table row', 'box': [3.0, 84.7, 496.0, 110.4]}, {'label': 'table row', 'box': [3.0, 110.4, 496.0, 136.2]}, {'label': 'table row', 'box': [3.0, 136.2, 496.0, 163.0]}, {'label': 'table column', 'box': [3.0, 3.0, 104.1, 163.0]}, {'label': 'table column', 'box': [104.1, 3.0, 224.1, 163.0]}, {'label': 'table column', 'box': [224.1, 3.0, 356.1, 163.0]}, {'label': 'table column', 'box': [356.1, 3.0, 496.0, 163.0]}]},
    {'file': '1803.02632v2.1.png', 'table_id': '1803.02632v2.1', 'paper_id': '1803.02632', 'paper_title': 'Extracting Action Sequences from Texts Based on Deep Reinforcement Learning', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 814, 'height': 108, 'png_bytes': 6606, 'png_sha256': 'dba2222cfdd894ab72cc59091f7384fc68f51e0b0226f2bd022a6d78a18661ca', 'n_rows': 4, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.294}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 811.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 811.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 811.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 811.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 811.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 569.5, 105.0]}, {'label': 'table column', 'box': [569.5, 3.0, 651.1, 105.0]}, {'label': 'table column', 'box': [651.1, 3.0, 723.2, 105.0]}, {'label': 'table column', 'box': [723.2, 3.0, 811.0, 105.0]}]},
    {'file': '1803.03670v1.1.png', 'table_id': '1803.03670v1.1', 'paper_id': '1803.03670', 'paper_title': 'IcoRating: A Deep-Learning System for Scam ICO Identification', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 522, 'height': 100, 'png_bytes': 5374, 'png_sha256': '480f7cccbced4c97853d9e816af105c813b70ebe26223b150b1b474e05fde2a6', 'n_rows': 4, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 11, 'ink_score': 0.269}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 519.0, 97.0]}, {'label': 'table row', 'box': [3.0, 3.0, 519.0, 27.1]}, {'label': 'table row', 'box': [3.0, 27.1, 519.0, 50.3]}, {'label': 'table row', 'box': [3.0, 50.3, 519.0, 73.6]}, {'label': 'table row', 'box': [3.0, 73.6, 519.0, 97.0]}, {'label': 'table column', 'box': [3.0, 3.0, 76.8, 97.0]}, {'label': 'table column', 'box': [76.8, 3.0, 186.5, 97.0]}, {'label': 'table column', 'box': [186.5, 3.0, 292.5, 97.0]}, {'label': 'table column', 'box': [292.5, 3.0, 407.1, 97.0]}, {'label': 'table column', 'box': [407.1, 3.0, 519.0, 97.0]}]},
    {'file': '1803.03670v1.3.png', 'table_id': '1803.03670v1.3', 'paper_id': '1803.03670', 'paper_title': 'IcoRating: A Deep-Learning System for Scam ICO Identification', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 193, 'height': 158, 'png_bytes': 4413, 'png_sha256': '0fd2c798121e9eb6683a76350811677f213c280698f1905193a5556b2de62b02', 'n_rows': 6, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.303}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 190.0, 155.0]}, {'label': 'table row', 'box': [3.0, 3.0, 190.0, 28.6]}, {'label': 'table row', 'box': [3.0, 28.6, 190.0, 53.5]}, {'label': 'table row', 'box': [3.0, 53.5, 190.0, 78.5]}, {'label': 'table row', 'box': [3.0, 78.5, 190.0, 103.4]}, {'label': 'table row', 'box': [3.0, 103.4, 190.0, 128.7]}, {'label': 'table row', 'box': [3.0, 128.7, 190.0, 155.0]}, {'label': 'table column', 'box': [3.0, 3.0, 117.4, 155.0]}, {'label': 'table column', 'box': [117.4, 3.0, 190.0, 155.0]}]},
    {'file': '1803.07835v1.2.png', 'table_id': '1803.07835v1.2', 'paper_id': '1803.07835', 'paper_title': 'Joint 3D Face Reconstruction and Dense Alignment with Position Map Regression Network', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 647, 'height': 59, 'png_bytes': 4352, 'png_sha256': 'fa17edb1048d99a90acbdf858f90fb2eb13a0f83ddcbf79662b150a183be3543', 'n_rows': 2, 'n_cols': 6, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.266}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 644.0, 56.0]}, {'label': 'table row', 'box': [3.0, 3.0, 644.0, 30.1]}, {'label': 'table row', 'box': [3.0, 30.1, 644.0, 56.0]}, {'label': 'table column', 'box': [3.0, 3.0, 97.4, 56.0]}, {'label': 'table column', 'box': [97.4, 3.0, 174.1, 56.0]}, {'label': 'table column', 'box': [174.1, 3.0, 274.6, 56.0]}, {'label': 'table column', 'box': [274.6, 3.0, 367.8, 56.0]}, {'label': 'table column', 'box': [367.8, 3.0, 511.9, 56.0]}, {'label': 'table column', 'box': [511.9, 3.0, 644.0, 56.0]}]},
    {'file': '1806.04450v1.1.png', 'table_id': '1806.04450v1.1', 'paper_id': '1806.04450', 'paper_title': 'An Ensemble Model for Sentiment Analysis of Hindi-English Code-Mixed Data', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 335, 'height': 183, 'png_bytes': 7110, 'png_sha256': 'd52970e8b7e6d08959d4ac8c088fe9db7ac79adf00a8f74cdb0a5465af400af8', 'n_rows': 7, 'n_cols': 2, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.313}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 332.0, 180.0]}, {'label': 'table row', 'box': [3.0, 3.0, 332.0, 29.1]}, {'label': 'table row', 'box': [3.0, 29.1, 332.0, 54.4]}, {'label': 'table row', 'box': [3.0, 54.4, 332.0, 79.3]}, {'label': 'table row', 'box': [3.0, 79.3, 332.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 332.0, 129.1]}, {'label': 'table row', 'box': [3.0, 129.1, 332.0, 154.0]}, {'label': 'table row', 'box': [3.0, 154.0, 332.0, 180.0]}, {'label': 'table column', 'box': [3.0, 3.0, 228.3, 180.0]}, {'label': 'table column', 'box': [228.3, 3.0, 332.0, 180.0]}]},
    {'file': '1806.04450v1.3.png', 'table_id': '1806.04450v1.3', 'paper_id': '1806.04450', 'paper_title': 'An Ensemble Model for Sentiment Analysis of Hindi-English Code-Mixed Data', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 740, 'height': 320, 'png_bytes': 18048, 'png_sha256': '7bf57596710bbfab6ec3cd868f27cd9d50965dcde918779e3658feee2383cd4f', 'n_rows': 10, 'n_cols': 5, 'n_spanning': 0, 'caption_chunks': 1, 'alignment': {'dx': 4, 'dy': 9, 'ink_score': 0.293}, 'objects': [{'label': 'table', 'box': [3.0, 63.0, 737.0, 317.0]}, {'label': 'table row', 'box': [3.0, 63.0, 737.0, 88.9]}, {'label': 'table row', 'box': [3.0, 88.9, 737.0, 114.2]}, {'label': 'table row', 'box': [3.0, 114.2, 737.0, 139.1]}, {'label': 'table row', 'box': [3.0, 139.1, 737.0, 164.0]}, {'label': 'table row', 'box': [3.0, 164.0, 737.0, 189.4]}, {'label': 'table row', 'box': [3.0, 189.4, 737.0, 215.1]}, {'label': 'table row', 'box': [3.0, 215.1, 737.0, 240.4]}, {'label': 'table row', 'box': [3.0, 240.4, 737.0, 265.3]}, {'label': 'table row', 'box': [3.0, 265.3, 737.0, 290.7]}, {'label': 'table row', 'box': [3.0, 290.7, 737.0, 317.0]}, {'label': 'table column', 'box': [3.0, 63.0, 288.6, 317.0]}, {'label': 'table column', 'box': [288.6, 63.0, 410.7, 317.0]}, {'label': 'table column', 'box': [410.7, 63.0, 531.2, 317.0]}, {'label': 'table column', 'box': [531.2, 63.0, 620.4, 317.0]}, {'label': 'table column', 'box': [620.4, 63.0, 737.0, 317.0]}]},
    {'file': '1807.06535v1.2.png', 'table_id': '1807.06535v1.2', 'paper_id': '1807.06535', 'paper_title': 'A framework for remote sensing images processing using deep learning technique', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 632, 'height': 108, 'png_bytes': 7779, 'png_sha256': 'd473ffddd1f3cae4e7e360610823b38da8fe1142272b4c5a2c0333e24aed9d58', 'n_rows': 3, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 12, 'ink_score': 0.238}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 629.0, 105.0]}, {'label': 'table row', 'box': [3.0, 3.0, 629.0, 36.3]}, {'label': 'table row', 'box': [3.0, 36.3, 629.0, 70.4]}, {'label': 'table row', 'box': [3.0, 70.4, 629.0, 105.0]}, {'label': 'table column', 'box': [3.0, 3.0, 87.4, 105.0]}, {'label': 'table column', 'box': [87.4, 3.0, 245.3, 105.0]}, {'label': 'table column', 'box': [245.3, 3.0, 483.9, 105.0]}, {'label': 'table column', 'box': [483.9, 3.0, 629.0, 105.0]}]},
    {'file': '1808.00179v1.1.png', 'table_id': '1808.00179v1.1', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 324, 'height': 110, 'png_bytes': 3582, 'png_sha256': 'aedd1b9067fc74dcb6cea15ab9da6700bab47ccce5b7a05a64b25c20b94580a5', 'n_rows': 4, 'n_cols': 4, 'n_spanning': 0, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 11, 'ink_score': 0.234}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 321.0, 107.0]}, {'label': 'table row', 'box': [3.0, 3.0, 321.0, 27.2]}, {'label': 'table row', 'box': [3.0, 27.2, 321.0, 52.9]}, {'label': 'table row', 'box': [3.0, 52.9, 321.0, 78.6]}, {'label': 'table row', 'box': [3.0, 78.6, 321.0, 107.0]}, {'label': 'table column', 'box': [3.0, 3.0, 103.6, 107.0]}, {'label': 'table column', 'box': [103.6, 3.0, 157.9, 107.0]}, {'label': 'table column', 'box': [157.9, 3.0, 238.7, 107.0]}, {'label': 'table column', 'box': [238.7, 3.0, 321.0, 107.0]}]},
    {'file': '1808.00179v1.11.png', 'table_id': '1808.00179v1.11', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 324, 'height': 82, 'png_bytes': 2982, 'png_sha256': '3f9ddbbdd76c6fc12001d72f05ea86802db85f7ddd74a8b196ccfc95e11f914a', 'n_rows': 3, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 10, 'ink_score': 0.235}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 321.0, 79.0]}, {'label': 'table row', 'box': [3.0, 3.0, 321.0, 29.2]}, {'label': 'table row', 'box': [3.0, 29.2, 321.0, 53.0]}, {'label': 'table row', 'box': [3.0, 53.0, 321.0, 79.0]}, {'label': 'table column', 'box': [3.0, 3.0, 64.1, 79.0]}, {'label': 'table column', 'box': [64.1, 3.0, 151.2, 79.0]}, {'label': 'table column', 'box': [151.2, 3.0, 254.5, 79.0]}, {'label': 'table column', 'box': [254.5, 3.0, 321.0, 79.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 64.1, 53.0]}, {'label': 'table spanning cell', 'box': [64.1, 3.0, 321.0, 29.2]}]},
    {'file': '1808.00179v1.14.png', 'table_id': '1808.00179v1.14', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 437, 'height': 135, 'png_bytes': 7057, 'png_sha256': 'a96088c40f3dc83764b2cb1f3f0754b2a889f47bc5b1d5f61163a18dd4f53c16', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.259}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 434.0, 132.0]}, {'label': 'table row', 'box': [3.0, 3.0, 434.0, 30.2]}, {'label': 'table row', 'box': [3.0, 30.2, 434.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 434.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 434.0, 105.4]}, {'label': 'table row', 'box': [3.0, 105.4, 434.0, 132.0]}, {'label': 'table column', 'box': [3.0, 3.0, 68.8, 132.0]}, {'label': 'table column', 'box': [68.8, 3.0, 190.5, 132.0]}, {'label': 'table column', 'box': [190.5, 3.0, 312.2, 132.0]}, {'label': 'table column', 'box': [312.2, 3.0, 434.0, 132.0]}, {'label': 'table spanning cell', 'box': [68.8, 3.0, 434.0, 30.2]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 68.8, 54.0]}]},
    {'file': '1808.00179v1.3.png', 'table_id': '1808.00179v1.3', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 346, 'height': 135, 'png_bytes': 4887, 'png_sha256': '688dade9d73da6f8cdba6e772c6cecbea3b20cea6525904bbcc77577715f0dce', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 15, 'dy': 11, 'ink_score': 0.261}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 343.0, 132.0]}, {'label': 'table row', 'box': [3.0, 3.0, 343.0, 30.2]}, {'label': 'table row', 'box': [3.0, 30.2, 343.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 343.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 343.0, 105.4]}, {'label': 'table row', 'box': [3.0, 105.4, 343.0, 132.0]}, {'label': 'table column', 'box': [3.0, 3.0, 67.8, 132.0]}, {'label': 'table column', 'box': [67.8, 3.0, 190.8, 132.0]}, {'label': 'table column', 'box': [190.8, 3.0, 266.1, 132.0]}, {'label': 'table column', 'box': [266.1, 3.0, 343.0, 132.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 67.8, 54.0]}, {'label': 'table spanning cell', 'box': [67.8, 3.0, 343.0, 30.2]}]},
    {'file': '1808.00179v1.6.png', 'table_id': '1808.00179v1.6', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 437, 'height': 135, 'png_bytes': 6941, 'png_sha256': '84ed2601d0b85d638dae918f9ac98a8d85a2f4dca6f3382e0b7ba2ab53d8c2b4', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.255}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 434.0, 132.0]}, {'label': 'table row', 'box': [3.0, 3.0, 434.0, 30.2]}, {'label': 'table row', 'box': [3.0, 30.2, 434.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 434.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 434.0, 105.4]}, {'label': 'table row', 'box': [3.0, 105.4, 434.0, 132.0]}, {'label': 'table column', 'box': [3.0, 3.0, 68.8, 132.0]}, {'label': 'table column', 'box': [68.8, 3.0, 190.5, 132.0]}, {'label': 'table column', 'box': [190.5, 3.0, 312.2, 132.0]}, {'label': 'table column', 'box': [312.2, 3.0, 434.0, 132.0]}, {'label': 'table spanning cell', 'box': [68.8, 3.0, 434.0, 30.2]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 68.8, 54.0]}]},
    {'file': '1808.00179v1.8.png', 'table_id': '1808.00179v1.8', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 485, 'height': 135, 'png_bytes': 6481, 'png_sha256': '86e7d3e2d279a69e6638f469336f2465e8283dc27887e97095403fa106207efa', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.229}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 482.0, 132.0]}, {'label': 'table row', 'box': [3.0, 3.0, 482.0, 30.2]}, {'label': 'table row', 'box': [3.0, 30.2, 482.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 482.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 482.0, 105.4]}, {'label': 'table row', 'box': [3.0, 105.4, 482.0, 132.0]}, {'label': 'table column', 'box': [3.0, 3.0, 68.8, 132.0]}, {'label': 'table column', 'box': [68.8, 3.0, 206.4, 132.0]}, {'label': 'table column', 'box': [206.4, 3.0, 344.1, 132.0]}, {'label': 'table column', 'box': [344.1, 3.0, 482.0, 132.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 68.8, 54.0]}, {'label': 'table spanning cell', 'box': [68.8, 3.0, 482.0, 30.2]}]},
    {'file': '1808.00179v1.9.png', 'table_id': '1808.00179v1.9', 'paper_id': '1808.00179', 'paper_title': 'Monolingual and Cross-lingual Zero-shot Style Transfer', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 485, 'height': 135, 'png_bytes': 6794, 'png_sha256': '1277d7724e752eb5cb7984cf9c0914bead9959718013a81c388857f83182127a', 'n_rows': 5, 'n_cols': 4, 'n_spanning': 2, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 11, 'ink_score': 0.228}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 482.0, 132.0]}, {'label': 'table row', 'box': [3.0, 3.0, 482.0, 30.2]}, {'label': 'table row', 'box': [3.0, 30.2, 482.0, 54.0]}, {'label': 'table row', 'box': [3.0, 54.0, 482.0, 79.7]}, {'label': 'table row', 'box': [3.0, 79.7, 482.0, 105.4]}, {'label': 'table row', 'box': [3.0, 105.4, 482.0, 132.0]}, {'label': 'table column', 'box': [3.0, 3.0, 68.8, 132.0]}, {'label': 'table column', 'box': [68.8, 3.0, 206.4, 132.0]}, {'label': 'table column', 'box': [206.4, 3.0, 344.1, 132.0]}, {'label': 'table column', 'box': [344.1, 3.0, 482.0, 132.0]}, {'label': 'table spanning cell', 'box': [3.0, 3.0, 68.8, 54.0]}, {'label': 'table spanning cell', 'box': [68.8, 3.0, 482.0, 30.2]}]},
    {'file': '1808.03399v1.1.png', 'table_id': '1808.03399v1.1', 'paper_id': '1808.03399', 'paper_title': 'Distinctiveness, complexity, and repeatability of online signature templates', 'paper_license': 'CC0', 'scitsr_split': 'train', 'width': 599, 'height': 126, 'png_bytes': 7560, 'png_sha256': 'aa5356c98b54d0a011b1eaaa3fc1be4f16ce53276b569580a4029dec71cc1f50', 'n_rows': 7, 'n_cols': 7, 'n_spanning': 4, 'caption_chunks': 0, 'alignment': {'dx': 16, 'dy': 8, 'ink_score': 0.275}, 'objects': [{'label': 'table', 'box': [3.0, 3.0, 596.0, 123.0]}, {'label': 'table row', 'box': [3.0, 3.0, 596.0, 19.9]}, {'label': 'table row', 'box': [3.0, 19.9, 596.0, 36.5]}, {'label': 'table row', 'box': [3.0, 36.5, 596.0, 53.6]}, {'label': 'table row', 'box': [3.0, 53.6, 596.0, 70.6]}, {'label': 'table row', 'box': [3.0, 70.6, 596.0, 87.2]}, {'label': 'table row', 'box': [3.0, 87.2, 596.0, 104.2]}, {'label': 'table row', 'box': [3.0, 104.2, 596.0, 123.0]}, {'label': 'table column', 'box': [3.0, 3.0, 171.4, 123.0]}, {'label': 'table column', 'box': [171.4, 3.0, 247.2, 123.0]}, {'label': 'table column', 'box': [247.2, 3.0, 321.5, 123.0]}, {'label': 'table column', 'box': [321.5, 3.0, 384.2, 123.0]}, {'label': 'table column', 'box': [384.2, 3.0, 460.0, 123.0]}, {'label': 'table column', 'box': [460.0, 3.0, 534.3, 123.0]}, {'label': 'table column', 'box': [534.3, 3.0, 596.0, 123.0]}, {'label': 'table spanning cell', 'box': [384.2, 3.0, 596.0, 19.9]}, {'label': 'table spanning cell', 'box': [171.4, 3.0, 384.2, 19.9]}, {'label': 'table spanning cell', 'box': [384.2, 19.9, 596.0, 36.5]}, {'label': 'table spanning cell', 'box': [171.4, 19.9, 384.2, 36.5]}]},
)

# Base64 PNG bytes per file.
SAMPLE_IMAGES_B64: dict[str, str] = {
    '1003.3684v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAjYAAABXCAAAAAAy6lA/AAAR70lEQVR42u2de1hU5fbHv8PNkXsglxQF7KiZigqImhqKpCQdOooKCnbSzFuWppVZmunPUisjPXnXyscbmoWmlclRwUuad1RAM0WUQi5yFYFhmPX7Y89lz8zewwzM7Bmfs9fzgO9+9/vs9+Na69l77cv3RUIQTTRTzU50gWhi2ogmjBFRpOgF0Yy3SCKSEKD6EcAkBMHmMh5I5DA1fDZwkTpVYEwvZbZ8ppNkGywnLZcj5nCTEZQOXJ01eYBboGrrzkMg2EV3THl2qedgvqPezivtG2Qs3oGBAbgnc2qUefoCqCpsRZIgppftjjcT+LEqr5a6R1VeLXWPMjyT+4y1kqZZAKDiobrL2YuPBQCkQRoPMgRm4KhhBczO2aSseTMBAM4dLuo03vuHUSYminbYDFISkepHbXcXDHVy/Eu5UeSDqAV3SdeuTpUmEJ/tGIE0zh0g3bmI5h0g+jYejtMPERFdHIK+y5W9LPtiOTfWrUE7iXKnOcczv4jp4LNvPuYE0mYhIlo/WYLX5s17d1q0JJSTJX/BEAQsXLZsbq8RWaScliEwA4evY3TSxGB0njQ+yv6pJg5FemhE8ycVyLNeXx5GJppO2HgoofPDsrwEfKJsrp2Am1r7FKuYf6P404bKTEwbqkYf5ebeGY2klzb5YY3cWPswiYhoWLzql7KD2xQDbhiXNlRt9xzT2PYCD8tNvEJEVBvd+px62mHxZAaORs8/iOhzrCGiaz5NHIr00faHNRJR4wiT00YnbDyUICKe2kbaPeJr5kxJD90h1dpXksP862jgdOdo6oXU1fcO02hIS+FgWviaHTdW3PnV6vkcoengKelmzTe24FIMZRqx7flYYA8A0o9q31NP6whzcJSP7QTAHnYAug2rNXwoDrRtUXYA7BaZXs44GkvJWxJP/vM4AOC3Abp70ixSywWVMOXEFzOdOBx5MJkHSxKmU3XpdWjZyGP3jcPJwGAACgDteVkYa48bZuYo7sXaCClu4lD6aLfzAQB9gloaE35K3rRJcN7CxKe/qkd2LrsRwNX3NYMUd2pVJdLN06om6v9SlnQVpzNl+UYiBuMOANwq7cex88duLjxYjaWF2kNVHSogeUkR5HkqNoc+RiZ9hjQCoCWA4wheFsaO4QUzcziyXTBIdSh5SRHktx8AqL8j1w6JDlrnvb8AgOQN7SF081iRASSOsPFT8qaN+9i9lQCq3JTVNG0c+ej2wFPYPb/uSHLyNgBA+orfEyYSAPwad6nunQ8aAODWmEWZn6QDwMqtHq0/DjM6bfIA0NIFXDuP9+XByu3j84bWSFWHCuhkb99Fv3x6VokJDDhuFM3Dc/2lwF4fwDWUlwUAcOG94SvNzPGP3qyN/oHMoZgDnJs4SbF53am+a9kh0UV7CyOGrjgtQ6TWkIw+RxU7FzfyIXGGjZ+SuyQu/D86gXVE9E0RvY57RLSmXSXRr57lRF2mMmOGR6QS3cQRItrfqYxIMX08EeW6ZRLRcqTR9UgiUgwwriSmDVhNRNu+1ytOiYie28SHRZHxRESx8apfTAcL6Nmob1WYRPRtmFEl8S/onDR2gOSqAZZ76PPZZ8sS/NYpSD1trAamZRxElIJ1qibzf1QeYNZlom+cy7RCooOW6gnA7z9aQzLsLxClOafzIHGHjZPSQEkMYEDnLQBKfJXPJN6Pdwei7XayRuSNAYIdrwG1ryc8AUim7TwCTOz9HIAEAEWXLwOSV0w52zzIGMl9tffkwQJc9YpraAO5XZ2gxAQAr2IjS5tN23ft6PiMQZaQt99+L/Xsjug8y3Gw/4/KA+T3BJ5+lMcVEhVaQt6uqZ2K3ljJGkIzokMB104BPEjcYeOl5E8byeTzV3Clh3LrYqXTyZMnf2vzB2vEM3aAvVMNcLmgOwB0lxzE/TNDVE8RI4J6d599ZLKxJTHygMUfcj9gKnbnweIxFhAL05S0kUbALjDGDrhlgAUAOuw/G1NtMQ4tYw7QDYAUNVwhUaN5Jq6/ccxncYVmSEFONwDRl5/mRuIJWzPSBi87bMbh55Ub+fCVSqXSHfPYd+lM7QfcYJp2Trm4Dg/17swPnddEv6wwzicdJHk49lQHnucBch4sHmMBsTABQC41Bqb6fH8pQLOAsu2GWADAa9gfBy3FoTOzttf1Q6JE2w0AksEbqi9phvwJf0NIPGHjpTSQNn4vbq90tFdudIV3eHh4ePiTzOYWrZFP4SEANNT/Ax1Ro+q9Wb34bHHKrhPG+aRVu7y6za/z7PSt4sHiMRaQjlX4GvXUpnEwAEkn4KsJhliUl5AiS3EYNJ2QaND2MlvDUasZ0gn3DCHxhI2X0o6vUAbwavmkl1QdPdumA0D9j4C0AbitNTrU5zQAnEMM2j/9OwDUAshOBZ6YHX/N2OKmav5cB74Mvs+DxWMsIB0r8TX+qQ2ArOyOfCwqq8uQRFmKw6CxQqKNlsNcWKvRUzOkXfdLAFCfxo3EE7YSE9MmO5eAmCergwBUogqQbjp4BcCq9kCvPwECIGsAQA2NgMuGPbeBxuVjX4RkY/o1gNaiFlj7CICsn9HFjSSU1/0XebCAhhoAaGhQ/UJDjRYQCxMALhrzQIAOtooAAPop7jVelkrIAKBu+t0lISoOJYFZOOpQr2oy/0flARoANKBRKyTaaI3jKwBg3cR2miGSTad/BrA+gBuJJ2z8lFw34LnDvVzD/0O0fC/RimHebt1GXyc62v/9re/8RES3Aj9anEGn/un5RNyZb4e4Bo0pJzo85JMNcYtkREQnnkv5bu5u+IxOi/lg50/vrTLyBpw+7FDN+XaIiCi9Kw/W2Tgvr9jis3Fe3nHr47y84y4wHRogbUyi8P1N3oAXJ4bCMSEpKXFwIHoqeFhy/hUM91FJSbG9hv+XiM7GeXnF/qQiaDnHwcTBfm5+gxPVxy5WH6DDyPJ5vd16zSRWSLTQqEf65KWHjs4eVaE15HToB3uW7CU+JM6wcVIaeJXJaQXXmUcUsgsP9Hf+ld2gHndZVnfx7qNKmeL6lVoyNm2uXyTetJF3yDH5tRwLSGP32sqMe5VpPpZmcRhrqpBooZ0nRdaWjRcUukMKcxsNIHGEjZsSRORgwsW0nerhN9fFpG1bzbh2QG+gNdDFhKMbGms/a8OXpl76WUAaWzfDsaUlhaksluLQCokWWhgkISEcQ/z9DSFxhI2f0gGPhc2M/LutGQ5Tlp5pEyxm4bCcm5qmfEyUC05rpza2/CiKqatb2wKLeTgs5iYjKG0gbRwcjOjtPXtNy2faMrGfTbA0zdFcM4ubjKC0AeXCw9ZcD+/0emtcWjz7I+emFAPCsDTN0Xwzg5uaoGSnjBXTBqLQ5LETvIjyOtHQPHmdaKJB1ICLJqaNaGLaiAZxxQnRxBUnxBtwkeOxWnFCtMfPjHuVWa0UyXn7SwDO1RaAGa9EiO4U04ZtJaknjvtPdpFlX4tc5IcHu08dC5jojNIjbZepX9CXbYYF06b4zQ0emi3FpgLpo9k+PC1hbUJYH+e7p4dFWTmOKgdVr7pb2GOWn7q/fGNVbcPMLgBAW6652r/rYraS2JjPtK4ikYjoQVhAod5qC0REtBFtZE1/VoTmfJJUMndKHxSyVkIYt4ToQue/uVumWTM/j9KYHwDHhYqWHqZFHBoHlSTeosrRHqfVe2aWEH3qlE5EshFJjbQ7pKTFlKZ83XcTSYzgD0lEdA+vEhHRSQxVDRg7HocslDZ1d+Qp7LTZ51FPRFNGc7cETpvn18zflEdk1bTROGh6HhFVewXLVdJO6R6iEjxLREscq4howDSzpI2pJXFHHNdbbQEACttMQKqFTsCtArVfQO8c4AQg8sAjzpbA5jvjk8lBVr5CaRx08NlywDUy77py258AuKAMwMYQNwD9tj+yxuO+KnSG7moLALBnzFDvtHpBnNSY7gEAT9ZncLX+t0vV9vUyAK5QqSkTq8cAp/AiUFHgCgA+Dy9bI22+s5+nu9oCAOD4IMcxlYcEcU1JuSsAeOAPrpbgkcpK+Syl0lbS5kShH4ArDt2hWehItjJ8AdDanph4XxfuTgoAau9DlrPvcBqjo73yOeSXMz6aqtRs53Wwx7j1u14SwjXVcAEAe1RytQTPmpzZkrSIw4G2kTZ2TgDOZc3R3FH+/sOx0L0uQKvuDwDgDiqETZvcVMA1ebWTarUFAHeTvtscDABITQQGBhww36dlMKQ7cwIACWq5WkIHakcIMHLpnO9t50JV/2r0x5qtvhE3Z437ug2w5F/3/SG7BLmwadNxtl5Xh/2BMefdAOCAfy4QXHAgUQC3uDHyezncuFpCBykEAMI3lvjYTNq877OPrfiXdN7e7oUz9oj7/LW1DdtiT7ex/htw1WoLOV3aBwQEvGqxeykt82ROKXXw4GoJHKMzx5lMzraZrNny988653zvPud/BTDnq8xjbxWhp7BnG3Av2FMEALvm9ABAy36p8LS8Xzz8iwDgPiK4WgIHKa6y2gmQwdtWsuZg1g47XLNjFnSSDZSfcQK8cBMAAgOB/KCe1j/bKFdboOweACAZJ0uzsFPKqgFJbB4A3PYK42oJHKWe65wA5Hp0tZGsOf3bKjtgfysAZXWoO3+lGMA99AROjKwEKo8vdBAybSq17lF0VlvYWc50v4gtCst4o55ZfqE4oB+AOTm3Afphjj1nS1gb3RXAncwUa6tblQ66nvxg+rSpEzYHAaXtB8J9xNEA4G525CDg4I/3gC+DJ0C4d1JZ8U+7ug95W7mls9rCiXD31r3yiWhOiIvrgDfM/3Khfvzz/m7PvJRCVPVM8D0iSo3MLpubLONpCfpyQb7wy8up3VZa952UxkFKdX4IEdV0TyYqefnzUyf7xhYS0bURxy++HVVEZnm58Lh9prX92Y4Ain+u6d9bAp6WsJ9H5Zzx6u9nu59pXc6i3iESAHhwqKJ3f4l5wve4pc2Cxfbi133i130mWoHUHqJBVC6YZltniSGzldPO43SREj9BFy9SookXKdH+x9ImUvkjgEVCuLmMBxI5TA2fWNuIHGJtIxpsSCcFLpVd1d9ODooG+45AWamTpK6L6Ewo7MS0QRMqu/vbMk+5Jod1BHKT8sOiPrV0SHT0cxpdG0tCBqvK6jLW7rFiKHWVdSxC83vIhFXQ9VR25xDL6N3Gb1WY+C7MZNPTz6l1bSwJmYCvEDlkdTVPxVrnVSZxKetYhC3yUIsXz9dT2V1FPBGRYkm6yfOabHr6ObWujSUhEzBcHLK6pVZNG11lHYuwRR5q8eL54FbZKRbGDLL8KVipn5ukXvbUdwb0JGQQUlan23MmuI01y42D+7KfgGtk2vVueoTm91AzaziWyq7x3ZcGCamp09ulkZBZ1eq+HwdbUtZZ1EPNPNtoVHYNL3eKEFRTp/673FlH5Q6TPMCSkAlq6umVtvpNiVXT5oTcCVrKOhah2T1kctpoq+xQP/f3/RM6Camp49C1qSRk1pTVXWrTHralrGMTmt1DppbEXVNSUjadqFfeV70w4/ZxST85WbwkvoJ3iYiyME/9oSoRUego5jbrRsw/S4QtRdnTE5HsrUaivtYsiYmI6npE13ITNt9D5rmTitW6HQ8uIHoLyy2fNnmYS0R0AUu1uqegmGmUtgqXCx8u9fT0xZ9kA2kzJ+ohL2FzPWSWhUp0LLQdsLTzh1chnKaOW9emlJDBSrK6nHrviooKeUNFNWxFWacr/DOrh8wg1HDeOuDfZ5wglKYOOro2bQkZrCOrKylYACDHd0HHOTairGMRWsBD5tD39HtnxdIlFnaJJPYSoNTPlTlLgZ7jlLq2R+ftigOUEjLhTD09gxMZCQB7u38FqyrrJMD+RF0Hoc4CHjKttjmPgaytk3ieearubP+zpZ8SZ0tvESn6LyUqcQ4jovW/EVGe49dEsZlElO8UKWhto5mewSEikrsOsmJtk9txytSpU5KD5HoOapmHWl4Sa6vszo3q4uYxdD5RVUwrV8+YK5ZNG41+rqZ7spaujSUhEy5cmukZHCKa1s/VY+gKq6WNRlmn66CWeehxl9fp6uc0ujaNhEzAz6PMI6uz5GdaLMIWeEj863Xi133i132iicoF0cS0EU1MG9FEE9NGNPGv14kG8Q86iyZepEQT00Y00bTt/wE+hYUoO6GT4wAAAABJRU5ErkJggg==',
    '1003.3684v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAkQAAACRCAAAAAAH5/gzAAAZxElEQVR42u1deXxMV/v/TjaRXWzxikgoaosliaWWtEQpat9pKVpLtTRKi6K0xVtvG14/VNSLUk3t+xa1hVqriEgqr6SxBYlIQrZJZp7fH3dmcu/MvTOTmMzM9Z7v5xPmnHvueZ5z5plzz7nnfJ9HQWBgeDE4sC5gYEbEYHsQ0XzWCwzlxXwiUrA5EQN7nDEwI2J4OeZE4awXGMqLcM2cSPtnn5D5rO0lnnQqSPMPe5y9VI+VUxVW9RkjPwanF6k47/Dlp0GD62HbYFMlnyZk+rxuuRZNHtPG/MLP0rj/q/opzNQt94GzM5UU1/Iug0pmtVCriou/W4XY0MdDK6DWlNTMtoHwmrxKYWxOpP0rI9Rr/cZcenpx2r9/bWGybPwE16FUPoho9sR5UhkquD2nM/y+WLxgSJNJD83T7a/ZHeDx7vzEsqgp1UKIqLLok5Y9/+QyOm0hy+H7JUb7oZyyfu6JXUS0/hvRpkHvr4w2NNnrPBERbXJqYUbxLhY0omhUU5bpC8YwIqInIf7pwiYsl9LtEvqb3xHLjbUQoqqoV7r8RES0G2PLKkYaaSEqY3eaIUtcRhZ2EZG6w19SRlT+OdHyVSvaAgBG9TWnuLMFB9hjIzKPl6W8KxwBwHfGvU8F+Rk3pXRzLcNkUboWaVUUkxeMOQOgz+V/l1WMNOa+72DsTjNkicvgGqaYOsvi74ky5gaO0nz8yMrTx/Rq7yCmPDfWw2lBepdF1ClfLdN8JxOgCHG3mJin+0cZvdMMWUZl9D/x0NJGtOV5b+297b0AoPjhE2Q/B4A7x2+rAaDk8WPkp6p1t6j/LrDIt7Z1cNequ4oM81WpuSjafVb6xlw05KmH+NkwTzflpQQVgJKMRyhJLdCTZnYtekNSRHwcoMpM5/cYJyLlCYCiv0sEwkvFaDJK+1uDvU3dBQWQfe6UMq30Tk6WiASRDtFWARTd1y7LnMJ2WdqIjqGx9qPLfwAcCK61duWKgFNQTo/13NU1GTjYuuaCr77Y222mkisW+88LQ9+zxFuT052cB+ccBoDV7QJe/R4Y7t9sA7Cid+yUD74N6KuSvHGb42el6uHXWYW/jRq1yaRuFN0/P6XjWZxpVWP+oW8vaspppW0xsxYD1MFxJIZV/wg8lTgRl94bq/5x9dm2q3jCdcpqM3T9XdotbQXq4ruN3pW/CdHdyckylCDSIboqcHvw/FOLYjUCOpy29OqsFbbqTcoaDjiT7hdDPwYkEkW2VBFR45rXiJ637q0mou5tYoiS8duLT6xTphGdAjeLza42nojUzR4RbauUTSUNp9NtwyqSMSA9Pe3QhKC9xFePGk3grhvqFo+Bus8ra+cQHfF5SvRalw3acjxp0rWIqJ+MkZpPSzGGiMIHClXSiJh6lWi9WxZfuEaMLkPb3zp0XisokBROROoOvGaGDxSTINIhOhmJnqeIaAl2ERHRhhBLT6ydUKL3/tLzdge/9KHweZYHdLl6G4Bbj2DAfd7+nQCQOhgIcr7x4gNRzDCgo/++PADwHrstH0gbXQP4taE3HFv9inpi9yTGxOy8NyrpbfDVK4Ux3XJmD/QCIhy2AJ7x72jLiUkrWwsJagAeEKqkEZHWAng1P5UvXF8bbX/r8NhHUODR1auAYgxPogdEJIh0SKmM91p1BqCV4fvY0i8bAy9xVa4+7eykUn7RFEBzAMDAAYrU6+eQoyvZBQcGAmjiADi65L24Ee3zSwSC7u0bBgAffLt1DH6aCMAnC4CTxPvBetO0nwzUg1HdEhpcyXE5A6DaLUE5MWlla+F91BVTiaukKQBX5IEvHAAEGc2FFT72EhQYG9iqacTb4w0FCyWIdIiuiofn5/NtRNKIyj0S9cAVAMCktePXH1zRBAB8uclldPvNVTvwSnp6JnOLWwBQvbAN3WxUx9/ff5xmfVa/+w8ozqkBYHLmDTw7MdPU7QbqGddtvVMaari6urr+/JmgnJi0srXwLrqJqeQqqIovHAAEGb56jSgRFHA9Nc9tZcS7apG3DAJlDTtEV0USBL+SEldLj0RD5xwodAUAjzaoXY17oAEAZkb/WR9nALVC85o891kjSy7wf4lsDoAWH8r2AYCJ/a+lvA0ANaYsr/FwZT9Tt+urt26c0eIPHBqjaqhhvp40E7WIIPNwSEcJlXjQE75uHD9Dr3SNXMEdyZUXLHi6cca4cBMKGnaIropiCIbV7BqWHoncVz5Zyn3KF24PLRtdH3gMHD0LKAHgFHpacn8ooTkAKIYruQVn79prTocDwInOa79Zp/lWswold6/46rkWAylGpd26gxb/iAWAor2CCzxppmsRwzcl0QoRlfTAE86JEdcGAFDzoeCOhBigyrSBN0woKNIhuirqvHoBALSvLTIsbkQYsGJhtBoAtlbTvFvIBQDnyiUA4p0L0n2BUw+B/PkD+gNQFgOg4hd+nG15yv3fG+vUAOD0/k+BCgCoOyN6y67YHADIrNNRODcufeIL1Gv5X4BEdcvh7D9tWG24rt1/HcDyOoJyPGnStYhN07mKn01dt7c1ABTnCVXSVFIMoBgqvnBODF+bXGHVra8A4BVYlQ9A2a5UweI8wECCSIfoqlBEx94AaJXGjK6EWH4Dlk6Gtlh75vDHc9NGEtHhnp7Ve2wgou11lhz86srk9p+XUMjwWT9u6TK3iOjs2z5V+pzf8IZH4OCnL7LEjwv1qtwyjYgig909OnxERHTPI5OIiHJaBtSr4+o05DlRXrNRvHuuDXzVw+uNTzUpnnp0u+6XC06K6HapTwDc+o0c0sERnxDR8fazN844ICzHkyZVi4j6V/sGwqPfyBHdW3/ygIjoYh9f316PS1XSVRLQ/+lnrTxbTtEJ14rRZej6W4fYxtz/mgK7eszZcuDz5bo7OVmGEgw7hCc0rnPUtum/ovogIqLQPRbfgCUiSty08niWfmbJjT+LSJ1NRCGjKe1G8QtuTZvU7A4REeU3O0dEqouNvzRRvFQ9IuUfT8zS4V6SWpghkGa0FnM6lq+StHCdGANtNLUE3OTfkaNUJ10vMKOZoh2ik3HvqrLwyp18Irr7D6WEEVXoycbQZhtgraOBO6MPAwBWHdpnhe0786UprHey8fs7yyqy+jluc2xxsrG4xHq7sq1upAGA6kiPl0+amZhy4UEF1p4VG2nsh1JBI9HR5ccqv764sbUOKV9a2qSNW/JvnScprPGFmS3Nmmes/5y327Gi6lYPnd5O6ox1BRqRCo4ocXCw3kn3jNQC/yCrHRo3U5pVD+r/lvBxRVW9tnZP2MCIwOgSVlc/z72ias53AzMipn7FUYacAIQrDN+h25OuMu9qvLxWBCD85f6hMIBx8RmYETEwI2JgAPMKwgDmFYQt8ZlXEIb/GYgej72f74JivIK/nJ1Uxd41kZVZiZTuYpkNRYs2RDk8h1jYcYgE1GRid0kt35+V7VQXO0+08z3grRVEX78J5w+PEV0YgsDPRDPFi5bLc4hZbjVeGCfbf/XDpk2bNm3S+j4YFXXmyu7PdGSxE4PJsgBZCxrVVVu/nDP3sTUEmjqU1gvniYieu/lzZ5P6PpDKlMgtj+eQLlYwotWa387r2oyaAJznak955dXvJVcj0qj+rNtcJW3qY1UjkhgBh2E3ALh3vncFANSNakllSuSiHJ5DnK0w8CauOXQ6Lu706+u1GcErZ629tVC7NxEl34kJpzq9V2OhMxacs4f3RL2ddxAAdTq2A8ClMMlMiVwANvUcItneD3p06tgxZVygNqPG5EXjdYnzQdXkakMa1Y9vjwTwyx57MCKfiOR4ABemOm8nAIfeksyUyAVg6DnE0HGIvucQSzkOkcQUAEj5faToxcIdw+VqQ1rVf/BuCSC0vV28sR6EHQCODO323+sA5btLZ0rkAgaeQwwdh+h7DrGY4xBJ1Aeg/ngJb2v9WtTSKA2j6N8fy3bLXaM6nQi6O2fx57fsYHVGRJmOTYnUM2kd5hL9udlIpkSumOcQEcchfM8hZrnVsAA2f85LNNuipp0N/yYiurKOqK08J9Za1bPRYamKUmofs4fVGVFXJNIfv1CmY2Oir7KNZUrkElEY9FxNhnBrtO1VLhPtxy3iaEVEtBvbibpXVxGVOC+v4G+hqE4an5VGRNR6ABEpP1HJ1Yh0qt+BYxoRjalXYAerM+4hdaAXqnZJvIlcb2OZErkAAqHxHDL83bGjhydA5zjkSUjqnji+I4ouOAALOg4xin2KAF4qGABCd2YA//ehbN8z6lR3R0AAgBYpv9vFLn4/xXZ67gkMwvbEV41mSuRCzHOIhOMQjecQSzkOMY4NQfxFzWkA8EQCbhZVzc7OLinOfiY7GypV3cvFFwAqI8Hm2x4A4Ncxbn8TAP0m7XB532imRC7EPIdIOA6xtOcQYyg5GcFL9cl55gIoURUZ974AcLPGF/Ui5WZEPNWb53G/w2r2MLEmWo7ALCKi1zHaRKZELhHtwEKN+3K0ICIKmUpElOs4hYh24NKhOKKQ4UREe7GDqHt3IiL3ryp2UnFN5875SQFRxDoioghvLdm7pmzfWHOqL3AuIqJ53HzT5nMiDECjKtyMp52JTIlcEc8hIo5DeJ5DLOU4xDgeaNzOcb5DBjUG8PepKM2IrMrLleu8iFN9ovsRgI6Mb2AfIxG1jSYiovuO6aYyJXL1PYeIOQ4p9RxillsNC2AvZms2m5qNIiqZu+xqTNPvNHtnE9t5eHf9pyxHIq3qF0O3X544IteqS3wjh9IuNPIBABx901SmRC6HpMu5jVtW0fvVJBU3caFcb87pw51njZysd6pLublXTcGs9Lxv+5p4eQ6lZZ/KDAuGVQ+l2fxko0nPIQp2spGdbIT9eA5heCnZHkd7Je3vl8i+BvkPSTZ8nJnhOYQ9zmTBxbchHGFzFRgYeZHBPowoXPNnn5A5tfIlZoaGg+cVxKZLfEZeZEt8BgaJWa0m9LZbwAtPem0W9RyW5zXKATZphLiVZMTEna413iUvPmfqgBcZq2jd3B4fvpK8ol7NRSaN6P7mjX2tYETqjXcL1O+/IpKOm9WzujsAjJDf8Pxs+Z305lNr2qoREhuw8RhORBTrOlZV/ijbFol6btkdTPVH14kevHFOJG3Aa5TPBmzGsNuUM8j7XEU1wsQGrJNk6G0HAIgY8J/uQ1DeKNvLV23UcBd3/xf2QV48ENQcqLVseqxhOnFNgLsCNG+9/B5i8xYHwmt93RHJjjZphKlJTxBODilvlG0+d3GqnXT3hXwAqJ8iknb4AAA2lvIa5YP9uxOqwCN8V1JTmzTCwWSQyRbcUy/5XAEAqDLvA1A+fALTUbZtGPVcErW/X6YG9vUUSRvlNdo36hQpAXjgsY0aITEn0kRNvuXWp5iI6HDvX49/OFtJZzqgO9GauviKKKaX2ysjR/5EpF7T8+Tedmdo/6tY/H8Lq5zU1tEbK0vrIyq9XhT548Wlr98iOtAckxd+sqzLjCIiou5Djy6KeXuMuiInFQVB6JS0p1eORFrVK0uOx2NVRURELZweV1Ajysk7S0bHbVtXTYnYoCYi2tMgi0g9aQQRhXUnoiKnr4hMRtm2TNRzC38Laa/BpZNSKi3gNcrsjPVFRFZUI8p9xrpqaNhr6uwGCgAFHw6tAigmbvkN8AEAl0oQi4CtF2XbdlHPjUBZ/1OHuN4PxdPKWZNk+4aoaFzENzZqhPTE2iMQWDmy6x9NgKv3mgFAM8X+riKTKMko27aLei6N+Al7q02ccLTvOQextJDXKC/Mrr7bFbZphImJ9cDCNQD+4miFDi6ix8cko2zbLOq5Eby/pBrqx3558bBoWsBrlBfWPTjoDhs1woQR+eA6gPp4DgDFRdoXvVT6nFqHxqgaGhoaGlpLP5DFUL8DXFRojzaoXU3BIy/O2Dy3YxVATdbmLuYmdAKgmN/plli65GQVudrQ/ms/V8KNm7ZphGkjIuQHVz8HAJfQA6hEAB4V8YJ5S0bZtlXUcyOorOBI0gHNAGQVCtO4+dxXpjZ07vflDsCeSrZphINk6O0cAAhwzErBpoI1W1MA1ZIhvYEWWQB2ehUApqNs2yjqubGX4kOXAcD9rM4ceZGf5vEa5YakUU8mTZzwzo+BNmqE6BL/2sDGnj4R84lotdvs1OlER99YtKbPfCURZXRccHDJgdoe3Z6aEWXbElHPLbtGLhz9wclr0WPStORFXprHa5TbEr81910GV1Qjyk9e5PDoKPXxAfAgu6FmJffk7wZe1z2ruyuA4vhAbuy8/7yhhJMxMe4ij7xokrsIS5/qSr6UHxymEE3r8xohx0NpFdEIOycvmo56rmAnG9nJRjDuIhjbA4y7yGDH5EVzop6zxxkjL4JxF9njjIGBkReZ+oy8yOZEbInPgP9V8iLEyYyo6qcAkPvAxUld7FgPyMp0URQ2gv0GWmSwHyPKiIk77TfeXZlwI3x+TTzcdOqsx6iQekDiyLSQLt+iPGRFK3EVDY03OregeEojQ9ofoF57zzV/WnXZfpfvhIS53Tn3ZhfYxwasIeIxjIjoSYh/OhFdAufuWT1io5osG2ixgncwM6ZkEH3rEmtI+yP18IVEfzR8INcz1npBJO0hQIw4+4NiMJKI4jGQiEi9MFZQKAobNZ8GmmFE3W1hRFGuW4ky8Jo2PSmViJ75BpUQ7fYuIqIPBsnWiLqtnLU21XrizDmoL456OK37rP7i9QgpsqK9Dvl+BMAdWdr0/teeAh7hqUnAlg4uAML35cv1cSYIImnPLxtzoYtXrprZt5Pgmh0GWjTAsGeDgbPoDQPanyrWGwBqFZ1kM+UKmViXYpvjZ5pPxe82aCO8ph9o8dOkxZ5ZUbvClbOaBO86EN0AOPh5/GS/p3X3hnztAgCxl+v9XPU/1o126Awovwv9QpuMK3EBcN2pGTKeegCAN271lOuXee14idNYb9jtxDoZA9LT0w5NCNrLTbMHFn4YVFkvColFAi1W+KTi/MywCc9FaH+3MI2I6Cq+lOuciBdE0l7nRIkxMTvvjUp6m0vlR07fWPiuSm9UKzHFVRSQFa3BVTRE2yWb04ZnwoD2VwgXjo9SINeB6OfhCvT3iLTnx1m9afxU0traQdOi/vUZjJIVJbiKGrKidQItGr6tb7i59lvnHfVpf54c4a0EnnI1Ii6IZHRGdbns4reuDXzdcF487D7Qogiqhl0+An3anw83BBXCW6Y2VBpEUk5HQdw2loxWwhhZUYKraM1Ai0Io27RWAvBFMvRpf95+jwDgIdrI1Ij6dFOCCyIpq/NE7Wb8+TVMkhUFXEUrkxX1UXj5+mMAd9ECQFYheLQ/Ra9UAEjxDZGpEbVY7QIg0bux3RpRDn9KgxxwLMV5bosOwb4DLQrh1fO4P3AnIbyTJvIij/YXeTMFoJ2RcvUhqxdE0u6W+NcGvurh9canmtSlAY08vbvOIsrtUcnDp8d1iwZarOi9s3f/dfZM217pWvIin/YXE56QNX2UUq7bHsIgknZFXiwrXjTQYoWf6rp6jVoFi7/gfHwwr30rhXwPpVVwEEm7IS+aJiuyk43sZCMYWRGM7QFGVmSwZ/KiOWRF9jhj5EUwsiJ7nDEwMPIiU5+RF9mciC3xGcCOx8IYazEvlUu5BvJvmzymTfkDLjIO48tvRELW4pNfz57wf88Nmb/9Y3GwtkjWjwaHJ+yaw6jecbPY4aPqIuRF/UuywuOP13jrtwf2sgErZC0mYwwRUUFE5UvaAtGopnyRgIumOIwW3sF81m2ukjb1ESEv6l+SU+TF6R+EIV2/PfZDXhSyFu9iHBERnUFXbYEhI3CYKpDDaOHwnYNGEtEr1UXIi/qXZGREhX+XRHFGxGuPHR7U57MWgTr4S/Mpvdo7iIFsOIzHt0cC+GUPDMmL+pdkhEp1HQ3bY4cvG3msRQAn8Jbm09bBXavuKoJZHEapgItW5DD+4N0SQGh7Q/KiwSVZgtceO2R7lLIWAfzxeffvNB9PT3Ec/MPhvjDNYYSOxGg7DiOdCLob7ZEztqEhedHgkixR2h57mlgLWYt3EbZ06eKhNVdrz8+lTCM6haHmcBglAi6a4jBadFKRjQ5LVZRS+5gheVH8kmxONmrmRMIYjPYzsW4cFRW1Nq6IqHRindaxSwp3edF5IpW/G59SGoYtwipCNNPr7VUuE+3HLSIKGU1EtBvbibpXVxGVOC+3yrdwB45pRDSmXgF/Wto8okDikiyNiGuPFY3IqaysRQBAwJ66PS57AsA+v0Qg6N6+YTDJYZQKuGhNDqM7AgIAtNjwexd98qLoJcg6BqOd7+L7vnlrPwDcbFTH399/nGB9JsVhlAq4aE0Oo5eLLwBU5jP8NORFsUuQdwxGe/UKooUHHgHAL5HNAdDiQ9k+uktD5xwodNVxGAFdPMaZ0X/WxxlArVDYisPo1DyPs9hq4JEXHXDDoYnIJcg0BqMDbjg0sf+RqPCkogsASmgOAIrhyl12HXCxFH1uKwE8QqgBeVFwSb7gxWC0HyPSZy0qAaBw0p2FwQC2POWye2OdGiY4jFIBF63LYZzofgSgI+MbGJIXeZdkiCIUAcIYjPayxBeyFm/2C4LXgJEje7XsfoyI4kK9KrdMI6LIYHePDh+Z4jCKBlw0zWG08PLmYuj2yxNH5IqRF0svyW11VjSim59nk75RwvbIl7wICwVcREWd6so+lRkWXOZLYIfS7C7yYhk4jOxkIzvZCMZhBGN7gHEYGeww8mKZOIzsccbIi2AcRvY4Y2CAGdse4aXuFe1y1JT5oP8Sz4V45EUGBvY4Y2BGxCB3ENF81gsM5cV8zd4ZAwN7nDEwI2KQOf4f6dulPivHvbEAAAAASUVORK5CYII=',
    '1004.5186v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAa0AAACeCAAAAACkpApcAAAeRUlEQVR42u2dd1wUV9fHf0sTqYqiJopijBorIliIBYNoFH0w9p7YYuwS85gnthg1Maaqj1ETjY8ae3ktsWss2GNXbLGAvYFKU2CX3fP+Mbszd8ouCzuL0cz5fBIv986958wcZubOzJff1RE0e2nMRTsEWrY0c44R0UTtKLwENpGI3ADgZbx3ffHFq+AiP9EA0BH330tnhRD03+u46Ei7b71c5iatSL8FvFYCAC4VcTPQm/keMflSyusNC39Hju98WLlHiXUdCj5CWkKKX5SDUTzbfuJpxc5vYE1nPEsCfCtYGm5kAhWN99zdKdfwmn9ewwzpU9/qLAPE2F9TesDnGhHR6FA0+5bybX/2dxlJzjeIfxzT707u2aHTwhwY8dIgr462XORtpvll+hx/eizuv6tCiG6Nb+7hftfc8jAQUeNv/TW2EXzen3gpr3Eeuw9W3mHIo0rojCa5REQ575jEwcy0L+jahZ+tjWFGIjLG2JEtG3vR0rFsmYb4HSUioiVuIURESV0x1dw0pzeuEhEdR3s7BpqHknrFHVa6b0WNPDADADyq68TXuIv2XQ7cC/86uCTKBYCLPc8iNvbCwcBnzpnVAADQqx0AwLNm/f9xExXK9IMnAHjaNVH4o0fKHvufjqdWGae0S+v/vrffxJsAgHrBeW/qtL1InhDcy1wcbv53wLX9AIDDjfIz0P2SvbHS/mx5LTZ8YBB+1B+/YASQMNbKne/qkSwAMDx4jNTMF5OtKmu3AYBuOJD76BGeJ5kkwQN0de9DZi+EaG/tuW6yw0Vu8kPkJj4GkHMjl6tKPRKvv8lvsDyzreVoRvhx/3b1WsBlK8LmkQOMSenI2XCI+2F15+Yl1ufY/+ap4X9OfM0POa/988TGh7BqTPbuXr2WzA4vH7JoUURQwx3oVu7Nn4AdsaezR48zYEvt1+bPnlU+nuv1eZnyLZMLL1sfI6b5N0f0iMTWuqUnTRn/e4tP9WzwwL56e0zLJxnNeyFEq/9kl+/65lfz9HAwtNTEbd8e79vP9OvcQw3mAMAPi/2LfhUmXL9QzVL0+B/3r1+XtWkA0n11SkNajhwwq+2uYQO/Ld+O+73a38S9c9p2u+aERAlzibJruZ0kGkpENLtsGtGOYk+Jqn5ERJTkOoPortsWImp0kGhj5SdEpsE9iExVOhy8X2YlUdhIojPV9xTqnHBlMQClZxERVSt9liizblsTG/w+15NE6712WfaCj/bX8peIRtUxEhG1sTnLeDtqEdFVjDxDtNDrCdHlSCIyNeLbQ7Fa3OH+FDqAuUS08CENxW0iogQILoQjt6ZIKuVW+YSuExFRYhxRPLraOyecS0Sn3Wpk01AiSvUfQUTGgNn8frZuRmQoOZDIGEf0vNx4IqKz+IMoLIQbIGwknR+SUbgzeHq64qPKwPdEFPYBEdEGrGWCN1V/l4h2hVzi98IS7driJ4g240re2Xo30EiU6/4eER3BSaJ4/9NENJ9vr4flsmyZqoQT0bekkC3myHWqRURdy5kbph4lMpbzylTYYTcrJ36diRMmTgOAU2keBwGUvMI39ex9O2h3hzX/LXKwCXDmTk0AqKnb3ByoxV8Uel/wKeQ7V7Fu3Si+y6T+xSwTW2zpKAR/52IrANFn2B5ctB076JLOHUGaHS6quwCuHjUAeOIZUD84tEb0vwYgNReAuz+Cjz8CAMzd7+5m1I+vAQC6AZ+eq32ulnysC5WZI1fsCQA3yzPzpjKXgIp3NnXLxxeTz8K/OwwAN1HK09PTc9l/+JZ2RVdg74/GbdgaA/zFzUxdPC4BCDBvsfu8V1zh5moVAOia/ZJx2lLj63uVCf4aysj6cNGa5kUsLWHflM2T+b8R8Iz/3Gt29PumtoGBgYEdgVY4BQAYPH/Awq2zqnN93nf7FTtbyMda6MYcuSEp55Gx91Ou5WLVoHLlyvVXnBW6WX0l9VvoB00BVEOJcKZ6QX/4tF86rIh352WtyROohEwAMOS8CcByL303LqJRTLfCzNbarpxjZPFv0DKqMsEbcRvivbBE++m805VwEDDpdPn0ebXopElPF4/uvyaHy2HXcVuyPQHApz7KljRvVLrt0inurvLO91yYI1dq2MxSD2a/x7WsGFULAH29LbWYPecWcU901aZeuwsg5PVdAJDzO+BpABIB9Er4Mha9Nq1sDaBu4BEAOI5WzAg+aDBx0M3CzNbF6wCADIQA0ANAPGKY4MvWPA0AOeuFveAsY8YHlYBHwM5D+fV5YSVQPK7j+deCg4ODywDesx9/x7U8Zw5k/6f92sn7XrnFHrm9Ted/tcCcLLpQCwB03fXr7boSXjA/Gcc1AQDP+ZvPAZgZBNS5xn0Kiy61sS6alvqhKQDvX1YnAsZpXdoC+nTzI44BGO3b8WkhZsvYIxUA5vYtCyD+AfB8Yof2TPC6+Ue2Avi5nLAXXLTuRXMBJLhn3Q8ADAZbPvQGAGQwADDACGDOcwB64Q12h1mT55kAYDV3Zl24RECr1zKCAaQhHQDSuF+lm93Kskeuwuh5y9fv4m6dy82HrS0WmPKewR+L9veJWExERNf7EBHRnoixi0dvIaLrFb6YtI+IaMQUIvpsNNdh5ztTf4mdqKftMb6BrRYRbW/jF9T+0SZ/n7fmFN6csNauAV9u3xPXIZWIwrqP+XV51IQccfBH6o5bPXktvxeWaGlt0LStU04Nifgs91hsQInYk9ZcHPpXseKxRxe941O+/dP/hPrWGUbrW41bvuUz0VvHfeEh8w9uHzHhZk+iS+8G+ITPIpq2luibliV8a3S6fDy2PLze69mlkSs+Zo4cpdUp/0aQp1uXTDoQ7le0zk0iGlXb26fRcOkO2/U18m5mFR0AGBKCAwAg1c0HSIef5RqcWsXtBX+NPBlGCSeMYaE6AOE1F93KqOomDR4PUqu4sHthOS0vG6p7ULp/vr9Gphd1u6Kv7CmuvHwivVqd4vbuBHfksurPbwiYTn7QdWIeO/wKfjsOr7no5fp2vG4e9+JizrZN/7xvx4bclyzg0PM3AcC4o5U9vz+v1Lm1c+YfRZt9Xe2l4jKOf1e9vtfV3U0H6/5pV0IjXJHr4vKSUTTJSVnlKrrYdV69YvctaMyTZvibME+Rupcxct0r4cJ+izRnq9k+aKwuNFZXu2+pG416LyESk1IaBOdjexO5Fu7umlzYUqG7dwqrm88U9Rnc3Vw8umzrenG2Mmbeul9rZGkApvl3PJ/HBYItHhgTE+gNAD1cgEcjfrG8+rEUmT69w+p53TrSMk+Q1tJVcMzYvjmr2ZLIPetfhV+L/7tocBkeaPnx6bz0LMOwqlYDQ/7+IggFf7e6Af348hOsF7Uld7tOaZ38jxCZuk8mOlnlHrHFueYAmlHyJwPr4T7Xhy8yfag0APcJJttoptBVcMzYs0ptRCXBPevfMVaXs4wWE/S0JJaPa1gy0bceu6wFlp/X2A5my3RCwAcyJNkanEREGQEVc2mDfw4RDexExBRH/LJt/4ED+5slUfaN3OnmoyUUmT7UYvaY+Ul5HUqhq+CYsS/5bHElwT3rX4VsmTr1JKI3Ay0/T/dcTZSMt60Flp9sOXgl1IVZb9u84UJx+ESuv1xjeSMPAJH9nntBKLoMBIDF/YNRhGf7mSLTB6WG2BGK0FVwLLQerVhSXBLcs/5VsD1rTwJYwX8qK0MAvPHESmBw9C9ZBRxSjDfyBKWwiTHlPteSc5ek3YNy9AB88Mi4yx8AXsvZB6Y4DAASD/e08gJJ2DDfxjsWqrL/r7ukZNu9A/azfx0A4Tzv2S2jM3AIbZUDczRbDA4pwRstBKWwyaV6gcMB4HrnifFTd0m6H7hfGsA5t5rJT30AwB9XwBQrATCNmGblCZTZEMDZ6d9NT7N7n3jHQtV/R+gkJdvuHZgH7K14e9zXn10R0fX6H8LHKwfm6Jxwydodb9W73+WkC/7atA+ot5tviY8+VhcZPRtF85tUO9UMAC7X39wU+EbS3cUDwPGzowKvwhsAXJGGDKEIACtqWftwJ9rw7MU43fr6O+29YPGO+ZrTJYNkJZvuC27pj99aM8Ulqcni5nzVn+v21l3rrRiYw+dWsYxnQNSZ68DDM2cAXR/+t2ZIdF3Ap3I5ZhP4AEDf0KYAukq6A0BO/+ivkA0P7j1OFlsEoB8z2Fpcog2Xddehvc+o/OxXTv/or/gfDEv6yEq23TuQLRzt4oKKLQZm81UNpi292T1FKTDHs9XxcVjSxgNIA+oHh9aM2z0AqSkpKSlpuHOxBoDoM28xmwAAHhx9RzhNxW1jAzd4whdGAMiFL1sEsElX3lpcog1rA0D4uvxw9WMDNwjf4H8a6iIr2XZfcPNG+fIAQhIPM3OxKkt3tjYqBOZ4tgQcUoI3CgSlhJi8DH+l7gAW3NvqDRTjzqRs+LNFAIsqWo2L3fDofi5/F+zfLc6x2S7mlEhNTc01pGYIpTzcF9z8PAIAoKg42hL1TuyQB6bCfUvAIa+J8cbKPEEpISbfwDOl7jpsPrvMBeddqpV5CAAPUB/+QhHI3RdtNS52w9i0DA9AjxJ275XZsZmXTb4zHsDFUuPfCONLo2y7d+CA1nrGfRQ1PzHoG+ce9QACcFUemOPZypgx2IxD+qRc+zeKxx09H8m1WAjKrdHCJo0BIOitPwFwJwPTvfGRwzN1wMZuujanASAxIAxMEbiYGWD9QY7ZMKS7B4BL/nZ/vrc4BvDEyxORkQCwtuZP3GcHrmTbvQMW+6XeA3iIcM579gmXR+WA2wgRB6bOlZDFIcV4I09QspsYngG6ebvOAzQHWaLul3s9Hjzoo96/BmPUxUSA1o1yBVvEPfB/2pCDHEmR2bBTNQA34qfn+SRv7io4RkpQY8sD3LN0SYlxz/p31AZ57wBox4DKnHe/mD3lgFsXIpuIAlPrPaGAQ8rwRgtByW9yJDYgoM0jogNNp6/5ZBUCO7Hd63IOahPRysgLTz7ppSdx8XeMJSKinB4tyvhWbzddXBQ2zJ0w48zKGj/k8Z5Q6Mo4flazF9c6qKGPf/NvRCWLe9apGu8Jj4WvPTGoR7rFe/L73x862KDNfRIFVrA3Twrft3gcUgFvNBOUcmLybkp108WSJYtaoSkfbX0WEaqTFPVL29h6Hc30uXg0IKK02h+f8nBfcBep8Sn1ajM/nzlLobV1anzf0r5GahSNZtD0CbVsaYdAy5ZmWra0bMH8gP+yWeQr4SKf0WgzeG0Grxn+djyhSCwz67HQUNz7pToG6oKgzsRK3RygPu8v+601n63EOYfPFO/pjZw7x25Nj3sxh10BHIUUwGRAT0tJDII6arLR2GgkYCgKh/40U58iscw/0Y6IiLL6vAClVmvgqATAZBBRAflkQFAV3upKR2OjkYChhUV/mqlPkYwVrziV0v1FZEsZHJUAmMJGDPLJgKAqZEs6GhONFAwtLPrTOvVpII8SxV7EZVAZHAXAApjCRgzyyYCgajwTSUZjopGCoao8HZvlMhXELuXUp/yyvRN419w5SaIJymtdMq3qmxwcZQBMJVMXBJWMxkYjBUNVyJZFLlNB7FJGfSplC0A7c+djXfsSownKa10yrU4wMTgKgAUwlUxdEFQyGhONHAx1fJYhaH1KxS5ZnczIjgr3raCeXRpik0gpc7egsiloXbKtqt23iIi7EV1BHBHRGXwhtBz9tN5HmexG4hLR0s9U+xsT8WhMNKlo9J2REsv+UeAdlp1baWM7+gHRLssB34TeQEX3myHAW8+TABn1KbX6S1esamkumzufB3S+1xuVud81a2jX4oBu0PLdolb1TUyYQg5gKl38VQVB2dGYaBTAUEeft1itT4nYpbJOpkX6kruwlh9BgPFCbUvnZ4BZZVOkEsq2qm1iwlQAMMu2Pmr1uVVdEJQdjYnGAoYuOhyl1n2L1fqUiF0q62RapC/NFloDuHxM1I1T2RSphLKtapuYMIUMwFQydUFQdjQmGmUw1KFzS6L1CWUwlLkjW6QvzfY6gKPVFZQMRCqhzjQRYSoHMJVMXRBUNBoTjRQMVeHcYrQ+IQNDlXQyLdKXzMTlZyVOU1El1Ckvq9skCYTpk2wg+8S5R7AAmIqmLggqjPYkWxRN7HU9LGCoOtlitD6lYpdS6lMilmnWtUTWqLvF+c5GXmWTVQllW1U0OTiaEtQYIgCTBT2F0j2oKbHNj5YS1FgUDQOGqvae0CyXKRe7lFKfIrFMs65lz84NiiFK6BzceaVFZVPQumRan6o2g1cERzn6UwAwhY1EyCcPgqoyg+dH47wzvKsAhqpGfzJymbLXBFZ1MvOjden8T4UMOAp7AExlELSgXyOlozHRSMFQjf6E9u1YM2jMk2ZatrRsaaZlSzON/oRGf2ozeG0Gr9lLd9+SLIZjMmp5gTO0P2GVwOQPPcM+8m1iPNOsyqmsClow35IoWO1Rnr8USg5LcsKW9ie734tvZ5k+fPNFaX9aJTBJxj4ybWI806zKqaQKWjD6UxYF40/gL4WSNUlOlbQ/mf0efo7o3jsvTPvTKoFJMvaRaRPjmWZVTiVV0ILRn7IohKEF/pIhMa1Jcqqj/clEs+lHIqKz0fSCtD9hjcCEjH1k2kR4pkWVU0kVtGC+ZVEIQwv8JUNiOizJCVvan0w0fz4HgEqJqs4yeEhT4Db59dOFJcrtQjkVJTxFlRYtTrtVQeEIDirwlwyJ6bAkJ2xpfzJW9scZJmBTjJrZ4iFNntsU1k8Xlii3D+VUIDEllRYtTrtVQeEADirwlyyJ6bAkp23tT8H6VPy42V+/L/1KxVkGA2lauE0eCBWWKLeKcorvGGIS09zGVp5aQNSgjZzZVKIx7aQ/ZVEIQwv8pYzEPIZRqswylBBPPpqbb8OjiZ7Uoz9ZSNPMbQpA6Koq/nANXYU37EU5lUhMtlLQ4rRbFRQO4KACfyklMR2S5IRt7U/B9JX+7XKg7QP1roQCpAmYuc1TaR4HDx48XPIK+CXKxVtZRzkVSUymUtDitFsVFA7goIIwp1Si0yFJTuSh/WmxhPd//O58853tTKplSwxpBgAsEMovUW4vyqlIYgqVjBan3aqgcAAHFfhLCYnpmCQn8tT+5OzDaSVRadcXx7ar9i5DDGnqABYI5ZcotxfllJGY4kpBlXOU3aqgcAAHFfhLMYnpoCSnLe1P9iJ5oQkA3cTdV2LUylbdwCM9IYY0Q17f1Q9Azo7YvU1jrW6Vp4SnQqWroMVptypo/uyJlyc7tCDMyUp0OirJCRvan0xLUV2GHwCUr6nalVAMaaaLgVB+iXLrKKdERZPV+uTbRJUWLU4rqqDIP/3JlDjtT2Fogb9kSEzHJTlhXfuTica96wwAuPukqYrvCXlIk18dnV8/XViiXBnlFJBK86rYEwX2kcUtGSBS0OJUUAUtGP3JeDJrfwpDC/ylULImyamO9icTTfYHA/edndfnprr0pwKkeTezik6yRLmdKKeMxLSjMk9Zzvx+KhSGFvjLPElMtbQ/Gbt6/HnterrCoT/tW6Ic2rfjv8W3Y/uXKNfMaRmz/3fIriXKtXPLqZnKT1R2LFGuZetvky2NedKYJ800+hMa/aldCbUroWYoBJ7w2fYTTyt2fgNrOiNfiqEi2dBXy16g9qdtLpIWTGg19M2rs94oPdX+bN1f9lvrKJFsqA1CE+pofwojsovI81AmKw0K1bU/1XRk8+2l7aXKTUP8jhIR0RK3EBt83UxpTcuOUtlQa4SmStqfzIj8IvLscvPC2uxO0P60w5FK9KftpcqnY7G51NFGth5+JK1p01Eqv2aF0FRL+5MZkV9EnoEymbXZnaD9aYcjlehPm1xk8oTgXubi8JHWx1hv3zmuRGhCJe1PZkR+EXkGymTWZneC9qeajuRzwtxHj/A8ySRZQ52ttRyUzLaW3hF+YgrUmHIXgP7BYyBhLMSSolBcZV6R0IRK2p95jJiHNCgc0v5U1ZEsW1vrlp40ZfzvLT7Vs1ykqNZsf4CXc/L4n4gCPRQZ2B+YV+W1uVg1Jnt3r15LwBCkUFplXonQhEran6IRlRaRty0NCoe0P9V1JL8+Vyt9liizbluTiIuU1hKFYrW4I0OB1nuXiHLcphBR1Y8kkqLMfUsQE1UiNNXS/mRHrLncROuq3JAgoow0qNran/Y4Kjj9CXi1qg14f755HVguUlILwA254o4MBVoMADyKKEqKQmmVeSVCEyppf7IjKi8in5c0KAqu/amqI6vvMqKwBXIuMgpbLAvBI9iM+c/t/n6/D7pfAEuBuihJinIEKRRXmVciNKGS9ic7opVF5Nm12aGq9qeqjqxmy9f3qgIX6et7lRf7bIVTAIDB8wcs3DqrOmwKerKSokrLxCsRmlBJ+5MZ0foi8jalQVFw7U9VHVmdwadnVFXgItMzqvJin13Hbcn2BACf+iirJGdJ/JVyQX9lSVFBTFSJ0IRK2p/MiEqLyOctDQoHtD9VdaSo3g8A8YgBcOTwTBdgYxG21iL26T378Xdch+fyhx8C8DAHADwNQKKypCgjJspIZDqqkCnT/mRGDJkrX0Q+b2lQOKD9qaojpWzFPwCeT+zQXsxFCrUW6zBr8jwTAKzmTgOGAg15AmCdXxaAOtcAYiVFOcVQg0EsJqpEaEId7U9mRNEi8ubNxdKgUFv7U1VH8plqWPcxvy6PmpAj5iKZWsH2hYfMP7h9xISbPSWCnsmNJ22dtqWsT4undL3CF5P2MQTpsdiAErE/c7KhwjLxyoSmStqfwojCIvLM5sza7E7Q/rTDkSP0Z3jNRbcyqkpvaMq1uHwivVqd4gq/BY9vVPY75xvorYMhITjAmqQoKyaqRGiqpf0pjKiwiLw1aVC1tD/VWQTeCkUTXnORwsbKtdC+Hb/gb8eGXKWNlWs1wwtVDtrZ5vLm9y7ZV6vZi2Z1jXBFros0icq12pVQoz+1bGnMk0Z/QqM/NfpTuxJqV0L8Y+jP5EsprzdUqL/3zAMGQzUAz+4U0eVUKKIdUCv0p8mlULKV2GdwdyT9tnB4Q7bCbCd3L8gMbz0ZQMroTaW6f/7is0ULzvu4fuqtjK9aKEwGAVVZ9hNWlEvN8qZyaVSVtT/Na7zXHimpMNsX+N5cqnfGNvPp7LXEiYhIH9PTSKtqJyvgqwyFySOgtvFWdehPRt5ULo2qtvaneY33sJGSCrPdQC2Oq3nSNw/ms1CyNdk9nYgaDVLAVxkKk0dAbeOt6tCfjLypXBrVcYpGusa7t62KClEJ3Pf+Fd1RAOZTbZtX2xdAw6WW76Ob334K+EQmXQaKVODvJ6WGTB0QLG1X7ZloYKsmjRsnssqlFnlTJgQnKYzL1niXVPTFQgDA7ihWDzQP5tNZlnrHBwACM8+YK/KS9VRX9hOAEv1pkTd1wlvd2eHlQxYtighquAPdyr35E7vG++dlyrdMli763sF3eTaAhOqujGqobebTeVbUlbidspwrVmQ9eQRUVdlPWKE/LfKmakyipNfnJNcZRHfdthBRo4P8Gu9hI4nOVN9DfAVvH2IVEY26JlINVWA+C+W+FVKDiGgIfhBJ+VhkPS03DRYBVZb9VJP+5OVN5dKojt+3gltuAEoV2wiY6jVi13i/MO/PdyBf9L0PFgKG25VEqqGwxXw60SZffADoT4u4VLmspxgBVU32E4r0pyBv6pSVMXrG38buDmtycFBEe/wRPUXIk4UABRBRZeddbImR6oECsMZ8OtFiv//wduLXbUR0m1zWU4yAqib7CUX6U5A3dUq22hVdgb0/GrdhKyt6uPu8VxyUlnvX9TX9hjWdpHqgAKwxn860UT/F7/34IcuAyWU9xQioerKfUKI/GXlTp7zL8Gm/dFgR787LWhP7K/duXESjmG5Ky733Hreor4+Pgh6oNebTqVahAnAzWMiWgqynCAFVUfYTSvQnI2/qjFkG0TaMOUF7iyzaK/wlI4VNJJrsf4MU/qqRWqH9fiLKDBxKRHQIm4gopB/RWMp6vRsRUfbGwppl7H8vlSi12AIiosdZRHR4jImIvrzG3uKjFxARRfsbpO0qzTLO8m97HmeZC6WdNcsAokttrIumpX7gJCq5Nd71BmC0b8enfIXokSuhMSR6oHLms1Bs8++3gRkVe1vQS7Gsp5nCZBBQdWU/rdGfFnlTuTSqGu8JR0whos9GExEdiw0IaPNoexu/oPaPNvn7vDWHqxBtnlX8S4lqKMmYz0I6t87H7D/176iHPHrJ4KsChSkgoFZlP9Vc+V2QNxWtNq+a9meqmw+QDj97M76zbkmZaqgt5tOJnwofb08NjcjTnyICikKgPzWKBtq3Y82grRupmZYtLVuaadnSzOE3T5G6lzFy3SvhAvmmPzXTroSaadn6Z9v/A/rGWAYOmmS9AAAAAElFTkSuQmCC',
    '1007.0920v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAiIAAADYCAAAAADzSdegAAAgcUlEQVR42u2dd0AU1xbGv6WXpQhiCaiIiooKCmIJRhQ1tgSjxIpJNPpiN8WSFzUSNckzxsSYRIwt0WcvL6ZYg9GgELGjqPA0QExURBDpZZfd8/7YNttw8e3sDnDPP87Ozs7c+3mYe2fm/OYTEViwqClsmAQsWIqw+P+CiCKZCiwMRyQRiQgQWXg+IhLc/McaKtQZYdhAw+IJYWd4deX5rALfoM4i6eGXeG5AaSoA5242AHCj3AFS224Gtkr/s7T0FTtLi2M5FQQtDBGBdCL7H06O/f8xPTwqaZgvmRy5uSZuqHW82wMG2AAT5ERE83uKnXq8Zugny32BCjJPGGinNVQQnjAG2g4j4hx0w6sPiYiONkUtxJkb9zRKEJGrF/CBYnHxJCO/OWk+JQy00xoqCE8YA20HERmYi5weUzJpqw8ADPmPfS3OR5ee+ky2zQsf7FaMe7ZGNnEz33nTtHZaXgWrC2Ok7fopUj1Nar9apFiOeNn0/Z9LfuqmtT1oj8m/W2iMN62dVlDB2sIYa7v+TOdkBvp4qz6MTwYA6e0bdp3a2AKyq6Vlrn3xMLu0rFsLWWppqX+nksslHdoC5QlTcTcFz7Qsu1lS0sctuVGIKOeBAyRNfHEv366qa81XTn03v1b10rnW6s/qAwIA3U+XqWdqVRk33ILaaP1Yln39kV9vd9Maqm7nE+SyhgpWFaYGZfSGwIXAZPWH6ntEdKhVz09Xdel0mqg40AYDiDY3BfZTcaAN4nYO/Wgw5hF9EeGIFhER8XQ5ADgeNQo9qnf1AAK2Em1ycQyurHHITSdaDAQ9JoqbQloHJKLzvZvNj4scAVQQyfc0Cf1qhdfEQs6vE9s0X/hRoHeKaQ1Vt/MJcxH+VTBlLmJBYQwrY3i6Ggss1FrxI3pXEpUE2pwlouEYQET3gf1ENBz9Z8voHPAHEbWBYrIjD8Bzv/0E/E3yHhhGRBS19clKyEYDgyQKJbQOmOogvksknwlUEG1HlzKiw+BM7k/DNYPoGcw0taGqdtacIvyrYGKKWE4YQ8oYnq46ABLuZ+k0jHEExCPk0wB4AgAaKb7yRMqnNugIpHNvyDWCTeTzH33rB9F7OHIVuJA1wYRnRdt6ImEu6R9wquQlX0DUHwAq38REF2Co785i9Q9n46X2wNIhr9a6oTWGdVQQpDD6cxF/IEfz6WGT/z6ALwA8g2sFXrobd3EC7IFK7bURcFwEANEd0z/ZhU/mm3JF4PxDz7++aQ8AWgcsuYSO6m1uFkB6EUDTe5mqMfjRNXQCMG2azu9Ma6jxsJIKQhRG/ywyHMjS3LwbhGzAFgDsuetrvuRSzfNs3sXezFvJr5skRbPDbnjnF0D7gH8SnNSb5AA3T5w4cWL0J/6qVfcBB+XiUzTUeFhLBQEKo38WCR144sKtQOWHo73QDigHgDKgrfqS0MjeKmZ+p5V3E97/+1P5HGfTpOi8b7j8bBC0D9jGRl6uyXggaorOZaGNPE+5WIuGKttZQ1hNBesKY0gZ/bOIaGNTvF6qzL/4OWgXhKsAcBX9PQEnEID/Gti5E6QoP669zn4+vvthpqlSDPlS0SPuAV0GIA0AZADQohcSAYBm3FGfhgfjJAHInWVqQw21EwJSwWrCGFXGwEQ6vQuCD5URVe/usJeILrp43CRKdvDKJKJvEExUPt4Zq2VEsRhARBWKyfFkjKCfXiSiMKxR76rUG+8+YeIuL7BPlKvuAGOK7gGzPB0uEd1pB9yQ0x/eoh+IJB/0rVb//K/G+ExO5SNWmdpQdTufcAOebxWeeEVjaWEMKWPsGQ1VfhthJwro5jM2W3GwCb7BQS3fyCMiks5xe/blAaebAn0eu7i4ubpkfCp2FbtGEeUPFYV2yaAkVxc3sUui+jGTY07NSqQ6Ork5O3yr+CAdPU/3gHQ/tlX0iKELAawkyp/VsltU6MJSzg4e/qN5h76hm+WmNlTZzic/xuNZhSeliMWFMaQMaigpKv+7UBygGT6LbMXqUazAvhHu2zs76z01KCvzEemuW56z/qkqZzgHBB6525cVOTs5iQCgrNpDb2N7l9o0VL+d1lDhKUuK+BRGv+0qYXgqdjo3fta8h0HnAxp21ZlRFVjVGXAy+6jszRkBaNhR91Xg8e8n93mxdNhSmwZeu2pUhTpV1MtXy+T3mziw8mZjKrAUYRXwrAKeBRpOBXykyNK5KbwpmTBaJThhIgV5bmMBxvSyYCnCgqUIC7A3A7Bgwd4MwO6LsPsiLMDjmwEwc1KPhixLQZ69nVxKgfrfFKXlu0exFAEKNqMWKZI1acZ4K7T94dwNHjztOvvA76c9xnY0kCI5O/89VOApIv/PTanNHB9eXx5BtBGNJTWUR8nXan38Aa//PwD8U0XevDfCkWMmJN6ACqkYaaCnRETPxxBPYRZhqGTQ+xLaHm22FhmZi5yYkH+yhrzKu6n1Mfrilxb/U3GbEz+Bz/3bKqZpOj0FAHthn0NocpPl9lh2lueBJqfxi7v2DDb+q4M6894wyyvh2MoihzlY56YOJw9cArBbCn5vne0bPcD7YBUAQJZ/D4DkwSMAKDybKLkDpC1SbCZ98AiFpYAsX0Gu/XUyU16/pmqqntahvn3j0RVA9948p8jp5+xHFx0DgORInynAxsDm6wF8ts3D+aMw7H2v8teJE7fjcHDzTeu+apmYHu4zB4BkXoLbwQG361OGqHqq27cNz7bs8Dkw0a/TVsGNM6da/734X/+8xfO7zrLeIkrEWMWH8MFEVGW3gigjkojkEUTUfppi1ho4Kimn2R6iyBgi2twyneidrjILzcqI1vA5XU1DDGl6yu3b8BiiIr/JRCTrkiu46WohIj6VUZbvCX6nq3vGAX38fi4DoCTJHRwB5KamAqJJnDmIW2ZEs5yxgBgAPEvKgKjUzPp4b0C3b+7TDpQAmTOaCK6lxUgZY4PWg96o5HWg+fnm1q3/bl3+s84WPfy7dX7r16ncLbtwlmMehWX/eAZF9TFF9Po2pWInsDNWeC11RcuWAEKyfuczRW62b+Hn5zcFe3TWOyUudVk38FXurI37LgL5xt47vCPq5x1Gvb41HxlPFVJ34bXU3cELAJxxg8+L3t3vdAFA/zpa6KmesVQDuO28bNnjbQumKJ4Mb5miU0m3cOOVNkgC5CJR/cqPLVMM9G1GVHLmWAE21q5LmYIGb8zjWYRudAEA0XjJQQBwJAC5VQBu7AEavRVzHXCS6r+qouSL19oAD4FfkutReih6aqBv/TqsSwsWYoujMyUActGdxxTZ9Vjx7wvYIgcQUgDge/cKAPHlACS9gK5/AARAonxvkrQMsHeuBpBmX5HjZSExqlDF384lihdZKXqq1TepFABE0/eFCjKpp7seB+j41Ha8XfSe6e7u3PUOEb0T7CqOmEOU12fZkZWHfcWDHh8csnjX4X+uJaLMVh8s+42ODXPzGbKV6Hy0l9fwh3SgxcojKy7P7P3Paktc9FZNGNTMLWjEGn4uen8Z2d7NM2qGqqecvp2P9vKOvkREj/0rhfmM5nz3AxenTyg2W4tMKaZ59Gc792tuPq4lzna3JO2cAECa5q9/rpBlSIMcqNijXpUUKXuq37eC+CUCLSkqTMwPD2Y0nnWrzooyQ7FqjD+rOmNhLD7pXlxS6Y+GXXXGoqYYfOlswtKG0lk20DyVCg/vdbZvSMKwFBGxCng2F2Hxf906szRqJTy0K1IYrRKmMGygYQMNG2hY8HHRW3bs4mP/MQG0z8RnmVnZ+T3r1W2C4vsOdnKpbQBQkO8gqmzPUkQ7aPPSoTPb3vqyg/gLE1MkZeeRg5ZOkZK1f+V0ebMpPzt/sD0xWTwxLABIj70TFrWqLv2Xml0Y/adH8hke54iI6Du7MFMf9xTgoKVRq3GZVPSyx1m+alcvYLhCjQnb5GShEK4w0K0a/rcyV6JNTpESi6fIjGwiKvFqXc1vebN8eQJRnUoRswujP13Ne7+1siRTNFfAp9NDzz4GxJHZGbweRb6k38A6NnUwuzD6KbKrdLhqZR8xoIGMqvNyUZ1dAT3yqOoeWZ5HalElASDGQz4PIls44jn1iHz7bAWg4csAyYUbMuFxWGYXRj9FftV4rjlu4kBGSd2axB1ddX7sZNImjzJHxyV+nABLs1ZncpoCuGbXmcdjSCe6qN+PcDz6SuWCxVI1XwbaOLI8q0+y4Bgz8wujNwR2wz7uRw5k9GzUVqLb+FVrZbpbIhGtxEHTWSuYbfA+j3d4RK0qZ7V2vqXyYG1XQCSfMUHDl63zLSI67vm4FoxZ3RVGu2Xh2M39eKDRRaJDuEVEg31kRNX2a7VW9upLRJSNg1prLaNEZZeBFfylyNCZWadFvRSzvnK/JUREV3GCKCyEiKjQYy4RybzWmd5vqpvC2BiwH9WMYo+1IaMgG8DWoYy78kFKf/XtFYuzVot8fnDib+8Zi1o/91bKagBA6t3OANBZdAhKvuxykUNSUtLvjW8JkTEzqzD6KTIEV9TLK7UhI8VhZdyVGfCwGmu15f4RVx53H+oLfBi4NA0A/qvou41DOpR82R00cXJyctr5rgAZM/MKo393deyiw1WOisVCOyMAlWZlAMqsxVodurrTBtdtgvg8hsu2iNdSHIA2KAUAaVVbFV/WEd7dBcqYmVkY/bOI69d5X6iy8VXDABVnZYsO5wCgwvKs1dnf19oAPzrye5ReC658CCDU5ywAXMAQ1RchzyQAQNVPgmPMzC2MgSe9L69dup0A4IiovRZkJJECIKmMu1K0MeE6QPGosDBrlTHx0Yzp017ZzNejoSIoMLKlLh8fBVw37MsCZCvHvKDiy5w2HboGYG0LKzBmFhbG0ET6t7Ce25J2vbGLiAMZnX7Rs1F0ytb+Yv/Rj7lU1Zm+a/bP2wufl01mrcwycVeicMH8XNFcGNXezWPAe0TFQxzFnkOuEf3S/+MN0XESNV9GdLL3om0LDnMlqhbCFY3ZhTFSTJN+8bHfQHfjAJXWynv5QfKbjRs7m8paiepmSdH9wkCdmdu90kBRbRgzVt7Mqs5Y1RkLMLMRFixYirBgKcKCpQgLhlqBoVYMtWIXveyilwXqLUdT8Uiz3NxW6M2X85TiJXcU/3o3E9XO06reQWcGUiQr/vfURrGuKLuTWJwhdArtt/h9/Ow4b8+Z082mukpuXI+Ma1obTytrQGc68XhjcYV0dns+UatzGEFERPeDjwnZvImIytoM59EmYBwR0aMwv5xaeVrVCjrjBbWanUe0yiGBT5sAF+WppfnyOwI/iazhcd9OsAUArwV356M2nlZW97zasfkUMFkSZ5H7Iv0FniIprRvzf5AAnK5bU4dmBMAVBfynyM9/wD1cTRUZ5q3o9qlcxZKGOrJcVP5nvEXMOwI5nBV0mTItPTjQmfViXMloIBkv8J8iFytx+D0lVWSYt8Jv4Sflu5bJwKGOLBlfzrVEqeh+23c1nJUyNGyVlh4a6MyqYQ9IPuu+hM/pahpCN8XPc0njuFYZ5K1+s71EdNAlgTjUkQWnq5e3EPXkb7p6G6Nycu4cndb6Jy5npfC04rJVGj040Jl15/EpC8OnlRK/rlbitm3biLmuVRxTJ7e0V4DW9tdBMweGAuJ2fihaFOMODLTZZcm/Fen2STwfIX3Pnu/vTsx4ERWzxjYCRNN3/Qp9jyu1HpjcrS8AARiQ9Fy54874fH7fUuTTD4OKuK5VMaNE2dfOavNWd28OATAwVUEdAWh8y5I6fD2L7xvDAW8pFzSc1QB9OdR6PEiJE8q7jkWBO3yHptjyPRfp6cZxrTLEW/2BZspNOdSRxeJmlXdhYWG1tLCE/2NpcVZ6TJlKDy50ZvXwDr94nPcXfPdX5COMwkTt8LdyiUMdWSzy7i4BcLPJkoB3eD+WFmdllK3iQmdWDEmf6hQHwAu3LfoO+JIvZihhInEf9UrfzlcAoOrIyJBnEl4HUHU82nJKREYCwIHOX1vgWKE+Z2PB4awMygEOdGbNqLxo89AP+BshPF70Fin8nKBxrTLIW4k2nT0C4Bs/DnVk4ZCVFfO27yINxs3lrCCVarNVGj000Jk1w33YST/grxuRz/F20XthVKDY87mx54g4VJFB3orOhi7et/yAFnVk0Wc003uJPQZ8wstF79WYDmL3/vPVJldKzooUnlYatiqZq4cGOrPqM5pXVycn9RyeY1FXK+Mw0YPCQBtt6qi+lhRpc1bG2Co1dGbVkqLUq9QtWMRQK1Z1xqrOWIBVwLNgKcKCpQgLliIsWDDUiqFWDLViF73sopcFrONq1Xp0APaPbqCyKFErl5bM6trgKZa2vD9kVtvbuwKafpwq3PMpbbkutl3oyo8KWd+eOd18qkNZWtGbo57uRJs1acZ46whjTlcrERl2tZrpnkJERNvtQoSLWkmGxcpob3Aef6jVeCKiBKfXn+71/z/g9frsarVNuRQj4BRZbl9MRBHT+StvjiUiognY+1S7lF8srceuVv4TlYtzBDxCbgx2A9BrRznPx2mN357uFB3mivrravWCamVvdy5HpbJzqs7LRXXWIwBVf1ajJvKIxyi8KwYAn9JUng90T1m+pYKtDPTeIIcmy8+xjt+VBVytTmhcrRy+hYajUtk5KeiiC5Nfl29en9wz3hh5tOHZlh0+Byb6ddrKhxDOtqRoPr/eeLi9L3oKoIGt9HtvmENLD/eZYx2/K8u7Wmk4KjV4paSL3kwl+s6lwBh5VOQ3mYhkXXL5GXJDOhERzcRnvM1F+uzfFz974Fa5tqmVXu8Nc2hEkTG19rsSpquVjYE7JdXcjxqOSg1eKemiOyFAh/JsY+SR+7QDJUDmjCb8/LEsv/kAkFzRbqxZw7t7+LPywnYiQAu20u29QQ4NgFiHybJkVE0Z+BGPt878LyhGsfWn7e1kkiUPORyVErxS0kWdADihzAh5BExZtnM6dvLFMESv/ke8dPvws/y9HEDsD6yLHXApSAe20um9QQ4NBpgs1CNXq8sAgBmbpn535KsgLkflpUUXGfC44q5F85HxVCF150uGd75OPPV2rvlYAMMRU7kBOrCVTu8N+37BKj5fsJCr1eLDlU4AIO4B38ZaHJWRklljrk4zopIzeURcW7UC7vjznCKeuAYDsJUp3Ye1/K74d7Va9+hTxVI5wHVvMg5iGXF16tdhXVowbxP3kUVA0en37XhPEUJ5sJ6plQndh5X8rizgajXqq+Ub5QCwr7GWe5MKvFLRRVIAUsiMkEcARNP3hfL3p/LT38AXrV8Bv6hVS9uCLGyv4JpaaffeMIcGQFpmHb8rC7ladQ/ZlHRs7vt3YjkclQq8UtNFLUc+frebW9fZZIQ8InrsX8nbtd31Yacvz4/K5en9IldjOrp5DowjovUui7LnaWAr/d4b5tDOR3t5DX9YS7+rOuVqlXGxuGPXRrruTagdiIWC+CX8PdB8dKywW28R/yVFub9QtCcMmVo9qfuord9VA3O1KsoMxaox/qzqjFWdGYtPuheXVPqDBepn1ZkZYvClswlLmb6on1Vn5omH9zrbs/Jm5rDJKuDZXIQFGGrFUCuGWjHUig00bKBhwVArK8SFX3LbTfD+fhS76K2zqBXwcO4GD95UWJS7vNmNjS32X4T5UCqGWlmWKJr3RjhyiC+bgB/DZEQkGxZmTpSKoVYWVaLyz+o1PKbIywsUJnBh5kSpGGpl0XBsxav9Z5aC+w73FxxKBeGhVlK1tZXaukq9YF1bKz4j8MBRABDNAaofPkR5tlwHq1KgVBZDyyBg1EpFWGmsqzQeVla2teIz3sawAZ+clSASR0KbLlux5KdBCyVcrEqBUmmZWgkkrIBaqQgrNXKlXnhaWytzEUV8zkVojyeApl8REXVsepWoNPQFuRY7FRlDOnCVcNxpzYpa6YsTjl3am4WFEBEVeswlIpnXOvWCPGgwESWEpGu+q0cpQo93T2sHrCaisNcUlzAHiA40ukh0CLeUDmhEg31kRNX2a4WVIpVdBlaYrUVPRK06KQkrtXWVesHKtlZ8h+e4cZQ4ZtkUT+XnKByOMcBO6cBVwgizolb6KTJkvxK1euVyZON0bygJqzto4gRgp+9x1UKGlq0VgJ2+9SdB9o4FIOq3YdSV/so1bm63AfmmrcP6R2hbPmvBVYKILfePOMJyqJWKsFIjV+oFmVVtrXiOAwpGbLDGXaa4pL212CmBo1aqUCNX6gWVrdVBE3CsOhc3Fah2CUIAhYVTIoZZiZ2CwFErFWGlRq7UC1a3tapCFW/7lk0oBID1k30BJD4AyuNGjdRip6RlHOxMQAON2VErg5USifOlszuWHvGYumgHjn95xils3GsATi3u3/56v2GchZRZQ0MygmK431nsaZVkcl5aWYt2/d7iR4Xgz/f6d3f46a9vPYDugf5tXDZHLHEA/vP2rOBLwzdfiRzxryT0/u72yjOi5xZlbLvQOHyjp0Ae44UpppLBV3muXdVGraCLXKkXnsrWqi6UFF0Ko7SLsrBuIgDdO2/9q6S93VOwU6y8uYFUnXXvvJVVnbGoKaTVYGYjLIzHL8MzDr2UDlZ1xgYaY62SwRbVNjYNVxj2GvwnhS0auEpsoGHBUCuGWjHUiqFWlp2LFN93sJNLbQNK79rbk0TUFoyjebooSst3j6ppg6zs/J7+dXEu8mD7q626fH4S+Tumtmq/+hcht/3hOB7febvom0lrIuM++Vj/f7bvbpN2kLMzNr7GDVI+H5daJ4TRL3a6gOFERJSOSGIcjTGARv7kQrPnY2r+vgAH64IwBq5onNTvJ3YS8CnEbU78BB53vz3KBoBNnP430Re/BIC8m0/ch/3/+b1AhKmzV/yOrXjdfZaNMY5GFAYAONhghHnifRFZ/j0AkgePoEONyLKLUfVDMqBhSzTIDeoRR6MnSA6AtEV6X6hQIh0VjNg5Vd0j1I9bZ8mRPlOAjYHN1+tQI1+9kDD7jVUtR8g0bIkKueHZrQiW5WjWdW8ZsnVr7xa9jmOcX9uvFQDN3vcqf504cTvQqte7n66ObHxHgxKpwSPAmJ0TkDk6LvHjhDqihSFvyRaxsbGxsSMwmIgofDARVdmt0KZG9jsWUnXgPMrksiUq5Ma4W1Fd5Giybb8gumd3mIgiktQATftpRETVIRKiSw4ruCiR2tppeIwxO6d0t0QiWsnLdNXswhg8i/TYsWPHjh1fQPmifAAOikpIjSXP3kAP2HbbiwCuL4/K1IhntyKLxNjs3dPa5c75DPB//gegieePgDw8AgovIvXNjxH2KB7T5z2Os5PG2gkwZuc0uVtfAGPryzMarS3U1IhnAQA7DwCIeRSW/eMZBVuiNDWaUrET2BmLus3RfPPfUz7LCoHYxL/x66j9VUh6Tncjh7Gg6cU7bHG5yCEpKel3BUrURbMBVxy1eg9S+tehS4VaPsZTUyMz86+j5NRCQNuXR+mYwLNbkQViLxQcTckVYITzbpz6XHYUR/SKc8VB2LJ7e3MFSqRydgLHN8KgnVMGPFDv3t5MuoVXTWavbfJg3UuANlsisoxbESzK0YhH7pjt6Dp651DSuVG0ZQpwfc67g5Ek1UKJRE8wLQpAGepTMYAjAcjVxRFO9d300ZaXACO+PLy6FcHSHM3EtA+jMfHnPUO551MpkAWUje26AkhzN4ISGQZvWnQ4BwAVdTZFilDMNe0JKQDwvXuFNjXSasHGXQcTiqBj2VOs+jvi060IluZoBjb5MRR9m3zWF0ovIgBd/wAImHt/tz0qNjfjokQqFaRSI3ZOoo0J1wGK5ytJzCyM7rXWhVHt3TwGvEdpMR3Fbv3fJsrrs+zIysO+4kGPuYZERV1bBrRwshtTyrHsOaw0NSLjbkXmu7armjComVvQiDU8XfR2SZj64bGTb40qJCKiuSuI6J8LiIgUXkREma0+WPYbnUSfxYumtoFE7eyktnY6H+3lHX3JiJ3Tmb5r9s/bC5+X64AwplRKPPqznfs1Nx9Xzmha0WNTL0B+6bWxcYbZEiNuRXWSowEK7cRAMbTm39I0fy8jmJEppkX38oPkNxs3dq63HM33G48BAOKP/lwrtyJWUtRgOJpu1+8AgOz4EOZWBGZZZChaH1wQ1MPl9q+DZ4C5FYFxNIYjL7vCr7VNLd2K2EDDmF5W3szmIizAOBowjoZxNIyjYQMNG2hY8HfRW5BnbyeXUiDMABPV3VABZ5DddrCTUtuG7GCklyLZB34/7TG2Y6BJMNG/h1ozRcxnWQR94CwxWTwxLADSzcduRA2dr3Awimuxvw6kyCth4S5/nX0+isfa1VSMNPUpz5NgorqLWqmBMyrpKTfBwUhAL/huCsD+fTmPL/iGrekTFHtYEbXy++oCj/tXg2biNiINeTWzDgwMwS/dDRjoz1ArvlEr7qweNZFXwosmMy1VdVZ4NlFyhwsOqZmh+mdQhKcnr9Bw31L02TYP54/CNOCQhhmqhwZFMJ28qgNxdc2na4p4Rq1iiDIiiUgeoQGHOMyQhioaHmNN2xVeUas0qPoWq+NgJPTpauddcvo+8E9+USsAuampgGiSBhzSMEMcqqjhhIa8EnzsHC/CSPE7vE9Xe/h36zTwxamAChzSmPXUS4Mi1NbBSMBXNADQfWOeD89zEafEpS7rBr4qhwoc0jBDWlRRvQ03KNGhahm0ySuBR8ppRfNv8H0Wue28bNnjbQumRKrAIQ0zVA8NigxES+88xcIffjDkYCTYiC4qcQAk8Ob7iubGHqDRWzHXoc8M1UODIkN3Q0ZeUuAwB0ZCh7wSdoSsdwCQ7tGRxxSRQAIgvhyApJcKHOIwQxyqSCq1phh8olbA6qZvAsClu89Ch7wSdrzcEcCfiWvseKuUSFh/875taPvnN4R18jjTfK7asUht1vOhrcqg6MKHSaKIuFDrlEXwbFkE4N50+zdsThWvctVxMBJ4vYhsmXe/jBWvvy3iu3a12NnulqSdkzFmqBYGRXW3pIhuX5L2bA998krgJUU3U7x6N2XlzazqjFWdsQBzkmDBUoQFSxEWLEVYsIDtBwD6Wfig/QQnQz9htEqYwhBRHPtLYWE44pRvKWLBgs1FWLAUYQE+bQLYXIQFm4uwYAMNC5YiLKwV/wOZMDjgycMlsAAAAABJRU5ErkJggg==',
    '1007.0920v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAigAAAFVCAAAAADgMkvbAAAvk0lEQVR42u2deXgT1f7G36T7Xgplka0UKIvsZRWkWEDQapFdFhWEyw56QQRRqeh115+iLMpyBRFkU1BZBYEqiAJVpFC4YFtRoLQU6L6lyff3R5LJTDJZWjJL6Hmfh4fJTJZz3n4z50xyPnk1BCYm59IyC5hYoTC5T0SUxFxgcqQkItKaikVmJZHalKSaVqnPGwDQkPGfzNKobg6tkBEe4w2bozC5JG/x3WUnMm41bNtOo9v9mPRtKDoNIKCzFgDOlfhC59VZ5F7n/yoqesJbbn9kNULV3hCRzRwl81/+fg/8a1q3+KMPN3R9JMvOrsKox79xqX9/LTDWQET0XI9g/+5PiT3k1YZAqZsGXZGmKmSE+rwRaT7s+LMjBE/mEBHtrYcq+DMnqZpmEFFQBPCKcfPFCXYec8h9Zog0VSEj1OeNSPNhvuoR6sdRhRPWRQLA4K98qnBuSrmTE9v6CLzypXEw9LJzlxD3nUZda6oiRijujZ3m2xZK5VSdz3sa43bvEa4//6/H7qR1LXb4YOLPMg38rjVVGSOU9sZe821nQIcuoE9t840xxwBAd+mc973NvQD9H0XFQX2Rk1lU3Lmx/nRRUdS9hb8Vtm4BlByYjCu/4J4mxWmFhX1CjtXqqMm67ouKug1xNde7vJPTq6u+a54qf+zXZtxt7jUBgK6d13OTuPIL50LaNhc8WJ959majXqH8xwmaCn5buaY6aZFCRsjgjSvNdz6ZfR6YyN2ovEpEu5r2ePed9vf+SFQQo0V/ojX1gG1UEKNF0saHXh+EeUQf9vZD4969V9Bv0cD++GHoXrmpOxC9jmh1oF+HMmfj8HmiF4G2t4mSJpHgNYnoRK/6zyXFDQFKiQyb63b5+LWI8Xm8Ryc3b/D86zG1f+E/TtBUQVu5pjqZo8hihCtzFLd740rznU9mxwHPC3Z8g15lRIUx2uNElID+RHQN2EZECXhglp5+Bf4kouYwToIM0bj/yLfAP2TojoeJiOLXuWSGfiQwsMJohuA1T/sGXyEyzABKiTagfTHRbvBm/z8i6ALRPZghfJygqYK2mpvquFBkMcLFQnG3Ny403+lk1heo4N/WTcUoPyB4iGEqgHAAQC3joXD88q4WbYDz/I/xakEb9+Dr/20EzQvY8wdwMmOsa99Pru+BA3PI9jUnVzzWENA8AABlz2B8IPBQw40F3ANn4bFWwOLBTwofJ2iqeFsdSjEjZPDGheY7naNEAVmWWzl1/3cdDQHgHpy5FWF95/b+gA9QJtzbG36LACCxzfm3N+Ht51y8ZAjY2ePvT1oBgOA1C1PQhrtP2i3oTgGodzXdPDDfPIN7AUydavU419pqX8oZoUZvbM8oCUCG5SO/gcgEvADAh7/f8WWZeQqoXYAt6RePPe2qG/V3h2Du94DwNf8i+HN3yQLSDh48eHDk21HmXdcAX9NmNdpqXwoaoUJvbM8oXQYcPHkxxnRjb0+0BEoAoBhowV052nm20hmfCapv7Mv/vGuYHeCyG+22JhiOt4XwNZtrDSWW0gfiJ1ldPWoNN0ybIm110lQHUtIIebypijm2ZxTNqnp4ushUhStmo2Vb/AEAf+CBcMAfBOB/Ik/uDx1K9gv3+TyHz3bOqIIbgz8ydor/moH9kQoAegBo3BPJAEDTL3Nn5UE4RACyZwofV7WmQl1GSOxNNcwRmWafb48Ou4qJKr9svYWITgWGpREd841IJ6JP0IGoZEwA3tMTjUN/Iio1Tp0nYgh9+ygRxeID7qmKamOB85m94ZZPssH8ATImWb9mRrhvCtHllsA5A/1ZW7OTqOKVvpXcw/+ug/cNVDLkHeHjBE0VtJVrqpOP8GUwQiFvXGm+8+96qOy/vb010Z0jR2caX2xsww5tm0y5QUSkmx1y34j+P9YD+twODAwJCrzwbnBQcFA8Ue5Dmi7tL9DRoMCQ4MBk7usqvyynZpz28w8J8P2v8YZu5Dzr16Rr45omDnnoeQBvEeXObNI5vsvzRbwnyPlXg9Z9u6wxCB/HayoJ22pqqvMvBaU3QiFvnDffukX21uuU/JMXHG0ZU/O9grlx7ZZPLVzzCQiw+d6huDhSY73v1ayV1V2cw3tN4GaoT3F+gL+/BgCKK8Ns7uwTaPu4qjRVISMU86YqzTd7I9GSql/HzJyX0/ZEdI1f4ebICLbCDTiUuVf/zPRo1HjdFUZI+EbKfjBY9/BiLVsz68gIj/JGqpYZrtX1ZYurHRvBCoWtwmer8JlQM1fhx2kUKFL1zdZU0yrVeROnyvMcExikzsQKhQns1wyYwH7NgP2aAfs1A/ZrBuxzFPY5CpMqfs0AMyZ0Z+bk3Pb2oYqARswJu4Vyaw2qVygZE6aPkemvOOfTMAAwrL7iX/JspBQvceHgX9vKxj02wvMqXAJvxBZ2Ea1CnYqqTXgMS4mIaCeerg7kVFXdmDelG7KIiAxjXiVKibl2xxM28Vb1Q77sU0fVeiPSslFjsa+Kv6ox1Vgvp4rkMKPsr8oPjGbsDCsnoikjJCqU/ij0tEKRxhvxyWxWnSewuWpnph2mCXJskBznVr+m5gV8m3r7Aoj7roRNIyT1RrxQto7sX3tHOQBAd/0m8ooAQJ9ZgPKdxwCg4uQ5PYDKG9mozCwFgNRFxkfqc7OEB4C/D6UbJHNFfyAMABqUH5HUfK4PlTk5KMk0WO/l91c1cqs34oXy4/0+I/P3AcDuDg1WL/+4STLw8SMHZk15p8kQPWjV0JKMPsdwtHPdpL3vnBg9kbDlhbIfxo/fgPPdImeDfwAV8w6E7Oh/SSozbtwOBoAwXJTQcUsf9nSpt+S1l74d+HwFfy+/vyqSe70RGxQzniVKxmjjpCNm2NGs+ptpm18eVcbMo3Si5Q3zifaH3ya6L34d0SX8QEStjHMUihtOxD+wpsl5ormd9O4dh4nIOA5fxLNERKfNP14lyRyF34c29f4gKuryiEGwl2eE4nMUabwRPaNsfhzo0+i7YgDQhKT3rp81GltiwuDVeQuikb9oeCgwQLsJCEl9Amjmc5b3WCMOYDkQXlgMxJ9Ol+hdU2ZkazWQ8rzP70Pg4A5A0OJdXwv2ihihvNzqjejnKN/VPw80u/Ld4wCA9kazbgHwDgPwW77vUQB1LgJoqwW8fIttn4I7MHyYJvPMceRLZEaIkaasdOuvmFnLtg/x2D1csNeuEQrKrd6InVHSWjVu1KjRJPN1j/E3EmbknkXh4ecBXEZdf39//40LABNKr4cIwWo6YFjV64vavaV7sxvfLmUIk9Bx2z6EhFwS7rVrhIJyqzdiZ5Qv57YHQG/uzQu3rMyrO2tp3evLHwPQBrW7ij/Z2km2+55f9XtzHAUMGklW+IXVzwaA63D/Nw5faYYBJN6HgsJWUvdMZd6InFHoXHsA0Iyp2MHbe7jv6tfXPgYAHe85AADl3wpPITqx391A4YdPNQdygO+PSWKGJiETADIiYt39zFkjRuQCuf5+Vn2oAIBkPCx1z1TmjUihbLpt/P8RrDUAqDD+zlPT+as27TiQD8B/9a4zAJY2Bip0AEinB9DpT+PbT1cM8A74BFQCSPUpzYpwsw3lKAeAuWkZAH0918vdNkfWeao2rl8Y52PVh+TrQEnSsKGCvTwj1CApvLG+Hvupa2hAp8tENLdDUHDv2fseDokcvI6I8js1iW7s7z2qiIgO9Vq0fv5uOvZoeK3EX9Y9EBw18jalN31lyRE6kRgRkZDDP7C98Vt7XvttRq+Fle68BCwfO7B+SNshHxDR5rhzt+aNr3D/R/jfLfr7yuBHioiI34fYMS+s2RT/cjl/r8AIxS+PpfHG1fU6pd1X9wQMKU+NTgKAq0Ux1gOzLjVK9KShv6Br60sFYZItzsnZU9yrs0aChUs5hwq7xFr3oWu7dX8XtvK22zN1LVxyrzeutOzrVfsAACv2flezV7h1bbeOrXBzoM5nLwOAfv9g1GzpKmtmv11eKnry3bbdAy/90He6piafUb5fejCg35ttauh6YhdbdiOztFEzbc1eXK2HFyq1WlYobBU+W4XPVuEz3eEHbnFQhI+H6oj9ONW0RJ2/ZsCGHjb0sKGHCSoEwPJTc0Pj7xJnCq75eht0XtHArVxfTVkrVihwHwCWtfHzh+JlosCkBsCub0g+Fjw+Nho4P+5ybPw7nvM3LVz6d1b7Z+qpAQAz0V4ienC4SxSYRwBgJ5Fg7O3Y9QZPAsAeT6f8EWHH1QCAmWgvESUMd4kC8wgALBXDiYgMrx7wKABseiYRFUY0q1QBALbDySxZcgpMRgDM8FK/AR41m9h1320gOC7zgrwAmD73KoCK6zcBIO94csVlC+1lgcOEkJeRApMY/ZIJANM/P+R+bpi+dLxU2G8TCSdPX11U4/IKAMHIkRUAOxYXOQlYFdNgJYD314cFvB7L0V4cHGYFeRkpMKnRL8gCgOnGB3Kz+v2Jv5fNf1Fn6beZhJOpry7qp6x6AM54t5MXAKNug4io3Ps1ogtxRGTobaG9zHAYD4VKGM5RYOLolwcBYKkYXjazWcBFc5pty1tEhuljLf22kHCO+qoAAEZ0AnNlBsCMwae+fgCyT58GNBN4cxETHGYDeQVbY1OARwJgJXPnrS970ji4lM4cXQvQTNv0g7nfPBJOlr5WQeWTBrwuNwBmKaXuUZ3vHfDoZP4929sBpOzvhScBYBdWN2z27AfvLQCA01faAUA7za7+pn7zSDhZ+loFLYrc6S87AMbJP3lx4PIBT/LnbBF2ACn7e+FJAFiXhsB/YhanAsD/jJCX1ve8ud88Ek6Wvrqutdf2BMkPgAEAVQK4FLBkye318yfFWWgvjQPISxZASkIAzKTA9b2f+sUXaI4iANCVtzD3m0fCqQsG2/XHRi3OatvIC4D5EYDscgDnNgO1nh1+1pr2Ekeh5AGkpAPAOPWc//t/AHSJPA4AJ8GtGraQcOqCwY7/vFQLfOMnMwDW8RaAr0NLAawoAVDR00J7GeEwPgql08FEgUmHfskDgAH5MLJviwPf2AsEfbo1A9C/NeoRc78tJJzUfa3axGr8zenTpj6xJkpWAIzoRp8le97a3TB44O0dg1/ctHvhUiIT7cXBYRwKdTwxonZiipECI3H0y2MAsJPDWoWE9X+BqGCwX3D44DNE3z/wxqeJSRWWfptJOHLYV7kvj7sY/7Ad5AfAbv7VMvRMSGRQYYD3xYqW/iK0lzgK5QCQ0ngCAGara3kxVpM6MwnnRhjMswEwtsKNrXBjYmJ5PUysUJhYoTCxQmFiABgYAMYAMHZ5XMO98ehgJ4OUA2fB30CD2gBw3s9bRy1s7nD7XG54P1QXcMrIzO0RhRrG9ciX5iTQkRVbpeR6dp7fFHy6OYDPDv7e7+H5Nne4+sX6If0cA04O9MvGPTuiahrX42Kak5uX+xU3T5CW60kdifsriYjKHzCI9j5+tP3ne3C4kxe8hR2exPXYOXkfHJt7yEF13UgT3Ew89ZECJ5QPAADf7lkAdOk3R4pXiH/mpw8BwLetRrT3Pg4e6+PsyX2k8mXxm9EI/cxrrN6t3riF65Epzcnq3N2sDiAx1/NGzItpVaaawLgee1yPjBwPp7KvxkjP9QSu1z2l426ZIB5z743z6b9KBQctyA8cZT+VXzVf2pgsRU3geuTkeDh9NEcjA9fTc8GpN82TORPEY+49AODA27+OnkiWg5YcLKtEKEH2U/rIpOQ3DgAWS1EzuB7nHI/bJ7O/rSXqkSAx17OSqKy9dwrRTOLHWZkzrGhQ982mOCfuIGdJgrUlluyn8yHJRPQWdvAtrRFcj4wcj0m6DRPkCXby+xxPlgPgx1lZlDnSGOdkOchZYpMIZcl+mti5L4DRELMUdzXXA/k4HpOWzdTKFOzUKenlpLcgjLOCdXwV/6BdS8x3vv5LEue8iKW4i7keGTkeUwvLa+fl5VXq8gqlD3Za2PXdnyGMs7KJr+IftGuJ+c4XeG0VsRR3MdcDudmWG1deApBW96Xof0vO9Xh/3vmpvjZxVsIMK/5Bp5ZEw5IoJ7T0bud65OR4jIpbtmzZsmWh7ZbNlZLrIeMlbJs3/rwqiLOy7b141pUdSxq3/hWA8d1usRQ1gOuRh+Oxkb64AJJyPedMH7Y9e78wzsrce0ucEz/rymyJzsoS7s6aVQfOArQCAktRA7geFzgeCb7rmdYzOKz/29JxPScGhAX3Wk9EROkTBBCPsffCXCvTQZ4ltRNT7GU//dT3g23ztiByBFksrRlcj1OOB3cF12OGeER7L5J1Zd+Sq7ltDWl16gQUWCxlXA9buMS4HiawxdVMTKxQmFihMLFCYWJcDxjXw7gednnMLo+ZUFPzeoozjf/7R3nftZlNLqnQtJK1dn1N1YxQB9rlbjjOplBubjl2uNHEQOT+cM+bHeAAaVKI+pIUcuLrxuaffqw/Oaji3Nm4pHpVYbskRLugIBxn+y3UJUwgIiodEHDSEdJUderLI8KLBL+F/zgR0c3YRllVYrvsol2eDMeJnKD84QUA/q+ULoQDYkkZ6gtSQk7iRkTMv/JcldAtH8XHHQngOAcjWWP8T9n0JsgNOdlRNH70sPmVBHCcg0I5jIcAfnYRAECfWYDyncdM1BcfD6u8kY3KjJsAyv+qhCeGF9lRAWJ4qU5wFe1STlLAcfYLJWXhoPfByy4y6uNHDsya8k6TIfrz3SJnC/AwI+J0cuLThjUrj/VYAU8MLxLXNq8FllQnuIp2KSgp4DhRXOPMe6g8feSVqRpg5RtpofB7JNO0zHr7/OywSW0f7XLCq81v/YDeR7sDmDIhCECf1N6XchYhtmXoxI7QzRxTS0ortL4ATv4xN/ISggDASxpKpPQ6KtJ2fr9jIL597tda6DdzwkbTkQ3b97fuljUqRWvp9xPxuNB9V1/gbYXr5Pc6jY1Xhu70RvSM0uG55xZuPrFxQKYN9rQlJgxenbcgmktvAkx4GIc4Xe4ItC7JhOeFF9no/ObNX18Zf+FRfqoTXEO7lJM0cJy33SNNvmk6+JQ19hR+C4B3mJ1CMyJO9wLw5yEJ8JjwIttp7LOmDUGqE+AC2qWcpIHjHExmIx68uMsae5qRexaFh5+3ez0JHugEjwsvsi9BqpNLaJdikgiOc1T9wcjuJcSeUHfW0rrXlz9m/aFdJe6K8CL7EqQ6AS6gXYpJIjjOQaGUHdHEx9xz4GkA5fsTTZfMfROFd/IrgwkPg9zhRRrgm8c1Cb8D0gY7AUCXyOPj+KlOhR9ON6FdwX3E0S7FFBcHANvbLQPc6Y23WJ5RBQCUTf/7tQ5YPfpMB2DpQAA6L6Dp5OvBAcHdwwBApwfQcS/MeBgqNABIpwOgk3jouTB+wHRQydGFmBubES1ZsBN3uRD06dS50aZUJ+i8eGhXD67femhWDTzbzox2KSoTHOdOb6y/XEh7rBlCh40bl9Bp0EE+9mREmvI7NYlu7O89qshMfVnwMA5xajL09oLOIZ1meV54kaBVfwxvHRz6wHPmm+ZUJ5MRTtEuRb/rkQCOc2G9Do9sKu2+uidgSHlqdBKs8TDN3RFeBBdTnZyhXTU72OnrVfsAACv2fsdWuLEVbvbV+exlANDvHwymmqUqvpFOvtu2e+ClH/pO17AzSs1bT1yllt3ILG3UTMsWV7NCYavwWaGwVfhMYAAYA8AYAMaGHjb0MMGDg52K9526HTUqmra6uAbH0+KscAcAWFV1l5BzYoVCaxY/NKPFxY9aB3/oYqHIyzzlzPnU+Mm5hW+SgAKzB4BVVc7DwKRzSFoAzDA97FciIvrMO9bVr42qyjxV/4uvG/OmdIMRyLLwTW4gnaoAgNnJQat+GJib4TjOIXfDcVYt+wCfm4xIdLlQCmUrlLK/Kj8wFcrOsHIimjKCv+XOQrmEcUREtNn0v7Wyp7r21AnyForFITfY4oAUvPFys3Gmye4cFY6Vfk251RUWvklaCsweAKbSJDCLQ9ICYJuKEsw7+wQDFtJJADrxw74szJOcEWA8vkliCswKADOHfJmTwAQZaRwfJ7MZjg2SpFB+QBuuNlfzSCc+6MQP+7IwTxWyRoDx+CaJKTAhAGYO+TIngQky0jg+Tm4zHBskyWS2M7byb/JCrCwZVrydvDirKkSA3dE4bBqBLQFX7oi6Ep2jDMvKurx3arNvib5peYvIMH0sP/fMnARmyUjb5pdHlTHzKF1ghsxzFM4hiRPAvIXLXXmkkwV04u3kMU9yRoCBzzdJRoGJAmDCkC8BBMfj42Q2w7FBknyOEnUyh9u+XUtAOplBJ95OPvMkZwSYgG+SjAKzA4C1tzd88/g4mc1wbJAkc5TB+J3bfktIOnFol2Unn3mSMQLM+EY2803SU2BCAMxuzgyPj5PZDMcGSXJGGb1od7mfKZPX2w7pZNnJZ55kjAADgDCObwqTnAITAmAakSQwqoSAj5PZDMcGSXJGCVp240OzC0+Kh1jxdvKYJzkjwADAEnAlZQwYzAAYwAPAjCdYUxKYJSPtcN/Vr699DAqY4dggadjjEUsXbyAA2KNpJQix4jKseDt5cVayRYCVw0gmWgKupIkBEwBgWzNgBsDMIV/mJDBLRlrT+as27TiQD2H4l04nd4mYHJI0AYyI6Ehsj/VHN03ZRESWEKsfeaATP+zLwjxVIQKs+peA5WMH1g9pO+QDErBfd046uQyAmUO+uCQwHgRn4eMsZhw3hoHJd3nMc0h6AOz8qduNBoTaJ50EOznmyfUIMA25mW+6Y9KpqgCYIAnMBMFpe/D4uCrkoYElgNWsFW7u4ePYCre7XnczH8fOKO40wi18HON6asLiajfwcaxQ2Cp8NkdhAgPAwAAw5g0betjQw4YeJsgHgN0+lxver0Y6YwLAApt4sypxAQC7+sX6If1UkfmlRAJYg8m+xan5zwyr3nlXKb+k8cYpABY/2i2ZXx6ZADaGiOiA/9P6aj1lNf1SrzeOAbBBXKEYThWppVCmZxJRYUSzSrdCTnYAsLHYUq2nrKZf6vSmKgCYoplfUC4BrBmOVO9iQRm/pPHGBQAMMPxVCpgyv+xzYHJK1gSwq+hoHKVNCJhY1JkYJWf0S3aPpPHGKQAG4MDbv46eSDjfLXK2fQ5MXsmZAHZpa+IkwIKAiUSdiVJyRr/k90gib5wBYDSo+2YT9kVxw8keB6ZAZCudwFz3Qk42c5Q+27aumDVgnYFIgICZHHjmNNFngbfsUXJGv6rukTq9cQqAAZkjTdiXMfNLnANTQDIkgNXu2u0+Q15LDSBAwGyizkQpOaNfynjkfm9sCyUKPAAM4GFfEAR9FQPDb8ZmfvOTUpCTDAlgwVFRHZfH9E8DHwETiTrjG2FllzIeud8bpwCYSKKXCAemgGRLABte9imsEDDrqDNRSk4RKk4yb5wCYI6kJOQkXwJYOM5AJAPMRSOU8EgKb5wBYA4/KlYQcjr+81It8I2f9OwXwnGGUNJBDAFzboQSHknijTMADBbsC9AVg79DNuILIglgN6dPm/rEmiiJ2C8AHADWxOtWBjaU8hAwkwNc1JkoJWf0SwGPJPLGCQDGy7f6PjEiIiHnmB0OTN7LY1kSwNqEhA9IIqKVgYsy51kQMJGoM1FKzpiRVnWP1OmNcwDMke5CyElM2d9TYjjEEDAXjKiyRwwAYyvc2Ao3JtSAxdVMTKxQmFihMLFCYWIAGAAGgDEAjF0es8tjJjCux6LSm5btBl5Vj7O6a8LAGADmuFAyVvx8uta4IBRfTi640MqaYXIeZyVbGJgUKVeoJgCmJjROPgDsVwwhIqJrHfaJMExO46xcCgNTb8pVtQAwN6JxHgSApcJUCzs/FWGYnEZFFMpRKNKkXFUbAHMjGuc5AJhFD1xWE/MF6VOuUF0ATFU2yQWAmfTdnwjtKcZ8QUh+CY9ZwsDggSlXcASA8YO+8o4nV1yGZcOEeikf+gXIB4CZdKoMeNSW+TLKQjUJjlnCwOCZKVewD4Dxg77eXx8W8HosuA2jTWoI/QKkAsBEr/0y1+jSV44E0Oa3fgD6pPa+lLMIsS2fMF/wbNi+v3W3rFEpWsGxC9139QXelneGjyAA8JKGh7i8nXLTLqx4UgP0PtodwJQJQcD/vjsCdPvBsmG0ieeKotL6Ajj5x9zIS+70RrRTwS1aNDdhx9bMF6xjwXjHeGFg8MiUKzgAwHhBX9mnTwOaCbwNo00qCP2CZACY6Bklsh8GCqvQmmrix1uZj/HDwOCRKVciABiwfFz/lLaC91T3qM73Dnh0Mm9DxBVlJQcAZlIPwbM7oprMxy5ICGFBlpQrOADA+GYkLw5cPuBJg2VDQdYLigFg5mtjVJX84oeBwSNTrhwBYLygr0sBS5bcXj9/Uhy3oTwPpwAA5soMUpRq4oWBwTNTrhwBYJWWoK9zm4Fazw4/a9mAWkK/AMgHgOWjwnLDivkCdDoh8GQ5xgsD89SUKwcAWC4v6GtFCYCKnrwNXTGU5OGgAAB2clhMcPj9o40/93fCmvn63hhnZaGa+DwYLwzMM1OuHANgXNDXZ4Nf3LR74VKiHaYNo01UfR7OcwGw6pJfXBjYXQmAmYK+9IHeFyta+gMFAaaNO+XhGADGVrixFW5MYKvwmZhYoTCxQmFihcLEADAwAIwBYOzymF0eM4EBYLADgNUsnfw+u+XY2l8P88S2G7QSn3HPCQAwVZ5ezegX3EmBiQw9i7JfrX9uVeNtpyAr9uWWoefIiq3uA8A0BGcAmFSCG9Av91JgtkZ8E6snIv3DsbJgX+4NTShunkCyAmDqKxQL+uVeCszWiBHzje+bWFmwL/cWyn+aJ7gZjnMCgEHF6JfUFFiGsfvdojwC+xLol2Z13G2LQwCs2/WbyCsCIJ5wBbp0ONu4VXHynF45X6ShwGK27wUAzWygMicHJZkGK/jNLh2ntMq+GuN2WxwBYLtfaLB6+cdNku0kXOFIt0OGTUv0AK0aWpLRR7kFgNJQYP/Gw/3fPl6BOOzpUm/Jay99O/D5Cj7mZY+OU14fzdG43xaxOUqX1SvmBaYSGWKGHc2qv9lOwtURrxSiHYEHiJY3zCfaH35bpnHYPEfhNt0RdSVixOZwAPU+JiJqU+8PoqIujxgEXlgnoqlkjvLbWqIeCSR1AhgsAJgmJL13/azR4glXNGNAFyC4ZSPkLxoeCgzQblLsVCsNBTY688upLbNnvw8gcHAHIGjxrq8FXojTcUpLt2GCBLY4AcDawwZtMvNeV9IGAxhwGvgt3/cogDoXlbJGKgos/PHHKXnUkknhptvx2D1cBPOyiUhTVstmaiWwxQkAFmGLNpl5rz9R33TXy6jr7+/vv3GBUtZIQ4FtAQBNv08LuUS0kJBLYpiXTUSaokorr52Xl1epyyuUEwDT2EebWuIf01Yb1O6qqDfSUGDbjRT1IMuZu6CwlYowL3sz+ysvAUir+1L0v91pi7drwNd0E9oU3Ifb2bDd7wBQvmdox3sOPA2gfH+iQt5oEn6XgAJLS29u/L2EjoARdUrGw+JeqElxcQCwvd0ywJ22OAbAKgoA2Em40qw+vgfAJ43gv3rXGQBLG8tkhQn9gsQUmH5sHgCsnNgQQPJ1oCRp2FCBFzZ0nGqkLy5wty0OALB9D4dEDl5HZCfhio53eXHrq9uJiA71WrR+/m5ZLgF56JdbKTDby+P2Byb/Z9+hZ4flEVHsmBfWbIp/uZzvxXGbRDTVfIQ/rWdwWP+3lQDA7KBN1/NiTKekq0UxGiUX59wx6WRrREospZ7Sx3bWAOjabt3fha283Yh5MQDsrlzh1rXdOrbCjcmFD7EqwRZXMznT9wkXdj12vmZ7wIYeF4zQwwuVWm2N9oYlArggLzCj2NDDxAAwBoAxAIwBYOzymAmeAIABAIr3nbrdbGQ0to0EA8CqD4A5zUvzoLg0sUKhtS8Pntni0sfR9d5QZaFYADDDV2k67exIiWLAFmW/Wv9cUuNtw6pLfjnNS5MqLk2KdDTbb6EMM0J/ISKiDd4dVQ2AUeHAlytoQ6LsAJiJ/DIsdfqszvLSxOLS1JuOBuu1y+tNW8M7qhoAM4wYR0QtImUHwEzkV/ZUp8/qLC+tUIJCkSYdzXboufFy1HjT5uxnoEIAjNs8tD0FwJc6jnR6uiTQja+UobUHgGmMK4F2QKXpaBBycW6xxfaqZ1PRI+advUIBcHSXzoSDVd7IRmXGTQDlf1VaMVGy0lCfhHUC0LWX9ACYlYzkV+oimwNWTkHZuDSJAbCDaGPe9P0vLHTX7g5GHMxIPJ2c+LRhzcpjPVYIAsF4NNSn9zVp/X/A+Eb3rpPIBzrc7J8X31x4UXoAbHnXJh3XrevVuOd+PN6oxTIj+bXlhbIfxo/fADTtueDd9+LqXLZxSum4NIkBsM7YKrjN0V0cDmYinp45TfRZ4C2yw4flN5pIRPr22VIBYHno/a6eMhoelB4Ay/T6kOiq924i6n2UI79aTSUiquxYQZTi+5qYU5QwXOAOz57zIclE9JYEcxS3cnEOADBvCNZeWOguDgczEU+XOwKtSzIhzochdOr2QiB9el2p3jAF+GWUFs0GTimTHACLenAnUDf8G8DQrTfM5JfJoCE+KBjV5wUxpwAl49IkBsCiTuYAAFb+6OOtr3gph0d3tRcQT/cC8EcxxPkwYNKSjdOwca5kPgShSRMAHdf9HC05ADbuiX8a/zBs20d+R++3vpPvaNC0gi+8BBxce94dFItLkxgAG4zfAADTV0/+bM/Hbfl0V4SAeLJwT2J8GNBg6Aoq1YVK5kOobwQABOCc9ADYkIAvcfj/9Hux52HruwW3xdovNzQQcnD8IBbF4tIkBsBGv7i7zB8AgrujYR0B3aWpSiAYMD3+WLqE51bv9sVG0+tID4AFD/1ill/QyI0Pkb/wTmsnAWdnLxiEozp7TikWlxYmbQJY0PKb7xq3SgCg4z0HAKD8W1QxEAzo13p5agcJjUhMrwCQja7SxIClpcMCgI1P/U8ixn+3+SHeHfx1QAZQPLrTa0BqqB2nlItLkzoBbNjHr64yAMDWOgCf7jLhYGbiSQdAB70dPgyAZtrWLpAQAJsWtB+g/ZNbygCADaj7TRf0rft+X4DLRev0J0DAnGtf+qB0TX0Rp6DTKRKXJgUXJ3Y9dqRrx9VH9815+fI4Ht1lxsE44qnJ0NsLOod0mmWPDyO6HVUmJQBGJ7puPzVtbIEMABjRnNeIaOF8Sy4aUXrTV5YcoUPo8+Kiyc1RYeMUnUiMqJ2YUtW4NHWmo9lZr3PhVEGbTrXMt5zSXXaYqFsrXpJ2cU5ecm63DnIAYECedzBQAMHcXJcaZRUfKOpUFePSahgAlp/eBe+MimIr3NgKN8d6u2tBYVkUmHA3r3BzgwalHD+wmPkLBoA5HR6vtvNhi6vvqoXnbBU+KxS2Cp8JDABjABgDwNjQw4YeJgaA3VUq5r2ztYFgl8c25zoTALYput4bp9V7enUz5GRrRL3bcfV8j2TG9Cm7nhz1JyTO/VK7Nx4HgEkNOXHSh18kovewnIjORkqe+8UAMEnMkAZyErQqd4rRjJVERONKpM79Urs3WocAmGpHTInDvwAgpxPvRoccD8j9ktQb5wCYhWbiYr64DeUjwKRhvwDApyfvxv3WuV/WssN4KSp5ATCOZuJivix5XyqIAJOG/QKAFp15N3o1EOZ+WSFhdhgvhSUzAGammTi8idu4gwgwdUJOEHuVlURkm/slRMLEGS/lfuJcCQDMTDNxeBO3oYoIMInYLxtZ534JkTBxxktpyQuA3WuimTi8idtQRQSYVOFf1rLJ/RIgYaKMl+KSFwCDiWbi8CZuQxURYNKwX7ayyf0SIGGijJfikhcAM9NMHN7EbejVEAEmDfsF54ybAAlTZyqYvACYWRzexG2YI8B2uAKJeQTkhKowbjwkzB4Bp/TXM7ICYGaaicObuA3lIsAAicO/+BPCcusMNCP9xUfCxBkvBaUEAGaJAeNivriN6keAqRNysm7Vrsf71Qup1+/xgza5X1ZImB3GS5nLYwUBMFjjTdxGdSPANKqFnFxn3PhImDtTwVgCGFvhxla4MYHFsDAxsUJhYoXCxAqFiQFgYAAYGADGLo/Z5TETahYAVnDN19ug84ouuuLjQxWaFjXWGbMR0F/y9dZRizsPBJNfBq10hXJ9Q/Kx4PGx0blfHDvi91RHtRZK4dK/s9o/Uw8ShX/xjIBuzb5z8Q89Zz8QTJW2AMCRFVvdZ5Dtt1AnkUBEROcRp17I6fF0yh8RdlxCAMxiBBX2MDgKBFOPNxZbiIiKmyeQ29LRRM5N/txPmPur9p2z+M1ohH7mNVaPb/csALr0myPBi3AGBDfXABvitQC0SWoeaiy2AMAHAOAugzx0MrvrvttAcFzmBekAMMG0H0DGZQCigWBqtAXAL83qAG6jwJwWij73KoCK6zdhBTfpMwtQvvMYYKGfrJOvpFPj8goAwciRDgCDy4Fg6hFnC4Cyr8a4kwJzVijH4iInAatiGqy0gps+fuTArCnvNBmityRcmVkx6cO/8FNWPQBnvNtJB4DBbiCYesXZAuCjORrAjRSY7ewpFY3HjRs3btwQDCIi6jaIiMq9XxPCTdv88qgyZh6l89koMyvmMPzLXZAT0QnMlRQAS4U5Y3ScMBBM3ZD6Ccwlot/WEvVIILelo4meUbp/8cUXX3zxIUyL/gH4+gHgw01bYsLg1XkLovlslDn5SvLwLwBA+aQBr8sGgIEfCKZqlU8a8Dqg2zABbqXAtFW7Bwc3hd8C4B0GAMNvxmZ+85ORfjIlX00q3QhsHCetI4sid/rLBoAZA8E++d/hyCV56i6URZE7/YFlM7XupcC0Vb1iNMFNM3LPovDw89ZslClDQOrwLwBYe21PkLQAWIgZrq3UQxgIpmIZbUkrr52Xl1epyyt0l0Eu/sQ5VVrtqDtrad3ryx8DhPSTRp7wLwC7/tioxVltGwkBsCa1bxg3/mwEYSCYii+QjbbcuPISgLS6L0X/200GOS8UvzIA2eVWew/3TeTYqOkm+im4D3jhXw2fktKQ4z8v1QDfPK5J+B1SAWCaoRsKQgFg+1AAaenNLYFgapXZlrg4ANjebhngJoNECiUfBab/8wGg414AX4eWAkCFxgw3NZ18PTgguHsYn37qYUm+0kybu0FKQy6MHzAdVHJ0IebGZkRLBYC99/0znwFIuXIfAP3Y/eEwB4KpVBZbAEBfXAC4zSDr67GTw1qFhPV/gVKHtwkOeeDfRDf6LNnz1u6GwQNv8+Gm/E5Nohv7e48q4tFPuzlWzFH4l1suAU0RdB2kBcDoyiND9+5fOKPINhBMnZfHPFuIpvUMDuv/trvS0VxZr3Pzr5ahZ0Iig3ggUWn31T0BQ8pTo5PE6Sf74V8eBYDRpRRdj1aATSAYPGjhktvS0arRsq9X7QMArNj7XVXDv9gKt5q0wq3z2csAoN8/GCz8CywBzK6a7ZjftnvgpR8GTQcL/wJLAHOkG5mljZppqx7+xYYelgDGVuGzOQoTGADGADAGgDEAjA09bOhhUvLy+NYNH2+DjmJs75qfmhsazxxjhWJU5vaffwwb3UakULI2fv6QegrF8FWaTjs7UjoAjAlO1syexlAiIsNSmy+HHhyulnWhVDjw5QrakCgpAKaQ1NEKF9bMehknLjfSbI74qKe8J9Z91QdLjkNKAIzJpY/wd6i52Ye2pwD4UsfxTU+XBLI/piK/uJS6CNbhVirSJ2GdAHTtBfkAMFYootryQtkP48dvgAXvMkoGuMuVkedws39efHPhRSkTwJhcGnpGj27d7xMA2LB9f+tuWaNSTBU1dcy9980FPu90sK6CrS642Xrba9rM+9f3L0SQcV6Vz/6Wyv7YHz/cCoBMcJezQsEvo7RoNnBKmYwAGCsUh+LjXQBkgrucKAhNmgDomPGznAAYK5QqRV/JAXc5U6hvBAAE4JxcCWBsjuLw6NpJIuFW0sNdzlvdvhgA9KgjVwIYO6PYPeKvAzLEwq36tV6e2kHhZiemVwDIRleZEsCYRAqlAhUA0OlPgIThVjqdEe7a2kXpZk8L2g/Q/sktpU4AY7L3Xc/3Q1uFhMdPJ0pv+sqSI/xwqxOJEbUTU5zDXbJ8n3Gi6/ZT08YWSAyAse96XAHAdKlREaLhVk7gLsiyOCcvObdbBxkTwMAWdVWpZS7AXWyFG1vhxuAudnnsohjcBQaAuSTncBcbetgcha3CZ3MUJiZnc5Q4jQJFqr5BWDWt0qgVAGNiAgufZGKFwgRZv+tJYi4wOVKS6bseJiY29DCxQmGSUf8Pi+GIO/EYn7AAAAAASUVORK5CYII=',
    '1108.4723v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAoQAAACMCAAAAADAEOfXAAAaJElEQVR42u2deXgUVdbG385GyEJC2AQkbMoqYAxhkSUsURmCyCIjEAUGARlHFoERYcYBhEEQBBk/dhlAEXEG2ZXNIYGwaNjBRGRJCIshCQlZCEl30n2+P3qrrq7qruqu6nTknudBb6Wqfvc9NyddVd33vq0hsGBRueHDhoAFK0IWLIhoDhsFFpUVc4hI4y33hHPnGv9VgagaMqvIYAKA1xShhrxITBUZs9+BSnZPyMI7wo//g9I8a7tmsKNTCy/fr9HH1M795X6DLt6RUdEtoH4tAPilml85NZScjycjZ9d1ihzUyLx5+lD20yNr7RhS6nVi3x7TCaqrtCvC9FUnL9RMCIb2Tsqt5VNFzkof8+cRyPrqiz/0MbUzvtg4qYvQQZ6Pe7t+2RpyoTmAjT+c79W/v5R8PBz6BVuXjfI5NaD3okAAwOzsD59IndPov0PSvU1s/uewFKGKKskufsIrRERUOmYKicQujCUienGotd1+ishBEgOmfwrE5WHoUUFEpO1tkJaPvHBXZsWQ9g+IiB690PMREdHuaD0R6ftHKypWkcFch9o6U1MdlUREJHBPGGR6eQxcmiNWuQPP/AsA/PltoYMqIfpMSf4UAALaaKTl49lYuGNjOABU33jqXQD4so8PAJ858D6xP4y8f8TUVFGl6INJuQ61wkWfvKKDBduiB3n619zib2ly8vFoZH/0/HPGVsMh69IApGcCAGKaeJ3YrNpvYJv57ko9laJFmHwIeAkAUHabUFYE6O/fBaC7lwfo72dZb3As7ZJMAwCU38tDwUPzDs5ZFbnZqEjPA6C9WaHq4AVtLh9dLpZPZcfW0j6Wl2zaCKDF9v0AoJnkdWL/M6xvrZ1aY1tFleJFCOAVANdfmrRn6qXPj+BEbJ03gXUt6q/GLzF1LFIsbe1H646N/psO37Wvv37lZ5FHjTs4Zx2Pqjtn/8en/zTW8PnqE51XqTp6XWae+Ug4n8qPRDQzN5sjEcC76N938SkdYr1O7LEe/sMKDxjbaqoUuK9Ho4Q/dsFeIiJKDfoXkX5my51EFPMSEWn95hNR7FAiovihZGlHN7hLpB820ECGFkOOZz2xzbyDc9bzfTYRXcOUC0Qbg/LVezBZTVTWzu8s0V/s8vGCB5Mo7DA3z6IuEdG2cAD1PlNWrAKDmT6V6CheM22popKIiPyECrPTFsOd8cbm2Mi3AZ9XFwNAOAAEVAOAEM7RpnZsA8DnvZgdQzWhN7ohy7KDc1bouTeApv6ZHYBWjzJqqvk3XO2LmFFnq9nn4xWfD5Ra3pQ1Xolee+lA0pFrk7TTvUzstuFA9yf3lhjv7dVTKXI59omcDEB/6d5Pcb4AWkj4dCUAAKL9vwPQTqyHNj6Ab0BbAIEoUXcAn52TOsc+H2/41TZCrrmZi0jjH+rwNb8m1plX4GVi96Zt2vRF00d7TZuqqRQtrKi2wJWUK8ZRkvzhnib0GoAIsd2BnP/qVR7B9zsuOWmXjzcUYRzOmZtnEQfgGwDQ9FpbfN67xKa1bPTkk0++aXo+VlGlaHk1aAr82KY5svn3kA6fa6moJQCN/c8rPD6Efl/4jy7l5+MNRTiyxgGdaVT2+Y0HsN249ZL1Mu0dYr+eFhcXFzeq5f4CqKvS0WscrWn9ZJvjAHDPeJ9FALK1IkfrAOBURX+7uzOHZ0GVKZIAgNYLr9/l5+MNRVhzUc5mY+vQhZlNAKTdAAAUo4NXiaXUdgCgGaHbCXVVChRhobGeUDrtbk3N+jMHAawFAHTIB7CjRimA8hIAKC+HpY3T2YBh6ZDBgK7I9GZmCe8sXTkAKi8HUK7i5TjV9Eb11B52+XjF/f7E6TOOAUDquBHzAEA/sgAAVv+poVeJ3frA+P8B2GBQWSX/cfn0wEgEDUpIGNY5HH2I6ET0km9n78NOIsrtPu/7Rd81DHnh0MCIiPiclIERtQaeTRkYERGfQxSX+d66LUM/0NKB/qF1+m0iMu+wnPXdy+E1B/64qXdI5OAHM6NCn31HnbdoUuLCQrpuJiKiG2Ps8yFvePNjS9MPfrq4tPFneiIiand43IIDR6YOKVBUrJsqkzvWqP5sJhFNax8c0m2SWiqJpM2svpvXOrvRzkEAkHfz6RqXQusEawQPzHzYUvAdH8dnPZaTWvUnrhie7mn6vP1sNF0+o4+O0nj1pFZ1VMqQesdUhGxmNZtZXWkzq8tRARYsKrMIFwzzmzWGjRWLynzRLvfTUIU/uxyzyzHYajs2Zmy1HQsWYA4MLMAcGFiwYJdjFqwIWbBg94Qsfo/3hCAviTmmfypgqwDzsUVKncCAqvw+oRr5VQ2ZVWQw2T0hC3ilK5dYFP0W4Gco920G/bUAv3J6yvkZ5edySoa7rkx+h78T5GOYudTr9q+zuyFk4nqi0ult0WeJhDNy5jSIccMQSX6HRNmvFTia2ikfWTR//IBZ9xwwlVGphswqMpgy59+eRryxUdzZIO2MVn91y5VLZoe50yfEIMtxfnKRw29Q4athpxwxlVCphswqMpgirlwQXa8ZaFrt3lza7Np7V2LdesGX2WHopFUjFUb+46NmqLHRd6TesyqriEzFVLr0YCJxhvdRn+5KPZZJOahaY1+lkfuefwCExGZcqSyVVUSmmyrdeDouOHVUlwkAutOpeo4blyUSnw0DSu4p9Qxl6U+0Q8VzaKTVAQhBjpLDonzm3iZTrkrXi/CTzWHV/xkN0LrBj9K7n7C4cVmPSIqFdtHmkT8oU4Pm/hx0qHgOyVn1AFzye0bJYVE+c++S6YJKGWZXGGpqJRDRlVgiMnQjWtmwkOhg+AOrG5cpfsOu0ml39mGtqw8mNh1a+hPvkGi5s3tpuTkQEaVgmiOmfORypw8m8jO3l+l25p4ZTJlPx7Z9Hg07T0TrqSBsMhHpI1YSRXewOX6r5vbcbCreo1WkCM39OehQ9rg5z4GIytrFlUovQilImUUoKXN7mW5n7pnBFLWGkxKdmkS1jXt5HM4VBhwHUPsqeG5cSKzx95XBCHlZmauxuT8HHaqQA4DZdXYFKoxUI3OvkemCShn3hKHmdZ8VegCBR/8RtDJulCETdQMDAwO/mgm+G1div+bdFrnzaaNNh+b+IN6hCjkAG377PlhhpBqZV4ZMxQZTxithZC2Tr971JwFcqz5v3oPNf32zNWp1FHxSv3P90/jB7Z57sSTA1XV6Nh2a+4sV7VCFHIB9F7/ywc8+bRREqpF5pchUbDBlvBJqBp81Oh1tHwwgdRtQc+rQnzs0OAwA2j38w5N8e6Ax8rHC5ed+mw7N/UG0QxVywKmTK3yA3dUURKqReeXIVGww5bxFs7TeFAA4e+d5AFj1CICuS+D6fZcArGhkdeMyFWHHGqhAM0Oe675NNh2a+oNohwC00Cqaw5XX8/488a03Pm+iHFKSSpmZV5ZMpQZT1hSxuxP9J/gkFn0cDGDX2ui2Ycn1JwOJf+vd8ude/Q/+Kzkwevhoy8Ex8XOBUY39/9jK9fmEnA6t/Yl1qPtT7uWSRk/3muowP1k5RBs9VdtfdDhmspCCKt3MXFCme5l7bDDlz1Oka2fLO7cEABRV97uqe9r4qHP3YQv7G4rbDX0ASosMdWdSq7VDbn/CHUqchyknB4lMFZBembkag/k4ODCwmdVsZjULFqwIWbAiZMGCFSELVoQsWIA5MLAAc2BgDgwM6SkHht2JGN9WZIu9T8iQHrknTK43oK51q/WAexnsSsMCCjswOIuoOABAzuS1YUCLFj8oK7N4xa2sdlPqWbYN6+8EPppax6YlMwybb5caxj+lJNKWrwgFtOHnEN/3gr1cJl8lZ5u/yx0HBicxfb/tiumZe134FixIXTdtGPEh0dkWv3FbMr9cyzDpEtFvvV1ASruPdkqRhtT1T9DTN+1zFczcdZmSVXK2+btU+pJ6axGW3axYrkoR/jmDiIojmlaYtneFaYlowqvclsz89i4jIroYR7KR0sbMKUUa8kP/IiLqNpGUy9x1mZJVcrb5u4RC0fcJZS7slh78ddNbuwUAiN37iNOSGT/dAYDm6VAOaRPKULCufSiALlseebVMvkrONn+XQg8m93QA3fXo7Bveumn94TAAqK9NsrbkIhsu+9QA7DV/PbMSSG4oQ0HBnRAAqPPwgjfL5KvkbPN3KVSE277tmbhjeVLfNA8WIW/ddO6DEAAIw1VrSy5yTNN3e/26Z8s/oRySG8pQUN2XjL+lK94sk6+Ss83fpUwR0pm/NB8TNC2h21gPFqFPAIDTFyebHuKKEQwAvii0tuQiA5OeT26/dGcNKIe0eZxXhIJqz+QBwE0UeLNMvkrONn+XMkV4vhPSYvsBTX7KB1Ch9Vghat+MM79slSHAuJar1NqSzdM1n+GTPOCekkhOKEMBPky7B+jOmxdXeqlMnkruNn+XIkXYZNCDiwMA/IoyYE2DjzxWhJx106HQA0AFQq0tubjLo5Yt+bnvoVcMyiFhsyZXCQowcOn42+kfxaO2V8vkqeRu83cpUoQRAckUC+BSvfrAxKAenqpB7rrpcOMfbhnCrC25vPGLaqP54bkpB5RDckMZCoBp/3c08d1sdPBumbYqbbb5u5T5xCSpdT2gOGmYBrj5W1cP1aDNuumwJ7IB4B46WVsyeUWpPQBo5vzvan+lkDahDAUAGjcGMpt0gHfLtFFpu83fpchbNEk9AXzr808ASZ2C6HSiB2rQum46vwzQxGcAQHpEtLUl94FOUwwAiHxGMaRNKEMBkgcXAoXHPvDzapkclcgvs9nm7lLuY7t8zSCizLqbiIhGzy5acrDVDfMnJkS0CDdV+MTkl2YT3nprwutNKig3KJqIUgNvEBm6LuC2ZL7JP24eEdGdP2hlI6V9FuGUIg35ns9lorlROlIuc9dlSlBplMnZ5jQV/NhuN9YsWt/7ABGRIXLtJ6WPVlSYi1A78oUnQtu8slzxInzO+AfTnqjkmdeJiLbFpuZPf11n05KXX9noCUkX143JlI+UOGbOKNKQP/c/dm5Gn2xSMHPXZUpQaZTJ2eY0FZxPOG3P9ayc1gEAkPF0p5dHNQQAzIjrZ3PU+90HqDqfMOf7kq5RGtuWzClw104/ah+jkY+UKtMJRSIy70BBVFeNspm7KlO6Ss62XQJKzFN8LmqDubnx8xMbl10uDPbzfBGySa14jCe15lywfi9EUh+0LMRnYMHCk0X4dT/fVZ+aN1L6oU3DzS392DCy8OQrrF7jYzCYq644FKDiGgAw41yD6VGWo7Z+f34xuxwzpIe5d3PRNExkixUhQ4K5crGK8epi8Z3rLTcGvUz/VMBWAeZjiwTAHBhYoNIdGNjlmF07K/1yrNTbK247MLAAe58QzIGBBX4PDgxmowTFHRhUsEtQwdTBTR8CkTCaWnhEphsq3TJ1UNSBwWqUoOwsGtftElQwdRBluuFDIC7TamqhVObiMl1W6Yb3hOIODByjBGWL0HW7BBVMHUSZbvgQiM84s5paKJW5uEyXVbrhPaG4A4P4F8y7GSrYJahg6uCmDwGkmFqoKtNllW6aOijqwCD+BfNuhgp2CcqbOrjrQwBZjgkqyFRGpQvCFHVgcPAF8+6FCnYJyps6uOtDAFmOCSrIVEalC8LcdGCwXfzOM0pQLlSwS1De1MFdHwJ5jgkqyFRGpQvC3HNgEFj8zjFKUDDUsUtQ2NTBTR8CyHJMUEOmIipdEOaeA4PA4nfhL5h3M1SyS1DY1MFNHwLIckxQQ6YiKl0Q5p4Dg/3id5EvmHcz1LFLUNrUwU0fAshyTFBFphIqXRDmngNDUqcgOvOwt/MvmHcv1LFLUNrUAW76EECWY4I6MhVQ6YIwNx0YYos/efB2uvMvmHcvVLFLUNzUwW0fAiehgsz8MpttJVS6ZOrgjgMDf/G71ShB4U9MXLdLUMHUQZTphg+Bo1+DydRCscz5pg7WbXdUuuo94bYDQ7pv14V3OB/bWY0SFC5C1+0SVDB1EGW64UMgKtNqaqFY5nxTB+u2yyrd8J5w24HBg4vfXbVLUMHUQZzpug+BR70nxGUqpFKO94TbDgyjI+efHH5rwfvMgYEhUVkODGzxOwtUwqTWr5f4rsqfai7CUISfLK7BRpFF5TkwcII5MDAkmAMDK0LmwMCWfDIkmAMDc2AAc2BgwQLMgYFdjh+T9wlZsICn3ycszgQABEXKekcxKw8ANE9EaFTUbvCpCkhltRnI14P5ujIcLiCdllbutuRj9ccFlFwunDJEuqQT51aUzwjR3Tw1aLbrX1jF90iA4du0cp9J5hUsSav+A+DBuqLS8ndaSrUPcMzkI+2cH6QgrTqEQbZZCAbnREsYtSXP6l8nGABG+sjLXAhpYr4RHRN069SLfexAxt2uyLQieUxhpPM5DpcxgojocOBYPW+p/Qrxk8oCXyAiyq7dvcLVWTR8jwQqfuEDHX050LRV0jyeiHLfySX6OOCwmH0ASAaTj7RzfhBi8pFWHcIg2yyEkdYTLWHURqvNz6kyMxdCmpn1APh/YLADmXaLD6a4TAuSx7RDSp2dcw0JREQ0Et/Y7sh+S/ykY1hIRETDkOJqEfI9EgyvJhDRU3VMmwuaxxPR8sD/EOXieTH7AJAMJh9p5/wgxOQjrToEQbwsBJGcEy1h1EaT1+4/lpx8rFeGzMyFkGbmCytnrc8Q8GZY4KQIHci0IHnMBYJFKOOi3xRJtj/Y6eDgJBjnOdxGsFJ+Dke2TwPw9W7j1o9NawPAEwQgGPkSTQ4cMu2Qds4PUpBWHYIg2yyEg3OiOUza4DOhX4/u3dPfbCIzcwGkhVn37YXjmth7M5h3uyLTgrRliiBlFOFd0wKYW0duGADg8mzLLt3pVD2vCIM6AsCFlJ6tXS1Cvp/DmrBnAXQ0Lq0q+3YEAGB48TDgBAZINDlwxLRD2jk/SEFydAiCbLIQCc6JpjBrwzsAkH4yQW7m9kgrE4LeDHa75cgUZooiJV+OrwYNLCci7bTPU5b0ukq0LT7oqYSEL4jIsLZ/0p4uxwVuCe907J3v8sxqvZaIqINfjunKWevZm7MXzvzVuLX4FnU2v7Jr+3UsoDLfnkREH2ODoyuII6Ydkqi0KXpc2R1f6OiqxEPydPBBtlk4uCsynUj26ZI+Pl9+5nZIKzPhwrKPlxXwpdt0KXrvJiLTjLRlCiCJiEjSGy+Z2+l+2pVVozQAvtx+sFVM1h/P+rz2WqteawAAqxem1UC1ARnh1jNSykK2U2Fq7qzBGgAVeleWPpk8EqaZniOL8lr9d75PRo/NfQGcr93IfNhPOxKf2x4s0eTAAdMeCQQmjUhu3/l//jKQNjrsQTZZiIflRGNwtAH4ul1N+ZnzkRzmxbSpmp2dDjW2Adl2KVemGWkjThwp5ZXwlYyMC293PEFERNtrniHah6tE1NL4YFIQNpmI9BErufZj2J1x40jfMaVERKvrzHF1jUlZu7hSU/MWfDOJaEyzUiLdu3qy/FEZfu33ci7Rbk0WkbYrFjv74xVmCiGJrr0xIxAvZjl7QeAgbXXYgbhZOEJaTiTipUvaRpnkSuY2SC7zIhHRc0NsQLZdiiYuJtOC5DAFkTKfjkcGphr7pfRdM3HaWoRHMCM5OTm5xRTOOX2CdUSUjflERNT4B1eLcFqfh+ZmHpoSES3H/4iWXSduPverdawg+mTArRtzFzi9KIkwBZGXuubS9b7opHfC5CD5OnggbhaOZRpPJCJ+utsj+QdJzZyD5OdLE5DDBfF3ixaLqEwT0soURMp8Oh5athYADOu6bqnVzeZijbqBgYGBX820/kh7srs/AD+cAYR8GuCCR0KNgAgAqI5UpGlrFRQUVJQXGJcjo1bMmYOS7QOEmcJInvODFCRfBw/EycJJGE8EwE93U1P+QVKNE6xILvPHYwAQilQOiD8cMmVykBamA6SfDHuHSwDw3rrzzXEcMGg0ALDhTbRGrY68Y1PKegHAeaMVhJ1Pg0seCX7tSgBAj9rIvfN3AGl1/97sne4VPwYAEbgGifYBIkxBJN/5QQrSqkMnCLJmIRo6blIAJ91pACqS4uwOcpo5H8llDiwsDgB0qMUB2XYpWyYXaWaecICUfDk+iwgDleT7vkNE3+L0/mTqMJZoNlFpg+FERGW7ubeEp4iIluI9ooc0enbRkoOtbsi+HJ+cZSCiBdeJKK+UaJ6/loj+gaum3fXiiQo1vreJKAqJdGxQAVFB+AbHVxArM6+U7Jg8pC7U+FyccNgRk4e06BAG8bMQQHJONDHN2oiILmIs7yAJmQsjjcy4DUREcWHldqB6Di/HXKYoks+s5+I94RkMICLK9cV1WpMRMpGI5vof+3cqje5JNIuIvgu5SESLz1lvV2OCdERE87GEsldwfBpkFSHHzyE3KJooO3wPkaHzONPuipAeRBR/lIgyA2IrROwD4Nh3wZbJR3KcH8SZfKRVhyCIl4UgknOikWnRRkT7MZl3kJTMhZAm5pqTRJTh/287BwZLlyKDaWWKI3lMO6S0Irw4tHVoeNwcIlodNDtjOm1vtOj7+efe7vp+Bd1oPHdeEhHRka6zN//1O/MZRSO7hwQP+piIfolISJ+ez/FpkFWEHD8Ho1VASsftZyaOLDLundglJKzvYsodtfTE8c7xWWL2AXDiu8Bl2iE5zg/iTD7SqkMYZJOFyAe91hNNMs3aiGgPZvMOkpK5ANLMrPjg0wvb2n5i4IOsXYp9dmxhiiNtmfZI2Q4M2YdoYDj0V8rbBFBRGIDyy00iTJ+mPGwhNGkr55g2Ppzj0+DmpNaCo/dj2vN/eOEiRbXXQMQ+wGl+QkwL0t75QQKTo0MQJNQjH8k50e5WbEt8PVcyd4BM+zGiaz1XHBjEmRakM6aHFjpxfBrYzGqGrJyZ1cyngUWlL/ksDgXIqU8DeyVk647ZQidWMWyhEwuwr5plwaLSgi1+ZwG2+J0FuxyzYMGKkMXjHv8Pg1LgBoTCTSAAAAAASUVORK5CYII=',
    '1108.4723v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAsUAAAClCAAAAACRy9+3AAAeXklEQVR42u2deXgUVdaHfx2SEEhCIBBAgUBAUbYAJgEBmQCCjokTBUFZRmVUEJHN4IAgCCgj4LCpqAg6wICAiLKqICNkYV9kTUBBIIJCaAJZyNbp7vP90UtVJ13dVbequ9N+9zxP9NJV59z33rpJqrqr3ugIPHj4eQTwKeDBVzEPHr4PIprBZ4GH38YMItJVl/NiHVm+/DX8md2/4ZWeUVC60vJ7fTY97ln3VsdDp2iKq9sIfAavZBXTuBoAgCu/XP7t17NmABfP51woFO/y+5kz2WYAyPsl5/K5MtQZ7aOJtrG6CJ+xycM2u9+7mo1Axpx7CF7JKl7c9CEAQMaq+5vP2nQHwOetE977TbxLxrpu7b4EgNNvtfjbinzEdp3jmxm1sN6e92bq2J8rzfVnE6bNKAZ8xyZjioG0wZW33RhcYF3fX82a9pa++o3AAv/s4n3Ht7yxWwJe2KopPMmOnDiTtVVQo4elMW/c7Uo7FT0d9KCl1b2QiMjc42d51WH90iYsrPoxeqL3gneJtxiShpnoy1i9Eja5/BphExEVt0p22KKfODIB1yxT3G+6gVankIYj0BK+EYCg6WYJeNFWDadfAf9zS22t7zCLiKhk5oEqO+3cMAxHiIgKJlpe2DDAF6vYwrooZAORHt3FW94OKiSiHqOUsHltIQhTPLvSKi67bFxkWQjmgcOI6J4o0nAEWsL3+2jK8kskAe+wVbvpl89/K/KOrTkJ6UR0YZK+6l5Tcg/geSKi7zZbXqiod837q9jKuq7mBqIS3C/e1DSOiGhiWLECNm8tBGGKD3zRNbnyVutC+B+OEdGR/Ypm14vww5xuta1i8Vbtpl/+efHWdqG2ZlpIF2DblncbVN1L37Br3Ho9gIyelhcCEzZ5/wzNyjq4aBCwD4+LtuRfDQOAqDsnfMUmZ4rLvh4iudPSiE4A4rv5bnbdrg+ZoR28/FWc0dXWKjzWLcT4xpZUJ1ekd0KhG1v+GYD8SOtLPTK8P6M21iDAsCB+mmhLrRpkGfY5X7HJmeIPxukkr2P2xFx5c84bv/hwdt3An1z070UFkruJt2oGL38VX7jP1tpn6nX9reVbypzstL8H8Ez9T4woDLe91Py892fUznpo8kPN0yJEW2q2zwOAy8j3FZsM7OMNmknuU5gX+tU7U17u86PvZtc1/MnsCf9s0SVHahGLt2oGL38V36hrP6FAjRUzR9xc62SntEQgZMSVLdhne8cIkTe8P6N21q5z1+QMuSne9Hb2dcBwHEZfsbnHrlg9XHqfQhx8OgAx/UaWoXqNwDbnXwzRoX9YqsReDls1g1ewiuvYl2pQzJTg0TXeJ2enxQBeCViCjL/4dBXbWXWt1/zwmEm0KWX+iCsX5ySjQXVcxRbsJa+6OCqhiI4G0PHi/uq2iq1zHgsA8d/one/lsNUHqzjEaD8tThwKRPc/le7stBhA9BNppwvsP7mNId6fUTsrgPoJR3eKt6UuSd/zWi46+orNLXZ2ef38/HxjRX6R04+8giMBoBayUL1GYJ3zgxkAEG7FqxyOWzWDD5S9Z8NC+2lxbwAYt/H9Xs5OiwGM2TRHOLHLb+j9GbWwGh4yHgwGIuF4+tW8OZDToqOv2Nxi669OA5DdcFpLZ7+WAzsUA4DJ8sukGo3Auj5SCoqCAQPqO93Jcatm8PJXcaPr9tPi3gDwUOetl2IAAHnBtku5PRMAAL3brtvmeJLh5bCwlh0NuNEUuIKOIsjMhSsjUJCxINBXbG6xExMBYGP7JY5za18Isw3BQC7iUb1GYF0fHYcEAzgb0cYpvHirhvDyzyge+Mn6Vs+3wXEAoBtmnmdhadrNusvtDZYfELoxAfaLO/wU5/0ZtbDWSdrdFPgtK7GnCHL71ivA4phnfcYmY4oBmIoLHecWKEc5AIwK3QnQzpfurWYjsMIPbAPgcvqiQKfwoq2awsv+fGRXGyKiG0O6h4f3n0tE01qGhj5yhojutGtxhYjo9oDo0LYrLB/1iz5bjN/i/c/uLKykf27+vr1dk6+JIOlMUsZPr/fJVcTmrY+/rNhENOrBsIiH54mwy4f2axze9olFRHQ4fuPRUUMLtRyBhvDG6YtPrG+3wExO4UVbtZx++fdHm1ruaCOxaXOHVpJpV7teDoK375K3s544SZ1jdQ6QeTvyO3fTKWLz1o3mzqbYydzmp99MiIWWI9AUPvtgZLdGkvCirdpNvwL+hb8tltgy73XpG0vfrP2mD571cMLqBFIum9cel5CHrf0I/Bpe0dUdxiT+cbfTDddJmvXWrnT4IKqyOoH0EZta7Oo6Al/CKzj7+Olxo9PX3y2UTDENPEC+uDOzKmtVSPlsXru5UQ62J0bg1/BKnx79MWucwu+R5U2SfPT0qHtW+WxefABT0RRrNgK/hlfMX6z05ruS2j57Btotq3w2bz5GrGSKNRuBX8NXp2e4+ZP8HJ5bVXjg/69VRcMLKnUxw/rFmKpB776pMKMasPs3/J/EDaTBGNSXYK3gu57/LPDgtkEef4ao8qlHaZ7QrufykrPg9M06faxt/dmbdz+oLVnhb8Bd9QHgbM3ACrpHPhhwY/MFin7SdnfokR9y7x1a/5sBSioAo4d3sbdtFeTOjhp2j8HLreCP8JVPMc6M7oR6YyZPnjAwGoskT0R+7bmWzo6q/ZS1SYdeDBjvdCdFb7w7wPz8zlCEXSAi+mdn9HrPBVjlMRhntt5eWp4WO77UIhd44arx5Ktz41wNrerlQV7QK4KewFZBLoR8dg3gnVzaOIWXC+Fv8M6rHMITRERUOny85JrbjBeI6JGn7E2KHS+xE/tnd6cHoaeRiKi8t9kVWKU044DY20REJf3+UkJEW+JMRGRKinM1tKrTsAwNDNamuIJcCLnsGsA7OYQS8HIh/AveuY+itvU0I2S+9GNRKUc/ABAkNBEktZOK6DM+czEABLfVyQUD8O43K+oCQK0VB14DsLpPAICAGQoqAMD/ht60ucbEFWSXYGP3LLzsCv4GL3l1V2FA/brSl5ZxoU6a0jsxxrut38xWBgbkzun+gKXVZMCybOCi5bHxhBayKwDAtQbPYr21XbWCnBIs7F6Bl1PBz+AlV3HmD8CjAICyK4SyQsB083cAhut5AEw3r9l2FJrFOWYAqLieh/w7ti3iLKM+F8aLeQDKLxvlrOLaqyqer5AEcx5rS/vYf6TQCqD1xu8BQDdWdgUA2DDo4fqbyi3tqhXklGBh9wq8nAp+Bi+9igE8AeDCo2O3Tjj12W7sS4x6EVjW+q5PgLMJUbbC9mb5nGUZz79pwLexdy3/6MPodMsWcdbezg1nfP/ekX+8YP7sk31dP5azjB+cfHSOBJhU7EFLW7MV9gCvIenheQcMSJRdAQAyegYNKthhaVetIKsEA7tX4OVU8Df4qmfXp9Fs2NMPYhsREWXV/oDINPm+TUSU8CgRlQe+Q0SU+BQRJT9lb1Lc3b8TmQalmMncesDea43X27aIs7r3WUl0HuNPEK2ofcvt1d0nRGUdAo8RvVoVTPIKoTO+sTWPoSERra8LoNGHripUuci4OIEoHc9Y/yWqIBNCNrsG8FUOoSS83OH7FzwROb1Lvssa89URluYL0aOBgIHzAKAuAATXBACECXtbm4l3AwGTEr55Shf+aw9cs28RZ4X/9CwQE5TTEbi/5FI9GT+Ma/434bljNZ2Aufgcp9T+3jcCADzz6I603efHlk+UWwHA+sHAQ023We/RcqggtwQLu8fhZbP7F7zEGUVA9DgAplPXD/WtAaC1jM/5ggEgLuhbAB0ke2gbANQIbgcgBMWyZrPTjKwZTsCkE5rBrqXRIxoA6g5e+vOeqFn5cisA2Ja9cuV/Y0psPgLHCvJKsLB7AV4uu1/BS67Mzu2Ac4fPWWhkf06tCz8PIFJye4jovyZ5Jd+I//f+qmDS+/eF/Xn4Y+gLfAkAul6fFh2XWwHIvq9Z06ZNX7ReK1etIKMEE7tX4GWy+xO85Pq8OwY42LYVciufR7t8c4EK7wNQVVtKRrBG4H+Dni+tAia9/9A6OwzWTrcHjgA2Wv7xqPDrzl0FYF1q3759+z533/f5gLMKMkowsXsFXia7P8G7+ilLS9s0bbsXACzal5oEILdcYm8DABwwVnkMxXWWy28Jy/1Obd698HsVMOmsenNvrLK0fjgxuQWQ/SsAoMhiZpNTAZTVAQB0QwybAIkK7kowsXsL3nUF/4N3sooLLAsSpam/19MtP7oTwKcAgI63AHxTpxQAKooBVFTYm8CRXMA8f0B/wGAzulUUV8oyVACgigoAFe7PKLKsb7xP6FkVzEXaqImvZwBA1ktDZgEwDc0HgE/+0UR2hbW3Lf9/HJ+bK1WQW4KR3bPwciH8D77ymxZHUqJR+8lhwwZ1rYs+RLQv7t9fT92OTUSkf2jWd3O/bRLW7/bhlMjI5G9TIuunHDucEhmZfIOob86kZWueml5OO5LCo/66koisW4SsfX+rWy/l4MreYdH9b0/uHN5pjMt32g73jQjrtspyW9FwJ2Au3q1ZEzP90Mn5zT80ERF12PXS7B27JwzId1HBoURmfJ1anXKIKDU2NKzHWHEFuRDy2TWAd6jgAl4uhL/By7tL/ve8NrnNNj0JAHmX761zKjwq1LmvP+fOfU7fuHOd5Ym75E37zpnv/Yvl1o5jcXT6qCmus475Zm31FTwK77rnPz+8bP6r1lUM/qwHf9bDb5/1qIARPHjAn/+a+exBgVOG87niAX82EVQE6sgYxM8o+BlFNT2jkGUbDAJ0Qfw7nodfn1Hw4AHuBuLBAx50A/m9oYsHD35GwYOvYh48+HkxDx7cmcmdmdyZyZ2Z4J96cGcmDx786o7HnzjMJvjM/KpRPBuXUPu3A4/00WhCll8NKZkQ5fVcLUqomomi93+71mF8I5/B0+dnwmpMYhOVZU5JigoFgKEBHof30MVaIwBB082qnJn2MA95m+hY6z9Y/lqbm1zNSoBxJlz2rB/8KxUMjDig4o/VqYI3JA0z0ZexeqauP7GusF4ehvfg1d0jT15t2beFRld3W56/EQy8fOsrhisEN7malZCs4GYmXPY8elIL4E7ziPM1mC+QVMG/805eOPBQh09Yuh7fLjpUB3prRQvPwnvwZ/Ewlj9/KQXzdBIR0Rc1ixm+n93kalYCjDPhsudmd90iov44w/7jTBV80zgioolhxSxdTyAiopWrycPwzv3FbuIPE3DrtjcvE0y7IgDgrvI07+ZqV4I1mpUbAIThhm/g86+GAUDUnRMsXY8BgIv7h3nhwCu9uju2M2bV1F+Kz5vdKS9P7jYGvhChybHU3w4DgAj8kuTVXI1KqJiJTGMwgFOB7X0yd6hVgyzvY53rztB1KwDmcat1XjjwClfx79tnoPSZmeNb2fMWnxFtbjLLfuiyJ+g2dfmhuRaruAihAFADBd7N1aaEmpkICAZw5GRqlG/ga7bPA4DLyGftfl2Het448ApX8UeTgV9vD8bn9oOSIv7TSoLl+4tYoP/s1K+1WMVlFpOhDqXezdWmhNqZKH+x77/gI/i3n7zeGIbjzM8OG6bs9cqBV7iKUyOA/d0j0Mv+SsuWTneMBYD4ZfooDVZxuEUjZES4d3O1KaF2JqZGbQ6Bj+BT5o/4uGJ18oEGjL1v00V75cArvLprAFQc7uV+v4MZFoosLX4W17V8L5Yhwru5mpRQOxOf//FdqI/mDkDqkvQ9r+U6CuoUxMoY7xx45Z/dnSxJBEzlta3/XOJwXjzd9k1cUBQMGFBfi1Uc0TgXAK6ji3dzNSmhcia2n/wiAGcC2voGHmjeHMhpwbiKjWl9vXPgla1iOnJvvTQ8AHyY1Nr60qOdnJ0XdxwSDOBsRBstVrEu+TgAXIyM826uJiXUzcSB/e/rgC2DfQSfuXBlBAoyFjDep5B9J9JLB17RRxNpWGqIjzBT7jQ3Oy7dT0SXgv6jzaceWSG/Epm7zWZ559xNrmYlwDgTLns+23Lkyy+P/HsLI/sHB6rgJwWcJprZ2cDWNX2PcXIOu9qDp/Qu+azW66dkD13w5cxSNzsapy8+sb7dAo3uo6D1iVm3Jv7dwDQG17malQDjTLjs2frn42LVHEI18GeSMn56vU8ua9dbMVXl+pN18BTfR1Fx7t4Q+jmyoYzfJgcjuzXS7C75G98Vd5M0J7oZg8tczUpIV3A9E56/0VwNfN6O/M7ddKxdG9YkN/I8vGb32vNnPfizHvxZDx48+CrmwVcxDx58FfPgwVcxDx7gbiAe4M5M8Hfa+Dtt/J02Hjz4eTGqhZqDh9+G1laVyhoM86orpeYR9zDXE6k5lJcSCU1UcTCISW4vKyytGHMfAODGuE8jFJURXCoOShZbHU/Di9hF02b+OrsiYGyUgmyREkY8HdXfqlJZg2Eee4roj94HmO8GEtQc0qVkCE3ccWhtVdGP0RO9F7yLSD9xZAKuSZeBS5eKMAKhjqetKgK7eNqK+k030OoU9/DCyIVhiEp6xKqi8SreHFFORCMH2v69bSER0cm+zKt43KffZ2RmZvS65KKU5Bj6fTRl+SVZHFAyKBkVFoVsINKjO1HZZeMiy+pzXsZJz69cIqKiyBijeARCHWWHUDm8wC6aNvPAYUR0T5TbroVs0TBEJbWHJyKNzyjW9ggGkPhCifVRkEMlANDqIvt5+0gAWPViC5ZSDUfbWqo4Kg9KRjQmAKG4BdRsrrjM9s1Z9RCWuOlcO9EIajaHl+AFdtG07d54DMC6CgXZomGISnoEPkBTm0oVDUaThYvNwDZmF4RIzaGqlJpkFjHJ4KJBwD48zlRGvUtFFbyIXZi2pRGdAMR3U5AtGoaT6dAWXuapwtF/rX00ffniV19xudc1jCAiOoZF1hdKY9Dz3JbkAlV3yZuSb7ksJS2XOrHwvYX5cjigZFAyK5T/NT7f8ksW16TLOOnZVE5E1DHwhngE9jrKfikzwlvZ7dNmrt/p8tR3J/8sr2tLtmgYDtOhNbz8MwqZNpUqGoyQtCGZsV1/VPeHSy1qDoZSgtBEDQebmOTQN3se2BjKVEbkUlEtp2GCt7Pbp60w7/6v3gm41HPVw/KzHZQwVabDF1YVmTaVqhoMQ6vuSzIfX9VYxSK2qTmUlxIJTVRwsIlJunY5P37IfxqwlrG6VFTLaZjgBXbbtBXi4NoAxPQbmRWiaOR2JUyV6dAWXt4JhZ6IevV2u9slTLT8CrA973eqm54uPIwuJhVnFBuj3ZRyPYaRuOGeA0oGJbfCzZrxRuFMQKKMRM+pfe44joDxjIIV3sJun7Y8xFgIfpTVtW3kDsOwv6gxvHxnpkybShUNxoi5DdBq18zDO1T8LLaqOZSXEglN1HAwi0nqJxzdyVjG6lJRL6dhhbew26etTnAkANSSSWIbuYMSxnE6fGRVkWNTqazBKMzqCUA340d2Y6VNzcFQShCaqOJgEJMYHjIeDAYicZ6tjM2lol5OoxxexC6atg7FAGBCAyUjtw3D2XT4wKoi16Yi1mDcqh2CWrqiOgAQza4utak5GEoJQhNVHAxikrKjATeaAlfEaigFZewuFfVyGuXwInbRtKXMNgQDuYhXMHL7MJxNhw+sKrJtKoIGQ187johemkVEdPWxcvbzYpuaQ7qUDKGJOw6trSrJ6USUE5xoJCKai8vSZeDKpeKoZLHW8bRVRcQuTFtu3a1E5q4vue1ayBYpYRymw2dWFdk2FUGDUdz+70RU9vzItJPLhueoeL/YpuaQLiVDaOKOQ2uriv65+fv2dk2+RlQ+tF/j8LZPLJIqA1cuFdEIRHU8bFUR2MXTdjh+49FRQwvd30dhzxYpYUQlfWlVkW9TqazBOH+kJDZBp+IueUHNIVVKltDENYf2VpUTJ6lzrM59GTc9y5DTaG9VEbEL05affjMhVk7Xzkbu/EVuVeHPevBnPfizHjzAn/XgwYOvYh48+CrmwYOvYh5/pqgxs7qQ9LJ+MaZq0LtvKvSqBuz+DQ/uBuIB7gYCf7+Yv1/M3y/mwYOvYh48PO4GcpDbsIQgmFHulxHJaVRxMLiBxCYfW0sBgy1FNAJR08Pwop6EZAaxkSjFv91AIrkN2z1tgmBGuV9GJKdxx6G5G8hu8hE5fZwyOBuzkCzS64gG42E3kNCTkMwgNhKl+LkbSCS3YVvFgmBGuV9GJKdxx6G1G0gw+YicPk4ZnPQspIhGIB6MZ91Aop6EZAaxkSjF024gjVfx00lERF/ULLYrIUjdX21sdtctIuqPM7Su5gaiEtwvfwxCrlsOKBqUrArCs5621jD5PVtTRCMQD0bJIVQOL+pJSHZexuUBE6UoP3YK4OU/PQpGN5DqEAQzyv0yGjl2tB8UywgYB8MAL/QkJDOIjcQpnnYDyb26O7YzZtXUX4rPmz92tZf+dhgAREB4SvPkbmPgCxHMxzHTGAzgVGB7IAgwLIifxpSrhsPJoBhDMYNoBA6D8Si80JOQ/IDsMvZsh54VHztl8B52A6mW24gFM0r9MuJcFRxsbiCXqiKGETiYdjwKL/QkJDOIjc47pPi3G0i13AaCYIbFL2PLVcHB5gaCS1URwwgcmx6Gt/QkJDOIjRxTPOsGkrmKUyOA/d0jRPdutGzpZLdwmADAiHDbK7EAEL9MH6Xi6E+N2mzxKular2ny2MEaLLkqOKoOijHYGOyjd2h6GN7Sk5CsrMzUqM0hlXtmOHby4T3sBlIvt3EUzCj1y9hy1XAwu4EgqSpiG72Dacej8NaehGQGsVGVFL91A0G93MYmmLmHxS9jk9Oo4mBwA8G1qohlBA5ND8PbehKSGcRGbYQUP3cDaSC3sQlmWPwydjmNKg4GNxBcq4pYRiBuehje3pOQzCA2EqX4uxvIUW7D8KmHIJhR7pcRyWnccWjtBnIw+Vhbzhmc92xNEY1A1PSwG0jUk5DMIDYSpfi5G0gkt2FbxYJgRrlfRiSnccehtRtIMPkILecMTnoWUkQjEDU97AYS9yQkKxcbiVL83g0kQ24j9y55Vr+MDA7t3UByGTx/o7kqeCGZQWwkSuFuIP6sB3/Wg98lzwP8WQ8ePPgq5sGDr2IePPgq5vHnD25V4QF/t6po/MQSe8ywfjGmatC7byrMqAbs/g3P3UDg7xfz94t58OBXdzx4uAozqocbiNVoo4GcR4Nce42vsysCxvqgf3XwgpGHPj8TVmNSqD/BA0j7eAMAmFddKTWPuMd3biBpo438v2buVhGjXuzj5iK3qN90A61OYargrn+trURiTY/dyGNIGmaiL2P1fgRPRMWtkomIzGNPEf3R+4Dv3EDSRhslq9iNIka92MfN4Rg4jIjuiWKq4K5/ra1EYjWL3cjzdlAhEfUY5UfwRDTbsoq3LSQiOtlXE6uKrFjbIxhA4rYS67+3d78NhCVeOqdpVW/lWmP3xlQA67Z4nV0lfGMCEIpbwLLYcAAPrinxH3jgYIzlkelDVwGg1UVNru6Y3ECa6HnUyHm0EPssjegEIL6b18VCKuHtRp78q2EAEHXnhP/Ao+zrIZZGk4WLzcC2JA1W8bF305MzPls9fTTkyVysL2ReawQGo42bql7KtV057Im58uacN37xOrtqeJuRp1YNshzrc34E/8E4683xw2Ne6/Xz1jX/8p0biNFoA83kPBqIfQrz7v/qnYBLPVc97G2xkGp4q5GnZvs8ALiMfP+BP96gmbUVkjYkM7brj0E+dAOByWijnZxHA7FPIQ6uDUBMv5FZIV7uXzW8zcjz9pPXG8NwHEa/ga9YPd/eNrTqviTz8VWNfecGApPRRjs5jwZin1BERwPouHJ/Hy/3rx7eauRJmT/i44rVyQca+A38klft57qnX97aYNTLPzxxIMBXbiCwGW20k/NoIPapExwJALW8LxbSwkpkMfKkLknf81quUh+E7+Czy+vn5+cbK/KLgBFzG6DVrpmHd/jMDcRotIFmch4NxD6BHYoBwIQG3hYLqYN3MPI0bw7ktOjoL/D6q9MAZDec1jK1MKsnAN2MH13oXz3sBmI02mgn59FC7JMy2xAM5CLe22IhdfAiI0/mwpURKMhYEOgv8ImJALCx/RKglq6oDgBEt/eZG0jaaKPkszs3ihj1Yh/XJXLrbiUyd32JqYK7/j1gJbKFYOSZFHCaaGZngx/BE5ExrCcR0UuziIiuPlbuMzeQtNFGySp2o4hRL/ZxU+Jw/Majo4YWslVw07/WViLxfRR2I8+ZpIyfXu+T60/wRKMeDIt4eB5R2fMj004uG57jSzeQNnfJu6yqXuzjrkR++s2EWKjW63jdSmQ38uTtyO/cTedf8EKcP1ISm6DjbiDwZz3An/XgwQP8WQ8ePPgq5sGDr2IefBXz4AHuBuLBA55zA1Wbd9p48OBnFDz4KubBw4/j/wBpsmOFoEAdawAAAABJRU5ErkJggg==',
    '1108.4723v1.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAY4AAAByCAAAAABhSGl2AAAT5UlEQVR42u2dd3wUZR7Gn00jkITQInAQCETgpCRgAhhaMAQ5gxUbTQVB4CwIwXJiARQFlaIngkaQ40BARUDgToSTGEKTIgQIIiWQgFJCIAlCsvW5P7bMO7O72dkSRJ3380F/M783z7zZX3bed3a++4yO0Nr104K0l0Arh9bcNZKTtFfhemiTSILXtk2y/avWA/w+xUlSd42nch2t/6r1AL9PcW3uuO5aiIt9uzacazW4/soB124U5b+EhViMwS1hPhoWYuSNf1pxF+WYeO61RvmTYr+4huU4uzhna+TQpJYwzl+fn3b7s2p+5vzYj6KrSfzye0VnOjzTsHrEL2WVVxifauN+ZSVvXyWZSZozknyejyzvuc/BzfJhF/pbg8tdLSoOUTxhVGeccXcAf8UHHmfZ/dHbq0f8qWLy7bCNrrPOc8fitCAAQX6sfosPef8z4Qi3BpHxOhXdo56eO7jaxF+d1hK1FwYPNleH+JL52cBwwyS1U3lBIQCgc5zP5Vjl3+pFTacazYOrT3xdt0tAZOqJw9Uh3ogAInBRbTlar/gaAHRPA+YLPwMwnC0BYCo+B1NBCQD9SZPQ3Xi2BKW/AkDRpuMWADgw0ZE07Mo3+/SylW7PMRSKCtJh/G+exGP1BgCROB8IceXAB15+ANiKO9SWYzwy+ry13YBUbE2NGQFktW48D9jS6YZJX7+9a/hjlvnztnad6+j9n4TGH3/wfrMcGCZsjFrV5yjw2YuV3w4duhgAs+69WtBjqw+/08xF0TXfSBIUHIcJQPMonnumIYD9Ie0DIO488FDAMDP5ZbVTOZfXAdDwfZLs3I+kPuR1kuyW9i/yKJ7ZRy6sdVGat1sP2HKm0XLOb/YjmdnRTLLNaGvqgyZl5Dd1LqmZyg/gPls0hDycStLSXVSwH0Zqs9VP5d6LkzuRGRBxZ+0dz3ce/WsVKx1lu7RsdCtgBsm+/Ugy4nWS7BdjJk2h95Dcjj1S76REkuSKurvJdTgilaM0eixJc70PvC5HTvRekh/LFGyH8b8cqsRZ2SG9IjDiztqWn/52Z7Hrcri6DKwzcCBzHpwyoo78VNY2CAgOawcgHFdQagIQGg2gAwDgvgG6E/u3o0zq/0NZ2BYADY54/Y7vEtepXfqdI+UKHQJ0jaNOfGLM6vAAiTtp61ovaXL7jmB1c8dnAKDr/dHlvU4rOum/ZtwRExMTcx8A1AMAWLJSltTvLvYvxA3h4eHhn76g5veIgm19YDID4Tmv1vog/RGLTKGe7xXwXnzBL/+NCJS4i4HX77z7G5VX5SseAgD0Q4V9djG5+sEv9PbaWNd3z2ftjccWwKLTAcCCEbgJ9ZNVv2LN6hdbg2NNgaM1p0y5tOi5ETIFne/l8Fp8Xd6nQTgY1DYg4jJtQw/TjjCgHo6qXFkdOm79oACJQA0COKd39YON4+Li4hpJnyu8+2g8cB7YsBXhRqAASPzLRgDQr1G1Zr93T7n1r+FeIH85UHfcfQe9Ugig+PZt7wUBX9WoBvHK3fvPAziFRJXlMA8uBYB5w5sAiRcBrKxdAQAGIwAajQCMEK4mDOUAEFrTBOBAaMWZeuh4DCAQ/vG6/QDei1X1W81o+AwA7DndDcDcqwAMt4gK1sMITQ+96np4J354aMnfx4x+eH5cQMTl2rUzNjUFivJTe6r8BD9h1mdxyWFrij6JBi7c27fz/g6jylI+PzQ9V9dz4uFFu+olfTJ9w7H4Hu9be3/zz9zwpIGPAl+OfzJhT//5e1OnBhekDdelpgLIfunWNgd7Z6i73/HzmNBRQdnlb0cAqz9Kahed23ispOA4jP1vYHjxgSuxrXqPU3lLwivxpB+sL0ReAMSV2rgwISGFExrMb6SyHHuSeGC3OamT9ZRXcrJV7f1RMRGez9zmw8a2YSyPBmA8EGebv37+tbVO7e0nHt1j7NoGAMprhhwxtAp3reDbHaLrSHxfHjsl6H6j+1va3UBodwOhgT1a08qhlUNrWjmgUYhaw7WmEHUawa6drLSmlUObO7SmEewawQ7tMytt7sCfg2B308qLgMb1AeDHGiFG3lhRIuXqRlzbYVuCnKM/WTnOrv5xaeS+eAAL/7e3d8ZzBXO37as7JAL60zuLZo/zZxB2GF0Jd3PBwcjg5yMUIYDv5n7uFHkrbll0qsLy+I0KccuXh4xBT8fIIv/ELR+fDr86LgZyMN4df+/NRHPgAfQ0kaT+VgtJfo+7SZIVw55RK+HMWUkwuhLuNmQMMfOzhGJ5SPJKvA0aFyLXKJR7ccvT+8lfbt0uF7/c9xUDF98li/wTtwx6jdzT+hcRjPeOv3dfjnnPYAZJ8kk58XVhkO/lqDxpshFss8M/J4vRzZ55LbScZPcx8pDkVEcRpnooh3vxtbNIMi9dJm65fwjJG2PEyE/x1dF6kqPuJ/9+guTlei1Mwg96/kJBle3N1i85fVvAaED9Or6fqSQYXQl3ZyVEAbhlyVVZCOxo0cDWQ4q8Fv/+NADEF8iOs2lFJoBlX0GI/BRf2j0MQOraqwIY756/97IctRYZHzUq9uVuAPoFYh5TwN2lpyMBIObXfWIIVH45yNZDirwWR5NZ71qAtRmy43wY3RFAcgqEyD9x88ZoAGis/04VGO/tuuSWF3ZPU5YDwN0BWVfI4e6awbSO8LAYAv8ca7/vL0Vei2NYi/G9f1qz5A3xOMxuceqlaf84AkiRn+LFlyIBIBpHVIHxId6+ZJPWvX7HzY6tnUONRTvWBmyd9/3K7JtX2JdPNdqXAMBJlIoh9jawg1tS5L04wr8blJvQ9dtQ8TjlJX/94vWgEz0X9ZEiP8UvIwIAglGGoDAAu/IyYwJ5GVjj33hE4s26LFn22W2BW3Z3nb6kcNAF+9Zrh84Chr0wiaFx8TD7nOWIfBGHIf7ZoNw7zorHKceOB4PQou+oSinyU7wSYVYw1IrY6kekvxHYq/KOk/LFzxyDmo0FYN4foE8hWi/ZcLsdcbxrxuOnCqb1RwMxnPOkfchS5Iv4gUdmvXOwz4a7LYJ4BJo1A5BYsE2K/BSPsgKbJkQB8AzG+3BN+4/kd2SD7NQOOLwzUG8QEe7OnJOTPf4cEoXwkL5+aWmpyVh6WYp8En98egPEb5y8c70gXjusHgDURL4U+Slex/q2qEQ0oAKMD/H+BQv5d6dHewnbfwGwo63/hXABdzdvDhTGJQrh1tMvAzh0w8stkxxRpg/i5fk9AegmfXskQxIP6XAFAMxoIEXwTzy60TkAOIsuUAXGe3MZuH+u9f8z0U9+GWhJvujzZaDji0xluuBTJDshm2RJBbn5nlKytM4CykKSbOi4+GvY39N1rWtxQ1SZ9RtLG0XxKaF6kq/iiBB5KV5SQZn4iJtJ8p16JnLbixaSU49V8d0tr05W+bZLwHE2/LoMBgBARebPdf15X1hhdBHuvhDbA1i35hTwbouHIQsBmK/YqXAp8lI89KF3AeDni71E8TER3wD8ZmQrIfJO/EJsD8jEMw8VAFyZGSwH493x9+rfGzvToyNTFpEkjw8jueuuZqh1z5AhD3StgzTfPyTRD+7bKKrt3bPJ4kdmbN3Stf8ZklfaDyUPZmz+4dm0c6QsJMfcEhnd5y155OYP2L145aOjvsvLGlYoF9+ZvGL3mMHlssgr8Svth1ImzuWp+RcnDDWQtsuDBPEHr+/bTwq4u2R9aacUnTL09Q6Rkhw/uutqQmedQrw050LnBEXknzjO//dKSifddXJ/S7sbqN0NhAb2aE0rh1YOrWnlgEYhag1/eIJdW+hqJyv8ATmr86uPsdk9sUpz12sNv1kYXN0kXUAVleOt+jBq337mqUtn9QnaPvbW6eGAzdw1K/aL3cj3Dn5zebJyMGAK1OzhpM61irbflgYBGct5MSMmAgAGBwnpKn4hu7iyt2Lbys9JAJxLf1XV4pKiMF5pp3vzVnWf/JkGJFwiyat9e11Vmrt6Bb9Vhb05oWYNAYS+YpF5qc6zjbu3mFZBpjn1lm9b+TkJgHPtr6paXCLyhPE6dro3b1VZjtfs/m6nQ0eTvP85kuT3SV7Db1Vhb06oWd8PXvz4BEkRGRv70debc3M39z4hplWQaU695dtWfk4C4JQgm5fiEpEnjNex07W4O7c353ZuWjfbx8NNBmSNbYuCICdzVyPDfIPfajS3R5tW7AGwTOK4bnjCHq1bnV8XkamrDrcLGgUAi0bEiWkV4k69Zds2fs4GwH00s1bV/qoexSUiTxivY6d7cXXz19IKx1kxjQtl5q4IHPxWBWomIWNPAUDBtiEBnXRt/JwAwFXtr6paEbLx2ne6F1f37shGS3sYj2xg/OcZabf1SgpLFcqR5C/8xuwWp7Iiyx5rLe3K22QKeSwaQK4pDFZkLB6AZexinSytqil7C9s2fk4C4Lp58Ff1KO4g8sTxOna6FVdXjtNwnIfq4hTQZemYTZvQ8OWnEED4zRk1yzs0Treqy4bmUCBjyzrUladVvWCK3sK2nZ8T6ToFyOaluJzIs41X2OlWXNVMnoRP7eEWNJKbu/IA7qO58La1/qILRQguJDmspcMxNY8kbx6g9FLVxxa6SFdNFzj3lrYN481k1/4kv9KdIfUpeMuNv6pacUlRGK9spxvzVnVzRyyK7WExmgFAnYEf/pQdM6U0gPCbM2qWAADJK4sVXqprdc1cpD01ZW9pW+LnBNBOAbJ5Jy4n8mzjle10I66uHOn4wR7uQbpLc1f/4Tcn1GzHZgCIsm1LyNi/WrhIe2rK3tK2yM8JoB2q8Ff1IK4g8qzjVWJ6LsXVzR2DJ643hFnPbetCHndh7hoI+M0JNbur7HIYYEB9uZeq6bt057THpuwtbReL/JwNgPPgr+pBXKZoH6+086kqxNVdBs5FljVYj5dItj1GkjyL097Cb1XMHTLUrKSCTF9AkunRRsqQsTw8RlKe9nx6F3srxR38nAOAk1FyXomXVCiJPPt4HTvdiHuBvY2Z8OxmAMgfOWiKwtw1APCbjQETULMLsT2A+28CcDJndojcS/UXRAKQpVWIC72V4hI/5wDgPPiruhe/ENtDSeTZx+vYWZW4Wl5tSYtXvs+b0fx9M0l22Dhy6vpN4waUeg2/VYW9CajZlfZDSdMr7+5b3m6mhSIyRq7BROvnaFJaBZkm9FaKS/ycBMAJlJxX4lbsTUbk2ccr7XQt7h32Zt562NKqV6gLc9fA3X5SomaHdtRLcXoilmFJ/4bu0+7Flb1d/bQAwLnyV1Uv7nK88Gjeqt0N1O4Gak0rh1YOrWnl0MqhtUC14MnX+IC9bf+q9QC/T3GNQoRGIWrXHdrcgT+c2xvwxLAu+K0pRPwBLfd8K8fF+eji9Ih5/y34JCTP6bHtApUoAxTtlnueHyHvXlzwxxMZQrukkFYh7mS+53DcE1IS4ii3EvTxa/5ZaGBw8Yh5fylEB5Ln9Nh2gUqUAYo2ls+lhZ1accEfT2IIJUkxrUJcab7ncNwTUhLiKLcS9NV878HBWG+NAkkhSkiehBrafimJSpQDilaWz7WFnVpxwR9PYgglSTHtWVyJFUqOe0JKQhzlVoI+ui6cafAwlsPdI+Z9tuCTkDzlY9sFAzyZF56N5VPzCHn34oI/Hm544s2RcXJJMe1ZXIkVSo57Qkry+JNZCfq6svr8gT71V+mVj5iHvxSihOQp3ekEKlEEFH2z3FOKS/54LpuHtEJcgRUKjntSSkIcZVaCPs8dA0wcg9XWs1QQ0qZv04tPiJ7sB2dF/d+SS0mzniQTQ87bzlX1O56c+OYLP8lD8q0iB63k8WTlVpwVLdDz8Ff9y0hyyL5Zb88qlUmKac/i8ohn8DhJ7sFsIVUZ3Isk38YCKfJj7igYR+bgIadHzPMAYoc8eAt8L4fseerSY9tL0f0dMwua/E8W8ocF9KYcbsTJwm4I62ldmrRfauHK1idlkkJajbjsMEcwjiT3YbKYSmxHkk9gphD5Xo43d5DmprV+VT5iPgAUooDkCY9tF6hEIZSxfGreHa7FyaMPPxuO284oGUK7pJBWJS5ihfvxvBUheUFMSYijDHb0sRwpnyxcuLAnlklDyY6JuuQ4Wa1bS5ryfDxZ8UKNZOuSJzPN8TdWghbW1+dbMZx1jN6Vw7U496cU81gfdDHbd4zCeUHSKe1RXIhOYIL1ZDVV1mnmHUXHJ0/FAjHydWV1qE1s06ZNR1jXVgGnEO1InuhOJ1CJUuiP5Z7M+k4w33OFM4refCot96RI7rjnSEmIoxx29OWqfFlmBwCc9nVpnYBSiDLeT+ZOJ1CJUljsu+WeTFw033OBM8q9+TyJK5lFwXFPlpK8BAVXQZ8WuszvAAC6QYZVikfMC30+vMn7cojPUxce236xErjruAHAOSQLYeqcOXPmzKndfk6mf+I1dda3V7P2QOK8MAA/RgujF9OexcXDXKwEdP1PAEBBvSRxBLn3lgFlm18JESJfF7pLrF815G50N5NtulwiyUnDrV8xuJMkeXVcY1+m8v45JAvDUk38seWo0aNHDY0zkcW1kshzddaQlq4jKQtJmiJ7WoPpOFn13OFefOQUkjx9u578cBvJE6GfiJJCWoW4FBXXSiKZH36ctKRMlXV6PugAObmTgULk21Sem1y7ZsdCkpkJEZHdnw4ohSgheQJqaIX5BAM80QvPxvK5trBTKy7440kMoSQp2ud5Flea70mOe0JKQhxlVoIBMN8LLIXo/nnqApXoygtPzR0i9+KCP54rhlBIqxBXHkZw3HOkJMTRrZWgdjdQuxuoNa0cWjm0ppVDK4fWoGFv0LA3rWknK60cWvtt2v8BkB2UJNd24S0AAAAASUVORK5CYII=',
    '1108.4723v1.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAXkAAABZCAAAAAD+swtjAAARL0lEQVR42u1ce3QV1bn/5UkkgYSEVzUggQoVIQhJoDxiMCK6EorCRRECLYoguiBK4CJQLKmlgFVEe731idhVJUipyqMIIhCCvB8SDCnPQAgCScgbSHI453z3jzln9p45e87MPmtRq3f2Wll8e+/v/H7f/s7mm8nMLzuIYLcfpAXbKbAz//+tEdFCOwv/5raQiEC3rC30/NxSgh8nOBFR0K27wgaR8nNLCX6c4MZ1nnbK4nwTQKDmLN8EtHyp6CUo5LPij0GYecoOAQCUnTp/4ey/3ABKTpeeqeddvi8qKnYDQNWp0vMnmtD6OekkeVn8tABQLeEGRCGHa8og/E+13PEiAOCT4mXNE38xvTXw28XtHs3uxbnkFf352qpxAPLf+bRXxsyO+OjSfMlqo7DUvFffeHN6D+0aVxRFhcyJBASo5gXBE737H8U3g2e0085VZL8bDQDuv5Y1uqf8HMYUvuDerHgxGt68cLn38x00PtxqGL8Bg6D2lya5PFZdyGDFeCW7RufU8HjYLxVrUD0RkXvwSZ0H/F/AFZbK6ZVEfwrfys84MrJc9GlipRBVR2AYfcODLznobyP5mcpZU1NwmYjIPeMY0aX795IxBQziVTEqnzhLdWOi92oY2Go4fjGDKPpfv+O1NuH3REQ3cvf6OG1Zk4WDRER1s5SBNaPlMq+wLI9YQ1SJQfzMy2H1RDR4mhDVNPMKrntMFhH9vB0/03TeuVzJ2obXiYgKh5ExBcS4DOPZc0TUEJvg5HzYajT8QgZB9NWx17zmHOwkojNzKn295pXvxW+IiDZ9oQzcbHNZJvMelrwWa4hu4Bf8VHwSEdGsqOsiVLPMe3C/xmEiOrhHN+vJ2oIcIqJrXcWBC8FZVjwYnX5WTUSjUMQ5sdVo+IUMgivs+nsivWZ+RH9gw7rFbX29KtsPSFpdCaAgVRkITflc5rrjYXmi4TFgN0ZwM7UXowCg3bWj8qgq7jvR9wJIHih2uuP1N9zAhgyJwLmsKK1TswNAFCq4MbYaDb+QQZD5ggFeq/7wwAjn3HU5gmv6tUgEzWj+AEBtrGdocIFMhrwsYYBjWfICbua2EFIiOyGP6sWlHQllv10y95SB06SEmUNPrv/4jxKBs6x42q7LHQAcC+XvO9TV6PhFDILMn1FvNHa7hl753fvrmgSB7BkMjI1724n6Vt6hO0/LZEhl2f/ikDvzo7mZFr2qAOA8auVRvbj1VZF//8O8Z9K3iZ0i8gftSnzt89YSgZ/poc9cOICDhdnauyfPanT8IgZB5iti1GKDkJW5U66uEgSSnwZETClbh91DvEOxFTIZUlkGLP24dNxVfurl4iuA41s45VG9uPXY93gwEh6c2iT2cnSbHbxrxBWJwFlWuNY8edgftSOe1ej4hQy+pT92i9fqH/YJUWlIotvX6WkiotLgoTRXvd38poXMFZaxEF1tkczfIdCyERfO5i7CChGq2RVWwa1CgnIx3Ca8wh4bWElnHkB/FxlSwDBeDwYRUU76Nd9PXm2R7NTxixgEez7CqZb5tPFA51HHdorKPIDOj+R/V6fuBWeEzN5UWQDEpRzaws/lvLVzx8xy9JFH9eK2Do8FgNtwXOg0ZWlbdNuae2Cz9cD5eL1txaVNkb6jcSmHtuj4RQyCzLevV8v8/QCQjTeFZR7AdCxpxe5J2stkSGFx9O/nABALbSG8c8Lk1qVd+sijenFDe18HABfainzqj6cCCFqYesp64O3rfYY2Fn7SAkXFXA1TV6PjFzEIMt/hilrm7weAIX3Xn1MGqhq8PjvuAwDc3zMvlbvPlMmQwtJ06FgFgDL04eB3jaoD6gpeCpVHVaMfedYBoBzJmrC9d09BykjnXtYDV7Oitr173gwG1rVgDNxqeH4xgyDz/Y54rgD/DE8CgKAs9yvK5+O9t8c1a5TvMmh6sHqBxZEkmQwpLK0ztscDF46npXLwG9eXAW8kTAwAVY1+WuQWgLY8fRcfNtCMZgBhY98AgO+r77NO4c2KFwMnJlQ9O+2ZiR90YQzcajh+Iwbf0r/1biKiinGDWrUatZSIFnSNjBxeRETX7ulSRkRUM7pzZM+VytMb7vfi5HUyV1iFhSp//drubwZkXubgqSij4Mjs9HIxqtkV1oNLB5LXHpo2vp7HbR7/YMdWPR9ZTtT0m6n5he9NKvVDATEuw+in5C+RZ+BWw/gNGATP+1xdN99t8MV/0bub4aa4OOB8mMSzSpXlaCH1TQzSwFdtru07MEiMavasUsWt3Xk1JdEw7NMHbySm+KPQg/vJCs/AVsPxixkEX/ey54122VKn8fOr+YvknpgJWATwPqimT8ys4ZpSwHpWAmMIFXyB09Mu3S78Zq+Q8buB6q2Sb2x8WQTw0qgWcQOgMMxKoAyi7+jICPFXuLje8Gt1jdlLcnvel8UXXoBquuet4FqggOWsBMggLsTbjmdL7rT378iQfgNuziJAtfCSWip6IwoBuHxW/DEYRH89UhL+RssAtAemLAJUK/IAmeiNKETg0lnxx2CrPvADqT5sjRl+II1ZkK0lhq1otTNvN9haYltLbGuJbS3xT+auMtTc5blJ/b3mwa/K7xof99loNFax+TaRtyAud7DVwf/I5tY9yBFEbp756g/gzfz88pc7Hl/Y6e+jUfKXPUfbZEWi+eKBC8tfsBwQJ/jU60E5jSmA/L+sgUY+qhmUBmd9bsb9/sWIGy+086ExA/fxVrWrbA275mW0iwSA8cGGkZvWo/fQ1qFY65JcROTKSCIi2o9HiIiocdLzfh5oaeE5wadeD8prTImud8vUSlC5QaOHWn7AWZ+bcY97mehw90t6GlNwvbeqXeXW8LYnu0PFkZOlK+zj47FZscb8NxER7U8iIvoO/+UROYyznHlOvqrXg2o0prRICZXJR7lBo9j9gLM+N/NFdDMRTR2jpzED13sz7Sq3hux3vyzYtatg6Dlx5AaqD2273HYiVitmSSkAIKULm73pQFyM5WLTkQBEohrAxkE1QFTauROeqf0XAaBbCQBgX4LymrfFnVy19A4GAM763MyqweEA0jbc0NKYguu9t6/NAZC3TrOG4KkPpw4ZUjK5i3Hkpplf89gDcZ83AwC6r/0SAIJmsNldXwEPWc48J1/V60F5jWnTP8b5flY4aBGc9Znl2hoNAD9rzpeMXN+YdpVbw3QAKNmT5S9ys2Iz2knToAi19wcjfemeZmVcqTa5G/y/uPCFb344uZaIXM1ERH1CKzzDjQlIPbEus46I6JULNCBTL+jiBw0rpQE46zPrMqYQER3Gcp1uzAxc5+2Ou/f8/MUvntStgYhcmdVGkVupNuc6h2Ac8gAA/VfFbJ87qPNbnqkDE8YOzJW821Llq3o9KKcx/bZtJ98PCgetgrM+sxoQCQAhqJOMXNc47apWJ4u83m38Rm6y5RfvI3LFt/TIB2vynrkLeM27512lw2X3vPvkw7/y/h1EU+9hjerE6YmzIzD8MpFjpot89rx20GhbGoJr+op1DHOIiArxosU9z4Hz3hcQUkpEk7o2cmsgImruVGoYuZV7m4Efrly5MhV5LIAd7VrVqNVm4wYiZ6FMteHkq7welGlMXz9DvpnXDhrHLgbX9hXrHGYp1WaR1WrDwHlvTruq1cmu7UyGkVuoNsU9OsXHx09W7m4+BYCgoe82fKvO970HOHFAquCo8lWNHlTVmBY3x9XW1jpv1vKSPOGgdXBN32PFoBEAmhAtHznfOO2qVif7UYL/yE1+h83L6Q2AlnxZGwOsHQsAeEiJGQBwO4B9PS1G7hji3Beuylc3Fn4SjKLgnhqN6bZTkRcXAChuv6BrDvtkpWjQMrim77WiO5YDwBX1F3TrkWsTqGpX2RoyADjzh5lE7rfWuEcp/+biQyLqeYaIiK7gIv+blDu52mK1qQsKKSOivthBRHvmuYlo0RmiqkYiRyvljiDL8+eZHTJ9xepsUFQQ/IDzfWZN7kdE9Gqs00K10YAz76pGIvp9WDMR/Q6ntGsoxFNkGLl5tVlVo/w7AivcgGt8LQC8/eQdAOrgAAA05nzfxuKe5+WrTA96tdMQH42p63q9Vj6qHZQE5/qcDDWnuASgz5Q/AuNozMCZtwKuale1a7iEKPiP3M+O35Xc+rZ7S4koJzEyavAM6r316UWbt78wupbo4MjOaPloVtZjA2KQbv25DRN8Mj3o9V4TSKcxnfbLqOgHXuHlo9yg4aMVY3DW52SotDrtePWsCQ4djSk481bAmXZVs4b1mO/9sG/kss/nDyfRd4dcSX2DAn8+z8lXYawxDfARuh9wYavYdH2g4WIkwJl2lVuD4+PMDj/UCwD7zYj9BtzWHtjNzrydeTvzdvu3t5DcW4c91PNzSwl+nOC2xgw/SS2xfT9v1/n/yGbylLi6MizUfZO6g+nLrnv+FD+iSyhuiZJMSqn1k838ubV7CqLH3t2d05dVfbp7R/yTLXF12+1LEiXZdEckaqRX7JRB1bKi1BKCT0xKaXlh7/B07xTrczNMY8arzeTBWbycEzN93K1qD45ilF5fdhqTiIgah912kEjmPazuiESN9IqdMsgsE6WWMXgHAGEvsWN5WJ9ZTGPGrIDA1Xh5IRoz9e6WNWaeNyC8vqwMk5XzcvCAXOZ1RyTy0ium1OLOGzRRahmDP/i/894/x82xPrOYxoxZgYCzeDknztTH4u9vwEWtJNhHX4ZOOClXbDilFnTSq+1rDwPIu8lbCJ4KAH/1p9QSg7d/TjvH+szyaMyeutGSWYGAs3g5J87Ux2K52nj2/BPBm4iIKJ/b8x8pYiEJ1YeDiLZitqJtmu3mXsmPiXb5WHSGiOjsVLevu+j/KweepQsly9dythlHRLQd/2RWQOBcvJwTM7MM0mN1z89ck5E+/L6k8DQ2dHjuQ8skr7D8EYl/zubeM9COhLL3ouqe6s5ZQDcA7uy/Bfm4m4IXbneGPsWpCljfa1XWRAFANE71U62MAMD5eDknztTHIrnnaXUMgA7/Q0RUhpRXX10ytsPbbpLc87RvTsoziu7lyApehlKLwa+6qOSOrzlLaR/PFbiLr1EMvNcqN33W/bw6w/qqdQovKDcQucwKBFwTL3Nipj4W2Sssry/zVJvSIeklsplXlVo66RVTavGaLROllrEMrJCIqB879Yj1VYtpzLRqM1lwbbycEM1r6mORzzzTl3nrfFVU93rZzHuVWjrpFVNq6c4b9KfU8qsxI5qKCs0c609FBacx06rNZMH15zNyTpypj8WKft7TBPoyxA4/tVH+d7e4lENbfKRXTKmlO2/QTKkllIHtKwCAVuqRiazPLKYxs6w2E4Lrz2fkhGhxKYe2+MQi8xdqAIT6MiAK5TI555RaeukVU2ppzxs0V2qJZGAj6xrCAQfiPHOszyymMbOgNvMDzuLlnDhTH4t05ovPdgOABuX4TqU15Qely2S+6VBwRbxyqGBaGgCs7fUWgOqWEcDIRY5w5ZRBZgEovhYLQONuDo4+48IB/Cv6bg846zMrKPNbACiJTWJWQOBqvJwTZ3Lukvc2h/ErIqIe/WuIiBY+SURFmKj8bRr+IHeFzdxJRKXhaZ7q54xKJaLKlklEVB6znsg94GneIqIvka1+WnE3LMUc+Dt7iOhc2IdecNZnFh2POEvkHriItwIBZ/FyTszkGKWusF+N6tEqJv1Z4vRlxY8moPXorKzMex/6WvKuklNqMemVXqmlOW/Qr1LLENz50htHV9+zzO2VgbE+sziNGWcFAM7i5ZyYyTMGpDGT1JcFoDFjSi3uvEG/Si0/4MX7YgfyH2R9boZpzARqMwlwFi/nxEy9u60xs99JwVZ92M3OvJ15u9mZh60xg60xg60xg60xs5td5+3M282s/R+ylDoJlShEmwAAAABJRU5ErkJggg==',
    '1109.4653v2.11.png': 'iVBORw0KGgoAAAANSUhEUgAAAZQAAAC7CAAAAABnIzcrAAAh2UlEQVR42u2deXxM5/7HP5NNZBESsdQuRaml1nLFDamtaFTVruWWorqpFqVU0UUXVfentEovLer2urT2cqu22kqtIaWWEA0iJLEkmUnm+/vjzJznec4yc2aztOf7euVlzpn5fp9nzjMzz3POeft8LAQz7rYIMg+BOShmGAkimmwehbsnJhNRkGNg/uQx2fEXgLJ+DgCwkPT3J48Avc0AlTTnlLswQlw+ez1d+jeuggUArqVeKd2Wfz73yJVSyYbbUqVrhVDy5hkguppz6+wNoEak3w/BucISxdaycZRWIsRWmFACl3NK2MMr3umJXn9OOfXG31Fh4ntTetd77iIRHRke3kd+qs1SouMjInoa/7nk06V8jRBKnpv4SFjoBcfGpXgkTzzn5S+1i7e5IBnxw/bQ9bdqovlbWURrktDxP8bL+ntOcdNbIqIj6EtElN20ciYRUbJ8VL/DM0REHXt60iJLd+ZrhFDyTB+863g45ymc9PWtasZOvExEROOxjoiItr3oSdkADIq7OSUcwQAQOybjNQAIlZ9I2fdPcYeR4F7tyHf9IiC8fosvpdmUbpRCeEB+LMoiBwAQhpsAgJUf3CMnjzWxTbFKaOrbr7vh/KG/Sy3vbB2oYxCLbADIWyX9u6ld+D0yKHmo7XhkP5sPAMVXMvmZ6eSufABAzq6t1nRhT9Hly7h1xg4x3ZlPJ3+65JhvN5+ya7XcJ2KBNCitVI1xYf0ltdjbY1Dakg0Ac15ENoCi9d38dnRvnS0K5KD8J3ic9GDT+3v6/INwvHn8i+zZH1IOFIx5wwbMWBRT8p2m/J51TcpPmTZxVYexVj7dmb+l+Wb70inFgPXVTdErHzmp0XKp3stzAeRFW8TG5ras+sDHQL/K9ReC5vW4dTrxZy8PXXCZbACnYuogG8CCIRY/Dcn5ZxftmbjH76svopN4IjMzff3wGquIiKhTi2VEJ/EjESX1JCLq2pOIvq91lcj+XH9KSyIie2tuDxHVLX+I6EaTbnYhnZJ6Em0J3k+0MmIT0fyqx4lGP1TsLOmMzGm0HXOJ6F+X6Hmc5xvLKTuUiOz1LxF9WimX6IfS17yb6On+OCIabz2OQURX3/JwVtaNoodOEX02MAATPXB82bIVGQPTHpO2zvQCaoQeBRAlvyL/+T5lAMuIpT9eOngQsAzm9gCI6NwQiHxzzQohHVEAjWzfBIiqVRkoff0mkHzwlEYHWtdeACCrnLKxmGf+cwtIH1QOuRN6lgLaBy318iMdd82OzW1C45ANzH4BALCVr/VxmjdVDx0tAxzv70W1EPdT/Ch+q14QEBx2U3jFwYz6AFDfsua96o0fbP/YUG7PI84XJWNtT2V6xrHOANofBNDzCcuZw7uQq7UkGDr2cMPDDVSNPTLsg28H46sRwK+5YTsAlD3h7Uxvzym16T2UQTbSysUBAE6U4ydMr9Y05Yof6Np5RrAX1Ty9zCItTMQ59Tdpb1DY8fCtb0Z82v5pO9sjvyg6+qQq/XdUkOf/ea0Wx+msr54OmY+NHVSNIaHTZ7DllgPSUS48PDx8yThvvynIXjQICInJxpdDpF3Pduc+FXOreFO18n8T/tXnMfKimj+ufSXgBgDYCu8/eX3K3sszv9nO9rDl2/U6qsRaOO98OHbM4kmJZQC7xiW+8t0W54YGqxoDRuw5tOYxAHUR16xZs2bNKno9KL9ffgBAXPaGDiEAYF37G3s2ddUtrybrnB47L4xZ/7sX1fwxKE3idwHAL+icugwoM6rnUbYHAKwAsBVdVImV6h8AgMKVuP7JoATgMrDxZ+U6BMCQa890VzcGdKv0+bYkAI3u2wQAhau8PlH54DkAiLu6RfpCTk1ok+N88stzuX28KTroAcJ9U4JLe1HN3aDkij/zVhsAshUDsN0EAJsNiPz829NA8fTe3TDnFgBrS24PgK0XgVuTn+ghpMN2E5Yvdq0D8FllhJYsAnAkND8zFjYb12DqcQI6V7xeHUAu8oTGEPLsV9UtAMK/WHMYwKwqXn9T+pYGgLgQ6cfrt6Y3bjmvK2QVPXrzujdFT0y1AKuejfemmsuF3aGeD0SVaveac/Pnx0qXSdm9sF1U9V4bU2Jju17emxIbl7KfaGO7dz9PmWyllZ3fWLr29VnE9hBR037j5y9NnlTIp1/bmxIb2/Uy7WryxrdTlxPR8irT1037dWSr13c5SkpXJzvFRjX7P6Lpy4ne7xgX/eCTaULpjKgr0gs3t5qwaMxaL6990dKGRURE1F9+p6P+wT3dd5I3S+J1b65aMWGi1YNqhi9IGo0LqTYiyrXa0w7n83uIqOkgSj9q00vMPF4sreqPHigke44HjRERsYvGGWl2by9IUsZRx5XJPMeOgrgd2cXOZ+0VfvTqPKXgyP48j6r5fVBcRNNBd/52sIdvc21N+zg7ZYy8QkSUFnbTt6vERqt5cPLoc9iKcK9F7WofdbQgY8U6ANjycMRtrhYS6Pe3cVZa+uPv1b23BuX+dQgHHv7tOwD4Mfl2Vws4OFGMYBQFBd2T4MTc7vcNG1S25bGKfgEn3FaTSwb8mxJ8O76OgYlb990H648Zmyve7momYnQXIkYhAJIs+POPiuMvAGX9G0n4S3xLYALeZpiDApO6NwMmdW9S9+aS+G5aErsNkfP2jOp2HSMHt/Au0Z+dwL1F3Tsia9n2bRWGRlpTjyZNLo/MJV896qfjcXU+lINyevBz/Qxk+rETclx+6fMYtmX/IiP81qh4fxT2uJSxC9Ai562kuu2zvPwBnYeyVsUuBfitLu3c4xFa7v5tZr06rDkyuWb6TSXaX/sPPwDehkp5cele5LyVVHfWMS8/Qv/rf2WzYpcC/FaXdu4J9e/XJPrFOf357VXrxgFN2r7kh9Iel/LwPEXFeQMAVhrKXZKh+g0q+xSWuQa/V3rZmMdRolqwsL20dRiApNW3fC/tcSkPB4Vx3mBI9pEJrtBr9hH/Qrnn216PxK0slB7bLmYj54YMfjtAcXVptgc+k90uonhTDABULNxyB0p5OCgy580h2f8eX/DjwIFfg6O6NWPQt8pbkNvahPbK3QAAWNuw4hef/l/VrQ7w2wGKq0uzPdJQ+UR2u1rbXIsCgBicuBOljE1XIufdtScJSHad4UQkUt2ap1ovi9unRxFtheM/d9lrP7Ejs8IyCfyWQXGN0o49Uifckt2e3KOfyU30JzCKiOgg3vJ5ojdWyqt79CLnDQ0kW6C6tWJS5gThW7SsL5BYebVEFluiT7WukNlHAsedoLi70r6S3fpRgDDp0nz+HShl+KagyHlDA8nWoLo/FtdOkZ98/3ki21xd4ThQI2N1X2mzAXumhQMUd1EagO9kt4vFmAQ8FyH6DpQK8eGMaGGXdq01OW9512PiqWH62t6N2daxOlUADNm+rK8THmUr8K0fr/901lMLg/RLSwVRLhzAkkp+H5TS0ue6ADF3oJT3gzJ23oEE7ADsFgsALBiiQXXXqsVnZLy8gRsTfDO6AQB6b31OaeVdvJMlp0y5tmjMkCRol17gYOPrIq5ZYFbIMRUuAcBFtLgDpby+n8Ij2eE24DREqlsjXviYHxNKbQAAln5W9ZmHExSHRmlpjxS+kt06F38KAEvXMwBwOrap75cYPS5lcFBEzttmg4BkP/Q7QBCpbnUcCU0STqmuSf92wwI7AFjzHMVvAk5QHBqlpT1SJ3wlu8UoRCEAXKmSCGD0sdMArRgd7Htdz0sZWdiJnLeD6paR7CI6Ve2tKVtEqlsjXl3JbWxvVqrkQ+lENLphZFTrF2lDl+j4zguJJPCbgeLq0tIeJ1rujuw2vCQu7N+hQnS97jOJbtYfSES0LCn16qsDrf4QNzBSii/pw42G4jRbvTDKiwFgO1Jdmqb/yKmtO0udrWT4clVeyZAT1lrh0Cwt75Hiwo3aloDcT7m87marxha/3E8xUErZU/MmlyktZYZJs5iDYoY5KOagmHE7ByXpz/82kwLzNgNU0lwSm0tiM+DLVeLbiMk5Wb+wyhF3xTGxk3iVyh50lwzK7cTkHKwfZf103zsP3b5DryTk7IvO59ufvR/YPr5LfCQA9A8CgC1zvjVM1Ik8H9u+PutcZoOXy4t1nmraPOLcro7Jhi9I3jZMjmf97J+GfeU50WcEBVS/TRUhZ3/xMNEf7XYRzXUcmrZERHQzoasuUQeXPB/bzup7inKfjNkl1ikPIHSS3bi4Qe/+2KB88/tu8JuXhitzLvFIg0dxEgOkB+8Fbfd4UNQdMTIo38UUEtGwJ+Udqz8mIjrUnuilz9dv2759W9szRET0tjQoqterB6XgbBHPXnDbz50houuxNYqEOh0+Hf/FGQ/AiQBjcmouzxGjYkd6vKBZ6R9Cbk8GACScBoKGdW6TmHh6SHUA2F2jrFGiTsnzse01f7sGRCWdSRPqlBv57tDqHpw8BhiTU3N5jghvf2Q7n8uJtzpUWWVRVme3WLOqJHhCyFX6+BM7sLoL8AIAnN45AAAK/tvPD3BelUIrgChcNlZHZ1ACjMmpuTy5+9jM5crirU5VVibK6uyW3Kw6yTNCbnCNV9r+tmrxO0ACAPtL0y0A8M+XLH6A87ZnlgdwOKS+WOfQzA9n5hq+8xhwTE7B5clzCn2IwSxXbkxWZeUIQLlbjmZVSS7nFA1CLv1vCGvjXN0sfp2IiH5dQPRwV12iDq54PuX2XowW69RfaqcVtc8anVMCjskpuTz2GYGd5TobY6qsHAEod0vRoLKHMErIWRNeC9re7aL0ePxzAGD7erDf4LzCIe3fEess6WdBj6jRRs9TAoDJueby5LiAaiz3GUdjTJVVIAAbaDb4jKKHMEjIHRm+quyI4Ru77woCsNpSFQBmPx/kNzhvQvx34WKdhgDQbF5WvKFBCQQm55LLY3EeHViuszGmyioQgLGaDSp7CIOE3LMflEXCpqlvbegCYGENADhWGJcDFNlygqN9hvMW/LGuhNjubuvfAUQjta2hQQkEJueSy5PjyoamiSzX2RhTZRUIQK5bC4aokpI8I+TyUtsAsEz+8UQXoGhLewDIypgI4Fi5iTVH+wrnrTm0JAhHg+pxdVJyr4cBVsQZWn3dBkxO5PJYvFM0z8JynY3JqqyaoqxSs6okDwi5qwVASYskuFm1PoBjN2IBIGn27NmzZ5eqP3u0p0Td1QJxe9fOWUHA9yX4Oo3mhgE4HlPX0KAEHpNTcHnIlWRyr7+8YFUTPtfRmKzKyhOAcrekZlVJMEzIXamSCIT2+QQALlz9O4A/ONH44pt5xok6gedj22kDs58bMfyp+dX5Ok/WBXB268wQA0vi24HJCVweHexeHVGPD+jfqckrfwjiqawxpyorIwCd3WIdUSW5hvEYISfRdwWDhm05NG9wOhHRKkxwvmxEy6iYR97XIergkudj202kY91QaLdo0icHlz04w+47jOcfTM49l3fhRm0L3xgu5tQOEglAOeRmVUkub3KpCLmTv9xq2NwCANbFXcsbIeq8usnF6hzbHduqvAnjmXcezYBJs5iDYoY5KOagmGHCeDBhPBPGM5fEZsB37uuOkXH52exxmci77kApCb3bOih3howDcHrOzoNlBkSiMGPvuZmjtF9iDAf0EsajBUejgsdGChDdtXl5+bYX6kBF6GkGh9xBzJZBPwHBs//3mC3oxXgj9+gDTsbpxR50JyKi/MEva79A3y7dDzCetcuAYvp3wywBonshi+iDsE2kJPS0YTyG3JGYzUA/HsG73mGSlb5OMQTjBZyM0+ckHRTflX46Iy7igD4NigqumxqaR0StR/AQ3czwb4my8DdSEnqag8IhdyRmM9CPQ/DsTw4govvj1T11JQMyasbIQ5ZAknFLkiqrd9ooLK60nl26/34tHVDcM7ecU+e8htEAWn4+I6JENeeLKhCASFwFEDQMABZJhJ52rPkutQyiklamPajM3nMLkEA/oNxIx8s3L98P4BubZyeP7sk4GY3zjozTZPK2bwQ6cYmcdbqEAxZlXULRmXx9LT49E3W4gvFyMqIAIP7GQe5Vfa/3An5GN4iEHtwhd8psBvpx8VnMQwCatfLwjN4NGcfQOC/JOE0mbzuA7iyRs06XcMAdjctNXv/B3j7/IPBafNVajvvwo6Sy6S5M1F3CeCWDSToegt1yKGCd0WwiREIP7pA7ZTYD/SAjePRTjfNvvPf6CWMwnmEyjkPjPCLj9Ji8I6gyoHdLrCbioT/eOl2yW/9b8kKHgTojAYsaWYn2h00TFPs8gvEaPUhENBIzBIhu99jmw+WJzEHouZQB2YvRbEPO5kA/J4KXg9YfFtPpSv/zbKKnD/A05cS8RETFsZ9ujTlARF+QvV4nItrU6DjR8jL7iNbgBBE1bSSlSIOiStKzbO893ipO9MXpHVcTEXGJkiXhd1ju5Pk7xRcTFYXOoluVJxIRHcL/KPtNotyE5CKhUy4G5TDGSqnjnHu+t2QSFbbC+yLZaP+t82NZjtu9VdLdDkpBg/b5/NLEkX3yqdfC0TGTiOgQEVGTJ+gcgtOJaHDNfI8mendknIDGGSTj3DF5QVVfIqA4VQn9OazTIfivcyRgWB/QiLzFwS5N1OEKxkv56Nk5tq+77iqrWFzUXlzp0d3BgEzowT1yp8zmQT8ngheJqlUBNFq4M9kjETY3ZJyAxhkk49wzeY0LgbS9DZWJknW66L/OkYBR9bDgmw0VtRT7YFCubnTPrYWvTEQj5Uvjmu/4oQvgJPRgALlTZnOgn4zgJYbFAkBJpHo0KO7IOAGNM0jGuWfy7gOwu54qUcs6XSABj744rhN22NopFPtgXK6uWjUgvTo3KNbEot1hQCxOAjKhBwPInSKbB/1kBC+kwU3po1XWo9WXGzJO06/cMzJOh8mjz+ryibrW6TwJeLPPQ9OAI6V0TdTdwHjY3iMXyN02ifuYFuw7fBnAeenb4yD0XISM3OFqAZ/Ng34MwUs5ZQVwCc2MDIpRMk4Qx/OKjFMweY6GkT/6Qhk+kVmnS3brsoE6TwK+9Mc3ocifX0Hg9WAcxsOaVeeBT2o8BQbRleqyuTJwLjWpDSASeprBkLsrVRL5bB70YwjeiMgfAPphaC33FyQPTT6E9T0iKTsr6beKANBl1Yh2dY62bZxe590HY7a3bYqWO57f2SitXnOEL3zl/Yb7Uy6PSyo/5uzFR/sOAvBm8hRLJ40kzVjEn4rtm3YQP/aIhPXcbznJAJeY9M+EiPndJlp+eXsfuv3r5PQDlu4T0haFLUyd1yNqaIe4tU3eAH76MnE+Xf7pVLlQuVNPuzyC9RY+M6fiOwljAUTUrAvg6aPXDiz9dUMoYP1H1pHoLrXajsLCV39pRa92mB8MADZ3gzLg9DwAaBjsqMiyZw8f3r/Mnp2fhQFDp+xtmzZt+mCg3MaR1urzEz728u6PHhmnhcYZIONggMnjEpvVX3jueh29yU+txafF6xmB8bI35DRupZyHDh6ixg2lnRqEnrsjx7IZ6McheDlbrzRveG/CeM3qLzTvPN5tcQ9ap+NPTrNs7Jq25vHjf61Buet/vvxjnf4ndK+7k3EPW6ebMJ4J48GE8UwYz1wSm4F7wRH1ToByeX+EhdhtwTWBq1fCLAV17o7D5S+RPNd1QvwEyvmVkwOAi19v/TlqYNOawPEB6U2TPwi8Mh5j4/inBK07h0gejCrjsYoMxmN1lOwePHSvcwvKec/J6cYv6CrdUu2/yO4r/WcAxmNsHHtKoXXnEMkzqozHVeRgPLmOkt1zr4znKSjnPSfnrkn71E2+038GYDzGxrGnFFp3b7seFKUyHqvIw3hyHSW7Z/AevQegnD84OU02zz6pcxv4XRdPA8ZjbBx7imF5ABPJg64ynrjNKvIwnlxHye55e/KoAOVcc3JaoJxLTk6LzSse272NKplrRinH5713rMzG6YnXOUXyDAej7QQYz1lHye55PSjgQTnXnBwD5Yxychpsnm1gRAulLS7fjEKOD957xzI2Tk8EzymSZzQ42k6A8Zx1VOyexxO9NijngpNjoJwbTs4Fm9ez4PkaJZ0AF5/MNVPHuzlFBeMxNk58Sp4lZJE8o8p4PG3HwXhiHYHd89imFi0Wf/PvjgB4dbyIzg2ByDfXrIDjVmn0kaeAGqFHecm83O6hyOudOF7D2Rau9fJujX51UcHTxWpbXNYM/OUdm4fdvYNQo8OwAm0RPCaSZ9yR3FmRV90T60hyeb6Yb+qBclqcHAfKPeyCk3PN5qV9UanGqJkfjVPb4srNwF/esYyNq6kpgsdE8owGR9txMJ5YR8HueeWIqg3KaXFyHCjnipNzzeY1qQS8vfbNLg1UonhyM/CXd2wpmY1roiWCx4nkGW2CVUxmMJ5YR83ueTEo2qCcFicngHL6nJx7Ni9iUetBu8OgssXlwynHBx+8YxkbpymCx4nkGb6CJVfkYDyhjsDu+eIdTJ9t3Pr7aygzavfRJBecHJrE7xoACZRzcHItrn/ynIOTi0qEYTav5Zj3354K6CXzcnweecce4GC8iHAg5W1rGHAJzYSn5CvqSQCwvP5sY+XFiiUt10sBQNX6Qp1dO2dZgO/7er0k1gHldDk5HpQzysmp2DwJ8Hsz4t31oi0u14wsxwfvvWMlGI+xcYIInkPbjhfJg0FlPLmioLon1+Hk8rxaEv+SUhURjw8Y0Ovh0kjmtOea9hs/f2nypEKHbN7Pj5Uuk7J7Ybuo6r2uMcm8zUh8Y8LQBFh5Z1u3enm/PFEnOuaR8UR5nUtEle58mEsWmnHq4nlsU6tUxqO9zZbvG9E/j3+KadsJInkGlfFYRV51T67Dy+X5blPLQDnXnJwalNPi5GBUL08nWSHHZ/wml0rpjrFxRm1l3R05VpGD8QJtU+sPTs688+jnO49/PU4OdzvN8lfk5O7677V/ODnz58uvMN5fkZMzYTwTxoMJ45kwnjmnmDAe/lwwnt+xuLxzQMU4ADheIsRG93uafy31Sum299RR9lBWL+QOYHEXvzu+NOpgAoB//e9A2y5jPAX5Lixe1N0fg6L0lOV2OB9xWJ4+Ogc9ZTxZa0+U1ZOb8UAZL7BYnHT7vRfaFBERFbazq0E+94WS+/gswqYi5zg9PPkRR+xponOulfGY1h4nq8e164kyXmCxOKng3JfxERERPa8B8rkv1Mn3QVGSc9wO9ogj9jTROdfKeExrj5PVY8X1lfE8mOjtE9u29xKLU9vSvlv7jWM6/rde83UehdJTltvBHnGespzTrF4oX+LQ2lt8ize+ZcU3Lx8N4JvvfTl55LE4mYsziMWpMbuIRbZBTnLF4X/rFNNTF2LqekyTz342370dro/BY3n66Bx0lPE4rT1NWT0vlfGgjcUxLs4oFqeB2bUct+896ZHD/9YppicXkhk+pq4na/Jh0/t7JObPpR2uj8FjeS7QOWgr43Fae1qyep4p42nOKTwWx3NxBrE4hS3tkblEBQ1C9ktzigTyMV9bqZDM8DH2T9bk69RimbNV13a4Lu48qj1luR3SIxWxp0TnXCvjCVp7nKyeVNyFMp7Bb4qAxfFcnEEsTsOWtsRXeNpx+zsKGr62MsMns3/MrRZnejladWOH61sosTxtdA7QoeumHrsIWA+giDO+1Wb1vKNZBCxO4OL0sDj3trQPTZ40eTp0fW1lrTuZ/eM0+eRWVWn+DCWxp43OQU8ZT9DaU8nqeauMB20sTuDi9LA4A7a0r3//YYq+r63M8Mnqepwmn9yqKs2foSD2dNA56Crj8Vp7Klm9Ut4p4+lhcS64OIbFGbClDfmq8aC/OzcUvrYLhsgMX2Un+8dp8kEnzb8hYnl66JyeMh54rT21rJ6XynhqLO7A2wCgIz7nCotTYHYkXV2t++7vF6D2tXUUcmrdySJ5TJOPhRs7XHhlKqspn8dk72BMGU/Q2tOQ1fNIGQ8aJB6PxYmKeIawOKUtbapjwhnVRva/5XxtHYWcDJ/M/jFNPtaqaztcQ+ScylSWPeKwPBfonI4ynqC1x8vqOYrrK+MZWBKrsDjGxW0ziMWJtrR728dEtVpERESnBjv9bzlfW6mQzPBxvrOSW63I/Lmyw3WxJOY4O6WpLPcUI/Y00TmxJHuJVPFol22/vpZ8iQTjW644o//8AOPpcXH6WJx7W1rB11YsJKjrOTX5YMgO1/ebXK6xPDclOa09TePbe1gZz7zzaAZMmsUMc1DMQTHDHBQTxoMJ48GE8cwlsRm4C2E8X3k5b4E5XZPc3CNXSiXjXlTG8yIlxBgvh9sEzOma5GYu+erR5MAq4zFTWeYkq6loZxjGYylcHeaGq9bmg5sLkiIvdzuAOTcmuR17+iDoZkAZj7OklZ1kNRXtDMN4LIWrwwg9tTafWxhP5OVuBzDnxiS3q38HRaWMx1nSyk6ymop2hmE8lsLVYYSeqgdGwAmelwsgMKcG9aQYFTsykKtCjrOD0pK23Mh3h1YHgD0ZgErRDkZhPJbC1WGEnqoHRk4eeV7OA2CO8XIMmHPFy2l61UJtkguhVPGVCwCsF7PFJMNgnlr+jrOklUNT0Q5GbWpZCqvDCD09AT43Z/SMl/MAmGO8nAzMueblNL1q1Sa5jhnQsflzUvwQYF7tinPBdcIDME9D/o5Z0jqdZLUV7WDYplZOYXUYoacnwOdyohd5OaPAHPOwZcCcG15OBPV0THIdcwrbbN6JiApDpvEOtfoNGbCp5UxlnU6yLhTtDNnUshRWRyb0tHpgBMbjeDmDwBzHy8nAnDteTgPUkz4usCtzuc3SABBWgu+EJ2Celvzdw9MXp/e7AmBJPwt6RI2GvqIdjMB4XAqrIxN62gJ8BhAjkZczAMxxvJwMzKl5OfegnsokV9UB9mmSO+EJmBetJX/ntKR1OsnG6yrawQiMx1K4OjKhp9kDQ9yXwMsZAOY4Xk4G5tS8nAFQDwqTXJ0OCJ3wBMwrrSV/5zCVlZ1k2+op2sEIjMelcDa1MqGn0wMDgyLwcgaAOWO8nAFQT2mSq12KpDWCoxOegHkq+TvOklZ2ktVTtDMG47GUoZxNrUzoBWsJ8LmZ6A/Pkf6dgU7yuVv+fX2JiAq+J2r0DNEEohv1WlqJ5uxb+SERUZ/Z9vqJREQFK4g6dSIiipzGZ2lG9y2aE/2osP1Ci4oOpHQkokxMI9YJFw2p3+aQJkREH8YWEVF2PlGuJfg8ETXGT9R+ARFR+xibNTqXiIgGOP8HW3nXE/3O8XYievt3qSJL4etsezyHKKf0ArEHBid6BS9nEJhjvBwD5tzwctpetSqTXNhswmajqwBWlMoH64RHYJ5SGY+zpJWdZLUV7WAUxpNT+DqM0BME+Ix8UxS8nAfAnMTLkQDMueTlBFDvYPfqiHp8QP9OTV5xXBNy5O5NiY1L2c91ICtxyrrpaytFdbjGlPf0GzKgjJf19Ec/73i4ayZR0aRPDi57cIZdR9HOMIzHUrg6HKHHeuArjOcemPOQl3ML6ily5c3ss7VKHY6Oj7QYaMiQMh4zlWVOsq4V7YwfOVaHI/S0SD8TxjPvPJoBk2YxB8UMc1DMQTEDt59mSbL8+d+nxfEXgLIICIxnhvnzZYY5KPdg/D9Ogv5kLNvT2AAAAABJRU5ErkJggg==',
    '1109.4653v2.12.png': 'iVBORw0KGgoAAAANSUhEUgAAAbMAAACiCAAAAADue3zRAAAcz0lEQVR42u2dd2AU1fbHv5tGSCEQCL0aCQ+EAAkgSPISYuga1EhHQVFBbAiP3gRUeKIC7/2wUBQUEX0+UZoiShEQpEgJAYRHaIEACYQQSNlN9vz+mNmZO213ZncTCM75J5mbe865d+7uzp2dz/nGQjCtgpmPeQrMNTOt7I2IEsyzUGEsgYgsBFjMa5r3rYxOrIXMz8aKaH5lETT/HPezem0LgNz0nKqJ7J/z0nKqJOkOpnBXM2nI28zr2ydIn085W/7hK9WSPLiegbxsp6f8HbWnzpnZr8WLl4nSRgT2F/4Sv4ro+MigVP3BWHfOX8WkIWv6Jw9+pgminh2U5Bup6WpsGEbNxYk9OAHjPQkL7484DQOIiK7F1s8ioiThpH+HZ4mIuhk6WaK7w1/FmJClVU8S0btYRERHI5y4drtza0aXsdHdsH5l89YPhC8AhI8b8I+VgL/QnrL/bwDboMeY3ry/8065/ZoC8IUPgAe6FVbWdPW/gxelbT6d76rrmWj34VfptifWw22THv+rbZiD6KuNvJPay7Y1tspdek99E1HcL/azhQBQmpPFXktP7S4EANzYvd16TtJScvUqCs7YHT15d8GfTm29AgA4v+W0XfH26cgcxJdcvoYbtxyusrgAYN2XXlq+62U/b8XWRNyla/Yf3wkAgM3//L3/M4Tj7SNeEf+4KeVg0bgpNuC9FWGV34plWzbG1Jo5e+raruOtrLvgv639FvuqmaWwjt0cuubhU7Kk97dlDq4/UmfJon833M65yuICoMWPF2TE7SrPJdvV89t5X51M8Oh7kDLYg5zCE1lZ534Y0WQtEVH3DquJTuEXIkpIJSLqnUpE3ze9TmR/cRCdSCAie2emhYia1zpMdCvmEbvEnfPf5nuAaE3QZlra8DjRmDaljpCszceHRERkj3piZ1bt1Y7UTFzqnUq0qF4e0aaqueW3B/mm7hminj43tH1tRU7Dltn77Pjq1d9mDjnxKADgTF+gif9RACFCh8KX+lcDLCNX/XLl0CHAMoxpARDUIxoInr7+W4k7QgDQqOQYIKRpfVTNvw0kHTrt9BIYerpz7az+jtRsXAB5k1OrAMk+q8rtXZb1zOzGQP2YMADYrpb3o7pz7swe5L7RzEELH8A34Lakw6HMlgDQ0rJ+TuO2DyQ/+hzT8rCjUxI2pCrcM4/1AJB8CEh9wnLmyG7kOR9KK2UTFxfAH3kBOwHUOFluaza1OBXAr48AAE7WVOkxcm78XfC9fiC3gZC0/cm1+gQcD9w+PWhR8tN2sUXoFBp6Sun+P9R2XMwXd1pZ3eWeOVzZxMUFcA41AwMDA7+YUG5r9kOXMCDrT24L8nwf9k+bSgAAZy91upN7fScWiVsAYCu+/1TlmTNzV4wbLraI+878ZkrPprjA/zZ+8cFI7ATsFouzT0eV/awjbnNUb1eu076cNQzAVp+4kuJg6+b7JdP7w9INALZ1CKL9t7rchc/PYiJ2A8A+9EhfDVQbnXpUbAEAKwBsRy+lZ72WBwGgeE3+gqGRwFXgJwP7Pmnc1nU3A0DxWpTbA8sWAH5oXXXDScyKjL/B/m3U+4UAsC0h/73cURnlvWZ5kmuM1QaAbKUAbLcBwGYDgj/+OgMondvvEXxQAMDakWkBsP0yUDDjiccl7rDdBixLdm8E8FF9/8olANL8C7PCYbPJhlCEYj77Te4nl5qJC5sNgUvWHwGwsEF5rVnEA9nA+p9bYFebP2NvFUi+igmbM+AoQFsbLXk5/sVG5fuY58is9Eyf2Nh53NFvc3dY4iefWLGvRvvn/28nOn169s2dls4zYrB5TtfqG9pO8f/u49gHwnbUeRVCC4B2UY0jg5Z2nhrAuC8+9eZOdPo0Ante6tn6RItU/Pf1l6IP9F56MKHPHC6kwzasvHy8IKh57ece3vSvHYGxA4ZiH+8qxMU+bhhbp3RpdjSxF8rr+dnesWOPRUSNfdHyLPB63ifAhMPiH/fUXNfsTNMOjz5dz0nYMro/02kX021ElGe1nzhSyLYQUexQOnfUpumZdbyUiKjk6MFist8wkFIlbuYJe3l+R1x0OJ/o0jkiKqq+81op+6eF421EnzxEn7SkG7by/o5Yp9WtCwBVgGbSFs4aOvGszW0dfR8AYAkzllUet175TrpSNIA6APBL2EOT2Fuxo7+tBrAtCc3y8O+JFY/hsZVUrLjuWFSjd7uxm9ql0wFgbw+0qLeimV95f3flqW3qFRDW51jFiWv8+RkRERUWSi8VRER0k4jseU7C3qUMTyl8UeLjU2HilivDY3JXJndlGsqFu0qwmOehDN5o6t+aeWgJMD8XYfL6pplrZpq5ZjDrYkyDWRdj3p+Z92d/zboYaY2LV2tJRg3r4J6j4VE4JhFQP+ivsGbZq3f8Wvu5YGv60YQZtZD1xWc9vbVm15dCtmYZw14cqMfT8Cj4SVD21rpvtblj5zp38c1C28siAWJfkhlYMDqCO7j66sdhAEDLjob4jg8GYF9xodD+/P1ufa8vrXFR1JLYF7r5zfdi1LBKW2SlKyqRHU2GK1r4SdgXBXxmfKTuTFF5YrNfziZ6J2CzEHXgLKIDUZeIKHvsC+2RRURk7TW4lL6Kziayv3KE6FKX3W7VMp3CYCIiWo3BarzulRFurlm/QfhRdm7233IR2dHUO9Uw2MxNgub47DA8UnemqDyx8wO/JsrGQ8IrNKyYiF54koiKzpbM59Zslv9NIuo8kmjd+0REh5M94ojlNS68rdHj+0Wm8gOuxlNYLa96CXYReY3u0apkBACMDh9leDe3xisfjbUJQDCuO45XdQ4AkLCuAKjUyJdvXBwdCqDjygL8ngkAkRke3VMLNS4AU4+SNllR5aJ2QVmiaPq678PV13BolE1SuuKoklGJLDbBVUmLSkYAQGBy2g7WVSjJUdbaOIYlZhXyiV76bUB+X2AXHuEPSzeHAUCd4m1MnxuZIQAQcesQ6r2/wA6s6+XRmjlqXACmHuWrSUW/DBnyOZiaFjUb+rXikf6v8f59834EgA3RbOmKo0pGJbLY5LqkRSUjZw2whXEVSnKUtTaOYQlZxXyiF4zVPlrfazfV8aLKDQGAMLDYeWVf4lblBIY1eT3xz7Ur33JrDyKtcemdSsTWo1Az7sOerWlRsRmvyRoyRhNtR39l6YpQJaMWuRl7PXNe0iLNKFzPaB6Gia5iMpVaG2FYfFYhn+hljC3YM779COGKfRKjiYgO4Q2+jCeLiKj1A0REo/Ae0bmHEBBvdbcuRlLjAkClHkVS06Ji07ImS9+CqwcAcfXX3VaUrjiqZFxGdlHSosjoeJHCLroKydRqbYRhyfPJh6jXHpy78tzAHDiY2QDuMZvkgjLr2GXAehAlgDXyHz47HrnsJq8vqXEB1OpRlDUt7x+TuAQv+P7jOOZ4Xe3jQJPMdQMASEpXOvBVMtqReVOUtLjIyNtFNBJdn3UkU6+1aaWeTz5E/d88Ra2s13MPt98I5YpGShDK9kh59/kPbJ/33l0DaSPW1hg54qc+u328U2NhX7K8V5fO6lUujpZHpTfM5zb0Y8svjzVrAGD4jtUD5KUrgdvf/2HRwqeW+2hGhljSAuCLevoyOuwCuoquQjKm1oaZW7h6PvkQDVj19js3cduKqtwbrAhSOnNM6vbi16eiNZ5/pwYiN89648debl3PejNHvVOJaGzo/4h2YF+pnfuwX0q/YiURkRUvqQe5EPOH5HjqESIie7OAXCKi2NfE4CcvEF2f77dNNTLXxHXcg2XOhi3NKFzPsoNi7aKrkOwCXuF7MnNzDIvLKuYTh2jgelbcvm0xEaVgAX8zWjuViOi/+J29nnGW0tiWF8LBzfHzvVPnKalHCbQBGZDWtCjt5fclr3lKbwUAloFWxb2Po0pGNTLXBD0lLbKMDnurZLFFdBWSOa214bKK+cQhGrCi/UeuAriA1gCuF8HS+wwAZIRL9p87Hs8D8n6d5lfZkg8AaNjSnb2+tMbFZgPYehS0+R9AkNa0KCzNX/qQblUu9/MRLLPLS1f4KhnVyFwTNwrnJS2yjHlcDVP+a8vWxrCujmSqtTaOYXFZmXzCEA1YlV5b6gPn0xPigZwGccCYYxkAfTuGu7wVc2U869deABY0eQr+/RcAwMXrfzf+mEda48LXkgj1KG/6IiPpGUtCgqSmRWn/iHuMOdr5+klbs+8bAmN/Pm1pHdNbUrqyS6iSUYnMNfGjgLOSFknGwzMOnw1JDqZr2Qnj6gCAw1UsyVHW2sQv4oYlDsSRj/Ey8PwsZ2x0JxpbY2ltoODBNp8DX334QZ23rnziD1ifyU673aBp4mikj58YsuqPL2sCxSMqDar2+28zG3qLSS09YWsRQDfDAMCW1pi7Vl+6EaW1pTlbT7fmzc3KfietTQM1IgtN/CbwVpTFzYwXb0VZ2GTA5RtRPrK5QZaVyyfxMvDM89BhahvNjPfqxtud2srGf+3HG207cW2n9hVEt7eYHLH5nNo0mNyVaeaamWtmmrlmprm9ZiaTWgaWUDYnNgHmXt/c65uGO8g3lh8veueA0cJr4u/Vgiv8mil5Ub3AqHFq9Y4Boxkf/Hao2uBgFGfuPT9/tHoXnZSsTnOAp5wx0KlIojKd7P89ZvN5JQI6n58peVHXwKjbvGj5A6MO+x19OM2HYa+pd9DWhneDSRXAU37gInQqkqhMp/yu06z0eYpuJlXJi7oGRt3mRcsfGBVfLPxYcwZqvCCkk/ZozUTwlDMGOhVJVLGT/cnBRHR/hF6GR4UXdQ2Mes6Lljkwqp7YZkX1qlra8N67zongKWcMdCqSqGKnLd+MAfDl93rvqRleVDcw6gVeVBsYVWqzawOjTnhR9cQ7fgK6i46snDsv8J59BSVnCrXRWzXFeJcmQqcMiSraR2FtALTrpHfNRF5UNzDqFV5UCxhVarNrA6POeFH1xDsA9BEcWTl3btI729ac8cM7e/s/Qywg26jjhHnvJtQ4Bw3FeJcmQqcMiSruM7Y2uTBlzsSTevWuWF5UPzDqNi/qEhhV02bXAkad86KyxGloMLhfR6wjYqfGyrlzAu8PJS3n1eKF+Za0thIdCJgtI3Sd611JrmcMdMqQqI5ON9B5Xill1PtZ5/WM5UXdA0bd5EXVgVFVbXYtYNQ5L6pI3GHll191AyRTY+XcOYH30LSnOLV4cb55ffxxs1/cJECXYryaidApQ6JCqI7Y088HTbq+UKTv/kzKi7oDjLrJi6oDoxra7OrA6LOyEbpM7NPwVQJK0+VTE+TcAVEtXpzvg/1BI2+u9IVexXiFMdCpSKKKA0XDhgBaL/8tSc+ayXhRd4BRN3lRdWBUnRfVAEblI9SRuG0xcGJvtMxRkHNnxebF+Ya0wLIvf6yjTujqMhY6FUhUwaoEhANAZaQn6bmeSXlR3cCo27yoK2BUlRfVAkad86KyxML9GS3dJTrGDiUiysNw4Waze3ciouDZkvmmBU4g2rFFOir91zMldJrS2MZ0iv0bEdEirNJzPXPCizoFRj3nRTWAUefa7DJg1DkvqpWYPmrOOGrLxDPzvd2/zWwgrQrcUYy/XgQWOhVIVKZLymkrgCtop2evL+NFdQOjbvOiroBRVV5UExh1xotqJEbhmIvVGEdWzv22RGyeme+rl770R+HS2lJC17nx4ClyGsRJoFOBRGU6jQzeBNCm55q6fswj40X/tUk3MOomL6oHGFXwom/+/C8NYNQpLypNvH/2ofNB3YJhPf/njaRfREeJnDs6fXqKUYuv6pjv1qS4BLq69bTVHyyh6+T5mQieckwqA52KJCrTad+oiY2X3vwo1FO+0Skw6j1eVAmMavOiMmDUKS/qLLHo2K7l8vP52hrOSvRWbVR6TqwInTIkqmg3tue0jza1bXVZu5bLzefUFczuJjl3mNyVDvup94n1jx3HXw81qcDmJTn3Mvxs9DMXSWa+uNvPivnZCJNJNQ0mkwqTSTX3+rjnNTf/WsjozUsBfnab733A9ZwAS1GzCrpmfyFkFLj8+fZdIUNi7wOODz4Xm/SOl6ckZVKZ46di2wed390tSSKmapBJveeRUU3bxyvY2AetsHsyKddMKntcC4D/NLtETNUok3rPI6OuctpnbfZsUq6ZVPa466JJS85IxVS1mVRDd4+j3xt12FKWyGhCfbWv/yhAGxn1+ONKPad9Wo94bwunVmqkfVxzFORiqlu+OQDgS5uH99T3IjKqmrN0fJ94uTOTRY7fetFEMVWjTCoqPjIqMqPOkVG1nLYhQR3kirBMFhl+6xU7PH/e/DwwYqpGmVSt61kFQkYZZlQVGXWSM7XopSaVTzrYHtFZzCLgt0b/B6uUSWWOW66y07dRZxkxVSdMqrE1ewdP042wV4moNHzR9rCDRLSE7C26E9Hm1sfpm2r7idbjJBFRbGsRiyKlk7qV9Jtkle0HSs91W0dEJDpyWNR3+EbEoiJKiUr8F1JB/alERIfxM12bTpQXmVRCklHpydlzVMavlo4l3CHjLGQpgzU7TEQU8wS3sfqzx6PZdB6+54ho2H2FHv5P8YqDjD4cIDCjCmTURc4TS+o1GT3/3QkKRVhHljKwaABotzg7QhBTNcikalvFQUYhMqMKZNRFzph6wJsbpvdqJZ+SI4v3bY/17wBCkZ4IgBNT7abNpBpas5wfY+PQHNU55O5U5Zkzc1eMG94UF/i/j198MBI7AbvFIvn/o8uGK5wcX3k3laBgma/9qOQP6wLY00LueDNf5YulSNwCAFvx/QCOvjKhO3baukhGpTNn0IrOQ/cEyKcksWXDvbdmKXn5AYAV1a1xJXsCgHCc6tXqNvcKqeHZvrEiIaMCM+oKGVXP2XHcwTflirDySXlg16WVE60/DABwPKw5I6ZqiEmFUii1AiKjAjPqAhlV5uQGPj3o7R+kirBCFhG/NWosk8oeP9kcwNnt8/0YMVUDTKryzqFCIqMQmVFVZFQr55z0Sz7tOryN/H5b/f06vtNKcO41j8nimJSh52dyJpU5Lp1ZPfHE7Gdft7Biqt5iUu8ZZFQ3H6vuLMNvPX/meWxPeKdaAFgx1XuISfUKMmo+p4aJjMLkrlBxkVGTSS0jZLQifzb6mcio+dlomsmkwmRSTSbV3OubhruQb7wX0NCb54E61QHgeCU/G91vOEBuek7VxIqzZvcCGnr5u+OrQg5FAvj054OJvcYZHvHFlSv6JJYzk2pfkhlYMDrCPR7krkVDDQCvaX0RX0JEVNzFrjJi14GS+ntbJ9UFk2ofOIvoQNQl93iQuxYNNQC8pn34Gt4lIqKX1EbsOlD3/t7WSXXBpH4XVkxELzzpCZNatmioOhvqAg11xoYq4r39w5SeLbRGvKYcLkMGmdRVnQMAJDxbEOT+PbUDDVVhQ1XQUDU21KmaqBobqoaG6mVDFfGCVtiGClAuL/rqAGWVgZS4LWA/W+haAxZeYlJLN4cBQJ3ibR58D8KjoSpsqAoaqiIn6kJNVIUNVUNDdbOhyngdJ+yfw//Ki746QFkhkDBYBW4LYPM/f+cm51QDFl5iUrNzQwAgDCfdrIsR0VB1NlSOhqrJibpAQ2VsqDYaqpcNlcX7kKiold8B/nrGjVgUc+UCCYNV4rbUvcNqR1bnGrBeYlJPYjQR0SG84eb/OhbVRNXlRGVqoqpyoq7UROUSpgVjxq4oepr/CGKdxTROTanFWukzPM0LYyMEKmKuwmCVCq3Amb58VhcasO5dzQda8HjIGAB4cO7KcwNzUIQAALCgEO6ycg40NNoZGypAm2pyoko1UedsqCYaqsmGuuRb28yYNmMutMVcBY61gwK3ZbIq3FAWTGooB1KWIBTu8408GhrohA0VoE01OVGlmqhzNlQTDdVkQ13zrRO/n5fiRMxVGKwSt2WyKtxQFkzqQ9wbrAhhHqwZh4bCMBuqiYbqYEN1oKEMG+o6nt9nbYeK/6NbAGXFQPxg6ytwW8ZkbigbJrVn7SsAcBkdPHt+Rh81h7tyoi7VRFXZUOdoqFM2VBqPuK/Ym7/9v4tCGyPmygdyDFaB27KBXWjAwjtMqqX3GQDICI/1hEnl0FBVNlSGhqrJibpSE5WxodpoqE42VBYvnb/WjY4XRV8ZMVc+kKB9Ksdt2azONWDhLSZ1zLEMgL4d4+vGXn9fSkMEPTZ4cN8HqyKJaE2PKas2TFxIRLEDJy1dlTStmPamhIf3vrrr0arVUvYs7xLSuG8u0U9d3v44ZYaVtiBuyuTnImGlbxrM3Tj7j1GdJpaophm7hs35RLPQsIcnEd3sUSmkao8jjLMkzelGb8zUUIuWxNubHBbSaQUREZ0eRsSPmGhLp8krxm0g4gMJgxUnuTtmytezviHp5AQ3Q3v94kFda4e26DOfiG63HCI5Lpm24NDqB96zE2U//e6unQ/2ziKi1Qnp18cOsSrDGn405z4b6hQNdcWGajhrs6GutVglYq7SQCq4rbpbmTKpVzfe7tTW4lUmtWKwoTCfU5tsKCoud2WyoRXvLVxR2FCTSYXJhppMqmkmkwqTSTXNZFJNQ3nwjfcWJaqp+pqXllMl6U6tgp18y2DN7hVKVFP1NeuLz3qW8ZrJIdP8heezWr1WC8COSb0iggFgkI/YSeRUYZwH8a6A6B2jRF2pvnZL9SagqTyxcsg0e8BpynsybDcRfcgvRiLbSeBU3ecbA7kntfTmM8lOXsbH9L/skuouXDAWQEALC4CU/X8zGMnfvVd7IHwBwDLq5rAmcV6Ip9vWblwOxCS++h/+ePqcxqjyaaNBp3xx/OOGwRbQ9E/ZTtGPZd6X3NgL92f2qYnJbkKimfKWt6OmHGO41uCyoUSVeXkbHT6qXDfMPGS6roA/Xv9QLhCScOYE4PNCj/i4uIzhjdlONUe9/VxjL9xTu6BEDUGiqpSog/Y0SIk6gUTVxVdVVF8hCVWacxGA9fI1qZP7NKoCMm1QbAUQgqvAywCQ8dtgZySqu2vmghI1CIkqKVGB9jRGiTqFRNXFV5Wqr/zlnT/clRAxHFgcVedDMIPwhEZVQKY7smoBOOLXEogEYH91rkXaSeBU3d6D6KJE9UOiqpQoQ3saoUSdQ6KyvBqqr7x+p3jYvjsRFfvNZiVXddGoGnsQdch0L8bwv62cKOvEcKruMKnwBiWqhETllKiS9tRFibqARFXy8i9X2OW+zGFVAAioxA7CIxpVFTItHp78Fv+5NelFWSeGU3Vfv9EwJeoSEpVRokraUxclqnBznVep+qoYgY9yEB7RqKqQ6eSI73jtr3WWhrJOLKfq9poZpkR1iKBKKFEl7amLElW46cgLyFRftUbADsIjGrWqCmS67NLGSvyvy5vIOsk4Vbe1bQ0KiOoQJJVQonLaUyclqoBEdeSFXPVVPRRx+xd+EB7RqGFKyHT94S98cNSnBYCSbcmyTgKn6vnzM/cFRGXQqQolytKeBihRF5ColviqVPVVEaoSAbhSDIiD8IhGlUCm14sA7P5toQ/wfSUAOHYrXNZJ4FQ9WDNdlKh+SFSNEmVpTwOUqHNIVEN8VaH6CptNctj6OoBvqxRCHIQHNCrAQqY5DeKAE0OuvThyxFNLGwPAJb62SOwkcKpu7/V1UqJ6IVEtSlSkPQ1Ros4gUWneQ30aI+SxwYO6x7zOf/HH++5NCa+ecoDhTbPjZm6cu6FeSNdckavVRaNq1p+JkOntlkOIYrgFiCYiorWYLOskcqoeM6luC4gahEQNUqLakKjrvDJf4fDa2aZVjoRGBFuM0ajazzzVIFPHVxQre9eSdxI51QqvkwrzObVpMLkr08w1M81cswpvvm8ASDTPg/ctsWxObCJ3fzbDPMEVxmbw92emmdcz08w1M01q/w8gvCTtNi2TQgAAAABJRU5ErkJggg==',
    '1109.4653v2.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAXYAAADQCAAAAAA2BxX+AAAbS0lEQVR42u2deWAUVbbGv84CAbJAICwCMYphDSCERQQnGAJBQAaIGnFDloegsoyOoohEDKOoM47oA2QbQCQg8ow4wCgRAQlGCMhOGJCEVcgC2YCQrc/7o7qq7u2q7nR6qS6gzh/0UnXvPXXorrpd95fvMxGM0D58jBIYZb9zgohijCpoGTFEZCLApJvzu76y8eBBGicZr4Sf6rvXv99XGPHEvbQ+0cvpXc8Bgu4WX525BtzT4Hb5xFt/rWnZ7EdevO9kSvvAT/Z5OZvzSzJ20Zm7hBd5UfmxD05sfTuUHEQE4sI8OWQPERGt8IsmbUMlm5xEvGd5uvBZnHLTQOb55LUAEamc2+cv+qwXAGDMEB18NgKiev1L+PzTtWAEuKnX/ON6m7fnv33P05Zvw1RdfCkn/P4zAOCXvu7rM1V3P5dSrg0V3+wXCAB0KqMMAFCVn4uq7CsAys9UAUBVXh5u5JjFS4K4X1HGzoqzAFCReawaACovX0HRNWdTTKy/XCh7H1iPxIRlLEWOchZV+bmoyikDgCMzdVf2beggPq27FMAPww/cfO2tSiC9W9Ok/3yYOXacedmi3b0XAlu6N5uTPOu7ga9XgN3vH6tC6v0tGqAlI29k99sNbO7SYumCz8J3Opli8BMbigGUBJksb4gjLXogvP3HwOhWUSvFsRQ5ylkIm/YmjiV89ebNbc88s9rLv1L5i1g3rGdfboy8SmSe/BQR0YOxK4lOYdpBohX1rxJRh2aHiK51H2Zm9jsRQ0TmvkQLWhYT/dCwkMjcdlT6pebrnLqkXkqmXVhERCty6SWc5zIqajKBiMxRucxY1jnKW8RN24io3QvevaQqD7Qn1jKvbrSaRUR0CD8SUXxYNVGV/wgiysB+IooeQ0T0LTYw++0MOUBES6koZCoRVYcuIKLork7PZC4lk7ltDyL6kISysxm9HnKdKOcjdiyrHNksLJvm66DsypNMBPKk54U4eCEKAKJMmwCgow/gW6cTgABcl/aKxWZmv14R3aKmb5uA34rrpKen/9LkJAB0dmmiO2HfYRwWu2Azmli8HvjiOW4sPkcuC2HTdV3egRyMA9LzefivMGfzqZMFwDJ/E/6tlvYKCjrF7Bewc3b9BXHPmc+iaUBAQMCaGQAQ6lKSz/ktw9aBlhdsRm3iP0dlcVOwY/E5cllYJ66nmwOJMzeX1xWeFvmhDa4BQGX5fbb7KCltx+x3qt6cOYWrXhvfAY17yB9Yl5JsNuzLZH9fywsuo0kjD2U/CnBjcWF7C5aP19OnvcH/5n8iJvYcuodlAEAmBqs2rwCAnRjC7HdsHdBoesLRrnelAUD5d65e8gGMLxz3Z/ENLqNhLRf/HAPYHsvGloBKIFtfyxyPzZ+9mgBgi6kdGixenw1Uz3tiGICKSgBUWQmgUviu7rwM3EgaNZLdb+ENABUPBCzddBjA/NYAKkqcTvBYFgGDW5RGAChGCfiM/P7niwgTwIxllSOXhbCpGsD9vwOkqwkkEe2I7r0qPWViChERbX34vcXDkyqIdj/asNHwX1c+HBg+snBGt6D7XyaKHv3mspTYt8vZ/VIHv5Wy+Y35RPRTn5mrXttM9P2QoLDBK52ayWTFhwb2+Ixo3gaiDwY1Dur02AkmIyK6EFhAJI+lkqOYhbQp4vFCOn33O3N2eHEmY2NhIWtfYau4YMuLP4raqt8fRo+oledK2/lx+5XU8ztZESlcvS5ea2vyxDIHk9F58Y6k7bHUtlQeiQiFd2/6Or+e0yNqpbG6pP3qUmWVsVCkOTmwdeiJTSOyjBpq/LWuhi+qfHyMk0ztD9LPhfa+cK29cZIxQtvwfQdAf92k019X2XjwIA0qDAYVZszbjYCGN349yWFl5xT0jqhVi9KzAID64X63+Uzm6lfToyP/sLzI69V52ldX3TbYrx8/eRDI/tNah1vkr3up86D13y4d8dAGsyP/r7XoGtDXHUhHOSwnyKqrSCX6FuNqsZZ6BKOJiNICxlXX3L+drvUTNqgwRzksJ8gqfwAYvu/TWjQJEL6QcaP+taHmnWvXtd5+LjnGYTlLVpminblY3IMdnupaL2W3w2HJaBVDVoncFRSQFsuDlV8UvkPVBZecSPQiuirorxyRDTv302kz07WIpUl4mridb0antufayt8bZbfJYTFolUxWidyVCqTF8GCnH0/a+V4agKyeYVNqn+ep9cPHM2MxiBdQ8WpaUOqAU1LXIpYmPkrb+WY7ev5kTplTLXNj3r2k2uOwWLRKRHwk7koF0pJ4sKygnUQ0D6lEFJNQi0vqKfT7ev3Cl+NWmrmxWMRrWXgW0Sv3V1u6FrE0CU9jtjPNdvjuJ0qtn8ZyY968pALo23Y5gPymwquylxIbAaZJKdsABB15FrjH/6i0b/HMhGAgzicFIeO+vgGcHdNUfssUdLpv80uJwNhufwIg/HFIYO0+GI179HzQXBRp4sZi82hYeh2IPXja0nXuwYOA6Xnpkd0uN6MX47oDgZGtmD69/Uc0pgmvH+6iwmENgBKt+q24TjqAJieBiR+ufx5fTGLfsvBgl39NgrO3iQMjgAVPD9jfkRuLySNhlCnncAaKLfv3iujWKe7RCdIjv11sduH4YABxB7Gd6dPbN35tclhKtIrhrtQgLWGh+ARCXMsz4eZifiwmD/OSPl82liddIpYmPvLbxWa/o7kif29/2u1wWLAmq1juSgXSEq7K98JF8rAhDsMW4/X6kgNtkA6YTSYAELG0uyyPMVbbhYjEedTEjWn6abfPYSnIKpa7sglptW6/BwDKXCk74UYnNcar9JMxbYA8YKswGxGxNPHRersQLaMOAEB5qpvoNZfLbp/DktEqC1nFclcqkJbAg5mWpB0FaCHKAFTW5pNfLJyTw32vZmP1NQX9VQ3416sCcMS/7FKo0LUFSxMf2e1yM9PSjC0APm/F5u+9CaR9DotFqySySuSulJAWw4Pt+tM/v371K4Q9tnd4aOjQPAcnkIcSOgQ1jEsiokX1Z+a8qqS/Ih4vpA2t521J/u3FPm9kDA8NHZonYmkSniZtr+KaZXR/a/27G/j8vUyFwQ6HBSVZJXNXdiCtiwUdzcebNKnn5DJH7lYa3lCd8ao+UdmxDpVYrtoilibjafx2OS4XtfVxil7zMhVmrC4Zq0sGsGGEUXaj7EYYZfd22fWDJ8XoKhsPHqQxgTQmkLiz8SQACIiwdXO88FhBw/7QTrTKfja3SdmvfLV7e6ux9VGw7a73u6g2ufjlqj9rVfaas8EtqwOpWL18noioLK5epo2bObGJ2olW1ZzNrRa28CT4AkDAO2VvwA5lpFHUnM1tNm9vjf/qKFF9ZePBsm/HIwAYtkfmeQDzmTJtE7VkIyUjg0+shNXN84SbJUB1wUUAFZevwCv0kQtl3/9G/D9Y9kfmeQCkfbBHYHy0CiEbORkZfJLQKeD3+CnfTT+87CfsjgkbDyxp22IRvEMfOXFJPY+eH330fmKzRWbi2B+J56H4XutENMjjl1QuGwY0EsEnBp06Vv9TouoZ7VKJqGc8EZX7JZPm9JGTolV0HuOJiM72i80mog2N9hFtwkkyd4wnorSuWZz6kwZlZ7KRkpGEsFgJq97tq4goE6lENDCeiKhBMqdapZ+y2/4VEr7x7sH7gmS2R+J54A31JyEbDjTqDHDo1OU9L/sCaMufPH/zBn3kyiU1dNDJTQz7I/E83lF/Ch10chMPGoUCHDp1AuEqR+QV+shZPAkAEIhchv25T+R5vBSByOVBJBPAoVNtkGt93aryEn3kyqf95g5TLMP+nBF5Hu8kenOHKVYFNGLQqVYd0wHgMgCgLgHILbenZ6WzshcLCmA3J597twvD/jQWeR5O/cnjwWbDg0YlAIdOmZbu+wHAYgBA16sAvgkug3foo1rf4c6aeSgnOK4eii42e20AgP/7y0td9g9ddiBmru+vLz3S9UTHhF/m7TI9NPPEqswmPZc01DYbOZkfP90VEP3kGABp7w9svLnbW/4Afpn65L37HxyWOgIoGDmw5+HOE4v7rG+4/a2H2x3tP0R3JTfZ/d3Dsj0Sz+OtZQ5V0IhBpy5e6ZDbOnUEAFw5Exl8OCisgUlj+ugOxZMuWMoOY3VJy6hElUEOaB5zH/d783nczqJVusym0s9EVf63Qslvr3O7QQ4YYeBJMPAk4yRjfOFxu6snORmyw1/xkYLgWOdu7TjdEpos6p2bNaCO/0XLi9wwxM4651VzNzIvaT42s3DPtEWroylrUv0Eux2cfihFfUONLb2+qOe0i53rRnU1O/wNSnBWPmlQwu2hngRNjOqsHP78nZZP8r9N1JOghVFdLR3+bhX5JB/nXexEUkgEhTxiVKdw+LMipqyEkkRlJgmkkvfUf9ltqyfh7gdmfPT3mCZnZVJIBIU8Y1Rn7fAHcMSUlVCSqMwkglTMntA7nmRPPamqawXR/jrJLCkkKSS57phWo8MfDU1gh1YIJQnKTBJIxeomDdX5JRV21JOK/+yPkif6vclKEkkKSZ75XaFcsZWHVgglIRCQhZE43aRb4H67TRe7OomgSSVf+gIJV6JzNu4SSKHOnsuPd/gTQh5a9O8TH4W4cLwTgLiD7bkkb4VlDlvqSYEdsXzt6hbgJYs8aKXDO/wJIQ+tEEoCwIJUVrpKui97s2FfFqurJx2dMiMe6dvx+mtfvt2vEWAmzjFvuZvzS2y2uRySw58Q8tCnSufszfvn2l3iI3hhJC5JvZfdrnrS9cT7k4EjwWqSRJ4wquMd/gBwaknWQknghZHUdZN0Wna76klT/1jrj7JlzVlSSHTM84hRHefwB1RWcmpJVkJJgjKTJIzE4UyVlXpeSz0xPbOi/ZiX8cF9Cfhw2/6K8A5z28kI0PbYfjGUt/10hb9ECj20QASFsmPHmmJi3H6/feerfi+2Obej/2ggc266qW9SjgQp/XtxdKeQXS2mfmt5zJybjj4rwmABqRic6be56aa+Sd1vtbVUpXqSCinkslGdyQGHP3ZopVASeGEkW7pJxhK2sbpkhIEnGWU3wii7UXYjDDwJBp5kTCCNCSRuD/UkaC2ipCWepJGbnJ/TekVaiSjRstmPvHjfyU/bB36SeGnNF4/YLV7285NHq26osSVEN7ldP7eYUOf6keJpo1w6E9jMxGn1JBFDcr+Ikh7wpFq5ydmuju1MnFVPEjEkTYAfzfGkWrnJ2a6OfZ85Z9STtPw7bM/jSWsuwHk3OdvVsZ+JA+pJkviQ5QmLIckiSmoAk6oRG3SGJ+Uvte8mx/vaZV8BUH6mis+DOUKxOopMaqeeJIkPiU9YDEkWURIBJhWHN85RDbrDk8asr7LjJmfta5c5dpx52aLdvReyeTBHKFbHOpPaqidJ4kOyCpGIITEiSjLApOLwxhqx6RBPSppmx01Oxddu2kGiFfWvclpOzBGK1eEzcQRP6vLXv76xbu+auBzGTk3NAy3ncYu/GgMwqTi8KQ3hdIUnvX1pZqUtNzk1X7uzXYH2N3I40ziVI+QzcVhPRtArksSH1FSIJBEl1v5NzeHNebWliEwGT2okPMoaSgofNwBgdJ6sbN/E+Jgnwht8snFxP3U3OTVfu04AAnCd7139CFnFKUdlfEIHfbNp9Fk0DQCwpuUP4hNurmURUWLt39rEf/686PAmNnBebWnw1wye9IHlOr505ZCHLXjSx/9ZMP/ZleKjjzWeJO3JxaO9uJdnNz/RTbFPQsri+eq+dtLBsL2rHyGrOFUr9SRJfMhKhWj5eHZPDmBScXhzPhJnbi6va40nSRpKv1v7uAHg8CQ1WzcgMpL7SE77vpstNzk7vnY2e2eqE2lLcaom9SRJfIhRIVJiSJz9m02HN+gTT3r542423eS6qPvaqZrKqUBaLW0pTtWkniSJDzEqRCKGJIsocfZvKg5vLqgteRxPOuIfY9tNrkzF164SQCWquTyYIxSrw2VS4wTy+Ih7EDzq6aeH3h//I3HWbdITwdWNM0qT7d+UDm/8jrX+Sz3aEd17VXrKxBQi2js8tPHw/bJZm7WPm2gcJ/q2yXsKLdXGfDXVrpuc0tcufGThjG5B978s9/4ze4RCdawzccLcTRIfEp+oYkgMwGTH4U1/eNKZlv723eTUfO3saDlZV0ehOGXgScbqEgxywAij7EbZjTDKbuBJMPAkA08yJpBGwF14kpbeblrRQreAuZuW3m6a0UK4BczdPObtpnYrzOO00C1j7qapDpEmtJBh7mYrPEoL6c/cTTZRY0kfEUtSIjsiL+TusKaFVGAh52kh3Zm7SSZqHOkjYklKZEfkhdwd1rSQcmSXaCHdmbuJ2kgMi8N6u1khOzIn5D6/VBu0kDUs5CwtpB/1JAZPkrSRWNJHwpIUyI7MCbkvbNBC1rCQK7SQjk4y4Rv3Di4FRG0kVoiIZXF4ZIeXMXJPBEZEdF3QdsBxcDpOCljIZoZgaCFRVkn35m6iNhIrRMSyODyyw8sYuTESbi6GFS1kBQvZzBA100L6M3cTtZFsszjclY/jhNwYnqWFdGfuVgOLYx08J+TesnuQFtKduZtoomaDxbFGdmROCG4zd7NBC1mP7BIt5PUJpBWe9P2QoLDBK4nUWRwFsiNyQu6aQNqkhZSwkLO0kFcmkI4vLDgkRKTkhNy2zOEpWgiGepKxumSEQQ4YZTfCKLtRdiMMPAkGnmRMIFUOUu2XR+bW3MinGn8zysXu87MK7nrAk/lrSuzA03cgZ+a+2/xYUuuvR7mIoOR8sWKK62UvnX/uUudpzQCYl14IuDE9DN4hdjy9qLcxupqIqodEO2ocZhtB6TLN5UW9/CdPU/FjIRlE5tHvEu1v+webVS2JHddt0Dy3qLc61geAT5LDxmG2ERQ3ADWz378XwSt8n6rGd1tmAN37T2Wz8neKmdHlBDJbwC16RgCOKfZ4FEHZ9GAhEBiTcwIpfesAiPn3Ded1hFKh37K33fAfADBNgUyZqMMyYBAUTqrHjaxM6/IKAIHIq04LAYAW5Tt4OzNZSAg8CKPUEJKZmYsAKi5f4Y7My2X/C4YM+CCjAjGQKBMlLMOo9WT1DJvCS/VwrMzs5uGD8hc/GN7+Y+CZVp1W1jK/XZeaATjsF5VfGAgAITjJ2ZnJQkJCiCCMioaQ2G53TNh4YEnbFotkDMj5BN3GyaxrCKDZZ0QyZaKEZWS1HguCIqMpMisTPY3oYMefiKi41Vgiqu6c68RfYRPtxSt0EtOJiA7iHcbOjCV2hL/YFkEYVQ0hsV3PeCIq90tmjszBBD3IySTmrH0hMnfKP9jTtwKWkdR6LAgKg6ZwrMyxJXseBhD8woZS4PTkps58MsrHx/0NN1FHWE/nzikMsQMWhLGvIdQQAOrUZY/MpQTdc0+m4ZOf/3d72Jwi9j0FLDOxeD3wxXNQavqwrMyPccmCsNr4sjXAmqedynFm2LcBCBLwiyoEQUVIqKigoKCgWAZhfiuuk56e/osNDSEf5ZG5lKA7yv4VAJj6Ly49wL6rgGXaxH+OyuKmKpo+DCuz7Wj96cLWFiMXUlllsDMpLv9jSwOgofAxv4kQqAgJDQsLCwtLkEGYs2gaEBAQsGZGzRpCoS4n6JZfqRsEU8J46du8fLwqLGNR64E9ViZ+ep++Q54EAEyO3X3aKbvDTYfW+OCoT4fmuQBwGb3UdIS+LgcQIIMwtjWEpHZk4YRFqMbpBN3zaT8uCJqVoqvCOIxDUSxqPbDHygSid9IkYSrZv/2CI12cSDDjl/k+wMa6pqE5AJAdGq2mI9QiIiIiorkMwqhrCInt6hKA3HJ+IGcTdFPZq58qAoBFY1uCUexRwDKiWo8FQWHRFImVqagEXgtKKAQA06T1zphNnXjmyuRJLzy7LAKvHM8G6JtXfBk7M2t9IAmEUdcQEtt1vQrgm+AySEfmfILumkB2Tpsw9/ufpo8qIkmxRwnLVMlqPXuHh4YO/ZZBU0RW5vuhwa1H5v07JLD9QiKiwoibTtyTsVSiCxGtizl29dVnKqSsVPWBJBBGTUPI0o7y+83ZMm9zy8CBhdKROZig5ziZ/dF0ZF91dDeTGmXCoSiSWg8cYWWuLpzl4v32vC3X+3Qz1WRnJoEwKhpCUrsrZyKDDweFNTDVMsFbi5MpPt0dHz4Rod9lDkcTvLU4mQ96lJTejIB+Q+sEtfnr5vj9GWmz9bzqoHWCGn2t8y5G+et6LdXBBA0GEgYDaYTBycDgZIyTjHGSga45GYfwJIY98jglJLuM3T51Vy6jvTnuQtWhl+ZF272xsGe8jwjBHHkhINFzejJkXtJ8bGbhnmmLVkfTbRFQPVBH8SSGPYpN1MxlTMfYkTZ4kj800fWxchnTOXakIZ4Er7uMpeLWL7sansSZiKkq4ciUkAgwsRCQDCzV1nhM4TLGyfjw2BHfOyfxU5WXhxs5ZquDUaGuNMKVHMKTGK8tXglndvPwQfkAQwlJABMLAUnAUu2Nx6xdxngZHw47suqdlfjZ0r3ZnORZ3w18vQJMHyoSRZrhSg7hSYx6jgQAyewRSwkxWjoyBCQBSzUZj9XoMqaU8WGwI0XvssQPdWh2iOha92Fmtg8ViSINcCXYmLIVrn0hEvi7XPYbrWYRER3Cj2TuGE9EaV2zKHoaHX2xVICzwqqJqvznE9GGRvuINuEkEQ2MJyJqkEy0M+QAES2lopCpRFQduoBrYrfsPbGWecUkwvYgZKnsfQQRZWA/EVH0GIEJ38D1Ed2ViE9afIuSg0qITi7UUD1JgScx6jmsEo7EHjEEEAMwyV2LwJJ9aEg1IsC4jKnJ+Mg9KHuXJX4sEYvNfB9KiSJtcCWH8CRGPYdRwpHZIxtaOlKIwFItoCExBoNxGVOT8ZF7UPauGCMo6BTfh1KiSBtcySE8iVHPYZRwGPYINrV0qAoysOSE8RjvMmZHxme5I72XlLbj+1CRKNIEV3IIT2JMxBglHIY9ghrAJENAIrDkhPEY7zLWXV3GR8jSfu8VALATQ1T6UBcA8iiu5BCexJiIyUo4DHskE0AswMRAQBZgyRnjMc5lrIFSxkfGjpS9SxI/AHZeBm4kjRrJ96GkrjTClRzBkzgTMQEAYtgjjgBiACYZApLFfWowHqvJZUxFxofFjqx7lyV+KHr0m8tSYt8uZ/tQoa60wJVqhScx6jl2lXBYgEmEgEoZYMmu8ZgjLmNqMj5SljZ77xG18lxpOz8bfagKAHkKV7qTlrB7RK3UC650J60uVVZBT7jSnVH2rUNPbBqRVYsG8QMz5ky/YxaNPZVNNXxR5eOjD1zJwJMMcuDOCt93APTXTTr9dZWNBw+SiJKMT5+WkWT5uWQEDIk2o+xGeCr+H6fZJuezU5QzAAAAAElFTkSuQmCC',
    '1109.4653v2.6.png': 'iVBORw0KGgoAAAANSUhEUgAAAUEAAAG2CAAAAADE8mtMAAAzSklEQVR42u2deXxM1/vHP5NEBCERYqktqlSJNUEtbUItLa1W+UpVF20VXbSquqCqqi2lim8tLW1/dq360lItUoQgitq3WhJBEQnZRPZ5fn/M3HvOvXNncufeO1OJe/7wOmfOmedOHjN37tz3+XweC8FsupqPmQIzg/92I6IoMwsaWxQRWQiw3DbnQo++Gg8EtpD5Kdbf/FxN5mzcnx428G5aFWMmSsvnhr798JFX7zm9omngrP3mp9jl61UOTa+t2NweABa93MrMoJbz4Oz5X7UHADzf2/yoavlfT7079Kw9vVveM9+DGt6DK272Eea6BAIAnUnIBYCi1BQUJV4HkH++CABQdO0abiVZhU+/fR0yErYXJANAwb7jxQBQePU6Mm7yMxpbSqn4TbIF9wnd8gsBbOp7MO+d8YXY2abGxN+n7XvhReu383d1mAfgt7Y1J03+YF2PdwvArcOMxUEVPo0AaEG/W4lddgEbWtZeOPer+tvFGa0tvnYOgG+eMzgVGWPuCw4ODg4e7v5vEpBCa4NV/PCXxjeIrK88TUSdui0iOoM3DxH9X8UbRET31TxMdLPto1Zu3akoIrJ2JppbJ5NoU3A6kbXJkzuv1PpBnHFsTl+NpI1qQkTUZpgwPppOaloJgc80jVkyr0bP2D8uq48KV6+5HVZyo1t1PyAiOow/iHqFFhMVlXuCiBLwFxFRxPNERD9jNbdue9BBIlpIGUFvEFFxyFwiimhFRCTMaM5g62FEdMOy1D7MqPa6ARm8Wf8LIppXpcCdqCAip5/iMFwT++k4dCkcAMItvwJo5gP4+jcHEIAc9oxu2MCtax/WJnzUlqE4kOm/c+fO3dVPA0ALABBmtLb0w9EA4qmrfTzr+a/P6P8Mf1Z+NIBGWenuRnWawYdxUOxPxd8IAAAf/5OArWv/t5g9o3LlM9y6gO0fVpzb/TlrMmoEBAQELH8PAEIAQJjRfBqkKADb7q1jG16kKQ3e13+DZcEQC4DT5ULcjeo0gzE1N+QLp1g/NMJNACjMv8d5qKzse7l1Z7In7b02c2X8fagWGRkZGVkbACwAIMxo/WPjGtwFYFs0Mm1vllH+U9bs0pvB5LRuABDb3f4zV31UpxmsNCd1lr373XNoG5oAAPvwsOLiAgDYjt7cuuM/AFVH9T/W6q5YAMhfJy4WZrT+sdvDAaQc7oAZAHCoYTAGdBij91qPUAdA4uZptqE7UZ2fYmf7L7ESEW2YQURrQs8RFT02kIi6diMiq/+HRLQXfxARRdx1hSinzZNWbt3ahjlE1G8/bQg8TESfHyCiFi8QEZvR9k2SbrmfqPgVHLjwXyKyvpJPRDukFw4avkmKm28myuvxlW2kNipKuD+4/W2/VxtdiIseBACxU3pU29BmfLndU+MtD4w7tXhfSMT3UzefbdTlKyCySVijit92/sCfW/fzNxHNg+JrvwFsG9/13mPRvbHpv/EBEU89Dzaj6TfJr4+1e77mnuETml+fWBXYmNMfAPodPeGv7zfJ0fdfSI0f2s02UBvVQi7fg0R0Ysns/2UKg3+OFzpZFvE8JR8rlK7LLLCeOpJre+TSKSu3mp/R8B58+17ryaNFRGfziKhohC3yKd9Zeq8Hi05eFrtqo8Koe9SR4Yu897s4sv08Nvg+v6Ot80HCuWDDfherjlrye1BlazmYjGkqXk2GD3dyKqzH/poJOt+DWqIa9B7cPPuPCtFT7vPSe/D33tdCS8kdVtWtGL4o8vHxUgYv7x6AspZB8/6g2XQ1348ARN82Lyfak68m2jMhzT0LMPcsmOdBlME9CzlJQOUGwuj8TaBhJYMOmJiU1iFMVwQr+Tqd8nE57cUM3vgxIZ7O32UbXGuf2q3TMKMyuGf5b2vDgMQhrwxSfVN6QVZu4ev3indYx/YOrQQAT/vIpuLmrZJOu/yPWHgp4NYodnVuXXwx1/ryPQDwbES7ihcSenbjF8leRMl3t5Ji8Jm9O+9ZnHH6u8Y6291fbjewloh+xotqf9Wlvp5KNM0/VhjPF74H5VM5jfpIp139qrMO+pjorybi3QTryCNEl7smEBHVBFBugpVfJH8RJZOmK5Pb32O7QWGd9houOs1HynB3M5iNtURk3X9TbQZnBqwiSkUnYfzGN7/viI/fEZ0kn/qkUR/ptKsM/hyUT0TDBgjj9V8SER3uTkTUY+7YhUnSRfIXwUI637s1dNiOKADY3dkVHF+r9VtMPS+uRQAq4YZ4W30YACx+KUw2tadhddm0q7aisz+AqBdvVbSN/7wFAI0SAaDGqw6L5C9CzS7gmIrfAQB2d4RsN0JRagqKknIB4Og48Ua/fWOC5Oa/7TFxrwKQ/4/9kqI47YraDD6V/R9gFx4Vxq8DQOLuwbKpvP8Nkk+7+jEfGwQAtfPj7A/U+XKWFVjf28ki+YtQk8EqA1dnAsiqbLE/IN21sDfmBcKPY/O2PPPMUm5jwvz76zf9EhhUN3yR8Ji4VwHn/jNx+2exAHCyXehI1W/CckDBjMgPhGEjANY3plpkU/99w+Iw7bylpgcCQBBO2x8Y0vCt6L/XLfsUAHB45vSZmdJFshdR8jfJlckUj/lE9H8pZDsPOu5a2EJE99rOg2xjQkb1oURkDU9hj9n3KtDJytuJaCrWEhFF9Vd/f3DPu+2GS0+by96XTx34jqhDH/m00/PgaYwiIjqEj4RHkjvB/wEbbw9fYaU1Tc5LFzm+iJK+ScjaJJKIppEtgwq7FmaLGeQ3JrwblEOUNJ1/zLZXge5/kIgoyZbBPm5kkKx/P/xYKjfOr5csmyp4q5hlkJt2lsEjeNf257wn7vp4dkwAel4hIjpMRNT2Sdki+Ysoac8CYBm6/wiOtLCPFHYtsP0K/MaEYZmrgCXPSR5rAQBX93Qtcd+xs5fSZNnmR7jT7HpLfdnUnNd8FKedtcq2vQJFqCyQpue+nH7soc2PWwG0BIDINanSRfIXoUJP8pzft9jcwz5Q2LXAovEbExr1+hqFmTUkj4UAwCkEab8Qr9Zu/yY2WtRQNnUiv1pGRkZRYUa2w7RyC0YuAOSJr+nlqdXRKPajvRuBPTtsOT4uXyR9ESp2otd8dNnkcr7i+dvproXvXroP1SLF4Yh+hxMfA/jHLABwN7/JRn0r6FK0xx8IAdvGUhTXXTZV6dIHAE7U+ODu0WzaVQuqlQIAV9HevuHi+AMALBO3nO6NvpnZ/kABqrFFCi+ipPcgEYCX0l98XHjAya6FgEIgEZKNCY/W+WZHFOCwWaFe0z8B2P5b3Wh5+49cA3ARrQDcyAOAEzdDZFNRc+bMmTOnSvic0dy0y/NCnyQASAyJsEWtYMkGANQPB1rN9wdwMug+tkjyItRl8PhJAh6unR0GIBNZQKVvViUCxVMHPgoUFAKgwmIArc8ChICFvx4BMLseAPi9vCTMAv6xgiwAsCyIPQbQPFsOC9W+Iav03loXuHA86gEgrV4XALiMQIcpAMU5WeCnXbbRJxIBWjPa1xa1XMwsAPjnxoPAgPsAnN8+048tkh2p5KuZk71CAiO/Ipq6mujzntUqNx9wimhz18++6TuxgHY9Fly1755FXQPD/pNO5xp8NCmOiLZ2HLf4nQ12wB6YRsQe29i7cujDi4iI4h+c+dPbPyJ0AO3tGxLS55rK38XPfbFrZ4c+V4goJ/wZIqJ1GOcwRTTi/sCghz7np13Szh+ijt94+5kCIWre88PiDi8YkkxERRNmHfqh+Qwrv0hyJO2083JGE4fzZuHRMNtn5p+bTYSr2IsCcOUesz+Q1sx6onr1Cu7dYT10mNq05AIVLOtT08mUbNrVHdZrv+V0bMM99cy+Wy3b2cYn9oR0rClbpHQkk9WZ96hh6ovNDJrNzKCZwdsig7cPcY/y5KuJ8kxI82rGvJrB7ewSAKc4HgACwpw9Of14WnB0WciOGnqvIYPXf9y1re4LFZG25a4pLRVX/LNs8eMGZpABcAcCz01Z/3ei0GdkKABce+Obkm9Fyok7fXcs0PfdSpIDcPSem1atJ3Ehg8QQIqLc7hX2OVnRLcbAfdQiAHck8Gwqu8eEAlralyj17WHtcKXEOwty4l7Qe3Ax/dgyVXIARu+5abXaTlftIl4iIqKdeMjJil5GZlAE4I4EXpyyDhhMRPeEEuWdL5qpIoNy4v5xuSwi6jxCcgBG77lp1cRdRauHv71xNhIBOBwIvDi1dfVfAFYWAuUbqAoqJ+4LWlYGcP83MypyB2D0nps28op6Gx4BgAtbzzGLgG2ChN96PtczCVWE318HtQYQ2VFtEDlxz7gUCAChNw/xBxDpPT+t/5tEbH+932sGUDC2Wcu1GxY0BoC4MS/fsyLrA18AsfvvXl7te4shOTu8tcjvxSBFAm+fom0NLy4IzHyxidqQDKbbdilU8CXbG+pUJ+4ANnq/1CKdNiSDR75A0aG4j4ZbgKWrNzVtd2XgXz7A9u572yJ7cOfuQNKNsYho/Gw3QxJ4YpRlbfvN7NP555ptbVdX4qeyrjf9abJP0gOLH1IZMxuVAMDXLrFF+fDrAHAeGdIDACtbVJVP6/8utn+TJHfplki0uup+ol9xmsjarBcRxbY6KWHy+r9JBACuQOCFqQvwTSaiIXfnEhGp+CZxIO6/WK4Q5XfE5zLEb6f30mmVxL3EVv+XvQ9no//1iKRf4pEJXDrRHED3Q03hyOT1NAGAKxB4YaoS6tcH0Cpxt8qYDsS97xcvX0yc0gfVZYjfTu+l04Z9k4T0PP0rrAs6LqvWGQDOopY45eAhoL2JABwOBF6cquIfAgAVpIvgBnHH6Dnbt72VIuJMga4L9F42bcw3CRCIFLy74GAj7ASslsa46InvXhGAw4HAi1N+LXJs/2PVVQaVE3cADRoAyWGtpIif0Xth2tD3YF6cpVv2rOcbAdeAzbvqhB8EgPy1xmZQBOAAbuRJCDyb6nuuAEAKItXeVZERd8T3ywQyd0zwkyJ+gd6zaUMymGkzVsh75cLHLctVKAJwtFzulRDLwoTfAHxdV8LkdTcGwJFWr4sEs7OpEZU2AbRpaGMAyEc+3CTu+HXdRWBWw2dlHF+g92zagO/iE080RJUnBw/u07rXH0S0ut7U3yYfeLXj+0VECW3Hr/p4tZTJ6/4u5gB4TvgzEvjNTe2NXL1/xNNZRPlP96hVudnjM90k7sd67zgwpluKnOML9J6bNl5fXHyqsJk/ZdlOyVczmvgYfoeVAXDIMTubytie1q6lO3dYZcT9+saMNh0t8gOI9J6fNom7eY8aJqszM2g2M4NmBk3iDpO4m1cz5tUMvEzcPSh7V9WsPq7Ht38GPSh7V9Pi5q1SGvPKc4GzO2fjajXufARG751xfPV3FtTK3nXo4J2+Gpt83WHM2Djj7Mps3B2NO4vAoipzfDeJu1rZuw4dvNNX84ksg/YxY+OMsyuzcXc07iwCi6rM8d3mJEPP7gAA7O7s1idQ/x1Xu3zdYczYePkGwh4hOxtfdgtqiPt6YdmflwC7xp1FYFFZT9cVtXPZO9dE/i5I25kOXmsT5OsOYwX47oKNq9S4q4ygKYNOZe9ocP9707+Iqp6MgrdjK6996AyYDb+og9feBPm649hRec7YODRq3FVG0EaaXlq0cgSw5kn7cN2YP6si+rUhy4urxpfDgfETGnD8vU/vpvtGN5p3NSamafTXut6CB6vXczrm2bgCOoda4o6AuEHxLTtsKac6grbfxZ2bfAcgtYZtlPtaTFXAMmLFlszHyyFrYJexQHB2DtDt0DkAlsrnOte6or/EU+HSIc7HHaYuSx6Uxk9/fOIqUHAQRa5PDPC3yXbFk1BBozE+8Y9eVR1BWwadyd79Y0Ajspb5guPvsEvbdTeJfN1h7KA8d87GoVLjri6CxnszTmTvgc3w3cqltQGOv8MubdfbpPJ1x7GD8twpG4dKjbu6CBqJu1PZ+7GR7/XCzsKujL9b7NJ22HXwWjOYysvXpWMnynNnbBwqNe7qImjIoE32/vOLM8Bk74Nhk73nxLSeDBxtnz3rFTt/D+wCqQ5e8x2kKABYHT4HAG5UDODGeft9rtWVK8/jv1wUhMwdM/xKIO4HAZG4VwyoYMmuAtg07uoiaPgUu5K9v3F5ZTnkfluL4++CtF3Qwetqdvm6XeMujKXKcztnd8HGVWrcJREYvXfG8dX+qnMpe9+KLuPHDW2EAo6/M2m7oIPX/qtOkK/bNe7imLFxxtmV2bg7GncWgUVV5vhGEHcH2buEv8t18MbfYVVQniuycbc07s4imH7U5j1qmKzOzKDZzAyaGTSJu0nczasZ82oG3uPFJWvZ8S9K2p0Sd42u8prN6P30aNnhEUl7Sa7yAETizti4CMw1usrzT2PwnZfXayHuJWrZBZauUdKuzVWeI+6MjTNgrtFVnnsaB99FDb1G4l6ill1g6b08mUElQ3c7cWdsnAFzja7y3NM4+C5q6J0Tdz9dWva13jjjKRi6C8SdKc+ZKbxGV3nuaZzBvKihd66c93FDyy4axwsdnqWLknYFCC8+gfOjhw5Nu0DcOTbOgLlGV3nuaUoG8/p+k9i07KJxvNjhWXrs53/GvEBgEF7BXp73o3enOWB1gbhzbJwBc42u8tzTeIN5u788NFXfJbqIdtOnT4mpOd/KG8czB3nBU556tf/BbjPPvOcV7OUlfvSk2VWeGci3ak5E9CpmSEzhtbnKc09jsQR/eSe6b7XfJMlduiUyk3jeQV7MoCBp573nFezlS9K+q3KV5wzkOeU5ZwqvzVWeexqLxcvrdXyT1P+lwcP7D2T67wRQ/bTY4ZcIknYG4R/CsGmrhmDJCEieoE37bmmyrM4je3wdiHvfL16eV7i0T0J1HB2+rvqI4ZsfT/CBNld57mlcLJuGfkFqqM57MyE9T/8qmsRLSttDJmnnveeV7OU1a98ZVpcQd5GN88Bck6s89zQWS1Fer4kXByKlo2ASL3GQh4ylS7znFezlod9VXkrg7WxcAsy1uMpzcnYulkxer/09mBdn6SaaxPNu8Q4sXeI978xeHrpc5XkDeVF5zpvCa3KV557GxeLl9ZoyyGvZRZN43kFeYOmipJ33nlewl9eifVdwlRcJvMjGeWCuyVWeexoXi5PXayHuMi07M45nDvI2li6RtAsQXsFevmTtu0pXeZG4MzbOAXNtrvLc01gspqE3iLiLJvFiR4mlcxDeub28Lld5RbrOgLlGV3nuaSyWXF5vEnfzHrXJ6swMms3MoJlBk7jDJO7m1Yx5NQPvK7RvbyN5Bt//DeG73+1pJO+SuMtYOZO/23oaNe6cLT2LwMXiXOs17eX3mJG8+8RdxsqZ/N3W06hxZ7b0XAQuFpvWqHH3mJG8+8RdxsqZ/N3W06hxZ7b0XATW46Z1egF7yUgeLoi7VJ3O5O/2nkaN+9bVowGs/EUSgfW4aZ1X1Db4ziqzc47yInUvSk1BUeJ1APnnbQrTjITtBckwxkVeysqZ/N3e06pxZ7b0LAIXy7lrvY8WI3mxMjtTtAMidbdx9X0vvGj9dv6uDvMAzFgcVOHTCM1vQilxl7JyJne39zRq3Glbw4vjp7x/WhKB9bhp7cpEzkhekK/zjvLMSL7L0c5nro1DROMqL7RC4WuDqv69Pg5ot0X751giZJeo05ncXehp1LhztvQsAuu5cK1X/x5sOWbM+z/sXd49iZOvc4p2IOk/QMNyxwBUPvos0LBcciug6a0kpBw6BFiGaM+gRMjOq9OZ3J31tGncs7BnoA8a9hiWx0cQe/y0vk+xzUgegnxdomjnYLqt2xxAAHLQPqxN+KgtQ/X8euKE7DwrZ/Cd9bRp3HlbehZB7LlwrffRYiQvytclinYOpgdIHgjY/mHFud2fs+r5QmbEnWPlDL7zGF6Txl1iS88iCD0XrvV+WozkhcrskCjanbYzFSZNSl/8zktRhhB3jpUz+B7BY3gtGnepLT2LYO+5cK3302Akz07HSop2x3b87BhUHbXnWJRG4s4J2W9UDODU6ZzcnfW0adzR95MCf7stPYvAety0xk8xD98hyNelinYG0+3dQgCFKAbm3QJQcL+2z68DcedYOV+9Xehp07jztvQsAutJXOs1/C6WwXdRvs4U7RxM3yB06/dLf69N5davr314/IoN78/W/LtYTtwZK+ertws9jRp3ZkvPReBisWljNe5KinaHllXB73RB4wDjiLuclcMQjTuzpWcRuFhKrvUmcTfvUZuszsyg2cwMmhk0iTtM4n7HX824+vW4b3NK46eriZ6hWlvqybS77kdZKDgON3nxuJSPax2fWO8ntRlMHPLKIKXHk5b838j7DVVoM3TLplhPGy/mFdpKlNgBL5f8u/iXiGIiKu4dodbW/Ge86GSm5ZuGKrQZuuU80cWeRl7MKbSVKLEDXlbBiwe8Q0REf0aotTW37r/pZCbiTSN5MYdu2RTraeTFnEJbiRLLl6vR1SXaLnTahQHqpNiWCHiFF7OC49wU6zmvFw6VCm0WgR1KvlzNp/gpn9+IiCiOiI6E2N6DBVfSKD2biCh5y9lieW3D1MtEVHjtKhUm3iIiovTdcfnndb8HqYCIYjFG/HAEFStMCb10RBERTcUul+/BoqqDiIi2YoP9gbNEROeGWYmPIB7KYbmaPQtvofdDnycUIIpJsR05MafDPtkudKRUhC3BxB/Wqt8z9ZtO9Zt+CTxTt/kiaOXFEnTLpoSeVk90ptBWosQOy1XdYf0hGEDNr4iYkNja5MmdV2r9QN/WP0k0unUxr8MmiupPvAj7VBQRWTvb34OHmm0losy6LxBRcYsU7QrtDHSeXkyJdf6QTQk9TrWtUaEtRmCHUlxe8s6j9JXDGwNfcBmkiFZExFceF3XYRNSnv0StvT3oIBEttGXw2KvZREQ0uXIW0el5OhTa0oLj3JS9p1wv3B2FthiBHUppuYqdR8FPff33ttBJEvjvwImHZa4CljwHR7U2j4n/6D7ZJvt7KXc5sHywDl4sQ7dsyt7T6IkOptBWoMRKy0u8s/AjAFiiv8k+yD/qwIkFHTYc1NocJt5yrOIo22TtfvMot7AKtPNiB3TLpmw9bZ7o4ITdjpRYaXmJv0lW22oS9BK9/797SZET23XYcIWJe43q2Ln3UwCAV7rtOhejhxdz6JZNSRZp4cWSguOOlFhhecnvwRPn7Ft0WjlIsbnK46IO27Ed/wGoOqr/MQCB6DBxhG37W3TTuUdbQodCmys4zqa4Rc7rhatUaHMRxENJlqvNYPHTGQAw/4U64KTYjpzYrsMGUJgDiQhbxMQFhcA7lfunA4BlxKq20MWLGbplU9wijbyYE3YrUWIZpFZ1NdMidugnG7eOejKDSbEVODHTYe/tGxLS5xrHjf/Pjok39qlSr9+19UGBTecREaWH5enkxQzdsinW08qLmUJbkRJLILU6XvxXBB3dXxzRRlGKLeHEF+vBDUx8Y94Henkxh27ZlNjTyouZQluREitBau/z4sxzbTFtYJjJizW3zyOzsvPCcGfW7TSk9forIfbDMkaavMxJrv0TXq6McRKTNJn7ZmDyYpMXm59isxl+NaOGtHMY3TvibFXi69tEoa2KtHMY3dOW6ACkVcizZ1+40uLNmo61wW2LlCzUVSi0ecwuTrFDOef4Dj+51ZJ2DqMbZYmurgp56lPnKHNAUIJDbXDbIiULdRXEncPsbIodSpnjK3MStaSdQ5i9PJ5Bvgr5K0lElB3SsEjuVG5bpGShroK4c6CeTbFDKXN8ZU6SmOwmafdCk1Qh/7VTOhAYlXRKVhvcvkjBQh0qFNqcsJtNsUM51307ZrDJ6t8BwDJSYnrOOZ3TmW0pjmcVwRJd1GwXp/0DoODqdYlE2+6O7p45urQKeb38AgCBuKa8SKEyuQqFNifG5qbEQ7nQffuoIe1cuXEgrt1W64pJxQyjA5wlusjid0WFvgQsaFJ7Phh7F9zR3TRHl1Yhj79SE8ARv3Anixwrk6sg7hyo56bEQ7ni+Ao3bx1JO3M6pzjfv4jWVozlMDpnic6xeGrXi4jy/SYTY+/MHV3ZHF35PMg80Fnbi9Ek8dnmFsks1NURdwbq5VN7MdoZx3fCi2OSVg5vnDJyBnuElRsHvdq9LRDYuC4AHF/wZ1eZOJvTbAcDgH95QJRoZ47rXwXo7rNCFHEfg/tVyAEg/6Xunzpd5FiZXEUVcibLlk3ZDuVc9+2jhrQzp3NcOtEcQPdDTcFhdE6czbF4Flpg7wcy/Xfu3Lm7+mm3zNHlVcgBYFzozwHOFzlUJkfJxJ2BetmU7VDOOb6PGtLOOZ2fRS3hUYbROXG2RLMtNIG98+7oqs3RHaqOA/ju8m+VXC6SVyZXQdxFUC+dEg7llOP7qSHtnNN5Y1wUbzgzjM6aXLNNRWDsXZM7urwKOYBfDy/3wTGfZkqLXleuTI6SibsA6iVT7FBOOb7DubvZWSIiuopLRNTqRaJxdDP0NSKiXVhP1vAuRER5ayhiItHHQeeJiHr1IiKqNJkoy/d1Ivof9v0eT317EtEVTCZaO52IKGZO7l1PERHl/cI/RdUVNdW0fUlczyWi3WOtRPTJWYeKDTX7EGVafC8SURtsc31F/VJbIqLpIUX2qDueyCDKCP5OOiUeipsu8ZtEgbRzTueWhQm/Afi6Lo/RGWTnWXyrGwDWVMmFyN45d3Q3zdH5KuSnnrn+yojhz34bJq8NXpyTJa9MDtXEnQP1bIodygXHd/jPUSDtxDudJ7Qdv+rj1RxGlzidcyw+tcuk36ZuqBPYI51JtO3u6M7M0VVVIbdveWhJMqdy+yIO0LtD3HlQL06xQylzfGXi7oS0c07nVzOa+KjSbF8/37jKkcqhlbI59u7SHd2YO6xOLNQtrok7D+oV6LoixzdJk3mP2tR2mhk0m5lBM4P/fvP9CED0bfNyoj35aqI9E5KIJppvJI1tov2K2mzmedDMYKluRBRlZkFji9Lv/Wb+LjY/xfDITvTc66xftZLW0JlH06p0u0MzmDhv96Gqgysh/9LeCzNHObVBKaFdWb7kkW53xjeJwl3hP/E4ERHlDnnTlQ1KCa1nf6N2Hhm1rckDIZ2cByva35wBX1wD+u7/L7Qa6t/hVYYKyb9asOdsUO6AK+r4zUAvoDjtCgCgOCkL+T/vQkk7sCQVN6R7uIRn6qu3kVJ6MgjgcdhsUICvHo19fdi0+o8XK+zA4lxTpBU3pHu4hGfqq7cRXzsHwDfPGZyKjDH3BQcHBwcPN+Sb5CjqDR54P9bbRlH9iein8hlU1ORtOqe4A4u5pnC7t2zmH2wPl/hMZqSi5ZtkVBMiojbDxBebbsQ3yZmmMUvm1egZ+8dl9VFduKUcRX8qTu5pz2Cf/kQ0oAURxdQlygh6g4iKQ+ZyziicawrnpGLLYA9xe4L4TGakoiWDrYcR0Q3LUsGAptrrBmTwZv0viGhelQJ3opbgluJT/w0AxUfsw+AbAPyC4GQHluiaIqm4IT1RiM/UVW8j/XA0gHjqah/Pev7rM/o/w5+VHw2gUVa6u1FdngfbNAdO7bUPXk07huxt7wLKO7BE1xTF3Vu2Jj5TV72NeIoCsO3eOrbhRZrS4H39l8ULhlgAnC4X4m5Ul1czdwHYI+yQqvH67BpX5z4BONmBJbimKFfcoCJwz9RVbyOuwV0AtkUjMwgAZk3wnzJwV2edGUxO6wYAsd3tCVEftaQ7C/T1ffbetgcXfvrdEwBa3RULAPnrpCvtrikSJxUAQHkCkJLPP5MzUnG/bQ8HkHK4A2YAwKGGwRjQYYze+y6EOgASN0+zDd2I6uOymAZyR/9T1W6D0uCdBSvWxmYCTnZg2V1T+N1bhYUAuD1c7Jk66m1kHLwOWCeh9cVQALRgGGCZvme1zgw2aH4KyB8xPdz+mXYjqtKX1L6+9VHxicGD/9MhGN3sNiiU2br+3fUC/AbedLYDy+6aIu7eSugbUq3vX8Tt4RKf6azehprv4vVoN+ent0/HfPzmDSL6fTURET3RKF/nd/GR3j/NGyRsjFcb1Z2qk3QrPIGIivfe9xER0aVTVsclF+xuh8cO5pM1g5tI259JhxOzreIzMwusp47karuz8Pa91pNHi4jO5hFR0Qjb6zjlO0vv9WDRSVEhpjqqW/VJ1izYCACY9/v6f/cedWT7eWzwfb69iuEHCeeCDbtHrTqqhdTXjqXEOueJiIr6ziHPNRWvJsNnFRsUcl5BEwy7u6U+qns1cvZNb9a+4pktD75i+Vffg7/3vhZaar0+UpNy6zb0+ZdJ0+XdA0y3FJPVmQ3m3i2D926ZexZg7lkwz4O4k/du/fPzvwp+rMWeP4SH/QdrnRy8tYPn/wqHAit2oxm+MsuzEe0qXkjo2U1VlRd5RG4snbK52LDYmiqfumwFdR/2+K86uTsMM5rhKrNQTQDlJliduMO49pvhxtIpu9WNGFvzvRkX7X3fq57OoNwdhhnNcJVZqMfcsQuTZOYxav1muLF0ym51I8b2RAaPYranMziwNxHR8vI5XOkcWwZHERHRoqVERIOFuboRRERvB+Y4z6A8IjeWTCUst/lfDNbA6lS38FbLPHwWlLvDcO11AEjcLamV4MIdxllEbiyZklrdeGwf9eB9f3s2gy4KrLDKLAAOz5w+MxNqqrzII3JjyRSzurHH9kwGB1mWezaD2agEAL7IVJxe2aKq7Y88MeqdsPbJKB9+HQDOI0N1RG7MTx2sLtwrFGJ7JoOW4GW26/0rn3omgw7uMJJWMPYVW2f5IAv6BY525Q7jLCI35rqci40Y2xMZvDR0bFICgITx7y00OHUZaWlpaZlwUWAFrDILWgJA5JpUFVVe5BG5MdflXGzE2B7I4MV+s4ZXWA6g46dPGv3mezQ0NDS0P1wUWAGrzLJnhy0Xx1VUeZFH5Masy7nYcLEN/01yoef8ezHgx5n+nvj4/pQPIAAuCqxwlVn6Zmb7AwWohpKrvMgjcmPW5axu+NgGZzC56/iuwJClmx7zRAZrC/VYDooFVm5UlJWbECuztBrkD+Bk0H2I/3JREDJ3zPBzUeHlIFfhpWIAN2Zd3ygAWB0+h49t9K+6pAbvEREV1/8PEdHaBh66oj4ecI7I2vETotSKEbaHpuK8HY/jDVvn691ElFTue6J3fY4SfdSmwMVvEhbRFpIbc10iKgp8QBLb6N8kDz5pqyr4sV+iJzPILGBs5Vh4oxmxMkvRhFmHfmg+w+rEHaaECi9cCRe+movdxYbF1k47lduZ+uVt1/T77w0Gfh513lN3WJUKrMgrs+DEnpCONQFldxhLCRVeuLHSwcTYHmV1a0ZdMGto62gnFu/OHNH47TvsHrXJSUxOYmrczQyaGTQrvJgVXsxvEpO4p8D0WUCpI+48H//fiUKfkaHga7tYF1/Mtb58j27izhWTYYcpE8SdG2f3mFBAS/tKartYRx4hutw1QS9x54rJsMOUDeLOxtYBg4nonlBJbZf1XxIRHe6ul7izCi/cYcoGcWfjP/AXEe3bTUQry68iuoWmRB+MJiK6eTfpJO71at8gon44xh+mTBB3bvx1UGsAkR3B13ap8+UsK7C+N3QSd1ZMhjtMmSDubEzbGl4cP+X909LaLkMavhX997pln0IncRcrvEgOUxaIOxtnXa/00+Sxw7ttAQD8+V6XBnFBQEBcp/iWX6ytAp3E3ccfwL7Db4RKD+Mx4k4bxr+11govEHc2zsKegT5o2GNYHsDVdiloNMYn/tGr0EncAXuFF+lhPEbcvwyYPG7mYPICcWfjSqhfH0CrxN18bZejz305/dhDmx+36iTugL3Ci+wwHiLuhV9X8Qmd/sMReJ64s3EV/xAAqCCi8Grt9m/Cy1Oro1HsR3s3QhdxB4QKLw6HMTKDF3pMu7fKgB8L4NfZB6iOq0Zm8KekpKSkZS74uF+LHJvTQ3UUtG9bACAEZ7KOPwDAMvGB09BF3AH8enh5eRw7wR3G8AwmR7/TFRhyfRMsiyKA32t0MjKDtcPCwsJqwdInCRCIex74cd9zBQBSEIm8/UeuAbiIVhUstmJD9cNdEPckjrjn8WN+KmH3bB/gl/LcYTxL3OlMw71eIe5snBK8jsjaYSgR9dlORMn+UUU0dBIR0aVH8nUS95N3Dxs+fNgzYUX8YTxK3JO7HvEOceeg+N7I1ftHPJ1FxNV2yXt+WNzhBUOS9RJ3rpgMO4wniXvyuBm1Tua39gpxZ+OM7WntWkJW2+XMvlst21mMJO7cYTxG3M+PfDMgb8V7995Z96gNfM1Fjc8DKHfLz8ygyUlMTmLyYjODJnGHSdxN4m5+k5jE3bvNCpO4ayfuEGToTPgu6RlE3J1GLPXEncnQmfCd9Ywi7ooRywZx52ToTPjOekYRd8WIhvHiwcU/evpDvKKzP4Co9beUxnsa2u4cl2/ga59mPS0RWffXTulAYFTSKRcRSz1xVyNDN4C4lzWNuxMZujERlYi7pzP4LxJ3XoZuUEQl4u7RqxkbcZ9kAWhdUsbdT/l7lbijcOkXBkdUIu4efg8KxH36hVEf/vosvErceRm6O81N4u7ZDIoa9yNH4dNxHXmVuHMydLeam8Tdo59ipnFfAmDnYxYvaNwVZehuBXaDuPvgmE8zD2aQ07j75G7eEjITXtC4K8rQ3Tt1q9G4Awm7Z1uAX56C2zVytBH3gjOx3X/wMnFnMnSJ8J3rGULcFSN6gLgTbfI55GXiLsrQOeE7L4E3hrgrRjSauPs8Pa0ZksPmvupl4q71DqubxN0LtPNq5MZwbOp9oJXJi7W2729E5n84/EWYGdTcso76hFeGmUGYpMkkTTCJu0ncYRJ38zxongdN4g7d8N1qEneVxF2CwkX4zvX0uMoLwZ371Jd24i5D4Xb4LunpIO5icEWf+rJB3GUo/BMxb5+4zqAq4s6CK/rUlw3iLkXhAnznezqIOwu+oGVlAPcvu1UWiTvXGHwvCcOrdZVHiT71ZYC4c43B95IwvFpXeXtz4VNf+ok71xh8LxHDq3SVF5oLn3pjXeVBr2fDu8Sda8wDnnOD103cbc25T72RxB3Awrn58C5x5xqD7yVjePXE3dac+9QbSdyBYzfhFVd5RZd5Bt9VYHjVxF1oTn3qDXWVz1055m2vuMoruswz+B5RMoZXS9xZc+pTr/eK+nzDb4loC9YR0WfX0pHqkSvql9oSEU0PKSK6nisdi/WGbK1mH8ee0hU1H+F6rmSsFHzHExlEGcHfGf+bhCfuG/4gT2XQJXHnUTiD76yny1XeHlzRp95o4n55Onksg86JuxSFC/Cd7+kg7iy4ok+90cR9/x9+uDH/rYjBZZK4K/nUe4LVnWieUsO8R62jfT+u8ps/wXSVNzmJyUlMXmxm0CTuMIm7SdzNbxKTuKPMlXNH6SXuLti4WF2dK+euwlH+DtO4u9K3i9XVWTl3RUf5O1vj7krfLlZXZ+XcFR3lPahxLwWu8i7831l1dVbOXdFRXrervPxGbmlylVdH21k595Id5e80jbtr2i5UV2fl3Et2lL/TNO4uabukurqtnHvJjvLGatyNJO4nP9uWsOIXeJO289XVhXLuJTrK374a938+7fbw9oeMzqBL2s5XV7eXcy/ZUf721bhjw82MbwKNzqAr2i6prm4v516yo7yxGncDXeUBVMg3/jzoirb37VEAobp6UVxVAFDhKK/BVd6TGWSu8sDiOQufOW34/Q/n/u9oNZ9VV7eXc1fhKK/FVR5e0bj/k0i0sUme0VfULtg4X11dKOeu6ChfSjTuN4koBfGGE3fnbJyvri6Uc1d0lC8dGvcKQRujkRYaF2X4HVYXbJxVV2fl3BUc5UuHxp1iFlfAmrdOVjT1xVrbzk0PpH37ZWtToa295Ryq3MzP1LibpMkkTSZxN4k7TOJungfN86BJ3HG7c3aUEeLOcXZWw915xXU9xJ2LX5aIO+PsrIa7csV13cSdxS9TxJ1xdlbDXbniul7izuKXLeLOODur4a5ccV0vcWfxyxZxZ5yd1XB3XnFdD3Fn8csWcWecXazh7qLiui7iLtaIL2PEHQJnF2u4u6i4ro+4CzXiPatx3zp91l/wrr5d4OxCDXcXFdf1EXexRrwHibv1taOj+w0sgheJu8jZxRruLiqu6yTu9hrxniTuiy+/6evb3xfeI+4QOTsAWw13FxXXdRP3au32b/Iocf8oJudCnWkWeI+4i5yd1XB3UXFdB3Fn8T1J3G9c+GfJgZg4bxJ3kbOzGu6uKq5rJ+5cfA8S90OYRHSu6lUvEnfG2VkNd+WK63qJO4vvQeJ+DbFEVGmpF4k74+yshrtyxXW9xJ2L7zniHlR1/iCgylfPe5O4M87OargrVVzXT9xZfM+xurHWz5Hc4kRdk3ZqbZlD+tw9d8hjJi/WcSv0RGarQJO4m6TJJE0mLzZ5sfkpNtud7YkOL5PpUq7QZl2my5YvN4QXc2S6TPFi1mW6bPlyY3gxI9NlixezLtNly5cbw4sZmS5bvJh1mS5bvtwYXszIdJnixVxX1GU7s0vXyYsZmS5TvJjrirpsZ3bpOnkxR6bLEi/muqIu24lduhEKbYFMe9QTHV71Q+enBF22sl26EQptkUwbX8f90stj303oBMzIaFDegkr94C1ezHWPDl9XfcTwzY8nKNulG8GLGZk2OoMXn1xWe+LyTsCOHZV9kDm0n9d4Mdd9eVp1NIr9+KONnRQdzXXxYrs2lifTHuLFDdMvnD8cMwVe48Wsy3TZThzN9Su07WTaowrt1wCa8LEfvMaLWZfpsqU42UCFtp1Mw6NVyImW/mT8FbUqsst02dIC5cYptAUy7dkq5NkPFnug0pUassvpsnmcbBgvZmTag7w4GPh+33xP3GFVRXaZLtuJvtqijxczMu1J0tS383vmPWpd7e8q5j1qfe1K4J2XQWOvPbY3vfMyaLI6k9WZxB0mcTc/xWYzibup0IZ2Ni5ydqagFn3SDSDuLKpzjl+6ibvI2TkFteiTbgBxF6Mqc/zST9wZZ+cU1KJPun7izqIqc/xSUoXcRb3wPy8BQKNEoBYBqIQbAGq8+tnQMD1RWZdFlS8vTVXIXSm0mf+5cwW1HuIuRnXB8Uu3QpvzP+cU1IJPugFVyIWoLjh+6VZo8/7nooJa4pOutwq5PaoLjm8ocT8wZdbSQnhToc35n4sKat4nXTdxt0d1wfGNVGhvPPbeqDqvwYsKbYn/uaCg5n3S9RN3W1QXHN9Ihfb8KB90O0ZeVGjL/M+rtdu/SeqTbkQV8mrt9m8Kds7xjSTudfok4ER9Lyq0GWfnFNS8T7pO4s6iuuD4RhL3STU79V2y1IsKbcbZOQU175Ouk7izqK44voHEPXv8qNDA1d4k7oyzMwU175Oul7izqMoc32Dibu1+kNJH+p7yInFnnJ0pqHmfdL3EndNlK3J8g4l76siNAJ59YJg3iTvj7ExBzXzS9RN3FlWJwhtM3LM7HvAHxjwVaapjtbb/rX22SnzwMFNfrL3lHi5qEWQqtGGSJpM0wfREL03N9yMA0bfNy4n25KuJ9kxIIppovpE0ton2K2qzmedBM4Oluf0/yo2hsqn+bNgAAAAASUVORK5CYII=',
    '1109.4653v2.8.png': 'iVBORw0KGgoAAAANSUhEUgAAAbwAAAC7CAAAAAA7qXQPAAAhNElEQVR42u2deXjU1PrHv1PaUkpLoaUsshUriFDZCiKCFhEoguBPUYogCrJ7RRAXFEG2qyIKyL3syBUUEJFLBQHRXpC9gCBrWWUHoZSWLkDbaTvv749Mck4ymUxmJjMVzfs8PJxJct5zOqczyUk+51MLwYy7NQLMt8AcPDNKI4hovPku3H0xnogC7ANohhDj7f98kNbYAAALCf/MEMJH74fxKS1knvPu6gg0MNftc0B4HfHV+VtA3fLu1M85cqNCe6OSGR8XC8uWWCtH0YmygUWFsWVxPbusLaR6aV+wGHXOuzj2ieCgK/YX6dFoP/ai6nFnHl2uuv340NAebicz+lTi/P1Y1B7Rg/dQ3oR70XJCBtG6BHT6zo20vumocZnPJeEje3FOX5x2ctT3eMXJnk493E7mv8GjXRhBRETvYQMREW0b7lZa4ztq7DkvJO6h/winZrpVASFOjuq+719O9gS5n8yPURnZAIBg3AYAJE/9i03SB/6+DQCwq43GhVJ8eeOS+TEikQkAuWuF/1MeD/mLDV5S6CLh/W4tvC65cQWA9VomAGSnbrVeAEpuXBU+UKd/SRcuBTafselJBtDp1HzlUdZf00r88l5VtGQCwJzhyARQ/ONTxqW+c774TzB4FXquygGQG24BAOxMiB4ALKhffS6AaUsiyn0Yj+Mto4cDwJaWm23LJ5bA+mZKePITp10nA37qfqDg7feLMPfh2g2mAy/UjFtMC565c7btTn8MXplKmQDORNyPTACLBliMSnxp0JI9Y/eU7tUm0dXJtB1ziejLdPoHLhERtUwkosLAyUQnEojI1oaIEnoQ0ZYy+4mSQ1Poi9rHiUY1LSGirj00k62pl0VkG9abKLvyQCKyxaXT7Bo5RD9VvOmHCxa6L4qI3rMex8tEWRPcTes0ipueIZr3YmlfsABoU38RgIwq0ncNAASXBZB+8CBg6QcgDAC92qE5EFavJirm3QbaHzzjMln+P5IqAZahyzch4pXv7gAXXq6SM6ZHBaBDwHJ/fPSibtqw+dGgKGQCs14Ttm3lW55+wpO0h45WAo73VmZznczwwbMM3HcYhx9UaeChmGZxIzcNtL+6fKwRgA4HG6BHZvy5NduR4zLZwctxABBnWQcMzlkJfPUSfssJ3rFjx67Kp/xyxWLLLk55EpWQiRNVooRtp7iLLzrt0X2EKiUNXlk57UlFNh3JjL899lLgF/i5o9o8YusHobM7vGS/NPkd1ezbbQtaL41qoyPZSWG+EBB8HIhNnIeinCq4gCohISEhy0b75ZOHzCUvA4ERmfjPAPu2QU9zv2tza3mStuZ/Y79M6kaKbDqSGT94VZ9amhNURnFiLQZwOm/i3uszvtkubKqHS/ad77y9dFzbSoCNXCSLxS0AKCq8D8DQPYfWdQMeQFSLFi1atKjun8H7/XoDAFGZGzsK9xWt60+y3Wlr73h0jyv7mV1X3v7xd3k2PckCDL7XBmDAzVe438ayBCC9EEDaCqDSyB5Hhe014g4AQGFy3ucvxwLXgZ93ukjWPDoVAH5FZwBP1Zi/LQFock8KABSu9c9Eb+owAIjK2mL/NpgU+2i2uPc/F3OSPMn6cgPCPRPLVJRl05XM2MFLO05A5+p5MQBykAsATbIArK6QD2DOHQDWh4Gi24BlYeoGAPNqBpUrBnAkKP9qJIqKtJKVn7/yLFAypedTAAIHfRVjAUIWrjsMYGYt/3zyelUEgKhA+5fmyfhbd8S7QhnFT97O8yTrqUkWYO2gaD6bzmQGThWOJ0aGtfg30ZRVRJ90igpv9NwJooy2EzdMWV8jrOPN5M7vL1//7kza2z0ysut1otTm76+ctIpoVa0pGyb/9mrrd1O7R0Z136+V7OfHP5rffbyViIguh90gIqLNrccseXu9P+5t0vLGxURE1PstadPI/tz+XuM8mSps+GDt6jFjrcpszpP56Ma0atzYl0OHzubZcqy2E4fzZdPC4yVERMVHDxSSLVtXsitpRdJzB7Fw+YTNLzem6fJR+x3qXHFLQdSOzBLxha3aJo/meQVH9uc6ZNNI5s/Bu6vCzfdj/b220Ta6/OoNIqITwbe9e6rAZ9NI5rtJ+t8r6tf5rJMFl1dvAIAtrUKNy6YnWaA5AN7EfRsQArQ6+T0AbGpvYDY9ycxPnncREgIAyzpg8M6TKUOMyqYzmfnJM+KBzj33wLrp8ubqRmXTmcxE/+5i9C8QQILFHDT2ptj/+SCtoZEA81MHc6GJGebgmQFzlRDMVULmKiFzlZA5VfDpVAH+W0uiFWfP3WgV41ZPACAkxsmPcDPtRsV2MFcJscj6NnU7nb9HeHH9oYz2jww2bPB2L9uQHIOz/Ya9oOfozG93/lKzfyhubLrn48ZqB1xZuuRpHw3e9dfnR7BXtoWXQ+6MjPY6qwd53H0kpHv5h22mu1/jWUjWWoWijNPoR0SU36Hcr+oHtE/yxSOhjDcHt8RV7ud8YRLR/vp/eLvQRFceLx8J6V7+kXHM3dRBgNYqFIeeoAwAhEzIf1fHshXDInz4nN7867UbRgPN273ubV4P8gT4Zi0JgGQPT8ROV6Esu6y+vRZO+vNEU7aOHI1b3iYYQMIPd7zM60GeAO/XknDLP4oz0lF8Lh8AjoyB84Ug4qaia5nIvgUAKLxC9pUpV519lBeqb/8FTwKy1SrSChbYzuf7eCxLUiIAoHrhFv/nCfB6LQlb/oEdzaqM/3Hq3qT+hG/fK9j04otfA+JCELW1IesbV184+9+1twJnnh+/9aMUQFqFohIvr1RdSbP/3cRpkK1WEVewACmf7Enq79uJUMbNMACIwKlSyOPuBYvW8g+iR9ovJjqNTUR0/xAiIrYQRG1tiK3+szuuVltBx8O3EtEUJIurUNRnuiNkLy+h5aeffpxUda6NiF+tIq1gSXxohdgZYxmWGdwFyymMJCI6iAneXbDoy+M9w+J0+QcQfqQvUDfoqHQsWwiitjbEEn6mTbWrSejf7DEAAmUa5rTdcVfHFMk2NH7rrXdX7F3W4Ry41SpsBQvOPS/vjC+iAMHCI598/+cJ9GgtyTuHG6ss/3gCQMMAoEzwbenY33KCdwCofAoYPHVlP3w1lN8EPAgA13aPV+/LdPkVa/nP18xvqzym9po6nfeF93jWcu5wKnKAy8c6A+hwUKUzPrn4RAkAFCPc/3k8wiBeGvPFv35+A47LP2CfO7ALlAuoEgJgWQ0gNnFeP3FtiH0TEAkAJxCh3lC3h2QvL6zv2czxoMhOq9e9YFu4uMvjbQB+BYtDZ3wRFYWPSoGzH8GXeQI9XEsyWX35hyIWDXgAUS3EV0OfOXTWvjZE/oT5Xjj5dNSrx7+6PGJjM7WjwpCOdxYciMUOwGZhK1j8EhHV0gHgGh7yf54Ar9eSyJZ/8HPoIuCsbCGI07UhtRrsAeDy2/616apjV7DF0p5brSKtYPH9wGUVAJau5wDgbGS8lzeauTxZBT4aPO3lH9YiAFRUAqDp7wDJFoKorA2x5gKAZUHKUYDmIB/CKhTVOBKUIHudAysAFAy7OKkxt1pFWsHCd8bYKEQhANyo1RbAqGNnAVo9qoyXSVkeIa/xUwXt5R87u1Ws1H334sfDYp6/SWfqTJi4Rb4QRLk2ZGOX8OjOi4mItj8247s3v0X0c+IqFJV4M5l/dez/6qLCs336dG2a+D/iVqsUiytYZJ0xcKpQ2LtjtfCGT88guh33IhHRioS0rDdftHot0ZHy2PO67qgxz6/+yK7vePIsOhITKdzfv1XfPp+/JK7EYpukhwA3GtqOVa5cznkj52to3awsOVHUMJhyhdP9tez6Af57nnd9w+3WzSzeP8/Tkcexo94PHsyHsaay0QyY9Jg5eGaYg2eGOXhmqA9egvk2gF+/keCbtPDJQhNzqmBOFczAXWd3d0m/uom//vm07qUZtgCfDp5L+tVN/NWXXC8Mh2zFXbToaFiZd8oDgO2/x4oChke7k5LVVpC3W+asBJA38+LVB0dUhS8MSC7pVzfxV4+17u5zvjpuTGtAtmyXtUufEvq2cQYR5XUcZ6Wvu2vdmFamZLUV5O3t2K5ElNHrDOU8F5HqE4nOJQwgIqIdeMLJEYlJ7jFOD90nOI1sU+2+XH2RPsQHg1dwvpgHjuj7iEIiGvycbNekoFwiajOUyPZcHyK6L1pr8JQpWW0uORHRP2O7EtGwc0SUF1m32JcSHU/pV0ea1kOte7KfIVu2a0HjcAAPL72DzatGAfhmjTspWW05ebu7bmUAWPfITSAs4dwJX07S7fQrh78y+lULf3WkafVp3cV2RHSXcb7wM2SbfTkMAKJvHcS8iKYAWrR2IyWrLU9e8F9hyU2tQiuAMFz34eAJ9CuPvzL6VRN/daRpnXO9dR4e/elnCZUvcO2I6K7E+cLvkG25MiS8mSfol7qX3v/4XbcIXFZbnvxfrws//varVQEcDozzjUTn8GcoPrhlwhALgK9X/dSg5dWe+wOwtcPe5sjr06YDcC7rPcTX66uuZKqU9Nbnik0DFn8zFFj9rP3l2rf2VEK7f/RbVlJpexB+e39cHa6drl0a/Doqds61pKQG7eb5/JOXh/IAUEbmxC4blwkA55Gdm9ngu8kB5x5d8oQbX8xSbVnyA5Xtj64DggH8emhUtG8+eRz9yvBXjn51gb860LROud6cp4OQ27Pte3w7Errrl1CHYycduwZYD6A4F7t7BqBux8EFbuSUavPJi77uxx1SOKDDh77UVwn0KyT8laNfHfBXVzStM663VRJoaO7SMgBrB3Z01z+hDsd2/2zQnKKvu6ZWLo/atQE0WbzLDYucVJtPPusf/AdrTPT3IT51j0V2Wr3uBUj4K0e/OuCvLmlaJ1xvWEMs+mZjdWGuLGK2dnTXP+EEjh3VY2vhG2PRpEJwJACUQ5o7CkCxNpf8WGFUNlBclF0mHMCiPzaU9bE4LgzpYPjrfRr0q0ua1inXe3T46ETsKHqcw2wtvBxq0QCUCmRbpw5wIaZJ4IO3hV/Tym5ltdcuw5JnXB4L4FiVsfeOAtYdWhaAowENfThVKNhiaQ+Gv57XTb8qaFotrvd2UtPJwJEKUJPCC5wvfErZqkO225/JAXK2jQtE9zNWAOlooZ/bZbW55AmzZs2aNatC3KxRQOqumQHAmrK+uGDh6Vcw/DVKol9d4K9KmlaL6339j2+CkP9FNfBSeAHdFTlf+AqyFWhYOWRr37Vu7SXg87p9gaHlfwLop4H13OB2WW0FwVtyOxfAiRczhw0d0veLGOPvbSroVx5/tfvbXeGvcppWk+vdjLbvjxkYCyvXznoJ3RU5X0NvjzHI1k7DMsiW7TraZdtvb7VPJyLa22LVvqG9c7Vujym5Xa62jOAd+nBYxBOfUHP7Jb2PoFun+Ksu+lWbpoUzrleG2ULO+cKXD2PV4NjMjdnNWgvbsrfeaNnYvYexXG295K0J3ZpP0s2ASY+ZYQ6eOXhmmINnhgndwoRuTejWnCqYgbsPuvUHDOsa4MWfRGNrC7irBs8fMKxrgBc+19jqMtAKMCwHy3LQrQKoVQslRWtbcinfNug+BbIrFfvGtwy9mNqpvTc3pv0Bw7oEeMVcnmhsPboxrWagFWBYDpZlJSVQqwrdKiha2/DDRH88nipHdlmxKoCgcTavoFt/wLAuAV4xV6K/Bk/BwXIwLAfLspISqFUbPCVF+8N0IqJDHWTILlfsOPu9hee8tv4ZD8O6ra9N9veJRcVAa4dhOViWlZRArVooKdo9lwEg9ix4ZJcrVnn1o4ExXk/SjYdhXehrmSXXXuJzSRyvWifECpx8FwZBtiIMy2BZDrrVE0qKtsb0z23AD13AI7s66N0AgyS3nsKwmvpaSYkrlfhcEscrdoKT6UoVOPkujIJsRRiWwbKspCunkqLtV/eNdifXLv0QHLIro3cPzfh0Rg68WiWkIbktbmIl2h88WeacFT22ovTWbX0tU+KykpiLaWyZaZfJdLkKnHzXk3Oeo4H2t0VErboSETVpRET0KqZxJYUI17m+ai9GSeULjyD4UStRNtp8WkJna/xPVqS45TZaXf+8lxcsZKvfgoimkjB4d2qOJSI6hP9lfkCUE9u+mIhWVdpHtA6niCi+iVBRa/CKe75ndbxgudC2/VnKjnidiEoiZ3MlNnjRJUTFQTO5ThC9E3Gb6NynxFeQDvRs8A7jHSH9aHE91hsl4uCtsVwlKmyNT7iS3sEreLAD+5vxp/u+FYJOV+kiylwgon735vNFOkRE1PxZh4yBxkhu3YFhdetrmRJXJse1h8jx8qZdUaYrq+Cd79YBsuVgWAmW5Up6g6dojwxZW3nokJ+fTuWQXZ7ebQwALRZkRHvJbRoAw+rW1xZKSlyZHBdyjpc37YoyXVkF73y3SshWBsOKsCxf0hcyinbQ1MqITZk0YWMnhuxy9O5u62MAwpHWzsvBMwCG1a2vbS0pcWVyXAVYKzPt2mW6ygowDrKVwbAiLCsr6QkZRZub9igAy/hNp7owZJejd7vn5AUDVkR5c7XpExhWQ1/LlLicHNchl8y0a5fpqth0jTDQIqtABsMy1JaVdAWjaLMKgHKWPOFcEccju6zYZG4wgOMRD3gzeL6AYbX0tUyJy8lxxVwSxysz7dplunwFb323HAcrKWgFGJaDZVmJY3SdBqNob9RqCwQlfQ4AV7Ie45FdVnzuAQDnt84I9Hyq4BsYVlNfy1lypZKQS8bxSqZdTqYrVnDXd6vyfjAOVlTQ2mFYDpZlJY7RdXq1yShaIWPBy4O3HFrQ74Ic2ZWKxeM+P7ii0TSb8dCtlzCsK+CWKXHFklourhOSTFfFpuvhw1jnHCyDZTls1pOHsad/vdO4pUWB7LLisd2Rraua0K35JN0MmPSYGebgmYNnhjl4ZpjQLUzo1oRuzamCGbiroVvdNKyvcFiXHSh1DNeLsFEZj4jeQINpWF/hsC474DsMVwnd8sQsQ2SlEo/fQp/pdvt7XaLLA0DvAECF6PUaunWts/UxDuuyA4a06xq65YlZxsVKJc55q9t0O9c+Fu3UiV7voVvXOlsf47AuO5CY5BfoliNmGRfLSgy/1W+6fX3+j9u2b9/W7pw60esUug00TmdrMA67LKGmoT5deAfdvnInVHi97vu0SghLSD7RCJtX7QfwTRG4kh2/nT8tFBqmW8V8bTAALBkQAzgQvfOnhaLKqwZN0kWdrYTDigXdOKxYwQUOq4njijwveK+u1G5xRjqKz2YCKDxvx0KzU7daL8AI6JYjZhkXK5XcxG+FeA0Azu7qA6gRvcbdYbHrbCUcViroxWGlCq5wWC0cV+R5ZV5dqV0h86/9X7F9MXdnqzkAMG1JRLkP4w2Bbhkxy7hYVnITv4UdwwFsr0+xAGpEL7yHbmU0LGNaGdyqE4flKrjAYbVwXInnZYgv166YecRBoi9Ds4hOJBCRrY0x0K1EzDIuliNkZfitc27Tge1c+q5Totd76JanYRnTysGt+nBYvoILHFYDx2U8L4f4cunsxf8jolTsJ9oacYCIFhoC3TJilnGxHCErw2/1D15hrQtOiV6joFu7zlZiWtVoWE0cVlbBXf8t59MVeV4e8eXSCcVGAEJwG3goplmjDt0GwiizrUDMMi62KSNk3cdvAQA/WGoDTohew6BbQWcrMa0/qdCwmjisDJ9123/LfLoiz8sjvly6EPmGkK3Tf5w9s+/iAGPMtnZilnGxj3F+W3fxWyEW1wWcEb2GQbeCzlZiWpVwq0scVpOG1Y3jAiLPyyO+zuN0uYkTby55e0CCIWZbkZhlXKzMb+sefitE8ZYOzoleQ6BbSDpbiWnl4VZdOKwbNKwGjstuVakgviqRtgKoNLLHUQOgW56YZVwsK7mJ32YJFvhjtyIBOCF6vYduZTpbiWnl4FZ9OCxfwT3/rawDIs8rQ3xZOnuxCEARSgDMuQPA+jAMgG457yzjYllJht9Cn+kW+ANhbGapJHq9hW6VNCyDYBkXqxOHFSu457+Vd2CjxPNKiO82lm69WKz9zM3RzcKbvkbJnd9fvv7dmYZAt7x3liGyUomz1uo23RKtxRhxpyPR6wPoVmJapYJeHFYfDavLf6uK+DpGbrnAU9Z6IYZDtxwXK5VU8VtXb7F1adeqGvpbE7o1n6SbAZMeM8McPHPwzDAHzwwTuoUJ3ZrQrTlVMAN3PXTrmrk1jnn99ef0er2jpD8X62lkHL9xz8O46023rB2VFgMNYm4NY17HpE+qlja+1nd6B+9sv2EvqG0/99WXwx/2temW0bDMVau02EKH6ZYhu5zzlmtHVoLbDItr5tYY1nZNfAkRlXSJ12vI/R6vONnTeISvTbeMhmWuWqXFVo/pliG7nPOWb4cvecCwuGZujWFtn3ubiIj2xOs15Nr23XKyJ36Er023jIZlrlqlxVaP6ZYhuyyPrB2+5AXDYjjyqmBrzwrf7C1jAH0YryUe/oVuwbhYRsPuuQMIrlqOyXUaymMYssvyyNrhSt5M0gXmljGvPPLqnHnVQF4VbG39VT8CgGU4GMbrpDUhSm5cVfC7HvO1cNN0ywVz1SottnpMtwzeZXlk7ai06M45z/61ua9aYi7Rugb4eNakSluocNQXez9td8r+tfnzRyu69bPR9jgM2fDhim79SxbOWNp8NhF99nnang+i1BNnNSjiX+4JQPspuwqJiFZ0Db2vT5+vVFqb06rW/dOIetVo9OWxZughtdjPxjcWP4JoXNVaHa/Pa13r/mlEfWo0/NL9r82rGEREtB/MafTJRTudR30OTp86PZuI8uvi0RNruuYQlRQSETUJvK7xtak4xhbV9PyYj0aflOWRtcNKHp7zeOZWZF455FWDedVGXhVs7YqKAKr+m5esOrbGKW0poYeM32WNxY8gOthwMxHl1OxPRCUPphtsuuVoWNFV62ixdW265aW2XB7WDteiNxcsduRVZF455FWDedVGXhVsLd38Zkg94DPekOvYmqS0JeraQ9Y4ayx+BB19NY+IiCaH5xKdmmO06ZajYUVXraPF1rXplpfasjysHb5Fby5YROTVzrzyyKtz5lWJvGqztRV79aKtPScOqMgd4tCapLSFAvSVNfa/vmkC1zNg4rKhWDYKBptuGQ0ruWoDlBZbHaZbTmrL5WHt8C16dcES2enUOuawtS1ovTSqDVwwryFbPwid3eEl6VKjWz9ZPBHOsbXfAoCl3fy8A7JWla3FJs5DUU4VOIC+fGObjoaOFPZWf2YO5RdVgEGm2+zs4qLsPGD3NmF40zBoSmXEpkzYuxEAFv2xweUf6uGPYfAul4e1w7foJXQrIK925tUz5FWLrV2VBABIRD6P8Tq2JipttRpLHNm6TZdeAIBh7XeeSYLRpluJhuVctQqLrR7TLUN2uTysnXgZievN4CmQ12F25DWsrUadtN/fQqWRu48m6GBrj52JBYA8NHHAeGWtPVVjftnpLhoLQ6vxQ1vXAYB2DWbXeNlD0+0BDroNDUlIAIBVcbMANHnBTsOSJa+C3VWL1F0zLcCaXtA23dqPyQoNAbr/0xoMpKNFOZaHa4dr0bOvTTnyamdeeeRVi3nVQl4VbG1J72wAmNu/BjiM17E1UWkLFN2W87tSY9Yi4O3wHjcBwDJ0ZXMYbrqVaFjOVcsxudBpumXILpeHb0dWcnuep2BuJeZVQl6Ld2owr5rIq5ytpQdTBv5z4+aRz2YzjFelNcbw7u0eGdn1e47fFRvb2LVCrWeu/xAR1mAOEdHNmAIy3HTLaFjmquWZXL2mW4bscs5brh2+ZKDp1nvkVcHW7o+nI/tK4pupYryy1pjSVk9jWXPG+sB0y9GwzFXrycNYBu9q5/n7Qbc5Z5pjas8Y80n63RiftMjNK4jB3/lvxt69kbg/NeUDmMDN3RnXr8QF/RUBJJMeM895ZsCEbmFCtyZ0a35tmnEXThX+BNirni5w+e9mzS08h3sD/5TYq64ucPl9p7nVNt3yClrJXcvst07j5oLc/KLX7lcx3yr2CKitgsSF8xvTpY29Krqg1QMuv1F6XfdMtxJ0y7lrmf3WOXT7WgbR1OAUR/OtYo+A2ipIXC2GpbSxV0UXtHrA5U/02eBpmG4ZdMvctQygdT54M0JWEmXgEUfzrWKPgNoqSFwthqU0sFcN8ja5tM86GqZbBt0ydy0DaJ1HNQJQHllwqC3fY0dt5SSu5sPY0sBenZO3zJ/LSXPp9C/pKucmUa8r9bHkxhUA1muZfJ90inb1mG7VggG0zqNX3vPATjzlYo+I2spIXO3BewNdnvgk1YoEZq+VxLKSV5ZT1x5vGT1crq2VaWU/qFa7U8b8R2o3mA68WLPRYuiw2rIuMH+uKM0FsKXlZtvyiSVcfoDpdZn7dmdC9ABgQf3qc1mfdIt2dZhuoaKgZc5brQgCrNNajHWxR5Te9qv7RruTa5d+qOdJemlgr87JW3sPmDSXtpTZT5QcmsLn55BfHgRumUhEhYGTWZ9ciXbdMd3KFbTCWYsHaDW4zd3vtBxyS1WeyvYw1FZO9GpDt6WAvTonb4UecNJcW8NEIkppclyWnzXPg8AdE4mIyk+W+uRStOuO6VauoBXefh6g1YJubSc7d8tQNd+KezjUVkb0uoBu/YK9ukfectLcy8c6A+hwUJ6fNc+DwNJJQeyTpmgXbppu1RS0HECrfZVXf2mNJ3eX0djDUFs50at5zvMT9uoeectJc39HNelAlp81LwOBpc7Z+3QBVUJCQkKWjVYT7cIN0215gIduoQLQuoiolvt+0tjDobYyolf7DoufsFd3yNtFAzhpbj1ckg7k8kvhAAJTMeuTpmgXbppu1RS0Muetk7C2Ld4dDETitMYeBt0O5IleF5+8Y2fgHHvlvLJ2dS20tLJhaDV+qDBpaNdg9pHGOq22fBeEHnDS3BpxBwCgMFmeX7WPZQlAeiHrkxuiXR2mWzUFLXPeOo2CfYevA7iEJpLp1nEPk96Wswice+0411OF0sBeNchboQecNNeyMHUDgHk1Zfml5mV9bJIFYHWFfKlP+kW70GG6lSlo7e5a5rx1GhW6bK4JXExLeJRhvEJtbg9DbZUkrtZUoTSwVw3yVugB8dLc1Obvr5y0is/PW3P5Pma0nbhhyvoaYR2/lLhfF6Jdt0y3DLpl7lrOfuv83uZLn+3c0arrVRG6ZbXZHg61lZO4mtBtaWCvWuSt1ANOmnstu36APhA483y9CofDo0tCWZ80RbseQ7dQAWidP4w9eIiaNVZNqbpHhcT1H4DkDvZqPkn/kz1J/8tir38H6PYvi73+LaBbN7BX82vThG7Nc54Zf/pzXoLFfB/YbzT7I1NGp4UvoFszYArCzTAHzwy98f/NDCjAXRqEhgAAAABJRU5ErkJggg==',
    '1109.4653v2.9.png': 'iVBORw0KGgoAAAANSUhEUgAAAdMAAACiCAAAAAA/nHihAAAcPUlEQVR42u2deUAU9fvH3wtCiByG4pEXhuKFoCLmGUgqhmVftZ+UZWmaV1ke39LMI7WytDwqNY9Sy0z7+tU8M0kFjzCPFC8Mv4J4IYICIgIL7PP7Y3d2PrM7Ozu7O8Aq8/zDzGdmnmMeduczM6/nWQ1BlUdMXNRToOZUFecXIopQz8IjIxFEpCFAo15TK17K6cRrSP3ufRSlWqVaL0gDvJtwa1fuA01r2HJ83tlsnyheG/Nv7+Ip6xCnlfykzMejHLiegipLrk5/xt3thmEl0x9R06+K7ne5xwbR8eQxnoP4tTpuvV4Z3hRBbwyJcg20dKjwkMoTKyf+1BS874haVGJoabH41LC4bCguWdjrV7xhYUsfPkFlNVOI6AssJaJz/pYP7fMw5JRuYbe9aqtV8neMR3Da91M1AED3feBhYa/+J1pa2OLGL+YMbg7AFS4A2vQprG7pULeH4qIY79Lt4b0/Hfm/gwCAPyVi0ITJuMzebseshNy25VAnlANhPg9vTmM9v9PntIt+vSz7BgDtrTsAchMTtOkAyrIzAIAuHcjU73R1/2Wd2Ue2M7PSo/TWHeTe5w4tvX0bD9IEh2iPny9z0nzqrmpxIPIhfo7kM3hzHoB73hoAwJEI/xHAyqD6y4Ev1/lW/yQMSA73Hw8gPny/bsPsMkA7Oc576zOXTBQ1a8+s3H2u/qqlXzdO0B+6u0Pd2XOnb+/9vtY4NVw54EFq9yNOmdIjz25ZsCklwqHnSJU5R8qYS4ewnIjWZNJbuEZEFB5NRMXV5tLFCCLSdSMiihhEFO96kmirZxzR6sbJRJPalRFRP9MJzyIsJyIiXdDAwxn1NuoPJWpVN4nofofndIZDljbII/q9Zo4TzpE2P5FG9KxLruVjS4ok1TrBM4duQd8ByKrDrdcEAPfHgMzTpwHNMADwAmhcrw6AV/OGQM38AiDq9GXJByrel7vVy4gFvAAAnn1DgBozd24x3KVOG+QD9HLZ4Hyf0ozhcwOAhh18ASBBzMFvn5jn7M/wNSNPnMGZtuYudQpoHzxh30jD2vULbQD0Ot0SGHQnLG3bIeRJ621rPhSFXfqFv/PcDx8+/GftFOfL6fTiQQAO6r96U8QmeGM8ezj9e5nXqq3G3t4i9zkJMz2X9nrNMLP5H+oZ5xAru6yvZXWq72c+5O1tuAino46Hh4fHT1OcL6e/9fQFMv7RT5HefIHd9Hup/nnbzS5O/GxQL3WfWz/XzdXkMl8KXKo+e3bOuvdG6P9lm+Mat/H9lacCcRjQaTRSn3/zoXv5LfQLrVCro3POeW9lDANwwKV7aXENbVyzFuy2vzV9ACC+kyeduN/TaT+nRABG5LzB/D8+RgAyi4HzG4HHJww6px9uEHwKAIq3In/x64HAbWCvDfNWLQAkIEa/FvpEHAAUb4fzvdBuDeC30Jq7UjAnsEcuu23cwkIAiI/I/zJnXKrT5vR8MgF96+cHAMjDPQAIvQtgi08hsOwBAG1nACUF0KxK3A3g24Zwq14K4KxbYYYfSkpMFBah2JDEe/q/JQUAgIRbwINZAwcAJSWAx6qdZwAsaeR0OfVvkwXs/KM1jrT7J+z+A8FDL995L50D6ECTVW/3GNukol/jyZSLE45rW77+Nj5vNgjz953UNm71cQtkD+gdfqbtqLwuQzaFtfE9VP8dHP/4MLqs8T/61rOhF1sPAv478a2Qk/1Wn4p4Yd5hTbdZHYz6dq2/lfzAs1W9kc/8/tUhj7CXXucORceggEDP1d2mux//WH/IgQ97tjgXGeN870+PTZ58wT9o8ljNG8DEvO+BKUn8xqN1drRIa97p+dcaSKit7PtTUck+kUdJqfm5Wt3FM4WCu9nkMiIiKj13qph0uTaoDHud0s+VCIauX9Q55TP8oqR8opvpRFRU6/CdMnbTkvdLiL7vSt8HU26J0z7DF5VatYAQAEAL4YZ6hqmvaxsAGl/btDY2WW/gpE8GHwsBUB8A9vl2/YC9FT3350YA8VFokYevp1Z5xqyk9CF0OqjJF33Y2fvqmQBwrC9aN1jXopqTXk8rSvYu+aN65LxWeNh4pCLh28ebTwBAvjdA+T7Sah/9nJbBFaUuLlWEMauGqiCuqCqRqsw2HlluMEKjnodK+PKF6ONLByUCUHlt9btXFTWnqqg5VQVqXZsqal2bWtemStWoa3O0Fk1CUtOynwqwyREA8AiwFETO+eyakc4QmLPn9O6mxEN05Qn9yu1OWVFdRykV+tGfdm8NAFKHjX1Zxt53Nh050HC4J7L3PTEvRHSPG+vXvRDpDIHJk/wlVzPavlvXuK5bdd3jwQR/ABgaFu55NbFPFKBbd61Q92YzAND990KJy3h/RZhtebVoRLoltr4ovoutklVsJnIJw4iICntVP25hj6jY8ghMgUhF3olnvXSZ8l70TTQqfXkO0cmgm0REdQG4zdAR6cafIbrZM5GI8nvP0NKP/ZVhtj2CO32vv7pL1aIBWRds1exmKEX7SqYjcAUAj48Kp1ove1MwMAUiFZGZ856EzxrXIVwVz/bdU4AOke8AQMjSD1alzNEAu5q2BeovngHQ8Dpz3DA7Uan7Uzm1aMBWe+dulkrRfrouPt4I/9hlyFyfvMAUi1QgO7vmAF4RaRcN6xu6uQOI2PEAQJ1xn44MAIC/rgNAYCqwf/MkAD9vUyqnprVoAF1KLASA0qxMlKYVAsDZaZAoITOMlRjKzwAU3yCusi3Dwsdhlfj4ATwLgK124yvgoLtSaPHjtUp+YKxwdjjn+UgdkUbFWgBeMFRZlsX5AkD94nh2pwYLF+uAHTHAt77tAHTsolROTWrRgN/7nyp678MSHG5fZ9Zv84/FDids+qBo36uv/siUkC3v3LjlQuDlhsFrubFdIfryM+Dy/81K+DQOMFaxicjrv4gSKCenRn8JQbWbsQIOiPv8r9jhJFefxcDQpPOUBV9E1E5n7HDOGyN1TA5l1AVwplqw4R8uxwsAfJECAEmLFizKAzCs6cTIf7av/wR0oOm1D+dNTVGors28Fm1b87tEurFDiKhr1FqiS9hHRC1GEwlKyHJrjyQiXXAmP8aVnyV7JxDRZ9jKVbGJyqx3BavXEL5gwbzYust1RGy1G18BF91pI+eNDH0SgZWGaolOus8V2DHWzhkiVYAbPIZJhqUUTCAiOo2PiCh4g462BF0hovSucO+hJcpFtwVllNrgD2X6OWTMJV1QRyKaT/rQHzScTkSUhD+Iov3LiErdlhhzmuv7DhGV+S0lovd9C4jSFrBjYaFERNT5aSKiNH1O+1nKaengD7SCnI4gIkrvHpVKRJsfP0G0Eymkax1NRHGhyQJvZOiTCOzOTKK8wKhS1o7ReeVyWtS2Fwe/ntE36UjCFCJKIiLqMJCILg39twf6ZNBVuKYT0bAnC5ViQTUj3z8TYqxFO309GACCNTufAVq7AK7uBcZd/85zPwygdgqAUfN/GYYfxgjG2gLAraOzxO+WFwpnlDUWb1vR3XSfxtua9D3hjUEDNWlnEpGH6xf6Auh1GjD3xpo+S4E9FQsac2+9K8DbEa+dc0im+f/KTbe9UQYApfCGgYztuDLL/+zo7bXHjN77QmINNG4MIHTtn1FK1UC9Nm31V3snGlb+0c/7XdyTAcMtAD8nSkcdDwA/NQAQGP3tsJK8OoIxPwC4CAuw7vOdBKvpuwa3N9/Jr8+WnS9Dt2ptTM9uEFTAmXljVZ+FwLxa47uf99TXPwow2BGvnXNEvru5+zEYy3ALAaAIvsBR7dMAvHE+8s35tREYN+ejPX3c/QCgOs4rllNBLVog7gNASXEzcy9HCErIxgxISn1eWFamAYAnUSBup3lzdu36u3vai+3lhUym2q0ZXwFnuz6LgZ0bPyUah0t6CqrqNGykjt/NJP3kgnMurQEAvvUyAeAWOgH98/LdAS1q3TvfA4Bm1r6UmLYF+v/W2orMe81q0Tr4JwLAcfQV3sKXAKnCErLnGqw4GCFSVtao5V8AUGjF8tsLRVNaFK+JYqrdrhgr4GCbPqnACmLbzQXO+ohW1ekjdVQS/1ziAmx7DLhbBGj6pQFAql8YELrcHUCyb6vqmnz95SYY/S9rAWSioyI5NatFq7Hil1Sg7LPBzwHaEgBUUgag3f8AEpaQVXvzhwCNoKxMX36mWRl3DqBl+qyWWPjQnnUTvurN01cgFo29OieEqXarZayAY72RoU8qsHdu/uyGwtX1BFV1XO2cPlIH5eKrd8aOGT10dQCyG3UHMOlCKkBbJrkCL7YCcCVhUTW32MUAcOPu0xhT43eAfh/ZXIF7meRoP6+OXxN9tpno8z61vNu8eJFob89PV/SfpaUjz9d8vP/RtT29Av4vhy43+Wh2PBHt7zJt3Xu7DIVHXtlE/NieGG//vmuJiA49veg/kzfB/0U61t/Pr99tMdOTt7JrF/7VFD4DX3mlX7voP4iINjf6bPfcv8d1mVpKiR0+/GXOZqE3VvVJBrYf3T+cNjIQWsbOLqPzXKQOzXsN5XkhRAXBrxIRbYw4f3fyq1oiKp2x+PTGNl/qiIpeHxWftHJYOhEd67j5xJgh98zVKvZq9mZukNm1ueRsgH4SceN+EHflucaVfDJjhoHs1roLtWtXt2zjSgPJ57dlF0tau9M9XwC4lRtk/SvIij5LgbF2TCNV9J347d0FXdrrT9KFo35d9G9sLh1/EBKuAYDchOzwkCpbWwGVc1AFKjeoippTVdScqqLmVM2pymxXgkSUz4mPgHovo97LqIJH9rdIHkZc2rrPdrn+yOT0YcSlrftsl+uKyu13VviKMtw8vs0PMkg3FOmz/TDi0lZ95nTZ5rpSfcyyJo8KR4Yow83j2/wgi3Qr8/syBhSIDuMZC3tEx9rGOHVqpm8Vp5tv6KEuTzJHK+czpyu6UnJadKV0EZvTsWlElO/XtJToV99iIhr1IjvIjyndO91eXNqcl1Yel7YZ8d5aqZfBx5q4WmK4eXybH2SRbmWfOdiLS5vz0srj0tYQbyNKblhgdfGui3lhOIAB1MtBeIabwbeNg+JItxI5tRuXNuelLePSPC9tGy4tjXgbUXJugdXFu855IQKcM4B6eQjPcDP4tnFQiHQrMkdyGJc24aWlOHCel7YRl5ZEvI3YOM+Uc7oY13kvRIBzFlBXhO8VXE95hluAbxsGTceUmyPZj0ub8NJSHDjPS9uIS0sh3kZsnOHHjTk1us7S6CLAubUYHc6pnuEW4NuGQZMxBfv32oJLW+OlLXLg7kZe2iouLR/xNmLjLD/OidF1lkYXA87NYkR5MNwCfNswaDKm5O9W2IBLW+WlLXHgPC9tFZeWj3gbsfHfGX7czHWWRhcDzs1iRHkw3Cy+zQ0Kx5T9LRL5uLRVXtoyB87x0lZxafmItxEbN/lZEiF6LfBCBDgvX+EYbhbf5gZbsWPK3svYjUub8NKSHDjHS9uKS0sh3kZsnOHHzXUJaHRLwLnycrcIYBhuFt/mBtkxhXLqKC5tyktLcuAcL20jLi2JeBuxcYYf53TxrrNeiAHn0jHaLMX6H1LRM9s8w83g2/wgg3Qrci/jOC4t5KUlcWmGl7YNl5ZGvHmU3Lig1yV03eiFOXBuNUbb5r3FQ3rX8279wiKO2eYZbgbfZgZ5pLt8mG2bcWlZvLSDuLRVE0ZsnFsQ1cV4YRk4L+934gy+LTmmMttQOQdVoHKDqqg5VUXNqSpqTlVmWxWozLYq6r2MKqhcvrcqcNVW7VcSz60j13LJaVXgqq3aL2eeW8hs81D2oQ9i/GsAwBAXvrs2fXfOy/X9Go4x21WBq7ZqXym71pltBspebkhWJNNdWxvzShltCslyjO+tiDbUVltnc7rcyueTYtW+W7l9SL3HLxvCrjN9tpNX/Hbw0KGDkWuY7tqfxS13wWDvGY7enz56XLXSrbuhHLPNQNkuo/r26N49dUQA0117ZYg3gM7rHWW2K5yr5jt0S3HVYk7I5Kql7fNdwHkmnbdbmpWJ0tQ7AIqv6Eni3MQEbbpCKWah7LcBIPXPV5ju2rnXvQDA//5pB3OqfBtqSa7aiFVLc9UiWDXkctWS9o1dwBkmnberV318+Bu61cuPPLUMwJfrfKt/EqZQTlkoOxCA7p3PNEx37equpM/fRYfmSOXShlqKq+bhaCmuWgyrls9VS3LdXAR8TAIU3aD63dNEazzv0sUIItJ1U4jvNYWy108lQXft0DZEROPwpUPMdrm0oZbgqnk4WoqrFsOqbeCqJVt3cxEwMbHqDMv/IqJEnKQE31NEtEqhnJpA2cWN0onY7trbNBlExV3wuaPMthJtqGVz1TwcLcVVi2HVkly1Da27uQjYmFh1+uU2ADxQgE4B7dv0en6kUtNgIZS9Q9MYANNdu/8Xby4r+bFfYm2H+V4F2lDL5qqLjXB0ugRXLYZVS3LVNrTu5iJgY2LVeQhGPBIW/rZ0ydC1yjxvNYGy1zYFAB+mu/akQQnFE6cj1OGcKtCGWjZX3cUIR0tx1WJYtSRXbUPrbu5X3NmYLMul6rNn56x7b4Qyr1sEoDZK43sBQDW2u3aTJkB6QKhD895yaUMtwVXzcLQUVy2GVdvAVUtx3ZyIxSQi5zcCj08YdA6KMNtCKPvCff03Bt9d+9CAPCDv4IxqDuW0PNpQS3HVPBwtxVWLYdXyuWpJrpvrAi6IiVFnWC4BUIIyYNkDANrOUIbZFkDZN+EFAEx37Z3brwGLmw515Hlv+bShluaq+Q7dUly1GFYtl6uWtG/sAm6M6SCjzqi68YCcKe292729te+HG3ZNXaIUsy2AsrdjmqH2lOuufS7m4N//jsqkcmC2y5mr5uFoKa5aDKuWx1XLarUtGpO53KteLUXb3KNc+mxr1/cz/DCqsbv2nT257bto1D7bUDkHVaByg6qoOVVFzakqak5VZlsVqMy2Kuq9jCqoXGa7KgHUx/dmNh9Sa8tAB9VkJWc/0dmZc1qFAOppmXPqnZ/V6D8yc5o6bOzLohvSflgz3v6cmjbOZvpsDw0L97ya2CcK4MFu3bprhbo3m8G2Z/hVBaDeFlZGRGUxYTL58l/xhqVNIe/azWybNs5m+mxTXQBuM3Qs2K0bf4boZs9EG/tsV3Zjaqv2FbL74ntERPRXmMy+3boT9y1tCrM/p6aNs5k+29R76Qer0oTNuHcsJCJK6kX28UgVBlBHNKwYgNrEUKp+phgeIDMkTVh5XAYNjPYbDzz16zt/Pf84vCK2XmwD1BkHDuzmdv/rAQAEptr5zMEUoOYWZAPU3AHOAlCbGAra/BsAaMbzIYkbM0hZdoZJMArA2maNs/k+26LSYOFiHbAjxr6cmgDUxgW5ALXxAKcBqE0MTUTMM58nahFhDMncGBNPcrj/eGEwQlsz6zXuk7Wia+OWC4FXG7ZZazujDYDtsw0gadGCRXmC/Yc1nRj5z/b1n9g4RxIHqHkcWiZAzRzgLAC1iaGNNQHU/Zphkc2NMUA4RQwiNhjGVti7RKdb7yeivIbDiaisbabM66l44+xjmEREFLxBR1uCrggg4PSucO+hJfvmSEKAmsGh5QHU7AHOAlCbGKKcn0c3B75gQjI3ZgTCifoNEkTN2Ap7l86Nyyciorne94hSlsmdI4k2ztb32SZKIiLqMFCQ00tD/+2BPhn2zZGEALUYPy0JUAsOqDSAWtpQzZdeooTBs0fU5IfMjBmBcLOoBbb+GHpej4ONmP3TGPw0CfYx2myfbSAEADquzGJ+9ens6O21x4ze+0Kii118rwCgFutLLQlQC4DrSgOopQxtigWgiVwx8FRPxqipMSMQbhY1a2tfgOeEHwAA9QcsG11U4gP7GG0Axj7bOKp9GoA3zjOPWN6cXxuBcXM+2hNjH7PNAtSmOLRVgFqyL3WFAdRShjbHAgCiUciEZG6MA8IlbUVP6NIt5iUAwNioI5djYR+jLeizjf55+e6AFrX43e+d7wFAM2tfSoxd9zICgJrFoWUB1Db0pa4wgFpo6MJlvWaEmoYkMMYB4ZK2vPDUrDH625rIlkvPhsh/o8Iw2iZ9thG63B1Asm8rfvfqmnz9hSnYtnsZUYCawaHlAdTsAU4CUJsYKhuSCwDLhzdgQjIzZgTCgZICCILhbWlLgPe8B+UAgGbMLx1suEHlGW2zPtsvtgJwJWFRNR7sdotdDAA37j5ty72MRYCaJ6llAtTcAU4DUAsNUdu4kR/v2T9hYK4xJHNjpXw8x/r7+fW7zQSzhrO1p59PowG3d/h6tVxGRJQTUGRLraKR0Tbrs106Y/HpjW2+1LFgd9Hro+KTVg5Ld5DZNutLLRuglteXusIAahNDJ8Po7ImysPZiTLjAGA+Ey7J1d9l0m96JizXO5i4PR/261DW9jB9/EBKu9tmuOMm73AHzBweonMMjJJ93vJdfFPAQ/fa0KlYl+mRi3Ew8SqiTKrh9I9itchgzNadQuUFVoDLbqkBlttXvXvW7VxVr9zJVi5SW5QGjv5L6bJuKzsWmnFYpUlqeB4z+cu2zLZPZBuKX/QLb+mw/yqS0ZQ8kXWD0l1+fbVnMNhFRQWA/srHPdkW1mv4xygWAyyy5nbj7n/jK0iY3xz2QdMGtIvpsM321AQAz5z0JnzWuQ8oAhCz9YFXKHMPz+kUA7OyzrTwpLexpnZpuKyldQ2EXWA+U5s9hL7O9g2ucvbNrDuAVkXYRQJ1xn44MMIwfbVobsLPPdiWQ0pKotDkpbTsqLXSB8YDhzxnonC4dyBS57nF4OudjWfYNANpbdwQ+yWv0bTOzXfTflwHY12e7Ekhpc2uSpLTQmCxSWugC7wHDnxtbhwPx4ft1G2aXMfrZ4I1n5EiE/whgZVD95YxPcht928xsf/WOBrC1z3YlktLm1iRIacaYbFLaxAXGA84FHjqneNeTRFs941j9TPDMGQmPJqLianMZn6w1+raX2f77O6Kn+pGNfbYrk5Q2t2aZlGaMySalTV3gPTC4wEDnutbRRBQXmizQz5tnzkjvaCKiGnN5n6w2+raT2dZOLDPk1J4+25VDSptZs0xKC4xZJqUlXTDzgIHOr1/oC6DXaaF+3jxzRvgrGOeTVKNvOMBsf/MWZ8uuPtuVQkqbWbNMSrPGJEhpCRdEPGCg8/+hnvEoXj9vnj0jvHMGn6QafcN+ZrtOca1coLQk19Xbvj7blUFKm1uzSEqzxiRIaQkXTD3AdyMY6Lw5rhn3ZPQbxfSMUCnjkySoDruZ7azr0wFcqDP9yUl29dmuPFJaYM0iKc0ak01KC1xgPTC4wEDnDYJPAUDxVqF+MR8fIwCZxYxPNoDqtjDbEd9888033/gEfzPJtj7blUlKi1izTErzxmST0kIXWA8MLjDQuWZV4m4A3zYU6DeaZ30MvQtgi08h75PsRt+wmdlGWcE9wJY+25VJSouj0pZI6RzOmA2ktNAF1gNjM3AGOk/s8OEvczaz+lnzjI9Z3Wfv/mxXA6/eOfwJsNLo205mm4jGdPbyfeZzRfpsVygpLbBmgZS2ZEyKlBa6IPCAd4GBzm/lBrnIOiN3rjT3OePtXyOf8UkSVHec2a4qfbatkdJQOQeopDQe5d8Th0pKo0qjTnBSUrpKfPeq3OCjl1PXjwBEqqe44iWyfE58pP7+dJZ6gh8ZmWW4P1UF6m8cqKLmVJUKlf8HywsUa/C76ZcAAAAASUVORK5CYII=',
    '1301.0302v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAA8EAAADZCAAAAAAlTm23AAA1U0lEQVR42u2deXxM1/vHP5NNGgkRYikiqNhiicRWW6QpikYJoqj926ILbb+qtTTU99tqq99Wba2l1qriV9rSxb5ToaglakspRRKyiUgmM8/vjzt37jqTWe5MMpzP69WaOffc85zznPvknrvM89YRmJiYPFZezAVMTCyCmZiYSkdE1IV5gYnJA9WFiHQE6Ni1cFlV2Zgat/XCxYZc23wpzJWO2CqayWX6Ppv5wA0uUz8H5/9yLCt8YD1an8S8WnbOfvlpQFAd/ttf94C65bnP40e0AQDkXgNqVAaA1HI+enrC4mTm/uPrS8X6GhWdObvczfD1MeopQt2KMfJYschKvqgNrwC7DKVsu91gcOXv+qkOL33zJQp7rjYA5F3l6leurnPiLCkMhG/Pr1aA3V4quCN8rlTe3HDdAfWwYYAldxgjj2Xn+0Gvbwwg/3o5XWGVfyxNuWCSiEBSGRdXH5mS9duERaujhcLLndaSFZWwmckxSabm2rSn/HxvmL7cDkXctGvc5zu+47gPf84ajMBLRESTohD7kcXJpD+ndEDgsORUB3oh6NjbnVHxxU8tWNk1TGKlqm/8kJF1ETFqcJx3fbsMvTPqevGpl2dHqw2veEbEloLCPc0nFBDR5amdUX3aBzMHNhl3y55xWDr2Te29/3rLnifs9dKZ8S1R6ZXJkyf2DwPnoSXVR6RkHZ34+bctLLpj1zD6YUIgYqYTEf31LKpOOGlpygWTKhFsHFfxNyIiWu4jmvTNGEVEZJyr3n3TZiYXRjBRWhLeN31c+AIu8sWLUaXI9PH0AHQqJiIq7Gq0PJlElIK+jvZC0ElTI2pWRm4XWzEEXyCiOVhARGdC7TH0fbSBiAw9o1WGV9yveRYR0f2nO98nIjqNQUREd6Jr3XQsgmUDMbVnXOC3ym4v/YY+RERUMGICERnHVzhCRESrfVpYdMfI7UQ0A3NMX1uftDzlgkmV6+C5i+Zxa7LhPUWlCcc+B4CMc+pLCNNmJpfKP7LNV9zyi+5VgD9fvGNw5i7+c9yE/Z8BgF8TneXJBOCvxS0Qb1MjKlYKDncVW8ka2MBcv2m3AjtsrI7zAuCVrDa8979bHgwAjy0//Do3KG8ACJl0/d+ODUg2EFN7uvEzRxywt6kA+HCOnpMOYO7CeW0BAEP7WHQH57IRWMlNcVZkC8tTbu15cMb0ukNMq+zXxEvu6PIAsMnSRUB0eRZgbtCYS/sAAIc6CGU3q7yAdeZv70dMPVfSZGosNSs/9PYWV0lvKfrSPN2Oxq9wV6Otw5XDu/3Bk624TzX7LRafWuphn2YDAQBMDBnv2J1mfREqBwMZ08OHmkpetegOzmV14k7/DgD45nnLU241gtfe68UXdgwE9LfuIPseYMi8CeD0FL5aUcpZA+SbAbp4mPuLUpxxG8VpBSzmtFVSwDJuOtsLZesHPFV5U6H5b/9K/XC9hckEru26bFRp1jxtgCEtF4WbD9rTKYUVAKtfkFTxbSf60smexiM2/gwAuleVw1tbEGdeetBy0T65iHDIu2oD4U6k8af3O9Ti/m1Ad2Dtvd58w+0rWHKHyWUjwQ1lZ5zlKbcawTvRmP9Ybgm2Nq+xZMG8sL2prUNfBb5958HOoUNXA7S47/0rHQ/KNgO/Jpx4MGmqHjgQVTX554+OJo1kz5o1VYWBG3MA5AaJbrfu6+Q7IOcX89d2k499oD6ZKHpze9Cmpy4qWjVPGzCv9/ZXXvworI/Bjk7JrABAxs3mkipPRIm+tK9jR+Ovo+dTHx4uQhfl8HajHl+rPnaL9tngPdkh76oMxKTa2OVYBAPoA+wQGvb7yoI7eJf1C1r7AMDpJt6Wp1z2Tpb0EjwK6yXX9hH9Dtysvo6oSyIRUcOXiIhoQc0col+Ds2Sbv29wl8g4bjAR0ZNxK4guYie7HaXdnaybs2g/FhHR8tv0Mv7mCq9MJNqLJNOdrEVED5r5HCd6WWUyl4alEr3R0kBEp5Eo3CwSpm1DuWwqjniTLtt0B8jUiMwKEdHnH4kr8PoUi+y/ZbYuGEC1eSrDi8J3fKXjqEpEF9Hv5s2rP79U9wfH7kXLB3IRQ0yfPsYIO+fqNGoPGdgOP5IFDyncYXIZ0b/wLRG9ccnylFu/k+UDyZ9fXdDlDtVvJgHiVUXOlMQKQLzXWunmgpeTKgG6sWt3Agg6/QJQ1/cMO29qqg4RywBkVBVK1g0COtb6Md9cUG4VhhWqTmZwXj4Qd/KytEnxtH0bURHeUd8KJzdbJLMCAGsHa3fhkPbNSw1uv/qJcnheMF+lFZhWk6nr1n13fej5Zx0zpTIQ/kQHo92ttVnzzbfd+IaLS17C8y4bgeWA/u/6lqfc+io6HMJ9hiwAaKbc6/ccvwMHDhyqckG6+eT1SACI1G0BgCZegLdfPgs6TaUbc+wP/CGekx/PrVixqu79H4WSlslnk1UnM/FOdNr3+5EjbvBskXjagu8C8KloX58Uhwz+DKwprXK2yPEhBw/64s/doTOzFcOrjQzzPSiEcfewJk6cOKajn4OWlAPhdQN1HGjPK+w1AIY/zA0ven7YqOHPn1WrK7isfcS2G9ja08qUW4/gHjhh/jwbAEKUe11FVX9/f/+vJ0s3/8nd7fbyS+VuxQOAgQWdthrmsxTbnha+n2tYu1atWqNFd6OBt2M+PqQ2mcbF7ddUlt3SXO4jnrbxmWeQt/st+7qkOGSwZijkVhwd77cAoIv9Mu+EYnjx+J0vOo54DXyrHAivv/G0Qy1GNQXOH0UPU0/HLRmz/Kd5TdRqCi7TjTSuwob+lqe8hAhOqraVv6+Z7QMAigvoZWiMyjExMTExNaSb6+MeAOgLn2CB5jJV670mx1f0pOabN+Lj4+OHNfxZ9E6tzyrf4QUqk/nWpDXTO1YCjKL7i/94iaet6itzp05cMMzOda78kDFu6ier8o/DT583cv90F1bM5uENrvCL6dROW3z+pcWCXT4QXpm/RHd0qMXH6wJHmiCp+tYHAIDANqhZRe2WlNhlL3ituBUYaHnKS4jg8vMzPuNDVTmT/nrgClo8vh0ACn+QbmwVehgAUtCDBRpc81NQAKOzRvURFZ1tBgC654s2mWsAjd+/dEM5mXmfDa8PpAPbhIdFF65Jpm135yX/Xfacnd1SHDIHWwRJa1y45vCYz3FX7XlooRhepdnpK7lK205ODtfAvxaP/f8WL9Y5PGlfNEb5BXc+5r7dt1BL7LKa3S6MH2p5ykv8hX//ue+uJgD4SdcQQFEuV6zPB4CWlwCC/5ItfwCYW1u6ufyX668AhtkDewMo0gMgPVtFa6izqQT0qJEXDiAHuQCw1nS91hvLjADOml5smNhJZTJ9HysGcNq34GYIclAEAFcH1ZRMW51Ji9du2p5jY3+KuEbkh4ywIhRZAYAHKLR7zIbB2QCwaGRN5fDGvvnvfQBwdszzMzlzOc45WDYQU/fzJiz7oZW9TZn2RcEbNyoB/ea9t9gIAOurCFXE7pBcd4zE6Y6Wp7ykp0lEtCe67coDa19cS0S/9AwK7bGC6GhCSEivdKLLdWbM3ENEu9pPWTlpq2Lztq7vf5mQXER08NngSglHVnQNDB+QxR4KafM0KbV7SGDMPKLZG4k+7FY5qGn/8/tjKjzW8ioRvdG8fGCHV4/GVwxsv5L7qckI5WTSxtqzf5r1+/j2bx9JCEPAc0MGdvDG6ySaNsppGVavtr/PwHs2PIXZ1rdhUHDcOIUVetBIz717LbGyZVBstaBqsYN22Pc0qdn2Mf/5ZdfEftlEKsNbU3f6b6fm1JlnIKJTiY0CK3T9twPvd6sf+yf7hCPwuSGDu7d6/R9758o09CED2gYjjms4psWSA7+8Nv2q6QGV1B28yzgVVPqPxSmXm7TwC//UY1m14iuo/WnRnw7n7l3duBehtrD4JzvCh50sy9SvxkWTaTivb+JHuRUtTVtBmyXtAOPx4UnJ9vZCZOX/Dnyq3XCPR9PpY4boKAurWMPB88YGnX019KblY9/J1s8fy23cspL6NpnLtrWqYs9vg1mODpajw6TvFnPvdi38+UcnetF3WjTL0eEOl7EcHUwyRZ25CgCGX525F3nnr1bMk25zGVvxMolUd9OkJm0CLu7sPs6JRtYP1DFPus1lbBXNVtFSZaQV1Krr5Uwvnl4WxjLdwR0uY9fBLIJd0YvMKixXpZ1y0GUsglkEg2Wb9fBss94zAMSyUCmrin20ehHryc3Hls7EEFEyCxQmJg9UsumNDiYmJjDyGRMTE4tgJiYmMHYhExMYu5AJ7GkSe5rE2IVMTEyukq3vRVuC3WWkZj7eTl4553RmhTjm21LSX8W+Jvyd/nI578JqFdxmWRWd6EFDc6PntHOV6jk4fZAi1cGt1cPrRL637qa8PG3VgHWK/W9+PWQhi6TS0t45TcLjvgaAgun1Ws/5x32Wb23+oFn4ZQBY3r/OmE0eNzQ3ek5DVylSF2S8+WJrqKDfLMHumk9QKeyWyPJruCDju41aj9YGIiK61zDDvb2QoxM1N+TU0FzbvJ1z5aSrrGR8D3p1oWq2bkuwO1+bC5ncpP7PpHConU+WVHGzaRk60dOG5k7PaeQqZVSWq+PNYsCzpZvvPzkDwMWbndxuW4JO9LihudVz2rjK0XvREgZe/lWjlGhoUvbhvUVXWUC5X/WmZb0F0PT3JJPilumQoBMlUESPGJo7PaeNqxyLYAkDr/CDxfuGTy2CQDQ06ZOVFR/7bzSLp1LQvxuuOIB18aHiSXHTdIjRiWIooocMzZ2e08ZVapfgn6rdyRJj6EQMPIp+/AaRYUCCUUQ0pF6JROe7EJGxA7sd5fY7WUS0C5EZ/QxizKQT02HHnSwpOlHMstTMkONDc23z9t7Jcs5VVu5k2SIJA6/L44DXWz98JyIaAgBunzwJ6EawE2JpqOvQMx2SvSDCTLptOgS2oIRl6SlDc6fntHCVAxF8tkjKwPMDgGjfrRKiIYA24VGRE3eOYdFUKpqCRs0BEWbSfdNhZgtKWZaeMjR3ek4DVzkQwct9VBh4uqCLEqIhAP+97wYsiB9mZNFUGnoMjwEQYSbdOB08W1DKsvSUobnVc867yoFss/944a3FJ+rjAGDU8Y+yKLchGqNyjKjexcdmzsxaOWk0++lTaco8KW6cDp9VUcM7w+UsSxcPzS2ec95V1s/Bdx8oyy5ckzHwigDgcHFPOdHw7Dqg0sTEMyyKSlPmSXHLdEjZgi5mWbp4aC5uXitXqUZwoYmpllm7o4LFdnVQTQkDD0i5DRjn9OsrJhrq9QAW3gdQ1I5FUWmIR+cJk+KO6ZCyBSUsS48Zmps8p5WrlKvoopEZp4N6NoidCATUa8yXHpt1Ejv7ltffOGKIhf+K1z9sfjwhfXKXYUCl7/73RMCmyGk6oOcPY7s2PBMblfKfY7o+yWj4ftOK+2PZA+FS0JkZ5wL3PN1uFoRJuer66UiZkmI4PnYYAK8VswCgb+CYpytvbTXVg4bmJs9p5yoHf+EvZeBdvdeQ/0sgIhrmPuZzoaiBPwsnlPavxm/ci9A5Mx3O9MIulqX9huwammubd3qu7Md+sozvLEcHWI4OlqODiYkJLFclExMTi2AmJhbBTExMLIKZmJhcL8YuBGMXgrELPZtdyF5cZmICYzYwgT0PBnsezJ4HMzExQetfFxbcET5XKm+hks2shjwubVhAmE8ZdEnRysxaQ7lXQ/NFf1a9AsCYDXiomQ1uGIELxqFM8JM761+937kly8hzZnxLVHpl8uSJ/cPwqaW0PaljAxKJ6HKnteJS2VciostTO6PG9Flv9+q4weD6ZFMqHbCigtYr9+m2cJ+r+sYPGVkXEaMGx3nXt7eh0sqTteKlANSeSkSU0x9VX0x1Xy/+nDUYgZeIiCZFIfYj7Q05OTS4dARw/xRB9J+I2TDoMuX0r3hYXvs39OGO8BETLDfZLZGINmOUuIz7apwry5v3PBHRdv9RWoWwzICoSNafEjQ/0pg29AoRERmCLxDRHCwgojOh9jbEmA2ex2xwZgRw/xSpZ7p794N6qLDce7BBVh5gWnH7z0m3fEb3BYCEY5+Ly7ivGefUCBDx/b7aqNHKJOOcpSJZf0rQ0aa68NV1AQBZAxsA8IYXgKbdCuxsCIzZ4IlDc/UItB6HMoK3PJkFBHZJO69aX1+EysEl3SGLLq/8agHuVBd7NPLJJotFsv6UoHzhZ2TpLUXlzdPtbIgxGzxzaC4egdbjUEZw7cIiAIFQP9Pu3wZ0l5QUp6fjfpo4DZgh8yaXg373bfPX01PU7d9ACwntQdSa/tYdZN+TkCDMufP52hm3UZxWAIgNmHESfJG5P6ac+KK9rCfM9xWnZejEN1SccRvFV+4AKPyrWA1VAcZs8FhmgxtGoPE4lBG8/2Y1AH/4RKpHMIA++LxzWPUvsPPxsCeP/NSq2sxZ0354+q0ivkpq69BXAexpvcu4dqaB+/rtOw92Dh26WtHcxfUJoyFKjy9qbWvzGksWzAvbKyJB8Lnz+ZIDUVWTf/7oaNJIEgwIOAm+yNQfc0580V5QJMw/NvTozqFD87nSJ6JEXW1fh2uI2z1l5Cjj0kUH2y5UoirAmA0ezGxwwwg0Hof6JfhRvKFkNtQeMrAdfuTu8QwJyCBj8++IiBpXO0V0r1VvI3GsBqIuiUR7vI8TbQrYbvpKDV+StHYRHTesX/hK/ArudoFAexBaM0b0O3Cz+jphmzl3vlD7ybgVRBexUzAgxknwNrskSnPii/biJN6Y2EdBsFjEf+ySKNp9wkmi5QF3xd1hzAaPZzY4PgK4f4rU70UTET1oFl+gRl0xXO3GRTAVtutuWLSViIiih3M3fDeaI7hXIhmbdCei7S1S+UJFBPdJSzs5PuYgERFlV3yNiAwhCyStRbeQbttb8QQRLRHX7h5qICr2nSsY2FjpGNEWXBBHcK9Eul9rGhHRKewgyV5ERNKN1iKYG55p9+eI6DCOi7tTdiKYhiLilMR9vPdcHsF0wqfpA3pZ7lftDDk8NLh0BHD/FIGI1F+pmBK6WTUtkFfYawQYzjaH34aoIW16im/hYWui6Ov1cz0AxJ+0fPIPDAcWDHnqeBNw6fEB8LQHvrVmkGxrEx7VNP7ZMZLaTbwAb798odnEfrq0Pw6bcBKChJz4Tyn3km4sWdzuTQH4I1/Z+TKhKWsE8gCAKhdGmbwHNzAbpifPdsSvZWZorh6BluNQfaty2T8/WbrpGtUUOH8UQK2561qINwQFXRR/vYTqtphPfPAlADntgW8tRLqNz50vrs39oRHdRlLiJADIc+LL9rI3Yb4/xI0oOg/GbPBkZoMbRqDlONTOwVtOfe2FM15N1Oo/DuBIEwAPdk8ddKKmsCE3r6G4XgP8rfKXYbS8JBh/AJDTHvjWdNJtfO58RW2xATlOwmTTak5859gClrsDxmzwUGaDm0agxThUzsGHD831Ar4vB3VmA+iLxgAlT3sveiB3/7kIAPZCvKZGzcgTAFBofkrrrweuQCWCCfeLxbQHeWvCNj53vowNITEgwUmIbFrNie8cW0C9O2DMBo9kNrh1BFqMQxnB54feGTf2pReWhqszG1Dwxo1KKJ5SoY7XqrMTjQCw9xZwP7lfX5hYDdDnQ7fk8E8AvqjFfQVaXgJI3FoOAIR5372C1Zki2oOotaJcaep8Pne+qKRID4D0BrMBCU6Ct6nPl+bEF+0FQJYwn7Mq0gMTwYIfCb+7HoAeBnEHwZgN8HBmg1tGoOk4FDfRWnHlzYkoP3IoX5qSEIaA54YMGdA2GHE0tWFAS6KljwW2n00U/fw7S9fGTS8kOpoQUjnh+NGEkJBe6XS41dT1720kMn29XGfGzD18a6cSGwcFxycT0aKAKWlvEtGu9lNWTtpKotZ+6RkU2mMFibZt6jF17da35wolB58NrpRwZEXXwPABWbyBjbVn/zTr9/Ht3y4mU5GpA9u6vv9lQnIRSfcyybzxeEJIcPzHgje2DIqtFlQtdtAO4kdi3j2sb9bkqKCWr4g6X2buRZ9ObBxYKX4aidwneM9lvTgaXzGw/Uru1yQjpH7V8MVlx4cGl44A7p8iaPML/5jIFdfyGqpcUN/KjhCd4vWnw0NU9r69jRKCATPtQbU1bps4d76IDSEzIMZJSG1azYlvf8J8ZQfBmA0PGbPBnhE4NVeOTZFGzIaYyBUaHrPatgaWowMsRwdYjg6r0hdr2SltW2NiAss2a1Xbep3f8pxmT8u0bY2J6ZFYpjl1+jfAG8VeWqXb0rY1topmq+iHfhXtdKIqb2jQiItaY2Jiq2gmJqayLJbxnYkJnpvx3QdA7B7mizKqGTMeqV642JBrmy+FuZoBTZ4HM4HdyQK7k8WYDUxMTHAVs8GuZPb5aUBQHf7bX/eAusrfG2ekZj7u5LvoWiMVxMwJi1gJm8EUKso6mxkcC8ZsKLM23MRs0NSQ8pXsu7OnvP7KeWeS2V+b9pSf7w3Tl9uhiJt2TVnpt9FeE5wkLciQCk7LxJywjpUQV7K3w6df8k+yiyTBmA3upSq4idmggausMRteySD6yG+7U8ns05LwvunjwhdwUb1Sc6sRXDIgQY5U0EDdEm3ASnRLdBwNEZdk+/gYs6EUqApuYjY47SorzIY1S3cDI4uS4Uwye//INl9xS1y6VwH+VgAPFlUyIEGOVNBAvrZgJXydQEP42jE+MGZDKdhwD7NBO0PKCK5OAMrjLpxLZj/m0j4AwKEOjt5nKxGQIEcquEglYiUcQkO4EgDxEDMb3GDDLcwG7QwpI3hQ3gDgIHrbnMxeDYEAJAVwp4FD7SFjKQgsBquJ8U2kBfNuSjNypIIY5CBhKkixErwtoR9mygNswUrIxyNDQ6j7RZSM3/hXgVBd0jXNyAMPMbNBaUNrE+5hNmhmyEt1oVf0Scw02JjMXg2BAKDCwI05AHKDTEsEM0vBzGIAALxbPaxbhrmtRe3CGv0PeL5W5AoTacG8m4oZKVIhTAJyEDMVpFgJ3pa5HwLlQVVyrARk45GhIdT9IkrGv/3D35JGkqm6pGsakgceYmaD3Ib2JtzDbNDMkMol+JG3Wr90z45k9goEAtHNWbQfi4ho+W16GX9LWAo8i4GiJxCdbLJL3FZ2lTFEZIy8zQMSRAgGFTPihOwKkIOZqSDBSgi2+H6IKQ+9Em3ASvSSdUyMhlD3i5CMv3ubdfwgOACE0DW1jP2M2VCSDXtMlClmg5OusnInC2g7e83V5zPVwr3cKgzj0r4VvJxUCdCNXbsTCDr9AlDXV5Zir0PEMgAZVU1fg/PygbiTlwFd0OUO1W8mAQDOLv6tq7itiqM23AeuDq8KIFC6mwUz5se0UxIrAPFea80Vr7YAGt1PA4CAHs2B8u9u+U5si++HyIRMlWNaP2nMbqCTtS8bj1mBFv1y++RJQDcCANIG8IMIhLRrokrOq+vQMx2SvcT91rR5a1I/SFxkwyUm3DECzQypvpOli1iz7RmDejL7s8kyzAFECITszMzMTBMuQTfm2B/4oxm/Z+Kd6LTv93MZKs2FO+JnBUrbejFnPbBqGNR2U/AZxPo9x+/AgQOHRCAHnqkgxkpI+91MYQIyrER4iwURT52Tt29lL3W/tAmPipy4c4zlQcRhq7iSBpoCAQhw4MChKhe0bR5WiQdqB4lrbLjGhDtGoJUhC29VVm597FfYnMzejEDoHRoaGsrDV4b5LMW2p6HCUuCzz+08EzBR1lb97l9An1MVarsp+AxiKUEOiupBQRel/Q6xRnkoASthCxvC3ANRMn71QQQFXYS25IGHmNkgtuEiE25hNmhjSPFWZVHH4iN+QAguOpDMfkOh+RgFqvVeM8vXm98kZinwz7+6T2zfoecgaVtj+5668qzQohzBACfICbl5DaW2dLaYsICVkO3F4yjU/VJiMv7cvIYugio8rMwGkw0XmXAbs8F5Q4pz8INjf6QD+BstIGU22JLMvkZ4eHh4db7y6KxRffgtEpaCcOHYNnnsVWlbvWt+uU84yFR3g/3kBDMIQtnvkkyoYCVke4lxFOp+sZaM39w110AVHj5mg8SG5ibcxmzQypAigiv03FULuHa2SycZs8FyMns5AgHA2VQCetTICweQg1xIWAo8FaFID0wKSsySJMb3+deqcJ0ZkCDZTWlGhFRQghx4poIEKyHpdy4gNWFiTpSAldDrpXuJ0BCW/GJOxi8q4wAQIkaFplCFh5fZILGhuQm3MRs0M6R8L3rYnIMH2va6KWU2WE5mr4JASO0eEhgzj2j2RqIPu1UOatr/vMBS2GpiMfzSq0Ltvuk/VgxstFCSGP96YCaZAQnCbvtUSAtSpIIc5CAwFURYCaHfZiaEQHngmBMlYCVMlURsCAkaQtUvfDJ+Udk2U3Wha2oZ+xmzoUQbdpgoQ8wGp11lldlw8hRFNddpnI5fwlKw2tbftW3fDbaRE6QgCHm/SzIhx0oo9pLhKJR+sZKMX+iaWiUdYzZoaKIMMxscw4ZoxGzwAJVhEIT1rulYjg6wHB0sR0dZBkEwRgUTWLZZeCoIgjEqmEqb2eABKsMgiJK6xlbRbBXtamaDB6gMgyAYo4KJraKZmPBoMxvqMC8wMXmg6lh4HswElvGdXQezp0lMTEworYzvRllo5/7j52PUe9eD4aKfS1NhM8EzM767w6yLbbjNc1oasnAO3jNIVnBr9bA6zf63C9AvTajz4mbHDV7p/A0LMpdq75wm4XFfA0DB9Hqt5/zzEJl1sQ23eU5TQ6qvZOfX76UoS4GpLK+tHRmqjXNLTORunEtMD0PGd+fNwrU2XNs83D9FlvJkAZ+qlPnzP90PrG9HhuqMcyUmcs84x86beBgyvrvDrIttuM1z2hlSjeAjda22ak+K+U0lJzrfxELu4cj47g6zLrbhNs9pZ0gtgh/83/Ml7yjJZ04Xd9+GOLs4n02dz4Yu5Ec3JToX9haqMGmlUsr47g6zLrbhNs9pZkgtgj9/reSzrCSf+Z7Wu4xrZxqE7OJ8NnU+G7qQH51LdC7am6/y5ZNhjf4HDK3VdAWLQHhoxnd3mHWxDbd5TjNDykvw35cRtVXeyToNPhv6ECJJPvM93seJNgVsF2VFN2d1N2VDF+VH5xKdi9K3m6rk1BpJRIZmt9n9K8/N+O6kWbjWhmubh/unSJ0+WvS6waYI7h5qICr2nUvGJt2JaHuLVMqu+BoRGUIWEEW34OqawnNjpWNEW3DBjEUw7y0gD2YF5RJdWMjCVosIpqGIOEVEwpzsrXiCiJa4uBfOmYVrbbi2ebh/ikBEyjc65r9s43tafO7y6+d6AIg/yWUXB1DlAkRZ3QEAif10aX8cFudHV2Y+Hz3z67H4+g22BNZEU9YIGd8BVLkwKjyqafyzYx4Csy624TbPaWNIEa3nCitnZ2cX67PzZBuCYMomUWyQJmC/hOpQZl0PkeyszI+uzHxeo+9CKtBXYMEHD8747g6zLrbhNs9pY0hxDs64Pg3AuarT6snOhmGVM7gPl2pJNzTA31BmXRfdDFs2usTE7ctGAxgXd/ByEos9eHzGdzeZdbENt3nOOUOKc3CX+fPnz59fIXL+G5BmfNf1Pc7led7YV7pHzcgTAFC4SS3rOpcN3WpWdXPC9NhGC043ZzEHD8747lazLrbhNs85Z0j9mteQnwvIMr5jTrUJAHD8+pPmfO0gvQG6JYd/AvBFLUnWdVNWdy4bujg/OpfoXJT5nE+YDt3Y9a1YyMGTM767xayLbbjNc9oYUv198LiTZ7xjur2F+21brhYV3xjr+6LX7tyPygM4NHu/rtOU8ytTqrReHHzk5WdanG+SCGD31K4Nz8T2/PXz/f7Rg4YDuBI3UtelC/7v9ZebH++19ESXPh8cQPvlF8V7m6oAyI46X47Fnha/OT0z49zfvtHtZkGYk81fRjetuL/Ga67shbNmbTHkhA3XNm/nXGkxRXbni6aLx/VtG6ptuZUd4aWedd2UDd1aVnVzwvS7C6exoH14Mr7bb9Z+Q3bZcG3zTs2VY1NUpjK+51xuhY8GhrOgZTk6wHJ0eGKOjg9jcvMesABmYoImOTrcru7HD29/l80HE5OnZnxPvxHpy+aDraLZKpqRz1gEswhm18FMTEyeIO8ZAGKZH8qqYh+tXsR6cvOxpTMxRNSFBQoTkweqC2M2sOtgdh3MroOZmJjgEc+D8385lhU+sB6td/1vAB0zdSUts204GLOhNJgNAHKvATUqA0BqOVeBPVxpwyOZDSp5soZ+euD3zZN3KpO3L64+MiXrtwmLVkc7kTfmcqe1NuSJd9DU1z2x6ZHPsrPipQDUnkpElNMfVV9MdV8v/pw1GIGXiIgmRSH2I5cYcsYGXOo5uH+K1PNkUTUAvtMVXAbjuIq/ERHRch+HItiEZlAyG1SqOmzqLovg0mM2ENHpAehUTERU2NXoKkOO23gomQ0qEfz0gneWpClrf4pVpvhKcCiCb3MJ7YzH7pVY1XFTeSyCicj4DBYTEdHMfW7uxelFEzCHiIhedpkhx23ApZ6D+6fIQgQPUa2cHljXYPq4w6EI/uIlW2s6YYpFMHep4l8pnYgujCW3R3B+RLmzro5gR23ApZ6D+6fIMjdJRWvv9eLrdgzkfix8uECgN1y5A6Dwr2IAKE5Px/00Pl+XKR09j2ZQMBsAGNJyUbj5oEVTPO9BSGkvQCLM24DCG+b7+W7DFIAxG6QKWKkfrhf/pPxwgYttaGri4WA2nPr0409z5IU70Zj/WG4JgF8TTjyYNFXP8xdSRo4yLl10sO1C4KdW1WbOmvbD028VidLR82gGBbMBwLze21958aOwPgZ1U2beg5DS3gyJELbh8oDkve9v5+bUjZgCMGaDVO0mH/vA/MV8lLjOhsYmHgZmQ+RaI30X8ZfshB2F9eKv3ze4S2QcN5jIzF+YcJJoecBdImpc7RTRvVa9jSKKgzmxu4LZsKFcNhVHvEmXLZgy8x7MKe0FSITAgkgN2ktEs7GJNMEUMGaDg6toogfNfI6bVrjio0TDVbTYhj0mHhFmA50iImrVT1a5Nb4RfbtfaxoR0SnsIDN/4TkiOozjRBQ9nLvrvFFMceAjWMFs6N+MiJJqWTIl8B74lPYCJELEgmjXmYgoDZu0wRQwZoPDEUwnfJo+oJflR4mmEWy2YZeJh5LZoLKKbg4AMd9lSEvDkW7+nIWT1yMBIFK3BeD5C00B+EOgMMRhK37P8Ttw4MChKhdgifgABN8F4FPRkikk3olO+34/ctAmPCpy4s4xuH6uKYD4k41E224d6Wp+QcVslN/hUdQUCEAANzujZfLZZACQHSUuseECE27znDaGlBF8ZB8ABOGstLgHTpg/z8afHHPByy8VZv6CnMIQFHRRQnEA1JkN4zPPIG/3W5ZMCbwHPqW9AIkQtp2H8CfAzZgCMGaDTG/HfHwIgOwocYkNF5jwdGYDEnLy/IAiVJYWJ03ZWmhKBZvtg/q4BwD6QitvteXmNZRQHGBCM8hV9ZW5VW8teM6SKYH3cCmAS2kvQCKEbXVFJ/9SwhSAMRv4g2pV1PDOgE1HiZM2XGjCU5kNaLHID0BqxcaQMBvKz8/4jI/DYWgVehgAUtBDtdUiANiLnmKKgxnNINfuzkv+u0wUwDJTAu9hsSmlvQCJELZda/QbABSUIqYAjNkA4p7nNX7/0g2gpKNECxuuMeHZzIb+jQH8tfdTHxmzof/cd1cTAPyka4jyX66/AhhmD+wt8Bf0APTcKnrvLeB+cr++YooDj2ZQMBvqTFq8dtN28eMriSkR78GU0l6ARAjbKi/efgaghShwO6YAjNlg1tlz3L8TOwGQHiWuseECEx7PbDDMrBx7ftao13WQMxv2vukzvv61PbHPA8D2D56uvDVqqq9AbwiJ/mr2tkv1O85DTER4/YClHab5CenoeXpDyn8UzIbcLnd99BnF/b4qr27KzHuIWcqntDdDIszb/uN9YGrfWkfaJIV22aAFpgCM2WB/L1KmpBiajR0GAFdmLZccJZoZUtiww8Qjw2w4dySkfTXVPVKPZdWK538K9U92hIXfJsZErriW19AHUoqDGc0gVUGbJe0A4/HhSckWTPG8B3FKex4SIWZB3MhsYjxXpcpjWmAKGLNBo15YPko0M2SzCcZssFUxkStsr/zd4l8AAAt//hFMLEcHWI6O0s/RoS+2o3LUmasAYPi1BwtYJqYykPF929wdj8V+0Njm+ikfN2kTcHFn53E6Nh3sHMzOwaWf8d0AbxR72XNyz0grqFWXZexiEcwimDEbWASzCGbXwUxMTB4jlvGdiQmem/HdB0DsHuaLMqoZMx6pXrjYkGubL4W5mgF2Hcyug9l1MLsOZmJiQhlnNrgjHT8Y2gGM2QDGbACcy5NFhvUzpk5PJ23T8ZMLOA8PP9qBMRsYs8H+PFl5T08votUJGqfjt43ooC4LnIdHAO3AmA2M2WB3niwaWfU9X8w8rNgQN2H/ZwDg18QFrz9mnLOyMeHY52rFcxfNawMAGN7TTmu+eKjV/5mUZQCAT5ZUcbNplx4krrfhNs9pZ0gZwbs2vgHgm++Vdd+PmHrOVSPaZPWWW3R5taCfXneIaftrYBL7a77/5AwAF292crttVx4krrfhNs9pZ0gZwV9UbAkgpj1sTccv5S9AFeNgqTZfyhMd5CgIriLHeRCTGgCb0A7CHo8W2oExGx5hZgPtrvv31A/evgBb0/FL+QsAVDAOlmrzpTzRQYGCOJo0kkycBxGpATaiHcx7PHJoB8ZseHSZDdno8LGBrtTcYXs6fhF/gZcc46BeW1RqygevgoLYyXMeBFKDjWgHYQ+PRTswZgNjNth7L/oavK8S0Yh6BTan4xfxF3jJMA7qtcWlXASroSDmmjgPIlKDbWgH0R4ei3ZgzAbGbLD3XnR5hIUBaHHlEGxOx2/mL2RnZmZm5kCBcVCvrcy3r4aC4PNAi0gNsAntINrjEUQ7MGbDo8psqOAXAgCPyZkNsJKO38xf6B0aGhqaCAXGQb22Mt++GgqCv+4VkRpgE9pBtMcjiHZgzIZHldng0yyfi5sqDqTj31BoDjyJ1GvLS5eNtpaBX0RqgE1ohy7CHo8w2oExGx45ZkPC5SIAtxEDCbPBpnT8NcLDw8OrK62o1xaXckQHaxn4BVKDjWgHYY9HGe3AmA2PHLNhbPlfAfp1TAMZs8FyOn4Rf4GXDOOgXltcyhEd1FAQBhPnQSA1wDa0g7DHo4h2YMyGR5bZgJTxb4cvzf0iSMJssJyO/5CYv8BVVsE4qNcW8u1zRAc1FESV1v+afwDtl4cKpAbYiHYw7+GxaAcdYzYwZoP9me6y92a2bg6N0/FbqG0uNRMdrLXKkxpgK9qB38NT0Q6M2cCYDSxXJViODrAcHWA5OpiYmMqeWAQzMbEIZmJiYhHMxMTEIpiJCYzZwMTEBMZsYAJjNoAxGxizAex5MNjzYLDnwex5MBMTHnZmg5G81Srmi/7CeAXYZSMjNfNx09vaOaczK8Qxr2unPFNetMrVdcpZupvh62PUU800IKgOX/zXPaCu1x2hXqXyDlk+0EEHOthR0QVOWWczg2PhFDrDtUOT99olnpMDJlzhKmUE73+nZ2h5ABgsOT/Xy+pSzW9PWkTHB7f2hl+ya67TVi1/1RTBN79e9UwccGXEuOdZ9GmhjHX791UfU77o7JkuydXks5S28dC+ikmNE789vJ/+epzbIb1NRtyTL+YuPHSy0pDyKLx+9NqnEx0xbBj5pw6nPumo6AKnG2tW9inpsDzy9U+bwktraPJeu8Rztzanrg08WR/A8h0nYntOcomrFAl+Fpk2xEpJLMEXiGgOFhDRmVB78QvNJ5g/dksUIAxWQQ1MtuVeOo1BRER3omvdVM7SSfQlIkpLwvum6gtfwEUiot/Qh4iICkZMcKgXKf2J6OP5ii6YFZdkPzoDrh0aLLauRfOwBTChmausMBtSv/x53/79+2KXS0qzBjYA4A0vAE27FdiJX/CVfTRBGDLOsZOo0/KHNwCETLr+b+UscR/hH9nmK26VSPcqcElUAkzLL/856Q6Z3fE0/z9pF+zhYviW6tAkvXaZ52SACe1dpYxgrxd7dOrY8cpo6Uk7vaXoS/N0h/ELEgjDJhZ/2qke9lmZpTGX9gEADnWQ7KQvQuVghyO48FoDRRc8b2j1sA8ubF4dMKGdq5QR/AoAXDk0RPY3QJw4oFPxrTvIvgdADEMw8REE/IKIkwDkXxUl8OIgDEJNJueViwjZLIm3JgVwmJ5DUhbH/m1Ad0esFaTXBQ530Cm6ILkp+leB7FDQCweOCJ1RukPLRQRc2LwMYlGSq8xBY6urvFSy0gHG12ZL5wZPRIm+3O1dY8mCeWF7xaAEno9gxi+INgGFHyzeN3xqEb9Qbx36KnhQw5dPhjX6HzC0VtMVLAyd0AbvydJZal9H9KXCwI05AHKDpPO6H0AfR6wd7CgsoiVdEGn7h79xbA7zobC1OX/giNAZpT20Dd6T4UrPSSAWJbhKCBqbXaWe3OCbZpWs9alXz0Ypb9RfeAtYvfHXRq1vDjzuhb3xR1shb0iHpKRGsV8A4k0Afkh5HEMGDdjMeaHx77EA+JrPN33yDWBVyx1VWRg6pIJbKDq3edump61VGr3im7HAd/2EkqND9deO/Gi/uc/OADjtOwbY1nYbyn3ma6ELaXffQXSDF+JEh4L5wDnfZktn4MNSHppNrTvvueQts3q3smrU7CohaGx2lWoEF71zoITr2KDLHXATQHBePhD3v8sNaHx8KyCwgZCHzrwJALo8Dni91fo7PslVoPiP3EsfzQ3C5XEsgB1U6jogcOjnflYrdYhYNhbIEDm5zRrj9X85YC6hHYAhK3yRceJNwN/XUhfSBgB1fc/EiQ8F/sAZGdUZQNLbpTs0m1p33nPlVrUedrycNaNmV4mCxlZXqUbwj7qwknrVjPsnsZ8u7Y/DyMH1cz0AxJ8Uapg3AYAfAET7bk1U/SM38+ux+PoNFoqO3uix5YGubsxbfzT/o5n0EirsNQIMZ5vbZ64ecLtmJ2B9z3bKLmQXA/CtCDFyQ3woNAOAW0eSLS8A3Tc0m1rXwHMtk6cnz7bJVZKgsc1Vqm9VrqhbYqdCZDAEBVFB2CQ6c19UbatG34VUoK/AQtGlGuazFNvk68WopsD5o/Y3tjsOwK6nVLbIsR0G2aEQAkjRGWVtaK5onodYlOgqSdDY5iq12C7eE1/yHyZIYQhPSIkKy0YLm3T8LQDKbahsaNloYFzcwctJLMZcq2q918zylb8w+ziAI03sa2j+GeC30LHApgfrAP+PpY8q1bAd4kNBxz1LyS+TQ3NV8zzEAiW5ShI0trlKLYLP3QvhbzoHWE99mffZOBMMobyJj/BTXxN+QdgU2BFcauvDxT2lu3M1gdhGC2oOZzEG16IURm8e9Ynati+22ddY95agnz/X4dKFMQDKyd41qGHtKAk0EQRqi9AZZWlo2jdvBky8WV+59rTBUyW7Sm0V/Q9/o0nCbACAByg03evKBQARKKGyQFTg8AsihgKAlNuAcU6/voBeD3AQBr4moBu7vhULMzia+T8HFmYJKOL+dp5NJaBHjbxwADnIFQMDCt64Uck+gw06dgyO6dSxY17fjh07dmyt2gUJckN8KHAHjk6Ezii1oan0WnvPSSEWJblKEjS2ukrlhc4fMIX7kB85VFS8ZVBstaBqsYN20C89g0J7rCAi2lh79k+zfh/f/u1iOtxq6vr3NhLR5TozZu6RbKL4q28tXpM4vZCOJoRUTjh+NCEkpFe6uSZRVvgD9g60Q+9Fn0psFFih67/VZoloW9+GQcFx41K7hwTGzCOavZHow26Vg5r2P5+SEIaA54YMGdA2GHH29+KTBUT07FkLXSAiOvhscKWEIyu6BoYPyBIOha3mA2d/5083vPktQvtbNqT90GCtdaebV5mro/EVA9uvJCKiyyNscZUQNL+U4CqzSbVf+Bet6VXN1j/IYhgCT1Qw4RfEm4Cr9xoqV+w8qOHuwmnsZOpBv/B/Zm4E9M1SdY4cJbzE6Az2C39rnlJ1VRliNuRcboWPBoazaPWcCC5skarD/iWrWI4OlqMDwIcxuXkPWAB7kg4+qQO2xzNHoEzm6HC3uh8/vP1dNhGepN97AjgzljmirCyQSnmpln4j0pdNhCetoknH/4+tokt9Fc1yVbIIBstVya6DmZiYwJgNTExMsIvZwFbQTExg5DMmJiYWwUxMTHbq/wHVbdo+qnNtLwAAAABJRU5ErkJggg==',
    '1305.1199v4.1.png': 'iVBORw0KGgoAAAANSUhEUgAAA8UAAADnCAAAAAAS1asQAAAyZ0lEQVR42u2dd3hU1dbG3wnpBWIgFIEQWuihhA4CUpSiQUAEJBakK1wRrCiC9SKIwL0UBVREQAQ/EUSkXBCQXqQXQRIQqaGF9Dbr+2NO2afNTGKQZLLe5/HxzJm99t5r7bUy5+wZzs9GYLFYRVpeHAIWi6uYxWLdWxFRe44Ci1VE1Z6IbATY+N6Y5VKcJYV3XfiKmsXi+2IWi8VVzGKxuIpZLK5iFovFVcxise6VvC3O3zjj7V0iNyenWtlCNNnfTvX25yUrDEo5BCCgsRcAHE/zRXaJxq5MkvalpDRskJ/BLq29WOHJ4AKZ98lzKSlPeRdgIPLvFQr4Vx8go7Z36uQFdOq0kQqPjnphILHukTRZcqZTJy/gSTsR0cstgv2bP+PS/kgTYHp+Bn63xHMfo1nBOPFuRSC9IKOSb6+Irl4tECMI/5koCChcebQKaM7VVCiqmIiCwoBJjsM3n3Wrh6h85fv/0MT+X/hkFYwXmwu4ivPrFRH9a2KBGIGIitJ9cfcXOszki9lCo6/CMOkbx21ZCbcMQvI1zGY0tw2ats6nYCYdUuBhyHePBwrOqChVsfesX1py8RQa1Vjpg0E77/owSQhC0NiOHhe+PTsKzsjlnX7uoZSUyHrJ+zPqRAIJx4MahwJA5t6zXhHRYdJO2O/XI2t7rb3ZLwDIPHU8pG51xTr1ZEpKVI2Ek5Vr+wFIPZGc3DZkx30NbQCyzxz3rlfd8Vc8N+HYjUqtSgJiBzd2Xr2v1eFu6sGhOyllmssT+i25dg3D6Kx/Uu0WPJP52J6qUp4cTkkNaodrCSmpjStbZA2Qc+aPajV9HcfySmuyQpKSHJkHr+DSbjQV8lS2u3LZl7Iiytw852f3i7LIIfGQLp3MbazvRJ9NzjNRsRbTX+uVPPPLV3yRVbYiLl73zmzkpa8LpG0cgr924/4IoTNNIMyKSjByb3dLuS++E+WFiV8/9EFXjE4bNOijsqGbiehU1XJz5rQPHG8noqzRPoM/bFKh/WhMIfuysk3++15Y3G25j9+qAd1qPf1C+ZBZOY5X6zv2RvMcojVVWkyd0qDeNiKirdUrvPpBVOndJHbwRfSs1fPqIFs9iPbGQGlCS7p98DDG6UZn/aP3xSeJ3gTq3iKaONixLJ2IFpQDVlhkTQy6Nhk6peN983NJXGkxKySpyXGpTXmEtWlzR3lLtfu2jS/C19HGQNQYbZFDYj7ubVX+5YntewLp4nltNrnIRFli+mu9Ume+tDlQbSHRHD+/6AxDXdCMNn6o3KbNHLEzIRDmRaUa5X13qwc6vpBL+4GHthO9gBZENBVIpayq+IyI3kMfoqvwTxh3kL5Gg1Sin6DuWNqro2o80YUy+JCI7NXwwJbVwAVahVYZRMlRXruItiHoFNH9eJ6EDq6WWERE8chWDoj6Ofaoe+DBUbm0B/hDOzrrn67i3L5AlyyaONixLJ2I6BKwwiJrYtDoKlF2a0wjcaWFrFC2MdXkoBfQRxxWTLHFCNxPlyIXkUUOiYeHfIP/IrI/D6RrOxGzyXkmKnMQ01/rlTBzezP0ICJ6aCGZ1AVRdUzUd6YGwqqoZKO8V/FABKQTpQFDiWgOShHR2UffJqKBeISIWuBNIgrDEqL0MHxERPaK3klKLzEYTUQ0AX5XiCgG7Snjgy8oq7xjZ+8VRBNFYyARfdp1t9jBSdT/NpFoOSkHRMMdVeyY0B3gR83orH+8iimtBTDC7qjigehEROlSFZtkTQxeJCL6HP5XtakiZ4UkTXLoqlibYhNR4USjGUQWOSQ2boo4IqIVQLq2EzGbnGeiIjH9NV5pZv4dcIToYJUs07pQClLsTAmEVVFZVLFb34A38Ad8gGYAfJABoNrqnJNHLv+OWwAi9twCsu+gAnDiJrL3Ayh38Wxj3QZaE2Tu7wEAbeA3Hvj9CioCwP04cpOOoB6A4cM1HTRsfLAfqnZ7HVHygXFCGZrRWf+8An5o8eentdzMGkk1kbG/uy5VHFkhSUyOMF2nWruJp7+JHvsigBumOUTqYY0DqGPRiZpNN5xnoiIx/TVeaWb+WM0zU77GlLE+FnVh2pkUCJdFlcfdLWEzXd1TT3x7SaWnW+7cDwCTfl57NuLLnMc6AJeBEyUA9O0Xqe8iDPgLAFAaAJAAlAAAHyDeD5B2BsQOvDa9v+h6wpxV+yooB2a7+8LorHug8j+1Th7bsq5bWSPJB7igT5XSYgMxOfRVrLWzLdhw44zdC7hkmkM71cNzBH+LTtQZXnKRiWbpr/FKM/MSrwz75n375vlO60LfWWlDWjs1zksV65Xa9vTgOb5YBgAIat1yGFVePMAGNAA6DjY3uQDUF77aqgmkAUAqUMPPy54o//lWO0g7O+3j+M0L9i4aLR+8ZtavMDrrXqj+8h72XUIV57jOHqC+PlU0X3iKyWH4gNfY0ZuNzqwcPxmoYZpDwqGflz3NohPhuzPnmfiaWfprvNLO/Km3r0yjkUFO6iL9+S91nXnBZVGlP/9lgXxfvP00XvEFLgIA1mRN2rT5q4FeACq3xFYAoJHnhdYZALAFlcTrgZp1cRgADuPB0ICHsZkAXH1B7OBo5zRb9aFbgq4pB6ZTEUZn3RN1/Y904A8C8LuztgQAu1C5sUWqGJND/57Wbtq2lWuCP/oSMM8h4TCwE44CQC4s89SqF0MCatJf45V25v4vYcGyUVbj+SMbaesNncFpUUlGef6mSd61yAa+IaL58CM6BnxJdLABaiQS/YDanbo+1n/KESL6o7TtB6KsSe1yhN2t0keJtnj7bHG8kn6utj+w1AmiHb5hZ4n+LINpdkrrOUXsYBeeyyL63WujciDsbqnbKOLorH9ud8t+02erXf5VIAYTEX2KaKK0AQH4ONc0aygGFY8Qnajks1WXKjG6HzGKyaHfoxbsMv5T6jzRGi+fLRY5JB7Gh/oeIDpfEzhu1wwuZpPzTJSnoEl/rVeamdPtko6dL7O6oEHoSasf1XamBMKqqCQjd/eoVwWWDAwJLBm49FZgYEhQ4KnpQYHBQe0oJig4MHgerW/m/fCjz17pFoxPKLeR9PdgKhFdfyGicccmr6aQUMWPPNajc7nep4loe1BgSHDgViIi+vPJitF1I4YlEhFdG1qhdrsmC+xiB3vKjox5rEujlepBemBQSGAFkiY0NTgoOKijdnTWP1XFh/z8QwJ8pV3l7L7jiIiyR4e0frzTtnJAWzLNmpjBp7p17RrR+zSRuNJiVkgSkiMwKCQ4WFxcxe41v2DfSUSD/IJDgv4wzSHN4aWBVWJ7dnsVwGThvDabnGeiIjH9dV6JaU00Kei8btKCH91sTRqcEjvrowbCqqhkI9265PcZmGm3yvgBAFK6XHqvgi0ncf9c+82SAJCaU0ps2PTAizMyUkqb3bcmlVD/uVmST6Byh5FTCoA9yz870a+0cGAmw+gs3LtnYObc9LkPl3wCAqx/V61JBV2qWCaH4c7a3M6QQ/rDGyV9UpMC/P1tzga3zkSz9DfxSpg5ZQQ4mXRqarjNpDPnRSUbadfF+oraPa3HGyRdeJv+7kL6Ou0uydXorLv2fTGrcKR1gfybppjKK84DwPFNter98x8Q93Z0FqtQpPXff6r8hY//LyoiN/5C3zfKGN/c+VCGF3xT7p6/Tkdn8VPli6bykNZyBf/t9cm4bCsbaPqO3V7CRjk+d9Vj69FZXMVFVu6mdYFVMYurmMWEFxaLBX6SLYtVXFViEoAOHAeWS3GWFN51IeYXs1hgfjELvLvF4t0tFosF3t1isbiKWSwWVzGLxeIqZrFYXMUsFphfLOjaD39QxGOV89t30tHrJYsOXyevs711/HpoB04gVuGu4tz3l37ytNeuRx6cLKK/458dOQBOXgu6vGRRN6EunoppFvjnroc6AsC1f33meHhB8sw/Lzd4sZzWUD0p2tj/70S21+hwwD7/L/+0MeHuTCEP0s3WTJqBLi7+qidXcdESfX4suMSrQYAxiYq4Y1ZPccjpHX2LiCitS7s04fQPeE7TTP9ao4fEB5+VA+AzwU6UOG5YM1wmIqLE/mcp6fFSuzRWwknFhii5y4Qs+jqWyD7gXaIDUZfcm0IepJmtmaSB7DMdLzv242d9FClldR+YS99GJ5JJEhXpdbGu4ndxwHHwl89w8eGH+1O0D0PUvdaoh1gXXWa/MT+BiCjjXM50qYpHJhBRcljVHNFKOKnYkP3xgURUI5zoh1KZRDTscfemkAf1cFXF0kBXpXg8zFVctPSuzx0iajOCTJLIM6v4SkBr+bCf7XhB1MVA4Viu4soVbhJRLxwTrYSTAwWk/AEi2reT6InuRERL/FILOh4uq1jSp1zFRVOVYoiIxgWn3sUkuifrYrlHvTRduUnsSF8C2Vdu4HYKkHv9sgR6vYPMH3ZIr3MSryInIV0y+HPzWbt71/OVM7MABOOay5OflmoEoGkr5G4sBQAVMrcoN/DqFOJvAMg8l2OYhzxdIGvf8VyxY/1k1ZagM7vSAb3vR8cLzzI55/DZZHBWYdPtv4IBIDzlkEkSeeY3Tb+gmnxYHb/gp+gK82f/N2LryWbhowHgv49sHDVsSkTPY83CRwPbG5ed+POUvf0GEYCscRtDVnY6Y+zy8PSp05O0p369XA7AEe/6/2kXUf5TbLo/ovVu9aRgQ79UvfDmv18/DSTeCgaAUjgt9XFSmMK+Qc/ZF8zd0WKObh7ydHNpXq+0+LYqkd0wWaUlsD72YMYrb2brfP/2jYxNcXFfAwA2frSn3yAyG5xV+BRQghwpf8qYRB66u9UY38uHB1CWyB7Ve/vl8suI2vchohV+tyknahydlV5T644Lic5gExEtiDhJNLZRrv4atf5SO30fdU5zRU1ERHsxlih3YGAi2aO/15xUbW6jzdRciq/4PzqNMUREhzBJ6UAzhRcPEX0ZeFMzD3W6sysmEa0PvSWbCo2oRx+NY6tq3iSyj3xS7zvVkq+omy+TfTYMzlfUhVAN6xERPY9pZknkkVfUXkhX8E7wAmwhZ9uUv9wPCAaAb6NKoUTjb1FNeo2Qo08BVX2OAQhNTgU6Hjqr73HJABt6BY81jJQ5uPMHgNcX0XH2z/7dS3NStbmD3U94oWqXYRkZDq6dTZ2fdgrnGwK10xI081CmmzS+T0mgs9dS2VQ/WdWx9Bf63QfYRizdpPNdVUJf2WfD4KxCqHdPXAGyDiIHJknkkVfUlZGosBkRAQANhHdDbwLwFh92X9cLKOGbCqDPjZiEVb8iSd9jNAA0/T5Rf358+A/+AHxXHBiY3l13UrEJQkQEgIbxO0McxKwcI1LTMYV6APyRqpmHMt3fkny3b9++s4xyIaWfrOrYob/qA0B92xqd72Y+GwZnFULFfjz0Qvy/e6AMrJPIs6q4M36TDw+gMwCIKNnnrx9D8i+vQkTAyUg6+7xWi0u3MXS4e5uDFXtcd/7zS2uDAACVZi5rqDup2pT0DQOAABwPdfz9zIAB0OGvnYgwD2W651HW39/ff4lCUdVPVnXsd0dPXr4ndb6b+WwYnFUYNXbW1l9euoqGsE4ieNRvt54cvy7LgWSmNd5DHZcfqsqOmln2yuzHTC1fnXewOrYDdpsGKBOblOwLZEFHXFpzeIkXjnnVBTJ+ebP/wYqak6qNd4NUR32UKVX+KgBcQXMXrgnzUKZbB6WbOp2s6lh1pABAdmYNne+OPzODuSaKoKpUAc5HNoTbSVTEq/i+yc9/NRQAsOHQm5GGHex2sVaGyTNGVgeuARuC24rnGw7wBXCyVB1N6107Z9qAVf0BmvhW5QNP/OIrnhRsYt/P8gWuoqmtx0EAiA+Lce6ZOA9lug3v3/gcgMz1sRaTVR1rEr5rIIB96Gr4CM4G4rkiip5+/WRhKSRtm+YNd5OoyP+OesTZl2u1A3B8yIB3ACDrjuN8di6AKkOuBAcENy8lv0aWDQBl5wI+ATkAjvqkX26BbAGe93gdAOe2fuYNAJnIBACcius8EpS2/XXkTChZBYtqjpnlJZwUbEZMX/8oaP2QmhgbE18N9P1YtW/NFLIBZCNXMw9luv7z+x2JBmZ2kSzFRsguIToW9NnwsdWQO/mJR3S+o9EfDvK06rNhcFZh1JrVF0phRtWnAJMkgkf+jppocdUJew5/XOW/uUS0rntIeNeFRHtjw8J6XKOkRhHVKvt7P5HieL3j0dD7YncvfDA4su8t+q7y5LXv/fZ8q9d3xYaVjj2g/C57woxDy+pNsxNlPtmlfEjdntOJqIm08UVv1gpsRLQgILjVZPWkYEO0t+l3+0c8eYeIlrU/fnNcXJbynZR2ChG9br3WOKTRKHUeOep0iTa3Gv/VKz8pPqqN9saGlY49ILSkDQ9++FnsxCy973S2yqR3tpDos3Fw/qapEOpY922/vdzxKpExiYr2ujh9BmbujlP2mu1MKEvpzee3BOwHnuk30czsVHZdX7pj2Dg4sTusVbk8/pERbG5vvd4sGgBwbW1qq8Y2l7bKPLTTvZgSZbOarM6xS7ejzC5Wso9GhoGfgVn0dGPd7catpMV3M4mKxLrk70m2389bBwCY8/OPRcFR96dbxBzjKmbl/0m2jY+dB4Dc9V2LhKPuT7eIOcZi/Q1+8b6pdZsHntnUbmTRuCRxf7pFzDH+LOZ1+Tvk08SE9EpVi85Tu9yfbhFzjKuY1yX/VcziKmYx4YXFYoGfZMtisZhfzALzi5lfzGKxwPxiFnh3i8W7WywWeHeLxWJxFbNYLK5iFovFVcxicRWzWCx4Fr84dd3+W5FPVKPl/QDs23C15pOlv+9dWD1JPu/4f+nyNgDpN9R3KpSwdMpUR9aT/Ymc9Sk9ovM6h1Th2xivQN2bNxN9vO3ZFCXG0qkB/h472byRgGmOT7jeIpJLwLOrmBa83e35Gqf/Uzt4Rj9g/NV3yx+fWHlFb7cBv1bV5hxOrJGMORYox/avLqTbh9YwEmgTl/26rfyQoKzjx9pPLIf4OTsP3TcwCKnnt945VcvKKTPlDPWaGra+WfjmCZ2u2fLoaLVb7cv5bkmIaptxZWvkH7o3E77bua1UvzpRYiydGljLLXayeSMB07x7ydqVkQWFfy4iUsDZxYRfbB9Zag8REX3pHUO0KiaXiHK7x7gD+HUmF3BisaWCORZs7KOPEF16cJcZgfYo+hMR3YipdJmI9qAnERFdil5n5ZSppjbMIZqNmfT1q3l1NDf0NBF9jNlEdCzc+P4h9CISY+nKwGRV8sJONm+kYppvYqX7+GcPeO6WAM4uLvzi6VgkJU5sDNHjrxAR0Z4YdwC/zuQCTixIxRwLNj9+QkR0uLMZgfYMHKDUZRhIREchZesPn1k5ZapOY4joNjbmw9HrwxxjzCUiGphmeF+akhpLVwZG5YmdbN5IhWclY6X7+GcPqGIBnO1Z/GLL3a3ECVUHOo5s/wIQ77jrbOb8PsoWE+Tqs39N61tAcPuEU0DZ5z8c4uhv83djAXyzStPSr0oJo82evwCgejywtI0vgPY/phmva7FNfPngeSunTHW7FIBsifKQN0evNRJeRF+zvBxXYummgaCVBX815saieYjUjHKSPR61R700pYf8XttgIOq7nwHANhoqNfgigKwrNwDg9q6tWedhCjMWeMCAKzgxXFGOK34yww782N0ZgfYOopTjH/9AyZZWTqmEYhOuseCWUTLFWQs/9mkpNHnAMuZqLC0NlHnJ6GRJJuxktYXigTBtGbAseupQ5kUy4J/lRdOtmeepuPCLN0GBOPjNB15C904f7cqC8u+fdrQPHwzMi6owF8C0r0oFfBCjIQlLMGORBwwALuDEIsYYRsox8GzVlzr8vnrxB84ItCtKKBgm7M8AHrVySiEUAzLXeF1c/LdxcXEjMCFuuuyWpLktI2p/AgyoVH+hRHHWw49rNBZm0arK7KYRDRcubFW55Xr0r1RjlvKOGkudgXKozEtGJ0vnTdjJSguVzCxMWwYsC546dLbvxK0fbgRgtmj6NfM8FR9+8XLN62WhAMr9VzjT7GEiyvR+j+hUeyKytyETmLHAAyYdsdgMTqzDGIuYY4cNnW8N3weyyIxAewa9L18+//PwqqsdN6FN5s8ZF3jUiVMqoVjgGse8SESJ+FF0y6HbZYYQkb3+VdlREX4s3nvPlY4SSswguuj9ExG12S7cqutiqRoY56WgkyUZ2clyC8UDddpqI6FH6tGH6GTIViKajJVksmjma+YB98VqRhUXfrG3jlPSL+Gb4TWvjp6mngkFAF8/AFcPHQJsz8IEZiyAjvXEYjM4sR5jbLBBVvWXvX595Io5gfbksmXf/xV3SvrwDa5Ro3qwE6cEQrGBawyNWw6Vem5FGnD+mbKyo1akZlmRD/0AlA1dBdibtXEaS42EeanoZL0UdrLUQvVAmLbcSOwRADCocTsA/Uzwz8cs1syzVFz4xZHCnestAAjt/+nvv4S/c9vEtHlk4/pjNg2BCdjXCDqGMzixHmNssDn69CdTj3Xa0NNuSqCtNmbMmCFtfaVX4R26jHzJiVMCodjANTZ1a1jScmDR07CCHxs0cOsFbOq9IhPbtXfJhlhqpCEnW6CTRXZyA0AkMwvTlhtpewSu7H7Q+GMBpUfTNfMsFRd+cVccVI4nA98CgK3DZ8kHTdr6b307cHbnp+0mYF8j6NgpnBhajLGRcjx0chlU3zhp7zo3CbQtQqydEgjFBq6xqVvVH/4U2UllYQU/NqhnwDf45ZPcn7FW/LvkNJbQkZMt0MkiKTkMEMnMwrTlRtoegVNmUVN6NFszD5OH8Ystq7hfuZ8y5Z1ab+A7x+HD+ksQygFwJvmdvdemf/OrSTdlR818c8zsp3XEYj8cO4HYLlnQw4nhwBhfNHw/JdncOf4AANvEB067SaB9sIq1UwKhuA5KN23atGnTCmJro1sj9hxe86gAP35l8YS29wF2q8dgBPdanOYX1HdJJolfXFnFUvljIZKTjehkfK7/qgiA4IHJamh7BKoh1UnMTNbM0+Rh/GLLKg6alThDzpqngROOm79kqJ+TfgTgaiaA48uA+8b0OWbSzS/t5n/w+WM6YrEXsMoPDeeqcOKzWQCuoilAE996N+aJLMDUJsCWDACIqG/rkQDklUCrdapJ+C7AQShueP9GAMhcLbY2uvVIxc+2tRcIyc9I8GPL72Tijr4fi7gfl3UTT5rEUiNhXnCbnax6YLIa+h4r194DwOqviHHNPE75yp6i+G+aHp/59tcEAGtttYDcJ28DwNxBFdW8uQng+5LpAOakAchqCWSnAkBWtgL2rfLKvKUrNwp3jqfibowcMfypBZEynHi6NzAiaD0ccOKc8SWreC06PsZxGSthjlUbn34zAODizXYYeyIeWgJtkuYONQlZLpwK+mx5PByEYv/5a44AmFlZmnw27ohuKbtjQxdFOj4bs1NF+HGYduckUznuXHZVE7QrO60dACDLMSV9LEUDAOK8VHSyJIWdrIbY0ULwQJm20kjTY3Y2bPM2HgNojqOO9YtmWDMPkgzONmaPp/KLt8S0+Gr70mFLiYgabBzy/rrNY3rfFn6V2vadtZN/qhjc5dbKrm8u/en1mWQCMxZ5wEQisdgUTixijFXMsWpDGc8M23J43rPnjQTaw31qB5d88GXp1b7eUcGhD/Tb49QphVBMCtd4XfeQ+x9LmNIyuOpj+2S3BP0VfF1FJouEZFlr+ncoF1KuQ///Sa//9R4Rvf4KEdGGXrVCQjuO1MVSb6Cdl4xOVmRgJy9TWshkZnnamnVQenRgmunXdtNXjPsW4Y+bLJphzTzlmyYBnF2M+MU4uf9Wpc4lAeBADB3dnxujBb7eOFez5JGQ8KDkAO/TWTXNfrPoHHRsCid2pTP70qKb2YB8E2hVp6AhFGu5xgDumLh1oTLcIDWruu0dDNxBSfGcaSx1siAnO2cnOzy4Y74a+h4vXq9rP1GmTIC7a2bzsGdgMr8YHgo6ZlmvGT/Jttg+yZZ5wEVPvGbFhV8MDwUdsyzXjD+LizP5lHnARU9ma8ZVzPxiFhNeWEx4YbFY4CfZslhgfjELzC9mMb+YxWKB+cUs8O4Wi3e3WCzw7haLxeIqZrFYXMUsFourmMXiKmaxWGB+sfuyexVtarEawPxRiwsGWszUYuYXFwC/2AWVWFO3ItR4y5zlKMrUYiGA+aMWFwy0uFhSi43sbNyadyc9e1QtML84X/xi51RiURqocWr1Hubc2SJCLRYCmGdqcYFCi/NGLfYsfrGacYmjEomm+G5kfnG++MUuqMRiZxqo8fuOKi6y1GIhgHmmFhcotDhv1GKPqmIh46b7LydKRGvmFyM/KFwXVGJBGqjx7qplABRharEQwDxTi1eCqcUFISHjyhOAINwE84uRD34xnFKJYQE1zvi/Aa64s/miFptBi+8OtVgIoEtqsWtosQm12Axa7Ipa7OZKeYqEjOuf3BfYgUfA/OL88IvhlEos8ItVqDGA//zL5oo7mw9qsYL8/QeoxUIAXVGLXUOLTajFZtBiV9Rid1fKUySys32ArGlN3wLzi/PJL3ZCJRb4xQLUmH77nKhFDypoarEKLf4HqMX6ADqhFruGFhuoxWbQYpFa3KMPmVGLXZKmPY1frGQc7X612fAUIuYXI5/8YmsqscAvFqDG2V8/Cyfc2fxSi02hxXeNWuwCWiwyhl1Diw3UYhNosTvUYvdXylMkZxxaTF58fsB1ML8Y+eQXw5JKLPCLBajxrBe8nHFn80stNoUW3z1qsXNosZYx7BparKMWm0CL3aIWu71SniIp4wDYohZv6JbL/GLkj18MayoxVH6xCjU+kVn69u3bOdm3k1Gg1GJTaPHdoha7ghZrGcOuocU6arEJtNgtarG7K+UpkjMOAFC62f718OjfbvUb/1Omn8Av7ueMXxzwzju3vnplcHtTFm7ZK7MtMJprDi/xwjGvujKVeNPp7g5+8cGKEKDGiX+9BeBE2beqjXWXWuzEFZXlWwelmxptjc6M6HU4XqQWzztYHdsBu81mSS0e5RfUd0k3kVpsFUBzxrAJtHiwFbXYYtb5oBZbr5THfNckZVxW25zdvkAYzjC/OF/8YjijEov8YgVq3H7WrFmzZpWsP2ssCpRabAotvlvUYlfQYmfUYktosUotNlsCphZbZlzG/iPXAFywXA3mFzvlF7ugEgv8YhVqDAC5qXcAFCi1WED+3n1qsSGAzqjFrqHFBmqxEVqs6TE7G2bUYhcr5WFSM65k982VgD+Pt3+A+cX54Rc7pxKL/GIFakxENKJlcKlOHxU0tVhG/v4D1GJtAJ1Si11Di2+t01OLTaHFSo8StFhPLf7B6Up53jdNQsYlPv3xju0telxmfvHf5hebUokFmUKNC5ZabIAW3y1qsTvQYktqsTNoscMFC2jx36YWe+4zMA8dpsbRzC8G84vh8dRifpItP8mWWbhgajGL+cUs3EtqMX8W8xU1wPxiFGlqMVcxVzGLCS8sJrywWCzwk2xZLK5iFouFwv6vIarw/jHLjRswDkFhVBXw7haLd7d4d4vFYvF9MYvF4ipmsbiKWSwWVzGLxeIqZrFYYH4x7iK/WBomMMK7MC5i1lfXK8Xx17kFlFLwqH+ZKEF/l9YOnrFf4hfPq7xiPwoVvzj+C5FffNw5v1hyBZb84ifDN09YbcYvjv/i120VhvimHk16sXehW/mMdqOqtv+xh+PFUzHNAv/c9VBH/r7Yae6JKeUB3+Mzv9g9fvFRDCAi2uj/XG5he/LSrPr2hLh46UU5AD4T7Hfn+U5FnV+s5J6YUswvLj78YnmYJ/FtYVvHp0VicZfZb8xPuFvZUsSl5p6YUswvLl78YgCoii2F7ZoqVZx12ec/HBLJd7wuck9MKeYXFy9+MQBclB5ELlnmJF5FTvwNAJnnciyYwWYDyVFROcSKmRDYnGvXkJZgN5+W6Tgst3cT/m8APL6KmV+c8cAoY1jOLI8dDNXS4eu+Qc/ZF8zd0WIOhN4F8LE6kCw5KqsVVLEyKSGwa5uUe+e9t1Z3eTUL0GGVVYP9cXs3xcWp7JbD06dOT+IidSk5pcD8Yk/mF1/xqae5L267YvmcUZ0X2olES8nXFw8RfRl4U+xdAB8LAxFpoiJziAXUsBBYqlPuMFFKk0fsBqyyYNCnpzDP+kvt9H3UOb4vJovdHin3lJRifrFH84vLJezUvFG6abPW9ts1bQAES8nX8w2B2mkJIjNYBR8bBlKjInGINahhNbAI7BoNBL295ns9VtnAJpa1ZIANvYLH8metc6kpBU/+1UfkPgH6ex+A0P79aesT7wwONeUX1+v86N/hFw9fXWbE8A09d3nBd0Xjgc27Q+UXL9zZ0SW/WOwyvAO6JDlxRaX6eiX5bgdg5BdLzlTUzjU4Epg9sNOBug5ssGzp8LUeAH+kCszgThg2ZfmzWDRC29zxB1CISgMAGjPjXU5H/NRHF2ONgahoAGg6LzGcK9WZ1JRifjGKA79Yoz4Zn0HABiu+yh6LzGAFfGwYSIxKGGBEDWsVEnJGPy0rg93bHH/sjnOhOpOQUswvLh78YlGhOAIBGwynFGIZfGxoLkbFZoYalgIrb78n19JPy2jgUGxSsi+QhdJcqc4kpBTziz2bX3wz3bSKCWn1TC0BHTNYBh8bBjJERWOmBhYOjutWdNfH2Ip23HCuL4CTpepwpTqTkFLML/ZofvG1Sp01HScBQESJm/H4OkW1lHx1mOdqmMEK+FgcCNBGxcEh1pgJgcXWK0DaxN699FhlK9rx43UAnNs63Zsr1Xw7VaBGSykF5hd7ML/4Tt3wHHWYOiGhnScS0dzA8QnjFEvF14het15rHNJolNi7Aj5WB5KkREXlEAtmamApZsAbC5Z2nJBJZMAqywYHYsNCO0+Vz+ZMmHFoWb1p/DtqM6m5p0kp5hd7NL942ksmlypXN1BsqKkljMxgFXwsNjePimAmB9bWtP7CP5NreZtjlc1pxyd2h7UqB34GJpjTBOYXA/jotXsdlab1F/KTbFnML86/zpe551HJzuE0ZTG/+G9ozlCfexuVDTP/F9Dh33X4s5jF/GIUOVawpFyUQI6XF1cxi/nFLK5ivi9msVjgJ9myWCyuYhaLBet/DdGeH2XMYn5xEVV7qYo7bOFYsFxp0iSOQaFcF6aQs3iPmveoWSwWeHeLxWJxFbNYXMUsFourmMViofjyi/OFkHWTWvw3ocXOlSrs2XoF6t68mejjbc+mKDFsTg0sdev49dAO+WuUdPR6SZmAGp9wvUUkp3vxq2IJ+vuf2sEz+kn84omVV/T+2/xiMyqxmJHz7qRnj6ql4RvroMVIXCZSi+PNqcXa+cMSWtwsfPOETtdsefWp2q325Xy3JES1zbiyNfIP3ZsJ3+3cVqpfnSgxbE4NLHVx8Vc9O+Sv0eUli7rJVbx7ydqVkW7ypT1Xam6J8Gx48HO37hK/2JRKLDJmRyUSTfHdKPKNjdBiN6jFBQEtduJTbuhpIvoYs4noWLjx/UPoRZqwuTIwLoD0kK2O/dxobN7ooT7K4U2sdG99PJlfrOSWCM9mfnHe+cWmVGJxWP/lRIloLfKNjdBiN6jFBQEtduLT9WGOQeYSEQ1MM6GW9yFN2FwZGHR1uOP/D7tTxeaNeqhVnIyVbq2PJ1exmlsiPJv5xcgzv9iUSiyoPAEIwk2Rb2wNLbamFhcYtNjCp2uNhBfR16wux9WwuWcgaGXBX3i5sT6eLDW3RHg284vzzi82oxKL6p/cF9iBRwS+sRNosTW12Am02JRabAktdvgE/Ln5rEh98WkpvHjAKopC2KwMlHmpKGMAwNHxwlbCuXRdC9kFYdpSIzOyceZFEnxxky/teVJzS4RnM784z/xiUyqxQC0GfICsaU3fEvjGTqDF1tRiS2ixAhM2hRZLUuHDrzULHw0ga9zGkJWdzqgD12gszKJVldlNIxouXNiqcsv16F+pxiz5DSFsOgPoMMT4SUEZAwC+fSNjU1zc1wCAjR/t6TeIhBYKD1mYttRI46pDZ/tO3PrhRiCPfGkPlJJbIjyb+cV55hebUokFajER7X612fAUkW9sAi12TS22hBYLMGETaLEsAT7cvg8RLYg4STS2Ua5h82CudJRQYgbRRe+fiKjNdvVWXR821cAwLxllLKuWfF/cfJkUTaWF7II6bbWR2GWPPkR0MmQrEU3GSnf50h7ML9bklgzPZn5xXvnFplRigVoMoMXkxecHXBf4xqbQYlfUYitoMUypxTo6MAT4sMOn0ORUoOOhs5Z/DSMf+gEoG7oKsDdr4yxsGokYYgllbGyU0FeKptxCcUGYttLIQDYe1LgdAEe/eeVLe5rE3FLg2R58RR0p3DLcAoDQ/p/+/kv4O7dhyi+uP2aTW/ximUocv1ODkPVdcWBgend5DyZq8YZuuSrf2BRajGpjxowZ0tYXMrW4y8iXrOev4n7xW5Lv9u3bd2qpxQYPMCxpObDoafllnxsxCat+RZJ1JAduvYBNvVdkYrvmLtkQNo2EeUFCGRulRlNqobggTFtppO0SuLL7QcMPA9zmS3uchNyS4dnML0Ze+cWmVGKo1GKHSjfbvx6Q+cZuQou11GIraDFMqcVGaLECH5a2jua1Wly6jbPRewZ8g18+yf0Za7urJ52GDXoMcZh5IzWaUgvFBWHaSiM92fiUSdjc5Ut7ouTcUuDZYH5xXvnFplRiqNTirLY5u32BMJxR+cZ13IMWa6nFVtBic/awiQcyfNihV+cdrI7tgN1m9SOv4F6LR/kF9V3SjYQ/8FZhM8UdmzwT5/PBZk/NUVwwC7yebFwNqU5i5pQv7WHS5JYCz2Z+cT74xWZUYoFanLH/yDUAF9BQ5RsXKLQYptRiEw9k+LDD7xnPVAeuARusv5KJO/p+LOJ+XNZNOGcSNljijvUfmNlAvKmV4oJZ4PVdVq69B4DVnxFnfGlPk5hbKjyb+cX54BebUolVanHJ7psrAX8eb/+AwDc2QovdoBZbQItFmLAJtFi4SpHgw8hOBXwCcgAc9Um/rL3qzRBwuJ3LrmqCdmWntZPw4VkwCZtooOcWZ2lxuo3+AEgbTamF6oIybbWR2GV2NmCbt/EYQHOQDjf50p4qIbeE5GJ+cT74xWZUYpFanPj0xzu2t+hxWcM31kGL3aQWW0CLZZiwKbRYkAM+7PCJvqs8ee17vz3f6nXhJ3tr+ncoF1KuQ///Sa//9R4Rvf4KEdGGXrVCQjuO1IVNb6CZl4oylnS2yqR3tpAQzWVqC8kFedqakCtd7o0NKx17gOjXdtNXjPsW4Y+7tT6e/DtqJbfE5GJ+cX74xaZUYnHf9jA1jrbdTWixkT18x8wDFT4MIPdUdl1fuuNsg+22dzBwByWFU6ZhgyXuWKvso5FhllYXU6Js5tM26fLi9br2E2XKBOSfL+0hT88zyy3mNMGj+cWerzysDz8Dk/nFXTnchVG8Pswvhmfzi4uD3F8f/ixmfjFHG0WZpMxVzPxiFrMhWMyGYLFY4CfZslhcxSwWC8wvZjG/mAXmF7PA/GIW84tZvEfNe9QsFgu8u8VisbiKWSwWVzGLxVXMYrHA/GKgUPOLpVECI7wL5dJlfXW9Uhx/icty+k2TxP9dWjt4xn6JXzyv8or9KAT84vgvRH7xcaf8Ymn+sOQXPxm+ecJqM35x/Be/bqswxDf1aNKLvQvfNUtGu1FV2//Yw/HiqZhmgX/ueqgjf9PkxufGVxfS7UNreNQ3gMwvdsIvPooBREQb/Z/LLXTPW5pV354QFy+9KAfAZ4L9bj/fyRNkH32E6NKDuzzEHeYXu+QXy6M8iW8L3eo9LRKLu8x+Y37C3c8WT9CPnxARHe7sSVXM/GJX/GIAqIothe5KKlWcdtnnPxwSyfeH7mjPXwBQPb5Y7FEzv1jQRfmp8JJlTuJV5MTfAJB5LseCGWwyjhIKFUQsm4nRzLl2DWkJdtNpmQ3DyosqfjLDDvzYvVhUcXHlF2c8MMowyJnlsYMhWDo83DfoOfuCuTtazIHQvQo+XqhAhmUpoVBBxLKZGM21Tcq9895bq7u8mgU9Vln1Yn/c3k1xcSq75fD0qdOTuETd0LNVX+rw++rFH3iUU8wv1vKLr/jUE0dpu2L5nFGdFzr2jVRLycMXDxF9GXhT7F4FH6utHRJCIYOIhVmp0SSqU+4wUUqTR+x6rLLQnvr0FNyrv9RO30ed4/tiN3S+NXwfyKJicV9cXPnF5RJ2im+Ubtqstf12TRsA0VLy8HxDoHZagti9Aj42jCOEQgIRi6hhNZpAYNdoIOjtNd/rsMoGNLGiJQNs6BU8lj9p3fmevfrLXr8+cgXF4VcfkfsE/u99AEL796etT7wzONSUX1yv86N54hcv3NlRyy9uPLC5wC+u2G13CcAFv1h8Fd4BXZKs569CfTv9luS7HYCBXyx7UFEThuBIYPbATgfqwoENli0dHtYD4I9UgRncCcOmLH8Wi0bAMI4mFA0AaMyMf0474qc52sCK7bWKBoCm8xLDuUhd6ejw1WVGDN/Qc5dXMbgvLu78YlF9Mj4DoLH01/gpMoNl8LFhHE0owgAjalirkJAzumlZtt+9DQBCcJyL1KWGTi6D6hsn7V1XHD6Liz2/GOK9wxEA5pYwMIMl8LGhtSYUNjPUsCOayvZ7ci3dtAztZcUmJfsCWSjNRepKd44/AMA2cdPp7sXgs7jY8otvpptVMSEtx9QSgI4ZLIGPDa2NqGDRTI0m4OC4bkV3XWAtaccN5/oCOFmqDlepKwXYkgEAEfWLxb9pKqb84muVOhtGiShxMx5fX9dbUrbDPleLIZbAx0Jrh8RQOEDEopkQTWDrFSBtYu9eOqyyJe348ToAzm2d7s1V6ko+/WYAwMWb7YrDN03FlV98p254jjJKnZDQzhOJaG7g+IRxqqXiYUSvW681Dmk0StO9BD5WWstSQ6GiilUzNZpEMQPeWLC044RMMmCVlfYHYsNCO0+VT+dMmHFoWb1p/DtqN5TxzLAth+c9e96TvmlifrGBXzztJeMFytUNFBtqagkTZrACPhZbW6CCVTM5mjY0rb/wz+Ra3qZYZQva8YndYa3KgZ+e547O7EuLbmZjThM8mV/80Wv3OhRN6y/kZ2CymF+cf50vc89DkZ3DyclifvHf0JyhPvc2FBtm/i+gw7/r8Gcxi/nFKKqo4FyUQI6XF1cxi/nFLK5ivi9msVjgJ9myWCyuYhaLBeYXs5hfDA/nF/OeBYvFV9QsFourmMVi/R39PwF2boxVQH5fAAAAAElFTkSuQmCC',
    '1401.1475v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAr4AAACYCAAAAAAnhrWrAAAoBElEQVR42u2deWDUZPrHv9ODtlAohVKOlVvKDQoCFVwRUFBhV1lQVNZdi4InKuouCiigKIgiWAFBfgseCKwHp8CCBypyFTlrKbRAC6WU3oUe0E47z++PJG+SN5lMpp3ODJDnD+i81/PkmzTNzOSTr41ghRVXawRYElhhHb5WWGEdvtdezLAksA5fU1H8g4lB26u8W9TMa1RH7yupF0F8Q85G4f+Idt3r6E248E1J5JOGreWfdhoIx5KSkjeFHzWh11qwFgi4r7HwomrNZWBAZ37MrwdKBw2QXwop5Mi4ZwEKd7GXzXvrLpo7aGs9H2t+afK4Pgbd/HYZ6sZP8JCOXKG6q/qBkgBxcW72XxA8e/bMf4b/6XPSibOPoJ9x62cIuUhV/wbEH+XImyMP4CJn9ugg/F18sbQhbpl9UJNkUzfMV7wUUrAouWkRUdoLA4GnX3j+4fYY6GTRyfdVkhcD2qa5GGg0g9suY934CR7SkStUf1UvK2lS3Y9Qj4goAbZtejNS9A5fZeuBBrFVRNQI7EcpfuioHMDFihFRgaeIiMje7yk8rpf6DZXsQgoWT3V3CNWjmIjsT7Z0smhp1EzfCmxvBRw2nNOIm2SoGz/BQzpyheqt6mUl9cL5tW+fMFqt126Di9Ze2bsDxBbxR+lE/4lqAB/1Xqx6DwCw5i9O/ijZDF6nLntZflmeFDQ3u1x/0brPzc316d+7DbaGiDccwW2nC934CR7TUVWozqo+V1Ln2pdFVRUikLOxpEufr5sMq4sr249VdBkSCQDI/O38jX9uCIC2JuW16C9eyEmtn18qnSyuIfx4+es/ArteeeJI/FfNPkW/zmJrxQ9JRS3vb6bI+OzcFW80BxyLt8wSGhQ5KSHpYqz4FUvx9qTKm/7KVb6s6h75xbG4ww3+kd9Cd1HcO2P1876U/MPns97/ck4TAKhYWRo6/o+d9jt6cFriyLHAym498FumY/AFtW5MOPUE93XUlVGpo6JQXSF9rqTRJw+fVTR6Bpf/N2nN2NS/PYYDPZb2HrSt03cAkDi2uMOKDusAzBxeOba8bxypWo9OfFVa4+jEV4Hi3lUTx/2yAGdCYT93rkRsPX7Lsw0GT+1WqsjY8Nny+QA2DGwovFbkPDvwgYqu/50PANjZZeGt/d4eyr3v3XJjtPTj1lUvAljWQndR4Kaw//lS8UMHxj0XUC6cUKv2Pz/v7ZUdNvTcptYSuLhxzIsXgTMP7SxX6yYLp57gto76Mip0VBaqK6SvldR760b0EQKGD7+rQ7O4FCI6gZA9xyJfKWlbp4TIEVP/LKUiqpyovFXdVKLnsJaoL/aSqlW4lGoM8arqe3xTShemEB1CR3atVdkbu8nRMSRHvmYbQ9mh4fnkuP0CvYzHiRQ5q27Fr0T0D8wnKmqObKITWCqmEKIy4A6pejz9WOOezhYlImrXyZfXvv98mmgUWlQQEZGjTtD7RJ8gTqklNQZRaVPsIfpsJKl1UwjHTXBTR4WMujpyheoK6V0lTV/7OhZ+uupo1vIOAALQuF/nvPd2prWrB9i6F68HEFIHqNO3bAOw4EzPbz8pRRpUrTb1VVWP8NENei14RX2tdexAaD/YDmQ0UeaNHl+yEN/3bCq8UuRM2RPSH0AYAPySFZ3zxx8VDdQfThY45JPC3BVrAJxaprcoADQ668PTRfaa8VeuPInz3wpK2CofACKRo9YSQN3JmA77rJn8NaosHDfBTR31ZVToqC5UV0jfKun84iEqKipU+rmFDQE4J1wkB4HVG4JMYO2g4WmtGysmhiBTK+ausc0PzeldrP54GSEBQL0m6qGvBH1Y8p50oCtyZiIsUBqTD1q/fv36f/1d/b4PlYpXvZ8FthzTWxQA7ME+FHxp+KO33DIpAh+yQ0D8n9fyqWbbd39xc3fN5/JMOI34bumoL6NCR02hWiF9q6TxWzcpAgGgO3LJBuSAyZmJXvjlwc4HwvCRYnAmemkWSN6/EhmfT9t7FwAcf20dAKCb7WJ2U6AgoKFyaKt/LH+8ZSvxhSJndxRfDpMOTVS8GgRUpKuS1I3KV7yKHI+q5RP1FgWAgma+07vi4zV3Algxbm9CX1WHRsuwV1+cmrFJeiXpJgt3RCO+WzrqyyjrqFOoRkifKunOl8Z9R2VvBI7uuekhALn7gUO7+45BIZqGoigB5VC28nHi2RS0nGBriRuQ5zgVIjQ2fQlvEMruO6j8pAOYbPt6sk7O6JeqvgSyd6AU6DHh4gKg6pX/U2fpnKJ8D0MZjx7uobcogEuZt/pO788bDQGAh5sozmoAoNYSADCh+c+xnQGodJOF005wS0cnMjIduUL1hPStkvpv3Q52aB4WGjNUfLW/bWhEzEkiuvzOjX1ua/d8AVFq8OoHHx5+40uFRBWTG/V4ZNRP7UIeUrTaY+qGdCJ7h7DQmPMxdUM60YYWgx4Ye+sXRDS7Xp+bE8UBlYvat7irD/ti70T7yLptlhGNiyPaFBMdFhXzpSIn2effeOfDw29DnXeI7ItjYobd8toVe4ew0Bj21m8ekojoYPvGQNv27cIBW6nuokTb8I2v3rrtbVs3pEkO0dEODcPqdpxNF2NCQ288+fkNYQ1Hylqy7YoPTBGmKXVjwulMcEtHSUbS05ErVF9ILyupFzbzt6tfrqzP/gReaSC+ySuqEw5NK386qFNob2IDAHtRlOJ93eXLkTbzOUuCQ4qqwkIDAKDicgQ/sqjtc2+Z245xexKDvHd6MC+wSsvLB2KDFhxdLl1jqnSThFNNqI6OOjK6oaO3lTT5wdlVGp9EXjI17myd73z8rbyZeBNf5LU/7c86el1J9740vtpi/GNPOEwMKx87bfhVsDU32fYP/aitH+voH0rarh3WreqV6Ndcj3ou8k3b1SBwclq/xv6so/eVvMYPX6C4vusxlxpYAntCR+8rec3DQmZU97bm069RHb2vpLO3btNhhRVXXUx374MzK6yAhWpaYYV1+FphhXXta4V17WsFrotPJq2LByusQK3e71uR2Bt7Ywu3IXCEcHOoY0M5BrSUujO/y3vc/H2eWbuDyT6gOTYA9tG1cO1zKUKnoIO7Syf7qeAeFdcn6npXXLoU4ebZd1ZzHDyEkOilz3wqNGybNDNSZq7DyqflmU9ft8HC+7eHANkjdzasha17NypDp6CGf8z2VIKsCs8W7FFxfaKuB8U1oe67URnu3XGWtYxoloOIpk4Wn1gxd/gY5YBLSHTnrqCitr2vEK1dUiu3HP3+/BW9grZHeCrBlCyP3nHmaXF9oa7nxDWh7u/PX3HvjrMk5KHkIACMO74LADJamnlQidOIWHX4NWT+NKFW/rb0/jBEryCP3UtCmz1br6fF9YW6nrtRx4S6vT8Mce/at82dm9f9MSe9NRA9+uPbAKyfsBIAcH6v7Y5I6axfbguMzA6sbF94EZERYk9hatHQI46bbCjZW9i3NVsw9q0pgzbMkzdaPVda9vKuzC5BN0tj7IdSO3ULASoOnure3YbiE3l3lBxs3SEg82jLbkBhav6Qcymd2qDseF4/6dpISkqnU7uorjYVSxyI6sXqUG6OlD6r3OZoi7NU1U54fWbOkdSiyKbS4lwZ1QiPi+tCXWlZI3Wdiaunrllx1ery4kKtroG4Zcfz+kVoVze49m234lXMjW8JAE9/kwvYHcLxv/SF2OiB28RBR0Z13pX/dpuVSB38eKbUc2bug5/uj92F3XcHxbzwibzivweN/JviAlw1V1o2/6EbRp3/mzQkZdCBvkd72nFsdP6AhDF5yFl8z7drombFfbM1+uUZwNkP753+XdArjxQXLBkmXRpJSTNHrGq0dZGcTbnEf5u8z05Tqs2R0ifGtdkE/CfmS/F1ajMk/HaaLc6VUY3wvLiG6kqTDdV1Ji606poVV709vLhQq2skbsGSYRna1U3RFlMLHV3nEK09RcPHENGkkUSfdCWiYiQS/VQnny4F7CR6sUjuORq8jFYWFjZfS5RTV34CCX0TPFT5cDnlXGnyZ48S0bPigIoebxP90rOkrONWInprpINy8RbRDvwf0YpoIkrCHqKq3uOpEIlCQVJSe+ybRLSYXZ4pl3iTaFuAdCnFihae3iimzw75gWj3QvY6EVlEbHG+jBrQFp4T10hdabKxus7E1ah73qy4anV5cdXqGotbiETSrG6KZZlaSAvbVtI8EhQmx5m1L4RICle1WUhf/vUJKntV0ZOMVCL6L7bs2bOnqfx8ytwnN+F95SM5lXPFyal1er+y3S4O+Fl8suF6lBPRLpylQuwgSkQq0SYhUQYRxQeWFUsCS0l3IZmIvmYKc0scQq7UIxVNRCSnHzec6JUr7LVw+LIt4sqo0eHrIXGN1RUnG6vrTFzi1Z1uWlyVury4anWNxS1GImlXNwkLPZqzLSVG/HnXrUtay081DohbjgPxX5etG6nqaQwgw9YQwPr+7AQ/bdaI5147CN250uQbD/bdPnSY+IyMNAjPLzkZEAwgDCkAwiH9w6JFlfyhipQ0lT33A/pLQHdzWPoXt5zIbBKiLke5RTplVDc8Iq6xutJkV+q6EFfK2sSsuOrt4cVVl2NK3HD3Uc2phUQTRsRXCieIS+HLiTaF0D7hF4LSbZvn05AvXnAoepJRSETfI52IKkqkdeJ/ISrrFlMsryzPZZNfP0OU1mGz+GkJfiUie+lW5BLRVmRRIfaLv6yKE8Ts8Ep2fpCSJiCRiL5iJwhuCfYrLBctPLKZpb/zqZm58utEnKcp8hZxZdTs7OsRcQ3VZZM3GarrTFzi1d1hUlxOXV5ctbrG4opn3/3un31zcoGntwQEAnY7cLqkH3Cw0rERdtgBtB487mHEvdvKpuqpBDB48CIAc8+Ly/xw/HYgbHXaRPnbf3kumxw/D2gzUPzl7jXiYwALCu4csBKgL55uJuS0oxKwVzkAYC9QuuTNQDuqADvsLOktd68A8HOF9OREvSUA5eYAANJZ+kmfXY6SX9+APBTIW6Qpo9rhIXGN1WWT0w3VdSYueHVbmBSXU5cXV62usbisBMXqcgQ6fQOd+ey67RWxzfZMCd83c8e5lNbd874s3/jnjB+GVk5KT2nVBggqegI3zvo4HNFST86080epB2x/Xbvpwlc33wYA2DB5zulebYEPEpL2XmQPo2Vz2eSGlSlZG8PFJ27Zhv+8unBTh1sD7lv585mPWs0OSnzpTOqfMt7ISu64772c5Jsj8xbFHEqeNm7isUnpqT3zJqWntGorJrWNWLcj7+sre470F96Nq5dIn3EhuXMzQM47tCMAICFJSt9+1TtN5NehZauS+3eRtogvw8RhOnNGLYrrSl02+WKSgbrJTsS1JWnUNSkupy4vrlrdrkbiJk1KT+2Zw61evec8VJ0uI0eF/CzNIiIq0OshupJeZbiUYq44ucRRmlqmGFEurnAlTc9AIRkZxWccXKOU9PJZx8Uzl+Re/SXURcvp7RPVry/kmtqimj7nwYPi6kx2S11dcaWsJsVVVc2Lq1G3uuJerTdMJnc527I21t2Y88RnbQbi+r5h8uoR9yq9YTJlMeLTamPhtB93nBuI6zuuJnGvzrPvhcxge6smtbCwYzfdZrvOBb6KxLVgAFi0BSzawgonMdCSwDo5WAJbgWqZA7jANQAAGZvzJ+hdLeXuk35q10XbK2AoiuX5ZdwkZ5xG0fJ/RK2vO9T3ahvr4VoQl3Hix/xXwkzvturHjqKgytB7ABw9FVg10ncaB5gjRmCMe9S9PC1f95fDPuW+qvBwOv38HJ1eAUNRLM8v4yY54zR+fHkRJj7k8IOThaEergVxGfXPv1FujrJBjUie6OwHxkcBQMi/VjXwpcYmiREXuEcBkvW7Jwjfj18Yptd5CYnq5fll3CVnnIT91zfp5MtF/mAabaiHa0FcxjEUmtptNSZ55mI9EdGFuCpfahyAmuEaIu7h9PMQGwAUo2kHp53K5Y3Mdmty0uvaGO0jI+AHYaiHa0HMrW+CsqkxyfNS7JP5AE2fE+BLjTXXvsUpeX0j80/lDw4RiQ4e1yjZX6fvsZ8fbA4AYDANkPt7dC8bwLMsADDtQ4yiEv0lBCBFZlmkZcBjJ0piRFwgXK5VhlOkxRV10Owp2NMMfhO8HqYFEVplmkeWW8nmFFwMCGlWdjyvX4CTFNDCWJKSFWyKuGtFRumCDPoIEbjipomrsOZOyULWNxprzr65/7n7FNLeubdYIjo4XGPfXVVnuiQ0Fc1sJZgG+HFN0/cnABqWperkiS1bgDtsTpYoWDIsQ8GySMtAg50oiBFpAblWGU6R+pR1XBoXCds4/zh0dfQwLYjYKtM8TG4Vm7On3fhDKFgyLMNZCi2MxZSUp4i7VmSUZNBHik5vrV6XvYs9XMJHGmsuJ85gP1ECchnRocY1+rxPNCaOjRZoBCrE60I3z7I8GfrxwheaCFSAZgnxXs5ERarXVVBIMRJl7EQmRtgCrFZ5lNinw9SQP1z76ulhWhCpVUHziJvJNj8ZheUvfuMQKRudFE5gLKakPEXctQKjJIM+8o1BsdGP5fpY3SD9h2YHA+g//tigoQsgWYr+GYh2FIeU2ICAS9pfg8FC9/as0L1A/UPsE5SQp4BiwJ4W42SJIEUqtoxi4Q/o7IGkk8oSwBZgtcqjxD5NHf4SOnrAtSD5FUBgS6k1ADcAAf+c9GGYtJls84HsJ966Q1rGWYrdRz8Hbj+MDScGAxj8+rmWTEl5ihjB9jswFogeO38I1s5SbEnggtg+Uf77rZual5FwjWe/KTqz+2U4ozk0LAsAPA8kJhgtIafSQiEK7ETq1KmBjRL7dOvwl+D0MCHImG7dug3h9kiLqgy2mQqR5rV83a6FbNQp9GAs51yOYBEjgT7K9kb+97WF+N4Y+K7HYqQP3X6vurvgnfi6O5SOTYSpb8uvelKL1oBd/YlhT2Db3QZLOEklvJW8Oz4O5wGlBTBbgNUqjxL7dOvwl+D0MCHITp3W1PD20mbmyyIFLKkY8Oo8TUp1iu441QKorOjqyI8CstFVVhLaXSu8cew+ZEHzZ+D39zzUCy8AfsIVRnSocY2ck488qDDlEGAauZtnWezCB9nHP+qks4TYUqVIpYZC7LBzGI0dlZAXYLXKo8Q+VsfBwWn+pLeeHqYFkaEbieYRN/M3WSSi0C8/Wi1SNjopAOjBWExJeQrbtWJ2EfRRbAtEaMh3GmthoYAGqwI2N9xyuEeGQHRwuEb7v322eNaaRpKzvADTyKxIcxXLkjJ5Q1Hmr9vXLXy1yUuJmiWSJqWntCqdlJ7a84yQikdOkialp9xST8ROKuTOcKkGVmsPBqeIfT2kOpKWjWruy+NVBQvp68G2x5UgEmPDaB4JHRqpIIqO3VS1an3K/UmT0lPrvq2VTB/GYkra3mNThF27U2CUwEAfFu9/dCE16R4AvtRY5+1cxekrV04XOXjAhIiIkvrnEJVtDz/PWi7kOmF2nIV2CdJN5QSjUS8g1spGKfqqTaB42xTWtCBSq4rmETZTI5KrFBpciClJTnatCPr4VbhrufvJBCIixy37a2CaW9MljBbwQH1eF9jtmo/hrLc137CMPv3Z/w5f56SxfnRce6hRZeKM7o9W//vcGi9htIAH6qvFiwfPCJKyKCGwQ6R3Nf/fzsZJj+IauB317MGC6P41+8SkxksYLeCB+rwusHs1V4vmqaEstQD6WHdTw7pdHRYsZIUVsHzdrLAClq+bdfFghXXxYAWuX183AMj87/H6reLqr3zUAyyf0/CkT9jvJ8XbWW6HX/u61UxcA5HdhjtrCOkC1UFrPValc183gD65v93iecOnv/wfuI30aVk+p+GuT5iRF9jCCcci2t+Y/fCPfnu+cFdc94BJ9+FOo6yGtKdOXabRWk9VaejrNq1rERFRxW0Dq4P08Syf83DTJ8yIIBy1iYhK2vco99evNd0W1z1gshpwp0FWQ9pTry7TaK2HqjTyddv7zoIIAAieUy2kj2f5zOCFNSYIw4YDmHz2szr+evJ1V1w3gUlP0a1CViPaU7cu08lttXftK1mPzWo0SGiIvYEZeMkgJyMnRaQvyxXLB13okvMJc0YQatI5IQifsgE/LJp1kyan5IqmA5J6NYzE9QAwCUCGO3UN81R+axorNbEQIWv943n9IrLKbdRGSKnUjjG64gzosLXanafYwxyTqzbZq8m1r2Q9tismUNToM2bgJcORjOkTkT6XLJ8udMn5hDklCDXpnBCEA4CL4/pM1uYUx2tM0bwdRuJ6AJgEZLhT1zBP7WbHW6lJhQhZC5YMy8DRx9psBpZ3XKmGcCVGV5oBreQ6O0/ewxyTqy7LE48pcQTcJb9gTJ8MRzKmT0T6XLN8WuiSM2EzIAg16ZwShI+FHNPLeTR4Ga1M8z7ACTfE9QQwKcOdeoZ5nJsdZ6UmFyLa2SGR6Ezgb0T7PuThV2GEPEP2UWEpdHaetId5JpcryxP387W6WfqpSGkBJnm8MK8tUWBm2iXHXixS7zTR64tN5UzY9OzcnKUTXM60WTdgHhH9osmZjFQ9UzRf3TCpK668ubKRmYvN1YissEDTMczj3Ow4KzW5ECGr4HI0+kGif1/mtRNGyDPkw1exmzU7T9rDhjZwnnnKzl+TRPaJ3jLF9Llk+bTQJWfCZkgQahBCXYIwb8KfXwCwXidnY/gRwOmuuG4Ak+HqTVfznWr/OM5KTV/3F77NyK0fqq+d3gxNCkUGaQ8b2sB55lu3GY3FjySPtUNXRz6AbHR1hvQB6D5kwX+MnlNRfPeT7/QKAxLkpi7IAUAKghBAZRnLBoN0MkGozPps8YpAoDRNJ6cN6EktYmNje/vBnX+64rq/uSbkfqP+4iNpGdud7gNFyIWwrAAG9Fz8aZyOdoSpqhkayXW2RtrDrg+NGh++jb+Y+wsAlMaPlw28GNInM30i0mfE8gHQeJfZUQnOhM2AINSk0ycI//vV++0B+2SbTs5KrSma70JX3JoBk4xuFUbrGuZxbnaclZpciJBVWNr24ie5f9JoJ4yQZ7Dl5BQ6O0/aw4Y2cKgJqql4l/zAlO9s2Rs3vxEuG3gxODKPUYRtylYl9+8CI5ZP7fWlgC7VPmHOCUJNOn2CkIaVN9u8ccmUH/vfx+UUXNHUpmjwJW2hJ27NgEkAgAx3dtczzOPc7DgrtcaskNCyVcn9aVJ6as+mQKcl028AZygnMrpy6RJa26oPS6Gz85oJezjc0AbOcw9IPbtlex5n4OUc6XPJ8unxhJxPmMcJQj6ntwFOuCWuR4FJXcM8F0wnK0SZtVRXO8nwTmPsJqfQ2Rq2h10fGt70dasF0y6/zQpf3TDp/5sLvzMHMBdpe9uf+6fXy/dNVp/Fdba5Xjw5+Ibl81OCsLbOvv6/uRYMYAlshUVbwPJ1s04OVlgCW2dfK6yAH3/y4CkHQfiRiaAXQmEj6J1VatWsEF7nDQM8xEW5wJyyKq5CE0EvhMJG0Bzypiuk+VWc4GtZFZ5zLPQqb+iJw3dhAdAozuUQs3HfrwH4eVzxdXH4thjrjsjOhDS/Cu56JrqGu8flhJJVb97Tp/NHPaZeHYevQD3Zqg9s+a+JoJdsCs0Tfk6ENP9ZsC6+5tbu8SveUOfaV6CRZFJJ4KxanxRgMZF4KuZpLADQ8G4iwaQYAmSV2wIjswMr2xdeRGQEh6L5o4mgR0MQVyVU9pFOrRQOg0opeAWZkAwNY+iYYEbIk2PMjFBs4TwmBQxOtXtqgzf05tlXpJFkUkngrFYJtJVEPGloLAAa3k0imBRDgCOjOu/Kf7vNSqQOfjyTQ9EA+JeJoKdDFFcp1JQfGsx/tATMYVApBa+gJCRDwxg6JpoRcuQYMyOUWtQekyIGp9o9tcAbwovmABKNpCCVBM5K+JcRTzyNVYxEDe/GCCaJJyIiop/q5NOlgJ1ELxZxKFohXXsBXXGZUMnYRURjJsgOgyopeAXF/6RmaT3ZjFA9i5kRshaVx6SEwSl3Ty3wht68n0+ikRSkksBZCf8y4omnsQTMiUOxJIJJdfhWtVlIX/71CSp7lUfR6Fo/fJm4TKhkZBDR8qAyWdlkpBJlpqWlndUqKAkpNkvrJeP4bTt0oMLO84geHqVsEX07hX0nYXDK3VMbvKEXHeVlGgk8Z9UYKlosXOdcrkax9AmmgLjlOBD/ddm6kTyKds2HLK5aqMaVGUplG0s+hM4YMKlZXk9lRsh7NGrXCdcznqwt3tCb176MRlKSSjb2ry4tJnNRKhRLSTDJQ4B/HtrSsvUtaxP68CjaNR8y6iULRQAyBIdBSVkbsLOkpCRFT0HCVLmZrRewZHnZq1rdC96J/3rHrU5Zsu8YBqfYgx7nDb16+Eo0kopUqoT0LyOeeBrLDjugRrFkgonhcACA1oPHPYy4d1vZeBQN/mYi6Olg4iqE+h2wr5AdBs8rbRk1Cgr/sWaGjjEzQvUsyYxQqbPCY1KyOVTuHs/zhvCmLaHEm8mY2clp549SD5G2kognhWGexEWltGqjRrFkgilUwuHET+uKnsCNsz4O51E0+NxEELXLujFxmVB5BZF7TswcJTsM3iZjbToKCv+x5k7CepXMjJCbJZkRspZKlcdkcZ6AwSl3j+d5Q2/bEko0kh6ppEuLKbgoNYolE0wqYKuyiIgKasI4XaVv3RTiykJVnHU45/A0Cgr/yc266JjWo1FXZxmDU+2eq8ix0EKxfHHDpJeEWvb7UgDU9+NbcI3yhh69YTLtxx3nrKPXf4R6uGTa4XO7Hh/S+1rYGAvF8huBvSaUNzwafbnXLRjAEhiWr5sVVsDydbvmYsYMSwPrb5slsBVwx9etcBsCRwgElWNDOQa0RLWwtt3ZgZVRt+PrOjZ77zb46ZKt/mDUzOXNTawua3cw2Qc0xwbAPrr2BXVVv0fFdVddfxPXdTYXFTv3dQuJXvrMp0LDtkkzI+vBbfcuAECjhPs2Nwaav39feT0geM6HQTV1eXMvP+o2WHj/9hAge+TOhh4/WLXMl6v6PSquu+r6m7h8NvflNPB1mzq5u/BNy9zhY6rl3iV8ZdPocSKi1fiBiOjNCzV3eXMvP1FR295XiNYuqYVvfXTMzbj6Uaviuquuv4nLZXMtp3lfN2Dc8V0AkNGyJqZcwaO/rQBwOmINALrYtOb0l7sfMkasOvwaMn+aUAsf22x2tzoPi+uuuv4mrq2Gchr4ugHRoz++DcD6CSsBqL23jIk1lXvaQ59s+wuocPS3i+rgyE1uuLzxrmPVzA/EvjVl0IZ5shDqyRxipp5rP5TaqVsZ38czfFI751KH2hfXhbrmxVUWoM4v9pgVV2Olp5qg669XfTl1Lh4cK/bSsfgqoqmFv9bJIapYQMPHENGS0Zm/df+feMf+1l5ha86Mwwza13ZwktRzaFTEimV1dtKuATsO37eU3aDT7BGiQyu/x0ai6RdZGmnSyTisXHjgoSeI6Ny9b+5bcj/7a3EyDis/OvD3f3y97Peh06ufn4gqBwVuVrxUTRYH6889MWBxyvKOCVyflO/7GXh/2W5i7Vz9ehcPnhbXhbqmxWUFaPJLPWbFJcOCxUaprRiJ7slp/uHfNLXQ0XUO0dpTgsKMlypGoiGxxjmA0cR6pTTjkj36EXK85IbLG+86Vu38RPRN8FDljVmKyRxipp4rgV9cH0egSe2a+o0E9pS4LtQ1Ky6boMnPesyKa1yw2l9PbQRnQk7zxlgAbE8vrUKaCEB/8O3ZdUknpa6BLVZj04jPcDk0Qu4Jtt+BsQ23Z4Xu3Xuq/iH29630OyqpH/TghrJ9sfLabFIQ/gxEO4qRsPcBQMkqBeE2IAoDgaiLNcgP5H2/dvt8xXYpJrPBenN3Hx0O3H64HtenrgRSu6Z+eEFcF+qaFVddgDI/6zErrnHBwfY7MDZB1VYDOV3ccfZozraUGOiRawbEGu8AFttq9b5Y4KHSTd8Nh3mXN951rNr5QdNmjXjutYO6uJ0aMePmyiyZqo8jx6R2nfprX1yX6poTV12AMr+ix5y4rgrW+OvVRE7DiweiCSPiK4mGjyG6FL6caFMI7ROh4nTb5vk05IsXHIqeZBQS0fdIJ6KKErbQv+o8X0ZU1fIvryg+MmGTGMmagEQi+ipC4fm4X/yLsimkJvnjfyEq6xZTLGeXJ7PBenN/x69EZC9V98n5EnGeprB2Tf3GFw8eEtdYXZPiyhP4/HKPSXFdFJyMQkVbMRLJLTnduHjIyQWe3hIQCNjtGls2A2JN4572UEVhGBDw0Kbb3XF541zHqp//h+O3A2Gr0yaSDm7HIWbqubINmbKPJ9Ckdk398Ia4xuqaFJeboMiv7DElrouC1f56aiM4p3I6RSCd+7plPrtue0Vssz1TwvfN3HEupXV3BS+V0qqNAbGmcU9rturlGACNv/o4yA2XN851bFhV9fJvmDzndK+2wAcJSXsv9tHgdhxipp5rYyyZso8n0KR2G1c/AGe+bh4U11Dd5kvNiXtzR3HCn5Zw+dlSYV+YE9e44J0qfz3OCM6pnM4RSDe+IuF4KSNijaO2jtiJiBwH3XV581R+bSgm84PVL9UsmdinIdCkOdr6UevimlbXUFxuglIfd63xTO0NZZsbctYm62aFdccZLHMAK6ywDl8rrMPXCiuswxeWr5sV1jsLS2ArrLOvFdfq4UsXzc7VG0kXgcylb19wp4aDC991Ky8LbaKiD/Kwfru1W6/fw1ff+QsmPcLejcpwl5cSeSZzeVU0lDbR9eQJZ4Xe4avv/AWTHmF3PRPtyuNNE+1Gmc+rMhTTJrqePOGs0IOFept+oJveyN69q8FL2Uznpc0TDUGooK6J148nnBXaw5dz/gLvFAZAtAOTR8rIVNnxvH4RqilqmzLeNEzmmTR5mSWZIqVAQzk4IzEFN3VNe8JZ4friQe38Bd4pDIBkB8ZGromaFffN1uiXZ4izVVNUNmUa0zBkjljVaOsinbzMkkyZUjAU44zERD80ANe2J5wVZu44Uzl/afglIiLJDkwcqUKmCpFIaoJJaVOmMQ1T8ExcXmZJpkop0FDMSKwYiaQDX/nz09Wt8GQE6V9OSIiU5Hb0AZ09wHCk/uOPDRq6gI1UIVPyetKUYPsdkCyjdx/9HLj9MDacGAxg8OvnMvaugMgzcXlLbEDAJXAphYgeO38I1s4CQ6OA+oeGWqci6+IB+g5eKhyJswMLh67JmxqRgr5pmJZnkhZilmR6KZVGYhqWyorr+xF92ii+Oz4O54GEvgDwXY/FSB+6/V5zUxooPh/ojlMtgMqKro78KCAbXRsgJ1qwNtNcg78TX3dHW+mVnJIw9W10H7Kg+TNiV09q0RqwV1j70jr7AmrnL/D8EgBIdmDcSHuVQzGbQ6Sgbxqm4Jm41SRLMqhSSg5kopGYHXaZm7rGPeGsgAnWLUnl/NW5GecUBgAJSSlZG8P/Lo+Ukanzk9JTe+YpCSYRkXJiGibzTOe4vJIlmTBRTMkcyAQjsaRJ6Smt2koslX96wumzblbAqzdEOc40C6PKYABAad3L5/8U5tYUOSqyWgYAQHlWy0AAwJXcG4qLIsO5byCOjV/fBJd/+1tKcy5ldmAUgMqX4hWDyy+0DLDuOLMOX/8JI0uyq8o+zjp8r8cbJo0sySz7OCv8/uTg3JLsqrKPs86+lrqWwFZYtIUvwjLNg2VLaIUVli2hFdbFgxVWWIevFVbUOP4fKlX0WMF1RA0AAAAASUVORK5CYII=',
    '1404.5002v1.1.png': '...',
    '1404.5002v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAcgAAAD2CAAAAAB3zUhfAAAuhElEQVR42u2deWBMZ9vGr8lCRkNIBCWW0lqiqrU1LRpLEfQVW3WvRi0traV4tZZS29taWkvr9UqJtNROq1WElmqQkkRJYouEJAiykkSWmcz9/TFnncyZmXPmTES/c//Bec6c537uM0/mzFl+c106ghb/hHDT3gJtIrWoSkFEwdq78HBHMBHpCNA98O/JB1KBKoM6n0SFMnSkHVr/MeHh4HZZvwEA6nf2rpSybmy7WLNJWM1Nb934JfvdBo5vbu3F+F89S17ba2w5VIW67i6L9Rz6jusyiN9muucj6zsSZD/uHQ7y/O30yfDBr+eL1t8stdXL9qt8WFRg+l+nPQZKnjwlmHJWIsFeYsHmVuLzwfk0G9sj9eW2B3UollwzRK51KontDOK3+T8e6Q6/fXC4mEl+RESmVY/lCNfOzLTVx/arkm/H7Lb5RERl3YKJ7lmdSFFi4eYVorD6VaJdNUylV0iFifzmC2eT2MsgfJtjJ5a4bCLJ1HuU8MPQ3tZU2X5VcmdOuh0yL0QHExVYm0hRYtHmFeICTESHGtn/63EoUmonOZnEXgarb7MDb5+H7DOkEeMWNMTNGF2POkDa52eT8+vUB9tG8fEbgR7PoDAmr0tT/lWZsdC3p3khKMD8f8Hl7C51clJyelU35xcnFm9eFp/Srp0OYGrISsUlIN1wUdfK5pgFl7J7FMY3fcLtxrnGTwLcHmWW6twaF91292gEAGj+wtC4R2yk4XtRM6RTeXMUnq7W5fzREY+yW9jLIHiba1/MftaHy1mhQmdvCDTF3/jfpKB6wQeB5AY4FZ0Krp3zasCwm0NxIsSj5aR13Kty43hLd/OCeyRzCrA+JAVXFw8oYPKLE4s2Pz88p+upV7LB1pAWh6jo6MR70Sdtj3lnTf9dW+suDNu5v97UeeD36NwnzVYhK/CdPwAAZ98ZeHWWjSxcr3ea7QM2tNqEv/qUpwWeqn+A3cJeBuHbnLu2Xwaf07JChSc73GeejmM1TRlCtK4tESUgk4i4duRbRDQh79HdRHdq3GFflXloNbn1EbQKkECUhtNEp5Blzk+ixKLN77faT0QLhpi4Gi4gj+iQj91DaxYWEB3Bt0QR9QR7RKYJPcuOfWwiIqK/H8ug0f42knC90tyjif5aSdR5GdErYdwGdjOI3uY8JAhyWlTo7KEV+QjEl5Qel3SFX8e2nx9zvmffFbszvWKAmmf6Kry8DcjmztWZ0+9aAODJ5rexedSlXgB6zbl+Um4NHugG1EUwUPeuYI+gW9H/rcZLdABAEz4MQI8NJumjGNeryZBVXbHrM6BQB7jd4z41djOI3mYPUU6LCp0+tJ6v/gyOP7e2aVfhwZBpPx7fJapvv6u62gB+fF7ppdagpAJmvxdYvGLOb5Te/IqbJwA9LmfIr8Gb+0ewR4DHf3c8qgMApB9/BfBuZuMt43tN2pWRVdMLmLAzP+3EVPZ1+xlEb7NFTnGFzk5kafj8OgUh4xZ30AOnABBmgWv/UnPN2asZt6lhUFBQRx3zquyY57ee2Znm7KcOAAoAc/4ocWLh5m1NOQBuo217vgYlIdjDsqUnPt8PALhVpyEQ39uhXl3br9kYBiB38aodR55jN7CbQfQ2W+ZU6emHwQQAeSM6fYTUwmeBeKNpLwKQjVy+fW050Cx4eK9vACy5ybwqO/y+X/IHABStGgMYYAAe8c4FfkeJOb+vOLFw8xe7bgLo+/cb9GJrMMAIGIwExPe6amv3YACzcblJsIelI6c+u+W1JAAIpHso2jFdOgffC7rJ67IaAbhz5fURtbgN7GYQvc0woFyQU1yhtfutjty2vbp4750xtYx3TENHAKYPL4+4/sLiR0cMwr8TOzw9nGunH2v/eIphbt4U9y4pQUNhflXBjeP092sN9z2XOd0PSZ/u7zYrGF8ffCPN+4OejxW0fzzFMNcyMb85cqb7tD4R8Gl1MDX8ufDPkM+ufREf+nb/w2/ve0Zy0IRPDz73ieei6BfnpS85M2RhM3aP0r69fPrJY/19B63wBH76LvTQW32lKxe8Lyh9YsezAI4F16KCVp++zmxhL4PobU76dP8LS9uyOR8TV9jcMoWyK+Py1PtkKiMiupUlaBeaipLvExGVXDPfEDO/KvfODhGl/xqVLWyXpZaUpOYXsPktEws2L7lqZBaulcsc1PoeCqL4otF2EkGvIiKipOfvEN2P8r7pcAYHK6mQQnuM5dIk4bH/A0Bd/tvJ1Y+xtIl0aZLCcY8Nr5u2vt5/dNpEPuQPltPjc+s97+v6B8vaROKfQQh4AAjW4YHP5EM7qK4KZAhmriN70IOOuQ/toHOrQhk9oB1a/1GHVolHa1sv1WgyqpZ1DoYNEVJz6yh8+rkBF897GNu0ceEUlLuLmqYcf4CgY/5RfwSL8ZR2N3ooK8aU468cvqJvw2dO9EiZ53HS5kTqS2eHchNZo9lnUZ9PB7xzJq7o7Lpp/CapUfqggYIVs4xLgdiXX6h/59a22ioMYLp++t/bOkmPJ69AplX8tYchOuR9JX9os4xLoRi+mmPmYAw9utqBskQkxqTPPGOJiMJkHuTl3FvaMthExa1j+RXRtacS0cn2bTp+dFuVQY9/thMnpcezl0TcgW1N+ZwoAdsV7Lt5/+zticRN85hFK3wAwGOZvFOumV1eKwTz/NBFsXCEDl79V/LX3D+bn1XNPh+7vJ4qIzz/aVcb48kskG3d2wc0wln51bD7p+zpx0LfXuaFDs0AoCxm8zkSLRQeOW44uzKTHezwjjTzkXrznUkQrcy8lna9KDUtBXnXrt1F8eHI02ecepdvJvkB8P2ZW7FsqkuvXSqMJ68D1wo/ApzCS/ILcHD/3KSwGeYFXQQEIAy3IEZRGD4GAJqu27BdtPLssDbHcxY124TkXu/eYJgbZ+ISfADUzi9m2gc6MWcCMf9ZNiFa/Ym0HE9mB66lcy+Mmb02SPb43P4p+o4UYzMcCMMt8ChKARKI42NoEhGN8kmjScSv/L1aDt1z+5Nocj6xzI0T35F7kUBEEbhububMIRo4lYgufkN0x/e0Ol/Mmfx3pMV49pOIO/Ct8i9n/muX7PMDbv+U4ZC6gCwAwL/L3U2mLzkQJp5daCxCUaJEfMzKP984IlwZ3HDLhJ9fiuxW7OVjlbmRGXrcB1CI6ubm0k+Y9a1aAf7PLfhJ7U+kxXhyO/Attym4/sQXE2UOz+2fwkNraNI9APji4x2/LOJBGG5BjKKI+RjvLX8tFq50C9uAuFU77u8ZIsHcyAtf3ANw18N8PrXH49Thw1lph+/gv9cA1D2n+qFVPJ7sDlzrhgkICJprkjc6v38KJ3Je3XAA0Pk3rKvnQRhuQYyiWPAxHRd/dlm4cuSZXxs37bT7VGchc6M4Ar3uAsgPrgYAaNfVaDRmmYx0d3wUgLuPqT6R4vFkd2Bb1wJWAvC6L/OPmNs/xdeRh+sfJCIqadCVyND1KyLT6+/zC/RBeHJaNhFRLuKpvNd0Ilp4mWh4HhFReZ+OJFhJvevfok1PLiei1ROJaPRJp64jx35ARF02EMX1TDWvCZxIRGPKiQprbVPnOzINfxAxI3DjOZpEXCDTyml9kai4wctKzg8CJzrz24/0AcO2RoePPjaOiLLDJq99e2aJYOEP1KqJVpspcai+z1HKHTlq7fRddHGE97PriYhuDidiVxLR5kFERXUziWj1ywu3Lprn3A2Bu71Xxk97z0R06NF4IqJfRj7S9L0rdO6Dnfv7f6nKDYGED3vqu4yPZkbgxnM0ibhAtrV7zt6/RgzKlb/vzP45gXpcT6A2TZlDb2lmY3fBwvkxP/qjOHroZfYnDaW3Glc8SLMrywt9gLw6AIpqFN9spHfyxrHpUmr7gIqrS2JLOvi64m61xHjSScQd2Nb9uPzAFlXtwbIDKIr29KNyn34om0gHUBRtIh+GiXQARdEm8uGYSO3BsqbqoQVcpHz14AWTgh/aQYOrQhnBGrPzz2d2aPi3deCo+k7DJ32BjH05YyWeuMjSi4EsJEaI6shGa6Rvd5V7qMfs8Mkcgm+EPe/WhlHn7hCPJDWRp3f3eU+8JtOv4u1GL//ZcQdqlZyf/txSzxrFs4dJ1PnFnNTGcA2zw6M6stEayShadvd+4sQR6jA7wmSOwTdc3H3nXP1BSUsedYxHkrhJNP2pLg5p5ph/8n7PZyJRLi5I3EKyqxejnNnhUB3baI2sQT/KIIrRbVWH2REksw7fSO/72dum01/fdIBHsnGvtXjKGiQ5opnDaBf09DVRnuREqgpftf2BiKa8xU7kDqurnRm0wDeCiJ7r6lBi2ClQkKxgxoCpyvad20n5Ojs/Dm43ZeMSAKx0jlnaplpyft+zpqd1vJoNE7lNdACQFVuvgy6zVGd6zCwyY+5736wXYziT3PpJRikHzjI0EY6tVhIegXoAtS4rTCzuIEi2bGpY5QvvRnevM/h7I8BJ55ilbdKWjNh4Oug4z+4wsfXKNwDw29b6y8YiIazZz8D6lpvZvma9mMs947qca3/LWWrHkqFhUB3ZaI10eP35ClAc11sVZodP5jB8YyUc4JGsfyKvB+gQtu3ASwD2+bRG6CG82GDeGw2AuXuNo/VPFg9dEYLRt8bu0gEF81CSVfdSIwC4PR9h/b/u+3STtkBIvQls34Al4YDh5VfeR6bXHvMq5VGIRwB4I9csRVWn+XhktT7YyWK107Hcb5HV8WQXyCbLPTFfcTHsTso/2Vn0n++/31h9GBFRcrWO06IMrEbRBSQT0Y8oJaLjSCeaVJ3rlIcjRGeQRTRqING0Er5vARLoKP4WplP8HXkIfxHRagh/ej5wkNXVTnwx72yVYWM8m0kqdtjZKoPo47sSFJWj+z5wkHxAmZJfbN267ai92aiI2fgBPMQDK0o1ACb/eumGf3Vx36vwB1SgdiwYGhbVkY3W2Iw/1x0PUIfZYZM5Dt9YCUd4JKsTebxnp06dOk0x/ICK0jY6gId4JJK2671i/SiIEZ12SAFg3O4stSNGYjhURzZaYyv+/u4nP0xThdlhk8mAbypeTzrCI1mdyC8HAMATz28gQCxtY4AR4NVsRJIvvA4MpkQW1wXf14BydHjpvwBWxDOrlJ+KvH0UwLE3zLo5PmNGA0W/vydY7XycH99t58Zld8wjyE4sLpBN9nhISEiIPiCkvoJ6uJ2U+R35Zz+vF24R0bRG+n9t5TGb6f1n7TjWV//SJgHEk/peE/3wTeZu5wbrex36rbc+9G+i8jbniUd0Eofq+52j/NGvrpu73Tq1o5zZ4VAd22iNnO/IkoYAgJmqMDt8Min4xoF9t8cjOSTPwmM2t93r8qs5iMdqGD9ahYqITllmYzfr1I4TzA6H6thEax4gs6NGGXZ4JBc9WN57Z3Rks2Dt6UdV+cWy4rga0+L6SGhR+X+WKn8eTCeom057HqkxO9pEKmV2NNQDGuqhfSI1ik4LuNYbK+svdqmz1H2IS7/lTNM7+tq1TeX6kdmlHXLNN+Z82jaBmio4HBJjAdqgyjE7CnR2+ET2eB8ruT0Mc5N2+1BewvIVoyR61by56EO9g6+d/Gpd7aLPd3zeoXq9ebG/euPGqJoRtVVjdjgkxgK0QVVjdhTo7Ij0fuzyPtZuEo0122QcmC15U+g88hx9rfMVIjK9sIeIZjQiIippNkQ9ZodDYsSgTdVjdhTo7Aj1fmyK7Ug+xmL+bPreU6RoKH6NkowAdILPdvXOv0MtnZ3CjYeBZ4NW8wvOh0Uq53R2uGQKdHYEej/2xXakT3YWQTcEN3fvyQMAGE5tjisVKurgdlQ6AGsaPAByr6bdYqa1Y9gFAKEducR3o/8FtXRvGCTmJr+gGrNzUxWdHS6ZYp0dAI6I7UhOZMZhoAfry8QANwaBos7Mw7W+eqvQmgYPAJxsPobVRVp8tt3z04/5NAYA47Vrl38c+vo6qMXscEiMGLRB1WN2lOrsAHCI97E+kcXTPgjNAnCpvGHXD6cChpcHvP9EC6+y/OFTe7QPn5IF4M03gr4yTEXx0PEDm41+aixxCwBQdnjnwf5Mrm5JH5UsCx5aav7QxsTfbudWqgazg4p8DbcAFzA7zhZoCv/5UYX4Ve6Jl5SJQYzzITK9TERkSts9qTqxwA1tw68nT56sf5AuIIOINnjc5/AdnuO5gIvdjogTZq32WsGd7JheeeKumswOx9fwoE0VZHaIiDK8VsorgxFukuR97IsKAro3wfsyMcCNhaIO/IwZVjR4gOWN5xi4c+hIAHU/mLGPz/xu8naop3vD8TUC0AZVjtmBMp0dAA6K7Uif7AwS+DIxwM399kLXKyDDu4UVDR64rd1w/2NuIjcDADoIzC+LoIdqujccXyMEbVDlmJ3LynR2ADgotmN1Ig0GEns9McBNLuc4BcQChoj57hy+w3M8BiKvzau3sMmOHAKA30dwvk93F7ULVYvZ4fkabgFVktmp23oAUPJ3qDw4rAxlABzifazcNL+8fF9uv2ZLPIVeT3enFfa60fZl1nEKF+d3Ndbe3We8G29GxS78ufDP0AWG7qXDNprvSwVOzHzW+6DnAl3Ggp9vj/VGbnKn2X5O3Di+N3RQ9x8K1+hw+O19z5Q2vwkAMxdxC87freZTHX573zP8eI4msV7gnjOd6y8v2VhHxk3zxHWJMe06vd4VwL4dO+v2n9ZC+fNIU1oDPRk9GeBGpKhjuBWgs6bBY3kydbX53YS7TzdS7wmAbDinajA7VVJnB9pjLGiPsbRA5f0aSwttIrXQ5FmgMTsas6MByo6FQwZLEPgsicyW4EpLJa6lnjyLlABMJVsqqSHPItNgSaTdoi+dHdpAZLYE11kqsS0LHySoZqkk36vJaoEUmed+o9UoVLY8i0yDJbF2yz0kWJotuUqehWuJfZBUtFSyrY3isKVSxH4i+mBfZcuzWMiDuB1iBHMkJtJCu6UACcw/LpdnEbQy1ZpIcSrb2ihwtMCBp4nop5kuk2dx6PJDZLBUEHugJPtgHKEg7mAeck7tL0Xa+LPJF29DCIJw360sKwKXWBzJBjFcHxIFNhxxHLQr5MFeR4oMlu6s6b9rm/+yschaH5KCq4sHFDDaLSJrJSY4VgSusThSUZUFTmijOFDgp/puYeOGdHddCQ58uMUGS5SF+UQH3UooDaeJTiGLlfzgXJT4Q+uUIUTr2qp5aJVyLHLZodW2V5Pjlkp/NXfz26Vs3+3ZRdlQvrJhsAQPdAfqmQqq1wIAT347sbUSAOBLSo9LugKXWRzJNjySH3K9miQKjPw1PuX9Ycs/clUJDh1aRQZL4GVYLMICBIGAFYGLLI7UVWWBUm0U+wWWzo/06XByxjKqVHkW2DJYsiCRC9jL5lmW1koQsCJwkcWRqqosyrVR7BeY0cILcFtsMFSqPEuFP7Lvl0YBQGkqIJBhecQ7F/gdJWC0WzgQhNnCwLMi6r2xYpJCCGKYuQhVgkklEoBxrsD2KdkA0rtXq0R5FjsGSwIZltUvbfn8a/SMN2u3cC5KiUP1fY4mDtX3OVo+/sV1nx7u9cZPKl5HWncs4n2Q1LZUsq2N4rCl0rGB2+Iix2fTA5NngRWDJS4M1xvipm8tHafdUsFaiWdFXIR6KAc/HB3UpjaK45ZKhtisVq0eKnkW7emHhnpooREC2kRqoU2kFhqzA43Z0Zgd7axVBrNzY9vFmk3CagoYHSskjiMgj4XT0rVYT53RvztcwezItiyydbHKpXKe2WFZHVlMEcvrsLVk+HtBPrND4eGzJnpcmauLf0uaxLEL8gAALJyWavp8HL+lHlRjduRJmDgefConmR1WlsW0JrXJ+dFdIJfXYWtZsq9FNR0QNkzWLbrZZkanrFuwDRLHLshjDkunpXE+ajI7DkuYyBtUkMpJZoeVZfl0oInyG6U7WgbH67C1DOgzeNiwIS1yZDE7HKMTHSxN4tgFeTgXCXUn0tLiiHsILGFZpGQihamcZHbe7U6Ui1kF7huJaMgE+fvO1DKFiGjpKXnMzkLfnuaFoABp7EYE8uSdisLZM8RtnHdqf1nqgWvMpln748g1SAxkSZhAPTUUhwtkZFmSymsDaHFUaS30HoBjNTvLtItoyXwtu0dKYzcikIf1WmI3Tl85YO4vHtNeLwBYpyW4gtmBLAkTKFJDcYrZYWVZGqEMQNE1UliLriVQ/MNYeTcE6B73g393XqJFeiNP4Om5942jNzzJbdx+FkInvrj98lQAuP1hh7ANpSq9yVJyKQ5JmECBGkqd5p9Mmxcaq7hAsyxLo+DTQPmRoiInalk8TCfvrFUXkM2dPPlIYTcWII+noQfeEDA6bggA3EZOWakHepkRH7iA2eFj6SeqXWCKUznH7ABuU3D9iS8mRr65OXD383ceUV5L0Zo5cm/RDUpiAY4F0tiNJcjjZ4XRaVieIY34QB15FjguYQIlaijOMTucLEvT3x9Pn363tU55LXuqV5M7kfP81psXzjeXxm4sQR4dKjI6yd4t1L4fI0HpOGNZBFupnGR2OFmW9LxnQ2smjXSiltgG8lGPw48eJSIqHFtGfyOJ6DP38lmUi3gy/8NsVP8gEVFJg65EdA5ZRIKNL2AHUWHTL4nu4ATRKXC/KRj1iMmpy4+xHxBRlw1EcT1TiYjS8Af7UuBE1WxkAycSM8KYcqLCWtscTyIqMKf1RaLiBi9Tj9eIdj9pUHLpxexWv74KfvuRNuDVnb+vmJFNxGE3PIljBeRhvJZ4RucCZi379sXV5SKnJSKKHdtIP+Jb9ZgdAaojYVmkZCKZVGowO7vn7P1rxKBcWjY/fu2QLAW8ErdbTw1TxOxkJHp08LOH3VQEediNL7bJqJ3bWOeKG8eVLc/iJLPDyrJcvNTuMZ0z+55Qs9kDYHYuBKY31p5+PPzMzuU1WHUVWjzUlkoAcOuGp6GJv/aJ1CyVtInULJU01EP7RGonO1qgylkq2XA+ErolsUZJNnKXb8r0ft8KpxJ/omgGXKCCo57OjqqWSnxLrtCOczo7NpyPBG5JnFGSWGJHFDMME1p1e7ri+tqJW2eoxezwRI1s5yNUiqUSBxWVLy2slzahOSpRZ0fS+UjglsQbJYkldgRRXivOdMGqW3iUj2rMDkfU2HY+enCWSixUZAyJIOo9pVJ1dswTSS9XhGsusBNpqnGRiGjjHkuJHRHjc1XK8sFHNWaHI2osUR7lE1ngG0FEz3V1KDEchYrWdSaiiFiX6exIH7XNzkeFMXldmiIvOb/vWdPTOgDIvetWvYGuY9j6NkBoQdrnZ5Pz69QHbsboetQBCi5l9yiMq9tBh7xUpJQ8Vr0sPqVdOx2fgVKTA+E8sxPh2GonLJUuK0ws0WHhOBTr30Hl/vaDdz5ilHNYJId3S2KNkliJHRbW4VR4cOskoqOLWKMlNsONl37w3f8N1GN2GKJGRcEdVS2V2MhL956x+u1vqVJ1dmbU2bJly9pJ0/N45ZxznuG0KY8uIK908k4TEdHV6c8Ag0sYiR1eUIdT4aECJND9VvuJaMEQE5PBEDSfiNb4qKWzw0nQWKx2+nnkgla3rY4nV2eHPbSeRN8yKm21sXJ1dmq8CoBe6xLLKecwSA5we/SCHgCAZkuQvXX62kmwENThVHgAAFGXegHoNed6Y3OGEzERAPzVY3ZYokZlwZ1dmw7XgzIlH+sdvNHHE9UGLBiJStfZ0b2bvF2gnMNYdTBuSRWMkgSwjhDRERot+QFIhi9UZXZYokZdwR1VLZUAAE1RH4BPSgkqX2enCHqBco75apR1SxIaJRFmSZI9AqMl6AAE4g5jxwRVmB2OqFFVcEdVSyXmErxpDoCSAC9Uos4O73wkUM4xAhC4JXFGSWaJHV5Qh1PhgQFGCIyWYATQKSQCwNGycpV0djgJGtnOR6gkSyVWtOeT3wGKfg2Vp7OTPqaBbtzUqWHdJmdzyjkMknOsr/7V5PN+3iPJ8MTqmT/99u9ZJkZih4N1eEQncYi+59eUHTZ57dszS9gMlDdy4nezxiM0TSVmhyNquNVOn+yUNAQAzLRU8lHG7LBQUdmoT45PfrvswensVFDOqWCUZJbYkSR7LI2WSrICCvLreOtUYnY4okY2ylPJlkq4kNyytaazoz3G0h5jQVP10EKbSC00eRZozI7G7GjyLFxk/cUudZby9BUSH3Zeu3UUPv3cgIvnPYxtbuV7GL36AziX4l4+BOrLs8j2LKoU1INNZpHUYXkWxyyjKuT2MMxN2u1DeQnLV0j5/wiID3uv1Wj2WdTn0wHvnIkrOte7NNl7PwBUn95pDNSWZ5GPUqAyUA82mUVSRzEPbv/syrtUvLcw1vyQ6cBsyTsJPPFh/7VJn3nGEhGFEREtwY9ERLfCylWXZ5FEKR4w6sEmEyd1WJ6F2z9JeRdp1IPRTzFJ/9jwgo2JtHxtkqHrEwVENImIyBhUP5vINO62+vIskijFg0U92GQWSWXsu3n/pOVd7FgqLYJuCC/MYji1Oa4UQsuk21HpAICymM3nSLQAIPdq2i3+WL35ziROWyIi/0Ng64v11JdnWTgYxXinI1RDPW4qNG0Sd2CTiZPKD7vyLhITmXEY6MEJs1zuGdflXHuDwDJp5uFaX71VCLAkB78AsDQIG03XbdjOLrdesGXP7ePDobo8i3MoBVyGerDJxEnlh315FyuH1mpTJzzTVoBvlD21iOiP9oUc+HEBx4nolbE8ycEjHQIaxHxoJaJRPmnmQyuRMajeO1nqWSpxhx5plOKBox58Mi6p/EOrKXg6kbE1CuQcWvXLvo4LBPDlrvQ9SVeAE+cGAi/8/UhUpldMTErNMwCaAOi3oZghOfZc5xYA3O4daiEIs7LeG0b24LriTue6UF+ehUUpUAVRDzYZn1TBpWLkyc1n5j3v+4jMQ6vuTfD4xlUGsrGwTPIzZnAkhxDpYGgQ4bu85a/F7LKfk7CHBHrhFEoBl6IebDJBUgVhT95F6mRnkEBrpR1SABjvC8APApDh3YIjOQRIB0uDCKPj4s8uw6XyLM6gFHAp6sEmEyZVEPbkXaxMpMFg/kbl8I0OL/0XwIpcDvwAYgFDxHx3juQQIB0cDWKOG/kA8FFvVsnIgHKoZ6nE2x85gVLAlagHm4xLCmX2TiMnAz96jHL8hsClsY30gyeXkVBrJX/0q+vmbufAD7rw2tcrNg5aXU4cySFEOhgaxExkjvB+dj0R0c3h5hVLe+uf+ZBcIM8iiVI8WNSDTcYnlbfv3P5Jyrs4gnrw+EZZphn44MAPw60AnZjksEQ6XHbjWAq9kEApqgzq4WQZUvIuGuqhoR5aQCMEtNAmUgttIjVmBxqzA43Z0c5aq4ClktXI+g0AUL+zNyxkXG5Fe7iVG4e68xs1fNLXZVNgAa8YPRQ6HznE7DhtqcQnk2P5xDE7fHebSJKsifTynx13oJYxcVWNNT5iGZcaPh/Hb2yp4zcqOT/9uaWeLplGMUPDojrynY/gGLPjpKWSMJkMyyeW2eG6s9ZMTrqec08X/YiITKsey7GUcRlXy2Kjez4T1Xk0aBOJ4VAdexImipkdJy2VBMmsWz7ZZna47qw1k0wpbNsTSabeoyxlXAROScxGPX1NrphIMUPDoTr2JEwUMzvOWSoJkklYPtncd747Y80km9mx8+06YsNNsYyLlchtonPBgdWCoVEP1ZFgdpwsUJBMiU8T352xZlLnOxKC57h/NwSM11B2fvXr1h/Lb70S5YpvSA6J0QNmVMcv8YV3dUBMsufV17pBDWYHQHHcYLYtM7G4QD6ZIssnvrvOvTBx9togtSeyJlIBlMUAd9u5lVrSygXzUJJV91IjV0wk51jUyPyu7fvFs+wpz5Go03w8slof7KTSMMv9FoG1VJKXWFwglyz3xHznajGFZz/qD7UnMh+BEMi48BxEFtUDdPNcd/UnRmJ41RO5zkdwkNlx2lKJSabc8omphbFmUvkW3fnqzwhlXPgXDtxw8WW8GInhUR25zkdwkNlx1lKJSabc8omphbVmUnciS8Pn1xHKuPCvHGzs4okUIzEcqiPb+QgOMjtOWiqxyRRbPjG1cNZMKk2kWbklb0Snj0QyLlzj131+gKHc5LqJtEBiWFTHMQkTBcyO7MTWmZ3HQ0JCQvQBIfUV1lK39QCg5O/QaqrcEEgdXQ9jpk567ZVtljIul971w+ipY591e4JS32uiH75JnUs6+8wOh+rYkzBRzOw4Z6kkYHUkLJ9s7jvfnbVmUizPUvXuX1sgMSyqY0fC5IFZKqlVBmvNpDE7GrOjBTRCQAttIrXQJlJjdqAxO9CYHe2staoxOze2XazZJKzmpre4Fb9kv9vgxi/Z7wqMt29uvVSjyaha/EYVI2NfzljhDftrsZ46o3935j+orbOjnqWSlWvDHH/nmJ0Mfy9Fdkoss2Ovv5WJpPDwWRM9rszVxXNzpC+dHdpAXzo7lJtI+jZ85kSPlHkeJ21MZI3i2cOE+1/T5+P4LfXY/6AKs8O27OrQOBcycBvrzM6SfS2q6YCwYVDG7NjtX/Em0ey2+UREZd2C+XX3kMD8w8Qc80aGHl1t3WHKxQWr0i/jfNRidriWLR0ahfcFT93jFq3jNnKYnQF9Bg8bNqRFjlJmR7K/JLNz0u0QU3yw0B4pgfnHYqNYmxOZ54KJFDM7bMumDo2SiTT91u/dIm73reM2cpidKURES08pZnYk+0syOwt9e5oXggLAK+1U2KiXeaFDMyDvVBTOniF267xT+8tSD1xjtszaH0euY3a4ll0dGnlBv/Teve7bGmxTDm5jndmh9wAcq9kZCpkdu/0rTuTxlsw3tXskZ5RkZSOmoy4CnF0Ss3X6ygFzf/GY9noBAPy2tf6ysXAFsyNu2dehkXOusr37Hz98zftnysJtrOvs6FoCxT/Ific4eR67/StMJN3jFEDccam8YdcPp1r5g+U38gSennvfOHrDk+zW7WchdOKL2y9PBYDbH3YI21AKFzA74laj4NNA+ZGiIudHMGzsmvjTUsH5ee6JlxQXCIADgBYP0znFD9nsX+GsVReQzZ05+XBGSRU2ygIA/Lvc3WT6EozhEru1GwIAt5FTVuqBXgJ/JajP7HAtXeSbmwN3P3/nEacHOD767SixFos83EZSZ6dozRw4ww/Z7l/x0DooqYD52C0QGiWJIzTpHgB88fGOXxYBjOGSeOuG5RkW/kpQn9nhW/Z0aByOx7sf/FN0gJaJ20jq7OypXs0pfsh2/4oTOc9vvXnhfHMpoyRgXt1wAND5N6yrB2OXZLF1sncLuJzZ4Vv2dGgcjvrrNh3qtkOgISMTt5HS2UFsAzjFD9nuX3Ei/b5f8gcAFK0aU8EoiVO08v1+aRQAlKaatXOMEOjyADFA0dr57gJ/Jfb7x/x2GIykDrPDt+zq0DgeASt+PNP1O25fZeI2Ejo7wEV/Z/ghu/2tXMmkDXh15+8rZmTzSjuJQ/V9jiYO1fc5yjsvDRi2NTp89LFxxNolcVtfwKxl3764upx4fyXmonNsI/2Ib9n/VGF2uJYtHRoF15F5CzqvLeFaEriNPGbnqWHyyxDK80j0t8nsZCR6dPCDLaMkANcTqE1TNyu6PBfbZNTObayrHGaHa9nQoVE0aNH/tv1cT1ESCWYnoWYzp/Zdor8LmZ0LgemN/wlPP8rddP+vmZ3La7DqKv4B4a57iIp1wSfy1g1PQxN/7Xlk5X4itQfL/5CJdJ8HoMcDPzD0eGgH7VEVyuhhvvx48MyOFk5F8MP5kwHt0KpUZ0dgl2UUczuQJnjono95If5E0Qy4UmdHbWbHaBWxcTx5RbqGAX5k1sduznVzWmdHYJc1WMjtAMj0qyZB8HwxJ9V8MVk7cesMF+rscC3ZFlbWb3RF5rnfaDXKEpGRk7wiXTPLuFRufawnFocisYU597M63i5LyO0Q0cxMKYIndiJ7hyvKx4U6O1zLtoWVw4NG7CeiD/ZZIDLSya0kqUDXRNeeaiOF9X1nPbE4FIktzDmdHd4uq0A0kab2mfYJnkM+LtTZ4Vq2LawcHnTgaSL6aaYFIiOd3EoSS7rGDPxIpoAt4xYeRWILU0FnZxF0rOkjg/KkjT+bfPG2VYKnKD7qLgBKOZAOVAKzI9vCSiIajjgO2hUiRmRkJa9A1yybqoPS+ngUiSkMzv/2I+Mwd8nDojzJDXAqOtUqwfPz2n4ZwI2XfvDd/w0qgdmRbWElEZ/qu4WNG9JdjMjISm5J15iBH4X18SgSU5izE1k87YPQLP7NZFCeF4fhjdHPWSV4QpYAMA4PmtNlXF/A9cyOFVJGUQRENP9ut8kCkZGdXEDXMMCPwvp4FElQmDM6O/ploFe4lgXKY5XgKQRwKiYCgD9cp7PDtWRbWElE5K/xKe8PW/6RCJGRm1xI1zDAj8L6eBSJL8zZQ6vuTW7REuWxSvAASIYvKonZkW1hJXHJPD/Sp8PJGctIhMjITS7oygI/RoX1sSiSoDCnfx85iF0QwjmEWdYJHgAIxB3GRwuuZnZkW1hZj5QWXoDbYoNBhMjITS7oygI/bRTWx6JIgsKcmUjWLsvM7fBwTgCykWud4DGgHJ1CIgAcLSuHq3R2uJZsCyvr0T4lG0B692ocIqPEH0vQlQV+msquz+yJxaJIgsKU3xDg7bIocai+z1HeNIum95+1wyrBkzhU3+8c5Y2c+N2s8QhNc5nODteybWHl8KDHBm6LixyfzSMydvyxrCYRdmWBH8kUsOmJxaFIfGFq6uzwKM9t97pSBA8AoCQroCC/jrfO9cyOPHUbyUENsVmtWgEWiIxkcqtJrNI1Uins7DuHInGFaQ+WNZ0dLaCpemihTaQW2kRqOjvQdHag6exoZ61Vk9m5Fe2pMzTpfOVvT50hsI1DKSuK77iC3RHDL0aHvKNUoYIqUWdHDq8D2J7IGj4fx2+ti1o1xt7/ziFrKGviO+qzOyL4hbOMsusdJeOOA8PLQD6vo5rOjqgOu7yO3Vt0DOLRu6NjN72sie84xO4oZ3Y4yyhb3lEyB2V5GXu8jit1dkR12OJ1HGN25E2kHfGdQz4uYHY4yyhb3lGyB80UTKRtGMi1OjtMHbZ4HTaFw18rmaU602NIp/I6yTm9r19ubb6bWBiT16Up8pLz+541PS1Gd8zrii9mP+sDUGpyINRkdiKYZriJsYziFuASRihCYQdGKOeyQp0dMCDR911p12hnriPLExMTExMLASSENfsZWN9ys1BG50SIR8tJ61ihHWviO7lqszsW8IvOvTBm9togwQLgQkYIlayzAyFIpPhkB0B5LADcqwH0fbpJWyCk3gTM+iE0CL26TF2H/OHf9EB4syFPz91rHK1va4Hu7DWO1j9Ze0k4YBw+YA66mI5AdW8sgWWUHe8otcaDQm8sKNfZQUDEa9/VCYEzE1ntHQDYlA+g3htf9cbuhUIZnahMrxig5pm+ZkzHqviOB9RldyzhF84yyo53lFrjodJ1dmCX15F5i27yr5du+FcXyuhk6GoD+PF5mDEdKXRHVXbHgp/hLKPseUepNB4qXWdHBBKpMpHteq9Yz1/LJHu3QHtqGBQU1FHHYDpS6I6q7I6Yn+Eso+x6R6kzHipdZwewz+vYn0hWFsecYkpksRkHYGV0evX6BsCSmwymYw3dUZ3dETM7nGWUfe8oOWHmZRTxOqrq7JjrsM/r2LkhcGlcgNeIbXTi3To1w34jovI254mIOBkdotyRo9ZO38UK7VgT33GM3VHO7HCWUba8o2QOyvIy9ngdV+vscHXY4nWUMDvGj1YBgEhGp/RWYzfb4jtwgN1xgtnhLKNseEc5N6hNGKhydHZs8DqymZ29d0ZHNgsGlMjoaE8/qhCzc/W3I9eD/1EyOv+scPgTaTpB3XSAIhkd7ROp6exoE6nhkP+/wgNA8IPXXNM9tIPqqkCGYFSFw6oW0HBILbSJ/KfF/wGhrd0shtRSrQAAAABJRU5ErkJggg==',
    '1407.3745v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAtAAAAByCAAAAABfd7mMAAAkFklEQVR42u2deUAVVfvHv5dNRBDEXRHBBXdc0HJHEXPrRcVyrzQtlyxN08ol0ywtK1vU3jB/aqKp+YZrueS+4JoobomCuAvKKjv3Pr8/Zjszd+4GF0Kb5w89c84z51nm4XJm7pwPOoImmjw74qClQBOtoDXRpKwKEc3VsqDJsyBziUinraE10ZYcmmiiFbQmmpTOGjpYy4Imz4IE82tobR1dbCnzKSxVB/+pbOhIW3Jo8oyJ0z/vQmYi97+Lj9u/8QpkMR9nDqWZgXtZLigoaAIg6045XV7dcqVoO+ex1K5Ugfs/PuHR837PQkEnbzhyuMbYCpR8oNanrf59BV0vNbi6y8GEgM65Dw75XS9Fw2f3rXzSts98AI+mb6827CNTBR0/asIwe9uOX348ptKICsi7c+rWkilc34l1v0f52eWmEPQPSyyGEhEZlrn8rBgxfGv5bGt0SlqKnkK91zUi+hLLiOhi1dJ18GN8ybfaxZhO7Ba8XgLZOIn+RESUM2qy0JWCqOIHWSbW0K5wBADdxHmjjio+vS9b8Ql/+Wn+gE4d3BCAIxwANHshp1Rtj8Iabr2T2ryl6cSGnfmuBGy78YsD1y+ThC5nPHPPoad4T5TfIEdZcVLU01zPSGJXWYFJpWq7bkjsXwCAX4aZSawuqEKJeVCQj8peeHa/WHENjT0CALf23zAAQOxMYUToAdKiD+UnAkD+6Ut6mc5TKc7tmYMu7EhhUhKyE/iohWjtK6OxCgCwL4QxUfDgMdKeSInVP7rPLU7jDjyUuyJeiiLLkT1ALwBA3l3ho0yfkIG8LcdYSzYYKmOP7epgP5A/ba9HVI84YOOHuftGjlzL9ABfrfEs/2kQQBEDs+M7H5N0nlJp0Jo56LCro2/jr4GRPs1W/96m+rxPZm/rOSMfUrR2lnCP9bkAYps6SiZ2BtZcsex730NCYq+0q/o2ABxst9+wfp6ecUW4FMUpaAD9Adx4ee6hz/YCAL5/ce+kN7/w7c9YsslQWbgpjMMIvrUYo4h+8r1CNLWVnogajSMitudqMBEZOhEtq51OtNsrVdR5Sm8KOVmCH7hGus9oItK3eEhETaqfJ3rS5kUDG619HXwDG4lo6nViTBgCwo/er7FBSmzwICI66HiWKMptr6QnXooiZSMWdUYMbo/tRER0xeMQES1CFNGv5dKoMGAa3ZA8Mm2orN4UMj9eMABemVlASMwNqVvqeRgTA+hGIX3moIpAqMP6Z+whXsVxmzOBGxOqAXDrHQhU+GjHbyUX7SisAgpu1wdjQudxo1ON+0MkJXcANDG0DeDe0EfSEy5FUeW5yF82vsAvfVp3BTAEADYGeMKx9UbUkzyyyZBT2bqed1EXGBSuS7gQjXSpW+p5zq91s9D/jMVf6S5HAVS59qw9lh4zb914rJsqdYRg56ASi7ZDwJ67tXf2BWQJbWGseOdybwChMTgg6gmXohg3cL7vEKC/VO3EXLEWvVIAOHmyHr1ui6Ey9gl9Gz0BQ0SHyMqd2G6px/XQR27LQl81JKKaq6ur67r3n7WCrjlwOeUUVJQ6PDziUGLR6kYbfsavLwEyE97GitdRg2tIesKlKI791s2Aq6euwlPsmfjoIjIPzGAt2WSobH1CP9oV1BmYEXGuPo4CBp0OAFaOYXqul583L3XN9DFNULktc+LKMc9MRU8IOXaD+X2PjMxGUERrR3ll1urR7u6AzITOOLENcZtrSHpx/KUozsuatQCcaOqDLLGn2qRvqz1YNoC1ZJOhsvUJ/WlhhA6Z37xWH0gC9hyDawEQz/Zc2gBUmjLoYstaewEgbxt4nWdGujVeFhvIP6oDgEPoCyZaO0vtF65NHAlAzQSb2NrNzwFAXpSkJ1yK4t42/bdJncYnASAHAA50XfHpygEyj2wyVCYKOp27dJmTV25rAziXLwQQ65xz3xutrgMk61meDSC/veuKHRcAfFsHvM7TLbnIEz8ex29qwzcPPQCy54YPBBOt/R9Fx3YGwJrIz+CGhMQWZAG6FdG/A/ivD6PHX4piXXPkTL1bSRex9yJAy5ED1J0esT5qb7rMI1sMlYH3oc/PPX/TPbQCPU4Onl4TAP737luBZ/v9dC54gWN8yGhdcDDTs/3HoGaeR2q+AxyY1b3RxW59AV4HT+3rxjsjH1zJdmtSY2wPAEBa66vce0JtA/zqu/3UabYLmGjt7WBurWmzuBZvYvd3R1yDhr4GIbGnFxxFh1VVceKtPi2vNh0kubJFvBS2Z+PMJzG33F6ogPxbf6eF7AOOzhroc+K5IVWDf80ITnEqSC4M/78KoiXThkwUs66sfb7prxY0daEMTwAFsX7ebE9Geadr+Q1duUciTwJ0kHSelffnU5bP5hptm6++ldlIuMsRorW3g3vaVIEJE4rEPkgLcGD02EtR/GzcfdTUcLlKFTy3oj1gOPvakLmiJdOGnpaCxr92x0r6jTb4YrCfWND/uh0rv0XsAgAs/2O7tmPlGZDP22Zk5voJH5CF/74EtL6YCAD63b3x1L7gr4kkvc5G7/2IXwp8ezVxwMIm/7IE+EdNb/qcW9y+XhOK8dtBW3KUnd/oSXeb868F6+GIQgeHf98m2eSEHB9/h+IUs1bQ2q5vbde3JpqURXH8GEA3LQ/FlW6ag2UgG92gkZM0gUZO0kRbQ2traE00wdNMTsq4BdSsDABXyjkVUAPz2jcLnXmMT8GNco551SsqFZKvPKrVvowwh2yUpC3XyXeA8sWi9NhHFUPsnUa7iKG0P+UM5PgUFPSDLVfWu8fUB7Dqz3Pd+k43r30oem12nVcXAMiZs7nagHeNCjrh51VvtzfLHCoJwo8dRL9g/devOkS/2H2R/G2E++t+7hNimUtkWxqLWFD/u1zg8HZV/ujg8k0qQ6kRGTkFkxoBoJUX3R1nFB1tkPTOj9z7/K8EtXO7Ff1CCI582LdqBQAYzv8oGVbccc2eUlWub6XdEtwkG/syuhQSEeV1N1jW3oR2eiIietIoWVUhcLJ55pDthJ/S2CRbGB6YSkSU3bNrtmLohUFWcYlsS2MRdvFm9pyTT2vD+KOs+v1UhpInJRN94bKXKL/vCD1tDEwumrHkaW+2w32uXR2A8xwD0Q/CQwoB2TRsPtHZgHsyfSvtlmhB/zCZZ029ZYW2oQ8iiIho3mF1hSBFQT96k9kvPSKbDGeelMGCno+zXOOOs3J3er9BRFZ4bVsaba8xw0sjiKiBgCFbwBS0NLTEdRNRMjoSzXfOIKJO44tmLPdm4RKhoHsu+3BFAhHROz/+cfjIkcPdEnilLZ55RPTmSzJ9K+2W7HLps4BZVnO6dEtd308GEHe/S9GYQyVJ+CmyPFzYkX9dv3Z4hFourPHaljTaLvs3TwXwy1bu6IR/FbWhGgSgAlKAiEAPAO0js4tkrFxdablcbeJnY/0AwOHN3l06d44f48cPrO/kAiB4ezarb6Xdki1otzUFrxWIRxbgP/Vmp84AaM58ntMTzYHeBJAPLDGH9I/uA4XJD1EY/xhA3s1Chd3ig35sl/U54o1fCK3i/UtgEHYcl0jWbZQneRpN5bGo4f3XsxWAth0AALn/G6Y6NDTzZeAYXkTaHXcAqPokxn5JmgQA8cdHCCnZ6wkANfMOstFZabeEb2jbv39mobBYFzg4y9r6tly9ukOd9rsx1KfBUkn5vUarj2JDaFUA2B12Lnf6rAKIIB9O5aMavi8kQ405VPdKu6pv42jranP/+OL06NcNP/1w7PnlMrt2AP3YLgdQT2jWxwHev1NDRgsPaDguEdutBkli0iiOK9NY1PDogP/tWQs/4AkJ372jUx9yBvK/ajsb5R2Jq5urxU/O+SWLl6QDqA/A8M4iwXJyqjsAeIKlNlhtt0TX0ES5LZzOcos/Cf6T4PgN0V2nnUTU6Sh7wn40Tw7XExFtbZhCZJgwnEgC+QRNJopput8Uc4gn/HQMWU0Uh8kxRKvcUqgI/B27LlFb4zeheRbVJP/2CWtozmum2wiSJE+jNC5Po4XwTF/jNHRarKf42n8SEf21kuj5fqpDJ2a0G/eEiFo2IyKaiK+KypES19DN1xvot4Cb3EHkB6LCNUwhIorBx6y+lXZL+pFjuZ/xah7Aknng98IWoJrXVsDQTobf6D7yYqe5DgBy3hpSCdCNX78PMpDPpYiT3U3bcgcAj9hXAH/nxJZA4+wEFJG/A/vBMMXlRQ4cJP8uyr2WutUhSWIamXF5GoscXgZODHaAf883c4GCtaNMDeH5RZGJwx4B8y8/APLPofi7D9YN02GgO8fUyf9wArNj2IWDKcjgwlbaLfFn6K3mXpoLcByco0ePV7kGYMSh29gX/msejiru/2aicSAAxNxpDgDNdTsABuTzZ+gn7hYNNnUAHF2aAXBFFmP3Ob/WzafsG4vSpk8mi18NwVfyL0vd7SxZnlTSyI7L0ljk8CrA1xdAy/jjwNK3HEwNAbqAyD199Aj78o3b8Qv7oUqxcxMIAG1/SwaA7Tpfsd8DegAohAerbaXdkv9S6IO2i49DTubpX/4XHPha/wd+V2xjLo/yAIC/4QoADi5XAAnks++i2xRYQU+X/tUXlb9jNwnFX0LzLEJlnqm5rTcJSeLTyI7L0ljk8Cq6eHOpv4TLeZXT0tIKC9IyjYYAAJXbndkNTF166MC7D9GyuKk5cZir3ksAsNpfGvDiPppzGZ4SYK3dkt+C5fRz69e6ysk87gMjJ5Wr8PK6PmRiK299PAGAgrwGLMin15QOnfoOtdF80fg7dpPhM3flu3A3Kzuc3rDFX/U0suOyNBY5PKcWWdyPUhUk35kN4HK12fWmKobyOxeecAG8EQegbl0g0a/YBR2WnukC5KMygMKDodKAZ42HAPAAz8lPsMquQ4n+TU/uEn12/a6CzDMydkEYRm7f0MfEmW2qRgPAafSWr5Gfnzve1gdTRePv2E0qLUpaw7X2xLzvZ4u/6mmUjbNpLHp4YTfyATxEWwQvXbp06dKKzZdOBVIy2aHcMxeSANxGSxwZmA6kH55T7I/Clj+4ALji2QTA5Sf8r+GUTEDXLwEA4r1lz2ystFuSBX2J/zZgShc5mQcIrba1DbpW+6or1HE6FX7cFA/oFw1+ERLIJ78AmO4xKBWmmEMFWbwWqKAAQAH0ReXv2E/GT3vvMABcGjtsHiT/9AAKCkSvpW5jSJIsjbJxWRqLHN74CrsB2j22If8UOCsDQJJPe3aoYt/9PsCtS8FdsGPbbeAb/1eKnJE8/oq91ATAzUNLnADc4++NObNTL8cD9NtUR1bfWrsl9tjuVKine4c1RER0YxQR0f4OM9dM38kNvvMJEX0wXfGYb1AT90qhs4mIaE/3z34Mm5tPtKuvR9Xeq4l29atYZ2DSdk/3xsulM3YM7Vbdo3q3oX8S0akwb+9+W/7jVSnsxOru7r4DU99v7dFqkmQ3qves9Ts/+Lb0geeR/nNOnv+y7vd6Ijom+Of38p4w78phZ0+FeXv3S2K6U2V5UkkjO86k0UJ45hw81XbzmfHDM7iD8e3dPXt8ThlN/W+zQ8mvfnns6PP97hNd7Hv4r/dCHhYxG3nDe9bwaNp/CREVzvkmZkOzrwxERNswk4hIMLsh+FLKtJH5Mn0r7ZbqC/4SmSfNyR3IQEUzyvfSApzsaddq/g7s/Uq7/thVQ8OuzkXJk4VxJo0WwjPrYNqhR+0ClZ2RHevJhmLOU+tAHYDHu9Jad9DZ4wX/yye8O1TnfvtG9qvOmk36PatDa4UN6+xqO1a0HSvqMnue4z+RjWKY1XasaGJa7rg6PpVmtYLWRFXWTH46zWpLDm3JoW2S1USTsipaQWsCjZykCTRykkZO0kQTaOQk7aZQuynUbgo1gUZOslJSkp2dDAUUYCcHshK4/139TLnCMJNU8EkW57eMV7qwW1dxiPsf51yqjiiHUmdGmY/YOrFPBAaHf2VBJ2w+fthzSBN7FfTjjccO+Ix2w6N9tRYGqhuUmEkq+CRLosArGUvhGIevvE/3KRzxvsv61zag1JlR5iO2RooQQea3t+63mFxd1scRk2R8IhFyBCiGjPFHsAKWZAQ/ks6WWjJsEgDDmts5hjcamAI5ofhv28VgoJJVYtNLbArtOIwiIsoJLX/axAkMM0mJT7IkSrySsSwOLCR6E5OIqGMlg33etjMJO7I2UcZhmjnTmgjkDiYPvUHpL3lGs30cMYnhE7GQIzm6SBV/ZNqYGfiRdLbYYrBJXOBvXyC61z3aBMjJHuSkWAxS9DwcZ8v5Cu3bGENEREfRgywyk4JsLGglXslYekwhojhsIaILR+30+qhJ2JG1iTIO08yZ1kQgd3BCAhFlevsXMn0cMYnhE7GQIzm6SBV/ZNKYOfiRdLbYYrBJRES0/WsiovOhJkBOSrv2eUMzyh7adfC3/ZdUCrxSXWONNE8ANeAJZjdu8eWzP2b1aVrcRFl5pu0R7NhyqRLcg6OuNhO7eGISzyf68Ss3lFMkSxpiR6pNhEVYkvEMxmeLLR6b9Ho2r3QyGwDqxwPYv/ksgF8KWJAT7PZNIQP7iZ0JOdRHoB3JOUH8oKQNBZOF25F1a/8NfqsnxR14qHKblyjsBBXoSqpmoIJXKhQpTJIR5qYoJU5v2izswIwyFboQiElKlHhmgV0iqJOXD8AdSWIPT0wywycqPjLJqhmMsEm1v/7GAGzvC1WQk/0KmoH9bPwwd9/IkWslqI9AO5JxgoRBUVshZz/o9RWA/Gl7PaJ6xAHAwXb7Devn6eXMpLyFEYdfm5UPSHQlVTNQwSulvMhTmCQju0bGbxw5cuQYfDxyRF//gEylWdiTGcWHboSOEgIxTYkSkiZqFC+CI/erA7jg1Fzs4YlJZvhEpoYE/JFlUZtBOptvGWGTRvm/2+3vbZGfqoOc7HJTyK+hGQZQI26BJ0J9RNoRoyMRgRop19DtFi9eOKT6DwYiop98rxBNbaUnOuh4lijKba+MmRRU6y6R/uUwg4yupGpGBa8k+CUZ4ZermYgiohVIZc0WZw2tzoziQ1ego6RAzFCi+KQVNQKVa3wKU8W2SEyS84lka2jZkDr+yJQxk/Aj6WyhpcAmEVFiR7h0yTcFcrIjOcmIASRBfUTakaSjTgTi17bvvffBhlPrQhMAeGVmASExN0ATQ9sA7g19ADDMpOBagMOMbb/J6ErWmZEoTKIRuVQBFGZhX2YUL3LmEROIZUqU/SLIGxP6qdCWiElm+ETqQyz+yJIYzyCdLbSMsUn59d9zOPLiAxMgJ3u+badkALHQnxZKHRNEIEl8t57qnQkMehyUsPUI0nHncjMAoTGNAZaZ5AIAQc475XQlq81wfolGjEVmFvZmRvEiYx7JArFMibJXBDOrbhF3IErEJDN8IvUhFn9kSYxnkM4WWkbYpNhXv158scee/gZ1kJM9C1rJAGKhP95KHVNEIEm8X7i2AzBEdIis3AnAddQQh5TMJJ1HnJyuZLUZzi/RiLGwZmF3ZhQvMuaRLBDLlCg7RbDy3u/iFxwsMckMn0htSIY/sijKGaSzxZYRNumNRVVQf+/Hp3aZADmVGDlp5RgW+qOzQARaOQYqsMWHwIyIc/VxFDA0wG1xRMlMooxGCrqSJfCQ3C/RiM7Iz4aMWfszo/jQZcwjWSBmKVErx9gtgh3n1zngogP3ZFFGTDLDJ1IZYvFHVohiBulssaXEJmVc6gJAN3fftb4mQE4l8YK/awEQrwL9gSoRiNM2ktyDuhBkfvNafSAJ2HOz+TkAyIuSM5PyASC6sK8JupJZH8QvfkUjx4zGajNmYXdmlBA6yzxSDcSIEiVPWnEjiD7+rQOwtRyHKmKISep8opRMU+giFn9kUdgZUjJlZ4stGTYpJRPldRxoz7e5OsjJHgWdz1UVwwBqdR0gFurD0o44HWaQ01biknIn3JofCOfyhQBinXPuV14R/TuA//rImUmnHwKGL8MHyuhKqmbU8EqcX5IRb/7cfHCopQLoJLOwPzNKCJ1lHskCMUmJEs60SwRXRz6eMH7cKz/58agiiZgk5xPx0CJOSTakhj+yCEtiZuCmlM6WWgw2KcmnPZyHfAMAd1O6qoOciv/Ybs/ARh5eIRNksJ8bdT+ed1CC+gi0I5mORPwRtImI6PIAf1QMHzGiX6tefxIRba6z6PdP/prY4YNCim4za9P8zXJmUmjijIjIQXPyWLqSCTPGeCXBL8bIrr4etQYkfNXRvcmrNLqJe4cvSDBb9Md2ZphRYugydJQQiDlKFHdmkSOQX2P+z74EiqgikZjE8IkkaBGnJA2ZwB+ZMKYKP+KmlM5m5pGwSRlN/W9T7mtvHjwfMSqR1EFOJUROKoj187YM/REGRW3Vr4quFjR1oQxPAHiQFmD8KyTxSSMns3QlS+AhhRGV1+XUzKI4r7SLLgmhK9BRVmCiFEmzMQLTDnKoIpjnE0V2rGdqSMIfWTbGzsDZlc6WWgw2KbJjPSDudHZgO52C8SSCnDRyErQdK7AVVWQbz8iqbJSQXW3HCjRCEv4BjFJJ2dUKGhohCf8ARqmk7GpLDm2TrLZJVhNNoJGTNNEEpbJJNlin5aHYv+00B//5bATzBd3toFaQxZSPP9Yc/Oez8bF2U6jdFGo3hUWUrWla2WtSOkUt+4HKuOfsTIUFNYXvVE/vedhweOXfwi0TjiRJvfTIq5u8y9D8jBsAZO06k+o3uB5tGsLMjUz+vbLKNcTV18RR/PuD/JibL2M3act18h1QR81f62hMZn2F7VAjE59JCt+FMF18uB3NOY8l1ZrSVwhiWrRPaNuoTjpSeR/6wdpDx9zD/YfyBTLz4fwal+bW+TXcMuFIkruRa/oriuRQOzcA9NNHfSY2uPZdY/dvhjBzI3nDkcM1xlbIv3QxeC73fX7KT8ILsckbjhyuOdYlKzZ9cjgXlX7Bz0tGOh7v3XORq5G/AKyhMZn1FXaDGil858Ok5AO1Pm0FIH758ZhKIyogK/FQxlWRAiSlpeREyTtiSUoiFMkIYSQeMwQkIxiSOUyTEe9I6pCUlGYBgerEuikylGDxbbvTDBhpa5CeiPR9g6wiHEkSMkTRMXovERkmeJ4kIqJVTkHyuSkWQ4mIHgf5cG9QRaBKvrQpdxgR0V7X1/VERIXhTR8SEaV37Zqt9Nd6GpMZX22FGpl7YVHhOx+mYZnLz0REdBL9iYjoXuAutZTbiKSyfhevgnckkZQYKJIRwkg8ZghIxjAkM5gmI96R1CEpKc1KVCcW+CQylKwgJ7FgpJe4FxxPBllFOJKkl6JIshsXEtES/MznJixIPjfFYQQREW3g/x88HLuk6uT6hmMjEdF8cD8VdNNpnDrIqXi+2gw1MlfQCt+FQ1rocETm+5Yf1VJuI5LK+oJW8I4kkhIDRVIijKRjhoBkDEMyg2ky4h1JHZKS0qxEdWKUJIaSjbu+47lVXzs/FI9wtO1FRyB5jv8IfrHzjom5UQ+HAeB+lVdghB30x0EADxd24BcjdcMjLluwWyI0pqII57skU7wnyhaa3RNVUx5VQu5Um/jZWCbvOzqmAu7BCVdRrq64lucRRtuzjY55AlJkNtimKZEmN+IdSR2SktIsRKoTo7R/81QAv2yFrd8UBmz+AwB0b8OYcMTjgOTYIonZY7iZI52z9hUA65/0E8x1dlefGxkIAIBNL/eoHJWnXO6iJYD1OT2Fjp60ysKF43214Krga2HyQxTGPwaQd1PYai8AikxDjaySu4rdQq6hsUfEg+3XUbG9WsqNeUkiFkoEJ6k5basYk5SMEUbSMUNAsgaGJE1uxDuSOkQlI3KSSHViZ5IYSjYW9Lvo2+Pz6HwEGxGOBByQDFvEMHv2fn6S7wOQfD8QwD6IO8/KrVCdG/jV8X0AONzF+eX0XXJf4jaFjQFwACKQrTkOmHef89WSq4Kv3Pjp0a8bfvrh2PPLwXKcTEONrBLed9nvjv1i+0wu8B+VlBvxkkQGkwROUnHaGpHzjoxJSsYII+mYISCZIS2pTW7EOxI7RCUjcpJIdWKUWIaS5ZtC2Zp0gxeA6t8bE44kHBCDLRKZPb2e2yD0ERF99wURUWtskhli5qY4hN+/n/jHOP9tRETxU4gOYYi4Du3866blk0JXG7hZxJuKa6hmcg3N+GreVdZXfnxyDNEqtxQZx8kM1Mj8Gpr1XVpD02KMIqJYtFmxfJpbrKm0KHhJYiQsOMnIaSvW0Gq8I4GkxK+hlQgj5pghIBnDkMximox4R7KOU5iqQk4SqU6SkoyhZNNNIVHqL+MaAl9KN1qJnUPiidI83yEivfcyol5V9USFzt8SGZr2IqK9La8wfURE1P4OEVE7/CK3JM1NcWiyZMmSFUfyiIjosxNEeh+3J0JR9E9IiJnY9hh3U4ZtUrX4mr8pTOwcEm/JVdZXvjmAiKJxlog2VzpDtAPXiCiopXhTeHFipvU3hazvTEF/gVeJKBZdD+xZXi3WVFoEahpnXIqE8cvYaSsK+jwRUZtwGfW2RWgOW9AXMIOI6Dze58aZ4626+0R5HfA5sU1z2RAnN/zd+z+yRyJMR26L0Bwjs5T/rp4t6NwWoTl0C46JRDSqXo5NON1LDV0Ar6FD6dDgeWO8IBKO6vY+4/FXustRAFWugcEW3bncG0BojJKq9Ld7bQDwOy2t0VIrQTF3PQassr3GFcD/znYBTOHuBywb0eNsUwC+Z28Kao/QwMhfyGlMdXufseQq6yvXbAbAFVkABoXrEi5Ec4AiCWr0yiV361cbrO+yRTVHma3aDT0V/COjlAvGpUhYv4yctkI4TlFEclV1khIAKBFGzHHYl28sL1jbL7oK2CaswTTpAiJr9znB7ENhOmZW3eJqZJahOkkzCQyl1cdDbFlDr3LCRgDQdfsx8xzkhCMWByRiixhmj4yqFDkSANAb0iyLoDo3J5cb1fHx8RmjeM4xKPdHAAhFgtATg04Kf6FCY7LkKuurq6KDBRRZhhqZFt53Rm5DvLfF8x7MgHpavAGWwSQDJ7mqcqzMiQrviCUpcT9WCoQRe8wQkMyQltQnN+IdCR2cktIsS3USlZQMJVhHTrrngM0cOLAXchSEI1VCkQlmjyEqGgAwZObOPP5v2qQ5QX1uAMAvU1sAoIV/pHnJMnwBAIbP3LHYEYZCF2Cr8wSFv2o0pg42uKoUFlBkFmpkQXjfJXm0K6iz9MyOHTFKi8RLkpJuBpxkjRjzjmQkJQCAEmEkO2YISGZIS4rJGyh5RzIAEq/URGFWRnUS3RQZSrY85bh2C7jM4S0z2Z/A3IO6EHVCkQlmz7GW3CdQhaXJ3wgX6VX1ubn71EstAEA3LD9KURSE7EKvRXG/IW72d5twftecmgp/oUJjssVVaxBLRlAj6wqakM08Vfu0MMJEKcrSIucliZGYAydZIwzvKCUTYElK0lqAQRilZMqOGQKSOmkJqpgmGe8oJRNgOwQlJTmJoToxbooMJWsKmicZJQ6tDeiHpwHAD6NrKwhHLCVJxBZJzB6GqiSuOICXvv1oLQHA77pG8rmRzoA01/PAoBex0gBpzNcxJR5rH2HctLeu7PjkvfvRYa/MVvqrRmOy4CrrK9/kGER6OaDINNTInCh85/3KnLxyWxvGT1ZkaZHzksRIjMFJjNNWiMQp4ghGEkmJgSJJCCNOSTpmCEhy0hLMYppY3lGST3uA6ZA8UJCTGKoToyQylCw/tjsd5gu3ASMGd3LEu0Qt9o5dsGv/lPA0I8KRgAOSY4s4Zo+8L7dxgTj7waDn1xxd/+Z6InZuOj+osXvF7u8REdGRthXLt0okoqmBFdw7vU3nBzXx8AqdS0Q/uM1MmEZEtK516O5bCyquMtATvdxfdRqTGVfZzp1C03dg6vutPVpNYgBFO81AjUw/5VD4HtPfD+4DRgzv1ebde0REp8MD3L26DDkpO4dJixEvSWQwSeCkY8ZOW37KIXGKOIKRRFJioEgSwojHK4nHDAGJaVrGNDG8I25KqUNSUpKTJKoTqyQylGwkJ50Notgz+qDWOrM4IPPUof8dXcIcXTmT6hNa0eLcqvJwD4V5AYA++opD45EH/LBginXPG6x0FTYilmx4YVL03ZLI02IEmeIjsdIvkw5KnCIFOYkVCWHEKYnHDAFJDadkOhsM74ibUg2ApCQnqYrEUFLYLcKfpLBRBpwpmXn7hZ6PaGSgMiGgMi6mHZxVaMXpVinZlo0Ssms/PrQpeXyzTclMHDauJSZp+3uhkZNKF2OwaXAJFd3rQ8t1/FSrSGjkpNIlJ/Vc6VtSU5edv7au0zbJlpFNsiVf0I+qQNv1rRX0s1PQ0DAGWkGXnl2NnKSRk54pcpL26awJNFijJppoBa2JJiUv/w/+BTFRszLMXQAAAABJRU5ErkJggg==',
    '1409.5313v2.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAk4AAACXCAAAAADGnIvnAAAfJUlEQVR42u1deVxUZdu+hsVhUEFWl0RcccmlFxQsfZOwRM00zSVbPtNMc6nsUzOz1LKkfFsszK1M89PUN9fKBdK0RSQVlEXNBQkQkB0E2Wa5vz/mnDNnZs4MM8yZGaRz/37JcJ4zz/Xc93k45znnXNeVjCCFFGKFi1QCKaTpJEXTDCIaJlVBCttjGBHJCJA5df3kHHQHojoISubsVbCMpIududDslWrQiCmlN68Lf9D+bBE8oE0D363bUlX/tuVQt/dW+cxu6K/qL5Xucx+juV542M0VatXwTpajmhikUM7u93Xrwt9n3yu55664ydTKGXXfublpVAMewMkcN3WfMKZdtS/lbo9JZVfGW3PaMJ+gYXvdtl7DGnl20iSczpX1fbjnn0VjGj89rMDXDoCIQLy4FfMEAmNiYhZ18VymIbNRswAtyfLIfgYRApv10MkTIaMfAcLH9HVBodG+t2KeAFbGXLcIr/hDM4M0ytk9JuatiW6DT/I2D11FJ1a5YEgM1cREAu/+QfT9TNeFp5jmkgemnjz3WS/v2QKojU3QsP1byCssLK8+DB3u4TlrW9zGsAnyheYLZLbNCnwC7z9exOJ+IqISL2xooANNC2umE11reDpVYgtRGnCA6IQsVWDvWKDSQrjjPc0M0ijnlkREB9HiT27beXkhEY1FVzURnQV2ExHtmsS1z22vIqIsv9lCqI1N0KA9yWuwulHTaSOCrhARqZ/FwgYKZK7NCnwCEZleO/n2xqmGTm8y6y6sFlwQg6ezH6MezrftlnWz9YN8FPW7uV/WPhcA4GXcPA6gFtgAABvncO2/FxUC6LRAELWxCRq0hxYkNGp1e3U+YnsBgMvXbRoqkLk2a/HdTLYUXMJoAJXxl1QPjHVD/Y67Hi+l/66M7A+g/vil8qAn2wFA2m/qyP5gm90j+qnirvg+6QvQ0UvFHR4ahMIfqvoM+j4g2lPbbeYZN83TJkHvDNcd/IerL190U4X2upnoJpsktLPg0G7/lu0bffHkk62/+G+7bYjoDd4gG4xCoCv7OW9PEgCM6Pz3phHAhuEnfr3SG5dvR3I7t0174PVHBnjMKOCNJYWP2pgE9dux/c7dJcal5TI3GetULUdrP3k87w2gNv5yfZ/hPvxS6YbK9CbUtv3O3SXCXzf93MnoxN/p6NHD60L6btMQ/dYxMv6n0EdUVP0yer6/5OcoHCO60q/zxuO+flUk93z/De0mtlm2fdKmQ6063yFagdUpH+EFzd9PYfqoRZhEdB0RRPEYutLcZYfYcz0RJS51w090Go9/JHSxExzagZavxL2OMeMGH5oLv1WrzhLxBmnuYudZWZl3aihmK9lNbw/X/lwN1zwq6FDcGq8Svfqp7jsnXQG4RcXzx6JDbWyCeu20EBAoLZe5SZhQhPB/Pd9jdPxvMwN/5JeKGyrbm1DbQsDE161YO7WLjY2NGTUwpobK26OA6Co2EWlauH1MtBnTSRWGBNL0lBeSHMwmXbP8BlEUjhPNx36icCTSVcjPXPZZpJ1OvwR/pCZLq020SxZaPG6+RmjtJDg06iSrpnr3TqU5dAHM0oA3SDPTyXXKlBHu+ITbUu3/o/bDbTesophFNBfed6valfC+lBgKALKd/LFwqI1N0KDdF8al5WVuCqYH+vN+q+rSoopIE9I6m18qZqi63ozbtPjCX7d87eQ3f/78Nw93X/q4+tf8wML09Hqv44BMppoE+KAQl5M8IiBLygkAwGzSNYd2A3xQBKzNGrBv811kwgV+Eb2L/wMAqneGv/GGNRfjp9cld/f/XGD1szNNcGhQyABCfZuO/J25QZoJj9274864LdzP/v5dG+Zq0XYCvlJunoXZqNizJ9qX95WIpPxDS4NpgQa8sVgZJhPkrTcNS2sBWhBusR+ry/B7ZteWgKxf5UF+qZjQ9WbcpsU3+3VL106yybt/SS0BHQSwWHut9GVXGHIXoGVL3iZdM7dh/5stXuznBwDoIGMelyYFuK14rIc15X55S3KF0OPez2MFh4YlMxaNiXeP1T8+vhaChfW9+P0EZg2w9jV22s/5b/aCrj3Q/8EzGyiWt/cLn3u3Gzv2rf6ZOcH6Y4EYCcI4AzYNC9DG/lKa10H7cVP9klvaw+yGbIFq6PUmWCmzX7d0OqEI8A5D/ZtuQP3f/Ia+soqCtkCpi7nnnL9O7p2kgLb8ruzGEUd2Pj8qIdCK27N5nSM2zNlk9Nd78VwnhdDQkPGZ75WH3mjPPhNcesCqY9sGpwG1K4ATOdO4d1E9r67/HsDsM+cGRPB2PvXnCACtRm8IgH6ZrEE1laDZWS+YuV7Mis14dxMAQPndevRDEcmAQvQzemi69IDJ3rg0TH8dljMK0lbjmS79Z1WsBdSLvua3tP1fLCdUj0s213EZ2nqg/Czq+Btby55blfFEleWvORZl7Yqd/NVSw2NwZnSLtoJDQ+G2Qpei+BQN0BHFmgy5daeKTsi5jYF3Aayd2Zo7Tb+MtuMATG6DOXqHfV4iAWXHX/SEbixWoppI0HwIZ64XimPdN8+vApD/TPQghD9V8AOQeuYBvVtqZqiCvemlIfx1y+7skrv5wb1bt26BvuEb64iU60NCogcuraWKEA+P7je2d1S0GU+qL7t1eGzQdv4m/ueJPor7ttYv8e3/zFO/dJV37+LhHXKDiM518fB5mr5xUwTuMbdSfTmkm4eiU8goIlrTTj6SaJm7osevusGhS1cfoLuJoTGPyoaUE8W0HPSvNL1GU6jJPdorFL2eJcoYgqHTZhLRVbdMXXOJ53IiIlrkdYf/ra4LRvce/0SHOVXEGwuD2tgEDdqVIZ7yXkal5aGZhrnzfveWw6f8O3yfhohqVncfNLTrq6X61WCGyvYm0KYM8ZT3MvV1oaW4BYyC+hpv4401NT4NnaA15S1a2flNuNHQLkWOH+umLo3b+Z9FgLLcX2YtakGOX2cZMP82//XvbT93AKgva8vfNb89ynNkIS0MxmKEapdX/cYHxRimMA/duSNQo2pt1Ak3VIFDbJCG0NdNvP5tRgSV3VNPDQNwu8N7b9uCWhZ0bKhEUGnEANyaF0FiYuIzE/q4ZO6Z/KpN3WzpOUQim4hCULnXiWyVl2+rAu73sw01fMEzEn2usVOpWU0nUVDLvWXSdGrMAFxXAoh06jAimx6qh+weTDDSydMJkZL0QApI0gNI0gPpzs6CKIuD6xgFAEBzqA5DgviNyQl3l4DueIuOevsPNxe1aoKrAxK8/Ye7TNlp0I2L7jJln956yeQcLpkVcM8lhKYs25QHbpq7Tfsx7vV3fVrqNbZJjwE+8s8RHdXTe9W4Ox1ljkjQ0/u9cSp/eHnOe0HeRj8Zz5q3S+69hNAQfc7BYYC+bEk/LftnzeNTDHeN9yY6/2qtHVBnezkswdneREQ0PIwMkynFFdFg7JlQo7jizooZf50GgJwgYfJP2Odye6DKnJKrfjKyez8hgbVT1bkW4ZdPTW6PqsSy8ODKq8WRVUn+oTIA9ckZ/frJUHm1OLIqObiHS25qUF8A2h3FGlHgxA1DARyctQP8vunm9T4Aqv8qjvCuvFYc7lOSURIlNxqK7ZGXKIv0Acqul49I0TwgMyiHqMWv/qs4wptXcKDofGCo6BNBe9jy62SuPgWuqm5lFfDxhqPWTn8+ps7qc7btMSSMdAt5bXPh+lH79gR8PAvA5YklQ85OKUbh+lH7dvu/P33v0cCFKwFmR9GGNGdvEaDUaP9u2b5zx3zne/RLoHRjdA6KtozMQObq0ZVGQ7E9Nr02OHBYHJC1ZvK2c4NPG5ZDBAR1enp6enoVmGR0CMCJ3W0/niX2IWYOW8pTvU+XfNB5B65HvZjruLXToI+JpkwnKmu/n6jQs7AI7xHFudRSdc+jRLRqvIaKsIroJL4m2hqo21GktVOZ5v4PifZn0ONTdH0rB79HROu9icqQRpSFc0RnUUQGQ7F1QUNEr48n2nw/EaW6f0U7yozKYfvaSbF169atW3uHEZMMi0BleEdbaTHWTlxC3GH7pUUJ3XH5nWhBud3WTsYXuyoZ4HIHiM/3SARaXwjHv4FATaU8/moUgKh3bgW5YSjgj2GAf4VuxxFiXfXnfLLINVOr1Gb7bpW4FUAAe3X2AgB37cWaPxRbo4gCP6XspEs3ALgrI/GscTlsz7LFCwCwo5xNhkUAEKWttHgniyIK5A7bsA675v045tuhNR7ecNyDgnl7y7MSFgI5sjYADj4EMJyZGy7uABS4BmYTs123o0jxfGHctRD9vq+bpijzh2JrHMvF6Qc3BjN0Aj/hcogdLIJ4aeglxB02l+nfIOmL76sPjHfkUrx09ReeJ7sAA6hDMKCs5xru15T4AwW43+ALRjvaGl7PbhgxV7/vPigMBEjvtqVS/GrEjaoc+cV05AFnwxkUk+UQLVgEe0TcKN5hm7bySFDwwP3npzryMWbhjWcmewGIivoSwJo8JZSAEio8OmQHQP83px24TVCqNdyO4gyosAiYc8TFFVAqdYMYOHIrgFP1aiihBlq2KgV+QS0MhtLoUGoA4Mhhv5tVEUCySvODtlPjcticoFJFgDY7bTIsAi8X24NLiHfYgqNmTMX0jzrJHPkY81d4tUbPnUSl02ZsXLwv9UlF1M8nhivGXaTi6Qs2/s9btZT6pCLq51OPKZ5IOvCgx9QMZkdRHr/des67/1qi6HxKnOnf9qUUru+yaa9uXzYX445MUESnEsWO2fXhOjySbDiUxqFefdEPMxfOinDpQeq5j25efjzq2UO/jVCM2WFcDlsTvDq7o8fkPZTwok/r6SfSJyiiUzkEXqVtrSMvIeIOG9HOsUR3/fPt+BjTaDpdeqiQqDq+VR4R1f5toNitzVQJ9mS0Y+NvfEz2XZOtqci6w+pl62/W1t4s19gBVX2zmjT1FpVDlAT5CCI+fBc6bKpyIiq151Nxo7XT6b4BgOLRXrntAbnhUzt5ZxOv2oLtd/pk+/YIgpcXt9W9C2Cf9YZLF0DmblE5RAk+gr1KyBw2V28APnAkubdqdpeJ/llbAmNkEkFFnHI0ACVWwZsquTc7uTTwIV+J7yRWORqEEqfgEldcos9Jzr1SSGH2MWawzLmzurmjyppzHXURDOliJ13spIudFE35YqcflnP7kxPuLhF9QH/vUCumFdcFiMfDhwiCB9EUD46B0hc46PWYY9/CGj1PjXHL5v+aV2fyMWiGjlQj2tPchEllVPV2zwPFn5omTpsZUuNQ756I9F+v/XgkuNexEoEsDarSBKH4MBXxodh9kwoO3+fzY55+j+YKa4eXLIbc/rfMvOKJF386DbpBRJqHD5jj4b+V73DBgy2KBwdBwaTAQb/HUntOJ5cG6PB02KH3EnRJBUA2w1zfZocE+wgexFQ8OBBK6JDKHKqzq06Or0Dl+WO1xXFJhKy5Kdf/KkDZ2XikXKDKpLgylJw9WgeAMo5lAwCqjn+fBXYPm+8OwqZfATAuDACKjiYRANQn7kwlDoIZksgROHEDABxkqGXapHRZaqvCpV95/lhtcfxVDXKPptsXyiYk/iEFqk6eVqZ8ns8rrAOmU+nG6BxweoPr7XD2j5ssD19H+eekACwjn2Xq2xqrU/o9tPg37yDoePis4oGBYIYkdpgVPIireLAGqtFIOoGDkcLBLgIHU0vxMqQRcXqDNOSTjofPUf45KQDHyGf2sHlpkbn4X8CTtToevk7xwEAwQxJ37dSA4MEWxYNtUBYjwaTAQV/hYJvAwWrpAeMh/W89FjzDw+co/2dZKQDHyGf2sDk6r0Hx7sUbX+N4+DrFg1gQjRA8iKp4sAqqsUg8gYOhwkF0gYMFHgWGLHh9PzdOCsBj5PuJMBzNtwD85y85DEHFgx/sF9YJHmxSPDhFW6FTOLSCI1XAhldDLPuAvSHgKP+cFIDHyBfjlkGzcxoAhKZAUPEg0xuSuNGw4EE0xYMDoRyjcDBzdlJCzWPBd0QxSjlCPEf556QAHCNfJMr8yZ8B4JfJEFQ8qACwQxI1GhI8iKh4sBKqkUh8gYO+wkFMgUPDS/H0CYroVB4LfvGoZd8Ty8PnKP+cFCCLYeRze9i2KFb2iH3r0Ik3lmkEFA8cxOJRy74XdSneoOAhK90GxYNNUN9ajARTAgfSUzjYKHBoYAANMwoKXP35s/5WB+T5esmA2qKOleU+rWSoux3kItabcMrsWpFW8cB9+rvU5Qe5mh6SXd6/s0lxWRqm3+SgGoC5/NLBANT8MeFae4mNCYmgYivMV+c3AaDwDQOl6SRNpyajcJCmkzSdANhfUiJNJ4mNKdnU33OokZJNvRRSQLKply52Tcum/uqJkkUKJw6qmXPFHWMg33Rs6lvnLa9z4mw688b8FfPWPZVtzrM9v97h5viiueM7xkC+CdnUX0YZkdNs6ps/V9xeBvJN1KbeqWLSfwJX3DEFbjI29QBKK1zkLU2Z0TM+7jr/+prTuX3c/iVOEcKmb+kNjKsEdJ7tjDs+Yxyf/WHK9XKfthCdK27GHN9O7vgO8o93qk09AJzp+tIFU2b0rI87xycvebrjU3kTRBrPP4ErDsf6xzvXpv4KyuoW7NWQKTN6zsed5ZN/+zwRzRNrFdPsueKc1lVk//gmYVMvdHYqGD7uKZkhSzk+3yMxMaP1BXy6L/vApRs8PvlDewYu/nmtWNO785rkothjG8FSmsFwxQ/c0hLW29iNK75JjcyuAHS5nk2cBDtwxQEARYVcYsM67MKPY76FHfzjiwrhCBizF7tPgt5RAjBhRs/zcdc2dU8Ojx8RLQ6775/BFQeAY7mO8I93tE29wHRy2fhN9ZtGWwdQh8GDB4fJKkfOXh2qAM5yLT+1Xp+SmRMvznTaCQAIbanHFYcxVxz24IrHjdLPtQ8KYS8Cd1yQLrFpF44EBQ/cf3aQ6EnFBcERMOa44kQeO2N3wYQZPd/HXdv+9ydA52EiMR+aP1ecM5C3q398U7Gp/22E4unrl/1aTTNhRs/5uHOk45mT3t/9wUpxlpDNnivON5AX1z++SdjUW/MKmCE1a7LaKUjFOW/jrmdN3n0KiSveCCg2MXWVN1DmY6+MxISR6HOQGAWSmaEU92BI00kKaTpJIU0nKfBPcO4dJtnUQ7KptzmGMWenSHJmrGjuqCuacx15ESn9Xw+kBwV2lh7k/lT8Yjs0LEQQ4uLbboHuHM68A1H1DeT1iiiigbzTbOqNluKKureLYYEQwYCLn18PwJxeAE2YM+9AVE/v98ap/OHlOe8FeRv9ItpePRMoejAiolhEn7uDNEuECIJu9o2xQHcQM995qML0ueFhRkW0zUC+SdjUuzVwg2DyTzYsTF8O8IpYNxfN/kbPZBFlDoGROV56wOkLAKAgpVcn5NfJNF2QTWofrQCg5q/iCG9OfZClkwOwegGIRZnXSRyAqnMtwi+fmtzenhVhMmeEDjIdJitHEC+q/yqO8NbLSsTqmYJR2AnF3GNMVl8A4K3jXp89X5U2vfOPwJaQnYwAQNDNHqJaoDOUeQ6E77Ruv2AzZ333OUxWjmBz6AzkDf3jxaxeU7Gpr0Qa6fQFV3CaiKbMogL5caKEdZxjvaCbfWMs0Btk5nMgnNO63RY0xMucyZNzd2elFzZD8Qzk9fzjbTSQb6I29QCATyk76dINAEAnANGz1gY++9lw7H+fc6wXdLMXzwK9iM6w9vStORCd07qdoogCucyZPFlMzo/fZhCegbwb9PzjxTSQb1I29Tx9AQD4qXKw4MjV3AC5oWO9Ifm+lWiceZ7koJWR07qd4lguP3M/PibPj1/k4GVlRwN5Z9rUV478YjrygLPhWuZ9Tqtu6Dd8bfu5pu8NRLaOjxvFt6eHg5zW40bxM5fxMXl+/CKHQ/zjnWhTr4ROXwCcB5Rb33MFXv+2xh+cw7kJN3ubLdCNmfm6TlmndfGDQ+UrK1SADpPz47cZimcgr+cfL6qBfBOyqX/sFKcvoCtT163dNjZWTUTq3pe10oQxO0y42TfKAr0hZj6vU8ZpXfSlOA+Vy5wVOnCYrBzBRii+gbyefzzZaCDfJGzqje/stKG+WU2aeiIiqs/WetUoXzHX2e0ice6x+FGbqeL/eumhQqLq+FZ54t/ZCWdugFn7t1rkBMXMykEw1ovKAcCliwIyrVrFPUgG/PA1dj5l7izX1l/8M6e8s95b2dN9AwDFo71y7Xv512VugCkPtgfZ0DFZOap4Fu6XeeLkLWdbsk6tevvirdMvDg9rVpgOysoxMJbynTQJNFTmdJ6OeE7rlqPajOkg/3jJpl6iz0k29ZBs6iHZ1Es29VJAsqmXLnb3NFe84eeeajc0M5t6DjXUVHvuT8UvttOn0UOyqbeEoGI+qla+PnvIHvtxXxMmlVHV2z0PFH9qmoOaV2cvVJO7l3yONO0/NkJVxIdiW4Lazhxi+6E08qm4qfjfHKJE2e5mZlPPoZqMO0gzpNFLNvWWPxU3EVXbjgMRg2Obl029DtU8l1ycB2//LJt6xnSe9Z5XXrjeq6+cpVC79VEA8LpmtyI4xaZeh2rkP8/jzONe9Y93pk09YzrPes9feyQpPHWAkqVQe/w+BahJGm636e0cm3oO1dB/nseZx73qH+9Um3rGdJ75Ud//A6JfB1TpKNREtKpngf1kaM6xqWdRDf3nubQrkcb8IxotvTna1BtNp+stwhbFK9kfp8DSYjRZ+1+TExHR3p45dlQ1ElFRrMdaKsNJogsoooOoI6LTyKYruE5EdplOLKoWNg3XiX6U89MWfToVFnCJqTuvo51jZ1L1myT2dCosIJFhrFyKM6bzzI9MBBiSx3/ffLqj3c6VzrGp56Pq+88bcOYhJi39H2FTz5jOMz/6IQOAqlpnTn9x+yE/LLLbdHKKTb0AKhjOvKEnP8QzkP9H2NQzpvPMj9AxGwCsLeUo1JfnDt277eNCu43HOTb1HKqB/7yRJ78S4tHSm6NNvdE7u3W/DeieoVzB/EDFoqqo3PsnaV65NvnWw6vbj1uQBwBvfWCnd02qPq/mR7SKc1+VvjzuwaUuqxNGvDugZLF3r4SOy+W/v//78KefBfBGeugDE+2CKktbHvfgUvcP/nh0ZfaaC+Pf78ykPbnb8qNDl/kvPzp02TDboK6tOVgy07sy5Vy3a2ATA77bcwjVwWntxMqIjyIyjJV8J8Z0Xuc9X58vaE5vn+nkHJt6YVTtddD2tCWbekiMAkiMAsmmXgpIRtBSSNNJCimk6SQFJJt6SDb1kk29ZFNPkk299KDgH/Og4OfHnC49gDmjelujLA6uY7S++JpDdRgSxG9MTri7xC6ozdCm3qKELu4c2GC9rSt4o5bi+fWAkVG9KCEP3DR3m/Zj3Ovv+ui/k22THmMf1GZoU29RQt/MaLjeVhbcSukBj/tvYFQvEvNo2ZJ+Wv+fNY9PMdw13tteqM3Ppt6ChGrCNQ3X2/KCN0J6wOf+h31uF7fOGX+dBoCcIOF7FzuhOsmmXi8ZmYMTOjRO1nC9rSu40dopv05GnZFN6q4c615ffQDGk15rfs7IAvhe8jZG4MQNQwEcnLUDADhzeLp5vQ8Yy/XKa8XhPiUZJVFyI6kAJJt6WCo92Pa1QL3ZNJl6W1lwo7NT6gudDwPf9NzBse4N1AesJ33pxugcThbA85K3OebsLQKUGu3fBGsOnzvmO9+jXzKW60VbRmYgc/XoSiOpACSbelgqPchyu8+43myabL2tLbjRVTfL9Q+iPz/nWPfG6gPOkz5NJwvQecnbunYq09z/IdH+DHp8is4cXjn4PSJa781YrmfhHNFZFBlJBSSbesulB+/uNa43m2YeV2/LCy5sU99p/BdDsO9dzqk+IXU78PBFPet69joZz1rJBxkb1jf+qj/nk0WumVpCM2sO3ypxK4AA9ursBQDu2kEMBfwxDPCvgGRTL5AQd4yGddg178cx3w6t8fAGAM1PbxrXm01zE1dv6wousBR/bV9OUWsPjnUvoD7ggi8LEM/8/PnCuGshzLMYxhz+OkzapulJBSDZ1FsqPTg5pIVxvdk0A0zX22zBBR5jDhmw3ne6zqm+HzI6AKp6tZ51vdaTXsBKXoTwenbDCK0hPmcO3weFgVrHfN0tRyUkm/qGE+Ido2krjwQFD9x/fqr2odMSgXqzadbq19viggucnWQLNhfdp2PdG6kPdJ70ar4sQCTz88IiYM4RF1etxTprDj9w5FYAp+rVWsv1lq1KgV9QaygVgGRTb6n0oCy7v0C92TQ7cPW2suACjzFrgxJJ59dO5TOf3rziv7wNWk/69AmK6FTGSt4Wi3U99FvPefdfSxSdT4kz/du+lMKZw5dNe3X7srkYd2SCIjqVKHbMrg/X4ZHk1CcVUT+fekzxRNKBBz2mZkg29cZu/6zdPxHtHEt0118rev1yvWC92TTZemelW1xwk4Y8dw382uu0/uy6DTxPegMredHt1Tlz+JpsTUXWHQ2ztf5mbe3Nco1kU2+N3b+qnIhKtduGlDVQcP16W1BwSGaGDcZX5zcBoPANA+0FJQJCI+qY8p8dkpLF8ahVs7tM9M/aEhgjsxeUCAiNqONr46Kk6eQM1GZjU6+3Yn4wwUWaTpDocyLB3PGSbOoh2dSLBSO3n039Ckghhc2xgrmzk0IKSDo7KaTpJEVzjv8Hh3/9noUPubIAAAAASUVORK5CYII=',
    '1410.1237v2.1.png': '...',
    '1510.01234v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAXAAAAFMCAAAAAD3yrg1AAAo+ElEQVR42u2dd2AUVdfGn00nlRaKQEgMhGIIJYSOoQSBBKOAiFRROgJirKCI2F5UFHwNQYofnRcRRSlK771IKAnNJERqCqTX3ez5/pjdnTu7s0k2W7LoPf9w9t5z505OJndm5/54joLAzZbmwFPAE/7PNiIK51mwjYUTkYIAxb9+HbdRBhTElxSbm5OZ40v/vKJuFu54rIVvhaF/XhviZloHoD6Un9+wt21Tkh+vcZyb1ldUHG7qKRIRqMp2plXHuZ9Et5nrdJCIiNLSZGI0jZcdMKqSHaKVdHbV65SdolKd5ZgkAzf79nWAa9++vZ508Jp1r8KxhqdY7jTmJTzTd6CKiOJrQ0j4zHkyQZrG34BOlexgbaTeTyM7RaU6K5twIvJAUyKi1BA0uV/x6JGmJNy8NXxVxmBHAG1Xaj6flwvSNEa+1uvbSnaw5iV/NHk7b9nFxe8d3H634jAv2z2HJ+A4AGBQXQDA6eMyMdpGp9iDXSrXUZ7JTlGpziplHDhiVzfNAKwpfLODM5yXtQIK907AnVN4wg8oOZPk4BdSG2xjfG5+3U4AHp5Iq9X14kDDDpSlXHnYuKs3O1zPmClQci3Bq3Ug5I5oMVMBDXUf7j9wQWm9Rrib6VTSTql3imUX8ws8nkZ6Sn5B+yaQnJ4lb5rnFQBcu86+QUS0uLsrmnTvHkd0LaB+XFy4+xw12xjihFFE9H8hsduWt4LSoIMOBzZ857OgOqfY4UST2QVSHKTeVK/Dd5/UHp0td0RLreH0KRCra9zYCXhyNVGcq2vIJf1TzA1yQF+ilfWBn0hyeha9adIiR+HvZDkREQVCuGl9BRRQaQCWsY00HKOI0hzXElEylPoddAQe14iewDTp8MnSO5J20Dq0KSDaiZdlj2h+whulpPx1cL6zo5BSwdRhiCIiema13ClGoS8R3RMSLp6eZW+amHVmbGMAqsln2NYhz37oDudu2C6JrQkAj8q+3JyJgM1O+h2YjudbAB8OGGtkuMSKX8dod2Bgow25ckc03+5OmDD5s+R5SZ8xT+KKd7HzMhB/faTcKQo/RS3907P0F58Oayjp6JbfaeFmpvHJbaqrl+5fR5ZhfFD7C8MRMPA9g46Hl/AUgMmTyx2utcRHUJ4DUP9uUltjRzTP9um8LxMBoPfLwPPNb365Dl/GOFd0iszptbfoU8qieEDR7JWd03CRbc6YWntoSrsA2aei/TF1kRLX9b5+xz3ApeLhuhsYkLhv3759w77wN3pES9meNWvWrFlzEoDj2/hfasqB8RWeInN6lr3Cj6MdAOCVOB9tU9G0VQU9boyPc8EmMa5o2irtc0bS1wuTD6w8s/ZdvY5mDuoMwTMcrmdF01a1AfqMr+CIlrKYlwCgJQCM+fDB1zTVo5xTVAEAmNOz8Pvw74W/qSI8DQBuUKJwN47dwNsuwF0hRNOoscsRhYrAiYc80vU7avTHAQKQ9pp0uJ5pBjXpgsMAQFNTjR7RUhY5YcKECRN6AIDbG1i5aTogd4puIADXAYA9PQsn/EanvUrg4Zy6swGgExJwoiMaAyeB+FSkZYqN2qfQnBlK4HbRQP0OLKt7dhGhaLK/dLieaQYp1tdZ/xug/DixsfEjWsEmexeNqgPInWIYMoGir2sgVc2enmWfw18Ys7F9zfbtvPr/LbxaGajo0OYa0e4wp/7Pjnsw0BPf6BqL3D283BvS6XpTQ5/v124r6XcQpU9s2PLpDivV7PAidw8vD/cM5u2NZgrKfM2vfZ8O7+TLHNHMx8L4Gt7uXu7e7sfkIj/ySCUiuVNUzvDq9kLfI/WBHuzp6U9j3gbE3YYOyL6rbqG93aGgwFcBAIVZdV2h3wgA6lI3ZYZrHcMOAECOszsMhuuZOKhA5VPhES29AUHFNWDsFFWPnGvhnnONGo7M6elPw3d8+I4P+K49N55wnnBuPOF2aY4fAej1r09DL9tNw8krcPKKP4dzg72QV0as6KHoN3TkabZ6wpPjTsTXGuWBgtTDuddaPGY5Ua+441Y4y1euQc4DkD5zmQ8A0A9XPB3f8ZB4sCjqZtxO4zkiIroXsovs3iQZUI/4mOh80D2ZBjmPMt6cFIb7RESlkaPK6MeQDNaz8K69cbuMoYLz67LHLOG/+pQQ0aQXZBrkPCq+pVokJPxj51wi6j6F9Sy8a18J6536mK0oG7u7AAjfXmjYIOfBtan2JrU8xAtAl/WFjGfjb5rb/4J3F+WDh8jOB4DSswllQkfxbUJxLlCWeRdA6YOHbLcqIw2qlCLNmnfzYJot81221wcAGpYcMmiQ85iR2Xc8AcA3P170bJ3wc8WAQ0jDFUu+8zsMWj64MLnHcQB/9Z+xbdallQdwPNx3PLA8qOFSiN3H2teb98eXZ4a/QgAOhR1Qb5xftqybX8tvgNGNn1pt5YRnZHkCgA9uGDTIeczIGo4kpPSa6NnqKQUAUlYqk5YOA6IiW56NCYx7gKWfJ3rDdVBKzcSwBTOgnvPrAnQ/1gnApHEegNjd43L3m+lzENp8TB8cjjjTAXmjuk8e8VS3GGBtu331rJzwPHgAgCNyDBrkPGaka/BDALiFbNGz5RXu2axZoCcAKLySuje4PzxnzlBvIMJhI171mwY4vHAd0ABLLq6A2A14XR4DBDhfAU2L6AB4Nm8M78lb8oCkqdbON4oFPkaBIoMGOY8d+nHiA6D0AlSMZ8Mr3LcX+mkugDYA8GeOyzEAdW88OD3dEUCQ9Deu6waA1g6Ao0sB7iQOABARD2D8/A1TsCHG6mu4F4T7iA77FhvkPHZo9MKJccp1USfrMp4NEw4AnYUTqg0AqajnBmBDo2vwk/nj0nUDgPB/fsrwFxrovrAOjptcrPS2esJrCldtMXwMGuQ8ydiYoYdL3vgAbVnPtgnX/GcjBQC0Qp2OGjpJ/7GDVEw3a81xW+dP7XM8abj1n1J8GqQBwAN0MmiQ86SDmzYFUv3bSrxq24Bo+8ReACjZ1rj1MeFsAcCVAKSVMN3smEbBFwCgZCuAXi2XXA6xwSu9qBQASK4dCuBRMdsg57FDjw7OAXKOzHViPJslPAelOr80FwDcVuy4BODbJooV53YDWCb8Gh4B+MW7SOwGUKoEQMoyKFac/B3A940BKKZs7mCLCyMmMRmgX2IcgcwmPSQNch4AlKAEAHZsuw0sDhjDejZ6l3J2SJBnzZ7DTxMR7Yr08h2wmojoQNc5a97eSUTHQ7/6ec4ObCWijB7zf1+ws5Fnvyxd9/Fna9aKPrW6t6f/sCw62eH9zR9vISKiLP9iW3y1p03hCY/eHF1KRAXBo6UNcl7JyH4NvFo/t4joSuSRP9/qk0asZ1nyymS7mx8kkFF3H7ZKa7L1eQB4eKu59yUvXw8F083ag+wg4e/wUdwHttmASP+9oGt7hWyDnCfaw13Z7bsqpJ79kFd3NAmv3AqV1AFfvujPd3zMMKXM1wKj9kXH3Lxif4BjElW2T4c5zR5X6ej+/U7On4V/yPZp9SwpSicFqZwrHZ5+N9j5n7CJXI1rON+152ajNZyDQOG2m4YvKXxJAedSjFlBivCvm78TAORczvTuIxeXcTXziS6cEDI/4Q9/PH6w8SvuyNz/xH9CgPsb1g6UTXjK2lUzujw+hJCpIJBcY9by3CLl9BYWf3l1E+OIiIoiapwlInpmqJG4kNftmhAyCwSSa8yYnkH0pctei4NAtzGeiIiOoS8RUZSxhIe+bteEkFkgkFzjIrfNRBnoZjUQqInm/zzjH0AImQoCyTU2IAAeeGS1dykHMZBdpm6eLEL5GI9phJC2X0IIWQkRMhUEkm18KW8YcByDrJXw8+/1/1r8tDv6QvHb7ysBHcYDAPiwgd8zGagCIaTrlxBC1kKETAWB5OkgZ6D0644fWAFXvrQQqvhDH00WX7Zve+t0LfR6bdwG6DCeCAAY+nNs7yoRQrp+lhCyGiJkKghkhA46/cvBDls8rHCFh7z11nubzmyISNE2FL02vBagmLJxP4PxAEhYfrp3lQghFhHSEULWQ4RMBYGM0EGdF6xPHZFpLSDf77emA85pmJj4O8EAEKzY0ZfBeLBvTIInqkQISRAhLSFkPUTIVBDIGB2kCFrfaOApRyvdNGs/c2OHxr0uUDwOLldZjGf/FfdZ5RFCbm5uG96VJ4TEfoYQ0kOEqMhiiJCpIJBxOqhO2LndVgOBPHVwTyDyAUBZ0ozFePrP6to98qUqEUI2RoRMBYHkGkt7qE65ALVx01qPhcWHFNov9R18TwLAWQxgMR5PdJ43JbVKhJCNESFTQSC5xuJzl9IB3JZB3cz8pnkFY4iIisbhEyKiZ6KJ6BffJCLVsy8S0UnHnUS0+Ay1mUNU1Dj0EREdw7O64W1eISKinZ4XieiLP+m44y4iegtbiWhuRyJa4j2H7afefYhI7fIRe2wiWuywwXLfNBPckojUXT8logz3UEmDnCfbGHWYiFJdwlWW/Wqf+HwAvIeMGhXVrv8+IjoTXbtO9HmiPb0/XxY9r5SINBjPrijvJoPTt/t4toyrEiGk7ZcQQpZEhMwCgeQaM8YuPH6sc9R9W4FA97KDnPQxHphDCFkZEVKYBwLJNcZfpPYhisdjE9kkQsgiiNC/fMfHJELocUOE7DDhphFCjxsiZIdLiomEkCUQIQ4C8V17bhwEAgeB+JLClxT8swRqCnady/J/8UnaXNl3dUZZofLs7J605iPr/DLErhOpdrD+YyGt/HDgtGY3Nrb0XHyukkOufbt24BbZnuRxU0fIj5mT9nGDhOVNfjpnqyXFdEUg4FDcZrD4j6wikILMeluonupzmoiIVjmFVn6UASuk/laAP/Cq/IDfQsuIqCwy1F5BICKigsAoYvEfqygCLcJazQlFm5AMA1YobbJwkHP5RkTh3xaArVB7BYGIiD4NjCIW/7GGIlDG3IBRmr+Umeb8OW/VHCTUQ74/Wdi2CPO3VxAIwKmAugAY/McaikAb86O0g3t4SsAdVUYaVMkPAZTcUgH6UA8A4O8DSWoAwOU5QkNZpn5dGA38E7TlDwBQzJAex2ASZJ88XJpaLYpAxT8Ltx8d/mMVRaD9aKV1XVew4I6A65x95VX1yqXHO8dBCvUIuXxzr9fWvjcB/Di7eP/o0etwNcx3BpZ28Wv5DTCicfBqEQ56A5F9vzhZinDJcQwmwddrfGp8ForqUAT670zNi28t/lOOIlDV1/D22Cz5HNafiEqcPiGibn1WE93E6/FEq9wfSfuihhLRSr+rRDHtyoiIWghrOIUPJcquO4GI1MFpREsa5RDtrplFtKkmgPrf6c2hP8m1cCJSdzd/Db+BWURE8fjIoEHOI6I/fyDqHEVERKfeCZucT0RtnyIimoavLbiGO6EMBqXWXFwBHa6T2hZoWZgi7RNi8wqAPvFJkOz7Az6v/lQIpL5cj4V/hqf8b3LztBlf6x1Hb5K0+HhAMQ62VwRSrhNn1eI/xhWBqp5wf6Tr/Cy9Iwm4zlMA3FAgM8vQh6Epvx2VCEYJNilnM7B2rAD/HDt2ou4NADVf+v76Qd/52XrHkU7Syb998Kz9E2B7RaDY15jTUgSt3zOwDNELJ95O/k8U6low4QNwQecvgH49KbDQjuHXsuVd19fpLtMR2P97KHPqsfDPjwCg6LUs70K5k7gd/tB9ScRYNWwNAiWW1MnOzlYps/PA4D8xsYcPvpFmUUWg4XN2lmgWiWwnCbhj7Iup2PfO8guBOAaoFcLd5gdd4bIpgy8mPwsW/tkivDbor/sTl5/jZo3587PWvD0+3NYgUMadDwAk1vvgyeks/mN5RSCP2IzFGveHsRJwx9D0+vIWvxwIpAN7jgNuSiBZFzmo0bIj4WDhn8QkDcPattw5EjYBtWYNvWJzECg8NjY2NtY7ODaGwX+sogj0wrcfriMA+F3RgpX20Qn6KAEoUSbtUyoB5xoqAJedi+7XBtr9BRAAZQEAOE1c669g5YNQNjIbAJa+0khyHINJ4goBlHZBdSgCoawgF/COPNAY+DshvKe1FIEOhXZec2zjpI1ELLizU4vr+A3Oere9V7vpTN8egRXa0mTB75/8Oa3reyqipKYfzT9EZ6Jr145KJ6I7npnCwbVwUJu9Ez7ddWDWkGwJHHRcf5KtA97fuPO9b6sDBCKa0sXTp+8XDP5jNUWgq+eyGkdouVUW3NE3vb6ya8rWLpTrAwDKy/5sge/bTSCBg86H0uVzZaEa5sboHLk1nG6UNnerXkUgEf+xR0UgvuPDDbxwEk84N55wnnBuHAQCB4H4YyE3cBAIgH2CQGpytD0f/i8EgXRVwA7PjvT1AICRDhLoR1skLO/bv++3eb0+B4HMA4HEKmDa/fBeLPQjdme8lEQ5L/ic5CCQeSCQWAVs5rI/jhw9eqRXCgv9iN1TU4gor3aAioNAMAcEEquAOUwa0LNHj+Tx/iz0I3bv6JYFeIanXOMgEMwAgZio6QCQfGIU5KGfJiWlADyZfXYOAlUBBGKiAgGoZy5QQB76OXq/PoBLTsEcBDIHBBLuXLr/z73+PSJ96IfppjOIIQ4CwQwQSG/Jmz0VgHHop2R8xGfgIBDMAIGkgdsVfgBgFPqZ4/urGzgIBDNAIGng6gCNIw/9/HDvdw+LfrX/F4JAkjjVoQgYFAQTbcfFDQ644tCag0BmgECSuMT82voFwRg7eeJbB+A3Vw4CwRwQSKwCBtyDRqVOAv1ouq+Nfjh1yuQxK/05CGQWCCRWASPahjkaGSod9CN2a4rGhXAQyFwQiHkqXB9VX78gGFeT4Ds+4JvI3HjCecK58YRzEAgcBOIgEH8s5OZk09nyhD1Hdz+nf1wiK6sIZNufPGPT0SMNJ7gUXM55fYj9/m0ZgECi0M+Y0DD3v08+08eIIpB6ze0i9cRmkkDYpK69UbuMEUREe91eLSN6TEqDMXW+6gNwnqs2ogiknnGJ6F7vk2yghUuDVaF42ygiIhqJH+014QYgEFPnq9+S2StSZIIERaDt3xARXYxgA/Wnqa7FNACHXoRdlwZ7tdBd08DU+ao3TTZIowh0uhAAApPZQHv54nMXbSFD9mjKflmI6YFFQCC5Ol/yikCNvlmsBrZH2uE3zZubo8dDhuwRyn5ZiumxDAgkqfN1cdFXi3L0g7SKQOMC3uh1fdv6z5jAan5KAYDULZSZeC1urAKS+l9M2a/r2w8BYfurKeEGpcHYOl8XE2cptnba01QSdKGuZs/E7dCIoyGd9zszgXZwhdfpGNZNnd1coU8PiWW/LMb0wCIgEJg6XxtGKDDYMwZGFIFKA99yODroARNoBwn39PdvuySob6Lh9NqyXxZjemAREIgR+kEIAHT8JUNeEejy2G++utJ3z3NqMdBe1vChxcsMGy3O9MAiIBAj9HPqiPArSZBXBJq4oC4C9350ZpcYaAdruOanuiRLCFmW6YFFQCCmzld0Tp4LUIo6sopAExJ6AlDM238jUhdoRwknRaGLk2sxDMmehL/eQq1Zp65UT8IVURcAHQjk7obicw7pjQWhn7YjXABc9WnFBIWHA8CW4FgoFXneAOAXLAZW+5KSI9z8/RwfJWNdphzZY0mmB5YAgRihnxdaAbh1eJGTrCKQ8/DFAHD30dNMYPW+S7k4tJVXzYh5RLTUfU7Km7Jkj/+wVeYwPVYAgUShH9XcxfGbnvpabUQRqPjlSYcuLh+XKgm0SWmwSljaHoquCVmyxyymxyogkFjnK/FU7a71jcJBuHm2MCRML5CDQHzHB3zXnhtPOE84N55wDgKBg0AcBOKPheAgUJUt956zM6mUDX3AeSGbJPzBusPHPYcEvOTzOPNC5SkCqX9OVDrM8JV6OvxH9GQVgQBY/uXVWQw2Y7RGHsjWvFBlFYEor9/cUloXLfFE/Ef0rKIIZAytGmrGaI08kK15ocoqAqlfGEVEzXxZj8F/RM+4IpCdrZJbUf28kAEIJDYc2HIewP+UgOgx+I/o7fg1oRY8w7dee8q2X3zo5klh75ulfFCWkouSX48DOtUfLfmjlQeClBcykBtKKYLtQCCm4XufdgA6dgXjMfiP6BlXBLLqkrJr0I8HXptTSnQ0GJN//2zTs+PURPTfAcvGTPz4fB0VqZdFHtrW5RjRwsUJpz+sQ5ui3JuNGrVWXFJuuEcriehYd/QnWtYUn0iPZJUl5T4mEhGdxyKDBnWddrfmfP7udSLRIyoKQM9rv0XlsF5ZCRFRW6d0m67hvzV/RKSeOpLR79lP9JNrNqmC3qQkUfVHVPNpoVvDe/y0OW56xGq1Mbmh/dZKeDmKQNno/lUZJTfaR6JHRKnd4NKzVOpZQRGoQit6bXgtQDFl434wlA9+DPKBY/sf8aSo+iNH/kh4IRm5oSuwFQgkNuTi1IsOCOg3qVj0JPiP6MHyikAVWcKZO8EAEKzYAYiUD2o+AuDkA0b1R478kfBCMnJDBbAVCCQ2eMDPD0Db5BOix+I/DAgEyysCVWSrkgSyx8HlKsDIA03LvIK8g++AUf0xSv6UywvBRiCQ2ODtUhsAaiBB9MDgP6IHo4pA1kv4vebIBwBlSTNJe73p374/a8lYCKo/HTt27NjwZt78M+mL/ndUCPjBCC9UrtwQrAcCiQ1ObQqE33Vd0UOuBv/peUP0AOy4uMEVVxJtl/Abf3fwPQkAZzFA0nHw6RWf/fA8wKj+iGo+Unkg1MQlQqGqfLkhi4NAxhWBopNKAaShI+PVUAhV7/yCRc86ikAwQvqUAkDqS408lm1OBsoWvDgILOXT9O3lG7fuzZGo/ujIH608kB4vJCM3VGYzEIhpmOKxG6DdE5oznoj/MCCQlRSBZN6kRPvB/flRL3Z3xBtEe3p/vix6XikRQ/lk5bTze7KJm9OL+aLqj6jmI8gDGfBCMnJD/sOybAUCMQ1nOm45N2VkrsQT8R/Rs5oiUIULeXaQ3suDok4rugDq8y8Pnwet6g9D/ujJA+l4oXLlhqwOAokN2Yczw0Ig9Rj8R/TsRhHol+W7AABxf2znOz62sPZXUgGgbPeAf+9unm33NM9+1bqT+839T09V/Cuv8OrYRM5IKWoc4PAv3USujoTzXXtu4CAQOAjElxRLTONkd/o0GVczn+gCDgLZTJ8mZe2qGfaecANyxQSzrV5KpXiTkNerVS+lEvSMPrliwjQ2fkpxEx6LIob83xbjQc72vixs+/1doEOvmY8RHx6AQ4/xOmxQy8r+E67hTcSCVVbHTWBNCRu7T7igT8MUrGLlaQAAHzbweybDThMuW7rKPnULIdGnwbotu1uG3X/xvAMrTyOEDf05tre9XuFyEjb2e4WzvAlbsEqKmyQsP223+TZauso+r3BPf2DJqL7nWwMYOkSRcumk5kJhcZN9YxI87XcNN1a6yo7XcA1vIilYxeAm+6+4z7Ljm6ax0lX2eYWzvIl+wSqt9Z/VtXvkS3abcCOlq+z5Chd4kyy2YJW08GDneVNS7TbhRkpX2WXCJbxJDluwisFNSpXA215Ds+w243rqNPb7LkWfNxELVjHgyqYo7yaD07f7eLaMs9d3KVJ1GtOmqa734QJvwhaseqzeh8uq0/A9Tb6nyY0nnCecJ5wbTzjnUsC5FM6l8MdCbrD928Kih6Lf0FH6qYC5zhzc7TtvanJ8PBKeHHcivtYoDxSkHs691kL66cms8Pouh1KCehQ/OOz/lx1hPmxD+sxlPgCOzo709QCAkQ6gH654Or7jwXbbUKCmQjuN54iI6F7ILumnspo3iGghlhDRFV87qlQlNmS8OSkM94mIlmrS14uoNHJUGf0YksF221CgpvLqB78uk37KnEREtAhLiYhGFdpPpSqxofiWapGQ0ZnL/jhy9OiRXilEHzvnElH3KWy3cYGaarxp9k6Vfkpvx3wMSYfdYD5ig2tT7crtMGlAzx49ksf7A8tDvAB0Wc927+iWBXiGp1yzmy8+2/+CdxfpJ2cW4ewJu8F8ZLmf6QCQfGIUkH3HEwB88+OZbuMCNdWV8HPFwLPST83aM/1dm8JeMB9Z7icQgHrmAgVQw5GERLJX89H79QFccgq2i03klJXKpKXD5D/B/jAfo9zP/9rUAuAa/BAAbiGbvY5dAJy9GONrFwn3bKaEp5FPsD/Mxxj3Uzr7GADg4+cfNEDpBehrXcgL1FRHwn17oV+OkU+wP8zHGPezXeEHAIheODFOuS7qZF29A9laoKZ86+xl/BPsC/Mxxv2sDtA4MbGHD76RpuGBdWZEoKa6dAt7l/MJ9oX5GOF+VIcitG7TpkCqvzThOy5ucMAVh9b8fbg5AjWsJeZrZC+ODs4Bco7MlVy7RgVqbH+FaxRsZD8BKEZJNWc8JjT5Sa1ATdNW5yQNQIn2/O5pb/U7tt32weKAMWC6r42OmAoqPPZetb9LOTskyLNmz+Gn5T4R7XipV32v+r1e2let/6nKuEBNych+DbxaP7eIiGgb5gjhVyKP/PlWnzRiu6tNoObx3IAoR6CGeSpcH6V5GfhwV3b7rgoOAvEdH278KYUnnCecG084B4HAQSAOAvHHQm7y71Jy/wYa1gGAq65OSmpmfGj6r3+R3/NNKnx3cjnTu4+ZZ2k6ImSJWSs2tYP5CX/w69WNnvGBAFbtu9Ar8m1jA8s+3fjNWIeTg3ovqKBa9P0NawdW5kdPHjd1hLE+0xGhys5qIggk7ToUtxkAspbnFimnt4AsMYSKX15dHoaeKiKikt7GS4mohoRkEREV9nu6QoLkmUqVUvoVrxrrqhIipD+r+lvzQSBpV0FgFBFRxvQMoi9d9soTQ5UAgS4vfR0LiYjoNeOn+DHOC84d58kV/ehRlUq4+ly+sa4qIUL6s6ZNNh8EknZ9KiR8kdtmogx0kyeGKgUCfR70fmL5f3Vp/+mmeQPZaMjyRMvcwkM9jN4tLIEIbYX5IJCk61SAsIvZgAB44JE8MVSpLz7ua5QvK8stN7WxSLdA9qFVesM15aeUDx4iO19s1qr/qNLTUZiiEQIS/bLM+8ZKWukjQpoJVBlpUCU/BFByS6V/OP1ZxSJY2upYpoNAkq7inzW3nJfyhgHHMahySkHyt9ku7577j9bfHX2h+O33lVLVnoN4Ejok5iCWdvFr+Q0wonHwatDywYXJPY5jZ0jDFUu+8zus/TG16j+/d6g//5MPtvV7p1TiXw3znaGnDPTdoL3TJ33p91yZFBHy00wgxJ595VX1yqXHO8dBemi9WX+cXbx/9Oh10J0eqgACSbr+O1MhiqKVft3xg0oqBcmt4UTFbZzOC2u4fLmp9vhFG34e9Yiy604gInVwmlh+itRBQ47db7BJs5qu9LtKFNOujIha1b9IlN9hkFrqhw81WtKKmDVcnEAb+3o80Sr3RyQ9nP6smiJY4mjTK1WxXX/+QNQ5Sog49U7Y5Hz9kaas4YDrWowtAYyWm3IQqZgiOAA+r/5UCKS+XE8sPwWFV1L3BveHa8IY9R/3ASGAx4c7fpH6whahXEkr6RO2bgJtbGpboGVhCvQOra85pD8aJoNATJdy3ThxSOcF61NHZFZOKcjYk3u7eQnzACBevtxUE+gUwDLgB2BSzmZg7Vim/BSANswBhz4MTfntKIOL9cFOWV+mpJXE2AmE2KcAuIEpXSUezmBWdjRMBYGYrtjX2MwpgtbvGVhWKaUgo1+V3uv41QkA1+XLTUXgT23geUQACOz/PZQ59ZjyUwDY8hkS9R8A8PK6KevLlLSSGDuBm3zpKvFwBrOyo2EqCCR6iSV1srOzVcrsPE1QnbBzuyulFGQUk3Ba2/7lp4FA+XJTI+fsKnUR7gE7nCYCwJTBF5OfhVB+SveLZ0YYqP/k5rUQ3ycwPpiSVvUeLHlev5mdwIiJh9Ob9YfxFY8uBwQSvYw7HwBIrPfBkzGlPVSnXIDauDmwMkpBDnK3UeEH+/yvu4CRclO1FqSvEbw98e/6A8CgRsuOhIMpPyU9aB6r/lMKAIcRCUh9GClpJTEjE2gfSySHY2cVimCVP7oCEEj0wmNjY2NjvYNjY1B87lI6gNtoWymlIJmEJ2i+yMzqCcBIuSlMefOtIwCQMGHEfOEvYuJaf4Wk/BRKc4UDKZWAM6v+c/gBUDhvyGCA9ZUFMFLSSoIIsRMIsUoASs2SwhxOb1ahCBYzGqZXqpJKAZUV5ALwjjzQGPg7IbyntLvECNGk/1h4JsLHs+saIiJKGkdkpNwUEa0PmHv64sKm32llku94ZgqOpvzUrkgv3wGric5E164TfZ5R/6HQEbNXbuwzt4SIRP9MdO3aUenGSlpJECHNBLpYv8FZ77b3ajdd73B6swpFsHSjq1apipUCmtLF06fvF0QZYxceP9Y56r4xYsh0EMiw3JTwCz5+Td38aVEK+bbuqrmbH2TIxYjqPx2DV/+d10I4JOujvJJWEpOdQPZwjOaQrgiW4WhFpUEgWSmg+IvUPkRRsVJQdYFAHYNXy/owv6SV8cP9m3d8lCp5H+aXtFKqYO9m8/+nuTvSxee5REPfwM4Mm7fz4PLhS9RVOrRVVd3MYEZtv6SUwREqBwcD3wIlrSo4HK9UxXftuYGDQOAgEF9S+JICDgL9K0AgmE4HcRCoiiAQI/kjRweJcBAHgSwCAjGSP3J0kAgHcRDIMiAQI/kjRweJcBAHgWAREIiR/JGjg0Q4iINAlgGBRMkfWTpIBwdxEMhCIJAo+SNPB2nhIA4CWQoEIiI6gxhpPEMHaeAgDgJZCASCTvLHCB2kgYM4CGQhEAjQSv4Yo4MEOIiDQJYBgQCd5I9xOqhO2LndHASyDAgEUfJHjg6aroODOAhkGRCIkfyRo4NEOIiDQBYCga6Nfjh1yuQxK/0hRwcxcBAHgSwDArGSPzJ0EAMHcRDI8iCQXKMIB3EQiINA4CAQB4E4CMRBIL5rz3ftwUEgcBCIg0B8SQEHgQAOAsEq4kAcBKoECCRSPWNCw9z/PvlMH4MgAf9Rr7ldpJ7YTBLIQSCTQSCG6qkPwHmu2iBIwH/UMy4R3et9kg3kIJDpIBBD9fRbMntFikyQgP9s/4aI6GIEG6g/jeyOz+d/vD+wNSoNAs1sbRkQCJUEgZraBAR6tVB7s2ConnrTZIM0+M/pQgAITGYDOQhkMggkR/VIg7T4T6NvFquB7ZHgikDmgEASqufioq8W5egHafGfcQFv9Lq+bf1nTCAHgUwHgViqJ3ijmn4JuiUNEvGf1G5w6VnKBnIQyHQQCAzVs2GEAoM9YyRBDP5TGviWw9FBD5hADgKZDgKxVE8IAHT8JYMNEvGfy2O/+epK3z3PqcVADgKZDgJBpHpOHRF+JQlMEIP/TFxQF4F7PzqzSwzkIJDpIJAo+YPonDwXoBR1mCAR/5mQ0BOAYt7+G5G6QA4CmQ4CMVRP26UuAK76tGKCRPynhkJg3PyCxUAOApkOAjFUzwutANw6vMhJj/4R8B/n4YsB4O6jp5lADgKZDgKJVI9q7uL4TU99rdajf7T4T/HLkw5dXD4uVRLIQaAqgEAi1ZN4qnbX+kbpH9w8WxgSphfIQSAOAoGDQBwE4iAQB4H4rj3ftQcHgcBBIA4C8SUFHAQCB4GqYGpyrBAO4iCQSYpAckiQWAbs6OxIXw8AGOkA6CqHcRCo6opAckgQUwZsqSanvdjKYRwEMkMRSA4JYsqAzVz2x5GjR4/0SmErh3EQCKaCQGKDHBIEV935OEwCgDXj/QGxchgHgaquCFSO0A8AYDoAJJ8YxcJBHAQyQxFIDgliLRCAeuYCBcBWDuMgUJUVgeSQINIrA7b+PSKSVg7jIFBVFYEggwTpL6ezpwKQVg7jIFBVFYHkkCC90dsVfgD0KodxEKiKikCQQYL0Rq8OAACDymEcBEJVFIHkkCDpYNWhCAAMHBTDQSAzFIHkkCDp4MT82gAYOIiDQOYoAskiQZIyYPc0t36xchgHgcxRBJJFgtgyYNswR3sUbeUwDgKZpQgkhwSxK9r6qPr2t4nMQSBwEAgcBOIgEDgIxEEgvmvPrRK79uGKf30aFDYFgbjBtqgbN57wf679P6gP/GcDL/1zAAAAAElFTkSuQmCC',
    '1510.01234v1.2.png': '...',
    '1510.04780v1.6.png': '...',
    '1510.04780v1.7.png': '...',
    '1511.06278v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAtIAAADoCAAAAADnf1yUAAAf3klEQVR42u1deXwT1fb/phulC0WWIgqlgCxKW9nKokJ5WFBBUURFFDdQREVFfOrn+VT0gU/FDRdUiijIIipSRPAJ/FjKjqBSdkVaoECBFugitE3anN8fk2VmMknvNJk2Sc/381HS3HO/95xzTyZ37mS+YyIwGMGEEE4Bg0uawfBjhAEYkMV5YHiJtPX+4omJpP+8JjG2j4HW7IgvptP7IqrlhcePhXwYYgQIhI7S1qTtsXnmiCpz43gAJfkNyJRoxFHaGqLX2kqh3nJ770jAH6VV4Qb4UTpMxGhj91is/fH78IdvvQHA4WfX9R7+ggG+rP/kWynD3++3hDzZXMR647+GNI8GgHtCRLitc/PKrI9cIegIzd4bE/p8tCfL+3qkRh3bOnggYJ11PPLixOZixgDOPDUzDoLWpR8cy09+uoWY9fmMkjLLhE7CjjjzHiwgIlA1GPszEZUi1fbn4serlO3VEmjAtc+F9kOJiKh00MtmmjdMyPpTWxQDhKytT+4mOvmPrWKOmIfcW0XfpBR4sm4BIPxlK5F11H+Ifu140lOQDmMqeHZcKvI9Z0RmffdhKr4jbquQdcGEAqJpEavFHJHn3avprFknQyBylC7b/BmAmPgj0p+WzC8M2fp73/YZeyj+P8BrxULWB2YmRJtAr3wpZL2ibTLQcvqzq4UceXP12RDc9eHLn3owTbnteLv0RADLfpoDdB/w1Hcixoh9stVHO6px2Wn9yhuJaPRlm3sOhQpYz/+8/5146PnJ6ULUsnCDahOv2pPDIWEAkPjL3zEA8N6ECCM82da2GQBg7eJfAXxtEbIOGQcAc8cmCllvvwgA7XPEHMlIiQXQZ+a7Ue5t4x+3vVh4bQSAtDEXRYzRoE31+XBaL1+67xLEpGUe7CJgfSkBiMY5MWpZuAjiSy0n/wYov8D5xrz7AQBtcQQADhf2McKR8u9HSS8+i+sKoGdfIesJAJCz5V4x7svfm24FfhwiZFx0PAYAmv+9S8D7qtVxANCyYr0BqWldYQYQgzMixneX3glsxs268x7ER+kvyzM+rfr5ih3FX5mkN86cvNpW0rlJAE2dbogjHz4lDUfr2uZlxBSP6Shk3R6A9al5JjHuB6c9s2TWH/MXChk3DCXpM3/wGg/W2Wsrw8bEoeB8DADE4c8h1RsLwmG9sTICwO6wJCHrcMD8bs+XxKid4QZxSVfte2fFhHFTcFOLJ3tJ7yy6G7Kj9IJb4ozw4/dmraUXJWc7fzclJLff3OtFrAHg6+RLBLkj14/amNJ7TbiQcYOkswBwBEWeamP/RFNmr1VtShENAKEoFjAWrGiHdUgEgB3Zk5qLcW9fsq774mgxamUmg3THY+MKSrjRSrQL9jPs3self1fhGaLCsVbfnO0q+5ifqSLqPZSIjiH0KBE92K5MxJqIKlofFeWmQ/f9MxKD88WMfzDlE1X0xVseQs0mIup+O+3G80RE2XjBA7fdmIiI3q9ux0NhTeXJ6WWi1tY/brylgIQcUWQySHY8XNbSXQafODbeBGwP7yq9cbDR5dKLROQCr71iyPfUx0/YHYlGQgKAq3O2iFgD+NGUIMq95/733t57/apbrULUw955JC/njaHwdPaUAgA9lxTEogoAKhErYCyWEqX1i82XRopamzrOX3VTlZCxIpPBenp4SVhWSH8AG1JtKZx3n60lwZSLde0TjPBif0XToqKiSktRKRpFNAGAhtgnYg1gTlth7kfebIb2q1/95Wcx6kkfZ6175jSu9rA5sgEAYrGvMcoAoBxxAsZiG0AK69knf4oWt0bT1J0rRYyVmQzeTbz1XS8BLv7wOorjAFiXbrfvPF2eW/75XEO8KDj+EoD98S+1mxSWfAEAqjwcHGXWQOX6dFHuh/f1A2CavMb9OZyCGm3aAEcTPZT0sOLSCMCMpnGXngaAU+glYCyUEoX18uwFIdgbclX11ubrKrdFAE1wSIRaGW7wXj3s8AwRLcbJyn8TEa2/z9HQDxN/FVlHnS1z+8Lj2qvFUCKi18IriOgV/ClkTdkYI8ptji0mIqJ7VwtRb7itiKio8WwP1OmziYjS4yw0tjsR0dtNKt1bO41F1tJy6y3/shLR1L8ErItNoXlE1A3rRB1xZFJ8OgXn01/W0jhxKA3AzuSWy/or1h1AIkzdBT4kha2vc/fC897uhRIAGB+9EqCVD3cQssZJxIhyh4+cDgAnzvUXol6+LA+Y3vY+D5Z3XAngSNb7YZi0PwegJZNCRYwBoAIVHp2QWR8cffax8Y/e93migHWjIWtbAcf2pfUTdcSRSeHpFJxP/zlKr4g6T0Treix8m4jo4pXO484rCaUiZ7sXkka7e+HpUz2+T0zc9W8R0S89F+8cf0+JoPUyvCjMXf7AuPXZGQ8eFaPeO2TDb/8ceNoTdeXL03ct6vKulYgWpe079+xoswdrmXHFPYMujb3q1vfFqG3HkRQh64L739m8qffQfDFH5JkUnk6x+awzuP64tOJUGwA4WdoJAL7dOc3R8sfFbqiNWwCKsgpTU0StzfOHthDnPrTjYkqqSdD47M9F3fqaPFPv39akrzT+mZ8u9O3m2dppLJKRmlrvyqZuKb50JMB+XFrd76Vv+W8y39XCjgTRXS0Fp5LBYATRHeL2i+EMRpCU9NJ7OEUMBNO9h4XN+A5xdiSo1tLN+EPPCDCEvgpggLcsAwzvY6A1O+KL6fS+iODDSy2T+ZPN8BaT/epSC4PBMo8MBpc0gwFWLmWAlUvByqW8L8370gwGr6UZjIASEKtWVdMgCKh/1o61Pmp9CTPQESOtEcD3Hmqqavpew0Gjj7b6p9HWXlN7SFjtOmJgRgJJx8O1pMuPVL5fNyW9NK6CiMbdUbvWXlN7SFjtOmJgRgJamgYN2oTW0ReGTf3zx4t1ba2PWl/CDHTESGs+PawJ9Kl/GmhtpAypgY4Yac0lXSM41T/r2FoftYFB+k1GuKRrBiH1z9qw1kdtYJB+kxEu6ZqhHBEAYJLk5erQWh+1gUH6TUa4pGsGIfXP2rDWR21gkH6TES7pmkFE/bNWrPVRGxik32SES7pmEFH/rBVrfdQGBuk3Gan3JX2u3O0LTzANzQWAnCY9ULfW+qj1wUBHjLLWmEWh+YQ/PcrzTRzx4spRQVQPdy88X23aF3mYyNp3qti1KV9Ze03tIWG164gxGdGYRcH59JsL4pqqmnpKuobKpe7UP4229praQ8Jq1xFjMhIEyqV1eQuAlvqn0dY+oPZNkP6dkeBRLuW7WtgRvquFweCSZjC4pBkMLmkGlzSDAVYuBQuGgpVLUcebeKy2xACrLfG+NDsC3pdmMLikGQx4r7ZU+sGx/OSnW6Aeqy0ZqJ+kj5vVluALtaW7D1PxHXFb67Hakq/0k/Rxs9qSUT8ufSyXiEqbtK2sv2pLvtJP0sfNaktGqS0tv+Y8EJOWexD1Vm3JQP0kfdystuST08PWFWYAMTiD+qq2ZKTb9SEjflfSG/NbANgdloT6qrZkpNv1ISN+V9IhEQB2ZD/VvN6qLRnpdn3IiF/uS1eMTX8d9VZtyUi360NG/LKkX2y+NBL1Vm3JSLfrQ0bghw+2mH3ypwaov2pLRrpdHzLihyW9PHtBCPaGXIV6qrZkpNv1ISP+t/DYuuWDEOCHBvVWbQnG6ScZSc1qS3BzQfxAu3GPPjpudGJl/VVb8pV+kj5uVlsy6oJ4d6nUU+qx2pKv9JP0cbPaEqst1YW2kIHaTKy2xHe1gO9q4bta+BYABviuFgaDS5rB4JJmMLikGQy4uSCeZvKWxWR4HwOt2RFfTGeaXx2lB3i7uT3Z8D4GWrMjvphOGgDel+btYPC+NK+lGQwuaQarLeF8RkmZZUInBJfakoEiR/rkqfxG9qkeqS1NKCCaFrE6uNSWDBQ58iBP5c+yT/VIben9yG+JCnANBZXakoEiRx7kqfxZ9qkeqS1dSgCicQ5BpbZkoMiRPnkqv5F9qkdqS3eX3glsxs0IJrUlI/WTdMlTBag2U4DveIQD5nd7voRgUlsyUj9JlzxVgGozBfom3vYXrmuzvvbvhQ9U/SRd8lQBqs0U6CXd+835R0cVIpjUlozWTxKWpwpQbaaAv9Ri6jh/1U1VCCK1JaP1k4TlqQJUmykIrh42Td25EkGktmSwftLskz9FB7U2U0CXtLlXdzOAJjiEIFJbMlY/aXn2ggbYuz+ItZkCuqTLd+4+AyAPVyOI1JZgpH6SHnmqANRmCny1paFZRHQ0Ii241JYMFDnyIE/lz7JP9UhtqeD+dzZv6j00P7jUlgwUOfIgT+XPsk/1Sm1pVzZ1SzEFm9pSHegn+bnsE6st8V0t7Ajf1cJgcEkzGFzSDAaXNINLmsFA4NxO24bVltgRb6ezDXgTj/fOwJt4vPBgMGpc0lZODSOInk4LrP/kW/i7NI2Bsi361Hn0ua2L228ygkCWpiEiutB+KPm7NI2vZFu0qN2r83jvtltuf85IYEvTEBFNrZOS9pXYjNdCLB7Uebx22z23P2cksKVpAGxr26wuvjAMFJvRx61PnUef27q4/SYjgX56WP79qLpwxUixFH3cutR5dLqth9t/MhLoJf3hU6a6cMVIsRSd3HrUefS6rYPbjzIS2CX9e7PWdeKKkWIperl1qPPodluc258yEsglbZn3YN24YqRYil5uHeo8ut0W5/anjARySX/8RB1dUTRSLEU3t7g6j363hbn9KiOBW9L7K5oWFRVVWopK4dfSNIZzi6rz1MRtQW4/ywgC9OphwfGXAOyPf6ndJPizNI2B3ObrKrdFCKvz6HNbF7ffZCSwSzotDQAWJ31cc85zUZHuXngWS/kdMEa1RR93+c6QM62E1Xn0ua2L2z8yojGLQvMJv7ogXhnTr+ZXjmpJmsZXsi0axh7Uebx22z23n2YkCKRpiGh8n5i469/yc2kaX8m2aP1Ywr06j9duu+f204wEhTRN3d0C4DeKMG7VeXzgtjvuANXIAUvT8M0k4Lta+K4WBoNLmsElzWBwSTMYXNIMhsEIfRXAAG9ZBhjex0BrdsQX0+l9EcGHm3gDsvijzfASaet5X5q3g8H70ryWZjC4pBH4GlKsfAUv1Zbu65EadWzr4IF14Y6xakHCGlI6qa3f77eEPNncILfFla+M1GZCIKsttQAQ/rK19qVpfKUW5M4fbQ0p76lLB71spnnDDBI5cqN8ZaA2U/CpLQ2a8a9ZuVQHaku+Ugty589U0ZLWR229414iuqK5QSJHU4VL2lfaTAFe0hoLj/jH6+gLo0GbGmgzjbkYJdhBXENKH/Xaxb8C+NpijNs6lK/0UTdoA/DpoT9Bt1qQuIaUTurP4roC6NnXELd1KF8Fr36S92K82Wsrw8b4+3mDUy1oiFgHcQ0pfdS0rm1eRkzxmI6GuK1D+Up3RupPSWfvn2jK7LXKz7+X9KoF6dCQ0kddcrbzd1NCcvvNvd4At/UoXwWvfpLXC48Fo0wYHjPJzx3XqRakR0NKH3UJtt0VgraDxpX7nluX8lXw6id5XdIpANBzSYF/O65TLUiPhpQ+6mgkJAC4OmeL77l1KV8Fr36StyW9bYOUn33+7bg+tSBdGlL6qBtFNAGAhmIJ08WtT/kqePWTvF1LDysujQDMaOrfjutTC9KlIaWPOiz5AgBUoZnP3danfBW8+knelvTVoyIAHIi7ErWstgTj9JP0aUjpFDkaNtUcAZxGT59z61O+MkqbKdDUllwXHndcCeBI1vthNaUsbH2duxfVogIVosNM2p8D0JJJ4g9zqLpQYgT1+OiVAK18uIMxbgt7rZ9aKNsasyg+n/CL33hUvjx916Iu79b8Nx41VlvylVqQm4uzbjSkvKf+pefinePvKTFA9smD8pWB2kxBqLa0f1uTvi0C4BYA3ygR+YC6KKswNaWWRY5qOyNgtSW+mQR8Vwv/xoPB4JJmMLikGVzSDAaXNIOBOrx6mOb1w2hNhvcx0Jod8cV0psGfNvEYDF54MBhc0gwGlzSDwSXN4JLmFDC4pBkMLmkGg0uaweCSZjC4pBlc0gwGlzSDwSXNYHBJMxhc0gwuaQaDS5rB4JJmMLikGQwuaQaXNIPBJc1goG6eAnBBJn0Q4ni2acGBwsv62F4X7ylsNFCcVkl4fl9h4wHujc1zC1uNNhkXY05uYe9EIUt5xN7D6Li8QjVzUmfQWWhaPYmIKD48/d6H2qLjmHsGhrZ3aE9vHxvytP31gfFRIzQFqk+PLNJ4V0m459HIke4lrstS524wLTdQQ3vBEGQS0eF+Cz0YHe63UBmxWyNRGB6XV9CYE13RCfbQzem20KqltPcEEVFV4z+J6B3MIKK98se7p8gmeLDGSAXPjkvVerS6C+FADyX9cZI1d3SOkdN3DplEtBRjFE+z/0D5THmpNUW7pG3GKgrPUMSlGs04iA/kMie26EQYhPOhK2PuC02MUuoZBgDn7+oAIBQhALoMLmvoOJaHy47r4RrH+tgnW320Q+tbTU0Y7uEL45cupsR5hn6ZSaMP29lZscjYr7Cxtbpx1GasovAMRVyq0YyD+EAuodqiE2EQzoeujLkvNDHKcOda+kxX2fspZ8QftdzAjakuwgsxtTPZJuWTpjI9tqqQKWLkKa7M2lqLZnqdoEzxUarPh6mH0fOoveMRLj8l6gccW3vY6piYo1blOc+OfVXVf9BUhABgPVLmgcJy6iyK/la2lucRykuAqsITAMynzsp7VxacRmWunZIOrTvtxpeKE7ZT1arCfAAo2pplPgrseVE5sK3VGbF8VLux3YgObS2DixNQtCmgHk2ZY3VyHVY2X10bnC0SHGz2gVymSzaeM1nOOXFG52RwGdRBIp4PyUIrTy4JUM2ho90eqv1f26BOcxVTCABc0U32Tt+Wz66Ozbz+kFQPb2RseODfZuewGcMv5ly3WaNyyvtNcP6hJGwDAKvf2j7yIXKl2Dn6lzWjR3+b0nLWjI8SsmStf93w5LKJuz9fi81pzccCGR1bfirrvalb/OT/TftFosT61LXWha9VzbwmofN7wOhWXebY6Q/fOTnrv6sB4EBq8ycBvDs3ruHrPfDNv8rXjB49DytsA9tanRHLR7Ub241WDvu9/Ll/W1ROSLC3SXFdkN5UjwazPcef9kno/B4wqlXSHEdsDiubr64NjhZb7TlmzD6QerrMsjm1JUs+J3AmyMHgOqiDRDwfkoVGnuQOQeWWst0eqv1f26AOc1cm5+L6fXxKRESfJxwgmtS1ioh6XHaCqOrOYVYiGjqCiGZcXky0svF5eS/p9PBUeBf1at1OSEQ39FpEdAhrtChG3EpE1o63b8q/dJGzdV/Uh0RVL3TKJKLUG4ioImyKovc1A+fYKdeH/kqUGbWails9RERVyacdZ8GxWUT0JjKJiNJGEB1MIyLrtUTU6VHpRMc2MKWNIGXEslHtxpLRDx3OEVkfu4cUTkiQtY24VZYL1WjOHBc1e5iIrEmnZbHZrBy+qhtkUahnzOEpkTxbMgtHsmRzQrLo7Awug8pJhPNh43R5X+4ykdwtqdAc7fZQZSGnjZCby5iGjiAi0rjU0rj0AjBw12EASLsMCHl+2RL71t+LIxoB6SELXXu1yPX4WPjcO4G24XvdUphiD197af5IZ+uYhMeBkDv+kJ6ODSCigdKB2D332Sjp8fTuQEyHVmj06OJS4PBj8Xbah7r1BzBS+iMGwOlduwDTgy4DS62KiJ2jOhEDoOyJkZcApvEL18idkCBv01wH2kZz5jhuzHcXgaMPxMtis1nZfXVpUEchnzHFTq2jo9PCmSznnMijc+3ryJDGMNXlw87p8r6aS+aWst0eqizkGIW5i1caz6Adcbspd/dWFANABAD0CF8xQmr6rThiE4Bmf2rM1uUeV9dXhQChERc8UCQrBji1fUIogI6qS5zy3g7K4/tvBJC+C8DY1xaMxwLHE7dPbZusCrJXYrcu6bc87DKwA86I3V1Y3XU8CQCSTMuvlzmh0aaJZHWOx0379kF8NV4ZW7LcV5cGdRTyGZPD2dFpIUuWynd3fR0ZcjOMp3y4zr8bl2VuKdvtoSpDlpm7eKUxb9aMvvObXqs4tNhXKkcRHxkZGbngBd2nqdIzp6s8UDRRDHAQCVruyXs7KP/Cpfb2lsM/oTJLI/ufBxGn9iPrlagZ6fdb1QOrD6aH3IfyhzRySMQBuRMabZpoos5x+xs+g6U4XhlbE7mvLg3qKFxmzCVbTgtZslS+u890E4/DeMoH3I2l5pK5pWy3h6oMWWbu4pXGUfr5jN/bYxNgNdku5lJJJ1vTlWjaE97BLYVJ0VoJ9Q4GVbrr3QF5jtePDdx8eKTjr3ZQHy8ONXzttfNznxsrPYdh9ljtxzg4I6ZKx5uzx0r/tsffAGCpuMK1o6c2+WjyHI8fnp1ziyo2k9xXlwZVFC4zZvfU2dFpIU+We8zWGFQ9jEg+3ELtstotR/tfUVKolylClpm7lKvrUbp0+gPtgTPAqs2AGQC2Vg6xtV192WoAqFim4eS5cqFYPFHIWltdtQkATknb3wTgdIW73pcn/Q4AFZkABnSescf57OPWnbcDgGz3aN8i4JKJI/YCkRYgR+OHGY6InaMqjbs33woAO3Cja2+3bcrRFDm++fKZG9K0YrP76hq0MwoXNvlAjo4yC3my3HyhWoAcjUzLhxHNhzsoXFbPoaI9wxaqMmSnuQuTvKTLUQEA4Q0rAewJL8tvAuw4DVjfuX04AIsFiJy1fDeAD1rLtn2lXihsfZ3Lvp6tCYDZAoAsVVoU5hLn/x2tplk7VwKYKU3NOQBLGpUpejspTbO2/gTgs1YATOO/7S478mes3gvQJ1JRWy4A+OQiAHMfoOtfADkGtrXKI3aO6jS2XACiZ36bA1S9edfNcickKNpKZLlQjqbIcdgjXyWaoIxN6mrz1bXBGYULm30gRTZlFs5kqX23p0BicB1UPoxoPuyc6vcVLgOKObRYFO32UJ0hWy7IzOVMFgsAhL4qUa6YPGN5yPYVKy9pF9bpI3PpN7eY5sXeFfLd9x8eOTily4dh2PH4trzNnVp26Dsp58SCXv3t1Xj/zGWm5atP9QHwffLt8np2EgJbnth+aku7NZNO/7llUKSK4rfxm/PXnCl9enfO2qqugKO1dfpzJafm91t4d2cg+fvjF5d0WrVh/S2O3grKVoP/mXd2Sft0AOi86BPZYqrNgGeLj3/ZZ/GGX+7a8fi2ExsH5R05fDrvi+ThQMqHxRuuT1xpG1hqjYYzYvmoNmOb0ZVdnz95YGr36aEKJ2zf9fY2Ka5rHJdQFaPJcwx0mPFFFOCM3W510O6rusHZIi0e5WzSQPYlma2j3MKWrCgX3+0psDG4DConEc2HZJHt8r4yAQBgn0Op0Fo72lOOSaE6QpYoO9mn3MnU9olteZs7tZTvSztQuff3CrLafmB3ZK9F2Xr8oNXbn9Z4pnC0Hs8250k7ylS4s5iyc0qtbnvnH7BtcZ6doqbbZS7/7dhF21/FZuvB3WVERGT+9azm+M6IZaOqjE/ss7j1X7tNRaDI8THtzMh8VQUtb1GzqQaydVSM50iWNhwM6kzLSHTkQxsKhzTcsrfbQ1WH7DRXM/n5Q+KOt868TccvZg93x7S7EsEA39Xir7CgUof1Wz1LSsu5ous57Gtp/8TUKQW//iZ+mA472fqLFxrwpNZv+PfCwxJmospwcfszJ5LCeU65pBkMXkszGFzSDAaXNIPBJc1ghKFasQ9v9VpqLjaCgBJVYQTAUTp/wb2fALlf3bnImyEkFuT0/9o3Lp+YP+oznjhGjY7SnT/NAdCr1w6vhpBYsGdjh1E+cTnps0M8b4yalbRd7MPbyxfhQI1ESmoqXsLgkkbtK8MwGLW5ltbUTYGmAIuQ6gigKVKi1qipyi1BxdLNWgoulWfO4GKuk1UpqsJgKEt6Rs+Eq+fM6du6z0rc3eqKj13FPgC8cmnC4AKoBVjEVEcALZESF5mbj25ePWHctIRbq1wVXH7q3uK1KS8tG/S8TSVHKarCYKilaXJDpxOdCFtBRNduchX76PE00a6r1pKWAIuA6oiNRS1Sotao+a5BEVV2fJYOaym40JUtson+7n6zVSl0w2CoIC08EgcvBeIb/wBYU6/VVDrZl7H9H9AUYKlWdQTaIiUuGjXfdIxDaLdv0E5LwQVRN6YA0a8sXwJoiKowGKq19L1ZeVhz+3cV2NQPGHG2R+4PGxUSJP+XPiVGS4wE0FAdce0ODZGS34ojNm3atMWpUdP4HICwOIWCi5pqIFagOlEVBpc0gFsbfo1171X9Dz8N0ZIgWbM3aiLciJFUqzoCbZESF42axwv3onTd89BWcJEQK2nGeBRVYfAmHoCY4fMnNIi+c8FNFKkh9oEbJva9dsjdEBJg0egOIY2a+AkfxJ+acZtcDcWFqqS0E88ZQ2gTb/SeqcMw+sdFN2nIhgAx6D15/NHqBVgAze4Q0qhZ13/W67Nv86ySk4UhqIkgDqP+lXR6/A/d0T/+3f7QEPswW4DnYkech4YAS/WqI7CxqERKXDRq2jyXsTBzdbFMrkZBlXUKuDj59uGKITUEcRjg22kBIOR47zSYzrQfDBexj90Td/+xceDGRQXfhaZCJcCCLdWrjkhiI8ddREqSlRo1uGTBqqwVs9/cNyRCS8Elo8cfR/54Lv3DMOWQKkEcBsNx72FRWAxQgkYAUHXQclUElcR56niyqKO7i+ki3SWc+LujY71d1mtWH8D66wMjJ2tR9Uyac6y0UxjPGCNgbqddkvEzAOCT//2o1dwzaQ7PFiOg7mrptvcoAFSt1BbAtFTyZDECTPRgx9tX9Yo6tKb/Yxp7f6s++L+GA964kueLEVg6HgW5Za3aan5vVCEUlSF8oySDpWkYvJZmMAIb/w+71Fxcvdcq0AAAAABJRU5ErkJggg==',
    '1602.02332v1.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAksAAAE0CAAAAAAZSihXAAAse0lEQVR42u3de2AU1d0+8GdDAhGSgIEgl4ARy0UI1wBCUQIUxBJNG7eIrfoi0lZELfxowVZakWKVtta30B9YIrTRCiIisSqgICq3glAUIQgVIURIIdyTcEs2yff9Y27nzGVv2dmd3T3fP2SzM+fMObOTndl1PnlcBFGiQlIJYheIEseSKKcVEeWKvSCqkZVLRC4CXLF1zeSKw0vAiM/ZReIcJyp0lWjx/NSHBnttV7n/bNqoaJjgZeY3NqG5YfGZg2c7DImx17S6TPq3dTuXl9mG/hW0eF86v7TIe7uTy+9fDBwd/rqP/n2vYXN1SS+Y8rM+qTnTHvleqz7GxaWvjl8Za+8PZ1Y+1nvMG++tmZ49tcJ6tja8gkQEMlQh2tSS97rDTfQ2HrZa3LCAiLyuYVdx06lv9RURvYBFRFSSYbZ6n2kU/aV7CffjPiKiczmZJ73MNpSvIIjI4n3pwx+d/cjHQZgEIP/fCy1/Ob4EvK8Rlrpwb1cATZAAoNcdVy3mEWuVjCYAkD7zxC+8zDbUr6D5sXSyzYPw563fldPCalGxzzXCUqf7MT/0OR1nV8NdsCWMr6D5sbRq/HdaF9dIj+tOn8aV0gbdQ6nqz56UTpSHP5bOzN98dERavP8p/Ro7rgJA3ZkK1JVeDdveTGIvrG9nBoiLOzbXlsX4sVSFbkD92XIAtafOKZ9Gymx6Bc2PpS23J42vfB8AsG7ADXPn/fqdMbNq2YdSHRyU8QQAfDLoo4YVc+tR+/ONqcXfOQzgjV9d2/TAA/9Q1/gg//NrM2d7sK1/2znr/7BrwqRwfR/yrf7MD0PbqwPEn15ped3vcuQFT7frfMeZGDyW3mzyJLbnZkwGCru1fwkAUPN84ZaJs+15Bc2uvY9OJ9qMCfJPt9zwBdGlAXc1cA8pz01EuW4i+qTJHqLi5htpaeeDRDP61RMRdX9Eap3rJqJ/dj1P1PDoj4jo26OKiA5jU9iuQ4mI/hcvERExAzyUS0QNw4goZxrR3p4fxdS192Hcc/Jk2fpHbnqHiGjQWCKqSZxHRJTToZyofnx+qF9By2vvlfcBt2W+e1n6qfmdfYAWT7+3hnsoVQoAmjp6AJDSNROtqi8Do/YeYftKAXD1sQnXA64pKzYBqfsfBG5KKonI76k2wIq9ewHXQ9LTBwo/HRlj70gHV65cc+KBQ3cDQCsAaNpMWpDbAUiY9Y4dr6Dpd5XvtjsI3HTi3fuY50Zhrdv4EABOfHkngNF7Afc9rtJ9O1Cp72/viWwAyHa99x2gZwLQpOnliOxhbYCDs/r3Gn33j6UPrQ8eSIm5q+7pFtcxTQEgJ8mOV9DsfenL7p0yMzMn85/kUlMPmzwEgK/RTn7UUDj0tdbDjB3+B8kAkND0ICA9RH1E9rA2wOTNTzdfNPp/GgBsKmk+Pc4+37lseQXN3pden9EbAD2//mIr5iNBdXeThwDQFcflR7MKP78Z24AGl/Tl/bLJ0vM34xIAeGq+FeF9qA3w6+vmzr3wyszJucDY6UOHjbsv1g8fqmN/qLLjFTR5X6IDvQHA9cNa+RuGWgDYjHH8Q7U6Zn8OADXF1X+eeDNwGtiwHUj2AEflNQZk7ACA3bgzsvuTGeCBlcD1090lAFJw65wpsfztQDMCUFHDvJg76ux4BU2OpRUXpH/vwjLpm4bNp4Arc+4p4B96PAA8lwHXyzvWAfhrZtJ1dQD2J109mQ70+xogeY0WS1YdBern33sXUOsBQJ7wnuOuoQYA2AEuvgKgdog0oJmp7guxdPRUclc8fc8DWJMmfSW0uwJoeMGeV1D/gXLrwLTr+pUR0Yw+LVKGPUFEOT/81dIVo35Twz3clZ/eOv+v+enpeaeJdgyYveq3q4lWd5q/bt5nU4f+so7oyI3PzP2EdslrbBj53JL8ObW0/e5W1+fvLBqZkjX+Qri+E3jvvhE3pN4w4r4P2QEW3zl7xdpfLqD389I6FZx+t2VKj8Ux853AF+4eKWkjf6H+fOa2uevmr+2YMuYC0eiyWYWvuUP/CsK/e+EGZhd9U909UfeQr1MXuyUAqD/k6dmUqloCgGd/Vjqzxn8vdkuM/H1h6gCrrkv8qrZrcpzcC3fuWNe0fakZLVwAUHbJhldQOY58H0smD8U9hmLOgd9X6akzeyhKVKAOZUPeofe+f1D/UJQoy/Oc9RUGmqAuIUH3ULzfizkHc70k9quYs3AoosJaTZ4BMCK25jQiDl/HEU4YgHC7oiDcrrheEtdLohBXbjdSFWpNGkKX6//QStZT4v03QHxX6U/Zp3ElTRq6bXIut3HDth4a333dwy9MmvXd71Y7HDRH/Fgiyd3t3/phwG38qx4v3WaxJKBtqjV4aXZju/A9NL77BZ8ta4PXPj8cwl0Yk+c41XL2CLiNv2XlaAPapnl/wXbhJ/GVu18/sgkwLad/CHdhTB5LquUMvE2jPynkOKELP7q/2BJA23tCuQuj9hyngtba3QfqeajJW866MxWoO3oOQM0x+QYCuYlZG7UUGcoDUA3UWoxH9qPBcFtFqtafPWm1Wc+pc7h4yeKspJhW/UiVsfAjlG6oLq3kmyqtnAKaw3IsKaCVCguuHL1tOws1ecspLdg96eGGpS9tv3UxtCZmbZRSZSgHQDUvCgBY8u3OPV4EHsjsVaSMR/ajHLf1r1SpenBQxhMWm13bp/3Li/7SeTO7YaUU06obvzYWboTvP3D0jQfuL+jV5WO2qdrKMaAZ9vy9Su4GTxW0LupYSfRBqwsc1OQsp7Jg2l6ivzc/zzYxa0M6Osuuwzyb5yaiysxJRFTfu4IBtrlultv6fY8uK1Vz3Vabbeh2z7aT7VYyG1ZKNa3S0Iz0lx+hBIGpFMVsU2Z+toBmOOEmYcP7kgJaK59ypwGjE1aYQk3JJsoLyvoCPa6Usk2scScjQ5l19F407ZHV1cCRR9sywDYFOm7rZzFSNQVWm3WlHhnW7uQEZsPK75pqWq3oLz9CudrwTU08rKNAsz3X3gpo/ayy6TYAbb6CF2orLegFIBmXuSaWbVgZqq1j8KKT5y6fguUzeGCr/8m/MkhV88321m0YetNqRX8tx8Q0tfCwjgHN9lwvKaC1DG2Tk5OTlz/pjdomM/+tB9vEsg0rQ7V1DF60fcFiuupJ44At9D8FKVXNN5uu2zD0ptWK/lqOiWlq4WEdA5rteV86LIPWW9B6oHkTxXIayroJ00YvQ62efXTU9iMTmPHkcqML+NYGnVQ12ax+w9CbViv628FqTExT3fwcB5rteV9SQGvfDhsBoOYd/o2IsZyGMm/Ct+FkKLw8O6LHov19mPHwowtkimZS1ctglA1DZ1ot6a/lmLSm7IacCJrt+k5ABq3JL7+3D8CCThzUZC2nusADwIN6ME3M2kBPZ7V12Gc9HumkNGXVAHY88jbVn/wvVqpe5obGDaZKORsqG5Z/VEyrBF3N6C8/Qqn/WniYpmwrR4FmhPq+yrnPME8cOnak4vjfeheg69AZR8uXDx7+r8c+PfWvLptmVHz1rzHJfRZWbvlO1u6pO8u3jvlCWfCfrWOfWXD53wfGKU1g0kY5p3b/S231G3e7/pGa+YS2Tor67E2P7Ty+vXt7AD1WLk5kxiNt87gyOi/FTQd4862Fxw7N67UwEbun7izf2mKW2WZbTt939KN66W9byhtWKvOOXxw/t+bm0dg9defx7b1GKE3u/UoeCz/CbdP2lW4b8rffXPhsV4HaVJv1vQnsLmxxS79Z/z347IA/N+H2WOAvo27O4a+5z5jYARa0ll/qpv9r4zrLaSiTJro2rAz19uz5xb/Wjwd+cVuX4es+c6lqNRh5wzCYVgv663VMSlNmQ3aAZuFQLKvyyAD84d6s8O/XRm1Y3FfpxPr9wKrqa1nxtGGI+yptqrF7dmx8Oq42jHhwu5Gp0+XZSRF5v2/MhoXbdeSxJByKuF4Shfj+OycxZi3jkY7mOmEA4hwn5izOcaJi+DsBRRg6Kj5VWEtE7h7dYMszaeIZ+rJ/FdHBKc3d3tY8cvuK8P1N2U8nJ0wL0ZZ9zUvpXtsT4dsLzrxHN9jShKFiEq2MZViVobCWUXiOY4VhkldjGWZlKKxl1B1LRmFYHBnxKKyl86wlH4jKAUneV7I3wpZqd8drxlIGmApnZKCmYgvrS6tQ8/Z2uyYprGVkrSUXiKoDkqyvlIoThgA0VKgCTIUzslBTtoV/uWvj4z/9Q+fv2XNPobCWEbeWbCCqEUiqvlIljaowlLmkggpVgKlwRh5qbiJ6s9lFquv2czpiy2caYS0jbi25QFQjkFR8JV9tTL6VUQGmwhl5qFkCvNGtJZr0fwNdbPp6X1jLCJ3jBmf1z56+6cdsNirgPpdT+s+tLJBUfKXP+qyy6bZt2/7V5iuonBE68tjqPIDElnZNUbGWMLGWzLwUa3l1ObD8fmhgsheA0XuVT11aE2VP6feYWVN2QzCzlohlawk2ENUIJP2GgSzATDeBmvXA1LMlqP54FsKVCiqsZZitZS4biGoOJP2oZZzZtGjc9vEFbU8t+r69l4XCWkbOWrKBqOZA0ldJqNACYLL18fCXf7fMvkNJWMtIW0s2ENUESCq+UnvFFGGohG/KqJA1mzJn5KBmPXDjzMIVxRsr7ZqisJaR+k5ACQ/lslHV6MwtSkJm54ILT/ZP7fe43Or9cakdvl/6p2+n3PI/UvjmHiVjk+ijoU+9MnMtvT8uNePOIpJDN9mozcp+nbt0Sk6895It3wlwqaDpeW+zGZ/qvNbKgyMiupB1jetAyQ+V5mWMWVX+lSbG7gm1KZs9akt4rCO+EzAcS5W1DYf2XSUiyplIZSUe6dm6ks9rqOFiYP3X7jknPThxqMHLaleydxBR/a5bnrHr/5kfU6ZhKLN5nZunX+vkwXqTJsqe0vaYSSlNmQ2pu0Wq8gOeGLlPwMt9leEKRF1T+L50cl3/bsTvMRTW0p77KsMViNq/pAwA6j+4M46RJ2L5PoENCw6Vff/5W8IwhJuKZ/Yc3PzwprGPIn6RJ2LZWoY1EPVM6dXMmxKc8H4vrKWwlsKhCIciCsJaQrhDMWdhLcU5TpzjRMWuHXAUqrRp/AHPMZ5FZnBfwB+5fYUf+DCIPkPgExEknjTtPeA5RkZkRrO13L/1Qz/wYWCRltb6UOkmxD5Rh0JNe/c9RyEyG3mOk6VgUigjLa31odJNyH1ikh/6MUmITHuPJT+lYHFo+iy210qGVz/Grsg0O8exeZY6VqmIQYYVmpBEvSvU8i5ZVakoQ95gwoga1W7kNZSGfvlEwyzqz5YDqD11To9Cddu3EpY6YBlwAxORqTWNapFpPJb4PEuOVapiUJGCbFm7Qi3vklWVijLkDSZuHPLkH1/IbVNm0o28htLQL59omMX23IzJQGG39i+BR6G6OVkJSx2wDLiBichUm0a7yDR8CDDkWWqskhGDEivMc5uQRKMrVLtkVKWmDDmDWde3lmhP03mmPJFy3RxPNPeJus80hlkMGktENYnz9ChUnpN+OnluduZGYBlwA73I1JoGLzKd+TnOmGepsUpGDKboD0kvrlDrUlOVjDLkDGbl95JQde9tvzLliUjheaJfPtE4CwBo2sy4ZorpdAB25kZgGXADnchkmka5yEw0s5F8nqXGKi3SGQHvGY5al5qq5BIdGYPZdAJoStVrTfyLgvTLJ+pn4fVrkIt1AJJaWqZZWoVZBtzAbNdFefplgjcbmewrfRJ+uUKtS01VcsqQMZgpPbHs9X+095Mn+uUTA8Khd2VkZGS4rYWlFbAMuIHZrotykZkYQDilN3LpzRVqXWqqklOGXG8lTzw5Fts8I+3liVRnHrz5Zg2AZGth+XVzc2AZcAOzXRflIjPBz3BKAF7JpTdXqHWpqUpLZXh5Qr95wP40+3hiMwJQUWMevNk+Kysrq521sCy0AJYBNzDZddEuMo1/T8CQZ6mySi598rKGKnmSaHSFWpeaquSUIWswf/bf15NwdWk7M54Iz2WuoV8+0YBD+54HsCbtqg6FKlRSLyw9Hm7mBmAZcAOdyNSaRrvINORaGvMs1djKRDZ9snzrmC+nKhmUYOIfjRmOape4fvmGzWuXzT8wrinURMcPpkmRklLS467pA89vWjnry/nNjN1Ia+QoDWGRBclNxzgL9H7rxJU13Tds+aTdDGn8XNIk+Onsnrrz+PbundTB9PmGD7MEAmxgTL9UmzYi/dKZuZaAeTill0xK+JHhWH6pmwtXB788BGjYM3HCHMBnomOwUZAun1/knTvWNW1fakYLl9ewTtM0S29hlgE3MDQNOv0y/uxAqFWluK8yfu+rdJKqFBXlWYS7/9hzcPPDm4Y/6hLvS8LHwUmqUhxLwloKhyKul0SJgrCWENZSWEtxjhPnOFEQ1tKWOlp69tYsCGsJYS0bHcK4fByKLbsJcdClsJaOtpaNJn8/es2smwiDQmEtI3GOazz5SzLrJuKgUFjL8B9LISJ/+m4cAgqFtbTLx7F5ljoGqWlC1U7qm2j5kMwqNeWkk4MfV4QLFAprGbljic2z1DFIVRNqdlLfRGnBrnJk/JzNz22EJjTlfsICCoW1jOjnOCbPkmOQmibUOKahidJCW+Vg6mYimo9iRQ5q/dgS8SispYM+xzF5liyD1EggwzENTeQWzCqT+g8HMEGVg3qVaDcoFNYy4tfeo7DWDZZBaiSQ5ZjGJr0BdpVTO+fw29GrRNgNCoW1jODfOdHyLFkGqZFAlmMam6TzqxxCS0uVibCAQmEtI/2+VFXdnY+i1EigFceUmrj4VbroQ3l1KjFCoFBYyzD9/SVAyrOEuaY04Zj6JtoqnXp8CgBXzYIiIwMKhbUM47Gk5llyDFLThAzHNDaRWmiruAo3lgC0GFdlOcioxLCAQmEtETFricKc/xz7z8zRCxPBM8juiqbU7KShidKCWeXGET+vPPH3Iau37LpJkoOqykSAoBD+sz8IawknWMuB2UXfVHdP9KYp9RzTtIm2SvnZng1ftmlznaGfwEBhyO4LE9YyXHYgiDzLcEVgivsqo+y+yiDyLMMVgSkqqq69N+Qdeu/7BwPqI4gmouLB7QaRZxnWCExxjhPWUhxLwqGIEiWsJYS1FNZSnOPEOU4UhLWM0/LCIvWLzhw822GIf+0D0Jmxpi2dcIOnH6YwAI8ZCmupF5OfTk6Y5ier9K0zbdCW0Zxr6ccxuhChzLS0Ax16sZZ6MTl4aba/rNK3zoxVbWnbOS6gSEvfmZb2oEMv1jLJH0GZFAy2jFltmWgzYAxVpqWd6DDcrjFWtWWCOU5UOR+PKtXSQ0NLLGieaamPtPSdaRk0OgzaWurEJIDLZQ1e94ovbOlFWxr2XzRqywRznChzPkVMLhrYuW9R0dBOQz7AfZnf+v8GaGiJBc0zLfWRlrIpVCMtQ4gOg7aWOjEJ1DxfuGXi7FoYqKm/2NKLtjTsv+jUlpY4cRMbSFna5M9E5YlriWjYNgM09IYFTTItTSItKdfNRFo2Ah2GylrqxCRRTodyovrx+Q0sNc0LQGd60ZZm+y+wiTv1c5zG+TQxmXXH20DbVv8EGgYN00NDb1jQJNPSLNISKWykZUjRYVDWEjoxCSC3A5Aw6501JtTUL2xprS29Ysso0paJljjxMocq73/weKdN97y5sNm226F3g96woEmmpe9Iy9Ciw2CsJXRiEkBTAMhJWus2p6Y+saW1tvSJLaNEWyZY4sR6DlV+77rX8fGL9euxbpzmB+EbC5pkWvqOtAwtOgzGWlrNx5V62IKa+sSWsNSWPrFllGhLr98JMKgypeC1x5u1GL/8u5QMHObdoDcsaJJp6TvSMgzo0Ie1tArypKruFtTUJ7bMPWylLS33X5RpS69v+CyqfGD/s/l44N2V3wWgc4PesKBJpqXPSEvYiw79sZYwBnnWAsCOunEWyZ8+sSUstaXp/otCbZlgiRPruYxLjG77zwEY3vZPwwHooKE3LGiSaWkWaQnPZS3SEiFFh0FZS4OYBHZXAA0v3FPA7hUt19MPbGmpLU33XxRqS4O15DgfgyoTTtyaC9fpm+8AoIeG1lgQZpmWhkhLCXMmzpYjLZs0Ah0iNNZSLybvTXjzrYXHDs3rtTBRm9JuNtfTJ7aEtbY023+BQVTn5lrCTExeTEwBqpAGmLlBb1jQJNPSl6oMGh2G0FoagjzLLqmc1Dz50yu29K4tjfsvoInHlx0IT6aluK8yHu6rFJmWItcSUZVpKd6X4sTHhSHTUhxLwlqKY0lcL4kSBWEtxZyFtRTnOHGOE4X4tJaOzKFE0GGTlp8szQilqNDeJ7Dzxfv2RmpIR4e/7mONk8vvXxySTZW+On6l/5sVFdSxJOVQIjzsUrfcty4MOGzSqlhCGVURklHl45IQPnapW+6PLgzZ8JJCmN0pjiWEKzfS3+UR0oVRFSEZPec4JYeSIX4aB2SsosExagyRIYEMuwRMki+V5boQTTPQyMpHP0vbnNrYnCtKm+WWaThUVHDHkppDCY34aRyQsYoGx6gxRIYEauwSAIzJl8pyXYimsvilIZ17vAj8MDO7iJWP/pa6Oa2xgSs+3a7zHWekzXLLNBwqKjhryeZQasRPk4SsVTQ4Ro0hMiRQUYNSGZMv5eVciKa2+GKbHxNRQ3YFKxHz3P7+/RilV5YxMmPLmUa0t+dHGrXUlmk41PnlUGvJ5FBqxI+VhIxV1DtGhiFakkBj8iW/QNKF2uKWD795BSib2NZUIvq+DJJ7ZRvzYztQ+OlIQBGQ2jIVh4o3nCCvvdkcSo34dWMkYYK1Y2QZojUJ1CVf6hZAZzTx0z+segivTrGWiD6qt4FNcmP78MEDKabSVMWhooK8XmJzKDXiZxFGqXeMLEO0JoHpVuGY6WZGEzeP/Ss8lW2tJaKPSjcyRmZsm0qaTzeXpioOFRXk+xKbQ6kRP70kNFhFeE281NQgjMmX2nKXeU9TCr44ejdM5KOfZznAS+Ox04cOG3efWTsVh4oK8n2JzaHUiB8rCQ1WEaY409wymq1qtpzt6a6OS7bk6uRjoGXZOAW3zplSZtZExaGigjyW2BxKjfgxkpC1inrHyDBEhgQqalD5BkqXfKkuZ0I0OeaZ+JNXs1y8fGSQo6+SeuUYIzO2Wg8wM9V9AZyAJE89NBwqKui/v7R1+P+++fM3kPEDItow8rkl+XNqiWjHgNmrfruaiOjMbXPXzV/bMWXM2rtbXZ+/s2hkSueCC0/2T+33OBF9NPSpV2aupe3KoqzxF+jIjc/M/UTp/f1xqRl3FpG2KsnLlQW78tPT804zi4lOpJwlIlrdaf66eZ9NHfrLHfnprfP3+PX5WN2c2riOGdvKvLROBaffbZnSY7G0WXbclf06d+mUnHjvJfGdgH8DML0Xjs2h1IifJgn1VtEizlL77tnCMiqrWizXejreySgfA70vzL/GWvE4FOJeOGEH4HwcKu6rjPUSONS51jLqfkfDhkPFOS4O9muYcKg4lsR+FXMW10uiIKwlhLUU1lKc48Q5TpQohNQONJo2OltuigrH32RGiGijJDctMKOjjaMAmCE+lhpNGyW5qceM0RDcKABmyH1cY2ljEmDEjNEQ3CgApjNzwPWYMRqCGwXAbNw5To1h1MtGFT/yqZZ1p0/jSmkDYOIoAU1uagmNH1eEL7jRbEiKoTQuU2am/CsPThWnDk6VdOKxpMQw6mWjlubIp1quG3DD3Hm/fmfMrFqjowQjN5WERhlthim40WxIiqF8x7BMmZnyrzw4VZw6OlUSjruvUk1fZHFinpvBj/pUS7rlhi+ILg24q8HEUbJyM9fNxT8GnlgZzD2GxiFphlK/TJkZM8NcNze2Rg8ulu+rNBxLm1t+TkQvE62+/t9E7+Er+Vi62PJnRFSfvkhbQ6mciUREb2M1UU5fImLWpSHDSUkCpTw3UUPPsUS0se9B7VjKcxNdyfw1EdEX+JBobEY9UV3SghDtV8OQftCbiCZkmixTZsbMMM/Nja3Rg4vlY8lw7a2mL+pko4YfLfIZR2Gt2+AoWbkJwBj/CPuDG/W0kzWU/LKH5ZnxM+TG5txUSQdeL6npizrZqOFHi3zG1NTDMDhKVm4CMMY/wv7gRj3tZA0lv0yZGT9DbmzOTZV04HcCavqiDidq+NEin7GqurvRUbJyE4Ah/jEcwY162skaSn6ZMrMO3AyjI1TSie9LSvqiHidq+NGYz1gLAJsxDgYoycpNAFz8Y1iDGxm7aTCU6jJlZvwMoyNU0pHfCcjpixxO9HhY/MinWgLYfAq4MueeAqOjZOUmPJe5+McwBTcaaCdjKPXLlJlpM/Rc5sbm4FRJB9qBt5fk9Gq5tf3P8Nb/e6zPnryln+c++9mz21zD5gz4ePbI7iUjxmlrKDWwW9bNzZcO+3XTDxZuTc65byIAZV1g2+yCzJ2DJ2Tkznp2G4b+PQM7H/tu30M93cDRUZNcubm75ac3Pj+m9dr+s5P+NX+r6/anDr2yu82gwlaNv5fHZEhVuecTPWfq7vnbNv0yZWbqDOXBKWND4wcXV/d7a+mLRpwo4UdDPuPA7KJvqrsnWkFJVm4C0NBmwImVjdyv0pDMDWX5pW4udWbGBMpQjU3YAR81MLsoivZrVBnKuLuv0lMHYShFheB9acOCD68b8fwt0fM7Gk2GMs7OcfVogrqEhGjar9FjKIW1FA5FXC+JEgVhLSGsJYS1FOe4WDnHhe0bOGHkhGkKVcnphlEGzoSPc+KxJKcbRhk4Ez4OTjRNSdEIzoSPc6aPi0ZwJnxcI89xVljMaxagmttnEv7HpBvWnz2pV2b2Jv4JHxfBY8kSi6libsm3O/d4EXggs1eR2krN7TMJ/2PSDQ8OynhCp8zsTfwTPg4RzB2wxmKqmKvMnERE9b0r1EZMbp8x/I81cpKGY5RZ6BP/hI9zio/zgsU0MTcvtYroq8VaK8acyeiMWZs1clIiJaPM2Ja27Ffh4yLm47xgMU3MTZ67fAqWz9Bacbl9uvA/g5EDq8zsT/wTPi5S10tesJgm5toXLKarnjStFZfbpwv/Mxg5sMrM/sQ/4eMi9Z2AFyzGiLlHR20/MgEWuX268L+b4O232P7EP+HjInUs9e2w8WEANR80fP0LXD99Z8kF+d/c6j8/Kou5lNtG9FjUcSLY3L586MP/lLVb6I0c4LUlbPJx0rTyjVu0njIAYEDGjvshfFwwBtwSi7FizjVl1QC2FZvbpw//a80aOSnxj1Fmtif+CR8Xwfu9rbAYI+aa4GL/Q82YNqo5a6GCNGZt1ci9ufvZbRj6k2WMMtNa2nL/hfBxYRyA2QfKE4caiCprGw7tu6r9S0R1JZ/XUMNFIqJz89gGV7J3EFH9rlueYZ5k1qYTe2uvffbNFeNnSdOWNn0+PnGowXKLVlOWq/yAR+RaBvz9kh91cQ/R70vZZ94aK3/PeVegYwi+ZbD71YYtimOJQETB3HPy+4FV1deyEBJzFn6tJnyck3zc5udmbHw6DaExZ6HXai7h46LINJ0uz04KnTkLtVZzCR8nfJywA8LHiRIlfJyYs/Bx4hwnznGiIOxA9FejAxVFOcbHIcKkMZhARSEt7TmW5OTAgJfBIaTRW6Ci1fiFtLTnHCcnBwa8DI4hjUkBj19IS3uOpeIgl0UDaSwW0tKecxwTLQjNSyrJgSpLZNShtkzziEfPAag5Vsf3wnbObkcijRxkDJ3AlLMPFetZDqD21DlAFZWGuSkG02RYNsvQ2DqWmGhBJo9QSQ5UWSKjDtVlUklLdk96uGHpS9tvXcz2wnbObkcijRxkDJ3AlLMPpR+252ZMBgq7tX8Jqqg0zE3xmSbDslmGIsasJRMtyOYRymlvGllk1KGSBCeXvGTaXqK/Nz/PpxoynTMPJYHJdNkYgclPR8s+zHMTEQ0aS0Q1ifOY1EXD3BSDaRiWDTI0tu9fan5nH6DF0++tAVpVXwZG7T2ifUvzlDsNGJ2wAkjd/yBwU1KJsQN5SVlfoMeVUr4XpnPmIVLYhiXAG91aokn/N9Cl0b8qU0cPAFK6ZipPtAKAps0AVOzdC7geMpmbK/XIsHYnJxiHFbpxxdW19yisdevzCDmy6EUdSkt6AUjGZRh7kXML9Q+ZLkMmMA3ZhwnG1EWTufVGpGRoLH6/lJp6GPo8Qo4selGHyeBUoqEXObdQ/5BpEjKBaZF9yKUumswtPWIyNBbfl6qquwO6PMJlkxmyqCslVdCkdL2ouYX6h7BBYOqyD5UzXx30mYr83FwRk6Exdiyp0YKsrpSSAxmyyP7eMqmCxmJ74XIL9RGGsEFgKtmH6wrkJ5pdA1BRA+CAKiq9zS0iMjR2znFqtCCrK6XkQIYsMupQSRVUjkVpiQeAB/VcL2xuIfvQcxkcZAyZwGSyDz0eAOh7HsCatKtgMhWNc6uSWuuHZb8MRWzZATVakPOSUnKgyhI5dagsAwDNI6bn/G3+hq9vvu0vrNFkOtceSqTxMNNlQiMEpm46cvbhbilQEWcLxgza1/unlUNXfaIiUt3cFJ9pHJYNMjS2rWXORCor8Ri8ZO2ecyxZ5EpdZl5ML0zn7HZCKTAN37WcPFjP/nj235X0xdHqBkZUepubzTI0lv/+EgCgs/KgSS8ArpYAkKT8AYGOxvWTBng9aJle2M65h1yt7zgEQMKgx9eH4nemHf9RrnVroA+ANKC7fvwdvfcU2nHFwfWSrdGCTOdetuNUDymcZkDXS7ZGCzKde99OYzykndcOTnWazvRxtkYLMp372k7wHtLe/epMpymspXAowqGIEiWsJYS1FNZSnOPEOU4UxH0CiLHwSmEto8xayuGVTuSMwlo6+VgyEYpyeKU/ClNeEjbOKKylk89xZkIxyW+FKS8JI2cU1tK5x1Jxo9YpdhBnFNbS9lxLk+eshSWY8Eq1B4NiVOWjskQOjlSTI20IjhTWMuK5li8N6dzjReCHmdlFfghLNrxSjYk0KEZVPipL5OBINTnShuBIYS0R8VxLutjmx0TUkF3hl7Bkwiu1tQ2KUZOPSutcN3HJkY0PjhTW0inWUhOHLR9+8wpQNrGtX8JyUv/hACbwHlPfp0E+Spzx6mMTrgdcU1Zs8kY4hbWMtmtvRhz+9A+rHsKrU/wSlkx4Jbu2rk+DfIQxOTK0wZHCWkbwWCpD22QAyzsCN4/960Oeyrbcc1bCkgmvZNfWPXPIXD5yyZGhDY70ai1fXL9owYNFCSbjtrSW/yjJrv74eXHcBJZrCWBKwRdH79Y9B3Nh2UULr9SvzSjGel4+Kq1tTI4U1jKCn+P6dtgIADXvALir45ItubrnLIRlJy28kl1bVYzSM4p8LOZbD8jYAdiTHMlsUapmpFrLlcD1090lhlHCm7V8+XfLxKEUcK4lkPiTV7Nc/HNWwtKlhVeyPegUIyMfldaey+CSI0McHCmsJRyQawkA5T2OtWafsxaWgBZeyfSgU4yqfFSWyMGRUJIjQxEc6RLW0kH3e5df6iZfMBzvZHxOLc/+LO4atfxsz4Yv27S5jl1bXUd95tTFbgkmrf97sVuiTftV2aJc5451TduXmtGi+rrEr2q7JpuP0ryuDn55CNCwZ+KEOeJYEnagUbWm8H0AwOL174pjSdxXCWEtnZRrGcf3PgtrKc5xENZSHEvCoYjrJVEQ1hLCHYo5C2spznHiHCcKwlrGbenh5pmDZzsMiXvemeAQfMj37HTkqIebpa+OX2m1VvyAzYSAXKWMDxudYwkr1hhugYnQwM3BS7Mt14ofsJkYkKuU8WGjcyxhxRrDLzARGriZZLlW/IDNxIA0oowPi0M/Dr7nWEKO8QM2vVhLE1cp4UOjspRLMYomtlLTlAxXZLbA9xxKgSmDSo6QXjtOuFalk5fGabPDNwOoTGAngMtlDWZYNX7AprW1NHGVEj6Uf1ry7c49XgQeyOxVJLdUjKKJrdQ0JcMVmS1wPYdUYMqgkhkSvh77xDvT9y39iJOXJtNmhs+0Zn/p1MBOoOb5wi0TZ9cCunXjCWxaW0szVynhQ+mnysxJRFTfu0JtqhpFo63UNCXLFZktsD03TmBy01FBpTatA80XEtU/2b2Yk5em01aHz/BRqfLcxAV25nQoJ6ofn9/AUdM8N4UHbDrdWpqxxxTmcdojq6uBI4+21S4NZKNotJWMpmS5IrOFFG4UIROYCqhkpvVw56lAwg/+AzDy0nTa6vCNfBQAH9iZ2wFImPXOGiM1jRuw6c1a+mKPk+cun4LlM9inelvYyh2apuS4orcthERgKqBSm9apTx9vAqAbf4o3nbY6/I8NfBQAH9jZFABykta6PzNfNw7Apjdr6Ys9ti9Y/Mg1Txr7VLqFrWQ0JccVvW0hJAJTAZXatA5JKSwJfkxbhZpGPgoAaHi5aNxINrDTlXrYat04AJterSW8JFcumww8Omr7kQn8B2C+D9VWMprSJ1cMqcBUQKU2rZtRYSIvTaetQk2LnWII7KSq7l52IGIcbHq1llbJlcpPI3os2t/HpFOjrWQ0pVeuGHKBqYBKbVqZPbcBwCkAjLw0nbY6fPOdUv3niXJg53Y5pHNH3Tg/yGasgk1v1tLEVUr4UPnJNWUVH/YlG0WjrWQ0JcsVmS3wPYdOYMqgUpuW6+V/fwBgCTh5aTptdfgsH5W/SvOAD+zcXQE0vHBPAUdNpWzPuAGbxg+UHw196pWZa2n73a2uz99ZNDIla/wFOnLjM3M/oV356el5p0n+iYguZF1jGr4/LjXjziK2D6IdA2av+u1qItow8rkl+XNqiSr7de7SKTnx3kvsFjZwPSvbUdpwQwns83HxnbNXrP3lAnZItD3nj2899R6KiejMbXPXzV/bMWXMBdNpa8PXWhMR7cpPb52/h1Z3mr9u3mdTh/6yjkaXzSp8zf2bGnbyu/LTW+f/NT89Pe8016u2B2LrOwGYDcRncqXy07l5lp2rfaipkuUHPL6jIXUJmVKbRuxXJrySmdaJL2qPo5hLubQKtVSHb7aUCewkOqaFdHqNx7QjHNO5x5JfdXEP0e9LA2721lj5K9G7Ivs30pRjKfxlxx5w5neVftfvB1ZVX8tCtHJFD+ogwKZDrOXm52ZsfDoNzuOKfk3n2be/yBpWFJl9bsMecEW5jztdnp0EB3JFv6bjSXRRXVKk9nzI94CwlsKhCIciSpRJNXkGwIjYmtOIOHwdRzhhAEQ0R/xKiWpkzSGieLy4EIUoyCIUJY4lUaIaX/8HsCwGfH8SGysAAAAASUVORK5CYII=',
    '1603.01595v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAf8AAADQCAAAAAAsFzc1AAAb8ElEQVR42u2deWAUVbrFT2eTJSFsAXxGJCA4yC5BRcAoiwOJxEFUQEBRlE1lNCqODAzouKCIgII+QN6gLAaHEdmEiCAQIBFQthAQhrBKgLAkBEjSSfp7f1R33Vvd1UmqU7cs4J5/qBSde77vfkm6O9wfx0GQuoEVJLdAzl/qxhURTZC7cENqAhE55PO//PkvJecvdUMqxM/9UUPuvqb6uMI9jwVVE7D+mh0XGz3RmL7pZ1lL+ceUP+s0cGju5+09V6Or4Plf+ALX1vwbX4yrH7bhSLPOhac3Nvqv6S+Sv/hHr1G3H/zkT+HTKjH/rCEjBxh4eE5y6qYGz1V37suIm1Cfu5+98KteXctfv4JupKvZqOukiso1vSK3hKq05kEi+ggziSgjKvC6/TxwZOTPRET0r5D2lejzOzxrrKu96E9EdL59dLbm/kN9y1rfXVPF3Pw8///45Ln1Ff9CzazILaG6+ERTAMEIAtDioYKA69bX9M8/VX4aPh1fmT4Td3xirKsqCAaA2q+ffE1zP7TM9d01VcxNf/7ZdQcjucJlLq3QLaE625b7oPXZgOvW/zoZHzNQuXKMrkyfjvbVA3xywyYD6y814qY//28e71ZnaZFyXZJzBiVHCnyuQYfSCgBg71j1846vP+zS3nJu31cKoPj0eeReFjf/0Hu5D7qwQphv6ZFLKPpuC1eTWmRu2kbnsTIWX3Q5wbNNncP5Nn375H08+6MWUXouW/PQ8nxVXUIzfj2vzfZa31OT202103fTn/+mLqGP560BAGxuV2/C6g+39XuGtNdISdxZ+Prfi7H4zcJ1gwbNB+B8dW3E0m6HwG7R7D5XszpvwarWN8+Z+WnDjcLmf3s77oOON3sKYb6fPrz2xWEfNnykVK1JLXLKl5FV321fxuLr0NxzedMcrk3fPnkfz/6oRezvEPUS/9ByfVX9O/gNtt+eLzi1Cu36npqUj5idPze9FwVZLxNtRD/3R/d1nUd0COu018uaXiByjXySiO4Yrjzwi4b7iZLalrJbM2/JI0qpeZFczR7dnN0gWfCrwKn43KsQj++/b8qlkmav0mGuJneRB+KIyNWpjGXb4Rv+Q65Nnz6ZD7c/avNxfbmHlu9Lh/Bodvax1cNjlmv2mxL6aqrQrM/GEdeXq8yfm+73f3J/oHP0iivKRxF7BwMxoRma64IX+tUCHCMWrWOfVjP/CtB112H2TnVs3xpA96BFcEQc7tQg26r3zqwQj+/iZpEIbrcYjbma3DqzaxfgGFLme+RS/dV9+1R9+P1Rmw/nH1q+L4D9ycnfnhx0oLfvfnNV8OtzCucr8+em+/5/RYP9QMzJFf2VD+8MAoLDrmiud51sCQAtHSu7qZ/W91HHkT1pyFNv/JoXthlA3YMAWln4apAvRPGteQFASKRXTQCAuxu1a9G993NlLNdoO3tBebGWb5vcmqqPdn9a6Ty0fF8AjV92X/jst6YKv5ur2j3rx03v+z/zjlujo6OHqu8AqiivJzTXvylXQWH72ee5ZndcUKcTt9Ax1KtSpUqVhW8AqG3h/PlCFN9R5zKQ/9MYr5qUjjb+o9rM7k+5/C/XEzvV60k6bXJrqj7a/amt89DyfXn57LemCr+bq9r5c9P7/v86qRUAen91bk2/9TTBZQAoLrpd+XjuUGDM7J1NsBlwORzKreaoE6u+ObFw/nwhim+9F6fXOz3zLwD4mgDMHXqo6ltvXfzy9aFxfpfrN3ZV0U3utwohPm1q+lR9tPvDmmcPLd+3rP3WVuGzuXOHetn5c9P5/qd9rQDAMcBZxrvbu6LSAGA7egJVioEsIH/a002As8APW9y32vzPWgAoWm7tLwP4Qtz66f457879CwC+JqXIfclArZf7Zvhfr/qMnGmefX1Ku7p3n6oPvz+82EPL9/W33356VH+gKePQ2vlz05n/oovKnw9jrgsAnMUAqLhUc1191jdZQOmkJx4G2v4XICC0agmAvaEF2bXdt6rMWbkHwPRbAeclCwZfiCJAW4jb97bXZy9aujYP4Gty1/3ZVQDOe8tY97Hp/5hPAPC94w7N6j59qj78/qjNF1/h7cv3zWMvMjTrFRdrq+DWh9oWiq/w3fpxC57odWNz36+OL4mPBF6devnM6l29tr7w8+mtjdclnTm4tcev3HW7tmNO7X/nrmnBQOtP8jZ1a4SQOz515i/u7Zgf8USQ+1bTjklZvy+8+/6Uv+7JWl/aVujwV02YuTLo51UptRqzQiJfdvvWWvjDxlVzJ+2LD1Nr8tR94OjhMyf+r1Wfsta+5/63/xV2ecuMm4dB0ya8+uR8mqv742l++6j031N7tPU8tFzfPS/MKjy7NuMh909ydb3to9JPbGnxgFrFWs361d01uT/yVObXrRJvt3/fV6xcOH85T0REJRk7i8iVy9+ikwdcZLm4QoiI6GrLNCIq3dZ8oqYm5y/nifKcrgN7CspdMvOr6f/J811d26fWR90fLyn2FfTV22/dHpnUmjg7f243xvmvb2crv8z8bPWK68IH8vyPIbXLOAYApSk9rw8f83SDnP/cPvnOu6sdWnf/SMf14SPnb1Q5RwqiY4KuHx85fyn5/C8Fyf9IQfI/UvLnv5Scv5Qp8x+1Te4Nblz+SyD/c3b0rEgBy16cfamg+MU7vCzUK9eck1WuvhxljW3+9OPZrf5aH8Dg9h2qHU97qKt9O7WW/8l5dVgHZAv4F5+cF3OIPgxby1twZq4BbxP90uyUFbaU0/8w5T0WmUZE9QGEjnfZt1M/83/iSaypcEVnhlfkFhERFR4tmSpk/lOrfEOUg/t4C87su8giIhr2mBW2NPIIEeXXjikh6jHzzTlH7NypxfzPTbcFi3lWaUAAquMCb8GZLeoUBiBuxVULbLHyvotAeNyRA0C9Ue8918jOnV4n/A/65z8ObMHD+n9bujYSAG4u2mCF7a1FTgDhOHsNdHqd8D9AKOCcEjtO/y9zLoYDQCQOWmGbml0fwJ6QlgB2T508Nc/OnVrO/4h5/idKH9Nh+GUPB5RNmquDeJmIaBcmWmNLRNuQREQtF7no22ZH7dvp9cP/3DNpwbEB5+DnYGgYADhQYJVt0dDu7wJYOMCBPuFJ9u30OuJ/HM0W3NIrXff1ZYRCr5QgwirbsVHfVQHQGgBiZ+dE2bXT64r/qdNhR4ruX9RUvh0KEWmR7dxT31cHkL5Jmck+23Z6nfA/zs4l6WFAbRzS/evIBmcA4LTZv9T0Z7ty98IgZATdmZiXHwY4Uce2nV4n/E/hjj1nAZxAG/0fmAlHACCrdntLbNO2Tg8Clt2ENp+HAdgf2dy2nVrP/xShyPz514hfHw0c3xfXRWvhuUrKzALo26RgK2wPDDo/csTwwV80wmPNARzdODXEvp16vyFIja1Rte0xIkpqXT2800u0pXfNWonp8x4Mb/T4Rf6afnjwvVmJE5xEdPi2iW9tIKIlt076/p+/jur4txL11vqOY798fRWtiY+I6jmPiIqe7NEg4s5Hppr/+/+nPtqy+Z6EbN6CN0uO23fh1UFOK2zpLmVnWxOVjJ+2K7nFFJd9O63M+Z9Tuc2UL+zivY1qA0DpgeI7w+hSJHcLv19uZs1Z6F27qV1r/1Znv7/SsZ3DctvM9Nod69u4U3n+C/L8h5Scv5Scv5Scv5Scv5Scv5Scv5Q8/38Ny1Uq5wubnf9nR+OFLUxzM8KDx1QHUt+Mj6oOAE8GAa7/ZBYHvRQl7vw/d+6e1eL68kSB6/nbRXfK+9j7/D87Gm/yL+LVhZ3xA0tpcescos/dLT5ARPk9xjtpfqK48//cuXtWi+ulPUSnHkwT3SnnY+/z/9zReHPFFn479BIRdRpBNHrW6k2pqZseOELkemwgEd0eRcLO/3Pn7lktKz4mItrdXXSnnI+9z/9zR+PNFVt4dusIAPcuuIqgYT27dO6cNbQRsH5JEoCvl0HY+X/u3D2r5eeTANAkS3SnnI+9z/8LOxqvLpx7MhwAoi7vwosAkLV1IID/jWwLILYjRB3G58/dsyZv+XiaC1gRL7pT5mOYdND9qfBoCY3Ad8p5gJYY/v27yb2HuLTXtObhxetfGOuk5IRqtw8c+BURFSV9sW3yAweJ3XLNit+w/N7NtPJPeH/G27U2UGkREVGbkLOmB4B5Fi4Mvp+I6EPMVe4nXCAiV522R8e+98ZvQg6eF/WMzaVsPE9E9AumcrVQQQy6HFiWkCe6U+bD1xHw87/g/A/laLwAbUMSUZsWRESjMIWIiBb8jYgoF50ml1LWLT8Kww58zt0rTR67D2FdnOI7VX2Mnv/Xnf976USl0dXcjMGfo0qJSkKna66vRo8jItqNH9n8l9TaQbQSB9X550aOJqLS2jOJ2rdhDGir7gVCxq8svMyRTVTUER8QERXdeoyI6DiCjxHRkMYCjF2/9eydQ3swRtmPNzRNHhr8WhU8lC28U9XHu46AXv+tyJw376uYq57/w7Ts8//s0/qeb39kWarX+f/Nm7d6nf93H403X8rCiR89fyLr/QTUBYAVjoYAUB0NGwJok7UVIs7//9Cr1PvcvVLL3qc+npzR7YdHXII7ZT72P//vPhpvvjwLJ83Y+NMrZ5TzsfNiAAA1wmoDQFXzD+IDymF8r3P37lqen1QXTdZO3LZGcKfMx/bn/z1H402fAlv4ttuAY43aACjZ0F3pstUV5cu2LgSd/++lOXfvriV6XxcAjgnrDsYL7fQS87H7+X/1aLzZUhdO7ZMH5G0aHwIg87L7x07iYSeAM4iFoPP/mnP3nlqqOvIBAA1biu2U87H5+X92NN5ksYVXLj8BTIsZDACnPJFoI6qnAJTyXFOIOv/PnbtXawntNw0Afr9wv9hOeR97n/9nR+NNFls4I37Tr691PUNERMsx1vOGKXbJjhFPXhJ3/p87d89qKXx62Ibds4ccE90p73Ojnv9XdX5NbruOiqdzQYLnnxpzN57r0Boiz//rnrs/tP1q6w4O8Z1yPvL8vxTk+R8pOX8pOX8pOX8pOX8pOX8pOX8pOX+pGyf/wSUrgH34Dz68AOJTEYANn30DQSCGf1sNbOKuQKgld88gYWMx/6HyElYEMRDRlSYJRGJADP+2GtjEXYHY/Ae2rQYJG4v5D8ZLWBHEQETvKLsvAsTwa6uFTd4xef56lty2GiRsLOY/GC8BC4IYgPQY5cSPCBDDr60GNvFUINSS21aDhI3F/EfZ4QUwOxWh8D8DlAsRIIZfWx42USsQasndM0rYWMt/qLyEMBBD87/if3Cc7kkgIiEghj9bDWyiViC4U/WeQcLGcv6DCy8Qnorw61zy7L4oEEPHlodNuArE5j9ot7XihI31/IfCS5AoEIPbFecrperuCwIx9Gw52ISvQKSl17YaIGysz38oI7wA5qYizHhBfXWzd/jyuiOG//BIWpB4Ww9sMm9rV64C0fkP/D0DhM0fkf/gN7wApqYiZBbVyc3NLSnOzRcGYuj2w2ATvgLx+Q/qPSOEjbX8RznhBTA1FSHn5DgAmfXGNU66JAbE0O+HwSZcBUIttfcMETYhZfAfE5c+gzL4j4Hw5j9GuvmP8M4q//EsgKKURJWXCDob7T+8AJUAMXwXjosDgCUtZwBVHfk1ALNBDH/9JL7jDAPOILYpq0CopeZe2tbpDmBZfzvyH9rwAohPRUDplUsAhIAYfm21sIlSgVBL/p5RwsZa/oPjJSwIYiAacW94ZLcPxIAY/m152MRTgVBLblsNEjaW8x/lhBdAVCqCKBBD11YUbOLfMtBtlfwH5PkPKTl/KTl/KTl/KTl/KTl/KTl/KTl/Kcl/QKIgsDXvwSWdGAw9sZr/0MIZMDMaxLOwXj4GAPNBDLbnGrjEbSMs6YR1qkZ9cEknmtAT2I7/0MAZROQqNicahGMhdPIxhIAYrAcNXOK2EZd0onbKoj64pBM+9MR+/IcWziCipZPNiQZhC+vlYwgBMVRp4RK3jbCkE9Ypi/rgkk64SxvyHxo4AwAKC2FKNAhbWC8fAxAAYqjSwCUeG2FJJ6xTFvXBJZ1wlzbkP1CBXAuYn48hBMRQxcMlqo2wpBNVXNQHl3TCXQY+/01dQh/PU85Kbm5Xb8LqD7f1e4a010hJ3Fn4+t+LsfjNwnWDBs0H4Hx1bcTSbofAbtHsPlezOm/BqtY3z5n5acONZVWSml0fwJ6Qyh7RUtepGkxKhwcA4JPRwv4byiExrzzw2/IF72pszGrHv3IuhgNAJA6iCQDX6EkOgL8M+PWf0PwP7fP/1//0zbUIFIzkFvZJAhECYnjE4BIvG0FJJ0qnXlEfStKJ92VAz//J/YHO0SuUE/+I2DsYiAnN0FwXvNCvFuAYsWgd+7Sa+VeArrsOqzfyxvatAXQPWgRHxOFODbL7lffFWDS0+7tmfGso67ydeRpw7kQJUDx/iMC3384mrwWlPnzax8asdvRViDAAcCj/47/zzZFqOewysPf/QvkPj6ZlAMDhS0cB4Ja3AJgXDaLmY3xWPD8hra4GBTFfDC7xshGWdKJ8X2qiPtxJJ16XAc0/845bAQxNTe5vkP+YMy/+QW/+A8DCW6DhPxQlxALAj78/DQA1PeCCKcEAnnWS+m4semUc2iCzqE4uUFKcGxwhYBDPf1gXTda+PXFNI62NWe34kTbqw5104nVpv/wPVU2bAsDJkM6AydEgvkkgIkAMVQwuqa6xEZZ04pYm6sOTdKK9tBv/gXJyLQyACxVYJ/XjeZHI2zQlhEdBzBeDSzQ2ZrXjV46EnSzqQ0060VzaLv9DC2dAP9ci0Bd9ysL6SSDmgxiqvOASt42wpBPWKR/1oSadaC5tx39o4Azt+7/KRYOwhfWTQASAGOw3sjxc4rERlnTCbSEX9cGSTvhL++d/JP93HKxIAhEroSkfZf1DoBr1wSWdcJf25z92n+oFqT9Okv+BPP8jJecvJecvJecvJecvJecvJecvJfkPSOjDAH2BQDkM3CD5H2XwH5WNBvHlPwAGfWgiOcyVBrnwVGGUwzAohrZo4BO/m3sN8B+VigbR4z946EMTyWGyGHLBtWeQwzAohrZw8InP5l5T/EflokH0+A+OxtBGcpgshlxw7RnkMAyKoS0cfOKzuYH+/8/Irtt7UfKfIYL/gD/+o3LRIGzhld/tq4XwuKUHWoDRGOuX/ALg62Ixr6GGAcCXQxvx7bGbIuRGW2ZNqfbzVcANn/hs7jXFf5gVDcKDFyqNwUdymC5d5MIgh2FMHNpSyWQT+/AfCAWcU2IrfRyABy88NAb9FHPi7+//7aCYV7S6yIVRDsOQOLSFh08CeiFpH/6jktEgPvwHozH4SA4xUpELvgoDHIZBcWiLJtnE8PO/nfiPeyYtODbgHEzjPxiNcQnpTwQhpsewQkFvanWRCyMchkFxaIsKn5j3/v+P4j9MiwZRwAtGY3CRHGLmoYtcGOEwDIqhLZVMNrEV/wHU6bA5Jd4k/oODPlgkh6D56yIXRjgMo1LRFhU+iTdt/n8I/2FiNIgbvOCgDxbJIWYausiFIQ7DsNxoS2WTTYLK4D+cS1EW/wF48x9Pu/mPLVD5DwAoWo6K5FrsOQtTokHStk4PApbdFDdjxowZM2q0nJEEJB52AjiDWDHD0EUuDHEYRt/k9MkD8jaND6nqULKFAk02sQ3/UfloEB/+g6MxtJEcpotDLlh7xjgMgz/iPGiLF3ziD665BviPykWD6PEfPPTBR3KYLw9yoWnPEIdhUAxt4eATn829xvgPYdEgwiM5dJELQxwGAkdbKgWfSP4Dkv+Qgjz/IyXnLyXnLyXnLyXnLyXnLyXnLyX5D6nrVVbzH2WkYpjFfzDUg1tRjcqAUBiDw0wGt+9Q7XjaQ10FIjSsP87M4DZazH94p2KYz38w1INbkUVlCIUxeMykPoDQ8S6B+R9cf8zM4DZazH/4pGKYz38w1INbkUVlCIUxeMykx8w354iAP1inXH/MzOA2Wsx/aOAMQAT/wVAPbkV3VMazV6tBJIzBYyb1Rol5smGdcv0xM4PbaDH/UUYqhln8B0M92IpcVAZEwhhCMZOK7ZjBbbSY/ygrFcMc/oNHPdQVuagMCIQxtJjJ7qmTp+YJ/QLgdoyZGdxGy/kPbSqG6fyHBvXwrOgVlSEKxtB4t1zkom+bHRWX/8HvGG9maBt15/9eOlFpdDX3In+OKiUqCZ2uub4aPY6IaDd+ZPNfUmsH0UocVOefGzmaiEprzyRq34a9gGnVvUA//8X1W8/eOZXcleMIPkZEQxoXcCvuwRil2jdEjGOZI5uoqCM+0HjvJiK661Gh81d3TGNmZBv/AP7DnYohiP/Qoh7uFbVRGRAFY2i8WwNA7OycKJHPAJ4d05gZ2Ubr+Q9PKoYg/sMb9ajTYXNKvDYqA6JgDN473Xk/gAjse0Dsi8A6HTanxHubVXwbLec/1FQMQfwHQz24FTVRGRAGY/CYSWJefhjgRB1hk+f6Y2ZGt9Fq/sMDZ0AY/6GiHtyKjoQjgCcqA8JgDB4zafN5GID9kc2FzZ/rj5kZ3UaL+Q//qRim8R8q6sGvyEdlQBiMwWMmjzUHcHTj1BCIyv/g+mNmhrfRWv7DJxXDfP6DoR78ilxUhkAYg8NMSsZP25XcYopLYP4H648zM7iN1yP/wVAPbkUWlQGRMAaHmWSm1+4oDgDx6o8zM7SNkv+A5D+kIM//SMn5S8n5S8n5S8n5S8n5S8n5S0n+Qwo2yBkpzzyQyBEb5X/4R0MqgjRwgRgq6iEQwfCRJ2eEUSgiiROvPBPFPKDIEfvkf3ihIcbIEC4Qg6EeohAMHSk5I1x7QokTrzwTd8hJIJEj9sn/8EZDjJEhjMHgUA9RCIaOlJwRrj2hxIlXnok75CSQyBH75H94oyHGyBDGYHCohygEw1funBGuPaHEiTbPxBNyEkjkiH3yP/yjIRVAGhiDIRT18Cc1Z0SV4DJ40EQ1DyRyxD75H2WgIeUjDYzB0KAe4hEMRZ6cESahxIk2z0Q1DyhyxEb5H1o0xBgZojIYPOohEMHQSM0ZYe2JJU540IQ3Nx45Yqf8D09uBwJIBlEDMQoRBgAOFAALBzjQJzxJ9Hc/yxnhnhG4MswXl2eiNTccOWKr/A83GhIIGaIyGDzqYQmCweeMMIklTjjQRGtuOHLEVvkfbjQkIDLEw2BwqIdFCAaXM8L9LBRKnDDQpIHW3HDkiG3yPzg0JDAyxM1gBDPUQzyCobzUYzkj7KZY4oSBJlpz45EjIWXwHxOXPoMy+I+B8OY/Rrr5j/DOKv/xLICilERULLfDASzr78s5BJ2NLg9pSP14XiTyNk0JQcJOD+rRZoBoBAMAEBcHAEtazuBvOlgZIpT4jjMMOIPYphpz45Ejtsn/8I+GVAhpYAwGQz1EIhjecueMsPaEEideeSaqeQCRI7bJ//BGQ4yRIRyDoaIewhAMX7lzRvj2RBIn2jwTNeQkgMgRO/EfqAwZwjEYDPWwAMEo4x86BRInfvJMjEeO2Jf/kGQIJP8hBXn+S0rOX0rOX0rOX8pkEdEEuQs3pCZU8v2/lPz5LyXnL3Xtzv//Aftf7XoB46DOAAAAAElFTkSuQmCC',
    '1603.01595v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAlcAAACeCAAAAAA6KHANAAAgWklEQVR42u2de1xU1drHfwOIyEUQxEsqYRz1eIMUTU0TVEyFwiOUmlDi5XjpNTMtM7OjWJ1jF28dL3npHHxTM/PVstTU4wVRMS95QdQTAVre8QIiCsPMPO8fM3v22jN7mL1nhtRhPZ+PfNjr8nvWs2Y5e82e9eXRELhxc7l58CngxtcVt0fFiGgmnwVuLrSZROQFAHyPpdBmzXr4FR+8r1kANGT8x02JuX6m/si5/6N8aYjvr7hVj3lZFty/Kf5ez49PUBVWegEA4BvmZb9tSc6Nur1NP+0JIqSRptoHz77OjT3lWhzZca3FsJCNSa5ZVwVLDp6ol+KHiouHf5s/SblQQdr4l2rYuipal7Wv8WjvspyS15Psve9fWfO/A3qbflYp2Gi0nzb3dMzMhipm25G5F17nsguZd861AnB94rJAtsH0a7Mb5c5s9k2Sw58HQRL7CQOJiOh+2uuk3L7FSMVtDQvp0TSLmaIcvEREtNNnpN5u32eTxZ82FXMwlIjoZnTTKypmW+Hc23idL0f+SEVTxnSGxOV30Xoi0sdHOzZLMv/PfE1vYj6fXlexPhOPfqb8P/oZN3nH8jHuT+OS/rXBbttazM8qBD0BIPiti2+qmG01c2/9OjeefQEBry0ZJq39srcHAI+Zrn4uWqlFSJCaDwHRyjdjm9ztjtgce10r+AT2qZhtNXMvY70uoPbjllusAuNer3O4i9dV1g6gn7RId/067hUaTFfaI7l6AJVXb6L4LqC/cQWArugadAU3AVSc10na6YquQVd4HwByprvburqEKDZWgPL2XDNW/bY736Be8A5aMnNrLWycbRRnZ2ovmK9Aedn3IZ1ru/b9r6jbVaa85YZtAKB5TRKO4EFuaArXFYCBwGc9wxp9jl2PhT19aGvHhunvz9jcd6oWAC0fdK+gxwFsiWy8YvE/wzLPdg59DdjfocHMbR8fGTHSsHLpgS5LmHbGmsNDRhC+fqd8V2rql260rPLWJ45iYgX2dt5tWJuuB7RTdgZs6pOnWvEbz7dhnltrYeNsY+6qwDofRsN0he2Jx8vfereSnWv7drQceF6m/A3E9/koW4sYNhzBg9zQ7O7bc9AsZXBXfG+80qf4FpEhciMRUeuGJ4nudnzOQLS4SQnR9qDbZGiZtP9Ko3VEMclERE/3ziDKw+sniP7te4tpJ9TsIqJWY91k356HHt+sXzIhLsNAxMS61/MY0SbfnUQrw84STX5ST0QJyWT+aVMxD0lXrlzYNrb5ZiIyz62McEwy0bkYIjJ0J9Pcf9fiFpFh/DCSzHVVnxE6rlgyxTfHdDlfum+ndUEAGv6TSPTKeLAamoUf2XWVTPoLz5rWFVV07adfuoWIiKKHGz99bKDiwIlEpA9eTBQdZWxnnLF+oXoiXa2/EFE2jrHtTDUL3WtdDSwsPPFqpwNEJMZqaNOPiHZGnSXaUO8o0Q/4Rfm6aj1//vwVWRXGS+PcygknJBNlBh4nohUm1XtNZxARncR/JHNd1brquWfHkga21hXd/mpsC+BTMntlPVgOzdKPjUd6HmETCdDnRgLe33RIeSqeqeuNLck/l3jvB1D/FwDtJT3beACe3m0B+KAMbDtjTZl77az8w4HFKX2OtYEY68Uz/QHEnQCQnKQpPJWNEhU7dskzw/aAvDAAPBXeoW3c86NNVycutgOAdpof+iid69BY9LUcWrEOQK1AAEFDh1Lm4PRRpYLXbNaDxdAU7686tAXOHQaApgvXRbE1AQF5uIAGPj4+PmveBhBs8UlZ/KmXtBPK3M+Sy5cBYqy/opFQY1jebXVId8eVg2FDGAB8Mv/muzjuFdPHgv8aJ9jD+6yque4SIL1+LjQ0NDQZ+BoANLHLSo+bvUo8WAzN7vN2wR4DcKgNAJTveXfo8SbMZ5XSVmiNkE7mT7lVjJptJ7UvRrnPugrCKTZWPX4XaqYuPx6B/YBB49g3MxrYEAaAvDrp6bdXvTUqBgAQgbsAUFnxJ3UPGSw/MVQYl+WGIQCAfrgfKXiVetBU+fpW9f0Dfd4aAM2cMTt6sNb4ORoAMhGPqMd2AkDFZjujlm/nUwkUwK3WFeFeWyHWJu2OA0DFJpQuGB4BXAd2HHBCPspaGACQuw6oNyn5tPGqY2g2ABxBf6diaRweHh7eCDiTDwAoRZTZq4wHm+tAZl2VGJcP7k++VA/QTa/7uMf/5k4yAEDmVeDezKRB8FnxwykAC5sB2jumB6llAKCtBECVlQAqoZe0M9boATz5q5uc+Sox7pzCPG8V4Mu7QqyaFdlbAXzeFLXq6ADk1Lp/JRiVlQBMP+0JCmacWx9rYeNsL7kHQNvVeOW3bH0BoJ8z+DnJXFflS8teVqCCvdQPKwaApSOamL2yHiyHZvc5w5HEMPj+JSXlxS5B6E30bivfJ4lW1vHvNoco+qV3Vq7t/V4FEdHubtNXvbWFfowPCO2fQXQ4MTg44fqB54PqJR7K6OUfNuj22x0CnpxgbmeuCX/xNuU/Pit976P/efBkcuuAoLiZRLTUd3rhFHOsRNkd310/ewMRbWg2Z+v7P7/abVp2YnBI4rHDicEhicdsKp5M/rN/3V5vCpfC3FoLG2d7U/93126ZttB0RbSj19+XJc7USufalq8jSS39g54Z8pPpQ/+wvo0C2gycL9a33zn6gx93T0oqZsMRPMgNTepH1bm+Tu0yfittJWzJLt1tqWjTINOuMic8GO50ru/aDkoMYmO9WtzSAwD05yrbeNOdQGfP2lkJA7hTx+sXbQsfptXl4pZeLjrXdyyaco7qoztYeJXxYPX6CmtKzboC+HlR8POirj0vWqkDN25wLee1I+HcD385y6eMm8L3LIXvj3p4Qufhwe+D/D5o34+X8uaeUNWcG78PcuMGl/M4MRo+D2q+V3nIFR+4rxjTuordyxcMOO/MeWe+b3enfbvECgpvdAnnC+2PN4PHI+HJ0XV1aM3WTTV+XRlWXPS5NylUvF71+33DX/9kWWMmPg3/d6bS47VQR+UBYO+S9QBuL79zv3JCKwtRlfKKPNEXp/09p/oBeDm6s+9v2c/2lsQJFVyqMruFTVTjTDpThpdmEx1redl8/doposu9siU1DPFZ2vc9LX2ZWOXJ5irkiYjKIhKIqGhCEdHH3julonLyynzZ9qSNT9HT15FFRNQQQK33DJI4bftxeF2V8nX1bWAFEY15Qbj+fh4R0ck4SU35eZ3p5LjhhRQi+lOo0nVlKU9E9EFEAhHN91lPVISnJaKy8sp82fY0u9YdIuo+joj6Ln5nRaFFnLb98Aedjtva7t4AYkbe8zVe/3QPACIKJDW1Hxea795wDMBXlY7KAzjUvD4ANCIAfrglEVUrr8jT8sgAAF2XzfVFg1eFOjFOp56LCiyiiCKi4hL/CAn9zkAAaFwhPKZpMm+BAfg+3roGAPB54JMAOnVzVB4o/z/jn/cYWvoicADPSURVyivyVHzRHwBC755g25vjVLOuFncKi8rI6Nas63YMbfqnRSKLaEYRkf/izMy/76zx66rotj8ABEKAUdKavxH7382rP7SuAQDa0/z3d/8x7ReH5YHPJmrMf+tBO7fTDFZUrbwiT3U8ybhIzgE4Of+T+SWSOFXt2ws9FxBd8tpCRN33s7SjgCKeDcgkojk1fn/1CyYREZ3ALKHkwtPwfkZrVWPcXxWj+yd6KmjyH4X7K2v5n78g6pJARESHpnYee1ciKi+vyFcVnqLaEhG9irlE7dYaaGPL82ycqv6eTPiz3wINgr4DDJ274/7/DKkHaMat3QVNQH73RleGYESHngCG1Pj3q3J4G78cMf8hBG3Emx5Zz12VqQFwB4cGe6B53zHlDspXfplmruwyZ/WFl26womrllXmafeYqoD0OHbDmJQ0G+U9m41S3v0rJ/B27kr6pwP5nWNoRJhTx6qFe4CcbAAQY+TwdBAIv55V5n5zus2OgwaoGAPwQFgYgquCgg/KL/od5sTQtV+8YoGdE1cor85T46V9/L/hHAuoDkQDQaWMRE6e6dTWwzlfYM0+/DVvjLVjEYAA4h0C+ZweAIOP/73LzfPx1Tn1E7Jx1+EerGgCo6x0MAHWQ65j8mYqQ4uJiXWVxqfE6pPPR7YyoWnmFniYvytzzxjVE4dA+4xLMZeJU97zdf9DqCbX9XlwzgHwsWEQNADyBMr6mACCw0TUAuIqnTDe63GcAaGbu+mWARY1xptuXGU9H1ndMvujiDABnGsx4YkIP3SFvIBh5jKhaeUWeJgOPPw5cCI9CYkmpN6BFCBNnvLrvcVIHfJCM1P7rBgDoGJqdAgmL2OzPPwHA/Rq/rjQJxwGgIDgauOXrgzqa0roAENaOrREt8QOtN3ANnRyTj4kBgA3tFuHOUY/rTYHfEcWKqpRX5AlZ8zICUbJvrheiXvIGcDawNROnyu9xKhu0MZC+WXs9EdHG0Hwi3fODiaj9CCIi2lc7h8gwCWtr+vP2XJ98IkO3D4iKfKOJaHQ6EdHFARVMDRHRHJwnIroWtJnI0GW00uftlvJERDr/Z4goIZOILnjH6FhRWXllvmx7muqRQzSrg5bo84NEVFjrX5I4VX6PM/F9Ipr2lvFCYBFFFDGr5/xvpnyN0Bdq+LqidTG5t6akaonK2qUSUfnwMXtPLk+7wNawxOfhThuOjht2R+m6spQnonFd/QP7fERFr3x6YH+XhCskEZWTV+jLpqfT8ft+frP3NSLSvbfgxLq2cw3SOG36kT1/VezlD9xBXdiiHS/daGM4U79+nZp+/ur61rJuHZhTmHlH7kV21sjVGOc180bnSBVnomRFAAAnTlKHSI2FqIy8Ul82Pd38sbhDN2PFmUPB3RpaxlkVi8PP9fFzfTyPCTdwHocbX1fcuPF1xY2vK241cl3F8GmAcuLyYVd88L5iOD/InzPw5wzc8AjnSwXnUl1qBvJ8hOU5l/pwcamlC3+70v71hgCy3okP9QOAYR5ioWPyIhkqcqmMPAOOOs2lMoirQNUybZQE4gou1VDJuVQJr1k0NJ9KXgjMJqKlpnmOZQsd41LNZCjDpYryDDjqNJdqRlxFqpZFbeUCqQ4uddMnnEuV8JrjC4moNLi5jmjism37srL2xRayhY5xqWYylOFSRXkGHHWWSxURV5GqZdrIBlIdXGp5OTiXyvKaP3ybWw/+MZvOtYXHGABYNSqcLXSMFjWToQyXKsoz4KizgYiIq0jVMm2UBOJiLlWaUtWcXVNI7ImawaWiWYUWgD+uAxMAoOBgiqTQQVpUMIZLFeVdGIgM4sq2URKI1fvV4n9fr/cGll1skt5v6H6fSROwfdHLoW8Fzqq15c1z/wi4NX9TDPKnRUQWtpFT2zot59VGtx/fHP2BN2DuCcz16nv3w8U34KZcqumcd9rHb2xc8d/Va4EsnTeAU17tgAgAholfaiSFDskDJ3frvEYGslwqI89UOxkI7Wn++3L/kpEtbbRRFIgruNSv3jf/yqRUFXuKiT1rBpdqtMOYbPpt9TSyLlTPpTJkqMClsvJMtXNcqhRxNe6vrEZjGYj9ffuAWKLK+mOI9JOkWThNmVG79iQiKpRfV2JKVaanmNjTndbVKUw1Bvi2Odvpy2/64FlT2tHy9nH3TclnmpmP7IqFdteVtfxJIqKOScat9X/7P18klWerVa0rS0+/wfMCEaU9cV9cV5ZtrAKxv29Pefn3ZruSvvmstpRL7WPmUmeKN9AFpwEg/855AGiSDialKtNTmtgTbsuljt1cf9zYHQOzPQBgeui3ptQ132vChD5ioXp5Exm6vCjUyKU2GXDIk5Vnq50KREBcMw72ttnGXiBOcqkJaWlpaWkxHdPS0tLSXpSkVGV6ShN7wu25VAD44vJWP1N5RnOhC1OoXp4hQwETl8rIW1Q7EYgc4mrRxm4gTnKpLVoAwEWvHlKJO6Wt2J7SxJ5wey41Hvjh5BoPnPZoA0C3N87UgylULy+SoVqRS2XkRXDU2UDkEFdpG/uByD1nSM35IBGp3wtcKqCGSzWnVGV6ShN7wm241EIR5ywH6miMhHtYOyD74EIP4LvaAHDmriknHluoXh5RS01kaPnRU9dh4lJFeXO104EgMV8LC8SVbaMoEFdwqey+/bErRGUdkgxsz03Ny4ho0NEaw6WefWLM2LFjUsN1RETbMJGISFroAJcqkqEMlyrKM+Cos1yqBHE1UbUMaisbSHVwqey6YlKqmrNrCok9awyX2tG00SYios2YTkQWhQ5wqSIZynKpZnkGHHWaSzUjrgxVK6K2soFUB5e67tcZkE2pauppndjT7blUdmewOqGhY2ftLOVFMpThUkV5Bhx1lkuVQ1xtU7LVxaWevDygZqRU1fDzotWcb0JiUVHgKVW5VR+Pw1OqclOdLxU8pSq/D/6h90HwlKrcOJfKjXOp4FwqOJfKuVS+v+JcKjc8dPzgH2S3c28ExfJXCYrzmRo8+LpSYJdWrxoY6y75UtWCo7a5VEDIZypJnWourA4uVeLInAIWznGpKslTprlhoWP9BOs9xG3ypcqBow5xqURCPlNJ6lRzYXVwqYwjJiRnuVSV5CnT/NpYx/oJ1m+Iu+RLlQVHHeJSiYR8piyiKhZWB5fKOGJCcpZLVUmeMs03uS/gqjJfqlpw1DaXCnM+UxZRFQurg0tlHDEhufC5qA301AZ5mjPdjoD2SK5ezg2jZzgvHE79bXe+wVrCpgYeqnypzmcxhVXmVBZRFQurgUuVOHLkebtVvlQL29qxYfr7Mzb3naoFxGSqmLsqsM6H0Vatv36nfFdq6pdY9nTYn+cBqU3bZkgEaPmgewU9DliPi9Hb+dFPQ0YQAO2UnQGb+uRZjsGmBh6qfKlgMo46mMVU7G/OnMogqkyhc4HIp15lHCkxe1yq1YlQWfTUgjxlmrcy7q9Kmo4gIn37a1KBxU1KiLYH3bbsJ+r1e2odUR52EdHKsLNEk5/USyUsNR7WfKny4KhjXCqTOVVEVNlCl3OpFiys3f2VvXypMubbPxLw+9sPG5lkqtdOnAA0abaXb92xG0qB/PENJAIl05PrAnEeay2bM3qFLwLNa50GEFRaBvQ+kS+RsK2BhytfqiTjqEPy5v5s5lRz6lRJoVOB2Ei9Kjhy9PmVhEu1RZ5aoKf/EMhTW80xKn3NOKxhJrU3tiT/XOK9H0D9Xyz7MSRrGw/A07sMQHKSpvBUNkokEhA18LBwqbL5UtWCo1VwqZLMqQKiKi10NZcKKQvryLoaWOerqXvmrdv2l62zACR0AoD/XBoOAEGwgZ76ZM7btnjhyxketpqj8aAlY8sr60oFLqCBD4A1TSzdmPVgdAE9AMOKjPhe7FtoQEAeGA08aC714/qI2Dl71o9Py+VLxSFtTyM4Gusglyr0b1ARUgzoKos9AwREdf/2cMtCV3Cplo9wQzrv3x7v6LqScKm2yFML9NRMnso2/2IUgPG9D+QPsRBojZBOsoCrLMk6dfnxCOwHDBqNKMFo4KHOl6oWHLXNpTL5TEVE1Y9Ncup6LlXKwjr2PQ6bLxV20FNTMtXcX99EvUmHTlsfxfCpBAoAIPbPi5sMtxCIemznSAAV2xMtesnplS4YHwFcB3b49xAlbGvgIcuXKmYcdUhe7O8l5jMtN6dOZZKcOhuIXOrVciZHq6Pn+uIafNcRPRvM7WmjT+ZV4N7MpEHwW7a+ANDPGfwcltwDoO1q3fjJXwECAM249R0tBXxW/HAKwMJmVt3MetpKAFSpB2rV0QHIqXX/SjAjUYVGddvkMwUAbZzsiRvNegC1hiwAgEu3ejI1AFCBCgB4oTWA85nzvRyTt+ivL7sDoG787qbAb7kxzzCFzgaCcX7bAdo+uoU4egtHppDUvV95DQ3VwCNFb2sfGPNZhO/K52ZogEH+o/uGbOn4LtDq720Ds2KtH2Dhb73TNf0AAMMXJFsJxG8e16vV6dgOVt1MegfnHNcMnH5ulXdG7vKgjDc+ijyWeP3tmFdYCdsa1W1tMkYuafxhxFTA94nWABaNHTus3k8HP/dmaqAdUZQTEN8idhJGpx+OPff+nDRH5SX9x5/AqbhnpyJjypFuNKXvSk+20NlAGux4VRu+MmIeO3rGkViokpuw4FIhIU9l0VML8pRtXpkTbvwDAreWzJATuHS3pcaqnzzJqj9X2cab7gRaSkg0HuJ8qTLgqMYxLhWyqVMdP9enhEtV7EgFl8qQp0rQU7Y5AKAkvyM+HhxuT8Cqny17QPirhp8XdS2Pw5CnStDTKMvd3UdzijXl4XYFohTuCjn+CnfjcRxDT/v1zU6f5DJ2leOvcDsu1UH09PqldrVcxq4+MPyV3wfxR/7dD87j8HXFeRxunEsF51LBuVR+H+T3QX4f5AbOpT4g06660TRVU+006QNW4lzqH2zlPSf0iAlOcCZfqkUjNeCoIi5VLFSbjlUll8p4l7KwqBlcqqqh2mm8qJ2hMLXAqXyp0kZW4KizXCqTRNVOFlMnuVTGu5SFrSlcqqqh2mn8yhCn86VKG32gYl0p4lLFQntZTJ3kUhnvUha2pnCpqjTsNC7zh7P5UiWNVIGjirhUsVBtOlaVXCrjXcLC1kAulWnF5Hq141B/4xIA7dWbcEm+VEkjVeCoMi5VNJXpWNVyqVCHqLoxl8q02hLZeMXif4Zl2nd4ICZ0FLC8ZeOlwNHUw7tSU8ugmktF1pWGMGYUlTRSBY4q5FLNhaJPVAuXynhXgqi6HZcqm7xVyPWqwGHnfkRU4fU+EVHyQGfzpbKN5MBRp7lUKexaRRZTZ7lU1pEkXWvN41LFVpqA/O6NrgxR4jAIALxrO8OlAgAqRsV9KGmkEhxVyKVKYFejz+rhUllH9hFVd+JSq0jeasr1CgUOPVyaL5VppBIcVcilSmBXFelYVXOpEkd2EVV34lJt+A4IyIMp16sih3CaS40H8MXlrbUljc6oBEeVcakS2NXkE9XBpVpStXYQVXfiUqtI3mrK9arYIelclC9VbFSkEhxVxKVKYFdV6VhVcqmMI0WIqr18qQpTotrOiMpyqTmR1lwqAFRstuwlp1e6YLiJSz0AKZcqr8E6EkyBw9oE4FoFXJMvVWwUs2jRokWL6rZbNNnZfKmsEpMlVV06VpX5UhlH0nStNZJLFVtpjcCmAodRtwBsrHsfMPeCWi71XOrN8ePGvrwyHBaIqhpwVAmXyhQyPuF6LpVxZIGo1kQuVWi1/bPzVwcMHQ77DoGJe2Z3PtU+4LMj6wvSDxj69nsT6rnUlILlABDpyTZSC44q4lLFQsYnqoFLZbxLWNgaw6XaTN4KpQ5vnm9R91RAqJ/GNflSq8wzqnGaS7WXJdVlXCrjqGpE1T25VFXJWxU75OdFazqXqip5q2KH3MC5VE6vgnOpqC4utRroVX4f5Fwq53E4j8ONc6ngXCo4l8q5VH4f5PdBbnBXLrXUeKjcN8yLTw4eKhrVQJ6P8LoqWpe1r/Fo77KckteT+JuZw+AoA3ZCYEidzJea9U58qB8ADPNgEVSzJ5GKdSGXKiKozOiVBGJ97jkHLxER7fQZqbeBMS4kh82Zvg97vlQG3TSDnSxD6mS+1KWmVyxWgqCaPYlUrCu5VBFBZUYvF4h9LjUPKURENAxfy/dThZq6sO/Dni9VRDdFsJNlSJ3Mlzpx2bZ9WVn7YgtZBFX0JFKxLuRSGQSVGb1cIMq51ObYO1i2YpMTt45NbnQbtA2OimAnIDKkTuZL9RgDAKtGhbMIquhJpGJdyKUyCCozegWBVLGFuiQcCBRIThO5KaKmAibKwpwiAWqu1RVdg67wPiCLqT6yVgU4KgE7HUxjaiU/AQAKDqZIEFTRk0jFupBLVZklVcm6ylufOEqCfZrITQE1FTFRFuY0E6Bi7f4ODWZu+/jwkBEk9nUHsw2OSsFOkSF1Ll9qBADDxDkaFkFlPIlUrCu5VAZBZUavIBC5/VWPb9YvmRCXYdyYCSSnSG6aUFMGE2VgToEAZWuf7p0hsKWt3Gd/ZRsclYCdIkPqbL5UIlo9jSTYK+tJSsW6iks1I6jM6OUCsculAgjp1PlpQ3ELDcBgn1boKYOJMjCnQICytQE5LwtsqRuZbXCUBTsZhtTZfKmA9p3xwq9GBJX1JKFiXcalmhFUZvQKApFdV/7h4VGLW/Y5AxhJzv37D9b/5anwDu0m7Rottkq+GV34XZYRE2Vl2lvVMmypG5ktcHRjkQB2FhwES6MK1Y7KA99rwoRfjQgq4ynnlXmfnO6zY6DBZVxqwUETgrpjgF4yegWB2N5fJZcvA4wkp4+Pz5q3fTL/5rs47hVx2Ibl3VaHyJD2wda1IlvqRmYNju4zvkq5ItiJMxUhxcXFusriUrHaQXkAGc2F3764vNUPLEKKv86pj4idsw7/6EIuVUBQj25nR68kEK8qvJ0CGJJTSm5+McoaEzXBnBp5iBRSTPXRN9vgKAN2Mgyps/lSAd3eONNvAoIqepJQsa7iUhkElRm9kkA8qlpXhHs6M8kpkptG1FSCiVrBnJJayGCqj77ZBkcZsJNhSBmE1CF54Mxd472AQVDNnlgq1mVcKoOgMqNXEojMuioxborCPG8V4MsbIslpJjeNqKkEE2VgTiPLKakV2VIRU3UDsw2OSsBOgSF1Nl8qcBnGvx/IIKhmTwwV6zoulUFQmdErCsTy8+fJ5NYBQXEziWip7/TCKUS0u9v0VW9toU393127ZdpCIsp/fFb6XqINzeZsff/nV7tN0xEV9UjfOmdLE/++6+IDQvtnEFt74PmgeomHMnr5h794W+jrFt/j0LqY3FtTUrVEZe1SiUj33oIT69rONRDR4U4bjo4bdsfYbFxX/8A+H7HVir7HsZQn2ozpRERk4sYjifVUPnzM3pPL0y448D2OlSezaNErnx7Y3yXhiiQ42UAs/FR1ru/aDkoMMpOcDLlpQk1FTBQyMKekFtaYqjuc67MNjsqBnU7nS9WuTrAmUEVPslSss1yqiKAyo6+aheXcBD8vys+LcoM7c6ncuPF1xY2vK258XXHjBnvcRIyGz4PSjzqPgOID9xUD/oyBG78PcuPrilvNtv8HA0wuH77+oJAAAAAASUVORK5CYII=',
    '1603.01595v1.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAUAAAAEzCAAAAABlTYUZAAAigklEQVR42u2deUBUVfvHv8M6bAIqqLGIImoKuKAZbpBLprgk5q5lWZqZZmZWpq/r+2ZZbrmSFeW+5L6TJqLimgqiJrGpuIAii7IMM/P8/pjlLjOQwL2z9LvnDz3nPueeOfcZ5px7z+d+zyMjSKkmyUZygeRA8yYiipC8UL0UQUQyAmQWOA7KrGFslpH0E65xsuMfKHnM5D1dLKqvH4x5SYxmLxx9GDSizs5ogRyYvurMFc+RLii7e/72kinP3U76mAnDRfZf3jqI4cAZD+fVT5nttz26+pMIiJPOYQAREZWM+YieO+3GO89fWb3sH6vA8FAM6ipI8LQnTEVEqj5h1TgXRGRkDHTW/lnKv815/u+h/8Xlz18593p1vurfRzw6Lvwf4PpuNgBsZgt+H1iuQB2PKkxIYVUYL3dVp6f3647GFuEdmJ4FAGgfILQDE44CvThHlDk5KM5Qa0uKCykqAOUPHiP/KaB6dB+AMvchlOmPAZRlKjn1NKaMEgBInlGdnm4b3L3OrjL9yJP6x0N2RvUoG4DiwWOmSwBuH09T8+rzU9MdhwBANonfbGIJwG5MdyHP7UAAA7C8q3/9NTj2gn/Hswfb1ps7f+bentMVAChmYHF659M4ENrgh5Xf+8ffaO81CTjVxnv2oW8uvP2Oet3q0x1WseppTeeHvk3Y+kXpsVGj1lfVgSe72A8uOKwtnGh/XL1prkqfOR3hNRaIadpgtb5LgOKTOLdd3VNZ9dd29G++GBjl2zJW1+zH6NP960QFIjjNHul/ufTTL8uZxvQX8hyTSDL8Rg55GfuIiEg10jmX1KE7iYherHeV6GnbvmqilT4FREc8npC6afSp+/W3EEUMIiLq2C2WKBUfXSH62TmPVU9vOkZEzcZXfRJJn0IUj6GawgnbS0S7nOOYDLXvRURldvOJ6dI6/xtEU1urWPULfN8mIlXIQ6bhLR4A6n3PaXZPUB6ResIIpjHWhXA7adSBg0iV9arGgVT2ci/V6gNERBT2lma+3UH57pOJSFV7JVFYK029qEFERL28VERK+9eJKBGX2PV0pmXVdOD/zhKpfJ2fEhGpW/QiorhWN/QZop69iIhc5hPTpR2eF4n24xaxqs13KyS6tYrd8pPN44OAb1nNFvvOJCK6it91jbEvhNtJuwqGRv/JBKhSQuGwvc3Il/qwTN1wYNCfBQ6nANS9BSCEc2ILG8DWoSUAOZ6BXU9relbNwXpf/RtAo7v7hgG4e/01AD2u4I4uwx2LtF0aFC3LSEpEAVMfGDt34/vYOJXdssewYRQ/ZO5YD321xLvBABAs299d2xjnQp5vDGzTErh5HoDvsi2t2AY3t1RkwVsul8s3fgagNuc0OetfFaee/mC10vVmfr6+vmM18/DfqK85qs9wk7ZL6pjwDXU6cas1GLiKSsprMXW3AoAscm3RZabaX5q+2jjc0DXGuZDKn0R06QUAZ1sAKP3jy2GXfRhDYVEzvIg67fQ3MJVcNrseL/04tkoO3Dw1BAB9dSjfAwjCHc1RfUY3oCvZXZoeczkQpwB1E1a1Cd1Opw1lnbJDU+iFEqa1QDwFgPKyJrrGKr6QyhYTaM2LAM2eOS9siEJzSwIA8eiDVi/EAUDZ3n+47ArqycuB9Ko9L6WEAIBsuGIXAJ/gywBQtkufARwJwMMy1jlFS98KBHKAo5lMNUQ2X5kcyv7TTtNURium2bZeiQBwAa/9w4UYd2CBxlEomZrtCeWMWg1tfk2ZogaA+AdA8ezogZD/sD8JwDI/QFGovfF+BgCKcgBUXg6gHCp2PZ1JBaD130DVlqs2PdH83xc/qgHZD4kHAazx1WeAVnkAdtYqgb5L9k5KAMn2JffrMNUge39bW3bLqhH5ALD6bR+mWZe129IB1cIhfXWNsS/kH25jLvT3h/PrI0cO7uCBbvRlM+fWROucXMMXEoUN/2Ldpm6zyoiIjofP+OXTA3S4j5vXa7FE5/vXrh2Vc7qfh2f/s7GvuPoPfPJZG7fWH+rrkd4UMPgJpTWcM/dEFWbhhHa1nFpnEdHUUBfXTpOIKLHtl9vm7WBncjvPPbjwgI9rzy3aLhHt8Ft4cP6fH4R/rmSqET0JKGW3HRL37oLDx6dE5xO7taOv/G9t/9kK/fUxF8LvZFUWVNsFx94uaqYbNbOfNpU9z1nG6pUnB9Su4YLqg/ymNtzM48ygWkluXi7Mp6lulrdwoEJ3TrW8VTPZ7VwKo+SLqrA2Ml6z9/Kb2v3TheicVwUHWvmKdEFaW3wzJMBcK9LlSmtfPv66XWFRaQDMQ+WORt3c//oN63Zgr56Jc6cI/1N5vp+LCrZQ2thYN1TKyQ62F955EpWTqBzMSuUiZJb47VqB8yK0Dow8YXl9mzPHChw4B9IYKI2BFvdmwnOm9IxHHQIs9JrUZCtQJREdeHbjwV2W6sCEL/p4uQDACBuof7grL57ixfjtlzsl6veacCuhhq+3oTpEPw+7SMSEGpy7WntpkUTq4fOILjW9p38nYlIS0b1XEjmVatTJ6v4Fwt5yh6Uba/1dZKD//AzsPRgLtI2cvF1rOtAoBGiw9JM4diWz/IQtOdmMA4BfxgYAmzo5AIh4p9hZYzpXDACB6ZxKIi8m6Bg9m/iXZVvyXcaHAJB+ZiSginMHgAZlujtdn8VL1cC+PuxKEHYMXBHmF/rzzy/7djhMQ30Cv6fDfbcenzhDQbS/Ob5aMc/zBNHfb3y2ccFOyx0DiYhUUXlEdB/vERFdwhLt4ZJG6HJzT1QBu1LNOmk4iWTYLiXKtjtARJ1OsRi9HtLfcIsnooWW7cANnxMR3cIUIqIrmKM7ntURDl0UnEpCO5B6RxKV1x1HpJpCbEavI/4vdyUiyrBoB5b5ZRERJWG6pvuf6Qypo6fJ8ep9dqWaddLIGDgy/g6ORW8vw6kuuKJn9IAW0j84+4rlzz77ZP4A4Kbh+Eq4aY8nv7l40bXuRweoWZUEn0QGOG3GH4tVh3CwD5fRayD9Tbhb/kQc2wgA4IESACjVd/m9hXURGDfn/GFWJcEd6DpwQ7Gjy+CNZSTnMnrNElNjPLN4/ylPeAIA3Os/BIAHuperC1O6AJDN7nKLVUn425hRyQv6Y9S+Lb0BQ0YPv+bnAGi+WktN159qqKksKgMA0muHAXmlgJOsCADgH8yqJLwDe3jvaYuu3t91BdiMXgvpZTFx1wBaZckuvAdXTWbq9XSAdk61xSO/zoD90KUAkJ3XlV1J8CcRu2FeMtiMVNkAGOj6bs86B9p+CRxZnvmg97C3gC6/TxzoezZ86Uc7t1uqA8t1vmkR+86qBv8NnA44N34RwIrx40d4njuzxoFdSXgql2/nChRC+xKYIaNH9qMW6ut16zpZ6oKqYkNUPW025+Cz8DYsPpB6oTi0vYxXSaJy0oq0pBeWHCglyYGSA63UgZYo+beKbQgiJLAu3cbg3wXW8+/qcr4esGywbm4HGgfru95x6BFUT4YV2dvfMDznSUxhSfmHzWBOsD46rL3z7cRXu8HAAvOBdXW5LrfcP5mI6BBeURsacz/MJfrGIc6cYJ3qAbCfpTZiEY6JPF8qYhy4a5FeUhlLRPQ0wPEWGRqXyLcR5aKj6A6cvPbQyYSEk5EZRLvdy4ho3Bt6W8+VX/yQQWTEYuo3E1iptFSXe/oaAMzJXBBkxFifALggz5xgHd4faDMGFosA65HeAC4vafmpMeOwosHAafSFGcG6PlVsqakDV7bzbxUbG+738hEM822yQi9+Z0np0wbPjv9fnLHWesoA1Th1jAMqeJ1G8V27maI7MBCAevJCGZD7xBUA3MHofK8uWbSkwKhFqEmk6mB983z2+UswgV1kG89Obz/+qXnBevAmNe1smmnEIhgXDnh1N+DtsQdQt+9UMnGoJyB7f9MxQOaW1qn+/aF4u01XAEMr+EJuz2zwVUVfVoeFG7KGPzLF3ZniiwkanumgoYl6frNxuAwDXacasQh3Hzhy9B2/Y9HblztywHp36MH6bP2JS68BQFphJgD4zAVAE5/94g5gXz8jRsiabvDpfdbWfGAdoQDQLibX0CKcAwc4bZ7+x+Ith14/OOefwHpUOwD4PfstAPAAgB37+0UDKD3Rz4gRQJ32p470gdnA+llFVwBuSGnNtwjoQNeBGz50dBm8sfc/g/WgIAC4a9dZ/yw32XWlDECqu4FR0Vl51gGojVSYAKz3gDGwjv4FRQ6AAnUMLBYC1j9/8F8/APjVEPqXXkzKAXAHrWA2sI5Wqx0A3HB/kW0R/FGu3LuFmlR+ISoi2umVRqTsN4SIKORtIiI66ZhMpJ6CTYYTbQLaK4lIucHhV0NjVDwRZTlEKMWfhQ9hsiaTIk8jUocvIMp1DiOiNWeIKMP+J7ZF+CeR6oP1uSgdAPWd1DIYee0k9pML4fRJz3W2ZgTr7849H3lz/sIxbIv5wfqWvyu5OWYbr1ylNqEy84L162drh9czajEfWL96r3c1jf+CFWlpSV9a0oeENSUHSg6UkgTWJbAuzcKQwDoksG7JYL1o2e37IR/VgynBOpAzeS17zU/92/Vym0lenJy1gPVhaVTwhnuiKcF67ifj2uM+G2f3nKWg9f05OWsB6xMyiKiodiOlCcF6aaZyCduB6jdGElETL3bOasD6/t0pnnCN2HWzpenAumNDrun4jksANpezc1YD1v3KFABckQOTgXWDtMa9NYB24eycwJPIyp9zPD/G2rs+c3sNOyWf8uGRFaO9PnWfY48D025+5Za3ZFcE0j4PDM1oYay1nqgErCcoHQAk2QXDJGB9vZGlPvqj0Z0Y14J3mrJy1gTWieg8ppoQrGv6w4yB+ei0SEXpPr+zchagWOf4KMulQX7FDiwN6VFiQsU634G3YZtFRGMalzA54ScREcE6ZnjtlsN0inV+coG/P4BWsWda63PdrAms/3jvoCNMqFjnp1oOtQHACSld9blu1gLWAey/utEG12xawFRg3eBqQ54BgAp1mZzVgHUg8cwyG2CPo+nAOjvllQLon6YA8BDtWDmrAes3Go8bP37cqABTgnUiWohMIh1Yf+ixl0jd4V12zmrA+sj0GAAINSVYV7ydm+zWJyhyihasex/9QBGwLnAxO2dFYN0cinWDFbf4R+1DeTkJrEtgXVrSh4Q1pSQ5UHKgBNYhgXVpFq5aJyUuDOvhwnxGy4ifIRoX5rNotuibR4wtnwvzGa1e/CwiF+axaEb0bdAby+fCfEbLiJ/F48J8Fs2Ivg16Y/lcmM9oGfGzeFyYz6IZ0bdBbyyfC8P0O5kbsGjhRd8m5MJGVmqOK+3ecReTCxuwaOFF3yblwtxRRyd+FpcL81g0S/S9RKRJREQuzO3yVSKittEic2E+i1b/9Vq/XOEcaGIuzEk68bMXROXCPBYtsOjb1IJr9r24TvwcCTG5sCGLFlT0bVIuzE168TPE5MIcFi2C6NuUXBhcSKsXP0NELsyw6LxSUUTfwu5kfmpt+4kAVBuXG19LKIMmivwjv87AGy8CyIxfYifiTuY3Rz2e8P740esCNB9Zq89xX+B2SkQXdm8saCfzSrkww2g1kJYRP0M0LsywaA0XZkTfrN5YLxdmxM+m48JVEH1LXFjiwhIXhkTlpCQ5UHKgxIUhcWFpFpZmYUiCa1QTrKttKi9bK1iHSQTXwIlV2zh2TZl+vOZqO90FVgzWTSC4JiJ6FhjFMWvKij4jVbQ1NJcsIET4nr+naXNP5wUDeDbBca3M0Og2yff7CzBtiHAAWMIza8oL4x7bYMjyWastIER4dcG6CQTXAM424oqRtOWYUDcAL6/9zlkC66hMcF3623Dut60p5991BQCvp1esaCdzUyVmJ3MAyydzF/+0ZSdb0lz+TaF/whOjmnwyBq82XNsLWzp/3WnvtHOeiJw4ZiOi+jS/MDVw1QPcfGl/V+Drilr8/uKEjua/PdscolkTv1zXj3NcV3YMfgwAmcgX/FlYxJ3MTZe0O5mjfP0YznGmPO/6A0BxGUqr2skcJg4RjhUTuX8hTLn/t++tKl8flVjXesE6xBdcXy+rkw8oy/Nt3QzKUwfFl308s8aA04xgHeILrnPvzgRw3Xtm46mG5YYNgawAERyIUb0XDMKo17RgfSSqBNaXacC6PywhRHhEBADsCF4BIM9Zzi4nLI51R8HJ7+ysEqzDVCHCAUD1rBBals8q7997B1jaaLR1gnWYSnANYMIVJPV4dboGrDPlN689ubzpz8P2kuC6imBdnx4fzm8TLpPAOiSwLi3pQ8KaUpIcKDlQAuuQwLo0C0uzMCSwDiHB+j+J12uO2E0J1isO1w2xwDqfsfNwOh+5w6LBesXhusUD6zzGzsPpfORu4WB978FYoG3kZP0aTujrdxv3CICYYJ3H2Hk4fYkZx0BUHaxXHK5bPLDOZew8nM5H7hYO1gULyo0qgHXepuYcnM5H7pYO1isO1w3xwDqHsfNwOh+5WzpYL4ILANhC77Kr16fIdr10tKGYYJ2bZeN0PnIXbhYWSbGehOmapj4zrWKdLV6nPbL7RGXh+JoUH6uIOkRZkWK94nDd4irWOZuaMzidj9wtH6xXHK47EmIq1rmbmutwOh+5WwFYrzhcN8RUrPM3Ndfi9NM85G4FYF0WdZkJyp3nLEer4SZQrDPZPGc5C6ezkbu1gPWp19MB2jnV1oSKdSarAetcnK5B7FYD1llBuU2mWGeyGrDOwelaxG5FYL3icN3igXUeYxcAp0tgXQLr0pI+JKwpJcmBkgMlsA4JrEuzsPQTlsA6/t9sBc8H6/wyRFCs83XpzFbw7E3hYZVg3QC0iwDW+bp01lbw+pzVbAW/272MiMa9UVFZjK3g59kXElGn98lgK3gmZzVbwfPBugFoFwGs83XpzFbwTM5awbp4oJ2h6Qa6dGYreKE2hTcfWDcC2iE0WDfUpTNbwQu0Kbz5wLohaIfgYN2ILv3czj/a7nDh5qxDsV4KBw3ZKzFehhiKdUNdeoeFG7KGP+LmrEOxzgfrhqAdwoN1I7p0Zit4QTaFNx9YNwDtEAOsG9OlM1vBC7ApvPnAugFohyhgnaNLZ7aCF2xTeNNtBS+LymCB9VJuGSKB9YSBBUDByVl2/K3ghdsU3nQhwlPkaUTq8AW6eN2ssmghwqfbJBPNaaPQhQiPiieiLIcIJStnNSHC+WCdVYZYYJ0B6fyt4JmcFYN1flkEsG4A0pmt4Gu+KbwE1iWwLi3pQ8KakgOlJDlQAuuQwLo0C0vJDGrNG5lPn4422oTivD5r287elFdU6igzx8109dL8tdko0QeuVBcyryOkBwJujerKcpOAvwNZpxTsuGXXfBALxK3zfsnbJu+Px+OE+QmXOL/ZyM3NSQa0aAvkHvoLwdG80JB4Mu+z+kb7Up1xpmYrH8cBJnLqjuaM4QrqrFcRqXuCGy33UK3uB3cHe59njnQC7OXABoFWY1L0l/cr0apa4Zu3Nmp4k1tF2RsJxvtiSrCuSRfYDvzZhzEkYLf2JYHGRWwpopPLU6IsmdcTtgOBBluFWs7aq/PfaDUdhuwh0Qk0KeNU+QJIMN4XM4F1Y6nIdwCAtGmQxbI2k8TakkgXwL9h5vpJ+mP96tVt1c9FqA9OC/vcXS6jzxx+kGERAryBl23+5kirf9tdcV8EmESycxxIEVBYYF8uC8VVWzgGoTw1xa5loC3w7HpRUWe3056tZADo3g1VmwrbdRoCQDXmGT7pwj58UoOQAjNPMp0eMUzIby5t6BsAFuedcQQuwwWAo1vBCZYDr/0ntkOFfUHNV6RjQ4GQ4980R60hRK/ZuE+m/Q07LPompOVJoj8bA0e6ReMlJdH58PrTZkcMqOgnTERE3wIt2MGlSe2IoUREUfBjfsKbTi5dk1Aq1E84JYeIEtxSiYh8EUxEajlCWS9FBV/MABKM90WQMbCkDr4iemDjVkSkCrxMexBeSlTU1CaRSN0YXU7sBe7QFQfXu0TqDypzYIojbC9y2wbeJiIaADfGgX7Tti6t1fyMgEv6hT6zSPMx/kR0C/BlTSCxpHGgsb4IM4lMQ4CKkoAYoiOdSFEfS4iIPkUoEYUhgkr/+xNRO4wiItpeiQMVYcBsbtMFwFhNpx30x5YmEdFXqJsjnANnIY2IiK7JkUY0yQ4vMBPIJNI60FhfhJlExn+befS11b0PrX5Xtvp9/PUAPgDwApLyagPoBMcZQNElsFSqi3/S3F49DAYAOO+tDwBfXULbLwF8+Mqgb64DwCtvOUHDRIvBTBofAUBbPFozS6hh8Mm3L2iwcMsjE18dlNrRNV//EudvZ/Tv9BjrizBPIk16xq3ptO16xOULPombkQHYAoA9kF4b0MqjMwmswO+RmgfC8wfe1TjQEwD+nA+HX+0BXOmFo8cAQP6WvUe+ttPeunMTlg8ZDLgASYLNIztLwrQPI12v3s6Y3GA6grWWvyb8cAm4D6TYtahl2BehHuUmxO1b2Mt7wkerG46WIwgoBoBnQBNm/SvQRl3MnNC2LQAg9iRri+jSN5VY0BLAw3MemDoMAJoDnfc/00DlLkDB7WAZMCv+9GBACdQVzoE6EXzxPJ9JAcgg9NV+2p2miwDkACvcv2/D7ovAXNgHSKAnTvIGqUTKFphGRDQCrxBRmGZAJOqJIUREWyocA6cDnZRERB8jmTn6PUKISOWN7ZRTDx8Q0Sib6US0Dtgm2BgYpJkdiH4D7hItQ7Ny3acREdF6zY00qy9CP4nMQbCaaCx6EhFddHa/TnTaoXYa24HpHg6XiLKCgBS1MQeetoHDhSd5mQcHAneYw8Uv4gzRdnRR0e9AIyK67H+JqKgVBqsFc6AT3tc9J/VS0AXn2tdI92lsB7L6IrQDs+UxRHTZZh8REd0e4RPawn9cLtEpF2c3V+d4IiK6N7Jh/wG9pwNYaMyBE9lPJaymHwx2HfyW29hCopKetWKIiE6EtHvV3WeFQrjbmDCs0+YW1Y/q6jo0i5hPIxrs6ubm4uocz+5LDRxofDVG4cD8CwAFtq5Gfv2Pa9k/K3CSy3XLR7Ez7z7HmFGYaRvAnfdy7gbUFnJB9Wl2U12XitOcGtlWpS9m5cLP50BpRbrC1GY4/v8liYlITETiwpIDJbAOCaxLYF2aRCAp1q0lxrqVKdYNhekih1qvLMY6o2AXSi8vtmLdQJj+fKHWRYqxzijYK9XLW5Ji3UCYXpqpXCKuA1mKdSKiBSwHMgr2SvXylqRYNxCmix9qvZIY64yCXSi9vOiKddPtAI9/jrHOKNgF65boinXxhOmoeox1RsEuWLdEV6yLK0xH1WKsMwp2wbolumJdVGE6qhpjXa9gF6xb4inW85UA7N1FFaajqjHW9Qp2wbolnmK972kA3X8XVZiOKsZYZxTsgnVLPMX69jIAclGF6ahqjHXoFey2QnVLvK3gGxjuAJ/nLIdZY6yzIquzN6a3aMU6S5iuEY0T0UJkivkkwlKsExEpXbvoRPIsBXulenmLUqwzwnSNaNwUodYrjLHOKNiF0suLrlivXJhu8hjrLAV7dbplBsX6v31FWlrSl5b0IWFNyYGSA6UkgXUJrEuzMCSwDqAoCwDg7G9n3Rdmqv3gDdyUuyXhZIN3HZ4lF3wUbc1/nmy6zg/mLmhwd8OVj2QMJyKKk79Twfv/6mXVXr947lNrGN2VS9f5wdwrC+4uwGqMXDMw9oj+qdcQoz7PvV7tr6sGp6LagdYNgrkLGdy9kpGuEU4Yd+Cu6n/cLpP9hNl0nR/MXcjg7pUMc9na/UUVF1JUAJCfGK/IApA8Q1fj9vE0NQCoHmUDUDx4DDZ+11mVuQ+hzCgB51SYlK6b5UkkdVv/sQAoZmBxeufTwHe/uDv9NwzY+kXpsVGj1gOKT+LcdnVPBU5HeI0FYpo2WM3C73rrqTbesw99c37o28ScChPTdX4wdyGDuxsO2KnovH3bqg97xKqJiFb6FBAd8XhyM4KI1J2IiJqNJyKidf43iKa2VhFR+15EVGY3n4jUTaNP3a+/hW3t2C2WKBXHmFNNMIkQEW34XPN/8CY17WzKYAR+uSaTiNG/wDrt2ndU5wfJABTMGFQL6GGz6eGVK4BsDKuWR9EzoNuVNB3RdHAEGPzOsroljwYa2V8z9Z2Mjq5j43AZBroykZj5ZcEnEdcAYOXI7pdaAH8WOJwCUPfWOwFtWvbo9y6r1qBoWUZSIgr4A0EI34oWNoCtwzNTO1AfXZ0fzF3I4O4Vj4GDStcCyIK3XC6Xb/xMHv8f55U93lSz7vVjwjfU6WR4Ym0DqwZmqkztQN1+8GdPQhPMHcbLIt3GeCAJwIuo004zpzjNnfvkl0/HapdJfhyL6TGXA3EKUMs0IzVpQyZoSnwrk34cC5PSdYNg7oIGd7epzIGE4pYvxAFA2d6ULYDnlEHXAMjLgXQULX0rEMgBjp6GIwF4WMZ+pGZZ2Tfp5UA6TEvX0Wq1Pph7Xim3LIoDCzQDl79tXjrWP/1hfxKAZX5YVQxA8TKA1n8DBHsnJYBk+5L7tdEqD8DOWiXQR2JnWxXlAKhcpT/VNEkfaJ0J5q4JtC5ocHfbOcDcOawDSRNjyu4eS4+Es/cfjoGHo4PCp6Znb3yp683MtId3fgoZCCB0ecHJ7gF2zb5XFG3tJ1vvNsQm5Le7xTubHT15ol/8R0npx1WtwVh9J517cKbxsakPb53pKdec+hz94vSpmg9zm7t3BwC0WZfkFP/+pxNkwG8h0ZxyzdLcOZUvqD48Sv09AGQ/bSpDoZPdLUWQZjooTw6oDUB1s7yFAxW6A8DjzKBaSW5eLkyn2FYmaU81xYIqi67zg7lXEtxd4sLSijQkKic5UEqSAyUHSmAdEliXbmOk2xiYnAsX3rO3J2V5A3f8G5h6xXhdKPBu4MAH6+NPu0Y3Gub+r2DqfPE61L/cKVG/18RQ1g7hmMgFDBSMj9cINwjA1HnidVJPSiK690qioUV4sA6L4OM1ZOpsvA4ABxqFAA2WfhJnYDHtphO7LPonzGLqXPE6cK4YAALTDS2i3Ujr8XhODoozNEiE4eM6OTvbavbEYup8vO6zeKka2NdHQPBu7OWiQfp82dR15xdF3iI6EIIP5n28tNunZURbopybjBz5KxEd7rv1+MQZCo7V/GMgEZEqKs9IlkoaocvNPVEFhhYBd+1gO5CFx1+sd5Xoadu+aoaP7wnKI1JPGMGzWoIDdUydmyXK6giHLgpjFnEcuMPzItF+3CKisLeIiHZjh96Bxb4ziYiu4neu1QIcWOaXZSRLRKmjp8nx6n0jFiHfTNCmFAUGPQ7L2JPA2ligGw7o84yc3ZjVrEnP1DlZIPnNxYuudT86QG1gEWMS+dnOEJ67uTExtblydr7VrEkfY52TBd5bWBeBcXPOHzawiOHAezaY/umGWZ09AbXu2b6wqJk29yNPzs61mjcpT3gayQKFKV0AyGZ3ucW3iOHAW7e5eFwBAPHoo+fjnADsLKv5k56ps7J5pYCTrAgA4B/MrSQ0WFcAQNYwHw4eR/wDoHh29EDo+LjL2m3pgGrhkL7gWM2f7jGCa132kV9nwH7oUgDIzuvKrSTkk8jF+VdwbKBLefZZVSTksR9/HXqpf85nEW8CEcsDndf1nSkD8J9uc2W92HJ2jtX8iaVY12U1ivUV48eP8Dx3Zo0DT9Yu3j7SDB5vFxx7u6iZHY+P6+TsHKvZF1RZTJ0vXk+9UBzaXla5rF0csN4uOLbaVmlFGkC5svpWSFTuaNTN/a/fqJ4V0l76gAq2UNrYVMsqbTohUTmJykFMKhchs8RvF9YD1qUkvRsjOdBq0/8Bp8SpQlOhn8kAAAAASUVORK5CYII=',
    '1603.02514v3.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAkoAAABNCAAAAAB80hTUAAAQpUlEQVR42u2deUATdxbHvyEcihwRFOWQhbquKGg9UEux2qrrVa0H1tsitq6uV9XW3XriWa1Vy660HuBV7bqCZ631RFoFD6gHiq2oKMqlVjkLAUny9o9JZiYkQBIzo9vO96/kzW/m897LL7/5zeT3MjKCJEnWkI2UAklSV5L0comIIqUsSHo+RRKRTJorSZJOcJKkriTpj9WVrrXT6vURG0tF8iZ+DIAtUwGgsrWv22H+tisBPoocw10M2gEAcls1a5hiFq8mvSx+CJDiupy3WlcKOLYqLW3UsWNxczw+9D9YfWvvWyYd3sRmOu1oAmBbUwCwP2RbqOZva/11brGRXQzaAQCa7C8q0pjFq0kvwI9aklZbPk3LtdGQjTtvwRVcTUoFNhAR0Y+2sm/1N5U6XCcTZGIznYrtzxDlydKZd91xQH+zA7KN7WXQjoiImuO8mbyaJLYftSSttnyaluuaQjbuvFkyaa7U7X36W5meZUOlSf3UxGY6fa94HTj059Zind3F5pnmRy1Jqy2fpuVawJBtTWrVf9PDg2OArMs5DV55wxb3di3Gj1myt8GZAFy5oA7SeLYCkJmW7dnXhd/MNO0fJAf2h8n4Nh4AoLtXbIJaAByDE2coTL/n1c1MXtXJ8vLOAThRWt66I6pOlpW945qa4d6rXp1+GDjyfH7wk8YdiUltPW6TxbmuluL8axmKgE4ywyBZgAGJb7LgCq4JkAQcabHeVzm65TWcua3Bib37wDOBJi71DUrpnwAURQRccI/3i+c1q1MXoqKioqK+V0ZFrUssjIqKZecXHAAA3v0k/0yHwXk8hk6cQbXCe0V54huPzeM9ix4engTEjH3vMPAsekTE5UEnCge3LarDDwNHntcPLmnckXSpZTdZlGsjKa6c5/Ol7cUuEwyCZAFGSJzJkrkSZQBDieZiENEevEpUBTAnZs50FUlENG4D0UgsIlL3cMjnmtWpNaGhoaF+jqGhoT4uoaGhfUp1p28OQOSAzUS0DaEqHkPbjjMswyA10V15rXMUI7wgxBBRX0QSEQWhSw7Rq9hZhx98R6zkhy5p3JHY1Oo2WZRrI6i5GKKhbQgwCJIFGCFxpmoyrSvdAMKJHn9xh+gXQMn5zZkewHX4F+eUKsoALhNRNLaY0ZWIiGhsBBENmKU3E+QARA44T0RFwHEeg2nHGZR22EdsYzN4HRFDRIOYrtQRs4noHSyp3Q9+sNbyQ5s03pHZ1OryaXmu9VAldognKou+ZBAkCzBCYk3VZdpcKRdoAzQO3pRsWw9QcRs4U7PIpXFx6Lz7lTRgviOQK//NzGlb1ZHtQNmpOXpGQ6arQ+XV4uoMDnqjCo2fg8eqJQAZNLX70bt6sNbzg3dkNrW6bRbnWh91swqegONUw2SzACMkA2/MO8F9BLt7VDUOA+/SbaCU+Qpcz+ObiPJjR3mjF50G7vC/YNfzTBiQlNnZ2XvtMrKztzrfy87OfqIblfQAzBdcI8cmHoNpxxkygDN1jwZGeMyo9I5uVOKGqFr84AdrLT+0SdMLUZtaLp8W5NoQlQGc4jvPD1ILMEbitllwgstthAVEh+BUTnQJKCUVkEbj9/BN53pqiFSjXKnIhRnZU5LYZnVrQrVbZiXa6HgA3adyF7IbPAbTjjOoPLCNiMi+1o/QCK8TNhNRJ+NdqQY/+MFayw9t0nhHZlOry6dFuTZEqTyxiojov8XVg2QBRkisyYL7SiW7Q57MXgI0Q+UT0E4XFEAeiCy60ZJvKkvYCdg4doLrZiwsBG6OcWCb1a2v8vNz3GLz8+/ax+Xn5+c/ctYN5RwAANaroFmN+a15DO3ZhjXINyC6ErTwGZ6axwtCLnDqZ+O71eAHqjtiBT+0SeMdmU2tLp8W5doQJY+Rrb4H7FlZv3qQLMAIiTWZfoJLdXcBHN3dnRqGTEolItLEtqjXd+jBqMayGZT8Z7d2/9QznfYa9mZY52G/EtHFXv692nVNItI1M0lnHUqIDioqdeOxh4uT+8d8JjlsiO0ztE27OA2fwbTjQxNf8x3YY7kdZJ+aw6OcHoohfWZMBnpRkYezk9tE8nNxajigDj9YrtX8YJPGHolLrXaTpbk2QKW81bRHYNgjg2S30QGMkHgmfZm3XqmEXHU9sMCxvp5Jo7JXFrlpv5xU6CrXb1a3Zt8+DIxX76yZCQAljroLBZYBA4O6VIGHDvUcbMzjVRS72xap6teT1xG7vh8Gjjy3H2zStEfipZbZZGmujaSYnrrYGwbJAoyQ9EymTrtFleZPW4meNdz/e+W9BH4IjHpp1iul5QwEkir6/F55L4EfAqPki1+SrqRq1gN4FhDye+W9BH4IjJLWdkuCtCBXktSVJEGqg5MkCVIdnCTpBCdJ6kqS/kgyab3Suc0PbchrlnNklmr4DMZU9m6Ja7ehy+6rXfxlZRWhEQ0AZC7NUrv6ydQV7T9wtKKLmuQbPl0VgOpQb2c8vtQP6evyZfBa5IvH4Rp5m88EzA6LBK6cd+3fEKKxdUGLHDWLtXLxEk8rmRUT20PclYxhx5uuKiKKxn4iKooMZKphNuIEEZW+651pvfvxTybvTV3cOI6oEHIfT8d4IqJV2E5EROt63xH0pwYWqZny8f0r3c6IxmaDFjdqDmuBTFtFiUZoBADuEyYdGAUAyPO+JwfgjvoAXBd7dkn3Y986xTSddNJqX5QlSxsjuHJc60B0DygOGu8NAO5wB4Czj76XCzto65A7U1Pgu3roTWeR2FzQokbNwwo+V/LpvxkAcD1I3/5+k5m8d65Nr1hvyN25CMDIyv1A142753vzNh35caXAPYlFxvYE0Ln4uEhsXtBiRs3HCj/tnvjDLQA41rfalGvsIV7vuZ01wGrhyZpXAXDEQ4Mt/8lcIBNpRll63huAzPW4SOyagxaUXAvWaiWVnPp7xa4GlHa2BssMT7fXzb4ujR+z3nrh/QQAPyEEqIwqoSaTdImMXvmL8H1Ii3ygcgIAlwyR2LygxYyajxW+K9lO2LTcHgeHVLc3RR4AYFcSClLemOZs5ZF3Xcgo2Jw65EvDU7cw/XV1Wf70HYLfKtEiSyAHAOcSEdlM0OJHzWCFu4KLwWEiosNH6Z5sD9FKojF/IiLajaNMg72IJKJ4nCUiOuvyuXWvLOaHFhJRFRGdwlkiikHPRJqLOMFXi2mRydhMRNS+rZhsJmjRo9ZiRVj65td7M24arkZ/iPa8yWr4nBRrflHiLh1XaEdQfzDXhhFvYnGHSTli3Hbzx0kX/AYAv7mKyNYGLXbUOqwYd7snJmQeMZxVJ/r15r3rgDNWjC7h24MNkE4h/wCg0BZwuAL231SEawT9NFmkF0oBoNRfPDYTNMSOWosVpysN9PjSxq668eqBDfwl6ukItF50qYd2OCBvD+V4A8hBR3ZDwNrTXwjblXRItzZPAKiLe4vG1gYNkaPWYQWcditRAQCZtoB9xJoMABUVJAOUUAPAxdFRfQGggimEPhM92HoLiH8JH7aMfrsWYTNmHIA9LUcAqGDcmbxt3muhQs66WeTMVRobJCgGiMXWBQ1xo2axgk27k8NbOAdNuJ41wr3xyAd0J4xo/+gmToNX33rPD8FTpo59a1gaEdGdcH90mTJ10oDO0c+sVybRnPHzMhXMOJT+aUgG0fWIls5t3r9PtMHLqemIAgGnoCxSM/1v9892ShKLzQUtatQ8rAX6f1qvpEm93aqd/AUhsy7a/9XpDxK1VCYgCdJ6JUlSV5IkSepKkqSuJEnqSpKkriRJktSVJL3orlSaxn9XdqcAQJmULUkw+ze4HTFcXzqx2qVj+a8TfCYeBW71mzldv2X6Fznlk8YCWJdANv2nWNu/gis9Ab2iGpGriHjSksVQcl4ndzsAduxt7ntn7QY44/jCBxuGCMyOCXODFX+D64w07SvVtJapRKRaEdiFiK4pehoui/N3SCciyvZ4ZOXfhJ7uXfbKIP2iGrGriHg/UHFk4bVYe874Wfu+4u8r8m4NKCXKRrzA6GTZbWsWL91qnrLzc+blwm0/+wKQz03NB9Dm6XLD1v+KGHvRHvAZ4mHl74eqwZRjAPhFNWJXEXHikYXX7Vlu9YDLLbXPpKFhr87Dnu+udoWP0GTlIrLq8+AiHwR6MU8eSLfR/evq+S5EROp5hqNS6l7MJSKaKsB3JHQQEakVk4koDUuJ3viEiDQN4tlFwt8t04gzUPDIwutDIqLiCN3TH76rX0r0NKaCiIQelZZ/g9tWXJBLec3G5SUCADZqBmuNnT0B4N/vApQWf7oki9c+LPyzZHbf2/uuqYFnKk2lWqlWVaJSpaoy3AWWFdWIXUVUS/2SoJoKAAuW6IbbJT2dALcPdP9UW6FWqYSapPl5WfUKLvl1jALzl7w/QPfcL5utAFSt2kH9QW43xwHf6Z3ifMZpH69bMGRf28R+hYh/VT48b6Gbx9eIdWmZYGQX84pqYsEU1RhUEc0ZL9L1SXWysGoB4EhwM+274p98/rPos7nl2rflgbb9rgjDLd8/2ro3A/aHwffNfWUA8MiGfWpeQwC2fYBTJf2bvKb/916uX2fNYl59oPikxYeNlmNMHKY2W/Nh64mYOiqpr5FdYFFRTfUqol/zp4vUlfTJIqhi/VjdyzxK8136T++3tEORyv/Hk52Ega79SGbVrlT5zBkYV3YQABSacgAoz3lc8GsOE4rzt8tuaLrql+d2n7PlEABcO/A2gO4HgcDgb4Cy8/egauRpdBdztcj+ezmIWZOqYqaG/+q85JOv48X5ZPXJImhTK/azkUHeFRiTshUAcHf69m4CMZP8vWDVrnQkd9q0aRccdgJAMLIAIH/bHPdm0UoAQMjcZUGN5ldjLm078RGAn3B+zZo1mR0AvLevTOnQfRcSetWwCywoqnkRVUSM9MkiaGsA+9IXbQE0dE4EgKShV90h1OltjJVvUZ6IswNQGvewKTBu97FAAM0X4qjvKu3l4oIZF85uUn6pt4/DruCJzQAnjA7WWkbOPmA38C8rFiSsrGEXmFlU44D0IF0VUQhXy9Mh/KQYv/7ok4XXz9easK8dmbssNoUAYP9Dx4+/FIa5J3s68ACLFWsdrHMz4Okc5gIU64hI08+njDG7ddFu3/UtESX6824GMA/URDei+/LNRESpRESD/jpTU1L/aKSRXcy8GUCUMl1FlLuAqM10IlLV38VVDH+FNaJcn/PIYmgLzmlvjRJRWB8iqrL7BxHhJKXaC/kYi51WvBnweVsAQG/Fdg0g+8ZzRDEAXGcfhEhfEdD4L2z7zEwAwKzuasB37gYlULAFAN475SFzHjohzMguZtylVAPAL+GKZZEfhbcCZh7ToFoVUad5yWIMEzyyGLoJ5ueLm16bgYUpRcA5hymAEkoEz55wQ8DpPpNWK4xKF952abKWiC6OdHLqs46Iyuf5fZqYsOr9u5N13Xbw3N2x4Rnad2lhCs/RBUREWcOJSL2u5/q1HxUQEVW4ZxIdb2+4i8nKnzxE0Sjs76W8ohqRq4j4P5zoyOJoFnKYp116xxLRxiFXj3c9RnRqgFOn9fSOo89igbB3J7Ryfn2SYMVLyuRsx+Dm3M26Buo7iqY1XjRq7jbWTk0fewD0pHHdu5ilF1ZFJC4599JAfr6yzytecwGk4iVJkJa+SZIkdSVJUleSJHUlSZCevCRJEqQnL0mSTnCSpK4kSZKh/gf8nlrQqvXgZQAAAABJRU5ErkJggg==',
    '1604.06285v1.2.png': '...',
    '1604.06285v1.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAZwAAACGCAAAAADMEGOsAAASsUlEQVR42u1dd0AU1/b+ll5FURRFsQaNwYJgjYr1p4LhxWhsmGjUqDGJIdaIyUMTjeZpRN8v9hg19pIQNSYaLCAWsIIKGhUQBRFBKQrCLux5f2ybXZnd2csKLJnzD/eeu9+cmT3M7J0z3/1GQhCtupqF+BWIyRGNxYgoTPwWqp+FEZFE/M0RL2uiMZiVruN5vKpVp6mTtse6aQNJFe+u9MoNeSt/yzOt3RjAFTsQFVri3tiWYccfczp1HIWBXrqs3Z2GU3LbnqCMO5KeswMlGk/Z/XuOk+Y2rMrcXBzvOMTmYurbSyP76Iw8Rn2D6IodiAr9Ivlx2+njHY1EJwUhGWgJevwcCA8BAOSuG+FlcEKga45oSkRUtLM+Jsi4nrT2aJJJVWc5bkNKiSjeFad0h2aECdpCxQ5Eib7YBF3zWdAgIvnDEMwgyj286cPa2K0foec3x37sKbutc7gez7l4MK8KT5wt2cMsAXT48eWhy0ZtqGIH4vc54haygiUNw4PvAenrL7fIq9CEoO1wrL6hdVDA6SpMTiLOAgCG1tMdiTtr3JYqdiDNgFMVOIwpqYD37+tGV3C2NhL0E7dfClTlb05zbBsZJwOsN7wOAChJ2HU4GUDRwaFIj429L3xLFTuQXMCLFesbD59Sk0ylPYAobv8cEFyFyRkqwf5uzj1C77zTAADt9Zz4NKHLe/nYtPwZ/po9+4jwLVXoQHLXo/EyVuxNwDnJiAoBz+8eEaUA9RUej9TUu6cWWVuGyqtwQkDhlor5/0Yiou1oV0h0BOOJqCWETggqciCOcPnmm/kj3d2/yCGmCcGsSU1xVdVPhaEJgZXezMkB5Q1BxmRA4hE2rmmV3uaE9F59Mh0ondqhC4o/wzgHYIjHzv/WMmITFTsQu54ozs1y8XBl/LF66pZWkZtQLcsGPJTN49XinrnTNkqOOfAHrdiHpKeQXQLQICPZx5hNVOhA7PoAg+3DP609jgn+Cea4m6x88xDoV43KGeHxgKTVB0emIwHIBJKOHz9+/N3vmlXybgQBW1mx9v1sTXXm7IDV1GqUnLPoCAD4YK0L0A7oN4k7+mL6lsrZDRfgNjM4wlRnzrGDWNyqOlUC1+cq0oDeQJNuiAYA+igNsIMMRccqaS8aAOn5KIpE1VWlc8/NDpJ8N7dalWlvd4mUAU9C680HJDvq7jgIyL5Oagx0QSLO+VVWchqBorF/DSrjYZuWxdvaArC1tYFz+1nJRETx9rUcnB1qOZyhqrYR7+3yqe3T0XnQfUWp7WNPn36d5j4nopwhkk7tbhmAV+xA4u1rOTg7Ots/IEroa1l/Utt449Eujj2UfZmdk4uTcy1n+2g9ILN62JbR0AJ5GfLWNmpPYamLqlXoVonPM/LvSdpavfIo4pNQiE9CRROTIyZHNDE5oonJEZMjmsj4hMj4FE28rIlmZUb7+kz5FNGmsUN13D25xT85Odl7Yk67T3ak7FONlnRk+/42pdsVhWiovPJtD17IP2yl02RAA4hau4859i9JMotP3QQRPKqvXcdoIiL5GpufWeDyMV8TXfZ6qO5/eo3oYd/z2k0GNBEVtgxkjf1s4FdS2h70Msa8knMHwYrGUosYBvhvLiVENGWEqn94JRFRwgDtJgOaiBYbSA4/Wj4imIhauRlFx63OFuI6nWGauetNGwD+h4uU/bh0AGiZot1kQAOxzeuxxj55YCaA3QdrzGzNbsD1GACQXkwsEwwqi3QBgIYlUUqHx8pVcuBwgHaTAY3iX8Ywx17v0hGAX3eY+W+O+rJGcxBGJN8QEHWo2xla27VJ6++JRnu8sUUfOhMfEhFdRrjS8aI5et06GJiv3WRA03f3qWsgW2x53Y73Qr+d9zfVmN8cWo4JRGs88omO1c6lvHqTiUjunaUXfRshRETxWKjypPWATS+pbtN49JXNZCA5/Og8vLm8jFI8jteY3xwQ5MgPHV4LGGCxCy4T9xcBaeP1L6Aqhg0ASPBC5ZG2nG0RM/SRTtNotGz7BEP7y48uQOxICzQfOKW4xlQIMtAUV/Jtzpw5c67ebWBK/j7g5/f1Y5xRBgClcFY6rr+/cvmN/n/9S67VNB79w8cGv0Z+tCM8PQF0SDlXY5LzAAORhvp2dnZ2O+cBLQethyxf/4mD2op/22KoSCEfLquHlpELLxzVahqNTiqpm5eXVyrLe8YSu5aNKwDYI9GsKwQcyznq2xOvo66aqjZtWELKWwZALu5ZAPAIXRT9gsReACRhJ24HcJrGox3TvwSQVP/LFjNZYrcrBIAy1DOcHN0Clqpf111SjapcS0o3StChUeREACXHgoChHhtsVxoASQKvAkCKqy/w1MEO9pJntQDA05vbNB7t7w8AB7x/YIsdtFhqA2TBz/BUOnlBb7h/ufTbzzsGXOX0F41s+9Ej3g8ZY8m9djHP1i7hXSKighnOR4mIjjglENF3V4iIFjquMghPtEsmkndfTJTt4EtEkxcREaUPKdFqMqCJqNSpF2PsrNqHiORdJwuaSusWsJT9J76NMwVUueSreatLipHfMJExNfH/agant4PHDur0ubJEdbJ76LY5R4iIKN1JwIKmPf6JT2eNkxIVeo8jouLxU6ISNk5I024yoImmdXNy6f8dG/qC34FL08YWCEqObgFL1d+jvsnQV+XKmsq3d8oR+aXnJrzzSb+lWqF2X8jHs7b8cJm7pO32zk1x8peaDOgKxc797ccEIuOS86JeOzmnfwFN+D+ktvW8yeEfEc1oHQJ1AUtpBeUuIdapcl0PVd9kqcpedOdUFmekLCdT6T6vuCErzc5CaeoL8cGncfc5TXCS09tvOY/3Q7RxWFFKz7PYO7/4xLhx2zUOIKrzSfmuRWWqkZud3T4FgGNBV4vnLJABZ3zqh/35nwujPhDZDAIKnzoFLLqDdzIz0/6c2vyQgCoXtVZcvNSOKMvLRBEOkeoR8h9ORAdfe0ok/2gsEVGPfluJ7uCEeB0zprZGUFQzbu7Z82v6uFtvCahyKU3toOkDOgFOrzXWAJwAvPh4VB1AMm3XCQDO198DmlvfEM8UYyoEGVAsCG8RIqDKBaCeeqmk2pGeNBjAgHgdUHy6NwB4S37vD6CtBWBpUyhmw5jkPMBA4VUuADtV6+I1jlsod23337ADAAubmwAUbZSJ2TAiOTlHfXsaX+UCsHmS2lGGB9AaUfxtiecAICtpJaaAbba2pHSj4ZV8qioXAJQcAuxkQIrG4eF9FQBKIlQjCuvkdh4ALmKwmAKjkpMPKQA8+2zzoU6Kfr6hD9lt+v0agNVNgI53AdI4JJvO/wFgfWPVCCArBBw37EsBypaNHApAKgNAMsOXtcLnGiuqRl+ivKyS1oQmhCXccxrgSE+y/ec0BHDt68R0C1/f5Xo/BJxa0Lf1jT4BQEq/DyT+/hxH7MdDOtxqO1w1cnHxGXTf4obIpQPrHvFZYI1zy2IkvUJvbbtYr/PG2vp3tkGufwObqFSvnsWPopvdrT6kwuj5AW6OADDWgiU2f2iTV7mkl5/olL0yb5ZpjygsI1FmXISy2reJaAXWENENN/7SaqWTCtcpv8s+TGj+0OZE8MiZQkQUjnVERMFFvKXVSicVztjw5+mYmNN9UpnQ/KHN6UnoYy5Buv3jpkbryiiJfROLlI8I44o0pMIioaTCctAWUwBg26RmTGj+0ObEIbDuxun04tRW885HS9O4RVdULqnwEwBIORfMFltPaLMrOCkvaxpKIa1YlRj377q0J9ChVXDwz1VCKiQqC3zKSGjkD23GyVHVVm/5E5H8TdKUViudVEhEO75gjs0b2nxXtqlrq1nx8YBkgmHEKyMVApDO/4g5Nn9osz1zTmJ2TExMjNdnLzrgjc+Ok+EzJxWzFJeWxUrHte7ZdLc/upRpNRnQRAc8iTU2f2jzPXPUlEK76H87rBnwvtwg4lWRCgFga3Pm2HpCm+2ZE4vNysv5A6Kn4VZRijPnRz33ge7DiYh+QZyin++kuEvuFc5tMqCJZE5vG7j/5UXrCW2+Z466tpq4B6gTMvyGdmm1XGJfqobYVwzYSxT8WQWpEAZJhbxoIOm5qyFCIx9aT2jzS04xSgBwi61riwBIu2lKq3w2MykFoF9nWiKnSU/AetQqAMh42pvbZEADD+FkYLd50XpCm5lIxJEdj24WObzuPrm/ptj62wbfN1xiGs7QFF15be+6tQ2XZP1kjaKuHbcDJVNtx9aJO7fIU6vJgMbhoNAlBnadF80f2rwVPDKee0lQYG91W/qaHQDIrjfTf3l5/Edhdx/OQ6o7F4vad5boNo1HS3cENjBYfeJF84UW5VVEeRXRxOSIyRFNTI5oYnIgKhWKBlGpULysiQZRqRCVqFQoJ0sxOahWSoUajcAYAbxAfnTuxoIXsk9aM8Z+z7ezw/3z/9dPVCrkEvs4GoECeIH86OxPson+YxPJGLsBAOuv5KJSoRaxj6MRKIAXyI8Ot9tHlI0ejLEHrpm/KZVqjqSX7hpuYTYygIhop22hsn8cl4no4jkiCiEioq3bmdC7bfcRFaENY+xgPoy56q2NRjQRUcmFG6WC0aV1xiiYIUeUjhEualLFXSKi5ClyNrSUiCIxmzF2MNOa0GpsWmu413XzbLMSGNPYe6veCUWuEwC4qF4kSaeaP1iw9IvbANASgHzGMgkb2hqQfu/3JWNsJIQvD88389ka1+ojDVj3bVIt2A5N/Whsq0EzgV3tT+gV9XoGRwCwVK03KnjSZv83Fqm9tvVX9He3q8OKjvv1VKcDjozohKQQSUSXv5rWmJtQUygV6mgEGuAF6kV3XbYjbUwOI3rnGAmGOc2EqFTIUQvU0Qg8LPFkR0u8dvw1pIwN3R4A/H7NFpUKOcQ+HY1AA7xAA2jU7XzpGBM69rQie6JSIThqgVZaGoGlUQMY0dKepbE2gCvuMMUOyn9mA0hRt6acObpruDHUY8NpfxhH7ENQshRqjUBDvEB+dPGla48BPEAHptgd1tkAuOnyuuHyTcF1hf1dqN1/KC/nQw/13QkmH9+TauL7HBMrFWppBP6JGazowGgiSrPxL2VCrz9HRKnWPwm4CTVGRlLbqWs7AxDBLxvJoCZpcqVCLY3AQwhlRWe/v+Lsma6BmWzo0q9Wxe9543u5yWUktZ069hQR5chGVlhN0pRKhRyNwJLNj9jRV7duiZczoxM3RzwyVqlQiIykllPHniGiHNnIV6ImSf84SS+d96C0wOmXP1OuU/un0NexvCXPum7RjLsJFSIjWQAvQPboCfKea78ypSSDdGUj9apJimKSppeR3G85D0faN9y05v89oznakcnvhkV/GwlALRtpQE1SFJM0qYwkxyn3eudMpvsejXbkTedoIlqGCCKlbKRhNUlRTNI4BQ+NjCTgNO6/NlqDHKfEOflNZCI/dLyiEDn9A5/eAEZ9AUAhG8mrJjmhDiCZ1mFifzhfUYhJ9hNPmIrLSGo52wEc7chHsWE62xaiJimKSZpeRhIA4ApoaUe66IzfFaAmKYpJmlxGUjkxBqApRMqg+///mqgmWRUykihnfXOTNnEAwJkXm0ZN8hUqFcrLzOPMUSlEfrlFr4wk1yktUKxvHnWtPbB6oGTjwBveoLWK9MjKAMmmnn8EAOt7cNQkywDHDVNntlCqSUolBsUkW5haqdBUpEJ+WqAQtPDXH+tWFxOGt3Gq1VeHWKLtPBrg7DZ4q1YhMqZ3+P5Ze+E2gi4EuboGPiY632nBvq8PEFFy04WLotTuv/p+uyEoTEpn36pdJyh2a1+nZu/mUmUpFZqKVMhPCxSErrTXH6sLkenx0uIr97lyghVWkzS5UqGpSIX8tEAhaP7XH5v6Saha9tvDA/DRGnJXTtisO2m5GzVCVSkVnjxwGcBuGQSJDepB158OMMfmbEdUKtSoBXJePGxYbFAfGsa/elmD5t+OOXEItObb3UGbDs59NHnFm/jeauDzJWty9m4vPjEOg96DYWJfgJrYt9Epf6KXmlS4XRCp8GU0Ek6WWk10AQtaazs1RkaywkqFui8e1i82qA/tvUtOv3rdY0LXwNcfm0Kp0HSkQl5aoAB0DXz9MTSUwi7NfLxDTkw2jHh1pEJeWqAAdA18/bEplApNRirkpwUKQOt5/bH5Jud11PXz8/Pza3jn2aILj8N3Kx/abgYzqbAOGCmJQQOlKJ8WKABtxf/643+yUqGpSIX6aIEC0Fp7YeaztaVQsgdVlMKI5oVENOwS0fjeRPOrgFTITwsUgjbq9cfV2X4f3aeBc4M+o49zKnkRgxfsOvLFanXZrvJJhfy0QEGxeV9//A9XKsyLzuncXllZNyw2yI9OinXtzo7m7IWoVCjKq4gmJkdMjmhickQTkyMmRzSIMpIQZSRFEy9roonJMRf7H52lJyOavlWZAAAAAElFTkSuQmCC',
    '1604.06979v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAkcAAAC/CAAAAADRdid2AAAfFElEQVR42u2dd3gVVf7G35tGSMVACCwQAkikxCBdBAliFJawUQzSsaEUFwFxdRWl6eOKFVAEaT9AygK6oAiskkXpQZqhBFiyJISaRkISQpJ7k3t+f8zc6XMT7p1yE873eXjIzJlz5p3JycyZmfP5vhYCGjTcDi96CmjQfkTDU4IQEkfPAg03Io4QYiGAhY6RVMLTz45naLMQel8z9NK/163qB4gHqzCgHxWfv3g56+L5omo3zNi96VKNWszbt+VwLexGU7zdqh/yqisdSXpSdVJhQD/KXvt8y5j3N96odsPDX4xIrVGLmd8+u7H29aMFzfsAGX3/6Wr92J4fuVBLelL1UkEIAdE5jmJIjbYrwNYathg7lRgRWp6drK5VhJAf8JLLLdh7/1dBn9KGC1VPql4qDBkf+dfwsudb4xZ9a9/laOYrXgASj33p+nh26js12zDvrOqp0ksFHWcbFIXbxwCApWug620M+S27RtttNV6FCf3o8q8X7QCAyrwcVGaWsasrrrEDuMq8HFRm3ARQcalSXMWWfRO3bpvZHRTEwXo0rUokU3xgTGzrGAgAVfk3FMtl7SrtyKf7VqfibqXstWYBp2dwa7iTqqDCVRmKKgzvR9Y3koO3Pp4O4EDnxrP//cmR4S8SABefnb33H8ngVh998SX7iiUHey4WVtkR23T5119Fsg+us5pEPplnsHq5OJBlQ+5k9DkokCk6MEfs6wkA57qHv6ZULmtXYUcAeu9zJu7zNaH1P+yKTe+U7x4zZq3opCqocF2GogojxtmnkcT9vCLyHCHTH6oihJBH+q8mJB27CTkXvJcQMo8ZErKrp6YSsiqgQFjFHv3MgRtNNhLSdSohqR1+NWGcLRVHvm5WRMgvDQqFR8YfGBd9lzP/xyUpl8vala0ghKzu6mScfT6OEGLvTQh5YAIhRHJSFVS4KkNJheHXowYlpUD/1IsAEHx6LNDK9wzwYue+AIYzm7CrszoB7e5kCqtYgi/2bnKD2Sxt2e+PmXBjk4ormpEUAsR7bRAeGX9gXOQ2YP4PgnK5tF35CgBhuU6U5aSmApYX+BWik6qgwlUZSip8DP0lpLX1S3rGknkqBcxbyQ5egLdfKbIPzxaKYVZ3BOCPUkBY5UFHU/8ZmxZkyghJIu5Ekd8BAI0uiGQ6DkzwGwyRN1LqpF2FFdX0ox5RnTvG/+Vl/r2d+KQqqHBRhpIKY69Hq3xgX9ZrXcPe3AsBAKjCeYSK3xMIC0VVwthNdp8JmGbOSFsiLguN/f39/df/XSSTK+arVcobqXLSrsIKoNLfmbC9swK+jn/O7liWnFQFFS7KUFJh7PXouhfeWvZHGxwA7BYLv741Sp3UElZxVBowrVfvQSM84Hm+PRp2k8uUR+NiDXZ2q7GTwvT6c+cWrnlzHDN9Y+U4pZOqmwpDr0cXLqNkwfNtgFxg10FBQYt2vwNAmWItxSpB6Dl7YpYH9KNOf0oGgIptykfGRUS2BjvLc9aP0jYC901LOgP424AMxZOqmwpD+lERrACQNaIZfOtXAjjtW3YjDLDaABBbFSzLks8AZDFz0OxqGwAbqkRVrOzfk9UGvBmcVGh8v5GIg//y7acALGwhkuk4MD66nGD+t5UCSuXSduUrAJzo6kza4jsArA8DD/0PIJKTqqDCVRmKKvR/7j+aGImAp0cP6+2N1wn5vsW8nR+ceLXX25UH/9LgvsTDqx8Linq2kOzvO/+7NzYhfCjhVkcOKfx75+CHJvNVdgwKDh+4mpCfE0JaDMn9KTSo3WKDn/vl4gj5tdeMNW/uEBzZPuGBOSK5PSGEHEkMC0v4QV4ua1dpR4R0+9HJc//Wge9u2PH2QkLIxZZz5u4hRHBSFVTkHnRVhpIK42dqVZ23dfAjxbIx4LX8DvazjRrVv4sq8JB5bNduR1uqk1nV+uf27sq52vOSr/o8tuL6Phesbf0BwHY6KkzxpOqmgs6HNGo+5BeXF7jbxLsB77o7H1IvFbQfGdWPrHH/+pN7LRQM3Fvf3X6klwr6vd+o8Fs8ocqtBuwTvqzvsSro9ci4ef6706a4U315s0FazPPXRwXtRwbyIqWB7tS+E6ANL6KLCtqPKHekhQofAHEW2mPUThH7z5P1mR5xoNciGqB8Pw3aj2jQfkSDBmi+ERo03wh97qf5RmjU3XB3Xm1pJhDc0rF06TbQqrrXpUWn80P6u7VBLQs78a7zAt3tRwWbUvaTS+wn5Nweef0fGR+Y8cKkkU6q3Fj/7Z+ddpNqN1CJavZrWux/Z1B4IACMYq/+JQsv33hwaoR5igqXFZfZJj+gKtC1cbZ78yEzh+Mf7I+LxyKdz2YhynohiieTqmmz2g0UQ55FQ12CkflGlrCnuh+7nDfiIikaGpqizXxNFyJvch4hn/glqwl0QYX7vIh/TOb/vW0BAHI7BP4AEo+1AyRZL3BX2UJcSyfC7hcqiTdMi3NLIwMtILNWscuzPopCyKqWo9LNut2tW9H3Wbz41ux4FYEm3NcA4OXx++IA4FBvBuCwdK0m64Vezw1da554w8jwGg8Aa8ZFscvbf0i7D0FxW893NElQEwIgEAVqAk16Dzk8YCUA4FAvZpnJZuHIesHnCOESjaC6TCSy4FJ6AAAqc3NxJ9OxKUlPKYNiFg1B4g0zYzIAZBwa7VhuUWEFEIRcswSNKHkWOIjBagJN6kchw74vAlAczHx9ZrJZOLJecDlC+EQjwljycGS7L4CRzWNWSzYQlHApPZjY2SVi7gfvbXviLSsA/JL4R/mb79oUsmjwiTfMjTYA7FPmcd/m99+IAHDKJ8Y0Rb6A9fNu76kJNGWcfeMDsh9LCCGrcshfcYXPZsFmvXDkCBGk40gQDKNvNXqZEGKPyZFtwJdwKT0c0T7iJCG3uwy2E/Jj2wJC7JNGKWbRYCV4QF6/dW9LVhzBdNPG2YQcfqv7hNvOBd6dCk3eQ/aOXinCLEUJHBw5QoSJRgQR+tJ3d4Cs5xvLNuBK+JQejggYGAsEztq+BWV/HX4fYJm4YbezLBqmh/WdSeIVFePiPzRRT89567JG5jsTaMZ3WsvLx07h1INqxUxB0s2umT/uhyz98fiizcC3zyls4Cg5UeR34MCBQ40uSGr2xw6kXo0BgBjLdjjLomF2/GSJFK+YEf6Dv5mCLNHrdv25yolAU773P+ezArueUCtlkDxRohHh6GHAN7AVNVbYwFHCp/QQR3BwOv7LZMLw8jvnNIuG2bG6lXh55fWdgSZLatj92C/qAs147gciBq/7wFfpbcjKcdzkT9V0HBOHnMz4i+IGbAmf0kMcxSUPoA1uA4Ct4n41bSvHmd+NKvfEi5a3n1zvhTNeHUy6yfapPOwHhCFdVaAJ1yNCAIwrfOkp2QtKG5ABp1lDAACDmy3dF6e4AVvCpfQQnAoA2ItB6BKeAgBHMVDxHalIgnlx9rYjb1NBOYCUQwu9gB/rmaSm/NipXABX0InVIxRoWj9KO0eAgU1LogAUoZjPZsFkvQCbI0SYjsNmE10SX/k2yqK4AVvCpfTg6+zNBu7MfmYIApduzgCq5g0brJRFwyHB7LjuePTIb9EHOD/m5qSJE8auiDJJTcigX5sDl9PiHmX0CAWa9dx/bkBYULevCJn3PSEfP9kwuOPQ80w2C0fWi5/ZHCF8Oo6UxLCGiccFbVwNyifKG7AlXEoPR3Qd+c6KDf1nVhBCyK7H/rE0cbZVMYuGI/GG2c/92zCD+aE0ZgwhXZgzH2ve97XnPjt4oGfCDVaPUKCrKvScqcVlvagua8iVFmobXGkhTunBRreY1ZdLHnCM7a7fivapmQSz5rFZ1yVEeNQ8ttSTpHOsRSOBtZeD7Bazms6HpPMh3b/SVYIGKC/iXuxKOL/96XP0d0dnsrsXVfBGpZcXPTsexvfXtvBG7dRN72s0aKDa7yI03wjNNwI38434AOi3h3YY5Zgzh/nnyfo8QAXN60fH2ZSnpVG7edpSwR+BV4C7AjIy83tG4R6ia2k/YqN1YVyE357M6D7l2Xuj/ucay8pvfnj9zq1RdZyuHdu1e8DllCcFR5E7ZWmoBwkiK88Eeb8VaOj3/qoGFwghn+FrQsiZcHWWtaboq9hfXke61gW4VqPv/REAfGfauc/tb4zvjhta+Z9oIMg6aHQV2RSbR4zkaQuHtQXgDS8AHZ8sq6/GsqKG6Gv17KxGdK15cG3s01dbx/MX3eDXmn911NQLpETQvOSbXhj25cwlRt7Xch8SCsptqcay3iX6qj9dax5c2/hV0WK9lmbfaCWClsUGA3h46ecBBr7P9n1YsPBoJUvLMiwrQNJ/ywFQlX8NgDX7JsT0K+swz20u9ZeHfnSth8C1Hhi3rgYBQPjtVBjYj+7vLFgoGMzQsgzLCuzp/qt9w9yqg3Hh44Bl0U2XiOlX1mGeRV/l/vI60rWmwrUn5386v8iTuo5IUH1vwnSF88bPq52PJSJalmFZ93gfJ2RrQDIh3QcQQip8PhDRr7zDPIlLUvSX15OudQGu1WicHbPBTrZEXxKdPlPH2RJBnToSQsir+Nw8ntZByyIIAHk1vgsQ1LY50AAA/OqJ6VeBw3wQFP3l6yZdu36kBUOCpnvO5Ugi6P2z2YD1D1Sa+b1fgNFePdsRQHxqO6V2+2NHj6jOMdN2v8z5yz8mH+nXSbo2FgC6bcnzmH4kEZT42StXMj5KQCMz+5FgHv3/0ER9u+DgdLHDvJK/fJ2kaw/vA4BgpHlKN5IJmr5o72+v56CTmTytYOpCW1wRj70qxfSr2GFeyV9eb7rWFLg2sajED7Cioaf0I7mgli2BrKhOHjKPrVnMHwBQsRWoRwDkVIjpV95hHoCiv7yedK1pcG2nJX4AzoW2B8evmhsCQQXlAPYPKQKK9s30Mb4flcPRR4rBOcJblqfsBPBNc6BTAYAtIWVi+pVzmIetVNFfXk+61jS4dmh7AJf2zvcBx69WOE4fzBXE6Nm+7QqwoNVYGPzcv31Ev4jgiH4j/sPRsg6GNqXLu5vf/54Qktdn7s55O5oFPVEooF85h3l2c7m/vJ50rQtwrTbP/ZUzF6Ru7Pi53cHTVox6oklwh6fmm/bczwtieNozg/ad+Fv/HI/iabNvRTPXuJuX2oacCg4PtPD0q8Bhng2pvzygI11793CtVmfn7OGwXhGeNI9NIujmz7c697J4Ok+rN/2qX/sWOh/Sg+ZD6k2/UroW9wB3pDf9Sula3BMz2fWmX/Vsn97X6na+EcqLUF6EBihPS3laytOC8rSgPC3laen4qBbmrSm+7utLKm1NZVM7rGvym4/R8HJKMUezwu5lwPuj7LXPt4x5f+MN6fryPvX6PL/TrV1m9P0nhJjj6MVuNuHZkTuCmYtnXzrzw3eFU9nIimnvzTZvft2eEUoqofl32qMYorB2UYw9c0zG3TGFko2lgKILmGO1zqV3o0fP77Q88mgf+T4hx6OvE43gQ/f8jggpbZNANAIznXOQ/orXqiMdLVFr75IplGwsBRR9XWYcXSQbDQQieeRx287VQJd+U76DdvChOzFfUaVh30VK/e+eKdwqBRTdtmhhm3CRbDQQiKzX0uG6sqG3H4C4n+5ADB+uu2NKNzrcqpGSSt37kRg9dMIUshsquHvalJFJ5T0Ii+VApNi5VOluzVVW18OV8c6n+kVVcigANK3YoyF86HqU/2ukGd9pHejhsTFHdo8ZUyphChUYRQV3T4fBqASZlOyBHwY6ih1ApGAvYufSpY9EtvsCGNO842pZZSd6uDLO+VTPyCsMAoBQXNASPnQ5vpyi9RtM1dHaafDjXx49THqKEJlhpxKjKHf3VEMmE5JEe2CCL+aASH4vEufSouYvEkKqHsyRV3amhyvjhOkyH5JBHi9gGiGEpGKORvChW+PsEysJ6ZlANAIza8xBytFDKHuDCjaU84dqyKTSHoREpQOI5BlJiXNpyITvS4CLkxrLKjvTw5dxwvS8kcCP+Y5RpiF86GrY1r4A4/NopbU9UeR3AIAMPeRi/CebX8C3EyHcUIE/FCGTAwHEOwYH0j0IipOesWSeSkERvxdZjJu7fiLWT5e37UyPsOxB/R/cGIKuEsHg4cPFtrUJKY2M70eL/uplQj9a9UkWGvsDWN9MbZM2A755wcEoOjZU4A/VkUnpHgTF9uWrBz3WW7gXWTQdsnhCuS1E3rYzPcKyMN1/dQ2YC1G5gP2cnrS34vX33IEPXYyzFQ1vAZW2W97BRvaj617K6KGQKayGURQZjAKQIZPSioJiARDpYCRlDU/qf/DicIXKzvQIy/T/bB7aJAcAstED2sGHro75r74H4Gzj91pPN/B57cJlBfRQyhSqM4oqAKIAmQTkFfliIRDJ7kXecL92X5+OVWjbmR7lMr3CkpAJABlhXaEdfOhixC1atGjRopCYRdO14zLV+1ERg6lmjWgmRA9Z7FHKFMoZRQV3TxVk0maTw418sRCIZPcicy6FZeLmLoLfGVfZmR75QekTLPI4/WwGQLZM94aG8KE7L7RKi8Fzme6DmSpPj0cTIxHw9Ohhvb3xOo8eHk8MaxD/KSFyw04Jo6jg7qmCTB5hMUcp3MgRlRwQWcntRexcSgghhVHlQjVcZSd6uDLO+VSH534B8rgxLq3gjTFWohF86Ob3tYkPB4U+/jErxU0w8644SDF6KGcKVRhFZwAih0wqV3QUC4HIKy2UGy5Y/J5a2870KJTpN/8od2dpr84WreBDOs9f6yi62AWfDIui89joPH+34uNuxSXlUaCB2pXP39NiwPGU5Fn0N0lnILs98LgW40vnZ9PxEf0ro+MjGjQAwHsOgH70PKhEPw8/O/08RgUhJI72FxpuRJwe+djo+IiOj2jQgEbvj0qyAAABkSpvlgrT8hvocFemWG0d60d5G/fva/qyX+npoqnPeAEyf85r69Y8pUk/ErfrknuoB1mHOg2Zbaf9X2dtXq+Fm6dIIsC+5kqZ/ZX7tf3efxojCSEk2f+lKkV/zv7DXbPqrIVYrTZ5j2XkbMkTM61kbaJ5fkcSAfbXThFy/bEUN1QonKl0jCaEEDIKmwghxH7strh8AN+Pcu7GikqysbTdhLvvR2wTORNcOvwaVNOmH73vW0wI6T2Rkz10NCHk/nDT+pFUwE9fEELIyXh9/GlbYc+waixA3cNqtbIO9XSsVmrb+ev3xwH802baXU0q4Pc7ANAmQ595tdeYKegs/lqVWYyKH1hQ0X6pjGK1cNW285vQhwB062VaP5IKaPbFAjvw0yBd+lH65sRx4CxAvxqcPHn8J5FPVQFA8se/i8hUUKzWSUjJWfJbqyvvfvT2BfOG/VIBL7R6vd9/t637UNtxdjr6fLd58eT41ay/e1wSId/Vu0Uqo98gFwkhA3pslJCpdRar1WZ8JCFnb6H3p1Uko9l/zBofyQVkPQK/R62a87QNu3V/xH6rLTvnMwjApuhQeHfehNYAkPmsslMnxWqVQ0LOFuPwMC+0emK8WRZacgHWNn/z2j84W+v7WlBUVKevox8XpAlqUADAJ7Qap04l68/qsFrOiRSAzDNUUMz7jHLWo9IYV7YeWD9a3rYzPcIynbBaiW1nICIjAXTKOGRSP5IJOP3cF5+eeXzXU3Y9xkdJ5Uv5hVfzz6Dkt7fg3KlTyfrzrrFaoWeoEKvlfEY561HIsVpSpobVqukRlumF1YptO0P8wgCgvmk+ozIBr8xrhDbJc478rMe82gY4xS80nrywcfbXTwMUq3UlROSsz4OlTFduZFI/kgooTnsUgGX27guDdLgeNcApgjvsPf23vss/XPl0tVadFKtVCiE5W1AOJF60AshBN7Me2AQCCsqB+pYSAEBkjKb3tSLG7DzSuyADa/MZdrXlm8s2bE1mUpoqkLIUq3UWAnI2v0UfYGLgLwD55eW2ZvUjXkB+iz6A7/AFAHCtoK+Gz/0nk9oHN4ifTQhZEjAj8w2WXS16KLJ1C3+fYbeVydS6itVq89wvIGcZfvVIt++PTRxVbN73NU4Ao6f8+fF7Ti57IUsnnjZnF0lsAAAo67H8YcB+/Pnhs+GclK1bWK1G89hk5OytvfndY2HiPDapgPSjd2K7W3TnRbYsY4byi//9E3APYbV0PqS28yE7n8kCgKpfBoJitTRc5mlbbX2zQ4+A9N0DJoFitTTcuXLnZZY1b+V1j2G19L5GeVrajygvQgOUpwXlaSlPS4MGKE9Lx0d0fEQDdff9kWE+tXU47MT7nu9H2Wv3Hgx6ptUIaT8q7zu5T1xYAgwmX2sLQSuK/e8MCg8EgFHs1X9s1+4Bl1OeNI9DlwooWXj5xoNTI6CHb5Z7PrXVwLOuBtOO7r602n7vdziH9nOsiADgO9Nu3vd+iYC8ERdJ0dDQFH04SLjjU1uNJ62rwbTj+b60oji3NDLQAjJrlWNF7NNXW8dHmXiFlAiY9VEUQla1HJXubXi+2tIg50yq9vCsoB3P96UVhdd4AFgzjvvFNX7V5DutRMD2H9LuQ1Dc1vMdDfM5du5TyyGvyvBsVf41ANbsmyzYmnETQMWlShUIVg7QMu04AWj5yh7iSwsAkwEg49Bojx3AtaiwAghCrlF+2c59ah3IK1Tg2YNx4eOAZdFNl7Bg69EXX7KvWHKw52LhPhwQLNealKB1AtBylT3GlxYA2gCwT5kneLg9Of/T+UVm9hyxgP03IgCc8onRc5xdc59aDnlVg2dJ9wGEkAqfDwgHtk5NJWRVQAHfCAfB8q3JCFo1gJav7KYvrbbjbEIIWfe2YCFmg51sib5k3jhbScARTNfbnxY186nlkFcVeBZoAAB+9QAObM3qBLS7k8l3aw6C5VtTI2ilAC1f2XN8aR2X6neEM7fWj7RgSNB08y5HCgIqxsV/aIwvRLU+tbyTLBThWfFdlAFbOwLwR6mCb62gNTVjWokvLV/5N4/xpWXjJ0uk8HEJALotyzMtIZuCgBnhP/jDkPHRKh8p7ip9b8shr1CEZ6XvE6CAt/IQrKA1NYJWAtDylc0GaGWxupVg4fA+xvnYLJ5WScDK6zsDDfKpqdanVoC8KsKzjruPM6dxHoIVtqZA0CoAtHxlz/GlZaJyT7zwJVhRiR9gRUOz+pFcwPaT671wxquDAdejan1qhcirIjwL1CMAcpw5WHIQrKg1CUGrAtDylT3Hl5aJs7cdV76CcqDTEj8A50Lbm9WPBAIYf9qUQwu9gB/r6Xlfq7FPrRB5VYRngU4FALaElPFUrg2ATXDX4SBYUWsSglYFoOUre4IvrehS7ng+yG/RBxjaHsClvfNN8yzjBTD+tOfH3Jw0ccLYFVH6PfffjU+tEHlVhmfz+szdOW9Hs6AndjjA1sghhX/vHPzQZDkEK2xNQtCqArQ8QeueL63Wz/3bMIP5oTRmDCGVMxekbuz4uXnf13gBDE/L/i3GGuFPixr41AqRVxV49ualtiGngsMDnY1MWAhW2JqEoFUHaHmC1h1fWq3nsVnXJYg+pp89HNYrAibOY9NKQF3gRXT0paXzIe+h+ZAUoAX1pwUFaEFnsqOO+9LS+xrlaWk/ouMjGrV5fBRHsQ/VPzUY9/HEVX2mRxzbj/rtoR1GOebMYf55sj4PUAE6PqLjIzo+olEL3h9V41Ork00tKF9bx/qR1KdWE5tajXDYWkfViu1gZXyt4VG4rLjMNvkBXt/yq/53poXrw9OKfWpds6nVB6vV3ZdW4+/9EjtYGV9r9Pf+vMl5hHzil8ydkZHvE3I8+rq2/rTKPrWu2dRW40nrYujuS6ttP5LawU5Z+u99+/fv65dpVj+a77+ZkDw8wv1dhlYQQsYP1ZXLZn1qXbOp1QerrSW+tFCxg5XxtUZHEwIgEAWO5Q29/QDEvXQnADrOq2V8ah12skKfWjWbWr2x2mp9aT0Lq5XawZrO144oeRY4iMGO05kcCgBNK/boOa+W8al12MkKfWqVbWr1x2r/Xp0vrWdhtTI7WDlfa3T4AtbPuzmmAOYVBgFAKC7oMs6W+NTGJRGRT62qTa0BWK1zX1rtsFpNxkeKfrQivtZwnpYcfqv7BG6oegHTCCEkFXN04mlFPrVBkPjUqtnUGoDVOvWl9TSsVsmPVszXGh89563LGpnPLpTDj/lWV6bTfc25T62aTS3vJAs3sFqHt6ygNTVjWokvLV/ZZF9aqPvRivlaM75lRK/b9WcW0wlmeJ1KBOs5PlL1qVWzqaVYLZzbwULC15oTDbsf+8Xh/FnGXJZC9ZxXW0OfWt6mlmK1zu1gZXyt0WHtU3nYDwhDOrMc2iQHALLRQ8/rUQ18akXwLMVqoW4Hy/KrPF9rRpQfO5UL4Ao6MXosCZkAkBHWVZd+JPGptZVC5FOrZlNrBFbr1JfW47BagR8tw6/yfK0pETLo1+bA5bS4R1k9089mAGTLdG8dnvslPrUsycr51DqxqdUdq63Wl1YzrFaj72u8Hy3Dr/J8rUnf15777OCBngk3OD0b49IK3hhj1Zmnrd6nVgzP1h2sVqt5bFI7WClfa/g8ttSTpHOs4Nhzd5b26mwxjBfxRJ9aPbFaOh9Sn/mQnuhTS7Fa1Dqe1hN9ailWi9o4k93zfGp1xGrpfY3ytLQfUV6EBuVpKU97L/K09J5GQ4ug9zUatB/R8JT4f7oJPwguL2LJAAAAAElFTkSuQmCC',
    '1604.06979v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAd8AAAC/CAAAAACa6IlqAAAcS0lEQVR42u2deVwU9f/HX7sccoqioH1VBElMRQwRj0xB0zI0TC1Rs9TUPFIzTSutSLvs8CqzRP2JZabpV8u8+ZrhEeQVCqhJgnhxyinXLrvv3x97zc7OwsrO7gw278eDBzufmfnMa+Y9x2fm835+3jKCZA+wyaVDIPlXssZrRBQhHYUH0iKISEaArLE+g0UrXhyiZCTdnwGAEq1a/SSJWIV1/q1IS0vL1k9dT0tLqzBZpvjkz7+brSDz6I7rgrt3roNV6zed1RAHs/fcRiqs82/RjnlhHe9oJ/J7dXttR5HJMre3jvvWbAXJK8emCO3f1W0fBzIH/NjQ9UN6f9KAtdh7bisVRARquGXF4GPtz3UvIsNonnqN5v+gGPPrF2FPwzdurXgiIsoOUxHRz3i5wTWo+/3NIYxrwTVm99xWKqx9/roE9/o/zY2B7jWFi9G8gkua/051rO8k+NP33WlyANFnv2x4O+a1ty1bUHdAOPbcViqsb19N/ec4AOCPfuw5expD46p43wQAkIW5N7yOkcdyLVpuj/1VWO/fGLdNGv/21ZUozqSrAKQuNiykvl6lex5kJOl+ouY23+8RtQV5qM28C6Dmeq2xHODGb9fU+oWytCr2dnUHAFVhDnsOd4VcW3AMr/tULklKVGQbHRD2ntejwiIZnCqs92/TMbtKAZR5yrQOjBtZmfn4Kex4u/rohAnfAwASPv0zZjIBwOHov6oXLlECwLXnYxM/TuDVvSdDfWMPfnZm8svqjd+c6r2OKQeKBQmee57I0C10WivoeG8AuBzuM4c9h7NCji0A6He8LlUrtni5fhTGOCCme163CgtlcKqwsomS8wGdwDdEtDmPXsVNIvq6TSnR4WbFRJ2ma5Z5qtd2ogwcJaJfOhYRqWeOJ6LLnolEtJzn9tVjg+KJMvBaCtFmtyKmnI1+l4nmP6oyLHSUiGjABs2KEaOJNcdMhSYFRBQfVkf76koEEan7kf6AcOx5fSosksGlgofvG/2CNgEo8NVMlS4e3RQYLN/GWCLreSDAKQ2oejWmOSCbse0oMDl0AIAYnm/QnqkvAgFO2d2BRyqzmHKalVcAg1KuGRZKA4D8ZpoVPZirp5mv0LQAgHd+HZLyUlIA2SRDAcee16fCIhlcKhx5+Aw2ddHFkIvdtFPnS51PAmh5lbFEFzng4FwBpNwKBoBg2b4ncpNjedq+sWm21RWACyqYckaPkmVdTEKpkSAgvynH6hXmK+QoqMe/vfxDuw5+Zqp+mmvP61VhiQwuFXx8n3zJcSOODNFOZMPXxcXF5Yc3mS9RAAAV8Lfmp9z5Mq7AyybtYRfjLRrkqOP6bm3RjyUIcKnlWF1lvkKOAqDWpS5Fie+5fT34JbVummvP61VhiQwuFXxcP62Gb/3ASfd1rTNa9GTM2zSFuWQg7gGAsuZhdECFPV5/DHIWxf0ViJOAWiZjLuBbxsNWSnzrmJnhunRp8ZaFUyK0B4Rrz22mQm519yKAKcUvj9AVdP9PAgDU7AVclECm0dI9fJIA4AyGot0jfwJAlY39q5dTvnpiIJAPHDllfHLm8rCVgrr8m74daD5vdJrugHDtuc1UWOvf9MsEDH2o3B9AKcoAlw37LgJY0w549B+AACiUAEipAtzX/5QJqJaPGQ5ZXEIaQOt49rB2W0oASqgYcpxcawGkOlXleDMEAT3Oa1ZUVsBIqrkKTQsAnA+rS9O6SgCKProDwrXn9amwSAanCqvejy4/5e3R8yui5buIPn2yhWfX564Q/dZ38ZaF+4noWvv3l/5Op55p1jw6OX6gh//zxURHBn68PjpWQUR0YsCqnQt2wOc5/t6P9NvyG1n8Zqjno7PJIGdXu+UHPjg/q+9bx40EJXQmIjod7e09LN9YKmeFXFsg6vlLHe9He4Yu2bb/rTX6A8K153WrsFAGlwqbdJHfvhckAwBlqr+3ycw7JUG6h/7twi7qSy1butq4f18rR3VF2cWZyliNG1WHQ52t3d9bva87me/fL3N1vKro6GJ0QNh7bjMVUvzGyhurrdWxxG2JtfEbtlIh+VcR8d//WFdD0dBEV2v9aysVUnyO87rpKqsqUE//0lW0KqTrFziaPtea1Te0ieIjvs42KiT/Aqhwt2btSjd+4idtokLy74MdH+sIIELWWJ+dMu2fKIUJbhFoxNeuZJD4I8kk/0r+lQwSPygZJH5Qej+S+EHJ+DTL4nPyf/6H/J5tZzojM6uwt78FFVi6nNZKUwubDpKcYyf/qj7ctvIledLwgctNAriSfziwxxK/Wbqc1nJ++O5pO/m3OK6sSjm7k6FA/d9LSvkcH4H8kT93PaODWr3lZpV62sNWtq/qid+oHRVSTERUOWRAZYP5PzPLGRF1THtytH34wYLZBUSfOSfoC8qHvKug76Otq7WBogoWvBKOHMbRmXOR6M7AJCvDW+pRswznND9uOU03mVluoX/NLJc33czyw+zk31UuPxEV4DH9IX3uBSJ62EcQ/1Zfr13F9O+vK4mILgy2QkX99+e8Tx7rofnVZlTc3C783o4ERwxbEwB36LH033adA/CjUhAxTdobT/9ZCQCBmTb9vrGtSv8kHESbAWXuXZTcA8Ck4HSMnm6mqvA2AEXuXTOcIAdiCMCA+NnNxpY/D5zCcN30t16PAujZVxSNozYrV6uBX6Ns6t9j6AB9ePox7A95aMPXX/klMik4PaOnm3kqwmcKEBf00DfgouW4EEOAgfjZ0ZwAxYqe7+iUHQu4ueSTt66Ko/E7KeD1yL/3bv3Ipu2rUOzW/TwHXyJ10KiTOa23Myk4AzKom0nhTxFRjeMHnLScKWKoMQbiZ6/nL1HyovDp93QTJej3uYoy2/xPkOcvERk9fyn7MTj3V1ijQm7BFa6Pw66CHJB5XuvXOieGQcExkEHdTDQDAOcm4KLluBBDAGAifvaz3su3Zo8r1E6UIXmMHAFDXqkWxQWsCHxDfmJ4rk3ff9udK9D9LIAfAHQDjCg4I2SwG+u+z0HLcSGGAGCE+NnxM17Q1jZPJ2v4KXf4+QHoHv+HGD6vpE7f23LG9CMjkuQ2fP4Oxnndz3MYDADegBEFZ4QMsgPaOWg5LsRQ8zrPQPzsaS3Czx7W/Grq7A0ArkgXw+U7bXlLBCa8f/qQLa/f8YsPKZw1j+p9jtP0wScMCs4IGWRGplAtuDhBs4ihGcTPljfAx2uTnQFvaBt1jt0qNLxlSxG4tyy9PwBZ7NGrUTa8fpsvz9+i+XUk5U3DN0YGBcdABg2vcgQgrwZctJw5xNAU8Suy9WOw+uzFfAA30V27sehrCgB56Cm0c4uq4SorBwD4Bdu0f3/GgjeOA0D61HFLAUBRBoBJwTGQQe1MoHsRgN1Nq7hoOQ7EUPOqwkD8lEoAhe0et/FRbBr1W1vgRnpEf+3GZrgfBujw1I4CubUGNYB2151iVgPA7aIBNv3+TLQ14N0/L3zR/isVER2K8vQZGs/i/3SMnmFmweNLDyzf38ZjSDEXJ8hCDHWmR/ySor1bRJ+jiuAJNv/+/NIXp072HpZDuo2d7rnr7IzxZYK8H9WMH9Las8uIVaRVUz3xld8vxE3KtkKFZV3kqlNX1B0HmIw1x6DgdMigwe5e79j0oqePu4yTE+RGDLkRP9v276dcoNAQhvaSxMLwELH072ecqQwJl1kZ/iDFb0jxG5JBip+UTPKvZJJ/JZP8Kxks/T4p8YMPLj8oBxBJjdRitX+iFCa8RUrjM0jvv5L9C/gFLiu74+REtcqHGj4MrEQpiNm/ud8nnvIYFTC24f61I6WAxsEvmIhh8QywTf+RGTuDkRam5rGKUvgX8QssMWyewSbxdahjKG15A/L+QFTZj7ZuPAZMVsTqT/fJvsucsDRJFGI856wbL9j9GY2IUkDj4BfYYtg8g4Dfr4yBBYiMUkDj4BfYYsTzfdIIWBAhpYDGwS8YiwFP4280tImSCkbzyAAs8E0p/Jv4BSMxpjyDfdtXRmYAFkRJKaBx8AtGYoR9/2VYekdn0xu9yCiFRsIvMMWI5vm7messER2lgMbBLxjEiOb6vSNnAAsQH6WAxsEvsMWI5fq9eoMJLEB8lAIaB7/AFMPXQbHGv6VQAED22DZMYEGElAIaB7/AEKM/KDqewf7357MfpODoSHfl7WRVJIC5x5aFX+zm+eWZn7SpMBG1d8bATmmRocB7g5bKntL7Pf71T0PORee/GdF5xFnZiNgebh06C+ZfxC8405cWDNnoAI0M3yOzFP4bA1cKLEajRjG5INUzqmPkPFF0kTOABfBJKfyb+AW2GF4OD0/+leI3pPgNySDFx0om+Vcyyb+SSf6FxC9I/ILEL0j8gsQvSO+/AuWng70C2iHWeHe1XIpv5yGgHSKNd/993U/63y+GhbvdSHpSmPOsfM2NnG6vtdJP06Y0D4dF7vaKv7I4oN1O8e68xF8RVQQOM0y0AuD0rlqY+Pax16j0OS/9ePyKqBdUtCOkwKbjt6MBAe1oVPHuq5gTIc/e6jDYX5ib6Xuf+KPp5vbjM7TxOcsT7sox5st3v2lk77/iindPDmAGa/jO+niqQO7FvseKAY+IrCva6bgQTwB9tlYK4l9t9Dozsr22IA+1Wdo+flVWGWp+PmWygsji3av/O04sZ1q7GgUAD+RrJktueQCAz70UAfyri15nRrafDPWNPfjZ6ZjJBOCr4QmzX/nMb4SKtYLY4t2/nGv8MeLCqs9XCRTbeSKnFYCLjtrxRF0dSOOiK3ZqXzED2g3R68zI9scGxRNl4CjRziYlVBu0gK6ZrsBbvDsf7avzm4h6M9pXwdvUtDvoulDj8xOdxnzdz+5diYhmYYX949sZ0evMyHbP1BeBAKc0YEeQFxxCd+iTc5gNdxc23l35/STjgh/GyTDSY75g9+iaKYP1+TSWXcoFFH+h1t7xV+kdGdHrRqdIFzng4FwBNCsC4OhVf7i7sPHua19lneEhANAzrkCoBHWLfX7WZwGM/mLaOuX3w5Ja2vv5u9nRXPS6RpsKmFWYhvJji1BvuLug8e6XalqUlJTUKkvKdY3p4wDgKVh8+6Y7BxifM+avTTz2eh662/v6vSNnRa8bRbZr3jNmr/HN/fpZmAl3F0m8e8GtdwBc8n2ng/aOHF1a7gwo0EKgN6QLP8iRJtdniWvfHsj2t7d/r95A9/8kvAyg5nA0mlTDOLIdAHBsQLTRNGMFdrz7TG28u4c2DrrIzcVexzMiAgB2Ba/Vbbb7OGcAl72ECdlN+mONDPhlrFbNiZXxXig9vsLRTv7VB7R3hMuGmIshwJohQPeD0Ee2K2QASKkC2k/N9XD16OVlCHfXr2Au3r03lA4ACtt3PmvXo6qqKNNv9rnOAK4nrncUwr1XJgyeCao8+ZZWzb69N72wOuBF+3x/PhPtB7dnXxjTzwGvM8fYNwzFf+qZZs2jk+MHevg/X1z6qF+Hdi6OY+6ZDspv5aj8fH9/ntHHw+uJT7WbrX13dcr2riuE+f6szeMZohufPy3q+Pk3BuWRzcfn5zb9mPwcke1VvTb0AdTnJsbEcqzAU7y7Tfp/LyV7920FcfT/3j1UEtpXJsb49t1xmqRM6w7+Cql//8GLbw9NywYA1eGhkOxBDHE583mXXm4ZRwfMlEnXr8C+tZGagqyqtgFyKf7qQfWvFF8n8WWS2dgc3gcQ2VjVR4pVfKRoVBBRhHSaP5AWYeX3Den5Kz1/JYNI49vL7jg7qpUOHYCiQmdZdacGVJ6ZVdjbH5AG8Bejf3O/TzzlMSGsA3D5heywQZ9Z7NRJM3UBick/HNjjL3Kggc0rqDfccqmcJ1D0Bptf4AGmqKO34ww0cWfq8Vvuoz/lZ7ys/12EPbYEGvjoP2LxCupxy4jOBd0RBb9gJUxRH7/goom2oQ8nD76PEyb67CO4Xx5BQKCBxSvsPRAP9IicuxMi4Bd4gCnq78dWvzu0/3212sLQmIAG31lGk9v6OQOIeLnSTQAt+35Obw6PiD1XunKLgw3i61SLRvRncwbK3LsouWeGPFAV5mh+1NwmFriAkqRERTZEPIC/KsELAB6q+R3C8wuww/gbygluvdicwf6QhzZ8/ZVfoqHkmz5+j6wExrUNjr8c7jMHAK49H5v4cYIRuIAVW7xcPwoT2QD+RrxCQbEHAHjhqgj4BV5gijpaA6kYXf1qgOtVE85AHTTqZE7r7YySkpZTiUgdnEdEEaOJ6LJnIhEtxx4GuHAlgojU/fgbwJ+P9pUxr3AV84iIUvC+GPgFK2GK+vmFyvkLtlS/pGJzBjLPa/1a58QwSrxe3lkJZE/0BeABAJNDBwCIMQIX8lJSANkkcQ3gb8wrVMNZMzxKFUTAL/AAU9Tt3yuLA/rPS/5CMzH6bljWLyc0nEE3dskrpT8B372kf3lOHqhrvZ0vdT558uQfLa+il39o8LyjU8EgGjQzWMbckO3bzwDQc3eBZsoTKgCohSdEwC+wxPHv3x5tgA+D3ks14Qy82SWBT30LZamv/sSAlym44JL4ntvXg19Si2kAfxav0Exz4VbDCyLgF3iAKep/P3Lb0m9isjOMOQNN0A2zZMbIC5nP6FfqgAqYgAsZrkuXFm9ZOCVCRAP4s3gFr9Z5AJCLXhABv8ADTGFB/0KfhX99yDmuvlHJ8Dbrjxs6Gts98icAVIE5Tn/6dqD5vNFpENEA/t2/0fMKRdWAbFgWAGR6h0EgfkEO/NJEu88McbbwbynKNJ9V3D4+aDyuvqKMPdI+HKd9568Nbq4AZHEJaQCtQxVznP51lQAUfcQ0gL+WV1jlqN3K/EuZAO2e7wBB+IW7M2dMf3Gjv3afDeJs8H50ZlQnT68n3iYqG9rEo9nQiwbOYH+Up8/QeCZ5UEtEtzwKiYhOR3t7D8snOjFg1c4FO+DznAFc2DN0ybb9bzE+NeuJhgYCDTy8Hxl4Be1WtkekFy2YoBAFv2AlTHG//IIpZ2BUcrOd8eK3C7uoL7Vs6aoHF8pcHa8qOhqxY9YN4M9L/z6bV8g/UNE3VCaS/n2rYAopflKK35AM0vjPkkn+lUzyr2SSfyWDxC9I/ILEL0DiF6T3X+n9VzI0qvj2OxXOUCo7A6i41URW075JnaufOZLXcXyL3aNgWwjB3iCDSPIxqMmBd/+eO7rpXs+nlwEoXPir77j36vTv4rxlrdNj2+0cZWMIwYYgg3rLzSr1tIdhLh+DXY3FK5x4O8rHHQDGy3nsP3ofX2h/hafU3UHxS5iKiFRRYXUhCPeTVcF8JZx18NB/pJ5zkejOwCSz+RjsGl/H4hV04/JH8sovTHp/y3wZABQH1zPw4feD5ADksbNQB4JwPxCC+UpsBTLsD+gGPLR6QQLM5GOwq7F4hcvr/dxloPc28xqf037Qb+fDAODH+gauz9TcNsL9wROCYH+O4c9KAAjMhLl8DBAQppC/AgBbpvjz+/1qMjQnzNFBgIElMMqtoLGgXQcBQDbHgCDUgTZoYQWOegTkGNqsXK0Gfo2CGPMxzAaAzD9eAL/+HeW5rRpAahcHBktglFtBa68j6olPkxSI0CMIpmgDG1bgqEdIjmFSwOuRf+/d+pH5fAx2NWNeIRCAeu5yGd/8wjTsIKL5/xizBIbcCnrb3gxAq6/IkFPBFG3QQggGWMGkHgs4Bm6QgQ9+IfsxOPdXmM/HYNf2FQevsPUtq1RwNrwnYTOgvBkII5bAkFtBbzFZP07vmDdnBeObiQnaYAIrsOsRlmNQBL4hPzE8F+byMdjVTHkFxdsz+ef3+wYdud1mfxRgnBxBn1uB4YKxYylxzNIpzQxF3TiTKjDTL7DqMZuZwR6JGVKn7205Y/qREUlyM/kY7Nt+BljJH36V+YH3/kHZZPV32PkcmyXQ51bQ2Q4AkEWuL/+LsbI3J4PAhBVY9QjKMUxb3hKBCe+fPmQmH4NdjYNXiA+wxfgbLy6Jn+zhAdTDEuyKAQA8paOxNk3Rp65mr8dOv8AwITmGsvT+AGSxR69GcedjsKuZ8gq1vw+GDfr32zx5ddYEgIslYNolzaOxHN1ZCILpegxYgW2Wcwz8gwyuMs2F6hesqTxi7dq1a9c2DV4rSAIkJq+g2dVL97xtEr8xGamPs1kChVKXW0FnqvElAPDN5DYGBIEDbVAqjWAFdj2WcAy2AhmcYlYDwO2iAYzKNfkYBDAGr6BVc0dD2/LOL1Q1/5DFEhxn5FbQLdUtYeqHh36bN6pEjyAcMkUbTmsgBB2scIqjnvo4Bl0dbJCBh/ej6omv/H4hblI2GSrX5mMQ4P2IwSto1ezFYutUmOsiP9KjZf0swbkwSj2rCgvlQBC41tOnX4DZVA73xzHw0r+fcaYyJFwmkv59Nq+g2DqslTT+sxS/IcVvQIqPlUzyr2SSfyWT/CsZ7BY/GSFrrOpl0H0PFaEwwS1Ce/1GUiO1WO2fKIUJb5GQ3n+l91/JHszx+eu0cu0ovy1a8/KksTZRA4REBNCo+AXLrGD7ieOtp7or0tMiYltZvyPWJmpouJkgAuwUCHa14riyKuXsTrApv2CZpWIsEdHdsLY5PDQFLE3UwHt8HRsRMEmBYNf8C7MLiD5zTuCRX2jwieECBwDwXnjrDR5OXCehboGX1x88fuLE8cjN+hQIHdB0s8N4lRBitm48BkxWxJoTZ7/7MxgjiR5v1N93WIiASQoEe1prAuCOIhvzC/dlZQhiUguUkaQJxqrNz0dllo4+0BfrMzDoczIwEzVAcETABikQLLex5c8DpzDcxvzCfdlOhzcN1AIOR/9VvXCJEjjQo9XSD97ZO2SRAmAU6zIwGHIyMBI1CGFsRMA0BYI9zQlQrOj5jq35BUssA6NycrIPTg/Yy6AWfulYRKSeOZ6IqHOrC0T3egxXk6FYn4HBgCwwEjUI0b7iRASYKRDsm38heVH49HvEI79ghX87r1q1asOJGiIiCutORFTZ9h0iogv4HxGFTdRkM9vFKE70+ouINlCJ11wiUnl/TdRnABFRloD+rWmXzYrJ6ja4SiD/kvrvoc8U1CWOx/xl9bWs5oFFLaTcCgaAYNm+J3Tlg7B/tKH4E//QroOfmcpAFnKTY3lp5llhJoiAUQoEe39TDNra5ulkB9vyCw0wbwD4W4MmyJ0v68s9PTMYxboMDAZk4YpgmQ5gDhEwSoFgf2sRfvawrfmFBnaIBeIeAChrDMNZlJV3YhTrMjAYkAVGogaBjI0IGKdAsKcpHq9Ndga8kWFrfqGB1sMnCQDOYCgAKAAgEVGMYl0GBgOywEjUIJDpEQENMcBIgWBvqz57MR/ATXS3Ob9Qv5UacX0aasF9/U+ZgGr5mOEAkJgLVMaOGsks1mZgMCALjEQNApkOEdAQA4wUCHa3plG/tQVupEf0549faOD9+eKydI+Lg8I+10wd/vJ67tNjJwIjPaYOabG/xxIAQMSXgW4bh78jYxZ3+rir14nIMETtnTGwU1pkKND/f6+ObJvcd/Vru3cK41+l9hC6degM4IXMOAAIEaRXKX7Bmb60YMhGB50avTjxdJHfKQnSnDM9g+NvlHdyNCpmZmAw0AyMRA0CiLcKEeC7fz/lAoWGyBoFv9AzOF6K33iA4zeUtZAMD2p87JFhV/Y9e1k6vg9qiJoKDqiVy6X7s8AqbPVh0EHgj46SSfwCJH5B4hcaOb8gI+kcl+7Pkkn+lUyM9v8leeBLxlAK6QAAAABJRU5ErkJggg==',
    '1605.04635v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAdkAAABuCAAAAABOlDanAAAZm0lEQVR42u1dd1wUVxc9SxNhAQURu2DBjgV7Qw2KYuwxFmyJRo29xc8YWzSJRmOMiSVqjLHEGGOisRtjj4otigIaiSB2RJEiArvsnu+PXXZn2cLOLkqS39w/9L2ZnXPf3Ls7783MORcZIdl/0hykEEiZlezfZSRDpCj8xyyEpIyA7NXMtS/RzUuAlhXVAqQwHMsoXY2leVYyKbOSFbplRAl7qXHp1hzkVMD+6GX3XowaBODzI3QIH1PYY15T/nU8m7jJOnfPBsS+tkE8dPTiFDiVc1AnZ2HgQBuALNtgJTwV2XBzeA7/RS8psxvX6VLLrd/XDoy9MfZ1GeIWJKo8A2SZ2a3ecjezNgYt2bqAYtEkebd0Eu0yU2463SG3z7HaXbN+tkCr52MbSfXfnUcUBGQZ0oQ9l+/KocKrjoovVldiYZgJx00RpW3l9A99TDKx2RgVyRX4hWTq3DrRJjCsuBovlw9SAKjQq3ShfxlfPKkIHAyz2l0Fm6BlZeEOQFZ1y1MbgCxb0ts9XODsKndA8dFBL2k5fbMqNmubUw9s8QVQ6dctCwH4oDgAr3njm922aZ4tu+bKPGsu3DbYibYAzzV9Ge6MoTPhU6bQzyBJ4CEo7eVkduundbaqAABRK973AwD4jVhwT/+B4X6TbFtB9Rn66WndT+HQz3EAoLj7ZxLjzqYDyWc0Phj381WVyDEfCgNiajqZd4cHu89maVq5V6O1vwmrPBlDzwJ65gO6fS6DB7YobRs8AKBEM327qypHqc7JRa5ClZOTeOmxOvaG2ubQCObLBxUHPzgGAFiL17UbO+dsEiyWBv162ba18fIKgzM0ravDStaaNUoNRI8L3j8/6l7N0z//lPXmQgApvX4OOtblmfgfVr6LsYG7rAnfBjwdcACAemX36OsTHsJqT0bQ+zOATgZAaeHr48MHlfz5om2DBwDUqqZvt/TZ1dqx0w1cq1p2deSYxotn/Lmz1u+2hkZvp1tigPZyfApVtBur4qTgIxVx1JYV1AXyuGw4OZbkRKdMZsq/I5nr0TaFDK12mPza9QXZayjJAVOsXhoMDQ4OruETHBxcvGFw8Agz7kaNIfnM509ySd0s8nn5fjTpqQDodQgf28tjOEkDoNFh5BGneyorIAswv2baRqp8Mclpf5JZGENyp+MVa0Jj0fHkdLKd+3OS9HVQazc+RhD5Aw5oenswxQjDusxyOnZxLMm/NpNsNI4k/UaTfLeUmjyCR4zCdpJfVxG16Fv7JZnZ1Ly7mzhMkr07MdVlGUm+1o8mPRUAvQ57yL9GkBQCsdFw8jZ2WwtpTWY5NiCXynEkiQ0k1TXCrQmNJcfZY0muxxaSrILn2q030VqQ2R2Ya4Rh5Tpl/sF3mgNAID9O8k7WzHyVADhUkgEOyMJFnE0EHjcSdZ35bQFwqo15d+dRBgDKfK++qKiet886T8bQgf5A0uGyeiC0ugL8XawhbBu8aRu3cn+33b30D3DrH6Gd6PvujwMUxTZHAGgQH9dAs/VvNBB85hEain5SobViWxq/UxHA6pnbOskOaket+weAHAMbixpwYgZ4RRmNnxpEA56VTLtDLgCoCMARVnsyA/0BcL6UAAgje8yo8833FWwYvHmr2fHLbseX6/tKZ3vRf9vuDCBj+6MywIBfDmgTegD9BZ855t/J1syi3ifT2gIvps4JAzKRtbdvvv3NHS83BnDR6nNYfweJbkuBXbkXgaqzTbprIYtrAOCvVg6Nit3oDEBllSfz0Ns/keuBcHCfW+ohV1sGb8HGd99RR/N1VwNQXW4jsw89xdMZAAZt+2Ey0Lv5ktElAeD2ml6t9J+5snNfcRtWUDO2kSRVIa3IB/iGvFt64OMvqfKaQ/Lt2iT3I56c1fAF+XS0mHl21i9kQrgFd3yvg5K8Ko8lF9V/QSa4daFJTwVAf4VdJMntJdRCoC/67TsWeYdWQlqyUo11zdyAcpq5EP3V5HrPBGtCY8HxjM0kSUWJIBXJe9UGZJNM79TwGclvsZckI6t8aQKjoMxG9SlRdmAKSd5+k+Ti6p9t/PhMlXcTLvSVV5uUO7y8x5C4lU3l4dup+vy1r5ZOTRGT2eap5NfLLbnLXdxrzYKwSyRVX3bbvHx6y2J9U015sgh9bXBFNBozdkzf+mhqAPTAw8fPHc1iaBWkWfth+Gtyz64jf9N2P5umBRgxfdv77WNIK0Jj1nFkV0+/pSTP9ZfLwz4nmfxWs03nvqk3MYO8OcQfjceMHdT+jSjakNn8lhX9gMxSmdqliksVteh73JZkzxuW/ani8iKiuvmItxIz1KY8iYLOA7pV7y9SnTSmIa2BtNqWxWsBNqhvpVoZGjGOn+xavzfNCgyxz/Bc6wBwNf1CsJo4qMOdAGV8YAFvGXWgDtWRd59eoCeL0HlAx6sFArLSs6qqHcQP3rTF3uidnRigWxhXsTU0lsynxz/+/ezBMOBsC1mRQb+esJ/A81mTCy8G62djzQRNMx4JRUv4LUIe1Jp3HHABTYoOOm3jbTnSu71WeHSk+597tdLAHfmxWG6ZuUXJg5IYbhLDTTJIPCjJ/gHmOA9Au1fjq92/CrpdUaWkXSFhSBoBSBoBaQUlraAkK3oz+Qwq66m+XdYx7doTzw5FMbb0By5OaqVjFSDliYssu4atOGo6GvYd7MeOT3jSzN/WAdl1sF2ZjV915krJCHdkJp5Iv1Hj4febunQA4oe9O+AlDiRj+Z2H9Sb6AeD6aLnjdHfg0eYTp+WDgqsA1yMSgzssFo35eMIaLwCn3g/3dQeAgQ4AcHzVdvuxI7/fv9Pf1phoDhb7/fw5Vukw3lfXXXfP9cUkX1hUWZp4Bn0OPUiSD4IOkuzUh+QuvG3m+bN6uf2PupP732LaG15nSUV4hIo/BiWT5AV01XgYuFEtFjp56sgmeEiSq/MWjCTJzKoazAKwC3own4KdlmJiOTQp2Ck2TBkdZyu4ubsOeMB88lLgA/MYZjJ7DX00jV1rSHbtQ1J98bmZsSSNsj+z7yaQzPAOyOV853SSrUYLhqGef1g8dPbt3GWazE5Yc+DkqVMn2yWQJD/SZrYA7IIym4GdlmJiOTQZojOrfiOCZDXfvP4urxySI98wj1HQCqp9om69Fexu5jM7C+FivLflM0AeknADa4M8ADTf8kJ/HZrVLlQ8YrHKebOrw8jObVq3jh/uDwCRAaUMrnE2YRcck0IMDQDg6I4pAH74Na+/tZULgJA9L2x7BrXnb3g2z+uonjw03Jt69oQiEcC1mXlbFBdiVACUj54i9bm4kVfMUQCQ43HqPTkA+D6/onM8vUcbu6IyDgDiz0QAQPbPBhOjjdg59ymIif58tQEAGHcsyVRohAeLsq+9GgBo3CJv3Ie9AKBsznHbMnsxG+imbV9v4jseX7atVOZrHClXqWUklm70Kv5xMPDj+9lHBg3aDHBtrxfxrU9jX1DZdSu/qnRC1MhPPfQDcNWpbnFHasZ1Q7tHOcitqX3f96oA1BMWyQDgywnCd3u2Yd/qO/fEJ4fzYqI/37wAAMebHFVv/VBlFBrhwaKMxwLufrBwxs28fvIzOQB44abYFdQ1NFq3aqrbNU2vax+SDOlDqiLckqkO+oW8EUJS3Yoka2gmk5Xl08hDJZ5RHdj7j4dltolXt53HFLJ+HZIcg6WauTB7bEDxmzZO4dp5liS3zCBJ/rmebKabZy1hmx/tdY8TJBdhpzYmuvPVBeC44yVyp9th49AYHCzCcSpaLVExvvzvOrrxJJK8gnni51l5tWpV5YZbADh8GzRIvWZhLyDpyhVANky/O21mH08g1GErZB63WpV52E/8NW546MfA/NhHgOKyhpCKF1OmbsweorJ3mlK8/y4AKDcPE8r1bMN+q2FbAP10Mck7X10AOCa0ESCvXsE4NAYHi7m1R+SbDgjoODJb08+Gi4YTnCVeGe3bDh1NaMtcfmoY0TQcQFP/hnVCu43Q7/kzzeUPAKVuAqhnU/hn+u5yBbp/9s4q5eauZzULnRvrygdMWvbZ/+zM7B5ZJQBYMVb4TbYN+1HkXKPA1TMIwL3YzgBCr5gIjamDrTF3VKoEoP53ZzTPjDw0vNpceNg2zzYzcVyF5dvqA4DriTluK0OHqHU7ElHa1dXV9fv/AfC2JfrrH+x3B4ApK04cm5yE+gCARuWBjwLnXLMzs98FAEBsjk9qamquMjXDDuwb8DLa5m0QgL+RT82pD42pg60xTxdvACiOGE2/hObHmm0BzuK3p72JbdnHPuh/uTwQV/zDD59tfG+49kXR+uG14KPjSttCbtob9b0Doh1qA5UrA4n+9XV73Da2GhrpYk9ic4+HAkDyvVkAYkvPqjLFduwqyDS++wEAXQBUuGvwjRWExtTBVj0qrJepQdbesnmVSdKoPpoW2pt3zp01P/hNBRCzDSg5qU80AFclEI/65Q4DQM5u24J/9sxyB+DXYjjVKw1IOzlb8J1r/t7lj+z6ycY+9waAkBUrVqxY4Vl3xRQ7sCvWPAfAxASnC0D5upcBIGencWjMHlyQdb+lAJCExkBKNiDrmgAA8d7BYjObBoXw5kAJAMpMIHemZ2WHTTGT1MCqFwAUzQE0+BsgXNftvQpgeUVAkS565DcGPX139KjB3/hj7+67wBcBgzXD0CDNcfvkgC23ncjRNB5AvxhUZabDHmzZ2sPRAFchSxuTvPPVBUC27ux+AF9XMA6NwcFibLT7IYCHRlTHk4qtAUyJjQf4yxRHcXc9F3oHyku06XdOey/S3dun+6Xz3b29u46s4daA/Ka4vMWinZ0/2LpvxnKSvFV53ofHSR5tMXPje/t4MNzDt/N3Ip/XaZVqQWR0+Mk/p3VIInmhdw0Pr9feJ9M7F5OX6HxV3F1PzsCOZTxq91hGkrsxM2/z6OZyr9c+LRjbwmhPtV3209Qf4fvG+e7e3l0f689XGwDybKMPts/fYRwa4cEi7+DON95xcfTAdDKz7iCS3BYSkzJ1kMI8hs1v3tOLO91UVNdwypXX/DVrpvvPA2V2v1J+ejC1oUgackHQii1d/QrzBfj9J7XVsaVKGcukdAF4lBroYCY0Zg+27Dj1xJMmQcIXWfszWzSUSaxUiVMhGSRWqmRSZiWTMiuZlFnJrGC4hchejS/ZvwpaVlQpKQTHIdrfbDu+Epv7r4KeyyKywnDcDtL9rHQ/K9m/XiOQfgco6wMA14s5KWlcYCFDy2b0KSMT8Pif33N2pkJW7XauM5TKWgCUt4o55vh5wnZGf2FpBPTKgMLXH+AfKxcwyuyjXde3yq9UBbDh98vtwt8zOiJ526mTZUa4K2KiQ+b66bj2T7acPl5saP1qJ85uflFxyEcAsmbvKN1zstWZfbY2PUs5roaQ0W8nj1+HCECrDIB6490s9TvVCkF/gPyMf9v0AtbJBbRiB/3w876t+fqw/K7nWl+0ySXJnPZqk4T3a+hPkk+DKzwUcO2vI0RTTgtNNEWFntdIFqERGJdMLnY5bMjot0sjoEPUKwPU46+SD9qfLQSNgBHj30a9gJFcAObFDsLhm+oXpBG4tnoiPtMU/jRNeI9DBElyGyIEcoIEhGncdcFakuSHJ0XEapnrdjIZLQ0Z/fZoBPSIemXAns9JMiq0EDQCRox/G/UCGQVmVi92EA7fVL/gelCfHPigS20USHivYlA9WbcuW1Hnfz19gbiHYjjaZQjAHSmAw0gA2Dhcf41Sz+5sA91bj6hXBpx7AQBV42Entmm9AF6OXqBY5bxW/uEbn05Bz6DcNiqHKvVdLb9dT3jXUSVN1tKqMuvZdICz54sZfv+MvsBpvG7I6AdgM49fj6hXBpT//As1sCdcNLbqyX0AikdPAeQmJyE3ISs/41/15KFJuYAlvYBIuUD+4RudTsFPF5v/7+JC3USs5bfrCO86+8nRNKFzWo3v/sC2UF9RiXAGFEsbzzJk9MMejYAOUa8MGBYwud1fu7d8LBb7dIjvcGBtYNnVwB8NS889sPh8v7cIIeP/ehPf8abkAhb0AqLlAvmHn79f4ApqNZldz+mSdp7V89tr6ObZ3g8fJh4YFbBbM19VjIiIiIjooZ1nSR5F3eTeKpEzV+T0JqOeGzL67dQI6BD1yoDElnBpo7BBI9AkjGSO0wKSbNnhOzIOR/Ix/kP6mJALWNALmJELwILYQTB8k/0CV1AkLzvVyeZYkqleE0iqvFcKM1tr2bJl607lGAgyE/SZ5SAERolek6j/6txNs5rOqZiYl9kuY+JPyprn2qb+0CIqJqvyMhs3eJorOj20AjsfZMcwknRfQJJhvioy13k52byt5sx36jQywfVpEDR17TCSh+tfzwuffpfBwdZlVjB8k31rKmo2mDt77iLAkPqvn0onWb5szNxSM0j8EiRwS/kukY7QMfoBOzUCWkS9MuDaqN2lRo/6rcdZB7HYBrNWbQfA0SXTCrkAzOsFbJALGA7fuG/dW7wZjZecAQyp/1ZbcRS3JRM+TS4eAvIY/QDs1gj4NLl4SKAMeGdRKVQ9PO/8QfuwXbWE8YLlAjCvF7BBLmA4fOO+dRoBp00Nh7aFgPkOLeH9pTwKU7TOjXQBvBGnZ/TbpxHQI7rrlAEjYtoAkM09cjPcJmzmipULoLpZvYBStFwgPd/w041Px/JvlpqVeK1P/r4PCKn/GsL7S7Hsi1cfA7iL+npGv30aAT2iXhlQXKYR9FSqKxq7GAEk5UCkXADm9QLi5QIGw0/JhqnTsZjZmFjN/5PaGDDf8wjvQBqEIr08rr1wq6HEwArzDD9aAbgTE9IGQka/HRoBA0StMsC53xcAcD+lrWjs+ikAfvHMAgCFEgCVqnyMf2WmsVwA5vUCYuQCGrGDcPhPKrY2PJ2C73rOh3rJW2wkSd4aRgPmu4bwHtWnptyz/TSdnEDDtb/Wp5bco/1kzYq2Ty15ydBZ4mrLDPns9B/Nuj4UMvrt0wgIEfOUAdlDRx6PWjsskaI1AsmtP9y/aF95ecdnp7uVKNk98rv2cv++zwSM//Pdvb27bjaWC1jQC5iWC8C82EE/fI1SQNC3USOQx2/XEd5fxivlK1FsGCSzjdFvGlqPqLe4Cy+CmshsgXx6u7rnVQ9fd5lYuYAFvYCpgy2GKf/wzZyOpBGQOBWSQWKlSiZlVjIps5JJmZXM4OliZUkj8B/TCFSWmOTSXY9k/41q8zCgjLtUcAMAZCZo+q7+TgCQ/sDZmbnKsl7G9PLCteTrT8o1LwygzATAQ0cXu/0cCHAvBNh/C5PciDLO5GPlPm4A4OmPp49VeMsNT46UWxik5WL3DujvZUwvF2/6UuqDg5u43TnbqYO+lbBpw/jmhVHHPuXHs6d4u5xm6+OmyR1ajiyMzL4CJrmJavN6ejlE1Uo1oIyrV7ps0tJkhpFkVmjxCxoudi+jz2rp5SIZvIJS6n4AnGerDVpBE+0od66vY08m9MMn2s2rBiPOnj8D++qY5Carzevp5Tb9/VlXzZ+7l41JHxbQWt93ndd6xu8AXIWztHaf93v9p20R+8XlW6XnAx+mAUBQz3tVQv0NW872/KTmLPSH54bKA+McAde6Cd/OkAEAn3ua+TO6sIUkCaD7xZoFzCqx5g8WFSIA8Bhf4asLtl6NBTZp6ZgowQRaEX/BfFnCkxBfSv0SgB+UAFB6TN5Wfcsu27srpiTkITtv1AGAESNPhgDAmVaJhb2mfVlMcqMQAUJ6uX1PKlxDr50SdI+hC8xX4g2EnaXUC9d0dewBAP3c1gMAzohzVuRMcvEhsvKupyKO6juXZoQtNftJc/Ryq0upRy1bsiwtXwvAnDKVOiXbklldHXsAgOebO9IApHuIWsMXOZM8f7V5q46xsFjIk2aRSzCM5F00WbJkYT+/1WrDP/1iTC8XtSYxKKVed6uavwTeNmgFTySv1D5qx3LnPKaQ5MMFPIXVJDckcSzuilhBFS2T3KjafL5a+iL+Xk/+zC7GEJJ3MZwkE1t3iDeRWSG9XFT478AxkeSwKlkko0iyUW+DVvBERo/JsGMhm10vNEubWXVgY5KLKTKzRcskNwyRdZm1ksh8H4IZu9KvlTtfFJYrj6nuYgW9HFaVUg8CgMZrk30FLeD3wTFyO+ZaTR17zUJnxPSrQVfr2fXm5NUzyfNXmy+8efYuOgp63p1u7hXu3eBk1xpHWEo98qSmTH6MoAUciXabZIeDvDr2AIAhTt/gt452DfjVM8nzV5u37xmUwJ4cDG4Ngz8DkiTsPrDv6bOwlHr3tAwXQAEfQQsIm9SiVXh/m+97dHXsAcDv9S0LnB1tAio6Jnn+avOF9pv9OHetcC2ZfVwmvCjcvGPnjYmglHr91S4ArnvVErQAOZrNHW3rDaiujr2WJD/82ds9RIMUNZM8X7V5ezOrJYRnTFy/u5Ggn/3unflB+m5i//JG9HLYWEodb9QCcPvEMidBCwol8J5Hn2c2gevr2AMx1wl0LpvhL2CSW2dFyyQ3rjYvrKUv+q7nSg9/yHtGDAxrNPkBScb2DIBn74iIrg3Cfid5oXsluPWMeLOVIybno5eLXsDqSqkzd/YXV7bVWaoWtA529azY6/EeL3nNVbasjfV17K+Hecsbf0Uu2kF+2snHo84bN6webREyyU1WmxfW0i/kavOF+0pZUEo9NtK7hV++1j/hzXtRM8nzV5svaPASp0LiVEgGibsomZRZyaTMSiZlVjIAgOM8AO1eja92/yrodkWVknaFhEFyrvQF/4/ZXO2TCsmkeVYyKbOSFbX9H0+ETOnuPWKKAAAAAElFTkSuQmCC',
    '1605.06770v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAgQAAADUCAAAAADGyGRMAAAgbklEQVR42u2deXxTVfrGn3STpaVYKKCUXYpsLVCKIEgUQZBiFauyDAwgDiCCMjCiI2jFZXBERUZALfATFRGUoY4simUri62sBVpAkNayCKUVWkq3pMn7++Pm3nvuzU3apCnNbc/7+WggOe/7Psk95J6bnG8eA4FHXQ8f/hLw4JOAB0BE8fxVqLsRT0R+AMDXBV4dr79encUBGEj4j4f3RrUeHwPxNQEPAH7qO67l+vtZzRTOXxqvjYM/5XQc22Tj49U3CbI2/LwneFRnPgm8Nl7JeaNFRnyrbx9H5sRnx3jq6gCkiDSMJCKyLiEeXhHK4/O/KAsRWYZHEX2Hpz1SXGNN4CssFHJP8n903hhfDvIB4BMPIPbQf6rndCBFIn+9vTIyhX+30W0BQxSq9xPDE6+IfzIdzLAAKM/NQXnmnwDKfi/nx6LGInzDDwBgmAlY8i4rH7MdKk9NgvX/LN0xbtyXACWMLM4csB/7ejaL/+Hdg5Oetq78eP89y/nBqKn4O4Y/+O8UE4w4FR06E0Cbvi8tes/YNFs6VJ5YGJ5AHBFRp6lERLSsZQHRtsbXie4dtJroLF5II/qswTW+XquhhSGtawyg+UdERMY4ovJIE9HhgDfZQ1X1haEiCl6JawQM9lkLBJ0YD7Tzz44E7i7O4v8kaypGZX09tWPOzPcBBAIoeNQfN54a8E/2UHlsYSjEkYKAfQCangHQxQfwDegKoB6K+MGosWg8ejQlP7VgcmMAQMAo0LQba3wVh8qzkyAbzeoB+KolgHqA9H8LPxY1FOtHATDc/+njRx8AAAR2waqvf7xDeag8OQlWTe6MJr35C+9NsWEUAGAoSsR70me+NBT7zO4fKsdrgnpmIBORdyYBQNn3/NX3kjh5DgBQiEjbHUWjerwJnGjk/qHSmAQmmACgx28Aod6KzccBLGkFmMwAyGwGYOangxoLy9h8APh4UkvAXATg+T++9kfJyhbMoYLrXycrvq9O+vjkH769Oi1H5qBJBqMR2DXvgU7p9w//+Z29hvteOf35wZCo/3vnp986DPiIHw/UxH6CiA/Wt+0d8P35/ws++NY+9PssfdAAI13ddc7kLx4qtyaAg00L5hNtQwAAl26GG/ih8JZJcDiKThyyRPXUPCSuH6oKJgEPvrOIB/iWcx58EvDgk4AHnwQ8UMd2Gxv5BaCXXx5UY20j+OUhD3464MEnAQ9wKpkHp5LBqWROJfPvDvh3Bzwcbi8rYuadTwMnqQUn8hoNcrFd7qm8O/u6kmD6PC9snLd+jpGZlXdP21o5CdpfNzYP2J0VPqD0SnLb35ykXv7qi4ddnQRZX3w205VJUDpwxgBjSExFB8NTdK6LkfrV1sRbPwkKl5y/3P2F5gCuJ9woMc/oBACw/vek2WdmqGeoZEvjM0T0HpYRUXoo84AGpfxQnOskRcQLroxe2s2aNS6zolH2dO4tQqqvIfGWwye5o89RwRPBKUS5M3KJ3g1IIiIqHPKqib6M9RCVfP2pjhDJ5K4PlTDv5PaUsr8bk861nANdDW2/bFfRKHs69xYh1f418Pbz2sL2aPSZ71gL1qzcBUwyxQOgSc3e8MeCFA99WHS1B/OXiKuoWUq5qF6lVrhRDVFnkOrN914HAo1Zp9GCADTENQA7N8wG8PX/PDQJ/NlT9n0AnU0pAVhK+fzOc1a7OuW5OSjPKgEAS94lAKYrfzpCmYuyrUqK1nzlT+TfZM9RtqaqyE9JNmULj+/KYfIEOrf86lUUZ1mVYqUUQaU8RO7JNLPVZfheMV+qI4O/ZZdq5NK6VZkJQCCuYnThk8B+jADwSXAPAL37eWgS3NWT+Uu/Nttij5a+OM8sU8qmOUlBiQ+eVWUJzPKBUZMI+42hk4GE8Ds+hibKXLYwYc+EeSbIwPOWiDtWLPuodbJUTWx6aNyBHePGScTb+58H1387CsDu6J3WtQssYp5A527t1XzBm/O/HzLXJIuVUwAohsg9xWZyXVmZlC/eyuDvuSfjk/+VVAOTYO/l5gCO+3UD/AHT+73nA7Sr3YV5C18+A4/9XA0R0WJ8LPw2SsdrRNZnx8qU8srWp4hm97AQEcUwC0Mbs7yDiKKHElGZ35vM3TLKHHXnJSLLk7FWhqK1hj++73KLddIPsshN4x6VO5w2EpG1P9Fu38NEiQ2S5DxjHBFR5+bHiG72GmGVxEopYshDxFymmVRXUibmS3UkyaeCkononRpYGBIR0QHMJiJKnRs99SYR5aP/IgtlttzuXnHnk6A4bD4R0TFsl17XDbcfItqMM+pJMDTUQlTuv4SIhgwlImr4JnP3Y0SUgsNEFDWGiOggNlB+8PNEZAlZRhQVyfRmm7KTIDn4KBGtIGuXoUSUFHlKzhOERE0QLhU2SGLFFCmYIUIu00yuKykT88VbWXLfgUREWTU0CUq7Dy4RroJ+HfZILtF5+GYT0cT2JW4Vd84ipl3sBgDdDJsfFO+Ke9yQdTwFBXZjBWa5SH2KsUOZAwAgyn9LHEvRdnfeFADQp23ProMfeQYXTw4DMDgNqjxbDMKWOHWK5pDuqmZyXUnZ07Z8sY70wJXU+MrgvNUVr4R+J6yYDeFrWj6c6tsQrVsDiFz98yDPf5X8q8Ag+wScku6yJvRb06S/xlgHtLI2ymwIOotsNKtXr169r14CEOK8qVAj+bUGywb/1fobWkj3hdjrCAo6a5eiOSRE1UyuKykT88Vb6YHTCK7BC4RVf2yVLoeaRB/ahkYBIQBQHxke/uEqAOiAmwBgLrtLpJQxN+FoB+wDrAbnH+aS0981ohudwFK0BidNxThbf8GC65+/OLkjLjjbeHWjsJMkVkwxag0xqJrJdSVlYv6dtlvpAXNN/kDD5mNf+SDd564B5akBQAjOwq97kfCPrGk1bCrpFZoCAAcxTKSUCz+c0AG4Cvy032HWbQQgp8zRtwEAkFI+3CHwzDZlI2MdcPusuPSW3Y4CQFmidulkDBfFSilaQ+yayXUlZWK+eCs90OruXwCgpCbmQMrPS3yA/91Weuj4VQAXEAnEnjMByEFvT06CUpQBQMNPv8kELO88NUKklP3rlwM44V9yOQQwm5lXV2CWLQAirwHY2KiEuZtBmQ/mANb3Hh+pAJ5vML3ZpooHlhcDMPU1rEjZCuCTMPlhs/DvMvkKUBz/+EhRrJQi15CHCLlMM7murEzMt91KDxgSktIBWl4D0+D0uD+fnTZ1/Mq2jYbvDAPOZxjvA6Y13AbQtmc6wt2vk9XfV29Zc+VUcYPOLZ55EEhaOKTJlp7z/CFSyv/9+3MRh2NWHjW+deStfYb+8b0AABKz3DQ6oXHeyCHRx7tPKej3zUl7lHnIqmV3NUjsMj8AEvC87T9760WNniALEJseWbDP2nvoP2z3fvdpVNfgvXc8D6Q+93Dk6S5xYp5A54aid3jbDg1W9p8fIImVU2whDZF7ys9QrCsrE/PlOhL4u2/eyLDUPqNCjd/i1u4niDoifJh7DHlzIvrRnKYrWwA4OP3ltitvfBLk9gQwOP3k64/8cD+WUracNncJoBtOF0Z//t6x0fGg0Ibay4bsm538KqBopabsmby+3xlTR2GJeSU/3P49rHe31ecLO7Fi2RS7IVrN5LqXboYbpHy2jiT5Ul4X68mmTevX4KaStGPUM0JQk5+cFx1RhXcBQy3ZWdS722oPDAHfWaTnMJd7Ygj4lnP9xk8xpzc/dqqqQ1BXNzHWjtOBBb4o9/Gp4pA6eTqoTWsCvtuYrwl4VGVNYOQvg1eHEbeASuanA3464AEOn0BX8Ikm8OGOlNoVVp+6BJ8IwIcKN3FHim5jfFR0g/MpDyme7+7l34DFUmD9/EKJ9W93oZbCJ9eQqIGbuCNFp84nzQH4v2pl7yrqEKPAUsg68zjRHw+kuLm9TAWf1PcC+ESjQOyhu2seA6mhiHjsYvvByjPiYgDAawvbotFnbcae9cWWdt2BOz6ck+Te6UAFn7RBzcIngY5wk7p7+m82XX1ParumALD5u4zbEWhMPN0VvxQDQIfMWgifyMCHaAan1iInWrJuoOy7/Uo5MjqihFL0HaX/HaPCUtDygw+twKbhtQ8+kYEPmxmcnRYZJPloRNKMKe+2ftTCypHRESWUorM4tnjRYna393+eN6ixlInt/n7/r9+vebvWwScK4MMYp9QSE6dI/Pa2fCoPn0PnFHKkhnZQip4Wht3WWmlj+O/S34+sIronRoWlZN+LgPtMHrHEK3lu1O2AYdraHdJdjQuLgEFp59RDbYZ56QAaA0DAbezdjI+e8U7AZ+73GxkLN0PQuf4tLo9y3FSIST0HAhCGBWpoYRLXhwfDt+d6tGflyA1z0tIAw0R9vhF8NcaAkYGzxb+av2SfR9nkwW8DgKnDP3z2jrjiiS3nXgWf2AMfKi1MYuNrAPyClWc8uaEDKEUnVwcA0Dsh1/aDFEuf87HHUk5M/b7ptKk/PZriU8vgE3vgQ6WFSZyel47CXXNhZ+8nNHQApegiUvcAQJDImZwsa5Kfn19uzi8EZCzlb+80RYek1w/8WNvgk/Z2wIdKC5PYbMaSZleWPaaUIzd0AKXoImILCgMAE5rYPr65OB/AyWbz28+WsJQuNzLuA2CI33FmeC2DT+yAD7UWJnHXwBVvr3pMJUduaA+l6CciPw4AcCq4M3CtEDAuXbp06dJG3ZbOlrEU1DcUAgBad6tt8IkC+DAXQaHFbFYktnkxYW1ikrBYkOQwDe2gFP3EE50B/J682A9Xw8QnYCm6wWIp8B/1IQBcujbQ7UvEzaPvbx7U/P7R24nopwf+9WlsvImIzrV5fcFuog2t3tn65pHp/V4uPxAb0iT2sJCy/5HGt8emrn4gsO2T1yl3wIKt72xpGTjkunR365HXX+oZ1GMG0eDsuQlr4l4tIyLa2e+Vz1/cQj8ODwodtpoRIDY9HBvSePAi+f69Axd/O2c9Qp84EBsSEnNV1pJikyKpLejRun2ren5P3SRi5IgNKXHYvLVbXl6iy0vE8lc/TFvX9X0r0Y0u7S4QEdG0voHBD/6betkWjkRUOmHK7mMJE7MrV1xH8IkG8GGvRUgs6bOiL2A9PGFUvEqO0NAOSoGeNpWcTA3p1xwAsObe9o6Szh4sjog21OWNphsThGXx8h82oTbvLJq/wJfvLHIUPdOzAcCybVit/irpYj1fzh04joOLuvRpcHbHwGcNtfmd4O3ngzh34Cxys0rC2vnwjaYcPgGfBJVdE3DuAJw74O8E/J2ABzh3gFpmeuFyD9Q1zqBWml5UrYcew44zQFUt8XRvelG1HnrbXqbFGVSteK0wvfAGL4pbGotV9hfw8M/V6NH0oq6FyBlI9heengQ6ML1grCtYlEBTl7pDbeANNDgDD08C7ze9YKwrGJRAreu1Fq0fyrXroGveAI45Aw8vDHVgesG6W0g1WF1RLxClddlp30FfvIGDhaEmZ+DZhSG8ljuQYYEGwyKAhq9tZmuodGUk/PKAfQdd8wZwxhl4+HMCeLvpBRjrCrmGUtf28RmBLGhg66Br3gBOOIPq/DFLrzS9YK0r5BoKXTvSG8wC7DvomTeAM86gOt8JvNL0wshYV8g1FLqGzurXf/ho+w565g3ghDOo1ncCrzS9YK0rpBpKXYG4J35atn0HPfMGttDkDKpnEniz6QVrXSHVUOgymYEXg+Ku23fQMW/AhpozqIZLRC/lDmRYIGrMP1euHaSowej6MaZRq5FXNwUH3r1c3UFfvIHD7w7sOYOqFdel6YXSusJWw4EuRQd98QZ8jyFqm3UF31l0y90teNRu0wtuXcFNL/RqXcHXBDz4moAHN73gAQ6f8OCnAx7g8AluLXzCg8MndSCuPv9pMABYV1ysVzwrFABoVXqg71x5bwHrf/Hfk2afmaHOjDB0Cp+IaqxLqNaH8guk3DlTonGZiMg65g2iw+F/EJFp+F8stD4iVxokcymFQ1410ZexjowwwPwnRd4UZqPpX4rlB3Km2qmLcWMSRLk0Cf46ysEDohoNVbV8EpT+Xr5YmATfBZcR0ZQniOgN/xtE1H+aOOjZLCIqDGlXTtYn/kJEd4USbfqAiOjY4EpsNNUJfJJYk6pqNG5rI/5Q0dr+AQCMm4qBhIggAH3XFNsekrmUnRtmA/j6f8AvFwFtIww9wSdMYVGNrEpkUORaok5GGgA6uytHkaBbHMWSFAwAd5TtRv7FQAAIvZlme0zmUj4J7gGgdz9nRhg6gk+YwqIaSZXEoEi1JJ2MNAC7o3da1y6wyAn6xVFyrwcCQDDOoL4vCQdTJNIkLoV2tbswb+HLZ+DUCENHphdsYZsa8VYqJdVidDLSdvseJkpskCQl6AFHgf3RuUxEZzCLiCgNrxNFdiUimo732WEHMJvy0X+RhTJbbndkhKE30wtFYeVHFlIpqRajU5ZG0wf3AgI7hkkJOsZRSgWEw4AS4I2TVwDTUZSruZQbSH3KB+2GTCl1YoShI/jEycebbClbLVanJO3iyWEABqdhl5jwtH5xlCBh4245goDY9/623PxlTEpTqLiUhmjdGkDk6p8HOTbC0BF84iTYUrZarE6p+29ooUrQMY7SWPjV/1IEA5i9NHnX33MQKT8scCmNAkIAoD4ynBhh6Ag+sSu8arJ4y5ay1dLU2REXhD9ICTrGUYJb5ADAFfQBgDZtgOy28iQQuZTuRcI/gKZOjDB0BJ8oCgtqxFv7Uto6W3Y7CgBliVKCjnEUQ0wWAGSGRAF7RxYABXte9ROcMBguJfacCUAOejsxwtATfMIWFtSIt0wpWwqrU5ZmWJGyFcAnYXKCHnGUMuHozD6ZCdDG2b7A5u8vAB+2Gw+bE4bMpUxruA2gbc90dGaEoSfTC8a+wqZGuhUZFKmWpHMPK41Ses375o0NcoIecBTl8SkbO6RFUJdHFxPROmPGtTnjTESUPnzPkX8MyiHRCYPhUg703nBo2tgbjowwdGh6IRe2qZFu7Uo50nklP9xHTtADjuL4+FzdWtSvpwEA/vwxv2c/g6YTRn5yXnSEQyMMvtG0Vu4scs0Jg+8sAnfC0Cl8wsNpfP4Cdz7hpwO+0ZQH6gSLyIPDJ+DwCYdP+JqArwl44BZwB0JMn9jHy5WrsZTMrLx72sIThAxfGApxbeVqLxdeOuC2ARO2MnekfjA6DUDmwK+hIGT+srzGNKq0eCgK35ryyCs5AHD93/Nmz/wVqKYfuE5AU5M3fHfimC2xx1KuIZGIvsPTVSdkPBR2WtxjZVTwicSV5M7IJXo3IKnK3045mARPjcWP3jAJHLMl9lhKIRKJyHroZtUJGU9NYbUW91gZ5fGRuZLF9b4hysW91fQr55ebjsc6b3jTT3QZSzFENfSaU5a9Fg+wMjJX0oIANMS1aloTfPPkg00SNbYG2cEiMohiybqBsu/2K7xIoKRVtEgUETNhwBIGFpHZEueWKEDZJRIJlctahAyTKCplIRlHLIqmyYqalZFTbGSLOEDQwtSwez5uhMyVjC58EtiPEdU0Cfbc5/9kwY8A8HHf1nd/AIwJ67baHhZhQJSPRiTNmPJu60cZrEN85ZQUiJJEETETBixhYBGJLYGKSlF7opx7Mj75X0kAcCo6dKYGISMlykr3MJCMIxZF02RFzcrIKTayRRwgaGFq2D0fd4LxO/EHTO/3nl89C8PMWUTJEE66+U2fISJrtxwNWEQGPL69LZ/Kw+fQOZYDITWtYkeiMJgJA5YwsEgnxTmUGa7AUk4FJRPRO0gkIjLG2RMyTCKjlGnpkEXRMllRsjJyikS2SAOMcaoanaZ6xBJP9DtJnRs99SZVz8LwX6lElrAGQvm5wUVEWYuI8oOfJyJLyDKiqEgiItpw+yGizThD9ER3IhoVRopRpBpEQ0MtROX+jxFRCg4TFYfNJyI6hu1EQ4YSETV8Ux62RD0J2OGKSdB3IBFRljAJYuIUPSkmTpEoK2VaSqKTg48S0QoGop4gLPM3aDx9IaQUa5ehRJQUeUoeICxK5RqemgSl3QeXCEvPX4c9klvlSaD5YdGmFqeAdhc3jQaAKe9+MxFfTNPwEWEAj8bXAPgFQ8WBaFMgEonCYiY+mhwLUAkq5UpqvOpzLxUhwybKSpmWRypgUZQmK0pWRnJTkcgW1QCmhqdWnJLfiSF8TcuHU32rYU1wslOrsLCwybbrgw5DP4G5oJmGjwgDeEzPS0fhrrlQcSDaFIjEgjj0ONHmWBwNPw31RkIVIcMmykqhAa84YFGCnLAyUopEtqgGMDU8FKzfSZPoQ9uq42Pjr2d3B0ALf8hvDADTRh7LfATQgEVkwKPZjCXNrix7TDXKMQUCbcxEk1gRGROHVEp7qN80VD3ZRFkp07IiFkVpsqJkZaQUiWxRDWBqKJ+P2xeJAldy14Dy1AAgBGer4eqAMroDgGGMSbiqHdHy0z1GwJ7wYACPXQNXvL3qMWiMckarsJiJNrEiMiZOqZRWd/8CACUOyRM2UVbKtHTCotibrMCBIYtEtmjzNskYrno+bobIlZQeOn4VwAWWPfPYJFh7XbgdgVVWAPD72xdtDYA9LMIAHm1eTFibmFSgHAUHFIhEorCYCQOWMByLyJgAKiqFxVIMCUnpAC0XpoG5yI6QYRNlpUxLJyyKvcmKGokRUySyRR5gLlLWUD4f90LiShoN3xkGnM8w3ufxS8S9vRvV75FNRLMjGgb2n0lEdDEwT3hMDYvIIEpBj9btW9Xze+omw4GIVwd2FIhMojBsiwSWbGFhEZEtUVEpaixl78DF385Zj9An6EBsSEjMVTtCRu7DKGVYFocsiobJioqVYVJsZIs4QNCiqKF6Pu5cHchcSe5f39u/756Yy9X13YEyzkt/unjayj5Qnn60jKz5VNwthYgsBzq/rjFKHKQdlzLM4k9mHSqgY5mFigZkOvyng+HKuJhmKj1yvthxT1uiUinb8uJpK1GByXr6eAkpLhGz080Onj4RqVIun7Jo/FYXU0P9fNz8nMAWR1d/lmalW2B/U5nYmCDwzst/2OTl30C7qtQTJitVraGTnUU907MBwLJtmLfvn3BVqSdMVrzeqMVDewwPLurSp8HZHQOf9fpfoHVJ6U9Ltte/f2HnKjWseg39mF7kZpWEtdPFjkUXlHrCZKXqNbjzCQ++25gHOHzCg8MnPPjpgAc4fAKPeqK4bLtSmagNzIufI/gEXj4J3PBEcdV2pVKR+tXWxLZA5sRnx6BqrjCVjesJN0rMMzpJfx8fFd3gfMpDgzh8Uh22K5WL6mdeVPCJGjlpDsD/VavHuQNsH5u30ys+2Dnp6JEDXQ1tv2znYjn/apAo1Iw99J/qbwUAWLNyFzDJFC/dEbHsnyvOvGHw+OngctNH1q4bCq+GTwK9awV/yywT7JCTZtM5fAJI+IcmKuLAq8XOKcUN6qRi5oUR7Jx5qXx4DDmpZfCJDf/QREXsSJTXWrR+KNfeKcUN6qRi5oWBXpwzLy6ef5TIybHFixYXoI7DJzL+oYWKsAKiXiBK67JTwynFDeqkYuaFEeyceXFtU4kKOem21kobw3+v4/CJTIxooiKMgKgXKH16oVKk8FzcoE4qZF5Ywc6ZFxd3FimRk2NERL0er+PwSR8VMaJCRRQkyvbxGYFaTil9XKdOKmReWMHOmRdXV6EK5CQCAHon5IbWafhETYyoUBFWwI70BrM0nVLcoE4qZF5Ywc6ZF5cPG4OcpO4RvHAy6jZ8oiZGVKgIK2DorH79h4+GvVOKG9RJhcwLK9g58+LKp+Vq5CS2oDAAMKFJ3YZPGGJEAxVRCAjEPfHTsjVAEjeokwqZF1awc+bFlY/LGeTkWimAyI8DAJwK7ly34ROGGNFARdQCXgyKu27vlOIGdVIh88IKroB5qXwwyEleqwEAnugM4PfkxX51Gz6R8Q8NVIQRsCWmUauRVzcFB9693M4pxQ3qpBLMiyzYOfPi0ncHEnJS1G0cEZW/+mHauq7vW+s4fCLjH9qoiLYApUg3qJNKMS82wRUwL65dIqqRk4xViVc4fAJPoiKeL1XVF8jA4RPUDObhWWLEu+mc2gSfeAIV8Xypqr9AHD7BrcQ8qqNU1V8gDp/w4LuNeYDDJzw4fMKDnw54gMMn8Ch8cotCfzhKXYJPblEIOIqKRvEgjlK45Pzl7i80B2BdcbFe8axQcOcTRylumYx4Dkexdz1xG0dx5HxiHfMG0eHwP6rJ9ELP8ImY4ji1usMf0KBRPIWjvLawPRp95jvWgu+3vgT0uv/5atpyrmPnEyklEV7meuKhkJ1P1vYPAGDcVMzhE2WylCKnOhTNNJJdSxx5oLiKo9hoFPctWFCR84klKRgA7ijbzeETsMlSipTqWDTTCJJriSMPFFdxFIFGqYIFCyp0Psm9HggAwTgD7nyiTJZSbLdORLONJNcShx4oruIoxjiqggVLJZxPzmAWEVEaXq+WheG60cCAsE1FABD89LfFQPaEZih4Ja4RMNhnLQxB5/q3uDwKjQuLgEFp54D14cHw7bke7ZlRtpAHIejEeKCdf3YkcHdxFlDy3KjbAcO0tTuAxgAQcBvkYelqWexwZTDJ6stzJ6KZRjR9cC8gsGOYND4nLQ0wTGRKNRgWATR8bfNGjaK2mNRzIADhr4GqJ69+BvJLxsi3e/W0o2zy4LdRigBhA3RJtXxOoDP4xNnnns5EM40k15JdFXigVA1HqbwFCyp2PgkSyIxyBIE7nzgNZ6KZRpJrSUUeKFXDUSpvwYKKnU8aC28BpXZN6yR8Yp8spqya7Ew0o0dyLanIA6VqOErlLVgq4XzSuUUOAFyp+me7tQE+USaLKcKtM9FMAcm1xIkHigdwlMpbsKBi5xNDTBYAZIZEcecTJc8hpwi3zkQzjSTXEiceKC7iKOYiVMWCBRU6n2D2yUyANs6uqhlaLXE+YXkOMUW8dSh6D9tIdC1x7IHiGo5icz1x24KlMs4ntM6YcW3OOBN3PrFPFlOkVAei1SG5lmh7oHgKR6mcBUvlnE9yPlt6mDufQJ9ki2svmYHDJ14TtRdHqXPOJ9AfjsLhE2+JGsRROHzCg+825oFbs9HUaOCvg3e/FaD64RMe4BgaDz4JeNT1+H/onc6k7KFizAAAAABJRU5ErkJggg==',
    '1605.06770v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAdwAAACPCAAAAAB1pTXEAAAbmElEQVR42u1deXxM1/t+JpvIIrEksYWgsYsSsZQ2EUEbyg/9Fi211la7UlRrabVKW7SWCmpvKbVXiyKxV6ISEdSSSKxZkAVJZjLz/v6YmXvPvXMnmcxMZMR9Px9yZ857zvue98y999xzn/c8CoIsZVXs5BDIgyvLiyhENFuOQtmT2URkpxvgkpXZVJoyu0z3zogAgIK0/0pWFKU6bStp6wpbnJQqSL7nlmlxkPgu+lCq/3uVd/YuIZPZ9xwdqUBVzaNUOpyTrP3rVNPFBG0zQvGUOY/tXErERHHOXtFlZWbqvKoJEb7bY0rownVtQ9Qpt951+jUslcty4s8njlcd7krpx6rPf7WomqaFQhg/n8fBPk6RSfU75D2I8rsB65gws9NEJJxQ7QlUE5E6PFD6Rq1Zas7dXfgxGr2e69xC+DEe/YiINMudNhZRUTIUhgEQtK/2vEZE32I5EV3yKjKA0tE2K8iGThneczeF2gGwM/Z8lH7Z8l+Vc6ne6p1hDwCKMXMHnyxcUzIURQTg8bv+AOxhB6BJl9yiAigdbWsEWXIRI1F7Uwryk66wq+zMNyZWGlP4PFcyFEUEII291gekFRVA6WjvKqkVqvo7/gQAxTjh15lnopTJQPxM/RfK6AQ1ABSkp6Ig8SGA/FsFrKqpknL0poZrKEn/W1cnZSN/96kSPYXD4k+wPSk8FDotPgDS4tiW+fA607pBACWibaIN8wd3EsI7fXNGiWCseq1Ww++BATWbrMd3GzzKzw/Ethl5RwYM2ARQRK9niR1OASdbeM/+c2H0kKGaNStPtVkB6FVNFOWUw+67Ol3XN3Su7xACgB+7Hx47YmGtnuqSHF1fHOV7srJtrYbfA/1rNl1vGApOiwuAMXmlBfOhXW0uTuIAGpow3UZxlh9FE46tngB8fiSirJpDiEjdLJWuBhORpj0RNRhJRETLa2QRHfR8TESvha4nuo4JsUTrXB4xqoVMafpwx2tqXSGa/Kqab+gIEW0vl0kF9afQzRKYUF3H+7qjRRjM9CSzynAi0jRNlQoF319dAIy3T0S0GCtJECeDABqaKNSGlSZU6Jv060j/1HHfARVG7sgBbo72RmpsLKAYzOlkzexTAQiz+wWAe/xAoI5jcnOg4bMkA9UixDPnKRAae5Nv6BKAbfU9YN9iG+qW7Lo6NExPPIZufwYkD/KWCAXb3+IIV89oVPhom2ujeIsYnv36UdS7c4d5YtjcLaOwZTLQ2q9Fk7C3h3Mq/2Y5nQRQ5RoANLYD7J2aAHDGUwPVwiTB36lPb0XSxTPI4ht6CsDzEQCHkl7muIvabE9GLPxtMDaOkgrFBba/xRCu9aFGo8JF+18zbRTnnrsNABQhq3IuANV6raBcVQXAOepzl+VhH2j0SsnwdnZ2dt7yifbZgv9fbaBamKxzgCai3ebK7SFoAhiTcQk5x6aV8ODeRme2J/W6/gRVlrdUKAT9LYZw9YxFhYm2uTaKc+bu6AsA6IpcAKNDT93sC+B6+blzH2+YOiwYALB2WCNUbmWkRaFq4XLPDtMiLtTDSUCjUDAF3mOXej9Y/n8lO7YZfwV2ANuTUb3iEt+WDIWov2uHmWiCqycOoES0zbVRnDP38k3tGiyaAwhpuDw+AEDCVqDixD6XAGcVkIjm1Q8DQP5eiUstp1qkXEtBzpJB9YA04JDgqefYG6vnr9WO7aO8khrc+QURCkFPutdYdTxYMhSMljYAJgpXTxxAiWiba6M4g6t+LxMAVg6pAUAx6reWAIAVzwAo2wKv3gAIzqv3XwSw1BeAUgWAVCoAKqgZVePTDCgBILlfDTiWLwAQ75h7v5K+ITWA2lMjftl1OAtAhm8HK4+pznrOhLV7W0LQE4cPN/opJEPBaGkDUITkIV97o+HqiQIoEe3i2jDnxUHA99v8WjntTfnZAwAyW1wtB2D3qsAmHieqjQcSQ4cogoOBY592bHApJBynF5xQvD7z6oboSoE/Lzh0o16HTpyqkaX1mC9iU1y6uKrunlVP+h6/T/oo4Hy3NReCwxfpGqoSFOGZHfzIQZVe0Ptn12dtXt1kzRcHcbPjbrmFudLD9OCp1bRXCV1PANxteKuykVBwWvoAGH+f+8fmB1eeuTSqOrwT37phACWiXYgNswdW6Nz5QIqPUQe20P6EH62YBQDZ5R2uKf2dAUAV71dJO9d8Ul8h0SqjatLrbPVVVWMnyhbMjHNbr24LaM4P6jv7ebxM53ty29d4KPRaXABMbP/uk/oKyQAamjBuw0qDy1y/brbEwnf9SgGrsDPiL+3N4M99MhKjZJAY37TKzsnzKw3PWlxKBgD1wTchi3UXMfTS9fyZw5+Xild1dk1t3Nrl+pGuo+URsi4Sg5e0u00dS+vClZ6UW7OOnQyQK6l7rox+lNGPstjyPTdYUfK/o9L9FZfp3klLsK1eUmSBnAgmizy4kLP8ZIGc5Sdn+clZfvJzbuk/5zo8f7NPk7R/nf2KNJ4Vn1EhVL7ElsTacgnJw22njtUc4oKMI9W/Dihc9f6WjW+FInHw6P5WtJ+zNOV+swk+ADSr7zg/m+jFlTyOyM5VjW0gVDJD0sav0r3B1Px+WWU3zsvQAGOqZCdUeN6pWNcxmIgoN6x8dFE1u/Qh2o2hVrSe3u8mZb3jcYZI038e0fn697iSselEC50OC5SK3/6UEUG4rz3O6fyZkjb1MDTAmCrBkJfK4N7GMCIiOolORdXs1odIE/PEitZHJxFRTqU6BbTbI5+IRrzDIcmdfyNKx2sCpWK3n3erYLFucDXvvE9Er3gZGmBMlWDIS/U51xf/mTQ3CHS1otH9rz0G3IKTruKX9k4Agvc905VUJQCueCRQKraUq22vPzy6YzKAX/fAwABjqqwuYhzDWwD4VDA+hUyQm6XOuA9xnpjR5K2if1H5SgBuSFMf9gCAavmRupJ+Of8DTqE7q2RR/37yeBVAq3YwMMCYKqODe3561+/YVDAuhYzLngIAXAnyGifKExMqFE9O3PcBcNGhafpjNwDwAAfxdwSU37WaxSpZNJ05Vuf2p19PZzIIeAP8UVmbUN1G0KJFX/f1WakRpIJxyVJMllm3PkQU3IeEeWKMQvGtExGdw2S6holERLGYw319dlrQyCcCJbPa191zM9F+kZoSa/xNEgaEpsrehCq5Q2giEe2oGEO0H9coyuMCEa2mTI/xRKSutFw/uN36EBF19VITFTguJYGCWYOb1ywsly5iGhFRHD5h9iv4782301klSwY3BfbJRDS4bq6UAdZUCYXcoRQvy7X21H4zxh1cKpg+hcxYRhSXJ2ZxytRMr93OcIc2eRzuzNyt/uYab52155UsEVfUqgWg+frToRIGWFNlcUJVqcu1/eBTwfTJUsYyorg8MUtTptbeO+AKeCIXAPIggExXDoo5yChZIhWcKgFAeSRIGmCOys4KFSNuSAWfCnbDRZssVUiWGYTpVWY+DMVtscMlu0ZVUwHgAVrrJuAdCs46AZVwnVFqbElsmz3V/hqriA0ITZXNMzcvUhHKpIJF6JKlCskygzC9yiyrZ04vtQP2lFN0SwKAxEqB2myzvJiLaQBuozmjZFH/etxUAkhFK7EBganneeZm3tEf1bS75+SgUdnXfXLH0ZGUild03+c+5LUrumanANUqA8CVcg4qegUmp2LljU75IoBNBVsxxgVQtnVe3fdiALC0MwCVPQCVGgCUCn2eGKtQbLk6IGw06NnJ6ZgcmFgXtHOyPTJqN4pBhfBpNYGUhODXWSVzJF+XBjZq8cG3QQeH+4sN2DOmnitued1QpzB/HwWW3d0esCHqlNuAwOG31pyKLDeo+RidRsKK07EV33dF/p1zKYsnXvvtyi9usfUATPv7Qkj41CJfil2ZGZdUIaw8Mu/6TO0E8Klgrdbok6X0GVHRX55UtA8/cBLt1l1fwOSJsclbxX0lF/gvACAgDti2ckW1+ak/O0KbbZYxJaAdTamypqpAqbjtK4ekxz/19Q+ZCCB6zHS/Ndk/uRsYYEyV5HtIg6n8D7XiiYj+REcNUTS6ERHRFQSzOv+gJxER5Q6eQETx/8PrBURE+R01Jj9pMlJw6UI+aTIpS6m5elH32HDnqqbwqX6RCqZYT1237DzbzIX162I1lj7osfJ495o4kjRgnilLn3O/Wk9E9MSv3DVm35kkdJXcjiajPxHFr5yAb4mI6CMyZ3Cf81P2C9e+FV8cPHkTAObc+sy/qDNfpURlTwDAV/U/vQxZYPPoxxBvABcWN5la9CrtIaArAMBlg2qQSg6m7Q9uZwWgHqGJcCp6cAH01B62/STmazmYeCFgNj/GjH5Nd3huAAA8EWucG6BKOctkRs/e/0X3lnI4X4DBTZlVjTsPW28GgFt7RCqtN2vufMh8Lrcx6IPz5eR42txlWbyzBn309EcPAAZbFmRmZGRkZOlr1hoPQH1R9/HV2QmzC81KQqnmRL3I7ZvtlB2AKNH3O/a/3RtAXqS4QncvLy+vPtzHFk2Aq+f0n6a3WnTamKWoUu1n1AvevtlOSVyWM8e7LVcAuG6w9eL2fOjfzQBAdQBnuZV1h40tBr0hXwotETpu5kXgZHuFiS8Opj+Y7wsAGyuKS6r5+fn5CVbM6KdGgDZ3AY2+unHXRGei54//8SF2FqmXfnzn2ZdobMeb+3a3gvSW74Zn7slVQR8BUG/9YQ2QhWwAQBayYLjyj9yZdysCCboFjIm7TfRlZuq8qgmzfbcXSbeStHHduLbWjiGLN+fh44AApD4wMMgl5UyXUOuC0jUbbudqPnwF0qD0JTWlNssTI+d5z/iSgDZfzzQJQxWGZt26vdW0HLAvuncDd49OMyi+TyM3946TOHKRHrXg8n/vv/+/Np4IpXNhHm7tNhAR0c3BJi3QFcF/IpCACdZeHuTx5ix8nEgIUvcB4PiZxrqgdM24i0T3Op6RBqUnB6p5V1RSTpHAM7ZE0/4/G8FQvTNV+/bBlMENtPrg8nhzBj5ORCQAqXdePmN1klntFwJK3/c9EVFcmDQo/YOf+ONdi6ScIoFngpLfetsIhirRrlD+kxKW/bsTKsIteNfVJuVqC0t0IPWhz1wA7zHmts+0enTHeQC/cguz/zwDgHqJkqD0x/tX8B/y8qScAsB4JijpNfJBVZtAYrCMHAVpaXiWpBFDzTmYOiu6UtWDh8h8YrZ1o3hzA5C6xSIGpdf4fokG2BcuCUrf20QKsmXcKWGJQ9Au24DZMIwcB1r6zP1i1t7O05Qs1JyHqQMAPq9aq0s6V/pHQLXVy3+sZfbDpVG8uRCkHrd40eIsC2e/YlD64DqTQv7bu3k+pEDpx9tIPi8YIOf1nolK2h+3EVA6w8hBjXziiJ607K5hoOYMY0ngBKLYxkdZILqmfu+T96tuteh9qx5vzt5zBSD1pr9oaGf9W1YGpSe/BqfXldKg9DdWMx9+/ULKKSLGM1HJ+kAbSQTjGTkAlzcDANfP9+9k2DkYxhIACRH/dGS5OxTuN9tXvd8XwIH+9QbO1RJVHetbdalWfV6VD4s8q/OHhc2H4QbYTtpNpXIBbOmvQC+3yRZ1Mxtn37VDnc4juBuost7Hdie6PwCANgs2J/fPYJ+fPCG5KzfjFMB4JiqplGYr0Fae/0QnofijDw81ZxlL8PfABDcIgejNdLXCrx/+Wbc5ZccU9akJAHAqqe5q00DpYhGA1AMAoFVEuhesCEqPH7m3yqiRh3qesZMApadV0D7sXgKAm9m3AKDGXEPkvN4zUYmtDO62vlpGjt4XOnKBdb+OZHg7A9hSA5rV68M76hhLjvi5TNwIMKUAt890ZDC38ejFATMAIOdGZieYAEqXeIHFgtTPKt8A4I6EEFgDlK4b3A8XVkG9w/Pm/BWuA6WfPMhj/Jy1l6BurQDg77uDtC4ZIOc5z14VlhQ428bgCvhPdFewnAYM1JxlLOk6sV378H5CILp+IVUTNZdfyGl+Pc8Z2DJk0igTQelivLkHA1LvkZXjBChRGVYEpWcnvA5AMfvItTAJULq3di3Q3x8A7jh0MHQKAOOZqCTT2zZmywL+E+1CZhTCeai5gLHEDW1mj0qWBqJffMydWLdq1y5/BYhsfzW7PUwDpbPfPcoDWJB685VOAK54NIIVQenlFTkAgFpNpUDpPg+kmhAj53nP2BIA6TYyuAL+E0Q9AJ7N7t2LZ+cQM5ZMde/zmOXuUGbrr8qVm3AX6BC7pvHITGkWFeSGIkDpD0ePGjlwjR94+LiWAmXy5URoQep4pxGAW1GLHSwDpbsehA6U7tsBcOy7BADuPnqjQvhRA1B6y38l2+Kd0jrJe8a4C+DfQNt4FGp2ePiXfx2d2DuTiCiw/4w1v4R+lk9EdLTdzA1T/yDa4bvgwBf/jmk3/Y9uFXx7pe3zcGu4giv9K9zd6831RETUozcR0f2DRDRJTR9+TMtV1GtGEdZ1YKAAovz3Old1b9xzMdHTpgOIiLYGJzyaMkBJRAWfLYnd2uQ7c9aW+VaJzrXaETPqvWy9gbxBIyLjIgYnE6V/8O2pk226MWufdLiR1KMQ45S2DcYz3l0iarXH0KnS2GRMwMjRqun6lJwGDkKeECnGEkM+FHWVeeMAzJrgBUxcgmX7p9RorPH6tYv5m4ClHXjaTkcUcvlspXY+Fm8ylhmVEcRux3Q9+llAkAIAYuOoRYCgN3X/4m8CW2/MknJK5BlTcqfNLUcTMg6eM2w7cJDZrZ7HRSLa+Q4RxS8kiqyyiSjWIedFBaV/x7wjiT1QzMozv7Sx5GsAgKrA3Jrz9uFn10dxp/cC87c4N+jRrEV/LDzkMf/jynghZWzwver64+bFTP97dDiquAQWJb874qGlf5cP+bqRvPcjAFz4fLeZUAxN3yltbY/AQg17FNjZyYOrXbBJGG9exdU1wmV2Eltv/6mZGzVwL3vlwX1ptuS1nwMgpMRNhZRqR0PKdO8KcUreBh9ldht8mXoGMjuJLPLgygIbo54JlqNQ9iS4lF4cyI9CMgunLHjx9n60WeoZDZXIHqoaO5l6Bs+LeobJwhMm4Z2YEe7lCgDv2TH5eBZl+QGIXPGbZJEwwVCmnrFOlh+fhSdKwlupC0kIm49nUZYfET2t102iSJxgKFPPWMc6n4UnTsIbv+rP4ydOHA9JYvPxLMnyIyL6khlcvkicYFj2dko3nXrGmkb5LDxxEp7dCADYMMyPzcezJMsPwNk6VaSKxAmGMvWMdahnYDQJbywAJJ5+H2w+nmWS93t/vJwrVKVDPWM8Ca8eAM34BQoI8vEskh/GK16uRyEAuPgtCmIj54xUANi042DDoPvvnrf7b18kEHQEWPnV5Qoo1z1Jm0rU6N8QAB3i219Pm4lA/4GhYgUzJfthw+1f2CW9voHNP/m1WUUAcI7sfyKgzRFHC7t5oYovXr4zN+Djj6dvPbclLAngc/pSY2MBxWAmoQ96KgQAcI8fCNRxvARDBSsl4QHKGaMN8vHMF9WmwS/bIgZKn3oGgDQzzD5FLQCifDyzZdlHdi/vW6HSop6BUWaY9XW0fz9cUAX1Ds8595clBi7nV87MzCxQZea8hGduaVHPGEnCA1AQGSbKx7Nkvpx+ZxaAy96z6k5+CQdXRz0zWpfT93fVj1Fx4tlLw6ofHgog/2APGKWeKVzBNOnxpdJJn4Tn4gwAl59oU3/LK3IqaPPxLGk/OBgAdjRdBs5AKQ+uSbwyDD2NZ7Y0Pw1slXqGz8ITMcMAuKebuzn2XfI5gLuP3rAoyw8A1E+zAc4AU8QqPa/BTRTwyjzYreeVWcfyyuzi6WneebAp6pTbgMC6GZtPRZYb1Lzowb0yMw57+pRH5l2fvzsBzusnfRNwvkfaJ8GtGnzVxONESCDC947q2OBSSAtEfxmj6Bl+IAbd111fcEHRc+bVDU7rEyI8OQUzRDkkPd493D9kIrwPjVH6ran3PeBSV5v0oNINLpaNHPlexX9O/+RkSfsAMDoWF8O6TNMa4IuESs/xxYEJvDICehpj/DQvAPWMmBkmf+0Dbm+bLav/0bzg7CQS91wX3ens/O04AAitvnTJFABOjfnFlifzmgJ4OrrcKgUAZ9081hnm3VfsmwBQeKACwO10WaOoSjWs8cv27Cn87DSUO/T3B8osQK5QXhnT6WlkscXBLZRXxnR6GllscnBRCK+M6fQ0stja8qNJvDIMPY0xfhpZbHBwTeGVYelpjPHTyGJ71DPGeWV47hmj9DQy9QxsmnoGgBFeGZ57xig9DWTqGdgw9QyM88pw3DPG6WlksWnqmUJ4ZTjuGeP0NHhxqGc06ucyZBpTS4pBPaMxk3rGFF4ZAT2NMX4a2Cj1DIcHZ0HoAASkNFLUMJaD0oWENkK4uqnUM3xN3kdTqWdM4pVh6GnICD+N7VLPcHhwBoQuIqWRpIaxHJQuJLQRlJhOPcPVZH2UqWfEyHAGhC4ipZGkhrEclC4ktPlSNLgmUs9wNQU+ytQzIjw4A0IHwJLSSFHDWAGULiC0EZaYTj3D1RT4KFPPiIQBoQMQkNJIUMPAuqB0gxKTqWf0NQU+ytQzImFA6ICIlMaQGgbWAKXzhDYGcHWTqWe4mgIfpahnSuOy3PqXUUePwmfWWCA8vPHvhwIw/I0rexU81JyDqWv1+/y+rCMPRO8W3jB6cr0VD6zjiw6Erv2hOwGIjpvsBQD/7DzWcocrrAtKj7s8UbGr9aHaUnD1GwOlWsiBKwDY8w8iTE3Wx9p7bSSFkx7/OtIf+Jbfknc3dlCmx3giUldaTjsqxhDtxzXthOrSmBwippQCm+ua+aNf3QFztPPKo+/6LNF+Obfy8EjTeH+IKN83WViU1yxMhwXR/Pfm2+kW8QqRcpKaqA0zbYojImrZW6KEqOF2qc20L2IaEVEcPtF9IajJ+LjX11ay/EqbegZiELoEKY2YGgZWAKVzhDYScHUTqWcENRkfZeoZA9GD0CFFSiOkhoFZoHSgQJVp7y6ijfEWl5hMPSNuk/NRpp4Rix6EDhEpjVKCGgaWg9I52hgpuLpp1DN8zbFCH6WoZxxKhXqmnhHqGS3UvCMHU3froKWeaVdbEogupp5pYRL1DOuJDoSuw4yfOb1UAezph7wYu7SaImoYWA5Kb95fRxvjwJSgCOqZCzz1jIsz02a20EeZegYiPLgehK5ldeFJaaSoYWARKF1EG8OXoJjUM1xNkY8y9YyYGmYvZmoXa5sOEJDSSFHDmEM9QzSqrZtHp28MaWO4kuJSz/A1hT5KUc+UxuDGkCZubcR5jZ6dJPmSSgQ118PUCweiF3j+QET0aRoRTSD6seuhBFJXOlgM0DgDQhfLhfXrYq0PSk9Yu8uYxYJalyUHl1LXLTuvKcrH29WVNrKbjUw9I1PPQKaegUw9I1PPyNQzkKln5C15IVPPyIMrU89App6RqWdkgUw9IwtkdhJZ5MEt0/L/wAXSbq+sTy4AAAAASUVORK5CYII=',
    '1606.09370v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAWIAAAC7CAAAAAB6ccv1AAAWlklEQVR42u1deVxV1dp+DsPhwAFRcMyclXJCcUZNFLU0h3Kqm1qa5mzpp1ld+0rvLbuVGXVvOeXVHDMzhzS75k3BBCc0kBAcEGdjEGQ+cA7n+f7Y57D3hoMDenDRt9/fr1/rvGuzXtfzO7xr7bUenldHaOZcc9Eg0CCu+kZygYaCs2wBSTcAECIfL1z4J4R4IQAdpf8evun+jAuvjlourgRzu9sHc2MAeAa5AEB8vh5m1yAn/rMqEC3reG5uu7Z3N/zJxOGGSl3uwLuwc337ugCjrST5eldvQ5dxfLCG+4x2qgMQdneh4lww5nb9KSkPclJ3DTFJox+wUGq+PZ50JsQVihZwtxDvBLrcrv+1BQ9yUveUi9f6YeE3Un5xdf7v1z1H87nbkZ+e0fvz2/WfeHivHs23u+PlqMpKYc6L5vbFgW636T4a+XCWOwBAr1XjCp892sT+sfBYkkvDQD8AxTG5uY1b50SbWjYGkuONQdWlBxLjfVo1q+i/TRWtODY3z9gLqcm5eUENyosHy7nzTVvooQqedzonp6dPZI12OvvAMdm5NbvYxziZ83hzALgZlVIjOHYg8ve9gqtH8EhDB/OzP4vi5N9vPhpcTTVH+wAVXu5IGhPIt4FWmeSCiSQTm9RZujTEa76VzA5wwYL1Ty4agFfzX375o9rV95O0bq7d4V/v+Y29VbFcrI6WHeCCvuSqOsB35cTriAEdJn0cWuOrYlXwk02BvaHD0cViHzrQDWNsY2wcuOgpzCW5OvCLH1a2hJmf9fBAgx49lpadX8mzjGhW741FAf5HlGFKBriv5S6BLB4F9C+SIF4M5LGoCVaQ5CCEzihmNPDkIXIGupJcj7Z55I8YV2GIldE4CH1JXge+KydeR7RPIc3dsUQd3NoUT4T/AFwpGft5aUcxCH1mFvMocJ4prutIXoCZZDNIy12p+ZU8y4MwJpKPYLoijHKA+4OY+V2BqdYFE0kmDXmX5BgMJskx8Cwg84FJJJfClyzww0ckrfXdsioKsTIax6AvyQIbxGXjsSNmkeS/YUhRB++IEJoWrZbHniJBLI2RDexiAtp8m0ZuoQJiB/PLBnaRgRhDcvmAI4owRxUDqCfldq8J0nNH18vLHwMANP3BknDqxhlkSj1tDYA70BmAO0zA6QyYowHUuZZU4bcURbQyVjqe3VrAFP10qeA94DH/NmOYEBD02/NoMvAtZafj+Zlw8xRaA5gyRTlHVwcDVGS5A4C6P3bPmdOtFYC0dzc++lK3qGj1nqlk63QDOO0KYNTzjSu+HMvRytujldmquQNXSgf3v9M+z+WX99elJy/debye3Ol4fsB1QF9mjk0dDFBRiNFmyyDr4VZAXs+zE5fqsbm859oCoRPve8tjj2Y3yx1/Ig9oUzr4Hfem+UlLPrmwf9WxdW9Knwumryl3fs1drGll5ph/Wj3A/R3JD/gnAODQWczTA9fKe6xBN0QAAKdduh+MbdEAAwjgzG1PAwDgMBoE3XPwuH75umaTwo2pAAwwI39v+fPzfAr7CSBlhiLMbsUAFYWYmUWptvPGGa8BwKPAYSDmElLSHR/kbfDfsBMw//30oxU4OikdDeiMdKBgiScuWcv7qe/jgIQV7hu87jk4s141A1cKBgLognhEdbrN/FbUPB5GFExprAhTXzFABffFMR4GH0+9bVU2j5pLcm9nt6eGjP9joDc+zfTy8jF6JYYZvbyNvdjR6O3lvZJMn9EwKLTDG7n3vmlzEI3mV326j+x7sA7Qkw7jdZyYOHDAgIbDz5JUBD9k9PLx9oqQBy/wMvp41bOPsdjb6G0MPVp7Wsdn+7ffTpLpA3Ud2iY6mp/0LJk6qd7jvTqssirCKAdQT+r+juTzM2t63CExWnwf3JG8JcO9Bq67e3qWf2ZhyvXXVSA4rEUGc5qHfVXMy6ulu8P8sty9VGHUAygnpd16aLce0C75NdMg1iDWINbsgRzJN9KJsfr+CeFtpFFVtE3b/y+qivz+JJ916fXqrqQfkur278pvRqPwPAHAvbGHBvE9J4qrjf171Li1rX5/XA1/Zquy5+JrWbO6562tkf/7N7iydsOZsa2QvqvjP2uJkChe7NjZ6/LhJ0MBWL+6asifXQtA5srsAvPMx5ye/e7hYokk+dlwE7kbX5B8501lx16fT4qlS5vlJDnY10wy65EBFaWqPFCrA8D9HStJ6wt/J08EXCfTZqaRH+v3OTPuvd7dkSSfzCA5D6dJLl+n8P/H7VOpEYlEkhbfISTJ/rglAsT9v/zrV8kkyR2+hSQnjyTDDFvINHR3NsT3mouvN6gBILz24wCuPyP7L41+YrbUalYvAEBsVm8AwBmjpwj5sPZ0e2tTDz2AkAn5XnUJwIgM0V49UmYDyD7ZWwegdSsA1ouFAPBmxie2fW1xbx2AcPQGgITLg/RCrT3F+3wBoF5hOP6SMwqIxGDRdhRBABBZ3BsAngNy/urdbsuC9kj4tksH2xP+/wCAcN92ACzzA5aJAW3sfovbBF8gLdMbAHxx9mm4A0VLOv2veJs2ABEIsX2nu82bjprr2mMLRtk7PRoBKD7YMQ3ZcUtaH/QTA+HTs3Xbu/zcCDkwAoArsgAc3Xagw1ajkBCH12oJAOCkatPAfcHAQXRQTynLezN0zbY+Ikh+2BgIDHt/zvcwSffzOhQA6Nrl3KwXVtcUEOLc6OFS3g3fNe2nc9HdRgJXYd9eWm/5AQjH7D4ipeBAAOi0Mq2WD4oBwCKxInQBG+oPPOIq3kmbLRUD+/BMgzHrZ+gAH4P9+7rmZwAI13cTCeEjBwHAB/GojgIAMMF2q+ffOXqvgN9i224BSPF+0raNaHMip5q0Yn+zE0DxwS6eIkE8NCtHDxTBH751UwDgD3Qp6mk5ogf8cE7A8+IIWypG3fo6AJYoYKidWf7JPCOA2KxeQm3V2i3TA0jwbQndoGQAuODX0RR9KhXAFbSDIH9OI1uG2zBbK1pvIk3j95PWgR0KSVr/9QFJ8j38577/1uNB2vIoksnuq0nGG5JIa/D75KAIkpf0IRYnv93d6zGQadrNs9erdfdc5gUAH50ZkXBpamsAuXOj5jU9/+PzIwGs2XY8t33wYoHOi4v/5t878b0J/6MD8O2ypfUWpax2R/rcwGDOrbmqbiWcst3H5LIuBtiT7sUTqS1C3EU9kj99xC+4jtRM3ZMXHKQDgJhYBgXqIDbE2q2HdusB7QZag1gzDWINYs0A14WA/X34IVvvPyO+vaW3uxDtm+YsC7lvlry2L9b2xaiSbKDb0IE0NtADYQOVTweqImygnM8v32g7q47aKRQbqFw6UBVhA6X9JYlZI30Pq5xisYHKowNVFTbQtGSSOX5NLEqnWBBfm0iSnWtbSb57osR90a+P7evwRz0ryRNYQpJsaCwUAWJZ0KpBvQySw/C70ulUiO95R6GiA9UvBG79AVQdNhAaFBYB8EaquDsKJR3IuHJP2B63uusPuFUZNhB+tegBnHJro3RCsLs7km8iniRHW5+tGcn/rZbLhVisesDiG3rjxpmtwa/8IcTdHdtssnJbwEX7x2OYU9YpznJHkl1rWUleepctxpG518hQ/KJ64ASGhoV9tuuaINejjCXJDsNtn0xt+xWUcYoFcY7rKJLMzriKCLvQ31VbX/FNklyC/SLdQEs2GalSY05oblmnOMudIhX71IgwdJU8VYYNBAD/vr7HWMYpGm3QfvB4oLtHFWMDAdgdu9EFv7u0UjlFO5IvoQOF2894qwwbCDgc9bkLsNND5RTtW5x5fIi0B7563g7xsIGvH9YD/NL6FADsgVgQj2wJ4GLECjcgcWy/aWD+obeUTrE2bQXjhzzmU3/US3kkv/c1layAk9us/XXNyO9IcvXgOsYer4u03Fne+Sxmc+slVpK27XugyikU4UplhZkKslKVYQPdwamxgbRbD820S34NYg1izTSIIeYNdIgmXOMsC7FB3DscWhkrrYyVti/W7AEfZgIoxffJuaZ3sxTV99FYQA8Q4tTtSr7Pxe3rz/d54kUftVcIk9k/ckshEKRwQqTrUZbl+wwy5LJCLCAnn7TJ7B8FD0ghECQ7xboeZRm+j6VaqAOvABDL7B8FD0gWCFI4hbq7A1Ba/Scmu7cDrwC2u3sm4B2SnKho2QWCduUrneK93an5PvbrPOFYQDL7R24pBIIqgxxU4V/RIb4WkuZnA26S5GCDyYFXgERRXEiS7dxSFa0bmCSRPcIUTmFExEpMrf5jOSjdRYumCQS46AEcj51TS9FSCATJTvE2bWr1HzkVi6UJZNvDT+y3SNlSCQQpu8WCWK3+I6disTSBJJtfa4dB2VILBCm6BYNYxfcJN3SBgCwg2Nk/HqqWWiBI7hYLYjXfx/KrPRWLxQKCkv0jt2SBIFW3YJs2Nd9HTsW9hEO4hP0jt2SBIGW3aN9iNd9nt+2TaCwgKNk/cgtzOl5oCm6b46p0irUvVvN9Pnq6prHXVGsFWUBO3hfL7B+5RW4Oic+YO7ZI7RSRDVSFj+RlgSCNDQTt1kMz7ZJfg1iDWDMNYmhsII0NpLGBNDZQ1d8XP9AyVuc/WA1HRBYtF+OB6NYAeP2MYyILxKCqwPr9abPLq9I/KPW1FdLf8CudYhwDlVvGitzvUqc8IosIVBXm9H+niOuHkkybO7kzbkh/zVbiFL6MFWkZE+pudUxkEYKqYh05hmTzWiRNFy1hEsQKpyhUFVUZq9aqrjUv1jbnwyGRBSJQVfZvnQPgm50APBrZa3konGKWsbpVIlwD5Bx+yk8uuyUMZUXmoiz3bQ+gU7Cq26FToDJWP560C9cAWPw6aiCzgR1iUYRrSoRqeKDJlZXeWRMCVCuRI6cItx423RqFcA2ZPI9cggNiCtccwxzeQo/FxbxQ/7+SS8rFpZziMDMl3RqlcA05NZ1cg21CCteY2vYr4GW4XiI5vmmBAuJSTmEIV/YyVjVmXzu3CjAaAURe2A6cLMnFYlFW5tfaYYARDRsCaPd1lELB0aFTgBtoexkrH/xkF66xLntLB5iQ6ZDIAhGoKtX0fgDgiXgFmg6dIpWxkoVrNj7XB0At+7dYKMqKnYvSNg8AiqGsb+fmyClSGasS4Zqc/w4BAD/7t1gkykoJF2VoUhGAFHRS9jp0PnSIM4/3tB3t2oVrzJMG6wDAAykQjbKSOPbmtKlTXlzVGFONewHufaWFdFCFQgClnEJs2pS6NXbhmvebGx+PJDk/0Nvv2TjBhGsUXJRjnbZGTx2dTbJwdP+6Pq2eCVM5xReuqQLnxbci0jsH3pVTo6poVBXNtBtoDWINYs00iKGVsYJWxkozaGWstH2xZrifw0zHbCCJ/qOrX11jAzmLDZS6fcOZ8c3NiTFvTBCMDZS5MrvAPPMxKNVqFBWsBBSuKZcNJNF/fsAKwdhAM9PIj/X7lGo1igpWIgrXlMcGstF/TAgSiw0UZthCpqG7Uq1GUcGqEoRr3O6HDfQMyujYXIanWGygugRgREaJWs2EfC+g9nRb9+4d8TXgHbI9sbWgbCDAerEQygu9bzFLLDbQX3JGAZEYrFSrQeVWtarIr+geLJUa2TPe3PTsbwrFmshqH7OCAjZO5FEUDuh0S6lWQ46J+fTjT2+pdG3EEq6JgPRCmNJt3nTUXNceQPHBBtutlxNdI9oDggnYHN12oMNWo1KtBog9PVu3vcvPjSpDuKZCrx7h0hU0J1WbBu4LlnLvyE6dg06FtJdlbTb/5LH1qzoPHWF0/XDDpRfS1Wo1G1/QYZj3HFSGcE1FEoWtihX3Y9qPn439wirVrTpJcp3unPSIYGWs0j06WZIxV0oU75euYKWsaiWIFJ6dDbQPzzQYs36GDgDCqwUCKGIiRBSw8e8cvVepVqOqYGWraiUkGyjF+0ldiY7NE64AjthYNeKwgYp6Wo7oAT+cG6hQq1FWsBJSuMbOBqpbXwfAEgUgNisEAH5DbWQIxQYyRZ9KBXAF7ZRqNYoKVs4XrqlALs5wGyY1ovUm0jR+P8n3cIQkG+FW8WiS7+E/oqiqDIogeUkfYmG8IYm0Br9PcnkUyWT31WRC08lTpkwe29gizgu0ig304cu7F8/8XaL/9F5A8u84vmiXWGygtJc+iTzUddANpVqNooKV8MI1WRcDVDmXh2N7thXtSD4mlkGBOqjVapxfwUpjA2m3HtAu+TXTINYg1iDWDBobCBobSDNAYwNp+2L8Wakq2ZeBev4AkODhZmbz0v05lwAAXg1L/2RWXHq1UA3Qu4D4jx0Jm7xjmgFY89/fej89r3R/2uZfD9Z7RZ8XlzVruOpX4MbGdQMdQnxh/LQXHs7kZM6P3FKIAcnsIAVPCJVxmBk3Ck9YSLKwj8Ni6nF4gST3GSYUl2KxjFB/tn5OktyBCQ/ppK2E86NoyWJAMjtIyROqlMPMuGWz8AlJcobDHzuHMZIaBb4tdTRbCuKUKRLS0bkPCeISzo+iJYsByewgJU+oki75P/jp7YF3vmhpgvDnbvvAdtuq2vFhZcESzo+itX/rCQDfmJXsICVPqJLe7rzWmseZ5VRy7nCBo6euoZ3UKDoeX6zwX96fZAWAuPnS5+L0G6pxLGkpsCQXPCTYS8SAZHaQY56Qk1+gu70Z/Q97e+/Q30zz3jaXeebclqETAYArh+Vf6BlpdxfN3eezve854Nu/mn4ZO3Y9EjrXelU5zqGg2gt++vjY8y9Xwj44NmxxWJaqxQNNrrz9j7fOAmmZ3gDgi7Nyq9KWu2Wkqa3bCSkX72yRQVqnjVbm4p7fbVk6s9/X0mL4Zf0scm/1TFsuXtUwgZzTvpjkY1IuZsgI9TjdQ78mz+EXp+fiNpus3BZwUdmSxYDOYjZJxmCh3KpMHoXHOrxUCAAFM56vAeimbvpF0evfqXN3660WOgDImj+iGtDPZZOtr3pOHhAak6R43LvUOD5xLwJN3H93+pdY5vyUtLJx5DkXNOk/2SSzg0pVtaqsk7b2C+IXAEDM1TYA0Ea3W4lZ48btvgzoexoATmbpDx06FFXT/ks24mbH5J2/IqvUeKpxWrkArvo8p0McCACdtqUpWnYxoAtRci2rUlWtKu0w861Oi6MAnIFBqhqXUPqBEaYVAHAJtQ0Gg2Hjmza3dWXwBv8eZYZTjSPV5Sp2NsIy50duyWJAMjtIXdWq8thAbuuCxvUCmiEXAMyFZd6kq+MUALSEv0r05Y2VvzXDIcCq0wHAvydK7vLHcZ7JnB+5JYsBybWsVFWtKuVbTGmpb/nB+WtAh1qHAeA4BjiAmMi3tHtkHwAU/mA7wvhsXDMgFfg5EgYzcMH2dPnjOM9kzo+C/VMiBiSzg5Q8ocqBOP609P/ZTwAwrthyASj+8LnB8nmPlGgbumZcwPp0w1e7TwH4vAFgNgPunhYAce4FN/zQ/jxAAOY89ThFZgA0Oz1RjGwJ4GJEmJuipRADmnP6AqRaVnKrUjZtx/r5egevJUkmjSfJn/t8sGLogiJ7f+yIlj7V+y0gucxrfvJckvuD56+d9yOPDfXzH3qCWxt8uOe9k9OD37IwqdHCv4Xz2FA/v0Gp8jiRQ6rXGHrk6z7ejUdlOnfTJnN+5JZSDEhmB8mth8QGun4rwGHKTvmZQ6sDwLXcALmcQXGiuZWe2b4AzHGN/e5iHOcdycucHwX7RxYDktlBzqpqpbGBtFsPaJf8mmkQaxBrEGumlbFClShj9afcKmmJQoNYswdr/wdc5pc4LXgopwAAAABJRU5ErkJggg==',
    '1606.09370v1.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAaQAAADECAAAAAD1pAoHAAAop0lEQVR42u1deWAT1fo96d6mpVJaVtkFn8i+1yKtKDwFBPUBsoiiZVWUVRQRReA9fSgU/RVBBAVlEUV2FxClZSmbZaeAUNZCaaEthZamS3J+f0ySmSSTyaRJefDe3H+aO/e7537J15m5c+/5zugIrdztxUf7CbQgacUbhWSs9ivcvSWWpI6A7t66L3nZXW9/+wpwT7vc3QPFz5POBYcABLfyAYDjtwNQ6tvqLvhKBYfMH/zrVtN5DU1X/f5Ar7hXki2pVNarvSeB5SynH3/cBxhoIsmJHUKD2r/EO1CgxqvAxx+Pa+ATNvaKp3AWtEeq6R6eV+AF9443bAigYcMGoQASVMF5FCSS+ghgmvBxyhDybggSST3qkuSF5qid6TmcGW1/bXTI95J7IGm6MhZvqILz+J60NALTVgpXTt+77VpeZxIuveU1tLbjsHeaN6cENRIGnb8zz0kPrPXHyyl36R23DrDde2j1gG3e9W/4uYqfOAAAOi96qfiZvfWt9eJ96T51mkcAMB4qKKj38K0/DQ/VA84d17e6DwCKTx4Pa9LwzgSpDKjhPbQ8oLEXnWuzuGWrsju14vDiFFzrecNSO/VQ36LCabWnELg9sEvPH5b12f1Z/TeKXpmR1r/+NoCr6rySe7j94Pw7EqQUYJD3YrQA93/kxZCfAMLSUOGzO5L6E6SxL9C1hO/Hk+THQCFL6uMLkuyBLq8Z+SfQbSf5GjqQ36JZIfkTXqrgiUOtc+fObPvA3/cdkzcmDuEzZkzuV73629e95R4mxNfFQbXf1s8L/xM+Sy/u/e2Nz4XKc9tbhcD/kXMbhwO4D7t/8kEToG4M8DBOwjAGL4QAT9Va/lmlCj2HLg8FdLXef6GuV9CCOsGQlxVeK8J797fcqAt35mHWUoLXdbi44EHhc4MNZSeOZJ5CnlBtFgT4A+0A+MOAtFyU/gmg2uX0Cn7u3epNsKA44MnghNfve8FbiKPxZvU7vApe/acwjN8CALg2KuIf51pa5xFhNn+QCaRt3bp1a99/17vnFmd6AUu8CBfcJfCOnklA0+97mHY3AVDY6a/4zwPwnRO7ZkCX+Ht0BS0c+MubeGvv+H7Sk58Jf3f+hTcDgMtOzGp3RDIAcNSFey5I1YCMfNz+DffYph/zSrLNK/OvvQEAuB/YDRy6gKzrsk/Zy6osWw+UTk+7/94LUk0wGT/Mw39k06/cU/BDgUFhwQFfCZXSvhNIcnM7v78/PeTqU6GYkxcSEqYPOZmgDwnVd2YbfWhI6EJef61Oqy6tJxVU3BT8UHClkLCQSiE7vQJ3KLhSSJg+LPgSefgx36rxTQ55wz19uP4R9e5VwKbf7bxI5VtiYVn4Pbrpl39e18Tvzrun7czqtJ1ZrWhsIS1IWtGCpBUtSP9FxXcagLh7y+e4uxquItwj+b72v3r3lvfND7Na0e5JWtGCBC2rQivQsiqgrd1pa3f4H8+qKD5DALpa9wFA9o1AFkdFADwVqDM1cNXXILICAwK8+HVMvFNc5/KMpNBHEc6DIGWvXXZqyAOlJw9NegU4+M2KSiP6RwCGhIWNB011kXKSUa9KTOUba2p1RUZS79Ve+MV+TCv1eT0K2DG5e5QeAAaaLxG3Pr2Y2WxMNcHhN75Qt5E1uE27kIu7u3Wx7Q4AXHws1HeSHjIjqYBz7GNaeqnINOwBV3Ce7Mz2DC8luQFfkDTgSeFgRvMbLjvOfc5AbkIiyalveb4ze6vr1BJ+24vkfMtjurnlWv905vcJ301emzC8HTLV7aRWA+A/1STtLpSS7oOMXNX8msxICnhWOIc+ptePkFce2+0CzpMglYU/TZIGtCLJKi2Fo+PTHQzX3bQ70C2X5JtII7ngG4+DZOoziOQDUSTf+OKX7Tt2bI87Z24adY7krYj6ZTScL0tQG6Su8yZ/ec62u1Cm+98kGTNSZiQFPCucQ5+Nc0jy8BMu4DwJUipmk+RfeIQkm1YjSf4509GweaFt/XI8SbaraiL5XqrHQdqKVJL7U0iOJUku+dbSVLtGLslncYwkVQdpkGx3kve3IckJoYWOIyngWeEc+rw7niQLGriA82R2lySsJa7CGACokV0GwJQwwcHualiI7YGssQBuHojTAXi4icd3pAXhLQG0jQYwGgDOplhZ+rWLSwCEIrt8yHbdb2SEAkBUwSHHkaCKtmrXp9acuSZgY3dXcB6cSU+Hl5HcVWkWSfJFXCa5eKOj3XdT5Hr/jM+9wxYyVWl5/p1/vXXKUjf2yLW2GYtJsoVftntn0qE5s+bcsO9OGnw7k+QsLHYcSelMssA59Cmqj0dPru+R7wLOg9mdcXvttaaLJ32TWwIAaiCzJvJ2LZY54/4h1z0ZXlrpuJnztx9m+Jx7dOnjQn1ls8riqlcAgP2Hx0e5hXg4baxubfstde27BzbNAYDzuOE4kho4hz5BSQN2NO/wu78ruPKfSamYdu7s1nbLLTM2bCQnpMsYNpcl2XWIMnnnTLoI3wskhzQoIkkW175g225o9oTQovpMOkySrZ+z605yvS6TLI7Gv+VHcoInhbPrc3rwxCB0y3QB5+fJLalXPdR//YX2D1jOJBwIt3mMLRtXBMB0YQwAPNvDpnfBn8/pvHMm6VGnDoAWS1K6AMBGXR3b9nei1gW5h9gcANouvBZl373XJ8M+L/22x+5IyI6kAs62z9ERGyJHjtjSe7ePIpwnQarUHEAJTwpBqolM0+xFtuAvGQBsfm4IADSy7b3LGOelZ/9KAREAEIzjXQBgSX3b5sVXfnZTgGFPSWcAYTge59B9/D+Si8e9ixaQG0kNnG2fYbMi0fC36dN+7a4I5+fBLelRXwB7IPxboQYylz4fbGvTFgBW9u/kfGrojZWtZoUAYBQcKUt6wqZ10+HlPjjm484cslf+rQCgBFVkutetC1yo10JuJDVwtn1uHn8UgO793//qrghX/in44fxYADiIqsgVgnRix9NyhrsekZ03RD3krWW0XuklALKEf4m0AnM6Xq4BAHanfOoDrHfrXGoxPwDAifCHpN1zDQB2PJsP5G+f6mczkno4SZ9cAxCsuwUAqNNUGa78QfoZnQHgOqqYXgeAkEp7psjdZbJDQmWO5u3vpPNWkEbqNwPcPLQRAFyBMNz12p0AnHwhZ9TIEYMX1QOAYhSrwuvzEIDzyQl+ku4C3KYNl4C59QdDOpJ6OEmf67U7Af7PzwWAy7mdleHKe7n7es1+/dux04D4905vGSBc7/rJJv4nO17WDKNy/gre1y94fohXglR1y6sl9RY1nAMAKDV/15AGDwEYdHYhADT3RcnL146GdW8UN9Y13tAP9sWdnPHREGl3Ae7FY3kHVxz41d9mJPVwkj4CXuKIEQMr701ZEKAM5/GmH3cf7tQMALChm+wkamyPrhW+q3Yj+Xq75sLHkmU9qnm8SZe2JyJaFiXn1xutonWKI8ngiXAOfU7vv928nc4FXMXvzObe56PtzGqpL9r2uVagUbq0ogVJC5JWoGVVQMuqsGZVaAxWaAxWbQr+H5+CF+/yFOFAsXbCoEIFoDLenYUrucL9ra7yWpwzs4ips3Bv0Fa9Ukw+dzpIWaO+rIpdh+fenhjB/N2FnyjdNZ2Z1bt+qbYnPtiyP215qlaKqITm6gYj1vRlRtDtsVEyLRIyq3o4AEj6/HvZJiX3PJP3LIk9QJKGoCdIksMCjyhZOzP75p8e8e4k7E97nqpIEZXQXF1yHKympgHTydTGV2S4shIyq3qCLcnChj1km5TcK0eQpHzUKcLPux3/IkmuwptK5s7Mkv/uUZAk7E97nqpIEZXQXF39qKLpuvBiksP7yHBlJWRW9QRbkjOlQRKbFN0rx+XuPXHr4eSykzZ74WdQpmTuzKy6Z+J3PsMBYGl8PSDQTnN1720AaHgW+GN1KoCVpSrwRNMVMQEAYl+5HWLfgoXNwwB0/GJ2iHo4AHvqR8o2Kbvn9pmUGSN+HrhA+NslpIQkGef7p5K5M7Oj1T06k86QZPpwM0PM5kyajwQjuXI02SfcqJb/bzUtqzyAJP/AT/YtzEMsSX6EXS7xpCMXTTR16CHXpOie+2eSZKf15K8CO6g4pbM/APyYvLSNgrlTs6uhHp1JDQGY3vhWbjt+yKxxa748tWwFuK3+pYWh+a+oUF8XTa/lhQJAOP7qbteCYF8Kq2onH1ENB+CzN3SyTcruuR8kCR91cT+BHbTP0PIqis59cXFbrJK5U7PMSI/ntU7Yn1aKqD3NFWoYsbegBwBf5DtwZe3JrOoItgcja8s3uXDP7cudyEc11fvZnBGCsQkJCT+eMpHklZnO6KuimWnTO2PXiOf3mGGeKkdK2J+2PFULRdSO5qoEJ5oewSSSPIy3HLmytmRWdQTbknFGUnK5E5uU3XPjTHLgox66GGM+W/xmWp4Xdm+6tH2KE/qqaDan5Yycf3y/wnLqJw/z9ERyxv60UkTtaK5QxYhtAKPwxoswR66sLZlVHcE28TUfJ00tFd1zI0gOfNQ9bSqZ7zXtrc900dHrtjsxF81KF6zwifq449stzDSSw497GiRn7E8rRbSbLc0VqhixrVEEAAaEy3BlbcisquCqF1e5AZSV3vANs2/qrOieO/ckez7qgc6We02sGnPRzC/GB4jEVfMX3NH0QQ9j5Iz9KaGI2tBcVTJiw6tnAcBVtJfjykrIrOrgrmW8CyCt6rsNxts3+Sm65/Y6koSPmv6g5fEnVo25aKZb0gb4paqlaXlfT08kOfanHUXUhuYKdYxYXY9zAHA2oo2ZwSoBsSGzqoOLTUxMTEys1DRxPBzwlN1zc+KQFS1+brFBmD+085HmxK6tK29uZ3a6/j7zp+sR2Z5KTv8iee/aRzhPktdC2pAc+gFJZjxVzKz7NpCmDkNVwImmx4PSSVP0TAucBGSSz1FyWqsS13i2I5eFPiq6JzYpuudX/qckIArAzVEXT4QMbv2eC3N7s4vD1zczf0wYFuXpmWRlf4o8VXuKqA3NFSoZsU2WvPJ5jX82nGRhnEpAbMis6gm2ow7hyBPdJjngKbrn7qaflI/6xDtyd7l1Y8+7pq9eeGd29RPFLQHgWpvUKE931RRoqyJFVEJzdbVJJ5pm/1wY3Uon1yIls6on2DpvUnDP3SBJ+ajj2w6UsVgz9qJL+ur518cEGVa89SAADujTR9uZrTgG6y/Jjm9PS1uacmRAowkuJmONzgPwv+0HYNmeRG37vAKDZOq7yuON3a0rF/pqQapAjoPP66s8dSEldeE9tfWN/xj1pvyxLwr2cPhCvcYW0rIq/iuC5AcgVnePnf13NZyX8WLNy0JxvKfK+3c1nLfx4qBd7v7LZneeM1WhMVtRsQzWjHdnmVVX4V/PURbB9NvO4u5xmzvJz9bSN6RX79qBKwdeSM/uXFNxHImFp8xW3I1s1IoMUtaoL6sie+2yUy80wfWNbT6zW27LGv7keOPsX1edlV0EeiN/zIDCpSm3jw1MWbX+oHKQJBbqma3OeaWiiKoorOoGXt7Cm0Wlo8XtLru6LRvVNZyDD87VXuH+VoWZqWpWXc2v+aRtc2n7DSRNTfvI9d0c9omRJEdgATmvstHFUBILeWarEkXUnlcqEVG1KqG6wWC9NvoaOSvgNyucbd2WjarCPQcfnKq9qmWwyjBVzaqr7ApbMdx5wrbRK4kyfX/1myN82IWTZN9nXP0/9O3tgtmqRBG155VKRFStSqhuMFgTgr4nr8H6tlG7+kxVQZK45+CDU7VXtUGSKKeeqFskVV1lHX2xrayxQPcZcsyx7/mIx8z/OVdrmGiKmusiRqaoBOvnU39zV4NVIpJqL6I6SPUeooi3MvB78jasbtjWdy/v0MM99wapVntVqcEqVU6dMTlIyhM+cbFHAAAYzpnpwhl7igGgdxPHvm/lfmJ+ujPG6ZB2Lc5wpggAUHLBOle9UQKUnb0tLKNfiwPyBdHTkhtwT4NVKpIKoJwarCJe/1t9gV3oaWmxqRt+HOCuRCzUqr2qnTjIMFWRFN4CQNk7jecDKH03+/ENvV4AgOj5nYf0qoVnHPueWNW+tfljlQ+BpMq71zXu81VrIHNmeN584JPM2fjpwO8L9qQ/lDC7M4Ckys35RemSL1urZrYq8Up3lAUAOOLXFMDhP8r8Xgl3Cw/+QMnstu9a26R1WzaqOgargw+WA1JH3Zg4jLTeLjlxpEUIvEtm5qnV0UOvksyPfZO8FniKJDMeBiDRtBb7TsPHNrxovzXkS8NJDi1YUp3kA9PJgaZnHlxKjulHkn16m2ad3Iokkvymo5rr0w3EfGzk2VpbSbZ4mCRfNV+VzWUfxpNsusLENY3Pu748SfG4Z1K7EVJ1UrF+YLEN0VGVew4+2B4QHHXnniTDVE1Fr4SEuRsvkyRfbWYkWVmYFZT8Pq4uOpkc+3bB7xJMY+TLJGNfJS9P4KCnyEvYzgvvsVFPkr0HkjRGJnxxmGVnFZitShqscrxSs4iqjbCqOgYrSZpOPfm0NAnJUrdno6pyz8EHp2qvroJUOjo+Pj7+5fD4+Pj4+E0kD/iYNZFn4w+r1SFdAsk8WJeqSvsg27FvY2RYApRDHsUe0hj2FXkzxxC+iFwWZODN3AxsIFkrgeRRdD1hHaPlPDVBykF9gV78O8nZPS+mT5tp0YQWJP+7iKfCcGS7DJINHsnrgW1t5lxCfc4ZqguSPZyjD5YDUkddsYWcMlWRFNDRarWF/QAcQhtgrykagN+g1b6OfcOCLM+uX+v7IymiHXDgVgwQhg2FzwJJ0YEIwy9B3YDTl2MAJFXu9Xrnd3XuMFtd8ErNIqo2SqjuaLqiSrudm7tL2qu027m5e5o9G1UNnIMPCmqvriYOzpiqxu3txU2+A5E1AawKiQZWCxOGnIYRjn2bpt4SImxcuR7Y9qgP8EPLxjmVfZDcKgJIGozrkdgWEwj80KBtXpjfts6j+9Vs+1ROhE41s1WZV2oRUZUqoarFK+lUticAiMBpoUVSd2CjqnHPwQcltVc1E4cWt6wfH1touSW9I7bHdyRZEPEJybbCLavXJzJ9f7RcIT/6lTRWSSDZ+EMOKyW7xZOXkJT+AfnADJLt3+JrhcYqCTQFLeMwI8nnp6t7sPnAv5jke/iL3P7MDfLGfYtJ5hSRZMpkE8mZZ/jEYpJ8IrzU9XOSFS9f53uJZCtsE+CkdWHBoIdb7kl9yCkipQesjroxcZBhqnIGfhUPrq5jpKlfr1Iy32eQieSqtqUyfU1PtS4mafq/f5E8goNkAfZdfJtk37HkVOTNOshL2EEaA7dcH8MjOMhSvz1ZUxSYrUoUUZFXKlBETzQYPmLE8BfqlXFBCslz/l+5xWDtkUzyQkBsmRlOrEvZqKrdk/gg4IkHREfdCNL3kyVB2k2SX/Wspo+ZKP76Y8ZsGjLNSPKnZz8asGnruL435fry1vCmS3d83ecHklxctYxk7JcjbpHc23LdtO+bLRxl4o/hBpK9PxuWw8XVy8gXP3jtEskpb6ldItjXdvWfIwfeJHms+/YDE7tkkSxs+gJJ8zNac7Js6txD3z08W9XanRXv2ouf7NrZoUemBU6sk+TIjqHhj//bDfckPgh44gHRURk4Z5t+rpmquHapURgApNXVX9x/89GGOie01fOp2Y1i/QGg6EYNAEWnHgoEgJtnH/YvPvs3HYqz6gAoPvFgMIoKogDT8QcDFJitOiWKqDNeqXNhVZ0ig/XQYbZqLsGzr7vpnoMPTtVe1RBRXDNVUbGqq86ZrdrOrKVESBq6HnEPN8IbO2HLI/tAK1Cdn/T302V33K+t2z7VguNOkLzAVIXGbK1wBqvHTFV4kdmqMVg1SpfGYNUYrCgHg/UeO4u0iYNWtCBppfxP9uT72q9w95b3LWwhLavi7sWDllWhvZpHK/iP6YJn3Qw0FtfWX88NNBXX1uNCaYChlj77RiCLoyIAngrUmRq4wjCIq4EBAZ5+BwVd8IqTDL+TYuTlCFLqvJ/vfym+/tGF30U+/2oTrJ+V8+wc/cFvVlQa0T8CMCQsbDxoqosHuox6VWIq31hTqysyknqv9vQ7SHXB7YS8JU1uZFWIpo7i3xbZcVsx8nImaUhky72QVWFTUjCFJE/jNYHt8B1JGmDOtMhofsMlwtznDOQmJJKc+pan7z6X6oLbC3lLmtzIqrCaOiRpiLLjEmTVWRX2SRoS2fLyZlU4LUcEuctDiCfJK/HC0SotzTS3dMX8DJJkt1ySbyKN5IJvPA6SRBfcXshb0qQ+q0I0tU/SkMiOS5BVZ1XYJ2lIZMsVsyrKI2kSigIAmCf8+egd4WiNTOFiGNFASRocAHCldmUASVX/BuBKb88fyEVdcHshb0lT1VdVA1pN7cW/JbLjEmSo1QWvTgB64QXXAKSy5ZvWHa+M0Ni1Jx/20oqDHgUAUuvqCgGcCjAHpUZ2GQBTwgTF/Axh6jEWwM0DcToADzfxOEijAeBsyiDA+Fs4ANQoTnJoKk9xSNKQHRTlTtJArTlzTcDG7uXOqlA6kwoBfvb5hwUA/m3Ja63B7JrAkv5Brt/Z3AoAdhnjAKCf55MfURfcXsjbRjJcbVaFaKok/q0gRq4+ScMqW66cVVGeIAXrCoAtnfWhhUBKY4tqaA1k1kTersVQkgaXhg7efAOaoAtuL+QtlQw/nDZWt7b9lrpqYmQ2dSX+7USMHE51wbF3zbbWq/WOsuXwCQCw//D4KK+99SW0E8sGlrJBE5r6W1P65mIjOSFdMT9DWjpEmcq5UOJcF9xeyFsqGa46q0JiKpOkIZEdl4iRq8yqcEjSsMiWK2dVlEsLTV+A5QP9oM/Hhq7W200NZOJAeANFJXFJKfjzOS/ujpl1wcPshLylkuHNAaDtwmsqZCqtpsri387EyJ3qggPQNV5W66k9vvay5T4A8E7UuiDvveQqtMDw+xJAf6Vs+UrrwZrINM1epJyfISnmW5KXilkX/D47IW9Jk/qsCqmpovi3MzFyqE7SsMqWd4e7WRWug5SV+JoOCC38+iVxbaQGMpc+H6ycnyG9UXnzVZ8WXXB7IW9Jk/qsChtTBfFvZ2LkqpM0pLLlilkV5Vpg1eeltwegN2yT5O3UwIkdT0NZGlw6b4h6yHtBsuiCOwh5i00t5gcAOBGuYlTRVCL+bYZzIUauqAtu+PNINoBLaAEH2XLsTvnUB1gf6LWd2VDTROFqO05yWwmptGeK3F0mO0QuPTlvfycv3pKuWCSnx6edBbhmvC+u1+5k09TnIQDnkxNUXDpE000bLgFz6w+GFQ7FKLYf1EUZqd8McPPQRpW6/3E/cPF47KNmPP/n5wLA5dzOOPlCzqiRIwYvque1y53+1YYAoH++nfRojX4NXSqJm9fAR+X8FbyvX/D8EC8FyaoLbi/kLTYN/WBf3MkZHw1RgSaaiuLfApwoOy4dVLUu+JIJ+6M5oesiXzjIlg86uxAAmvt6Td4zqUVlANhdv7r06IZuQa6UxCtqV02iC24v5C02uZFVYTVVSNJwJkaumFXhkJQhypb/R8mRXsmx0BisFRskaNvn2vY5NN6dVrQgaUULEv572EJ1tayKuxevrkaO/F+a3XksR62pS3uZdwdZOeorAr3Ct67yUo8TM4/Upd3gKbopEe0VBqQqEEUjbwQpa9SXVbHr8NzbEyOYv7vwkzgFWydm6tWloUyOtKdA2tZVSkRb2YyODEjn4tZq3LOjbkpAlLmW5dk+l5WjNgQ9QZIcFnhEydqJmby6tNvkSAcKpE1dpUS0yGZ0YEA6F7dW4549dVMCosi1LGeQZOSot+NfJMlVeFPJ3ImZvLq02+RIBwqkTV2dRLSEzWjPgFQQt1bjnj11UwKiyLUsZ5Bk5KinI4Uk+U+MUzJ3YiavLq0uSGNJkku+pYyos7SuUiJaohEtRXYhbq3GvX7dSXJ5oLWPCGI/kheClBkjfh64wKy2GiK8GDfO908lcydmR6uXP0hnSDJ9uMlFkIommtQFqU+4UQbZtiUPscKbh3epd6+s8gCS/AM/OYLYj+Q5W0hOjro4pbM/APyYvLSNgrkzM3Xq0nBBjoQjBVKsq5SIlrIZ7RiQSuLWKtxzoG5KQBS5luULkoTuuLhfsPn18y2voujcFxe3xSqZOzPLjIQXyJGOFEixfjBS5fzRhs1ow4CUtLjiTcq550DdtANxzrUs1+VORo56OsYmJCT8eMqkzI50ZiavLu0uOdKRAmmtq5aItmMzShmQrsStXbjnSN20AXHOtXTzTHKgOx66GGM+W/xmWp8ZMr+a4owdKZrx55TbnXtbngqSh8EL5EhHCqS1nviaj+p8BCmbUcqAlLYo8yZl3XOkbtqAOOdauhkkp3LUxSntLTHavenS9ilOzCVmc1rOyPnH9yvcUpeGC3KkAwXSWq+qWiLajs0oZUC6ELd25Z4MdVMK4pxr6e49yZkc9T6D9R4THb1uuzNz0ax0wQqfqI87vi18SZXq0q7IkQ4USGtdvUS0rYS1DQNSWdzapXsy1E0JiALXslwLrBK6Y/qDFkJqrBpz0cwvxgeIxFWhtrwvvEGOlFIgcw3SemxiYmJiYqWmieOhns0oRc412LZIeJNq3ZNSNwU8KYgC17I8QZLSHXMFVhc3+jyiwlxiplvSBvilqlDN+W0kvEKOFHmNApvRlhJpLLwJd9iMEmQBTtIi8ibVuydSN814UhAFrqWfZ09JQBSAm6MunggZ3Po9F+YOZmfmbBLuEAnDouAVcqTIaxTYhzaUyFGHcOSJbpOgns0oIgtwkhaRN6nePZG6acaTgihxLcsxBR+zRfz8+O8yBmvrOjO3KRceM6+xZtfO9iw/qXjxVcvH44vXXpU22ddVzejz1i067IBs13J9WeIuk7vuZX2dmCrtJAGxH8nDZaEcybsTxy13GaQcJ69aPD8wk2kHSZqe/8HTJDJPxHt4V+Mpvy4OqiSlZeWoTSYVCtTnR798ZuvHwZq6tLeEchW2x/qusr+tpS1NOTKg0QRX09JG5wH43/bD1pXuKRdrNGP3l/EuD/LMi5QdE301IkoFc8E9laMu1GtsIY2wf+8HyXca4M3s1TtR4u5quIpwj2QstHLXlliF9ydpl7t7lcHqMVEVGrUVFcxgzXh3FlB8hgDgX88hmd30287i7nGbOzmZraVvSK/etQNXDryQnt25ptI4UgPX1FZDYIXlG3gB2ive+blJVEX22mWnXmiC6xvbfGa7KJo1/Mnxxtm/rjorv77wRv6YAYVLU24fG5iyav1BxSBJDVxSW4tCXqwfFhasA5q0BlD4x/Gc5v0ly54Z68+G1n/mPms9b/pb1V08E1gg7KAd8BZVbV/VJ3dbznDV3iFvfp/GdiaSQw7Ow90FVjNRlewZXkoyv+aTtu95br+BpKlpH9nOm8M+MZLkCCwg51U2Kg8lNXCkttq6e9z6Tb4huaRyx2W/94sT9RcXNP70tw999ass9bKnsEN5rU2EsIV2xIsB/IOAZQp4Uoi8jV8Ouw8rbZZrbQ45OK9ugVWGqMqy8KcFbihstFbnCe8tfSVRrvOvfsIr0rkLJ8m+z7j4f+jbW4naauvuBsuvMNhEJuLJEmYEm/mXJM/7ROwnu8Df/GJwToaLIEkgbKBl8GIAoMYqJfekEEd7jPwQtkGyOeTgvMogyRBVmYrZJMk6+mKpaU+B7jPkmEzn8xGPmb/m1RommqLmKsfIFJWgRG21dTehzQ9btu/YHh1rIM8FYC95XgfrK3X/AD4i+wMHhfrqh1wESQohhZbDi3l66NsrCxTds4M4Zxck6SFH59UFSY6oytlIJck09BN4oWeFdwC3bGYgybUmmc7PC11IXh5AHsOhotO3SbL4vLi3kldMlqYXkuQxHCRvZDmjttq6O3oWSc5+MIfkJPiUkTzzl3gRfiX2AtkMkcI/1NEme10ESQohhZbDi1npcqvCDkIpSI7OqwvSd1PEEynitvnT0+FlJEufaZxDsmTSkG/7fkuSo9D+8wz5zmloL8oBk4mV509b0SKVvPLq5JEkPx5PctP02BNfv7u8bTLJxMpG0/zPWguB/e0BF/ekbJI7wk6TZCtUy5ry0szj9hdtBAvkwNymf55zESQphBRaDi9mxfa5C3YYlNyzg1AKkhPnXQZppKhgPdGSPlAW3iUz89Tq6KFXSebHvkleCzxFMuNhANKkALHzNHxsQ7X2W0O+NJwcWrCkOskHppMcaHrmwaXkmH4k+/Q2zTq5FUnCxKGj6121m7WmkiQjUL/HtiN9fd+TXjvHdfNtdcI8aVhCV0Gyh7BAy+HF1J64am6lv6W4cE8CoRQkOefVBEmGqMpU9EpImLvxMkny1WZGkpXnkGTJ7+PqopNJpnMXSPfYjZEvk4x9lZcncNBT5CVsJy+8x0Y9SfYeSBojE744zLKzzqitju5OhaAq6gf8QmYB6yWNZ1Nn+z/wI0lOfp0ug2QPYYGWw5t7hOSHiMxWdk8CoRQkWeeVglQ6Oj4+Pv7l8Pj4+Pj4TSQP+ORbb0l/WM0O6RJI5uF9S7c+yJbp3BiW66AxhzyKPaQx7CvezDGELyKXBRnIm7kZ2ECyVgJ5FF1PiK60nOcySLnBNYX/jSj4lpGshGdtDd4GfiRXx5a4DpIdhBVaBs/8cAFMV3RPCqEUJCfOO6cZOyWqAkkBHa1mW9gPwCG0wV5TNAC/Qat9ZTqHBVkeTb/W90dSRDvgwK0YhGFD4bNAUnQgEIZfgroBpy/HAEmVe73e+V3zg7oaauuaojaCddS1Gr4AAnHY0nRlTce2QEfg38+dGvVlKpAJHPdrUskplh2EFdoRDzs+69cX0ANH1Hnnojg672rFwRlRFcbt7cVNvgORNQGsConG6mcAADkNI2Q6N029JfwqxpXrgW2P+gA/tGycU9knuVUEkDQY1yOBbTGBwA8N2uaFbes8ul/Ntk/lROigjtq6xkJY7ZJmEsjWVQHkX2yqw2N/hVy+D4FADi41/hhANpAY/n+tnGLZQEigBTgpHqYm7+oLlAGR6ryzKQKcwshQSY6UIaricH5n0UD/AIDC76dHIklgHW4YJde5F1KFD5+8qYcpOQ7Auucx2YRjzYGMM7FnPweQFAdgfV/dVENyHKL8czGZgDpq62mYuZ+DkXkLMNzEI8C1B5uPBgpRpgMuA9F4YufOnTt3vgfM3+k8RjYQUmgBToqH2j6DAZwBuqjzTlrMcAojq1wWyooWP7fYYP4wA7+KR1fXMdLUr1cp830GmUiualsq19n0VOtikqb/+xfJIzhIFmDfxbfJvmPJqcibdZC8hB2kMXDL9TFHcJClfnuyppDk9Yhs15ypYJinnqbn8DG5DHXyyK1AfXImZpHFHRBxyWz7rYt7khRCCi3A2eAdrJNK3mqBviZF96wQ0huQGU56yG5klbO77ydLgiS8M+arntX0MRPFbzRmzKYh04zkT89+NGDT1nF9b8p25q3hTZfu+LrPDyS5uGoZydgvR9wi97ZcN+37ZgtHmcgfww0ke382LGdx9TLyxQ9eu0SSU95SQWxrg0XmT4VjQ2IeCRpwkWRR10oLSdO8Bq16VgsadNG83hQaFqYPDUlWgJNASKEFOFu8pGZtu4XXSixRds8KURoUGh4aViksONkKJz1kN7LKILkkqpLMTr1JkscLeGH1V6dNTmmr51Z//pvwZW5fIcnbBw0kmX+whIY0E0nDBZI0HLzN29kkjUeKnVJbHdy9dVIcuPDgcZvVKhoz918wukVmlEJIoWXwslJzXOI5QjgtDs6b4RR2ZqW6nOPbDsSdF/XkgD59tJ1ZxYmDS6IqVLJcy100aqtbqS9/P112533buu1TLUBuBMnn9VV33LWU1IW+WoDc4oJ7SlSF16itGoNVo3RpDFZoDNZyMljf1676d2953/ycpBVNclorWpD+B8r/A5odgrHUh1OyAAAAAElFTkSuQmCC',
    '1606.09370v1.5.png': 'iVBORw0KGgoAAAANSUhEUgAAAd4AAACMCAAAAAD3xJdXAAAb8UlEQVR42u2dd2AU1RbGv01I3xAIJIAQQmhCgGAIERCQEFExAXwivShFaSpSFBBUmk9En1IeRSlPEAQLKlXpEDoSugkYhBRKIJH0kLp73h8zu1N3d7bELGHuP9kzM/nm7Jzdmbt3vt9cDUFtVbe5qIdALa/aHtZGRN3Uo1AVWzci0hCgeVSuvw5+o84tBw2pJ+cq3qpVmHLBBQBe4S4AkPDAHWWu4c5+MAousC/cgutoHKamqdvAwyHplWbwgpo+lVveu3NwSI8hGzUA1h1NKA9ruc7Zy3t3Dg7pPbpAl5biM3paPQepFV3PCJ3wqo/96f3VB9eBJqCMAmDRJKVdK1AFNR9/YA7zctYIqvym4I36IJiIKDUMQen2y7FqZ4LQIdcR2ZEPQET6O5MwUZFexV571/tjzmbmLOH6UF2zGk7DzekOU2s/GafnOLLTVG/R0BQn+N3b9Bc3jDzxMPZJGgJHHKfWCDjk2PzGJFdy1woA8PSaV0v+dTrEGJf8ft2lYZg/AN2FgoJGrfLji1s2ApITfMJrAEDJ1QTf0CZOUN5yoJ7j1LKB5g5MLmLtE+HlTjFq9cosZPbKMUR/tuxfVDgnaBYBD4ZE9/pxY7+TS0MmFo2anzgo5BBA3zcclXXxyeG5lV/eE8BQx1X3SzT4xIEfliuAbyIqv2t1hUjXH3i2lGaPJiL6DCik0hB8RUQUi+g3dBQPPHeM6A10INqANoVEu/BqZXat6icn/3VorpvrTL0julZ+8+e/N6Bu3Rl/k6O6VlNHB+O80ndbraK/Bi7r007vm7iCCfoeCfeG21PJO8YAqIGTu1wQCgR3BlrhKorfxjBv4IX63y6tXnnf29uvAZr6s4cFO0TNswuKs+/51fd33HU8KyDVCYY1DM1ra4e0Lx9nXjfeXn7lUvqfyGbCNp6AGxAJwA3FSMxCWTyAOrevV+YIyH5HinlGAT29Fr1VY5ijFN/Eu3Wd6Y5R3V2+mLIXAJA53v/l5CeMPS1fwR+kA4n79+/f339ho6o1NNgHcOSQjle0h/N8e4HWP8TqT4YCKOySNHqFO74zsV0bIHp0VRz59QOSHKn3i3Pd7+25lPl7LAnvugO3TWwW1BFxAEDjU6tWeesAt3LxYB+q0u18yi7NYO9xvTERANAAOAlcSMW9v2UHYzbW2rgNKJuX2KCKlfcxUBx+XI5KuZ1fMT+MLnh4+nq5/48JyvpPJSLaE1nt+d4j7r6gxRfZ3t6+Pt5XF/l4a32epggfrbd2Ff39RsPw6HbTCirph9EFr+revt7VvY85RO6CV3VvXx9fr5tEF7u7Bo4OvWDnDyNG0M/nKeXp/dO38x9k1zbfLygs90MVvJ2fm6IJrfbP385X3RpQ3Rpqg+qUVJtaXrWp5VWbWl61WW6ucwBEPSrvNupRkkMUM6wxW/2UV8U2mx3WUJt67VWbWl61QSUE1QaVEIQ65vzwjDlTnHXbb5s0KcHi2mNqvxAON+MU7o7PbjSgMf0wMC8NqFcLAK54VCujQGHYlF/diQNRdJ+L67nm3XFzo/KyeiZu/x2tExnIvNLLfAJb+qxJbgVUn7DCHpCv2ENTUYfVAdIOyc7q8tKaD1+Y0DRpaQvt4oF3t17ZpL3QBMDX+89HxbwoDN/l/dfiBl1wY8WJCzWH+qAwNS7v6uN3N8Qd1/YNGWTq7m54D+bv4RU/8Jbqf0osc3kroHnz/QAQ1mHBTNvfepH3KyG+vl4aILQdgMKDCffDBrlx629tu6EN+VcNY5w9b7oFi6JRQiQtllsT+GSgS9ah+2OUZ4fslf3EoANvkSR5G90a+vF+p4mI6OtqEUR0uT+6lhMRlXTXS0NjS43QERGdxotERHQnbDcRncFLpvcz9Tfmb2GTWN7S/Gc/KKUNfYho+g4iIn3nP213a3An/2+IaF3NjhsPDIgqN67+svmSfQtcfb43xOUv4Kh5ewUnIZSWyHUG3DyBjUqzy96x+vUa2MxfLVwkSd6gZ215FxkS1veJIKLLK9/Gf4iI6A2Shsb2ypdERHQZLzPx1q/4kbnyfsQvr77fUCJqGmAsL/3Q1/bybjccv+F6omXoWUq3vHDCsDbFxf8MUTTcDOTme7BQXp6EQFoq1xkA6n2vOLvLseMWQFhewSJJ8rZSCpkfhLDojWYiA0h+/NusF0K5DUQhexbZuUK4oPtChfs7FVKbFx3cchbA5jJuyUtj79a19eR8PWKGn6eGpruv1iBlCua6obwYgYa1N/RZB9ojEGU3ngAA/LTVghxfgi8tJ9e7Tu22vX0UZ9d6J1LeE67mL5Imb+u1d1PBCENPp4sWAOC9vvOrJ7iTvihk2vZW/Leyo2XT6h0Vdi9++nQpL/zS7wkA7fnpR/4y3ubyDuwH4IusEx7AylKXCCD4mt5IJ3YddX0wkIDazIf1jw/XdTAvx5fgS8vJDRlkVXZQvmf7ynsALQ0vPVYzfztOX7DgQ24LUQgAOCI4MvEhQG9lu1s6kd97pEMhN1dpc0fxuxidj9hc3vEBAI7NOecPYB8C7i+91ewl7sRTbS2AbZe9vnEHgOzB39SyIMeX4EvLydHRc56tIj2UZmfFnu0r7y0EGF83Y//O3jm/VztuE1EIAH8NN7xKXlN2fWV/ieziP3hB/bmGV+drB/G3yrvf4sf5Lsld1z/DLQvebnPPORRA/qBJTQEgFd6j3qk1f/asubxP09SEA+GbWgCAbug7ESkW5PgSfGkZOUwfGJke+9j/OinMzoo929dzjhRe4OnySiI6X61VsaFrJQgNrcWP7NZ4+tDeFYGXSdjRIrp+kteuGLpWpZN1RB24rlUaXFOJaETjImPXirYH2eVz/gDXiZguyG9E94BtvJU3zn7u1vQnIqL33iJKttC1EksYpGXkFl8iogWonaEwOyJKFnWt+Itkk7ep59wfS4yvswz1pPmYzisvFxqa/x5+QReIy/tHiXzP+Yu/SFDe+whhOu8HuPIe87CnvFlejzFd2wC4lhNRdfFvtRnAT0RbupVaLq9IwigtlWPaHmCewuwslNdE8jY8OqUnzhtfc7z5jPaf8R+gIQoBT8GTAjr4ikS/lr9CJJbUysnJKS/LyWcXVHf3BwAv8IYryz3tGdT5uagxc3EPQD1XAB64aFh1Z1k8gI7AQvw5fvLZU6fOAwmn8kxriSSM0hI5HO3/IwAf4JLC7Cw0afK2XnsHztxVwvYIcrh/rfZN+KtPw1QIBAqOSnex6B2XZYJr7wfsj7Bb7wNIDHy/8RRWt00hAOjA+7GUE2hXecF2mKIT9cwTNQIB5Ka11qB7kvftGvAA7uNm888AZADL/P5rGjwWSPCkJXL4IO54f6Ac/PdhNjtBy01rrTG7ZzvK67Os/2L2gUBrXwFAzKB+y4+nNoE0NLQ6d81pJqXh+Sd4sWEYsFs3ANjSehkAZLn5An0+KnUH7vF/GmXaVd5rYDumw5el5/uiOA9PAZlt7k1YjkKUa4DbQCf06AEAG4djZRczWnwJvrREDkEuwwH8BUQrzE4w8tDm3oTlZvdsF0K2xP0bPRHRrs+JiDa/ySzVdX1eJjS094azF0r05i1lo5TwAWZGrcq1XYmI7nmFEtG9GtuJ9B1e40ataPxEe669XhjHjof1xWdEG9Ewm2g/EEL0ET4lKukA/5vsthssXHv5Enxpqdz5hmeJ8tuiv15hdvwLLSMnWCTasz2DkkSHIzqsP7ZpzCYi+r2Hn7bTeqbvO0IScm1fSyKiM32ba2t0HcgMWdOZPg3h/a+hAzq7YrLp8o7rqPV7ZiFRXmjITSL6vf2W+HFD8njlbb/NnvJGYA37qnCSd+enPAenEVHRs9VXEemXNw7vVcdzaBrbp9T6+vpovePMyPEk+NIycofbtH/Or/6yUqXZlXlq/bS+1X294gxygkWiPdtLCF6Jz27Qw5qHm+ga725p5WnznR49hQs2PtUYQE7c35FhADCjSy8AuNUhxc2OG+YFt5sbr2MPktybuvNX6jNuBTZwsUaOL8GXlspl3Grkb1V2Fpok+X+UEPwibbG95X1/ruCxhWx5Z3nPUt0aZvT+iWdrAG92u/OYfQq3POUeSpm1Lw5qQ2U7Jd1XjNXZp7D+bZmF+rFLvdQSOoERNnyStQ+W+HTYeX44SzAWsmnYDgBYO7KjWkFFhH6Ft0Lrnld9OxMhfpbWPvBWnZLqwxce8a5VN80jc656lOS6sdfeKHpE2uxHSY6ioJ6cVUoBFQQiVHBTEQbID2uc2Xuv2ZBaP/e9U+iOsrKWAApveWhKgsuTAV/jY45TCoAQHyGIYAE9sBGGsO3/lSAMeqqoGTzEyhW3J2t/GM28N69uwqqgH+N3HFhb0P6FeQBS39oROPjDwlUnj1IKO/yU0Toz+qkxPC/UotLpSFrPoAeMo+jGiPGDbT4VMDDEphbaxfEyB2/1Lc8HkwJk44yJXzGfrnV3Zlo6/cW9FxPgAwBDXJC/JC29zdt1jOu4OHtVXlHZm48rOJsOj4j0Tjv5XLRQWbQnbiOLJ2duU/36m0X61znnFa39Q+s6zYePbsjqSW5VbIvQEZEuJoKI5rCmdKLIC0REyQPxMbtgxXBckwER+OjBVoyytVsggiHEawfPIzrb/I40zpw6JhLsBERShEFyT2YleySiiDIHXafcfn4nDau4OPPNTKJP3fcpsG7VAeD2gV6oLN4Tt5HFZ0oaN9W/dYnoTndjdqUxQ3X0fVimEN0gBTcE+71LRESnI4goBW2YNLJGEhFR+vwnmzIL9J++gZvmQAQi0sfb/OxPEQwhalv9SohoTD9pXJxSvshQXinCIDmAE7/67cjRo0eikonGJxNRvn+IAeTg4kWePxBl4ikF9Xh2+Xurk8XK4pjbyGJ5jZvu+IKI6GIPw4p5bnlE1HmcEN1QUt52zN11/ctERNGIJyKi5XvZ8q7CYeZe/HFhebP8CyTltb1laEN07Mv9MuUdEENE9K1HoVzMlbesZrqFAziJIXQ2EFFQvSwiegl/sKu4eLPHD0QP0EJBPYbKKYvjocqfCGvc9P0pREQFjQ1xgwgioqnaQqL9OEtEZ04onIWs+ZbfAEDzFgCMxNcAgAOGK8VA77UAgBOdzIEITNP9nQ6gPPMeypOLmEWlZxKYewtpB6/rAQBld+8jpwBiGCJWBEMIdPf5AUC9ksPyMQ9hgKV5CQDcODEUQFBJKQAtDNMwcvGg/P7AcfSCdTMeGJVlYytb/S8W64EdMWyYc0sLAAEFF4zoRieFtxQmI+aZhSdL0Q0A+vpuKgZwOdTQ76s+YEsugDxfjTkQAQBwJTLgLeBYeODs3z79feBIAmjVSw9udDkOlE7d5/vLM9cA7Aqrt3r5fxvGWYQheC0zWwsAfoYn2ItjcAiDhaPWBIB+4icaAEfT6wC4VK01u4oXuwGln7d/X0kZLi76bFGuSFm8J24j5XojQiZH/bl947/ZxV6uxBTvKuhQyM1ZC2YkKf1h9OSmcQcPos77bwKA96DV2wcA6yYYV49et3kc8HNfmAIRjK3luSgAXS53vpYxExHNhkdj5ceJ1eHRK7nGhi17WkSmDzjrgtiYFmemNFlx1ywMIYIY8uEDAK5gj5E4hlUIw+Y2NQHAxR3AmYtTDDvmx6d/PtRui5JbIhcTJ2l+eXJvsEBZvCfhRsr0PA8PPhrW4YDBmeLR+j4ApCBHFt0wb6XL3jy2Gdg+83H0JCrtz65Jn0/65u2J6FMi4bWXAxF4197Yl4mIng/QEZW7LaEcv4lEpPNfTltqxhPtRBIRUURbizCECGK4hGlERBcxnVktjLlrrwRhkDOVlQSlGl8Xt+lRxF9njPV/9uydqeBieZGIqF1fqTI/FmxkPjtu02vD3/HEc8Z3tk2TTlTSCQsF6IayaapqDBpEcQPmjq4BoFPzvbfr74rh/Zp6bdqlsEttxP+TIfZeJTQzen9CXQBX90Kcy3U/BqB2El7uq0m+dJL9trWRZtDoDDcTcXZNNG4sWOsLHQCUcxPlCGNj88+w/AXZoWnI/eIP2CpwxRtjTfON9V84ZXlgIgwA2q/KDBAr82PBRgr1Lo/dXnvc2L0vnmQvpn3+8/qKsg2xJ2vDBw0bAmi77kS0omvv9wCgifoq/zwAaEbqv8GP/XjrX6m2BnufFf+XZ7lp9IA5RDqkItDT09Pz2+nQr+q0sVZnQxGgEIbgjNBFAFAMP/kY1iAM67j5K9fe+VVwBubHtSLj91jUOnWE+bQliJX5sXAjhXqvf1IbTfbN+X23YdWUZXGHJt9DW1l0w9y1dwszCPg8c8wwfNa6kVp+77VOr43z3SQf5MA8CXogvRijFmtAn7bqfBMcA/Qajex9MBEMIYIY/OreA4C7eJJZIo5hBcJQfriH4eXOi9+64A+XUFHctEv5KXfAH9csivXJzXcHShm4gKcs2JNgI4V6eQldAWhmH0gynkeDg4HURm1l0Q1z5U283oTpr7RljuZzuydM5i7UAEZvHfW55L/EIEJSmnRfbR/bNwpAyZ7ui8c3ATKAvVrO95/l5msKhhBBDJrY8wBwwz8CyPL2FMTCLrbl8iYWGE4eJ08s0QDbBjGaXFwc75LRALjJHg9zre1gdwBX/FoKlBk5YyzYSKEeafKrA0DD1qze0S/W+SH3yOfVZNENc+XVDdlTA8DKkfWZeOTuy8YaJFwhDXrWy28EIBf8L2y7c8zfXJQCQOqgZgDKdABQqgFAZTp4rh54KQxY8qybVzmAy25F6R2AUkYmo1EI7/TSb8m7jw3TAPhV8zjQrJkwxSkRNxqDfp7iir+DW8bzYwAlKDFsdy7C4hG8A/bMdHVYj/GgB8dmsJrG2DVmWgMgLaFbV4ti/VoCSIn7qhpfmZHjYsFGSvUGLv4QwO2sp1m9ndtv+mFxyHAA4xbt6Q3a81ozZT3nNvte+2j3wUl9c9i4qOZH7Ksrz/tr2/+X6JMtRAufq+Xbqt9VMYjARw9+7+PvH5txvHeNmn1OreuubdQ/mw52mrn+3V1EW4I++XX+uQmdZpTvjvEN6LmOjByCHAwhbd91S8iaOqyUqLD1MEFcMuTZur6hLy6SRxhk+qbbMZMdrGM7MwZNLs585T/Hj3WITbfc1S3/YPGF71p9rhcosykaY8FG5rPjNi1+dczhi6tGpBr0/og5cu6d6HtEAnRDCaVwNoIux+siwo2XxL3tasNhIMLtguYaALqrZaHulOcn5RCUwRAZvxZ2CteYjiGLMMjckyndGFvHYtIXLlJ4mEbJ/ffEU/6d6sgrczFvI0u387lNr515EBbJ5XB/d054JzY0ohsVaKWzHkQQNxGHYH+TIAwqpYBKAxHkOQQ7moowONCMYzeIIMsh2OOXUBEGh1rpDiRMdKZ3trp+jGqlc6RT0koQoYKbDMKgllc1wqqUgkopQKUUVEpBpRTUk/NDNZeCU6AKKqAAS8+1MqAKELEKte84P6qgeI4FM+yAQ7ACvUsFQQ/2/u41ogoQsQojtzgrqmAWULBMKYhYBNuwAiEwIJwHwpCdGDgwd3I26ImhB362ZtOTv5HCRxXErIKTogrmAQXLlIKIRbAJKxACA/x5ILjsxMCBuftZRj0x9MDP1lx6JsrLRxXErIKTogrmAQWLlIKYRbAFKxABA/x5ILjsxMCBmfJyemLogZ+tufRMXHtvMOeAyEYAEBx9kLkxvpk9zb425kg3ADjRORVm50wAoImw9dQsnbdB0DZ1dgfQbZRhdMqD7yxVOMeCyxgAWD+6EVCXAPggS2ZV4ASFGQvnehDMA8Fld/oBADS5YZUeLxsAwmzNpWfi2i9AFSSsgqNQBSOr4ABUwVpAAYCAHRCzCLZgBQJgoPinwUqAA2V64myUkhMmyitAFSSsgmNQBY5VcASqACsBBYjZARGLYANWIAQGhPNAcE0EHCjTE0MPgmzNpGfi5CxAFaSsgkNQBRhZBUegCrABUICAJZCyCFZiBQJgQDQPBNdEwIFSPTH0wGVrNj1TT4TlowpCVsFRqAKfVbAfVbAAKCihFMQsgrVYAR8YEM8DwctOBByYzk4IIIihB2O25tIzOazBRxUkrIJDUAUBq2A3qgDrAQUAApZAzCJYixXwgYFlb5ga0RADB8r0pNCDMVtz6ZnYhxBVkLIKjkAVBKyC3agCrAcUpCyBkEWwFivgAQOSeSC4JgEOlOjJQA+GbM2mZ+LbK0IVJKyCI1AFAatgN6oAqwEFATtQKmURrMYKeMCAZB4I7oIqBQ4U6ImgB362ZtMzUV4RqiBgFRyEKvTJl2EVbEcVrAcUBJQCn0WwGSvggAH+PBDC7Lz4wIFiPT5OkeXtyc/WbHomTs66ITkAD1XASBhZhYQrBIWoQn0AZYUAUFpmRBV2XgKwJAgcq+BvRBUa8B/x2m/JhxsIMKAKXXitNTAl8QZYVCGITc06QEFAKVSPOWhgEQxyIqxgkWVT6TifPeABA7rCPECcndvAxQALHFijZ8Qp/g7qws/WQnryHUoxqsCxCo5DFThWYZcjUAXzgIJFSoFjEezACgTAADsPhDg7HnBgMTuenjHRwtbDBOSE2fRM3DGSoAqKWAUrUQU5VsEeVMEcoKCAUhCzCLZgBSaAAWETAQfm7hhxemLogZetmfQc69ZwGlRBfo4FlVJAlUAVVEChQh627ySoggooVNTD9p0CVZABFFQbO6oMqmBqjgW1vKoRtsp1rYJVSqEqygWrNnbVxo6Hxs6uutdh9fTMD5GdXbF73Tp3uB2Gdqe1sePhsbNzBnET7nWbbOycrMRC7mQ2dvEsAQps7A+RnZ1nXzfhXrfJxs7Jii3kTmZjF88SYM3s2w+BnZ1vX5d3r9tkY+dkxRZyJ7Oxi2cJUOa1wsNiZ+fb1xW615XY2DlZsYUczmVj37k1oSa03X652sr6MWfnt7Pb4l6HAhu76efmw7ls7OJZApQZYZl22gXRn5woMVxMfGsWEdGl9w2G2BFeOUSUu1JkiB31jsQQmxiOl4mOtsbYX//9Xe8ReiL9VzGHt3c8RkQlU9b8/llUEtHOFliwbF7Nw8IcemO58XWSibO3wVY6d5BVJ2ciIl1sltH52rN9jqwsfyPTcvpaT6TM/Hg6e/1fmCZvhC0KQder22JzFSBLAj1JDmy2uhIiorbVMmw4OTu/nd1W9zoU2NhlNoLT2djFswRY8+11fju78Gsm61631cbOyYot5E5lY5eZJUDpt9f57eywzb0OBTZ2+Sfyw9ls7DKzBCjtWjm/nR22udehwMZuYiMns7FDOkuA4kFJ57ezwyb3uhIbu+wT+Z3Pxi6dJUB5eZ3fzm6be12JjV3uifxwPhs7b5YAq0/OD4GdXWBfP2ft4IkZGzsna7SQwwlt7FeH3R8/buzwNY1s6Dk/BHZ2gX1d1r1uk42dL2vcyBlt7NysAMpt7HiY7OyW3Os22tgtPpHfWWzs/7jXqrLs7PLuddXG7uBWSXZ21b1esTZ2VKqdXXWvV7iNHZVoZzflXld9zo5vlWBnN+VeV8urGmGrZnld5wCIelQuRVGPkhyimGGN2WoXpCq22eywhtrUH0ZqU8urNidr/wfMzyFnOyIiGgAAAABJRU5ErkJggg==',
    '1606.09371v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAd4AAAC7CAAAAADuu6BCAAAf8UlEQVR42u1dd3gU1fp+N40ACcFACF0Q6REIoYWWSFEkiNJUiohepCmKoDQLYm+I3ItK0wtXxIaCEZQiTXrvNZpIJwQJSUhItr2/P3Z35szuhuwkMyHkN9/zwHMyZ89555uze86Zme99PxNhWOk1P+MSGMNr2O1qJKcZV6E02jSSfs4hLgk2rfQjF6eLAGCi419JsFt3HsWGXJwummisvaXcAnz/qH37vvO2Jv1D05Y8X5I8sK2NjtS807y9dkchpF7o7b618mntNX9cI+CBN2cMqPHf2n30WizcD9hbOr+AgQCCsr23moyn1AKlbbAVgLy/axRQvUvX2HDEbym8RwUD+eSiRzc+Y/k6vKebocspkvwc+HdxDe8BdN15Pd2EaLv12N2V8mk1tdp6lTj2e7CnAGRyLvAjybyP4be8sA75AOSLi57d+Izl4/Cm1EBCLknSWgeHimt4J999nUwEXiQ5tn2+jqnFSYLpaoHDOxC44ixUtxTSIR+AfHHRsxufsXzbWuU9cr7CF2UAAP7NKzctrnXju5fKA5uAeADZjfLdIartdyOi7igQeyOaVQIAxOHC3kI6UDCQTy76cr75fMi3rdWc3Zjq2r8Ex/sBQNq+a3VbBAFAUlLsHek7cztEIOlkbKXs3RntIgHkbqwQawIsZ+4yAUDGoQuNmqkciHNZjwHYBL+OAHI6yzjg4RMR8TDJMEg62T4ch06ExQUr+xBwL+3NiooCDuUkos6OugVsx5IuIl66QmfaCu2dkGdPdTEVHcjNRRlE7tFLNyqwfJmczVVhuigt4ekkzz8W9vyH0fW2kfyyXcdKH9wz7ZVyG75s1zFyasPXhpVdSB5vMhFjST6EzSS5qO7k2dFD1E7OJJnhhxhSgcODMXWm92lT/U3yeJNJGOuoDFvfY9DHNWudV7SWca8MKvvsu3e+yNx72wNRHb4sANm59JJ8AVgptHd6llYOP2sCJLoog8g9eulGBZZPw7sR6Kw4kFyz7H7yMmpc540q+0aj4lkOwONV9o1G/dO0VyqTZmn07TrUIK+acIjkyaCtzK2G3EIM70pgAkkB5+kDFSpfpLUhXqKl0bc7Ud1ZGbiKfAIzxcYy7onI8vvJOWVsZBKQWiDyQCDNUboPOCa0d3p2JABvawIkuCi2FXp070YNlk9r7xagteIXP+bcCy2A8KDzB7C2fIs9mFITflUrlW+xB8/Uhj07b++qzH6JaAtsYaWmAH4xn7T4NZlUphDrl3NdEnAeHZP5QlX4XUU8VmX2+wZtnZUv3g+cQKDYWMK1PZU6pQVsfzTyAzaiSRUflt57KgMALDtRtZHQflVmv0TEouki1NYCSHRRbCtcMvduVGH58usdDcx3la9fJ38FDpNMAtYw+3KGHw6StLpKScDv2ZeskfienIC+JLkECOn+GwszObeBXzpJAecX4E/yKPyuMftSXmUkypV5QdgtNpZwlwFzvnuzQfxRkoMwpkDkk8BYaeaaJLbPvmSNxA9kKk5qASS6KLYVLpl7N2qwfBreZ4FNzqKl/v/IcahnJ7nYMYGtRLjNNctUspFLgFRyA0JyyNaOm2Tr1AjA9HshhjfTHy2lOcyB8xzqk/wUrUjyJ0Sa5crVCMsTW0u4UxD57oyvj9hJ2qvj+wKR5wJLHaUhCLmgaM91CMkhv2ukCZDooggi9+jRjRosn4Z3NrDOWfxv5QyyGwaQZF/cS5IT4XqM9RL6khyMTiTfR1cy0x+Hjhzk/s/zLFtHYkQhhncVMN5ZdOF0xSMkB2DC9UTyQbxo/cHiqhyKp+zL5Ec/Mm5fPOg6eApIZdLOmyMPBC6TJA8FYD4V7fkOupAcNll8yFVoINFFEUTu0b0bVVg+De/Fcpjk7DtyKckhmETyUln/w465ZZbzc63xH/KfUGwn+S9MIH9DJVvnT84FYBtpCfp3IYZ3MpAozWGznFdhGmmPROKXLXjFH0cTI63OSksoVu0um+lqK+COc8y11g3kF2hMW4uPb4psr4YokuSfdTHVTkV7jsI48swd5+SPFx5I4aLQVujRrRt1WL49tVpoClhsJ82Lqi0iyd9xv51Z3UzznXPLQWmDP4324XiVJD/ABObEo/P5Cv8ko3seOaNGuurhtee0hemKNIc5cOZjIq1PA8kPzOFuNLR2/MxVeRHlbvR/Td7fy7j7AzrayTM9Gps5B4/wgzbmmyIfBJ4h846/Hlphnp1UtOdcDKW13wzxRqLQQAoXhbZCj27dqMPy8aHklvtR96GEWv2OO/78pnaLx2v02EWSvODfRlp6K7bq1bThNyTJnD4hg9v+b1B4ywXkrNoDBrcemqx2a3U6NLhM+dByQWWbK3DsEyv0bzdmYfn4+/Jo6VM5dordVWkfWb7LCKvcgYC7on7ssI53T80kMzpX6/Bwxs2QXwsEYCpTxi+y/YeXnMfk9jQPD36k47uKR6GFBHJ3UQCRe3TvRhWWz6/zLySn3tWorLTf/iejuuuPyxWDHIWJH/b7/nTZSNfDnPSrtQNx1RYBwHoxr3q5or7rlnCQk1rbH5bztf0AnC9TWahMZVWxiYibfjk8wnHu5wMjC/GWXWoPZF26M0hZqRWQDCL36N6NGiwtozXa7P73WCNao7RGa6TvdT2mNQylLhD220g/U5tvjCtaoky7ydluD6Al0GRMziVpcjYiJY211zDcxpGSd5pKyMmYSj9yMbp4pxHGbkzOhhk3Robd9iyFkmSPx7Qud2b7fV0A2H88ZvEbGwFl8fJzc8M0R+UXR0L8J5YHgPR5mTcszzZUHtQHSvBKvVu+shS8RhjP0jfOOX+LBBD4qp1kVvdXzfyqN8Vi2oQRrXFRc2Rzz8E2ftcsjWTas2nkB0FrFQc1dFHoVXZQpVtqWApeLXXkrRre7p9OmZ9Ckvb+g0neHaEo5v5tnanD8L4RmEmywyiSM4O/J9PQXnFQQxflXgUHVboFkkWanJfdssm5yhhnYf3SvQC+sSiKZe7UBXRes1AA7ebOKIeqBFAeVxUHdYESHCyEWwVura5t32Q+DQDm3UdtAKxpqbCm3ACAw1NdH/KsA8CkDalirdyVhjYnrAWAVrHKoi527VwIAERcPwA8ljUA2IpeioO6QBXNq4KGd8aisLJvxwCc1ycnueNWbImuMu23D3Y9+iTx3ZTcdUOGfOW1DsDG1uvtS6bbpFqpK03s4MwPZ2YA3FD37MvvTj4FRVEfK+tPxxU74SD0mWe0ekV5UAeoonp18wXhRBxJewfy0xoZ5OqK6WT7LgvJJKwj2dCx9nqt2+i/l1xWbq1UK3WlxdobtcTOnxr8zWvo8KGNyTV+p1gkqcfa27wpSY7BDJLcMbH1yOvuBzVbe6VelV5xprZbq01h+0nO57Ww50jawj8l74+wkdbAWdLweq2zN7mf5Nrmx6VaV1eaDO9BkmzZl2fgf5rksLtuiEWdhvdn00UyLxbvO24bTvZ4MM39oFbDK/Wq9Er18BYwObepEx01bt1w7MsI2rJly7bKpwA08QP8g7Klz3itO3esKYBuBxpJta6uNLFmANDqp7TyqF0bQPPkbRCKOlnvj54+m/xuAhz0BVODxWsesLkd1ByqiF4VMLzBm14r92m3ofbTqBIcHBz89SQADg6eTfqM17o/UdWt1tWVFt7v+AMAQnG0QlA4AJTFUQhFvWz87E0bXkhFc+eflVrvWe1xUGOoInpVwI1RUtnp09MXvfSvxqjUyvsnvvBeVx9nHQWp1tVVnBZf7oysIMCMSgH3ZDu+T5UhFPV7CXMncLpOc5g7WncEAeFIEg7qAlVErwr49R79FrhjXL8jzauvBYC8ROVv2wIkw2tdjaj9AJC3TKp1daWF780/DwJwPKwxev9lBpCKVhCL+tjmPhlAxh+vBiB3z6HLAM6iuXBQH6iieVXQjdFnOQDM7YLnrzgEYFYtwGwBQIsNQIs/AcJrnWn+9l8BzKkp1zq70sL7/o0B/L1pZgBGlV8NcPXw+hCLQB7yNB/eFYlngU/qPg5U6Lm+JnDmaFwn4aA+UAqvVLtV0Jeu4TtNwzbHx6Bn4qh7Gx6Jj9723n7TQ1NPLApaeHRexde6TDfdD+917bY8s635iSatpdrTrq40sOHTd8WfePO9YUCVNWPMdRbU+xhi0fxk2uHQnvXjx2l7zYceSd+/ZN+qQAALJ+yO5YTuC/zFg7pACQ4Wwq0CXudnlg04Za7v2DCdv97APdbAcrhOeH51wKVrDfzkWrGror/rPrYjPNYRt31t05XWzeBehB5v2f9ZdS061unngYOMbmZyO6jd63yh18J6ZYTSGdEahsGI1jDMGF7DjOE1zBhew3w0/9eBEkPsiy/9yPHF7BTJOONbXhotTg0737jvNe57DcNtHcaenHKlbZ0S7ZBdvy+snf6lfHh3fP3rsjpA8rDRA1FCWArKyP2Nn30PwL7o7A3703drDLp5Ss+I8gAwyE/BTdCDEXGLWApXsYzkcvX5C3RiKbhF7mfXSyBpH3uIvHDvdo2RP3ftSEUWgTrqQElnKWRhGUn7nuu3dnglloJb5P5b9RJI/vIxSR7spjHyc3N/+2Pz5j/iU0QWgTrqwO3BUjDF3OI1RWIpKCP3d9StDAA7cwCgXrLWj4BGAMCif9URWQT6MCKKjaUgWt55577eduUi4MlgAM6s/8vund1QHJb7o2NHUOPjT+zALz017v5ZAEjeNhg6cRNQ/CwFwf4aMG3TO2sB4HjriLHwZDDAPGFt6LKuSfBWp7E5WQpK+/dzjhfgw+q+EH8ycfHbGmPWA2B/7j0TdOImQHuWgu+T84k2KzoD7wNA433xAE7+shFovQ74/J1jFVCmV0rFr5aubtT64iN7/bzUaTy6x8aZlrVZo5yt9leu5SgEbxy4uVnbdYE6XPhv7rkDQJmofwDgb1zTZ3jl/jP/afTDm34pnRZ11fnX+2R0ZwCPOv4IAZB64ABgGoaMqf0qAN38lqBiVjbQ5cBf3uq0ta8HmtAnZLzimOWrYa6iud6Lfpt7XdL+upunjAYAvHHsEmDeD6tOy4zUfyZ2POKHut1H5Or76720Y5rb59vUiW7a7UEHgwFA5VPo19eUcmg7MrzVaWsOlsK8tAjh2OxnXN/VwyMTK48aueah7Zo/4vjFVNvFIvjM8lXCdr1iqqX+XSyFhdu66PrrPYGwAhkM9nmxiyt1yI/doKFJLAXh2LG8SteuXbNarmXh6fcqo97a13et0vyyL6wLr4QF6EWI0JelINtdyEZBDIaJ8/bXwxbAbvqzQHaDNiwF4VjauVcAHKvyyl3Dj3YCYJq27pTWe2frxm5wIyxAX0KEviwF2Wo12gkAN5A/gyHrkyfqAZeBNVtvzm6AdiwF2eJmz549e3aFqNnjy5qyAAC1o7S+5seuh7uzCKAzIUJfloL8JGPe2iMAP3MMsCUbngyGwLJWAIcDb1wM98Ju0NJkloJb5L4tOxMIfPQTADh/tbPWV/0CQuDGItCHEVFcLAXZOv3+TJ+aO2I/ef6nH3a/tQe9/uvBYMDCF95vtrf35UlxQ0951GlqMktBGbk/+gAOdbtv4uyRIwfdsXPbnCCtr7rFNbwyi0AfRkQxsRSUdv5KE/uxypXL5stgsJ2wNAliZlhB7AYtWQpeLWl3TrPWJs3fspsXJ0R6Ehagw+t8g6UAI1rDiNaAEQhrmDG8hhnDa5gxvIahOEPp4gyx/dIoth/n/PXGs2TYtNKPXJwuxhu5FIz7XsMMsX3cZlwGlEgaxG0yvPpzGbJmnbl4z/ORUITzy9QFffIbAHDRIGCffy44Z1yEbrkUZF9EQoZIWIDuuRQ05jL4fh5pj/3FjP5h25WpDCTqgrr8BqqQHTQI2ge+Qe5tcEGvXApCsgi5JOaNKI5cChpzGXw/j9EpJLPC61oVqQwk6oK6/AaqkB00CC4PyyM5or9euRRkX4SSQFgoCcOrHwmlVrWrJPvgCGvGkOSEkGySg13V4kFtkbd/3TaBJB/pSZJfl8lWieUz0GAvpd+xl+TubdrpOUNkHFgvX0ZOil2ZPwG+cxm0s1p5ZgAhuOyVLqAfh8BFg7CtDQOAankb9eYrwHveCC23VjMCul9/+9Mrv04+PKZq+p2JMW8FAZz/88RLwz/q4PHhvybXa5bSBACOD97fb6nUWlNHN1uDABwKiJLD+dsDOLjeGvBUGJQHtTQXDSItPQQAwnCqq15YTl+EEjfUPTsvJOOpBtB0ayUnQWgceZC83rKXXcyfoLTjoZtIvueYnOP6ia21ZAiS5C6MV6YycCVYUJnfwHfkfV+QbRNInsI4kjyA13XKpSD7Ipfc0ipotfbKSRBiniDJ5Vgq5k9QWrvOJJniGN6EfmJrjYc3955uN5SpDFwJFlTmN/A9C9kLNufwHsJEkjyISTrlUpB9kUtuaRW0WnvdkyB0wUpF/gQouAz3enIZtEuhINrUiOXByvwGrgQLOuU3EGgQoY5cA1aE6oUl+SKXCpNWwYfhdU+CEBqahPzYB/lzGbQe3S8u/FoeEOkCAnVBFw6BQIOo6AgHzkWYTliyL3KpMIQFH7ZW7kkQMrMaIj/2Qf5cBo3Fs1Yc/NoPR/yaCHQBkbqgB4dApkGMD6uaCgCX0EYnLNkXuVQYwoIPv14hCYIZADahp5g/4WquT1wGtw8W0bZvm+UH/FxGpAvI1AV9OAQyDQKmhBQASA6P0QlL9kUgZBSGsFDwer+sbjbJPnvImOoXyezovnZyZchBku/vY1q5GPHDf5Q5TNrHYQlJ3tdDaO32wSJtrY7fNWLkyBFD6lg50e8w+Xq0meScbSRTAr+keFDrByrWkE4keTT4L9Ie+5ZaLF+BZF/kElMrJpL2tsO13Tkv6/HykpWTZ5GMGThlwZIur+aR5PrYqYteWklmRw1RfHpz55k/TPgOEf25q3d4eMJlqbX7B4tykVs69x/kkZ5/7HuxSypJWl/95MC3TWfYFQc1Ht5R7ULCur5P8tu4o1cnDDGrxfIVSPZFLpG7Wi3dM2pQpgosH17ny4yDVlELz2Q1dE1D+bEP8ucy6PGuW6QLSNQFtRyCQiBf/jU7Nlq/XAoCDUMgZKgiLKhmKbSKWmhEa5TeaA2LFYahlAbCrkk4seLh48Ylu51MxeRsgz+sfn7G5HwbTc4qbtf89YrdMcxgKRgGg6UAg6VgsBQMloLBUijNWyvNAtEzDl+p0AWlWPu+ZCvwB+gciH7x6/89oNPwCtr3CnF9PbTvXSb1LcXRiwr8ukDJEfsaiu1rJ6p/Xz8d1NiV2veCuL5KQXpVyELfchy9oMCvoYsylIykpdi+dqL6CXoNr6x9L4jrqxSkV4Us9C3H0QsK/Bq6KEPJSNqL7d9yUX3ftO8FcX19tO/hLuy/YvnROxASt+xEU0GBXxcoGUkzsX23QHQ5Dt1TVD9/TX0mbUgV/vRoUeTodln7XidxffgQRy+chd5IGj21chfVd+nmexPVz19Tf2Pr9fYl010j7tlC6hVF177XSVz/Ztu6i5EADgVECWehN1KhzGNBcA9El+PQpdD1BbWPk+Nb2OQ6j6j2jf57yWXl1jrXXo8W3qLbC8F1WjyZJHm6PYI6ueJhdFt73fvehfGKs9B2eyFCSUhq116/AkX1Xbr53kT189XU55huLYGQ+jWdf3u0kHqFFtr3uonr38zy/tXtbcVZ6I9U9PteD1F9l26+N1H9fDX1zx3rAaCbxKvyaCH1Cg207/UT10eBcfSCAr/+SEVfez0C0aU4dC+i+vlq6v+JqspnO+4ttIlud2rf6yeuj4Lj6AUFft2Rir72nsabJHlWYgmdOktenRmwkTvwhfMzE0L/JDdjt+2ks06uctpZjFXc93q0kHotytprCXmYJDNC7CTJTjOLce39ZayNPHxUPgvd1l4ZSYO11yMQXYpD9yKqPy8/Tf0aUfsBIG+Z40/PFkJsPIqqfa+fuD4KjqMXFPh1R9JgcvYU1Xfq5sObqH5+mvqm+dt/BTCnJmCxAJ4tpF5RdO17N3F9HbTv5U2Os+8TQ/4ZPWrk4wvqiAr8ukCJSOp98/JCcMvLfWruaPNoRNwPu9/agtiHfoppGra52nMANrx8b8Mj8T3x4wvPNNubsGB/XKsFrjpXlfxS4pkHmp9o0m/3W1tMHaa19GixfK7Ua+Hflv3Se+rbAJA3ssygO3Zum17bIUifXUut9r1vyELfMfsAAM0OCmehqey7BCUgqfQt3zhnRSC6Ig7dTVT/ppr6l641kOcG9xbeottVD6+sfa9SXF/T17DyWZS8971ehxfG63xDdNAwGJGShhnDa5gxvIYZw2uYwvxfBxBfQk4mvvQjxxezUyTjjG95abQ439j5xn2vcd9rGEqvGrsbqeHIbwwYHGlcXNxysf8ADdX1nZwG6wj7R5WPP7ApVFcvvQTxy7L3ish+6CDxL5yAgiGhB5RSYd8p9o9iFdu/imUSp+Gj5lZyKvbqKNftNYhflr0XDmrKUvBGGBAYElqyFGQopcK+U+y/mNXYs7BM4jR0HUcy9Ue7jsPrNYhflr0XDmqJ7JUwIDAkNHRRhnJT2H9L3fBqubUyxZQHgGthAKr01ZOoXOZOFy1vRft0ICQu5QQwr1kogHaLc8SDWprQrXwCO88BToaELlDrl44H8M3PzlWwbmXtn1pZ01JhTf4HQN7fVhcHYfsNd1KDU1wfAJiZkgFAIi9YLv2Da9ehZxC/IHtftMh+dYQBfRgSMpRCYd8l9q/p8G6JrjLttw92P/mUfcHnW9t+BgCre+/PfellCyCQGhychlVDkr8bMrhP07s2CLyGlc2qzf/0P7U3Qccgflliv6iR/aoIA/owJCQobqh79uV3J59Siv1rvLVq32UhmYTnD5D/LXeV/Ln+VdI+epAbqSGuH0nGPC/psUvkBXuDvlsuVv1WQ4agZxC/m+y9FNmvbQCjB2FAwZDQlAS5C+OVCvuS2L/GW6v7I2ykNfBhktuxlzk1X3Fozf+uVNdPkIc3C8soSvLHNNfYd2F4HbL7brL3zoOab+qkbl0nkPT4i8G476L2w5t7T7cbCoV9Wexf661VEz/AP6gpgGBk48C5KACIMq3woq4vmCjJfw/0DuJXyt4XPrJfHWHg8NCPPzzSdc1Ddn2gRIV9Wexf6xeCwcL/Npx0lPyCjntR1xdMJC+E6x/EL8reFyWyXxVhQDeGhANKUNgXxP71fWpVD9cBwJJ3txd1fcFESX7d7pNk2X1B9l44qBOW0zKPdgJgmrbuVE99oGSFfUHsX9/X+S0jtgPAbvTwoq4vmAd5AXoG8Quy90WK7FdFGNCLISFByQr7gti/xr9eswkALRYAFthQfu7I8XfB9t4jvWCa1/1IlIvUYLEBgNkCwAwLEDz/0UPNgFndAXMmdAri7zYazNkyGViReDYMn9R9XHFQS1N26ziBwEc/eQ0yQ0J7qFEzVz8Irh5e31Fhy87U+sZo64MV7+i9Y+G9IbX7pE+KDm3xLLnm3nfm9p5mVqjrO8T1V/UMrf5wyoz2IY2HypL8q3qGRvRYqN22Mm9Q96qhTR6aKcruC7L3wkEtN7RCt/IJ5D4xYuPBecNOa7tzFqAUCvuS2L92Yvve7cK1BgFe1fXdLD9Jfh3edauV2NcIWQ1DojBAqhT2DZYCjGgNw2AEwhpmDK9hxvAaZgyvYTBYCjBYCgZLAQZLwbjvNe57DTO2VqXT7Db8/yCh3CqTUwvIJWVQvw6WPi/zhuXZhjqmUYAXqsL8c8E54yKKLvh760zleQhUBSHJgDKoXwfktGfTyA+C1qpNo6DeRYGqYB/4Brm3wQW1Tt3GwyszBYQkA25B/Togzwz+nkxDe6pMo6DeRYEVsTwsj+SI/mqduo0nZzm1gJBkYP3SvQC+segHW5UAyuOqmMxBHxPSKCzpEAQg7qmcclpsraQ8CBIZwYOoYL18GTkpdh9JDU6ygjUtFdaUGzqOuSKoXxd7LGsAsBW9UIxpFGxrwwCgWt5GLXbOUh4EiYzgQVT4tWXk9DdfSew+0YyCSQ0usoKjZtejT+p296cM6tfHAgHzjFavoBjTKKSlhwBAGE5psLWS8iAIZAQPogIbRx4kr7fsZS+Y1CBnWnDWrNNi7VXEsjtLiqB+vVb9HRNbj7xOtWkUCuniLownT2EcSR7A60XfWtmb3E9ybfPjIhnBnahAxjxBksuxtEBSg0BWcNbM0mt4xaB+/a66/WSPB9McQV+1Tus7vA5WxCFMdIzEJJVYXibnc8eaAuh2oJFARoA7UcFlXbASBZEaRLKCoyZbr/lMDOrX8XFfg8VrHrABxZZGIRQ2ALAitOhrr5QHQSAjeBAVXBYamoSCSA0iWcG9vcYmBPXrapVa71kNFFsahYqOQPJchBX9qVV9nIU7GSFfy8xqWCCGSFbQ+yGcHNSvk5k7WncEAeFIAmDd2E1Xd1xUhbCqqQBwCW2K/uuV8iAIZASvjgLAJvQEcDX3Zhj5kBVu3qiQJgf162S5ew5dBnAWzVF8aRRMCSkAkBweU/ThlfIglJ/7fTIcZAQn90AmKgDYdAnImda3D3ClVkflwCs/K2RacNbY4NkIRUhtIJdGlV8NIahfh+m/5/qawJmjcZ2gWxoFwD1hw/hjyQB/Gu+vxTPn7S1f/v6NpRTICF6ICjEDpyxY0uXVPJLZUUNuTmpwkhWkmjoD0t0aFWJbKTMF5JJbUL8uz5yHfrR1S9uEiySZiKk63oGJZItv445enTDErBYrn9f5Uh4EgYzgZq2iFp7JaujrQ01fyAomTZ52FCaoXx3ygYOMbmZSm0ahiC5e/jU7NtpUfCyFVlELjWiN0hutYbHCMJTSaI01CSdWPHzcuH4l3Ao7OdvgD6ufnzE5l+zJubDve/1v+0AeY3I2DKVAzznOVEJOxlT6kYvRxTiUnBB2w4zJ2TBjeA0T7P8AlMxfKGnX9YgAAAAASUVORK5CYII=',
    '1606.09371v1.5.png': 'iVBORw0KGgoAAAANSUhEUgAAAYIAAACICAAAAADPfbOHAAAZE0lEQVR42u2daXwUVdaHn85GyAqEGBAIYdUABpBNHCCRTSAMKDsGNaMBBRVcGWRwUHHcQEFFEZdRB18XCAiCgoAaQBbZt4SwSGQXwhISlqx93g/d6a6u7q5UdzrIzK/Ol9xb99x/neRUV1XXfXLKJBj255qf8ScwUmCYiCQaf4U/yxJFxCRguu6vB76K0Ge/qS8DMk5Ef7oFeDuxaJsZqrcr7+acAFrWVHud3HPpyrBqyi05+y8Vj/B14ou2mS2NyLhwX+hUi6lfqRDNf5htx3lMgI5rAeKF7ejRCsi29rJDoO0dq5283m6J6bzDluebEFri6c7QE0y97rc1CQoacbGyOg26t4/1rzelpBIBbWgSBdUbxNUFvq5Ix+sUiMytDQsszdIuLbmhzJXT27RRbXmGPuLrFIjMhUUicqwm/Sqrs1BE9gYxtFIBzYU3RCRvIBkV6VTi85bRqyH7LM2325eQ5FJqDUnqaU5bfGEZmBKB+i1ZdblyOnQDWrZgwYVK6iQCkU/QgKo6EYm57tw7GSkiIgduOQTvuXSKZrHjlov+bPL9p8Bcl9YiIubahBRVTqeViEhZOMFFlQjIXJeIUhGRPRRW9It5fTnm4KmkfT/sAzCnvb2F8mM7d3teozZBALInOzom19TVsv2PbQWtWgHry8Ju9f2H4OApy/7XnSUxyAc66wsYVUmdrv6k3PYY3apV3bfjjLrN4skuA95tlbSGG24GODmy2fJjD7fYCOzuMDD7nUTa1AI4l9J4xe/9nwEy6BpYFechkgB5j6jZPjh/nHyCPrMqHc+JRS1otYaqOxGNHClr4TeR31rky80MExE5XL/6DpEz1LskOyNqn5LSm3hCRCQ7JnSHyPvVykQ68Zr4/kQ0Erbl7F+dQoetldV5fNqkYcFd55krFdBIGDz6xsDLen4xr09EkjGVm2FfY/PoWeGns0kEZNzxyW2gVtCJnbePy/9XHfzOkwSUPXD6pTaUrb3Zj4KtVXE1lgyCJ0KN+KXJpkrqNB/I6ZWF0YMrqxOReDK/Q4jeZ0RefQr2ky0SxXSZ86DIfMgUke9hj4gchJVL4ZBIpuVbwTfw/tfTmidliqwgvMT3n4L90N9HOo+KyEr4oFJC++GvIt2elSr9FGTUaQ7xv2QdfWsjZBAdD6ykSUvgV2g7jWZNIIO2NYHNxFy43HhRCxNk0DWgyi4FvtIphEwf6EQNrNoHFBlJJrj5l6zRM2rAGpJMwF5uNQGLuKN2Jm0t3wEu//RX9tNxkv17wrltva/jFHQDtkGcD+JZVLXrBZKRBMTza0wy5GZa/gJ1aAycXu7/NpHEg6wlcf4/IdbyG5VlULiFxLmzquBSENHGNzqtooHt0Ig9myqjE9m6qpdsSladamyGeGJmgqy0HDykslO4NKrw/Vb05SplD53mlgUPw/0BOwSO9R9XUia0mv+wr1Ow5xRd/X2jkwhQCjeYR2VWRucvfh4t2Xh+OZ4SGBIWGJEvJ2ssERkUXD28elCNPBH5MrbNvfX6bBYR88SIIbeN+zQ0qXeRiCxr1jm1S9PJ+SLTA5NSynx7OZ4SXC0sNKjaJN/oBM8R2d6M4YPuLPRSaEpwtbCw4GphB3QG5OMlGzl38cbqluaV07H+lJyItRwOF87UigYg/0Ks6bpfsinZcbhZG/9rFJCxamasmhlmpMBIgWFGCowUGOb/PFTJSqJvLek60/FpQAZNh0HTGd8LDOO/h6bLPwp1owD2VQsokabqvsr9xJUgSmjK/sCAspLIGM6frSbFza+es3vUDPUqZrM4PjswV/44Ukteu8mepeCPxfu+CNvZBPhk9Y6kfs+o+yr3zUs/oW+/R0lfuzJwzN0xHHpjftzwVw+/t2FnzZRQio5vPjrzca9iXvdsv+hQgHv8ADLem6/4c3x4PPjK49FAwVtHT90yIUb7j7cwq8TvsWgnSTB/duyqeXRTByc98dj376TDmfFzI6nsk9I9Q+laKiJSdIfZVV9ExGxfmEy2IEOXQupbRgeeFBH5lYEiInI1dYJ3T0rnlN9NiIjI5SbJCn5n5Isi25qfFMkd8ZtcHBK5UesBZ0Gv54pl3gAnSRHzY7tFTt6x0cHJnZBisn3/ap3cp8Z04FTlgcY9cyYwQ0REHnHZFxH5ZrqtOQ/LE+Q+bBURKZtomcRgy/DZkd6lYPzc5WvXrVublCMiIi8pU7A4skhExgwRGZsjIgW1GpW61zEPSRGRptFOkiJL3xQR2dXTwcmdkGKyff9qncLfS2e6ToGnC5cvL/9H3xYafSgstDX7By582QTmU6S3A7Z0ULiVSFBUDS+/To4B+OzBOIBNjWorhr74SxCQ+MCVkGWLM2sSlvhNdku3Oj+lbwO+LFFLAr9eAWhy2MFJRzz2/at1qjX01bfjkM9K7i/R6DtYjZ4H9wC/TghMF2B5X+UJdCXc6V0KHgU4vCEFoHDhSMVI2apIgLpFGTQoKgbCOONe5/3INkD7zipJgHpvzjLD0n4OThXHo9i/WseHDyhu+/vWV7T6DjaEhcAPw3sd2g1yRXkDtA4Y6F0KmgDm8a+aAN4er1wAyr0QBhDJAdadigF2B7RyKyM/Nzr2j1cmHVBLAqQ2eiJp/7ef/8vBqeJ4FPtX6fj0GdHUW6Zt1+orbaD/QpCrIYNZCLtsK9qbRw3v/Hzl7iK/vKUmwI7aDuByAaEA/lzELwjYsmu8+zuZ/HOhC6Y9+1D3Hx0lAQjOuH1dwoxvIpyctONR7F+l49MUVPsP9xVp9JUWlZSZzY62DPRPh2X9yzd3/PzLrysHshQ/OxagZF6q43WIIAATVwEoerCnxvGXz6ZhfjTqNaZQKVm+gyZP+63r/4faqYJ4VPtX6Pj2SWmbqZlTXfZnpaWlpaXN/SYtLS0tbartTPRdMlHd92WRr7gp9osdD5Tt9jYFS02xALMfcYw/nDKAUiz/7jQ5enGwe41QYmOB1oc3KCUttue+N6fv7bFyoFnlVEE8qv0rdHyMck1aMn2Aq35ye4DVJ+4HsNzt3DUuffKlcBiyKn3ozQ4ibYsge3OClyn4tBFAVlFUHpSW5Plbf+UalsOvkEiAj09+r0WWRwTVAqhOZne7pNVGv16bJqtefH5Fb0cn7XhU+1fq9PNtCgL+0/b+bi76zZoBHA/oYh+q02XdshbAXWMXBo12ELkR2NTCywyUZvQEyD0+Bci6YUrjJy3bI+ucBviDjsCyXf/nx14/t/sIuOUyQBm1FZLWc1RmV8A09ccD/RycKojHcf+OOr46EYnlAWH8y4dOuOy7uicaPwC4odvu7CgntffjvUxB1qVaAImzZ8+ePTui1ewngfOFYErOAThcqx1s3PCWHyzR+BwM+K0YOE17haRFp7qpAIDYVo5OFcTjsH+Vjq9SkJll+fl4V9d9ZxvETTUtqbitfNNFigG4+uSJml6m4CRh9u8Cl/MBzjboAjyZdRhk0ZP+ZI86N/bhh+79KM69ysOhP4D8kNZMKXm2QRcIHD4L4MT5bo5OFcZj379aByiiqLI03eaekWGdPxMRkd9SXfQt9uU0h0mdLJj4CX/rl/MtA2IJuSslZWinGnT3lqb7lsnlzYdvC4vs8ZrI5VajRES+Ssw8/9SoYhHrf1MlaOlsbp++9eF78h0kLTqF94/J2PVB6hGVkzshezy2/at1iu7pVSe8xcCZVU7TAV8dmqLs/nqT5cK8srcPV0iKP0929wj0zPeXO7c16dTJW3O2Q4JryYNbriR0MKmc3AkpJqv3b9e5djTdrpN9jVUzA2g0Fi4NMzgiIwWGGSngv42gSDRd92GarjMdXwklWj8FSXK929TrTMdnQkkYN6XGTalhbh9WX86x/AyOc3IoOAJASGyAN4ida8augUeInS56zRvETY3lmf3+xBSc+3r9z/X/FsLZH298JQEOp461gQq5X61bWzct6PKeixMG+XmM2Llm7O71CLFT0GsukDkrs+bEx2nSdIAKy7P11Xycto6SnnOk+9xTee6elB4kVUTkas/qW0QW84AS6LIU41oV/ECZXsSuQsZOG7FzR9M5IXN2Zk3Nx2nTdE5Ynq2v5uNcCdl1FBSeiu5zT+W5TcExHhQRkV/oIWLeeskhOykiInKPpfqgHsSuQsZOG7FzR9M5IXN2Zk3Nx2nTdE5Ynq2v5uNcCCl07PScmu5zT+VVuHDZgP1gaudqqBEZwwA9iJ1Oxk4fYmen15yQOTuzpubj0KTpUGN5tr6aj9PWsdNzqOg+91RehRecn+kLZWdPuRg6QWsvEDstxk4fYmen1zSQOTUfhyZNp8bybH0nPk5bR0XPKWa7p/IqSsG2SXe+wb4O0Y85Dx2cP+BBvEDsNBg7fYidnV7TQObUfJw2TafG8mx9Jz5OW0dFz9lna1B5Giei3TMo3Znx/EMm4rcnOQ4dSZezWdnv3WeLeuqyaf2VhRfVfQcbOGbhC8jVkMHfL2xtZ+w2jyo5umkpHtF0VmTuyWhN5A53NN3NC6b55XT9rAdOWJ6t78THaesEZ4xcl9Dpx0Anus9hZ7o/BQlPPz3pq83/1zMHFKvlAES173C7Oa+ZySvEzg1j5xFip0Df3CNzjnycJk2nxvJsfWc+TpvKc6Tn7LM1qLyKTkSxSzb3KbAutJ49e/as5WAIi4tr/W7zHllUjNh5wNh5gNgp0Df3yJwDH4cmTafG8mx9Jz5OW0dFz9lna1B5FV6Oa/U+sMx6JxMdHR092D4yuHCuwm9S++kO6rZ+cmpqampq4q2pqampqUMBuMuULpfCYQjp+xwYu7YtIXszemk6gI9Pfh9aoRPaNB1ZRVF5eXmlJXnWo83ed+LjNHUY/Wptmqx6fvMKNd2ndPKYpgvjtKWxoAhQHHA12K0DsfOIsdOL2NnRNw1kzoGP06bp1Fievf+Eio/T1lHTc3a6LsA9lVdhCgozTFaasq765pLdYroSFGBD6p5q4ojYWfuu7onWjd8O3NAto22UE2K3Er00HWzc8JYJlowAzocEu3FyZwNeKg6ygHKJiQDprWZbdRT95B02Pq5ineqmgohyeu58SDAm+2y7k+4TkZV5Kxx79MUEKLnsMHQRINb//GHmnfUCsXPB2HmG2JXTawpkzkLTKZm1k+qbCLRounIsz6Zj7dv5OB06CnpOTfdpUHmuH1Bk3dWIiEEpKclt7lwtsnlArVrJZ6xDuwbHh9foOVVE5oRMznlKH2JXAWNXEWLnjqZTIHMWek3JrCmQOx00XTmWZ6Xy7JiejY9zL2TXsdNzarrPPZVXqSWb0ytlQI1rwdiZ9NN0mk4aNJ17c8XnqYQUOmp6zj7bHZV3rVbNKsfYGTQdxsKlsXCJwREZZqTASIFhGDQdBk1n0HQGTfc/e1PqOcplGV+x9UKjoY1ZMNQJ7ToZFGAu8W9M2cGgKke5KleSrJIFzajawmgaKBcgHz/X55GmB99pHPPyUCe0a96a9WGj2jWm5KMVmd37Pk1VolzOvJYedEpPQbN723UIObqxd3dlS4/OhQ/yr5Y8epNLysuHKJd5XITlHYnzAlq7QLu2YOVxCjpVNcrlxGtVgE65LbDmBGzFAIHPmR1aOgqj5T6aK/J60CoXlJdPUa6ZfGZtDW7tCu0qp7LknqpGuZx4rQrQKbcF1pyArV7vPvthjqqlozDazOD5Irnc7kx5+RTlyn0ubpS1+dgEXKFdLu7cqgblcuK19KBTegqaccM4nFo6dOoIEMp5nCgvn6JcX1zqXz6rc4RLtEvbfIhyqXktXeiUnoJmeFmobUTBUFhPf2fKy308AbpQrpQdg9Ntm1Zjq94R9G9col3aNmT5wgT4YWL697tbq1CudvpRrnkmZUuNTvX9udGxD8IuPtBcl459nq1iyq6fSgMeiHRsVagDgVD8RvspCsrLEoW4j8cLlOsE9qt6Uzdol6b5DuVCxWvpQqf0FDRjV9bjpm86rmyobOmK59dFP9+aHupEeWnFU8Hl+EiX7odFJHmwfaQDX6hvngbm5Owc1369ONYhlRQ3C5c92CfbvpSz/vEi0/Jsk8qO9F6q8x1WRQ2OOLVEZDcTRUR28fej+B8RkdTGV/Xo2OeVD+wSEbl1kENLQ0gRhXl/n7/mWtv2KFzHo+tyHLukYZ+tFowprxQIjCRui4WinbM2MKCseEpLICwO3k3psc0VSzJrL8Bv+b8D1HsBYMiPC//x3eNEdV+V1cIR5RIoy0zwBOVy4LWc0alPN3T3pqAZCQDtP8iNVrT0oWWm5p/X67vJ34Hy+nRDG/fxeIFy9cFSkXHsh2mffP9OCye0K5xSy4bSMqhylMuB19KFTukpaLZpreU3yVS09MUDRHXY+gMqysu3KNfwf3xXGAwQ1pF6tZ3RrtioXMuGQ/WhqlEuR15LFzqlo6AZAy4WBEExUYqWDp3iLqWbgqAWB9WUl0Y8fh6gXHFxcXF1IPTdc9MtW644oV1cKcV097Z8ANLvxvNyabqqpdkpLXtLVRitUgXNaD0nCNgXGa9o6dAp3Lr7DHCM1lYdexTu4/EG5Rr0zosfmAHm13aFdjEjZgLAtuO3U8Uol6LlATpVcUEzhsQDv6+ZGaBo6dCJ6PdTfTiamdjVqmOPwpcol4hIRvvWH/6yYvxzR1Kc0C4RkeP9717+w6RxyocaVYNyKVq60Cn3OqqCZqXPzdr5Vcs3zMqWnsJouffNWP9Lp+RTtnjsUfgc5cremh/fpqY7tEsObivpdBPXAOVy4rU00SkNJEwNbGVtqtU5RtXSUxht5y5pm2ByRXkZKJeBchmrZgbKhQGxGGakwEiBYRivGTVeM2oYxmtGjZtSw9CFcmlwdPknAwOltKRu+UP1LStPN7snatEguKw4KPxCLudAuG2h7/dL0CjUl0zdNaDpPBKu0Fm7wFqABxzdH/PWrA8b1GiENQWTT79YJ3NqgwWDoPGFxJigjJzmXQr/WBN36PzXG9fJ7zdavM50zO1++5hQXzJ1WjSdnWar+DWjGhScYsgzms6Vt6XAmny8N8x/YqiuV0hocHRbuNvWXtKuTETK+rUTKatxQERm8K6I7I0WkZzhvGx1e+9eDlaaqdNN09lpNh2vGdWg4BRDHtF0rrwtBdaK+6WUydcJubpQrmD8AYKf7zJpNQO23uwwZP9IzevuB/hNHQcXhjUD/PEDWva+Wp3gVjn/nmQCkEsRiqIJSw7ZGNMR3y3uBIR2W7G9HWC+qS4QYv1UBs94TOPo3Tc3NtSE/PMTRctqn3/UbSh/mzi1J/98JY6ITxrec9Bfh45intNQwl3HG/eMQ1c8Lr1nAvDqqnN+DHv7uTmeLFxqlEQDOGzJRoc4ONNGsT3hTEMgbczaRIANfzmCr5k6DZrOTrPpeM2oBgWnGPKIpnPhbS2w9kFCOHDb3DdCPPh2rFESDaB5+nIA02MQeJtiu6XswfCQjwHY0Nn3TJ17mk5Bs+l4zagGBaejrJrOGmzWAmt5x8MAoi/t9OABhUZJNACeoF+P1zYWkwhN2yq2d24IEDEs/SKQH27yfXk0e80z5+pnNpqt4teMOsx2pOAchnbNnD7zIjpfe+rkbS2wVt1fLH/ubH0p2D1jxqsjkqcuDyd+u9uvzh2/qPHTpNtjZ7sZfvDql8CiQXj2ClK9byB1TdMBv/69S8OMSHS8ZlRFwVnnOQ3tynr8mbiOR3TGo/YuL7BWrdU5gN/J03NH5J6jU4ByIiIXvnyoGdaSpCIykznlzVPTxNy8vYi8LvIIxyrN1Oml6VQ0W+EtPa/q1XGg4JRDntF0Ku/iJ8pEOiWLyBLTKZGizrzmAU3ngqOzW2azIKgxYoSsGfbCgy4vnaa0ibsTdt9C1TB1bmg6Fc2m/ZpRDQpOOeQZTafythdcGzBj9Hsl85I3esQRuS+JBp8E8DWAKWluwQ7X0+8L+IiVvcp7Pmbq3NB0ONBsGjXT0KbgFEOe0XQqb2XBtSdnr/n5idMu+P8Ab0qiwUk/0ocDcKe72oUx/T+fFmg7qHzL1Lmj6RxotopeM6pBwSmGPKLp1N4OBdcaNoQjcR6lwH1JNA4chazfmgAUuPq/DhHgwcUPvKF56HhfHs0lTRcSTOFWvzP1rTSbomaaDgrOPs9aYK18qPVID2g6pbeqwNq6Nz+N5OLaNwL0nIg0S6IVAxwZUQ/K7skDmPO3erainLbKpJn7BPrULYgDLpKPr5k6dzSdgmbT8ZpRDQrOPuQRTafwVhdYW/btMZjV6F4dj+n2Td7FksHVyTsRs7oHW17aSv9PrNeWrdN28uPdoSUnNpUlQdC/nolrH/TtUcs/2nz3+R/7wl9ZXCetB2Q/vqW44/2PBkxoCq//uC18WPxLN7kKvH4ny/Xl7gl3KfUpPro/TxNJp8SWAlsrpHE88OlTWzrLU70+8ifl8AcACf66dOzzLDr2obQXNidlT3s1VV88du9yHcbuZHfP3hPv23thxxfbVwS6Z4k8t23tZM/WsnZtTdfgFaQm/TSdmmbTS9Op59mHPKPpXHlbHkGvyGvb2fSno1zeMnUGTYexcGksXGJwRIYZKTBSYBgGTcf/NE031TgS/yybaqXpDDOuBUYKDPtT7f8B+64EPWhRNzwAAAAASUVORK5CYII=',
    '1609.01344v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAjoAAACRCAAAAADvZ00YAAAipUlEQVR42u2daXwUVdbGnw4hIaQDCUH2IAhBQAQVFMQRhAEJi8adQRBQRx0HFZQlURgCbqACgwMz4o7iK6LOT1yQRSAIIrIMuMAMm4kE6A5LEiCLgSR93g917629uqrT2bTOB0xS957lVtlL1f8+x0NwzbVQLMJdAtfcS8e16jUi6u+ugmvOrD8ReQjwhO/zDuXmdWxwJr5yTsKWT+nPl8SgIAFUEhsk7V/yOjVyLwhnp0h36Zx69ouo1MjC/MGj6zn36M/oE78nfseKMF46BbO/rH9TZMXxzhNjgS8+Wuqx7ebbWVd2O3Ns1OQNvrEbzC5GyeH6t25Pmv/I9e4V4ewUERFIZXfcRURF1zxGWkunYJZ6kIheHq4fG3yqwjT5jLmbiMqGpxDRB0MrbHtZ0OsEEZ2/oQfREZgN+mBoBVFRQgFRyhJyzdEpMrh0Ro4kInoxslAzvPyqoB4T/UR0fqRurI2p5pfOuNFEREuR56i476L2ExHR4R5EflgO3dnVvRZCuHQizV6SCuJj4Mss7XMZAP/m4uSYJhmH1qN33Nkt+WMidmYnX5n3TcSg9V064vA3EYNasUnd/7qgHaIykCWNRcX2/8Xc0pD/KqZKDns5+hC16g8JKNmYNw5nNxfftelY70vBvbNEDp8ONOtY9IOnSwIAYMFVlwIAOgxkHthoFlv6t2Rj3jgc2V68HqX57fpBqoUX5jxJ98s5gLOf/ntFvX/PGjpq/VPA0bRb793/xYWrAACBLeMCKExfhYqNGW/m3olXZ94+cPSPbNqr+9on3/t1F7CxmPnvey+6+hS0UyWHtnPMeu+9l2/rusYD2j0eCHw34c3mgwbvEt5ZImWLb7qAwOQvAtKk7Z3Y7AVQ5SLFZhnQ7vEAUAREvfMhWC3Mn8Mkf59fzrVvWMlz5sxZUUjHG58goh6f05e9t18oXUNbEomI6ADKiG57hmiHJ6fsRFb9o0RvjOFzz2/82/WeRcTHrlxJNORl0k5lDu2+Yd1VWHh266B/EJEPRJSF74lGvCB73+HJKTtBlOf9ngJT+aw242UPfsi5SLF5Bj4Q0c7WRDRzAvFaJH9BknTfsIzesC5JBwBsimoGoNNnIwat6NNw6Hx+NAoAPAA8iUlo9k7UeiD/YnYsp+2AAfhs9INR7PehH/wt/tQ5aKeqHQa1+l6g7yvJbVPhkRx0AaIvyN49iUloBjR54IX3Vw/jk3pnsR9W3qLKRYrdkmUgf1/zAJtZLZI/h0m6b1gKiy0lAKVe+BbnfxQ9FgBKV7NjZQAQDaAp3TN+/NRn2Z+nA8DNF51hY2nI/zImt0NhQDNVdmjf2kXt1iQte4+W/vz4J9kbxe3Nh7cdAACUb4Vq9LHF+R9FjzXMQNQSDYSW5O/90gmwTwsDL9oKFO8Yi20L4oe+dRYJxYGjOUAcSlD2YzlABGBA28+BsiVs6uqfAeQ1a8bG5mx6JJKyAq+XaqYyhzYtQADwZr07AALxf0j2TuyuTdLIMX3Fy8gf0+88BwALHpBm8NGZC+KHvnWWZ8B9AUSiFsmfsyR/h1ZvFjB7lvKWYMZXJ451bQwgevic4z8tmn0d9n6d51s+Ljlxx94tD3rRMG974edY2yYy40Bu54T6I+YdOvjZaHYrdjltLdwz78UWkMY2yl5blpnyXtvBmqkByaFZTqp8Cp769PjJTWvf/v6dq+BLy/INy3nql5x+73yU0/xq5v3ijAO5nRMAoNM7/5D/VxjQ5i8ninYtS+kBX1pWzoBENrrl13m+5ePOSxn40rJ8ww5m7M+/8ZPXf27cU6olV/K31zrJ37fNnmVwN1llxaWJAEqjfy1oGQFQbmIUAJSeaZ7vaSxuNpecTxB3k1ue3h9zeRTE2PKzibgQpZ1axhxW9kEE985t3/b7VMdz91/czqMZLRVTapKBopZS6yTdu8lhfoYVvrvcju29X2akPR3tntVqO0W/nf+tup+ad4975dTAK0/df9VxzX3Vca1uWCSA/p7adk2756WWW3/3zcE1uGyya+6l4xrqzpPzDHcVXHNmGfzJeS37vDNrlntuarnNCvG+ThlFhfmVT71jwWY+wfY55Od28YRYiO2tFFW0bqapB68p/GfH9oMI//x1dOOUljM3HR02trfhtJkvvjEm+FgHmxe0OxY8mh0aDW6miuJ7r9Y4PTJ+W6mV16Ubm5b+C5otFSaFwPFWCnMzjWcczj9/HQ0cMswwddOaQiiqWnZEjBhBRDQ33hQRu36ZjbHWmxfUOyQ0OxaMdmjsb/S+1umBaKv9FuWNzv13nW5LhWkhTrdSWIU2i2ccjq2hLvV0y5qcFlUFlKDBN6x69eR/DS3KztiRX1p8eatYZ+bQwOp5AFza5wWt0ygTb+zVCnFdBiscRN65Jt+iENm2p7/XDEDUG0ETMwttFs/Em271TiGuy2Ajz4qaHBaFKrqbbLEy0i4CeRMCired6mf2viPtn5A2FLTamDcO5ZsAIPlidki40WyYsGMVOX0g7YgADn/XqufHTW4FaNNRhTdVFrmby9d7+2i3VIjDohDNPgoYbKXgc/gOCcUWEB5aeDWNx9dNH09ph7+JGNQqd3P5em8zg6J0Ndkvqrru62QtXrx48TeKXQTyJoRtt8alrNlnOJbvn5A2FNDu8cDJSZ62OSNy+SHhRrNhIpid2fvDmr+O+SfYBoZl0wf8fOXIvQC90kzpDcosgBKDLRXyYV6IZh+F0VYKNofvkFBuAWGh5aBm8cS66eMpjLsugUlRJQixqGraEZF649GjR49OT1TsaOCbEIpbriOiy5cZjBX7J9iGAh+IfnyTspsulQ+JvQyaDRN+y886Iwdu2TT/mrXEd0R0+5wqGv6HKFvjjUiZhcKnvKVCHJYLUe+jMNhKweewHRLqLSBbElVBzeIp1k0Xj1JT2Q/ctR9kXJRynZwVVV07ImLaAIgHFHsO2CaEnf4bNO+mYqzYP8E2FHgAdGxfdPO94+RDYi8DDDdMmNlFf0D/xnf6G7LnopcdxtkGbQFovEGVBYy2VIjDciHqfRTQb6Xgc/4p7ZDYrNoCYhxUFy9OXjddPGanN4w0cB2uoqr9sw4N6fNM5LcojOVvbLEorQ/T/RMeoNQLQABXMYHb286hV1rLh+T3x9LMoQr3QS353E/86/+QH+aeXtUUWm/aLKDfUpEqDisKebxT9saXFOMeHnrgUgAo33qLyqVv8cJt/zd2U1O6R/2xtjTTOKgynnLdtPH4+2QMtK7DWFS17ogIBKDYc8A2Dlxx1Xqg6GS5wVixf4JtKCAQMPPg+/V2F4lDYv+BZsMEWd7PDgQAJGI3vpIc/Of+KfP6AFpvAFRZyD7lLRXisKIQaR/F/yaWmmyl4HPYDgn1FpCE4sDRHDmoWTzlumnjScMLn2sj78iASVHKdbJTFKr3s47viaSkJ3z0t6u8f91Wcc+o5S9/0GfWkVHeB84saddzJeX86e/vZlzS42v9WDp0199fu3st0YFbGj2WRcfHeCfQRjzw/BMtl/JDspvyEemP5BJ3f3yM96Eis886Jye3azf5JAVGDdz27PEx3glEqzwN67V4sETrTTKexfEx3j9/T0RE+ZOaJ6VPnzYqZbvisLKQfZ3LiVY2/0qE/Oza5758f+o2IikxNmf5na98OmMVUfbtz74+4xi/0zIi/ZFc2atpPOW6qeP5Hk1okfbU6CSc4K6l1A2KkmuyWVS1K10orew00XnVXwoLyV9gfLev6LSpH+2hgO+8sXsyzWfvljJJl2Kwj8p8f07XewuaheqwKGTvm0REB9YrBvkzswLaOb8Gio9JdRfn60KbB+VH5HUziCdM4TocRdXkpVNjG5rN7cs/BYjolSfDE2vZMzStlIhoUWG11FYt8USQKj1FdRBr/3BnS0+B9/H6YYn145tJKd0A+A4MqJbaqiUeD+Luw3INtXVHRG1TMHUVVVFXsPbq5XWCMzBB8wkvRlO1Xp2E4PqvFsPkQw7PQiWr00Qz3Ifln9L98sl+zOzXfsJ2Yy8z4z5kP62+q/2NmwDs6N3jeaORX4zTXwS+wV5HORdM6nRZ+ownxz5fbNOFugCjFELxWtpt6GNTe/Wf9vjAlKpKHP6HVh98bv5DODIw0WyI7EE+C2FfM8tzbvkMywGvU9pCelDywjZDjsWQ2QnOwCAY/hLEhbIAU2zIoddfbi4juvY+osKrbX8NcZq40H+VSSSL1bPkciq1ZkEYrfDwOtEPv/krADrex5BjMWR2HL/Z6cmUIC6UBZhiQw69nhgTCUR4AO/wX6sq8W/iAPzFazksyp6vSq1ZEEYrTLzOg88uvw/IHAi1+qfEsdST8JoTmf5OQyMcsDmmCqYqVoZRKofyAkkXTlUkx2+v37aVblrJxrxxeVtLh28uuq6VqVc7ThNa8DnDywqUYFJHjvEYMDsOE+f6r+AkknAqa79KJs6CSnI1eHWSP+sl4zPkE2bGaBlAF5cvWrRoUWoiET31RMW6ricp/6kmr+09nrST6NvB3+W/0ULx4jXmygDRpHKiJaPOHb3hhxOPXbnotSvofwvivvrqXNFsENHqG34qvuV57iqoiLGBDOW1y5YtvCWjiHgQycXHD+QVL3yS9t3YJeu/wwb6z49OydYXUDQbdGJqbPr3O/q8ZeaVOSU/rJwSEV13P8mBSao1557zgdcy5L9WIvGDndBx/CYiyo76p7Tc3Gn+NJTRhvbPsNUTZ0G16MGrY/6MqlOsGZshTpjunJvfTXbC69B2bKWsBaRR/2S0iQ9E55puIVqzQ7gK4dIRZIoSaBGUyq5GRTSjc4CeyzMogHwg2hlTQbS3/kFjrwLA8cPKqbh0NGASw3i0zE5IiXP9V04iyU6FbKwfJJ8FzaIHq074M6hOXjM+g58w/TkPE69zTZ9Ffd9+XKv+qZAd2Hm6JzDENpsDKwVTJdAiKJWeHT4eRU23XFfcxKAAifCJiAAua7Ih2dDrGZtO9fSMJzEJzZjQqSFT4zBxhf6rtNyy0yilioM4C4aLbl6d8PeaQXVizbhTfsL05zxcvM6j4565kACZNYnW0CbxOBejdBXqTuV2UbtTlUCLDLH8+fWGtya/WjwkyIel4gRjr9c4dKoBkxjGY87U2E98+jIAN190pplYbrXTMu6BnwX9oltWJ/uzWjLmVJwwU0arkrwOcEfTm8YAGvVPRpsQCN17fQyU/Eu4AjndbCqTKUqgRYZk7t79Wc871n9+vUEBDNu5cAb4NDFFCcnIXgWAQyALp8KlBkxiGI/4qxbEcZC40H/lyy0PlWVjoTgLmkU3Cq2sTvanr05eMzZDnDCDc26iYOqftTH3WPe4mZ+cOp7Ylal+XspFQ7sOW3js0Kq9e7rKFGS9krypAOqr1D8lBdNzaVm+YRHDl/yU/cndSVzM1JeWlTMgKqg8pqGCKQtSlJaVM8DL9VXR4HDfXlHZHfoaFNDGl5blG+Z723tyzcp3W+Pbub0u0XnlGqy+tKyclFtMnALAS8s3+7NzrxDCrgczDuR2TmBCp+KvIobjxIX+q9Bo7S6Gcu3X+LSsnAEN+FnoqF50o9DK6uTQ2sjKNWsvzWjNT1iE/pzbUTDVaYQCRfDmNmikeLEqLGirU/+UJUvlA3pXoTyIUAaR9FWB4uhIlESau951Q1ERvABw8Ogfw+GUj1ELnbK/hh5D1n81CKWWjRVnQbXoxqFVQ5g/6yXTnjDdOf/dPDnfNqCElb14vLfKs6+OGDUe+neiJXjwX1dOPC/dELus6pe2OmLUjtAur+Oa+6rjGqpdEM5FvVyrlaiXc8bIdj6mriuhylTjskyohHJTtekzVVqaSWpsffbXro+Z35QOpqDkqFm12vbfe/QYQlMwMhUwqpQsk1PBJP/czPqDI8uO7XxwipMwlSivFkkz3XEXEQXmXHvaXLbIglsKFfXitrW14Xw7CkYmoJQTWSYnqkxm8YbfRES0YbSTGJUor1ZJM3kAeNIvnQRT2aJKv3KaihCZubajYGQ81YkskyNVJjNvkREAMLCZkxihl4daKM10//UZvGu1AKtkxSTOLVXGJGxJ4EcSPgYAe85XXAdbCkY2BIx0skyqdt29jFSZDBAvh4JJG2L6/vFTCdbiZBdbb7XoVOXLq13STJJdgl1cV6jim/uebdXu9rdlcSGmoBS6ySJEXMtpzZ+6PfTGCwCA2U/ss6dgZEfASCvLpG7XbaTKpJVlciqYVLDr6wUXcLXUH5tP4eut0mcKQ3moTdJMI0cSEVEu3hBMkQCrmLiQUFAK9bOOolk1y4fjYztbB155odyegpEtASONLJOmXbeBKpMW8XIqmJTa66OlvTMZNSb74uut0GcKR3m1S5pJsp/RS6aZdGCVXkEJoaFePB+Oj+H8lP/bVs+eglGxHQEjjSyTpl13poF0kjaoU8Gk1negCW/8/b4MkTlWnSqufn2m8Egzfdr/8mMqXSEGVkm4V1juWLeL2p3K82kk4WNAxJMXp37r1TNYBpOvsiNgpJFlEh4lnmuqXpVpqGFQZ4JJwz2QYC2lrwitPlM4yqtd0kwEACtWLo2QJYs4WMXEheR+0CGaTEzxfDpJ+BgovumjPcbyN3NrBSM7qkxaWSZNu24DVSZoZJmcqUBJqxzhYeuj0ZUiUuozhVxe9b3q+Od9j8lTWs5cVTbhnmvumXrzyfSFKaeneye89MHuI91TV047kvhz/YWX9GO3BDOj0yMLSxrvikPD1VP2NT/yFwARSzod2bS2MToPml40AzlibmgJFjy9LurJemVHCzZdjiSWT6OVk3LbHXr44PMnZz7d68mbXpS+4MStnb4rdtO7V/jSvA8+0kM3mR1FpKKGabenAgdX3DSIR3v66pQRV57Zc1sn+NK8D8/nc7B3Saudz8klAmDlxYkhmnBxNuL55+3BxCHDgIPPF02c1F5MEWs2YtD0ohksXqjl1dSTcxuoFzSEkAxWKXGvMD2IEPmo+CUYMVgWR0UNUndrNSGl7nCtbtetCsvLMw1qL55lAeolDK28arybXEn7NrqiSqWZ6pZKUs3Fq1J9JpO7yQgTWIU60t26ugipao5X5c27XdTLNVeaybXaz+u4rzruq06IgGmZfJuT/HtLccbBhFAeldiKESwo90LF7omvos86TlAv+DP6xO+J37HCOoZT8Ej9qmMvRrCg3IsNgss2gFXaKyk5OjO2d9kPUWsq56lOXjaVQ72EDpW1OQSPYKx1ZQw+2QwqvByx8dXfDMBKd6z0ZdNTnbPKo15Ch8raKgUeyTEMwSebQe0oZgUDsLTxbSh92fRUJ61yqBfXoVIKZzEcilNRFm37bN6f4FpX6pZ3ckhFcPOgCsUs2FIE2xDT9zZOlkmglwa8MlT6MiK/lJ4kV020nuqmVUrVi+tQKYSzmCYUF5MykoRy9oYltK4kqTCudCWHlH+yCCq8BFEEk1ag385NwzJF+Uy4i8XXmFrpSyvupfHEXBl7qmtvWJVEvbgOleC7GA7FqShDSSiHDyJ4DE3LO1mri/9kGZR7CSKspQKwWPlMuEsFXpkofWnFvTSeuCtDT3Xt0qkc6iXrUHG+ixFfHPwylYSybwqtK0BJPMlIGfspxiKoyktQ8S4GYLHymXCXhQmHOh5L7cmGq9/IZ52gqJdCh4rxXUwTahWjoiza9tk1VQxtyztZq4uKE7wWQVVe7Ih3DfeI8k9Lwl0q8ArGSl9GPJbCU6xPuDLxhLq1cRgho15Ch0rwXQyH4lSUqSSUfZNjaFre8ZDiJ31QmapSKWYZCmspNbEkAIuXnykJd3HwSolqaZW+dA351J5KmQaYEuH67bzqOEG9kLQiokfhv19V8F0Mh+JUVKRmQggmx+g8aHrRDJmt4iHln3RBZaqKe/GleR+eL/uIH9kHsbd38EBBYAkAi5NlrfYuabXzOQ53qdCwl44cyp10xXjhUEV+6T01lJgx7uk3djcZTlAvoUMl810Ch+JUlAUbZu9usqx1xcAniWmSQyqCa4MKqkqrmGUgrGVAYEnlC9CLxTdEtQRoZcxjsYXkrkKl4GrbZROGx42ycFa1Pf6UQ1oEd6JzZXtsDep2/fb0dWqA75JDWgR3QlXZHluDul116Q3LhS5cc1Ev12oO9apquZ+wvurwhnauVd/HZAdyP2plpi8+Wmq5XyzYcZNLx5x5sXIYIuTjmoNLR3dfJ2FhXmAuUH7LltXqA09v0g696GVfxFyAXuj3eSJQfIrMzuSTc2B53MpavjwiYi6AjW/pvg5bOHz4pWTc9g8XCET13k12IPejVGYy7Xdn1ZIPodMzlg5tgkSuIfzPsBzJ/dx/fUbHko1543hjOs7NSAJKqpZ8kkcFAGTPNsT0vU0jzFS6MW+c0tHh71r1/LjJrdDgOVIq8jgmsuRTddJzr4Pw8DpyBzcBn3AA5r+jm+yjs72fPK2W1zmO5VQ0G7wxHedmWP89ZUs+5lEGgGxAF5x5EdCL5LdoNkjh6N27jr3WvvBpLZ7DUhHjGHuj7qRHroWH13Eg9yMrM0n97iSJIcbNcAElRUs+4THLSq9Je+kw5kUrzOQDySQRdfucKhr+R4vnCDUlPk4CZjSd9NxLIVy8jmO5n5/Ri/WI8yQmodk7EjcjBJTklnwCZ/E40WtizItWmEkKyB1ddhhnG7SFBs8RglF8nATMaDrpue89YeV1nMj9fNr/cv5jNEQft91cQAmCTlHgLM4+Mw/3yPRQI4Vf2dGQH+aeXtUUGjxH0f5OGicBM4+rO+m5Fi5ex77cj6zMpJRgYtwM67+nbMmnlheyqdfE5Yt0wkwgRdT/3D9lXh8d5KNqQgciJrKk6aTnWlhedRzI/SiVmeBL8z66WJIYYrBOpCSgxDgXX5r30cUcZ3Gg1ySYF60wky/N++hi2dGwHjHnL7p5YYwaz+HckBgnATMCq5ESdi+DKnj8Garcj5rZUdIp1vJC1g8iLISZdj31TsvyUzObztHhOeqxgr2xkYdrzu4mqy2WP43wAi2A936Z8e7Tdlw3bKj8j6elgcdQXiIT2Y1J5lf5UCSxBSJb9uTUZks0/YM6FWYN+O+xse4FUJ3QxY9vJqV0q52PPz/c2dJT4H28vntK6xgl6PI6LiXommtuFz7XXFUvVK4nX3jb6VWrVRtdF5Y3LP+U7pdP/NJ64hfjqu9is5MPAGD/tcYP4vf31f3dP7V7z3yg4OHk27eGGs7RYP/E7j3TZ6SN7jTPUe0z4z5EnXpyLkkzWdkHQyuq+tmao3wk+8a4J59hr75HeqQGiGjUBf0hu+GcDQ7Wbi8smla1RJrJykKmtkKy4PlAPD6DTWGo6A/XLwLQqn7o4ZwNDtJuryo0rVCTcgUSJqXTZyrZmDfOjLGqWjNJiFvh5vPXNVf06pOYMEDdqw8A0GnJfdf1hLrVnmW0YHgao9tMcwPj1ZiwlISYKbWhmkszBXxWaU2rmpBmSmU/MExKp89UNBumjFUVvGGJfEwTkmxnbMahwyOXyUgYY8J2tiZKvfZVlc/JRPd1OEuTyUBNybh8YzxNO9gsN8GrcWEpCTFTakOxmRw+C0XTqualmdhycExKq88kUVtmjFUVXjrmCUmXTkyA6FTsYXGcMWHqXn3i0inqOjIwWd9Hz7R8QzxNO9gsN86riWAMMVNoQ/GZ0pGQNK1qHvUCAJzeUMowKa0+kwRZmTBWVWanN4zcbJoQu0flAZomfNWBH2dM2C5Vrz5hsR9e/Tpg1t1OXz7M8TR5sGlujFeTWTcJMVNoQ4mZiUloFg5Nqxr7rLM9hikd6fSZLBmrKrPtMRwhM00IAOhCgjgez5gwVa8+2S5bPGGAaXc7ffkR5rfexWDr3IZ7FMHYh3qhDSXPjA6TplVNoV6Fz7VhmJROn0nipkwYqyowng/ME5LsQj6wJSZFHGesmejVp9BUOkEA7r1th6r3nXX5csVKESf1YJjmxng1mXVjh4Q2lDyTAIRD06rKv/nOAmbPUt69mv7FidOZb6fteyZ+xLxDBz8b3Tp7bVlmynttL33ql5x+73yU07yzLy3L1038hn7zMl45NCiM/4+Y5BNb3ywh9kXpwrHjWz58q3UjdnxwxPAlP2V/cu28fSUDj7667Ypm387tdYk0cvKnP3RoCc+NX48FoofPOf7TotnXBSl/cI4IJ/zoBjdqZJybf9aG3COULAc7mHEgt3MCgAaH+/aKyu7QF3zmxdKRiGELjx1atXdP14tr5WUze1awBxESJqXXZ4IpY1XFDyKCJHS2nheq42rOy6z9nRn2ZRbN0A8LZblYBsFkbSjtTOeaVrXpyXnDBBVhZWSMsfJUT8pBEmrsVSNhbDy3db1NgLZEZ9EM/bBQlotlECw2EmgYZTTT60WL+Ii6qtZuw4YWTmvpKfDOrgMPE8OlqeRqM7nSTK650kyu1Q5ex5GqUVXAMI5fdfRYS35uF48+Re04xzhM2PiZmotclR+T/Q+tPvjc/Idgi9bxDeZv+6cmduiSPmPiPe9WIOy8zmQ/ZvZrP2E77GItSye9MUGfonac5vfgkezyM0E9qR2FVKKhlXYb+tjUXv2nPT4wBTXw+DNoezQVraPoTHbHXURUdM1j+hnpYeB1LFq76bCW8kbn/rvOIEXtOO3vDprI2eJ4zD1pHDkv0dhsNHerSl4nqKqRitaJ0ug0xd7xryKEt3FY0NZuui/DpxDXZbDB0agg8xw0katkzlGVLdHYbDR3q8ov5wbt0XiHMlkSScHEaKwgPkYQMMoWZIEt+WMidmYnX6nRb2JSSTbNvLebwk/u5vL13j4aDkfgL6yYIDiMrUD2pKbU2E+wyAZ4kGFkw8DK5m4xKjKIzzVjk8LyhqVvj8ZZFIUkkqBY/FCJ7ZxZ2WmDgFJULcjyp6GMNrR/RqPfxMZYQhdyazfz3m5KP/4V+GqbzOH4oRCX4sUY4jB2msipEzaTmlJ40mA/hpH1gYNFttC4kpq7qckgPlfDJoWb19G2R+MsikoSidMlyksnec6cOSsKZQJG3YLsAMqIbntGo9/Ex1hdOnJrN/Pebio/fig5HD9IHsfCGuMwdprIqQKZSk3JnrTYj2Fkg8DWka00rqRLR00GsblaNinMvI6uPZroUKaURBIUi8IuSYeqsdg/VX3Dorg+k0q/yU5vMbm1m3lvN70fFYcjxjHmxwSHsdFEThXIXGpKeNJCRsaR9YGtIwfXuFKTQWyuMZsUts86uvZonEWJV0giKbgU88ZiPn0LsjIAKv2mI/IYO++upr3dfDo/Kg5HjGNhdwXBYWwGsiE1pYWMgoA4ejzIMHLwwGoyiM01ZpMQLl5H1x6NsyjdFZJIai4FihZskAmYbaoWZHEoQdmP5VDrN/ExsJZmYv+a93ZT+SEQ1DJQYhwLa9RtzV4TOVUgU6kp2ZMW+zGMbBAYlpGtNK4kZ2oyiM3Vsknh5XWwnLYW7pn3Ygv43vaeXLPy3dYMk2kk4S93l6Rl+YZxuuSytKycAVEAcCrjqxPHujYGZChl79d5vuXjkpG4Y++WB70N87YXfo61bbpKrArzmsPGwJTXmbUx91j3uJmfnDqe2EaHw3TlWEuhwo8vLSv74hY8C19aVs6ABnxcRylsvMBhBHyjitTVViBRaY4Wz5E9tddgPxH6yEaBZTzIMLJJYAAvLd/sz869QkMGsVOhZpMQbl7HqD0ax14U+Is1l1JcmqiQQWLSTKVnmud7GtdTES5ijM0HEWa93S4Y+FGjMQJ/YUXouq2FFsiGxJMW+7GObLCwhpGDB1aMkJc5PJJUQURSwtoerfY+Oa+5xmh1uCWbNepVA+3RUIchHvzusB+X13HNlWZyDdUOmPb31LZr2j0vqBuol2uuhWcLn2uuuZeOa1Vo/w+KHmhKou1PjQAAAABJRU5ErkJggg==',
    '1609.01344v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAATUAAACJCAAAAABoC/BiAAALp0lEQVR42u2de3QU1R3Hv7smOcEC4V2BqrHlLS2PBBWhXR62HB7COVR6ICg+UBE8YELFHMOptKdgVUSklmJ45IBSoC2FEEMMD5ESBAQiSCA85JHQShSqSQgFQ7L76x+7Mzu7Oztzf3eXDaT3/jO7N/n87u/+dmbuzP3eh4OgEjs5VQhU1GKViMilosBILiJyEOCI1r1N2lDswciKVFeoTIozyTuw9evOaa03jI2G/erC4hauAWer+9z8IIckIlBAevmpf9d//vxrKcROCMkp/HnhfyvferlL6c0GRlZkaNQ2pbiJyD0ihYjIsygYCs2x8GRflyoioqntPMwq3HAwsiJDr9D3hzgBOOdMA4BLpcF/Ds2xOI8nZSYBwNBLDmbDHnOQR4ZG7ay3geiXDAAbQ/68keHJuVM9AABtBjGrEHuQR4a2oV3WfwgAjukASrK03PM7zngQkHP9wDG3nSfYQADQcTS3CjEHmWTIJf6pE0Ne21NLRLRu5O2dJk58j6h25vL98wedMuR4skfszHtgt/W94lITDHr7hIffjNx4MLIiTdrQdS0AfP8dIiLqOoWIiJbfdZxoZm+3P2dxx2qiLS0qre+w7ycA6LqL3/jecDCyIk2iRpVrp3QG3jREbX3Lg0T5OKXnVCXNICJ3q8U2rXnFsglJaH6BiC7MZT0G6KAnPyt9g1sGzF04Z1WtTJFEnucvs6NGRJ6P2zar9EeNPHQ2NxMH9JwdeLGoqKioyws2nhBRzdNYTnuyHrubFTUNpDe3uy/+dLyHD77+R3KP+5VMkUTZuGQJhrYGfwUAx6DsmkP+PM/S/qtbDzD8UznaJSYmJv4l0+qWuQkA0PQVONB/HudNwwDWvdvc2Xb+uiNsEEdK4OyfR/wigaNX2G3oeu9hGK75MlYAL81a/ZuBLQEP+XK6o3VqampqansL03UF3qMbD/CaMyMYN8AJtMFXbBDvvQvsftjBLxLX1j7JjlrpGQBADXoBSKwDzqLm7cd/BFwEtn7iy+nVYRsA1OZZmC7+3HvMG9KdFzUj6FiZAnzY7kE2CGftphmtciSKxNvpDnbU3GlVALDkyY4Aep8GCPFN6gGUxF+raOXLSVyWfwTAojstTO8o/hQAji3MYT6nB4On38pvJgHG3Tu6bLNEkQX3teW/GyTMm5WcmpB3PgcAXhnyO8cwJK7MeP0nxaMvZrom+XIwIu+5wV2PDrLqHti9YfGmoY5de/95F/OBMwg8/+ymH8uA8Z06eYZ368UlK0pfFOpjC+ygK06hkoPulD7e37muJLkVAPeJuh4JdDnJnwN8eaWLw6qn77O+OFry3YO+f8pNLxPtIgwEy7MW3HG8tjcXvJz2Rg+UJy+exi1y+/Y4fLskI2WiFRiuEZZJVob+cacceG7UtqJtT55ggxUdS4gKnYdlfKVj+NrS1xj1gJeu2nNkQudf88H6zmUA4q/Gsfuxc75NrX1lylMyPeA5eTtGjh1nc3U2Ut3gcomzZ7MbU6RSW5TaAqUi3+QalcsRLWuOWweUJl2I4j1NXaEqqaipqEGNKYIaU6SeclWSGFOESAbmBKeqD45Xd09r5flgTKzABmsNtoxr8dtp+7OGJ0Zq/j9TB34zaV7fZw5lfBIjMGatgfzAHLs+q6I26deIiOr6IT82YFT6AqVUZE+XFV7t+JEIPdmaMNv3KdtRGROwAaN2BnuJiGjnnyLz5GRSvzrfx919YgLGKmrOqA7MCbz2n69eqjU233PFAmzQ+5r4wBzL328nHvQPBbgQC7BBz7U2SxN2pnfrXhTh77EGfpmnaftYgA375PFo+bIJSSdHVQBAxTxZ03swMOC7bsnOZDDoA2jz7IyNnpsiamHfqK5kLF8+GXvz/7WrTPItpWX1de3uRDXNdUtmJq1AHVjQe/A3v+y4xhEWrKoHEJ8k4SuHDKOH5noP57GCiGjj3bL3iu6367fG/I+MlkJNWoNe4Hqn/UT7cDg8OAAAhsr4yiHNx4DXFYyRGwoUnHodr+jgO2O2LIwc1EYXhR+C8PdaAFKvM1wyLnpDgYLTE+t2jfd+yp5wW+SgYyVsRhfJNxztI24NpIcCBadhkzO+AoArczv0jxIoPLoo5n0e0kOBQlL2on5T76//7FxWcpRA8dFFMY/a3L4PHy35bsKcyOW+22ZOPlza9JHO0QLLs9aEH13UwFHrC/Ts6f/qieQJKcnlMrEkYDIA9AFl0184fXRN5k39vAaRsUCMXmXdkplJy+5oH2A6uqiBesCVbqB0A6WHqqg1xqhFrePPdeuAkRWpWgPVGigVuQHkZ3EyouUdbPviLz1378LjlZ+M/WzGrJsLjKzIxqYii4KRFdnIVGRhMLIiG5eKLA5GVmSjUpHl5Wcu2ZhUZAYYWZGNSkWWl5+ZpOnz2qMP5e8oODnqRHtQwZ6rPxvjjIKKbLBUkTOboyJrpL0v4cC8c1U/HJ8g5Sto+h+aCT6vCa4LYX3Wt3DU6a1ytd+S2SIVVqBOmvkiBJquUSHkq9kiFWHaUIOKbKrcSqnIRks8FVkj7VXkMCBNfIborUSPlK8lC8yi5ozeuhDBqdfVCu183uLiWAoAdR8ELJiDImtUhPM13CIVIWHfe7/3uGiIL/zvtLssda4VYq3v05I9AZbszrVQUPch2Bcx8Gru9KdrpHx99WKl2BU6L24fEdHR5DLv9y/u2S/5lDv5jgoioprfbwq0ZBe1EFAnQ3wRA69/se2hdTK+bt5OglEbnvfYy9s/mvOLcu/X8sFHZN8N6hf8YN72wlefORdkyTZqQaBOhvoiCBJtCV43QMTXC/NJNGrFRCVrcrSH3LK0Cio9JPv2XrXzz++domBLtlELBHXSxBcRsHrkMaIyLOb7ui1z9uypyFgtuiaW2LoQnFGbBkuhi1RYDvfUSDNfREDzNSoEfQ1dpMI+anXJABBfF3nU/JaOvTSw+ZQ3xaugkaa+iIC0Yv7HhfetkPGVaMWYZuP/1lDreTQsaLZGhdLeldqiVGSlIisVWanI6r6mVGSlIisVWanISkVWKrJSkZWKrFRkpSIzVWR7AVZUmfVb4qrIPpKvImsgW0U2ADIqss0mAQxlVrfEVpE1kq0iayBbRTYC2WJ6aASbBASm9k30yZ0FB/2W7Lc6CAQ1UmCnA3NQpBJhfA271UFI2K8/6+s8xzEit5to7Di5c208vtR+vun1Rkt2aksgqJGexw8SnUYhGzSvhJivV7O+FTvX5DcJCEpPYBf8U4oZlgJBjRTY6cAcFCk6nK/htjqwUZHNBFg5Fdlviasi+0mmiqyBbBVZA6RVZBMBVlJF1i2xVWSNZKvIfueZKrIXkFSRzQVYGRXZaImnIvtJpoqsgWwVWQckVWS7TQLENUajJZ6KrJNcFVkD2SpyAGCqItv0gNtsEsDZM0CzxJ6L7CP5c5G1Ik0rIeSr2VYHInqo9SYBnL54K0vWnfgWpBjIVpHtfFVqi1JblIqsVGSlIqv7mrpClYqsVGSlIisVWanISkVWKrJSkZWK3BhVZNhtGG+5Y7zl79cTnwe1Mj5LF+YyQS/pyc9K3+CWAXMXzllVK+draPUt+3K1DeMtd4zniMGaJbaKrJFsFVkD2SqysdYyKrLAjvGimq5mia0i+0i+iqwVyVaRDbU2r77TZnN7+x3jITYzmGEpCPSR7LnIepHcucgGXyXnIptMxpWci+y3xJ2LbPCBNxdZA9lzkfUS5eYim8mociqywRJTRTaQPBVZB7kqsg6GU5GtV7QW2TEeQgtTsywFrGhtIG1XtA4Dxnfq5BnerRfb17BOh/ZKGreat9sx3q6nr/pwadMHOiPIUm56mThoIMuzFgSvaC0CPpz2Rg+UJy+exvbVtPphdoUg8R3jeTvIaJaYc5E1kj0X2Qfy5yIbay0xF5nMJuPKRs1niTsXWSPZc5H1IrlzkY21VnOR1VxkpbYoFfn/R6NyOaJlzXHrgNKkC1G8p6krVCUVtRuQ/gf5jUd46kkbbAAAAABJRU5ErkJggg==',
    '1610.06272v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAWsAAABnCAAAAADjmDxrAAAQcklEQVR42u1de3wNZxp+TiKCXCQRkaA0SlGXlsS2KVbq0l023VK0VbWWXoLSpZe1NkpQvdBW2gRB2R/rrnVZrcuqRQhRSanctKpFiJImQS4SyTnv/nFm5vtmzsw5MzmnbdrO+8fxnfnmfc4zr5NvZr7vmedYCGb8ROFllsCstVlrLhKlFzcj0aMHlOgpkESP19pCbmVa6Oek8KOhWcjTtMwxxJ2gCrPWP1XMD8oWWtX3tAnZKbz+Kmp9ctZ7F+sTn6oPat8Xmg13NCi1Cq+/hlqvf+DY2p5f1CNCW4qwvlQY3Tu0kV5/zlo//75HYL79y+J9WUOH1dSfWqfE4Na/6td4/UW2R2De6fYMLAuLNtSbUp/I3RaCJbZf47lxz0gLEDzov/WG0OIxLZ7Bub2Gchr8iHwqNlfjWn4qQodb3ES6fTESANpluM3pv98CAB64zz2Yoo2ZmPgOpQyuL7U+M7ka1oLjCI1r5CZSubUhAPgWu81pxpcAgGeWuQezMqYrIod8uvvbdvWk1lEVQPR9H3oAKTjgMgBcauc2UpZHjqx2aedEwAu0dOGvbry2dEoHQOmd6gmfT6xDw8PDh7TDysp68r0GAG9vz1w6TilsiX3nn60vF3wTJwBA9dTSjePrz/c6cYJHYMZEDs89NPYvXetHqU8feAIAMLYJUozMUFFdA9KLm6EL42pf+Iy77Sk0PSCaQCeb+fv7/J2I5rbwD2gS3otuhQX6N3vF/uoM9Zczp/q9X8AvfE7VnL82569hroGZYdbarLUZ8nuNxDqnxkovbkasR48o1lMgsR4vNhHNNr9yP3rMduv62gxzvDZrbYY5Xv8yxmvzHt0cQ8zx2gyPr8t8d9gnTjb1uWJ4CABkH/fr3wIAcPJY0yHBdYGr3fFwAK5lDUZ6Ya9mPgB8vFlLE8GWntu6TxB7m5Yf0i9c3sG4OY+SkwNkPBwgRSDlHpprBRcWJycnJycnL7+RnZycvOxwDRF9l7KkkKpSk5eWOV8rqJo4v/DrOH6ndMtZIrLN3lqSO+bfRGSb9MqFk79P0ze7L4crhXfriCZbiIQ7Xa88rqWF9sOEj04kNt8svi2Zuu+H/RFL+A7GzelaQfFH89o9KuehgGRAij20D3S2T1RU49Co7tiZ2toSdbd3j/NE29sjjgp+hx75Tmtti0sg2ojDbEvlAJwlojVrieh2p1Ki1b2IKKPlTT21VsCV9ot/8vVLRDR62ryFCxeOSuRbWmhTrhHRDN8c4e2MhyqJJuEy18G4Oa311d3FvR+V81BAMiDFHtoHmtCWKCqeirCNFvkRnb9nABHtjAy8Qrdwxfka2CeNy4iKV1SxLa+vw1kiGreMiOhPJ4j6/oOIbH5b9NRaAVeaIDT+RkR0Y1wt39JAswZNIKIvMVd4/2bIVaLpyOQ6GDdXa2BirRNkWyVIBqTYQwqHc+NjMwEAIfPuBwC0nX7gFoAmg9e7HtDmDPAHQp71lTak39kSAND6lW1AUUFXlB1rBcDSVJc2ywFOiBcAYOYcb76lde12Vw2AJvheeP+PojDg8zu7cR0StzqGBOkayOHc2LOnfQycKbz/slUjAHhi7kuuPvVGZvz6M37XX2sibqjc+s4hAMDk1Y8NnbZkbSNcrPUHgMCvdByEA1x10k1qEW9BBwCfRt8BcC2tWmcCQCZiuMuujdm7GnIdEjfdIfJQQHJAij2crqNHxRMR0SLfbWv+6rWSiHZ2qQrKdjWG5CHmMFHy72rEDXMv0wGcJSIq7gEssBIdxRoiol736hhDlHA37rtAthHjiYjo1h+swlbW0kSz9opho8za5yI22mQdIje9YwjPQwYpATns4exAxVqjVSi61RDRzi70/Kuuap2PPkRUgmXC+8P/JqHWlX9duq01nqimdCwnIurRXUetlXBUQ0Sf2U+VSVPFjayliZbQW3biK+35eA3XIXHTW2ueBw/JATnuoT5e8+F3qWhf7lp7e8w6q4u/rTboDiA44IA4gowWe559cMLQvPhNSxGIcgAob6rjT1UBZx/uIrEPAFZJYrNVLmVnm7P2BvHvg6ZvTuU6JG6GbkkEHjwkB+S4h677xoH3fG1vPNgwzcWuTcLsiMKTDZsKpkye/B4SJ1cX7RkPBKS+uBctUQYAZZE6DkkBRzF/BxCEYgB5p8V7D9bSiv3/2e6HHAHjrf0AumAX62Dc9M7VcTx4SAak2MPAfaNf0ZnyaABeT29ytWvfcwBqK+8FQBaMGwdg7c7E9rjV2BsABpcipNsPAKw3HtZzVAyOLABdagXgEqIAZECsMGtpxIkdq71RuKkrQBZcmDFwAHADzVnHcxI3vbVmPMgCBskOkmfq6ntdfrz8amYtUHLeegpoUfDKovKvK8/g6e2uaLz2+XXgqO8kIDdceEymClVAmxYHAWDfGGDqHhuwPyhOz1FJcLnh7wNeo8cA2NTxCQBnECLsw1rqkT82aN7sl8d2tlNqE70YQJplPOvguLmKWivA88gNfx8MkgHxTF1dhywH4JVF9BrgW0L7fCy7VgIRVhoZUulCz5c67NTePnuIKC9iCBHRt+M7BzwYT3RxyAc5GTOWE5FtyvMXDvc6ou8eXYSzo5W8uCPnjZiviIimQbwrYy1VNNtd9oP8QgA5PWFr/to2q/kOxs3pufHKhGFBocMnlnE87LQkSAbEMTWo5yu91VL3/HXBsaAHAgEA8xNk/6Ff5De73/4NPH+84SB/nTPOEtz8BAC2E2c73+cNAJezHhEuXlnLNdr8BMB6/FxEdBA0uOmbv2Y85ieAg2RAbI+fQqd6e54ndaoeQTMK4llaLq/56h6pI+sdmmcp1QXxx6l1bfeO9Q3Ns5TqhGiuN5rrjeZ6Ix/9pBc3o59HD6ifp0D6wXyuwBxDzDBr/RupdfqW82VVVVVVVgDZH264Kmz+bs2GsjpdYn1cBlzbLbxbUQLIcXWF7XDqJ9c10fRGyX5VWoyMOi2HUenUEQChg5sC3+wRJpL/nHsAHQfRuutew1sYGK/nJNr/N3M605x7Y6+89fDTAKqntR5X/tIGf+Mj7PVg7wjrjdUjAABH+3zdHuBw9aEVzxzY9tPFi0eqoukcr0sO5P+r23ZHWoyMGi3Ve/RFET5RXSzNs4nW340uUVFRd/jR8vCGE6n6IXRIM/IsKVMUsOV8R1mD7qc/eSWAgxZCJxqTMKig6bxHZ+oFOS1GRlUGoYo4uy3Rld7Nr4sLWEt9iZb6VBAdRaah53aZooAt5zvKGvTXOsGJFkIfGidhUEHTPx8iq7UIxMioyiA050PCd1RKVhQD3wL61RwHDgYaMzhhigK2nK+lQzB4JlBqIfReyikkDHI0N4OR0aKlcW5sNlLQg5z+sP1UoFNYGnCorzHPBKYomBz82LC0KWsb4UZm6/Wz3p5RWaeDqU6aOyeVAFRufcq+RcLVW+vMDyFKGBzR6hgiECOjSUt1DCFKDrTRAUxMiG1LREQj+1ON/wLDz/6LigJxOd9B1mBgDGFKAEcthBE0u1JBDa1OYwgDYmTUZBCa4zXRelTRAQx4tJO91osbVx/H54ZrLSgKpOV8Bx2CEWcEUQmgooUwgiZIGBzR6lZrCYiRUZVBOKn1ghZ2CvtfJSKiHBx917/GcK27pxIR0VPLiW7GI4kqMImIKODJurpQnMOsimk2sToSrgG0TX8s10CrY60FII6MGi0iIq119D3d7P/2749THRvjntC0rL6GtdqCoqBozxogINV3798UOgQjQQ/2XQAEoXhTwRTgIhKD3r0p4eqH2f+f7b7I6aqCVrfztQTEDrJIi5ZG/fb/j12tR68eDUu/jKwXDBMRFAVsOZ+XNRg9KFEJoKaF0B2ihEEFrQ6UeCkFI6NJy+E6pPhSdUba/EfjHsHNPJzMyMjIsNYAiD1YEGuYjKAo4HQBTNZgeDJBpgRQaiF0hiRhUEEzMl1gBaCQUjAy2rSUg8rLAND+1WqiReJXfw8RfdOg5W3DHkSiooDTBUiyBsPjNacEcNRC6EPjJAwqaDrHa0m9oJBSMDKqMggjz9xdbtTM8Py1pCjgdAFM1mB0xllFCaCiN3ADzfD8tVxKwcioySDMtQK3al3z9kxz/vonivqhWfhNhKlZMNcbzTA1C6ZmwRxDzDBrjd+iz4KD7YHoqcAbD+RnNB3QVJehgdKTgRkaQPRvULgoqN0e8kl8AtchfZ5TIPZRSqsIwQ1CzcPB+ff64pKUlJSUlJQVN4HChRMXFCVdlm1zEp89HhnYuHFjP+HJMXphfVyXoYcBlI8IvqNlZAWA2/GLBrUaetpZoT9+vVeyIh8AUPpy7ePNewrPxh2NLwFQ/MK16Ky7tzhBkyXxCVIH+zynwTIVtFA9acOg6KfKuY/SpKXts0DHAruP6tAI2/htzuaeFLYHzFOBCQRmdbMSbYi2ac8WMUmAwpOBeSRIKgOFi4IKGp8kS5A6FBIErbknlqmgJakwVDwcDPgs9BhjpdpR2MZvc1Zrhe0B81SQ1vVr/V4kou+x1ykF4dgVngySoYGkMlC6KKig8UmyBL5DR625TAUtSYWh4uGg32fBltvfC94z4K30XoBLkQIAVU+FcxWBAMK8jukY/ZX5kqGBpDJQlyBA1QVBqVmQdxhQOyhpSSoMFQ8HAz4LUR/E3oluu/s6eC/AlUgBgMxTQTQeCIUVgBUXdByegyeDaGgg+Tc4uCioX2fZk5QJsg4dagcxU0GLc4Nw9HAwcB2S/MjdvWMH/8HIjx5VJe8SmzfhDQABNwGvz3a0ocdPrERIj+8AZNv0/Dogly/EukOfpEQDePdljpLtvZhRzmDEJIcEhw7XlyLvxYxS0iqkL0c/hZSH0hvIEdVoObu+jspObrbk/rHVBugs6ywBkn1hqZaAwBNtYJmw6giw5GAhbLv99fzyAJcvxOjleQuerMWRSF6nNKvhLqcSISHJMcGhw2XMarjLW0nLAu8+wOjPVykQ1Wg5vZdpFv/R91u3JBugw9ke8J4KkvHAA58teHPmI+V6JiPVPBmCpm9O5fwb4OiioBKSsYIyQd4BfYYNClpyNwiFh4P+WtOIk4Bl2MjtMCpSAADOU4E3HuiSNOMNL3TXgaXwZJAMDZh/A+QuCmqHwBsrACzBsQPQZ9igoCWpMFQ8HAyM19Uf39UDQINgGBUpAGThPBU444FTZ54EMiJH6MDiPBlkhga8yoC5KKgHb6xAFi6B74AhtYOClqTCUPFw6Or6GemyjI5DT9QQ3YLXooKSVQ0+5ra5WEd/FWfsMqmwJKKVHaxEeyOuE00vJqJ/dqwgGh1ko+r2m5xf4t8fR0Rcfk5YEpE1+isietNykIiIViCbiPI6vzZ71ksD12n7mLGknLAkLkGGJnye03sZ9lFyWnQquJTokP95DlGNlsYSv+izcNt7fDjg/wG/zUWtRZFCXsQQ3lOBCQQ+HZdz5I8fOaPADA2kfIWhgaQy4CQIWmgsKS9iCJ8gdbDPc1ZrLlNBS1JhqHg4GNAsXA2zFVbe1cDA/DWzPZifAM5TgQkELqSH9vbTO+Ms5csNDYzMX3NJcusHp2jO56/ltCQVhmt+ps+C6bMA02cBps+CqVkw1xvNgPm7ST/H7yZZzN/ygbmObtbajLrH/wE0nmK6OafW+wAAAABJRU5ErkJggg==',
    '1610.06272v2.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAccAAABjCAAAAADLbZQiAAAQgUlEQVR42u2ceXRUVbbGv0pCImQgBAwQEM2TOYBAghiBRSDYLby4pBtlkKa7oWnDpDg0D5FR0QahFZ6UTD7ohoV00rZEGkEQbGYJMgpJABuUGRFDGELIUFXf+6NS956qurfukGAIuXutrNr3VH1773NO3aHu/eXYCMvuAQuyhsCaR8vuhXmcUf6HioWoOvldUP2MyppHGyskrYC8gtkrQX4XVF/RHljH1bvfeKvK55GaDSbjGDHXyV0nndV4Ht+OPup2Sto2i1knvfyc89jyjFaDyTgGbGvbQWsGt/6i2k5j8fuO/3V7oWtDCpzSy885jycLtBpMxtFveU9G7n1vb8O0g9V1Hj++gtXu7ttaNBNeatj58d3StBAEP132TnXtgD0Zt/9a469zuAWRAKKwtZresNqXmxWDha6aPo/XziICQDiunK+eHfhgWMM/4NSmmj6PN4E6AMKBwmpZ/5WMsRhtg13v50MqN/25E+7X7J8AoEkb/waTcQz/6ACc7q+pyePq9TIAqBeMit7qeT3UTIBlye0Q32/959/9V5XM4/wV7tfXQgCg63r/BpNxjFoEcAuAA4g015OntwPAuabm1Cmya+qb4FjUZgYQBC6aWyXz+O677ivlbR3VGkzGMWr1m1y4BaAQMU3MBWhbDAChqPA8mrLPnP0B9Mv9btkbdapiHu8as6WuzAfwPXqbvAJYWLU/OkaPAoCSlwoyRtTs+6sTQrOKUJgVMrE6Fn9k6yAAwO/qwM6aPY/tvgjrOPrRsg1J1a/0ww261Wk7EcDM1kGRJ+IeLW54KGLkBJS//JzPrWyHOmo06MuuItMn56XLsXG2GvLc6l49PwKwxcUBFg9QEZscr9VgMo5lFg9g8QCWWbycZdY8WqbfSE63RqFa23SS1nWOdZ1jmXV+tKxyrbLv51w9lAoArt25TbtHAwCO7g3v3dDHUzFZ5dpxLKZnI+8mOaRWdgDf76yVFumr+nBAjK70osgTUrN4Mf2hPXX71QOAY9l1U+vqrd43i2PtLyLx44G+ftUr9MNvfzy70G632+32D2/k2O32pbscAE5/sOgSSpbYF2swElc/eavLAgDIH/tj0oGWHwPgjJMDHpmwystTM1lV8Kpj4P2dF4lNsqeZHSgZ8/cnkp4r9FF9lX4VetLLIimkdvFCeo5dnZbQfydQmj7viSb9j+iqXiFL4TP1HoiL91DlcvWK/aCPTa+VmFi7QWIHrFvc1JbYMrjTafLT5kjjuUfR6Zj4SZT/yXb58/xuT5PkCz+SnBSWQ65cRbK0dYHoeYeQTVZN6lVEjsEFoUn21ORSdrrSJpMZ2OmtKkrFf7z7qpJeFkkhlYpXTb+iC8nsuBuc1t5J/j3JpV69dwifLAU90we/db58Q67evx9+gUhOfpBMTOcVZHFeOHm6bSrJdfFRl3gblxi4DpLdnibpjB5F8hu8SQ5fQpL/vU/0VLILqlkxl8mJ2C83CW8GKL58ID+rfZPM/7DYW/XWRwHnUfqsdyp3SKXiVdP3eI2kK/xjR/iLJH/AJvXqvUP4ZCmYLLwnV+/fD5J+x9VfTwEAxMzsCgB4cOLW2wDq9F1t5Ir64TIAdfAD0PRPWcCVc+1ET1v12pVY4OuH2stNwpva9kZqBBAzMsxLtfuhOH1FK6XSLF6wm3uaALDV3XTqVhSA2KA9eqsPkEWuXrkfftc5nTu7z5tTyre/aXIfAAx68xUD87gfAPYjGRi34tf9X1646j7R01YhCMg4uiEUUpP4ppZd35+++nj4tal1RFXRmr9s11e0UirN4sUrDEcEAESdaAAnACfO6K3eN0vJ/BtsmG6DWL1aP6hgiekkyXlhWSt/H7SM5LqE4uij+o+r7uNUl2QHyfxOwBwnvTz1I5NHxVV/bJzh8m4SvYAHtjwk7yQXPFomqt68wK2Bz49eGSSnvEMKxaul/worSbLLI+w0mORBpKpX7xPCO8v1jmfoemYEKVav1A+l46poJeNe+VvCbwEAYQNXGvwFMi10QzBw+9Xns5r+z3OloqetAoYuzZsz2OHVJHqB9ywEdweGfr1cUO2Kj9NftH8qfcV7/tmvGAAcxMJtF+H6PCJSb/U+WaL2NYNt1PJdEKpX7Ueg/TGc3Bz0V5LrErgzrtDQ/pj5ZCFJPreUvJGO+aIX4BtdrnL7WODdJL4ZaIe4hTEkGTlYVt162UUd+6OUQU7lDqlUvFr6o3iPJFv0IHPG/3nSEUxUr947hFKWU5gmVK/cD5Ja9wH6tP3W7TweusPI3vjlvz4NQ067KxtXApGLwzaNlz1tFd/pkgokYMM4qQmip2F1Yt2n+AI55L5zLwBnMSP63TDt9P6pdBXvsTjcBICbyUDCfCAXHaCvep8sfLzHHCAa+ciUqs9U64fm/ZzwK8cLkwAE/SbTwDTuW7siGBcz292uHQwAfQsge9qqM5P6pALXcb/cJHqa1uMUAEfRI3LImcMBrFo3o7meov1T6Slespj2PwFwXv8FDh8fDGTHPwN91YtZaAPPNwFwHokYLlU/XK0fCvNYmFt4eX/HEFw97TzcEQ3P/anukm+Ljrf+TVc9nXA4AeDY756ZycIjw9Gs4bYUAJuHCZ6qyaqkDwDssI2QmwRPMzum9roWja/Cxvioit0nLs30XiKHE9BTvJD+pdmuIHwZnYa/rB9kK509O1Rn9UKW3N6vjw8aOgxAZqtB8KlesR/+Z8elAIIOkFOBsKvcXMu2YRnQ2MlnY4o0ju+XRv0qusGA0TddD7tjHyTP9ns/J3vSUoqeyhlGUB0ZtebYqmYrhCbhTbUTlCc7ycW/Oryp+0avkPxuRJvIx9N1pBdEUkil4lXTu154/szOLrvI9cNzdj35T++uBRo/OUte437k1RfX5vw5+QTpVb1CP3Q9fyy4HVeR52c8eKx+1xhvT/sJnnPvqcZJ0eYfAJ7bE/1YVGU+QFQoPoD89N7QJyIAnNndoFu4geePQpa3JwOuff9p0zHY4uWq8XPksnemWM8f7wFb/Kz1HPkeMEeHVhaHbPE5lqHm8Tk9y/9QsRBVJ78Lqu8J6/87rOOqdVy1jquWoSZwj1BFEH0ovsDmwQR3X+xSvxaAWsECQ6hfLnqm5DKIaEAu4J8efFO3Xh42FZLS7yMq++Nhu91uz7gO4KQbgLT/C7l2+2ZwlX3hZb3dUEAQvSm+wCZhglsGxkfVrl07/FuBITQglz1zcglENCKXey/hmzr1wrCpkJS+H4HqffJ5jWslJtjuP0qubomExMTEB8K5tFHoaJb0Qosdms+R1RFEL4pPFazwQRCHvjxz7ty5Q2ZQYAgNyGVPSa6mV6A2jcjl3kv4pmr1auikGknp+5EAtUx/kLzU7f5rnufOi8LIRbVukV9hP3XOowKC6E3xBR4JGRMcT5LXhzsoM4RG5JKnKFfRK1CbRuRC7z34pnr1AfgmJZJS6SOB+JxGa4v+z+P3mQ30LNsLbIsysbiUhCDCFDc5FgCmvBEMmSE0Ipc8c3IJRDQiF8yDb8KkHr4kpeHrnPrPrn4VAHDk65EvAa1jd/TC9h7Gl0qTEUSR4tPPTbYAsD7pAUBmCI3IJc+cXAIRjch9rj4yjm4IhVk9fEhK49eriWtoA96L2f39SAC2njvg2D3NeAUX+c3Q52DvtTsEQVvWNuPAfcv0Sl3vJQ8BgOIFGwDEdPoewFHXWaNytxdsTh57qM+nn85JMJ7dYx9t/8yeBPN6t91wLzYYecP478f6N0oBfJtTTkCn7Ck9WJhi4p6FjCDKFB9gBEFc0iYIgA9DaIBgdHvm5BKIaFTuMQ++aVbvS1Ia3h/PNwwDsLj5vze67wXePrAnopPxCpqhA4B6kVufd6eKx+bu+pT/OLDJ/SR9+RgAwGNb5jS8OWSy3sc5krzcMycfmfJHpE5YkjzeoFyw6ImDeoyDeT0ARLkXAy5sZHweN7Z3v/bujcOtaqNtgx0Hepi4ZyAjiDLFBwPcJJB3pPz/AQWG0IBc8szIBRDRkNyzGwn4phk9FEhKg8fVL/89TvKT1gC2ntnZpu7NSwiiTPFBHzcZhouZALLhnsfDGfAwhEbkHs+U3AMiNjAm99iZSbNRjm8a1lOZpNS/P+afL8ku3Tkr7SncyMOhnwA4ywCkTL1m7PToiyD6UHyBTcAEj8PNHckMoRG55JmSyyCiEblMS0r4pn59+bDl9n59vBJJKbRp3M95FQCaTygh53mmeiPJkyFxpaTe+wAKCKIXxafxS1rEBF+G+x6QxBAakcueklxFr0RtGpALvffgm+rVq6GTeY37KZKU4sga5B7L7cJ99c09P5MQRAWKT4f8woGn3D85FRhCI0/vlBBEA9SmObmMb6pUrx7i7clQIimt58iwuEfLYHGPlsHiHi0+x9ofYfE5sLhHWNyjdVy1jquWWfMIi3usbFME/vSjgyIjqYT+GUhvDlyUlouU1p2UI95BatNnVUildSdVhiHAeo/Axbmj51yZf8GrTZcpAX9G0EGJkVRC/4ykNwcuSqymFEiOiDtIbfqsCqmw7qT6MKiv98g9UR2GtLgPWWKbvvvkSsCfEXBRYiSV0D9tuZzeFLgos5pSIDli5VKbagte+mwGHgat9R47DXPSMQRZYpu+eVQA/oyAi16MpBL6pyGX0psDF2VWUwokR6xcalNtwUv/zQDDoLHeoyu3dxCCJyHYdw1ImAH+7jz6p5DeHLgos5pSIDniHaQ2fVaFVFokUnUYAq33mPh+ykNo/3kPvzUgzQB/xtA/P0ZSB/qnkN4UuCiymlIgybmD1KbPqpBKi0SqDkOg69UFT7XsltL3lzYz+48f8GcI/fNnJHWgfwrpTYGLAqspB5KdO0xtyrym0qb6MARYJ5A/LR4Qi2HFXm16/y+ABZ0HlpF7Gl2g8+2I/qKng6wvI7kFO+UTw24sJclOHXSC+eXpi36/KKspBpUoZlfRH0N3klexRAwkOnrST+7mXp16/ksk1TsPVaHvZoBh0Fy3s376P39Y8/ECc2e06In/WAw8tmXOrClPFbYSPZ0/a+OxGX7oX11j6Uc+Pqp/XnrmIiPZJVZTDCQ60MVNun/3LW8NwEDnJaHiZoBhUN8fXQMOkuSwbob3R9esLSRz0Ld8Owcf+XkBL/wfm0AyH2PlL2I+3iDJRr/VI5fS/xjjIMkX+ypmV9PHjiHJur+UA/l2SGt/3DK0mDxKMle8xFfoPNSEfpsBhkFjfyz5JAMAQuoZ3hUVgT8D6J8CI6kD/VNIbw5clFhNKZDQIdxJalMW0ntTexj8v0w3s1v131dG3kbQvHNXl4d8IrTp2x+dSSdIzrJtI4dGu1jSPJOip/2NnphP8vVWt0iyaxpJclkLJ7mp8TU9P+Dk9J23kuQrXyhmV9MfrldAbo84LQcSIuqoPq/N1OnTXunzEckJOE4G6DxUhDmx873iBBwGlVo86z2WBo9oBES8L7bpvM5RAv4MgIsyI6mE/mnL5fTmwEWJ1ZQCyRErl9pUW/Ayr3E/Kq07qTgMWtzj5VjXxaKHQ8w8P1MC/oyAi0orHfqjfzqWizQHLkqsphRIaQHKSqA2dXOPWgSk9RzZ4h4tg8U9WmZxjxafY+2P96gFzzCvTSn/Q8VCVJ38Lqg+BVXOPVoGi5ezzJpHy/zt/wERr9JZO2zqrQAAAABJRU5ErkJggg==',
    '1610.06272v2.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAWYAAAB+CAAAAAAyv+SIAAAVOklEQVR42u1dd3gU1fp+N5sGqSSUNCkCAtIJHTEgWFAQhCD9ekG5NBVFUUHpKFz0CkoUDSiCRAF/UhRQLigCxgRCJ/SqlFAMIYVAkt19f3/MZmd2sjM7s9lcee6d73nCc6acd9/zMXPmzHm/74yJMKzizcdwgeFmw82Ybv+7K2w67iY2Zc3E8tT0vHZFNMNEo9P43zZf78J99IWvj9Xabe7K93zNNkuLxdg1zs9UMqe7wumXuhcUbG5r3zj+qnigxWwXZ+8fVFCQGaeNyKHhZl+T1WpNC1A56XpX/9X1tDatfPToqcH+52R5WR3heySHBVl9gF+zydsHhww4VaiEUXI0GGmlG/kZy4DhGRmpXz9raujq7Du7gAvKZJzYFGX9AIzPyrKpNSEFWKC5veWhRy+7mVwGfEiSPwCTSdJWf68aSl3RzWQG8CZJckGga/cEaHazBEzZsuM7ntPRYs/p0et9c2IYFhNALrDUAmB7aCudCFfOYozv1f9EjxmxJ7W27koe0fP1NvXKw5IO724HJHfembWxN7B4JABcOHi2VotaQE5qAQYeOvZgNICczHMxD7pAWHFxgX/v61ElW27dejJsz8m4tkHcezS4W5jQx53d79Okvge8zhy8EP1YKM6f9rf4dMFOK/w7YnP+rYTaALIOnQhv2MYk4Sn8eMaJyO6B3qHn7U6DB4HnyNP3/G5GTzI7MpcsGF9p0tdvBY3O49HOiE15spHpZ5bMrvToosltQ2SdxsSbexqNJ0kW9DBh6+PTP/C9N2vAa59EVDlNMgDt+i98Kbj3Jb2dRs7ffSd+mVhlNbcM80dvsg8enEQ+5Yu15J1JPr0+Gou/S3kW9DBhyxMz5pvr53iDnvf7ZrIdgvL4xlQ+CZ8LXDCC5Cj8i+QiPEOW+IYOtnZACmeht5U8a5a5OTTWjPH2zSZod5FshVoHyafxJskAJJNcik4WnW4eiKmk9aGALDIFNS8VtHuHJNkea8lJeMrGpWjoxNP+483xpTfoVYSblwCLi2POcwMwy9Y4nTwHHCJ5BjhJBmANr/9sue2Hb4WnhvwRmPfAeLLkFMl4TCDZF31JvoZBjrNvApt1uPlyLk8A+0gm4TOSc9H8kVeEZ1gC1jLPD9+Qt5L2OvMUfvxJzPAGPVbA68mAECze0KwWHovDZ6k+bYH9QCiAMGAvANyPql3NR0pQzXX1kEQA+4cBABoAMNn/tTnOCAvAAR18Rv6Gg8CbiYmJK8wFAF4bczDzbZM4Hi5BNFB5XCs5T/nPloeer/fdHDz4092T3wbMz844P3KsCQgErAAsQCAABAFAiHL90TZgW1PJO6r8UqAF4drp3En9J6oCC+s69liqXB65zOHnMOCOUJLx9PEivYp42R4J5PQCMMIH54cCaB2IswDOwL+N45y61XEGAFxMQwRUwu0Pm6jgn7eaHtDOZvnNe9AqFAcBICMVwKSzp9t+OUVkEo09ALAqzxVPL9GrCDfHt8JwPwA1H0P/KgCqvYf3LLC9h7djHeeYFyGpCJxSjOwy9bN/bndJqR0LLbDNw5v3a+WSv/DFsFCEJWNKDnB8SACKZ21aF/H9vW8nO5gsNs07B6yaU8kVTxfmET3vPwLJZJ/TJMnvsFPYsblDzYTabTeQaTUqB9cYTJLc1r5mr4dm+8H0TukTJjIUCIyMDAWAq7xZPSQ4YiTrhYVU6c6u4SGhDcmARUse7du0xWqb2wGdHSwMQBOS3NW9TvcWD/zKt8OCfM/zz5BKoVEZt6uHBkcmkbu7Rj3UuN9VKU/7j9cODa7S0wv0WCETobZrUcJ/4Z+OxxxvVClz41jzw3ElIDBA3x2VV9nXo4lQ5oSZFY9lh/or8yw/PWO+2ZhvNkQqwww3G2423GxYRZrH4+YEkrUM/7m3aRU1bv5PxTBUwICuggaFFdJpWACQ3kIoJ1r5yXiBk/dn6K6+kWEraP/GnB4jAAC3Lt1XeiTnRLU6PjoRJOXZFyXnTI0pJxnNbFQ5aSbl7TmN7OihhSz59z1YRBZdTH+7Rhv7gV9aNhk/8MF89+K6FEFajvMdm/xNO2B5ytR7sd3ttL4qGUU2Cv5Q4qRKqiLVk8k4RZKHzIvItW2GdEa8sH8mRln4Kua7pyBFkJRtfl+R7Afkk9errtbkZiUyymwU/KHASZ1URaon+7EPAJo+DKDP7hWP2HenTa32gRlHUKIPQVLOqZToOKVq7yvlIaOHjSonzaS83jfXxYgDfZoHYGYVp93z0TwAWLG3iz4ESflaEz/xnKY3ykNGDxtVTtpJebvT2AEA/u3m2wO6Ztnv0wg8s7DTgxOuaui3pAgytNL7U0vUkQoZFTYK/lDlpESqQpXtVYIQFn9D2rJCoM6copM1QvdooCBFcEbT62bXZNTYKPlDjdNf4mZeT3m2EYDnpS3LBerbyJGOZ5AqBSmCE5puN7sko8ZG0R8qnP4KNx+7Q5JZ/RAtbZnNjAEkJ8NU6JaCFEGGptfNrsmosVHwhyonDW72+khjZDoARH0ZeEMa4WBqgGAAPmCRLgTXaOUko4uNdzh53c1cAwAIjGjmBN0XlwHko264LgQFtHKS0cPGS5y83Wl0wku5JJdgo9N9+mdc2AUWNcVq9/2WFEGGprfTUCCjwkbBH6qcNHQaXh83m16s0rxa5Gnr+seBU518S0JOx1lenIzIbSNbJxwqWtlfH4K0DMxK8i0MsTW0RhwpHxk9bNxx0kbK6xOhl6NN1qvX4qqWrXAlK7aaSQMFKYIymqaJUJXqCmwU/OEJp7ssgMCYbzbMcLPhZsMMN/8VZp7ucdUu9r9yWxdvYXS5S8i4fj2ZZlxtuGsDCAwz+mbDzfhfDe4y+mYjuMt42TbMcDP+axfhSb3cJtIPgJ8Z2J8W9rg0OOLG/m4AYEs9EveAG8niWHpYtzBnNMHO7fTrGSIrqZn9N51KgGX9IyG4trcHANuOYxEJUZrbZydmN7Ep8rZqcfMfGwSFK6DxPsD//va+wPmNPn2ii74o8R0arIq1VXij9MlsxOcrv3Cjz+zOpa3cdmxp024Ast/q3nrjiI/UZtOLX+DUS30+aCZBEw4UvRw3vGDw18HSkpqPS39TUgIAFCSao625ywDkzHzi6YOtpozR5mQHMcEcTZG1VaNINc0vPr5S1fhm6Blnir/P3PI8ua4eevJCW7Q8pi5SDXl51rvvvjtoOrmsDcn0mDz7gas/ZHfqTZIvXCM5KSBTRRea2tRKft3aJkET1Oieb5IrsVNaUhOpHL8pKQlLaySMGjj7IklO6lpIjsUlbUq/g5hgjqbI2qoxgODNWmT8KF7H2vlB5Pn7u5H8vk5oFm8jy40WOJ4kc4dbyM5vkLQFfSMR03qTtIaPJnkQM5UpWIJeJHkFmyVoJMkNlfLJ7MV3pCU3WqDoXCc3O1bZmBNxlXwdezS5WSRGOjWlbFu1BBD0fQsAEDGrHQDUen3bbQCVe3yl4bYaBwBvzTAjPy0WgClss3y0VLcEQGWoxBmeuRUKoLpPmogm2IxuwUDEcwHSUjntjevVgd21m2o6WSTm3BSltrrpm1sJK0D5CN7GwdhAABgwc4J7JvUBbGx9D/CHJRgAQk/I3bwHAPaggzJGVVgBWPG7iAYAyN0z6qvjQTenVJaUPLSiBXmsMcokDLNWHt7kr6mWSMy5KUpt1T7SsKzL/3n5YhMAPD4iU9O6eHcWbgKQBzMAhOS5OsX2fodBygARLc8BOGz7Q0QDAFzmwSGDkdQ11VcseTiE3bq+Jp/O+AwAUrZvSGqtrZoTMUlTdqu1VdO4uej5CV80/psw7nh6uSYynzbyAUBhIRCLyzeqqf6bzCoIH/9yGbYfgkNENOHqgfkBYMjuzyUlzyw0oyZMoz//FQCGJB+dN9CirZ6UmKQpqm3V5Oagi9e3HFkhlIelWLVw+bwhAISiAAAKwlycsXrvZtVxc/ut8+a81auggYgGAKiJZgCqhGyTlMrxqlAHW4SN8NdXf6KtmpSYpClqbdX8etL9/pNCoaP/Dg1Ujh6qAQAxyAeAfBd98E/frQtAptriKo0XAEfQTEQDAFSuLlwVOZKSZ1NlHTvPA8KRDf6zTTegMTY9r62mSEzaFJW26njZDrp+fA8A+AxdpYFJOmoAQETTPwFYcx8ps8hOxvplAbisBnVgJYD0OokimgDS+QwAS2FzaUm/jwFejAVwEfH4fdJcALlKS1upEKOkKZK26nNzwa6Cq3ssuHHeegCoceHV+QUnC49j6DoNVI4jAgDw0o824KfwnjgeY1/sxmIFgGPPhM+a9sozjVQg3htDFM+d6y9BOx6TDEzZfRP4LWCstOTGLFZZ6XhMMnyGDAOwqsEA1Gz9EYAdphHa3CwSOx6TLGmKo606QxWTAfjs5RQg4Aa3+Jk2fQZEW9k/otBtqOLLuCi8sr3wj993tvmVPBG7hGTW6KfCq/Ybk2+zr561T+UNaePwzF8f+z8nNAHkk6cObH7gRzqVVF5PSn9TWjoRu4S88eL6zHc6nCB5aPSaYytqLtMY7y0SOxG7hGJTHG1VM7fzqzm3Y7TPN1/a28sel3Z+l//DwR5N8f6eWrVTkAwNAHAhLbx9qKzkyXyzLeNUoxZmALDuOhPdOlzrfLNITGYa2mpM6xvT+sa0vmGGmw03G27WaQn2v3JbgrcwEu4SMsZIwxhpGJ2GYbirvuJzeFfQQzXKavYS7V2D2O44V4Im0++dwdXe+JxDBCTxC87RAJ5glW5qCYooczUfSEpKSlqZC5xOEuw7HElK2gKuSPrYzdc+OP10v+YTVwC48e3sNgvFA8Wj5j8c2+cQwHFf9WzcZ6caiHiuA815vxxcdabgFcvT1VotKt3MHnet9d77vnFC02wyLMemCKpn6mh+tF98Y1O1w/zqPjSOj4+/J4jJUf5jWNQV9XeoTx0tX0GyuGFOGc3eob27FNsVAghENJl+LwNXy3KVhQiI8QuyaAAtU0cyLMemy6AI97DTapFZnard5DacIrkogFzkd4v8zVlpd+Hm4Z+S5BMZcs1e1N5diu0KAQROaDL9XqubnUMERNFfFg2gyc2ycIPSTddBEdpWIIhaX7jEXuw+F0go2QX8EtrCzY0R9+pa4PqFJorauwaxXdTpndFk+j08CREQ4xc8QZOFG5RuagiKUB5pRPYXAjMOLan3EtCw+g5ge2ezGyLPV+n71I4XVgQqau8axHZRp3dGk+n3OgZSKw+v9i8V/ZdAiF/wCM0Jy7Epgnoy0ohfQ+D9iNRzzwEwJeyAJXWqOx7V93dft25eYyhq73nuxXZRp3dGc6Xfa7EyIQK29zsM8hBNhiXZVA+KUBk3R+YVAycz7bdCl7TifQVd3NG4/co/1sa9NrgYStq7FrHdodPL0Fzo91qsTIiAEL/gEZoMS7LpJijClUg1rRZJzqvBbTjFnyaSJDPx27+CS9ytpzE4mcwbZf/SodNTKnP8O5MO4fXDeJ8k63dWe+rYz3VGk+zX8wgUVoPCQsnGYwVl0HQsSuSE5dgsBdU70iAf6m4faXB/IWmrOrd/D3fLllyLsJDkiz1ceyITKdnCF56i/uauZZlIcUYT9+txs23OVpKZEDG2DrlDHpahaXKzDEu66QSqa62jn352BC+0XgOYEtLT3c5d3a5kBoAeVWWSvai9axHbS8+VotFZv9ds0hABOsUv6EeTYUk23QdFuOg0/nw2Km377KCe1tyPsDotLS0NS0kuDEe620V4Wm0jyQn/Fj5n15Mkj0V9SnJIuI1F9VaRn9W3kpujb6pcQI5zRbRjUZ9K9ovgbq9ma+sTJOeYfrFjHG00ZdrUCd1TZGiarmYZlrgpgurpNF4BgHoTizi/dCzyI8nTvjHFbt38x+MfZqZPSpaq94L279DeXYrtCgEEItqJ2CVS/V4MCHDbN4shAs6ivzRMQWvf7Izl2HQdFOHpV3wuBUa6n2/mvmOR7SJUtXcXYrtJIYBAhqao36vNN7sIEVBHU/GHDEsR2pjWN6b1jWl9www3G242zAggMAIIjJGGYbgLlG0xZ91JlXZko5dNd4dWZVuS0S6XvOEmNd6F9Ly4XwSgSWeHq3qqCry+1PgBoZdTztYZntI/VrJLHcyRs84ZzftlTXxkKGTZ6PJ0d6hkoCthyA+4TY13kY//26iuEdCY1I4y9VR56k2N/z4ttNmg+oFYK9mlPqch5qzLVGlHNros3V2Psi1mtMsOuE2NLys9F3bDKdKlzq4632yvp8pTd2p8y2FWWgZhrWSXupvFnHVnVVrMRpelu+tQtiUZ7bID7lLjXUjPs1NwinSps6u62V5Plafe1Pg2Rx7ygXkSzLJseWUTc9aVNG5Zuju0K9twL6ArjR3KSM+ptWMAQFtSu8Ts9bTydLvoe/wokuzQ8hzJHwoku1Sv5pum0SlT5r5xi+TVmuizfcAhx9oKr86fMX2RPfZkwzK1Gyobk0iW+AxXxpAd0LLQA/k1viwt3ppg24ZTZCaWkmTDzhqv5tJ66jx1fJBD8OmeaL8u03fZqNXNR9FhJ7mwbQnJ7JbAPGvpkdwWv9OWOIIkeftRqyqFlgNJ7kM3FQznA5rcbG3TwdFTzbwk6G+/YTlJtmmu0c2l9dR56v4gR/zhhZEft3umSPPIXsxZl6nS0mx0Sbo79CjbIoaigK5iEun51zoxpfqZlqR2hznqqfPU3WmQpG1N4Ltar+ZbGEuSIQNZRpUmyTOYSpLNPnFzQykp2w4MVwfcXM0S6fnWyzYKV6VLnV3xahbraeCpo9Ow9dtHksM6aXUzq48lybBHZRo3be0nkszGOJJHnIcrmpVtEcOl5K3uZqn0/HniuHHjemHIuDsudXZFN4v1NCjwOj7IUfRt3ZYAfLW/JTly1ktV6RyAJmk2uiTdXckOHB8IpNdJvCJigCYJhgRco2WsX2bG5VVNAJowfDiAFd9Pr4cATUntpSbWExrliqeuTiM/vUGfjBLehs/8Czc+9/1W3OXmaj5QJYfcHnxeokoLyvbr2SQnN7hFciKOq1/NSsq2iOEkoKtdzXb9W5SeBTYkF+MwXevs6uEwQj1npd0VHS2dhj01vtg8IgoI/lCyy11qvCNn3aFKC8q2mI3uSHfXrWyLGOIBTanxEulZYMOzIxqFdBzlWmdXc3NpPWel3RUdPcr21eq2y4V1ffVMhDpy1mWqtJiNLkt316FsixguBHQPJ0LL6uzaMZR4GvPNxnyzIVIZZrjZcLNhhpthfMXnv/ArPsbnZYxOw3CzYbrs/wF6Tlinz8BfUAAAAABJRU5ErkJggg==',
    '1611.04741v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAcgAAADZCAAAAACBN/+8AAAi8ElEQVR42u2dd3gU1RrG390UQpIlEHoLASQI0kISiiCBJEgJhm4oXgHhWkAQRYqg0ix40YsVlHIBiUiJRhG4QpQaeotgAtJCKEIK6X3Ld//Y3Zkzu7Ob3bC7GbjzPo9y9sw53zmZL5kz5TfvKgiyHgUp5V0gJ1KWlEREC+W98HBrIRG5A4C8TrpOixY5J6SC9P/JcpGcsLMVJK+Rj47cnT1AUbKh4NGiocJlP1ZRMoCawUoASCnxhNotWIp7/85tD3fSaJo0B3AtxxMVjZtx23QHiooa97PrZAfkRF2JjFSiRmRk31ZK1cy/HRIyI8O2UTFOR0T0Zndfr24TSCIS7OytkS2BxpGbiIg+6axQhW3mt5V3r4Hxtod0eiKJyActiIjSO6H5XUfEm7HQplH9gUX64oKJRJJMJNEJoIVaXyyuc0i4bZw9iXThGhkwB7fmOiLQGduabfTHou/164ebZJe2br2Q/pNhvm16C7eppHpDIAA45IAwJ47Y1u6xBA9MOir1k5TXgRX6NfHTNxRSPtlhpAEaAyg/eU0Z0MkfQHFqYWFv1ZE6nRVMrTa5qCjwicLTZe0CgbQUn+DaAIDySymq9q2BksQpuH0cTQKYOjYOoz5rJ5QPO9GS+2zrEC7VsMAbR090B7C7dKRg5+il/aOo2KcPMtOKioObW52j69ZIeg/4kuhSy4YrV4Z7z9cRnW0F7IkYgW4aprYgSImFm55+fyCml0ya9FGD2vuISLelQdcvlvo/l0ef9qqB5r16rWTrmDjMqBeJFgDtc4kWTiYi24dw5RpJ9CkQS0TUb7lgjkT0EsZTQZASkURrGwLbLc3RhSc7TdPSru5f7OE2X0e0HCimipb4hoh0rfDUgR3ALUFtNCKmaek08HQS0TR0J6JN6FhMtAsTiKg19Cc7TB0fR5hI7Wigf4UhkXYM4cpEFtSCWzrRWd9ckznSSxhPRNGIJKK/ge2W5ujCRCIyMjJqwns3iIiuPfMuEY3HECKiEIRT2fv/EdaOR81SohLgn0S0En5Epf74iIh0Td3zuUQK6rg4wkRSSXfgZZ0+kXYM4cpE0izgTaJ/zDDbOfpEjkckEZUC2y3NEWS81+p8/cYXW+3QXDx/9y/k6j/2Qo35ZrUdvQAPIAyAB8qA1ByoTwNoeOcad2FvUmeIY6KaP3W/+XVbsYFtGcI1mr5Ct2ZhwfY/RXaOiazN0d3lp2lZ737X7PkeR08bPtYVq1WZnH/fBVLdAIyODYSFurriozXa9WThGz3aV20I16jFqG356+8Nbi2yc0xkbY4uT2Rx78uTV3pii/D6x7TWRB2BiMlsRenU9SZ1lq6jOmyL1h1rX4UhXHgFsg0r8n+2vhs0lc3R5TfNky5jtidwx5ZaTs174CAA0CvpgBfUKNkjrLOigZ/bP4Rr1aMn0lr1sjRHLxCAvyqbo8sT2Qw4BiSnIyO70lr+QU1c3bifAfWS1GZAN6TgaKiwzvyqKrci0/DAaNoMu4dwsd4A9DcDxOYYhmyg9JOaSNdZnaOzz1qTa9byVnnX8k4yVuwJcx/wzMR7g3zx7yQfb5Wv90Fh7bve3iof70srfLx9ffpQiI+vt+9qouxpAcERXecUEVH2IEXXjpeIqWPiGEet4aWq6Wk4i1WPnmXnEC49ayVSt25ZYbZzSr19VD7eWaSernpyVOShhkBvC3MEEVF1PFguya1Xw9ZadnXV+BlLxfUVpnVVH1h0CNc+WM7T+VuZoybHow7+9qhZ0018jgqCTAhAJgRkQcYhZcmJlCUnUk6kLDwUpHm4vBfw0JPm8qWHfGiVJSdSlvw2liz5bSzIb2PJgtTvtTqWELhT4gk1HsNfHu5atV9D5GTXoIog8bbX07K7BxrKfxd7Qq1uB6D4dg1FeYsado9caHjU6tnMG/LbWA+sk7+sx6DBryL+0F6PF4c3xNVPtgXGLhNve/y73QnGRJ75fV1R6KAlALJn/9Jg7Lv2JzJry+FDjab4UNb+Ju93AYDrE18Za2+QqvRxqHJXF5SqX20LQLfmtlfJzPqCkkvfxorGcSKiIu9mOiIiGmr5BawcJPAfFuFjQyksWbS17rNKRr6AMUREuq88vyUi+gkv2D35qvRxxINlo7JezSL6l2cikW7sEqIzQX+zJde+jbUJ84iIaCBOExFp51huWsgm8gY66jOfM8nCu3QvVfYmnfHdpQ+Vh4lId9r+R/1V6ePIRK7w2kaUhSeJfvIrJ6IXR7El176NNcTjBwKgu4t4ADgVZmO/FhEXzgIAvrdwcEuweQoz/acSoAjxsf+soQp9HKlGBMAHOcDmXp4Awn8pYUquvSFQO+rKBQAnXvOIJwD/HQSArhwrBQCo791HXhEAoPyOydnbJKwHAPweAQAVp1K0hkP/lf0ZAC5w9LExGhNMKK+oC4cBbfZdAJqsDGiu3wdQfkOj32wIrcnKgCat1IBaHDtYkW7sww8haOMCjSkcDRzBEGgT/QCgcfkBvuTqOzuj8AOAPbH9r54HqMQH2BNzrmz2AjWwq1PjNV99EXAQuDZ64cEPEgX9Rqg2lwG40N4NoNXDS673PgIAB8L26TYv1m59q+z3557bBD4aH8xMzbEPF8PqTweSghss/O+/Tk16Qbd21ZHuK8GH1m85GTuJAHyy0a/m+yGGPvwQgjYukQdQ8Uno28jK9QUAP1zmS65+9Tzb7Qki3Rxah3eIzsUR/dwmh0j3yjgi0gWNSLrbaAtdVB0komXsGkn0T2wlojeuEtFXTfOJ9tTOJTrgdoYowTuRqK1+jeSjGYOZrZG0HBOJKHwkEdGTERuIruC1ZKL13jlsaMOW34kuhRORrpexDzNhvo1L1kii43PCXioiuoyZRETJWMSXXP3Gct2+KZdwLhhD3eKBnUNQOi22DqB4efPvgEJ1rVeju7GYFNwHQKyw40SsB9S3WgP580fWAqKUm0FTo7oCvm04jJOJZgwm8rsJHQBfAIDqwj+Alh7pnYHHS9KY0MYtfwIZycmAYqKhDzthvo2L1H1ZXPrYbJTBEwAUKOVLLr9pPgo/YFc06kZcTEWBH5JvdwCADoqdANARAO4d7ydyCdszaO8d7BoM4Gy+Z1JS0tF6l3E79QkAUcmPGxsJonW0dFsCLfgP7ZWAm+cTALxQzIQ2bikGugUGd5j5+xSxIbg2LrvxExS3d5BWBa3+PQEVX3J5Iocp4qlIBYxC/MXHgb/gBQBKz4sA4A8AlyBGjyom6b7F9lEA0tHAy8vL67u5uIpGwkaCaP4WZnAL/Vninv+/lgnN18Hr4LveX0U9rxMZgmvjOtUNO72ntv4PsAx+fMnliWzUO3lnewDDlD/8PBRojSIAUJc/pj9EAGgF0V/xfyg33PP1BdAOdUNDQ0NDG6MNbrEt1gmjWXjlPvvXkN6WJseHZnSlcPHJzBXfH9Z/Ek7Ylaro1rUCgD+u+DXKAIB76MaXXP88chRmxABo0Of8pbpA1/rHAOAUBvJnlY+fAGB21G/69OWpzwFA5yaJAFC+A007nAOA8gTASw1cF4tmpvc1qy26KvChGaVsAerMHGlYCW0ZwjkqO30+E8AtdFZEpwHAdf8QvuT6RI5A2zr6hPYA4PPNtuuAdtmzQwBUFACAYnXinwCtNE3lJFzoDQBea3aeB/BZcyjWHNsN4OtmQJerAAmi6YNxykcFABS+tm5HVwBQFwNAhRoAqdUA1NAyoY1btABWlgCo6KHvIxiCb+MC1Rq8rxlwMyX8KbyReh2gH99wY0qud77qvpqIiO646e2R9vb74JuYhRVEvw5W1R+4gYjocJ8V22dtRX3hjafSOu8ZSvt6zt84excR0bGuC7YtiSeiay0WLT7AROOD6d/aGRoI32Hjxw3o+vrfREQnY/z9ozOPPFO7TszxDf18A4bnzg1WdXmVC81tCRydmzBwweZd8z4z9OGHYNu45l7r8x8fSeoefZeItoSn5Mx6rkJQshrSGc8jT7TV253sfdr4lCovyPQpy53s9rrUevVqCir3dq3HbS8KMhwf7+UFKQFAfSHQ31I0+561caENKqjpfrmijRdT86BDVPl5ZPIfFNxJAQCZu4t7BpuUrIWUHyxDfolHFmSKTpacSFlyIuVEypK63BYB6CvvB9epr5NCym9j4ZF5G0u+jpSvI2Xh/+XrIkx1am9Gm3F1fxzh4LAFf3t4kEbd2B6fnNyU7NrSPz3QKSWZyPkZSxqlLGy+3dGJvLfp4BHfES3H2JPIO3Ebh0oxkbofUtXK6Qa2/MDKbZDE934I9XOIloi0g0McH/oUhttOIhuw9YjY6v66CDFwu/87FbQpxvAdEq2jJfd1EQCwKUIJQOmMNzK97Fjvs1JhxA8lJ5rUYIkHFh/Tf1oh0TXyun5nhwVW785KgHS1L/4MgO/VAIDjLetJ885OUPx/AUAxHQDLk5fdIpQVANrsOwAq7t23CIUbyHO2s5jEumrTClD+0xEWWwd0N0qllciv/boACO0JAGU/jJXoLbrXMTjyo2MVCAfLk18dMH3HzPNr9+FIeP3JwOqgxqssQeEG8pyF0cUOT2JdvxiS+OqL/woYymDrQOJHJ1xIkttyZN3f8taCD+fpwfLPZygk8yVnJtpSG0DDL4jlyVO8PyfSzm2bQERhA4io3H0piUPhHHnOb+RfqxvJlcW6bq+RR5qgWXSNOGydBnTb4miS/EFPdvLQa7mWrjf9jYjOriPqHi2dLzkTKPf7l9oAHxPl+c0gIq3/V9T9cQ0RnUICEfUfQETks5TdTgPqa4k0Hp+Rrv0AIkrsfJHZKJJIsa40qiMRxTYjYhPJbZVMIm/CLZ2IJrYqpYrXtbYn0uU3BGqPGUMHn108ufbZfM8kAPUu3zvxqhuAIJNDPbcdPPB9O3UggKhk7Gc2miqljVhX1M4B4G5ynelykrxS+SAgAEDnDUcjvpymlOydna2xABR9vxlxrl86GngB+K7pJQSIrdbcdgb45shzdqOp1v9LrCumbvqzQ+H+D02vWVxOkleiWp7+AFATKY3K6+YBGnWem0p6iYzXv3QzAKVoh7qhAAANMkzXbQ3Ab2fEkediG436Wym6tcGrnzW499Uw7vO6yZCi3DsW63+36mXdfhtAaoO3W70hvUSmXmsNAIXojM5NEl8AUL7nmfZJeioeAGqUAcgo10Ph+u0xfHcjeb57kMhGgy7fFO2K/X34T3psXZqKea/CE8hAaJtwAIjv8KUULz+04/IAYNWkpjz0rVhzeg+AbwAAnXMA/FirFKJQOEeeMxshJM3TxzQV58lbzF69OSExH4ARW3c1SW6bXvbZA9CeKW30O6y4QJKXHx0Tp7z3676ZI/IEPPmRkOU/zN+JBCLK6r1497JdTX3754pC4Tx5zsPo+jutMQHwHjb+2V5ueJ1Eu+Z3CWjV3Mv92SIOW3cKSf7g91pPhsaffnlcARERvdzD1y/yI1tCuvjB8pkQunBaG2IEpzno+879dhnNE4YBwP0bbWqdV9X3UYhB4Qx5LrrRMk9e2m1ND0B3ZkLsQjDYOiT4YDnvYHZYJ3tDSocQuG1IpJP04+pfAQAr//sLZELAmVJD48zwwX+mA4B2z0DgUUVIJPEX+d5PfwT22uDEAU4tb9/N+8rvfV5RPJJ/kZJJpNpdQRrnPh/MSitt1lL5iMJXMkUnr5GyIJPmkElzmTSXBZk0l9dIWXioSfOsi9lNeqBysJvnx4uZ3zild3EaoOIMxW4UAS19jBE83XVqt1ZFtz08qELxmAW3c7bsQHj90bNArySRad+un95DxPLbBOxm+PFWueENPQ+kBfUuu3cw8GrO1mOH6UYTfavMblkRT77ow0R4LqRVdtyRAzUmdH6MCc66nbNlOA5el6gFujM9zTu9Jm75zYLdDD+urX2ZiD7GV0T0Z30iSovFB4ZmK/+BKwI0XI+jXES4SXDW7TxHaAZqHV4XMT635IVeXRbozvI0V9rgBWtUzOnPIQp2M/x47rNtALhBCeCJp0sBrw7d/qM/2lJRLXgJMAsv5l82uIf4+KgUXjcS5CJQOczAdDcAUExdPDHJ9GezUVXpY1Vxa/cDkyoWAjt2zwW69p3BluCwB8sWLb+vp3P8eGYXpr5TJgBMuXoIAHC0l1P8xJnBxQjyhIfIAt3ZnubF6XoHTKPlN6zw4x7sqdFTAIBY73X6RPa0NggXnHU758ti5LjZ4DxBfnPfNf2c+SpLZLqELNCd62le/uHqQxMWVICz/IY1fvyxYKa+ZwsAqPVsfD6AApW1xw7G4KzbOV8WJcfNBucI8opZiaqEyCsAX2WFTJeQBbozPc1Dmtwh0o6O0XE24eZgN8uP67/AAquMxbtL6TBWEdH6DJqGW4LTjebjx48fP34oBpAxOOt2zpTFyHGxwQ3g8dqAi0RvdNEyVWZkenVZoFefp3l4E0A5Z8ePMNqEiyg27fuX2mRM/0R8a6+gdQCyGpht6BYXFxcX96n+gy8Agds5XxZ1IrcyeO3CYiAi+Rpfw0SQtAV61T3NK8chPQEgxGPXSP3yoAHg4ScAuz0Zflz0pGDKnPOdzne04ee4d3whNymmLEqOi8DrxoqRIxRp548hH6LYOuyzQDcbuFtg8BNRz5hboEc6AlxXBMU1HXTceZ7mCtUVfWFI/fr1649kN613x1bo+fHCc+K9n3dfi739bRiGdTtnyqJO5EaJDK5b3TOubi9TbN0YAdK2QK+ip7mNgDIVtNUXtpdz84UR7Gb5cVE1HBK31MPNhmFYt3OmbI0rNxt83WTMWX2uNZIAnUJhqLISoTILdLPKKzUXL87dOHtyuKMt0Ct6a457Av64MsgZnuYVAHBMM1j/qXFgYGBgIyHYjVT9alSIzmInUwAm574w1JYfhXU7Z8qiTuTg4HV+cD1BXvjphNZAJrD3iLHKSgTpWKA72dP8VAag+3jEcM4m3AzsZvlxw5RQzv3YFwkY2LgwEEA+WGza+Clfv5ipi4Vu50xZ3IncHF43EOQeNTUALniU3vU3VpmT6RK0QHeup3lU+pzVcSPfKSfO8tsc7Bbw47RzTN+GqoZ9x/xGRBcH+PuGfkG0LJ7oo6frqp4Ydcl4p3VEW5Vf5Ft0YWQ7X1W/143BWbdzpiwGnYvA6wbj8/jmy3YvPTu15zwN74UuJNOrzwK9Oj3N04vautvDjz+YWLdzpmyRKxcMbiDItZfU7T2pwE/ghW6dTHeVBbpC9jSHTAjIhABk411ZciJlyYmUJSfy/1UyoIxHAlB2B9D3gLwvXKZFi5wTUr6OlK8jZUHKgLIlthz2OIpz7LcDuHMWNpe5c9sTKWDLUTVHcZ79rip3DnHw/FHnzh1LmjNsue6zqjiK8+x3lblzS+C5Xdy5yPSrnzt3IWnuUTmmDauO4jz7XWXu3BJ4bhd3LjJ9aXPnziPNEx6Q/a4yd15lipvlzkWmL23u3PGkuYEtN2La6nv3kVcEsLi2FUtxnv2uMncuCp4zRYvgOWuabg6eO4o7FwzsOO7c0aS5kS03Ytq7OjVe89UXAQcZXNuqpTjPfleVOxcDz5miFfCcgd7NwXNHceeCgR3JnTuWNGfYcgOmrQsakXS30RaGuzYDt62B51Xhzs3Ac5ZBtwaes2Obg+eO4s75ge3lzl1JmjNsuZFqvdar0d1Ynru2Bm5XBp7bxp2bgedM0Sp4LjK2GXj+4Nw5P7BjuXPHkuZCthwA0BFgcW2uVAXw3HbunIXNWQbdOnhuPrYZeO4A7pwb2MHcueNJc44tBwD4AyyuzZWqAp7bzJ2zsDnLoFsDz8XGNgPPHcCdc5UO584dTZpzbPm6yQAUAMtdc6WqgOc2c+csbM4y6NbAc7OxzcFzyXLnDifNebZc6P3Nc9dcyV7w3C7unIXNWQbdGnguGNscPJc0d+5w0pxnyznv7wKAdRm3CG6LgudV4c5NwXOWQbcGngvGNgfPHcWdMwM7jjt3NGnOs+UGTPvXwar6AzcIXMhNwG0r4HmVuHMSAc9ZBt0yeC4Y2xw8dwx3zg683k7u3LWkOceWm3l/89y1ZXDbWeC5gEG3MLpwbHPwvNq5c5k0h0wIyIQAZBxSlpxIWXIiZcmJhEyay4JMmsuCTJrL15GOCOlePT9O5k9XKWBYcydELv71dG7gs61oW6x8suN0aRc/1XTJ+52GzCxz+Iq/5rFdI5dFfP71d8sf4qRkjsk3K0nsi0CJiEgzolMuEVFJ/z4l9vSrHJfWveJ3goiI1ruHOCRgNXwRaNasF8NwV1iyIWR1JHIJzugLtz1esqdfRqWtV+BbQ4piQhwSsBoSWXZDs0KfPr5kQ8hqOLRmfPhkV32p6YjVqXZ0rJQuznqn5XjD+j/DIQGrQzVauJmVpLlGbi6NMBYjaD2gycxESZrOhBzmoWhzutjIADON9JGLoo0/T29ftqE2+w6Ainv3BdwwH1A+2amK9qOVsdga+7G7a8PFS9/e0X9OBYsAc1C0OV3MMcB8I4N+Rzvu93oN0/BIeP3JwOqgxqtYbpgP+Kjc2XHxGhmMH43FM2hARO0a/kFU1HWIjkWAjVC0OV3MMMDGRlzkbYI3s/iGYQOIqNx9qZAbbivFNZKI+JVR0mukkqfrSqEE4D2wE+Dz7s4fWQTYAEWb08UsA2xsBCMSyAIybMPaAOBZA3CEX7V8aIXh7Yos7uwEAVx1BHbhbL5nUlLSUT0CrGeYR94PSfv5MEMX8www38igQGRy5VxBQ/bHfHC/ajmRAIAonDUWzyCKq1aprggRYH9xuljAAENAFA0EjyUvM2kIE+pYKyfyQTWu1q8VhvV5p/s/ueqCwrZoh7qhoaGhoY1hhKIxZ3bcO73rADr9Pcp1JgywAEqKbbjLCF7muZvDwqQRmc06OZFVVZ1lmRv1pb3JcwM5IvogBosgwOZ0sRUG2OfLLOM7QOueFzSsQQAyyk3aC/lrOZH26uVZbx4CgJQpYxcDAA7eA0oWjhguRIALAIjQxSwDbGjEadRn724iANitaCto2DkHwI+1SgVoMcdfS03lHNHNl2DDIxDXP8b67p3nhnglfvHmVCUAhAYFtvZe2+ttTwD7F/Rr+2ffwdjz+WGvkDETgB9en9bpTPTac+HvuV2PmKQIDwcSP+xfd1fwAg++Ea+Ds9yntr55oO9YgGmI7OH9w853fDG/58yvDyuemn9p46l6YatrGwNCSo+xKiZlXShu3qbvTKZkQ8hqeh6pPXJJ16aPwdohtMOGm4Wc27YpAixGF1tjgC+ezm0WVQumsPD9G21qnVfV9xHENuOvH97nkVJ4sBzaYQMgP1h++AFltQayHn6Kbm/0pZ3DLsqZcMCfZfUeWrVwg0aplA+tD/8aKcNX8ks8soxyWwSgr7wfXKe+Tgopk+Z4JEhzeY2U10hZkLKDcvWpOE3/r1dgZZPKTcmu3dcBFtgSlk758Cby/tYj+5tN8kb2700+7GS15Z24jUP7WrbAdoQpdTXrwMptdrhgo5pIc4u6golERKVRNU9VgoRHxFqxwK6KkbWk4Csqbh1NNrlgVyt8BSt+2m4A4LWodB6sO1h7WLPAdoApdTVrBQDbXbCldmgFg2f9BTuR8OtK3gJbEfKQ5/F4y3oA54f9Qon3w/rq+X4MAiDOmO/PMJ4O3LBkgc3ZaAOV2nZLUWU/jAUAm12wJZvIM/MGfAIxB2scCNun27xYCwCJH52wYIGtN6Vu0WPu8o/D66VXYtstRX0+QwEAtrtgS+5k5xbCli//MLbhKp24g/UBtzNECd6JRAO6bbFigR0+kjSdK4jOeC4V872W+MnO2XVE3aNZP+yH7mQH6PTmm/O2nPwuKg0iDtY0Naor4NumGYC00VYssH2B/KEeKHi291uozLZbclJvmmg01bTRBVuyJzsBP7cYeFpl7mB9O3UggKhkoHILbM9Y0MsFcW7Wfa+lqC+nGf64bHbBlu7Jjv/Tl3eKMOZX0cgiLm5qge3bHuu+39QY1n2vJajU8rp5eXkadV6hzS7Y0r38gC8yYO5g3Qa3YLMFNv6cPncAktT9rBlnS1BZt98GkNrg7Vav2+qCLd1Elh1QRKDw01cMjLlvbz0S3rTDOQAo3z1czAK7NcDabxfHdlkKXOiGzk0SXwBQvifmoUiknrON7/AlEH3ONhds6R1aDYbUZa/cXNJJhDFXrDm2G8DXzSqzwFYXAzP+/t4DpWsbifheS1/a4gLAdhdsSIvZuTj/j7RaUTWRd6fh7EiIMubHpw3qfKn9yKPLGFzccLr7762BoZ47bv7HD6feS0LP52N7h1Pm/msVHjy9jofleeQryX+6hT49B1tXrWz8fsZ/PB52+EqMMb+XF6S03367ir7X1f5guVIXbJmikwkBWZBJc1lyImXJiZQTKQsyaS4LMmkuCzJpLl9HyoJMmuPBMPGCm0DjugBwsYa7mh6r5l2SfyG7VsT/bSIfBBO/99PFzb7JrQGs/+1c38GzH2wmZpy5veD53e++HeTaRJoR5XwFuylzxjd+LoCvHgATpwuj8ZSGiKi8n+5B52HGmRsq7DAwf3qkS+ErM6Kcr+BLVgzOHexpfguTiYgoCZGVOIcPME/kqtfwMRERTXvgeehOF4lW2GFgHu3aRP7kV05EL44SqeBLVgzOnUTRVQETB/BB0IJUx4yvCPERrUiQ7PJmIMp/KTGv4EuVGZwrnYWJGylxWzBxAN4b1RPUzAHfYEZuNC3XZGVAc/0+gPIbQleekjQTiFzPmTPW5foKGwzMuSnDxAfdubC6GVHOV9gBmyudg4lzlLhNmDgA9Jh7+kPug9GM3GharnciPzXpBd3aVUe6r2QSvqxeq8dSAXxv5LL0nDljXa6vsGBgvqpHwOP/BsY267CBnzIAsD7oTobVzYhyvsIe2NyBaySDiTOUuC2Y+IVVRGUd3c8Y1kjejJwzLTc4kb+WTLTeO4frOC/0gzWje2pI8xYfLHwkCa3L9RXiBuZ59aYQka5DBjtl/RrJ+KA7EFY339lmRDlfIdzksjWSwcTNKHHrmDgA1PgWz5ebmpFzpuUGJ/L0zsDjJWncn3/m8bembIv8DduGgmEpAYF1ua+VOfu9sL0ESJ/QwHzKvA+6c2F1M6Kcr7ADNlc6HhM/ObDQ3Ikct1OfABCV/DgsuYp3WZiyEICJa3lHgRP5EwC8wPU8tdwNwIJjdLy72URstC5/MX8b8O3zIubp/K4RWK07XGZEOV9hB2yudA4mbkaJW8XEDZoXuvwoYOJa7i/oY9LzZX8A8Co+0Qcib83aZF3eesDXUOc3EAHbeTkXVjcjyvkKO2Bzd+dg4iaUeCWYuHEu3wZP6AMIzchtod+84j+wbWrrJpvXvTz8j+vPwAxsZ33QnQur+5kS5XyFn+2wudIpmDjrRA4hJp4gfsYFAGj3wdU7gDXXcguXju6eNmTbYGCeY/Jdh0OafnMoXGiebli0OR90c6t1hz68iE7jifKcMrZCsMlViWQxcZYStwETR4rhZsDMpwAIzMiNpuWGPmoAarPjZcY49pO6GELrcn2FwcA8u3lvk+PAP78NVAjN09VqgPVBdzKszhDl+tnxFQLY3LrBuYMuP1KHtUStEePHR3cZ8BsRxTdftnvp2ak952mIrrVYtPgAER3rumDbkng68kztOjHHN/TzDRxtOJ0/GeXn23MjERFdm0hERHv7ffBNzMIK+nWwqv7ADcT3CRieOzdY1eVV4eBjmRu0J2P8/aMzmUH2xvj7R2dy0yju8JzJ1G/7ZhM75WMx/nVjzhBl9V68e9mupr79c4n29Zy/cfYuZ73ouiU8JWfWcxXEzY6v4Erl4/o3UrUfusJSSGc9WGYo8coxcRFZcy03G+v572xpZtHA/FZzsymb+6A7ClYX3dlmRDlfYQNs/siQ5sk73oVMCODh16HukPUoJPJkTzmPj8Sh9Y/OMnwlv40lr5GyILWv5pUBZTwagLIs+axVlpxIWQ7V/wD0WD+tt8YdyAAAAABJRU5ErkJggg==',
    '1611.04741v2.2.png': '...',
    '1611.04741v2.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAkcAAACqCAAAAACCbbReAAAnKElEQVR42u2dZ2AUVdfH/5teCQRCQCAklCgQQAggSAlV6gNK70qRoiAIWBFpPgoCIq8UARVQpAhSFEQILQQIVWoSakLoSYD0tsnueT/M7syd2dnd2Q1PyOKcD3Dv3HvOnJk9m71TfudoCKqoUmxxUk+BKmocqVJahIgi1LOgSjEkgog0BGjUNdIzkKdw1kuHCWhI/V1T5emIS+l0K+ccANemfJQ/jgfg3UhmZtGR7OxqrS2Yir+VnT3MxXE+kezzADwbOQFAbK4bCp0b2W5Em8J0PCrY703+qYQnVeqGaQp3v251fQQqdZLQoYMLsMTY1bcBqnYYI56TnExElNXMFaMsmZpTBcgrfUdo9qxf79DBCRisJyKa9oqPR7M3bTZBFFuzJoCaNWsGAGhphxecJL7t4d7u7XFN2x/tVsWyjVIaR0RUviWCtIZ21FBgknTCezO5/3tajiM66FhxRETe/sAsrjn9LXuDwBsgIspcV66yvSa2+2J4ChHRnkBYiaNSvD56H7e3GpqLpsqMnzX872vFjq/jrTbW+WPWRm7d4VxMU77DVz7Is0/1SP+st9YGAECX310d9/7Riz2xkLuauIKXTIdPHnt+V621trtixPGnYenKY/QJuGWXatHYQteFGq7dsq8D34echn8OSf8cFVzY8OdNALk7e+DuiRO3DZuzonbdMDQL47Zsv6YzLP7u7d+b4oiB1OYHFLye+BQMTYmGU4cHdqkevIJW5Y2dQc6OG0etmmIhACRfNV6P0eagkU8uNBuWgdULsrBv2rTd3Pb1A44trT0NALC79sikG30bRgPA6ZZN9sb0/8IRA2n4dKT2SC+2mZtHAWxsb5fuASCY73SLgSNerxFR+Uv0G3CZiD77jfK4dfYvqJ9DtBtvElFNGNbZQ1Bngo5OAjeIaCda5BNlhTrFEJ1387lLpH/H8dbZ8US6fkAnLc0cVYx1dnioC7bb7cUQ4EPFR1Kq70O+EYxFQM7fbxhvZkzCUC+ga5VfM0Xzbi1wQh0gHigci/7ugE8v/VhgtPb1KoCmnUMukZzWvYLI94p3t3ntmunF0HYDtM/Hc1qX97H+AdYON95FjHuCwjNnzpwNLLopmlffA3AF8oGrD1EFAF7AxSdZZ1HHgdfanjuC8P2SYpkIe3XWKwDyBtulHQwwC6sUR37eP7Js4Xe6dSOM3QdA3P79+/f3mx9s5so+EXAGAFcg4RbBw5Ev2irt9sWUfcU0MiIYOHfVLtXuQILwVKCTQz4XMYjP+K9WhL7mw//dAdqPYsfz3lkjVqgN5AJADlDLzUmf69BX/2G/ddfH1C2ejbEAdtWzS7Vxx/2nr4UaOnuaO/T7RxNd0ydN4HvVmiMKAGh8EuCBQuTulcyvXRcXAOAC2pX16oBLAKBz2EDq8n9Pw8q9pfbFkWZVIEZmG/7OL5/omHFE6fnJBFQegn6VhCNbX379TqBwTlxVoBlicbyJRM35Z6/V8cDxrf4/ACvL7vgHuD0DSHCkN2MoTZti8Pfd94prLD/+p4ZZ9sURQg7XP9Zydy6g29TtyzCHvO6/7O7h6+m+iiguMJ7ogFcZb18fn2ZEj94NatS+8YfZRPSoq6Zx/SuU5uXl6+11ZYGPt493eyK6PbhKg7pBY1KJiO4Pqd6zV9cPAcxznOv+8+4evp5uP3Gdwn5T7bpoP+/uDsDd3V0DAAn2PqfN/6mli6ZGo4ABidaOxPHeY8sp8jO2cgI0cjMynPkVFR6Xcc3J8PTw0EB9j80uE7l30n1qeFq1ob4PqcYR1PchVYHKi6iixpEqqqhxpIoaR6qURnGeBaCteh6egbR9XkygLQAimql+pqoUQ2Ya7kOqooq6PlJFjSNVoOYbUUUVAGq+EajP19Tna6rg35Vv5PS+5NqDy2/rjawkbkP5ShoAT1JdXfSFVOk2ULk8AMS7uxRSrUxJn1ORbhWbsk8SEh+9EqwGgKPE0afJcyrFzqy2pTdSN0UfqTTaWxt7OWJmIBK3Hj/iN6BOtx3xG3zO1wSwZv+5tt0+eCjpc0akW8Wm7JMTv/61vVTFkX7dnTz927UAQL/6rkfu5ABm7Pe4QqeJATbYYKwBALKW3H5Qf1KgLW4MC2/qdTvmNZ6ilPZL8n3IneE6ItJ1CyciuoSBRESPw6s+IKLzeIOI6FI/tC4iIipop5frc2IyS2TKLnlijREs2fch9RMvEt1vF0NE+kFziM6G3ufHsjrN0NIvPa2+zCjYYKwREVHqwJuU0dcvxgYTFAjAdYbwIUj7JZm3pu8HRER0MpyI6DqGEBHRJgwhokvoQ0R0acUkLCQiondJrk+yW8Wm7JKs0hVHf35DRHShIxHt8CsgojF9+c+27xAiqhVgNQgEG4w1IiIan0hEWf4hRcpNUKdln6xmX6iV9lkb/+vftQRuId9U9AtSA0fY7pd7pneta6FvaavElAPLyVwAqJkAYENLNwARI3O9uKGDW88C2Fhogw3GGgBg147YcvCJ2H6lnnI3Kr4jHpL2S/I+ZOjWPQCgEWErmQhlu17rCt8stNC3tFVsSns6VgegKDUZRYnGrD90/VCyoRHDZwIquEdiHRQ+fIz0bADpMVHapJKPoyrffKsH/uwG6CL9AKBywWHD0Pd+LwNo0sIGG0KLk2oFWgA+SFFuolTdz34f3TrMj9FCdK9zi/NHoknNPzrzlaW+ha2sKVr1Rm5Cq2M42qjizD1fnxowggDgcNOD+g2zdcDenufyP5heCAA3+82M+jKS1cHuBpVXL/suKAqL1vl5/je85OPorZD32179Y/1/gdQ0HwDwwzXDcR0KuTP9q4+v2WJDaHES/SAQwEWXMOUmgAuLFyzOYMak/ZLkjjaVBRD4nWFR0/vBg6Q9Y0P+IHZ9RJRf3+WssD4S90l2q9gUJ8uqZBDtLZtG9Gr7tUTXcYCIDjufJdruFUk7az8h0o8fTETxvlFENA/bWR19aO+jDyptuhJBRPqWz4A7SnoVbq21RHQNk4mIzhtz+6Wj5QIdJVTZbx0a4m0wLUFOYYotJsI26Glb6C1+RNov2fyQaRvH1ga3Rr6OOosXL14dXUCSOKJzLvXymThi+yS7VWyKO91+7xGRzn8ZUecAHVGR6xIifd3ORBTZMD636mdERBewn6h5GyKiRGxndSi8IRFRlN85Ilr9DOLo+rBpHnjtAdFFLl/MBXzEDdyGcxIRvVUjz2oQ8DaYlkCj1e+YRzaYuEBE1Lg3PyLtl3ieUf2hAN80IrqO7mxoMHFEc/ERG0fGflpqampquswssSlDPtFp0dHR0aGTiDpHEBF5f0l02/gVPI5NREQ6zWR6wGVOuoPtrA6FDyUiymuIepP2P4PrtYstUulGBzTTUSKmEhGdxRfcyGOEEBEtxgFrQSDYEFqCTGmfbT2apYpjkCIal/ZLKP/RZgDQtF2Zdc7yvI+bLDgu0+8REBAQ0MfsLJEkoaKHh4fHrx8BhkQjOuAGDFj3VW6bk1s8rsBPTgf+AOAR9bnXso7D9SW+Pnp7XgXUjJx16m+URR4A5BvdLOPmDwCeiFVuQ2jx8uP9v7xtcePEEQDw5fcq7ZfoOtuQcbYzrORMdfnZ9c080/6WxMTExPVmZ4mkDso3adKkSZPKECUgucM1aiIbAAoLaqEGcmR1NABwPWv2qZTFG6NLOowyY1sD0MxsfQ1+lZIB4CGaGY66fg73raig2AZjjb/yv/CrOy7HKXejZyctAC2MOSKl/RKNozgu41UWGsLsQp/7RL+8cc+0Xzk4ODi4kswsGWn4QiQAFPwhuowNOwcABdsbB8QAwGl0QbWXTgJAnqxO7Cag3OQ+l0s6jjw1WQCAoDBouicCQIJ/OPAkH0DPm1oAyWii2AZjjTOBmONLnICd7srdaLjCDUC8Xx1wNth+iceRbnA6AKwYUQVABtirRi2XNi7W8BWZbEgmKu3LbxWbAgB4rN51EcCSaoC2EAAV6gDN6pi/AHxf1XvlbwmAbl7/HtCsirwM0HLksTrQcskCl+cC0DYv6ThyHfAtANx70gaYEpcA0LYpznhUrRWAcd57Ado7urZiG4w1zsSVoY/Hjxs77Idg5W70rQPgVtRiF3A2mH7JX/fXjxz9xd8HJ/dOJ7rQ5yWfMu2mGQb2vfGib9n240919PNpsY6IiG6+RUTSvuGCVbJVbEpYabf4dN0Hu+nYf8qW63libTuf4H5pRDGNp/82ZysR7Wv35cqeM7VERNFtFm+ZuhkBfXkd+rubb0CXtUTbu0zfsPvjJSW/zs5/c8zhC6veSiIi2hQR+2TqUC1RTthQIqJTTbaeGTc40+oiWbAhtDgTjblPu4ENJopmfHt+U71FejLYYPoln2/kbDhdOqMLb1RC+T7uZYea7ulheij3Z/d+eqjxy3TvUV19XIUKniY6mZ4u17S1PZ7Fe2zXT+c2aMq5kvJXTgv2nKVHPWraQMlLaIINxppt77EJinEn/FuwbwhI+2q+EajvQ0J9H1IVqLyIKmocqaKKGkeqqHGkCtR8I6rg35BvROVpVYHK00K9fwT1/pEqUDnIEpPM+66uVFRY2U8K6SKH+UI5eeUkAr7Vjf1b2UCIt/oh2yd6kqslqndy5Dh6+EvUMZ/eIQP9pJAuaqRFBLodTgxtlf8wKvjGk80x0XTrBW5WSrPU9q+OcZg4EhGwKe+t9IPtPC39eNnH+UPv4pjg3Yj+pFuANwAMZkLn8PLfHKyuqEROc/StFNLVlb1GRAuxjIguBxBR4gB8aZi2fBiul94DssDTpk4d0xTsq9VKeVpttyE62twgtRgmBDdWGK/GmNGcmt0dsK4oKx7MUu6X9k4AnGYCSOtfG4AznADUey0P8Ahr9hP3U0fZZRypkN/ukPpA5W9nAPCduFxUwpFGVJzjitkxVm3Mi1zhhP6+M4phQnAjfuWeI9HRR9qyJe4WO/j6COYg3ZSXme0NUqoDGD3mSAQAHG+ZBMfkad2ri4cU87SrGvgCaL5ykZfdJgQ3nMYAwLpRwcLgiZAKjnU/uyg1GUUJjwEU3CqCBUjXlX1xkXtXcoDXjwCA4y0c6sthAWRVytOm3/UBgIDs83abYNyYAAAJx4cIY/m/D4JDxRFHxJ4eMVL/w4pjryyHeUi3ViNme4vqAFCm/9YMAJm+GoeKIykBy/wmKeVpPZ2J+0yv2G2CcaMmAP1785iz+H/vaRzrd63VpZbXUz5FeO0yIxqi8N1B5cTDzTaMO3gQgZ9NkNcetXbjOGBbb8f6sfY4PCi6wSsHXGVuejx+actcp8TW6zpYMeEe9hgAbiHdbhMSNzbWZ079uQrV4GjPaX0vDQNCXJMaAi/lJkpHByRuHFs7eeIied2WoT8CSK3oYKs+bc1pTtE9HsrEEU70d0JIpzH51kzMiXsIaM+hyH4TIje0n4wXBgp/eQuO97y/rhPg7FYPgIeAmwFArBZA2YHfXz0UMDtdVlUz+sxFXKzvYGF0afg3Cy532NfLlMD0RlAQgIYJx63Z6Lnw7TsJX3U3Bd2UmxC58acmSBhZ+q6TA8aRB/OvqOL1GherkO5wlx+wr5ODxZEMAQtbeVpMWRp16P1kU1jQHiQXwNoQYSCuoHx6enpRYXrW83Hdf98JWwcAsADpBvZYP9fV2bEOywiyHrhmcsWmmKcFUL06kBTc0G4TIjeKDncURlLvfgYgruJnNaY8D3F07TYQd7MmYAbSJQIwasfIRQ52XJ6arDIAR8Ay8sTLA+j5hdZNCU+L6G/W+iHjyCIXu02I3IjL9hdsREQAwNawpQ71u2YgYgsBFEIHIIOjb5MGVpFAulxKhQJ++RRPQJfKWcEAMpDpMHHE8rQACrgjso2nxa4/7gDfhgyz34TIjfswFB7nbADQ5WTCgZ6v8URs0BtpHzXyfXnC6Z5B8Hp9SP+WznhfBOkS0a6BbQN9A9sO3E9E8Z39fZp8RzRvK9H818r71ut7xVGerzE8bcHgTpV86/ZabDNPe7nbkX+mtU8ujgkW6/0Dnxoeq3E2aFxzH78O858FT4vnANJFyfO0UlHK0z7+O71Ri+KZYNzQru8eqPBI1Pch1fchob4PqQpU7kgVNY5UUUWNI1XUOFIFpZQXidCo5+FZXLA9LyYiDHHU9rD6oZa8zJr1nJjALKj3j9T7R+r9I1WeB57Wnvquz7wmrGMWpZUnW0uVF/bHkT31XZ9GTdiEt8YPKlGnS8IvCU+btiozr3DCi8YRWbIVlnla03q0Er5WSZlcEYJr2Qv7n/fbU9/VjI5+ibRhftIOjCzGU/b/QVFaO/2yxNNOSCX62i3SOCRHtlrjaSX1aE34WgX1aSUIrjkvilvvyJ76rmZ0ksdKG+Yn6c9kF+ND/x8UpbXTLwv1aRd7/EaUileNQ+8ZyNZEa0EwxzWTiFqOM61Hm3+raLGiOBLckFbFNedFidSnVSbbTRrmJ2nCS9nq5Sn5xfC0lQiAN54Yh2TIVljhaaX1aKV8rQI3pAiuRS8UXa8ZC7vyJVzZ+q6KVUx1bh+8qQeAS58aNvANmUqzxjHdoweiPYiL0YoNSwZNneYnQpeYiYIdx9im4Luxhi07S+q8Er+gjKcdmNUPOIYexiFTshXWeFrF9WjNuyFFcC16IRNHy5oENVy7tkW15nsxsGqtpUJhV76EK1Pf1YyOqYqpjnZqpO/2DteBzZ/kHxg69BemIVNp1jgW3zRgIiA4JS5GKzYsHjRxgJmI73pEThjzdVAvndD8w+g7X8OWmWXivAK/oJindQW0i5p8ZhwyJVthjadVXI/WrBsmCK5lL2R+IxOdvyW657KbiFoeJaawq6GEK1vfVV7HVEVG54egeKIpL+uI6EXjssjQkK00a5wU0YdYp9gpJoaZQRmnhYlb3NOpKHQq3WSbRt95b5hZMs5b9UtpfVoiOvFh07Hi5db6jxWYaFiPiOgdLJKrR6tsfcS7IVsVV84L8+vsrm2JCiuMIdJNJmILu3IlXJn6rvI6MioyOlvLnSHahWumcSRbaZaf1L2P2ClmiolhZlDGaWFi3/pENKAqkajJ+S54wwzJOG/VL6X1aYmI9Fe7/CeVGSyolqTAxE7NA6KCFpgvV49WYRwZ3ZCriivrhfn8R0Oi7uBA7y0FONoaOH83DADCNLsA1AeAhyfa8bee0h89evQoQ6JjqiLSMUifx+GJO6MhU8j7nwy3o0ePHq9wDUauNkcyQ7QH6RSRYeOgjAPMxLJPALj4AaIm57vgDTNkxnmLfsEGnlYTun5fV4b/FJGtUMbTfhqww8N+rFcOwTXvhWwc9fLciEPf6Pbgr26iwq6GEq5MfVehgCyrY6oi0jHe8FrVYn35llBaaVYkoj1Ip4gMGwdlHGAmvvPoMrIOfShpcr4L3jBDZpy36Bds4mnLNz2zVxhlyVYo42mV1aM164YcgmveC9nrfp831k9w9+73a1fyEBV2NbxlwNR3xZYCwwljdUxVRDoG+XDVuZo4Cug1GgD4cZRh+4+j6qC8GV7POEm0ByuGYc4BZmLFCUsqPlz2OgAwTc53wRtmyIzzFv2CQpBV26rohBvgj+v8qIhshSKedteFX51w2amu3W6YIrgWvJC/fzS06xd9MLTLpq4AGgfEDAFX2NUgTH1XVJbTMVUR6XCS9e34mkAKsM+nlUchkAAAXKPhC5EjARTs7ck6xU+SdUreMCw4wE681Ybf1aE2PSGte2vwRhiSdd6KX4pB1vwzTilVgTtoaIBhWbIVCnnamONLNMDOgbwJm91gEVzOhgUv5O8fday4szHaVFzUBgBT2NVQwpWp7yqvY6oio+PqWQTgkmveA3+8fAMgAIaGbKVZflJhjmQPwhQTw8KgjNPMxOofrNqwPTIDAJgm57vgjTAk57w1v6AUZC3T7WBV4HZsRGseZOXJVijkaYV6tDwLWyBAx4rcYBBcgw1LXsg/F3lvLhF9/IGhlKyhsKuxhKuovqu8jqmKqc7WavP+mvvPOy0+LqKb1WfNPkxEfEOu0iw3dqqnv3/3FGEP4mK0EsOiQVOnhYkZLwfVqObh0j+biG8Kvhtr2DKzTJxX4JdinjZ1+MJjR1/p/kAAWXmyVSlPK9Sj5UwIfK1ynlZAcA1uyHth6flaWhYRZWQYu/diCyUT7p7X5v9zO9e8jqmKiU7R5XMFpE8nItKefWx40mhs3L1iUlCXHzO7B1PDlp02TswNiyEi3ak6s9gmq3lFT+IhOeet+WXlivvar6tPGg763No159njL/jxoSITj9YvPaYvzlM+kRtpO364QFa9cFwu+6nLtlXcVdLyPX8yTQuzoL4Pqb4PaSqNLicBgG5vF7ZpYZYqZhj/f7mcXlC3mdf1A23Ga9imhVnq3yM1T4SspCbmVQ1xkjYtzFLjSI0jlRdR10eqQOVpVXlOeVr1N00VqHkiVFHjSJXnSYhopnoWVCmGzDRyR+oaCWq+ETXfiHr/COr9I1Xwr8k3krLjBgW9Xg1POdtHxqVHZdor0UuLfVS2rXhTavyjF5rLTM287+pKRYWVjS9jn96XXHtw+W29kcN88Zy8chIBXx4xvZUNhHj/ixKYPJM40n2x4ZvhTjE92s0rbvVpSbaPB7/+3LW9kuwd99av6yWJo8Sf10xsLjP34S9Rx3x6hww0xNGnyXMqxc6stqU3aqRFBLodTgxtlf8wKvjGk80x0XTrBcPXpFlq+1fHeD/jBCaZf1zRBvfhE4SkzfmoEjN6d2eCT8jrZS2bSN1zFWG93bkv6dZrLi/18WKHcw7GPm4w0FW5F6YK7B5szDdS1LtBGhFRbqc2ucVNqSDN9vFaH3MzDdk7jPlH2g8wmdFgkmSuQU7jDYHnCtcRka5bOOnKXiOihVhGRJcDiChxAL40zFo+DNdLPoGJ5KwffHnBX3+MdfvFeNa7IpoZ/T50SeRXzt6bLZpYXqbFxs0h1a8QEe0p0+GvHWEVTzHDa8s1X3+gf9sixV6YKrB7sDXfyBycNbwT6Dq2uHEkzfbR3WwcGbJ3GPOPdDaNo/BJkrkGuQTBaF/uLd+T4fRoDIcCriAiGpJL9GBus1rcW3/6r9/FnZJPYCI+61mVNhIRjXXN4fqfgI2jW07+p4nawzXDgom/oUkmOoxaBURJnt7ZREmaAOGd3qXooqW7njiu1AtTBXYPSjhIVpK/etXwrm+V3qviSu5qJtwbsJx/RDpX7tcxCQDQNBgpLzObG6QAwOgbRwAAx1uWgvXFPw9PEoBXCrlT/PsO8XHonxwAKqIwwYKJBQiuCDR3uvE7sDKvqTcQVD31F34NOAWzXVGUj4oKvZBRYPcAG+9nb8jjlzDtaQ2ge3QPgPbhY4iTicCQYSPhMYCCW0XitCMQZfvg03dYWJQ9egA2/wigv2WSuyMnSc9m+jCV0K17AEAzEa7sorw1AGCA149cHEmL2pvmOgFA1w8lWzgkoxJ/StJjorRJyuPIGd++dhGI8q4FAJc/XysabT0yYhAQiwqWaLRz8Abg7ovDwBH4AkBNHDGOrtA6hQPVr1+rqdALGQV2D7bG0SHUAJ9w4hCORQSMAlaFVl4BNpkIYEzucXrESP0PK469shwQkm+AzfYhpO+A2YQlXPYOPv8IgMj5JyW5Owq+WnXkzelaPtOHjLyPbh3mx2gRgVqNmM0tqgNAmf5bMwBk+oofecvkOgFwuOlB/YbZOjOHxCvxp2TROj/P/9qQEalpeex/edSczT+VBZA26GfxXw2XHw8HYeclz5/dLJjg1tRUgBOg01zPAyeNo5EIePzZW/8tqK3UCxkFZg82r7MbYZuxeRYViahpZyIqcJlLTDIRgxgybEw6T7TG64ko+QaT7UNIJsKujyRJTrjsHXwOhs7NNklzd4S/cI9I16+n3jhXZn1Em8oCCPxOSJWwwth8MJeisYKI1iSTaH0km+vksPNZou1ekWYOSVAynJIrEUSkb2nDOnu/HwBM0RNRUde1lChaHxHp33/NuVG8RRO9EERE14CqlAeM4Db5Gkf9EdL90MV+zp8r9UJGgdmDzevscPxqbB5FJSLq1JmIyHuukJCDF0OGjdeJKAZnRck3hGwfTDIR0TpblOTEOMTHkWnujvBB3NXZVslyXRRHlLZxbG1goWwc6UObENHXJIoj2Vwn+rqdiSiyYbz8IbFK3CmJ8jtHRKttiKNLzSoDwH/yiT6ZSCZxRAlnF7nW+t2SicseuEk00QUvUAYwivvg3YyjLsAeomRgp0IvZBSYPdi8zq6GVP7mAYKkv4T1xZO5DBv1AHggh02+wWT7YJOJwFySExORyd3hBgDhrru5Hp/4hJdYLYCyA7+/eihgdrrs+nz0mYu4WF9BrpO7cfUAdDz/kvwhiZTqA0Cz4EZhkw+MVv67dqp54I3oZsCf8/H7cbkazSGNp0y90WebBRP19oa99lHvIB8EwBNcNoNc8Fcg5eDcCahYBmuVeSGnwOzB5vVRR/xjbJ6FSZoACe/tATbFB5N8g8n2wSYTgbkkJyZiLneHxve6JPEJL2tcsBkANG1XZp2TPbbhLj9gXydYz3VyA5VM8omwCUxYJX8A8Ij63GtZx+F6xXE0tWiNV6sTv5TDH1fHv3/2xIlzQOwJvqbw/aVnADQH5luy0ebC/i7fTclAGFzLGuOIX2cFoLIzAHdcUOSFvIKwB5vvZw/+9G8tt7qjXS5v86uqIgVvZTLJN5hsH+aSiYiSnEAmx4isUOaLksQnvNx3wtYBAIDOkE/TGNhj/VxXSXJxWfdq446lQxIpaQDguufs2WnrPhgVofi6v2Z5QDO02UvZd0IXAEgBlvp91wgZt8M0QLtrXvfKwh14bMFE7pwqE4ORSOgBtNqVAwB5aA1wNtrH6QGgyPJ1P+8FxAqcG+webP17VG5eyjqute/8R8EA3AlAsoKUA2gcEAND8g0m20fDFyIBoOAP6fShl77oiaF/bupqJseIVLQAEFNk+OtVOTg4OJh9knDtNhB3EwCQxSUEklxfABiVNrIXTLKLmLpXJewcABRslz8kU6XYTUC5yX0uK/579OJdLQCEeDXrePTo0aNHPwdWHG2E1BcbTACQgyINcA9oYcHE3/Pfuwf8iRf7Ap2RCkB/F51hsDEMD7KA/Ey8qsgLsYLBDXYPNr8POW7qtCMAEDt60GwAaPgEwLYyeUJCDuGT5TJsFAIohI5NvsFk+2CSiaCw0EySEy57h5BjRC53x+lkQL+w9xvGucanv1yAJQ2sAugGpwPAihFVuLF8IeVGbDwBXSpnBQPIgHAcsrlONKtj/gLwfVX5QxIpcaaW5wLQNlccRxMyJ+YBtMD1C9Hmi8nYA2A8vvCDdjX8v7JgIgidK+LMJ/6/uwCj6lyKAbaltO5ttNG0N60EftcFzVDoBatgcIPdgx35/NeHzDh5YWH173RcZvhWs/+at7uKT6c0JpkIEQkZNoLeSPuoke/LE4TkG+J0I8b0Had6+pfveVY2YYkxeweXy0M2d0fHpA9Xre8zo4Cfyz1d6xkEr9eH9G/pjPeJ6keO/uLvg5N7pxMR7RrYNtA3sO3A/UQU39nfp8l3RPO2Es1/rbxvvb7CMyO5XCcU03j6b3O2krlDMioZT8n2LtM37P54iQ3Xa7+GBXTqEdzrKhER9fPx9fX28YqivE5lVhGRflmNRj0CPYbctmhiQaXubXwGcBkcH/bz6fem76hMIqONnMleLV/1GHRbsReMgsGEaA925InQHbuir92Gf+77+FbtMhd9A7yVICv300ONwXvvUV19XIUKngDuZYfKKKe7+ACZKMNuK7wUbD6DVFL2i5bXd2fD6dIZXXgjG+EaWfcepoc6WTgksVKmp8s1bW0PW94go9R73iFmH8brU+5WrOpkxUTuTc8QfrmXecs5WPy8KPeaWy03W7wwVRDtQeVpob4PCfV9SFWgckeqqHGkiipqHKnyPxTnWQDaqufhGUjb58UE2nL3jyLUz1SVYkiEmmdUve5Xr/tVQWnn1541+ChDPpoDHy2KWSpSxSKtS767pnhx9EzAR2vkIws+KhazVGQpxCItc5CSUVjlIE3ByfOHbld8qbubUi/yvIaH+Pp6aoC6jcXfcbFnMPec9tmAj9bJRwF8tFyfXSTyVGRpwCJt4iAlo9Y5SBNwsqBf2Jwvw1Ftt1IvhHJZP4vDo6vkjV9z72c/G/DROvkYPkmmULs1kaciSwMWaQsHKR21ykGagpMzME5P2vbwPK/QC/5dqmHiQhOfQC6OnEoL+GgT+ahwllkqsvRhkZY5SMkorHKQpuBkFL6/A9epyNug0Iub4Vv2HYk+0iJitWiZJPHM/P1sKfjIko/2go9Pi3w0gI/8LJnK6Bb2JFCRxcYieSrSMumpnIq0zEGKR2GdgzQFJ3u5RlQGqoApDmjZi5sD+nZq3er0k22irBBSz8zHkQR8ZMlHO8HHp0U+GsFHftZuk8rosnuCCRVZXCySpyItk542UJGWOUjRKKxzkDLg5JSsw65AHNBIoRfj3wJwdNYu0etfJp6ZX2ebgI8M+Wgf+Mighez6yFbykQEfjbNMKqMzjKV1KtJuLJKnIq2RnpaoSJs4SGZUCQdpDpzUtUJwhkIviIgyq8yQrLFNCU1z62wT8JElH+0BH1lKULTOtpF8ZMBHfpakMjrLWFqnIu3FInkq0jrpaYGKtImDZEaVcJDmwMm5eOW+Ui+4xflN8RpbjtA0s842AR9Fv312gI9PjXwUg4+QqYzON+ygIpVjkTwVqYD0VEpFWuYgWT4RCjhIQBacjJw5/HBlhV4AQNrCF0QFsuUJTTPro6cOPj5d8pEHHyFTGZ1v2ENFKsYieSpSAemplIq0zEGyfCIUcJDy4GR03wVrPc4vVchBAtiWV4NdLMp5Zv5+tjz4aCQf7QAfny75yIOP3CxxZXS+YQ8VqRiL5KlIBaSnUirSMgfJ8olQwkGKwEnOBo6/ub09sCtZIQcJYBvKG59o3Q7TQOyZtb9HJuCjcvJRKSUI+8hHBnwUz+J3wDdspSJtwiJ5KlIp6amAirTMQbJ8IpRwkCw4abBxskvob+NG9vm2lkIOEsB1488PZ0LkmfX3IaXgI0s+2gM+spSgCHy0lXwUwEdhlrgyOstYWqci7cUieSrSOumpmIq0zEGaGYVZDpIFJzkbZztn7V25cs22x7WVe3EX7iI3YLkuBCyDjwL5uMlO8NFICUrBR9vIRx585Ouzm1ZGFxrWqMhiYZE8FWmZ9LRIRdrCQYpHFXCQLDjJ2WjHr3IUe0Hh+IFrGN1gPFPEQUrAR1vIRyWUIOwlHwXw0WQWvwP5PRWDipS1yFORSkhPWSpSYxMHKTtqiYOUBSdtpDGz74Vq1LqiUN+HhPo+pCpQuSNV1DhSRRU1jlRR40gVqDytKvi387RqfVpVUMz6tOq9I1XU9ZEqahyp8jzJ/wP7mV3X6nheWAAAAABJRU5ErkJggg==',
    '1611.09235v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAb0AAACoCAAAAABRKW8EAAAegUlEQVR42u2dd2AU1drGn02AQEhCCh3kU+mCYEIJoaooKOUKSlG5inTIRQTBTyIRQhMu9dqQrggoAQRFkHCRe1FAWggthJBIElooCSkkJEuyu+/3x+zMObM7W7LZnYDfvH/AmXNmMs/M2TkzO+e3z6sjaPHIhpd2CrTe06JigohmaWfhEYxZRKR7KO97Ou1u7PLISZl3CEUP0wl8+BSVRxfdumbyWO9tH75zzT9ODSiys9XdFblO1bkpnFDkRNgX6Ip8psvB1vFd638prkRLhuw6OCajHFLl9z15xPU1EhkmoICIbp61aDRXfIpV9uvKGbCpqBxhS6Cg3wX5nC5HWxdVXS6uFNWzlPL8JpMdLU6Lse69HrFERPnBBUS0Z6NFo7ni7qo8+3Vu7T1eUTnClkBBvwvyOV0Ot26wXFypzrdEpvmnyI4Wp8VYj5wpKQAQ0AsAdlo2miuCx9awX+fW4BWVI2wJ3OmqfE6XU1sLK92uB+g+CoMdLU6LqWRV03zFgNYA3q8G/aZ14YWorkPRocI2TQGpwnAru2ZD0LmUkPY5dWV1wM14XedgmBsfd0vvMUUld7KDG1FWdmnbkttZDWr/md0qAFmpjRoClstgom9lNat0sW1lQWAhAMCnsuUxGc3ykRlf65lqwp9Lu9vS3zldhlvZNRtaSnhwK/uxmslezcQLRFipBMWFxpwcn9ZgJ0v5/Dqhxfram3m7Tffow8YO3tiVQj/Om1eEc+8EtYweZ4JUkTE5dDuMo2909+23W1aHB+9HBgaO3CU2uufaY4qSpod+B0NUxDgkTmz3y5yz11sc+WFb8ZAFsFpmoo+N6fj7J/OeFwQ+8B+9ctMs/6VWxyTIR/Gk9U/cfWMvEie22x1zJq3VCed0ZUwO3W4l4Vhk+0XTE3a2/NW8esbk0O04PA/r5i34Z495YCdL+fw6pcV6MP0jXAc0vUBEpdhIRPRepft03+8bVkGmoOUUN4iIfv1cVkcftDIQPd5fanTLfY9X5L2AiN4MJzL4d88heqHJfqKVVYusl5noexhSsnOGIDDzbaKiNl1LrI/JFLSciMZFElFuSAIZ/DvnEb0w1K5MpssUtNxaQjEiiWin9xnzfU/YBbYRUd+hxE6W8vl1RovCN4aIY9lbR1zpkSdVRH7tC99m8fy3aV/Af9fcC6auL8nqspaN8AZil1k1li+YokohAOAPwNv3qSCgaV5PoKn+nvUyE+2PpysPmCcIzOkGvJu5pbL1Mel8AaSuGgggsMd0ePu2rQE8cctJXTpfBUlV0QHAK00+4nchRFVwJ0v5/Dqjxfq+d6c2ggcPHt09dpxY04zm3w7OKraUHjV3ZtDIObK608bGADoqN7ocvCIdq24EwKuRDvBCsfUyL7qxtM1TLbFhfVwDW8d0AnUBoO5mkxceA+Cld1aXsiQA0LU9QDqlzaWT5boW62tvNgCgc8dUYbEwBV91av9pTH2xvTBF+L84+taeces/kNVVQql1Y7nDQhEgvNzQSf9AYZkXXZmt45U4IboXfrZxTIABAIwk/8vO6rKSBAAorWzrgbHUWmrZtFj33hEjAKBuY0AHE24fKpr6YW8d7qN4m7nC/HC7N7jPgh17ZXVhPkkAcFZqdEtwiuBDAG473oYTLYvCwZ1mwRRn45gidKkAcKmLV1l1KYcJgPF0N+WTL56scmixbsqYRgCyzw0CvBtfQ/r/5BfXAq7fKS68Za4ASF8CWkFArWayusD5K3KBwg1So1uCU4T6mUDOyXuASV8MoLgYQDHIapkTXQxhzCF9CWhs3nfe+Km69TGRvgR4ctpKA3A+fqn458hJXaQvsZYE7CdgQ84igPTF5pWKIa4kniwb59cZLd4xljXrhi4p1h/7aFErAPWX+RweV6P6ZyV/Hor67NrIQKHCK2FKcpreL/Vc1vHvPwnh6zpH+H1Q+tvej6qdExtdHCtjbCl6fF21pB+f2HvGa05KevqLY/dlJT+95dP8C1WKp8iXwyXRp6ZmXkjo4ouEKclp+tyPIm7ErZ47oJflMZ2Zkpym74zns5bn7123qU38lJT09J7j/p19qU2QTZVMV8KU5DR9FQsJrTC705Gc737Z0RQJU5LTrgdPSU7TF0/NvHTvqX8cvJH8nI/5ZNVWPr+dndBiPReT9FTJybR64cJ3w/zMppUA/eXgetBX8ZIqAKCguvHPwLo6WR0AU1rtAL7RDTNEMkUlacG1Mx+E+Dv825xoB80W+kOCnFQp06V4GF8PTw+x99ZEOFmua9Hm9zx5GF+/o5ERj2qkIZ3+P37K/xLX3oFYH0PdWVrvaaGNnFrvaQGNCNQCf2EiMCZG6x3tqUW772nx8Ib1/N5bpQgo0cPXqxCPL7RqvRv7RpC9ZYtY1aAfct/7trwqLx6r0VN443T+ePXn6wAA0g9V7ufPl0y/XwzuURfAkcwOIZUBVPb2+OnLOd0TALdv05ELDbsGQqbw9NEafYIsjsN9Ty2yKPT78QGV1GhlpKKvGjmGIh2Qh72uEm2dWU4y4sHYMdeOPXuWiEyzduRceGsjEeknzM9M6VfAlXIm788+UG8FEZlvml5J5Nm4u33uk68QEbF9Z4/ffjKm1lbidJkip1053f132XF4hpwkosuTiIjqhBMR9TM5hCLtk4f3w4ho5JFyqpr5tJHo+/Ymom83EVFJi1wy9ZtBtAWHiJWinisiisQNomFT5i5evPiNGA93Ht3ee7eL0HvSvt+9Q0RRPomcrg0diOhY/Xv8cXis9/7YxHrvo9xy/vVfJhOZWpWWT5Wh+iQiuoV9RCNWERH1PUm7qxUQ3V2jJ1ZaEHyb6EPEE71HRJQ/wkCeD3Pvifs2Bo4norOYw+nqNp2ITNW38cfhrrB6agkMZ+W+RRkn8+/HlwJF+35IBQDD9TPXUXIt4TZdPlGgsAxQxvEC2rtJmPTf1xu40KJS+cb2y/cDANT2Ogo0nLYTyLrWGrN7+gHBo33AStOzagMnHn8a+AcARM/2Vu/pQdy3rnEpAF/cYroKjjYAoKuxjz8Ojz1ztmzCyp1TzCykRByK4KIIGFouI7/PurQ+fw/6QSCkfusOxPUup8SaMAIw4gowMejVgb+/u6lqfnzD72b+M6oIYCV4AVvOb60CNAWwp/1jKj+6bzm/tYoufi2AeEQwXVcNfgAQcIk/Do89tQghjJwSC8mIQzO4KAGGlsvjexMdqHTdSMPbtWvXPKRdu3bVQtu1G12u8Tz0dSJKQE8iuhsKLDJSEiIOEX3esZQrEW0aU2+L+aZS3NtIpN7IKdu3sUOEgen6A98SEXVoKzsOT9335L1HmEtEdGkjEYVNFH9OQXUmENGYHgrLYaOIMrBL2Hj1Z0T3O5Zb1dG6N8g4328AUdE7X+1siKEPLqIrEeVgFbESEVFu2BDhJvuvyaRu73H7ntEll5iuI1hNRBTahj8Oj933LKMxADTrMH9SDE8cWgKG0nKXFOBPn1Ch9t8vAoe6lXt46PTrogXR/QubA6M7jx+QNC72q0ZoAyDI/79gJQAI/HDrSgDA+haqf3MW97311L5AMF0Bwk8nCmvwx+G5b+sWURkAvvpoSy9dHCzYSaXlsa9Mb7V2c0PgSgHoTGkitj2TCAQ0KpfIVv8CLqANsuK+BfxX+ux7r7Zwt8mFr1iif3boCbTCLxMBJJ2ro+qbfm7fB3b96IPE1pKu+igAgIII7jjU6z0AKJo6szdwH8W7BztaNW6Pb96+qgDWXcUV36XAj4Z4oPHH5dF4Jvl14NgTg1BczRsAXs5Ft8sADEVtIZWuRL3QE8hHLQA4BlV7j9v3yZ82eCMztrWkK/jpbADG/F7ccXi694xGMxMNPcDDkTy4SIDVcuWYt33P1n8MmAN8HDYQGVnflFvjkj1DdSULF1ZBozoHnwWw/y18/FxeIP7wiYRUeqz9lwB+140EgGQEq9NvBiMANJL2fXH4oLlUeG4E04XJC01eOBDYjzsOjz5zfj+qp19A37H/Jvq1n1/rUdlEi5ou2TD/jycnpJ8a4tdi8cnBfk0ml4yqX3NkmuUyZfqH1KmO8AtERJ3yiFZ+Wv5nqT0jEg+/tJ2I6GqfzxKPRa0mopUDz+zrGkdc6dz4HRc3NdpARERTcF2FR5ab4wcG1nxtQgHbt8mMVSdwukzvjr1yqMNh2XG4K5yci3EAR0qRNmB7M1DW7KMJQNag34CBC5uXf4boypGaXaoLn7WEiyHhwQBw7WhgpwDwJePxy/XaCy+Ib5zqr1Nz6OT3LYWkMON4lRf95MfxcM6krd+9AwBuNi70wnfpM1Da/oxOm997VOb3+qX/QkBh9BQv4SXL0Qiddo4fnVns/A0ZfrjXvyeAVWO8cBIdtLl1jYzQQiMjtN5TI3poHaONnNq1p4XWe1qo03vfDH75pdcPAsA7vV8ebISCRyGA3JcajYAL1oSrdgO5b5dRo+GHAuCO2cIg5wAAHNmWUaDX6/VGvvHi1zvyVT596d9+XyCTg9MrNptPASvBk79jsODAmnsXCTRQz7UmRY9CIiIKH+qKT6KThKBcVS68G9bz3cYzeDES9Cc1up+3cxgSlijJYfgfK3kwrOcYfCe+GzcQALzqj+RelFQLAfD3qkPF5YYWmyXktJGvoBRF2Y8BcaPK/AjaIr/1Ow0AGKpHCrOMqVOCqwIJzVuyxvlHz3g1HDfqhHovd2hQ248Qu/tMVybn25Mn0GjRq8n+2CiVVL32KKfqywKQ8KWVwyQXr1lce6M3upEQtLj2ZlixCAz6Exs9wNs5CoYlSnIk/I8reTAUnlqCBsddAYDNb4KhgJB4QACGc4nmR3qxWb92nb6wkMQVMncdLQbkqCDgJkIQUIL+PMDbOQqGJYpyGP7HSmo/c46l9QDuUSDnkwjAzP/B9OXfEi9OugnAhk+iaE0IK29CFwnBB/+aM3ul7CsgB/2ZGz3B29kPDkYU5TD8j5VUHjnJ1KKhgWjlEZm5I/MoXNy6mKiwwVAl70e5NSGPCpaJEJSryn/mCpkGjbSguMzQn9Toft7OQfAwolkOw/84EFDdkRO6MdfjgKMRsDJ31PkC+TNGVQWqt4CS96PcmpD3JvwmPj5+6qz4+N+fToiPX1OmD1jAyUbQjV9/2OLLR0svWeOKg5kw7fXzV+3a08G7KzDsxHomhwRTLQNxJdW/rb9dZQ1Oh+qggAIC8SVNpbJCM7MmPG6CBTroKiFYCcAT2C+vFKE/sdH9vJ2DkMOI61sAYPgfBwJCzd/vAaj56rabG6MBBRQQANizgry5MLOZpTUhQwVdJwSpc7dFQCDuymrN0B/X6HbezkFI0B+Tw/A/HgRUufcwZssXD4JtoIBhPskvATDKmwVvwmYAEKFLfQZW1oSuE4J0vQGA62gnqzVDf6zR/bydo2BYoiiH4X8cCKh67z3beOF/YIkCmj0Kg2Z/M64aMk70kDVzPolPTls5sBLOx5+QoYKuE4Jew94CENt8KMfgSdAfa/QAb+cgGJYoyWH4Hyup/MxJRAubCS/JeBSw+Yenhvi1WEzGz/pv/PR/O/sMzmPNRNuarppUKqxgWDRw1dzep4h4VLBMhKBcVc6knxI/ibgkY/Ak6E9qdD9v5zAYlijKYfgfK3kwbMyk5aV0hG0U0HQ5oE5apeDqOlvehIrWhGUgBC1UmU6mtnzG4ud4EvQnNbqdt3McDEuU5Ej4H1f6K8yDloEQ1GZnH7r5PY0QfJR7r0sYUG2Udso1rkULjYzQeg8aEaiNUdrIqY2cWu9pgb+sRyDcbwDoXKx5LRi8BR8z3hNLkk+fasH2yEqSDaAqcsroEchngXbRANAlF5kjulTiLPgUyDvmEahWsD1KJYYlqiKnrB6BXBZoVw0AXVFV1BOpRMyCjxnvSSXmEahWsD1KJWYDqIocq/ve7Y6s3CbfVlJiQECM6HhHVcaoZSMBwLRxJoDXH+zA2p4AOubvAysFnC0A/HFTvYGT7VEsGZc+5wU8F78f6six7xFoBKX+cM4I0Nlt/7mXIYJ/cC/e5ziOPF4fACQLPiXyjnkEQmVnQFbisERV5Nj3CAzJGfhDm/++nCumcBaTErvPANCpKNrxpvA1ULTgUyTvJI9AtZ0BWYnHEtWQ4+DSGR04He8dn9frXh/UmXURQ15d/PrfAbyTCBRmdweSWqxC6BpPn6WlU7mZCdOyiDdOCGyN/z3ck0rA5t92f9Fe1Sd2tkehFByaDuC86SpUkmPXI/AsthLRyiePVJmTaNSnspTS5TEALPNTy6GNRP9FqnlpRpdcYsZ7nAUf79OnWrA95oYNKbWwAfS4HPvf1uNxdMmSJZfDIqLmtq45o76szU0GgM6Mm8O4pa2n9gXaJO8kj0Co7QxoLsmxRI/LsT9y+uHN9gBQFD3p2KFVxV8CAvjnTgNAhxF77V3gKmICl/qIFnwK5J3MIxDqOgNy+xaxRHXk2O+9Tt6n2wOIvxTQv0+f3iPFLNDN3GgA6DhGjACw6eeYJpAs+OZak3cyj0Co6wzIShKWqJIc5QG1Znvh/+jQIqK74ze+ZCJK7E1EjefR/v1ERNE7iNL7qPUOgdbgPBEltfx41sz3X9hM65oaifbVyyOpZGx/iYgW6A6qdstje2SlYYEmetAklm9U9V2LzCOQjMt6fr50as7GAVHfrx1+iczgX/kMAF1QlTaypX/ncZwFnwJ5x3sEqhNsj1KJYYmqyHE4k2ZKq1WDT+FsBv/KYQDonvk9a/JOyacPKjkDSiWGJaohx9V50HIYAGqzsxU+v6fhfY9y72l4HzSCRBs5NTJC6z1oRKA2cmojp3btaeF671HmHUKR9vHHI0gEYvuu8Lwbo6J2uvJLxsRlN3VUq4fTXyWOv58WM87RSrIswGKeXkDOBVZE8CmCRWYRqrKJVu/O4voaiQwTUODaq7eF2OxkJpGzRESFVT53+J4zRjIEZHl6BVsgNXz4bAeXIlhiFonUZBOtr70FkV6A98JYFz8NIQhwbkXBVLC6E1cOMwSUPAKFUMeHz2bMnlML7R+89VQrACieKd1qFp/9pNrzAyNfqV8B972UFAAI6OXpPe90es2ay6KnTRtbNRqo/ZIsOxTjAitk3JT4REBkFiEnBdXvveYrEgHg/WoAM/u7fuYq6E7SWejFPMLAzZ935wAiM6g4KgtNvFWgmFSYYwspmXMRhFOGgEKo48Nn+2sNSxEsMYsAoCKbaN17M2+36R592NjBmzP7S5oe+h0MURHjcMycRxgP3o8MDBy5C2ZmUOlvi02cVaCUVJixhYZPLnAugkphKwuwSj58NntPShHMmEVLUlD1pxb6I1wHNL1AvNlfqfcCInozXMojTB+0MhA93p8GDieiN97nJsF/FotSE7MKlJIKS3RanTDRRdDenDHLAsxlelXHh8/BBHuHCAMRzbnBUW+yHMJqM2URx7K3jrjSI483+6sUAgD+APzxdOUB85C1bIQ3ELvs3M6+AHr8qPC5YE3MKvBEQ6CxIYHfa7joImgvzIaAFh88VXz47MfMKr94A4efkD+hDFudtOh1Q0V837tTG8GDB4/uHjtONPvbbPLiE802BoDTxsYAOmI9jl4B7oRZ/ZnYrvFck2gV2OUMl1QYigmIFWN9pEKlSAPWrbjO23pqX3WgaMdSWJGCQ7tNrIDem/0lAHTumCo3+xM+7BDzCFdCKQDGDFp/D/8b36SzTCoMyVTQmRle5SzA6vjw2QtzimAZswgV2UTr3jti9AaAuo1lZn8+BOA2OKfAJAA4KzKDll1o+q2aUpOUVBicqaDjsM4CTDqo5MNnJ8QUwRyzSDqoyCZa304yphGA7HOD8OS0lQbgfPxSAPUzgZyT98Q8wgicvyIXKNzQKOqrYiBnnbS5HkYApmgDWBOzCqwck5hz9hoA0VSQcxG0E1wWYMEjMLn+agCT40zwvA+fzbg4PHDurKnDW4oHrhd0NeLzF6v9zNlkw7BNx7e/+B8iZvZHtP+ZbzZHReKV1eY8wmRaEb5i8f/eFZlB88bn33oMYZGRw57AyxJOyFsFckmFtzVdNanUwkXQ1jOnaAgoeQRearBWLR8+m8GnCBaZxUsN1qrJJlrPpCU9VXIyrV64v4XZX0lacO3MByH+Oo4VrB0gMYN2cEIoJxWWmQo6mN+zmQVYBR8+uCeH8F9gHpRPKqzNzj5qs7N8UmEtHjkGgUsqrF17GteijZxaaL0HjQjUxiht5NRGTmguc45c5qASUwZGkskpLgDphyr380fFM2VyuE2ymsPD5TJH9o3nnGfKzFs5w5QxkkxOcXGJXyuYKZPDbeplwC2ry5xC7Nko/4H5z1SWreo47r3be++aZ9Qllznzm8Z+M4i24FAF9R5Tw0zviIizmlN9bt2ByxzKA4e5uJVEkskpLuCXA9OBF9d0qHCmTAa3cVZzqj+1WLjMmRPLMpRMhMJEYExmPKfIlMmyz5o3L+C2csiUKVNc4BO/VixTJofbVMyA68BlTkwsK6FkEhQmAmMy4zlFpozPPituPpNt5ZgpU6S45IlfK5Ypk8NtambAtetTxhLLiiiZBIVJwBhnXWaDKeOyz0qbS1s5x5RxJJmZ4rJK/FqBTJkF3KZeBlwH3/ekxLJmlEyCwuywZLBkyrjss64yZbCkuKwTv6LimDILuE29DLgOHB6b0fzbwUJiWWEmWYTC4m2xZFBiyiRuzFWmDBYUFwAu8etYVCxTZgG3dfp1UZ2CN2Y0r/De4xLLVgbAoDA5SyZlnLXBlEkT4zKmTNhK5wrFBVgmfkUFMmWWcJtqGXDtj5xFUz/srcN9FG+DBIVFtt33GtDJ+zQAxItwGBSZMvMqgOXmNraCExSXDzJjzXAin/gVFcOU+SAzlofbCMCZLVApA66N3jMKvythiWXNKJkEhTFgzAyHwTZTxnFjElMmbuUcUyZmm5UoLoEp+/hEHsTEr6hIpkyC25LrrwaWTCColAHXvsucmFh2rYiSSVAYY8kk4zkbTBnPjTGmTNjKKaZMIskYxSUwZXzi1wplyiS47VKDtWpmwHU0F2ORd5aHwiRgzDYcZo8ps4OUOT9DxBK/VnDI4DbVMuCWcSbNWSgMGlOGh29+r5xQmMaUVewstpNQGDSmTCMjtNDICK33tNB6DxoRqN33tPDItaf506FiicBv9hRS4PhnAbxz08tvi7fNLfd9fPWrgbDhT3c39g23mr8xbzmpxFhAcxXn7aaa+ZwVqaiUilamhnnReeQ95/3m3kVERGToudYuFnUN22z6032KVe4j3RgRyEoSCyhVMW83tcznFEhFhVS0cjXMi85DuVA+xw7hLexbDpg2y97rEUtElB9cQHR3VZ77VDEiUCoxFlCqYole5Xye50KBVFRIRStTY86f60EyYlhVIR3i8U5lTLfA+dMFj3UjSsy85aQSYwGlKubtppb5nAKpaJ2KVq5m2UhPP7UEDY67AgCb3wQknzlIrnIiJAgAlPQnKfnTGa6fuQ4UFhYWFhaW2neicy0UWEDJ20198znGBlqnopWp4b3oPPXMOZbWA7hHgZzPnOQqBxESBPB7VMbBbgdh7U+XMTl0Ox74j165aZb/UthzonPxdakSCyh6u6lvPseRilapaHk1ci86D83Omlo0NBCtPCLzmRNd5RgkSBhtIrpQ5ayCP50paDllvk1U1KZrCVk50ZX9bsyIwC6vWLGAYqPZ201N8zlrUpEZzAklXo2FF51nMrjRUuwmGm4iSsF+IqJXe9Ed7yVEdDyV6NJGIgqbSERYT0TUuS//GLZ1RJWauUTUYDklriEaVfM6y17rvt67iK5ElGN+sGWNuWFDSi1S0arWezO65CqlouXUyPPneoznfLvKGpwO1QGiz9xxk9lVrgnQrMP8STECJKgDgCbHxFvfHSB48Pr/5ppNkZ8aiQ3rNzdg2WvdN2BILCAUssDKU9FCPTYwUCkVLVMjz5/rOSKw5qvbbm6MBsB85kRXORkkKKxRWcmfDoBOlzghuhd+7m/Tic7lUGABmbdbhZjPCWxgK+tUtG9KauRedB58UzbG+MWDYAARulQAuNTFS3SVk0GCJgBI7qaT/Olg9qcTonBwp1kwxUEJDSxnWLOAV6IWQvB2qwjzOTMbyERIJaZmxLYvvvhiCGK+8PHwe85nGy8cAoDzmRNd5ThIEDhAwOE/F8HKnw6kLwGNzfvOGz9Vh5UTXdnDYJSXZCygwQjw3m4qms9ZkIpMBCvJ1eihh8d/hbKwmfDcxHzmRFc5CRJMp7YJ0Vtm9b2g4E93aohfi8U/48UZHwyqtIQsnOjK/NQiEYFcSWIBpSrm7aaW+ZwCqaiQipZXY/ai8zQRmJfSUYL6RJ850VWOgwQpPaSGDX862EUDyz9DZM0Cct5uFWc+p5CK1oNqtPk9bX5PC633tNB6T+s9LbTe0wIq/AJslnYWHsGY5cQvwLTQRk4ttN7TwjL+DwbUKW2OzyzOAAAAAElFTkSuQmCC',
    '1611.09238v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAe4AAABmCAAAAAAo6C2LAAATlklEQVR42u2deUBUVfvHv8Pgwr6F5UaSe24huKDmkiVmWpq+7q+VmviqUVqWuC+o5U9Ry0p/LuWWu6ipSKZlaKgQIiGiKIgiLijKOgPMzPP+MXfuvbNwZ7vjm3KffzieOef7nOc8dzn3zmeOMoJk1cecpCmQ0i3Zs2pENE+ahepg84hI5uh7t0xaHEgXc8mqT7pLrxUAKH32JvPxo3/6CJ0NKzIX5ag9A2Wlyq4fuDnE4y/LPIPL8sc2+DA26ZOsWZNtlVlXvz8efbyl6gY2BmJWF5mLctTeDUhWd0ygXn3Z+BeTO0TZJJkWfUdG/j3GAQAe7hrhY9TCZKWFnxos1QxsDfYT0eN5rdL06+9cJGEz2cDAgWpK80QiUi1u1YmorPZKK5T0rc9Not1zBZtUFYi9urQGB4lUsXXmavi104bRv0faKvkltuuKq7HO+HOm0mhitBUmu5gyZ+MDwA8uALzm1+2U1ohfn1zQVvjIMdsAwJwf0gMAyCMT7wAufjYrlT1oCBwbJ9imqkDs1YUfagLyvos/bD6SVxsXji12SHrqiqNrDzP+nKk0mhhthckuVt67xz3/id6/Y8xpxZh3d+mrKQHaBfsX9imd6g7QuY4WxWgYiFi6rXBE787tYb8kAPhO8KqyMsb0TJnsYmW6nUcfvACUxe3LBADlho3KkhICV0MX95wsugGAMvelqnkNBGytZiBT6lhX+7ciN+Um6H76RU6RVdIKQ3kjsbA0qVJPKC4MuNTC2bL1yeiDFwAg71CCQltz5+fDBabbWqN7A6+ADb+yRKMsUdorCUCVm5KLilvJ9ygzoQjI/zNXV8lNjEFStJ+yAVbcSr5H188XW7kyb4iTSH3fp+XscA1w6CodiIoqA1ujHn+7u2v/w0DBoH1tf3vzEdtAyH5HS53jTdq/6TOCfoIqMjQcrKJOiRHG2Q87/rEk6jWjU+ZYmIUz2BAnAUXEpsCHI2IBlE+b5O099lCVp6KFurcWDv0EbPhpUcV7oo7YKQkANz4J2ou0KcFHF17MbXFm3x7F0KVMJTsxhknRfsoGmDYl+PD8lKxW5y1aqu1ArLbwM6bRx86lVOr+IxFVYisREVtzbAgR/foN0aD3iGjENLaB0FLN30lvdVN/JVGlfCkRjexEnCKjpBOmIgytiJml6/RecHBwc7/g4GCXoODg8QLrEr1AKHwSET3ySyaa3kpF1GiAYXOLdWkH3l86s2ezYxq9UdY3XndaLrkeP+uKGp+VRCqP7gVErzc5TrS2dhlTqZsYo6RofFbyA1R5dHlM9Powi5ZqrJXDA5NCXOHaLOk9rlZX43Fo0bstu72I1JjdAHosW2HJoeuVX2b4VOTsBwAeAHSKjHHCHmhTY6DuLoAfAaxXfoSyXucsPGHK4YHMdccBePeYEZcf/ZUc2OVr2Moa3WF9kTdtRlAdCIdv/VAByFwByF1f9gGapvQGmiqLXLSVhinQ68IFCLlrOy8g8KoFz908u4sgNKPF93zzFbxaXU1o5KK5PmMXIgkJOcD99haF0uFazstMce8Q3Vh1H+oUGeMLNzZ8eF8ExL9q6QTeRRDO4wUAeGG75oK6MQDTSycrdOt95z93LcyHb91Qd3Wrz5QCADgFyAAnKAwaGScF4AXoBDQE4KS0Lt2/NeqD72fu7CM7pqspyWvG1ihmR5yNX6f41h0jQ8BvIGRjdsQx6a78c4jePQWcIqPEF67Ba5pTDEqpTMOeV9IAzwAL5vC3Rn0AqABATXBGpclW1ur6ev4BQC98eyWBc2+DfxbIjFuU5DUzTop2kccEaLqfuXSnxBxxKft0bhhQCsXhf8mgwb34ZmzNyHED+vULG4vO8gshAJJCmAaC0YS9tWqii/agZ5detQjAPQAxnlpFaJXe0AkbaGy8iRzXFcABVRLQeI75KUyJOeKCUFnmKwCudHVqXysdAC62A0hmj67PVWVtLnzdcWuXpOaUi+B1Hhrci29glBQAvACteYkKKKAGgHMjV/VFocIfyL2vKLkLeeNbyH6Rq7nzXX8Z/JshIPL70S4o2BjCNBC+L20LG7nFA0D2L6sBUioA1MsDChK9AGIUGVesMBTgX5cWAnPaD8KN/B/NzB0/ELz02dpBzvg76Ty8Fy+L8EHJ5mhk9FowwQZd3XjqZ18OWjRTN0pSKgGbJZVQA9DMVgGkrAA0SgUAhQKAAsRUMhNjnBRSVvAC1HU28VAsn2/4RnjalsdXL8buWJPy3duAh9vXFdfiI7++NdYb9aJrnQ53YmtC8lPzz+1Y4oeeigVFp48ucGEaGOgt0HdQe3RGRKXq6uYr8+VInpqRlfs6Gm10ST8QGJvS8VYmo8go6YRPfJp3Kbkrf60ya05t7GrUSXAC9QMBXstfWRi7cVtbINR9euWp2JkuKNj6mv491xJdZH6+W5mS3N4TTY5o6iT1ZUZ5eerlaxda+9ommfbZ9qKrqUd3TP+5/ajkqRlZyppTr2ZnvzEhLj+jzc7VhZdqlk/NyFJ2YSbGyzApKVMzspRd2ACTpl7Nzu4d/suDK219zD+I6ZsiLY9IoSYiepxeyaspUlek52kfq9SZj4nXQOidORGVHf/hZCG/ovzyPbqdVajhKTJKOmEDu9+diAZmkJWmzixgS4Vi6Cr27yutepT2DLVK006McVIMAqzKnkq84afsWagMSZE9BbqOGmp1whuOhQEJobKnQddRQ61O6e7aHnAZ91ToOmqo/1SUTGLVILFqkj2j6ZaoZkicuWSQOHPJpHu3ZP/jdD+znPizb2Y487ToOzLUmxuA6BPk1G8SwOfE2T4XErz6MW9nCy70FvSn+eOyb48XDHs5xkz6unzWq7eX3dLZ8TX6e4gbg27mOEmrxR+NSO/9g32c+Zf4kYiIbvnfN+LEGdhm0mc5F7r/QUQP9y566R3Bd+YFnxx/cKLud3q9HGWmfJVP+PDW2Z4X7VRW/mdx3tX+xSLGwM4cJ2mTeKdhwp8LsGpr3bL5EFU4ERFFuuVoczyQTffmDkR0tl4R0b3Yh12F0x3Zq4xoEm7zeznKTPma20ZNtCNEY5ewpv8sop2IFzEGduY4SZvEB5tJt+WcuTNgmhPf0BtAx8I4oE5fXzOXG8+LxYAH7vB7OcpM+FKv6OUE9Eo6bpfw0RMzgDfWdxAxBnbmOEmHTJCTeTwbgpw4ihPqA5B5WTSuGfl1gPON2ljXyzYz4et6qSeAOk4Jdgkv6O0O+I6v5YAYOElD8QqVplytUKvKUa5SVbJMOY/BV6WmkR7/bwNnDnOcOG6q3AHA84qlh9fOv3fXtLKXzYeyga/noAagRo49soVJDX6a+1VkmSNi4CQNxfe0kw/Nm+NbZws2eDY/wTLlLIOv+fbttMsRdzha31pWDS8gz7DqnhP7UybdmrEIcgDwKLIsoO2nDq8JsbqXbWbkyzcoG8Dfmpv2qObRxVEjsabXGQfEwEkaio96pfXkhstdT36IyclRdSfKZ6NNtybHg3oGe27bdiQJiN6cWBv9mjfEr0X98Py8y9af3eUw+tWTl8boZyKkBbdUFr49G/X/6cuGq6ztZZsZ+/ru9zxoYt097HpPCHk3YNT5TQ6IgZM0Em8Vsh0oTciG6rm6mesGAfDuMQNaBj8KhbPG1QbcWgAehxZd0nTra3267yIIcALjUA0AHbjr4F7digglAFBi8cOs9xe711rfyzYz9NX512VLZw8oaW6PZgDaAvDx+M0BMXCSxuJj9pUqavXYhhOvs0z5OQ0YBj+poilYWr/1c7PqWZ/u3xr1AZrjljbbSgAYA93SofJPplAPxQBQHGjJwfvlCQCtcNSqXjaeKCZ9tVoVucQJbe0Rdq2jnblHDoiBkzQWH14ec2jAmC104jV9ppxh8OVMM8Xsu0fCN023Ot0pMd+7AEGeaQCA8x0AIOytVcyPF1hO3LfNAwDqwj4WRJMT+SWAQvhb1cs2M+krZSeAs4FD7FJ+9ToAVVk7B8TASRqL+7+15WznwbfjXOUIlWUCekx5+1oZzCU4Jta339L9sRa/VduEw0REZ1/6moiItrlfJ6Li4eVERPSo48AiIqKsCLb9xqZqori6WhqzU3+h1yzqkCtEtFT2u0EvR5hJX6O8NVTeZJd9yik+j4hOud8QNwbtzHGSxuL7ZEuIRtVNJaLpr1USpbqnE5UxP8T8sl0ZUbbrm7S1r4YoLczCt2pXxzRCyKTJo3sN0b1q3NJ+fcKGUbk6bHhWk+WnfpmzpIJ7yfTRhJz4DqeJ6M7EQd7PDf5PcdUOUifuv7wtYLNeL0eZKV9HPkg73XevvcprB6XEdTsmZgzszHGSxuJKv+tEcUFERKplg9YtCvuL6Nf+7q3HPSAi9dcDtq7+vEutf60ZGLljw3tXbAePS1NuBLXgrvqKM7kvBnvyG9w4V/MNd8u+71afu143xNtsL1HMlK+cM891tX+LoVsJ3p09HRQDJ2kkfr8OQA/8td//ZPkZfH2iue75fJazr8Zdfc37BZmEJkom4Q1SuiWT0i2ZlG7JIIHHkqHagsfz50sTDek3YpJJ4LFkeJrAY82ZSw26eQu4Ux3s44H7f70JiIUAW+ZLRwqfyevgVwNADTlEoYR5oLWORa4+4PGDiXsT5/vvFngp/wjyBnVd94iGAAsZ54slhYlZSDili0IJ80Br1sM/Gzw2u8G1H7S7UDd41x8w2KBaa1sTzyNg2bsZHliw0B8h5f9+uVXVR1ePFoWt368PYHFCilOD8HHnHbixAeuLhrSbiV2HU7ohc6pvbSC5eUu7Lhtuk47pF3ge7JTkJpMrWWENrN++XseZV770Dp8zn0xElOb0BdMqgT27X51BRBq3PaT2nkhEF7FQ4OzW7WyqcosgoruIc+DZze6ietilmOjheiXRx0REhR+o7NVmcXqmwHmwU5KdTF7pyXDmVoPHssaVAFxx14LDUBwEGNaRwsBkAJi9QO44D6g24LEsaQOAJIQKaJavWrhgLUEcBNiM6XyxpDDQFMCRkIZie+J5QPUCjzXRoSMEnvt+PRhAQxM3ioMAm3nG1PliSWFnAFB+c1R0V/oeUI3A47k1jwpcKj0TAyCbuOm0OAiwsLG+WFIYALCupfjfE+h7qEbg8e6/4rzNPecH4rg4CLAF7xQCcZwjhQEAm1qI70nfQ/UBj08cOuCGtKoP3tDPAXjjoTgIsPCJwvpiSWEASE99Xnxfeh5QbcDjxIObayFvV9UpyK0PIBfBIiHAgunmfOlIYQA4CwekW88Dqgt4nN5yzry5017fXvVz9xcPiWhm81KREGBB43yxpDARTUeGaJQwV+B7qC7gsYb5LySSBXZviDiYtiT0imgIsODuDawvlhQmoqnIFY0S5oHWPA8SeMyZJjGz5StyQCwEWMg4XxwpjNt/DXDIe1vOAyTwWDJIrJpkUrolk9ItmZRuyaR0SyZx5pJB2uBauphLJqUbEmcOiTM3ucE1sH6wr2Xst6M3uDbtSwyvHE4voi7LqTMFPUweiJtz8/tBoizVROPMiYjOyDIt4swdv8G1KV+ieGVxehF1WU6dLfAweW0CYAJKvWMlqC/qBtdEVNYbgunuET48Kteol2PSbcKXKF4/uk9EkbXSRNRl98xmC7zh62bSRLqPbBUt3VZz5kREUduF0z3LZC8Hc+Z2wtuGO3ixOL14uiynzgHr3PAF0j3eynSLvMH1mUb1YCVQ7XirGt627dFSh9OLqMty6oLAOqVfI6C8UlOugqpCXa7csFFZUkIAKHNfqloYMTcLHjfEySBYtcF12f7lp2CG/S6i58NlT2SDa2NfoniVJQFAEkLF0y1MCv8pw+3xHFe2wBs+a38kdc97P6rngejz3b9t/ffb5TPrXaUD1zDHDQVjOw8++tkuzwmDu2f3HzrFtnRbz5mv+FRmIfv9BDa4NuFLPK+a6NAR50XTZTl1HrDODp81xWoZugQlDuvboH9rBA0fGYR3/2/4aAAY7z0DH5+L6iOEmIvPmZ8OrGcp+/0ENrg24Us8r3NrHpWLp8ty6jxgnUPyddZFBrwcMhNe732vhkrJXXlTY94C0OOAIGIuOmdetn+Uxez3k9jg2tiXaF53/xXnLaIuy6nzgXUWyecOCgBNzhKmZB/FId5DeBISli9ffr29IGJu9mJuijPfEfcyGM58iD4aHYpdtz4CbmK+94qqfhpHXV5dpmW/uV6OMlO+xPJ64tCBWkgTT5fl1DlgnRu+wcujGkCLN74e8PtqpqIkr5k7RoYAQNnsiLPx6xTf2pTulJgjpjjziS6ogjP/4AMA236e38QC9tvxG1yb8iWS18SDm+XI27VIPF2WU2cLPEyeXS8AQMarMuCjt/e2kgGQQYN78c06yy+EAEi64jmgX7+wsda9VbNrg2taj7/JEs7c4Rtcm/QlilcWpxdPl+XUOWCdG75uJkdqiOI9s4lIFVivhIiIGkfR8eNEs4PKiB5OFETMxd/gmihrbEuPLuGWcOaO3+DalC8xvHI4vYi6LKfOFniYvNbaJc/eOe+tS0REtPwzbd2epusiKonU0b2/WfFpwVYhxPx/zJk7foNrk77E9SqeLsupswXe8HVX4mw/ZjG46h3mt2OFeU2dtdi5vxeK3QQQc4kzfzotPeNdZeRKCW+oJrZxDtZFSLsmVhe7He3VtbeUbsmki7lkTzbdEtUMiTOXDM8mZy6ZdO+WTEq3ZI63/wJfRvK0Nt3NggAAAABJRU5ErkJggg==',
    '1702.02925v1.1.png': '...',
    '1702.02925v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAA7cAAABtCAAAAABfQskLAAAdZElEQVR42u2deXQUVfbHv52EgGSDhIBLCAQQEAGFsIggQQRHtjgou1FRNCziMqiM4njAZRRX5OcyEsVB2Z0MoCDrQZawySYIAWYQIoIECGAWAmTr9/ujl3rVtb263R2n4d1zOKnq8O177/fWS3dXV3/awSBDhowQizBpgQwZct3KkCEj+MEYmyxdkCEjZGIyYyzMvXavjJh8daonS9OuJssBwMFc/66M8K+TkFVfpW1fzZbL17cyZIReRAQ7wbiRnQAAxScjI5wV4U1QdTgyooI1u/qsdv4hfyOdLFw6F6LOmfQd7HV7/nO41u2p2Rs2R2ekNkHF5ytze/Z57g+w4cxTM+Ko2pLpv+a3ebqBH9nXf/I1eX7/PlAR9mQiSZvzYt/EKAAYEUb2y/nZiVoXn0mkqWnO8bOy75xH/WBqx9q/br27J0lNcY5r9ves4ksV41v4cZCZ9s0YC+Z5qSzUK3dv7kA/10ZJZ2eQspl0UvBsZkfkk9XDjrCiQXFbaWrGGCtt2o+Ym5X0frmczU6nif/hnnMPm2rOL+fwVxnb1fwkSW3uHARmZeycpboBgBovO2lqU+dgdZgUjC9g7O3INYykNu0b3L9gxZARWOne3If73VsjWPWv28u/VE6jr9uxeYyxkviUSvK6fZ28bp2DHmCMNUukpX5qxoqNOTkbe+TZVHN+LYkrY4xlDiKpzZ2DwKxet7luOXXvj1/8LI/at6lzsDpMptX6mrEC3M5IatO+4XkfKHiRX+9BLNCeEPsDniTXbOTPq5Vlt/8ORKflHaLewbaUelTp99kTAMz/hqYOy7znjm7djo5qTPdrXtdIAGlLL1LUFOdUs7LtHKeuP+6NxxpT1QTnuGavZQCicJ58kJn3HeR1+/XguxIWl4X+OaWGZeUAonGGqL/87+Hk3J/G3QqgQxeaejwAHN3yAL33qjVxAHBd2fpQc86/IDjHNTusZDCwGf2pVln0HeTzUhvHhw/+dOW9Ib9ucyojAfwU0Zqo/7+nyM8x2LqU41nRRY82p8mbAnA+NduPpzgFv0cDQBz+2ze0nAOw9/vKiEfjqs05vtkaQPl7Hf5Gtcqi7+A+3uYlh2M45of+421YJIAde5+indPFj/UaklMXn4v612svju65ll79/DZ1/ei9BFEAEI6iEHMO2HvgmecbdzpWbc6pmv3hr90arY8jqq36Du66XTAM6Ja0tBRXQpSN6vV3mrJi9kh62mJsGxKGlN6Zl6n3UP7iWH/6voxI10mJSyHmHDB3uAMDoyegOp3zNtt56pxjw8/S1JZ9B3fdLj0wa9ZXKReXXhHrdlLiklo05UdP+GFzFJKTAdxydAt5Co5kf/qOQRUAVCImxJwD2gJAh0UF1emc0qyj+ZzVfapIasu+g7puD7RomJSUNMp9RjkGla6bK6tCcdnOPLk8imhDWUJhYWFlRWEJRR0bGQ8A1yCXWvmsFL8ar+N6oL2MuBBzDts2uo676nRO1WxCx52rKGrrvoN6Xmr+hDYA2JsrCusASE5w/937OSkEl+2yvXPDsD+sFeXEzom/AThQ/29NKM/YItqUAkAVqG8kVa7v5VfncdeeBoBT7gvfQsc5pBeVRALlSKg+5zzNlner3BYJxOMwRW3ddzDXLcttAwCO4VMWPwLAMXB2cSwAZA8MvWW7dct0B/DNMIo2LQ0Aslt/REud/np5JHAaHajPei7E+9W6o9+PAHA0PjXUnLtleCSAg3E3VZtz3mYv7ww7kwQcxy0UtXXfwVy38353/ew/ZebDYQDeXf30PwHsOnH7H3RiCeR3kg9l9BoLdnHTC+TkVaXFROWYaasGgK167Eai/iSi/fNrQurRJmCLJoRT1ETnuFkRnHOrB90E4JcNMyKIuW07pzQb23diEvBrbtod5IPMvO+gXeeY0yH2mluPMcYmtI2K7vokY4yd6D9wxaoXxl1gf8B1jmUjel8b0+reaTR1e/d5DvL1yWNui4676y2aenuH7J1jRhRTU3+LSQTTeL8WpOWefzajnKQ2d846t5lzVurKlz/Ys+Dm95zU3CbOweowKXjo3c2bOvfLp6lN+wZjrHo/N88O76ro3ALyc/P21IUbznZsS05dPqdfAz8LP7O8tEs7RwhafmBbfJcGZLWJcwKF79nL2rUNlmmSdwEJX5BqybuQIUMGJIdVhgwZct3KkCHXrQwZMuS6lSFDBoS+r6CRdEGGjJCJRtX+/i3k+0CQ7wNJy+X7QDJkQHLP/YgzS35myX8m0wmK9p2N7YmrgacdgHKdIfjnlowRd8rHluCt26rX573/UNjW/ndO5T8hfXTkWBXdynefi/y5X/UJ9Lr15XXb4Urr0b4VEPiXxy85H29Gyy0CxNYjjSvl2i1cwX8LQMh91Rw7nNC2ohbBiBu3TTPNBv9cI+YyWnPnTY4VNnN/dPjEKOKRZpg7MJ8rqLyv7e+MMXaxd/eL3M1L8Kjqv/nuq+Lu+wP8uQJfXrcVV9pCzYHAn/yJsZN3biXl1gdim6vV5dosnMN/60LILdQKO5zQNqfWxYiLtk0xzYp/DtPUXEY97rxw6vK+D1SxhW0LaGrD3AFat69il2vjRI3RfEU71Z/98d1XRb9Ar1tfXrcVV9pczeG0l77PGGN7e5Fy6wOxzdXqcm0WzuG/dSHkFmqFHU5om1PrYsRF26aYZsU/h2lqJaMud1449as1ihljXcfQ1Ia5A/M8+fSbt7s/hHTDfVlPKWQDh89HrR2p1fkawM3rfvRibdf+siW5dRGdtvjQzYA1T9tXXVN5u+yHiwDQ9CgptwgQ21etKtdu4UD9cdBzQEztFVPa5tRhmQDwpRlG3LhtimncwJQqhFMrGb/P3gVgfgUtdVbbGAC3zXivNkVtmDswr/nnXfK+dOjJ/glUnDqHwgtA1dl816vfvGKULdns3q8sOI3KPA8d8NfvjziDsmw1vG47XGlT2vcN73/gBJb2JeUWAGLr5FbK9adwawh5gNuGHYy4Sds00/xhuysZrbnzJqkLT0QDQOKFPaTCDXMHZt2uQxN4adHr8F3b6z77+MPkDQc7Jj4JAB/2XzM+8+3ke/d3THwS2NSu/uQVb28f+ggDUP7smpjFdx0OxrpVeN3uG3LyG0CUK61VczEy5S89/vPtnL/TclsDsXVyK+VSCt877Z1pRb4OiKo9YkrbnNqFEZ/qoLVNMk1rgA2xJyNbl3L8pTdf+C8t9TXhzLXMDlHUxrkD8zz5BOp4NuviOPr1bbljQtNPTqXt7gEA2c+fjhvVakD77U129wDQbV/Xw2cmIfXGB3sCs7NXteyYP2RXEM71a3jdbq70BBGutCntu9b64TltO6+tQcz9w6J17bOjbOVWyqUUvvfAM47FnVY3UlUhqvaKKW1zalhixM3appgG6FchKHZnLD7X8l+vheXd8eVdlNQ1W58DgF9QSFEb5w7MeglTmNiXEAY4Yo50vTZ/KFx8noXN4xDebiGauPcRs+9BIKXGfgB1SkqBnnuOBOHxVpfXLcqVNqd9lzd9Liyn/ylabmsgtkatlEsqXIX/NoeQa9WcmNC2KrUFRtykbYppsME/1xO7Mwpw581Sv3rgFFD+o4dBbE9tnDsw67YhvGjpAiQDQBvut3XOA4jg6butwoDwyFIA959Lzfsmh/YVFiDwukW50qa0730Pvf/O/rtW3+sk5bYGYmvUSrmkwlX4b3MIuVatiCltq1JbYMRN2qaYBhv8c12xK6MAd94sdfq7jx8/+mY/E4quido4d2DWbS/s9mzuQi8A4AGW487uR8m6ifwTTdcLcgDOrC5zEroG5byUHq9bmCttSvt+fGo9NF0zZftKSm4AVkBsX7VSLqlwFf7bAkKuUXNiQttq8rgFRty4bYppsMM/NxAndNy5SoA7b0qGn/DRhnV/OW3CYzVRG+cOzOvbEZNWlke6Xkkvi3jc9ytu64+fXv/Ux3/WVU7M+rEpNgFOR8C/FFeH1y3OlTajfRfn3gHAMXmt8RfUGeduJgDE9lUr5aZSCufx31YQco1aEVPaVpHHrTDixm2PJ5hmi3+uESvc8r7W3HlzMnyjRsCxxrdQ1MbM+8Cs27pTx335OABg9Z6XGmvONndPN3xN/sHYpsAZYHV0t0CvWxWv+3ztWra40ir1+dqqZ5bXOEpiASC5tQ21J7cIENtXzZVrr3BXbg7/bQkh16gVMaVtFXncCiNu3HYxwTTY4Z9r2ubGZM2d11quRM77s+JQtPG9CErhJrkDc72U89nYDYwxtj9peCVjjLV5xH3t4j2MsY3NZ8xdtLrQu8/u7MkYc0ZOYexS9BjG2JQaG7/IZXenB/h6qdxaRxhzdnmdMVZQO5Wxg00yR4/OzGjsvlqoMvoOMXVB7VTXTVPxC2OMscdeYYyxE33KxNVK7n4bGGPHItMq7eXmyhUu3K3+dAtjLK/GFxoHRNSKmNI2p2ZsBZ4SHZhv20TT3ANTVaGv1pimZDxd51vGnJ0fs2O5cqxMDNvH2JR25bQjzTB3wLjnc1Je/mHvu40+rGKMrewbk3jPLMa2p8fH9zvDim5NbtKwVsSQC679zQPq1E3fNuvO6MaDf2fZDacuf233uC4vbE2PT0jfFVjuucLrLm2dYc2VNlKXts5QA7EvP5y5fm/WyGNMXK3k1gdim6tV5YoX7lYr+G9dCLm5mmOHE9pWkcd1MOKibVNMs+Kfw9Q0LqMed97cNO5Y2d934+7nep6mFW6cO3Cfm6/afMh5Y3edN/cudfrsNsC56+Ghk/VkhypaRbLiuKB8bt6E1+2f+vCOi207OohqPSC2X5Vbis3w35ZqTkxom1PrYMSryzQ9A3zUGrGSUYc7L5763MrCdl3o0zbIXQ28i0VZrhOQn6xYKnkXkLwLaXlo8C7a7T8GAFWr7oEMGTIC9QgV5Mdb7HinVafah9d2H+uQj7fy8VZaHjrfD1SQdykpJUxy4eS6lZbL7/WSB5FcefL1rQwZMhBC31eQdsV0k3Z1qtOkaVeb5fJ5snzSJtXyebIMGTJQ/fzk0jwgxgsG+OUCkBJl+17HjXR9tqH4ZGSEsyK8CaoOR0ZUsGYBLv5o3tnOjRFCqHT/1WR6uH/scYkh/x9zSrNuzy/cmsN+ud61c6ZTQc/bM6NMcOW6cf5z92eSTs3esDk6I7UJKj5fmduzz3MBLn7b3OWLG9MQ3iSWtZc4LkDhNlbr/UZYTaKHe/smqdVO2WSuc5YTAOA8c91yXhq1MiUK91zREHDvvFP2jzTu8DI0TXuVc95QvOHe/ORBHLbCletEFup5Pv+wA27Wb0lnJwt4nMdiGoSbwLLmiOO6FG5Rtd5vxNUkeri3b4pa7ZRd5rpiOQEAzqn15mWFD1emROCeKxoK7p1zikBNVwo3NE1n3ea/1qmZq0Xn20/guBWuXCeGjMBK9+Y+eGjmIwK/bFmJ6bo1gXATWNYccVyXwi2q1vuNuJpED/f2TVD7OGWXua5YTgCAK2rdeVmouSkRuOeKhoJ7V5wiHGlc4Yam6X6c97HMjWkAsKXrMQquPL/egHkL/qQ5C1btry9MINwEljVHHBegcBurdX5jQ02ih3v7JqjVTtlmriuWEwDgilpgXho1NyUC91zRUHDvilOEI40r3NA03dc5Q2vPBABscfOWdXDl7PBW15YHcc7F14PvSlhcZmqTVw6ww+tOq7aqzv4GoPzUOeXOuSTlO3LdcLCy3xgZAE5iWXuJ49YUbmO1CJ7bJDeJHg4/2OMqp/xgrvsHALeel1YtwEoPHu7d6xThSFMKNzZNd93GDskuAlAc43qMPKjFla9K//Hy8y9VwIs45+Ub76gxuGilmU1eObC+4/fOea9UKVub0xJHAVnNr/uH986VJCxr4MWj3TYDODJ48oY31gAkCDeNZa0Qxy0p3MZqUzy3dW4SPdzbt3212ikCc92TmgYAd6sF5qWj5qZE4J57NSTcu8cp0pHmLdzYNH3szahZ88cAi+5z7d2kwZV/+9wPddHjiZFzvYhzTpyXHI7hn86/17hSRY4Nvba3R8kDXXspW5s6AcgcGQXvnQ/1JvnHGwdiUbN/Xp1DnZZ1B94CSBBuEsuaJ45bUbiN1WZ4brHctunhauy3PbXKKQJz3ZuaBAD3qAXmpde3d0oU7rlXQ8G9e52iUdM9hRubpv9+QNfmMwEU1Pfs++DKLz0xtC7gGDNvLYc498aCYUC3pKWlhoVycjauV3sg+sYkbsv13QeRNaHcuedn0aT7Y4FeYfPwSLvuAIYCJAg3hWWtIo5bUbiN1aY8daHcdunhauy3TTXvFIW5rqSmAMA9aoF56fXtnRKFe65o7OPeFado1HRv4Yam6a9bx2M7f8JPbXxv9uDK95xoDQCtHcsANeIcAJYemDXrq5SLxnALTn7iwM0Aeu1pyW2pamqj+rm7KHLTpk1b6v331LY7LWmUJhBuCstaTRy3oHAbq80Y2WK57dLD1dhvm2reKQpzXUlNAYB71ALz0jXWMyUK99yrIeDeFaeI1HRP4YamGUzioYjPsbq3760eXPl/XFthkQcBNeIcwIEWDZOSkkZhgbso95+KSuUg5+Q/41r3jcqWKuJVP4+hfq1atWrN/eshxNnnniv4awLLWkMcN6VwG6tNGdlCuW3Sw32w3zbVnFMU5jqf2j4A3KsWmJcBPjyh485VJO65orGPe+ecIlPT3YeXkWkGj1gN+s95rYbhNXFNcQEAKsqa6bzBM39CGwDszRWFdQAkJ7j/yP2cpCe/EcfdNypb7hMilap3j1w/b0KCiyRbgVLY5p4r+OsI+yxrDpVebk3hNlb/xZSRbZnbPj1cjf22q+acosDiVcRx2wBwr1pgXho1NyUC91zREHDvnFOEI019eBmYFqF3BRWAUUsefc8wU/vErQ8A2AEdYhTLbQMAjuFTFj8CwDFwdnEsAGQP1JPf0PpHAChbPlDZQs3LAE7rvZF0y/VrHgVQtmpAyx9cXyIGUBDeNlnWPuxxAXS5iZqvyr7aHj1cCx+3rVacosDiudQEALiitp6XRs1NicA9VzTMHu4d52vX4p2yf6Txh5exadrrpdYMdzJWcV1vxhjLQK4ernxR4hHGKgcM4RHnHo6y+xq6nehaxRhjhckjGWNs52ju/3DyreHfMcY+2M5vvdyBMfZx7CTuzj0/v4veyxh7azfbWHMfY85nMI+REN62WNYa9rguhVtUzVVF4p7boIfrwMdtq32csgmL51ITAOCKWndeFmplShTuuaKxhXtXBuZ2inCkcYeXoWmadXvwT/HRHT5kbGo2Y2/dnRBz86BDOrjy1Xe+MSN9crmCOHdHTofYa249xhib0DYquuuTjDF2ov/AFateGKe6UtIrZ2xr+5e+fjVbtVXQ7ZXlU7+7Ibr3Avedc0m+7zLpy+e/Y4zldJ/2r2cXInEQCeFti2WtYY/rUriF1UpVJO65DXq4DnzcvlrllF1YPJeaAADn1HrzslArU6JwzxWNLdy7d2Bep+wfadzhZWga+XPzJwubC363EDu8q6JzC2P5qcLmYT5b5365MfanmMQovXf5f7vQ3HXzb2dbOQ/Uq3cNDcLtF8taj8Itrtb7jbiaQg9X+iaodZwiWU4BgCtqvSqs1MqUKNxzReMX5Z5ypCmFG5kmeReQ8AWplrwLGTJkoDq4cDJkyJDrVoYMGXLdypAhQ65bGTIk9xySew7JPZeWQ3LPId+UkG1Ly+X7QDJk4MrmnsOKZQ4A5V+eTcqwAL0VHDx7/W2aW4v2nY3tKW3HH8nRDjFyuQSt+7NuvSxzAJe7j++WFt/PXJD31T+f1K7b/Llf9bFct3Y564HkSQupS6b/mt/m6QY0tQA03YSj7R/Cm0Au57jdAsT2YKHmBeZlYjmhcJ57bl/NzZjAyOckhrm1VzlbsszZR62deRlHndMtJG2f1rv17vstc9nlrAeQJy2kLhh2hBUNittKUutC08U52noIb2HuOYVcrnC7dYnt1YOa152XsOUCqHkz7rn9tvljxT4jn5MY5hZetxzLnD00lDHGTo+2kKTqrtt+1uvWLmc9gDxpIfXYPMZYSXxKJUWtC00X52jrIbyFuecEcjnH7dYltlcPal53XsKWC6DmTUwjtM3NmMDI5ySGuSMoLPPSaABYHLzTwqkIBvdchCctpF62JLcuotMWH7qZoBaApptxtP1CeBPI5Ry3W4DYHizUvMC8TCynFK5UTlBzMyYw8jmJYW7R1/xalvm+Sd6zVG4UuQ4CvfSYm6b16/dHVFwt774PTd1NPtdw1lGVV4yyJZttLFsST1oIo92wrBxANM5Q1AI4bjOOtl8IbwK5XOF2CxDbg4WaF5iXieX+FU5RczMeZp+Rr0iMc4uuW45lvjNj+9qMjC9evLw2I2M2hyLXItDL3sza+PBL5UD5s2tiFt+l4JiUfRVN3U0+1+Gs48P+a8Znvp18b5X4uiXxpIUw2jn5DQD8FNGapLaEppsz1/1BeNsnl3PcbgFie7BQ8wLzMrGcVrincpKamzGBke+VGOcWfJ7Ms8w7zBlUOQd4tGWPTwEORa5FoH+743o8MGzwEsfs7FUtO+YP2eX5K6Hs8zR1L/lcw1lH9vOn40a1GtB+u43vbyXxpIUw2mGRAHbsnZBIUltC0025534hvO2TyzlutwCxPVioeYF5mVhOKtxbOU3Nzdg+I98rMc4t+HhrzDJXUORaBHra9UDYxG8XoU5JKdBzzxHPL/h9L02dI5/7cNaBhc3jEN5uIZrYeJ5M4kkLqQGgbFSvv9NyW0LTTbnnfiG87ZPLOW63ALE9aKh563mZWE4q3Fs5Tc3N2DYjX5EY5xZct8Yscy+KHNAg0CMBILXGd7j/XGreNznKXw3VvoemzpHPoeasA3XOA4iIg50g8aSF1AAwKXFJLRDVFtB0U+65Xwhv++RyjtstQGwPGmreel4mlpMK91ZOU/MztsvIVyTGucXWrZplrgovihzQINBdJcQchjOry5yErtwbzfy+h6auJZ97fgOMO7sfJesm2lq3JJ60mBqYeXJ5FDU3LKDpZhxtvxDeBHI5x+0WILYHDzVvOS8TyymFK5WT1D4ztsXI5yTGucVe36pZ5lzMHOVFkRt9xy0rboGJWT82xSbA6XD9D999AFryOR/1x0+vf+rjP9tatySetKB62d65Ydgf1sq+WgCabsbR9gvhTSCXc9zuOGtie9BQ89bzMrG8D6FwpXJC29yMCYx8TmKcW2jd+rDMPY+GFcBRBUWertWVA8DWyr4lH4xtCpwBVkd3AwDffVdw5HNNrOueDrvhS6MW40mL0Kixdct0B/DNMIJaAJpuxj33C+EdYY9c7sM9d1gT2824536h5q3nZWK5zcJ9KieouRkTGPmcxCS3yPVSvizzAXcyxtjD3Rl7kUORaxDoqc1OMVY18D7npegxjLEpNTZ+kcvY3elMta/Q1BXyuS9nnW1sPmPuotWF9q6X8qVRi/CkhWjUB5tkjh6dmdG4kqLWhaYLc891Ed42uOc2yOUa7rkusV2Ye+4Xal53XsKWC6DmTUyz1bbmSCMw8jmJYW6BdevLMt+VHl+n1zuMHWk05ZX1HIrcB4HOWK9jE7Pm3P9yGWPZDacuf233uC4vVG5Pj09I36Xsq2jqbvK5Dme96NbkJg1rRQy5YGfd+tKoRXjSQjTq9u4zFyS1LjRdmHuui/C2wT23QS7Xcs/1iO3C5HK/UPO68xIfmABq3sQ0O21rjjQCI5+XGOX253PzFfsax6tR5NqzVhdaRABA1aGKVpGs2Pvy2ncfGga6Ki51+uw2wLnr4aGTbX0e2T5PWkztX269+xHnaOsgvG1wzwmFc9xunaqqCTWvNy8blgug5k1MI7TNzZjAyOckBrlDg3exKMt1qdYnK5ZK+IJsW1oeIryLdvuPAUDVqnsgQ4aMEHm8xY53WnWqfXht97EO+cdfti0tDx0uXEHepaSUMHkQybal5fJ7veRBJNXy9a0MGTKqJcKnAOhxxbTT4+pU95CmXW2WM8Ymyz9fMmSETEx2X3chQ4YMyO/1kiFDhly3MmTIUMX/A2HsiZb5YdmEAAAAAElFTkSuQmCC',
    '1702.02925v1.3.png': '...',
    '1702.02925v1.6.png': 'iVBORw0KGgoAAAANSUhEUgAAAg0AAAEBCAAAAADeWcmuAAArcklEQVR42u2deYAUxdnGn1mWZV2WQ075ZDkVgyKogASFoIjGDzyCUbkVFRX4PBCM1yfBg6iJd0SNKFGDglE+70TEeIFB5IiigsQD5PDA5dpFXFh25/3+6J6u6urq6qrqAUlS7z9sz2w989Q7xXTPTP32yRBcufKrwLXAlVsNrmRFRP1cF1yhHxFlCMikunbI7IErjzxoZvbUBdGeEU6lmhdLGXJnCld8FcbfNX700QCAyq+LCrO763RA7WdFhbvpIJVc5dd161LN7laN/OMl8zYePLzps2d8vaMIu3d3BrBjQ73MrrY1a4AGbXOjvvweaF8/wWnOzva13nHTAzKyR5RWbkxR6xL+uKSNP/0dUTu1OrpVm9nPreqIk6wHYMfcpVvbn9UBz5zlP5JwnCi7f64vleuAVk0B4JN6qqdBOjjXvMBDu7M70NNDZNcNIFltrjvO++Gf1x2L0rEPE1VNOgz9bxd/LzT6n9cdi9JzpnziH157/oaa5f9zW3d68fJS9JhMRPTlqWhx+eZ1159QVPcr/7c2Nkf/69fFagp2vvjfn+GA62+98exDx30bfUS5hD/mliuOGPh+7rjV5JuvGdTnmVoiIomdON2Q8Mfjj8D+l1x99SWnNsSqyCSJsg8fMHrJ1sUTfv/nbkQUPY6xm5OdcGYb3B209ubhKP2ciOhXR+K43xkNDppHRNnpB5y3ZOt7lz84s3vEQfxqmI5m1f6PSzDI+2F7r6yyO0S0BIODn1/oXktEtQO7E9ENuMO/tecHRERrhuAW/4YHRuEzpaZg5yMMJSLa3L31N+Ijxkr4Y7L3F/3JPx5GRPRa8fm1FGNHrisIv4fTiYjo665zo5PMjm+4iIiIZhZ2I8lxvN2cbNXoy7k5nIW+NUREu47Pmg3mmpcd1+g9IiJ6tDC6GuKvG/42fNMb/o/FKPZ+KO2YSTr1FHOXIjP7FwAomAIAo/G4d62ztUs3ACjucvQfvRvo+4Y5fUWF7NQBgCa/2nCl+IgKW3UAIDP+xtHvcIMGnPHHOYixo6Vb4p9rW920NjrJex+4rxcAYOTpgOQ4Wbb4ju+4W/tfvuAeACg6NGM2mGvevQ/e5500zh1o8HnDN81G4anohafRRclq7/Tcsx2Atv0/+gcAYPYw/94xn88HACw8VkNKZqcD5ptfKE1oMp6/Am+Pt2zsROv4tZFJlk9uN9K/91JIjhNrdzWaNuZvuKXT/67UNcQN5ppXPrn9CP+pvMxgNTx91glNn9uV7hK105xXACBzKQCch0cBAK/39+8dUjLDa39vDSmZnUp0MvdUPOCjBdzhV+hmY0eolz5Hw59GJjnr+1Ny7e3dUHKcWAvmAT8P/a9/fPe5uzU9cYO55s36flDOQ59S/dUwv2/dsyrmplsNV2DgCb99txr9AOCMBrN2Avjo0Dr+vQ3PnlMBoLKBzguOzM4zda62MFWGN9jBZ0+fdoGNHaGW7gROjUzyb+ic+4WiP0qOk59QAOFzyk+vXnqr7mpgg7nmvc481HtYezWsaVMHwzA73Wo4elbjN645ps00b10P3foigMdGB3dfUDUbwLNnaCgJdqq+/Xbd3LHznjvRwlQLeOevtXOeefDS8Q88X2hhJ2TtkQevvCP4z8tP8is0D37rIMmxshaPHNL7hujNUw6/+R/JnsKD+eZt4DwcrL0anhoK9Gn90o50y2HImtkXH7zx0jvhXUc+Cuxe3zG499hOMwCUt9AQEux88tRTz24YuepUq4/ikfU+r+jR85jstoMzNnb4Kj3ooI7sNZefZCFqhM92agz+Jz0x+88nRW+u9yecs8twMN+8QtRafPr00gGfAO03vDTUeiWsOLgIaDx0KL199o0XNAbQu9O8rw78C3clmxlz1YddPzxc68QcttNhQuwjJtZX8D5mKm0H3D/ihGWHJtpR6zY/DidWBEf8JNst8a7pH5xft7C2+vrDxOMEowVtLiOgdkWbGgB1c5+CHTFl8pTbkmeZG9xVaF7OAwBs3V/ztWHlIWWtW7e+wL8SbZBb1DW1+qvh0UL8GQAyxz20/X0AyJyX/ROeOZP7lXMKH8E8nVf7sB3FI2rUenCP+MudDyXbSdbt1SBYUtwkT4b3oj7u4TGP/vW+QyPHiXXkYcCqxac0b968+S+DG6/pcftCnXl6g8XmnYz3g9+4TfdMMXvigAEDBpxzyCvbAKBN03Lv5s9b66+GrwvgvZXHz1EFABhV8Ni3pfyFbMtTnqioW0dDK2xH8YjJtWlu9z7sqDE+TLaTrHt88Kk2P8khB/xlp/cydDQObJaJHCfWf7UHFh36zJo1a9Y8wV7N/1T33CroDhabN6TlX3Inmm2FmquBVhwOAJlh1c8BQGbwskoAwJzB2ovh03XAyi+8bwX8d3EHnvTp+JHsE3EAF2w9/3QNLcGO4hGT6zc10zOh1UD4oUZpR083V9wk69+/+Xbvpx+kx1pXOX/o3Kpdu3btDsi1DOh8y+dfaQ8Wm1d/Wvk9/r0zztFcDbO2ev+eghlZALij5eUAsGzDMYkOKlANAGuHHgjUDt8GAA+ed6B333n4KPh/ueITAk5utb0dgApUKjUFOxWoiHvEJFvbL5/x4lFgGm3qbFmNmZukdkx0ueImecZ9N03PAsDTzaTHybJVE79iJ/cV/idPE/qaDBaad+a9v55JAPDXzCHQ+dZqQY+G+x2xlogmdq1feuylREQbThn8yqvXjP8+6XP7Jae1QckvRpx9bB1cQXT4a2Omzn1jwhnb/Hur9p/q//TJz5uU9riP6LY5RL89qWmDw85cFasp2Fn+y5+UNjz+Svkjxkh8cHo7lP5ixPCfH3XF10REy3/ZuUHjAVOI6MGS69ZMktmJ0w0JLzmjU2njvkPe4+9nkySit3p0e/iduZdNXjtCfqzs4oizejVG/9yNiwc0Ku39uPcl3GjtwZLn8q3uvR5/Z9ZFsyQOdHe70GfLdvc6BGZbLZZ1p4+W1nY/MnhpnndUM+xju102zqPTGudVODzJVUsrOx+xf/zxj7Hb5ZOlW1sPaCgRcXuf4PY+ub1PrtyeaVduNbhyq8GVWw2u4OgaV8gDXVMA4DhKU1Mo/zVln5DYm8JTfnRLx8F93uA+b3CfN7j6t76KzNqOq93XJ1a7N7uSvEHku8seamRrJzc2+/CG4h8msB152cfXV2UvPMhKS+borQeetnO24NqBzesDwHD/v8XW6ZVVuy85xG66bPD2e9d9c/jlLYN7RnXvWbLu3ZP6GwtFDIrKisp1JdwvpZVY1oqIiMonXdQT3yivPpA8NjvsJqJlnb4O4K9LPyT6+vh39TWZlszRjo6D9G2FNB70u5C7kC6/pJzod0Wv2cyXG1w+9AuqOLMRm2BLAHUnZzVVORcRg6JyvCWvK5F+xVlRk3dERLTzy5q7bVcDG/t8o11EdNGZuXteuouIaPkAfU2mJXM01XA1MI3LHnpl/oIF849b499zd/HTROU4xmo1sMHj1hDR9ibta3J3nXj/tQ+v0e4i50I0GFGOt+R1JdKvOCsgooQzRb229mctNnbWsUUA+p3/g0dI470fAKDjahstiaNF7ZvZOiu4CAAev6Cdf3wAAaiPLVYTZoNffn7F/ijt99yq3E7YFuPthCIGI8qx5Xcl0i+Vlb1xFVn7WiMAaLXrLf+GA++6Jwu8NDA/8jv/b5j12EsAYPXCEbnjodvPAv6OU6zE2OCyXdUASvEdUgpFDGorW3WlcC+shvKtpQDQCJ/6z//o313x7MP/fGJWfuR/f1nGemxHANnLZjKBukD1nT2ut1MLBi+oKQLwYWGX4K7lb9QUnt/IWChiMKps3BWFlb2xGrajPgDUCbYzFr81bEHXXq/XzYv6+83K0gnMPpzfivTes28eNae+pVRucEERgCXLJwbvopavnJB57uh5bU2FIgYjysZdUVnZG2eKnSjy+O5g43d1xysLFpzybT7Ed88cnU6g+tpx/GGv255YO2yTpRY/eNcFA34T3PHksAwGl060EhIMCsrGXVFZ2RuroYFHe9Ugx6B8dM5dt398wrzTs3kQn/Y/KafwUqZN+BPaTk/M+2/bz3y4wdc1f579VYquANDj2XIbIdFgWNm4Kyore2M1NPZeFHYid7K68LZm6PjaDYvnptdeuavptm3banZv226r8Fh78ZamPZe+am0oN3jG139l55tF873/FSsshCIGQ8rGXVFa2RvXDY0O2AgA38L/S1SVK/oCyEx5/dP07yrKN1wPYGWL6ztMtBOoeWsAd9boU7OoCGiCz2xOOfzgl5c/WYCPC3y87rSK7UVANZqaC4UNRpSNu6K0sjdWQ2bQ+wCwukl3AFtKivfLbG8IAG26pNPdUlKMfv0AYE6XadYvLt83YWo7lxZ81xpYH/yJD6PLI27wuwvvzQAvDPVk0W1YEYBPGnU2FhIM8sqKiunKlpJitZXkM8Uu2P+FF3/sxJWrAXp2Yh1gU1kf1B1yDwB8teVnFlrsp01lOaapdkelpTPga3jQ5KayPkDDgW+0Btat6NfXYrLc4FUjN48be/GoR9r5Js/sDODLt+8uNBUSDXLKyR/z5LqSm+umsj5JVtQf4u4afuIBDQ49/W6bT2q5sU/1W7Fl0shqItrRZSTRznMvemv59NFr9TWZFvtpR5eR3p1jf1ra6ITfGnwyzc/qRVznfarfZSQRlZ9zx9/f6TXoG7vvKYLBR/kXbDnZmsn3fPDUYXdqf0/BuwgbZMpJlvyucHPd0WWkwgr23l8W/u6vO3ofyX0c8tmSH7r2zOwLu12qnxgU+kLwg+V0ZNeMpXD84JWLmvRuqa/KCYkGU7crxopjreD2Prm9T67gdtC7cqvBlVsNrtxqcJV+NaRjrfYEqdVvn5DYm8L9fnRL/Rxd495huneYrtx1AzTglbzSNtl/ORop6XsUA5gDBnSNcGyiJTqy0YqlayIwi1GFrfHIjx20k/2/lbsLLm1uQzpxViIqChopga6RwRzp6Rrh2IiuERzFadnRNeKx2bdWIWs88pMM7chUt584uZpmnpZMOkFlRVCJo5F06BopzJGarhGPTega0VGclh1dIx4brYawNR75SYZ2JKrZM0cQ0UHNk0knKKyIKnE0ks5qKGu1hYgG42Ob7hD57s8eSET0ZL0d/s3isZamryU6itOC+q8V+GoTiIjosZm5m8Vjo/mGrL37ZC/W8tn1nib6AT8xUv0blhHRkoVR18mDmRWJSsiaZsoZgNSYCKR0TYS2MSnBUSqtKLwiHltbC8MtVtDOHxodAaBH73RWoioK7ibhKlIb5oABXROhbUxKcJRKKwqvRGgbW2sC3GIB7dCb7ddPL604v1MqKxIVBY2UsBp0YQ4juiZC2xi9JQ47SqUFCV0TPbaxFoFbzKGdys0/eebmgjV9Hz8hjZWKiIqKRtJ4I6UBc8CIronSNobFOUqtFYFXIjCLhbUo3GIO7VRi0dkFaH/iRTvTWImoKGkkjdWgAXPAiK6J0DamxTlKrRWBVyIwi4U1CdxiDO3UR5s2ALqtXpjGSkRFSSMlrwYNmANmdE2EtjEs3lFarSi8EqVtjK3J4RZDaKdhURMA2M8IyYlYEVXUNFLinm4dmANmdI14bFghRym1ovCKeGxjTYRbrKCdwsN3AEAtmqXqkqCippGSVoMWzAEzuiZ0bFxhcCWdVhReYccprPFwiz20c9rU6iJgI3oE3my6xKlgS0mxmkZKWA2rRg4YB/rhnWvS0TXdV3fI0TVtOy/lj421mKNNbTsvhZ2WjK5p23kpf2xR4WZ5cMumtp2XouHAqyygnbF3v3oq6NUxBwfe9EknZoWpMJl4Gkn9sZ0U5khN13DHxnSNCK7EadnRNezY4rPIULN8uEUX2pGpLu4xZ+nY4ZU5b/GkE1RWApVgilIa6ceka8TjNNs35Fq2dE0izGLXrSRoR6q67e1NPbumfgo0VRxdA7f3ye19cuX2Prlyq8GVWw2u3Gpw5egaR9c4usa9w/yx3mHuw2AA9kQYyL4x3+weNJi1/zvT5lkgiOAFYRLAPK2EAQlRfsI8TyVQk/ITtvMNoQvChM2hD8+FtUFmJX88Rezuex2eItjTL5IAFmklAZAg8hNxlIGSpwjUZPyE7Xw5dCEy4SSAJKrqu9AyCBWLkUeeInb3vUZ32J5+kQQwTythQILIT8RRBsqN/YGajJ+wnS+HLoimEgGSqKrvQssgFFaMeIrkM4VxFggiwRpiYoZ5WgnL7xCTT2zyVJiaGAaSZr54Y84yALN3R02JpqHddUuDzAr7SUOgYM9lgcSzGOZpJQGQkJKfEPGGKD+RIvskFoAwNx24sDTIrOSTp0iTBRLPYlikleSAhJT8hIg3RPkJ+/nGAxDmpgMXdgaZlbzyFGmyQOJZDJu0Eh9IyAM/IeINIX4ixXzjAQhj02EX5gaZlcp88hRps0BiWAyLtBIfSEjPT4h4Q4ifSDPfeADC1HTYhYVBZiWvPEXaLBA5i2GVVuIBCen5CRFvCPETaeYbD0CYmg67sDDIrOSTp0ibBRLDYlimlTTtufTV9PyEiDfw/ESq+cYDEIamBRcWBpmVfPIUKbNAYlgM87QSBiT8d1p+QsQbQvxEqvnGAxCG0EfYhY1BZqXQiKfQ+PSJWlp+GrPw2iwRTf089JcHNldRdYMKIiIa8ZquZkWmznoiOhJv0gVHERHd3qSGaHOV4q8aKCbFqREtx/mBsZTzvbHuLiL6NT7lTW2uIuJNa6vmXGgYhMIKbypQaGn39xtglwUCQAjW4BMzzNNKuPwOlnwSpJWY5qmE0kDCYSBp5oux9V+Fjy4IkSpcXItuBS6sDDIrnKlAwZaniN19r/F/he3pFxMzzNNKOCAh4CfUlIHyewoObxB5Cvv5cuhCJFIlCSCJqDIXGgahsLIv8hRimaeVMCAhgcXQssXUNMJA9IXj0YUk0/GqlmklzIrjKdxuF8dTuILbJevKrQZXbjW4cqvBleMp4HgKx1O4d5juHaar/9Trhrxmguy7/E/2X4cngua+SAmbAeOEkVHde5ase/ek/v7NVpkgcWklgDkPwzRoxselda6qnzL7BLJMEt6UYegL1y9xpNDKZAVhRureJ3zBI2EzzNNKqCWAupODXPfETBCDtJI4WkT5rVWgUT1wRC39uWu5PgajSisJZZKE0koSQ1/CqqxfkZFCK+MsBb8mzii29xp0jYzNME8roRPvv/Zhjg9JzAQxSCuJo0VUq4Fp3FS3koiOHauPwSiEw5kkvKnk0JewKutXZKTQyjhLwa+JM4rtvcZqkGRdWKSV0IjwzYmZIAZpJXHpGypbTKN1dyKiSaXaOSoq4VAmSchUcuhLWHVE/MgRepaCXxNnFNt7jd0uaRIzoJ0RglTMjjkPE2hs21AKAM2//wB5yD7hM0nCpuxDX9LGxURmpOx94Z5LzOBr+Rs1hec3ykcmSJTZMedhAo396pD3xmrVMQBSZ59wmSRhU8ahL0G/oiPDrUxSiMxI2fvCPZeYwVtbOSHz3NHz2uYhEyTC7FjwMEyjy2YA+BLb8pN9EkA7ginT0BfWr8hISStVCtIZxfdeeSZchzpriWh0h6p01w3LiYiOOoP/475law2vRbi9sDsPHxAYqr6ilsjwuoFpvJD5hmhXb/j7wj7EVd7G1Kvt5kvZf558arncFG86STXcr9DIaCullnK/JpuRvPfJ1w2pEzP86goAPZ4tR34yQXhmx5aH8TROu+PC9atvHZTbX56a3fGhHZkpk9CXcL9CI6OtVCrIZhTf+4I9mpjh16L5XqtX5CcThGd2bHmYnMbEaW+/ecXGXFRAHtidpj2XviozZRL6Eu5XaKSklUoF2Yzie1+4JxMzcnVaxfYioBpNkY9MkBCzY8nDMI22bYG17brlIUeFQTv1o6aMQl9C/QqPjLZSrSCZkaL3hdqJGSmq27AiAJ806oxc8EaKTJBwWok6fSNRY8FdjzVCxfw7C/OQfcIyScS0EtPQF75f4emG7tJREJNi1L0v1E3MQIq0kjM7A/jy7YcKg8QMi0yQmLQSGx6Gabz84vpGuKf9KKTLPvGhnVAmCZdWYhr6wvVLnC53l56CmBSj7n3CVTJjM1KkldRMvueDpw67M8sIj4RMEJO0khhaRDUppvHxwPn/uLL/RtLHYFTfU/CZJKG0kuTQl5Aq1y9xutxdKkvcr4lJMXG916JrEtmMjM5Wi5WLmvRuqY+M7L3dLpvnbjuyd8YAg1EKJ2WSaKuK/VLfJbPEfk2cUUzvHV0Dt/fJ7X1y5fZMu3KrwZVbDa7canDl6BpH1zi6xr3DdO8wXbnrBvwLcz97RHZPcTtJdI1In8CKrhGjXDQREalWPrJrYtWsuJ94Kzk0yVw2EAoTO9CFiVg6kNgvZe/VX/CI9IkdXRPJl5EhIpp0TV6ya2LVErkf5bdWopUATUqWRYxQmNjRzq5h6UARXiiu9xo8hUif2NE1kXwZGSKiSdfkI7smXi2R+1EJi1YYmpQsixihMLGjnV3D0oEivFBc7zWya6Z3bQDgpw/dWWL+qskCXCL5Mi3GW2vlI7smXk0SFWMjCyFAxlCWE2IxOzDJrmHpQJHYHFXv1ecxkT758Ssv2TWW7Amss2zsZXliB/owUZAOZNYv9WuDSJ/krTQREeQhBsZILQ33o0CTUshyxA70YaIgHUjSL0Xv1auhnkCf5G0x6CEikspTdo1KzZb7gRpNspXlY3agCxMF6UDRGap6n/CO56aV3wLV76Mmv6vhyWEZDC61STnIU3aNQi0UFZNqNYRiY6xluZgd6Ifr5NKBojNU9T5hNQj0Sb5KExGRVL6ya+LVUnE/iEWT7GVZzA60YaIgHSg6Q1Xvk7TD9EmeShMRkVX+smvi1NJwP4hHk9LI5mJ29GGiIB0oMkNl7xP/0k+IPslTaSIismqUh+wapVoa7keBJlnKhmJ29GEilg4UyfpR9j5hNXD0SZ5qS0mxLiIifdnkaBEPFklTErUU3I8wyxCaZCnLiB0jmGi/zPaGANCmS4SuUfY+4Uzx8ovr4dEnSEHXIJxdk2M/7i600MpDdo1SzYL7icp6YnyWjamsL8TF7Jhk13DpQFxsjkbv1R/iMvokDV0jZtdIERFNuiYf2TXxaoncj0o4kljDoUlJsojxx4gdo+waLh1IoGtie69D10Tok/xstVDQI4ma+ciuiVdLjIoxmS9Dk5Jk41U1iB3ZYJYOJM4wpveOroHb++T2PrmC2/vkyq0GV241uHKrwZWjaxxd4+ga9w7TvcN09S963ZCElGT3QZomFQaTN4ZmDzE+SLcvUhUXk1wcUiJLhDHMmwkydEQ6xIb/CZAVEXtJRddwg6MdM0GAwi7EkVpKjK4R+yPDdZD4rZU6LkbjWxyGlMgSYWR5MwpNlqEj0CFx/I+SrgmQFRF7SUXXsMFix+IQoBhVzoU4UqqkomvE/shwHQ26Rh0Xo9EdhpTIEmGmGq0GLkNHoEPi+B/lH0MPkBURe0lF17DBYsfiEKAYVc6FOFKqpKJrxP7IcB0NuoYhHi8/v2J/lPZ7btVhZpckAVISITz0EBHIQBWRDrHhfxiyImIvqegaNljsmCECxLkQR2oqMbpG7I8E1zG7irTMTwmQEgnhYZo3E5uhY8X/MGRFxF5S0TVscMrEmfSMT0DXRPoTxXVM9kXaJK8gTKpICA/DvJlQhk6IDrHjfwJkRcReUtE1bLBlx/LiAgBH10T7E8F1DFeDafIKBFIlSniY5s3woEqYDrHkf3hkRcReUtE13uAUHcsP4xPQNZL+CLiO9k64mLgYgy1nZWtliTBxeTOxmnyGjhDeIqTP6GbX+CEzkiQXdapOgjA3WOiYyVVkWEgcqXMVSZ+NurIYJ30j7Q+bu2HmHWySVyCQKhHCwzhvhgdVBDrEkv9hyIqIvaSia7jBdh3LE+MT0DWy/oRwHZvPIk2SVyCSKiLhYZ43w4EqETrElv/JISsi9pKKrmGDLTuWJ8YnoGvk/WG4jvl1g2HyCkRSRWRYzPNmOFAlSocY8z8hZEXEXlLRNWywZcfyxPgwumag0B8R17F4bXh34b0FwAv1TE35SElm0BqO8NiJftOmTZs2rWGXafpY7mlfVMMDVbo9GNAhW3YCWDC4AqiYP7nQBFn58Dv4yArDXrbsRGq6JhjMOubL2gpxZaC0X8Z72W3TJdSfLTvDczdbDUFczOZxYy8e9YjxRzI5pEQkPMzzZhiowugQT8qC/+GQFeYxL3RNbjDrmCUCxLsIETt6Shxdw/VnU1mf8NwNzhTV55V/1GDgwcdNwIjV0wGgq3G8z25/Toc+dv4DrX7T8SoAJR085GvcB/hwwElX6Sq1mDe+ut0jHe8Cxty4+LhVN982Oid1zsdb35/1j7l1jU7Kk5b0pkknPlKH85gztjvNasgNZh3zZFkvDYW4kUZK0y6+ePj+7y38QxHfn5IOncNzj2UqsCe2WjCkRI+IUWkyUEWgQ2L4H/WkGLIiYi+p6JrEwbqqZkJKuibSnxhcx9E1cHuf3N4nV27PtCu3Gly51eDKrQZXjq5xdI2ja9w7TPcO05W7bsCPFubyb55VA/1vtBkhYhM1AyAez5HRNuriFaJ4iSGpwxEmjEKBdapOFGmJQDpKpEXFxkSmp0k6cdk1wgyVNFI8XcMIEWXUjE52jQCbyGibhOyaQEGGl8hIHT26hlEomqk6KmEmJkI6sUiLXFVwxU9PSjops2uEGcamEanpGo4QUUbNaGTXiLCJjLZRazIFGV4y1XA1MMKEUSiaqToqYSYmQjqxSItcVXDFT09KOqnoGnGGsWlEarqGI0SMo2YAqPAcCW2TUExBgpcYkjo8YcIoFFim6kCGtIiQjhJpUQlFpqdJOjEFcYZKGqkg30yNhpRF/ozKjCmpwxMmAYWSj2JiIh6jRFpUQpHpaT4rTEGYYQKNFPvaV7uLiKhb4XdEIz6463d3bbPYUe6/pnNSRETf4EIiomW4W/vVOKQgnCl+u062Gz9hBz3tOrnHNqKq9ui76oVBFcHNyqkmCIfFagdtkTyglmpIKDw9oZVxlpiCMMOddX5GRPQ7zLDIvFuMiUTUZVaWnu30pfVq4KSIiD7FBCKiD3CDWcdzCuHV8I8ZZL4aFl3V8+LviYjWHoOivtXsDuVUk4RDYk9cI31ALVVOSDY91spYS0xBmGG3w4iIxuNO89XgEyIC0GK1GjjYJELb6GkGCqHVEEfqaNI1AYXil3KqScK8mAjpyJGWGFUmJJtehHRS0TXiDONoJI3VMLE/W88X4bs0q4GTWoNJ3pliqlHHA4XQarjrc7JZDUSb6vWooQ97l9PnJ+Do2tBdcVNNEA6JzWkjfUAtVU5INj3+WYmxxBQiM7zzlHVf3DDV4kzxyNCdRETvvk1ENAlvplgNvhQREW3FeCKiv+M+k44zBX41rLh169atW7uftLXSdDVQH/yFer1NRNkb8Bf/NvVUE4RDYqf0kz6glioTkk2Pb2WcJaYgzpDoy5mPVFyKpabpqDlCJEXUjBTPscqficFVzEkdnjAJUShAqlSdMNISwmPUSItKSDI9HW6HKfQRZ6ikkQqVTE0GeGFomqgZUQrYUlIcylMxVwAfEKMZ5hITCMMyXvy4mTRT5cU4aKekOJRAYybET88L15E1QqEQMrWlpFidRhT/eQMjRKyiZgDI8JxNZX1CtI1mhQAfETcxJHU4woSjUDw1+6mGkBYB2lEjLWqhYHqeQT3SiSnwWpvK+iTRSLFnwqO8+7smRc1oZNcwKT+BheWpaGoyhUhAjCzMJel7iiAQhmW8eGqJqTrKd2AsMCbIqvFk2QPqpapzQsH0PCWulXrZNZzWji4jFWlEOtk1SIyasdpqkUDb7PHdLowwYRkvWqk6amEmJuIxSQk0gqroKgVdI2rFpRE5usbtfXJ7n1y5PdOu3Gpw5VaDK7caXDm6xtE1jq5x7zDdO0xX7rrhX6ryB9Zk96nsGo7jCIJjjIoJpItyUZuxya5halHOx9QZVLkzKRR9pEbkcrTxn9yjRgaonkwduoYFxxjRNYyIMY1ygb4Zq+yaQE3kfJJDZpK20UTAmmRFqaqP1ES4HBn+AwXZFBkQ92Tq0jVccIxJd5iAcZQL9M3YZNcwNZHzSQ6ZSVoNIlijoShV9ZGaCJcjw3+gIJvEAbFPpno1lLXaQkSD8TH9DcuIaMlCw+4wAfaT5p9kh76Z1t2JiCaV7jB40pja2QOJiJ6st0P7j8UnrYYJRET02Ez9Pz8vVX33SW8z5Ox6TxP9gJ+wLf66lnKPKgyIfTLVf4OecRyxwTHQJGLSgzqxZqyyawI1C84Hezx3BuCQGkMuB/YpQOqryCB/JRQcY3I9FQS4pIxyUZmxyq4J1CSpOmkrfe4MwGf7RKJmQtE9OhUaoH4yC5MTayq44BijN69BgEv6KJdYM1bZNYHaZ5FUnbxUutwZhLN9hKiZcHSPzmIIDahUP5nJdA0fHGN6HmUgiFGUC/TNWGbXeGoyzifldYMErDG+bggjNWEuR4b/KK8bwgPin8zk7Jrrmj9fHAqOMS0W4JIuyiXejGV2jacWSdXJS6XMnRGzfcJRM0J0T3KFB6ifzILkxBouOMa0WIBLyigXhRm77BpPTUzVyU+lzJ2JZvtwUTOR6J6kEgaon0wduoYFx8CWqUkZ5aI0Y5xdw9Q623A+2LO5MyFiKMLlGOM/woBC5ZNZoJFYw4JjYMrU+AEuXPiNXZaL1Ixtdg1T41N1LI0pcmdsFblsHz5qZstOgI/u0St+wJadSHgyY6+LPulw0cUXXzSyXQ1tbPwiUbbXGMOrKibAfqLyku5ERHQbjKh4mRlP6qqCj4huOLLa5GKPqa0o/oIo23sqM5bgTOMq8hVc5n06rKsoVa0p7UtENOhtIlpb1K/Gl/vDQiJaU/ePyZb8R+UGlJd0p/gnU/1ZJMdxLO4xZ+nY4ZWG3WECIl3DCBlzuoaZ8XCTOFpE9aRxagHn46klOtNYDWGwRkNRppojhhiXo8B/oCCbuAGeQtyTqU/XsOAY/Pi7XQQzVtk12pyPxXytAmziVUUuR4L/qC2JA2KeTEfXwO19cnufXLm9T67canDlVoMrtxpcObrG0TWOrnHvMN07TFc/ynXD+MVwbMyPC8Vk4zWze8hKzFd/Wx7Jw5e88XkzYp6KiZaIm5hrKVgdCRsDGyhGhHY0E2di8njEaB7NqB7fikk6j/x7mOloVk16pUPXCIRJJDHGJLtGwE3itPToGpE1EdkY42+tPCgmAu3IEme06BqSRPNEonqgsqKbzqP4DvPs4ZibdjXE581EEmMM6BoRN4nT0qNrRNZEZGOMV4MHxYjQjjRxRouuIUk0z1S91eD9mnY6T/y+yG+ajcJTaU8ULx+zFSjtt2YV6rUN/53Q9zYAocQYAy0xBsZci1d7Y85EALNfCC6jLjq5b58+q4PQGdPyc2b8cJ6Xfog8oKEQJNE8mlE9/q9F+tNi/C1j2hldRT591glNn9sluaN2TSV2Pf93pMubMU+MYVoibmKTPhMPDqVkY3woJgLtGONFXGCNGM2jGdWT+zWz/khfaM6oobF4noiIHuhVdsidREMPPOxRot+f/NCoC29a1rRG55UzPm8mkhhjll0TioGJ01K9oAdq2aZHfHndLVf/M3yvEDpjIOznzETCeaSJMypVLrBGjOaRRPUg3op+Ok/sdcPqCURvY4h3sK3ZGCLKdtlI9Ey9bVTTaRJ9oX8elefNiIkxRtk1QgxMjFbiFqXFmEjbcOzttbT6wL/xd4RDZ0yEczkz8nAeMXFGocoF1ojZNbIsG8Rb0U/niV0Ntywiqm1d4rf8qkY7iNbcTkRnHk5EQ1rb0DXh1SAmxphl14RxkxgtLbpGxpqIbIy+cADFSMN5IokzWnSNmF0jjepBvBX9dJ7Y1dD7j48++mhfzPaOPsejRDduJKIxBxLRiMMM2i7Pm5Enxmhn14RiYOK0klbDxP7fE21Ge8/c6+yOaOiMrnCQMyMN54kkzsSrcoE1YnaNNKoH8Vb003ni3lOsPKSsdevWF+TeVXT8+R+wu6IFgPGbPsb2N6+CBV0Tqgtva4aOr92weC5gp8XhJlZaKnDImo1hUIwM2jHAizi6RgRtIuBNooLYHyWeI/sscvbEwwHQra9sawwAGDt4+epTAaDFJfe2+Pb+X6TNm6mM5qloax0k4CZWWipWx56NYVDMFVFoxwQv4ugaMbtGM6qH/doYs3Se6AtNdrD37w3wd+3vPnDchCwR0cwXjF45F16bJaKpn4fOFJurqLqBd4U74jX9V+OcVkWmznoiOhJvEm2uongt5ZkicHZj3V1E9Gt86qkR0XKcT2l4ipaDiOiCo4iIbm9Sk5PlW6Gp2nKQ8OPmKsldKkstBwn92VxFNGAGEdGARrs1zxSztnr/noIZ3tcehRf+qV0GANr+avqs516rSJE3g01lfSLZLCZaXAzMprI+sNHinY2t/ypAr445OEjCyYXOWJaXM8PCeUwSZyRCELNrDKJ6andUhvuzqaxPUjqPuLQW9Gi43xFriWhi1/qlx15KREQbSjcREVHFEW06lBUXnv29IV3DESY7uowUs1nM6BoRN4nT0qRrBFaHsTF2rw05KEaAdqSJM1p0jZhdI43qgcKKmF0Tm86jT9esLwOAqqMf/imQXXbukClpt1okZLMoNUXcRK6la0tkTRLZGD1hU2hnD+520UznMaVrnp3uXZg+8MpLbu/Tf/zepyM/XgsAta+e7HYJ/XuW0b7IJbcfenTJZ6//bFzGvTb8W742GO6SLV9T1bp9gdsl61aD2zPtrhtc/ScVEfVzXXCFfv7nDa5cwXGYrtxqcBVb/w8a6t9Dr7dMUAAAAABJRU5ErkJggg==',
    '1702.02925v1.7.png': 'iVBORw0KGgoAAAANSUhEUgAAAg0AAAEBCAAAAADeWcmuAAAs1ElEQVR42u2daYAVxb3FzywM4zAssig8AVkUA7LIIoiAGEBjwCUYlVVFIWzPBQcfLk8CGqIY90hcUBIVBKM83JKIuAIGlUVFFlERGEAUB4YZFocZZu7/fbj3di3dXV1VPSrPV/WF6bldp0/9u7jdt2/95mQQXHMt1TJdCVxzs8G1oEZEfV0VXENfIsogICPWvUPGD3DnUQ2aGT/UDdEPIxxLtVosZZC7UrjGt+zwlyaO6g4A2L8rJztxJKsVqr7MyT5CJ6nk9u+qUYMqjzSpm9petWT3ycMbLLp416EcHDnSFsChnTUzyk+s3ArUPjHda9tBoGWtCKdpOwcKk9sNGmcEHTGwpfvkNM3jt/Oap4Z/yG+nSke3bC/7uUmWPMiaAA4tXr2v5aWt8MKlqSNJ25Gyx6brsn870KQBAHxWU3UaAjuni+d5aHFZK3p+SNB9Ayio7a0xIfnD57f2Qv74J4jKJp+KfvfI+wm9P7+1F/KvmPZZavOWq3dWrv3PmV3plevz0W0qEdG2C3Dc9Xu339Y/p8bXqb12N0K/27aHakp2vvrvs9D4trtuv6zdhG/9RwyWSPW584bTBn6c3m4y9Q83D+r9QhURUYCdMF1BeP3E03DsNTfddM0FdbDJN0iixBONR63at3LSn//eiYj82yF207KTLmmOB7zS/mE48jcTEf1XZ5z9J6POXvGIKDG78VWr9n14/aNzu/ochM+G2WhYkfpxFQYlfzjQI6GsDhGtwmDv55e7VhFR1cCuRDQd96Z+e/onRERbh+DO1C8euRxfKjUlO+swlIhob9em38hHDJVI9Un8JeeZ1PYwIqI3cq+uohA7wbqS8Ie4iIiIdnVc7B9kYmKdD4iIaG52JwrYDrebli0bdT03hkvRp5KIqPyXCbPOXPESE+p+SEREf8v2z4bw+4Y3h+95O/VjLnKTP+S3zoi69ORytyJz+2UCyJwGAKPwdPJeZ1/7TgCQ2777X5O/oIN10vqKJtjJAoD6/7XzRvmICltZAJAx8fZR73GdBlz814UIsaOlm5e61ja5o9A/yIceebgHAGDkRUDAdrRs7r3fcb/td/3yBwEgp12GWWeueA89+nDyonHlQIPnDd80vBzP+W88jW5KtiQvz6e3AHBiv3UfAQAWDEu9OmbzMgDAil4aUkF2WmGZ+Y3SpPoT+TvwlnjXxo6//bLQN8iiqS1Gpl69FgHbke1IBRrU439xZ5v/3qhriOvMFa9oassRqVN5ncFseP7S/g1eLI93i9pm4WsAkHEtAFyFvwEA3uqXenVI3pxk+XtqSAXZ2Y825p5yB6xbzm1+jU42dqT26mbUOcM3yPkHz0+Xt2edgO3ItnwJ8Cvhf/3TR648oumJ68wVb/7BQWkPvfP1Z8OyPjUuLV0cbzbcgIH9736/An0B4OLa8w8DWNcuK/VqncsWlgLYX1vnDSfIzgtZN1mYaoa32caXz1842saO1FYfBi7wDfJNtE3vkPPXgO3oEwpAvKaccdPqu3RnA+vMFe8t5qHmE9qzYWvzLAzDgnizofv8em/ffGbzWcl5PXTfKwCeGuW9PLpsAYBFF2soSXbKvv12++LxS148x8LUcUhevwoXvvDotRMfeSnbwo5g7clHb7zX+8/LD/JrNPL2OilgW9lWjhzSc7r/19M6/OGjaE9iZ754OzkPJ2vPhueGAr2bvnoo3nQYsnXBuJN3X3sfkveRfwOO7GjtvdqrzRwARcdpCEl2PnvuuUU7R266wOpRPBLJ5xXdTj8zUXJyho0dvuWfdFJr9p7LDzIbldKznUqD/0nzFvz9XP+vaz6DK8oNO/PFy0aVxdOnVxt/BrTc+epQ65mw4eQcoN7QobT0sttH1wPQs82Sr0/4J3cnmzFmyqcdP+2gdWEW7bSaFHrEyPY1ko+Z8lsAfxnRf027SDtq3UZn45xSb4sfZItVyXv6R5fVyK6quO1UeTvCaGbz6wio2tC8EkCN9FOw06ZNnTYzepTpzh2l4qU9AMC+YzXfGzae0qxp06ajU3eitdOTurJKfzb8LRt/B4CMsx8/8DEAZFyVeAYvXMLtckX2k1ii824v2lEcUaPtAHfE3x5+PNpOtG6P2t6U4gZ5HpJv6hOeGPO3fz3czrcd2TqfCmxaeX6jRo0a/db75c3d7lmhM85kZ7l45+Fjb4+ZuleKBQUDBgwYcMUpr5UAQPMGRclfb26qPxt2ZSL5UR6/QhkA4PLMp77N529kjz9/XmmNLA0t0Y7iiNFtz+KuvdlWPXwabSda95feU21+kEMa//Nw8m2oO05omOHbjmz/0RL4oN0LW7du3TqPvZs/U+PKMuh2los35Ph/pi80Jdmas4E2dACAjGEVLwJAxuA1+wEACwdrT4YvtgMbv0p+K5D6FHfCuV9MHMmeiAMYve/qizS0JDuKI0a3P1bOzhBmA+H7SqUdPd104wZZ6y9770n+9H3gttZdzmNtm7Ro0aJF43TJgLZ3bv5au7NcvFqzih5MvTrnCs3ZMH9f8t/zMScBAPcefz0ArNl5ZqSDUlQAQOHQE4Cq4SUA8OhVJyRfuwrrvP+XGz4j4LwmB1oAKMV+paZkpxSlYUeMsnXg+jmvdAHTaJ5VvAVz9wTaMdHlGjfIix++Y3YCAJ5vGLgdLVtW8DW7uG9IPXma1Meks1S8Sx76/VwCgH9lnAKdb62Wd6tzzGmFRFTQsVZ+r2uJiHaeP/i112+eeDDquf2qC5sj7zcjLuuVhRuIOrwxZsbityddXJJ6tezYGamfPvtV/fxuDxPNXEh097kNap96yaZQTcnO2t/+Ir/OL28MPmKIxCcXtUD+b0YM/1WXG3YREa39bdva9QZMI6JH827dOjnITpiuILzq4jb59foM+ZB/nQ2SiN7t1umJ9xZfN7VwRPC2soojLu1RD/3Sv1w5oG5+z6eTX8KN0u4ccC7f7drj6ffmj50f4EB3tQt9ueZIj1NgttRiTVdat7qqa2fvrXlJl4Y4yla77F5CF9arVmFxkJtW72972rHh2z/FapfPVu9rOqBOgIhb+wS39smtfXLNrZl2zc0G19xscM3NBtfg6BrXUA10TSaAsylOm0bV36YdFRI/pvC0n9zS2XDPG9zzBve8wbWf6V1kournZ/InG1P0ApHvrnu8rp32gYe2f9Ph+uODVBL/s/FI5rWN9LX2zd5fduSaUyRVAMtvGdioFgAMN5jYl3c9PW/7++f243Xl49ic9Kd3lCV+d1Jsk/zuzKrRCWFWfAKq2oeyVkREVDR57On4Rnn3Edq7aOhXVHpJ3fcDVA6cM7WC5l6or1l0TRHRn3Le4FWT7dHUOM7WtUVEdDyAGlMTvK58HPPxUuLaT4l2/fJ9fZMhqvzunlXFCYHCiiwQWns1eUdERIe3VT5gOxsmbCWiA/VbVvpUEpeMIKKTGulrPpD7PFERzuRVk+26x19btnz5srO3msyGc/5yyxNbRV35OBaz4dX7iYjWDtA3GaLK7+5ZVZwQKKzIAqG1BxFFXClqnmh/EfrHSxuORX7fFzedKqu8vXANgAVH9LUaE4BaKOZVUzc+YwHg6dEtTKwdN9GnG7xt1D78HgBab4ltkt/ds2p0QpgVWUBZ+x/wLrJZeQWAfHzne+WxuqcB6GYANQ09cCnwb5zvV70GALasGGHnkekGbxu1E+5/MAG8OjC2yXhjEq2Y1T7qTZWsrxRV5UREnbK/k1USDU7bduudN31uqFl+XrcSSTV9pEHFBm/oRDTik/v/dH+JoCsfx+JKUdYSfTa9PKhU36RCNb27YDXkhEBhRRIIr73GfUOc2ZBcv4UCn0oJet1TRVtOeNNE84Mpp487KKum2rybDW21n5+gRW22+XQDtk2EC89ETp8KA5MK1fTunFWD2cBZEQXCa/8jzIbDHQaU+VS2I6uQiEa1KjPRTHx+3gVFkmrqP3OzQkNba4mIulws6wZuGwh/efmNuTj3G32T4are7rxVg9nArIgC4bX/EWZDQb+DfpW9aJn81VtmmntqdqsUVZNtYXPjtywiorH4TtKl4G1d4U97FtHm/uhepW0yXFXc3bOqOxtEK5xAeO3Vf82jOtqcXf8K+HNOdXLqA8Ax2GCm1uD01a8HqT7V0tDWB8sAoHb6+Gld+Tim7XczG6L1G9NXLq4Gk+ndJavmViQBde1/0Nnwj7XP1sR639+fyO5wCACq0FBXqKJ7lwoA9fFlgGrlu8ca+rrwnAoAFWgg6MrHMW37N/QBkDGtzxfxTXq7e1ZtrUgC6tr/kLPh/RUPZQIv1xR+WXwYuPCrCgC70U1X6fDqT78DsAOdeNXiJMe28WB9Q2OdHs0B8Fndtrxu8WHxOMbtmIwDAIDm7eOb9Hb3rHpKhlY4gejaR8+Gclj+hZdNI/dOGD/u8idb8Cp7mvUGxtd6HaDXx5ysK1Vn4NtNge0b+vbhVPc0SzJNu5Bv6OyStgC2LX0gm9Pd06y3cBzzVmPIgwDwdfFZ8U16u3tWPSWdE8JZYQI6tVffcJUPP6dx7XYXPWBzF9kleYCOgsqh9iOJaGW3havHD99v8D3FFff++70eg77hVZNSRK/gVsO7yMqpD37y3Kn3JXjdpBrbtriLPHzl2HfXzh5VqG8yVNXbnVlNKgWeECisMIGI2uOn+8vCJUv3nN7RSPOTtdS5YyDXXDFv0PGmtjZ+UL/n8YG64cfREP5y1fcdTzcwGarK7c6sGlliVmSBkNo71gpu7ZNb++SaW0HvmpsNrrnZ4JqbDa5Vw2yIx1r9EKRW36NC4scU7vuTW+rr6Br3CdN9wnTt53Lf8H+Cpjn6qoZqoWskTMSwpUEQmfBg7AdMSR0fqGKupaJrYo2XWfFBOmY4ETcm2ZCftolSAPDuI897r9Cc9flZU2rZrJKVMREjuoaBIBLhwbEfut9aeT5kUCVMy5KuiTFenq6RIZ0onEhU5cbkMyTDMpF0DREdaj3Ie6Vi4Igq+nvHIpt1kTImYlIdDgSRCA+O/dDUZD5kUCVMy46uiTNezoosG4kTiarcmHyGpFJG0zVENIObDXfU2E9Evcbb0DUyJmLSOBBEIjw49gOmpI4MqphrqeiaOOPlrMiyhjgRNyafIamU0QrABy25hU6zO9YGcMbj9+WZ30WGEzJxWjj7Ee1DJk/MtaCga2KNl1mRZQ1xIm5Mlob4qhz+n2HshZKd+QDQ6OAnFvcNQZiIwVty+kohISIc+6HLmgg+eFAlTMuOrok1XtEKJxuNE4mqnJDPkJ+2iaBr6O7t1MO7UhzOOouI6E+YY7eC3oeJWMwGGRERMRRdTc+HAKqEaNnSNXHGy1kRZKNxIklVGhNvyE/bRNA1H80hbjZQp1OJiCbiPsvZIGEiNrNBRkQEDEVX0/MhgiohWtZ0TYzx8lZ42WicSFIVxyQY8tM2arqm4oYqYTa8nPENUXlP3G05GyRMxGY2SIiIzH7oaXo+BPIkTMuerrEer2SFyUbjRKKqJOQ3JNA2arrm/s0kzAa67/ztX02fYXuleHLoYYo5G95fSkQ0Ge+kft1jKRElpuOfJprMx/l9uV+HaalsyX56873txytb8WSP5HQlInoMf9ZTFYUEQ7L1EEuewoa79u3bt6/rufvYsthtc58svRarLT5hAv9Y+2wm1me2i/MZ4sLSAzkc4ZFmP976YqARqZPyUfnuAPg4EiMtz09F78oPckSaxn68zMoASdYQJxLHJBqSShmpUGvnbQA2Hndbq4L0iyeeCBS26GTzPUUgIQNbmgVA8WEBQ4E5qeORJ8WHYaUVTtfEGi+z4pM1w4mEMUmcDl9KHYW+s2bNmjWrTvtZBamCLR9cCpQum5pt8Qnzs1Zjx40bO7JFpd2VYia2ERE9toKIttb4KxEV5XUlGnM7EdHOX5dra/I+XsN1yafIeV0pXEtli/kZtJSICnP6VqbUYo2XWZFld9d7hSjRY4ymKjcmZiipxJVSZUmsSmV+H69gUzLXEU3vXGHzZJphIubVYSCITHgw9kNXk/eRJk+SsEiYlh1dE2e8nBVZNhInElW5McmcDldKPbqGiMafkV+3/90phfUDl310Y7/d9FPSNTLhEY6hRGvKoEqwli1dE2u8zIosG4ETyaqK+gTQNmq6Rmp7F5d07pnh6Bq39smtfXLNrZl2zc0G19xscM3NBtccXePoGkfXuE+YR9EnzASOHgbiKOYpZGuJahtj4keqUrbGPvxqfNjwFD4Gwpxa8AAFn5aSDwitVhpvSDyxM/f7SYxzkLdNmmRNLJtJ6Is8Rl5Jj6fgFPz1UViJXhjCr8a34ylkBiKSWlCklchaYXyAclAe3pAYdgfRmja7vKXu0rbR9xSiNb5s0aEvCBcST4AeT8EU5PqEWtFc+zTDcjYwnkJmICKpBUVaiawVxgeoBsXwhpfqlhPR2EvSr8jbRrNBtMaXLTr0BeFC4gnQ4ymYglyfUCtaq13E1fiw4ilkBsKcWmCAgqyl5ANCGsMb5vfKAdD36u9TveVts89nvDWhbIahL+IYxROgx1MwBbk+SiuRd5HCanzLJjMQ5pAAAxQkrQg+ILh5eEPVG3UBoEn5u8kX5G37YcYqmzBGKyVPwaw+ke8Nf74uI/ZsaA0gcd1cT2h5ZQ6AT7MN1ivVACru63abT+uYLEpO6k1namvROy13zM4vvboNivblA0BdpNbRydv2w4xVNmGMstLatyuzr66rq2BWn6jZ8HHDZtXz8WtBB/ZntzNzAKxaW2By6/7hone6LKzl06rZfi8AbEOJvtT+vb944Q+ZW/s83f8AagFAFkpTn3Wkbethxi9beoyy0tqNkzJe7L7kRE0Fs/pEXCmOzB1VPZOh4pYJ4i/KRw/4o4lAj5nzCoft8WvdsfFboOJjVBrMBnxwWSZanjP28GHkAEAGylKXRWnbdpjxy5Yeo0/p2WEZGJxfoK1gVJ+I2TDrP6vpi4xXM5qLv7i10Uu5Zs/K2sxb8usqn9aF9/5ux5a7BsHgVrcWmjcH0GnLitqoAoBK1E6+Im/bDjN+2dJj9Cl1BIBui4p0FYzqo3a9sbxBSUlJ5ZGSA3FngxzWEZxjAq20ElGrYNbSd27YbZIhwBI76iXfBA4jdR2Wty2HWQ1lS43Rp6SfXpKukkl91PcNRf7V+HZNYCDMqQWBe5C1FHxAyJg9vKFu490A8C26J1+Rty2HGb9s6TH6lPR4CqFKBvVRz4a+fQFgYftZcd8aWFhHcV4u8P6KhzKAl4dCO60k87um6RQRxlPk5QLL73+qLkqX3Zdt4ObCGRU5wG50yxj0MQBsqd81qcZvxxgmX7bivNwY9fIpdRqmw1NwCnx9Ir1oXN+qDu23nQResIaXvrGnWW8px8QwrYRpJeM7/vHKDuDBlpeb2GKJHQUbtwC0qCArpca2rRqfSZIsm0nGSIiQoMSFj+gpcPVJewm3Ev1fasIn+HTAuVMsbouvKlpXe+DJZ08CcCQ9urxWbYERW2YDQEf9mj81eVVPmnzOk1mcVl6rtgCuWL/v4/kfLa5h4uy4JRMrWjzZ+n6g3VNXP9Lkj62npNXYtlU7wk5iqmxJWb4UpkKi0pjbV5696Q8zR2krcPXJa9U2wsqPtL4hKqzDJK1E0grhA9S2GN7w3b8O9ezMdZe3jYQNh6mVVmLJU3gKYfyE4yncahfHU7gGt0rWNTcbXHOzwTU3G1xzPIXjKRxP4T5huk+Yrv2s7xuOSjgmcZTJckUKrtcPZDj6ewqzmI2o2Aw/HAPT5JOYQSAAZERFO8sDRsXiZc1wIq5IgfXSwZ08K9VJ10TGbETQGHKYiAzH6NI1HhOiGwSil1aineWhJSxYEeiayBCUMLomqF4+3AkKK9VJ10THbETMBjlMRIZjNDUZE6IbBKKXVqKd5aGVXSNY4WWjQ1DC6Jqges2Ing3MSnXSNYYxG1DHZgSANjAldWIGgcCHqOhneQAGxRJkDXEirkgB9dLBnZiV6qRrDGM2EBlMIoM2iJsvEtehSZYH9IslMjGGOBFXJH+9tGgbz0p10jWMQ7Fto/50w6InPp83HyGgjXnzOJs4DhmiIsArNqxOcLFEJsYQJ+KK5K+XDrfDrFQnXcM4FNtzl/vusOUde7xVIwy0sbn4cJyNpUOGqIjwig2rE1gsiYmxwIm4Ign10uJ2mBXDESnvizRiNqIIb3+YiBg2oq3J7n20gkD00kq0szw0hDkrsmx0CIqsyhVJqFeQst8SZyVgROF3kZmaHIrtf+R1V9x/z/r+Sy5KKEAb42eoHmdj69BDVGR4xYLVCSpWEF1jhhNxRRLqpcftcFaqka5hHIrtmfvdzIZo/cb0lYsRCtpYtDRnY+nQQ1T8GIw5qxNQrCC6xhAn4orE10uT2+GrUn10jWHMRlTwBgA/HGPWKuIFgUiISpFRloc2tBMga4gTcUUS6qXJ7QhVMRmR+hJ7e41yIvo9vrC8b6ioncwDHPEGEe0tIyJai6vJ/r6hNCNrBxF1xjspvTCHKlsD5hARDah7JPVkchCl3S37TQlRSb05Vk+fJCucLK24JUFEMzbrqnJF8n7c6912HB/59IlZ4UeUUrC9b+A4FMuPg0MeBICvi8/y4A4BHIEpqcNxNkk9G4ciosLDKzasTkixeFljnIgrkggTaeJOzEp10jWMQ7Fss8aNG37shysey0nzKwI4YkHqMM4mqWfjUEBUBHjFhtUJLpYga4wTcUUSYSJN3IlZqV66JipmA1FLLeTYDB0CRampFwSim1YCvSwPPeHIYlnQNRr1CrLkWXF0jVv75NY+uQa3Zto1Nxtcc7PBNTcbXHN0DeDoGkfXuE+Y7hOma+6+AdUEriSO7hSboxT/gSldE482SXMccuCKRUJMKLgCi3Qd3o/ImuhFw0SRRAFMjEl2jYwk8cPTLB1TkLTi0DVxaBOO45ACVyITYqAProSl6+jRNT7WJCgaRpun8EgiX1iPWXaNhCTxwwssHRRWZLwpDl0ThzbhOA4pcCUyIQb64EpYuo4eXeNjTYKiYXSFGUkkMzGG2TUSksQPL7B0UFiR8aY4dE0c2oTjOKTAFfOEmHBwBTbpOp4fH2uiFw2DCJJIZmIMs2tEJEkYnmbpmIKMN8Wga+LRJmHNIiEmHFypnnSdammMJIrJEAlIkjA83dIxBRlvinEXGYs2QVjginlCjAJcsYuJCQ+A0YqGQQRJFJMhEpAkYXi6pWMKMt4UYzbEok1CA1fME2IU4IpVTEx4AIx+NAwiSKIYDBEvJA5Pt3RMIQhvsn3eYJEME9TEwBXzhBiWNuMLc7GKiQkPgNGOhglqFa1vzFx+/rfBYT12QtLwtEvHFHhTMWdDHNoEYYEr5gkxCnDFKiYmPABGOxoGESRRHIaIE5KGp1s6phCAN9k/i4xBmwTQLLBMiAkHV6xiYsIDYPSjYRBBEsVhiJiQPDzd0jGFALwpxl/6sadNAmgWWCbEhIMrVjEx4QEw2tEwUSRRHIaIE5KHp1k6ptDbjzfZzwabZBh/4wNXivNyLRJivLSZk6UwF6t0ncAAGLNomKAPYBkH6gBA8/a+gB1rITm7RrN0TEEwFTe7JhZt4nEcHM2yp1lvm4SYcHDFJl1HoGtSHg2jYaAmifxMjEF2jSAkDVevdEwhAG9SWVE/xF0/cNlHN/bbbfWktnz4OY1rt7voAaLKqQ9+8typ9yWI6FD7kUT0XN8NxZNHVhhoruy2cPX44fuJiGj8Gfl1+9+dkmLb+k+mmR/mManGOTV/Mn34yrHvrp09qpCI6BXcmvyOof1IoRRaqryQPNyg0kFhhdM61H6kwgqIKGq1SzzaBGE0S0RCTICmKbhiSdcoXtEQZiSRaYqNpCojSerSBVliCgotR9fArX1ya59cc2umXXOzwTU3G1xzs8E1R9c4usbRNe4TpvuE6Zq7b8CPiJskjjK65icL+MnWgWPMklfgJ3PkvBk/8QEjysc+FgYR2TXxsnrSnX10jdmA+e5+mkaL02H1kgWUeE7o9zBcXIwyeQXRZI6cNyMTH9GaIuWjEwtjl10TL6vH6yzTNZEDDs2ukWmaQDgGitrLAqFkk5quYRiGOnkF0WSOnDcjEx/RmiLloxMLY5ddEyurh3WW6ZrIAYdm18g0TSAcA0XtZYFQsimStUodt1mTYiIajPVm1WnalYhocv4hWlDzeaLv8Yv0K7cVEBEdbKVfcaZFRO8/y/0R9jBzytkwgvtZUHsTa4ho1Qqr2cA6TyIioqfmag9YVOW6XzaQiOjZmodUf0IeitrLAgGC2n9LFoBx8gp8ZI6cN2NEfPgon1ixMFBm18RLwmGdZbrGcMCsuwWIJNZLFlAL6izyMUxegZ/MEfNmzIgPH+UTKxYGyuyaWFk9XGeZrjEcMOtuDiJJ9WolCagFdWaDRfIKJDJHyJsxIz5krfixMOHZNbGyeqTOPF1jPOB0d3MQSaqXLKAW1H3eUD56wB8Nq8OTOT1mzisctgdBGIqhVhBNY2zOY2gkNY7isZgNQmeRrjEecKq7OYgk1UsWUAvqzgaz5BUAEpnD582YER+SVvxYGIRn18TK6hE7C3SN+YBT3c1BJKlesoBaUHM2GCavIIDMYXkzZsSHqFUNsTCK7JpYWT1iZ4GuMR9wqrs5iCTVSxZQC+otFTdMXoFI5sh5M/uNiA9R69+xY2FU2TWxsnqEzgJdYz7gdHdzEEmqV5YkECGo8bxBnbwS2ttLTeHzZmhvmRhoo6cpZcpoxMLYZdfEyurhOwuJM9EDDs2uGd2FiOie+pUsu0bneQOrFydAe8vEbbOnTzOxjYjos1Zjx40bO7KF4bPIKZnriKZ3riAatJSICnP6VhIV5XUlGnM7EdHOX5drV5xpERFV5vehlFSoOdVseGwFEW2t8Ve/2u56rxAleoyxevrEd34N1yWfJOd1JY0By6rp7rQh9yuiRM8ZaSV2VpSWWL2YQFKB2zaZDQzD6JK69TKsDiNziq6499/v9Rj0TZrwEOkRHU2B8hFwkzBzenSND17hKR7j7ym4ziJdEzlgWTXdnaNpFJwOVFQUw3HUZJMOXRNnqQVH5sh5MxHEh19TN4ElNl0TK6uHdZbpmijEJTy7JgJECrHE1UsWCBF0dA3c2ie39sk1t2baNTcbXHOzwTU3G1xzdI2jaxxd4z5huk+Yrv0M7xsSR4Ng4qhNrDka6RqGrcQJdJHBGsA8akawEESHWAim+viGFoOu4aAYH01jJMvTNZZgEpddI9VLeTLD6RqGrSgDXZRL1X1gTVjUTJSmZyGIDjHOrmF95KHFoWsYFOOjaSJlw+gaLTBJmV0j1yvsZKq/w+SwFWWgi7rsMlgTFjUTpelZCKJDZljMhlQfaWix6BoGxcg0TbRsGF2jBSapsmt89Qo7mersmn+8tOFY5Pd9cdOpcQJdGhOAWihGnKgZcJkyAeEtNoLpPtLQhIwc43swL7FGDowxlOWib+T6ycqIyq7x1Ut1MjOrmamRmwzWxI2aCaBDbATD+sSiaxgUI9M0hrIcnGMJJnm7meE52VpMTYxAFwmssYuaYRYC6BAbQdZHGFosuoaDYiSaxlSWh3PswCRvt4B6qU5mxGoeFBBR+/kJWtRmm819A9EHU04fd5BtfjSHqIf5fUPawheYRET0CaZHCCptsT7i0ErQ654q2nLCm3Zrn4ho3s1ERFR4JnL6eOuLNGQRIuSrn6gcaim9m79eYSczOgHxcIcBZUS0loioy8V2s4ESn593gReoWXFDldVsSFv4FFOSq0hvihBU2eL6iEPbjqxCIhrVqsxyNpQ3S652+/LyG3Nx7jf6sggRkusnKYdaSu/mq1foyYyeDQX92Kwci+/sZgPRnprd0otY799MVrMhbWErJhMRrcGMCEGVLbmPN7S9aJlclfyW5WxY2JyIiD7tWUSb+6N7lbYsgoV89ZOUwyx5u/nqFXoyIxntFLYSK9AF4MEaq6gZ3oJMh9gIcn2kocWiawAPipFoGgtZAc4xB5O83Xw0jfJkZmswNTECXWSwxipqBhwPI9MhNoJcH2losegaeFCMTNOYy6bpGkswie32a5mmUZ5MxWx4f8VDGcDLQ+MEuhxenfldU2AHOgEozsu1ipoBlynDh7fYZtdwfbihFeflchk5drMhlVjDB8bYyaajb/j6Feflivk40MiuEcJuivNyI05m6JWQYSsijGJ23yCDNQxnMbpvYBYCcJNAwajbmWQfplsNdA2DYhhNoysbRtew+ik4Hb8ltptM14SeTPVdJMNW1IEuEd9TSGBNWNRMhCZnQcJNbLJrWB+mWx10jQfFMJpGVzaMrmH1U3A6quwama4JPZn6dI0q0CWqtwzWWC7fYBY0cBODFSDy0GLRNQyKkWmaKNlwukYDTFJn18j1CjmZjq6BW/vk1j655tZMu+Zmg2tuNrjmZoNrjq5xdI2ja9wnTPcJ0zV33wCOh6k+8iTxIxqv+r+R9aNF11giJ0zAMoAFATyMLxbGTiuMrgmCgcyza2RTAaZ1hWw5HWE3nj5S4jk6dI2SDQn9FocJaAWwRGomeRg5FiZMy46uCYKBLLJrZFOy6ShVVnAtTgcKKzJ9FBqjo0vXqNmQ0OowAa0AlkjNJA8jx8KEadnRNUEwkEV2jWxKNh2hyhVci9OBwopMH4XG6OjSNZbICROQCY+aJ1q8C6d4GI48sdcKpWsCYCD9xuokm5JNawvZcjrCbgJ9pMRzdOgaS+TEE7AMYEEgDyPHwlRHYg0UMBBssmsQjssYCllyOvxu4liVeI4GXWOLnHgClgEsCORh5FiYeGo+1sQHA8EmuwYKXMZMyI7TEXYT6SM1nhNN10SwIeoL9EoUBBAeQX86W60pMDQeeaLQsqJr/DCLgbBYJ58pybRCVRDS4XSgsiLTR0F4jgFdE8GGKMt+uMOAsgDCw3g2CAwNR57YzYZQusYPsxgIi3WSTcmmFaqCkA6nA4UVH30UhOfoZd7d2uil3FiBLrc2einXOoCFa0LGjBALE1PNS7FhT2j5lB1YZ9cgOIzGVEhKvdE8F9xuctaPMkZHg66JgZwkBawDWBDAw0AiT+KpBbImHMwC6+waqHAZfSE7Toft5qOPlHiODl1jjZykBNpaB7AAAQyNEAsTU01iTWSYBdbZNQjBZQyFLDkdtptMH6nxHB26xhY5SQv4CA/jJjA0afLETkpJ1wgwkHELrFPKpGfaTMiW0/F2O5mvXHFerhrPCb9SbBq5d8L4cZc/2QLja70O0OtjTjYrDhMo2LgFoEUFWcCeZr0BAOUoNy121aH9AIBdyAfApGy00mqXtAWwbekD2Um1OgPfbgps39C3j81sEOqUMpU2mTZtKFRjyIMA8HXxWSklzXMh7paq3J5mvXlBo0+YXCiMkg0JvcfmBCTCIzCAJUrTY2jEWJgwLTu6hoOB4mTXMFNpBIiF0eiosoJrcTpQxuh4lVPnBunTNSo2JENnqYUeEaOnKcfCVC9dEwkD6WXXmJqWVDkhDU4nyFK4lZAYHUfXwK19cmufXINbM+2amw2uudngmpsNrjm6xtE1jq5xnzDdJ0zXfor7hokrcfTBJYmju5byMBM/QL1+WE4n5DvM4iftv36Gj67xwSV2YTgpRkTCS4KybKCdhMMhRADi0TXSMHmkheasz8+aUstCSCqVLqcTnl2jxHOCv/SYjYYVpNc06BofXKIMw1HTNTJeEpRlE/mtlXd8DiGKT9eIw+SRloqBI6ro7x2LzLNr5FIFcjpG2TVhqJRiXeRlw7E47mxgdI0PLlGG4ajpGhkvCcqyiZwN3vG5gJ74dI04TB5puaPGfiLqNd48u0YuVSCnY5JdE4pKhc+GXRNfw6i4s6FZk2IiGoz1NImIiJ6a6700wkLz/Wd7DCKiN7GGiFat8ObtQCKiZ2seMpkNI/wek21BzeeJvscvrGaDMMyU3WRr2pWIaHL+IT1VXmiE4hjhlm4rICI62MpfH7l+Gqtkn7+0f4MXg5aQVG3dj/KX/g0juqZaiJg0IyLhJTHJHTmgJxZdww9TQFpKduYDQKODn5gLwYrTCc+uUeI5wXeRy67JuvSxxRcBwKNP78obW4Bhy+vdOAoP/2vwe7nNBp27O8uErgmAS8zDcFKMiIyXWJI76ePzAT1APLpGGKaAtByTRcnPb5vONBaSSqXJ6YRm10TgOUFvNFsmES3FkBSp0XAMESXa7yZ6oWYJVbaZTF+Z0TV+uEQZhqOka2S8JIjcibYlHt/zGI+u4YcpIS2dTiUimoj7zLNrAkrl43QMsmvCUanQ+4Y7PyCqapqXKsqUuoeItt5DRJd0IKIhTU3pGj9cogzDUdI1Ml4SRO5E2xKO73mMSddww5SRlpczviEq74m7zbNr/KXyczoG2TXhqFTofcOrG5966pmW37+a3Bpb+jzwzBUA6hUDyK5rStf44RI/0wJNHkbGS+zIHeH4nseYdA03TBlpufDe3+3YctcgAwbBq5e/VDqcjsfQyPVR4zlBs2HjKc2aNm06Gs+lLlW/egxHSo8DMHHPehx4ZwoM6RpAgkuMw3AYIyLjJVbkjnB85hHx6Bo2TH+gTsGspe/csNtgYX66XgGl0uF0QrNr1HhO0F3kgoIOAOiu10rqAQDGD1675QIAOO6ah4779i+/gSFd006GS4zDcDhGRMJL6tqQO/zxmUfEpGsYQxMQqHPiiUBhi06m2TUBpdLhdMKzayLwHP9lJzE4+e90pCItjpwwYVKCiGjuy0bX0RW3JIhoxmYiWourU7/cW0Y0YA4R0YC6R8yuzccPIqLba5QT0e/xRVKKRnchIrqnfqXB5Z07PvO4t4yoNCNrBxF1xjt29w1smGm7SZPLflNCVFJvjraqJ8RZ3VvmP0bYbVbt0uRzlTeE+uwtE+qndRc5L/XUczV6pZLVptd6kIiIlrV5/NlFS0o0q8PCb7gsFnV+ilIzmTbDYmCSYS5cNov2SWPHZx6TalzUTpzsGmY3KTslcx3R9M4V5tk1criOeAyL7JrQGJ3A2bC8W51jTiskooKOtfJ7XUtERDvz9xARUelpzVs1y82+7KApXcPgEnV+ig5d43EjKdyEkTvaJ40dn3msDrqGZ2hSdpOy6wcu++jGfrvJPLvGF64TxOmYZNeEolL6dM2OZgBQ1v2JM4DEmiuHTDNdaiHDJaownChNmRsJJnfUtsKPH4uuCWVo9i4u6dwzwyq7RrYacAyz7JoQ7saUrlk0O4l5P/Laq27t0//7tU+d1xcCQNXr57lVQj/PZrQuctU97brnffnWWRMy3HvDz/K9wXCVbNHWsqYtM90qWTcb3Jppd9/g2v+nRkR9XRVcQ9/U8wbXXIPjMF1zs8G10NuG/wUK7geaz1AcnwAAAABJRU5ErkJggg==',
    '1705.02407v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAtwAAADFCAAAAABChHrPAAA3YklEQVR42u3dd3gUVdsG8HsTIJQ0EnoCijQVAUGKSFUUBEFFlCI2iqJBEUEFBCFIU1CwIFIiioLSBEURUBA/AUFAaaG3RIpSkhDSSXbv74/dnbY7s2c2m8CL81yXF1P2PPOcs8dldnbmh42wworrM4KsIbDCmtxWWPG/FiTHWaNgxfUW40iWAIDr6rw7Pt56Z61APAAbnf9dP2GzviJb4ZwG1jm32Mnb2fNEtvW/jV+RsecqHbiEx5bkcUkFPYc4l7MeuxzRdoTmBX8MOxE/yGuytD4HOnx2TY1r4vR/bKg2tsbknTkI6zhQ51U+616+qsWlMwNGrSyn33f9EqamokS1IMeFHDz+OHYOPTF68Lo3//6ku1Dj+T9m2yuNrL/myzRGDrgPCYtL2cfdBQBIWdKnvOn+izVStcT5px3BDd4pzJuwYJ7H7BYfgSfzEX4lF2WDMnHj27qv2jn0xOjBXr9Qgur4vGV0jnNpQfuIAuWef/aQZGapj6gTLXrxaoemN2/jc5LMK3OzUSPjutc+YCcLXkCGYd/1wvEWFpN0HLt/IMns0jPIU1gm2Ph0aEMHySu1y+ST5OY6F1w7PsAcoQSq/os2Urbk9I7HCveONMcej22iI5AZ+m0er0TUtzP7kxoeM1GO7NIzPKeBt9OS6P5pK51LZ2Mig5V7/toLAOX0/++Pveb+ToxGNACUCo82epVx3VOeCgKC344y7rvu2V9VlANgq7UwBUCZaFPDFNMt8SKAko/k/AEAZeMruHY8MaeX+f6LNlK23HTux1qFegeO1MKXfk+Uc/0fKoWSpUODUOb5htTORDnKRIte547tMhcAsO82zY6V/82TxiNHACC8Y6GSZCG6ih/NejpWAMBFLAOA77u5t0c9F2E+mR+NVv/flODCjd5X79T/yu5v43PN5eWG6SZnovcvlM/+egQA1t7vOnU5+s1eO5Cb8GluZiYBgIe2Z7g+3FdtzXEuFexNvOa/cbl6AmSv++YoROuuNysRAIaVgV7fBWIM8LCykgPHhEarU+gyAOk3NVjuAJgRlpu0Iz1rZ37B6d2nTffe2Sgv+c/zjgOHHGIz8/gYGwBcOfXXOR53ddw9itJoGn8Xr/7k2Y0eOQAw115Q4KN1ZAt5+QG7e8jlmah4H0Und5dqCQBySjq/bqZ2/6bhxs5pWHWE306cmA2gYPL+E/W3A8gZMr9mSp81ABwfP5h4cMg/1/bcdvcEe58pf8uYQQ7Buseea9h2zGZ7s2B477tI/JgBKD76fxuV9GubXwXalXlw43lgVbeeZ7YCuxpj27PNf5s88Z6koY2Xm+6+s9G2uKZTR/618pb1Ai1mvvaM6/vli3f8EL/b2XH3KEqjaRhb7kIf53mJMgcAZNcv0XmXj9a31JaX74p2D7k0E+X3Ue8XSu0Xyu/XcEzFPPKrJPa9gWT3p0n2GUbm40uSZOUml8h7e5EcFEcyLfovctptOWRmzDX3hXIeugwePHjw4DKtlD15uUQWs0I/p2Ddv7ewAXX26/XdR8xDl8Hdwwa41mJmkMRAB7m/1B6B/qzEJ+SrjkN4mRx3mbyMnldWjqaj/Ayh4VD139koB3EkVwbv9tVy1TtjbU+51grC7nJ33D2K0mgaxiuXyfblMjU5iGW81OH/BN/Tyi2cf0pD7p6J8vsYI/iFEsCAi98CyTcAAPaufABAu28Vu1tEADX/BY7O6Q4gst1IpI8eUBood/M1+HH9wsyZM2fODFf1JO6zsihbdycE62657eLSfsntLnnvu0gJK3baVFvusgG3Nn1D8LwkPcJWr9FyBzPDgDA0KPnwRNjKmu+/q1FpNAPwUG2fR/+g+fiRXyxzLgeXbeTquHsUvcwLL5F3JQx4MutbdQ4AOPHS523NvZOeQy69j4LXuQEAN3ac2/NQPdc1RGxNBs43UeyuDiAoF9iOKgBQZZFj55U61/4XQ7kndTnpXNSFHECo7vOVEPXYYwPbLhnkte9CP4TVvRE49/MT0iUUAKi9mjbf5yUPfX3up25Az9FbQ5zvQa2ADIat0QZfR+/XHvHrBrWMVb/p7lH0Mi+8fSE98yJwJeTLvqocADZPdESbLFkecmlU3e+jmcmNZx89vtr1S04oHm8q78g8W9f11ji/owCAnQCCr/3JLffkkzcWd7StdX4k+W43/mMAuKv5UWlaavouEqOB7RU0X/BKijR8bNGK408APUcvjRwOACgZmNHI95knAii1qMnTPwdB2XH3KC5VzQu9+GlpSQAZS/+toh48lPr1jlc/Nv2VWDHkmWfrKt9HE7e8dqv0cZCr83cG73J+7MEGB85tUv51bTsKAIdbBTUJOQQA9mt6cks9yR4+opMNWchZJlT3FufuKqqPTLnvwjeeLa0vLToA4FAbm8Bda53C5ofbgNpNlmeEe35l8msgHADsu9rYBF5683u/zPA+ivK8MPoKH14SAJ5wfK3d0zFyScJKc12Qh9w1E+X3UXhyHz8OlOr34YMAcnOJGqM+yQFSPwWCa53CyRvgyM0BkEPgpldnFwD7dr6H8uM/zwGStl++1iZ0LnKdl51yIfckPacicPp8Tua/QnUnvUoAF/c+Cq99P1Rtro8SnP9zLPshFmBuDgBsILD52FT4bAyUfnBnNwDoebYpAOS4O3QFIq2V/Xc1An4msCB1qlDL55u9sQXKjrtHUZ4XBjGtoWsmf+5Q5kAOctB0WP/9Il0A7M4BlIfcNRPl99E1rL6uliT1iq7Y+28e60GueLxy6MNTaZ/e4aP3hqeSXFZnzpD8HY+F1h56ZUC1Cv1PsGBq9zkTOv1J0v5hty8/eP2ukMcuXUtXS/b1qxfWYEAyJ3UNjXhsntyTqXXeXTDp95teOClUd+0FfRf+sfy+X0ivfT8ck2BQz74nq6NJ3OC4xxqhOflnz9B6I8hGf41ZPO6B/fTVmCT5XS0HSR4vnU5yfdfQ2wZc5J89Q2+e5ru1uv+uRsTA1xePunu/UEt+Ui20Sq9UZcfdoyjPC73Y9kB45fdI/tE7NLTTdEWO9V1Dm33EB8vGxvsegK8HdAgNf+C5n0jFdFtWZ86QfPl9/MY5rJppIHbLq+NERedPW+ln65TQ7op2/SLtOB5e+USJqHI2XMO3vLp7kns8qipySwUJ1X3g1is7TlRtEeaZLLq8v2XyZLTor4X2Pc4vbTubBmqIPnta/OhGoyjNi2ILachdM1HxPnpOA+t+7v9g2D57xrqf24rrMk7gJP8zDyxYn9z/qdiwJKSgyrj/ysS2JrcV1mmJFVbAckussOLqRwkA7WzXVZds1rtqBdpZZ6hWWKclVlhhTW4rrIBlBVphBSwrEJYVaIVlBQLWV2QrrB9xrMB/4zo3/OD1ICC1XUNxcFtEhwhgy9lm0SUBlHQ/W7Zra0SX8iYSAI7fDka1U+g6Qhn2/VHunsqqBc12kcYF33UMw/k/O0s7Tm4q2TVMuO4t+2NbR0rbNatF1G9NDm0jeYevL4ZnkirVcf9+kXWmrmLXgXrBAA/VCxKkHcR4PUOp7dqhHUgy77lnT21rv4d0nY4HHXApfnGvJu9q+5vvnFICpg79+eKGqrMkB1Akg2PcitT9T36pWNBsF2rMNATHVi0rEXu5L0w6e6RrhmDdF59fviO+4lL3Ds1q0fRbm0PTSN7hK5bd1Gx405u3kMw7vW1S5WbKfSFRbbvfU6lRvrdp4G1yz8P3Ti2ilfCMirmmJ/fYBnby66YO9n1lwrRp0/rEu7YvaEZyW7XLPnNKCTjq7mwyDmfMZPhiIckrN6fJC5rtQo2Z1m5Q74mnpenVdTS5GJsE637pPMlRIYmuHZrVoum3NoemkbzD15NIqJfHnBtDj5Erm/VtgztUkxsAmiXzvzq5C8oNIfkv1vFlkkzv55Zr24wk6Sjn0xuVE3BK1DlyBHaaydBvDkk+sENe0GwXasy00crtP5TJIFPm5YrVbY98nuQevOXcoVkton5rc6gbKXb4iAYYSPIZPEeSnKCZ3A91HbDB4X0aCHyhdIl4V07v/hs8f2APXFwdACb9kcE1C/Ndr3TDbdyz7JfLSfKfVzmOZ4UDqBS0FYMBYMx41xl3xtYYALaIdeIJMPJCJWD7jQ3MZIh9dSVw4dRt8oJmu1BjTYzvEApEDQwRq9tWKx9AWbg8HM1qEfVbk0PTSJHcOHL3IxRABaz2uvvd7xPusfn5C6Uk4h0Y2fgrFIxqOcjN1QHpXT490eWJ8t+4nu53w232gWfalu36g/Tn1Y4KsAOwIxl1AKxuWt21/e+CUAAIPyyeAAgCFu9bWspMhhfLP9L9t5cWlpYXNNuFGgN57781frbrQmf6ztivxr4zKluwbtvOBAA70dI1udWrRdRvTQ5NI2VywyhVFgSQhTN53nbvfeOZcT9R2ApU8XKyiJcfPIXk4y0kro7PdyI3lDhtd52WuOG2tY+SXP+R9OdVP+du3JvkX+hAkjmd7JIAiC9IslkjnzmVCRY+W3Wxw1yGlMbAVLtyQbNdqHH67cl0PNrfufUAWm4iP2qeL1w3aW/WUvlPCWhWi6Df6hzaRuriDOJh9CYd9YGz3k5L2v56fBwevOLPOfcR/EySj3QkWWkKyUEtSGICSbLJADIJq9zn3Ie/JNnkRW4p9VaiPfeo9OdVn9xbq5yhfVLowyT5/lBp8xbMJcnGDX3mVCYg05r0zDeTIfuZT1bGoleevKDZLtSYzCe53vUV8iBak0z18S8lqOse3Ur13VWzGvh+a3JoG6mTG8TfsRHbL42rD1z0Mrn/z046wjDRn3NuN8/2h0N1o7TTXmp1BDgW0ti9sW6zSUPiL+Sg5agJt1UYXU3686rHneunThnTLbMeAMyXzctwZAJAZoSpBEDkiKWzzWQYeNfzDx8YtOQTeUGzXaix8zeJmvgZAFADDQGUD9soXvfSP9cpL2xrVoug35oc2kbq5AZR/cCIt3vV6ogyXq6r55QhYIvBt+KnJfIn90LsIslB4Xay8hSSz7Vw+rMkue+mEV+0XS5dLZkVudbBFgOYlZ+yemT5OOnPq/7JTZJMxCKS+7FS/gsf40myylNiiROxiI4p60kmorOJDOejCkhySGdpQbNdqDEdd75GMgWDnTsqxZFkRCexukmu75tL7pO2alaLoN/aHF4buYozvlL+3WmSfXC/l6sl9+AVki1woz+f3AoRDyEEcE6xc+3quEbrerjXJLjt8TVRXaasWIOVrj+veuxeDGBbzUcBbENlCdmLanARgD29o3iC5FFvA0hHRefdZkIZcsoEA0DnCtKC8/DKVd+NwdMxAE7jDueh2xwHUJDdSLTjO75bEIKzS9x3yUmrpvttbuQUORSNqCnOOD5+qBfA3zBEtTWrTZujQAbKAPgXzUQ/uT9yfjA7Iu4g+do9+eTe0AMk73yRTKl2C5ntor/f77V647a/STqiJ5NnkUCeqvT4+eb3O8jETvzS9edV/+TuG+lgXu0lJPkaDjnPWavMIT+tYyfXVfUNwEkJ7E0Pk5xi+9WZQCxDk40kOewnecHZWl4VacwRKSTfqJflbLy7fBr5f6FJYnXzwC1vjhs77N5FrrqlVdP9NjlyyhqkRs4aFO+KLwX/JTrG4kWH6pN7BTCZnF0vhVyH8KNiXyg1vJ7Ms/Hn2z9fNCoOD811cnXk2bDoyuXQYr8LwHPDbRMeHvV1wtOH+aXrz6s+uVf3S9x8/3Kn8w/nb3yHYxJIx0vPJW9qttl3TjnB3udXHFxYY4Fb6RPK8HeXDxO3jZqrWHC2lldFGjN1yHeJk1sedh96dvfd61qvFazb4QJq/3J1XFo13W+TI6fMITVydkDxrvg4Lelza9/ba690kDxSsWqFsIiYypPIf1rGHiUd0294oH25Bw94nQYCt7zKIt6VE1GVzuZFh7m+WZ54eHld8ML4rX+5L7c74bascvZjkVVsyHD9efVveU3eUqFVOQDAmT+7KQtK+qPUfaECSeUE9j+OV20aaS4D/zoY3SJKuaDZLtQYjh1Hb7ld5sRPbY28M1y4bv+icP32qEHdSLy4lNPVdYfJfgI3lgw8yjP/hxUA8E+tzCDrfm4rcF3dz9315I8EMse8Yt0UbsV1ZwWmL0gKxeVuHawncaywrEBrclthPWZmhRUIiFvSDtebo2UFLE7NOi2xwjot8RXFM6GsaWtFIDi1S0dN/KN7jkkHi6XYySetN8yKQl4K5FeLbq174NDgroK/Lr5R/QWzx01Z0sf5s2danwMdPhNslPX4vEr+nZbo0Q7CuIAecSBGJEiEg/Z4qbt8X0dV+A9azGFejyhhVkH1cg8mwjTtIKRSGPMSftEO2ScybqqknpdphyvWDPI6DbzcjZHX+97zJJNbxNmF7ub4tYfD9B0gH8i32bfoJdxq04MBpR2EcQFd4kCISJB0Bs3xUpZPuOkhX4dW+A8emMMW21HRujUv1zAR5mkHEZXCBy/hD+1w5ZUaXRui4ffK2df4tpd7t80QfRLnxYh/SZL/hk8UmnJNdpi/vSlljnRHWQ/xyc3O6wJJOwjjArrEgRCRIOkMmuOdW5PSyufklmkHD8whuwOOitatebmaiTDfbyGVwgcv4Q/tMBxL6ZiOoNXSzrcwqICvYobg5N6Nt11Lw0JOCVS/p0XhbuMzM7nnPx5A2kEYF9AlDsSIBLfO4OV4vie3TDt4YA4TF/mY3OoDKl+uZiLM0w5CKoUxL+EX7XAr6pP2GNwjPwdbMZfsjKmCDyvMRVfX0v15X3gTHTRiw8L7AFw59dc5Ht+eAaUGIZ0yOQWILCkVCk7vPg0ABXsTqWYhVIn++f6HVADg0W/22gHgwXVXEDDaQRgX0CUOxIgEt84gfDzvtIMWc9hyYzXxukVeLt5vMZXCmJfwi3Z4utozQFB1HHPvnIFGIcDCn4YKXi3ZhJvgflLyNy+ig1Zs+KkpgMQX7/ghfveJ+tuh0CBc4RYgvpBSIWlo4+WA4+MHEw8O+QeQWQhlorxhcZGR/VcBqd2/abixcxqA6DKbEDDaQRgX0CUOxIgEt84gfDyvtIMWc8he8biZujUvVzIR5vstplIY8xJ+0Q6vn3kVyD0oD/gGxMxs3W5So5KCT+JUDHKfBp1HQy+ig1ZsiE4kyYKwuy6R9/ZSaRDOkAQIORUd5WeQ027LITNjeilYCGWi1+oXkDd2I7s/TbLPMJJsNjtwtIM4LqBHHAgSCW6dwfN4vk9LpMZazOGtM9zo65xbcUD1y1VMhD/9FlIpjHkJP2kHchqqnnR/7QBqTsk7Ujl8p+BpSZjD/elwCeFAiWgACAMQhgYlH56IsFUT9jta3y/9rZESAQDBZRtFADX/BY7O6Q4gst1I9yu2xwK1Cv4KUqSCrSyQPnpAaaDczQAQ91lZlK27U5nowvR+wcCS6di78gEA7b4FgKrn/bjeOevXs3CsCQ0DgDm3BHndLpyg79wDU3sXKHaOLfVjsGHrnOHPrYx9/fEr4sfz1tiG4NZA3+3znds316xmpuOal4fvqAHb8/M3+9tvqSpzg68aLOHR+DB2zY708QCcf29sHdVi542uXflAiRGl6jx4eRCECGPcfuLo7c6lY7gdnqJDy1ETxpbv/5Z722W4zgKrAwjKlTWIRQ7XNGq12y1AqK9P7rxSR1quy0nnoi7kKBPtstcC0BzYia3JwPkmABCZCb9oh8oZfUY7aYc479uFEwCRI3q1eREKIsHH4yQD2z+LDq/Nafmy8PG8NZYwh+ecJyXvmem4x8tdTERrP/stVWVu8FWDJTwa1Q/MfDurb8f9TtphT7dnPiq12zVDw4LtTWxARfyVU0ZocvdZscbVdA16q3/5LgkAOWOGbNs0J+dj157yyKio+fcfCwDALp3RPffQyPoJi2K9/Iguf+J98sbijra1qn9IsgRcBmEoHm/qfl1GY39+qqr/PrAfDQEc2FvZ63bBBHynWQegPn6UJveGVd+GINHom9WFtV8AYbND1r0sfjwvjcs6f78KSgMALDn1EvA34iPfCxGpW/Ny3tVmKhCJFD/7reiSmcHXDJboaFzZ+NQoYA3aBQHY9eB7T+NoH9dP4rZ6B0IBBIF5ZYS+UD5y5zTnACbN6d4KnqKDRmwoGZUGXQ0CgFKAUKdqEnIIAOxKFmKZnKdJyAEA2IM7g3cBwE4ASIlBwGgHcVxAj3YQIhJknUFxPNHbZBT+g4w5EOi3bObMmT0RPzNEqG7FywkomQi/aAchlcIXL+EX7fDXvd3z537wei037fAIzgLIQK1IQdrhdO0+uSQvd2ycRi+ig1Zs6LaAJO0RY0k+1VytQagFCDkVHdHvkG83yiZPlu2sYCE+VCR6t1IqmfEKOaZxNpnyPElH5F8BpB2EcQFd2kGISJB0Bvl4ztZki64+fyGTaAcJc3A35jzsE2UVpJc7G0tMhJ+0g5BK4YOX8Id22OGaw0PctMPF2IhTzGuApaK/UPJCvxZf/JHQ4GXnb5pa0UErNnz6Askdj4XWHnplQLUK/U8oNQhSIUDIqY7/2TP05mm0f9jtyw9evyvksUsSC3FSkcgxq8Wsaa+nkPbpHT56b3gqyeO1HQGkHYRxAT3aQYxIkHQGOc3hmATyn+e7R1bo8UKGIO0gYQ5OGYEn+t8SdtcgwbqllzsbS0yEn7SDkEphzEv4RTu4r7N/4KYdeLR95Z4311xshnZI2ZxSuY3bDdCIDlqx4fydR0roahBQCxCaVHAcD698okRUOZubhQjS5KkU7lqoGAEAkwvGBpJ2EMYF9IkDoft+3DqDP9KCwn8QwRwEaQcNE2G630IqhY8aAkM74N9/YiraiuwZyn4PPFpMAkRBg5+qWw8rWIHie1jh7U+yi0mA+LB/detts6JYaAfpB/sF82zFIUD8Ov/zIOsxMyuKl3bYU7pecRS89NEg6xlKKyy3xAprclu0gxWwaAfrk9sKi3YozrAmshW4PmgHWNiDFYWd3FzUZeLGsfd/L/q5OCbqVs2WlFny3VR/tKo6x3cOuUXa/TX6iZY/dKjY/d0HP1uRDgDYl/C14h4wx6bZP1wylwAA5qWq9mlWDVunblBuV1ejE/KLVEU4fv1kyb/+1r1lWVJGbm5url20fcE3GcB5xb9vtGvWojTBN0mqQV2yR0qDv6BPbz7imoz5R7YkFejsE7lxKgC0g1Nu+Mf53H5mqY9MWQ+Fwx6MaAeNSCAEMxgSCeaABY3lIOQjyC9SF6GxFkzWHa82Lny311gQjrhXk3e1/c0c7aApWZCXUNEOXNKk7yvVYhO87isu2sEpN6x2vXGVPzJlPRQOezCiHTQigRDMYEgkmAMWNJaDkI8gv0hNIWisBZN1a4wL3+01FsSCZiS3VbtsinbQlCzIS6hoh93BT9j5PfCLl33FTDsMNDG5A4Y9GNEOapFADGYwJBJMAwvKyS3kI0gv0lAIGmvBZN1q40KgvcaCaDOSpKPcssLwEGK8hJp2mAcc515IM1S5rwhphwJHnj3HXpCHvIKC/ILTu08jN+HT3MxM5+kQD7nVB6X1kC8nd1sPMvYgZj2IYA8yIqAWCcRgBh9EgklgAV7VBgjQDpo0GmvBZN1q48J05RlbYwDYItahEDyEaChph26dX6iJvcDdXvYVHe2wrFFwz7NvRlX6Agnh9TYkDW28HKuO8NuJE7MBoGDyfqf6AJX1sFNO7rQeFNiDoPUggj3IiIBaJBCDGYyJBJPAAryqDRChHbRp1NaCybrVxoVIe5UF8XdBKACEHwb8ZjEEeQk17VD5x1m2Q+OC323uZV8R0g6JWEe+2Ypk/7MuuSEf7tOSJm71QWM9KJI7ys+gEnsQtR48sAdD2sFTJPANMxgRCWaBBbXlIOQjSC/SUggexoSZutXGhe/2agvid3xBks0amaQdVCUL8hIa2mFfs7IYm6HLPhQN7VC/6SIga+tJFFSo6pQbFNHCpT5AYz0oktvKQoU9iFoPItiDhAh4EQl8wgyGRIJJYAHe1QaI0A4eaTyMCRN1Q21c+G6vtiCIXAAooEnaQVWyIC+hoR1u25427a0Ge3XYBxQR7fDUqFlBIe0WvrnhXs+UbvUBWutBH3sQtR5EsAcJEfAUCXzDDEZEgklgATpqA0RoBy9p1MaEKdpBbVwIVK6yIMKRCQCZVUz3XlmyGC+hoh2YHolSwyYn9d3nlX0oMtqh97CVJbvVnTRmwxRlssyzdT2msNZ68Io9iFoPQtiDCxHwFAl8wwyGRIJJYAF65ANEXAhVGk9jwkzd0BoXPmkHtQVRDRkAkNES8JvFEOYlFLQD7/m/t8YgKCYtMTXKC/sgcM5tv7N8KknyZKnuJFljMsmHWpDEMpJcuIrkxprS66Nc13Yeum+o43KZNeNIkjEzyAIs4LEEkpWnkBwkXzGcdjB5T45zUU4eM4NMDZlBku17MavMFJK3DcheKjdPC4knyd1MDp5Lks5raG2X+jzn3vU1yYSaeckxJMk1T5J0kOT2lwrIM2N8nvW5EzjXvsRRdwJ5VbC1+5zbQSqr0Q3Fi+Q0DvIk7iW5BU/5W/en+J3i/bbHvk9yD+a7EjR4iWRBmYU0kUNRsoNUpTSO6WhFOmLwI3OC0ZRkdUTamdm69RHlPrFz7qDl0YPzAGQMqj8fAKqdBVJ3XAZynGdanEWgYl35JGO/88+n1leyhT3SvwcAMPcKEFzrFE7eADhycwDkyJ/PJeMTU/ecgio5c68A5cd/ngMkbb+M9JyKwOnzOZn/ys0jJ81KAzIXoMaoT3KA1E8BgHtr+/xf/90XiCtvv12qRuVfAeDnJ4FD1eYCB5+OnDBu+NO3CCdwfctAriuBvCrausB5AfNQtblQVKMfihdJaQ5Vm4saTT8G8Jutv791H4LA471S+6C+TwJYUq+XK8HQtQ5gQ2RXmMghl3yo2lwoU/qIMDQB48+8eD9Kdw16FdhyChOD8NPmzcuV+wR/fveHdiCZG32cXNeYJJ1yA5fVmTMkX6U+aK0HKfkyZwsZexC0HjyxByPaQRYJDsckCMIMhkSCOWBBthzEfQT5RVKawzEJSmvBv7rdxoVge9mCcJoWLz2XvKnZZhaCxRDlJVS0Q2ZclQe7lqvztcNNOyj2FS3tcL4SwIsVFWnSz9bxclKvtB70sQcx68ETe7AZ0Q4mRAI/FQIzrYWqkV+kTiNkTOjXrTYufLfXWhBJf5S6LxSFYjEEeQk17ZB3JrNmmAD7UEy0A4rQevCCPdise7ytKDbaAUVoPVjYgxVXl3ZAkVkP3rAH65Pbiv9F2gEi2IM1ua2waAcrrvfJXQJAO9v11S3rnbUC7eCc3O1/vZ56FR9vvbNWIN5yS6ywLgVaYcX/YHj+gHh0QrI9vCZSw+95LAg7h54YPVixM2VJn/Kmj+FXo8DGwW0RHSKALWebRZcEUFLxu9i8HlHiCQDHlv2xraVf2bwl9BL7/ih3T2XVgma7UGPg5KaSXcNMHVmnbu2qQPvfDka1q2Ky9MDUYJQDu7ZGdNGdXV7uxpiJFaQjuXXzdDK79Aw9gEER/+zxfleAc7tOoyIKI9oh3lM08AkzqHkCjQYhRCRIOoPGcjBJOzD3hUlnj3TNMHFk3boFUQtdlkGo9MDUYJhDn5jQox2+xhqSPIlnXHeiegcYlDfH6HTTuV2nUfFNbkkX8BQNfMMMap5Ao0EIEQmSzqCxHEzSDo6uo8nF2GTiyLp1C6IWuiyDUOmBqcEwhz4x4WNyM6yyx+T2YTiIbi/eyS3rAp6igU+YQZ1Aq0EIEQmSzqCxHMzRDvyhTAaZMi9X/Mi6dQuiFvosg1DpAanBePz1iQmd+7ndkZtb073oZhYKTu8+rbYWACgMB5e44LIfXNs9G7llh2I64ZZ1AQ/RwDfMoE6g1SCEiARJZ9BYDuZoB4zvEApEDQwRP7Ju3YKohT7LIFR6QGowzOGDmDCY3HNs77iWJGYhaWjj5UprwRmS4eASF9z2g2u7RyNJdiimyS3rAlrRQABmUCfQahBCRIKkM2gsB3O0Q/rO2K/GvjMq28SRdesWRC30WQah0gNSg2EOX8SE19OSl+fNfat3l53u58VkZsFRfobSWnCF23BwiQuS/eDarm0kyw7FdM6t1AVUooEIzEAPVUGtQQgQCZLOoLEcTNEOB9ByE/lR83wTRzaqWwS10GUZxEoPSA1GOQyICf3Tkrr33td/0WrpH0+WmQVbWSisBU24xQWN/eDRSJYdiimUuoBSNBCCGTxxBrUG4ZtIkHQGjeVgjnawIbg10Hf7fBNHNqpbBLXQZRmESg9MDUY5fBATJbxurXSjeq4rmAUAHlSDK9zigsZ+8GikkB2KJ5S6gEI0EIMZPHgCjQbhm0iQdAaN5WCOdqiBhgDKh218TvzIRnWLoBa6LINQ6YGpwSiHL2LC62mJ4ttnzAxyVuRaB1sMcK96PM2ejy+ZcZhL4Pz2nJWfsnpk+Thpu7bRvptGfNF2eTFeCqSTxFpEcj9Wyobmo4MHD+6GvoNzhRI7E3B931xS/jfXlQm9x/moApIc0lla0GwXakxWiiPJiE7iRzaqW7Pqo71jynqSiTBTeiBr0MmRgvEkWeUp01dL5E+44SM62ZCFnGW69+E5cG4T7gzeBQA7V66J6jJlxRppuzbWro5rtK5HMf4+uXsxgG01HwWwDZUlK6XfspkzZ/ZE/MwQEwl2fLcgBGeXSNiKlFD/xKJMMAB0riAtOA+vXPXdGGhzHEBBdiP3oX0f2aBuaVWsffKotwGko6KJ0gNUg14OEIhqcBGAPb2j+NWSHIVVwNwcJbPA3CteqAaX4eAWFyT7wbVd20ghOxRPKIQDt2ggyQw+YQZ1AkmDECcSJJ1BI0uYpR3e3H4J+D0kzn1oAZxBt25B1MILyyBeeoBq0MkhREx4/EV+pP+tYfX7uzDvP3uG1htBiVn4s2fozdM8qQaX4eAWF2T7YVmdOUPyPRopZIfiOS1RCAdu0cAlHAjADKoEsgZhgkiQdAa1LGGWduDs7rvXtV4r1S5wZL26RVELLyyDidIDU4NODl/EhAHtoP09xxuzAC+Gg+NExQil/eDNdlDJDsVzy6usC4iJBgEmEiSdQWM5mKQdcGpr5J3hpo5cOJLCgGUwYWQUtgbjHHrExFV6zCyQsoN1P7cV19T93IGUHaywoohpB5MRONnB+uS2wnr63QrrtMQKK3D9//PYVlhxXUzudrjewAorYLkl1jm3FdY5t3jw/N8Oa2itwDVJOxQqHO9tvy9sTIe5hp7DVaAevNMOgjyCEXGgFQ98tPaECHzLEm5EQV1twXcdw3D+z87+1m2WdtDUGhCWofA8hI930OAmUYHQPkYx6r4CZlfoZew5FC31YIJ2iBfhEQyJA82qj9ZeIAKfsoSMKKirTUNwbNWyy/ys2zTtoKk1ICxDAHgI/XdQ7+l3E9Fes15pIckevYw9h6KlHkzQDkI8giFxoFn10doTIvAtS8iIgrratHaDek887W/dpmkHTa2BYBkCwUPov4MBmNzah9fws3tyX7UwQTsIwQyGxIFm1UdrT4jAtywhIwrqatNGmyERNIWapx3UtQaEZQgAD2HwDoo+rCAcVzKRk5mZWeBGIPKS/zzvOHDIASgXi5160KUdhGAGQ+JAs2rc2hMiEJAlZERBtFqBus3TDupaA8EyBIKH8DEmAZ3cmyfi04kTJx6EC4HYFtd06si/Vt6yHsrFYqcedGkHIZjBkDjQrhq29oAIRGQJGVHQVJv3/lvjZ9O/uv2gHVS1BoJlCAQP4esdDPBpyTL3aYmj/AwyB3EkVwbvVi0WLfVghnYQghkMiQPtqlFrD4hASJZQIgqKatNvT6bj0f5+122WdvCstZAsQ6B4CJ13sFCnJZkXL168eLHg4sWLFy96+8fMbGUBlEYzAA/VfkO1WOzUgx7tABEewZA48Fg1aq2FCIRkCRWioKg2fEcN2J6fv9nfuk3SDl5qLSTLECgewuAd9Ps694jjAJD8BAA0nWj8Y1GjDbR5LBYj9aBDO0CIRzAkDrys6rfWQARisoQKUVBWWwJATfzc2t+6TdEOXmotLMsQKB7C4B30e3J/DAC4fa3Ia/NLeluUnpF67qGR9RMWxRbhbzj13wf2oyGAA3tVvxpoVn0m4DvNOgD18aNzkmhWfbSuhgwAyHCdMC459RLwN+Ij3zN4+v7C2i+AsNkh615WV8u72kwFIpHiV90ANqz6NgSJtwn227NWsfbqwVc1UvXMzxw+3sESKNpwALDvamNTL0JDPZS9tK50ERax+1BvT9rBBjEeQZUgedS9HVzEAW1Qrgq0LqWACGhDv34AFn4fX1uEdkhTVksbeDoGwGnc4VfdwI7vFgTj7JLbBPutrJU24faawZca0abpmR85BN7BwJ7o5iAHQE4O4PQcgJ8JLEidqlosdupBn3YQ4BF0iIND1eZCKR6ItJYhAmFZQoUouKo9VG0ugvo+CWBJvV5+1W2adlDUeqja3MKyDM6xKxwPIfQOBvJqyfquobe/k9qvYqX+6X/2DL15GomBry8edfd+UrFYxNSDKdpBgEfQIQ6cCWTxQAgmkCACcVlCiSi4qj0ck0CmDvkucXLLw/7VbZ52kGs9HJNQWJbB2fvC8RC+3kFx2kE3bt9tvN/22dMnoyO0iyhK6sFmhnYQlR50iQPNqq/WehABhGgHVbWOHUdvuT3Y37qBq88yBISH0HsHA/AM5TsjfE3uZ7wsoiipB5t1P7cVAbmf28fcPoGT9FyERT1Y8T9PO2xYElJQZZx2EUVLPVif3FZYtIMV1mmJFVbAoh2ssMKiHWDRDlZYtIN1zm2Fdc5thRUo7hun9GPn0BOjBxeX8yDJAFpNIHVXB5O0gIpIEMQhZCBBdTwxF0IuWVm86SNDzUh4IhPm2uPkppJdwwozdn7yEJ5jpq9jFO7eEgHtQTeyS88oCufBiHbQaAIpyyfc9JA5WkBDJMSL4BBSa83xhFwIuWR18eaOrKUZvCATptoz94VJZ490zSjM2PnFQ3gZM+86RgCefhfQHvQjZkZROA9GtINGEzi3JqWV0OTWJRKEcAipteZ4Qi6EXLK6eHNH1tIMnsiEufaOrqPJxdhUmLHzi4fwHDMdHaNoJncjc5O7WGkHT01AaHLr0w4iOIQKSFAeT8SFUJSsLt70kVWMhCcyYa79D2UyyJR5uUXPSxiPv76OEXDawft5z55lv1xOAvDP9z+kAkD2um+OSnuLw3mQZQBxTUCQdhDhFlRAgjJEXAhFyeriTR9ZSTN4IhPm2mN8h1AgamBI0fMSxuNvrGMU+eS2DzzTtmzXH5A3LC4ysv8qYO8z5W8ZM8iNZRaH8yDLAOKagCDtIIJDKFt7DL4vF0JRsrp4s0dW0QweyITJ9uk7Y78a+86o7MKMnV88hMeYGesYRX1asvZRkus/4mv1C8gbu5Evl8hiVujn7tOSgDsPhrSDhyYgds5tRCT4xiGUrdXHE3AhFCVrizd1ZBXN4IFMmGx/AC03kR81zy/k2PnDQ6hz6OoYgT4t8ao9hK2asN/R+v4L0/sFA0umA3GflUXZujtVBEQROw+SDGBCExAmEnzjEGqYQBkCLoRcskfxZo6sphm0yITZ9jYEtwb6bp9fyLHzh4dQ5TDWMQJ5ndur9tBy1ISx5fu/tdleC0BzAHU56VzUhRxN2yJ1HiQZwIQmIEwk+MYh1K3V4dOFkEv2KN7EkTU0gwaZMN2+BhoCKB+28bnCjZ1fPIQihy8do6hPS7LyU1aPLB+3Ae4LmrMi1zrYYoB0tSRmBsnKU0gOakFy300jvmi7PLCXAkmSiVh0PqqAJId0Nnta4k5Ax5T1JBMhZ9iPlcKt1cfzzOUl5JI9ijdz5PmPDh48uBv6DnZe30jBeJKs8pSf7VkpjiQjOhVu7Nb3zSX3FSKHtqzivhS4cBXJjTXTQuJJcjezykwheduA7KXeJ/e0g8l7cgJ8nXvX1yQTauYlx5Ak1zxJ0mFmcrsT8CTuJbkFT0kJPsXvwq0Vx3NQmUs/5JIVxTv8OTL5JY66O97gJZIFZRb60Z4Okj06kcwv+Xqhxm77SwXkmTF+5nC/Ae6yrsalQM4iULFu5KRZaUDmAqTnVAROn8/J/Je5EgFRxM6DWwZQaAJuW6HAXhjaAUI4hBJIcB3vULW5Yi6EXLK8JM5SeKEZnI1lZMI/2uHN7ZeA30PiCjN2/vAQijET0TGK+pP7y4dHfZ3w9GE6ZrWYNe31FHJqnXcXTPr9phe+6Rlab0QROA9GtIOsCRyOSSD/eb57ZIUeL2T4TzsI4RBSa/l4h2MSBF0IuWRpSZyl8EIzOBvLyIRftAM5u/vuda3XFobF8IuHUIyZDx0jALQDfGoPGeXsxyKr2AA4TlQKB4Dc41FVkVvK+98ZhXcebEa0gwlNQJBIEMEh9GACIW5BLllTfGGOLIpM6Lc/tTXyzvBCjl0gxr94n6H0pT0YR+GdB5t1P7cVRXU/d6HmtuU8WPE/Qjv4E4V2HqxPbiss2sEK67TECitgPf0O6+l3K6yn363TEius0xIrrPiPiVM8e57Itj6hrcA1SDu4ns8/OiHZHl7TlpXbqp/8M1jijNPZg54AMH0Dg7rEJY9LKug5xLkr67HLEW1HYPmqFpfODBi1slyhaQdxXECfODCiBXzQDh7QBIxpB1URYi6ETt2CMITcvuC7jmE4/2dnmKQhDFgGj5QGn2ZnkirVcf4cm3/yQkxsCe/7iv7eEgHtQX4+fyZWkLw0rr7yGeh5NZ2PRJ+qdI4kP28Z7bpNcEH7iAJy7QN2suAFZBSadhDGBfSJA11aQIB20FTji3ZQFyHkQujWHS8CQyjbpyE4tmrZZSZpCEOWQZPSIJbd1Gx405u3kOSSJn1fqRab4HVfMdzyKqA9yM/nf401JMnZ5U4qJveq6NvzSNJ5P8z384K+cj0u3vcGku2WkGR6VEahaQdhXECXONCnBQRoB001vmgHVRFiLoRu3UIwhLJ9WrtBvSfKN2oJ0hCGLIMmpX58h3p5zLkx9Bi5O/gJO78HfvGy72o8/Q5A+6+xhe/JAMLwj7xlQOWhiv1V5+yOV5wzxXZx3ty4z/ng95EjABDeUfjw9vfuDgLu3vkzNmYBKFnrmOsv9i/HAuidt0I8AQBMV9+jOr2/cOtK96vOX9TV6IT0InURXsbQTN0Vpo959dXnSo8R73fr2V+PjpH2JHQA0Dx9nZnB9yhZnVI/xqBNKZRunzkV2GFfmIQbgO1e9l0jXyg9TYMST3y3S/GCHk+/s0Wx+uyvRwBg7f0AgHqzEgFgWBkUknYQxgX0iQNjWsAH7SAETUgv0qQRcSH06xaBIQwqF6QhfLEMYpG7H6EAKmA10K3zCzWxF7jby75r5WqJp2lQHb8oX/BB7JMZ8lqXagkAcko6P8nHnmvYdsxme7NgFJJ2EMYFdIkDX7SAMe0gBE1IL9KmEXAh9OsWgSHU7fPef2v8bJqjIXywDKqU+lGqLAggC2fyUPnHWbZD44Lfbe5l3zVzKXDRc8NmNlVuqIKzytWIL5JeUXyu9//8CvBtd+fa3Zubb57U5pYDwgeLanwSwD7H36i0q/G37e6or7pwM71lH/EEwHvDVd/MNau+WqvCSzWeIb3II43HGJqpG8j96AkTlQetf2Ts2A0DXTsuIxgAwi6bGXxtyeqUBp+E9+IcwN+AVACJze84PnqQ933FPLm9ag/w8ox/HtTyQbvXPv1OcUp+8Vsg+QbXWsttF5f2S253CYGgHYRwAT3iwAct4IN2EIIm5Bdp0wi4ELp1QwSGULYP31EDtufnbzZHQxiyDOqUBvFh7Jod6eMBlAJw2/a0aW812Ot1XzFfCozr1KlTp06hnTp16tRptHrXEnwkXy3hTHzLtwYMGDBgBzlvB8nchhX/Hey8WrKG7NSBB1eQfW8gec55XSt4tvjT74kvTx61FyP4+Fzy8iC8r6ji/kyhJ69dCbJecVChwGhWfbXWPJDsWY2XULxIkUY5hn7UTZINZ5votzOOY6xzYR+mk2SdNjSdQ1OylNIoLk9+pNMXr6CMnY40kvbyuM3Lvqt0KVDzWKXi+Xxpcve4MZszhg8fPnyPa3Jzb6lucdLkXo5j715xTW7nVrYcHgDawQwu4IU4MKAFfNMOXqAJI9pBlUbMhdCnGcRhCBepcOdrJFMwmOZpCB2WQZPS6Er5d6dJ9sH9dLS3TSB5G5Dise8amdyK5/Pdk3tX0BrFdW4nZPou2kqTO6/SK9PpmtyNnL8JdZ9VeNpBFBfQJRL0aQHftIOyGgHaQZFG1IUwqlsAhlC0t8e+T3IP5rsSCNIQRrSDMqVxTEcr0hGDH5kTjKYkqyPSzszWrY8o910jk9ve9DDJKbZfyfn4gSS33fShYv/Ixc6XtWtFknx/JskRwcdI9qjsICOGOkheqHVeeHL3jXQwr/YSsslGkhz2E3mwyhzywC1vjhs77N5FPjsgJXD+z4d9rgTyqmjrFl1JulrL1RiE/CIpzcEqc5Rj6F/dr+EQTfR7RArJN+pluRJ8WsdOrqsqQqtLOeSSnTXIKX3EPLxEx1i86CAfClpMbgZmkiuAyap918bklp7PP/LUjWgaN/iJux/dI+/c0yOy6uOpJJnUk2RSr+iKvf/msR7kiscrhz48lbUX9F34x/L7fik07SCOC+gSCfq0gADtIFcjRDtIacRdCIO6BWAIZfvUId8lTm552H10QRrCkMWQU/o6Lelza9/ba690kMyMq/Jg13J1vnaQ/7SMParaVxy0A3xqD4V5xh/AgVuv7DhRtUXY1aEdgMC1FqpGfpE6jdAY6tctAkMo2zt2HL3l9mCzNIQhy6BNqR8pp6u7hynvTGbNMJ19V+UZysJpD7AeVrDCekDYmtxWWE/iWAHrSRwrrLAmtxVWXHsRHA+g/XXVpfbWu2oF0N71PIDtuurVeOuNtQLjrGsLVljn3FZYYU1uK6y4duL/AUj6+3vx0VxZAAAAAElFTkSuQmCC',
    '1705.04915v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAagAAABsCAAAAAAa13kKAAASmElEQVR42u2dd3xT9frHP0nTUrppKXsVbAUps4OlFpFli/VCwQKCIiAbhV7gKqgMEXFwEQdokatcy0bAhQyZBVooyCgto9JSWS0tnZS2SZPn98fJOklO8iU5wfzu6zx/fdfz+T7Jk3PON+e8842MINn/B5NLb4GUKMnENCJaKL0Lrm0LiUgmXaOkU59kIprC1QIqzSoO6GvSVp5Z7NfPiXMWXSpu1lOEMU6a2jWPqFspo74ybbuz4aXVIk+T+/QmQyXvvyM223JgGWOHGcvyYnL5RIV/1dusrf2aJ22uiT5jXTxxAzNTfze0RX8TbtOPZQxzEJZleTG5/KkPcGds459CslnPNdzA+NPtH0qfaQxzEJZlTWJy+UTZZTsfcqAs4u8MwrJZjck1V32a69VcQZmRpda31hUVoi73HoDa63X8/sz5tiTL0o4o8w0D1cV3uLNVzqFC9riq8jW6s1xOWrVpUR+Edi6mgIyMi0nA2SUTtf/Dk4mvEkDJQx/kPnlc23qsW6OFv32U8ep4zTdrjvdYDaP+LW/VHBgz5nsrkivW+9d/P0I/8FJU8EwAOBx1ULNxMfdZeLdJq4FF1sKq/SD56CsLlACwN/5szdwFKl5RH4R2LuvGG/Ruk1YDi7iYBJ3J5WxQ9GaiHBwg+rJ5OdHegFKiuAQiot79viPKwRvniL71KjHup8cnW9W8HENEmj5GA2MSiOiw2xminV77KeINonNPHLSqEdHsFpF6RLyG6MfQEiLN1NHEK2q1DXMxBWQ0dUyCsLMrHlF5I4AQ94son5/gB/SXb9R1+GaOBULc87sA7R/kmfcLW+G5c4BsnFGLDwCa1r874BPaAgCykk8+Y10kphkgn/fTDlRPT2wAyKZsPGBcFJ7LVkC6qX2sOLviYuIJOeDmUYU/yj2OAWh41aSnIwBPWOoXXgW36dax//MTTVpvZg8G0P8cAPw+NsvHhogHAES4/5pw7mY4AITLfnnWqGh9LisBGU8t6OyKR5Qnd2lFPhp5enp6bviXSY9wv7DkkXe9vuz/sobf+iea6IoHLnrNYlua+ebgCheC3OOScdH6XMIB8aYWdHble30dEBQZGRkZ2ZSxf52wVE7l4lN3V25K5Q8MxQ3dgEGzNm5kuvNAFY+jHe4DgKr2MeOiTps/l+2AeFMLOrtyoro02w8AtT8x9HuqgFxhqazNQINZCRf5A5uHnwWA2p2AD3osnGJjSa0EgLS6WHQPTgOADAw2Luq0DXOBJSD+1ILOLpgopQoAqdTwXPvLBQCrWgIqlVGPCoAKvH50/ROw9sRm9QMAyp6GgaoqQLY2bTeAr1pAqQLm+iaUWg0soxDQfDJsKLy/3poLqJe/OMS4qNfWzwWWgIymVlUJO7stcrE0nZh+suBE2wNJhVdPDOjUKyn31obopzOmpd84/nierudK6qBFq6pOZ8WGavsBdP6s/OizbQRVL1+/VnjjP52G6gZmTEu/lTrAu8XAOTfu7WinnnXhSmq/1M1F29yihCPb9sNn1y+/1/EzBdCh67zbl5Z2/9SNV9RqG+ayYrpBew1TY1r6rdSWRQLOrv7g8Nb9MBlTvyqzTaDwsIr6iqvKUE8LAwvKwtjPKvn3H9ctk2+XhZkXVZltAo3nYgqIqR3SE15IT3glkxIlJUoyKVGSSYmSEiUZXIiU/btNInUhkbL4HwMwM/YVho4O2jHM6li7OUhLUKVIppEDGnJzUMFBiUeXqPmFS5pkLWy5bRiQO27qKIGxdzb897l+9kGV619wPFGatTc9H8wK5rUdXr0VSH0rNtgbAEbL7VRglxCSMtRLkyuqVTMeF92biOjHCDURqWMjiGgXxgtfTQYmWGjUrBJ20PX1S3T4SqYZtYToTNht47aqdnFEtEb7UvrarcAsISRlqBfNKCL6yGO/2N4gIho+l4iITkYQkeb0feEZ4iwlqtAKV6LrG+R4onb51xLRpOHGbUvbxRHR61//djQ19WjfPLsVmCWEpAz1lZ5biYrQW2xvBQDkcsd7VBvYQybuhNOQRGPb2McDQMz4B176pvSQhgAgnwQA6ye0sVuBWUJIylBvQgC8USK2txwAwrb/BgCymdBRgHVFhajL0yKG6rwK1O46rvfhUZEG7FBVcA9l9wF18S0AyoJ7fC5SD1XaZ+r9/gDQtPawvqnmB+5qOgMAck+8ZL8Cq4SQlFF9ZOUI4DiGiO0tB4DZiH32wzQlYqAlEznW8VTiqwTg8yH7Z0z6qNUL2uTwqUgDdvhr56Zrv/y81ZHjMcETgOSwpmt4XKQOqrTXikp9AMAfBubos9e5R1HtAGheXy6zX4FVQkjKuO4OKFdEvi22twIAojdOOXgQjd+eAXT4oy+AJzP75Nydj4jQsf2wfW6h/4Qnnu9+SruAXbMs2w/1huQFcNXExPZ9vwKAuNj2GUntVhckHosGMGmct1EfkFfyFidnr1XCGwDcUK5rOduwpaF7U6cGDiqwSAhJ8eondxzqvt1bbG/u6pSYt2lyaOHMFQBHJupZx4vAljB/uHXbgrbaL1OC1KPM91qfJncSgQAA8KgHS1Cl3VbDYXUy6E6gqu/HGXqVb011UIFJQkiKV++xPCV/VLHY3tovvAEjR9KRFxdPCIApBYmAEgAKf127gXosqwPg7m8k1kn4/qFezl7zhRoA6uCrbfhiutE8P8taOajAJCEkxa/LwlKaP5fuJq63HAC2AICs79eVZ2FGQWJa8UVUHpqnazdQj0OCg4ODE4zFAm1DlXZbAPf5q4H2k5FdG1RWVlanKqsEgO9CHFVgkhCSMq0HRZ3eK7K3AgC2JwIABsHSuqzRjFWNCr78B4yoR660rVafAKybwB28Rl+k63Qlrs9R829SCAAFiNZegW++DSC70dttk4C6w/0dVGCTEJIy1JVP1qV7AIHIEdlbAQDZ19pxF7UuFmQPPR0PPvU4HkDt3ngdoGrGPtarAVBYC9tc5EOYLO4sAOQGRgAlXp6IiQGA7eFfAED2/cCHU0CJlydPgU1CKBhDveq0/G4L4IbFt9IRbzkAqEeXAcCaV5trKUAjChKt5yZv3LmfW6CoVDzqkTM9+6is0OayBMAOv2qjPoOc3ZaUnQvQjiQ3FLfU/aRXXcVNeRs+D6VgkNApMEoIBaOv+8UebAH8lRXzlNjeRESd9k9cuufgrGFlRKfiAwPj7h5/PqBBfPp3z/i0GVFa3rVV25aeihfv06n4wKD4M3Sw1/z1c3813Ny41nrR4sNEe2J9gwd/R0RU9OTi3ct/be4zoFTbZyxn/z2kzTFZJf8coySqCh/DtUzp6eP/7IdE9BPmP5SCXkKvwCohFIy+XvTyJ8eP9Yi7I7a3jACciaDM0+qIbpa+71VHr+0JaM68krhQiIo0Zx/vXQ/1u+Ab7C2zwUU+nN3dXdXLYoxQpsQ1dkyBWUJIylA/d566dZaJ7W3zweGO5D0AgNW//Sw9vYMLMxPdLuYDgHrvYOnN+jvN9qP4jI+fiPbKOfD0VJn0brl0ooCivOoWIRKu5PqJkkzi+iSTEiUlSjIpUZLBGtdXlQf4tta1XL8PhHiLtv+kE+FLFzaNXGxvBYCSLWmpdL0Z13A3uqhf70newtylFUITToMvLeKTmh+yVfKZwWzMo3UFZgkhqbERUV5/pQ3sZ0x1iutNRER5iVimvfm3eixyLHOXWpiSR2g+IvjSIj5ZOeAdJX0fz8Y8WldglhCUagzA/R0Nj+oU1ZtL1J33oh/jhmk+mo4blrlLLUzJIzQfEXxpiVrUDH+JiB4LZmMerSswSwiynAO+fGttngnVKaq37if4EycdjQGAE33yH2bvyEcEXwLm1OLB7WcAbFIBLMyjdQVmCUGWs9E0mFGdonrrLluJXtwGQSd6Qbsbo56iBB+01O0dadymYy+dBF9axCe/8u8KILIXWJhHGwqsElZYTphTnaJ66xLl9+L2cgAVvtytVyOKEnzQUrd3pHGbjr2Ek+BLS9QiHQq5seCDN68CLMyjLQVGCSss5/mVH68s51Od4nprr1GUijVE9G0haa9RUYOIqFbxnuEaxds7UmvaNk3YsGN3mmzmu+nGG3a0dMSuYhYR0Tks4upl6POxmnKb/05ElD4vavJ9hxTYJISkiMI3amhH2HUi+mMdUY840b31K/Y+YesAFDWCnmmCGUUJGAhNy+ylc+BLS9RiBdJflCNkwKQaMDCPNhXYJARZTmwYJcNQnyRTqlM8b32iZBNPX8CFTmy3LMqKi4uLy2HOXjoHvgTMqUVvtGoFoEvuCS21uO85tSMKTBKCLCc6A0DkjiITqlM8b0P7y4pvsG8AW6APxV46Dl9aohT9PAIBoD6ywMA8sigwSAhJIf0ol4EsE6pTPG/DnrKNh6S8525C0hooSvBgSmP2kmuTwYnwpSVqUdGpikt/Qxbm0boCWCWEpBBfXukBKBHEpzpF9FZoVxQAJuwavwKWKEpz0LIpHil8aYlaRPxSpQdQiMgaBubREoBpUACrhGAwXUZ5ALjk30HBozpF9OZOfVmXCBjctLINgHJUgE9RavefNN47EibwpY69dBJ8CZhTi1O89wK0d2IoE/NoCcDUK4BZQiiY4R0AXD+yUsGnOsX0JqJLgwJ9Ij8nWr6d6MOBQb4dh182pii13KUWpuQITT58aWAvnQVfWqIWT0VuPz1ldAUj82gJwNQrsEsIBFP3zqfnNndcoTGhOkX0tsJMGChKQHCTyUcFX1qiFsuOFEd1BsDGPFpXYJYQkspOD+zV2IneEtwC6QmvZFKipERJJiVKMilRUqIkkxIlmZQo/K/ugPknuctVqhY+pXfdFRqVR2vpbXHRRO07v762b5+JPn9t+v1M0MjeLpooU2qR1l30cZvnDaBy1V93Or3R2AEFZgnbe1hq1t+o1rz2mOjeWg4Cl4iI6De8Qi5qptSiMvYlNW3pXERUNPIalQ/3T7NfgVnC9h6WmpkXiG4/kya2tzZRg8Dxf4cw2VUTZUotLnGvIKI+U4im5hFRZWBInd0KzBK297D8+d9EROf7i+1t8V9DN++DctBYAPNva5I6zChFU291Xe9hbqieWYYmnvLmo4OB20tqZPKxfatnlqGxb41fz8FOXpeYUovJnX0B9Px6hdcvu7IawCdm5+WO9iqAVcL2HpYnHwBAu1zRvS0eUTdbB9QSEW3rdpOIRoUQ0d2eMdVEmqHNNFT3Y9AGItIkBKmISDM0mIiKXx9Y4NQDqq7BKCKig9BuRVKKGCKi5ThOvQMLiGgsDtqtwCohJGVUX4OVaqJNM8T2tnwYNF9fvgwA/b6vOYBmfgCC3zyyBZA1qy+DW3zfl4sBWYCPAoCsmR+AoFUtEzTOPKBMqcX6bsR9vbiM1DuNAVxQhNutwCrBsIfluJDZfa/8lPK+2N76U98sHwDQ/cF9zKylz0fg+6ENjVkiA/nSSl1oBki/GbphrBMTZbptZL3wewBwHWWQewDIOJ8UbLcCqwTDHpaeh0eldu5xwF1sb/0R9WlKSkpKygJd9f3QV2puZA0yDLz5r8SR+o/Ejy8/YabU1n23M48oM2pxSXYBoDwLjniqndD/fccUWCRY9rBUtpsjTx1SILa30F+Q11/f+92Kj7WVu5+gZE/EBwoAKP2k7szVDxPMH1rLW1xwZqLMqMX4T15brfo+Lo07tOcH7/J0TIFFgmEPy8zJPzWcMnnfC2lycb0F5aKTPk7QvaJGc+Ys+yOq/TEAaDDnzeTmWy09wK9q7MxEmVGLSPriyKHZhRzhte72bm/HFJgkGPawfG15Q7Tbv+jUHpG9hfPeGd149cl+uh/xNFi47YiFA/pupDMTZUYtAq3HTPDLb9MFwC/nN9TDxWwHFNgkbO9hWZH1FADZwqeuiuzN/uVH5qv/mVdD7DLv3xow25mJksXlATpqsQZA6tByoPzoOwog7cQqOfBjPXYFlNTwFBglhIIx1OvLOBq5VbjY3twqvSfOaX+gO0q3cE9GvrY0tSMR0W4sJ6LxzYioVN6VLl6gcS2IiGh8ayKiU613OvfORJbnNSJNr6VERV4RRDRPnkm0qJuS6FLbSZMnTxrTpo5ZgZMwKDBLCAVjqE9cTER087lakb1BRDT7GT+fDgmZdOgfbX2DBnI/u17W1Sd2FxFRzdRgj9kL3hzb41sNPZje0nvcGaJl7jtfvzyttd+rh+nBlCDZGwv++ca4HGffQzKhFi/GHv1jTr9CIuquPVc/LIBpUGCXsLmHZc0rkw6fTx6XL7a3nVzfnSvhDR/57XMTavHenrJuvWSPWIFhD8ucjAedo2Rie0sApvSEVzIpUVKiJJMSJZmUKClRksGV/oJc+gtwSH9BLpl06pMSJZkL2v8BhOHX3pWahD4AAAAASUVORK5CYII=',
    '1706.08653v2.10.png': 'iVBORw0KGgoAAAANSUhEUgAAAf8AAABsCAAAAABBH+Q7AAAY80lEQVR42u2deWBM1xfHv5NNZJFYItQaaxGxJNGqJaH8qFgqWlTQUvvSWn5dKD/aKkpraYsKWn78FE1R1FprEBUpESG2RGIJSWVfJzNzfn+8mXlv3rz35s1koc07f2TuvDfnnHvPydy57977eU9FUKQSi50SAiX/ilReIaIgJQqVUoKISEWAqmLHAKV3VwYVLpt2q/7WoycVKf1/ZRcH/oGcR04OuhL7JtDednIooWZW2ss/fCnT580m+PlNJba2xi3rgeFofU8A0UefNB9Rc3doziNHR9KU1PUo699/EEduzu0Ct0kbiApnt0HP5eyJu922kyXRbajzTnTmxRnf7Gwn/iFQaaX0FgjPS0UE4vYDnPq9v3jJknr4mYjmjH2giZ261J/JzOgFN6isBMJxiEYIU8h9Scc5vBdjLTZjSrULRES01UHJvzXpN43bNw3jiIgOoYeO6Fd/LRFp+/kTUTQGUxkKiEjg998ZzkzBramKc3jgpW8s9SWr1377EgBg5CClR7dCeHHL+8wXQP7kKutVwNaedgDsFjCZsavQ63+VyRt/Vwu20uc3HqkvTleSKl/4ccvrCwAL781vDiAxGQAQ2PhZzf9kRZ1WJwPQ/pUKaNKfQJNUqD+lTcpB8d5zhk9uz+tvMNe5mpJW2cKPW3BtAJdXtvkAAFpEHAIA1fRnlP+vt3hU/cIfuBHoNR1nO9RecGjZxWFjCAC+7X9s2oRlDQdp9R/9Ha0MWk4/KGmVLfy49VYB2gm6cCcAmIl+r34ZpUZQ+c3/8UYxcRiiL4URJQQRka4LEVHQECJ6pedmots4TkQ/V8kiTYvZdNeg2AG7KmbQ9A8b/wnFbSUm60s7PAF4f2uamfIc/5nIkytXANU7AOAGAO5xowAfx2sAdrbwgH2HnWjCziVolC+zTXMwZnFLmVd3ib44LOmnic2fTP+6guZ/eNKpcYc2vQaMYw+0tgPsnfIBeGYAcODMRjSOTgMArDvj6KBVz2ujZFaemMeNpuZv8QCwfwAAz+HD6fTQT9/1ZDXimzuVX/9/D4OYQslwIsr6T6ADRmmJKGQIEfXpQ0Tk+jkR/VkljnJe2MIqbsJo/cTBadRK1yn9v0wxj9suDNARUeEsoh3Mqd04wen/Z2vLsf9vWDOdKdypD9zO/fRi2sqfIgX+c2pPW/3JjDWj2QPD6vxWxEwcdEK9Wirliy1TzOKW9Z7bGhWA2x5ABPOZPijkaDyyK8fxv2pwTA4AIGIwEL8DqD5jyDUB1ZPdN3yx6XXOAdc1T5czpQIlqVaIWdw+fvxFAwD4b3Xg+l0AQC7asQq3Usr19/+ro+//CCDmwSsA1k5xAdQvAyjRAlCrAFCJFkCjcY/dqrp14gwAQr+d6T3ODsCuWlZVIm3vHWr4egP9O8N6B+QteBQ+Zct17W1fL6nQVRdT4cXt7PrAqQC0O77ZCGhHHPEEsG5MPSAbagBIHt68XPPvcX5S6AS7kzlfA0DLxW08IoP9Eb3oEvqP33RZNWhuwhanzfHhnu1cvnAoSdeE/sDOC05r+++101rlHfSImSu/CtpF21eMtovq32OpMwDMffJZnfgFDX4OxeOtp8+5hfoMlw594trzV6qHuSI/+XROQkuOvjx1o+wZ69SrubcK3z38+Q3brdgmpnH7FEWDoLt/uxjVAacvPmgc4LQv5Qdc+vwKjg92LXl4QRtcntf/RKS7uX1LAhERZat1CVcLhUYPBb5RRKS92GqhyeEbW9ecyLBm0KQJ9cskIiro3b2ATNc7xBY8eBb+0A9YH/kdlr1eYt5uW1ZdUHYX42Jxu0S62E3hMToqe0GpxsG/9GFe1/Qv3aD5M8QwhQeOE4nojQ+YnPpLTHiITVjtXW+qLzFfYt7uxZuJiPIaV7lFVln5+4qc+R9x6XAtGQC0R/qWqv95suSVjkypXmj49dKtd/RILoV+Ra664B+w/9dnzwcLD57aENZnSukWPwp7Goo96cdSrHfsv4NqL5dCvyJXXf4R+78Dd02tpeq5fUrpLvRPsvPHTXGyFOsdl4qAAaXQr9BVFzxP+79R8T873MWP3YZiDGqbrnfI/v3vuGHtbJc4a9ZLhNtt7apLZf79L7MOyDixVQg7G9c73Jo1a+pW+vWSClt1ec74j4r9tzN19zpWGYp70Mlw/XnSyz2T+9W7VixuQf+pJXGc61dGX0TdYILfbl1/RBAR7ZNtpVGpArGAnqU60/5n3v9/Z1j8IJqHuabrHWILHoLXfyfuMR23vPUSoXZbv+ry9//uP/P+f0S1w2p9bQ44jBdZ75Cz4NGjEbNoYfN6iS2rLs9y9bDUX1ux+V/jBHxFSPWlU7aMBwAcvfJJY+D63aZm6x3WLHgI6ctT//jxambVpaGtVgzsTN4DR0dSq+omAe6NDCfv5QE+rn+D8b8ROLBZpFERnjvd7GqniYiu1X9LQ0QtO2USES0YQ0R0FgOIiO51GCplQf8pRrj6IurC/X8kAjVEpNnm9F+rrAiwM0mfBKPKhDUp8151cnyoP/fECz3npTxf33/BOJhMwFsLMqyWgYqY1Xubz/w/Yr9q9K2WiKjtsXGLDp+YEZpFFD2wIVxeDxvaxR4zxS1Eh7Zw8+w27A/9W1ZfVF04/73QNiTkNd8qwH6rrAiyMzcQRESUNAyL9WfWjsLt563/JyH+902fZQBwccol6zuTtP98DwD054uu1kCz2nMJuubdHQEAMf4Ud0nr30FlI3YrR98y/2uFFRO51nZIBADc8+lzGMDjjfszbqkAgL5KXnO/fpnSw6WGj1Uk9PufaGf71PceAypinZp99+7Gsj9Ufn6laFRp9cvSCoBxE84EAcD5Lsl/k/lfk6lvdXS8FtBzH4lPARTf029WTTlxV2c8xSAhcfpVf+1fqczY4vbJJ6jcMsxlEwDgfGcZH9alqI3l3LvFvEK5qJvnnzP1TeGDCxK7noOe+4geM1a3cd25l9YCUM8+5r7n1dvgIiE75xQdHzlyK4OKADgVeEK3/VNtpc5/taER2QBy3GWskpx7bffyQ6MeA8DDsOUxn+3mFspL3XwgwU59r6mXTXTEM5PlPt6/QvSjSwbRxoY3iGa115oiIS0n6meWhhDRKfsYoj0uxyrT/t84NAgLCwsLG4Q+RESpn1Mk1hHRj09oKu5La0e8kETUD0+J6PQLx0g3/BVOwbJza9XF939k/jSxOfAVZXm8R0TaGmuIqI+Xlkjj+DoRRSGGKKL6JaIDuMWeWs3Jf8gQIl3rPkR0rN2NypV/Zq4wyZh/XYsAIlpGFvP/yH0TEY1vR0RPvRcRlbRbwxYsO7danTEgNP9jAA4uZzudBVDrFmDgPtoAcEY+MCRUlXQ1CtmmSAhXHlzvC6DXlUo+AFCN+/Cq39W2lj84r3gIgDOvAZiTPhZwuMIplJu6+e//TgBQBa/PvZyM2s7Ozs7/+wiA/p4AzF8toAvvvK1mF5geNJE7qANFgNEOG3G0t+XPHerhAaTeDAbwW4+6zDFjodzUzfPPTn23Qs2AgICAACEjH36wbX7X6oDO7CJ0k/61Oe4ryQfg3X9btqO9xY89Tu0I4KSquyY/9WEH5pixAOBhxJECm9XVv90EcIfk5J8FDtq9cAwAiveZq+WuerspkAYcPcc97FwCJOrL9XwvA0DxHlTmmysCeDdz7CA5F+KtARxqXf23W9VUDQEAF4wF4Pg3XepMsFn98yZB2chsnS0n/9oRWQCwbkw95w0HrgJY3QCAugQAlZQAKIEWjlU1AOIcC1NrGE5pAbS/AxCAknxAtSHqIIDv61emhGcjR/+aDQDxNwjoWze3MeeUiHi1SQcO/O6Lc+1dB90CQCuSjAVgeRNvh262qt/sqM2rgpPNPOXMg/qt2MkABx7AyU96tLwW3A/nl0aqus1N2BJdw/+HpUfvNO367S8zp/rFhGy8HNRvuf5UrcBwz8SeY1RBQdGLzqLzj164MPW1dgmth1Se+z9eWhL/yC6g0+JrC6/fVwW0nzAjWv3i29PwZbMhWHY8Rt2w1aKW4toXZ8++7tVi9mTVWDwe2+vFOw+H+rMFrPqo6uBVHrBVfUHiVkzXrRGovdmFhClw8CBBhDzQXLtcTLos3lF1zFPu29QbWoX/latdFJtL9CiZuZqL1TM3+kJqXvaONitsVie/7URtdqWXyFj/Ue7/WlEVsUK72YqBWNhhkM3O6xwISGiV8tMsB8vrP4o8hzI+++A994G268/cmPjglf01Hcrpe6B8/8tbW5deW1Ua51n27rqMWsIxUPL//Oe/vNb/lft/Q3n+hyKVVuwXAgiuWJ/Bz4EFBD8HFQl+xlEIhvL8FyjPf1HGf8r4TxEoz38BAORz/q3sXCqgCjqyf54iorMrhzrK19fZPeP8N8kM8nY6ldSia9Hj043viOmlvbe+rO6IFTmnn5crAIywMzOcuzolte373pajtuV+oW58MwCZ4TmFJdM46yy6X66X2E33klubU2t3AYBuwwPnghlewnUUE47vUf6BLilR/+pplb7RPc8735pF7/yw8e1BfP+n1vMWEX2FNUR0zUt44SJ99oRApEoiQMZXy4sm6wyjUXPD6cPvUvYbHlGWll10068SPeoRRZQ+LZ1omRO76TS393w1bR0od/0nv2kIEZHurc+IYlo8EqijeEW4vr0BOM7XSepDzD3fO9+aoDrHOz9sZvYk9n/+NYG5EcY6IqKwAsEMFt3TrBTL/5OJpq+W8//e+kNnIiPPBCeZG56cRES5NXw0FvK/fwURUWwvopXOu4jSYdzyqnsjjIiaecnN/yIm/3s9iolowhsCdRSvCNd37zVzNiQJtlEy/4x7vne+NUF1jnd+2MzsSez/TGvPeeOX1kioz6jSCJYQIMjf9mM3AQC2vNvY3PCBvfHV4Ra0J8HCvcT/KACApolAHQLgigzDmRMRMQB+KpFZlws+zBz59i5OAILGFriY1VFcuL5rTxFrowz3fO98a5a888NmZk9i/s/xZc6bbgDdjioEAE1aGgqSdCLODTCQEQGKk38D0GkAkHg+TOBUg2I1ADekWTBRb8UqHbC/HzA8903gHPobznzv0R5AQGd5VSn65S0AgPaYBwDULT5luY6s8H3La6O5ezPvsoTjnRc2KXtm33+TBz52xpHvRnl94LHQ8eDHcVPqZDba57/I5M7zRb3bfQeo57T22/NbeHNg59ai4yPRZ5ThVU7FmwLQvbdVCJCJ1DgBuOrga8HEO8tm7t5wc9t2AI6A+uuAeYbRzUmf++Fu2WNbyIvhN+8xtUjPdAMAD9zqZ7GO3O8Ox3fsCY3DWA8ZbTR3b+adb82Sd17YBOxZuv+X/veffm2eQaSbPIKIWnnHEuV17K/TfyCViOixYxsygYGMCEjLidZsmtn2MeuZP7C4iFkWLSS/AqduaiIiuvBh4MQ8w/EsdFmupcR6v8v6/f9zE9FLIUR0CzOIiK5goWAdRSvC+vbdrqPdLe6RlD5E3Jt7F7AmUH2TlnPCJtga6fu/6vNfUH8eEVEsfifyf5tB+yNM0vQgm0xgINvyX9wgWTT/RW17FVq0cHvUv53xr1T9zYv7DkjXH0+BfTIRvdOkUEb+1TO1+vxfxYdMuz8SrKPEhYjBdywRUcdQktKHiHtz7+bWRG7bzLacEzah1si7/9uVB74A4Ks6YDjSE7+Z/vJWAzDkqX/Sr5HItnkKYL9+p7KQzPXa62xJP270iuXXXj06SAcAqhbbjr6mp1Fc0bAhgHaJ52XU4ruphmi4MzSLBu7y6shOqRp8+wFAwO50K/SN7s29m1uT9m4aNqHWyFv/vcmwPXZONwxH3N1vC8y+cGAgm2Szj+ipTY8OWr5pzviltdD02MKLh5m3NQMvHWFK1ZxqAEBVxFuuxPXimllZWZqSrFx4Mvd9KoKHrDpyhfF94QwT+Hj5+qx7M+8C1iS988Im1Bp5z39qijwAKCk2jgpzcluaf+zD8MtNcRbQqZgBzqZ3YfJqSTSneomdOhD7Pztcs2stqZ8T3w2AasHxW726ai44ATWg/y91aJvPsGkyHkiR/mAegOu15zWZ5VHnCQA8Ric5ddSLmuN7YHauE6BGTfn6rPuZfO/m1qS988Im0BqZ+e/oFRUGIBp9AebZE6dhOobMcHFG7qrJehjIrasRAeKiQBa/eXk1zI5luDgDiDq/WgX8Olxav6oqtxoANPQtumSXVh+4j3Z6CwMXqZ2AJwiQsRwaBAARvt8BqpDLAJBYw99QEcE68q7eOL7bveUE4IZHK/n6HPd87xxrsrxzwpbh4mzSGpn9fxGKAcB1/a5EQLt0aH8Apx8DBQtCBwMAipkP/NWgK0xgICMCZHiVI4/gZixzDSNh5NPJkyaO2mhh2sRx2CoAeJjRvVq/E/WBlPigbnoLk1yPAHRknNwnpmjzcwBg1vVEgHbPstebMa2jsHB8441WAO6dXulghT7rnu+dtSbPOxs2Rp+1J+v678DwYG937+DhvxPR0R6L1w9coCYi/7fmbNzec34xERWP6F3HvfWglUT5viOJKKLB0oOf/zml88caoruNFn56ithXOeP/fZirHyPzDOufDOBnyULR2xNOxYa/k0yUPvqrc2dfCkk1WKCLARGXJo3IkTn/O+llN49XvySiHUHxGbNHqo1m2DpKzP8bfZNm/qorO9p8rZPUh6h7nnfWmtT8P+udDZveu9GeDfzHo6wWDgAQ4Ls5Jbel8L+gNqGktRPleABASVzjGtxXGfsW1NtCvIFS7Xy4HV3gF6gCgCux1MGPM9GSdfqvQD/r93+kHczvzL39l2Ad+RY4vq9fqNHZW7qNEv753vnWBNX5LZewZ8P+7wDfzVD2/1Te/T8lGiiCyrr/+2hIwoHXbyjx+qeJ3H5QC3to7OyU/v8f1v/L5T/tobCiCv+jyD8x/xXMfwQ9BxYQ9BxUJOgZRyFI4b+V6z+lC4TCf8iQKe/oV49y9Tcyr1lHhbLmP/iYxLNAQ8oEwCiLhpQdCSLhXW7+MzYaVg/Td0SeqTPOVR1/LWiBN8qW/+BjErKxCZb/YEvgUxGwgv8wJy5kEC8sdmFTQ0ypDX1F5HsX5z+kvMu8i1U4aqnZmxwPJyJ66l8/1eKjYAQ4EAn+g49JyMIuTPgPtmTOZFjBf/CJCxHihbf+w2IXshrCX8AxoTYMFRH3Lp//kAij3PwPHYHDxs12CGNuFK9/tZIDkeA/+JiELOzChP9gS+ZMhhX8B5+4ECFeTC1wsAtZDeH5N6U2FnHyL+xdPv8hEUaZ/X9qrQHbd/ThHWyCMygDDoTDRvAxCZnYBIf/YEswYzIgn//gExdSxAsEaBVbGmJCbRgrItu7BP8h4d38B0HgUS/ArjdfrbmH/yCRHLQQBUGs4EA4bAQfk5CHTXD5D7YES0wGxPkP24SDXdjSEC61YUNFxPkPSe/8jiTSFxMPfrFjwBjthpXbOhoeHRCqoUnYy+v/J9ofJSrsOpWzx3nWxovLg28R7QhxaRYW9l/jq6X9/9qQDIGi4HshC4U+6Jbwa0g2t8SpVd+ALHn9/5cpzP5vorArK5atyDLdEm+p/9cWExG1c0iT2RCef646WxFx70L76JmW8ioiFUaBOPAf9UJEiTOITmOYMf+hqanJhyb67DMSIHrhgCDiHIA0/2GGScjALrj8B4cEEaIi5PEfQsSF5fzzsAuLDREKBKPOqYgV+RfjP6TCKBAH/qNeiGjxBSJtfZc8Q/5brVy5ckNkMUuAGJ5Bw4IgVuWfw0bwMQlZ2AWH/+CSIOZUhEz+Q4i4kJd/Fruw3BCBQDDq3IpY8/0X4T+kwig0/uM96gXA/jo3AJ8H+w37cJvM4Pz2cjS5T4WBbfwHH5OQhV3ETdxXa9LEo4Oi7NgSl4qo99oFe2v4DwNxEZ7uZWVLWOzCpoYw6tyKWDWhy2kpD5sR8y7kyJn/VJfrLRvUr1//XeywONFkIwjCYSP4mIQs7ILlP3gkCJ8HgTz+wwriAmK0ii0NYdS5FbFSBPkPKe+yrv9+mtUWAC05lOUp/UEeCCKXA+GwEXxMwjJ2YcJ/dDWW+glQEZDLf8giLqRpFVsaolfnVkS2SPAfUt7l5J/i2wKA6q2Fe8ZABNTQPxWGBUGs4UA4bISxKBu7AJf/YEuMBRMqQj7/IcBvyBEurWJDQwzq3IrI9i7Bf0h5F+j/eY96AbZnMif6Y5MOMD7chBED3wCYgiDWcCAcNsJQtAqbYPkPtsRY4FIRsIL/MOc3DGCKhJjQKtY3xBR2YSoi37sU/yHlnT+QPDfAs/rAC5t7uDUcnPlRB/f20yIDqlVtn0xEs/xc3bpMp9ghL7pV6/Fv44S5nm9gxv8sCCLOgUjwH2zRCuzChP9gS4wFDhVhDf/B5zdYMEWiIia0ioyG8PybqOsrIuXdCv5DPIxlvP+DA4KIciAqKf6Dj0nIwi7A5T/YEiSoCIvtFiIurNmCIaMhZb3/Q4L/EA2jsv9H2f+jCJT9v4oo+VdEyb8iSv4VQSV6RLHy/A9U7ud/KKL0/4oo+Vek8sn/AWRhlmtJmoRgAAAAAElFTkSuQmCC',
    '1706.08653v2.11.png': 'iVBORw0KGgoAAAANSUhEUgAAAgkAAADSCAAAAADlbwzhAAArp0lEQVR42u2dd3wU1drHf5tGSULooHQQVDoSQKQkIF6UUASUYsCCSNeLYL/6goqCohQFlSZ4QUDkAgpKEyEg5VIuvUhJaBIglDQgbfd5/5jdnTNnzsye2WwoMs/nozvsOc85zzxzsjNz5nzn5yDYZhuAIDsFttkjwTbWiGiUnYW73EYRUZB7NNwSG3UbtECjAtJMDN3JBgAOUv67JZb/jgMQemAy4Lijr7wdZF8n2OaxEP6L9HNhIa7c4OpwHgsLyaX7LLZ3bdXOq9Wero4fn7Zz65+xGUw96/m2YnEAO9ZcqPlMqSXd0s+FhlJe7j1Rgb5i1Fwn/PlOC0QMmkF0Y2QdtB2vFpxoNd/n+cY1o/zzO65uH/7FDw2kTk/5P8FRQJq4LQIRZPBbhHX458djx1bAj0T0dr+zeXuHjmusHKNnRx0O3HWCOA87EKdsZDRzMV8vQz+fuzGk2DYiIpobYo8E/waCNoNfVN5PRLQSbVxEPzV2EpGzQ2Mi2oGuAb1i9Nw7aK0wCisbETUczNedd37h6/dl8ldfNgMA9Oli/8r7ZVwGMz+oC+Da4ELTHMDctkEAgkYpxyjops4sOTT/aBzuo62U96r2cW++bB9Uf4zPYObjADD65Hs1ASSeAgA0qXrL5hhTtybknAKcl5KBvJQLyEu64S5xJqUje9lmT8X5mR09zTUvZh9WP4zPYGxZALsn1nkdAGotXgkAjpdv1Uj4/LuoIh81xuEmZV7GH43Kjlr56faeLxAAfNlx7bABn1bu4nTX/A0PepzCvrUPqx/GZ/AxB+Ac4JoeBgCvosOjn2zNQUzBzTZzVzv70d29FU9ER2KIyNWCiGK6E9EjbecQHcM6IvqxUCrl1RpJJzyOjbDoZl9m/c2uGEUZnIjB7q2FxQGU+1J7jAryilFrF/bsARzPA4gAgMj9fYFqoQcA/FArCsGNfkB1dW4iz/6zzufsji6Dp9+9Z6x7s2fSgoE1L7z8+U2aWeKtadVGddp16u/9d+0gIDjsGoDiVwCEMLMbVXdcBAB8vTE0xJnzbh37yFo1fQZp6LXvogAs7wSgeK9elNDj/ReLqx4Ha4YV3NnhJLooG7m9iIhS/69JCPo6ieK6E1H79kRE4R8S0f8K7af0e79THWfhWfdERAJKp7jss4Nl02dwETq5iOjGCKKFStES/M6cHUY6C/DsULlUirJxvCKAYxnvb784ccEmfb2ywyb/a/jUZ9Uvepb/JUuZiGiKCqUd9p+4ZdNlMPWViKkOAMeigMVKnfa4wXicCyrAs4Oj69z0YgCwuCuAg8dfQ4nh2w7or1jXt+6s/SJ8avfx7wEArvsbzsVlx6nyk5Xc//LMs0Nuov3GZXX7nmD/5+lv6my/eQbfOj+5EgD8uzJw6EQNAMhAA9Xh6OkCvU74bM0/ZwPYdfYRAPhqSFEg52Eg1wkgxwGAcp0AqvQ/H1EkoimTmG5fvlqufxCARaX9CsY5Zv6EZ4O2dmwzrjAAvHPhg/IHR1X6sRvOz03YHNGtWi/zg5D41ZY9JeLDce1UQvqR+xl/OXevLe0X1q5mOQem/PXjU/634pdxGfxjWpOhAJwLv5gJOJ9ZXRzA1y9UANKQAwCnetUsyLtIorMdu65c/daQTCKipY//a/4vb02m7Z1Lloxb1ql4ic7b5rSJqPr0VUprWLl6pcIhPTIZzw3RDWb8seqV907F+3FyzetW/yoR0fXHWl8n7Ty70UQ718J/3Zc45+qvkp6n12fAn9n+gK320WSwHerFxT1RtxCwnKje2v5jVv0+vFsq7ehcGUWfjO/RIhivFuQTKCJy/Tn/uyPKZlqO68i+GwLv63W3EpFz+4OjNV8fnjv19yv+XWZ9gF3ukRg6kIieel05uo1NbqCNpkKWTdP6m9x/6zPw8RwiosyqhY6SpVYCZEYZ3EmuvbOm73JR4A1EJL6LdNSq5dksBtwvrLOywsMAgpoMW6n5+oEH/P11ujD2kYeUrQrdpr9SG4lB/s+zt/kE/vt7ZvvH1ATyFYV/ZpTBxnDUr4/bcG1zowOnAMC5+vFATbrfaOvZbEuz8zHPvvw4ij2cD/+bOduPv8Eq92pLXx/964YZ8e2HBCiU9epsZQ2sz8c8+84soFM+/G/qbD9up1Xufp/jLv53w3FnwKZjGmGJZ3MXymrn2aWvEx6a8dXIovutzNOLM2B1tv+WrQ8v4OsESStTJrA/T94pkxsIAtCz/aoNvx97OXukhUYi7stVHpD46S+Y7fe/lbvnNyGwU7RPYpJncymaeu5i1peJvMr+OR7I9nnvMHY/cxek+Bu4G/0muDpiMRHRz5ZaubN/E4KAW3cK1HbcDv/zbO5CO+AHAHDETsvYzdaaHWLcgtuaRQIw8J8dIgqCb2bxik7dAGRtkG4lynEnWxX3FWPCrRoJ2o6fKbYqx/07tSLkJYN5dm6iXRh6myrK4ZSbp0/QN+PHbH/arXx+le8fpFO3FyFbYtzF75StNXverAocOgHdPLuViXaRv5z7W+c/Umb7S+SnFdzh5Av74Ocm26ATr93fGsDB/r3fh3aeXXKi3V3L/RTD33n6/M/2e/ChzLOhoZTjuO9aEhBZxVN6MhOoFn4HjAT1wY+/lvj84N5+uDk+a/R8n46F13755pAgAGEfvV41Ouzn098COz/cg3Vdw3P/2uaMNZlHGHsgYlPre0c0Vf6p+su5e+19ZHWB68yxbJTws5XzcxM2R/RpXP3SvM0bCj3X4L4rP2zdRCfvdT9vbZrS9pEB4bf/vYPmwY9VcGOyLCRjcH7LS5j29W858vPsZmdIyXl6H3dPFloR4UOHoVDUST3xsbvoq744dptdJ4jnEzyAhT9ThymHAACdd/r78CG4deuAzbMHZp7ez1Y8+JD3s27St285AIAyi3nQott8tjk/gMVSWUjmLrT+xzcCALa0uFOeO2geueTsOOgE3MRL4mUA2Sfdy29P/37C5S1SYJj977iv1S4lK2eeY+sv2EPAM1dZdJYyEppLVHadVi99M05kcxsF4q4fCcwjF5re9Xpiy81wEy87Xujnmvn15mZfAcgZuTZy6aPHwMIwP7ydta5Pn7kKJANgQ5PfXfPfd9qDAABQrMfiNADpkRILPDc/sWT8yr7nAeCv+PG7PljCbhSUu/6CQ33kMrVCGtHq4ldV4uWfe4hmF71CNLPyYaIRDZ1aGOb+ge53zXQnog3Bu4iWFl17V65t3o9K8fHx8fFd0J6IiJI/pE34mohmX6ChOGPuvfjeJKIOuExECfeuJVevR5gN351bdTdes3R1wcCawGeUGvUKETlLTiWi9mWcRHmhTxLRVuwiWlxiJ9EKHFWLJjMjIa47kat2eyJa2+DwXToSlEcUSepIcNWKJqJPyedIOBc5i4heakBEl8uNIcptMFXd8N25ZXfjZ5EewGJ3WtgfAEofBTzESx0AhXEN6N7NkbRvK9K0MAxrZw89DqDdHvu84Jku6f/Gvvr76vmu+G52dwAbnwDwdko/IGQPs1Fg7vrrBPWRyymULVy4cOHv31Ruh9T/OwHX9ObzSrWA9kuNHUd5++hr7NmQmVjzmO96K9tEAcl/xgL4pc09ynfejQJz148E9ZHLgygVHR0dHS1q5I3X573XsgTg0r1zbJb7sybO2AdfY+U6zksLDfZZ7XzyQwDWO1rnXUv+q5HynXcDwF+LV1/32z3nlz8BHCeZkaA+cmlw71oAyP5Z75Yx6bkawEVgzWbNfEoukOjerlB3NwBkL7WHAAAiAC9e7ddF5sa+NoCVtUv8crSYozIAYJt3A1j3RYvyA/x2/7B6TBqu1k6TGQnOZ1KhPHIpPGPFPgCTKwHIyQVAubkAcuFEaJE8APtDbySX9BQ5ATQ8DhCA3GuAY8bWXwF8U/GuPPJpSHd/Kkk/eJiAx+/JqMqUGViZOinAit/qYnPD8C5HAdCEJO8GML56uZBW/rr/+ZAzsxDW31dc5r2U9Sf8oDxyiQLW/6vN/QdiO2DLuE2OVu8c+W5HycbfjltzvEbLL//z6tD6u+Jm7o7pMN5dVLrJ9OKJbV9wxMTsGPMHms8ug21Dn2hwpHb3u/DNnDvHHjwXFN304wOjD51xRDeccGT4jpwHnhuGT+7rjk/X7cqp/OCY+429t48ceahMrZGDHf1wvl+7B47/1aOxuoFJbxbpOikK/rqPSpyLl11TBdHrbkK0j1zOHjF49pJ3YHc2uVK5b3N2XWb/mXzYabPS1r2z9mYQnTul3BPudWNH7o3kzLSFdSb47U715xPVWZSSq2vAflvv7fC2Xgve903ojNGNuvjdefkV0UcePL1gRAjfQAhsu6PspbRfT0Z29t//1ZmJZx9ZXiqkgP4i7N+Em/abAFdKWUd+Ok8NjnRdKS3OgT0S7qSRUBBZsN/lbptqwaMBxN6q3mNvgxYQexsEEnuLsxCr3EXG2H8Qd7nF2HeR9nWCfZ1gG3zxDteYARZU9M7dMxeJHvu5gvz3vdk7EHSzkmIwEqpfjSkXtiGpVsus8wlVjxv5XXxlWgG8g8z13ZkbrpfuA4CMyaeT6/2znKhIrolNb3coEw4AzzAZ3fDVIpk4NL6u/xzKDXrZ+4IAPjCh9W3cpOjprf9oC8A142zh68PLWN4RJdSr09Nv5A5TH1JI9a4JWXOkhEmBeB2js/hRIvoMU4noQBnxNHnKyAFNkGwKv3g/rUzWu17eR3SuzVYiSul1gtKeitoqKDKd7lfrfe25LGZKr9WIk3ruwPpmPPZeDs3t7N13LjBxIOUAhL7nIiJX7w+IdtU6Z7ojgqcWSqgpw1KIPg1ba947766GzB8pUVKM1jFeGqC8UuRrIqL468IjmHUyb6LRSLgwUPtpZSQsn0BEtLcdEQ1OIqKMktXy9EWmKVTrvTJt5cZNmzbGJjGlYyRHAuPreiqeiO7z/k3wgYkDeWzq2zOUjpdFZRPRgKfIbEcEI0EJdWLhRUQpeMS8d86dCZk/UqKkGK1jvNiQ+Uf9i1VEPz6FqsAX/AJ/Fqj89zoA1EgEsGLZwRKIiFl6pI6uSLKJoAEA8N2LVdXCbdUk3xjK+P6+eBeABbmeIj4wsZX1EmTzW4QBiOl3vailHXGHWp4AhOOKpd6ZkPkjJUiK4UqV0IeZf7QC6NhW5c0BeRcv4nqSS9y5B4Pxwi+eT0tWYcIkF7C8A4BK2TkAInBRXyTZxDAASNwSr5Zl/UeW22V8v4lqCCDai6vwgfkw59ooALgne4OlHfGE2ivjaWAzOlrqnQ9ZvGOSb9dxnx2IVnX84feh7+QQ/VIPQz54dVLb17PdFZTfnBsthxJlj5i5fXzsUSJaGFf0vvj4f3s/rZ0dblRDqyM/xaURkTObiKhByEV9kenPqraeM459xeUnp6lZnPT6BMXXVarhyXc+fvNP9WsuMHEg8XsmfDohlYiS8ZLyCrGJZjui658NNfvx6FTz3rXuXMi687g2KT7e0eodCT/VvELkGvwMEdGD5fYSZT7U0cW2fz60jgaD8SIP9/txnUCnHkFYqxzvP7djhFGR4QlWU2/eW0zJ/2aRlZGg+KaixXgnJVb4TVPGBiYMpO58Fy2pdZLoKIYTEe3BaLMd4ftnQt32RpOBmWTeu9adC1k3EjRJkR0J1yu+S0S0F78RUePnFCJ+sab9s2kaDCZ/I+FY39cK4x+eyLPqtbthUGQ8Eth62ZVOMcupXnVaGQlu39MIPkVEz1dnX1isCUwYyF4iooe6Ee3DG0oG3zTbEc5bE6rrz8c7pZBp71p3LmR+JGiSIj0StmAhEZHTMdw7EtLxgr59FyUuexM78jsS9jVPoeOPoql7yduItplGRYYjQVNvcWWmZMJxsjIS3L6XUU3Z3XVMGRuYyeq1AbhISRipnB3GmO0I582FeqlQdJ5p71p3LmR+JGiSIq0D9adCtQSFHfZ+FRl5TD+RwWAw+bGXxpVGjbWjt68CAMw692u4QZFkE3OqqQWHskulpqbm5aZmyMXi9i0WVhIAiuCgWqQJTHzlvxEAInEQxZU3dWUhSn5H+FBLNdm52kLvopD1O2ZRB6oGMgEgN1udEkvP0L/Q+43pu2vgD8DlUFbTzHoRmk9JSz/YCoBj1LqjHQCs2Pt9EA4E1RYUSTaRt6GdWpJy9l0Ah8q+W32ETCwe35B61xTAS73/1AQmts5pGWFADkohqvwFADiPppDeETXUYS3ztoUBJXHMQu+CkAU7ZnEkPFRmazyAHXgcgPK6qQRod+FK0cIZkwa7MZiIll74hYVgJK2II6MYAFSuC2DrlskO4KdewJWihbVFsk0cyizpjRExMQCwuO4UuVi8vp3H5IQBFxCtNMMGZmwNeocBOBz1IBxxuwEgsWRj+R1RQ03fGXSxInAGDaz0zoSsPVCaHZN8g3cWsgEgfNqiRMA5rkdHAEg4D1wf1a0rAGQrFXCpUksNBuOFXzyfFiy05yQA+OtKa+BIn8uDBw3sO7MqLlVqqS2SbQLn3O/tvVSppefu/lq6ZCweXwwKXw3Q6v413c2ogZnYUw8COJkwMQQYcSgRoCUjgq3tiBJqsQ6/VwROH4xpZaV3NWTmSHmS4N0xmfmEFb1iy0WWi+31GxGtafPxtM6jcoiIGvd+e+b8tu9lE2U/81j5yNpdJhLRtbp9iBZXGvfrh/8b0vytPKITVUa/v4HUTytXjFnPDdiwd/rzp4jIrfRQ390DW2R6ocbW+xnvKFP4dfsohYMejoh69BOpK0aPL9H26MU7Bz2T7mlGDcwkkLz3Ju1ZWOdzFxHRwpiDV0b2yTHdEX3/7lBTnv1s8x/N4pLNe+fdvSEzR8qTBHXHLPMO51JrKaeQ6LpzTmfcLzydOI/k1g6j9CgAyN1ftST7aWmFxbEd1+s3Eb90RFgkCF2tlzMvrhz8XKnC+KYmXGpS3+pikUPbSjZ3N3Dx12vNGzlMd8TkCOzZS43qOyzm0SRkYVIsrm2OrjvHXrNkr1kCkGtrBcPWfAHWxB1Z8eRhO2F/W5P+bXQiGHlBQfbZ4e96dpDmIoNhQ5T22cG2u2Qk3CaaL7emBaHmy80OJOYWZyHGZqXt6wSbfLENVp5AMTbkeffTtIxT7mel5R1/26yYACI3D4pxBd2WI+HKTM9z1ZSFmzaW7x+ec/BAzKhyBUe+8MAJzToQEfyGb7UAtZ6OneEQFEiSL/LsCKTIF8kwPJCOljGySr7w5AyfVut6kdNROkd9J3EvIqLLjSsm+5R/MSVgTMgXHjjJ6RDvpB/qp/h6AqXW07EzHIIiSb5YYEfkyBdRGMbkC9e7ZfKFI2f4tPpevaazHs9glXclHuKVl767P/0mYEzIFx44+SA0nYhaDPI1EtR6PCTCIyiS5IsFdkSKfBGGYUi+8L1bJV94coZPq/WRcG7ISjzPj4TtqGTs8c1A7afMSHh3BBFRZnUi+g27iGjHFk9RxcZERCMjrvkYCWq9SvdcIaKuOOAZyx2IiL4vdE1iJAwnIqI5cwWrg7VFhqvc1T8hrlthGPojsPV7zzpGTe/8TgndmdwtKLSI6DoeEBT5XscokHcBFj39aKmlvGRIOmrBCIHxi4BhqBCO3kg9GwEAZTJ9vM2cqcdBIjoEBXLki4UiGfJFMgwjSMcq+cKTMyZQjG4k6OVdAGBjq9Cn0/hFmD8GvwlktRqmfuNVgvHIv3g+5ez5aq/G/vnzvI8AWl/tzL/GvnXUU1IkmJRoj5i3wNTblFwOwL4Q9zKxlKsRABCFoxKB1ADgemWcw1oRa3snjp+YJuhWMowvXhF3wO2U+JqZzV0okPN59LuiIl/3Di33tzh28R00rlnshQbIHdq7BAAkVQ5G728WeN8HeeM8cg4tW7P0MSDtv1dV57mLVz/QJLnHrqCePR+I/QaA51POCm/oval+s3WhQPrlB378MCip1XePKiWF6l4GgJNINW+BqRcUBmDH3hHuy+QMhANAMNIkg1lQr4QfRZ6BcGi4Y2nTNVV03cqFsbt0JYMpYe1OCU2bu/8uWf/Q4nBhkc/nDpH7+wLVQk81AB64ngQAWNgLaFlxuVfM4/DChUvO9jnSCUC5pC2qa/GMa0DbPSf8vYvMqfFa0KaO54F0bOsRhGqPDchyl3xw6DyQsxu+1kho62W/2O4j78rMMABwaFWqTUJ5e7AfRR77vrcDXSNG6LuVCiN37vMmpcxOiUeCJnfNxs071fuSsMj3EyhO3gXA8kNz5vy72vXlnhrVhw8f3r9lmHJyL6Z6dr/cOOmnTdJ/drztf3bC+AOPruniQjgqVwbQINEzzDp/9tKZxLFx8AU7a+u9U2aZR5oxUpEiyUOkXCzL3S/Ct1bksfoAEL0kRdetVBhThprNVTA7JTQud45a89Y84RQW+RwJhXkll0P3V6pYseKLWOhzZih/CIxKhejpjRFTEta/ekGjNi40th4LiegQFEiRLxaL4It8kQnDHNKxTr6o5IwZFCM1x7hgRD0ANHZlanHzihwCY5GAYagQAb1RpQpwqqrPkcDU00AiPIICOfLFWpFv8kUmDFNIxxL5ksORM2ZQjMzcNh2sBwCO3jmi12NcyRIqwfhBwBRxKH8ElesCnU/kwAOcZAHY1DUNSNv4nq+Ry9TbumVyEPBTIaUFR1ySF0GBBfJFv6eG7IhqDb5WyRe1W9kwYqZMmTJlSrG6U0boe1d3ymQcenOXtXPfRXjImSxNkcxI4ORdgPnuu4OOmOUCVB0TACxWAg0C4wcBw1IhPHCy4uczwKRqfX21odbj2BkGQYEV8sUKOwLf5It0GF5IR9O7RfJFR85ooRjz5w6bOxUv0XnbnDYRlbtefbNRZMNhm6KLFWl4iohG1A+PaPEy7e3+QESxNq9558c9WAmRBoGRIWBMyBceODnQYeP/Xmt7wSeirNbj2RkVQbFEvlhgR6TIF2EYhuQL37tV8oUnZ5iiAlb6YBAYCQLGYUa+cPTG5VWpjZo7fLcgridEUCTJF8iyI5LkiyAMR4BXqjC548kZIRRjv9XfXrNkr1myzV7bbJs9EmyzR4Jt9kiwDbb6j63+Y6v/2AZb/ceeT7DnE2zD30f9pwD4IhUkCmjjUnxSPiGmwLlLtVQg6j9yoA4sie7IMlBacR1G7UcOXoKOYRJRQ1IiQu5KFigkgbu/IkhsiJpo5Rmo/Kr/GAvk+H4WaSK6I8tAacV1WLUfA3hJ/CxSZZhE1JBARAgGEJMchWTMQEmJIBm666O1wEDlV/3HWCDH90gwEd2RZaC04jqs2o8BvCQeCSrDJKKGxsiMBHclKQrJkIGSE0EydNdFa8JABVz9B7ICOdZEd6bXjwTw8LTPi0oLCHFqPybCNzBR7+EDkRUR8lTi9XsE7Zm4+yuCxISoidak9wCp/8AyqANLojuyDJRGXEcLElmEl2BIDUmJCHkrWaCQBO5+iiAxIWqjNes9AOo/UxpXqj979sMVm62inhVqfMlI1HzVrNL9nxP1qlBntpTSh6HoTlZwayKiTzHLgvqPXu1HIHwjPjt41Xt06j8GIkIwke9h9XsE7Zmp/8iJIJmJB2kCEfceQPWfpOBJRH+F/EJELf7QSNSklu5PRK66F0h2JBiI7jSoQ0Q0BJ/Lq/8I1H4EwjfiDHjVe/TqP2IRIRjK92j1e8RqQibqPzIiSCbu2pbEvQdQ/YeeiCXKLT2AyDmck6h5I+oaUdJ4kh0JRqI7PzmSibKb4xNp9R+B2o9I+EacAa96jy4QAxEhGMv3aPR7hGpCpuo/EiJIxu5ctOLeZTRf9pytCwB1HSu8X7XFL9oTczEA8QlnsK7bj9n4oxW0oM6AtEXAv5+VPi27+SK/GSgVoxKARBLwEs8w6QIx55OElWQpJAN3lnlSd0/OnYvWKgMFq+o/6FJkAdZPcK7Erx04UKdG+2+Qm1ZWeiQYiu5IMlBejEoEEvmGl3QMEx+IlIiQrpIchSR2tyyCxLjzgfjPQEmq/yCi67xhhcKf/v4JKsyDOoO67k3sJD0QTER3pBgoFaMK14NEEvCSjmHiA5ESEWIqWaGQhAyUdREkxp2P1rR30+uEzDJDiYg2YzkRNe6trPb/j+Y64fINIqKVeHsnrS80Zz0REW1520VEY44TEeVWGDzcJasSSHvRT9l4PzSbiP4PR909bHwylSi1+CwfLeREKpfV8e45vXJxaoxq4z6vE9rNIiJqF5WrD4Rp1sclT7k4IkpzBJ8hokZY7/Zn2jO/dFb6UFN5+YZ+93y680kQ9m56xTgWk4iIaEmZE0R5nXoQETW+N5noWqNuLiKicTipzCwXbUxElFu2toucleo5iYgOVx8wcOCAPlWVibHR4ZOk9SJpJV5xv56p+M9Ermb9PT28EbSfaHQjX1fN1P99IqKzTyj3unkRrdQY1cZ9joRvthBRUui3gkDUZn3sirtSXAIRnQqLyXP7q+2ZH0rFXU2l4q3dPZ/uuiQIezceCZbVf4jolQ+J6K3XiXSgztmIS/IjwVB0R5aB0ojruEEiU3hJmAGWYeICMRARggHEJEchGTJQciJIhu76JOSHgZJR/wFSQyKAdBQTFJ2pJL/Cwlh0R5aBMhEQEnNN4gwwDJNf6j8m+j2C9iwsNRHtnpWVKoFgoGz1H3vNEmz1H9ir3GGr/8BW/4Gt/mOr/8BW/7HPDrbZ6j+w1X9s9R+bfLHvIm2zDbb6z9/YrIIzBav+YxWBYbgOf7V71CZ4VoalWSBPvujUe2TYEwZ34evLhKG687V1HA3MyRc+jSa9F5T6jwwCY6L+I6ndY0K+6FgZhmaxQL7o1HuE7AnXAoO76OqLwuD6Z9y52jqOxgf5ojsC4iQUqPqPDAJjov4jqd1jQr7oWBmVZrFCvvAdi9kTrgUGd9HVF4XB9c+4c7V5jsYX+aI7AuIkCMkXsSWX7jR/YXvuy+rYiIAiMAzXwbvObxEGIKbfdWnyRcfKqDQLLJAvfMdS7AmDu+jqS4TBuHO1eY4GPsgX3REw7j3Q6j/IDwLDcB3+avd4m5BmZWBJvUeOPWFwF0lWxchdugRC8sXKEeB/XDbVxcBfP1rY6QXnjInzHprq/rZbHg3CMu7sMDB4jYK+eMwQgTEgYGCMrXDIRzJeIiLahYmy5IuelfHSLFbIF13HYvZE8APtxl109UVhiJb7K+6C2gxH45N84ckZgyQYXCc80nYO0TH8cw/R7KIKM5Q4nCgBPb0joVty8qmVA6v97EFfPGaIwIgJGBhjKxzycRTDiYj2YLQ0+aJjZbw0ixXyRd+xkD3RtaDiLnx9URi6/r3uutoajkYKnNGQM+IkGIyE9mWcRHmhTxLRVuwiIqKPtxE5KxbN9IyEBydOnDhjU7aKvnjMEIEREjAwxFZ45GMf3lDWpL4pTb7oWBkvzWKFfNF3LGRPBDcxHtyFry8KQ78bHnd9bZajkQFntOSMOAlGIyGGiCj8X0S0GxuJiKj5t7Nnz26FBZ6REGd0tzDPcZpWDSiRRQn/cX8zom0mEdFxzCZ6/4J5+vY1T6Hjj6KpU+NKRJSEkcqP9BgfB4Bp4vOOp0+MHsORlANwUepd7p66uo65GE1+3y8Vis4zqs+FIepfcRfVZkqE7hOOEzsSmDQaJcGIgfJf/ccYgZEiYLRch1/aPUwTHCuj0iyA/+o9UuwJi7vw9aXDKNVk52pxbZWjgQR3oyFnTHoPrPqPCQIjQcBouQ6/tHs0TWhZGZVmAfxX75FiTxjcRVdfIgzGnavNczTwLR6kVQsy6T2g6j8A+uwf0xl9li98AoBGq6ZjhWkbY+TVf/zU7mGaYHSArmSBVeTxbcbqPZoYDW/iVNEdtr5sGIw7U/tKlqZEUjxIm0bT3vWnmTZticgV9n9EtB2/EdE896uJdqKFU/lsycyMenkQMkdgBAQMjLEVHvk4WPgEkav5GHnyRWVllBYYmsUK+aJ2bMaecC2ouAtT3yQMrn/VXa2teDMNS5EvfBoNkiC8YsyX+o8ZAiMgYEzUfyS1e0zIF5WVUVpgaRYr5Aun3iNmT/jnDl7chalvEgb/3MHrrtZWvJmGpcgXPo0GSUBBvKPVGIHREzAOksdWxNo9DhPyRcfKMDSLFfKF71gYI98Cg7vw9QVh6PpX3fnaPEdjdaWKMAm2+o+9Zsles2SbvbbZNnsk2GaPBNvskWCbtJGt+WJbjPsJVCzdIht1G7RAo26DQEbd4izE2gyUPZ9gzyfYhgJV/0nbf6lYW9xG6j+4hQpFQv0deX9jiKlAtIcCrP6T/P2/n/B/JLACPTw6JItTqX487iMtnKMVyeEYKHkRIbH+jrS/2yfA2kMmaQy0+s8/ustfqJio//DokAFOZab+w+E+BvCSePWaKpLDM1AGIkKQ1d8R+htDTFLaQyYMFMdMmaQx4Oo/cfkYCYxAD48OGeBUZuo/HO5jAC8JRwIjksMzUAYiQpDV3xH6G0JMctpDxgwUz0yZpLEg1H/8n+ZSBXp4dEgWp2L8ONxHVjgH0Ijk8AyUrIiQkf6OpL/bJ9DaQyZpDJD6jzMpHdnLNqvnHI8X/FT/4dEhWZjHGDmyAiOpIjk6BkpSRMhQf0fO3+MTaO0hszQGQP2H6IvHp/V96YNdpfLcZwevl7WzAyPQw6NDOpjHt/oPh/sYwEuiswMjkiOErwQiQpDU3xH7G0FMktpDZuo/WmbKJI0BUf/5sVAq5dUaSSfc1wmsl9WR4BHoEaFGLMzjW/2Hx33E8JIoA4xIjhC+EogIQVJ/R+xvBDFJag+ZMVACZkqcxoCo/zxVj4h6VvRcMWq8LI4Er0CPADXSwDw+GSgd7iOGl0QZYERyRPCVSEQIkvo7Yn8jiElSe8iUgdIxUwZpDIj6T/ErAEKiTLxgUf1HKHPDyuDAp/qPquBjQTgHvEhOpML+5CHSioiQsf6OlL/XJ9DaQ2ZpDIj6z5BLB5Cx/g0TL1hU/xGhRhqYBz4ZKB73kYeXWJEcEXzlU0TIRH9Hxl/1CbT2kFkaA6L+U3bY5LLnpz5p4gWL6j8C1EgL88AnA8XhPlLwEnQSPQL4yreIkIn+jow/4xNg7SHTNAZC/WfuT5qZJY2XtesEj0APK3OjiO6wikJS6j+Mgo+JcI74SokRyXnxISKi8SXzzEWEIKu/I/Q3lO+R0x4ydNdpD5mk0eDskIVsAAiftigRcI7r0REAEs4D10d16woA2UoFXKrUEqjy+vT5S9emAQByczkvS3YOEcqsRs9JAPDXldZKDzjS5/LgQQP7zvQxIcP4PfUggJMJE0OUFpgS3zYofDVAq/vXBEYcSgRoyYhgdyBqjD7MeS1du2nNX/FRA/F4axr25V6sw+8VgdMHY1pJpDEQ6j9pDStXr1Q4pEcmbe9cslTnXayXtd8Er0APjw5pFYVkGCgeHzKAl8R3T4xIDsdAGYgIQVp/R+RvDDFJaQ8Zu/PaQyZpDIT6z42mMx4GXLue6zlK72VthQUj0GOCQ0mq//C4j7hFcQYYkRyegRKKCFlYLCLwN/GW0R4y61zETKGg1H+WTFeux79audxes3RXq/80OnAKAJyrH7dX/uDvr/myZvJvRWLHiqn/HeNrNy16bF3rwQ77N+GO/U0IkPpPStKNitWC7BWtd8NIsNc229cJtsFW/4Gt/mOr/9hnB1slEAHQfMkf94DbDoVwBd0sp4DttivImn8Bab74yT0wSAK3UF8aEzDWfBGhA/Ct+QJoCYP8qq7IgBvePnS7LQ1tKL3r/HVJQYFrvshwD2aaL9xCfVnMwETzRYAOSGi+8ISBWHXFmHfgCQMhcWCo+cLvthjaMOYdeH9dUm6C5kucHyOBQRL4hfqymIGx5osIHZDQfOEJA7HqijHvwBMGQuLAUPOF320xtGHMO/D+OiGcm6T5YtUYJIFfqC+LGRhrvgjQAUhovoAjDPKruiIDbqh98LstC214euf9dUI4Baf5ki/wgUES+IX6kpiBieaLHh2QNC1hkF/VFRlwQ+2D321JaMPbO+dvKoQTSM0XFnxQzg7m4IOJ5otI3EQCMzDWfBGgAzKaL0LCQK+6Ysw78ISBkDgw1nzhdlsMbZjzDoy/XginoDRfGPAhrjv5BB9MNV/0C/V9YwYmmi96dEBK80VAGIhUV0x4BxFhwP/bWPOF320htGGu+cL664RwCkrzhQEf4rqTT/DBXPOFX6jvGzMw03zRoQNymi8iwkCgumLMO4gIAx1xYKz5wu+2ENow5R00/johnILSfOlfgYji63hGwhYsJCJyOobLjQReH0UrbrK4ssRYMtR8uYxqykrcdZY0X3gBFQPVFa4FH6orOhUWM80XzW6LFWRMNV+0aRMK4RSA5ks+wQceSdAu1PeJGZhqvvDogKTmi5gwyIfqCiTBDW8fmt2WgTa43rVp44RwCkzzJX/gA4MkCBbq+8YMTDVfQjh0AHKaLzxhkG/VFQlwQ9OHZreloA1t73zatEI45nOM7dsTEYV/SES7sZ6IXF2VgtH4VnB2uMyc8hjwIa47+QQfYAgr6Bfqy2EGTBMbn0wlSi0+yxMigw74PDswrATHK7CByfEOKmFgTBxovTV9eHfbBNow5B14fzYpEmeHnFwAlJsLIBdOAPOvKgUdMcsFAGlIU2ur6/ChAR9yc2EVfGCQBN1CfUlMgGlixc9ngEnV+npCZBgGn6ayEjyvwAQGKd5BJQykwQ1NH57dtghteKEIjT+TFN+/CfnSfPGCD27uwQf4YKL5wi/Ul8UMjDVfWIbBiuYLxyuIVVeMeQdedUVIHBhqvqi7bQZtGPMOvL+alILVfLEKPjjMNF/4hfqymIGx5osYHfCp+QIZgiDA6xPUPvjdFkEbZp1z/rqkFMw6Rqvgg71S5e+6jtEGH3AX8A5SZhF8sH8T/sar3C2BD/ZIsHkHeyTYvINtt6kFjwYQe6t6j70NWkDsbRBI7C3OQqwyszTK/oO4y22Ue2bJNtvs6wTb7JFgm9b+H8Bm2L03w9G3AAAAAElFTkSuQmCC',
    '1706.08653v2.12.png': 'iVBORw0KGgoAAAANSUhEUgAAAgkAAADSCAAAAADlbwzhAAArP0lEQVR42u2deZxN9f/HX3c2wwxjGxRjJ+sMZpB1hkiMJZSloSTZKxFp+6IUpSyVCol+hEoIWbMM2ck6yDJjkMFYZjFmvff9++Mu55zP2T7nzh1LzvvxqHvc+3l/Pu/P+37mns/5nM/zvCwE00wD4GWmwDRzJJgmNiKaYGbhEbcJROTlGA33xSY8ADXQBI9UE0kPswGAhez/3RfLf8MeCN0zGbA81DNvC5nzBNOc5sO+kXbFz8eW610V1rN+PrlU3WB9GRsO3q7yfFX8+ryZW/dMnMGUy853KxQHcGDTtRovlFrRI+2Kry/l5T4W5OkZo2Se8M+7LRA4dB5R5pi6aDtN+OB8qyW65xvbvHIDDtzeP+rLn8O4Tk/5P8GRR6p4IAJRyOAP8Ov0xidTppTHr0T0zsDLeUdHTA23f0cvTjjluXmCch4OINp+kN7UJnp7FQbqdmN4sb1ERLTIxxwJ7g0EaQa/rHiciGg92tiIfg+3EpG1UzgRHUB3j84YndcOUvOHv/0gsJpF9HbXg1/q/b7M+uarpgCAft3MX3m3jMngnQ/rAcgYVmiOBVjU1guA1wT7d+R1T1eWLJJ/hAfo1JX8QeV+jsPXzC/VHWMzeOcZAJh44YMaAOITAQCNK9+3NcaUPbE5iYD1RhKQl3wNeQmZjk+sCWnIXrXLWXDJnc7O6poVM79WN4zNYFQZAIdn1B0LADWXrwcAy2v3ayR88WNQ4Y/Dcapx8Gv4q2GZCes/29/7ZQKArzpvHjn4s4rdrI6Sf6K208nvB/NrdcPYDLa3ANbBtrl+APAmOj316Z4cRBbcajMz2zmOno6jGCI6HUlEthZEFNmTiJq3XUh0FluI6NdCKZRXcwyddzo2xC/3epr1H5sxKmVwBoY5jpYVB1D2K+l3VJAzRqldO3IEsAwAEAgARY/3B6r4ngDwc80geDf8GVWFtYk88886n6s7sgxefP+xKY7D3glLh9S49toX92hlibUmlRvWbddlkOvfdbwAb78MAMVvAfARrW5UPnAdAPDtDl8fa877dc1v1qjJM0gjMn4MArCmC4DiffpQbK9JrxQXPOJq+BXc2eECutkPcvsQEaX8r7EP+luJonsSUYcOREQBHxHR34WOU9rjPwqO8/GiYyEiFqWTbebZwbDJM/gLutiIKHM00TL7RyuwVXR2GGMtwLNDxVLJ9oNzFQCcTZ+0//qMpTvl5cqMnPXeqNkvCm/0LvdHln0hognKl7aYf+KGTZbBlNcDZ1sAnA0CltvLdECmyOOKVwGeHSzdF6UVA4Dl3QHEnXsLJUbtPSGfsW5r3VX6RsDsntM+AADcdTec66vOUcVnQxz/cq6zg2+hPfOmcPyYt/vr9Pd0tV87g+OvzgoBgP+rCJw8Xw0A0hEmOJy5WKDzhM83vbEAwKHLzQHgm+FFgJwngVwrgBwLAMq1Aqg06Gpg4cAmosT0+OrNsoO8APxS2q1grJOXTH/Ra0/nNlP9AeDdax+Wi5sQ8msPXF0UuyuwR5U+2l9C/De7j5SICUBGYmza6SdE/nzuLls50K9djbIWfP3vr8+5X4tbxmTwrzmNRwCwLvvye8D6wsbiAL59uTyQihwASOxToyCvIokud+6+fuP44XeIiFY+896SP8bPov1dS5aMXtWleImuexe2Caz8/G1KbVCxaoi/T687Is/tEWHz/trw+geJMW6cXPN6hN4mIrrbvvVdkq6zqy20MzXsc0xxroRu4F6nl2fAndV+j+32kWSwHepHR3esVwhYQ1R/86DJG7aO6pFCB7pWRJFnY3q18MabBXkHiohs/yz58bT9MDXHdvpYpoL33Xp7iMi6v/ZEydunFs3eesu9adaHOOQYib5DiOi5sfZvN1zjAlptKWTVHKm/xvW3PAOfLCQiulO50BkyVIuHTC2DB8l2dP7cQzbyvIGIlK8iLTVrOg+LAU8olllf/kkAXo1Hrpe8XauWu79O16Y0b2Q/Kt9j7ut1EO/l/jp7m0/hvr9ztX9yDSBfUbhnahkMhyU0FA/g3uaGJxIBwLrxGU8tume2dR62pQX5WGdfcw7FnsyH/71c7cd/YJd7lZVjJ67bPi+mw3APhbJNWK2shm35WGc/mAV0yYf/PV3tx4O0y93tc9z1fdvPWT22HNMQK5yHh1BGus7OPU9oNO+bMUWOG1mnV86A0dX++7Y/vIDnCZwWHOzZnyfXkkkmvAD07rBh+9azr2WPMVBJYPVc+w0SN/0VVvvdr+XR+U3w7BLts5jpPFyJJs6rmG3BRW+L/xxPZOteO0w5LroKsvuruKv9Jtg6YzkR0WpDtTzcvwlewP07BUobboe/nYeH0A74GQAsUXPSD4tLLfBRr8FhTYsCUPFf4KMUBFvN8rVdegDI2s5dS5DlYbZKjhlj7P0aCdKGXyi2IcfxO7XW51WVdXZmoV0x9DaV7F8n3zp9rLwaN1b7U+/n/at8/yAlPliEbImp13+0H2068nZl4OR5yNbZjSy0K/nzuY+/+rF9tb9EfmrBQ06+iG/83GMbev6tJ1oDiBvUdxKk6+ycC+2OUo67GO6u0+d/td+JD9257OtLOZbqGQlA0UrOTy/cAaoEPAQjQbjx467FDxjW1w03y+cNB/Tr7L/5q7eHewHw+3hs5Qi/1Rd/AA5+dARbugfk/rvXGqWxjjDlRODO1o+PbmL/p+DP5+6yScjqBtuls9ko4WYtVxfF7grsF171xuJd2wu9FFb91s97dtKFxx33W5skt20+OODBv3aQ3PgxCm7M4oVkVM5vebFzvv0zh3+dXesMyblOr3P1ZKAWJXzoFOwUdUJvfOL46Jv+OPuAzROU1xOcgIU7S4fJJwEAXQ+6e/PBu3Vrj62ze2ad3s1anPiQ67Vewg/jLQBAd4o50aIHfLU5P4DFSl5I5hG0Qed2AAB2t3hY7jtIbrnkHIizAg7iJf4mgOwLju23F7eet7k+ssMwx991zNVuJNnPPGe3XTOHgHOtssh8+0hoxlHYdlGY+qafz2YOCsRdPhJEt1xobve78S13wUG8HHh5oO37b3c1/QZAzpjNRVc+dRZiGObnd7K29Ou3yA7JANjeeKttySSrOQgAAMV6LU8FkFaUY4Pnro4rpq3vfxUA/o2ZdujDFeKDgnKXTziEWy6zy6cSbSx+WyBe3jhCtKDILaLvK54iGt3AKoVhnhjieNZMTyLa7n2IaGWRzY/k3ubjCImJiYmJ6YYORESU9BHtxLdEtOAajcAlbe/ljycQdcJNIop9fDPZ+jQXHeg3btRdfc/S7aVDagCfU0rQ60RkLTmbiDoEW4nyfJ8loj04RLS8xEGitTgjfDRLNBKiexLZ6nQgos1hpx7RkWC/RZEgjARbzQgi+ox0R8KVovOJ6NUwIrpZdjJRbths4UC/ccPu6vcinYDF4VS/vwCUPgM4iZe6APyRAfTsYUk4tgepUhhGbJdPPgOg3RHzvOBcLhk07ljosfr6Bd/P7glgR0cA7yQPBHyOiA4KzF0+TxBuuSSijL+/v/9Pb9svh4T/WwHb3GaLS7WA9E2JnUM589uX2Is+32NTe/1y69sEAUn/RAH4o81j9vdcBwXmLh8Jwi2X2igVEREREaFUybixiz9oWQKwyZ45Nt/xWgOXzC9fYmU7L0719dYtdjWpEYBtltZ5GUn/NrS/5zoA8O/yjXfdds/54x8A54hnJAi3XMIe3wwA2avlbukzX6oGXAc27ZKsp+QC8Y7j8vUOA0D2SnMIACAC8Mrtgd14LuzrAFhfp8QfZ4pZKgIA9roOgC1ftig32G33j6pGpuJ2nVSekWB9IQX2Wy7+89YeAzArBEBOLgDKzQWQCyt8C+cBOO6bmVTS+ZEVQINzAAHIzQAs8/asA/BdhUfym09FmuPVnvS4UwQ881h6ZdFnKhZcNxlY+2c97GoQ0O0MAJqe4DoAplUt69PKXfd/GlnvFMK26sV5nksZOv1n+y2XIGDbe22eOBHVCbun7rS0evf0jwdKhv8wddO5ai2/+u3NEaGHor8/HNlpmuOj0o3nFo9v+7IlMvLA5L/QbEEw9o7oGHa6Ts9H8MmcB6fEXfGKaPLJiYknL1kiGkw/PepATq2XRuLT6j3x2ZZDORVrT35C3Xv/mDEng2uOGWYZiKsD29U692+vcOEAM98u3H1mENx1nxC/CK/ZZitEL7sIkd5yuXxa5d5L3onD2WRLYd7NOXRT/M+kU1aTlTbunXU0nehKov2a8KgDO3IcJN1JXVZ3utvuFLqEqO4vybmyCsyn9T4IT+s14F19eldMbNjN7cbLrY04Xfvi0tE+bAU+MO2hsldT110o2tV9/ze/j7/cfE0pnwL6izB/E+7ZbwJsyWUs+Wk8xbuo7VZp5RyYI+FhGgkFkQXzWe6mCeY9EUDU/Wo96gGoAVEPQCBR9zkLUfaryEjzD+IRt0jzKtKcJ5jzBNOgxztkiAaYV5H7EJLtQRmdokBsD9RfTIFEIx8JVW9HlvXbnlCzZdbV2Mrn1Pyuvz6ngJ5Btv2bX+zd/e1krtdrLi7/9ty0zNyRT+i69w9vXOTinqfbymug+ScCvccFGA5EegjbvMv+d0fpPC8gfdbFpPpvlFVqVhQgh7tSt8XRKJrIiW1dIwnssrW1+Bki+hyziehEsPIyefKYwY2RpAm/uF6NLtZnVLMjI+ntP8ihRV1dTY5MJvrMb7NuDWUB+H5gk9eQ0ynGSj+HJvPed3AGIj0kW98PiQ7VvKIZSHKf85T6XNAepWaFAFW7IbgrdVscjbK74MS2rpIExX2MNwbbHynyLRFRzF3FbzDrQt4MtZFwbYj01ehImGzvp+25GCKq7hqKM/x/IUpGc90a2s9+Z16CUg0f+qYRUYuhvCNhspBw0SGtCsomosHPaQYyLIGI0ktWyVNo1hWgejcEd6VuT9YbCSIntnWVJCg+rfd6A9E/Qq8r/voUquStB7/AzQ0qe6vYF0K3Lh8NYOnvzvfLEYAA3NKtoMzwTwZVVqphbmhRAE8uvmssEOkhsKSFH4DINZrVrG1+GwiMTDit0KwrQB53hW5LolE0kRPbukYSZCPB90nRP1oBdHaP/ckBedev426CTblxJwbjgl+crwYt6zcHWvtdUAMAES5KpE/688AudOaviqkh5XIgAATfOWIsEMkhYN0cBACPZW/X8g7JzgEQiOtGm2Xd5d2WRKNsghPbulY0shmjRCCyGTZ+3T94bNBEX6wbf3x4uduVVodPljxGPqt92NfIeadO6Mo/5tYAfl6UtaUfOvR3vhodCV++br+7QtuqXJobmDqwpjBCgZwvIt7Xr+Lo1jyfgUGyGgp7k33kn25uJBDpIZB8OxAAgnCmk4b3zjw/AMd86ik16wiQy13ebUk0KuZyYlvXSoLOXenVb+0rgagRA35Cp051ftsUikGtT60Wh5K67zawaPnGWo2Teh3yQu/etaK+A1yvBu1wacfDu9Nu1vr1I6+EVj8+5fps34ptjZbrz/yPnhxlWdlkUyW2hkL1bgLABaQYCkRyCCAdAQDgjVQtdy8/AAeOjg5WaNYZIJe7rNvSaNTM6cS2rpkE5fmSY8Z4t8L7RERH8ScRhb9kJ+KX2ws4ZoyXUyUYjAt+ecKNGWPOm1aiptFEdBHeiUQ0oGqm+BHCz3RJ1p0xHiUiatRDXsPvliSi7Gb4lGPGKAQiPiQiOoZx9oy8rRdIVv12mUrNOgPUmTo73JluM9Goujud2NZVkqCv/nPkcj0AqGdZ63qrLf6QFClfDOh5Mzzh953afyY89vUIZzgBqFgRQFj8bvEjhBdv6qhLWYYCQMSKZFkNXT9/9VL8lGiUNhSI+BAAitq5jjwU1avj3eBV/krNOgPkc2e6zUSjvnzscGJb10iCdsX/2KkWL79TQiqKnpWveYkwmHzYyexSKSkpebkp6SjmVxIACiNOXKBU44Mb9ab8O+zfV5y8htFfx25785rkqU36gYgOAQDF7U/eyoLeutr8K+sClJp1BcjpLuk2G42GOZzYTqsnQXueUA13ACA3W5hHpqXL1/nGzT1cDX8BNot9CjH/FUheOS358vsATpZ5v+pon/oZdq7KMXhzWubt9QNK4qxOHV1T0/2AHJRiawBQqRKQWDnMUCCiQwBAULlrAHAVTXQuBI/+5IUTXnVkzboC5HGvznSbjUbRpLliO62eBM15wp3gEUREu7CGiML7EhGtxm+SecLNTErzHklEv+HA+p1EYQOJ3iXh1fCG4LLRRESTfLOJ6H84Q3QzkyjV4n2JiBpim04N7eYTEbULymVroB3PphClFJ/Pvbe5bDRzeDOTiF5pREQ0rWSeZiC737ER0eRzkmZvZkoC1OiG013c7ZuZCoEpuYud2NZVkqCu70BTHM9LXRF8niivSy8iovDHk4gyGvawERFNxQX7ymaRcMoMHEpEE313/BBH9FJrondIeDU6EvICWxERXSu+msjWdJC9BaLoWCJK9IvM06nhu91ElOD7g7yGcV7HiSY2zOEdCY5AhEN7NXH+54lszSZrduVU1cFDhgzuVzlP3KzdXwhQvRuCu9BtuzcTmLK7KFds6ypJUBsJa/tElS1aNqrPn0S0qc0nc7pOyCEiCu/7zvdL2n6QTZT9QvtyRet0m0FEGfX6ES0Pmbruo7+HNxufR3S+0sRJ20l4NTgShj4ZGPTUp0S0P2L5waEvpDlaoOQXP9/1V9PoJL0a8j6YeWRZ3S9s8hpOdNrx91ttr/HyDq5AXIf2amhZZNytMf1yNANx6FSESpq1+4sCVO2G4C5029G6JDC1+w5CrtjWVZLAxztcSalpn0xE1Ft4Mf0JxYmF9XRuHT9KCwKA3OOVS4pf3d1hkRJ7o7H4UVdHjlLDUI5dvSf3lmxWVqmGmxtSGjaz5H+nyvV1Gc0a8m8vljUrBMiTCMVu6zUuOLGtKyfB4N7miHoLYe5ZMvcsAbmmVjBMzRdgU/Tptc+eMhP2nzXu30YrvJHn5WWeHf6rZwduLtIbJkRpnh1Me0RGwgOi+XJ/alDUfLnXgUTe5yxEmqy0OU8wyRfTYGjPkmDDBzjuvqUnOm57lrPgkSFfHm7j6gjvSLj1vfM+bPKynTvKDQrIiTsROaEs7hn5wsutCGCJ7cdLmbZXqxtDVuSBsG485IoUWJFSQmxY0CRf2G6LkBrO1qWkjAHyRcXmonSO8EziPkREN8MrJOnKv2gSMAbIFw1kQ4V8sb12jOhKmz06yIoe+SJzUyJXZOSLwJ6wlBAbljb5wnZbhNRwkS8sKWOEfFGxXi9gg/P4LGLsD313vLpNwBggXzSQDRXyZc10IqKj7XSQFT3yReamRK6wNYjYE5YSYsPSJl/YbouQGi7yhSVljJAvypZUuj+WsW9WxQ54moBRI1+4uRUXWLLvMgBUizeErMgDkblxkCti9oSlhNiwoEm+sN0WITVcrbOkjBHyRUneBfjl+adKrWQlQ9JQE2oIjJsEjBr54gZAUn76TBuwppMhZEUWiEE36HI6TFjQJF9k3RaYGN7WJaSMVhplI0Eu7wIAO1r5Pp+6gSn6q/fbQFarkcI7LiUYp/yL8xVukC/vTRl/BjJuRbeKozOmzUgFMKDKm1H/rF78MVhkxVAgCm7OBsDDnsiMCUvRdiaVBXDMp56s265PuFuXkDJaaZRdO7Q83uLs9XcRXqPYy2HIHdG3BAAkVPRG3++Wup4HmXkVOSdXbVrZ3oG+OM2FwLhJwKiSL/zcigss8d/ed2do0y2+hpAVWSByNw5yRYvTYcLSIV/YbouZGK7WpaSMVhoV5glFj/cHqvgmhgG17iYAAJb1AVpWWOMS8zi1bNmKy/1OdwFQNkGEJBRPzwDaHjnv5gVk7qIBrhPP3l5eqNJ+cJbjjQ9PXgVyDkN3j8RPfS3oHjgaQE61t7x2dr7q/LmHHwBYpCrVuoHI3YQGtKzp1MWJfW8ofSINS9WyX2n3sXK37Z/wtS50RDeNSusJjLwLgDXlTgFVLq/p45wrjhKd+ESeYiUYeIJ8Wbi7rQvZ+CZ3UfSe0nzky9zk4ONDVpceOmRTtz1eBpEVSSByN1cDuuxJ+Y575Uw5Exa0yRelbruYGI7WWVJGI41eiqqXkCq5nHwipEKFCq/Irx48i8Boki+c3IoAlrw6tTSqbZ64f4NBZEUaiMyNk1xR5XSYsKBDvsi7LWZi9FqXkzLuki8OWzq6PgCasj6luHZBBoExSMBokC/g5VZcYElaXCsAlglbHFAzL7LCBCJz4yFXNDgdNixdcIbttgip0W9dgZRRTSPPSKC4+gBg6Ttx5csK69BF/AUlmGEOJZjAlk75F7EMjI5FRgLA8npfA+g6OccPuIYIRwM7py8MQuqOL/TiDevrB+BUUG2ypBcDgIr17DVYog8DQHzJcEOBiN1uFfEXNaB1CXrQ63oF4BLzp3eriD8Ki8JStz27Z1mA3/uIu23Ps+sTrtbFGdVLo8LZgZF3AZY4rg46Y74NEHRMAAA3QloKVy8iJRin/IvzldesGWkAMDRgI0AbB9VwNLB29SVgZhXdBzI8VxvAhdgZPr69ZwLAv7daO2oYfTIeoBWjvY0FIrjZq3E1oOFarNPWCsDFuEi7Nks2sl2ZEoWlbqf73Rw2dEj/7yuLum1vXfiEu3VHR/TTyC5W7upSvETXvQvbBFbsfvvthkUbjNwZUaxwg0QiGh0aENjiNTras1ZgsTZvuZbnnUQGkQSB4SFgDJAvGsiGCvmS9dLg7UfnDkgkbWRFl3xxuWmQK7L7DgJ7IlBCdn8hLB7yhSVXREgNF/nCcjv5Il8MmQiB4SBgLAbIF3VkQ5V8OXvgbmhjiw6yor9ThXVTIFfkNWgAK2xYxrgZg+QLX33mU/3NPUvmniXTzL3NppkjwTRzJJhmjgTTYKr/mOo/pvqPaTDVf8z1BHM9wTQ8Euo/NvJ+mLmmhwCnKhD1Hy5QB3qiO5Imdr7TKTgAAF7w4mWglGAlXeEcyCgiVn+Hs3OS6DkpJKmxBXngKV0CS+Ob87T6jyqoY4yBYpv41jnJ5Vf/UYCV5MI5yvcixQwTo7+j0jloJUiXQlJKBFtQEZ7SY6DYNKp8cwWi/qMK6hhjoNgmXp+zfsfOnTuiErgZKCVYaTLnSBBRRKz+jkrnoJUgXQpJKRFsQUV4So+BYtOo8s2BiHx01H8qKav/aP44rV0VVwKBkStP14Xb6j9sE16DAeDHVypDl4GSH/EL50BGEbH6O1ydk0avRCHN+UJvCsYW3HcX0IOnFAgs7m/OQ+o/UBavgdvqP6yNBID43TH5mRNxCOdARhGxOJPxznFTSBKTFeSBp9xVSlIcCdUbiv7RrNLGroezxr6XC6xrVHbSR++vbj8uR9rNViMxO6Ji2MKFzUKe3Ig+Fap/LQJ1vn2yYq3pQN8KvA/1VNW2qQbA9vpUjkc2CIgSCyvxCOfIKSIGZ+KjkFT7xA1zyQrywFNGlZL4n+VOv9e4RWQb9gIRUe2yR4nuNOpsEz/L/apvXaIE75lE//r8QUQt/nLUsB+jiSil9CAistW7xjdP+Hu+SNFEdkJbPJ5jplFviY1W1LwgOVKoXG/32t5xjYfckR2SpHOagbiiZ5oNq0tENBxf6M0TZAUTm8OvVQ5XHsUhs2mcwTtjdEf9hzpGEeWWHkxkHcWI14wLyiBKmGZU/Ucp5OyQRI6RIIjrMDI7ysI5qhkQie7IZIdcyjwcI4FtVlF/R2kkyAqe7f+WP55O4pp5i0LmHQkeUf9BTOwlbOnxazb+asWAOoNTfwH+70UYZKCUbI2lIkcdgrgOI7PDK5wj1xqSyQ7xUEhqfeIWIWILHn9x+rQTT23qZjMYvWfuSnOq/6Bb4aXYNt26Hus6MaBOtQ7fITe1DAwyUEq2sArPtYcLUWJgJQPCOZAxTFKciZNCUmmWW4SIKcgJTxlQSioA9R8Edl88slDA8z91JH8W1Bna/Wh8FxhloBQ+zdvejqMOAVFiYCUu4RzIKCIFnEmfQtLuE7cIkaQgJzxlRCnJ8+o/RLQe7xykbYUWbmO0b4hyyw8bZTOu/iNqwiF7cxQDOWoQxHVEMjsawjmK8wSRfo5cf0fcOZ4ZI6MepKi/o5QIRrgnp2gqERHFbNbLAqOUxKTR6IzRkPoPEeWWqWMja0h9K6N9Q0QTA2a6of4jNOGUvVmP1zlqEMR1hCMt4RzlDIj0c1j9HWnn1ANxJohVD1LU31FKBCvcM2gSEdHljtm6WZAqJTFplATmcfUfInr9IyIaP5bVviGiy4E33FD/YdEhotV4l6MGAVESjrSEc1TuOwgUEau/o0whMTWIEsRFISmNBBZ/UoSn9BgoNo3SwApA/QdI8QkE0lBM4aNLIR7ZYZGzOLosDDFQirAS504VEUXEo79joCtKFJKit6ygAjxlmIHSzoGp/mPuWTLVf0wz1X9MM9V/zLODqf5jmslAmWaq/5jqP6b6jzlPMMkX02Cq/5j2QKr/uIPAKJMvbqj/sPpBLMLCGTjbcH8e9R+RvzvkjLiMG2lgmpDgPkxSUPDqPzwIjAHyxbj6D6sfxCIsmneghMBlDXOp/wj+XOSMqvoPXxqgmXYJ7sMk5V6o//AgMAbIF+PqP6x+EIuwaI0EUeCyhnnUf0T+XOSMqvoPXxqgmXYxd8MmxfhIuDJ8PQawI2E/QrS9Qh67RUTdcYJ/JOz5SXlHa4VwIqIxgRl6NbgG5584REQHdjv/vbTQL0R3UYtnJIgClzUcw9EVkT/brGJOoJE3jjRouYszKk+K5o7W/Kn/IH8IjBqc4ob6D6MfZIgHEQJ3o2Fpx90hZ1TL8EUjcZdmlE1KQan/qCIw3ASMGpxiXP2H1Q8yxIMIeItCwxzqP2I8xg1yRrUMXxok7pKMypOiOWNs3nYh0Vm8cYRoQZFbREQUP4ooFr1dZ4ceSUmJ64dUWe1EX5ymisAoEzAGyBdFYESLfElBi2lWii//pzIPoq8S6MBbZA2zQI3qic6Jx3CQMyB1d440aLhLuRuFpGjNEzoEW4nyfJ8loj04REREn+wlslYocsc5EmrPmDFj3s5sAX1xmioCo0jAGCBfFIERLfLlIrwTiWhA1UxFHkR/JDgClzXMADWqgQh4jD45A1J350iDujvD3SglRWskRBIRBbxHRIexg4iImv2wYMGCVljqHAnRanPExZaLtGFwiSyK/c3xzui2d4iIzmEB0SSd7XvTz5EqA/VF54vnJ07GfI7d0USDcf0mqtgr2SJ5JkChiDzOkeAIXLnhwbiuF4jTX6FZ8Udq3XCV4UiDurs0o6ScFHUGyn31H3UEhoeA0YRTjKr/yPWDjPEgLryFaZhX/UeCxxgnZ9TKcKbB4c5mVDkpBaH+o4HAcBAw2nCKQfUfVj/IIA8iwlukDfOo/4j83SRn1MtwpcHpzmZUQVRJa8bYoQMRUcBHRHQY24jI1t3+wUT8oHB2uCk556giMEoEjAHyRREY0SRfJvlmE9H/cMZeA8OD6JwdhMAZAEUM1GgE4vLnI2eg2jxXGrTcpdyNKCkc84Q2bYnI5vc/ItqPP4losePhRgfRwmp/bSla2nRhJaSNwCgQMAbIF0VgRJN8uVZ8NZGt6SBnDVIeRHskiAJnARShAY1ARP5c5AxUm+dKg5a7lLsRkqI/EvKl/qOFwCgQMAbIF+PqP6x+kFQTR3skiAJnARQu9R+RPxc5o6r+w5cGVXcZdyMkpaDVf7QQGDkBY6RhN9R/GP0gFR5ELwOyhrnUf4xhKPnjZgzlkU2Kqf5j7lky9yyZBnNvs2nmSDDNHAmmmSPBNBh8uo6p+QJT84W8IHs++r2zCQ9ADTThAQhkwn3OQpTJQJnrCeZ6gmkoUPWf1OM3irWFyUBp9I5XxshTekf3Sf0n6af/65jPkaDCQPHiVOrqP/zqOe7q56gUck/GSFaQHygTRW+bd9n/7qhgcIBgnlb/ebon/0TFAAPFJbqjqf6jpp6jp/7Dr5+jWIhLxkgpEWxBDaBMXf3H1vdDokM1r+iAYAWi/hOdz5GgwkBxie5oqv+oqefoqP8Y0M9RLMQlY6SUCLagBlCmrv6zKiibiAY/pwOCFZT6T35MTf2HW1FIVf2HXz3HXf0cxUJuyxixBfk1lUTRL2nhByBy4N0iso8KSP3HmpCG7FW7hHOO0wseEujJh6IQjKrnuKufw2XcMkZsQf4MCNFbNwcBwGPZ2/U7JvtNkMyommHj1/2DxwZN9MW68ceHl7tdaXX4ZD/JV9c+7Gvgq3Xd//IPiX76mmO66/IyaqoCPTvz/MAnunN0a57PwCDpEQAM+OzNFfP+WbwEKDD9HB6zyxgtshguyJ8BIfrk24EAEATXA+DVO6azt3n1W/tKIGrEgJ/QqVOd3zaFYlDrU6vFvUjddxtYPvZa0Ct1ujTa7816GbTDpdWe7OvlB+DA0dHBugPh5CjLyiabKomPAAD+2/vuDG26hXN07luxrdHyABSMLa1fwnhB7gyIok9HAAB4I5WjY55Q/3muPhH1ruCcMUq8jMwYtdV/uER31NV/1NRz9NV/ePVz1ArpyxipADyyggoZ0FT/OYZx9m/ibW0QzGPqP8VvAfAJ0vCCB9R/+ER3VNV/DKnnuKWfw2t8MkZKBbllhxzRF7WTS3koqt8xj6j/DL9xAunbxml4wQPqP1yiO6rqP0bVc9zQz+E1LhkjpYJGZIdKNT64EcWRCQBZCNLvmEfUf8qMnFXm6uxnNbyQf/UfPtEdVfUffvUc9/Vz+IxPxkihIKfskCj6oHLXAOCq8xlZWh3THgmNgvfEADiAZwAgBwBiIc3krSL+2Na6q7oXv0VGAsDyel9D1gKwZ/csC/B7H506wvr6ATgVVFt0hFtF/FHYkl4MACpyzL2zDnpdrwBcYvBDeyDum8P/5J2SnL+RzoKGMiCO3hJ9GADiS4bbK1HpmNbZIQvZABAw55d4wDq1V2cAiL0K3J3QozsAZNsL4EZIS6DS2LlLVm62z09zcxkvg2bNSHMcOZq4EdISwOl+N4cNHdL/e70FmedqA7gQO8NHdHQjpCXg23smAPx7q7V+DMU6ba0AXIyLbCUPRNR3TRMVYvyvIJAvFc6CxjIgiX70yXiAVoz2tlfCdEz32sGw+k9qg4pVQ/x9et2h/V1Llup6SOxlcLVZjYHiEt3RVP9RU8/RUf8xoJ+jqP7DJWOkfO3gLKiZAW31H1oWGXdrTL8cHRDMU+o/mU3mPQnYDr3Ue4Lc657vVFFX/1FWz9FV//H0ZhEFGSNlb0W9IxhU/7m+LqNZQ4tOxzyl/rNirn0+/s36NeaepUda/afhiUQAsG58BqbhP6/5smnWn4WjptRW/OzAtDpNipzd0nqYxfxNeGh/Ezyk/pOckFmhipe5o/VRGAnm3mZznmAaTPUfmOo/pvqPeXYwVQLhAc2X/HAPpuYL7ocGTAFpvuSDe3DwDgY26oNP88WIAI2oLItJcAWiofnCw0sI7mzQXNCGZuuMBgwKXPOFh3vQ4B0MbNTn1XxR4QV0NF9YTIIrEC3NFyVeQl3zhQlaGdpQ13xRilaiAXMPNF+ie+aLdzCwUZ9T80WNF9DRfGExCa5ANDRfFHkJVdEWNmhlaENd80Up2snKI4HznJxUur/8qc1VsQMFxTs4NuqvuQvdjfpgeIdPBlUGgK3LRwNY+rvzg7XNbwOBkQmnOaIQld13GRBjElyBiPzZ8oUqeRtong2ajcZo66IM6+9PyJ/mSz7BByfvYGSjPvg0X4wQE6KyLCbBFYiG5gsMNc8GzQVtaLauSpTIfht31sOQdR8v6/Kydd6MxY1mO97tkUdDsYo5Owzx3kSU2XKEyPvLZ+b0f/XDQ6XyHGeHDZ1/3jri3Rzus8OnF+17m5PwKhHRIYj3AWQ/E5GiW0PMkemfTU8hspVqcOHdT97+RwA+s4mIwnyuc5wdRGUzq6DV6d+jU8lIIExbTHnds4PgzgatFI2Gu0LrzgzzzBPyofnya6EUyqs5hs475gm/17hFZBv2Au9IcCmUnMEoIqIjmEiaii2GNF8UxFb0NV8osTn8WonHMmcg6povuiOBCVUctCwaPXemdakGTMFpvojAh+iepAs+qPMORjbq82u+KPICOpovCpgEXyDqmi9cI0FwlwStCG1ouktaZzRgCk7zZVB5Ioqp6xwJu7GMiMhqGcU3EgSFkgSMsZ8dJmsrthjUfJGJrehrvhxrlkznnkITq9FA1DVfuEaC4C6uSDEabXdJ64wGTMFpvuQPfBDxDkY26vNrvhjhBZxlFTEJjkA0NF8MSc5IK+KFNlRa1yJKPKv5kj/wQcQ7GNmoDy7NF35eQFKWxSR4A9HQfIEhyRlJ0LzQhlrrWkQJz0iguPoAYOk7ceXLqrv4ASCf4IOIdzCyUR/KvEPXyTl+wDVEGOUFIC4rxiQMBOLyF5fn5yWEUIUjA9CGausqRInaanM+NF921Jzz04pNKURET3clohXB54nyuvQysMbo0HyJ8z9PZGs2WUexxYDmi6LYir7my6BJRESXO2YbCURD80VQceERbRGO7N5CNHyaLwqti1R1ClDzxQU+OLgHHfBBg3cwsFGfU/NFjRfQ0XwRMAkDgWhovijyEqqiLcKRFrShrvnCti7OcEFqvhgFHzQb5t6oz6354ub+BBaT4A2kQDRfFKENLXcucMPj+xiNgg/mTpX/6j5GE3zAI8A7cJlB8MH8TfgP73I3BD6YI8HkHcyRYPIOpj2g5j0RQNT9aj3qAagBUQ9AIFH3OQtR9jXGCeYfxCNuExwrS6aZZs4TTDNHgmlS+3+FSB5P4D4XrQAAAABJRU5ErkJggg==',
    '1706.08653v2.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAdkAAAC8CAAAAAARYouDAAAr60lEQVR42u2dd2AU1ff2n00nPZQASkICEgWpht4JoUZAei8CgthoFkQQKQqKCChFQBQE6dKLgtJCL1JCDxACQUoIJIGQZJPd8/4xuzN3ys7OhIRvfr5z/slk7rnPnDtnd2Z25nPPmAiG/SfNxdgFRmYN+79lRNTE2Av/MWtCRCYCTIXxXJvPQeX3GAt3eCYyjsb/XXOTrkj/18PNmuNaDpZ4D7ccekmH1r8ZHsjJqQggI8nTlF3Ws+DCfpzI/S1WygQAN3LdbVvOuebpml3SX7PQOwNqP0OzxI7vvFehV7H1naQ7MfgmULoYAFz01LNPM/448SisWzla012alYwEzsMrzE3tPAti7PLYBvB9exFR5uhXETVdaLjWaAWp2+bhvqg5nojoRjsED0+hZzGotl77rDFKjZs6sVulYXeJaMlQb4R8RkSU1gXBQy5qlktxH6a2GcfNSnqfDkzKPfPutEjZTrw8uRd8rxIRfVQDTb/RKGddWOrN44+ODp+/LFKWlcRxzVBm/NSpo6u3PeNADnLZ44jhFh7XsTKrN2Kg03R8gW9tS7VOExVgZoni0IOIKCWyzB0iojWoZSEioicvJ+uQW4jiZpWNOG5W0NsUaSEiS9tIhZ0Y1xWNcomIsptZNcpZhwUcJSKiX9wiFbISjwFERJnRRY4ryymcZ73gxS34ljcxq9uf+N7p4WMAlnKXAo8qVyvg04gXXAGg6EdJHwJAlzbHFwMAZiwqrkPlr14Pdue9WWzLolwAuExQ3IlRw2NnAYBHJZNGudnzf+DOBP3bKgnaxu/1ReaYPPyeFQVhivRxGk3ZqLh/AAArez6vC4Vy2A8ApjlenyQDiL/TSEfnO8X7YlWemyV2nTv31wpT3IlfRXx2Qc+4kseH97YJfKCSFYTgcp7vVKQe3mdOBCwP7gDITb6H3IRMrsWSkI7sjQcZ3zfxCwDg76gCSuTN3des4is+RHAZHvfoY4DGT9KjtqZr82IbsvPaLLGIdTsAwPS+Yqv30pz+OTpCW/Ekxp6bhr4qfnvQJq+ZnbE0oMiXkbhYq8T7wIEawRN2fHOs+5sE4IfXd7035JvQDhbBuZPfiiwAcZVcCySv5tG7/DY0j2dXrXX9hFv48OUlB7AquoQevf2N3Lum/QEAmFsztNqSJfVC6v6JHmVemiNt1mAj0bb514fNcHDjp+4nJ6bqCO1vVLQvei5y7HZyTKsZGq+NieLQ2bbUm4guNSEiawMiatKZiKh+1BKiePxNtNYzlXIjRtM1tvNbWE1Eo64SFcQV1E+hF4lGVbcQUTw63bmTuGNo+GZ7425UTu5k0SN3fQTRPnTn/klwnUV0220bETU4IG/WoLcqEEDJHxR2IsXNJ8qq4naS6F2NcjWwRiUrRLdQa/r0qd1LzreS5isosd07fRowDQDAHRP84voC4e7ngNURAXCtsRrlxNdQvwA5t8oXzLE48HEGEHX6GgDg4qpV65P6XGpnb2zW51yDCbruvKzqATQssyUDABDWciMQHLgJsNZqIG/WYN0TVg6tcO99R98hz1/RL1vHjQaLE4+qH344ZtWx36ITNN6pkFrtsBqvRrcbLKyo5AK4emQAgQ8BuAWI3etF7Lz94ra2BXSW7dzJlHD2MNK4M+sISevY5a9U1SW3pdRFIDxpSw8AQO++t0L+7rT2e88DjZSatXzwevSgfd0mDgpUbq4+YfyEaVq1wo7f55cfBTl0C91UtvUJP43nWT/kcgu5FgBe+z73nhvdz8r83AAAC/DOg3N4vOdjyYXbm9ZfsbZLAWXWurDe8mINHLUWQRFdahdeDilTpswg++VvhyIrsec7yw5sb6vU7NRWA4Cp6YLHp2Q70WZjak4/pDW41jjFL09zJAigaMsrW7VeQYUWS+YWrpYBEP944rH7M1fGyv2C35v92Yi5/SRr+7osuevrW0CZ/fij5eMbBgHWfLmBvnJUdHR0dL+Xd6RyPxQ7Ln/q6dP1t2zyUmp2auu4P62QKduJ9gPkr+79M7Ue2ktusx+6U90cCQKAL+5pzayp48l0LtSOAM6vAoJGdD6ncL3deNGXi9+Qrn2x5ZV3+hTU3eJZ/csD94GdB/NBjc5XAQBTT/MGbkWfuCnt0WfLqjbKzU4PAdzp/zGqyXYiiPsoVvzq6m2N0fnMSZ5lW1zcTy7IW9ZeU5TmXz3flhwOACeT6gPAvKcAzHWBnAwAMOcAoBwLUPajhSs27EqT9n4TcQ0LKLPuRXIBxLln3imKNMi2nAazHrUVj7i/r2Mxd66JDt70GhoHz2is3OzMLL1SAWD+my/Kd+J5212KEdrvo3SZ/fkyAoDtppflgvbBZg27Oamq1l89REmvd9zx55h3nhARbWj92YptY2bTsfZFi8bcP9guMKj9kSXNfMO6PkqrHlouxMut2xNx58ygKZQPpvgzZV3ItO2T/3mn3ph/Or/i69/sQ9HPgs4VfYOix2mVi63pX6R6IhGNqurj2+B9IiL6YDIRjfnIUbOT8KrsGjzlj90jOqXKduKx6ADfeku5RxkDtI92b2SdpQdWDFmhkJULb4TDv1Pv3jHVW/3lQE75yTvFn8yp8zJ3j6eI2xVzBS/5RyKz9qK6gPVk/+4TxA07XyuOgnoWbbmUU8mD0gMK5tF2qpsvkA7/vOmdjKS4E5bIGib5TsxreBdPPCoT7Q/dgvas5u2B/vqF3N2ZeTu2wGAq8B9iKmqcSwQAy5+tYRgKJ26Uxw/M8emVanvH/914mMn4zhZCuWci3JITMsuEuxiE238vswa7aLCLhj13c/0CQNPCGFnTQi1XyMNryt2pmGB8wP9jNsF2p8IwGDO2DDMyaxiMuXiGwZiLZ/yeNX7PwpiLB8dTypjlzBShPcg+WeBhsrubNYci0qVzzTKYj6CLN2IDqzyHoVhdnlen5xtoHkNUyOzYe5NKnZ8QsrYTu3x93qHTQb19kJ107OZMOzSYsO7Q/oDuFSPubry4wvd0eQC//HWqaduPUO5Rk5IeexMiGmbd3Rd29WZjvwceAHD/gwXck9W+kbW8bx5uqXcigWr/vfPWAHi0MD0z572XtSpynWD9/UKOy/s8hP549s07VYaX1Bep4GldeivT+hYzl5IWn/N1/dhHvE2dIdrTbI9UrKmJqWCnlLHLR9GBYyYGDBecT6MjBzSI5ppZAq8Q0beYS0TnSpC5X8djRMmjh9TCHa5bSQDu4626mAon/TPKxxBR8nvJRN947NI4F4/rRI9bjDfTsvb8lnpco7QuAYfVIpXr8Z7W988S/dvsMN9ibtvbQqurJou2qXXmodidj1SsqW2WZZePiIjoaKRkmYfUH/SUk+tx84fbJli+S0QPhhARzcR8IqLeT4k2XSDKupE7056ZFnM/XZSgk5Zx0n9K+Rgimum1higZ9TXuOq6TtUtvInqphH3tsAQielw0PFclUrke77nlOyKiM9F8yyT3dCJq8Da7Tc2ZFbkLkYo1leTkR+PrLsKUMnaZsxzyKKYISn+147M2lexHzepMQ9X7ZXGhDeBZVlgX/I7+84Z6/yPhxQGgFAHwwUNtkrZOu9edBLCSn0+1deP5IPg22XDpVT2R8p5HnwJA+et8y8KqfgDqLpjhLWxTq4ndhUhFmhrvVLBTyuTTy2J3Aq3gZK6Ze12moRFw1c+9gC9Ksn7npnX2eNwVOIjXdXX6MaA6gJr17OtDss0AfHE/b7G8+N0sK7CFnyeRmuQLACWenGa2qXNcNuMjFWlqvYIauaZtVMvGkR5NJMu2zEYCHaA812zq1M+5RVEhhnrA3qFS5zO7c90GBuQ9k9L+339gAzvcAfOMmuM0idg60Z7wWwt90wZG8GPM9QBw1q2yvkjtngO+Gbl+0eXlK+zri7gS9x26VJ8JFHpCtF8T8ZGKNDXPxWOnlDHLcQjp3a0utijOEFOaa2Y7zzLGnycrr7DS+ogbeqlUh/3/WUxUhzsfHfm41tAnmuTsnVLRYLqFrr8oxjuPYZRapHI9wTOxPjwaMVUQqr1KRPQOZogC1XKelbgzkTKa2utU0KOVQyvAdkEkLMehM1kSWzrOLJ1yezVLW2bPEBG91inPmZX0N4+0CHvAerl1u2QNcnynm3BNJKIB5TLZ67Uq0Zlqkcr1BM/4vh96oeUd4deG6Q5Rdj18LQ7UeWal7kykgibpmGUZ2OPHy3tKTEyVLgMuoR8AsJxF6oMHDx6kyeaandf4qLcqANRcn5zXg7Gk/5x3mWGYIpbvbGNxrsF38kFoKIBq19nZVGNLbPTSFynvGdfvu+nnmu/swM8saP/tW7euT41BcXGgOkKELFJBU/sVFDulTDS9DABQ41Xg0jG8XqJEiRKdpV21zjU7sh8A/HA+j4mV9L+QXSw1NTU3J/Ux93+xWif+dKohdPL3KMrN5GPCWfzvdh99kQqeb00rjvK7vjgmTJYfNWffnpH3UE0aqOYQbSvYSO2aOq6g1nUHYJtSxi5z9gKAI5WwNhv2+Zas2K81+jfWEHL7tMcegBnF8phZSf/kpHEALgSPK/dew9wjHkBRxDvVEDqNqpIBABbm87/1zG8uOOdSSUekvGf6+UYATBP+viLMIi5bFkgMq3ZQ2KaWYSZL3d3YSG2aOu5BVeIqEdxFkniZv1NhrflQdp49O4/7fwZaaTjPRi8mIooOyMnjeZbpn8KfHEvGEKWZXG8RUQ3s0XiJUjKGiCa6ZxPR57hilzv0qZWIplxViVSmx3ua/dK4ogO7yKa3/41UotTAxew2tVe/srmnZIoilWhqOs+yU8rYZftMt8xRt4Up2GbbSqW5ZlmQzt3Ptq3pUhHAjX0z3XR+V+X9H4TY5/1ZMtIB/7a7ywA3zzfROOXNkpEO4G2fPwH6c3AFm9ylPinD3h7a96cwPZHynu7dZwHA7YeNYdPbuvkWMCu8L7tNzWZz55SESMWa2r6z7JQyYfl4+1B4v9G7d9c6gYiyu+7s+LJfYNQwpblmW3s0LelXsmkP4adEdq8WpfwqdZhJRLnjZ51e9eoMffeNFftnVO7Dtb5d1zeg+deU3O/bgwfqxNzR9qWwdaJjNdedeLtXul3uNdslkVqkMj3BM6v/kL1nFg5IJLveubb7//kw6p54mxq/s3Z320D5SEWaGufisVPKpNPL8s0uHClar+QzPIt23P/0GapR1aRTLnXfg1pVdW1JQU/wjD/+tGotJoaUP1Jr1DPly5N3PlJVTWOOgMFUGAaDXTTMyKxhRmYNMzJrmCizhZIkb1Ko5Qp5eE1g/OoxfvUYhv+fSHLbu1W8Q91kr4Zxd6fcnNIBiu+JySewPF8g8HwmyQW5AkHUreT6fEjy5FWx+0sP9siISxveSTSQu8v2HfTtFN4jgP+nT2Q55Pz0x/moNh/mHSznOwBiwtq6KMnr6YgS0AJYQ45pM+S4hNdmlPVR3yL+WxifToReysXHftq2hA8A9HJxHtizkeRx6ElEtMtroEX6apiOjt4Tow0sVyXJJYS1teckopMR/6oA1mokOUuOi3ltRlmLnhCTmP8WxqcdTCclLn6+LWlNHQeWXyR5PLja572w2lHJetE/vbgPjwawXJUklxDWGwOyiWhIFxXAWo0kZ8lxMa/NKGvRE2IS4+LC+LSD6aTExX+wYMf+2Nj9TRMcB5bfJHk49nbTdNEGAHkDy9kOEsJ6RQMPAE0GPvV2DlgriLDkOCAiywVlXdS3BBcXxqcPoZdx8S5DAGDpoDANgeUTSX7bDuSYj593SpflA1guIqwtuwIAoHT2Xg2AtYKIlBwXyHJBWVdMOnFxhybj4t8DgOuHemsJzEX15STyF5XEQokkj1/TfhAA0MKOT683dFZWWgEsnzl9ZhrySlgnP/IFgABcEQBrPSKxd0qCJ8c5XvuzqWOuiJR1xSTDxYXx6RuplIsvD8D6wTSTlsDkR+PaK97evRslx70nWQaO9cm5eURaGDVxHT24cGlePxMAzP/qgj88X08IVI13sCyxF0aYNtTeWVbziE8VD2H+ewwfAHBFmmflFAC4gVQ9Ii4eAI6fGWW70ExPeWXtZJeERkubM8p6YhJHJxqf3pEeXb/ntXXiK/2VVYKgJTAX9ZeTiF5UUnv5ytUtpd7Fataqb02tYAKAtLGd/YFolxU6jzq/9TSho+8ozf45ywaIDoPw4M7hmZh04S5gPmV/m4J2kexB0V/yb+060s0F4S2GZLHK2uUk0YnGp3ekdaYtT+z5gF1j/nSYeMh6ngjoIsl9w8KqzY1ofgEA/knzOHDgwKHi7CHivDn/wXIJYe3HvdomF34aAGsHInZyXMxrM8ra5eS4uDA+3Qi9jIvfYgoVD7lgSfLOWQsAIBHBXl5eXr99wsj94ub4jSTIG1guJawDuU9uFgI0ANbKInZyHBDx2qyyVjk5Li6MLy8IvYSLXxIuHXKBkuSBOAsAFVGsplTuXxeVN5IgT2C5lLAOKHUPAO6iNjQA1koiPDku4bVFyhrlImW4uDA+fSM1K3DxuXujZUMuEJLcdqfiJIpaKSMn84UeRERZmwTvyw2IiAZ7cnT15IOawXLVGVsCYZ2SSUSDXiMiml401zFgrUaSM+S4hNdmlDXpCZA4y38L49MOphNJuHgbMn/G9n5nh4HlF0lue7dKqOvD61j2wGvR1rMAZofw3ok9FF5xkmewPJth0znCmgOsR124DtD6Ua5aAGs5pi2Q41Jem1HWQX1Dwn8L49OH0LNcvB2Z/9f2ekINgT0LSX6mc0W/wOgJRDTfe2zCaCLaXW/s0o+22b27NXDFSNkbSbSB5WokuUBY2wDrVU3OPxzdx6wCWKuR5AI5LuW1GWUNegIkLua/hfFpB9O5+8YCF29H5jdjrG2Ws4PA8p0kv7eT2gcCwO0nEUre6m8kUcC19T2Lvr89o14NExwD1vrkGLKcV34WPWF82sF0AApcvHl5TEnVwAySHAZTYRgMdtEwI7OGGZk1zMisYQZJDoMkN371GL96DENhJ8kzEri/XmEOb24+Ov8gsKljyedTgdwwvZlNWX1wT5k3vfHg7xemOijdcHv50g6OMysHxSUVv/NqLFYtkRT4cRkUrmK8iKyIOBO7ZpJcgMTlMahsyZGp1EUXdoRqBXb57eh4DCAiyowuctxRxZOo7o6LwshAcUnF77y+553FqiWSAj8ug8JVbrnzItIi4kzsOkhyHhKXx+BwSyqjdVwXXdgRqhXYFTJ7C4OIiOgAmjva7a26qyVFDIpLK37nNbMMVi2VFPhxKRSukglBRFpEnGHWdZDkPCQui8HxlhzLqdRFF3aEagV2lSeFIbict8OmGBSXVvzOqzFYtVRS4MelULiKCSLSIuIMs66DJOchcVkMjrfk2FTqogs7QrUCu8q18R60AWB5cBuA+W6K7WHcnnu2U8cNO0Fjg8dz7qYg9QlkoLi04jeeHauWSDL8uI5y4oKItIg4FBh1HYHKYtCyJaciSjtCvQK7w6PxiVKt0okONEArogVlMZmIaE/kj39990Uuteq+86tV7QZYici6oO3ezXUP0NZXMHXOpKC9RItyWLbFWqz6jbFffXKZnvVoTESU3bpmqlwyy7UxEdE3WEyWbCKiam73ncoxIpnhaHRpU0yanMu5g7c4GGim8/B6n/7um+9SiWQxqG/JgRwrwitLd4R4Sct5ttb06VO7l5zPnbNrtSKibLfJRLTX9STRBu9d1Kr2KqJ4/E1Ec19MI/oz8BFZIzoduFNqlXTvKFf8zlNm7eXGZZKSAt22cuLqcqyItIg4n9krGMG9u+YLPTXJJTGob0mNqXBUF12ou65Sgd3xdzaxYdR1IqIWrYiIfCYTWSu1IqJd1S5SqxIWolz32USpAR8QkaXoXKLIako8mmLF7zx+Z23lxmWS4gLd9nLi6nKsiLSIOJ/Zs/iYw8o+0VOTXBKD+pZUMuu4LrpQd91xBXbH59nQTcdaPxadiZMuvAog+vQrQCUXwNUjQwyPK96gUKz4ndebZhxWLZMU8+MMFK5ijIisiDgUGHXowuHZGDRtSckc10UX+HLHFdhVrqCKtryyVbTiKkrZF7mwLWJ4vKiSilLF77xbsVon/lSQZPlxFgpXMUZEXkQckDPq0FEnXRSDpi0pmGpddIEvd1SBXY2P9MU9/jorF0AF3JL5sPC4Im7lJq/4nSdjsGoFSYEfF0HhajffeBGlIuI2006SiyBxcQyatqTwu8dBXXRhR5hVK7CrfGez9pqiAHgSgHvZAF6sfAoAsjcwTtVe2AUA2ZsdD/maGcA91Hy2zGadOHsfwC1UE0k+zAIQ2zENSNs/3g04fGi2C7DJ07kgL1LExM3TCK1sk2OOejEJAHC9aKRTuWrzPQBcDKgoiuFhlqMtOTNBhFF+mMXuCHaXaPrVcw59uXoU3C+d8TWJaK7/WCI67LqNiGYdo2ZRRGT1+IKItvmeIaKv/yGq8qZYZxpuEBHdC9xMZK0z+FmvjWP2EVGiR5NcVjLZO5KIPnaJI/qihpnoYrkhQ4cO6ROm4R6UIDJ4IhFRUptsm5wQO533ukZkrTfFeXg/HiKiBPefRTFwekpbcibHiAjKnJywI5hdouHa+MIb4fDv1Lt3TPVW3K+K5IYTt0/b9qJvi0dEh1/7bM2kdQfbBQa1P7KkmW9Y10c8PP5HW78SrZcocd8Ml/1M942ZcuOSIuICP86UE3d68cmLCEXEOTmWWddMkguQOBODlExnypU7kVOsi87JCTtCtQK7lifvKTcq+J/1K+FjAnA3NUJ2AHcAj0NDxW9dz6IZrFoimbei34KItIi4EqOuoya5/i2ZdFVgF3aESgV2g6kwmArDYLCLhhmZNczIrGFGZg0zSHIYJLnxq8f41WMYCm1N8tQk+1KZQCfMOP4bCLmeSt//lzO7YaBHdIWSJsy5vbaLOjOO54SQQ0aSS4FsoUlPDXF7eEylb1lNcu1jEGJSiMG+S7SHx5Dk8jrrdjnVEuey29Hfh8YREe1AM6sTZrxgEHINJLkM/eab9NQQ58MTKn3LapI7HIOcmONjksUg7BLt4TEkubTOOrOH1UqcyzP71RIioidhnlecM+MFgZBrIMmlQLbQpIP8FsITKn3LapI7HINMT4hJFoOwS7SHx5Dk0jrrDOiuVuJcfjR+0hoAvrgxpQIKB0IOGUkuBbKFJh3ktxCeUOlbFrj2MQgxyWIQdon28BiSXFpnnQHd1Uqcy6+NmwYDODXz1Y/4EwjPjN/cfc0KALnJ95CbYFtrSUhH9saDKDCEHDKSXApk8016yG8hPL7Stzxw7WPgY1KJQUd4Akmur8662hVUCwCWIdaFHrb/d50o91uxn02A+dNKVTdsW1gBB4adG9rhVHlu7Q/bOx7wColpec9VUmuc9oTfWuibNjAivy72hALdA74ZuX7R5eUrZE1CqW6noBETHlfpe5lJ3qJjDHxMKjHoCC821wNcnXShznp9uduZ3bluAwO0zhEgopkYZp+bJTDjP4VeJBpV3UJE9aOW2Nau9Uyl3IjRdI3yCyF3TpIrANm2Jh3ktyS85WOUWhyPQR6ePSbFGLhdogdMF0hyCSfPlhaVMebqJDlRok9p+4wCgRmndUEniLbiimhtlypE1L1MPiLkzklyJSCba9JBfovDyw5JVGpxPAZ5ePaYFGPgdokeMF0gycWcvCizcsZcnSSndzN+CACwBWCYcXROiUzYFMsVR+XXBj4E4BZQwAi5BKBWALK5Jh3ktzg8rtK3rEXHGPiYVGLQA6aDJ8lV6qyrlThXyuy6re06AcjaCzDMOKwL6y0v1gBikvydB+fweM/HKGiEHCw2rQhkF6t14k8d5Lc4PK7St6xFxxj4mFRi0BEeGBzdYZ111RLnCiR56ge+c00A4iXb/3jhqfI4AFhNDFIV/N7s4Ltz3yhAhBwyklwKZAtNbbST36LwbJW+ZS3axyDEpBKDDjBdhKM7qrOuWuJc4Ts75u6XIQDwa5D4dtes/uWB+8BO9n08exov+nLxGyhAhBwyklyKfgtNOshvUXgXnthmrojJbx1jEGISxSAm0/WEJ5DkDCcvAd0ZxlxLZg8sqPUuAMtv3wcCMOcAoBwL4F4kF0Cce+adoszash8tXLFhV5pS6XCmtHc+GFOg2737LAC4/bAxV6qbadJRQ5wNz17pW1aTXPMYmJiYGOyVxO27RHt4Qp10ps66VE69xLnswiwaVWJi2lT2BLaQiBlfFzJt++R/3qk3Zj+zNq16aLkQL7duT/INIddAkkvRb6ZJRw1xJjx7pW95TXJHY5C/kFGAxIUYZGS65vAYklzg5KVyqiXO9Tx5t1zKqeRB6ezpN7P2orqA9WT/7hOQTwi5JpJcCmQLTTpqiAvhCZW+ZYE7GIOCnhCTSrnwvJQ4V+HkVUqcPyNTsX4hd306b8cWGEwF/kNMRY1ziQBg+bO1ATGg8OFGz/KBOT69Um3v+L8bDzMZ39nCJffshFtyQmaZcBeDcPvvZdZgFw120TAYJLlBkhskuXE0No7GMGqSw49nqG48AcJdUoTm0q4KDtZEAIB3qJsyU14oUe18DkqQcyBsdfkfZ/bh6sOxdOMF7p/7tZOj6g9Jn3fodFBvH2Qk7ku/9LKCQ86q2P2lB3tkxKUN7+QCOVMuR7XzZo8mfVIKwE/BtYNdHu5JGcJ+IHefT6nawx0A0tZdcXuls1M6kAlKynczoLpm9FuQUx7t3nlrdJHkLDcv7SSUOFeTk9+OTuiOr2yL8/oinojoKDoQEdG/Vf9QdohDTyKiXV4DLQpMuQzVzhstk9sGsUREDQB3L2A507QkqO7yv7s1zSWiHf7Nt2+sHHzMmZwQlJTvZkB17ei3IKc42ozyMfpIcoabl3YSSpyrySlk9s7k2i9xjw+s37yLW1zibO+L3rhA2cH27mjqhdUKTLkU1c5jZj8Fn1mg9GqmZQ5amympCA4RJRbxeUKUaCrxyImcEJSU72ZAde3otyCnONop5WN0yTHcvLSTQLeryik+2hs8ZH8TADjUIFHS0uxrdYdw7O0GSJlyKaqdN/t9o32pXcni1dox81xujMJEd+RmIRhYkNnUBwgte2PZ+05+7/FBSfluBlTXjn4LckqjPRJeHLrkGG5e2kmg21XlFE983b0XAwAOiQnqLVfhX1fNAbjN0ToSplyKaufJzn2+xL7Ya9HUHuwEpvlml0igbPyV8sB+Dh8rj/1O9PigZHy3AKrrQL+FMSqMNuv3noAuOYGbl3Xi6XZ1OcXM+ndblwYg3U98m/9EFtBOzQHxa9oPAiBhym2o9rRnemjwqOevwfy1QezsBQeEV75jF0qkjBvwZXYFgI7DGwC8cNSJIB+UwHfbWgaEj2x6efPyLyFv0iCnNNrvPzABuuQEbl7aifaE3/ps6pgrzuSUL1YHZa4EsL6TsCbhp/kffqvmkLhu7fz335m30Q0ABssP8iurBD1LYi29PxTgoU82l86KqX5Y2Da8B0aPPlN1ApCdzWXWDakaVFdWCQIewwcAXGFnfrz21o+t+u0Gf4UmDXIKoz1VPASATrmjnzQsuzdA3ik9xWft5E+HRv3tRE4ZoWkQsfhtIDlYWOP7Uo6NFXLgUKwm0s6lVnD0vTR/euCZjsXjX+rPL3eNqgJkftr+gv1yPx0J85piztpJke3N/Ic127koF1QWPLgavpn8+vL158S+vrSUQpNzOfloc5bZvhO65OrUjh/e8+fisk7pOLLCBeEthpxXl1P+zpoGnziLs+zdhhJNWwwbqebgGxZWbW5E8wsO4mRR7bxcPR2aIfwzvAqA1/DgR/uKILi2AIL9sQRFgAwAeAoN5au5oGR8twCq60O/hTGKRzvnXZe8kOQ2bl7aSaDb1eUc3Dro5/YTdraQfor8nDigc9YCB2GyqLZ+uzxs5MkjR04B54+kI7brWgA+wFn+Y4fSrgA8cQbugfbMBjuX5YKS8d0CqK4P/RbGKBrthexiqampuTmpj/XJ2bh5aSeBbleXc1CTvOTryye7S2+SNXPmgEBhb4tNhGrrt1sR0wHcB+YE/FBj/L6DXYFcoDiAtJuVTYi6wE3+RDDQcGsGAGSikVNVW1BSvpsB1XWh38IYxaNNThoH4ELwuHIjNcsx3Lw0BoFuV4/OTemuFIBBGwfOcLRZxw6BOEumpx5yUR7VzptFRwPA8r6Y3xAIcekL4CoQBSRXuffOXPSdc+exH7LSUR9otTUZgDUJrZyq2oIyxZyCne9+6O2FIqbH/gAQWlnUpFVOtPTQ2wtNmgDAuspzAM1yWSdc7pfhuHlpeGg/xewB3ENN9egUjsbnLxLQuvTjMABpSAeANJidOaQBQKjrw+tY9gASplxAtfPDRpfpDjz5AV27AGfvYQdQqxMtAH63hI4HBlWMOwysv9+ok1Mhe1AC3/0gpKEICtdDpjNj5Jd49BuWjHQ9JDkDx0vDY+h2dTnpra2LrYr61vyBaNo6oq9bFvN7tcul450ifAMbdT/q0OFM54p+gdETiGi+99iE0XKmnEe1n+HuYldfPz8fX+99RHur1GwZ8OIcMxFltvBfSEQZI7wb1PfqeZOI6G5X3679/QalO5/GyAfF890cqs1WDtdBpgtj5JdsZDrR23V9A5p/rUOOLTcuCY+h29Xk8vvJ+72d1D5Q4bwhQbWf8Vn0/aQwydH96RWPl+yz9NNvuIb5aJATgpLy3Qyorh39FuRURqtdjuHmpZ0Eul1FzmAqYDAVhsGogmuYkVnDjMwaZmTWMPs9qLKmwhiZqVDLFe7wyhokufGrx7D/QOVqALi/8SqFvhFSCOPNJx47y9NUAK6FPrOWKSu+6+dy+PVm07wKW7gcj81i1hL0W7VKN2+Z3v3C/fyKmIBKrwGn99wMfiXGg3VI33zJHNa5pMzVkQn+iiQ7uwVNoHvSpuu+4W8Eyih56ODm5bejcztVfURE9LRF46f0PzPFSoQcj81i1lL0W61Kt2BC0bNfKbtr5UlfRSJkG9O+u/r07ZuHeiyTuDrUY/wVSHbRFjSB7j9GzN411dVntZSS18PNK2R2Ek5yC0nuQwtZZjkem8Wspei3WpVu9rGMzfpaaTzetpI5CkVOCxB+qZVEREPdM8SujvRYfwWSnd2CJtD9hkvR40RRcE8TU/K6uHl5Zu8W4fdYd9P5QpXZw7/ViSEiWum5hugpXrGvHzeKiOhJOSLqrUluZuTanftj99drkkXUGEgk2gaujikREe3DCCsR/YzjYldHeqz/WG7WRxi+55vZLSg0KxwCgGlEPYBTRAkeOEp0w4Sr/OwcZo2KnPyCZEUmf5KKol8Ay4PbAMx3UwDk3E1B6hMAqYf3mROf90nWxmOLypMDkNcod2rXundp0ajh8YfrPYEO7k1KAy8C8XyzK2a1PAvs83lJ7OrIWH8Fkp3dgibQvdHAJj2B8yheSUzJQxc3L/vAtMPP9sW/EEkHGqAV0YKymEy09RVMnTMpaC99O+v80c+LPe/v7Nc3ifvOEhFlt65pr8FMmeFodGlTTBoR9T793TffpTqTO3+fiGL94rnqa0REK4BJwomxGGAaONFrtdTVgR7jb/VEdyKiGIQw9d34LSg2K492I4psJ6IaKHnvs/5TmIMns0ZNTp7ZGlhvXzyJYCKq1YqIst0mE5E1otOBO6VWXWpCRNYGzzmz/ywmPrNCeXJJjXK1Kt1iS39xvPCPpSHC0oR//woAgFFWJVclPcE/E3iTiKgD/MQduC04apaGZx3Z0rXGRSKiogiP2XO2q+vnfBuzRk1OntlI/GZfPIBSRNSiFRGRz2QioshqRET7Ak4R0aLnm1nzSIuQWaE8OYlrlKtV6RbbeDAV8iejDjtXMa52aQBol6XgqqQn+KcBg7h97SHuwG3BUbMsvOsnZ7i/9Ds3pW4H0T1gk72JWaMmJ8/sG5hlX9yA2kTUis1sHyKizGp4dfhf9Hwz+91VYjJL9MCzpv2HwNl6yXS1OWpbbP8PwX2ncg+LvCBc7O506cfWkz/q0y4jtjaAiXJXJT3G3wz0ICJqgSCRv20LjpqVrhfHAL8TlYBrLhH5o6N9PbNGTU5+BRWNf+yLJyGjhIsCgNe+z73nRvezPs/LJ4HHBlueHACLfqtW6RbZ+sxy/L2l2C7Tl3idnsO3jc79xbvhkWVB2CxzVTTG3wHJbt+CNtD93zknANQFvhZR8tDFzcsz28v/DxuCSlvd3uIvtHKZZxLxjyceuz9zZezzzGxy0rhx48ZduDxuEcy1XzODw6xF6HejK2jfwgyHVbrFmRV8DvXfMMqErZcBpMURgH/KFwNMfY6YnkhdlY31bwiGZOfk2C2Imh1Zs/ebpAKeQAoQBYGS5/TYNSpy8swGTbu/lFvaefqTMACeBOAeOwHq/CogaETnc88zs03mzJkzZ45/5Tmj2PLkeJjF1gNXr9ItsnjY6cejrSPWvD2w86yXgOSXq74H4OUkMwCEe9eWuDow1r8VBJLdJsdugW12aBnINQG3gXpAX9x5DBslz+mxa9Tk5Ad562j/fURE58r0zCUiGl+TiOb6jyUiqvImEdGG8Awi6njiud+DyvVtREQUs4+IEj2a5FKydyQRDZ5IRJTUJpt+PERECe4/O5crAtuL5k7YZ8VsI/oLCCeixRjylMj6ZWCi2NWRHuv/tCIOEa1FIwsvx26BbXYY3hR8Q5RdB0VvEVk7YTrRcoQ+suuxa1TkFN/Xszx8/NEz35b9gXNPbjhx+7RtL/q2ePRHW78SrZcQbWj92YptY2Y/93tQdh5bwKyl6LdqlW7JT4CfuAV+ttJlHkyn3yqXaPF6WIfLEleHeqw/Q7Lb5NgtaALdrXPL1Xi9pFfvmySm5HVx88pP3i0HL1krNOafLqTcqOB/1q+Ej/06Ir2I2xVzBa//4ZN3BrOWoN8qVbpF9uR2hMplESXf9gl3V3GV6In8lUh29rmQBtDdej8puIyLAiUPzdy8wVQYTIVhMNhFw4zMGmZk1jAjs4ZxD42/ANC0MEbWtFDLFfLwmnL3oCYYH/D/mE2w3akwzDjPGmZk1rD/tf0/iS1AxPDYBeAAAAAASUVORK5CYII=',
    '1706.08653v2.5.png': '...',
    '1706.08653v2.6.png': '...',
    '1706.08653v2.7.png': 'iVBORw0KGgoAAAANSUhEUgAAAh0AAADqCAAAAAAjA6izAAAw6UlEQVR42u2dZ2AUVReG300nnRJCBEJCLyEIoUgzAemhSA8EEAFpKiLYhQ8EBBQpSpOmKBAQEJEiiFJDNSCE3kMJNZAKJNkke74fs7tTdmZ2ZtPjnD/MzJ179pzLydT7zKsjaKaZhNlpQ6CZVh2a2WJEFKqNgmYWFkpEOgJ0hXXtoStZFz2FOpb5ko12ZtFM2hxKYlJjhza1ubUI2vPdJ5MC+lWljf1T7zs5GLLsqyLnmpNDFvnFMTu4BDho1aHUEleiqa2tRfDCcOX/Oo+tfvW72u4L+j9cc/CI+6CQqshauftC2879fjmyv9Kbrniy96VZwfl1VQoqJMunH16OcnpbW3ObTp6nZBjjdYKIiH50CCGiGIQz29OaGYjoGoYSEaW3KxWTH9mUxOuOvwc+2Wdra5Gzb5cuZA51b3QBABe4MNvdq+kAuMAeAFympn+iPe9QZA/KDcYGG1uLnCVMDow03kOME9xTcFcq44pWHYpsY9/Xyv6WaVtrkbOoZ+Gm/6JW7tK77UdnrToU2aHWjn1TdgPA4sb+DVavbl75lT8RUan6IkFrsbC9qGNadF4hudepTzrO1a5KldjN8UQH0Z+IiOLsFxDdc9hJRC0PW7QWh6vShtjIWz+H3salSCKiu2gyZ86s/r5LDfmSTcm7o90QAbSqtP25G4CADlvfQ3nv37vA0KSlRWvxeB6VI79D8AcA7kRuWhmoPe9QYNsrXAIC47dHAEDk4LuV9/ba9J3z4dYircXAAmIem5eTSkvt5f97lU4nPbTrDqt2sVblSpUqDTfel/QotR775+Xswh9dRFqLgXXCafPybAAeyGZWsnnHlDIdru7Qjh3Wbf2E+gBo1q5kbwDuPde+4+zWd11nchFpLQbW/7Odmc7MYrIDAP+yCcza9Uq8/dzxSLtnsX6RfaE+AOgG6H8DAAw6N6M7Bm3f0Fm0teib26KEBcbFVUMA6HqeSgUAbO7J3S3jgK5tgdyzPEtj7bnyS9xnmz8ZNfsG0cbCvWdZG8b8exItc4iIssrXNVBO5fo5oq3F4Ek6fev0s4GIaOdcIiJK9h9KRHRyFBERncdgIqL0oZieH9lYzknwTQr1dToQV7NVxsODAdeliurxuGVe3BpbNbnT29WvRVX1nXkGhTe/4/D7V7Nq/e4PTPz7hq5Bo+8AvOczCfg052vxVhSD+R0HJzqMrXbnQNgAZvXeaMeRdvtTv3YDLn0WG+fZrhSS7/l++Fp+ZcPLyFD2nxrA3A8WjwUutHks2vHJ7LTTMQ8qcIrjnbV7mgHA2jfrnSlas3+SHdyBVHiiGM/+uXQyqVI7cwZ07VRWs1oFko3FVWlSvxoA7GEHoF6H9FJiHT3erbQwhrvh2yU/NWNO81uvF7ETtzeAAqmNfLQ6dXj/bTVrorBmHT9+mfuoRfzY4VzFHvxXRQGDjIvvQjOU3Dnpjq9wVloDdO1YOgBkP36MF3EGiL8q6mry09xTG1OU3JmD1bkrzfHnosE+H3pNdfzjk3NjKyRV2RYyw4l3J9W+wSLgb/ZVkdMP2pj+Z+aVbvvgRGmEvT10XZcudX/dE4wRr17axp1YkHIiCcA9+IhXl2YlmXZKf7t/aUA3Omov4NopGHD7344tvNvfuKMAHEyPdzUreccOadrpTHwQAATpdphup9tiZ2/uHhUB9lXR0kOODjn6SfWU/riXrmQNpg6CKVvF2aoYq+Og5B5XmFmMdk6XzDezHtcsd+u06V8AwJjB/4aWu1RWcQApVGSoqbx4TpFHzzuKSjZWzyzV8AwAsjLNVxOpaSLPYfpX2JkBAHBviorlStgBQbsqlbJGPsciAcSgEwA9ABxEF94eia4ugNvi3nMmAwBe5DYeE8/zLN7RkfS66s/jAI8qptZbz4BAt2I1wIKE/IpXPuJvjmZhARERbfG5QZTdrR8Rhbz0gOh5w17MHLXZuEVElOAaQkRECx2W5RARLS3XIHdv4a581hLuo1dQ3OdhcB65mO5Mes3J8Z6x8ZEP2k66ky9v85Bfb+EECSnIp4hlI/S1IyLM18M3LOJvItrTZuay7lP0RBQy4NOVUW0nZxJR5sD2FTzq9phP9DxoENPnQOMGKw7vHjf5dmRukzDxPJcQyswN7Y+ZxqYlg3EtnwYD+feOVpCQ1XyKUDZiZ5bwcHa5ffv7ycOYnZxm3kn70wEAnNaZml3PGRdCYy6fjK0ztTTW5vZgZuJ5zP8Gxf3wiQ4A6JmnifUpTiZIqDjlY332z0t1zRXkX0/mMqX2oLFtSudLjCOuHwIAHG1ZMq71ik8+yueGZRXaE6/+rquY0WyuvI/hjt68nHYjU7BQUC7yKp/CykZpdewJv7zj9UuFUx2e/TanAEj1UH6vfKTzljm7Bj8EgHuRc05N28JdKDAXeZVP4WWj8BomO5soK6cgaKdzqBwZGRkZ2QMdiYjowXSKxlIi+vERvY27ivxsfimOqAueEtHBl/4iQ0QLzgLlkwupq1JBQlbzKWLZFC0WzkR7xbHVYajZmIi+JqXVcd9jFRG91YCInvrOIMpqsJhdUBaKDS6kq4OXkNV8ilg2Rb466GvEUuwuxdUxzCmZiGqNJ6KRdveZbeYFZaHY4EJ5dcjnU4SyKR7EwhCHldjTXvHuu9p4AQ+uhAHY2caP2WZeKDAXeZVPIWZTPKrDt+vaFEd7pXs/fNAIwH7dq9nPH9xryGwzLwC4t/nPF7lyod95BcB1KpB8CjMbu2LwQVUAw5OG9VBxl14XwK66pXde9dT5AwCOmxeAvd+1rDAyVy6mVw1NQVLdlALJpzCzKXrVkYJU479MvBcuEdDJLy2A0yZvPvUSgB1/B+HIy249rgKgeXHmBWBOVV+H1rlxcaVRzjNn7K/ubVNCavMpzGyK2tdsT866cN+ucdOZ56devKtr/PK8y+Nj9LXfeAdfVe+Nr/ee0vvXmVHLqp9/Jk686FNz4hjdMDwc1q729Xv9QtgFLPi4VM8FXrlyMeXmGrxrWKxkLAUJjbSeTxHKpkR83cXCT0ZsGtH928zdXGw6cRcePEvZUG9erlxQcBRRvY0JWflCShaxbEpedchZtd+JpmzNXSi+MXQJd74qoOoovGxK5LeOZe2tlD9ueXTPnY/3V96Mb7G9rENJz6ZEfEVfnR9DQnldbkNJtvcwJJYrCvNK8z2b/1h1aLOONY0FzaB9+0czTdtJMxSKtpMdgLDCuqOdUnT8TMmjMKaUmGzCoF2Valel2lWpZsgbFi7tNvOvUyVXWKplmVrLVsh/HtJgrlwD2avsau6guquqwNTHU/imJnqL6kjYEH2owgg3Stj/0pcvW6hlGVv1F86HTvHN3ywOLNloWoz+tIuPGwAMVJgZ28GiK606727/kVI6Me3bOw/qv+crFZhSs4hicEgT1zvHOij8xii7t2U/wccf1QwrYFgR7/JivI+aWcfnEEFEZFjs9DNZqmUZW5+GVHqQr28UnlcLNy8vNcYaptQP20HYVd8lMod+CU5QFkpCxA1K6eN1TCowpe9ZLBLwBeA42aBwVNi9Bf0SJo5sggdqBpYXvWHANKJTNe+rYeGMalK6salDA1vBqJYVta8jr7XMhxEfrM3PQ8d8zvKlZf5uOtD/flTame0g7Dr7r6d26Pfd5KWK/PxvVgA8f6wy8Jq9eGDq4zFa8OvxVdsFKO3O7i3oZ/HxR1XDim1/rAYahY3bZAuDP37u2FgdADwo1y1qQ0d+Y1Ucys/iOB7Iee5vNxIAfhqueDjZDsKuy4M9ALyybK6rEj87tl4oDffQ3y7XEw9MfTxGKz9WTXd2b0E/5yq5GVZEtXQCEDrshasNz0pd2p2LBiCulpWK/PxqZsavAzhr7wDAzaORiruzHQRdk+PdAcDnmbJv7lbO1ANwx2OJwNTHU7jGjz7nLy8A8Ms8YNP3Oypj36sAcOgd+77f7+ZPhNxk/3E+pvHdOB3vIzOAYdwa5bdJbAdB11L2xPxNXG6h6Goy2wnAWYcgicDUx2Oy2H3ZDsMUX06ye6vrJz+sCUnuAOCFq11sec9SHrcBIM7fHgOw3rQ1/eHDO7tH7/mtff4Vx+lylYWb1tdXiXCzHThdnYOeAsAtJCs7IzgBiIkd5yMTmPp4AMReHP9hQNPbSovDvLe6flaGNQ1uAGCPFBX3LNdg+gbH1xhCRDTzOFFOJddnxtY68+fPXxGdmY9zuvTv5xA1490aZFa+rc4P24HX9XfdA6LM5vhK8VV+Rv126TKBKZ0bxk8gloioUS+F2bB7W/abr/yeRRj9WXxERBSLj23ShbuHKoCFWlbV8fl9elz0tsUhbbtxgr1iYzvwunb/5q0lWWvCjym/tPzMZ6uLTGDq4wGAYABovDzBR9k9i3lvdf2sDKsHozmXDQ+b3uDfRXsUglrWxcyyycnJ2VnJaey21WpF8dgO/K4TFh3c//4jNFDqZ9X9P9zkAlMfD4Djh5j/nQvKbjTMe6vrZ21YvZEOABnwsuWq9MnukFaFoZaVED8JwMXyk6pOMG3KPtBOnQ+2g7BrlSrA7QCl1bEjdp0dztvVlQpMfTwA0D0lzQnQQ9nHO9m91fWzNqxeFR4BwEPjc3CV1x3jnU4RkaEnszYVPzCt4QUzJ92X+Z2n6cy5cZg6P2wH89LTdCI69HoyUbL3KoWhHP3UQEQzrpsDYQNTd93Bj6LdKiKidl5ZirJh9+b2M8YzX92zUt6wDm9ERDSnTLYKyjqF+fZk2nurtjUCEJXEbO6KVQYOopbflvM8FQCeVG4FAPfhrq4728G0xHjase0usCBwsDIvlwc9HTN61OCVAeZAzIHBtngYN33qALh1cL6ySe3s3px+pngykWnzsE64eBOgLRPsFZ9ZYqfEYldPN3qaEHrFj1HLasioZbnFvtpoxLQL7mfbhszJ9+IYcwZn23X4CK5V6wBAltrqYDuYlhhPQ84nnY76d7ejMi+RN5cDQLC9qTsbGGyLh3Ez4ot/wi5Pnz1UWV92b04/16p1AP2bCec8utQIG2/bsNZdPWyJ35fVPip4tSrk6SQV/dpwX1V+2A6Crk93Jzdsrivw2T/CBC4eL9PcV3E27N4y/WzJ5vEfz5s31CnXhdPmhmlzw7S5YZppxIJmWnVoplWHZlp1aFbEqqPQWLjQouMnNI/CCC1R2Wh3tNodrXZHqxnygnZKTHB0MGRRTT7lVKCYU+4tv+iiokQt5Ya+srk64jYfPeTVv05NPuVUkJgTgKTlqelZ79SSXIcKusjw68Usu3fN02UMP91NN7ylXFOZRxOpxq4gQU1ZxYwE//+mHMRYLTX0lYCNsoF2OoOeIpRTnmJOVl40J7yTQPS1019S6ypoJ0prP1lPa7qbAZ93zxLdb3NMIe0koInksSsopqasYkZ8M+cgxmqJ0VfKsrEehigL19u02G8gdgunfWxAZH5Xx3yXjUQJaCG1btXPuGW7DkVHHwqLI0OfSCKq7mNq2T6PiCi2nbJQMm5l8+ZPsH7VVceYOCJKKxNomkix1SuTiEb2UZYNm8M0x1Qiajma2zpDcXUIs7EehuwMAzHKKb8xJwCoQADckCi1roIu2rf5FID1WaaWEy8AoNpNZX6ENJFq7Ari1JRVzIhnbA4irJYK+sqCjcoV7SRKOeUz5gQAiEjrCxxBV6l1KKeLvvd6GUBjs8ZWxXkLDMD2LrYFZiu1JKCmrGNGPDPnIMJq2UZfKQ1DtjoOtXbsm7JbsDF/MSfGHAH93MaTJNehiC6arQPtD7z7+axPrppbhga+H3Zl29ovbYvL7FelRT/wBYeaYjEjJZ3ZHFhWC7mjr5SGIXdmifO3x4Dv1/dgMSfoL27NV8zJfALYsr/RZjfpdSiki1Kf1t403S6u9U+vGbe6HBgQHdxsr6PtoanGrlhqaoKPYsyIe7A252DJauWCvlIShtyxY0ME0KrS9uem9UsbNmyJH3S5WwHcwjebvfb2gCfS69ZN/+kYAKk43s8Oge1HZpi3V/vALrrrQ5sjY/zaYJnD25mOWBlwAgAdQwxYrQ42h2kXHwL60zDJe2atGZqLQbYehtyxQ0A5FQDmxHmOW3Ntxc7H7SXXoYwucoO/P4AGq48aP4hybtS2cqNH7elxzC5PqCUVxqGmrGNGXOPkIGS1ckFf5ZZ2KnDKiW9lm5z8U24dSugiT6cyAFDKzAe9Nbscqv019Z/dtoalGruCJTVlHTPiGjcHPquVG/oqt7RTgVNO5oN3q+zjTkAZXBNfh3K6yKH+cwDIgfGvLfVCawC6KXuv2njXohq7EqOmFGBG3P8ibg48VisX9JWyMKSPHXShPgDoBuh/K+CjRsbJs48B3EUDAIkZ/HWFdvFZGQBA9xt6AI/QGEjMAErpmD8z/yDVUSVmcP2qtGNHv7UDfndm3OjC4wDgZpkQRZ3ZHKJ7pgAphyY7MPGELlq0aNEiz6BFE2zKxnoYItWhZ2gnAeVUYJgT4NllXyXgzoXQ1gyWw12HSrpotNufAP05ogYD+Dj2XwAA9xJfVXwxaaSJbMWuAHCpKYWYEc/YHDislm30FT8b62EIn7vu6VnLw7vtmOjGnqVevk1EE4Ld3Fu+S7G9a7t7tvmgYNR7EoZ8c+Rws/AHRPQ8aBBvXamfbfiMWfin8eaTowemGj1RxhsjD8QuH3pbWSiZA9tX8KjbY74pEI5fVU/SGzFjHWx2syH0QuLEQXqF2ZhzON/l0L8ftH1EbDw0+hV3r9e+sjEba2EU0dk/Z2KpYbBOel0F7ZR88EmTYE7LtZgXwU10ts6XkcOu1IylFcxIYOYcJFgtjXaCNjdMmxumGbQ56Zpp1aGZVh2aadWhmUY7QaOdNNpJo520O1rtjlYz5CnPAgB4vvtkUkC/qrSxv2THmD2Pagwsu6VX+lN2W2k3lGza6b9mYkdDWvm/zmOrX42q7b7gpFS/zx5Nq3BheeVNJy8sOXqmdKQbMuP/uTN/fF4cAM1EjoBTklZlsvBzkKWShGpIEl4kjsXCvWU1mWTOLEJcS86PiAtJ2kldNpa0kwz6pSNR2skwxusEERH96BAi9cLp95AcIsrpEkJEJ9CDiIjSh76X+7dwHCJHwClJqTJZoZ0EakjqtJ0s9pbTZJKjnYS4ljptJ0naSaVSlQXtJIF+ydFO82FU/DJ0l6yOPh8SEdGJEC4d9WRA7quDJXKEnJIY6WOddqL2iz9dwYGTpLyIj6fF3gJvSqvDAteS86OCdlKXjQXtJIV+yVTHY/fAHOPi35LV0agfE3dvc3XoM4nG5MkbfGMGf+MUEcUcNW2uFEJENNH9uQI/44mIaPUaIhKAe1JexEOx2DvSWjriftY7byR6gdqkxI+FC3YkhPGoy8biy8iTJhARPatKyr91HPUs3LSxlTsA0LVj6QCQnfAI2XHM/OWam3cBgO5dc7foPUDHvLwiEnBKqlSZJKkkddpOKpWgoBjfsnEkhPHkNj6r6JflPcte1DEtOq8A8OeiwT4fek11PDzm/Kgep6utK/uDDnh/Y5e2HV4NcWIfu0SHAD3ysDhof+Dd5e4pw2raosrE1VLiqSGp03YS2dtGbSULXEuFH3YkhPGoVKqysKFfv79lxZW1USoY/IbYyLv+rJFIZBgzkIhatF1NdA17iYg2eAPwXchQ2ZUj+72C7Xk1N4w5+iWj5Zwculnxb9PmBvWIiMZirkI/az8hIqKgKANtqXnLmhfxUCz2FnhT/hX94x81GfWMXZXzI3TBGQlhPOqysfzm/u0WcGqtJxXXHU2wnrP2otIkRiLgb6KOPjlE2Y7fEhFR0vpRNYBvjNcdObc75HF13IH9bSIaWjXdiiqTFW0ngRqSOm0ni73lNJnktZ0MVzp1S7BJ24kzEsJ4VCtVCarj2uAPXNDhgYrrjgCOgmISzsQHAUCQbgeAunaAvRPDxnlHfH9lv88XRmbPzn8cgJyzeXdmMTE+N4/CrMp09+ascJRTRyUxakhbEmzyYrG3wJs6fGtP5xxb/HBGQhiPyjER2rkh8+acf21PD4Pyd7SdcNq8PBtX4MKAoJcAZhE5AH4BAF3YsjTzvg3rAZf/ybvqsOCUVKoyGakkCzUkdV4Ee+dKW4mDa6nyI007qVeq4ptV9MvyqrT/ZzsznY3XxA6ohmcAkJXJf6C2mXnE3pFFMF8CcLxuHj7iF3BKUKfKZKKSLNWQ1Gk78fe2UVvJAtdS5UeadlKdDd+so1+Wxw63RQkLjIurhqCRzzEAiEEn3k4XbwAA0nhVS9/XycObFg6nhMQM8EgfKKadGix1AnDJq44RV1LnRUAXcb3BRnwrMQNq/UjSTiqzgYB2UoB+iVzDfOv0s4GIaOdcItric4Mou1s/ImrTlogMTlOJqFbTJCKiKW8S0WF0Yy5gx/vlyVXpbNwiInrkvY3I0GwEESW4hhB9ZHeOaGpDvTI/uzCOiIi+P0pEcY4/GJ1IehEPhbN3gmsI15u6q9Lwg0R02yk02xiFrB8LF+xICONRlw07tsYwRnxBRBTfOVPFk3SiAyHNfjocNTKKwZ/azFzWfYqejnTzLt39+Oo27gF9k6j+XyNm7N43vlcyxXT3h+vrkZF9m3mjbe6rg0PkmBkfhs3hkD4qaKfsyQvObKg312ACfKS8iI+nkC5ival8z2LGtZgoZP2ooJ3UZWNBO0mhX9Zop0snkyq18zSu3E+uKTx0nQqhcydzQhrq8nOSioBTkiZ9ZGkngRqSOm0ni73ltJXkZv8IcS112k6StFNularE0S+NdtLmhmlzwzSDNiddM606NNOqQzOtOjTTaCdotJNGO2l3tCX0jtZBBlWR66uAeCmxZijQ03GGs64o8SxmVEW6lxLiJTclLlSVsSK5I/STuu2yPqC36VFk0tI+vO/+p2y+6lC7t6uiUMTkbSTVcsSPHaxwj1DCR0KTiO8i3XVIoIdHKR1QtxFwZv+d8rXDnaylI5aNGWUR/qwwLFmehYuqSM1zUkC85OarckJVGVnuwtLPvpfn/LFtlNMaIkravuItb95kN9rl+dofW4PK/6MkFDF5GzG1HJn3LKxwj1DCR0qTiO+CnQLyM2X2DZo2MwSVd1pLBzIoi4U0kiAs+bdwXFRFcv6ZdeIlN9UhVJWR5S4s/KRVWE9ENMrxOdG58NGzwKuO26XcnhHd1vkkKQhFTN5mhrrqYIV7hBI+UppEELxONNpgA03GaAPp26LUGSvpQAZlEf6sMCz5mYM3GX3AJgGSJ4SEyYFGFkA3Ll9Od0ZVme0vjOsn4gHlkjv/PjxBAJplXQSCdiyN4LcuS2/iBvhXSVhjQyBQpZYDANjRIglwD427zFmCCk2iGyGb9hyKPtQ8dIUOB/H9XThORHqU2nScq9hLSSEJw5J/3mGJqgB8mMWCeMljs1CVUSe5Y48FHc4CB91Er1MOMd+Mr6ZEoUpE3ka1Wg4r3COQ8FEIudzo36d961YxiVucgR6OoX5ARXA+Ca4mHfGfFYYlP/vnhB3azj6aSUTfN69cay5RZMW6P0YHYdQfX27oNtRARN2w2Lz31Xw4szzAW0REpzDfuCE9EK0v/x6eosxPZllAN+wLl1+Y1TjemcXgjP5EROGobD0Ui0CIvrpDzVSdWXIyiYgaODzmLHFi7dQ42cp1x2Miiva4xpwgiIiigGlW0oGV6ejcnxULS2b2D4uqpFR6k4hy6j/iwywC4iXPq+MqxhMRncFUJdyFpZ+/vQBggkGsOtKBN4mIesDDeiiWgfy7ilRWBzN9BxMsliwhF8lQUitO5pRbKwSkWElHvjpEfpYXltx1B/rHrR9V49G7cwHPUZvTgBtjygPwODcYCHQ8D8ABOfl7h2+hKqNOcse3lh+AeT0yIfYReNPJNNOGQGxUy2GFezgSPio0iebc4/zszMPNjnralI70z/LDkn/PwqIqw9PXAeuYS1AWZuETL3lvFqoyVrkLnv3ziu/16KbA9q9EGksBzwHgBdxsCMRGtRxWuIcj4WMJuUhZ0jcvsaIwf00ZcsDPtnSkf1YQlkx1cFEVv55LKD2LqVQWZuETL3lvFqoy6iR3Jmb/6Nrq+JrS2CbS6OhtGs7y6gOxUS2HFe7hSvhAsSbRlvSq5sel0X3mrHY5s8imdCR/1jIsyerYDA6qMubcEctn5f19d5qOY8kO+VAdQlUZE3fR+qqyO9pqZQHdoOO6Z4KGlHMEtGKGMx2t1QeSED9p0qRJF69MWqFSuMcZ5y/ylgDomzbSQ5Em0RaWezn6xm8TdNhxxZSNqnSkfpYXlpXq4KEqYbUXnwuGLPGSD0/3uaoyiRlqJXdqxesBINBVIFmUUCv4HaAjEgAY4pV8TkIYiE1qOaxwD1/CR7km0TWYFINOdKq5cfSw3guqm7JRlY44W8MJS8EdLRdVIVpgt47ZzIFZBMRL3j8rveByg8jQfIYJvJDlLiz8rMLIF0SGL71v8+9Z/gYCiV7UwVGiTWidoyAUYSBERNnurdXcs1yqOnLUqJGDArI5S4wvFnKxMiqlYPy4z0nTqXanKRuJdCCDsgjZGjYsJXe0LKpCRJQUkEFEfJhFSLzkeXVwVGWeBw2ywl1Y+lkX5NO+a0CPK0SU5eLu5e7h6VHqIFF6e8/lRPSwr3vfNzyGpyoKRRAISajlyFQHK9wjlPCR0iSycBGClca/UNOf9BVzNuLpQAZlEbI1bFhKeBY+qpK4REJBmke85Pk0BKGqjAx3IeKHEu65BUoLEqfesg9wUxiKjLxN7ud3iGsSWbh4dq+mXAQi6chHIS+FpJRnSbnRCF/3C9Bm/2g8i4h91Tg1LSMfi0MzFMtvHRtvak8d++t/2hBB+9ax+En3XpCjNq/0P3xm0WYda9WhcbSaQeNZNJ5F41m0M4t2ZtGs6N3RPo8DPKqY1m49AwLtOAI9fhmW7QZmnrKrvwP+Q+o9Brv/YHUk/nIsmm69ZLylbZrQtsXIVKNAz/PbB1Mvu1q2Z22IPuQ3wun5uZT3euVmyFjsxkKUhhXyEeOPLCxaWr1HqAkkYyY+SJS1kmSeSpRZvrOJ64+ZxsUlg3GNOAI994N3i7afwwAior9chuXY/haOxW4sRGlYIR8x/kiNeo9QE0gBHyTKWokxT7Jfws6Dd5MF6kJq1vGD6U2rM6Np+Ppt3CWuQM/WZaLt18AIjgzEL7YnwWI3QlEajpCPGH+kRr1HqAmkgA8SZa1m/CeqQ/RMMOI6Q0ccbSlsaXNbtj0QB2w/irHYzfJgDwCvrDVBRvs2TwCw/neI8kdi9+kjO7Vu1erm8ACg/NiZIwLYFo4rKOSDxFgrtcxTSfq6S3/XVcz/fnPe5u3X4fmKTDtwz9YvdgMc7MZClIYV8hHhj6BGvUeoCQTbNG5UM08l6S2cZ79fFnoBqR78SQAnA4FuMu24trH7cNtDic52AnDWIUgoSsMR8klIcgcAL1ztYpN6j1ATCLZp3Hw3Tvcffkc7fPX60QD3Cx5xK7NuLO0r3X57Mz25eHnJkFyMmp0TgJjYCT4IegoAt2CU90h9WnvTdLu41j+9hjRmZr49Uqy6W1+/NADEXhyv+63pnioWrhSby4EB0cHN9nLeR54uV/m//N2wljVXAUjgzIJ3r169mrtMe9nGTVoYkmvk9k+KwW6mXXwI6E8j2/hfiuP97BDYfmSGCAglZfpPxwAA1g3Qoae7aZowx5VyE7JWNjJPJaY6dCNOnsXZ+pwtPmHtx7wv0+4eENBgcc3XLuYyHAa7EYjScORrLPgjqFPvsdAEgg0aNzYyTwCQ/OTJkycpuRqignQhkecQh5XY016wsZmHfDt6ZyzLXXGYsBu+KA1HvsYChII69R5LTSCo1rixkXkCAHT18fHx6Z2rMSpIFxIPv327rp3uKHwa3cZKO7yRO+mvHbHr7HDerq5AlIYjXyPkj6BSvUdEEwhqNW4S4icBuFh+UtUJqjPclGmmCm21gnThIPb0FMDwrcPmSj5dlWr3xlnSvXCy9XXLsaPf6oDfIxA9b7UXUg7NdQCQ6OqC7jP0Tox8jS78NMsfQYl6zwBWvcfVBRxXSq2ULs0TMLJWia4uCA0FgM1Bi2xI0S/31wIF6ULkzHLhEgGd/NICAKQgFQBSoLfSngIA/vaJN7HmiY0xXx70dMzoUYNXBmDHtrvAgsDBAJ5UbgWMdvsToD9H1AAmXLwJ0JYJ1t6y3QdzDd2nDoBbB+c7MJ64rqxfIiMTgGP/BQBwL/FVYzQAgJznqfgPvme51LGMe+OFRLM3E33VoaxHvT6XY3rVdPdu3f+EVHts7zoe3u2mENFS18/iJtr6wJfFboQiNRwhHw5/ZJN6D9eVQj6Iw1qZkCdx5qnkPUnP49k/j/ZQd2/kepKKhSgNR8hHhD9So94j1ARSMl9GnrXSZh1DmxumzQ3TTDNNY0EzrTo006pDM606NCsMs58KIKywfj2s6PgJy6MwwkpUNkQUqv2RaGZhoXn/NEx73qE979AM/+Wvuzzeep38X1cyPe757pNJgX2rYlNfAMnxps2VvDkCYqn3HR0pO8vPCyVFoasw1bgKMk+xo2HOjKh5r9kdG9dmtrU5ALRqcqe3q1+Lquo78wyAH4c5tavhq8Oie5v6cATErv508Ih7r8CI2lYPgCvLNy1vl7j/6UhAVNKKL+ll5UDK0GpqdK54Zb/vwtPgCEdR8TC+GpfsWHKluhJ2XUFQL2eJdGWyYWMWEwsTofKELuJ/v+ke+Lq3eBSim6SUvyi7V3ASEdGL9q++kH+NZxjreZyIiNY4NCAi+s7/HBHRLrQx8AXEYtBT0avEloCjC7BWQtKKI+ll/ZUkQ6up0bni2urSr6zd2y8sW1Q8jKvGJfuOlifVtcSz+fpfAqtcFk9XDstjxcNExMKkqDyOfV/z279m2bv9Ih6F2CZpFm4aTjEL8Y6j5KtjPn4yLvVuQEQ0czUR0bMA56sCATEzS2e9OgC/XyQkrbiSXtarg6HV1OhccWwROukpvhSOiomH8dS4ZKuDK9W1G7pHRAdQPVMsXbls2JjFxMJmWK2OW3ZlYojawjFFNAqRTdLV8bCU+cf76y7IFcdj9wATNnugARHRZw+JiD7ADCJq1I85vPRWVR3dRnyynpEO+QxhREQB+M7cehDjDUT0A2IUVMexdc3CiWi980aiF6ht2lzZL5GIeuK8rIs4J5wguqXDdRGJF6L5JjWuDCvzO14FbhPtBD4ieg2BRJRhhyixdOWyYWMWZsPmKediHzCbKAI4LRqFyCZpUjIq3cyrt6UfgZwn9wDoHz4Fsh4+RTL7bfqoZ11N3Zt7AkBYeQCn59f7EFICYtZs4IpZEW5Sklbykl6C60YjraZK58psS/V2IUCVa1erwZoal6xxpbpOww2AswcXJmXThSLxMEuxMCVUXuthoQOACyhXVzQKscAkGfxu+MEskoQQOtwSHYmWVcF02lEbsxZNK33A1NyVowB2zXzV0lh3hC8gpubYEXVowffRGRKSVgJJL9ljB1ehS6nOFWsN4fvo8zdmmI6cwmMHT41Ldm4YR6qrEoKIyODC+eY0m65sNvyY+WJh4kpkIgOyFaX+INEoRDZJHzviYZ7cVRp30fJwEwAjrzoA4Rdrxrz8lvNDlpplv4Jh/nNeeHJ0CwBoGuW975MW/iqn5n68zS8j/OVjQGYmXJk77mRzo9MvXqAfpozta90Ph1Y78XGrKge8BMDdOPnvd9yG67B2E2ODp4g31/UB0iLGKziEOQMwLEHAe0AIUgFcz+CKSJrTVQIJjvOxyEYplUcTOvZu+G9niEYhFpjksSME60yLh1GBiNp3JCJym05EIQ24ezaB5Vf0b7v5mSo7af2oGsA3ao4dC84S0SyUe0wpwHBG8MyJbT7X1A8Auln9a9O/n8PqtxmudOqWwPuDrt8uXT4UB2AX0SPgd9FjB3PBeUPhFxqmo9l9IjrvghtE7zrgJZF0rV5jm2PmZcPLU87FzVNzHav/Kh6FyCbpq9LXscC0+BuaElFHTnUM4u7Z17jnkojBbw6JOE9EZOiKzURE24z/Mft9PJK41XE+0/rk2D+BaaQHIoiI2qM0q3bp1u15dFMAX1gbjHnXiTtqT5wbc+9RJrS1JtXnA/tsIvI03oiLVEdiqZcMyqpjj90Q5v/1YFC1j3rO8UYDkXStjgonZk42gjxlXXwC/CoehXhg4tWxCENMi5PwmaA6xvGFUIx7ph1EuQQDEdFGdDMQUfoE2sA0bcE+bnVMlJUROdRnIxEdBvoQeaMbc9NXy9zcyvkJkWFNaYRYGYwLs5KSkpJCOiSZpp63AkccfGVEhrXDWF1UYoqkqlR1rEQrZV93OeQ510CnFxIR5cTti8/RGT+EI0jXyn8tL2ZzNhZ5iru4tzCGiLYCTcWiEN/E+LB8kj7ws9165tke7XB4y3wCYohn3gPk/p/vzHABAPemqFgOAJLHuS/WAbjmhc39OQJiZrsv+9B38sEjfYFsoBzQagdH0irlTpDOLOnVtPYzK6dZllZ7p1X2cScLnSsjcCdtbS8aACDbQm2NCQQ8NS5ZO/rGb22BHY+AF9MqvhuAOEJXkx9uulACCVYXZKOQymtz1fWeN5yBp2JR8DdZ/W7YEiw3PSb5nIioewcieoDpRBTyHm/PX03HxKfGo9IofEtERB98R3WZRwUPEc85dlxpKftXMsjuIyJaCWwkWoj6RJRTHpuIHvtiLBE19MwkItK7DVaEb/iGE6Xo7O8SUUPsJ3qaTkR09FMDEc24LuviBHSpROn2RpVW87HDGAgR1WB0YK0dO457dBw16s1eZRcQ/QrEE32LWlkmP9x0ZbMxxczNxpiOMU/ZbCrCKZloJTBILAreJuvPSg0TPQ8SEZ2vNCCbiGhyYyJa7PkZEdUXDMlCh2U5RERLyzUgIopGk2wiyl7r9DNfQOwwc5q41bCf7Dic9j9FlNYAfQ18SSujypVA0stKdTAKXWp0rjhD0AtziNbCP0lMPIynxiVbHVyprhh01FOMa5nzZj/cdOWyYWPmiIXJKpHxXczA10SZzVDmrlgUvE3Wq4NobeDkE7HfVFnIXCQktPrij9k7K7q339DFw6fTat6eBxo3WHF497jJtyOJiNqhfnh45yBnYDtXQCymuz9cX4/s19Ie78uPw4H6jTt4VVykF0hamVSuOJJeVqvDSKup0bnivrwY79qyhcuAO+LiYVw1Ltnq4Ep10ZwK4a+6979NbELcdGWyYWPmiIXJKpEJan1x1YZdfV0i75BoFNxNili4nCOXDTVeNX/t5umtGp5nPXzcxF5bXz6ZWufl0pbb+QJiiiepPI4PKCMtaSUh6SU72UW5zhXPXlx1qu4kvb+YGpe12T8vbpQKtJdONx+zMTyOL1/JTjIKkU0aC6fNDdPmhmkGjVjQTKsOzbTq0EyrDs1QHL79o9FOmkGUdnIAEHagkAIotB/OH5s6tSSlNLWk6MKhGGj7FststOsOzZAvLBwffSt89g0FJwpXqMRbwSnSiVVHzoyoeUPsjnW1xsL9xkXfGPZtSuVNvfBwDcO+FVJ1cEThLFTgJEi4YmAs8cZl3xSp5OWpLpxiFo6Lvili3/Ljs5qWxhGFE6rASZFweWPIt5S4xBuXfRNVycvjbGxl4bjomyL2rWCqgyMKJ1SBkyLhin51sMQbl30TVcnLw2wszyyPZrUwTiio2Gv5OLkJds86AcDUWzNqALjJnAybBBT+A76RAPDT8ABg3+ZTANZnmZp2bL1QGu6hv12uV8zOK6wOHU+RzqiSN+yFa0E9K1XOwnHRNxvZt/wwjiicUAVOGQlX9Iwl3njsm0KVvLysjv2oCrO82n4cCfUZDiyv6bcUO4P9Vixe6H/Q1NxeB+SMNCx3AoD30eW1r47pUfgPXhlRuNk6RgXu81mfXGUvWB/4AjjrEFTMqoPVoeMp0rEqeQVWHcpZOIBF32xk3/LLGFG41Kdum6Z/OqrtXnUkXJEzlnjjs28qVPLy6o7WjmUM0mEHMMXi5AxA53GjJR5wd74zyW+WcbF/x90H9l17N3NiERhO/aeHwajARdkhsP3ICy5C6bniZFlrvrFYYk4zilXy8uzYURkJ5iMX/AW71BfcDr/9fKEXgO0A4B3x/ZX9Pl8kc3e4oC+U8TSKwomqwDHSc8XJWB06gSKdcpW8PKuOdvjXtHgK7QSNgimym3d06wUg4wDwCwDowpalnebu8KNDoYynURROTAXOJD1XfIzVoRMq0ilXycuzM4tyFo6LvtnEvuWXmUThRFTglJBwRcxY4i1EwL4pVsnLu+ooPXvsT0xR7DnzeQAA5wwAjzJFOn/y8NvKAPCzP3DxRjXmQomrdX71TuH8tRlF4bgqcImuLuBIzxUj4+jQcRXpEl1dFKvk5eHcsNETPzgEABdGDPgCABokAtjimQ5AzxNDO7ysydsActZ95w3kDEwGgKVvVoRZR+52RMVCGU+TKBxHBY5RdGOl54qbsTp0xqUnlVspV8nLw7dwum8aDh3U1eWvhR+PtQOAcfunNTlb3+O7mOE/33rYOeIN845fIKMHDHevZaI04PTlhwGNnbbd+QE4Of0M9vZ0y7p3PCescC7yTdVRfs9YfcDKavMA16p1AETeXA4AwcXuJe6YMzjbrsNHnCXXqnWAuquHLfH7stpHKNAZKypYODXsW0HN/mFF4SRU4ErO7B8Rlbw8zqagM9LmhmlzwzSDRixoplWHZlp1aKaZVh2aqXjeEVpoYiO6kjWYupKUUmjJu6nUTDuzaKZVh2aFbv8HjD+zo/osL/sAAAAASUVORK5CYII=',
    '1706.08653v2.8.png': 'iVBORw0KGgoAAAANSUhEUgAAAb0AAABsCAAAAADbUDOmAAAYl0lEQVR42u1deXxMZ/f/TvZkJosliVYSiZ0GjZBWpQ2pfWuLEAStem31qtIFrfq1tLxUS1G1tVr7UvuulkhsRYkIiorYJcgmsk3m+/vjzp25s2QyI5OIjzn/3HPvc54z3/ucuc9ynnPPlRE2embJztYENuvZ6OkQyQhbKzyDFEFSRkD2tMY+WYUedCs2Osho6zmfbXJ4hrGPeDesFMVlTDm7TqYH9qrJtb2zbjs5qArta6LospNDIWvnJAsSLoEOz7H1Hi5G2JMXl/FsYvGXHUfUvvRjfcWs3neXxR5WxITWROHiXUmRHT9+sObwAb/33HB/34tTG5d+1gI+JSrdDy9E1YInLy5TdKrhnsdJkr86hJI8gc7C9exXVCR5Ge+SZG4b1xOlg/cMj3t/9r2//8mLy5Rmz58jPPcDOwGAC1yE64paMuHcHgBc/i933PO63rtTtT9WP3FxmVLaxKB+6onhKL2Jos6ZP/55Xq23NurNKhvzn7S4TGnlo85iu4YrTMgdQMfn1XqHXneMytwFAJjXLKDJ0qUt/F/djWi/2nP1i8ud9qGByDovKl7s1Lj2M5/TWcvV0WQsegsnyfazyFsO20m2jDcsLm90IVirc56IHmquH0nyBprPmDG1t+98VSnhPbMrhtXRQLjf1hw5AAS22/QhfLw2d4KqeUvD4vJfRBeVINH4YwDX+61bHPR8rve2VrsABN3cGg0A6Nf/hv++7ut+dI5/3VhxOVPgiVQNn16pWLGAzTU6nHR/Hse98/X8/fz83henlW+5rsKB74t2YkcnY8XlTB1wWsNPA+AOpXCi1H0mK7e7tO25fPZWjWkEgFN3ZngBgOKd5SOd5VErOtLFWHE5U+8J2/OdBTbDAUBAlTTh7IqfrqAC957HOSeTGgGArE/BRuFCTOKUbojZurqj8eLyJfnctFlqdskAALJ3TmUBANa/oyOXd1AWaeU556NsLeWYNft5lJiYeE1zlpyYmPiorGd1y1sJx5NoWUSSLPRpqGKRf6Mi48XlPSOe7fS7iiS3zyRJZgS8S5Inhwql59CfJHPfxeTSzTkN9/d80yN8nQ4m1w3PuxsbeKU4o6eOWuAp8jcWHo3jtRfVBcFpka8N8UdZ7qDFf3SpsN7mAGDsn//KmjT9EQA+9P4CGF80vZhilPP+XuxYhxG1rh9s1Uc4vTXMcYjdgazpcgAXJiQke7RxRcYt30/eROm2H/X/ZEVel0h+h3kkz3kbN3va2CHNcUdyIbk3vlWzP/XH5fLxUutSejbJzMwK40Pn+d9n/6GFo/pn5W8Xrb5YNpi1pPeqA8AedgBeapfraszs7v/1m3NCesElOPmXcTIA4CMP0SVbvuQFAB4VaGhu0EDnSalbF+UQlZT6snRRmWq0knMNe/1Lg68cAgAcaQkb4anFlDm+Kjl5HeDlo7kAoExNxeNkVbGTZLclgvVaiFcKTiQJq5vr+/8VqinT7kGZnGtr9LKzXu0QyUmLGru7nc775PNC7Gjq+9XkL7a0/bRAd877+kiB8ei1PhNAlrt6D4QL33l8NfwwUDB2r/vGNy8DiA/xmbRz+l+937MFAFszItDI8PwD5pMkN9d5SKqG9yXZwDeBfNS0i0otcIck7zq+RJJ3JjMO80n+eo8f4AbJedUzyd1e6VwccIEc83IRSb4WuZS8jH1lMWupaDv/5QLP9Go994PelQDZsJX7ALcOjQH5l9s26Cwvko+ouZZ1lwBI8xHOMif08ADa2K2EV3YOEHnmXwBwT+wPBDmesz0zVuw5i4/GPXMzGACCZRp3XCS260hUF6d5ssEnz+JsI/XZ35lO8fHxR6peQo8Hocmb45AJAGhoB9g75YiVa1j3Xv7PuuqsjM7a8CLUfs7YYiX+EWb/dk4XNIsF98vFyA6YsPjHPR+pT1Lg4wJgRXWoFi3t1FqciAprCY2rNkVm3Rv6yrrqrIzO2vBK9FLXwiMAKMyvLV7Jyq5XjKxvl+WTHcWVRANUaSZwny48XQvxgEpmpDFYkYOfrR1LbW14JXqpm3ofBYAT6ACgAABi0UlH4mGeMPUB8H76oLfEy01e3AsA+VuyZw2sBaQCew7bhikzKDN+0/5S7zHkIR8A5AvWXgWKpvXqAiD2LvB4UnfBTZ4vCNz3DweApAsEOryQHQggE1mAy6JtZwHM9nd0VQJIdMy9UxkoKATAwqKn20DZU4Z0nXCvglrvzop+P5VuxbAtupWvu2+r6D9J7mn97YJukwpIhvYZv3hl5MR8kvl921Zzb/jWD2ROcAx5oX1lRbM55LT15P/aVXF/qedFcn+LCb99sp1c7z9tx+S/R7QYpzzc1atSt2NLWysCo9ItmpOrZuseSzfFT4v+l5k9PY9abcVgXXhs18P8FUPJUUm3kgpJkqEDmXKu0IIFyc2LKpJUnjudT1VGKe7n3lDdY+maZ3gyyezKQUprWc+68Ni5hzWjkl58UcMGWNQJVBcO9i8BkHmWojfZqHcsHW3blFQJioiNF1+yUmdnXXhltLdeqCxnaKJ/NHGCcC4eS0n++QUAFEi1DjorwStKzkL+JsnMTvQvW8V6ezpf3Pb2hXK0ncY/umZ83r6YmGWao6WkH6obd8cXwFmH4PmvBtT/HujjF7z0ydGVGp5Ac7rsHTlkesBbmgmd6F+2TjSuUkkWFpWjJ1HiH62nHlDqPdnAYhiqS/6FMWRG1cEkVcH3SoWutPBIcp1zBpV1x/JfcdyT+Jet8Q6RvT3gUJ4hTFL/aOkosN0mwMdrMzShush/v803gOegdY+BlIE+TxMdAGBNXU/Yh6xBTSP+5WcypkzHP1o66hd7A/u6r8uHGKqLCd6bXAAMyVwL/D7g6aIDAK+HABw8TfiXnzHrqRa2WF7FOhv1+qG6WHJ7hxwAarX/GYWZPk8XHQCMuH8O2Qc+NeFffsas9+knyyeGVwJUgndwifrykidQpXhn+WNnedSKfHWo7raEFc44dx7AsOMJ27paAV2p4AHwGTn789HzBpjwLz9bsdTZs4ar/aOKcJdC4CoA8WgxxXSc0gMxHdShukePzJYBm6MBdKm+wPn7UqJDqeEBB97opu9f7qfxLz+D1tP6R1/By1eE3QjxaDG18dn8Dd7wmXkGAC7GtBkOPo4fB8DhPzO+kZUSXenhATUG31W4KsI8AaDQHpAvGDqmpuhfLmHbokJm2/njow8an+q8+HTEFPurke/JIiIA8Wi5OkmoLkL/BgA0TgCAW/WvVSklOpQeHrIiHjoUpim7/yI/MSVe1nJSU+yd2rbK9pDPHUu2XIW0HoouFjZ0YpYngMLEwMqQHC1Wl+GgALKMhXve8C81utLDyw1b9CqgOjWw9yTtxdsZdR3Mee4qpvXw/OzOblgovKL9086ttkxXzxyFnEsBgKLdHZ7gD2F79p52ZMSJGQ3D3C7ve2O4zHLL2az39ONa0pJz/YLsnuS5s1nvmYxKso17ePYzPEbIntavy54ndVbWFwHh2Wv1tELxJz1P6qytr5Vtzmkb92xUcTIbZ6cIRyc/NwDISQbcNe9jXHsEBMkNhatUs/rQqaJ9md64ys6Qq2BUchMYWC9tddyhaoPlTDvw4jcvAw/XSPNBhKVFvjZEbiBckHQuYpKvdSCLySjixnfylgNAX3XjZs++fqfRh74A0hdm5RaOrGdBM/xxvtDuv97SSwd/WmvAWQJPH0P/0OZu14+2i9RlzSHtfanBLrrp8ni0t7EmgBlRSYmIJknVPKffS84HoRZ+EOp3xwpRSZJkFPPV+FoZhECnjUwjpzvtNTvsJ7vtxAIu6ya9lFOrswFXsjotPAMMvgAcJ6r0WDPg6Yd2q/p8TZ6qe9tIE5gVS30ZQh5CTrWLI3lnclhtAYpquvBerDHh1WKlUlkv75pS/VIuRy3YeSgu7lCrZIMQ6B9c1pJpeM1c66l69iNZWyd7yRSNzaZYYD0tPAMMbeeNX5RswJoBTz+0e5NnPskhPY00gWWx1KNnjkiQARg85FAEABxpmVKsbE0cskK36VxDG7IxBAB+ez8Q+iHQ1QhAjofmKt2//hSAVdLYyGNBVQ04i+AZYPAZYYyF5aHdK1s6AYgY9NjNoAlgUVyLS5vEOMBYPghNILGGsmDlfCQjAeDqkX4wCIGOzo4CDqOLuZp+9nwZQLMW2it5f/Qx4CwjSzHA3NDuor2eAPBC/kHDJrAwMsIf+98A4NFrzRxPST6IgvENG2/cvrCOVHSd/WfWtV4tAKpRyzRz2TilE4CzDsGAI1Aws9kXZirigaAbCxWZgyT/rh9HyQw4C8kAQ8J+pcMgT322ZJLcFwCkpSsAwBOXOhk0gYUxZT4Qusr3c1cB2NBdfXnZ+pbNP27aS/305d69e33XsD0b21p/yryqkTY1qZ0TgBMJo7wBHP8svMZBc5sn64F83eTxQyO1ga2nq/obcBaTHoaE86M/CQxL0WPNIOl9AciGHADsxVBRaRNYaD1CZSQfhF4g8YXVqzfcjLnY1frGKxg/XPeCEAINvDJteUqf++ZaD8d62SGo7ZA89YXCZe8acJaTHoYVfWR4RzFGjzWTxPsCkAcnwSWaa7wJLOg5b6nzJsgGf3q2sSYfBHp0lyWfPSr+O2qOLqvV6laZ3itn6hBoQFZ3efWOx8xbzssREACgydIj6hXY3A/Ev6yWexJPlQ6GxgDQbGGatw5rJmnuC3AXcjIo4W68CSx49m5A3R0OcFiMPW3LKJC4WFqql3JbDIEGgCrNT+42T4uHU2UAcEWScH4+v0pGRoayMCNbyz0ZPgmGY4eEpk/SYc0k6X15CQ9dHjyNNoEFz979XaHhMMwHYToNhPVIebCN7tw6YYUdztnVDlcecwIq47KZvsBGOUKaEfXSIO3mFwDO+3xRM1TDjbEQWoE+hm6Z2U5AAarosGauGYT7aggA8Kx2DwDuCl/A0m8CS6z3jXKhTMwHsWnQTKOBxGVI5x+p4+seurlAGwKdd9Iu1Q+4gSZm6uk2pcAJuIdmgiYh5HJ98Fx1mqH1wXMthqaD4aGbC5r0cQJwwbMBpKxZJAntfujmIut8GgCuVg7VaQLze85MIbdH9odLtjSFkXwQOmkgMq32Ho04gkP8/MxtCF+BEfJSXIx5MHzY0P6LAz067fcDridFvG6mxmHy3QB3D64jZrgAUJSTZcBZAE+K4b5/ONCzAYBrsT846LDmkOa+1JrGnL8KcMMYe2kTmP325Zm3AqF4u1/f9k0/uk3j+SC0aSASetRXeLT+2GpvX2qTUZDcggmCJzI4hmRT9eyATBvw3eH4VzrfMdvP+Vez9SeH9c0SNZEc9qrC883/6XJmqNPCk2DICY4hlRNnnVn90kwVdVhz4GnvSw1vdUTSw7ExBTpNYH5eajNIGkiMMtuvLFjeuZh9izMJDGksM19dRuz95o2tjE4fw/ljlVv4GrBPsjubuiOnRYishCawxZTBtrduI1Sg3Lg2slnPRjbr2chmvec2s3EZU8TzpK4s4NlWDLYVg41QMaJxheX+pisMeNucfeecXSfTg6JqYl0UgIyb4mU/L+DEnnt1+lbZ0B1Ztx0dqSx8wbMi3Xies6wi28U8eMZ6zqIpK79/0+7oqNbTSvoeFJdM7PBB7csra/p+ewbAr4Oc2tTxlWHurXU9MeHe19WSFvqvO4lLv8UeVnQPiq5fQl+y2CfMx+7hgQdDAODMges+9Ts76dVI//qzavqSJrsmdQXc3HxVEfS29kOYuW4DgtzdXWVAw6ZA5vpLDvV7uJnu6XRVpM/vqReHlbM/6UHjaEcL4Gl0ZG25WBDYQ+IU04FnQp2h+1TZvXE6ST5u+8bjEnL6jvA4RpJc5tCEJH8MSCTJnWit4ubQIpJFnUJJ8gTeMcMP3BJwdAGWk8yPCv7621D4b9eD1hFxepKm3cBihZ/rzt471V6+RlOg3Tv9ndzp8eaOTcE+f5lUJ1GRvnXRf7ywSrd8aaVXl+/r1UppJjypjv0vz9ixZajTMhqHV7w6I9b7GqfUqYkdS8hZ+AN+U3M9mpDkt0tJ8lGg8yWy5yckyeOhuh8cL8F6wAtrSHIihqlYEAnXMzoS46Gxnihp2nrqCtfsKp8gI+Go+SDeFrF1+quY4ip/RKbIvNNNqJOqSOw8bCr0rDcXHQp40xVHzIQn0ZFdbRVJDnXUfmpUCs+EOkPr3XXVRAj3liWZMl6qIlDM2HmwCUlOuEuSH2MKyaa9hMezhwXW6zp43Crho6dvACnkduBTqcD6BqL1tJImrSdW2A9MI6OB05o/Xui6PYfiDrWIyCMnCLHmgfjRhDo9Fcl61kt2wnHymgxXzIcn6ojFaBXJX3CCxuCZUGdove/xhcguwMekMu0myfw798mCO/eFb0wKNAsjNTtfr5PkHhXJv+1fyicZbbdDsKsF1tO2x0zHiALyDHR63MSGx0XrrTJrQ05ToXBQRArZCFXzxaKR00lyZr0HJMPRlSTfRE8T6vRU6FvvU9gpSV65ZAE8UUc80CaBHCjXPvtSeCbUGVqvK34R2T8RyviWaE8uqIHJ3FYfU+d+XemgWNwF87TvM2hGmmaywyR53A6R047ki++qmGW9lYdm/RyXJ7wyQJIrga+1xQ+DTyaL1pNIFt88kgokuQmuO7QDSyrJOPfLJFXO6E2SneFfUkesVaFvvRD43vt84JQkmg9PoyO/CiAb9JWLpG+UwDOlztB6IdggsqfgQ7J5e5L5DpNJVd3u8XeqrdZKrjU2Fg4XmNVeAHznWGI9/4/XzPKof0Q8LwpHYKZkArKUGuvpSRpVJ61A1Uft7EMu6ApkVZ9IkrnAeyT5FtxNtraOCn3rVUZQ5wNno+y/NBueRMefngAwRmUUnil1htYLxQqRjUc1km3bk6R8MsnQJlLJ5lhpYJAU+QvipxfSVw2tA3xnvvVmnSU5FVVT1eeT8cptyQTkv9QYQ1/SqDppBfLqqZmOtf/QEZiIf0kyE3hfsJ6T6daWqtC3ngOwk7wHbDYXnkRHYtgLANA1zxg8U+oMrfc2ZonsRoSRbC+xXoxUMkot+VN0//cGRJ8jSVUXrCfJLeq/6wFv93Sp9c7llxTpsVvTW+6xG5ArmYBEFEiNoSNpVJ1hhXGA1HwPXV9UkWQBhHcQ26JSiXEoGhX61vOGvZKkh3agLgGeRMdxedecuDAAX9EIPFPqDD1lbfC3yJ6CfjihToBaB7Xk8EWDf90xpyEArN/WtTuAvINYAwCyVguyT0ur/Goi0iouah0AOXBWOO05Y6nLGTFa75/hH506duw0kHQsS1/SKOlUuD33JIBXgf9JJDbk1pQBgKMXcgDgMUzlODaqQkveeMEegDMSYBY8KY1V/uoWfmxZJWyBEXim1Bm2Zt8JuwoEFwe3OfxH41VRGqYc6f359jwXAFCEoXpVAMgYpZgnA3DZE+t7AwDaQ+ebELdN+FUnxh6OApQQwmaPDNwYCWy7ByDzerDsRt0ZAFKBuZ5zQnQljZNOhdaX3G55wRl4oFYHABvEgNnwbTkAkAtTMYY6KqQkqIs8rwIAJXz0b6Rk+rtWFUAWE1b/kQSdFp4JdYatWWla6m8Ct+fMZ4EAnAngXr7hr8rnPZghcI/VV8bd/cYfAH6vhPP/qt+JkQbNXrpu4ib87foDuAJEAjjeoe7aYYN6zKoNpNVrPBJt4uPj4+O/BObHh+hKFkM6FXKglAG3gBZqdQBwWexJ2iMNgOom2pty6EpUSEmtrj/uZAN5WXgNZsGTUr2bBQAQ5BYmQaeFZ0qdYYesGusRS5Ln/PooSXJiM5LzPCaQbPSerugchwVFJDm/ahOSjENzJUnlcqffWS8snSQnvSdMf7qS5LWQXiZGgtMBp8jsJohSkSdFj/Z28k8gSC2yTBjGpJKmByp1hSmYTua/gso3JOpcMUwQetwAR8h1eL3IhDqpCp35oqBO1R0zyOUISDcfnqhjCYY8JlXfeKVIb1YDz4Q6o99CWR408XjCdzXmCHeTFv7Vjmnbqyvaru7k7t1hqY7kwWZNFsXvGjUxpR9JtkGjzp07BjsDW9lo7+Apu/aP7p5BnugWALe3+/VqaY+PTN3PwUbN2nlWn1tAsrVm+CJz23osFGZJCnd3ucItVkfSVPNoKqjm1Qzp4uvS7zol6kKxWHQvRSmiBrq/n2VKnVRFoYvCU+Hu4e4aq1WXM9qt5Wsufa7r3Ujxq3+JjhXB3m27BL71jxSdBF7x6ozvzhYdvqiq84YmKfKDa3U8zrp7y41tWlw8mdXgZSNvCJ4KZeLJotAQmSX7lak3Ayub19sYkzS1/alKvenjpztMPLpVVwMu65p9oLwEdUZU6NDjS061nZ4MHtNuyYP0clBL4RWnzra3bttbt5EtpsxGNuvZrGcjm/VsBFtmY9gyG8NENK6NbD2njWzWs5El9P9UYVOOlq9chAAAAABJRU5ErkJggg==',
    '1706.10239v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAksAAABTCAAAAACqzBwBAAATMUlEQVR42u2daXgUxdbH/zMkYbJDIICyi4CQCLJExAQCMfoi+MaLgGwBcWNTUfGqCPpy0StyVQR8EDTqo9GAy0U2gSvwgCQsiexLEpAlIciahKyEJDPJnPdDz0xXd09P1ySTiXL7fOChTnV1/evMmZru6vqlDQTddPOIGfUQ6Kbnkm5/NiOi+XoUdKunzSciw1/wesmgX+Ppv3G63dLm0/Bd5OQWDugkFktPFIbEeXeQZZd9fanGcluo1HsBuK0FAJxs6mOhO2/N3pVyZB0XFfj6WC3UxgN6qMFt1XCsY4onpweMqt8J3db8+9xoBE2ef1LmfWcCgs4SEb3aB0Peb7DxN27vSjmyjg/OGYzQqUs8oMcLuURFklwiesjbuUR0ACOdeE+MwaAaIqLqoVZJhXWZRwPQuL1rdnxUkKeqh9e8cb3kq1H2gpmcXxjGvbh7KQD49TRI/AXZt1Dvmh03sclT06Nfe3PZwm7znH1w627x3lU6VnF7L5dqCq6hJuc6gOrzNbZrsDPplfbq6kuOO3jzgaxar3xGTiRJTBQSkGx5wqKoODG3IdU1Wu8l6anmPGcd203F7bVc2tOn1fz/vH/gyaesX6zcO2AFAGxNOFL16jwLAJwbMz914XYhwZJG3syJ2euFVHIiSXK3wQq57/WD78krfnijakdi4rcNtDbcaL0vTg71f7efsmPGVNzeu4+7P+5rojN48SjRVwFFRBu6FhFZZ0wgopPBqUS0COuI6JO2pURbmxUTjWjwa2+5JDoBsU9RyImVRFV3+xwiek5a0X2aZy92G7V3m52KJSJrNCk6tsuTuxvj2jv4xCSgs29eb+Cum7mofG5sc8AwffUO4Mk+gwGMBYDSuaNCgHjjam/8yMkkSUwmpOk3mFwNrylsvN6vHT0KGKZA3rHEVNzeu/buaQSa+EUAMKECRy9GAkCkYROuZgx1rIceLvXbs2fPvpanvXLFJJXEWJZZLuSe+VnCA0kvKGzM3u/t1CfypR3PQN6x1FTcXsslE/NvLX4X/mf0O4lTENd689DKZDKZVr3upTUAVhJjX/kohMzp/8E+LylszN5Nqf8X8En8ZKu8Y5mpuBvnGUoX3AAAS/WduIOZE3qgRf8/wSLAZaNCiM83fZ4YrFT45dO3Vu9n/BcsKE5+9elYWcfyhHDubpz1pb7h6QBwAMPQ/q7fAKASAHrfvh0Aqjc2aiqdvsAKIWG5osfCs5ekCk0WIOcW6z3re6D5S6My5R0zd2FO3V7NJbMFAFksACyoReBnP+YAtYsefwSGpO2ZAK1AJWD6fNNxAMvaAxZLQ6eMTBJQCjMA5I1rywrJsq3MvTQIkCi85yzgwY0tjdu73VbcBGC+T94xYBbkyd3eXxPY+7/NmidkfD00qMPI4tf7BN/zPNG2oQs/S5hvJiLaPXjJv1/5AeGjiWjnwLnJr26m/QlhLRIONeSagELSgYQOCPjbxMejm+BlRkh8aNDAZCIiOjeFSKwgOtfxHwt2eexpXKP27rB1w+at3jxnmaLjbSO7BzeLm6HU47Y1zL6yyyXd7Bdilwp7WrNbtvQHgEs3uhnwZ9gLpyrEXmE50SmswWbNxum9zN/ntLmrCfoeRej7KqHvq9RNzyXddNNzSTc9l3TTc0k33f6quaSzodC5Xd2gc7vetJPrBppyAx+Fvr6k/8bxzoepKhWX3o0blvoAgD16yuAW4XbNyYXtEg1S3FbC3BacLLz9Pu0eirMKmw1RptKssWoNNkcFGAAgZOYKN5+4WKmJy3L9zGqsm4pbyFyHQDWXqgY/HxMbNsJRvrLqm4fjrqz65mFHLuV+89ULHLl0KSX5UWUuLW0XAwDIn/WZsF/O+vlF082XwgHAv9oEAL0GvMdFZNCXmUFNXgsEsPuN4eGBADBBGDNTntQvKuBC+kMS+Dzj0ihppEQJAIDyZReu3P1ia1tp14of+WTIVcCa/Eel9dk7AYBDhlwF43DWWjUYgPWnbIvxBceZRBnsQbzBYEPg+NA49wksj7TmJuYocFsJc9vrRWctFdhp3FjFMXn9aomo4JWpUbgiNBr/NtGhbpeJtk9Z9vHE34mIrNG/c+wTMA+fWEs/9CogopW2MQ2xVTHl1gB835LyqN+9I9XtkCBYwbhzVDo6NF0oVXQZ4fIhuShDrsL6wnGiy0PTiXhkyFWwDmet1VQQlT/4lpm+TSCFDPYg3mCIIWA+NN79G5MVCTBiFMkYkn5Oc+maHKP4H2UuTf6UiKjqfM0Sm6z1odVENHU00aUcol+6VRER/fgYRy697VtGRNHTiWjWZ/9J2707bUiurYopP/jJG5/nys4jC58oQbAZuURUHta5hoiI/qmRS6IMuYqfPyIiOhZPxCNDroJ1OGutpoKsoycS0Z3hpJDBHMQdDDEEzIcmM9XfuIqguv6ocmCnxZtWAEDTjg7P6mg/ALFP3QwIDQT6nD4QA2DktKttNM+V1CsYwH2fLQ6AcSoAJD/dyX5jIZZbzdQ8kShBKG9an9UcQbHrTkUAyOjckleGXMVvNwGgSw7AI0OugnVot2aCsXPNIQDfOfYdijKYg/hlOELAfGhu3cfVFl4CYL56XfWIijz7ZvQLO89ZARY7pTO/XrP/+p6vZFttjAiUdbQ9FABuq95V3WIXYBQ2/PtEaedlycUgAAi/cRR4HgBy9k2018nLrsfqkGBztK82AwhCPoCqn8Zzy5D32vajpVbg5+F1U6Fw8Abj09B7APQfqJDBHsQtgyMEarl0MHH/jsTEbbHhTwNJ3W5bqdK6+r2ktCfmmQGYX9kevO6BM2Cw011RO62rF9QCwPZ//Tb2SeYWP22A7EQFxUEAEIrTfgkDgLQOUQCA6DTN8Pk3IWEcp4AuAKyzFhlEjEEsH1vywZJSVycSJdgcu6+0BnDcJxLAx7MM3DLkKqZ0fnnI7xtT3gWPDIUKiUOrNaOCfu38x7z35oiIlEMGGzFuGRwhULuP658yuiYFeOheAFOnBKq13njgdkwcN2a9Ad+u2XpX1JXHDxnHjr1ryKcAkBq/vy/KJ0bHA7lFb6Bf10ni/cfZSbITlSMQAJqg1DBr4aDCLzYIU2tHbdKgaeR1ADiPEqH83d3NJfW28rHslwzr7t1mm58zLgJAxtU1ABAWJ5Ng/5r5AThwbHY4cKRle/dksCpMu8bv7jVghy+XDIUK1iFt7VpF2fW7/v2OMXdQ8gNyGfKIcQWDIwRaa5XNAMCvqVp17O2A8bWNa4Fm5RVA3NFz4v3hzPi+QFDXdgByxwCdfTPFdvnN5CsQ8AMAAyoRMycwcts9gjssX3tefzv7KmA+AuGPUJjfmCFdJbOVV403YGTQbJv3TGZmZmbmxfzMzMzMzCyFBGbifTr+XcDy7RT3ZEhVmLv83bj7katcMhQqWIe0tWsVZch43IjOD06tUsiQRowvGDwh8KlXqvkBQD/fzaMw6jFD7vF0MY1xMXsYgPijgJ2iZfjZ/BDZiYKFC6QaBAOB0Q43Ty4lfPjsCsu3I9KFa+OfDR0ktfZyLwDon1QgLJgI0+L3Z99UkeCwueHrTcDy54zuyZCoODFtY8vp07Y9mm7kkKFQwTqkrV2rCESHDgB6f70vTi5DGjG+YPCEwAPPUAzBZwBr0sCUFtGM9yzayChahp811Sjmv0rh6yBdA6vh2eo+e3nqry9fQ28AwNedpZW2ckaaEKEsV1OwUsKXl7cEAtnVLUpKSmosJeW8MiQqnl3UEl22/2P/LzwyFCoYB8cgRBUhfmEA4O84mpEhiRiXDK4Q8HC7VOO6uqw78FrSkS7YA1gNBgE77Yo/1Ju0KpM5QttcA4CruFd6X9KKJ5k7dgTyOvUGgJpd8dJktJUTSsv9ADNaqJ/FiYRNx1YZkWksuPgmgOxWb94xm0+GREVZ1iAAhvk7Tg/nkKFQwTg4BiGq8Lm7QvgCt1TKYCPGJ4MrBBq51LQKwDXVv3xhBoD0muEoXzqjC5APbAuKEbDTtpFHAKB6y0gn7Vpflc9tI44AQE5YP+ntBEcu7f7o61CUpi32AYDsGzYYqCjAxJZ7j/cDcDK0h4vplZEgtE7ft8wAbBgXGwsAayKX88pwqEBRgMnfUB4CAB0ieWRIAlEUYGIdHINgVCT80+wHXEN/4TyMDEnE+ILBFQLV3zhzGQD0LgKwNqQSsOG2Uub2wDXA+uFjI+HrXwPghG/llTAbdmr4PH0LgE/bOSha5jeu72HHxS2ERJ2dnQPQ2tnSx6KH+2nn0qaNfwBLOwu/+pchLLEWto+RlEf3AHA+dYmrr44oQWh9KvH6jOnTJn0hrDnWVpRxy7D3isL2MfAduxQALhUN5pLBBEKQITo4WjMqpgduBWjrM12F8zAyJBHjDIYkBPYPje953KGEsGbxHxAVxCzYsmhz26AHi/cnhLVI+FTK3MbnvZaUMuqtaiJa037RlncOzxw4p8aBnab3nffj22tEirbTmGJ7w+09iIioesKDbYJ7PrqEiOj72KyiVxLNUhn9N2g/Q8kcnnb473HXhMJGzBUeHUUmSso1by09+n3EYvujrJkREREREW3DIyIiIiL+Zj+TQ4LQuq8QoF5ERDT9vqDQB/7l4ukFK8Peq3Ciqiem7jqWNCWPU4YYCNsgHA5Zay0V+/uvOTh9Qpn9PKIMScQ4gyGGgP3Q3OR2r5/vGnI8ODxQZaEq70Z34WtSe8rS04/KQsFgp1dLujmf92rv+EUxUedvqRjYR9rLxQHnfbX3wl3/paTPQFtDc8qI1pLZ1VHOzggb2FpjrE4k8BsjQ67izIGbvaIMvDIUKkSHdms2GCWphVG9nMlgD/JgMBppj+JHF5ZyHDUvYJ6+rxI6T+3azLE/3a55UNGwVH89l6Dv0dVY5FwxTfPPM1unfeyvf0D6vKRtO7JmaRzxedvhOjug5xKPVQRqHHAzQOdQ9FzSNevXS38Ri9U/Nv07rs9L+rwErwOXounE5X9NLhWkrc3gOKx4z/pdTlJpljaVGDLzr5ZM1tq6VHlLXAMOj2PPSc6UGeNVHZ4FLkW60QEGahOXDDnIUI0AWDhRyk2CEy+0CytOKqu0PN+93qylooqHtXQSlDoFA3BFS3IHw+kYeP8m83o85crhQeBSpBsZPlFJXKqylizVSBK0UcpNcrKWDmEFzxcQve+3vd6spbyKh7V0GhQO1tLZ0S5oSe5gOB0D77trrQdvuHJ4ELgU6UaWT1QQl6qsJduKJGijlJvkwwtFYUtMPxIV4P56s5byKh7W0nlQtFlLZ0e7oCW5g+F0DLzv/DL0C9RwoI7AZSIANO3ouGjadH8xEBSbewq/XQTsfOLIX69CGy9MuQlJKwBAq5kLn+kkPTO08MKfb9rKorA2BCAQRXwyjFOHDYqJyRFZS1GVvIpDhUpQ6hAMhpZswgueOgmG+hiM2i8YrS28YnfnVooOzwOXDN3I8omuiUuWHFSnGlluEtx4od3GlY8B9uKRerOW2vCnUoXzoNQlGNq0JE8w1Mdg1HzB6Mmo8Bfs7v1jnySbA54HLhm6UcInuiQuWXJQ0gqACCey3CQ/XgjmzeXmxf3frDdrKa/iUaESlDoEQ5uW5AmG+hgU93ExJ6LP5M9Fv64hT/aG5bnxzXscHsK6J8UJjgYALhm6keUTXROXLDkoaSVBGxlukh8vZOy3tb/2XRNYf9ZSWsXDWqoFxf1gqNGSbgdDCbSqri8pXjAaxLozHQ6PA5cM3cjyiRqUHEsOsq0AGZxoOzPgBmtptwGLUvLGF9aftZRW8bCWakFxOxiqtKTbwVAArS7Wl1ReMKogJuFh4BKAnW5k+USNXGLIQUkrQAYnCmd2j7UU7ze6pbR9OKNJfVlLaRUPa6kWFLeDoUpLuh8MOdDqYt1b5QWjTt446lngErDTjSwYqElciuSgpJWcsLSdGe6wlqy1iDq4tb6spRMYlEeFs6C4Gww+YJQvGCpj8NQ7Uj0BXIp0YzsWDNQkLu3koAQnlBOWtjP3hBuspX1Kj6nJ8APCcAb1ZC2VMCiPCqdBcTcYnMCoZjBcjMET77UE7MDlEzbgci+kwKXTm/rWyh/+9H3LjMCGpv4G4dvTIVKbuNw9shQoTXvLB2yroioAvVc64ET7mV3hhbkiXljF1lQdPJ4P4A9X0DQjQ8JaVkE6FrFKW4VNhvOguBuM2OXLly9fHhK5fDbXVgzVYLgYg5HjBaOWCsYtOuBx4NJBN7J8ohZxKZKDTCsBEBThRCk3yY0XCsJChu9sB1zIih1UX9aSqeJmLVWC4nYwuGhJ7WC4GoPmC0b3J4SFjchniMltCWFhI/IbArgU6UaGT1QSl6qspdhKAARFOFHCTXLihaKwgskf7t0zYMQVqi9ryVRxs5YqQdFUIQ+GKi3pZjBkY/D0O1I9B1w64xMVxKUL1pKhGrkJSx688Ogx6tPL4AHWUl7lHuQoH55r1lL76LrJUBtDY+5R5AMulcSlvq8S+n7XugCXTohLPZeg79GtA3CpE5f6vAQPAZfOiEt9XtJzCXUALp0Rl3ou6bkEnWmC/l5L3XTDX/C9lrrp93G66bmkm26esf8H55uABHWo+NoAAAAASUVORK5CYII=',
    '1708.06828v1.5.png': 'iVBORw0KGgoAAAANSUhEUgAAAmMAAAC6CAAAAADHDRVvAAAhXElEQVR42u1dd1xUR9d+lt5RUNFYAI0dsAAaggZjf8Xoq6jYY4tGP1swPa9R00xibEkssUSN3Rg1ttijYu+KYAMRxSigdAQWds/3xy67d+9WYO9l18zz+wEzd+aeOTP3cO/cOfPcIyEwMAgKGzYEDMzGGJiN6cZs5Y+142XpRxVCQgIKFky4yAP0MvSDPSsZmI0xVBEmXbD+Ptixy2jJyFyFduw+xiAk/ij5vYTZGIOQODL02TFmYwwC4kmNEdjC1i7Y2oWAWNwqvE5pmqMyR4mpLXw4CVlWcV1AmmnvDZQ8t/fOtnMD8DDR199Gsz67jzHoxcmO9gNzDigzx0OPyTfNkakSpyNqjgVWNKmzDNgXVGflkp8anIB0xmH3nV3ucev/8nqDZguA4fVarq2ibpBAgPLH2lGV/bg/negEohWZ47aXiXa6HFYnKLQHERXbfUlE8ib9Tz2pvYVWNbhFFNNaxqlPOfVGE5EsMK2KusHuYxaMLYOBDvX2FAAATeraFnBrXE+VAKoBgIMjAEjck8JrP4lGtbwCoPO1JHCqeUzYngckTazFnpUMfOxJWLv2N/8XewAgNaElgK7XmqkSvIsXCACIeh6c/GcscsCphrGFG4GNw9h8jIGPhKb169WrN1bxZpmI2oqjqgQPXgAA+YqwDd7hvGp1+i2lwhIPsHV+Bh42xwQCoLl/ZVcDGuOR4qgqUTahLlW+AAMAPlxxtRFOAfJXudUmdj6dFP0veK/MvX3/4YN795F/O+lhyr1EME+esbex+EAAkAyR7gRQN+AqABTvVCUARwKQVsw5J2/R242AdODQA3U1oFOzJXFB/wIbe7p+pG/ggmN4tmGcb9MfDqHSnry1L7mNbcpS/O2N1XJAsvLsfgDL66kSQKtMADs8CgFAmgsA9s6lAOLsC594q6sBkne3ta3K/xbx3vkvIpKIiG4hovLyV6CG1PxayxdbytpFbIiHc+sUIooJcnULn0JEZ9t+tu2L7dxERoc5+7/dV9etW9aBXu41e64lou31v93/5ZVJYR+XqqsRUZZfUdUu/4hlY3GIIiKiZPSovPxBQ3HA/FqnTbDkdb4nt2S8xLNLOXT9fp5cXaf05tVikmdrVqPnX1ah2nZW68l7a9OWHmYXu9OiO127Nj/h7Q1oTrRsWwKQeHKq5SS1xarh+Ff6kmTPHgOQPn0OlGakofT+cwDFD5SvSQ+PJckBKMuSC3knbxvYxXunarYrS85F8a7TKHn6HNn5ACC9GC9TlClTOprgligaiPv0JZzXfReSm1fk96+0MY677VSbWrP++v7i6DHyVctOt18KcNxuirIL0aNJvyfvp96HJ4//vkHf3WVeO1rR78X9DqcBVUq7Cc2SC9GjCVs/KTo6fPj6l8zGenQ7O2d61b4hizkfqz9s2LBhw/oq52Mcd9vrndcS3cO0a0RrXDKJ43ZTlR3V68n73TGbSpvMoCSV125J3Ryig9WyiJPiN6FVcpSImk54Cf2uaVekVNUu36qa83frQUTk+iUR9agpIyq1/y8RncVlou3VLxHtxV1Sl2m88X1zjkhWzyWfiIgGBBJRdD0iouBWRETZnlOJSOa1hJPiN6FdsvhltbGqhp2FPKdb2AC2Di0BOKEAiOovSb5xFjmcsgINT17tW4B/6p7BAFAtE4Cdp9prdyXH4RSAGnc5KX4T2iUFzLNgZb6kCOWPiXDi/JYB8pVre70ZrlEm0/TkARgbu2UwAExafzMg7++5aq9dCmo5AdhYl5PiN6FdItPbD18JM5QKYtZsAW3shPLH6HywVNdRldtNIjHqyUOtyYtrPV3yX7XXrjm8QxQV1Sk+9JesHsvvR4p59sHSyQhzjvCpcIlAu5rNvO+3Kp+VjkV8d5va7TZR6XZz62DIkzd752gAf7/RR7O81SuHxwAoPthHneLL0F3iVALcN6r4xUNpjYd67+iPfwocUFLSHEBBqqOk2Lc0GXD3Lav2IB/wd1VrPTUaKEhWNuRnbORz4p55dDZUwWPSUuP314IDl7L8BjWkbdG5D4E63gBwy9GuhF7l5/VJ0FNv0igFKa/AUJfFX7vIQa7yr2KexXG3SUsAUEkJgBLIOG43L1WZTJ8nD74frNi083CO2mvntHLvDQCL63NS/Ca0S2QAWicCRv+JP10+amHErO++AS4vCfAbshkAnn3g325pQebW6cGN/1FWS28XOG1rpvq0RfU6AM+3Tg38z7a9e5eHRt4wssq8cdhSAPff2KynQlD7uUbvnCtf3Rf1becfl2+ch6e75gb6JQHAmgG+43Zq5fVBdz2Vu9hgl8Veu7jYv6m7Z5dPKC6quZv7m+9x3W373qpWvc+5tW+6NeiX9VEb99aTOW6302VlfgOz9Hjyclo3aFjfyW5QvsprR8fCPl33wT5O6rRWE1olfgOzKMl39pzjRt4r/wyWEZGsVzAR0Wz8oDwceo2IKDka3ygPLB2Be5zTUoIVjp17GEVEVNjV+aKRIeweRUS7MEavczX8juFVAvlEz/NERLTGLpiI4gaiYykRUfGbctKR1ydGVz2Ou1h/l6tg7UIb2u42XW43I3gRcJaIZBeaz+YeTb0t10rxoaNEevm5sX4M+ICIiM4HExE9QKBCRuZohYvwy3avKg7Iv/8/POKcNnK54u8jjCUiolPoYqRjkVFEJL+Ur7fCtv6Gr+ZC/KY0tj7BRBS3bJryP+L/FLbDy+u1MR31OO5i/V22iP383sEeCPJ30zGtsG3Z2qHM7WYEf9V9DYBN6ORL3KN1m0q0UnzoKLFv62WsvfspAIBQPwDw7Rx3RfEaMkRZPC7xJADgTLjGWVl7eS7D+rhj0vw72FVvWb+/nxo6NWOmv3J/tWSq4u83TT5L4Nbg5/VBq54m8VNPl1+ivdZtbqYAgOxgT3Haa7L9LwCQTAEAjMYaAMDRsvl5tMtqxYCHaZy1uyXPVv7GfzT8ssg+e0KaoulrBSB79oTvtFWV24Ua9OFvyo8su7wd3AAALutK3uZ+24Cf1wetepruYj1dfolszH/nB7P3H185rMckcdp7D726fHdWqlz96+++qQhAXAvbste9QdtzAOS6a94jT7bXlHL54x7zuX5ZzF/n6fx1MDi+VgC4FVpziqbTllseftKQokfRXPUKv1Lx97WPLmm8KPDz+sCvp+Eu1tfll4tfmX7+eKJMtH5sqQbA5ydl7h1sJaKYxLLJCcViGRGtSSONyckbK6lsPhY6b97caJ9lcuL6ZW9HEJE8nOtHjYwiIoqI0nTaqsuJ1gYbmki1wTbevIqoKNDusno+ppnXPx/j19NwF+vt8svFr6zZLqKReD2JTt48oXHalPmK3CisAUoeNVIVhzdZDSCDx2ZMr6ZedHj//Y+3XNjYNRlqOiTSrl0DJKOQ82mUB9DVZlNZbcVTzj1uBOBvf1Oz3Cvd8NqntuvC8TeMLDaQ17uWqVGPS/zU32XGfasEqg1efufvmnOyAQBhTQ49xr5enFn6uEs3cCOQd046j3rW4M8LPfPUdEi082sTMP3oOFzJcTh16tSZGnd555f5VDXKDduYH9TFWWWJ1rPiZ3Er8fP6oFGPS/zU32VmYxXGVgCQdPol76pifEfLf8PvAzgVRtqtwqFuvLOc+D4zr+5396rpkHA68bnLkq4j5Smo5eTk5LTxI50OXYWfVVVe6mRI0Z64qkp/q0p9HDLvDLcWP68PnHoaxE/9XWY2VmFsV/zpAeVr3gibtU/d3DgVfHpvyLG35Z1VK5cvxw1p+PCDDTM7VAfkdC9vzoX0hZtjm8M7JCQkJKQO9PtZVeXZBj8vEO2zr+zxlq32XNn9Zv82d1cxP6/3yauutzmma9euXUc2/SvbYJeZjVUYCUkAgDy0Uq6ydb87abj6FQrA2Kwxffln+fCXsoqOSzqr6ZCn47cA1adH3Wz1ymEAKN6tp3WN8gyDNub6c8YiZXL1yDLdgObfJD5W68rJ69/GyjtPg/ipv8vMxioM2dBsAFg2uq7ywGjEqTz38bcI6Fknzw9q/ywAoO0VlHltpQBQNPHhF0Fcv+zSFwCkr3H8qCUlAFBSAHB8qpxy4EqwQU0HLP58PQHAfklTAPHKddTpHZW68vL6wK/Hcxfr7TL7NlSF+xF4eNxXB45N769ydBVW/0qZutXDyy3kJ6JvtxN9193bveWA22WVDjcnIqKE//rDo/+wYZGtexzRoEPu7PnZpn0fL1Z7WC/08fLuc/lCHy+vyHQNn6raF0shfxrxDB4Pbr/u1Kbxm4joQldPt7B1RESUNEpHXp8Yfj2eu9hAl8vAvqNY7n5cDqa4S7LgNuoFx0Ntaxi/+zU80FzX4dslLRwo1xO5znZ3pY0Vc/jH+U0M7topK09t/8De2MavW5ey6nX1qNL9Y8zGxOrHgoeLzK3bZy6fWcMeRWZjYvVDGvHHK+ZVLbPnCWdrsDE25xcLDksnyMwqUD7hR2f2XWt2H9PA0fip5tRsZd1e1rGfn9mYiP0ocDWnZi9cwGyM2ZiVPpXM3F/b2UL1t5Pyx9phVf3oZFFioF6DnQUGBmEwS8g1WAYGsLULBmZjDC8J2HyMwVrnY2ztgq1dsGclA6rq+2MFHCO20V5Kzrj17JXX2LgxVMbGGmZF+DgcT27SoejpCT/tkDPJv62Zom1j6VN/8RRDXfm6R4XydxTfJ5L/kVBiM6VmWdGI4FCXh2e7d7bUkVarztdUSM1zd9+W+kX5aKWVyNl+165ZlIvAgvibFmXV7hLRD1hCRDdr6toZGjSNdyBjxvhQPBFjH6x8yg2if948S0SU122mlNb3UZX5ALCfKbfU/bwc1fmamk9zLUWPtZ63f/cEh/X8tBJ/eXTZvyug1gXjH9qpmCA94p6NV3zxZRkR0bAXOk4J5ttY0YPSheLY2J4FRETXuxKRfMAwInpV/V/QbcknK5Mtd884R3W+pubTnK9oXu3NREQT7As002Wfq3J2zSdKkdTMMmZjFRSkhyee3pqTCUo35X7s6Gsr0vPmfCoANLoP4Nj2GACb/1SV1Zr0zTg/y52UcFTnayqc5leenicA7UsSNNNK/FIY6go08M1YL6ggLRuz5062Our8rkzVoe6CRXJgTy8Ayz1bAwgJs5aJL0d18WCLRd1vACdcX9VMK3ES7gDQCCcFFaQ153+VmwmTzmgRtHPfisYA5tt1y/96yTNFwecrHJptrCn6hRr1/Xs7Vt7ZsAmgv/0frXDLGdNEXXj9WKndGE9LtTG16tqaCqZ5qPfzI61H+25dV00zrZyKX4QLADjhvLCCdD/YlfMxXd+VoeBpRNdaHOPVF2U+Rimvw6GjlIiyET5PRvfrHlEVBWyS044mDyyWw6dSXUtT82mupegRTwCIkfPTCs4eMJqIqC/cjc75KybIpO/2aH9XBgAQv+L8m1VyN5A2et8mtvdTIBfnBtnAv9v4orKijUMk6OcWY7EPS5XqWpoKqLlP0zoAFvQt5qUVGqmmSsXCCjJ8HyM53d/1ES4SFbZCy2lHFO+Vh2s/16ovyn3sRlgGJXZBOxk9h7+iXY0wSjQe6RZ6H1OrrltTs2jOV/S861sFse0AzNFMK799CwwmIuqG6sbuYxUUZNJ9TPu7MgCO3nSpojhi73xbA40Oz75wAB4OXgDgjHhl0bmTAOCuylsa1KrzNRVQ8xmla1w6nFtfHbs108rXu2ooAIAXqCWsIMP3sRnuiUSxuCiT331ElLnQ7jhR8Aw6Z7u5Ku5jOW6KOUDHhUTBzYiIlmCTsqymQzERTcENy7yPcVTna2pGzfmKurQgIqI7kqaaacq+ISei3uhMRNQW44zdxyooyJT7mI7vygBwQ/tZ71bFKoazJA8A0CAA6JMkBZCGECCzCECrZQ4Abnk2t8zbGEd1rqaZRYJq3jRVCgD+Lu000xlNgyYD6IEMAPJU9BBUkB4bK1LM3nR8V0bxCZkP3KOyuPWLUSzChbKPXgQAjzPfAN51PQjQwXGN8ax+BwADmgN4cGKhhQav5qjO0fRZ/Q6Caj45d0ohQPPsv9JM30jDXwDGNo87C+xI79hfWEE67m17B3fycffpNPiIru/KHIj0qN8vfY+nW7OlZfWLh3ar7d6i70Lh5/xFb48/fn3FqBQiogsh2y+9OzSXqCBgOBGVzlx0bUvL+Rbrr1SrztG0IGC4WTXXUnRjQM1uvf363uGlC7t5rCAiejrQbeDb7mNzjfsrKybItO/26PyuDKpub9+9iy+CQhUftck+8SyUG6894ZxXmA8sd4+iWnW+pmbTXFtRynjs6m+vnVZtp3hg6+dqyh7FCgliHF6wfbBsHywDGC+JgYHZGAOzMQZmYwwMFmtjEcofa4dV9SPCosSwtQuwtQu2dsEAxuFl+HeiyFEiPoc3b/HDJ4HTfMA4vCapnrUit7BkclN1ycpUpxfTawo/ZJUZMTVvt9BlpL+7u7MEaNG2rHRVrXa1bDL/fj4eQnF4BydRzgDPs4zDa5LqGZMziL53OKwqGfIF0eUm/wjhE9cYMpNHzDCHV72X8jdVcThg7wRsEIzDOzGZiPK8/EsZh9cU1Rc6bSPKwOtlJbs8i4lo/ABhbEzdrukjZpjDq9r+OkLOtTGgzlbdWtkZ4fD6mnAb3bsrvjrcInbebin08+b8C0DF4b0MYHMJ1ExYi57EqFWvTQBckVlWsincAUDEmBcuwrZbmRG78vR8tARo/0tCCJKCP/Z0ktBHDis507K3fGq0esvVxPmYFoc30ddf8fKZfUsa9kSXzdW/LQXghnTBbazuxPpTbTQ4vFYzUVarPjjKHjiN3soC2eGeAFCn+HgvYdutzIjZYtHN+UEK3m5S9AAACzLPOHIqDB0Mg99RNLCfvzhm1YV5ne4SEf2wKP78597KZ+VMn/rdVDwaWTERUSu7dMGflYX+6Hj7z8gcIrl36weffvPRHXXZsGsLvl+QbbHPSrXqRETFPUPKVH2Cd4iILmOhIM9KVbvlGDFtMcXegGTMHKetRBSfTkSx7ve45eGbTi5aHltkslYV4fASXUAM4/CaxuGlcx+GTsgvO34X04mIrmG2IDamarccI2aEw0tEuXVnahSH139/6yKPZmcqYmPbq18i2ou7RCc8rxLRSoWN3ZyUx99KHNi1UAQbuzfifSd0f0L0ELYpRDSqoarV60REbftbrI2pVCci+Z2eb2WUES/xIRHRdXwkjI2VtVuOEdMhJq5dHQB4S3mrmokkjeJFN4hoLmqkV8DGTOXwUkznfMbhLQeH95ljiPI1PBkzFM/KrwSxMVW75RgxwxxeIsp0fkXHksdB4AsBObyr/9nvCsbhNUl1AIB36KWDilQ1FCqYYJ7CtlupEePxdncUNtRY6o8d+DsAV+BGBfyVH36wYWaH6oCc7uXNuZC+cHMsgB7TN23aorl6cX2jI24mCH2dcuM7ApDM6ngXdoEFACBDWZTlPt2kAKTwtkwTU6subddWCsAL9xQlnrXTAOAp2gnbbqVG7Eojb0Ay/JwkX2FjqpNy4gjAzO3TAJQCNcpvYyZyeM+eWWwD/OkIxuE1rnrRpRvpAB6hFYDMIkgikwHgvlewwEPGGbFyc4e5vF0A9+Cl9F4rOLz1bUYASAQ6C8XhvT38+cR3J4xY5QfG4TWuukevY/WAh/ERHZUc3piE+wDtiLEVeMjUI1Z+7jCXtwsgFcr7iZLDO6NeNJD/EwYOEIrDq/SMBjEOr0mqZ4z84fSp9pFPyji8tCUiPnPGcKkwviTOkKlGzBh32AiHlygYq5Srb0oO7/HAkO6edX+WMg4vLITDe+06tQniTJrT9xeEtZEItblQ3a6pI2aMw4v8x0346qan+nmxfbBg+2DB9sEygHFGGBiYjTEwG2NgNsbAwDi8YBxexuFlaxds7YI9KxnAOLwMLA4v7xiflApROLy84L/Wy+EVKQ6vLp5theLwlluQOTi8PFKqKBxereC/VsvhFSsOrw6ebcXi8JouyHwcXj4pVRQOr1bwX6vl8IoUh1cHz7ZicXjLIch8HF4+KRVicHgd+ZpZLYdXQM251FsdPNtfCju5Ag18H6yfIqQgc8ThHZw3kEtKxUsWzNbcqos3XLyQuUNXzh2swbqoWBzecgsySxxee0A6P+R/EDWYrRasJg6v1nCJEocXoNgrTi1D1TviKxiHt/yCzMLh1SClisLh1Q42Z60cXtHi8GrxbCsah9d0QWbl8HJJqaJweLVtzFo5vObUHIaot1o82xxgrMI0HMrF4TVdkHk5vBxSqigcXt1BM62RwytWHF5tnm0F4/CWQ5CZ4/CqSakQjQjLhdVyeMWKw6vNs61gHN5yCzIDh5dPSoUYHF4tWC2HV0jNudRbDZ6tgnrbQWEahegoqCAzcHg1SKkQi8PLgXVzeMWKw8vl2VYqDm/5Bel+sM/FIiKiQrd3iWi2/clf43f6FxBRv0tEgZ8SFdYLzlRVjjxBRCkOEcLPx8bNISJK/U8xERF9iwdERBkuwUS0/AwRJdv/aqlzfrXqnOHKcAk2q+Z8RVdj/Asi+dfVUoiuNrhMlNcKA+VERwB/InrRHGeIfkdHmbH5WAUF6Z/zl5fDyyGlisfhVQf/tXYOr1hxeDk828rF4TVdkDk5vHxSqigcXj6sl8MrUhxeXTzbCsXhNVkQ2wcLtg+W7YNlAOOMMDAwG2NgNsbAbIyBgXF4wTi8jMMLtnbB1i4YGGAVHF55VZirnP2PVMGgk634HF4AOL50mwjdo9U33Ww/dNXVqLDRbM3J4eVzdkXi8CLjrzsI6M/9wr3h8Ln6OhD7Sa+argAw1Ma0DpiBw0tEVNAoUoT9/NJew2S0NShDR6NmjGYrMIdXi7MrEod3qUfY5q3+vrc5xXrC58JwB5YpTaeTiR3Quo9lDWoMwBY2AFp2L3TWtkp7Xaa6UJR7wbeHn9tg0I8zl2k3unv/WqBtp6m/W+htbJ9/IFBn0YzDQNB/Uxt29VMX8fPmQ/7QhYOBtzCmvwsOTpLsqgWfTr3jHbhVSkrqLBpUvg7c+qWBqwT0+RoTO2AODi+Ac/41xLhQK4LcAbz2y3wXrUaFjWYLc9KP+ZxdcTi88+BXC3jNJvGPITApfK7eDtiMB4B1Y/1M7IA5OLxAEVdv4ZCd6gYANfOvaTUqO+wJAHWKj8PSObwigku9vQpXAI7u4I6QFhvXlA5MBoD7Z4ZVeM5fEQ4vfpwqEWPInG1J8X9x+3V+oxlZbgDgibu9LJ7Dq8XZFYXD65IJAFSMc9zZOI+Na1IHGgGQT10vMbUDRoLlrN9+sFnok0GXbXBnz3Eg9KjyeNQfP7+prnW1Rn1RrpNjwHMAeIBsrUbz4Kr4x82x0PuY0/EhsUHtj9oDuJ4wXbKz3SHVNISfNx8ctkbl0K+IGQggODUXQGKRxndJPooOfRL5yq9h5esAgM2B1TkmZrgDRtaXquUVAJ2vJQFp164BklGKw/ErznNMrGT9KJEu1BcJTwHpVZRqNVoEBwCQKIJBWiKkjd63ie39FMDGIRL0c4tRlfDzZoRP0zoAFvQtBr52engf+MkOcnXxwH3zBk375HafjPJ1AJB+MpFTYqwDZuDwLkgkovZirF3Q/N4Pk2Z/hdVajZozmq0YHF4tzq7wHN4TAY0+7DevGlqRsfC5MNqB7Q3IdPq0GTi8CcXe2dnZpSXZecLfDGJ+PvH3e2lopdWowNFsYUYOL5+zKxKHF29cP9Lzp5gcBJgYPtcQCXmtfzno00bmYx+uuNoIpwC5JNF5zpysdR+MjQB6TA8L7zVYPd1O/R+AhFr/axgj/KXy9QVS/FppNSpsNFvzcXiP3u3VJyfPgcvZ5efNz+Ft1ywfePFF3Sl+SCb0BpDzMEACzDxxeqD+8Ll6OwCUHu/KKTLWATsjHN6JSg6v27PE91F9+rmbEUoOb5hqhhcRAQDbA34W/krFLljriZyT8+00Gs10cYIk8qqA0WzNwuH1AIAGAWg1RM3ZzXRx0sibGU2TpA5QUm8PfIf+dbEHTQcAGYFpk5YYC5+rtwNAQn4ZLcmUDpgjDi8AyApyRbhSe3c/Ahb5j9BoVBGHV9BotjBnPFxO/Ntyx8MtJ7jhcxugRy1c+sTrDztTw+fqjYGMf+CmOGpaB8zA4SWid19z8+zyneBz/pu9Tl55v3OaZqMKDq8Zo9kKzOHlcHZF5fDOqx35hlt0ignhc2EsBvJufKr0GJvSASuLw/v8QHabMD3rvWaLZis4h5fP2RWJw/siydnf1pTwuXr6q+6AdEOkj+n0aQnbBwu2D5btg2UA44wwMDAbY2A2xsBsjIGBcXhZP8A4vGBrF2ztgoEBLA4vQyVQ5CipGkFm4fCKFwKXw+Hlc3ath8PLjyAspOZqDm+hy0h/d3dnCdCiLSoTh7f8gszB4dXF4RSaw8vn7FoPh5cfQdiMmhvi8Kp3EP5GlYrDWw5B5ovDqzOQrCA29oV9LhGFv0tEuzyLiWj8gLIift5y4/DyIwibUXNDcXh3q5468srF4S2HID17rXkcXpNuyLUmfTPOD+JxeDe8UHF297yABodXlbc0nE8FFBRYR1/NzQ8Can7l6XkC0L4kAUgK/v3QydiTYRErJerwuaGuQAPfjPWCCjIPh1ckcDi8fM6u9XJ4hdScy+FNih7QrWOHi5k7HCsZh7fcgrRs7NU2nExYnRmH3Xd2uQcA89d5On9dtpP589oNunMIU9cXzluYIyaHl8PZ5XN4LdTGRvm/1+nO7g1fa7+oC6h5qDeOtB77xdZfqwETRwE4NXuvV8Xi8FZCkFk4vMKRUKGPw8vn7FoTh1cTQmrO5fC2AJA3eDrnOwDFxQrTsEO2oILMweEVkoQKPRxePmfXmji8fOqEgJpzOLwAMO/xKK5GqstfLKggIzYW9Tw4+c9Y5ADt/NoETD86DgBwpOuXbtxaQQAQsiND8OvU54d3Ht2fG4kacIcMAEoVEwFo5S0OcSMXzLvZ5VBfOb9ASM0vvOaTGNsO2PMdACDrh1e4vEhnlIU2dRVUkDni8IoYAlfF4eVzdq2Iw8t/VAiouQaHF9hR2JC7Ql/BOLzlFmSGOLyihsD1HT7WI8WvlRZn11o4vNohhIXUnBs+F8AO1SWqTBzecgsyQxxeEUPgxvbLAXJOzrSDJDIZUHJ2M4ugkYdFx+EFL4SwkJpzw+cCuAflu2Cl4vCWW5A5OLwCklChl8Or5uxaG4cXQLFycvysfgdBNedyeAGkQrmkpeTwjm0edxbYkd6xv7CCzMDh1cnhFJzDq+LsWhuHVx1BWKm6+TQ3yOElCsYqRaJycXhNF2RODq8ODqfwHF4+Z9d6OLwQin0sMRyHN/9xE4lZ4vCaKojtgwXbB8v2wTKAcUYYGJiNMTAbY2A2xsBQWdjOFkpyJ+WPtcOq+tHJosQIvXbBwMCelQzMxhheEvw/wENlfKiXWmQAAAAASUVORK5CYII=',
    '1709.04959v2.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAmEAAAEECAAAAABFT7zjAAA4VUlEQVR42u2deWATZd7Hv2mBcrS15RTosrIIiCgIFhHRRRcVFvFVlAVEPACVQ1S8OAQVAUXxQFdFUZTlUrAciop0sau7oCAUBCyIlHIIlrsHLTRtk/zeP2aeYyaTNsnMpNf8/kk6yTPzzDdPJ5nJ88nHRXDKKRsryonAKWeEOeWMMH1Nq2ExTovEPlfNUF1UlVZbqWO0fZ+rZqjOu6RT9lat8p6QOeOwN741cuL/9o8o/PjB8Shq8TjmHD0/ahiAN9Ioqt/YSr+TZ5bflRjhTWpiq5plVWpUbr2DVUS+w9delU9Es7CAiOjD1jEZRERHmp4wbBPEaiNRx3YSEdFbmGfzhuC/z5rY/Lqk3oS8kaqXWhD/YI1QD3C1WrzlMQCN0RgA8FbssBIASQOaVuZ/w+27AADD5g2O/LY1sfl1Sb2pAakFfwi/KO4b6a/m83ZMC+pdtkJrtXLT8KELKqwL2th4l1bXmNSCHyFud0f5zzvve+WWnuob7a59jZJzLrJvl4uPn/5T471R7aIA9/FT7Wr92rk2gOz0JlfUkx8DaP+uth2j2ZO8Sz7qXogGLs/x042T+IP2d9cgtvMbCju1BeBWulSs9kzX4dolJ061bHrgTIc4K3pZSVIL/hg2z/WK5u+3ku4pAAB4H/jjr/X7f2Xjq7R5bPLsSdtXd/gW2PzgVf97aebfgKJHP2595q5v5MeQM2Blp+/+nsuetGYffT5z5nkcGt9lhXjQ/u76x7br/sQOU0f5ALVLrGe6DiNj3JVfTdtxoOMWK3pZWVIr/6Pap3jsww+mD+mXTkT0Ib4kIvpwK9H3rpFEDxOtG0hE375t54fSIowlotXRO4jOYlDJ6ilEo8YSUW6j7fJjA+4jorue4E8qxWIiIvIlzhEPGnXXjk/6cmyP1TpH52L/RcS6pN7oO0yeuGvyiG4cbEmolSO1oEbYuwcPHvUqf0gjjCbgc3qY6Ic60zO87kxbT3uwgIh87fsREWYQEe3DeiKiO26WHtuJz4jo/b/wJ7GsqOUc8aBRd+0ZYSK23xYTUddx+hHm12GiZmOI6MFe1oRaKVIL6nNYU+O33+nrHrwaQI/JM55LHDE9AhfOO6eRC2gDAFtwIQBcuNQXxR9Lx6bDwMmugPokufiDEeuuFFs7evFEw1NFAfskdfhPAKLcFvayolMzcy4YsyT5wT8BRVMf3bxhXtG79r9kpbUBoLbyhwcAvCQ9FouhyezP2uxOYXY7ABAPRq67ot57ZtnNrnW6LhVmG3XYBat7WcGpmbrgfPlLX2YAq79p2G/Wqm/sfY18ALw/X+dif/dwZQLAbz2jxGNXR/+s/N/xf1/4cGKDcp8/GJHuauv8kxP7uHAORSmsS8qNX4elKwaW9LIypBbECCuC2++PrCwAwOO9vADNJaBJO3tfpPUELMyZzTvwl6fe9wC/pL8uPdZq8ntFQM5H/EnRbY7g4J8BcpeIByPSXW1s+UVNgKMniwqPsy4pN34dhs9dBKCILOplZUit/O8lX94c9/J3zyjv0D9+8GPcC9887pueVm/NO4lA1MIJAOpO6XRuwz/tfb1iJ3XduXnTRUh7M/aV719pBMx6Y1Dfkxv/20F6DC80vPX2kuyXxJNefqbJ7tex/ZXiBVFPsQcj0l1tbM1nv5LX5Ojnw56fwLqk3ug7nD7b+8nZ2WO+LRl5kSW9rBSpmT+POust2ZPtI5vPJX1Zebpl3swcv8e8mdpn5e0plRvkBeiuHeeS2gsHGdlERV6pS+qNvsOWhlopUqsi88NcC+4P6zHU6PlhlSK1qjG15AAOUjiP1eyqHKlViWNY2vIYz4XPh/5YzT6GVZLUnFnUzixq513SKTiskVNOlXG14nknBadsqeftu1oxbVrNinLatAjsc9UMtWp/0ieX80nf+RxmY62Y67wRVZsRlpOm3Pq+f2/5cQDwbXj/qzxl2S/zPz1RIZ3/qG84rXL7thoOAP/9vIKz/3nu0lxdvADwYQ7gWVkAnPymxoywnJUzu72tvDpPegY16foecObhk8nb2qUAoGn77+z89JIK6Psf7jbhNEtc16IIAEYP9VToKdbDn/TvePsGTbwA8OOoHKBwYOKfWrQ+h5rAfAOAp8FYdfrcqztfqve3AWNva/HC9CZILr7n0o5Y0nYAEj/q1D8h4n1fMizMhkkAgPl5FYriLd66Ba1m37E3TooXQNFzBAC9Lsm/7P6WNeYY1rRvQ/Ve/M4CIA7HfIufAzCkeBXw3TkAtdvsj/xBIOUfptr3vKVCk5/fG8BV+alyvADeGAEAuPb9T6e0rIlXXCedagpsuehyV5tSAPVxHEh6ajVw6shl9vSv5OiO30En9+wEig9vO+nbs9fHHtraPh4lR7afoKwtClaH7DWbiqBbBgA4n7oykx2Pd2Uop2S5+zaVVGDwBZtaAnBdkKpd/MNFLSLVAxGtPtkKPZeMApb98lkdV/p8AOnoAYxLvGPA/x5ZUteeGPZM6vIJPJN7jNLwfwCAhfdDAgwFCygvU4rjivC9+38Zvz56DADeGnjN2QocYb97YgEg/jfN0vOrhip3it+c/sL79l6dENHqk7X8mn4w1fM29c6SB5svU2eiebv18BDRmS7AbK9dMxBLo2cR0dDuWv6PiNydPCQBhhILKC1TSuCKr15WRFTYcjAR0b9xKmIzEP3rRywiIurWWRPv9D/oO2QS5V9xmHwDR9g7rVNEq03W2gr5etjdH+yZPUQ5CXuuztpooOjJh1YnTRhq1ztOrUYAEAcAddENwG0XPwMA+LJfNIDo+p0vAFofBzLnDQCQ0GuStEytsQvqo367dCB/ysi6QINLABmsqZhTSWVSvEdznNrYWn2PjN/aCq7RH2+09yyPR6tNtgLOJbWVMHHwdeMAfLYttQGAB65/EL2fntfjMZtycBnzf1j4KmTAUMMCsmUzDgMYncxxxfSStpXlA3A8CgGg8ELNe+Tr8uvSGuuvtbUPLuNkK3KE0SvdegMdsXYckLbm8xhkXHZq3SIg7v2Y1Mfs/Y+Hjv87kXuJPibBArJlcQkA6si4YnRlGWEtUAAABT2kZcuPPAL8jmkJr91w3WwgAWcicjDVJVuhI+zw5Bt7A/loAmz9YmE0spdfVlQvGgD+nmtXAjEE4IQf/7f0bt3zergyr4DKArIar+KKz/UBzqHoqxtj9vYF4K0EI6zh5acBePNvlpYNHw5gyZfTLvYdbQngKK60tw9StHqyMvLnkh4vALRKfhfA/1wj8Ot9CTOef/K+DmjV7HsAWH+Pbf/s2UDO1rNa/g+0bLAWMJRYQLEMOlwx8YV/FQGHtpytBENs/DofkJbQX4oXAOCGG1F33wNgeXubf1hPilYkWyHnksdGD0hofOeYAqJdo1f9uqTVQvKp39dsJ/q93z8zNk/+wLbTnvVX/Gvp5LG4LYsID0xYNvmG3URE2wcSEdHWf8RePL5kZIvGIw6QZ/aAeTP6bNMsU2p229cWvvjjX8YcJO8/b1381oRrYv6RR/RdhZ5Lku+Rhw5v6LZREy/RgREd4q4ZRTmPfpHxUo/fbP6VTRGtlKzVFeqMEO9PWc2TtV8Q0fZfG3VvCNsmmpQcaNg0u7hRnAuuBfcdbKT8MN9jN/X3f6bvQCPj37Z1ZzVsDnedKAC+rPhmB2o1bODCd3871bhCZ+8c+qnOTbEBHvNtzexwRbTds3d4tFKyNXp+mGD8Sq7cbu5T6ZZ9w1bcWxDtzA+znZ6Mqpr839qbTZ72zHmafhgaDadspyer0DFMZvwGvNDJZKafeDxT6jlzXO2nJ6voLOrZE+DwknDm6cMhcp1Qbfoc1quGfZDpFYl97uW42ZxjmHMMq+KsEVW6FTlVrUaYZfSag8H5V15u5aDZ8OuCVfmQcSt7wav0a1u8az29Ft6KLC8/RBCAQrMBOLjo0wL7Nq3JFcD5oa8MmVoZaLaSUXNuann7Lhm3sgq8Om5oK0ten1sKq+m1MFZ03HKXmgEiCECl2VA89tObkocW2jbC5FwB4FnfrGYHj++qeJrtxU07opJGjdziknAri8Cr7TmGF1HrNYLl9FoYKwrQOxNlgAgCjGajgZ2fwfKvdtg3A1HKFQBSR2ER1lq+k8GOsKbsTcX7+sgo4Ia71t+Ma2eyR8U9U7W6V7n02n9MbaCnmRWttvxSAQ8V8TsL6iEOx1oAwBsj0gCsTfsUuOnDbpH7FBZny06G/Ek/61w8gKZRm8LYkgYyo8yVu7wyquae/5G7sJDch7bmn0sv1QJoKJteKxtfM6DXtraPh9iQ0hUt0KUsE6tRewfamfKfs4esfg0YIghwmu2F3rFAwwdibD85V3a0tNDnLsxTd7JCR1hjeAF4cVjGrYIEr2TITHXKSaia6pzjmjoBoPEKRK+Vha8Z0msL75d8eGpXNEAXU97ppX12iQNVRFDQbPnpSZ8898rk87afaqg7mjGzIGXmKFX7V7E0W5chRLQdvSXcKmjwSoLMmFNOQtVUIxjT1AkAreUcKodeKwNfM6LXlBWxDbGuyEAXW6aX9gVW4CGsCYL+iKBKs+1Bjw1Eb19VauMMxJaS20/5i0vZKpJmm/t9NnzfxMZJuFXQ4JWAzHatvgVAr89lVE2tOFxe+/aZEoDGKzC9FhhfM6TXlBWpG+JdkYAuvkzfibg1M3b7rrXhSgdDBBnN5kL0tcDdWz629xDGdxQV/csooq7+dnazgrumtJdxq+DBKwaZCeGcAXnQJoAvryx6jS1U6TXe2pBeYytqoxfwMaCLL9N3wkZxYMLEwdeN4zRbK3QCkBj33UO2vvi6na8UIwwd3wR2oxNdw3ArcQ9B83mycE7+Cqcwux071Gh9eSiHXmMLVXpNau0/yZCvqLZBV1BaW1qml/Yl2SIOFIggp9leb6p8Psu19xim33lJ+1dhI2zH3iHA5tYDieNWFAZ4dXX0z8kA0pMlnkpxzrXjvjwGoLHf1wmBXhOtswzoNe2KRFcE0MWWXSo6ofSubvyt/fr1GWHxSyAQQU6z4bosAJ7zne0dYdLO+78EFUOz4bUxhJKXX64jcKvgwSsBmQnhnOCpVOecaqATABq5i1AOvRYYXyv2p9fYitQNia4IoIst85P2WS0O1COCYDQbnt2SB/wYM9bG6xTuImnnye0W2r8Kpdm+Hp6xse8KIgm3Cha8kiEz7xu93379yRwNqpbSdt6jpd/2j71s5GkJQFs5KLb9xLLptbLwNX96TV0R3xDvigR0sWWCglN6t/j2yZ/Ov+83a84l/RFBiWaj9wfsSL12nX0027ZBse0n8h3dNqhB2+H7lZ2saJrt8A+NezbQ4lZhgVe+A00u0KJqyM9uW8sQQANCpdek1np6zX9Falc0QJe6TOpEfnbbWgUNvPsTLnRZPj/MABHEkU0JV8fbPz+MvQ7qwV/zEqAmzUC0jF4LvCJTQJczA7Gqzw+zjF4LuCJHIlizZ1FbRq8FWpFJoMs5hlX1EWYZvWYTBueMMIcEcUgQh2aDQ7M5NJtzDHOOYY7B1ClnhDnljDBdycCaSrNB4FaSpa1i6sxceQ7CTz2bz6sS0csMm8q18aB/SDlU4Ha73RX8i7N65s22ESaANU6zCdxKWNoqqpY+LG+9+79zSu1j0KwrwbAJro0H/e2g1vH16tVrsC9CnQkQlJ55g22zdziwxmk2gVsJS1tF1bC6mokdDRJtZNCsK8GwSW42FnTm4w3rAtvbd4hQZwIFpWPe7BthDFgTNBvHrXyLS98DhsxaVXEjrOFDkWPQrCvBsAmujQfd+A0AZ8dPjVRnbAwqXJqN41aSpS3CxYA0z9EdRwHQoZ8K6JslpQBAe7cUCAatcpbMsPnVwwAw9QXLfgbUACWUWEIWFEf8/FFCu49hxW+epWajXIJmy08f9cneBnnP1ocrHczSFuHa/OK3azbvPrPh0KSVc8Yjf0jypU+1euSDNj0Az0uXlA5ccdWaffT5fjzboPKeZC37ZW0dw6DbAvg6+U+WbSvjhTXzf+9UOnDFVUDOiKvvXPvU8sQ9ry2dNckzeUWHzSwoluiumU/9aWrCe1GRm4EogDVGs+lwK9XSZuNPvxsVA9J8iXOIRvchSqt11EtEzbqqbJs9eBZZRbNJmjuVa5PJwKI+XrJuBqIBSiixhCwoNVFB8XGW0GaaTQBrjGbT4VaKpS3Sxcg3V30AW5KANp7tUQDQXcu2VdaSNHfwV7LN62DltUoDlDAwS+iPEtr+LsmBNUazaXEr1dIW+ZJ+QqfnDmB/TBcAAnir9MU0dzBQsn1s8SR9f5QwIEvojxLaPMJkYE2l2erLuJVqaauAV0iaTfjQbZM6zl+a5Ge1sx7PspphMw56z65msEX1p0fYSBNUbcAAJbT7XFIC1nYsA7C59UAZt9r6xcIYZC+v4Fds3ddjO6feqQ/VhxMbKufh6/Dkl6Fq7oyCxmY0g10Im3JuJrnZtEGdf3JiHxfOoSglUiNMAtYYzSbhVtzSFvFSgTSQuwSoPS0jZ+cRQGbbbMKzLCktw6ZwbTIZuBcNrdycEUooWEIWlJKoRPGpLKHt55ICWOM0G8etJEtbhM8lGZC2bVDsJa9SdlyjZg3QfbeGbbMFz7LoXFIwbIJrk8jAx3GUrDuXNEQJJZZQCYolyim+bSpLaD/NJoA1TrMZ4VYVOJXpwO0r2oFOvbBpu2axHXiWVfPDjBg2EfQf22512RUqR9gES1gmSujMQATw8VerAOBYm8IoZwaiMz/Mhup/cC0BhVMfdya8wZlFbUvlLzwUi7O39nZmUTsjzJmn77xLOuWUQ7PBodmcd0nnXdIJtYq+SxIcMVtltd5VixFm0qzmiNkimkmwI0xF1yTK6ue5SxWIzAKajenTgquP+qZedeHqsDdWScRsAFQPm4AFRbyWIYLBoX42ZhLcCOPoGqes6OFP+ne8fQOsodmYPi2o+sPdps+qE2GThGEa3mwpxcMmYEEer3WIYFCon52ZBPWtnUDXOGW1aOsWtJp9x944i2i2pOCfumRYSE83al5JSvGwSbAgj/dRyxDBoFA/WzMJ5uvxr+oVEJ350E30GBFR/nAPXTeJiHwNUsibMJqIdmK6uWkAdw4O9pm+K/OJCClhftWvNK8U8/Rp5lJkElHuFG7LUeO1KFSjasalOQ8stjcTtYI6hqnoGgRlVbBpEADXBakDLaDZPHui1P/V7PQmV9RDSVRUaa2SOuSNKY4G0YlTLZseONMhTlK0Kf8bxbUJtUpOnGrZdP/pjvE4ldkqSVpLcVR0aXQtjy/aE0P7d7XtGK1pfizddU1DuI+falfr1861WRuAdu1rlJxzEb+1s1QPG/whNjIdKtszz/HTjZMAOnzi0th1ZwbXBkC/ne0QB/eSj7oXooFLjvRYuuuahiITFoq5TIL5HCaZwhhl9bsnFgDif4MrfT5M0WxCn8ZkaymdowdlP9uw6SLMj2+fpjevLbxfvXO+Y62//4yMcVeunb7z6CU/rEwpGjRLWsvKHtF992HXn5u+xQxkonnxE2MTEkas4YY2rnlj9jW7LGxyqR422W7H4jUdKt+zQ+O7rADy+310oN+wxJXpADwv7T7QcQvTzUmRqqFwax0LxWwmQRzntOhaUR8v0Y9YRETUrbN5mk3o07hsLQOpRM/2JKIR2XrzmmJWI6RQXu//KpzWX3OIbrx4PdH7dc9La8mNfYuInt4oGchY86c7eoguupXjW7wNs68FtrBZ9y6peth0djsBsZlDBENA/dRIWSisJQsl3ExC6PWvuJaIcjCPiIjeHE9EP+ADIqIunZRnTOmZG+50zLw6c4iIeg+mfVhPRHTHzUTJ9xI9EXWASicQETUbQ0QP9lKenzJJ2UBK1j1H1E8Wo4loTGMfURqOy2sZc6mPvI/QTnxGRO//RTQ/Gf0aEf2USUSYQSS1+aHO9AyvO5Pf2jjCNiwmdYRRKRF9iw0iXrOhEtszlXjsOpLoENZowhQjTIlUhKK05KGEm0kIvCRH1wAAH18CIB6FAFCoTJX8bFtqQriHc6FPY7K1n3y4d+W5opheS5B2IwAtnbbwPuV24x07GvH+AVGtXEAUiuS1jNnzPf7dB+nY9Nprr2V1Fc1/9rYBcNXFgIJviTY9Js+4rPGUFmC3dr5H3q1n2ES8ZkOFHvXbVxbqp0QqhdJG83J0N5dJMJ/0NeiaQlm1QAEAFPQATNNs0f6ytSFPrK59a7sXp6bN0tNp3KxW5/srn3pX4rQMlG2X93znhnWva/AtpXktlOqAONamSLWvFdliYZNLeNjqyHY7DrGZRwSDRf3USKVQamuCNJlJUFdcZVOYQlk1vPw0AG/+zaZptq4xe6Ho03q4MgFFttbklkWbr77zj9T6fiQ5N6vdnLB8vtF1fWktGPPFjy2jZXxLbd41Zg8A7PRvs/qbhv1mrfoG7Na+Gp7yzjvvDMK0d2I0djsGsVmLCJaJ+qmRBgzlC3OZBDXCZFOYSlmNX+cD0hL6m6bZErk+TZKt4d5vm7ri7hhxpxbEks1qRUh+YsRu8XARgCKQZi0DE4fcp8G31OYJL87NBQoXMnxLtGH2NastbIGuKcCtZdhYvOYRwaBRPxYpD0VtyUMxm0lQn9YkU5hKWfkeeejwhm4bLaDZhD6Ny9aI3I2yiFK76EEsyazW7W36v/pJ05SHPSNbxt2b+e5Vsf0+I2ktNPFuWbXGm5Nvbve5r044w/Et3obZ1wJb2Cy84so8bLLdTonXPCIYPOrHMmGhcGsdCyXcTELr9e/LU9XLvke/UH8u5uCyVQVmV6uOsX3HKetwgY/Im5mjLjtBRL6Tfk999Mtg1sfXci6XLcnTNfdm5hu1Oest2ZPtE7f2XtPnG9+8ON2ji9dsqHJlXf4bke/E2C660/g9pdpIjUMJN5Mw7X+o0MlyJhVtpg1vVXQGYlmon32ZVMn5YSYVbaYNb6h+qJ/tmVStY5hJRZtpw1tVnUVdBupnXybhmeQruHp0qsjmVbcueLTiMnFIEIcEgUOzwaHZHJrNptWSq/KsxTmGVf1zybxc2IHAOHBRpR9hEmHEGBiDRTDpZTo/9JUhUy2liiohXISyAS5A4Y9kV5nlFRrZFZERJjxOnIExWBRy6bxMz/pmNTuocTiZpIrCB2kiYtzyB7gAqPyRwI9sqJDIrshcrRAeJw4WGSwKfeNaL1PqKCzSOpxMUkXhgzSRMG4ZAFwA5484fmRLJSGCFcxXS5wwEgyMwaIwvkLTGCdaLtASMGapIhMgzQOL7WeNDAAuwR8J/MiWH8cNnuyKEGvEPU4CLDJYFPYY37+rbcfo0mKfu7AWJAJGRxX5AkFFRlSRhMVsbR8v0Bt1Yyg+fvpPjfdGtYsSHQBwfkNhp7YQGI6twJE/wGXMH1ldguwKCu2KzOcw7nESDIzBojBLJYEyZhakzPxaJmB0VFFAqMiAKpKxmIX3Q6A37AmbxybPnrR9dYdvRQeAXfcndpg6yscxHFuBIwOAS+aPOH5kcQmyK0i0KzLvklqPE2NgDBaF8y7JSaCWc0gju9JTRQGhIn+qSMJi1LWoAA3fWBHGEtHq6B1SB7gwSu1ECHBN6O+SBgCXxB9p8CMr3yUF2RUc2mW+gr0eJnmcmCbLYFEYJURO+vqyn7rOA4/8668Aoutfmgi0zesNtHWfRea8AQASek0CEu6ZR/C5e/J1xa2Zsdt3bV9pLYoTSmysLroBuO3iZ6QO6IVR0lpsuHyqcY+pWW5s3cJfoWXpkXPKyLpAg0sAKb+OyUuBc5sOwtO4uezYiuw338LjxDVZBovCuSgmRE66Wviqcrtxpq9RGVDRUl8Uxrz3/Q0KVaSsq8fkGc8ljpgurwVtDDbm6pxGYpleGCWtxfrSuseULM+vet1AoWVlCbJLzu/eyXOjYnotedYA7YrYCBMeJ87AGCwKp/QiJzCHkwFVFAgq8qOKBBbD14LaRhsrrS0t0wijCrPb2QocGQBcgj967QYJP7K2pPeaINCuyH1rxD1OgoExWARzIiet7Ko8qkiGivRUkcBilt4daGM+AN6fr3PxZUIYpXbCXuDIH+AS/FEdGT+ysgTZFRzaFbkRxjxOEgNjsCjkswx3kSCByO2GJLvyp4rKgIr0VBHHYthaVIBGwo6wnoCFObPFMiGMUjthL3BkAHABUPgjDX5k6eV8TnYFhXZF8FySEUYSA2OwKMTTHsXLpJJA2wY1aDt8P3HZlR9VVCZUpKOKOBbDQBoG0AjsCA9MWDb5ht0yisSFUWonQoBrwiFB/AEuwR/J+JGl55KC7KIg0C6rfvgqiDJAYMqiYkJkjVQSSCZggqKKBFSko4o4FuO/FrYxLPBl5emWFWVkExV5WSdCgGvCYo0MAC4j/Mjia/qC7Cof7bKiKuv8MGsQmDLW4lpwvzM/DDX4186tQWACr+UADjo/el6jRxj7iR2b1pL28riT051XH5Vl9g6qHFVU3lp693Zeejjz9B3WyPkc5pRTwVwPe95JwSlb6nn7rlZMm1azopw2LQL7XDVDrQafw+ynIav15zC744tytGI1qmxCUkMdYTlpAGRoz/f9e8uP6xYxStJWtA9VmIbUFlfZ8XgB1c2m4SVVhpInbq6CQVIjPsJyVs7s9jYACdrLfdIzqEnX9+RFgpK0Fe1DlaUhdW9FXGUn4gVzs0m8JGMoReLmKhgkFZG+4uppMJbNwmPQ3qs7X6r3twFjb2shFkmUpL1oH6omDamrxVxlJ8XL2EjBS3KGUkrcXAWDpEZ6hDUVh81rZyq38TsL6iEOx1rwRb7Fpe8BQ2at6iieVTEXW1L+E3Kb1ZH/EZv5vQFclZ86UI4Xb4xIA4DGbwA4O34qsDbtU+CmD7vJiVv6KSxOt//hxGfHJ/1Jp5oCWy66XDqtMW9kC6VKju74HXRyz04UH9520rdnrw+Sqc19aGv+ufRSAJS5cpcX2iepy3A+dWUmALjnf+QuLIzs5fGCTS0VlR2M3GyCl1QZyhiDxM39H2au3OVFaaHPXejW7D8z1X35VQ6kHLPXbCoCANqZ8p+zh8St5SOMQ3tRwLJfPqsjFsmUpF1on1R7JnX5BJ7JPUZpQEdAoSG5TawsGpLBkBopWaRKqOyM3Gycl5QYSpG4BVUOkmqzqc5o0ljP24i0zrAlDzZf5tNrxBRK0ja0T67S6FlENLS7BnTkNCSziZVFQ3IYUkYyI2Uw1ars1Hi5m43zkjJDyRI3G2oQSKrlprrye61GoHGG5XYdVKpbpMrD5EV2jTBqOouIRnUnIiwgIl/7fhpT2wwikiVs/El82W+LiajruIoZYVqVnRKv5GZjRjatBE9N3PwIE7noR5hNprrgf7dChvYSJg6+bpxmEaMkbUL79ACr9s/OaeSSaMg2KI+G1MOQES2msrtQfo98HXocVctQqolbcFGsbCRVlbJBb6pb6mPAqClwtIwRRswZRq906w10xNpxJGnEVEpSXmT7iSNk0BESDVkbKIeG1MCQKpIZsdKo7KBzs8VwXpIzlCJxK7ZeDpJqs6mujE/63Bl2ePLLAPLRBJJGjFGSZBfap60YAnACMuiIUGhIAUNKSGbESlbZQe9mg2BPGUMpEreiykFSbTbVGY4wjxeQnGGtkt8F8D/XCInj45SkbWif7iiQDeRsPQsI0FFmKt0oh4YUMKRAMiNYQmXH4+VuNghekjGUInHTh/3ykVS7TXV+n8yOjR6Q0PjOMQWSM2zX6FW/Lmm1kMQiiZK0De3T1Por/rV08ljcliWBjhJTqdjEyqIhBQzJkczIfdIXKjspXu5mk3hJxlCKxM2FGgySar2pLniazbc1s8MV0QC8P2U1T07QLDJ4lq0TTUoONGyaXdwozuVacN/BRor9+bGb+ut7fKDJBQqrxp/ElrmzGjaHu04UgPzstrUiPXvn0E91booN+OXqtlvVU5kjmxKujtckbkmoLBel1P0X8fkONI3XPr1RIoCCBt79CRe6+G0NmR8mQMeI0ZDVc36Y7Va2Kjo/TAIdHRoSVcBUV9WOYWnLYzwXPl+OVkx+knMMQ8RNdVXYzaYFHR0aElXAVOfwkg4v6bjZ4LjZHDebcwxzjmHVjDWiCLRwqgaPsNAZLAd6q0wjzI9mw68LVuVDg2BJzr9f5n96wv6easCs0BmsMKkt1ZT3389N958xbBKnZmD/U7Os1vY/A5qtZNScm1revktGsATNRtP239n56SUW9smYNpPBrNAZrHCpLdWUN3qox9zw4qEKTs3A/sezrNb2PwOa7cVNO6KSRo3c4gIYgiVotiVtByDxo079EyzrUwDaTAKzQmewwqe2kgBgfp65C4ciVM6pGdn/RJa9qo39Lxiazfv6yCjghrvW3wyoCJZEs313NYDabfYnW9ancmmz0Bkss9RWT5O7JELlnBon1ySaTWRZoYhgxK/pZ52LB9A0atPNHMGSaLakp5oMwKkjIShBSiTBnGreKzl5umErOnW6tDNz7xUzXx9z8knFGKx01zUNhdiPyQCZtE92921tH6/dLntyib/sTtoeM+XlnjpzpUXcz6QJUQqnZmT/Cz3LKm7/Uzm1xvAC8OKwQLAkmm1c4h0D/vfIkrrBb1MSzDHWTOBqjDbjiBrH0EQFxWBpEKyF92u3y5/sL7sT2xOmvLcGXnPWwsSX/fJZHUP7n8iymtv/9DRblyFEtB29dQiW6vw70wWY7Q1lspwQzHHOSuBqjIVhiJrA0JhRNzgGS0awVAWg2K4wB/rJ7sT2JFPev3HK7O/pM4BL5dSM7X8sy2pk/wuKZtt04R/kfTH2dh2CpdBs5+9/b3USBheHEkazMUT0YC+ZPxO4GqOtFERNwtDYCAuOwZIRLBV649sVTxbLWPHt5dWZQ0TUezARfWfhCFM4NS25ptBsUpY2IYLSPkkZJN9L9ETUASqdQAaB2GlhZpza1d/OblZw15T2WgRLpdkeuP5B9H56Xo/HQjlyqoI5ibMymEGpXF/wx9CCY7DaSAgWVwCq25XNgVx2N+MwgNHJfHuSKc/iSpg4+LoRBvY/KcsaYf+TOLWObwK70UmDYKk026l1i4C492NSHwsDf9RzVqShrWobYGgImsF6VSBYQgHo8jcH8mVxCQDqyNuzQVUmODUD+x/P8tFrqo/9r1YQNBt27B0CbG49EMOHA1jy5bSLga1fLIxG9vLLiupFA8DfcxEeZ5UMID1ZxtUU2oqRLeeffK4PcA5FX/1DXaJlsDqDM1hXQGGw4m/t16/PiGuUG/hBb5onSwvH67Z3Y8zevqopz7o6PPnG3gqnZmD/41mSjfa/vsL+d4Ww/3UcdNnYCNr/dDQbXhtDKHn55ToSgsVptlbNvgeA9feEckLDBHMSfyZwNZW2UhE1gaGRW/k9mCAZLIFgcQUg365kvtPL7sT2JFOeJVcJvAAgODUD+x/Psnrb/wxotq+HZ2zsu0LW00k02+/9/pmxefIHIZz2SII5wZ8JXE2hrTiixjC0lYNi208MgcESCBZrIW2XP9lfdiewN8mUZ/aTvghVcGoG9j+eZTWy/wVHsx3+oXHPgCZv2v5ro+4Nw53KxDkrjqvpaTMJQwOCZ7AEguUPvYknG5S0PV9WfLMDtRo2cH33t1ONLZofJjg1Rq5JNBvP0j5EkO+TnMHJpgCdbgJnBmI4DJZpamvLvmEr7i2IdmYg1oz5YaEzWKaprTlP0w9Do+FUzTiGhc5gmaa2Dnzi8Uyp58yirhk0WxgMlmlq6y9TnSORQ4I4JIhDs8Gh2RyazTmGOcewGnEuSRXY2oHeqPqPMHPcmUlqrcZDb5YHYDDCZC2YylsZgVcccJO4trLrzFz56/GfejafB+tlax/1RepVF66ufKY3HiX0PKCMrnEyUHK5IZKgoOUB+I8wSQvGeSt/8EoAboJrK6+WPiw/q/u/c0qN+DVzsrU/3G3QZ9WJcCdE2KYq4wkCfjygQNc4zSZcbpZWACudBAraEIDfN5WTbzhPNBZ/EPn6TyFahg3iDt39+IxXX331rmn03OVeok+TfUSPnCSiyTEZ5X9Je2ZennamK7dMfC05FF6eH/KXq1JzpTVSwvyaNoyNB/c7rjxBIjrfG5lybrm9Rg2ZeZSIaNESIiq5JJcWdiOizS3OWvzN99eLyzCHmAoglFnUsxqeIJqIdKKv6hUQnfnQLe7QY0RE+cM9ngaPEtFxpJI3YTQR7cT00MMQI+wBsee+K/ND3gnRXG0d7ggLZ+NBjTCRIBHNXIpMObfcKexpw+cREd2yla6bRES+BikWj7AHyhthYQcQuKLKELFxUxi/w8ErDrgFY2lTnV+eozuOAqBDPxXQN0tKAYD2binQ2dKMZGuGrjVD2ZrKuQEgt9fjKTmy/QRlbjoLnPrxqLJc0Y4Vl/qKPfCUeIvF+qTmx778KkfqBXOVha8qEwka8ICikp5aDZw6cpmxy638Kjmy/QRlbSmQghJaOx4U3zGWIPScoJUBRCEQbiVMYQbglQDcZEtbgFJ5s0Pju6wA8vt9dKDfsMSV6QA8L+0+0HEL/GxhetmakWvNWLa28H620fMda/3954xxV66dvvPoJT+sTCkaNAuC4FrZI7rvPuz6c9O3+PpEc5WT471gjcJXlUkJGvCAAl1jNJuhyw0hcYIsKH9OkMcbiBO0OACD45qqBeO8lRF4JQFugmsLdEBXeTNf4hyi0X2I0mod9RJRs64qNuVnC9PL1gxca4ayNZVaI6RQXu//EpEn7q85RDdevJ7o/brnJYIrN/YtInp6o1ifaM44OdYL1qgMVVk575Jygv48oISuqTSb1uUWwrukhKKxHfPnBNmOiQTZu2T4AYRIsylaMM5bGYFXAnCTLG0BV6uCaS3nEHUdSXQIazTYlN4W5i9b83etGcrWGLWGlKx7jiif9UYT0ZjGPqI0HJcJrjGX+sj7iMzTseaCk1N6wRuVoSorZ4RJCRrwgAJdYzSb1uUWyucwnqnYMX9OUN0xkSAbYeEHECrNljBx8HXjuClsmAF4xQE3QFjaApc4Be65A9gf0wWAITZVlmxN41ozlK1xam3jTJ/6OyqtAES1cgFRKJIJrjHvfX/Dv/toN6E0lzi5NoCEfXUPW1UmpGtGPKBA1xjN1tvf5YbQOEFpx1yBXo8AnKDFAdQKjFtxU1h9A/CKA27glrayNiPmAD5026SO85cm+WFTsi0skGxNdq0ZydYEtVbn+yufeldsw+VPcF3e850b1r2u2YTaXOLkamsaha8qEwka8ICCGuQ02z3+LrdQNYm67LScoLJjAThBiwOI8setuBaM81YG4NWOZVAAN2FpC67WfT22c+qd+lS0tjD4S8X0rjUYydZE65sTls83uK4vtGPAmC9+bBmt8ZapzQO6yr4IX1XGE5SUbP52O0azNTZwuSFsIZueE+TnQiJBDSdocQB+I0zSgnHeygC8EoAb59oClwqmkbsEqD0tI2fnES02pbeFGcjW9K41GMjWOLVWhCIkPzFiN99GEYAiaCg2DEwccp9mE6w55+TUXvBGJlRlcoJ6HlCga4IM1LjcEAYnKO2YHyeo7piUoAIK2hSA3yczSQvGeSt/8IoDbhLXFuhDqcqbbRsUe8mrlB3XqFkDdN+twaZ0tjB/2Zq/a81AtsaotW/7x3Z7m/6vftK0rf+IvXi8Z2TLuHsz370qtt9nMsFFE+8mjc+NNWecHO8Fa1SGqqzc362QEvTjAQW6xmk24XIL6ZO+nCnfMT0nyOPlCSoGN1MBhHQu6flh0Xr11PD35an52jt09AvFb35oaWphqKeoRJR1+W9EvhNju+h+sWNPKRHRo1/68X2ZeUREWODLytMuoqKMbKIir2ju39oIGMzMUe+dy9WuT2ruzcw3aHTWW7In2yduQ/plFJGgQZ82L05XLrP40hevPUNERAeXrSogc7wk27HiX0/QHwfyfSJntaQEzQYQJi9pw2S5j79aBQDH2hRGhcKdBeNaM0mtmWpeHWYg2uRqi/T8sP4H1xJQOPXxqFC4s6BcayaptQipylDTXG0Rn0Wdv/BQLM7e2jsU7iw415pJas1U8+pwDLPJ1RZxmu2CRxEydxaca80ktRYhVRlqmqvNIUEcEgQOzQaHZnNoNucY5hzDqvIvoziEWiXG1ariCMvLdQg1VBlcLbwR9mGO7p5qFjOi2YR2DOalawDOD31lyNSqQagFXbJ+jdNsjBqUaDaJDDQXql2SujAsdYZX+n9wZcr3zqyY8Rfl1+CnqeNyT/FDDx7ZfP1Okh8M9wuO83XnyH8+MZjuGUrHdvIFR68noiNh80NHrw+5ibRxa0gQnpsINWf8+tNpzecSUS6ik5rXTyGi06NXbJ3W5DMrQrUwDbW6DyYiuqReqQW/p6/o18Q9YRbjGrHnua5NcrmFWZJ0DQBSR2GRRtC2ZBjM6MTC0LIFsMOFX0K/xkPlkjbhwBPGO/Oh2iSpC91SZ/j0N0akae4JsxjTiEm6tqZWvwflxQGyoM2kVy2c5qutvjAg9Gs8VC5p4w48yXjX1K43drOSutAtdUafwxTcSntPLX+azbp9V6iy0kKfu9BtklCTEbWt7eMhwCz1gTL5OA1dZ00J/RoPVVCD4nJE+WRgcCUhbJo9ZZI6GXpT09Mug4wLAvDsylDyyN23qcT0CFNxK809Vv40m1XFqLKMmQUpM782R6hBRtQW3g/Bx7EHyuTjNHSdNSX0ayJURg0Kmi0IMjC4khA2eU8NJHU8veAkdeFY6vw/mgncSgKvhPfJgGbrafJDacs5EqWmcC+mCDUZUVObq2AWf6BMPk5inyz6pC/p10SoKjWoE7ExMtBUqBLCJu2pgaROpBecpK5cS10QzPfG1i387sk1r0MUgLnfZ8P3Taxlnstdq28B0MvgVPjLfuxnoA888q+/Aoiuf2ki0DavN9DWfRaZ8wYASOg1CQn3zCP43D01K1Obx+Hy2rfPFA/URTcAt138jPTksQvqo367dJs+ASm5yaHe/cGe2UM8QPzWVnCN/nijsvS5OmvN/+51rUbKTkPeU5FGdP3OFwCtj0NKTyxTi8eRP2VkXaDBJTqmJ9xP+gK30oBXogxoNgtKB67BDKGmWRlv3sZgK+XzcdaVkps21ISJg68bB42IrXwyEKEgR/KeuvwldRpLnT2SOr8RJnArGbyS3uL9aTYrSg+uAeETavLKRPPaRlsJzMcp1Jd1pebGQ33tTUYNkixiC4IMDO/7odLaxpI6yVJnj6TOb4QJ/ZokYpOK0WxM12ZRSZY2aARtGkKt540DgDI8a2Pu+7FltGZlfngc34rExynLhJXtDo0dzppSc+OhHuKSNlnExox3pjcnEDaxpwaSOkNLnbWSOuNvjdxw6+552Pr9aDb5wTD/1dxFEnxFbrcEXoVDqMkrE83dWjyuLD6O03UWlpobD1VQg5KITSYDzYUqEDaxpwaSOik92yR1Bp/+FdxKuifMYv40m+bBsM4lFZaKwVfbBjVoO3y/OUJNkFxS88tGnpbZtTL5OEbXWXguyfRrPFRBDXKaTRBupkOVEDaxpwaSOp5ekJK6ci11ZPq7LpM0WxDwlQZwC5NQYysLiMeVw8dpqS8rRhjLzYgaFDSbda+VQNjEnhqHKdLTX18RsJt333HKOlzgI/pPqCOsks9AtI9QC4aPqx4zEPmemsfVgrDUVbUZiLYRakHxcdWixJ6ax9XCsdRV8mOYXYRacHxcdTiGSXtqHlcLwlJX1UbY7AkV2byazdOPWBoOCeKQIHBoNjg0m0OzVcJ/N3I5xzB7ogm/eRTgYDRwCCP7mpfFGkkMDBc6CaOTYI1slDwxwgVVBSoKWCJBP4DLwKGlY5NggY7MDK1lKtkyWCPOwPieX5Wz+57FROQb+9Thn//6PyLBGvFFlmMxEuFSaaCicL1GPEEDgIsTRiLxaRo2qaxQ31J/hF6twjpvG+3b0evN0FomAKVyWCPGwCxpOwCJH3Xqn4DFW7eg1ew79sa9yFkjvsie//2kqgUVBeoaT9AA4OKEkUhcwyaVWcPqDpb/bJBouG9LhpmhtUwASgGOYTOXIpMkoxMXOnGjkzBn2SV5YnXnYDu1Vw8sjswxTCQo4mUTpY0cWsxQZqmOLOxjmCmfVtmskU7oBG50EqxRuJKnYIoTLhyI8fiKvUVeTzGKPZ5SHR5TKaCiQEdilqBBvEaEkcwmwRodGcKmtVjzY19+lSMlzHRazKMVQKdVJmvEGBgmdAI3OgnWKFzJU/klES4MiEnpHD0o+9mGTRdhfnz7NB0eUymgogDFEzQAuIwcWhKbBIt0ZOHTWkpzVafFE2aNmEcroE6rDNZIMDCq0EkyOnHWKGzJU7klES4ciMlAKtGzPYloRDZp8ZjIQEXhzt5hCQYAuPwdWpxNskxHFi6tpTZnOi22Yd5I9WgF0mmVxRpxBqboyYdWJ00YWgJSZrx6SLBGYpHFJREuAojpmLwUOLfpIDyNm0OLx1QaqMjw3IklGADgUgkjmTpS2aSApewoXPUBbEkC2ni2RwFAdy0xpMG1wqK1lOan3hgeDSx/g21YNIpbM2O379q+6g1CYI0EA8OETo/Fc6MTZ43iw5c8lV0S4SIBMfdOnhsV02vJs2k3AtC4tyoNVGRUPMEAAJefQ4uxSbBSRxYeraU0l3RaaKN5TVSPVpsAOq0yWKM6jIHhQqfHWgijE2ONWoQveSqvpM+5HIgZ8sTq2re2e3Fq2iwdMlNpoCKjEgkaA1x+Di3BdMFCHVlYtJbaXNJpqRtmjVSP1qsBdFplsEY+xsAwoVMuhNGJs0YWSJ6Mq6sgXCQgpsktizoOumxsan39aValgYoM3yR5goYAFyOMZOpIZZMQpI6sfl5qXSMdWTs5mrBoLbU502l19mu0Ov7Wfv36jLhGuQmFNeIMjBA6CaOTYI3ClTyVVxLhIuNE937b1BV3x4g7tchMJYKKjEpK0ADgMnBoadkkq3Rk4dBarDnXaakbFo1Uj1ZAnVYZrBFnYLjQSRidBGsUpuQpCL5DEC4STuRulEWU2kWPzEQKKgrzXFIk6A9wGTq0JDbJOh1ZOLQWT1bVafGEeSPVoxVIp1XmUOAMjBA6caOTxBpZIHkKsH1OuEhAzAki8p3UPzVSUFG4VyukBINK3IBNsk5HFgqtJTU31mkxj1YgnVY1mR8WMaioUs4PC1NHFhmfVlR1tz7VBKgoPB1ZhHxa1eQYFjGoqHLOcQ1HRxYpn1YtVG/rU3DSrape4ejIIuXTclgjhzWCY51xqgpX9DR71nt9Dcvx+kjs8/UOzeaUU867pFPOCHOqOtX/A0ruxmKOAkbVAAAAAElFTkSuQmCC',
    '1709.08718v1.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAVsAAACtCAAAAAA7GIOBAAAeiklEQVR42u2de3xMV9fHf5O7XElCqIiEirqFCEFdQsRdKUFoaCmtW11K61Gq2npalD7qpfoI6lGXUlqltIhrhSBCkEgqlTSEICH320xm1vvHmTlnn5kzaSaZSNLPWf9Ys862Z5+VmX32Ofs7v6UgyFZNZiGnQM5tHTQr/UBBnM5r7GWjf/Ba0mg7nZ/4V0HBJKs6cprKa/GaF4Mso1o3fI5vqtCfb/+cgTMa566A8mb+wOXdRcfiO2rCd+lerNj8AMV2dSO1MW84DLGJSX11ZWRfAE/QyLCJZLCKRobmgJ5ERMow2JwTHTgEBAqvTgPFVCcsq+GQMiKKc8UZIqK5yyXaSAarZuXMt9aroFwuigyd3Xe98Mqpzkx82zNHWQLouJV7GSvVJvb5Xss8gGTx5LzxTPe6eFFJwAUAwHB3ALh8QaKJZNDc1zL2/YABQOHt/PxeThcadFQgLq/APRAA6GGi2l/X7OkfWd4vWfz6LKweUJqU4NS2ZW3LrQ92FC3sbA3rzW1QFDkN6ZfwgheA0it3Lbz8XMEE1TcKCh364ElqQaF/M+DpxccNetwYYvb5NrENBmQTXWsBHA8ejcAy8rNCOBHRlR6N31seNBIoJlLOsZ76eecmQXPwBWn2Nuq8YYXrxJxaNt/GKgDY9vjgDhF91dMWzXr23EREST4emzYF2S/RMME8Xwv0J9rqAewn+tZv4+GINlBV7m2lc+scFBT0EhpvKCMi0rRA77OHgftEYQgnojgbx3QizSygmGgFQokewy514XXaiQ6FREfxRm27mK2z5L6kEURELaG9bK0BCknpg81skIahPxE9BPbTY8vviCjFvLntSUT09H34XSQiCkAQlXz2LRFNRzgRdcFEIqL9QDFRNywlIlfsJip2xWoi0jS1yq1tyY193RMAFJfZNN595SMiCsdwUW7D0Z+IioH9lIj2+zKJfqjkm5Yz37quPnpzyLUWANATtkv4eH4s2gitvC5nA6o8NAFuP4PqKgCPB3f9a9mM23kH3T1/4Fda+wMTbHG4LPFmxh/INva/fP2vh8FnyOJqWCcoxiF3HwDAjQ3/RWBuGT52/PWuanvZq32BDOD2yZMnT45d7V27MrsuDlC8OOXoLNxgw5kzXUNTO/mUk5xTC9yRuqlHhvnXCbCF9m8q+gu0tNAUCa8cXu7+NjXbNUEBdACCp9bGNdgFdAIATNnkoo0Uz9qOwl53pm6ywV6+WfGs7bxfBgBFd79cm3J665Xv/mX2z63mF6C/Ydi+P24BgBoAcET58anTO8ItADTrjnMAQDPTally/8t9RIrRB4AdVCg6DkTdwfs2wAOACQJ2IAB/AMCtkCJFy7fOOjwx95xAGW9fxJxBEkc21//5GnBvGZBC8DzbNmTIqAlrbgGKXW67DgGqT2971rLc3gmMVAFPl7h/ACAQCbjYBfAEooG4NDzOEoJAV2QBxV/WQ5qGcueogPvF5lrfxtnaArC1tbWyavVKJBFFOdg7OdqfI6Jiewcn+yZE9DC8+YiRQxYBWEXqTtqO1hBR1mwv/+DOiwpq2SphzKQ9/vX9OzkNukdElDVE0blDEhEd72o16JXJj4Y44j9CkFRznF4e0/93D6DX5UYzA14d0OlgJd9WUcl9h6fO1oW59ezsCgc8XNFEUZZ59RvNM2cAKCxzqXXz7YMmFsh5oGmte2ZaWNhQAQAoyna3hX4QZc+sG+Chdb16CqWdKtPWzWzPGE20E4M++BwAMHH39U7y43Cz5vapv+3J5gASQlxuWcvpNGtucX/tj75e6pT7Yz9wl7Np5twCKMlQNLKXU1ktuZVN3ueVc/sPMiJaLmfB7La8CvcOsslzAmoTV5P30NqaylRNjNy5KndkuQ1R8C8bONSBcyxM5f6187YCgNxbWc7BUu0yE7NekN7HTknN6uZd9dw+2nnuguNon/F6uU2ZPHMCgJI+7/Tq07xBXINwB5SmX7m3br5BE5NsUkBX+3vRA4MBzZZ0u6L5DcF60kHT7x33XTjjOcUeWadeWOkHZOz+bohkblO/2z5HOreXdv960Nsc+7wxGGUY/BlvEhFtbK9JnZhyGSOJiKh48jzDJiaZBwDrZRoizYRPiWJ9H7KedLAylozJRETFIfViiIgGhhpp5zfPyIFnOGiWvchbkHhrzdUCIqLXw0QtsiYYNjHJBnz9wZZUIqKfXUqJ6O0xrCcdrIzdx1QiIopCfyKiYcZyG2Ast/mVyW1FQURFADd1OQohFdm41TdsYpI1mqV19vS0ARD0ZpG94EkHq2LNuP2E2rJOUMYkqAGosww25M6fAAYBUD16ipwCvgkln3kMAMiJPqdM4yLRxQBQlvkYZanFUu+ijnQBgCalZwVPOlglOwN2C0E3LnbQBicOoPQBVUduKWJUUUqvC0js2nAOcHXilVMTJxbqcgtgJHDUr8mWrzd4neOa4GzX05o9n6iBL3e41PssAMDxEddL3l+qQpR/o+W/fXElbAoBKOn9DgDgxro163KBzGxHAHDBHcGTDlbFYhcP+lJ4pRsXO2gAHzX2GpjJnzhwd+zyc59HmolZYufbr5vmEh2vn00UFEpEFDqSa9EsfFx3/KKdZ31HR2U03ss1OWsZS3TQPpKSgohI05PoUKtnRJqZrxHRy8H/I0rGKSJ6ZN2OiKj9Hg395PsX3cF8IqI4fCx40sFKzrdd16xZGebxjYb4+ZYZFz/ogHlEcW1Psyee6HSOiFZVZr4t/3ObuyTUGQix2AM4ig4E7vp+30DdPOt0t2fjjDDAEaBZIZ0Bx1aeeBwXBygmo3h2WANAMWPPKcDp1iTAxzoegEfqRQDYPUGBUY4LUAIbAFCgWPCkg5U0v/feW7z3yu6QVF2AGZcwaAAJEZf7sSc+xb8PgDBz8wkJra7l2kQBcJf4Klp4zSVAneAHoAMfTb89GEBIHBDo7d8u5JVpiEtvDwDtFUf6A20tAEubQgBoyp0yAHSJyHTiduTL4CR4kAxWxbwONR98VdsFMy5h0MDJSQmOAPgTf3Rp+d9iHJWZb7dbpaGRnZ2d3W5J9sG/HZB0BQBc+difaKz17M59ZP91yOuaPzgIx8ImEdDyOGphTf47ADghoT73iSyBi+BBMlglcx1454jWZcYlDBqn4u3nAwB/4klVeMvy/iAPLdrArYvRwy8AuNSW+67qrBXua73kep98kr3j/aktUQAAqtIXJboYkZtvAyjh5tL4MQA8QqDgQTJYNXOEbjnAjEsYNAbN79Fz6HiAP3EVCqtjDXbnHjq+EAkApYeNXQj/20YcaNr+OgCUHkTCXqDB/ND4zg2jASAGg8Utn5UA6PiNDYBElzaKYakAkOIaIHiQDFbJSs4qdHe7zLiEQcMR3ZbPSINw4s1euswROebJbS6UAJA2vinsthy5CWB9M0BVCADKPKYFihc8aMAHAVUhFFuifwXwX09gUxEAZXeHzT+kAOpV44YDShUAUqkBZDXrBWBMGwB/nVtnhQW3UwD6aYEl40kHK2PaAZfMvPepHwCVCmDGJQxaqQLedwrNFk5cEREZD9CmSmXX4GnCCC/Yvxo+rqcl3iWi0z2W7Hj/KF0Z4eo67EnsCNf6IWu0LcLHdquPYKJjQ50aDv4faZtQdOelP3x6gOjg4KV7ji5eT0Qn+n2+ecRyJV14pX6DEZf+18/Re2w2FbafSERly76K29vuSw0R7Q1KeLZwolLkSQdNttuv+sB5dHj4sE6DThLRlRGubiNihXERaQd9bJhzs1FPfnFxfGkTf+JE5/us279wHxqOqQau5kGBr8Kkv9ajHF8LIK+e1R1lK+7i9TDH19i8fvuSaw8PAMCTXwt7+CvEnnTQbMaMixu09Ik/yGqrue3uXg/yPq+87yDnVjY5t3Ju5dzKJudWzq2cW9nwvH6HXnjsarbP2BbYPxaV4yv4G3ljnMXztLyHNlYalWULqJNtrFT0ojTuYXyoRpkQHgl5sH9+BXNL25YNnv1i8oYWHp+PrSRfoTNpzoIhPnbcL9a89SIgcvFk7mYz/iLl0c5zFxwnBrSAauuxhOAh70ES9zCKhBhnQrRISJSvMp5O9LavwH6ZZpbzJSIi2mnVsfJ8hc44zkKzXpr4mHOT6GG/aGLdzIVvd0WGWX8EFYNhWs6gm8Y47mEUCTHKhHBIyE99drw96a28ivAJ6zft6AYAmPjznyZ8OuxgCQB2H/dafJIJcz8wybzNNvV7Nb1FiDeAoz4dgCZfLYxkXac5nhtizDsr2Ol+gezYUv9xj7Wka7yRRHxUt1mXvpxgUYE5IXOZ90StO2cezMRXHIQk8XG5CABapohc2+bVOPcqqqHP1afX7HjWObLh368T9hQM1wV7OENAO8DgEFpHEubQ8hUCOgHg1hLpUTX9z1ca4JehYrfaTW/chWkaVAUJmfab3ZM5Z9wqsAY7Kagj2HwroB0CB8I7IpgDIr5CQCcAYN8HJacmTtxpQHxgss+7ff84vOszsVvdJh536cqI399YqsTfICEcEyKJhLhZeISjgUUF2A9/iHUueLSDxyEEIISBOcR8hdCE49paTycyJD6I0l6GTW9uN4FxaZ2Zr2UCzhJOJIZQAl54QKQeO0JTDhIiMCEmISGGue2KPXpwH7dayHGZS0Rq1695h2hQQzVRmfV6hhVM6xWcQkwTUW4pPZeI6AYRUefRRJQ86T07DMwgsVvtuWXGHTCBW0gc4IZa5PkhEdENnCRN20FEFNkxkShgHsXPymdyQN37EBGllptbw2uZdwwnF/DN79ZWauWH7XRoB49DsEAIA3OA4SuMMiNi4qPhremH3WdMPzEy2gKM+xyMGbcNAARYHw0FYBQJ0TIhpiEhhkcH778GAJg56VqQe6IbdGhHGhrZAdjd9LjOkYA5ALgO/OlIKdNEYsmt7APACQl93/rCHS0jP/342FAwrvnNidPxAMrUukWZeNwKp+TykBCc8raf/x2TgwohIYa5DVt6tMQOABwD0dRdWLfwOES5QAjHV/SQarJtqj7xkZfQG4Bi+ak7Qxm3GnLr5ZbJOX8aUc2gvNYoBwnRMSGmISGG30CHr5+u4bwiUZzHIf4GCCk5qwg2aGKnAlKgT3zUU+Rzp94ejFsdq9pRsRxDcWCUwTElAESXDS0PCdExIaYhIRKz2+gNn0ZoAOAH7pflWrSDxyEYIISBOUR8BcuMqACg058AGRAf1mFfAcCDZ33AuABKUWre5K71mAcAsekvc6fEjDvmMaBZO3pUOUgIz4SYiIRIXeDOdum4JerY3GVp4QLaQQwOoXVYmEOPr2CYEbcRsUR3m3/8yVkyID5K3nj77I2IyWnEuqWvDWjs1HbkOrOuFNKHj/rt+OJZBUQkhlBC0hZF7ApdVmocCSGWCTEFCTHCJyRdzWvTqQGMciAVAELETVS3vF2liI/kmCK/rgp9txqMkmNV3VpLHkoraG1VHUiIzH7I+w5ybmWTcyvnVs6tbHJu5dzKuZUNlWE/ClMBJ3438K8CwKcC2h6sMIZRkYzshKz6fU0fosbin6L78Wxf9Hn66wXuxZPAzOCX365AbllhDKMiGQ927Ripl9v89fcyOszzAEDb4h0tFzkAQHZEXrHqHd396dlNP1Qxt7VI9yM1DJ9r3U2TkFyxZyGsMIZxkYzgMPHrzPF3KXeMSzSRcmi4mvb5ZRJR5juZRF/YRHJNClsOq/qTmprR/ZD4vtm1D/yWe8pABc6oYKUna1SGofhoZQs4b7d8TY1Vkd9YYJzTMgC7tp4BpuhK+ayDWdgPLZdSvLjcAZp+wORr2bQ/fwcAXOxZzbP9kZezAceg1CRE+DkB6L6rCGhMABzwjPs2+phTnrQW6H6E2W/jctvDkIfg1Tzunb6rQyZYYQzB5ykJdWoeSn/Wwgqav5jHyc1KlQAc8SQn3REAGhbEAePzxwIXMBwASn6cYM6zrQW6H87jDuQCyHNSGPAQOjUP5cJIp4P9kwGxMIbgC5TEhuGR77z9hddINQBErr4cNoV0FMj5DA8AN63a17MkbjRJ3BdQ+WWXDwHg/+aa84Fuzet+ZKyg8/iGiLY/ptm4L+IheDWPrV6JRAs6qUlEQTA+T0nst82hMt+FdJeIBgXu1TIXWgqEiOgKFhB1bEdENAtfEhFdWtR1egER0bVtRN3McC2rGd0P6dxqfLsQ0RfE5ZbhIc65XCeiLUQHGlwlOoI7JKIgBF+gJMZ0IKIwT33mIl1Xc6ekQ0gx0SFFBlFpD6zmWJ4/Br+SSaR8V22u3ApcCpUDeegYDxMhD5M0VRTTrt7ETZ2ch8BDINDbv/38U9OA0KcBqYfOIxePLvXj18mMfy3XJioq6qL7HdR/BsDKRZ+5aOqs7X1Jw5/tgBFr37qfsnIYuCuXwnfXiSFqbJxt3ttGr0NXBucbnlT67XYAQuJeAnAyZIUjO3z29Mx0z/u61VacGAADHkKn5gFNRI9dbj0BsBQE4wuKIbOy4pF/ZpE0cwFse/irAwAs2HjuzLuP0VEbdut69fjtUrecnJwyVU6+uZJbK3Q/PIbvWmFtaSiRoVPzCFoUcb0logCND0NBtBB8ARBp9M76Ro++ftXYKuzGbgvEW7QFmjcH0rw7Qtmr7JIN4Ipkh/QPAdxu9GGLBeZKbk3rfhABmJr95kgYSmTo1Dzyv3qjJfAEOHGPoSAYIkKgP8702fLZNsPUPisBgOiL6y2AQ7Y4PyoXyP19mRVKrt58AuA+OgZt3Lhx40bn9hvNltoa1/1ISCRgcJN8bwC5yBPxEDo1D+t6ZQBuWRdnuDEUBENECPRH8/cj9hyMzNVjLjgKJGni05kzpk/a6o0jh+8DX/lMApyHnvYE7iUE9ebmj8K8que0luh+JA5ydeyygWjVAaLVA92c2o1JYngIXs3jQLNVv664NqvH4jIRBcH4Okoit5NXi2Z2VuMKRMwFR4F05sbgRxQ/9Pdr7wU/JiLKfH3thahuwzhKdEZ3R5f+q6u2Sqi9uh9geQhBzUOdpGprQ3ku+hQE4z8o8FWgOHBLd0AT+0bY8vIfVh3L8e+hvVGIu0H+ftVGgfxzdD9+ijgGANj02y/yvoOZzT8+DQDUxwdDrktibotZ0zbQPvlUn5kKObfmt8zUYk8fC7mejmzyPq+cWzm3ssm5xT+b/TAi7wGg0mgHqp0FqSPshx5GISrzIoF2VNTKLxcjACIAw4LoxEHEqEhdZj9EGIVemRd9tKPCVm65GAEQYVgQQRxEjIrUafZDhFGMuPp/qCohARj0AyOACMCzIE5zNr3GeWJUBP8E9qMZ/gAUAeapsVVuPwIgwrAgts11uyIiVAR1kP1gQA8wGIW2zAvPg7Boh6jSS8pTAKV/len1Ji4XI9FSBIhAkgVhURHUPfaDAT3AYhTaMi98dRcB7dCv9BIz5U3N1m8udNsk6k1cLsawpR4gAmkWhEFFUPfYDwH00McogkIZHoRBOyQqvcyLI9pu/0zUm6hcjGFLfUBEzILwAhY8KlIH2Q8G9NDHKIaFksCDMGgHQ1booq8SUTRixb1pNUW06iD6LfUBETELIoiDaFGRush+CKAHpDAKngdh0A6GrNBF2wGwQ6Febx1gKMfBt9QHRIyxIFpUBHWQ/RBAD0hhFDwPwqAdEpVe+GOi3lz11krlASLlsCBuXa8eR11kP8CDHgqFBEbB8yDM/y+v0ouotwptRmgBkUwpFkRARVAX2Q8B9LgACYxCx4OwfRir9GKsNyMmBkQkWRABFUFdZD8E0MNVH6NQFYLnQVi0Q6LSiwqACmpxb0y5GBi0NABExCwIJw6ih4qgrrEfAughxii4Mi86HkQkp2FY6cVrVPa//J06vcP0Ji4XY9hSHxBhWBBBHESEitRB9oMFPQyMre4CabLChN4qYZVGReSaL/K+g2xybuXcyrmVTc6tnFs5t7LJuUXtrPnyJNvKmpT1PE0rgALAKP8Rc+Jxq9fcfhpd4X5gTgSkptgPKR3Rc8sm2SF8v/RN8uWpFvPKEfaebifBKHzwZnrZjdmrAireDz0OyyEiIvW3nyxexDESmi3zln7E792cGVvh+/q0D/vBc9nKlQs7Db1BRIkz7ENNPLXdQ820p0NE1Be5ZGoBFKP8x6EANRGphwZUtB8B7GCKwjDKICbLgdQa9gMALMuZiMvnICSO7gy2AGCxvML9CGCHthLMMoBVBoGpciC1iP0ws6Vw7ELXCs9YAthxOR3QFoVhlEEqLQdSC9gPQzzCaJUUHSHB4B8C/6E96HvgNwBQzAEgxkj0qq2gnKIwrDJIpeVAagH7AX08wliVFJ6QYPAPnv/gD76Lof1XRysRpIeRCP007/6vNWuD3NP4ojA6EyrBiJRBKikHUvPsBxFRf+RLAhx6VVJ0hISAfzD8h1D3ZW99AB4bSIyRCP2UdVQSxdqsYGgPHXwgVIJhlEFMlgOpNewHn1sJgENcJYUnJAT8Q2jI1H2h7O+ntwLWijESoZ+nHxHltgwuY2gPXW6FSjCCMojpciA1w36UBzUw9U+kq6TwJVDe9PZvF/LKNABCQ7buS/3x4+ncuE+m1kfoaEXqzWgO/OD7sQkDzcjbZckXheGNqQQzYu1bm1Q7h0W7V14OxOtQ88FXnVBecReutouJxV0qMd9KABziKik8IcHgH3xDQfljHwAo+m7Ovy6BkSickuHYFtu+39lEYghvrXJHy8iPrxyDoAxSBTmQmmY/flSMBujvAA5QXmuBkJDCPwTljwNhAIBBKJbASLhqK/Fz/jUIUap+em8irgSjVQa5UAU5kJplPzLGjMkCsuxsjQMcfJUUnpCQwj8E5Y/bdwEA+egoBj+EaiuFYZ1WALecdbSHzthKMLwySBXkQGqY/Wjo/oYbHiWFW0sCHOIqKYK4B49/CA2Fg+rXcgDgmylNxeCHUG1l7sPvrVG8tbGW9uDBDrYSjKAMYrocSM2wH4Z76EeiZ1hMs9rrAACRKwe4HfVfan1x1XlF7yVJO2Lcu0bUH7Dt6xftD7b90AbAmaX9Wsf3Hfrz5oB2LuebzIWoofYg/P6zz7uLzeF737oAP7472y922NbrQf+2FPo5E9wriJ6cuau0LurWaSegnJJ5q7BZq77zUTrd9rUGly9+4gUkLFrsuOfa940AADPj4i27DFxUsXNMXHIj1TmkHnIeeLzfH4j5d5Si5/LO/MkBuDR7SMekto4bzrt02Xx5otpz7kzdmQFRS0d5XgoMaxi03wx8wpPT+Z0DUC7AwVRJeVDgqzCKf3D4RGwA3bqqDvBXGIAf+tVWJEyoBMMqg0DW/ZD3HWSTcyvnVjY5t3Ju5dzKJudWzi3+wewHAMyaHIiKl3mBOUu0mN3yHtpYaVSWLaBOtrFS0YvSuIdRJMQ4E/J3SIj0I/On1jP/7qm6loe421tcml7LZDxbteTdd5LEHIcevfHc7I8lPeE4YwtR8cJ2CF5jBPcwioQYZ0L+BgkxktsIuCsrVuZFT7JDJ8vBi3EIHIcevfE8LQbaHaD8bhrjuIdRJMQoE1I+EmLkC3zytazTqFCZFz3JjnX6YhwCx6FHbzxPs9PJlzi2VBinOipeq6ZiSIh0bjPcJ2FvZSQ7dEyGIMYhcBxieqOGTFHj64QfxvZ3O2hYklyqzItW+oMznsmQEuMQ0Rs1akZJFvMiIdK5/b239dhcThNY4DIky7xopT+0zIbAZEiIcYjojZo0YyQLykNCOCbENCREahJOmU90DhyQKHAZ0mVeOMkOjtlgmQxGjEPHGojqujxXuwXdZSqcyCjJYhQJEZgQk5AQyc/t3vFAL89fCgEgd6Q18sb1+gCon18IBMfdBab49wEQpts6BeCRehFQ7Zws9NFt1a60CVmibj+9/QhQXkdZja94nW5NAnys4wEg6AXAYtHhn7gjxbPDGgCKGXtOgWaFdAYcW3GMd0LE5X7IXRLqDIRY7BGnwJR7h18aJwI+6b+MByBwGTy2IcVDNIW+LIfCd1fTIZcsmYhAb9S4SZEsAIwiIVomxDQkROpze7t1M09Pz6ncSkHgMiTLvLD/TZ/JMBDj0Kvr8hw/p7rvSplauvaMwim5PCREx4SYhoRIZf77BR0A0MrfcuoDApchWeaFvTUUmAxjYhy6ui7P27zcMjnnT08jV5281uUhITomxDQkREr3I6EDACgmKA8CApchXeYFgkIHw2RIi3EIdV2e+6p2VCyHMxwYZXBMIFAAY0iIjgkxDQmRyO2ebO7f4dimgcBlSJd50Up28MwGx2SIxTg4jkOf3niuttZjHgDEpr+sV3uGJVCMIiE8E2IiEqK/cDjfxblepzQiWuDn4NhzDp1Gr6VLprWEUrrMCyfZoVXoEGQ5BDEOQaCDrevy3C19+Kjfji+eVUBEYoGSkLRFEbtCl5UalwMhVg/EFDkQU/gEY2VeUFExDvPTGxU3So5VdZMWydUnUMyFhMjsh7zvIOdWNtb+H9/EZE99949nAAAAAElFTkSuQmCC',
    '1709.08718v1.5.png': 'iVBORw0KGgoAAAANSUhEUgAAAdIAAACSCAAAAADUJBQyAAAfCUlEQVR42u2deWBMZ9vGr4lsspKVkkhstcYSlIZGk6i11JoSWi1Ve/tpaenr1fK25aX19qMU9aKWUmopiqqd2ElFlloSaokQ2ddJZu7vjzNzznNmTmbLTCb1neuf3Oc8J/fznOfOnPU3VxQEWc+WHOQpeNbkqLuiMEEb1Qt21m28kjrEVROm3CksHONYU3aDGZiFEva7biOPGpHIYpGObkZHO8ArOjq6h7dDn7PitkQHxGnj+Q2AErKTogE4OQFwcQCwWzQwC3UzOtoBLtHRUc871Ircq7ZhosxM204OJNa5I4KISBkL5xOihj1AF37hqB1L2tZ760N1IhBByutvY7V4YBbLHY2IiIo3B2Bsue0STZ9nt5JSOtBT1FA+pafwub1ox5IGbCGiRCCCiNRdFogHVtVKECW54n3bJYqwcUkNXR4FAjfFJ97lx7rWhDOnKnugsKB47bG1B9ZqKL65bqtE589U9+UR2znQCyhKLijo7nmmbjsFEvIL/boAoIcpqg7arZ7+mRXSwuHX7NjaQFlqkmerJjYvabafO7PULEEzMFVCYWFI64JLpS1DgPQk9w51AFg0qBGb6b9fAyi7cNshOMwHyHjkDGVAAzzIcixr71CFRMWHx+P+OTwXDNvNVuUH3pSW6JVDdKUxcChqCLpUUJgj4ojoQrd6H86LHASUECmnOY37omP9yGn4N6m3BnRctsBndG71HH01B14i0gwsv7kD5m185fM+mFb81luLAuocJTJnUPzxki4BHYgoNTRwxYpItzlq2tIFaLyeaI2bS1hpVRL9J8IFQRERK8h2syVdUq/IyMgWqLesgohI3Rg9jv8C3COKRRxRgrPHfSL1ZKCEaAGGEmXCNf2Dq7QRbYuI9uPNai8pNzCi/oiaoqJLwCuniabgBSJzBiVUIg0IIKLFQBEpQ7GKSN0F/YiIotZT1RJRE3DnUpvNloFP6dOZCIsnIgpHJJV+/l8iehdxRJ0wmohoO1BC9AI+ISIfbCYq8cEiIlI3cMyr9pK+y5U0DrVLiIqBd4hoBbzNG5RQiVtAIBHdfvWfRBSHAUS0C0gguhCiNKukEok0JbXdbBk4l/os2n+t75XGABABlzna1QWX0VLYKPh8DlCej/pAcjbKLwEIfHC7g92unNq6Ak5AZwBOKLV0UE+ABgAa/1KRci3jT+QAGNgyZdEWLPrQyazhSCTSyHazZejySDHi07xtswHAl1l7h8A8qPn0wK+3g9dVvNYTyACSawEYHhtiv4thT9EPSwf1EIgC8OSfmxu+0TX+EgA4fDR22wLVmY3mDUciUZUGVtWSwgWavyv2Gq+Jg7pYWHJ/sesECto0UgG0BaLG1bQHnpYNahMc3wWKut8Yt8IZW7l1o+beW6yeVrvqiQCUTF5nu9kydEGu3gtE6611i0YiAKgAAPuUnx45uiHOAUBQV5wAAJp0t+aU1KJBHdqDfzUFTt/ATGfgAbfS6UOs2z3ZvN6lErmiHMWHbDhblZeUMibEY1pv/YZVdXZfAf6aC6QRGh5vFdN38MjFiYBik++mPUD5/OSGNaek5g8qJ/7DgYpFswA0BM4CCXeRmQUA43yVb9cxo+tKEnVBEuI72XK2dK+XElxcALi4uDg6Nnv1MBGddnfz9HA7QUQlbu6ebvWJHsY1Gjio7ywAC0nVXpNnMRFlTQnuENVxVmE1XO5uq+3p7e7p4e3uJwwsx83N090tdam7m4f7SxTu7uHmsdrkQWn32xmeYR/c5tYd6uzY+9Wxj/p64GsiovkuGSaMzFiirL6Kjm1TbThbCstegT/1cirKq+3qWtTr4YL6ioonl1aqs70AoKjCu+a9QbR8UMU5fi78wvyMlRYPgU1UVOSvqOrAzHx6ZIYOYTYXxOEqPcM6F7qEMn1v/y3GWkWqITxo+10ASDryfOtnmRQ4mn5A9d6kxn+LsSqqyB7dW/Jz82BV2r3hs/2e5ZJmvuJR3u+fDv8vSgqgNEMR4IZnXOqHAc5/k6EqZEIQMiEoq4aLiObJs/CsaJ7l96Wy5AOvLNgNzeaeahy8lBM6vDG2Dzf1+Uw699M1RJwwLzHLK6p6Lkmr8MdZxByqHNxykrLq9LThSB/vvkXBrwVVa0lp7dw+U5reXNY48AtTS/p025ljDd9yQ9aR574MY9ZnbP6hr81K+nj6Kv552vEVPwnVXXPftfh9f53QgBrnRAY6H09v3r300YmQWw82bRhUpZIWfPNXRtv3AqX7V/1ry9dvOJwd8PJCVwty8Lus/jm53GGav4kPBNWTvc5xdIxjOzMgc4wlIiqJqX1RtP6VoURE6m+s/djryQcTOoN/jF7UpL8w/pHziS43fygODUhV5wYRLcG3RHTdn4iiYqs0stdvU94w77OS/VcMCcshIiru9VKxeNa/MZqD2eWCXnOVtHGgqezRUmzQREPNKOk9jCMiotOIFq3vP5SIKPNda5e09E7FUqGk/2JKutu7jIgmDBOHBpQ1gdvtlUREccVEvatU0knpRFTgE1oh1f98XOaC+07iKRHPkGQOYZfVw+KIqKm/ic94n8wNGa0Jp1lw1AnCnxJrd1n9qOvSqJawcC6UeRy5JcIZQOTeYlFo6ADenlkIe1zVke17MQfwiExPleg/88sXO3JRgyGrkyufIckcwi4f3TEDwI97TLzi3VI4QLuymxeA8kdPkVsIAMqLSRzKoA0qnmSiIr1E9OvH0Fe0DQAgcY5NL41Kfx7JnKwOewNA/bLjbGhITiyp30NzPr5TAkCV9QCA8tFTdh5yz55QGiIRgsqUADzwWKL/LSX8lUUUrQMAunksU3+GDOUA8J13ewCduplY0t8FAND5v8D+sPprvl0WfAK0enBxWvczEILTHQLmHfj3hdi3mAvGyx/3/grMNgCAbbNLj4wevdFmJf3f6Zr3j6U9puJJjgcAeOMGGxpSUxbR69YIAA4vOh/7Fp2J9B8HrG5efyUzD19t8K79ebiBdKcyAgFcc2wj0f8x8G9zmuAYgOOdj6q3fKbSnSFDOQA6Fnrvky8/vmHqFe8DCNdRTYH+/VpcnNFkxSOs/CLZCy4D0uvwQffEiJuP5yC82ZgoALi2BBUJxz99VwEIGwMAYmNb9PzOdh/Sq37aO4K88zkogDsA1EIeG5qn9OzZCG82Jup0FwATxrpDmIc/9x4HOh8xdKvvDODiHzP8Jfq/Dx51qYt7wImYCx1REBehO0OGcgD5T1tsX+CQ3mNDtGmfUkdUiJYVnrcj6mXE5s0Z6gXEOGzhAwCeiWOAUCfumzxhH3748dYLm2PSAXYbm6t841htGJgej1I4A4ACJWxoZkmHa3arDgA4uzDzkJmQACjGGklQNi7mc0j07yCEJXAATY7pCHg0a2hODiAf50Y4ILTXhFLTPqUhF7kLhJUnnRxVyn+0BtoCwJU859MA/G7wAQC0cgBqORcJvx28p1GfS56ibWyt5VOEv8sGgCfHLlbAkw3NE79bDjoAKdAlpEPrmFfHG0kwx3+3KyT6D7r8RKC2g3E/uQ+AmASzcgDuCA4G0G59fJRJJe2z/QoAYNKYK5F+Kb4AfADgLgJcAWxucEgbcASjwH9y8nll576Rd9ltbKzkMt9coKI8txa303W4v+dSeLOhedLfLWjnwfXE1we+/WbMeoMPq9Y+/NVFPBSNYnZf4a86EINbqGd+DsDL2QcAaiPJtJLGfrK/1BUAPLqggR/3oQfQEr6dIAoqkQcypbdZaxsS+cn9fwBIDvhH4xkAAO96mQDwCF3YsCoizZlIAQA3a3/2Wc6GmeMiDd3G/LHZAdcdWun3P2rOQSX3Kp32Ob6DZrhX2QxVngNwbFvE/cn5mXYudf/26WIuEt3NtXvuMACU/cIHldxQHFdEQW8b13IgzTaf0sjly5cvX+7VZvkMANmlUPRPB4A0n3A2tPTulwBkljFrkrYCdd8faugbxWfjv3EA9rhI9F934eMNXPRbwkchaNDmKgCU7dKbIQM5AAy8rQSQiU4mvokZsmz+ajUA/MT9ESjzAcB1zb5rAL4J4gMAynIAVK4CkAclAJRO+mt+GJhtyssBoP0twOqv8crAz7WqKB8AsoK6AzOS0wDaOaOWKDR+cytkE3arXTaAnV4lwjxgRTEApYEvnaeOfjpp4rtjvg+R6n/iBx+eBICk8SM/AxRrzv4K4LuGujNUWQ7NLk90PwTQofHNTIY+j3dqt+b0welz78YRHezn6d9nPRHR0W5zNszczwRnXq1Td+C59S97hAyPfy0UXkPi4vq37/07MRtfGOjjO/Ay0e1Gn3523KoPBMtG9arn2WrQUiIimtjVwzt6EVFRm9FEtDUyKfuD0UoShwa17/WegZ6BPV//nUS7lfOk+2e/LtzfwKNXjnYedvX5ZMv+jw09stY8HgqrpP9NoXPP/7Gk0TIVERGd7fjJT/N3kO4MSeZgdvlCpx2XJo7KNwfNTr2U37J9Xb1b1sLmCnFgQOJtyhNDfKrtjeHjX4u6dVDohpbp6Z1mXtc8/d21OfJrO95QNnO1YCj8EeVMqrrZS/yXGh/lNncwPENS+5B7IqtzGGScDDLVIEsuqSy5pLLkksqSSyqXVJZcUllySWWhmtHsonTAs5F26U4hEOpuPE1aetYLIXqhWLZGnmExqm0zNFtNtWoAmp297ewpuvOcpvcuT6JenGBCSc9t/nVXiF4oVpWR58rQbFp73aPWLPEoNai2ndHsU7P7+bsDwCgHu6LZ6bH4QhOuGIObpj1Fz8YuiVBHVUOeK0Ozlf3iVLQt7AnbqkG17Y1ma+06etoZzc5Y0KUpZ8mu/vcU3DNtPwqEOhZUWtLesbZAs+c75RNRxES2VYNq2xvNnr7qwMlTp072TLcvmg1g/K2TAID4iJp7ESBwyqvDPAF03VSsj2rbG812mNCnR/fuaeNC7ItmA4h1W8uVlGd/6eZZjlDjueS/jt5Wa1vLHpBEyMPZqvR8lO0+wyDP1lTufQ8A8Bf+mQePatsbzcZUAEiLj4N90WwAXiN25AHI99S+pDs08GrpzE/KwXPJyg8Oe+6K5pzvbw+fd+KLw7qhAGcvG3B46oR/Bw9SQYs8W7WktWsRtyOpHJoNAdW2N5qNJgDU0xcqUJ1otuS5lE5hJRGtyyTuXLqnWTaRetIoSo0kInUE0ffBKUQz2quIKMXzBBEtxC5RSN82yCM6VCeHtrvkUkXzD+g2Ue8uW4lu4oiVTqeaE0u71kREk/EV0SOn1kR0ZS3RC/2J6Ab3/x8S8KlJ2Vby53x+nJ17E1GZ4wIiUjcfcjqj3lZ+CgzrAmZoKYaPmdUdsFMbXkYA0fFal4l2uR0mev5d6RzifeB2ORcRi1WU1uB3M6ysIpqvBfAkgFsqmRJbF1BM3HKE55LrFBQBUQm3AbzV4SUAsRCHApy9rbk3anXYhsZgkGdran7yI0B5FRUcms2g2vZGswFAOXuS6KBoezRbuqSK8Zeu4VpbzVLC/TYA0Eaxr0tIhzbvHxkPDH0anr7nFPKAR+de1t7fMiGu5DmfPn063u8G6mQDcPSGJMltDQ1c8s69tC/7ww9AAy8W1bYRms1NAYyj2QCwVxEs+l6fDprdGkBMQgvL0Oy0eDMeCL7h+D1+66VZ+JNDlR2cU1xP/NPt25g31FCv7rbJNwIAUgVmmAlxFwGurq6umz/C5KzrKDg2C0BlyHNVNWP5iWP/k4l24FHt3NyK8twCG6HZmimAYTRb++RjfSjbEANz0Gx3GEOzzXDNDhywaYFTLf5EXggA5WVNeS551uqrTXAaUCsag//QMSEDZwdM/Sbg0bev2fKGplEj4G5IOz1U295oNoCK4zFsi33QbBABGJfz9iDtio7+ZwHgIvpoueSC/7zZBHgM/HYmqMV57swAMCEDZx97ac3na21Z0VOD84C8k3MdOTSbQbXtjWYDSC4UYX92QrOTUgjoU78gBEAe8gH3VT+lAaqFIwZouWSn2hUAEp1KMnwUqw9fB2gFSsCEDJzdaObqLbsO54mRZ1gRzd73yz3gP6FjoEGzGVTb3mg28BDif3hpDzQ7pbePR6dlRAt3EC16xdez9bBUot9e/mLVwHlKgUveEbTw1wVXJnf7uILo1EtLt3+wDf7DRCEPcOe1D24c5Oo4opBFnq2JZl/vd/LKh1GZRFo0W0C17Y5m0y+Yo9NsLzRbTw9zmzuyXLIqtbyVM+Vz5+wHWa3UyX5+tcUhB2eXdFnTFVBffjPWZr52Tw/mduimMB1rrk40W7mpf6DOqr89mr1z9UHueHVgr/x2Gs8E1dDh+l0AUB3qI082nhE/3ouLW3Vxu3nkpUkKebafGYvlJ+klDUNlyEl2zZYlE4Ky5JLKJZUll1QWaj6aXZk5NoDqRq+trgfFzihHU/zp5Kgq9w5EdpYLKZujEgAdfyPXbINotq45dtrYSSOtgV6L8sBcAFqXvzboWl2pLuxdh779pmLHyd+cJgwOxK2vfgqJXYhKAHQT/bz1qfGah2aLzbF3422roNc6ecwDoHX5a4Ou1YbUH+eIiArdGnIY86CHBgB0k/y89anxGohmi82x1ZcKrYJe6+QxD4DW5a8NulYb0kZw5FcfXCIiUs0yBKCb5OetT43XYDSbM8dWhLtb5fBvQR4BXtblrw25VhvUAKefCYA6AzsA4GJnVNXPW48ar0FoNgtiA9CaY6uyMgARrMyj1zzMXfEkExVpTwGU3akQZ+NpZ1VWhuSGMAGA1mOXjWDNlatOzM1EAOffc9pBAA70BVj3b4Y6h6XUeI1Bs1kQG4w5dkpn/2kACyvz6DUPc3PW2hffelv9/cozL6xgs/G0c0pn/2kSGxoUDy+z7HJpj6nGsGZDGoafARyK7XXrGkDF7ixgzlDnsIQar2FoNgNi30PnxYu/jA1cyZ1tI4eSwGszSDMPcxPRi1HriW7ivQSidW7ZbDYN7azJo7ehcV3ADBG7zKHZlWDNxpVVqzWRehatxVyiq5tYwJylzk2GxllqvGah2SyILTLHBjwAMLCyFmkWYG7w1tp32wEtitPZbBraWZNHb0OjKhsX87mIXQ5MjzeGNRuSb8+kVFztgEG1dgD7BrCAOUOdwyJqvGah2QyIDd4c+0KfAk3MwMpapFmAuYW1rQG4okicrS30iWh+Q6Oa47/bVcwuN/AyhjUbPfLu7w/fqJRk5HszgDlLncMyarxGodkMiA3BHPvGPk3IwMpapFmAuQXQmced2Ww+EkS0yfw2By9LscsGsGaDek2xgwo9gWHYkdICDGCeajbdDQlqvOag2RBAbIXIHJuTBKwswNwSudlsVSEdNPByS3122RDWbFD1up/a1wrAa5N+dn4HDGBejiIrUOM1B81mQGyIzLFRGawswNwSj7okslkiLbwsYpezS41izUaOvNMHAgh46VqqLwuYs9Q5LKPGAdQcNJsBscXm2EB5ERhYmUeaGZibX1sOoBwqUTYN7azJo7OhYQnwMsMuZwV1N+JabURD8HxdrrRdRVbhLHVuhp83Q42jZqHZPIidKDbHvjDQx6f/Yy2sLEKvtTC3sDZ4cM5HHTzbTxWy7dcacl8Y6OPTf7fehgbFwMsCf13UZrRR12rDemE1ERE9qKX5D328VThLnZvs581Q4zUNzRaB2LqqBFbmYG6zs1XRJRtVRLPPP8/9u6XfXtG3Cmeoc1SBGpdds2VBphrkksqSSypLLqksuaSy5JLKkksql1QW/r5oNoDHOY5OpKzdUPqbhSlZz1XqPVEZnX3xt8xmo3x3DjE1T/VKbJptc8S82tFsAKm/39leGvfaMMlfSP9h3bRKS1EJnT0nc369pHlB24eYmscUCcQygzHrtJiGZotNs2FF12yJ/u2CZhMR9URepY+lw94z8Mxais7eE64iIlW/cDPyGEd/BWJZwJh1WkxDs/VMs63nmq3fv33QbCKiaBRUOuRwQ6WQorOHzSQiovPhZuQxDujzxDKDMeu0mIZm65lmW881W79/e6HZVlYax/h2DrFmUoFYZjBmXatsk9BsK5tmG0TD7YZmQ88su+JJJirS+fe/RXfVOtbYUnS2trH5jgMAoJgGQEx5C3nMFUMsMy7VOi2modlSptnWcc3W799uaDZ0zbI5bPqCxvC67MvVJ9/8RMlaY0vR2Xzj/6Bf9KKzSkTqUN5CnkZdP1q8JNLvruklZYhlxqW6tMdUtsU0NFvCNNtKrtn6/dsJzebPpRJ89REiCn/uAZFq+EC1QC5L0dkC1kxb6wAIXEZiylvIU9FOSXTZeYEZJyxd6ppzqX7k1JptMQfNZkyzreaard+//dBsAJDkq68DQORzgMOsX3YK5LIEnc1gzYhN//HdZpnTvoKY8hbyDHJC/ojus8048OoQyxqX6sD0eLbF3q7Z+v3bD80GAEm+uggA10e4036BXJagsxnfbKDO69/9ecz/s1wxl83ncY4FTczfVMuMSdchlrUu1Q282BZ7u2br929HNBuAJF8tcHsKz5sCuSxBZzO+2dsAQNFzVcFVCcpb4XkTHq2w9seN9c2ZdDGxzLhUsy32ds3W77860Gz9kv68Exx+aJCvBuU/j5bw7dSpU6dO9W8WfHbh8dIfT7Eb8I3ctzWB3igBZs3cNLd7XUBNTB7g+rSPeuP0MdMnXUQs7/tjswuuJ+u12MQ1W2JXJaBxF1xPluh/lNdBJSpHs03KYTaanTFsWBaQ5epSOV+tBICzFf0EclnKSlrwzU7mvh9VgHZiLpvPg6LY9guARC/Tp5kllhmX6uxStsXertn6/dsFzfb3e9MXj1LjnCT5ahUAXMwE1EuGDGassfXpbKZRNSoXAFa+1UDEZQt5MP3hj04o+b6eGZMtEMsMxpwV1F3EMpuBZjOm2dZzzdbv3x5oNu2d89f9PgMKiUiCrw4ZnkMxd2et3jR0bhlDLkvS2TzW3Pbw+H8dPPr+kFwS2W0LeY6i+ydzxjeB0pznbjyxzGDMnGs2w2ObiGazptlWdc2W6N8eaPbjowUdww3z1XcLn3dkyeXKrKQ5rPlyOCVeUoVzdLGIy2byWBPIZlrs7Zot0b+MZsuCTDXIJZUll1SWXFJZckllySWVJZdULqks/K3RbACYPNbY2wvrWEjXHBm0C7dQahI/Xc7/C6jvCwApLo7l1LQ6S5r9vdEXUpyFtCV+19aRQCyPCe/s9tfZV3hQi1k2wzXboF24hSM7NbufvzsAjOKOho92p2zxSGgCYN3vV3v2mwkLbMGN09nSj55Xw8/os+5s7LLI79o6YqjnQABOc9V8k7Bsnmu2Qbtwi0a2UjPJPfnGxOHoUUFEVPay2gJbcFPo7EpKOmIUDhobegF2WeR3bR0x1HOvb2evSWeahGXzXLMN2oVbNLLpqw6cPHXqZE9hdIkr38MSIiKaYoktuCl0tnRJH04+wHnYGy2p3TSiHxHRZpciojidpjipjUwq6Wpw77dOn6lCSZlO3yciovUbiSlpUXOXJFNKGlQ/m4gG47rOXmhK+jsuE9HFeFMJwZ+GR/vuKtNbzVDVWgtpjW92tcsk6tpM12wpu3AtnK7DppvY6VQASIuPY9vdNpS/Wa4Pv8N0W3AAhuhs6ZKe7OE0PI/7V7ICNc1Q1byFtMY3u/olIpb/WLp4aR6gcc0Wls11zdazC+fhdDGbbvLImgBQT18oftfZ9aNLX0In/8quwS2+BkY2bLMehmzBTaKzpT7zae8TnUAsEbHUtEBVsxbSkUPtctxlieU2W9S0s/kd0rpm88tmumbr24UzcDrDppsHjW/6mG1NXElU2tbxMnfgFfLn+o0nInWbTMO24KbQ2ZKf0q2vA90b7i0CWGpaoKpZC2kP+9zCsMTy5pEKDPaYAa1rNr9sPpottgtn4XSWTTcHGlfOnqS7hcsPeKNMJ7/329uLgbtvBsCgLbgpdLZkSfcmr1//Q2jxXoClpnmq2kILaauKJZbDAKDTzifQuGbzy+aj2WK7cBGczrDp5kDjexXBepu0n5c0Tzf/hLyfgB/egBFbcBins6VKmvx8UMOGDcdhK8BS0zxVbamFtDXFEMvnTnIzqeWUhWUL0GyRXbgITjfVzFun0/WhEtt83GlxvE7+Jr2/Q3legIm24IbobKmP2o8z2gKgLw/k1oGWmi5/WXC7DrXUQtqKYojlgXkFzoASvpomYdkCNFtkF24YTjcFGkfF8RipZ3Y/dHjzJZ38Ewf/kfYqTLMFN0hnS7lmJ7UFAMVI5S4w1LRAVf9lmYW0VcUQy+1WOgNI8W6pcc0Wls1Es/Xswg2Zf5sEjQPJhT76fQAtv7j1QCf/gAarTkbCmC04jNPZEiXdksP9HIC1aoaaFqhqX9ZCutxOn1iBWB7WEsCdE0sdNWg2szzDLNdsPbtwFk5n2XRTRwbgoc71Y5Lmax7v94A4Pxzf+SFEYcwW3CQ6W/cS+FQnr9rt7xLRjDB3j4hpDDXNUNW8hTTnm22X2xieWK6Y+5+Era2/UmvRbGbZHNdsKbtwHk4XM+emjoyIfsEc9p4kxtuj2wYiIro9VgS/E9F9jyyjtuCm0NnmcLwsVW2JhbSVJRDLyed8ugWy13fCcpXR7MrNv03isZWb+geamv9ekFkUemV0toxmQ6YaZMkllVW9+j8exI1az1WjJQAAAABJRU5ErkJggg==',
    '1711.04434v1.4.png': 'iVBORw0KGgoAAAANSUhEUgAAAXIAAABsCAAAAABS5GaKAAASk0lEQVR42u2dd1hUR/fHv0sT6VYsoBAVG4KKNWKwYAkoxhLBbmI3atREf29MscXE2E0sryiPJYbXGF97N1Y0mKixoRhREBsiKE0EFnbP+8e2uXfbvcuyT8zvnn9k5s5899zDMHf2zsczMoJktjU7KQRSyP/5RkRzpSjYyuYSkezvPZfLSJpYJCu3OfArCtJV/zr5uPztnC1kxrxdedyzxk1a7IxeyLN2JJyrNdaVsk7XWdTybxbyt3LCvJ3OpAWEFj8763dP/3rq6ElDBAlZ4ybNOWPy8cmzm4ghIlKuddrGu6JcTWZNSBvhxnFP4XWXiJZhLREl1TDQei8+FCps/CYFmllnjAbCQf+X4Ax7AJBNzh/tH8odHLcFDKDbFTbIcwY3AmAPOwDNexZV1msQdbmJUC3jN2ktZ4wGwtTjc3rVydwFwx4BruypuHnlOTsHBD03sL4JcRUtqneT1nJmjyUrFufwmwkA8PDUfSUA3JyjuaKpQW7iWbnqUSS/dEvBaWN9c+zAFDqj9NkL5L5i3VFkZwAoy8pEWVqR0OGuvkm1/5Y6o+uvDonxQJhcJPriFCD/5IT7nu4pwM+fFZ8cPvxHpgbLt3pWXhQCgGL7v04NvaBrUxHWsBVT6JgUVHvj2h/qndW5k9y2xlTgfKuac48s+SP6A4GD1xendP6v71CvyQpgiE/gFlHO1FP314bEVCD0p/cUDFP/tBSjiTbVSyaa2VJBRI0nEBFTcyeMiJSdiGht3TyiY1452jYV8fhU2UqsVz+fAgacz6i1g3UwbCAR0dvdthCl4KQpYd5Nav3PrT6WiJSBmcL80zij7a8LidFAmBzlBCXgVVAIdLt2X1etrcm8dg2QjQby5gz0AMLt4m36zdT9fqdaGdGsg24AAPebIwB/xySBSzYoGf89P/zlNZA+qqYoV3T9tSERsS5n7QnqAwMHyNJuJCJPV62taefXqnl437HAn3lO5wFUv2vbdXoLrjtaa2YH2DsVChN5gvqs/+OX7ByNbRPFOaLrrw2JhV/4H6EHoIztuL1aJ7ZaW+N89iuXteEjlUhHTWdnZ+ef/s+2Ia/KdYdZAQKAwKfhI/Rg/W/Q698ozRM3yJn+2pBYNsqzj4aEArNjrzbAeUApkwFA3BhdzT2X+fNzts4aE9YU1dowHePG2GhqAaDnoEjLPhoSCtb/if2vp/YVKaLrn1JZHRLjgTA1yheVxcpQsGpUA+A5cPwCnEuBVKYmdgdQZfrAJATXOQEAJfuhbmNDYx20wBaVxcpY/9Gn7oZzqoC9LBYqout/SxMS44EwEPI8yAGg4OO4/a0Bx8plAG46FmVURct7ALE1614DkHeA88aDNwCs9oW6TQVaMUrUP8nzAa6DpYUAIC8FQKUmJxbuTTL+w2HcNj8ZAGT7hgp1humvCYnxQOi9kL4+9/oDt3BXepEVNqs2APx3xkdBVyI3XQ372j612weysDBdTZtNIc09E2pPA3D6866Nk7pEAOo2FfS+/ND2Z8mvXZrWGtsdx75PcA6JGcU42O/b8+i4OWVxgqzznDtbL1VvG+tlRFbvJnX+A0+aPKgGAK/btzT9BYNxRtt/7wZNSIwGQsAegOJOaTMnyvcEUHrTrypTk1/Z4a68kbP6yf8qQAZdGxtuUTAOlsM0/gOPfC3vz4TEWCCkXSFIu0KQdvglk0IuhVwyKeSQ0CHJ8CahQ/Pm/fPCLq3Lpbkc//9oLQBA4dHLOX6D36Kd0W/IbRSmAe71NaUHrwB/V8vVspKz63Swachp01fvTm549/smbqv+ZiHPic0vKp3SWFtWbn1UpBzXEHj5c2ICPaijqn3eLqvb2+PNhfz5tA2eXBFQXJKb/WxXpG3bPFVIyLUSmo7AiJC2Lg8Te3bjXDa73ayc5Pk7ERFtdggRsuVqXT7L1HZz1pQsoiVOJ7SfPPUG0dOuiUREadH4Rl29bgRSzOhmfTK+LTJ4IvKIYQr6OSiLiII+NusaI8F09Abg+KWSc1nIFrqaGVNGCQp55gRbhXyl806iLLytKR9YQUR0PZyIKGNhu4ZKldtLPsIjM7rFD8pWqgOiE1ngmE9EnSYSUYj5kDMSTMceaz/bmMa7bHaHP+tL/2Hq5cI0QX/te2w2r9QiAK54qSn//hgAGqj3XsbeOwcA+K2TeaVK9e31RWKD3AF02P5akDOMBNOx5uRvxvrxLptdscS/itRUhroBAKUkFgGAIvsJAPmzF+AQURXJZ/EspuB94AL6aMp1V6xSAgciVKVolzhVyDuKEtWK5D52A4Aar66JdEtkR/2Qn0RT7S9yI4BjUVeLZ31eigthNcYAsQG113OIqArls/jmCMiXt/lCUxztP6PLX/u3L1KVPAbvygOQ7y5u11krUtmeVBG5o7rwVa16PbOEKHA6Xl+5dGWeSNi5FXayxX2NXhIpJw0lora9iKjEYSGPiGpss7mc6OLsthNe6Yrpb8Ops5xUczklYD0Rbc4k83M5EWlnWq1IcHMioslYrprLrzU7JVCC6RgYr6TdAQ+4n2BuLnfgACBFH0VXAWQT408CXgDgVAmAWCLKatZ+8fb0IdnaorzBp3YJfZ6pS50C4gBkiWRQdCILbj8D5FdRBgC4Fft7V4EKTMefhsjQ322muInFDzpyNwfXHgcCQKDsIK+tKCLKim8AArYff1czJm6OXLE0qfvxfmpQRzb28g3caCFSUScStWzco9RvI1EdAH4NX+gmVILpGAQAbXZniQp5b1zV/rwYf6nQJzunZPCZeBFElDWtWtvLx9Q/jltcHQ1OzPvjqLo80mETjvcQqceIzFxz9vSMTAQDOJnkMl24hrbjxXMA4I5bokIe7X1IA4rkOqABXgFAaUlDzdxfZkgmzhbBlrdrLQdQFSmqcv6tzgBkcztrUEjvPtvzHO3FiXJE6g8f45HuFwyg1/T4+B3CVTQdo3rIAchRTVTIXddkrdIEciRa10gEgEvoDVQiAJkl/A4247OKL994DuARggG8LEZlWQEAoF4gACIAY3I+7CdSlBFJ6J8H5J370gGAG9rPnZguUEPXMXi9E4Bkz6bi3iQOWv3VjwQAh2WN4bphZyqgWDy4DxD8EsBujyIeEVXhfJbGPCJO+QAPb4V1VqFUjtGrAODJy3cA3EomoHftAj8Aecg3r1aiAq0YkYP7HwGr/Eeob2+W+8AcQRJMx0FNATw4u9KBvWz+HQvRmZD2W8/Hj48nIqLjXb/ZEDVXTkRZofMPLz5U161HzoW+XlWiLm7p6ub3fg7drz9v/hnbvGMZuezC+faRGURUGDicqHjU+DPXY0enEyX3qurW5geixbuIvutZzb35oDsmdUuG9qjl3qzfSmJFkiLO/flpt0yio5Eevv2fH/B0a7JOkIS2I5V9uerajubLlZzLPDOyB5B8Occn3ENdeJoboH7h+OJBI48b7jVcuV82rMpnmd6iuHadWgUxn55y6XVQW1k5P0Qr8uJobquOlqgxHW9frNrRW9oVknaFIEEVkkkhl0IumRRyKeQVaGGQ0CHJPWliwT8EHQImj25nuuOl45mNhlbbPUD0J+Y/dXSkstLanhVwN0orjCAl2VeshJGQv9wE0yGfk7mg1q25vr+ID/mzH89ecBvgH+NZXnRIn8w5s25n+ZUSPouo4QoAQ838/gpWP8xo8bE3wKJDQiQMv7OJRXW5qddC+0IURKSICLHkXdUl9C83OmSIzClsEClU14TSenVgupiTiLlPeYM8E7nokAAJIyEfPBRHTX3eoFlERPS7RSG/iYHlRocMkTlfCw+5CaVpG46cS0g41yXNjMSkNCIqqOpfxkGHBEgY/uPJqD4CJvdEUlVv79v62fTBw6BDBsici/7VraFkN75359DQ1DHmbu3g2zmAW1jaHX3myLSE4ZDvfL97tT0G3q8r0vJRsvcCELDrCADIprJsEQsX6bIwUcrpTMB4KiV1tZD0THx0iGPF/x0CiyEkxqYAQOpvw8xJ+JbIAbjhuT46ZFrCcMjPdXZ8P+8oAG7mox/6nJgyfkm9fgrMQET37xLlCIOOLWLhokOaLEw40/aUMn6+QptKif8oUVcLS8/EQ4c49v00meUQEmMNACinLTYrlpDhDeCGQ6Aec2ROwtBskzqd6CyiiYjYzEe/VMqlsoBP6D4R7fAC4P0Dly1i4CJNFqYz9leI9ricYFJBceZyXbXB9Eym0CHuXP5nHFH7SOEPFONKRLT9XwJV/sBMLjokQMJgyL+5SKTwcVF5NNuzkChtKRENakFE0T6qNjn/mdAIWEavfb4gIrqOX4moRy8iIteFRBQSTESkbNaLiE4EJ+d6TiMiRdW13JAz1b1qKIjKHFebdk/5V+++WQYCJZ+hEBdyo0pEVOKbLkyjuEV4ERHtk2UQlXTEd4IkDK7LD9RKBvwfH4gB2MxHXi8BOKhXsF4xMXR28PwxyVq2qDt3mmoBAI9v9wYQfg2nDaWCutWIzRAlCEaSBWyv++5F/W8aaz6yEw8hGVQCcEBWT5jGnBp7nQFELRu3rvTHyMTqgiQMOXq7sa+Pj88Y9ZpFl/locnYSCk7PBoCfAUDWZUPBVaNsUVUAuIda4KVCYmyzA1stEEZi0CHG5ZJqubm5ZaW5BbAIQuLaFn9hAnFPD7ty0SEhEoZG+X9mtgBA3x7J9QKYzEc1p6yu+WztewCwS/U/WnqhSI8t0sBFMgBohEfgpUJi7KmdwWpjJg8tu+jEoEOMZT3+AsDtml+8NbOcSgDKzoQL8ufg9Z/skGTXDED9+oCKOTIvYWCU060WACAbIlfB+trMR6ff2bgo7j3VqFLl8CtAMMsW6cNFdQOvAkDJHjaVksbuPoShaghCh5iXE8VA2Jo1a9as8QhcMxPlVAJw+5UgYCHxt9V2wL5KLDokRMJAyOPVwEwfxCnBZj6qPys2fs+JPABQDM0FgPUf1GXZIhYuUmVhkm1MPAzg3z5sKiV1kqX0mLpshiTz6ZlYdEhH5ujyMSkK8yEeQtJXegohAOid4S8mTZwwYpMfiw4JkuA/TxPaeFRumU5EM4Nc3TpNJSJ67JZNRER5Leu95evsMPgVUYsTY78+emr6gFwOW6SDi3ZEuNfovYWIKLH15zsX7CKiUx3nbJ11iOhSVD24vDdscCd7zNBVc2AkAeiQjswpDByuujqxg5tn9+8EvmMxpbQfcwRItFbFL4iDDgmRELQHoMp8VNRuYwdAeWVU9FxcCaGblxUhrWQ8tsgQXPQsN8COl0oJhjMsiUeHLDcTSvLtkd6wFB0yLyFi22V3rAoqXnfkgLQrZJtdoVZJ6QCgONZb2tmx1TC6tLRZO5eUk+9Mkkmj3Gb3lJVW5ONvJ203Q9rhl3b4JZPQIekvV5pYJMMbgA5ZflpbRZJDb3TIzaFDlh+QVh5yiEkQBINZiGyqxJfglwERe59m0SGzB6SZyk4kghzivdZiswwZykIkYheu/Ep8CX6ZRO19mkWHssczh+oMey3MR012IhHkEM89NsuQoSxEws0KSnwJftmoWYYOmT0gDRWSnYibZUg/C5FNlfgS/LLIFYs5dIh/QJrmkDYdMKSFjMydHieCHOJlGTKDElW0El+CXxY5lw8oo4nYS0S0rr1v4+VEMXWbbyb6vveGEeMWXKlWxj2tq2Tmpj+WdrlLdLAJvl2zoMoZtqVyQ8SZ/R3O045Il4bDhm3jTSyaqwmBmHB40Y6+o5WmJpYif3S+sy8yjwNA9G6TK35isYISX8KQpPC5XAA6xIZcd0ibBhhiWuqfHieCHOK7x2QZMpiFSLhZQYkvoSdpbXSICfmuKpeJDuKuFhhiWjJskIGQmyOH+O6ljPjUGT0zDANAoswKSnwJfUkro0MwcGYcNMe26VqaOj1ONDl0c8L+6hMnHO+XaCcEAKpgJb6EAUmrokMweGYcNMe26VqaOj1ONDnEzzIEwAQAVMFKfAmDklZEhzjGHtKm2jDStTR1epxIckibIOjk3QgBAFAFK/El+GVRIdeiQ/P2fAAAfepuqLQCAE6/EwWDh7RNUh/S5qYhSnQtg+uc+BBAybEovexEKnJIcxWCEgR5AOosQy9dnIsv2z330QeAbKPEl+CUxU4sQtAhANoD0thD2lTAENPSwOlxlpJDnCxD2b6hfABIhFlBiS/BSYEkbl0uDB0iOhjTxdvdu0vMr0S7fBcfXvjn5I7/OqQBhtiWGmRInZ1IFDnEW7HoEgSpGB0GABJpVlDiSzBl02YhOmT6kDZeS9Onx5kih/S2KPhZhixHiaygxJcQmAKpQtAh60FGEjokEB2SICPbo0NWg4wkjkU4OmQlyEgKueSetMMP6Yw4yfCmnxEHCZCTTAr5G2j/A9KqqKz5iWOcAAAAAElFTkSuQmCC',
    '1711.04434v1.5.png': 'iVBORw0KGgoAAAANSUhEUgAAAQsAAACeCAAAAAAA4nDKAAAUQ0lEQVR42u1daUBTxxb+wiZCAogiWJFFBaoiqOCuBRHFiqV1x7Uq1qXuWLtYLVVra/tsra1CQe3j1X2pG27VqiAouAuIUhAiuCCIrLIkkJz3I7nJTQiKNjduOT/0zpm5meFk5szc893vhEfQi1wM9CbQ20KTEFGY3goIIyLeS+cvXtyI9GtEKUbqigohIHBkSrcfA87m6m2yhYXdnZTF0tRCCz9uhleeI/u/qR3vaU21MApSk9wl/U2M78kL+TbwW5Kr3oS2DsY+VvHmDLPhpDVRGVHWl+/Absl3y0a1n/ngKfc9YRRZfbc9c88yEY7Gt/LL8AnI1HRXkYotiAZyZQuiVAQTET3yss972p31jmI/phARSdc++X4N/sLUvdvvMv9Fjy1gqmkyGT+lrEUxhSEAWC+6+8nTmtY7iqBLvwDAwxvP4Tun3joDADjX++VxbK1x5vm3Ji9zANj3PPvIaLNNMlv0VHiVzMQq5lp0T7HriS+mSXRkizK4snqsefAIJY+B2oICVAqlKi3rDkpSmAcgdfHz2MJi1J5SAGUCxnn/FXS1etGXNQCQNTIs7tsTMgNFDa3M7nNWN7bYbfiZssfDHi02rP/VIe5IF9tlK5YcHPCpWLkXyJv88o6D3W84+ZZDr6SbXW3mADu/qD45fvzmyF4Ob/8EjLfvEP3UfYQobwXFI4KI/ptPs3CHiA64FBFJZ44lopuCOCJahX1EtL5lKdFfVsVEgdz5zkwMy8vLOTrd+SC7R6nrsIQ8ux1E7WyTiR53GSIl+SgUTSTjzB6S1GMvEZHPcCIit+lERKX2k4lI0jG/IftI3gqSunoT0Q8ks0Wl/RIiomT8TdTjHSIiIfYRlVjOJSKJ9XqObdFuzZo1G+JFpNKjl6es2utD2UaxR24LVhNRjwBJxGEixkpyW9AKQRlRRnjdno00O5upn6Z4pHSUl67ddQcAd96h/g+SwhQntCulJgkAmmVw7TXnM1fsHjuym/jh8PA6TUx2dx7XbXDdzwtZtnUGtoY2+Aw+0Wgjjg+QF/6RbawGJjeRDktFmxw0NzU1Nd36mc72EnaP1uwKgSBTQxP7tTs8NXxKi6HhVFVj0YAzuExsh2xZYWwoL7TBYwCoEbVFa1Qo2rRDU2/d7qvsHlXO5GXlbhqaVJ/+MvhqS9WP2BQCzPQ7mzW6YfsIEYCQ4invM4ouNokAcBGD0Ort8wBQBQCeb50AANFBndlCU49iAIjD4LpNKGzJcq9RYtaxrQbIBuD79vpUj4bZIu0mAYNalDsBKEUZYB65KxuQrBo1BLyoE9cBCkcVYLrhUAqAta2AmhrO/v5SlCr/GGWP4jK5Lu4BUBk2bChko1A2qV1s4WjwR9p8KYCaCgDodAsgALwZu7o06NnsZoA13/tXolV7iL4f2FTQYUQ60fF+30YGhYmJiOLfWbN74U7YjCCiUz0X/2/RYboQZN006DIn+0jy8Lf5Fv0+UZTlPR4bLLAZFE1E5DXmi43b/JaKSDEKeZMv3cw6EW1szO+56kKQtXVgAVGW49fLYomIip2qNfXc0MjJ/RJXxrXcK2wvvdGsWWMAuPfYlQedxnLUevR2j84tdzN6UhOl1KQ6WQNAUfiSlyuKpJ24lrd79DOuuawu+GGU0+sY16qpfcYbvvcuK692eg1jfMcD0w99cPOZbgkYkLhs/ssWadXGGpHAELUGz/Z9FtxzN34dbaGPg0OPFeHlxooGTR2h9VF9/fWruUaa2Wh/RMvwatrCyel1Wif1rZGPJ3WDAkcDAFMn5qB78Xi+y9ime4dhapHY7XJokK72ESkZvhBbFNlNDZdd5W44e9p+shkKT771nQcALM5fbpcW1Wr3JaQZ75yY59aE2xGVr83N6zjPFkDcF4NtzAFgrGIyF8yNtNTuPqJBotBMrIw4TiIiqvJvfJGIDnhJiEgy2IuIaOJZ0rqojuhhcBaVjrBMJKII+Yh9maqF07oij7OeFTJqLI4x13cQQkRECehPRCMWERHReS8iIv8dXNtippCIyq2da4nmRh49Ex9/xlcor6q+XbtGq7bQ7Dvzmk3ADnVlK/wDIFuGfHd1kh+COZZDvYoBvo8wHTCYNqhvnz7ZIYy/buRoqIt9ZNfI/k33idSUp/EuANc9RwGANwcALEy5tkUrkRgAHwXAbADIPjeO03NnXRlWSzOwX3WNXLILKCOi8wbwW3VOJKuadITrNSIRERF5GhXIi4FF7FodrBGhgyHGYLuinLJ69argwLCjAgDdtlmd+ryXwzoAgIDzeWFgAuBi8lz5oW57xybQ7bz4NolIYm/2WGVe5PTxyyYiouLt012A1UREX6ZwPS+IiKo7+lfJrkStcoi7eaERH4mxuwk4340JZisdDjgOuiQAYBUcTHGjloVYAZbNdXEeXGyzXz7/YngOOj6D33BrZW9vH6K+k1gPzDgE7AQAnm9k+VUA83Rhi033jzBvjEU7Q5fvrgHYHtoRAH13tMRKRc9HPrBHhjgFoAqAiQ5McSh5qwGuG7QHUBvrr+NnM0rrCAC8MWLVF1mqY3l+wI0s2dEYnrp5YEo8t9YAONAIAG48lqOoRdU6ssW2Ytn/Q7BJCgClMpyuembucg9AMrYEACImt9SJKdLHP5o5Y/qEjU4AcB98AEBhqz6yWhFEXK6RhAUZNZ0POAAL/zZPfqfLLzcXJ+PA8MYouWf7d38AJisXOXmbHMz9XTfTYlx2FAB4GAJAjdwWZq3bARBPfpgqGOziO/+FRVove1HqJYlXZx50H/sVbwm0fS2jztDHwaHnj+htobeF3hbPKfdEelswEhKjpvB5c21RB26Ke3Nt8RLBTXVONnJWk5mDkWbsiIUVyaXqkbJZE/NnozppgJte3FmrzvPIwx3xZ1pMNalILZ03TDlpijaCscXi/OV2aWGtditskR1+7lqTceYQ3b2Qu2Z+3tY/3m24LRYY75yY5/bkNgpASPq/O1XSj9oqH6g3XecbfmoODmN8qRhDRHTCdIpEA3akghXJ5TzeJyKiqknznplwVQduUsOKlICQdE4K0f1+iUyVePA4Ce30eMhl7NdU5kT8h/2+R6H7e2zhKdnVZj8DAAYq72yYyaeX6eoCPCvh6v6dJ1YL5oSPlV8edu4ItPh5KVO16kSEAUYJlurCdzojFnWxI1WsiC01YjS1evYRPBluYgFC5+8CQJtsphzlIQDQY0ulDmxxTxm6YmFHKliRisQfBwKYQu3DfNRmPwIgui1/7TD3VJZUUSVkKFsNh5ta/vSzFIhh3vwuucsHAJvH17i3ReauoBDm+kxf45Glx2TeDoP7f58o1nAkigfAvE+f0Ll52NEfLk6eIt0YcbZ7OADxwhOCff0zmaoLoyfL9guLxg0d6iTnBb7/HNyyUl5sbEiyPyCdQ9+ZiT67d4XP9o+WMprs+URxGC0r7LACYPurqrdtNW5UD8QQi8TTyy+aKBPzrhH916yIaKPDTaLQThJl1UkiIppz6mn4iAIEyekFk74K+J88OxARfYwfOcXNmnp37SUtcVGErnYEA33sY2TUkdHC7dNd8uf8qHJHty3bdw5UdXqpEwBn4xxP4O1KIWBVXgH4XctSVl2XzYtmDf7axG0+MYgf8oApLr/xABBfRS2na4Tv5OS53rW/gtoacyM6+g/nSvmTg1Xwb/+ctllWgpLCwsJChtFg4DAXgCRF8SHtDQBDkw4ATFEBDH/kJTwQLyNAyKpkpm043JQ68af/XO9//H2GgRm0+qM72d8Fohn3/mJ4dSQ0YEcsrGiIjY2NzXDFDZ07AOkXWFuz8l8JII3quaVpb6gq8Sxw00ermqHNia8vHGMUoeviTi/I1x44YVRvjRVSNGFHLKxot0jxdwHAWwCS2tf3cZ9GXW2DBEDKU40aNxhuKkvrC4AXdjJDwadzdARynDy5nxdWSCFU1qpjRyysqIWTk5OTnYoj/q1dPZ9W/vOHbYAC4Pjzcnsb88oBAA7ucqwofmgpUHpmqRGHtpCzmhwMi7KxuVAdO9KEFcnRJFSF3mvCEK7ENQCopgZADSQwblwLINW4Ks+aqWroOz1yQMh49M8AcK/oHTlWdOjgHeBn5wncPY8kD28nsPIPI6IIs8XChRTvbdG4Uw4RhXqY83vPoY4npn5z7NT8YSWKOy4GOcDsg3HjRna3gp+c6nT2PasmQUnR/fgOQ4s/6yzoNJv2tFp1ZMWVj3t+foapchpZ/PR3DkRjB9gJ2r+/hoiqP5wWmxw1KYeown08EV0ffObKJ3752nvn4ElPyPnHKchKa1iRJL2mvQmVWf4LfCTzYqVHV2XHj46VdO7J02NF+ji43hZ6W+htobeF3haaxecN3MH0e6p+jWhRpNDp+511KT0vUNQGERu+S1E1waurWW7iQD+OeUVqlJ56RZHi7mm57rTAKyIiqmgTqKyzBWC8VMo1r0iN0lOv5E9Xv+COV0RE9A3bFgPWf7FByDmvCFCl9NQr++pcgDNeEYAkZ3Zws/nH30514tZ3liTGiXOgQulhVIwwsI8ixd1Tc93h3/OKgOo/x0Cn78b/aDTg8cr1hUB8rQmAFCN3pUom4i/ae+w7HOWCnZurT45HwATlhdZFOQgAv8xVjVYkn6o1mmLJXVwr3YeIpL2Z4gWEqqvYsI8b4ybcpnPowS4glIjoyiai7ix/4b5NSntdb3PnL/KvXQN4k5hYY4j/SjWVCuyjExGF+K8EULN5kqp+6xgehvJDuVsj3Zw6d/B/bypYlB5GVVILwNgSw4fxhCmJrLxX0AmvaN0stS/OAwC8ox7acOU7TeO+MlvvP1HKovQwKgYbYsM+0Bmv6IaoaUlJSW1NSblcn3QGAARI48xfZNwhKlpjFEtEFDNHQpSaxqjuC4VCYR7RQsEtonhclEjJbTrRRsZfbOTEX8gHQbGzZs2aNaux4ywGS7YxERHRHKRwxr1Lu/UJmsxPuu4DJJ5bywMOBKcqVTLYZ6Yc9uH3YVLcKXLdgQNeEQ84EAwfHwDY474OQJGZKeA5xgTATct23O2p4R+bAeIeQPp4/5mgyoTPUxUqAGDBPt0VKe6UF9A2r0g+CFlRUlEGoNCx3SVgRDsAt+MijTiLFuyP9OpgGd9iLuB1ReahkpUqmfy5YJbH5cCNV32+Mcz2m8zz8QGguNDyiBSDAADMvHbd0Hvgp5XdO20GJMua+qavmLKAx5ktyhobZYhdTJ+oYsE+TIo75YUOYzk3kqx72urjWvpYjt4WelvobaG3hd4W0GNF+j0Vr+BvN7FZQi0MUcH6lgzMUJdVhNc4H1+anCVUkRNXlu4G22IfW5NYoWuf6gdxTrfYGeh0NCIWViTdcNe0cr6NJsoRV1gRwxK673GMJFYZRLQa64nouo1mVpGuctBJxywnuux6n3SYgy4VcpLU/kgqnCZ7Uz+CiGhcpWoGOl3YgoUV7bcUEdG0EaTbHHQy6ZeDgk7s+GLBE1hF4Bwr2tbbBIBPTKVuc9ABiLkFix4w7sFS9VVlFUkK7wEQP3ikkUUEbWNFkhOWANBCFKvzPfVSNfAe2nZmqXo6sllFZ31sQoAo1xYRGlhEWpP4PFvIsKKHxXwAsESGTt85EG6syYoYqamm27YZp07BdslsoHdCNwDTJpkDfVJ7ZxYshpeLxWRP1Mwa00TrOehCbVAOcwAw5A6L0GgLflsmS08dGR1wLPZU5hzRQgBWAGDSCAAEV9gsoiZcYEXVMnoFD1U6tYWNLwbUZ31WBjr2+lJjEUH7WJFARr6phUDn/qK7xi5VMtBBheALNcIQtIwVWckmRDUsdW6Lfo6atHImc4ByolI9+0bp8b2F2spB1wjXb8DSLh8AHiiI9S/62UwlA10jApCvOa/J5o/s3QeRNnPQ8QKFAJBt7cVVDjojTbwisZqmWpHqTTL2LyswrCLPowD2WlQBEPNUWEQ4vCTT5II2uB0srCjUK7s1aG+ooRwr0noOujpn8IvDXPlWfUefVygOBfvaCmx9g/8mIlVW0cM+y46sOtySP+BwHRZR+++1dAaX/+CUBxHRDp+0ooXjxQyviEU54p5XhKdmoHt028UiRWBjrj4BKvgXvTmI5RQcqejJXfI7jqJI1GpHH2DTaP4rFdcy4sbCmyMKi+/35QOvw28r/OuZkde00asW79THfvWxX70t9FiR3l/o14geK3pDsCIAxREjXJnD/am0Rx7B6tnMipd/ZvcGYEXFMRs+ssJ2eSm6SY8tJ0f5qrF7at9FPEf4CJNRrsXyHBSPcgFgCAMAHQZWacxAx6nc/e1y6xKmsH5S9zPj3GJiL6i2WXqUe9/5MmBF7ociFL/8cTsUy4xRWw3VdEN/7n9DsCK2RIgNvADHzIw2bO31r6LfEKyILSdg82jJpJUiFxW/OeaP5lzm7+yyIXyhWSo7C1wEc8nOQNc1gIhERis0ZJzTJtothNx3WsM58HTKSMOvVPxmNAnBme8E+G3btqkPK2JloGNjRWoZ5ziRMghn+3ZcJ1l+kOU32374RmFFjDR5aDgAaG5RFq1IEPznuRNvFFak/IrQwhBAIyQzmn9mLriclHQVSEsq4zDG10+jVuXXip6IFYkSHweItBvk87shy4WK5gBKc915uOP6HwAFwDrLXzu/tFhRRoBh+wAt/wDxBOSVA9Vl6AU8dPOYDfgnJCQkJHwFRCR05igH3ZOwIlYGOs8iKLAi1YxzqP4gtG9rG2/t2qLrMIoE/pQ4LAVS8nEUXP/unXawoi3Na0ja8sS/31NrTPmWfIGFoHEcEVXMN+vdy3RMLhFVDbCIkrUYyRcIzPlmcS8rVoRF6TG41b7ETNuxnMoMk7Ymr1gU6beTu2ne1XjosSJg8pW1VWcDoceK5I7IfrOfPt4JZH1A2C/o+6rHO7UilY8jS3NOGuvXyKs7IiIKg17Cnv18AT2GqLfFmyb/B5lHpRoFv8yaAAAAAElFTkSuQmCC',
    '1711.04434v1.7.png': 'iVBORw0KGgoAAAANSUhEUgAAAUMAAAC4CAAAAABcetG1AAAULklEQVR42u1daUAUR9p+hlsYDkVUVkXwQPEmeB9REaNrDMYT420w8UiMRtdETSJJTIw5dtXNKlFxxXhEDV+IUeMVDYiIikYDAq4KiBEVQW4FZph5vx9zdfcMDs40x0i9P6S7uqvmrcfuqup66n1KQmBmplkxCBiG9cGIKIyhYLKFEZGkFttDCbF3mZlhsxEmPM4EnNtozm6XAj5OwnsyMvP6eutOi5LzXAJN+vXHxy4VeE9uSwdC6gwAMVwggd35cLidbbb6JMcDgR/eEd5Ce0YjmnOaNt9xAlXDhL+l3NpiTmLBhcXhuwKojkwUF6CflBmCterDzTNw01CufB6GRC+ZgqFygesFIiLaYcOvgHJjrUFYlQtV3W/QMwPtoUPXPv9Vtf5U6gIHQw+vrZHzatnG8G/7AABmjeZfyE2trRe5SheqMMOeGexT5t46AwA4N7AG/c/9yGeaur9+h38lurYgrNqFKiy6+v1yiON2FYb9ta3mzYQyzXFFtnaMIktMUZhagb2lL2t+fJAUAO6cTlcCQPIqveIVmcWo+Dme64n8wSMUlpqHoZ4L2moq8rIByB48Aipzc1CZWcb3zDiGLpOjigAUO0vUCceDr5Qv/0AOAOmTwmLXnlQBu3Xck4xB8SZW4BT8NIf22wDZspPO0cNvAvtXlp+aPn0Xt/hvx5x8+82vvMYqdJ4c6e65bdO3XrFbBnh1+hcwvVWXSLNd0FUzfohHKLDV1zMcZ/2bhR396mLIHNJ5ZqxfJrq/huIQTkQ7cugt/EVEBzvkEykXTCWiNOdYIlqHaCLa1LKI6LhbAdHLJvQp/jjAPY3wSiNa2lNBRB3nEXGL/9G+kCp9l1E61xOl7/iz91vso6JWc4hI0S3HhC5F4AKnmtR7JBFV2KwhogGBkUQ3cUrnmfE+BcBA3+0AcpupzsreCmkMSObvPQXM8X8RQAgAFK2a4AIEWe01dWTKawbcSh4DgVfTdaNObfH7fV1h7b8fbbmeSJzTB7a4HwKXeVElQPqCZma7wK0m3ADAzh4AnJNnAD62157xO0Uy91ISkrqpz67e7QoAXSWH8eD8MO3I/I8iu7Nnz55resM0DL3xUHtcAEx4FJB5MA5F2jRd8W75AGxc+Z4AavdCy/YAe6aZ7wKvcC4una0Aa7vHz/qtN9MmAidGqE/+pxrgWNml4TpctfdkoZmDg4PDnvdNw3AUrmiP1wHKrf13u3PHAbriF+ZdQ8nv7/E9AZqo7vMct5nK5C7mu8ArnDfYU3Uz1f/WU1nzMbvX2FqrT9qhFADkFe3RFrr/Dj+49zKjUwxZdaTCXnVYaAO8t/VKO5wFlBIJAGwP1RXf7O2NzR5sepXvCaDp8BYExqeHiOACr3BVX1FpINP20Go8h0QAQgteH6tJeMEjAQASMQqtO10AgDIA6PG3kwBQ8YtpGDr9J3eDxq2ZKNkwqx3wEDgRDwc5kMEp/vcXt32+/VWBJzob2mlTcncRXOAVbk8AciqET6QcyKjOu5ySRsAozxJvAEUoBpy2HMgAFOsmj4Fk68lrAG1GGeCw7XASgI2tAbnchBpM3Lh6FwHAr5KOsG1UCSDZtux+E/S8BRCn+DbLt+6NPlkEnieQFWsa7/kHXjDxXeC5wC0cPfIB/ORSBkAmB0ByBaD2zOjYJm1kE2mvb4nWRRF9+ZK7c5eJ14lODFu7JThMRkQU9+L6H5fth8dEIjrdf9XO5UfoYnAT9+DLJnybxwT03Xl275t7iYiiWq/7dc0fC/uvqKT0Nh9/EqMrnop6erVt7WAzuZR0nhwb7ewxKlJVTIF3uclfzFwXuNXMHfTJr+uOtJSOOPKKW+Pg85HDpN6TCrSe8a2686L3Cn01TWd2XmdlatOmjQAgu9RXYs4cbNqlglZBqv5AcV3e2Y6KXQHIk71VHUZ2qa8EZX229QOUl2eFhAk8UVv+5g/NaJc5LvAKf3S7g0uSs4cTr35az+pqbtnE3/pp6zEAwOajh/QvFqW/gK8me7N57Keb/7UsAFAcH2Xg4pe9ikvKvevZPHb9M5/o5Z37ON48NXKBgYsjLyecXF3HDloGJ5WbWdbKx/Ar8zC7qy3DEIzXY1abGIaxdQ7M2DoH1h7CgtY51JaVPdIde1oDwMLZKp4yP9fWRiknXzzmPLdWjsX37GyUcuu2pXdtbUkmacnLr7kIxU07Gzm1L75na0uVck/X5xrDjM3nrjae5oTHWbHF1zsCyI+ACsPMqHNnXEP8fNG2YEhzu5hM30HlD2K9bz3YFRsvnR7QNm93fIz9rB5DePk1FyGPOJYS+Pd/PNgVGy8d7zPFGIbK/0uVWy3yAKDc+VeZ8o32oqwVqUET/NYFjCUionvdjxERbUVTmfrKVYwjIoXbDSL6BpuI6JoHESXiZdXUEobo59dcpJK+SiKiRIwz7lHJiI9ktCuYiJSLkojuDUswpV512B46ql8Cz0+zAOC3qXmn1VesYQWgYHIH7XGXl8oAB/WsvPovP7/mIqTtJKqbjFeN5jT71BafJAA44tMN8NzwkaX2KcOyANxvOgP7+F9xPTkn3R8ayc/p/qv9w6ejlgL44SCAC3cBoF2GRWJ46BZc+gE4MGm4ezRv7t22H+dksLH8pth3rj0B9OoPoOW/NiiBQ6MtEsNL5cArAM4Mtp1UdIx7pb0/56R/GyP5TekKfvf564MvVtwAgNk+7w793y+7P7e4ua/MCHl6+CTVoZc1Xvvuh7FPz3BxOgCUGshvihU/6vTjGqvMwTuHAw4xr8V173vK1uIwlLaXQ6o63DcFGNTq0GOnp2bosxsAbh/Uz28Shji/1wo+I95McQBk7Qb8J27MzhaWhqHHUIxQL2w41CIN8Ll7aIqJ+au2lA52VVxxgpcXgB6R5wKRPO+XpvPnnRibYGWB7WFfZwBI7di6VatWoYKeufr5AWeoCfVKwYKEHVU+Ji52TQCgEVKAN9Y1RbuTH188ZolcwDAAwA9LuwGgL44WupmUH/Byz1Ud3GrFv+FelY+JTbfHAKBAUxSnDAYgCTt1Y7SlzjlQSjcAkLwmM3kNrGTcZRVpHzWOl37jTtV5gtNlAHLQC40kJar/ia6WNbYpgkxzuLdA9XcMtisByHRXUI4KXYZi9d8iQX4A+Kb5YgC4fHcAt/SsKS2r9mC+03GAjs/tANuQDQCQnf+iJX0vJ473lboNDrlARHG9XBr1zCKipd2dpAMXnRjX0dktcAER0eEpQ5s7Nx865TciShzf0dl1+EpKnuAndR72Lie/2u6OGXf0+IqFpUREicFecHx12uSB1nj3KR5d7BV1af7UYiIqn/VmzJ9bZ2eZUq/naQ6Wbl6W9+34TFkKY/N6q9c73Ux80r23hPF6YPPYDEOGITOGIcOQYVhNG/KcYsjGNuxdBuPoYYCj1/LtyM+zk5R35N3a2Klm3FGSdd1gWLLxzv1ui5tDXI5ey7cjbVpWQOBX2lsr7l68s36JmMjpKhC3crSHEwBMtRJ3zkEbem84Bj93SjoVTXRNEJ2j1/Dtyqk7lbxby2YvFnMShFOBcDUcQ8Xm6LWh94Zj8Fd/0RYuO6ynKsTm6NV8O302Z6aEd6vDNw/FfAw5FUjbcvRMXNyZoTvE7lOi8dQY/MMDCgDpkMzrqBGOXvnh0CBeglwGdzcxMeRUwOrNUYMHDcoI9RYJQz1RAGEMfmFCrCwLaF0hAyDFQ20KROToFe+NFRDzcSeAkWJiyKnA2wCQcW6aOGMbfVEAYQz+P3e6Nvo8AIi73xxAkk1XbQrE4+jl0x37CG6NAzBWTAx1FUA7AMp31knEWfelLwogiMG/PoSIlAM1U8FYKkipbp+SjBe2bV7mmExERBlLiGIRorkyofwtn0Y3OLe2nja5Hw7VxOz6RSxVH+1eYVoJ+s+hniiA0HKuXgUks1UnFaFBn/NT8Cwcfft2Ao5efeXJ0mU7y2cquOz8D/tfqomBYUVokHqBiGzlArHGhxPGSzKTEmCA+y6sBGDr2sfbv0vQK3NVaas8fnYALwVicPTXt7X0WbL+G26IvpXXOwQoUrqLi+Eqj5/Vi+4OSbzE+tbTEwXQ2hgPDw+PCXCIXe24KWimEgC23/vVCbwUiMLRv9AS+Mx3dTLvVv8uwPWL4kKoqgAAINIHImmm0TLnW0RxSFQoqeM8oghNexhB9zIzMzPv042/iPLX28QQ0aFFCqLkFE7KM7WHOkWXD5OIiJQd7Qo4VxKs/CuEt0bEi9oYqitARCSXvkoitYf6ogDaGHxPb29v7xZI2Qc0XjLhGpBwbqMVcNBelyIqR99v+ZXPhDd+5yfmU6ipAACkljYRa95GXxQAwhj8zU8AyPrh+vRHC+bPmxHhrU2BOBy9hoxf7bj2KO/WsqXZjUWEUFcBAPdMX0Km92QaEAXgx+BHj/pg75EVG4nUKgrddSmicPQaMp6KR9lL3UYlaQj3aZP6uiFQzDdZWwEiol+wysRiDMyLGhIF4MbgFzeyuSHrwNX000+xwDlY2e6Xm7N5bDaPzTBkGDJjGDIMGYZgHD0b27B3GQ2YoxdEvje7A3i6A0CavY2c2gu0/Hlx8QY5ew3DX6KidRy9bJ5/DAWR78E/p+2VXm0HYMdvV4aOXg6AIlb/fWH7G//uJN0QwouLN8TZaxn+3H1xZzzn2j1OLlo83uhLNiOgt+OdhJcCYbY2Sx3F0Qsi35MnYXAlEVHFMKUBLX9eXLw+Z89h+JPxGhHRSYfXFcY8ag7A9iOlmfWqw/ZQGPkeuDhuAwDYdZYAelr+vLh4fc6ex/BbAUDQ+P9GGXOh+6aV2258KsFz0KeoK7HW94PUZ9Hy53D2+gw/4IMYYz/cbOHaud7PVb/suFM+S16llr++cTh7/Sh8IBs9GuDYpt/7l75AFVr+hjCElrPXj8LHzQPBoUZ/8c/1X68vwnOlIRl2eM0Y9eTyXXhokzvAQES9/M55rbapIAo/K4ryUq9vnmm0ofszdYkkus+JNs/TGNv+e8ysMLidAPQi6rmcvYDhd+/Ve4CysIPxvmLPaxKMky59vr5TeoalhBncTkBnKTItZw9AkQQAh1IjI7/3eaJ5LqXe3j02+Q43vh1QdwDo9VOuxWJoMPJ9Ra+vzwHQ204ABuLitZy9wSj8CeVbjHlw/ozKjxSLbQ8NRr7bfO8/60VAbzsBGIiL/xuA851RRRS+G5KMeRBcVGIHyOBusc+hIPKdVLM6fmtvZQNCLX9UERdP3/lVxfC7IYnwpPJpHvQItwOQ5upnue0hP/I9Rd18LRlsQMtfEBfP5+z1GP4iAPCyzs/ArrynOTDRD8Dt2PU2Fvu9zIt8vxjkKu2/k4iI0mfra/nz4uL5nL2A4f9zgp+zW1AYEYU7rspc9lSHKj/acHVfl3+a+71ct3OwxiLfeVr+z2w5JyjYyPrt1PNN+jdn89hg89gMQ2YMQ4YhwxBM654ZUPta9x9/DLbOgY0PmdY96mEcPSxd614ncF9jcfTPu9Y9R+DenDh6m3qkdb/39MinaN03El/rXi1wv+wkkLbFy0kCWr2Dad0/m9Y9R+Be7Dh6NBSte47Avbhx9GgwWvdcgXtz4ugbsta9UOD+h26NwbTun9V4AveylWctcK1IHWvdCwTuRYyjR4PRuhcI3JscR9+Qte75AveVMUEWPX9YN1r3fIF7EePo0XC07vkC92LG0TcgrXuewL2ocfQNSeueI3DP4ujBOCmGIcOQGcOQYcgwBIujZ2Mb9i6jfnL0Au6cz5MDSDyR02Gq+0/j8Xxp3YuKoYA75/PkwKqcT1ukhLX+0WwM64XWPWc/enHjArjct5AnPxigICLF6IAa5ehrS+uesx+9yHH0XO5buCf8rkArAFZhNbsffW1p3XP2o6+pPmVYlh5PnqGiw3t7o+Y5+prXuufsR18jGKq4byFP7ht1FAAkiwBFXjYA2YNHACpzc1CZ8QhAxe1KWIzWPWc/+hrBUMV9C3nydzF6+JcJMgxB/BCPUGCrr2c4cNa/WdjRrxLnvK6MCI/vuxkWonXP3Y9e5D6Fp0FPRETrEa453OcGoPm3RES9RxJRhc0aIqIBgZFEN7H4KtEOx3wL0bovxMCvFZTR8jczy7Exwp0LLWTksZjTNxdVLAPcAMBOFR/r/McMwMc2qwfQ6UlmY7M5+idL/xEyZOZZaw47r7z7hsjjOt5+9GKvg30a9+02ZQrFTv4k1I3fDHS2AqztugBwwGNYhtY9dz/6GmkPNdw33/YDgGTolpIrEAr+6P5VWIjWPXc/+hpZjz3MYGpUCABgJMo0zWklaoqjd9w5cNZ57joPbdy8WN9ouv3oa3POIVW1JUgJegD2BCCnooo7K2IOy0vruda9bj96sTEUcN/g8uSKqYUAED6nJdAjH8BPLmUAIJMDILkcgBwKADdGWnceORH1W+tetx+9uGMbPe6by5NTt5NzPzt2esn4QiLKHfTJr+uOtJSOKIh/xa1x8PnIYVKvcQXv+zv3fJvK/A4STf6gnmvdc/ajr8U4+ssBlHxJEeCv+ox9dLuDS5Kzh5P+6tE9S7NtqHVkUH2fg+XsR1/v5paXXz+EW50LHdk8tunWzgH0776OTA/WDJvzx8ay+JcbCBdQc+8XtdoVyDgp0y39VcLPzoPB3mXT7UnplqIsbcwCe5fFs6ExDEMwjp5x9EzrXnyte3OC55nWvcrMCp5nWvdEZgbPM617lZmzCT3TuldX3ozgeaZ1rzJzgueZ1r16dsmcTeiZ1r3WTA2er7cY2n/fe+Zl++pp3XM5ewHD794LRdeqo3VvRvA807rXmsnB80zrXmumb0LPtO7VZnrwPNO615jpwfNM6177UJun7cK07s0Knmda9+ZvQs/mscHmsRmGDENmDEOGYUPHMIztF8CsHuwXwN5lZgzDmrP/B84iq66cW/z+AAAAAElFTkSuQmCC',
    '1801.00005v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAfMAAACmCAAAAADrUDlnAAAV40lEQVR42u2deWAUVbbGv85OEghb2A1B5qFsAQlhl0SEeUPwOe6IMI46istjGBUHBx6KC47MuIA+dARmFCU4LqiAigpPExIgUTYjiwiGgCCBBEI2QtLp9Hl/dHXX7aqu7uquupXG3POPN1Wn6nz3Xrq6qvr+/GwEEa0sIsQQiDkX8csPIsoUo9BqIpOIbATYrPhOt7XQjYMtDG5YbOFz02SjcL6205aAKVupFWgIqCdoBVEBM446otHU1B9AU0lsZGPXdrw6dKI+JsLRlNBL6t7sqQGPaPfAqzauQ2yxhuP2mGZ7+y4AaspiyZbqU48qKWgFgT/nW54fkDphDQBceOzSjOdPchvh/Dcu6/3oqj3SX0t7jQt4RNrIZ7l+rKzW8NWjKf2W7gGAkvv7THvXtx5VUvAKiAjkP95DRjMREdVdVkGhBgKnVEeO87SPpTfrOKlz7A+G61b126WxxzoN7qhFhtRa+0Czlh5Vkg4FXlL0zLlzMlYQEdGT+cRzzjdioad9+2u6zvreDYbrvg+tf8jWafBEl2TXf+3TG7X1qJICKwh6zqkkrkM5ER26j7jO+VzkupuVHet0nbWpQ5nRurMGaeywUIMnRqCWiIgWF/rRo0oKrMBLiq779ksXnJsL0GNPAQDsO/Y3AwB++qrECQBNp86iqs7411le7Ch3c8PABADAyTqAyiqY9ikATaXualEZHxmtuiVLY4eFGjzRB0cBoOTMKG096qQgFeh7VnvkslVb8c7EZAC04vr6I+O2AfY5m9t+dPVh4NO07itf+d+ULUb7W7NrVJznfm4kAOCN9VcWbXt805O/I+CtDeOKtj2+adGd9O7ThTcukxLH5huq+eWM6/bun7HY9z2lRRq857wUAC1aoK3HV1KQCvRder7CoIobmomIXulZTfRF+3P0z5TviR4e2kzk7HfD1rJu7xi9vm3EE572+JVERI45NCV9OdFpfE3Nf6Hs4cuJyvHwVqJ346VbnFXpBq+r63FaY491GuRYjpeJaPUHzDdHg7cen0kBFQR/bQeumrFv7MIIANXzb2wHTIx4G+1rzwMTvi0BbG1LxnYrm2r40g75KlveHgAKJ2Bv8j1AGZwouhL7Ot8DnEX8WOAUnK7EjuUGqxYM7KKxxzoNis/52bzrAcABAK/1eNZbjyIpFAVROvPm51yeBgC7q2O2Auh8CDfeYCv9rhDVADDYjP7mxbouXo1VXVHeDgAGtv35p5dtwNfRQ9Ffau/BbAA7hkeZNedXae2xToMcqSgFnnzcBoCGvj4CuG/xeG893kkhzbned69t0AYAcAxd4uLi4tY8CueK0Tmdxko1Tehuza4xrq/zV6uBOAcAdIjaEjEeQH5GnKe9dVAy4Mx1vyxxxBmrWr9Lc84t08BEiq0UuX1TAODooYEAjpaN8tbjnRSSgqggRfVHp+Gu1twVe/piK+C02WDG28dtza5Lu333Q0CXGumzP7QDUL/+GVQnudrIzwKw9edbUJ0EAFVdjFUtcowH6hKBmqKqiR2Bk1uv6JtnnxQJCzUwEduztOGfbwI4uyc3tfA/eiNvZOMXbSZFgtEjJ3kiOAXB/sYypMdmAGjcULv0932BcmDTNnO66/46f2YMgK6npI2ZAD6ru7n5Oaldvi8LwNrL0n56BwBQYXC8CwZ3Ru52YM2dvQZc9wO2rR53/53tl+XASg1eX+g18+ZEAXAgfwwigTx8eNXnrucKtx45yRMVXOa8GnbXBWblJ98BeOmS6DYOAHujL5R1hL3GhN8zPo0dAeDCg3+bCmDYbgDAz4czAewc3H3DeKmdi/EAdvza9sbNAIDd6cbK/jgAFz6bhI1PrhkwaPxS5M7tEd9t2A2/gZUavL/QbcMAoOvVpTMm9gLlXvWH9h2+B6OHSfJEkAp0PUrsvbF/YoeJC1yPbaPnv/nnT4nWXrJ449O7Hxj9l0+z2yb/ZpWx93CVtw5D9NTptwyJxlQios39iYjo0/hzRJSb/vZz7va8UUREf52+dJ3rwOHrjT0nfTFs/dyzRAOfJ6K7biciZ9dN0i7LNLDxeIrrHRsdjq4joiNRZ4kmv8jqYZPcEVBB8O9eFXHioJOIyLFvTyM5q8x798qEI+UAEVHDUSIi+vmgp322moiIvj/pyjvew26w7qniZqI6fEPUmPwWER2yVdM5u7UamDi4W2qsHEvOM/T6KKLK2JIKp6yHTSK9CgzPOa/37Wy88CddafMXmVLX2aOYaMl/Np/+r/olA8g519ECGhRx76O07Ee6fT5RzlDnI/71BFbgJSUK4RmzMk/2CJxVuXmLKdVsby267gf7uojqU+ui+q36cWpkC2hQxLjPV3brix0zgf7dF93iV0/QCsL0c067r3EEzGm+qdCsus1lzURE9ipyljtbSIN31DUSUZWnpa1HjwIvKeG7BvLL/bMDpazsmc13/WE4aAioR48CLylhvO71fELAl2jxvNechoOGQHr0KLhY5hxirTMfKZFPAMzvWRwjq4V6mRUGI50VNnOOLNc93EKIaC2xULqHEwHBKIoQcy4CgksVAcGlimc1waWKEN/nF0U4W4lyPb+r1b70U9ngP3UF4Fx5Iq7+wWR4N1E+e3mS6f1wn9T55vELznt+pSjJIfJefc/T/l16RvxPhb+eAM0t6gwDU6bo2LkVNReaZl0GgP61LzFyboJ2KqNcPsr472oVt5ZQ9U1JhUTOaU8R7ep3kthmxZyZGSgz+bcl+aTOP35HdPIq7+o8fs8733eK/EdXANGPOUl7izojZA3KjlXMqiD6e8xmInv29GZ6N61CM5VRLh9lwpqJ+0uJqLZjHwetS2okopk3ETHNhqOOJabPuXzSj18kIiqeyJbkMueL2Dmf9Mq8laVeu5Vb1Bkha1B2bEnce0QVGEP0VHQNEY29TzOVUS4fZcKaiU/W7e+AxMyPDg58e2wMgMy76uMhN2N7c7jOyif9uh4A+h4BU5LHlb2oT2fmry4PKPcrt6gzQg5lx7oRgARUAivS2gIYtfyFeI1URrl8lAn3cJc02gEkorx5cxIAdG/MA9PkHD1fXOoEPs7mXLLhg2ktdQOm6tittTcD23ANqk4kAkBy3bdaqYxyz1Fm3MMVOGIAfBc1qOJcIgAk4VA20+Q8Inf8/aEPV/6Q8zb4lnx5tjeXUfyVI+quJH9b1Bkhhrpj0YD9heEL0CaSXJ/Lg2O0Uhnl7qPMmPOIGAA7ih9OPowEAIhENWrlJueIy5tWkDbyy2i+Jfd0vsR7yg88aPtoxKbe2lvUGaGGj459/WHusLUJiB10FgCOoko7lVEuHWXW83njHyY+gwbEAIANF9gm77D3fSSi4JpTXEs2rb7De8OaaTZcn/iwny3qjJC/VtQdG7k459i0M8BTB04B9j1waKayyt1HmTTn85PXxaEtmgHAgbZsk3Psvf3F5/Zdvem3Tp4ll/23YhzSAGD4hxWoOnPmzJlqdosyw3D46pitX86myc249vl7jh95dgo6SzrUqV7KpaPMmfN/ndyYALR3/fNqQBLb5Bz3LO6Mvpuf+OZzjiUPNHaqqqpyNFXVum+F812TsR/XJCcnJ9/IblFmGA7fHeuUsfML4OFlW3IfOo0hkg5VqlK56ygzuNRPitdEYF9E/26nAeAURiBJbvKNmv1XArAt/PLQZH4lK04sAHCgy4JLpWv1tdW1MYAdnfB+I4A4dosyw3Aox9I+zlEUA3TEYQC9ewPHUodIOlTDLiufxR5lfM4Lt79kA9bfapuyBwCOdEwH0+QbbWy17QAgZRDHkpmZALB20DIAqIyPw5BpMQC+T+qP7lKKvAWV8XFefxsNtmOV8XFo2BlR3gs4jiEoeHFVEqrzX4hy62DHoDI+jlFeIx9lwrvX7y+dee+9M2ekOmh/XAmRc/QiIrZJtBhHzV/XL5307ieJiE5MbvQuaf57OEfilUREFfHpRK9tJ6LS6NflvfKWivh08pURuga5Y65zT9lCRMdiMh00N2Iv0RNX2H2kSrmycvkoE969SthrGhG9k7m/cs4MO7HNxtsmdWs74LdLTJ1z+aQNv5+ZV7zijmPe1c2f8/tGJSZd/Tei84NmEDkeW/rtOwNfYN6my1vOD5pBvjIMaPB0zHXuituf37Z15JQyon3Z+bsfmXDaV6qUKyuXjzKZYynfeH70FTZlk++6gcM76tMybCGUNFL3QFHH0V39blFnhK5B2bFvi+mKNBuAs59XXTHa5i8VPo4STAMg1smIdTIQ62REiDkXIeZchJhzERd3CC4VrZJLFRwLBMcins/F87kIcQ8HQbqgdXMs+vkJhMKxyKfnyLEo++CbUmFJFzPFaHMsKh1+OBb9ioxzLLr4iZA5Fvn0HDkWVR98Uios6RKUmJA5FpUOPxyLPkXmcCz6+ImQORb59Bw5FlUffFIqLOkSlJiQORaVDj8ciz5F5nAs+vkJhMKxyKfnyLGo+uCLUvEiXcwUo82xqHT44Vj0KzLOsejnJ0IKz+l5cix6+uBFupgpRptjCZwq6wpCkY45LyjrCgXHwgAVQfAToYX79GxJbkU8UbzkuSXV2qSLmWLU52LUeOvwVVbSFYSiCN0cy+zkWg2O5etHx/XO47foWTo9X3RG0YfiAw/+OXXEMWiRLmaK8cWxuNUodPjhWIJQZAbHopefCDGk0/NFZxR9UFEqCtLFTDF+OBalDj8cSxCKoszgWGz9cnpOLork9r6wX07PyUWc0RnvPrgolRUVyVUOANFJStLFTDEaHEvPyUWRSh1+OJYgFJnEsejiJ0KPThk7v+COzjB9UHEsSl7ETDF+OBalDj8cSxCKDHMsdv38RCghn54jx6Lqg4pjUZIuZnI8fjgWpQ4/HMtD+hUZ5ljOB8FPhPJl5zk9R46lge2DT46FJV0q4+NM5Xj8cCwqHdocC4JQZJhj0cdPhMyxyKfnyLEwfdDgWGRexMWPBCUmZI5FpcMPx6JPkTkciz5+ImSOhTk9P46FKaLBsci8iMSPBCMmZI5FrUObY9GnyCyORQ8/YWDdgHx6jhyLsg+BKZUgxITOsah1+CmrR5HgWCDWyYiAWCcjQsy5CDHnIsSci7gYIgpAps2KSrYW6qEtDEbZFjbznSl9zrOs8MhdSC0TLVU33DRIkQXxfC6ez0WIezi0Ai8TtDq+xhQ/Fm/Cw7yQ3Ug4W7FA5Snj/OBAU8Qfk7VJF54cizzgYcqxqLxMTPotlXUjCQ1hCeJ3NaWnTO2kx+y0+lpt0oUrx+IZ8HDlWFReJqbNuexGEhrCor+u0lPGedN0IvpVsjbpwpNjkQecD8eiY84v6V5JRNdjH92STUS0JvY8sU2iwjUjecx5r3QiojmJ572r8fn/QHrN+f9hFxHt2C5v+Xfse0T1uNz9d1CKAmhQnksecJoeIJUZfX2KQERm+LFw8jKR3Uisc3+R4rWkoQCGj4YW6cKVY/EMePhyLCovE5NCdiPhirD4usfJ7XP8f579yyFt0oUrx+IZcPDhWEzwY1F6mZgVshuJde4vrqg5e/n7T0eUXvnm1cxGL7cTrhyLZ8BVri/hwrGovExMC48biYXuL645R9EtEegzaWYDNEgXzhyLNOCcOBbjfiwqLxPTwuNGYp37iysSkJICYMiR7bIfi7fbCW+OxTXgKl+YMOFYVF4mJobbjcQ69xdXtIvpCABtGD8WACzpwptjcQ14uHIsKi8TM0NyI4m0yv3FPSiDzwNAMzq7ORYl6cKTY5EHfECYciyRrJeJuSG7kVjl/gK48JBrF9ljgNMY7uZHWNKFN8ciD3jYciyMl4nJ72RkN5LQEJag6ro9ZSri04lOt99A5Bx5ty/ShT/HIg94uHIsDElh8pwzbiQhISz66zKeMi485Jvha3fed1uNL9KFP8ciD3j4cizc1g0wbiShICxG1itUbTmTkQZ/pAtPjgXaPI3gWCDWyYh1MiLEWmcRYs5FiDkXcy4CgmMRHAsExyI4FsGxiOfz1vh87mwW35AX3/c5DDENBfOykxMA4Dbz7wc9ZIFvrwxwYxrU9eRBsJJpUFVRbfDj4wFuTMM/pBNlmf4bC0MW+PTK4Mg0qOoxnIGFTIOqimqDHx8PjkzD7OWf5RcU5GeVmj7nDFng0yuDH9OgrsdwBhYyDaoqqg1+fDw4Mg0PEhHRqtXm/37OkAXTQ75RDYlpUNdjOAMLmQZVFdUGhrWYrleKcaZhFgAc2T4dFpAFLRcKzsAapkG1R102FGcUHfdwBY4YKJiGbKbZF4Bz9mobF7JgRWL1Xf0AoPgrR9RdSdbNsbKeZxCggAuyYR7TkK094JplWR8PnWNkhjcH8O/BHcCBLEh4/+l590740qdXBucpV9bzDIJ1TIPXgGuX1fLx4OvNAfu8+8GXLFB5ZXAOn/UkzgCwimlQ7fGVquXjwZNpAPCxLQVcyQJ5bb9FoWAJvDgDwCqmgdmjxTQwrIXuMTLFm2NVH/AlC+S1/daEypuD4QwAWMU0MHs0mAb48vHgzjQAcORN5EwWyGv7rQmVNwfDGcA6poEZcA2mwaePhxmf88LtL0UA62NtU0plpqFUXkF/oK4jn5EvscNFFgz5h3ttP/+obABTr3tqampqN2YQXAle/TfMNMjnqmzQGHCXDq+ylQ1Aw87vyiH5eOgfIxOYhs8w27x3I0zIZIFvrwx+TIO6njwIVjINqioqpsGPjwdXpmED5nOZc5ks8OmVwZFpUNeTB8FKpkFdRck0+PPx4Mk02HOmdOXzG7JMFgT2yjD3t+uW9ebws0e1wY+Ph2AaINZMCKYBYt2rCDHnIsScixBzLuIijcgnAGRZUSmrhXqYFQajnBU+E57leg+XKf7pt5rIDP6djHg+F8/nIsQ9HFrad0JogxXeHLKDBo9w+U5YzbH4olQU7h0mOpJcdByL7KBh+u9qsuuHxRyLmlJRkS76HEl+oRyL7KDBY84l1w+LORY1paIiXfQ5koQlx6Lj+/yTMeeAxMzSg3h7bAyAzI/rwTRXpLUFMCqnnscVt6hPZwBAlwf+encqvwt7bO9Ir7+ZjmpkMNoMh7KazwHXENaNACSgMpgxMsyxyA4aHOaCk+sHgiVLuGqznGMx7M0hO2hwGHzZ9cPbo4Jz6PK5MM2RRNubQ7XHhzCWY9E5RoY5FtlBw/yxl10/rOVY9FAq5jmSXIQci8dBw/ShZ1w/rOVYdFAqJjqSXIQci8dBw/ShZ1w/rOVYVPyIX22tkWNxO2iYPfKM64fFHIuKH/GnrXVyLJKDhul3UrLrh8UcS5KSH/Gn7eLjWAx7czAOGiYH4zshe1TACm8OpnfdfZt3ZJroSKLtzaHWofTmYDxDghgjwxyL7KDB4z2cy3fCao7FJ6UiZSg9MVojx8I4aJg/55LvhNUci5ofkTNUnhitkWNhHDT4rRuwmmMJ2QtEcCxinQzEOhkREOtkRIg5FyHmXISYcxEQHAsExxICx7JQ/NNvNbFQeicjQnyfixBzLuKXFf8PGjujq5q8P6sAAAAASUVORK5CYII=',
    '1803.02632v2.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAy4AAABsCAAAAACpoLmNAAAZlUlEQVR42u1de0AV1fpdh5dHniagokigCVcTEFDUMDFRr6lZSEqm5YOemkV2ta6PSC3z1+2WmpmiaJaZponXElPygUqQjyQVMUkQFQUh5C2c1/f74zxmzjlz4DwGC9rrH2bv/e211x7mOzN7zsw6EgIDA4N5sGO7gIGBpQsDg/ggoiS2FxgYmkMSEUnY2oWBgV2MMTCIDge2Cxhsx7Vbjg4kVwR2zJE72isUisHAGXKkxmDVeUcHe6VC0ce9oMzBEQqFYxgg++WC6oFo+xNB3uaxN5wsqOjWp69Evu+JFlamGcynT4hEnva48NqFgcFGbInxBvrEZFHCcADDRhLRk73tfIYU5cfE2AExMedodcyDgE/MdKKTvfsvXjY+eLHDEbO4C5+Xtnvk+ZcGDD8xphsRxQBwdATQzg7AHlGVERU+L3UaOful/hGH/3m/ABtLFwZRsBpIICKiaGA/ERHVe94gIiIX7Wfy1+qQcu9HFUSU0xFmpUuqG569TUS0vzO6EVGwx/abqvNAFMkuzESymMqIUt2QUE5ElOYNoXRhaxcGUTClHXbUAkACsAkA8O3gboKRm8ti7QGEbjSL99ikmumfewPA6G8dAaD0s3gfCQDA8cGNkaViKsOxSTXTNngCwKM77dlSn6HF0DEWtTsBoBbYUw4AKc8JR+YiEwAwzssMWsWLcscP1dmBqCcBKCvGc62SJ26LqUzxotxhiWaw6MdZujC0HBKAFABICYd8K4Arv40RDgzAlkk/ywHH9b2bZz18CUM8tYXJ9kCFlwuvuVepmMoOX8Kg+/mDsXRhaCEM90fmJeDXqnVACgGbn3EUDhwnwc5Bbg8tyJ/QuXnWQ4C/rjAmC/C+xW9+coeYyg4B3XWFJ06wdGFoMdjNADYDKTP7B+PCaSi3zOTafH19fX19Z6kL4R/ZA41Z7/fZYAZrMcDdbbbv2rLKigFuUePgw9KFoeUwXYIt8obt0yQJQAoO+gdxTadPnz59+vRyTSnx5LO+ABQvnmye1AmQ3TNlToAc7GtKhnsBv1EHStPuDuyKqfNlX3+UksBr6gIA6KAthm+hK8d3pdGH3zRL6g/wrr5ud2pRZQGawWonODg4QCGPe46dXRhacrGfkgB4PoHqz45ONBX2cQ4geWDGvln4tXnOsUCBrpA3smWVjQHyAUC6YH5CaurJN4ezizGGFsN4T6TljlUfnf+e4GIqLPOI+u8MeDTPGT4Cpy5rC/sHtayy8BicvQDAYdiwKEA6rAdLF4YWQ7tnoJzqCCDGD/IE03Hr7gAA7mJo85yS5M6YWaveLlw7p2WVSTZ2xowqjTj2RDJDi1+NzQQA+xnoG2k67HJkuhz4Y4HXv83gDDganBm1rx5Qbh+zvG8LK/M/2vf0oD2NQOO7diZfD2NgEAeDh6r/XrVfpanJae/u7Obs7nyCEl1cXd1cXEPpyWe2hXUI6+f2z2vmkTZsinKQ9Ajzji/UVOxo7+bh4ubq4eIlsjIiatwU5eAcOrDTpOKkwQI07PUwBvFQLXFTb5R425sMKvaxQ2WxKsjJfN7665WuPdq3uDLNYDfudPA1sb5h6cLAAPY2JQMDSxcGBpYuDAwsXRgYWLowMIDZ8jEwMIDZ8jEwsIsxBgaWLgwMaBUultU3HR1JIfcxeLi6LK+8q/Hj01Xny92NXgooKCwf6M/2LIMQVHZtK11KvszIdJ0Q8JRBuhR+sXmOcbrc+uqLR43SJfurtFSWLn9z3H51vQcAPBMxwPla1ijtQXJ07Tc2kPG2VN9elNvN8bZJG0+casMNaX2it1l3xgxwCrFCz3SGvCZUOyrOuK4CqVxBtaqJ50SbbGRorSh744UBuEVERJ0BOC5WaRrqeo61gYxHWzNysYy+HG+bNk6cavJSojOBN5vtLfCuvlR4QSNsNuPYXF3ZxSZytclGhtYKtzm+n5xSb4Y8caPHCN21xsc2kXFbNKPTUmBJlW3aOHF70z4Hwoe9uvNPd+BPtbqRobWinc7bDp1m8S/TA7xsIeO2Du86A+BruW3aOHHbopwARM+sd7b1zti1w1dU2u26It2m7FSukhfFFRuL9b7JOb/ARA+9xsqsDFkRO8zaOBq+nSwS0zqPfgD6DxaJTpnuAQA+jUdhW7rI3kh3S43JVyfC+8nHpi2UAQAlx9YXDMnUrX90xSsTkzKWp3P9d/y74dDUqV/yQlYP9euyDoe6+j2UrWv87xaP9u9FsOOpLeLXj//zseaiafWrEnE46UjA9YXvv3VZLHFld1wBwAOXrVjqnwe3fN/ol0c0t5+SiCK6FhMpJ45XEdGn3aqIDnS4QzQ2jl/Mc8sgohX8pX7Qi0R6PZRTnMtIFbKba7wUTUSqKLZEbkP4WLOc7rtNRbsDrxIR/ZJCNHCsLWTarUpE/UdJBd1+tE2bTtxlJBIR5eCd5ro2c3bpUFMHDM+5AgDRXQG7+Xt3A1UL4tyBEXbb1EFccUbYUADxxjy8HnabQqaq1r8fyzWW5uQAkunsk7gN4qvJEsS6zgUg/1K0/3A1sifZIWDkCw3iiGuAEwBITPm/mLfUz+3lFDdBUnguC1UA1JQRjvvi8EuV0wkAXppzl65Ykp1kgpPfw2ln2JRIvgd6pH/YgyMee44dW20QIQDQP7nMG2tmi/YNpQv8/ACEfv7TcFHEuUEJAAq42bJ22ewAVfLgrZ5RvDqJWz5QhE5SqVT61ZvqOl3xkkmjNb0evqu2h/IbpRlvO3864lkVO7jaHLKPAYAbcnGx0bOyslIhr6yxndXdqSMAtEeuOOI6qE8rDc37BDZ1drlph/nJZ3viBKCSaFZpVB0E9IZnf16crihHnSBRSoJej4YjC586241rzG+/ZMmdLfMSotnh1dYwvqrGCZDBE2U3FgG42GlRj7k2szoE1wGAEl7iiPPoUgoAJYi04exy+RpqVk7rCdwGDmZqnNCzFGOA0K7pANC4Vx2oK3b/x8+AgQOgVA4U6PWgpEVLIybJuMbc7cB9iXEX2NHV5hD6mROAPI/eiF6zZs2aNe5918wVIwuvyACUor844iRjCwGgoGOEFelSpU6Moqe6wbG9AsB5x7u3OgKnSgHVhxNiAemG788BWNUdkMt5RUly+gWA1vITpt/vAPF7KBa432/3RW6iSteItfUAZIPY0dV20IhGAHiyN4CrGR9rLmKUddU2kHFbL7kcAOjAc71s0cYTN/diAUC759o360Fr+HrY6WU515xHuciLs5Wvf4RvX58dcmbsxrPR79qPTPn0AefUPoucAODIwkeCLgwbc+rdE5KopHBtETixMNY3OzLeO1r3OEHB8BmS6GhdDyzadT3wLFLm2Ac//qamcc/6iAc9jvu8yg6yNgLZjLLzdd17DUuEconnsEvLZr6uvpR/OeeCff9R860l47ZwatZb/hur17nZoo0vbsdna33eK93kaHG6GEB5Sd7HiarVa6Ci2iDdWqe4NpD/pZOuWFzeR3XRy4uzHJSf9+8o1INrrG7vcFnWS8oOs7aIi9kdB3cWn7Yyo3xAiIjibqfVDQ5r/ltU9vIxAwPY25QMDCxdGBhYujAwsHRhYGDpwsDA0oWBgYGlCwMDSxcGBvy5PmMVZY4OKjkFsl3DICaqbzo5qOT2PYCKcidJQxBss/YThU5lB9QVAm46u4urtUCAiwXpUrjrp2Me8b3/UulSMP3lyeyAQ+sy5IO+813JlxmZrlMjegB5U4oihn9gCdWd5Oq78leC+NZ+1tAJElXsyDpOV7tqBowsG/7QCy4WvaufI2zLZ62rXjNB5nDswUz29ntrM+QjQ1u+U1C/qK96eovKMju+V8qIPnBK17f2s5DONFFhPJZrAtY+g3xL39W3N3tBY5arXjNB5nCMP72afWqjdRjyrX1aryLk039vuLxUAgBSSAGA3p3xrMQiqq0bjwAzZEn61n4W0jVB1Ddyk/rJSap1h7TlbPlSRQgyh0PCTJXQ6gz5YGzLBwCqxaMftpCqCwFwQQUAI2s/s+maInruhWPRAPBTVJHVd8YUZaVQFN4FAMXt26gvVL9LrywvBiAr+YNvuae9sMvPumsqyDwOfhRnCagsvwVAXvIHKmvBTPxaM5TzH3/Y0j5P1UwEMjEOMLL2s4zOFFG8cwoA4KfBVt9IPhHWKWn/ByfjZxDSwjsvWbZo78j5MiAz2jsBSA70+Yyz3NPiwPizDfMWygWDzOPgR3GWgHkDvOcA+0J8Nnz6iV8GM/FDK7XlAyCf6hxpOYcjIPtv/0WAobWfpXQmiNwn7aoCUO0mscGW76HhnxPl4xAR9e78K1Ft+DgVEQ34JxE1OizjLPc0+F+vCiLVy0+TcJBZHPwoniVgdBwRqQInnLjVZTsz8WsdhnxkaMt3HnENswPaX7aCKnv+gBdrDaz9LKczQXRrGR3HZ0S0uZRm47q1tnxu558BAhwvAHAeHQK4vP39bgAdAMCpnVH43dnx9wGSl7YdEg4yh0MvimcJ6Kr2bLoS1eVWPDPxQ6u05QNQP/eNLQ3PKi1nGbhia9HkckNrPyvohIkQFZgCoKwTbPlWv48dYO/EeSENxz6hHpXl5eXlVci50RcA+kq+b4rWBIdQVNwfEYX/Ow7erxIEA0Ckf1jfxEPMxA+tyJZvdxkA4NKCgIcTsz+0gkYSuPXgo0oDaz9r6ASJIHnu9DmcC4ZN6aK+p8Zlr5tbvlDYOG9vb+84/KaOt3PKa/L2oDCHUJSRJSA6MhM/tFZbPgAI7wa8G/j2eWuYPAecPmBg7WcdnQAR8KzDRhwcCYj5+y7VNbpnDUgBnavezkYAUvRELQDIGx8QCmqGQ2gkI0tASACAmfihNdry6a62t0RNy3ayhEM2RJHtBHREvouxtZ8ldE0QdR63dZmjvWjpIgOADIwB0K4BQGkj56rnow4J986aAuAURgsFNcMhMFLNypc1loCuQ/hScn//F+5LzL7A0uWvjQpnKRA6WWvLp8Ogef/37lJLiBpO2932Ba4jNDoaAHb1XQOr6EwQEQFI2DPzv7DqYkymPmQhkwMguRIAMkqA+qQJsQBCKwDsdr8LnaueGi7rvykAlCsmjTMRZAYHP4pvCSivAwCZ2tONmfihFRjylXcfAn1bviqo/39vOy/fbwmV+5jDvsC13OiH9az9LKczQZSbR8Bonxp/Hqkp2L9jUJE+b53sevrPY3+a/XPJTz0OzS29/NNIaXLEb1d/mzditQOA4G9v1O8OOnjs6GPSkNVVx2L8tT1795t/M+/d8JX2wkFmcfCiHII+kdXseEzypVvA7Ozi4yNPvHau4LCyHy5dvVJ6fVNwLDsw/4KQPbt+r+T79JJBwLfBE4CwjefaZ7w072UJTs/ZKLv546UY1EwsaJeaEdrZfKpRKwpw7fngL9wB4OXlt8t+LImymA6AINGlqavyUmWRdopRffDB0mMOB4+EetnqM9a/7+fXanSWfH9c7eV+zs3bRcJZ7mlwszLQZJB5HPwoPUtAbl3DTPzwd7Ply/mVwkLE+OkxW4nMTheblZrHIcZIDAz4M9+mlCtsH8k8DjFGYmD4E9Pl4NhL3z+RZ9s45nGIMRIDQ4vBnIsxJeyhsLPtrX7zOMQYiYHhT00XBgYG5gTDwMDShYGBpQsDA0sX60EZbfW/0nZnhjZmy6fBrOmRf/lj6tX41ifa0pnp7OeU+U4OcnqAF1V1vtx9OK9cllfe1ehJOsMgAEBBYflAf3bsi5cuFRtN/8b4X8Ukb6XvkNYn2tKZ6ezn5Bt/yB3+6L94Ube++uJRfiYUfrF5jlG6GAYBALK/Sku9R+mi2nBDWp/o/ZcjU225flf1/AMWkwq/k5wML5kpB717apJn2rWvKEJphmht/3vu7Nes3aC5M9Paz1HNQEP7uVFx+uWQ1wTYDIOIiCqQaqZUs5wXm+g+eSnRmcCb4uxS8chUc84R3Xwky1JSE2uXH58uPwwTDnr31CTPtGvf4uftzBCt7X/Pnf2atRs0d2ZSrVOca0+JsbFJk2WTlY5mSzXLedE09qa9CYQPE+c34EUk2xcQDPisXGwpqXC63PJ6BtthwkFPEuFy7446k659d76fao7oVPwJos2yG7RgZvyXSe+x1FTbqLdFOQGI/q5eDJ0ikv18AwB6FlhKKpwu30yM8Uxt5Nz2jpSCc9BTm+RxJnw8/z4YW/px5nqKslIoCv4A0HhV+yCl7FQuz8lDZ7yn68Nz7dMPxd4HXZoSbaBZK1pnHCggWjcoJwPKwmo07snkGwkKzUOjjUdqYDdo28xM7VkD3QBQV6QSZuWKjcX6T3JwSgy68Nuss0JUpnsAgE/jURGyRUyybh+tVAHfjbGUVDhdjj3sOLHqB03h6IDDqm1LlFoHPbVJHmfCx/Pvg7GlH2eup648NWOmauNnmQPXAgAlx9YXDMnUnSG1xnu6Ppxrn0EocGxgU6INNGtF64wDBUTrBtXJAD4Zl/7KCx/4PX6MMxIUmIdWG4/UwG7QxplBcM8a6gbQ+H7ysWkLZcasXPHKxKSM5ek8Sp0SbczqoX5d1uFQV7+HsnVtVlohlt1xBQAPXBbj8lZEsukBrw/7be/W9ywmFVrQFCQSZSBeXThqf4Yo1Tmdc9CLjtM34eP592nBVfHM9TSVr+UQbXauIKJPu1URHehwR7cAUxvv8ftoxzQMpaEbmhBtrJmi4/SNA41Ec4NqZdDOdpWkCHyDrugZCRrNg9PGI9W3G7RhZlqXRKIphrrHxunppoiuxUTKieNVfNaxcfxB8twyiGiF3lJfo0QXo5ziXEaqkN1cm7VWiJeRqP5Nh3dEWJ6LSlb0EJwelllMKnh22f4UMMT3uzoAoFkjwgHXXr5csyv0Tfh4/n0wsvTjmetpKotCgX/UFwJVC+LcgRF223QX52rjPX4f7dcHhqG43cG0aAHNcDUwDjQSzQ2qlYEdgR6wD9uBHvpGggbz4GkT2BMizAwmzBINdQPRXQG7+Xt3G7JyxRlhQwHECxBzMXabQqaq1r/Pe73bWivEBjipF113RTi7iEom6/kvu+PjSiwlFfze5bsueUDAje+eAnDj4mgAI3IMQjgTvhhj/z7wquImSArPZanN9dSVDwKQog74pcrpBACvywbGe3p91DAOve1uWrQ5mo1E8wfVGLR1qADg4GF00ao/D742gT1h+8zQpFkin9IJACIc98UZsOqKJdlJpv7rvC5OO8OmRI7htUX6hz044jHLrRDd1C51CriJcISLSXb+xb1eL7148PEsC0mFzi4Xg7r7+vomqG8z/Y4uQv30TPgM/fv4VXxzPal+UxE6SaVS6VdvQt94T8CQzzhUqjAt2hzNRqL5g2rMA2aVX0DNkfkQ9ivUEvC1CewJW2fmBk1ZoURzujVnsnxDVl3xEjxMHQn8Lr6rtofqTdhKK8QO6g/tBtOj/klkz6/wQs/0d07+YCGp0OfM13ODAdD7+ys7AL1wnd+mddAzNuEzAWNzPR16w7O/4L1Sgz4pCQKhnapNi7ZKM39QjdROr6zqVPLpEzA2EmxmGgJ2g1bPzM9T7ZiK333N28FUHWTIqivKUWdKKr9Lw5GFT53txrVZa4Xo0aUUAEogxsNJIpJV5z4MQJJ06PKjlpEKnF0oNxgAJJNlqQC69T0LAI2p+g564d5ZgNaErwnUrJymMdfLNGoL7ZoOAI17m+ijGdM4tHOJadHWaBYSemTohvdS1NnSjnRGguZNw9Bu0OqZSWLPqPNnV2yzO1gGAFmKMYasumL3f/wMwOBCXa2E14WSFi2NmCTj2nK3A/clxl2w9LCUjC0EgIKOYvy+iIhk7SVqt1e/vhaSCqTLtjvqv+OQogIkG7LSAKzz5Rz05HX6Jnx8/z7tSkpbxTfX01TKAcihBKQbvj8HYFV3rls1AL0+mjGNQ8N/MS1aQDPkdfrGgYai9YRqPt/vn5e8LTW9CvpGgobz4GnjkRrYDVo/M3zY+TUAOHPjIUOzRLlcnxKnSgHVhxNi+axyOW8QSXL6BYDW6iWMWgnXRbHA/X67L3ITVdw0rLVCnHuxAKDdc+1FSBcRyRzjVwJAccVQS0kNb5Ud7+/evl8REc0NcXGNmkNEWeELv1m6i4iu3P/OkqN0cnzHjmNvEx18ZPn68Ukyynysw33jsz9/xNV/ovZmKL9qV/cVact+mTX4rWPaSr/YO2+GufV7hYgOD16wZd4+7dA/jHHzHv05Een6KDRjGoUSpfduSrSBZq1orWYh0bpB92llUFU/vx7dpQ6TaonKhixJW7Gvm+vIO5nG89Bo0yPVCbdtZkREN8bF7j/w1qxagz17cnxHz/Fn+JQjiuYnb41b3Mhj1QRxgxwf+vHON3bA+0mOX6tEE7MwyLkf0cb2roNXaNtSRy/ctu8tax4f2x6dW/HGVJk4z+GJR9Yw7YWjvyZPL7KU1Kx39UsqA+0AGLrw8Uz4mvoyVtBcT4vi2kBJk324MfVDlT1+6C2uZiOhdyM3DAJUZ6bFJ4FvJGjeNAwtC22YGeWfkQ8MMmsHF9XqnA8NNOmKxeV9VBe9vNoLSDU1DRusEG+n1Q0OE+vxHRHJ8k/VhwyQWEraeq0tPrq2soVH2J2sfkZg7f7v2tjMGPB3e5vylZ9vtvAIYReKAEB5YHRbmxnD38846ezbe+xbdoRT/+kT6Zx/aOjLkrY2M4a/n8/YodxXW3qIssK7vgF2bXFmDH87W746l7b6b2m7M2PpwsDwdwERJbG9wMDQHJLM/d6FgYEBYC6WDAwsXRgYWgL/Dz/WJ/ofWElfAAAAAElFTkSuQmCC',
    '1803.03670v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAgoAAABkCAAAAADCy0eKAAAUxUlEQVR42u2deWAURRrF3yRcIQchQLiRU3ERWG4BxUWWYxFUFCKHLAuoHAKCoMiCEAUUISAoyqGyQVg1EAmCCIosHsgZEDBETAwQLiFAAuROZubtH3N09UxP5qqB1e33T7qruvpX35cv1d2TzouB0KULAIL0FOjSS0GXWiTn6Fn4f9cckgb9XkGXfoHQdWtLIffYLbvUXcwiCvRVzleVc9G+qm4/5Dz3IZA+N9MU0ciQX9R1ZKgvgLXvuamFNV8UmKJfarF9XQ4jR/fE+59UMM3ponFczpDUHv8q4zyJWzpdvzB6RlKoy1FySJlzzhhjJlm28wfdrNJtesBj00BeSxhSVTaO2up1ltwwmyS5HJtIXp/TIoU+qCOOuTnifFgrM8mSpiGlJLmn2RUXB3Z6ooyz7HjIRBrHIZfkb8e0R8khMb5ztULL1tq/VDHegtg0kMuwSjbOxQWi4Gp9YEdvAEA1hACoEjux0xnvF4W0Jljn5pC6/VOuAij/WOEBAKgcW93FgfXKOsvrfw8CghdEAcCR49qj5JBQbVROkmXrYt3I4FsQmwbyyVVPyMa5KIVvuwE80FHVNrrmZO9L4aM3WnxkcnNMjHkTAFzFRgDY2t+nK11aGgBE9AKApICSgHp9VwMAfrrH7aGSiE7IqGeqyMa5KIUvewMnmqtvJMo9+dmPAC5u2Vdoafht6+fZbm/l6g+/uBsAikvNxUYYS0zFANM/PS7UR++wjQBuNG6ZaAaYG26HFJ05dCM/uRQAjMdTyr4hvOvdFAB4PgRF739QlJdHrVFSSACe/iYNAHb0sS2iX36aDgDFRnOxCUUmY4lbooD0hOiANJ4/eh4l545cZsbBXE8C9ABXxqpgvT4oqo//oHDSmkbXhmwHUPz8+MjIUVvKDuCHLhhiuUJ82jm4TxqO3xG9DNkDPm21+2859oNCHt6dBWzpH3NhH/BjG9gh+5/u+N1r8x4EzO88nPLzpN/KIs2+3KrbrD2mDsHYksbN8+YVaI2SQgLQt877AArLW39Wjv+j6t2zxpiBhF7BTc8hptz929wS7UjPiA7IM5PbJCJlQrvPY4+eanHQgwA9wTndPYxo167dXdXatWsX0qZdu6dIfoztlp6teJ5jxpPMqXaEfKGFkWzYv+xbwik3yb+E5pFkTtgyki/sIQeMIDnkeeWoJKwgp5lP4jlyzk0qkJuIKUmaSS66p5DMq1vmrdXeTgag2QmSpVhHUnOUFNLW7ZxVo5j86AyH3UGSz5XLZ35YPEnTiKY3+c94T4g2pCdEZ6S56pukMbzLdfKvT8jBaT9BrH6LzO9o3bGXQiLmpGEnST7Wi1nBcSQPpJcZQdGzJD/AepLkuD+ZaZpIHsMGkisbK4cVhD3I63PJ1nVN5qmkAiHmkuT1Cm+SZI+y08VrG0ZWqJ5jLwWtUVJIW7fztCGBfJ3W78sv60i2nUCSxZ0HfL2EHhEtSM9ic0Ky7pska44j+fQDcnDaF4ivegLf3+/YegltDqIWANQ6YP7R1ARAx6ZlrmrbLkyYMGF/RcszxLjUb/BVbyAZ++Li4jLaCsvaI99c3tIfiLmw73BbQIEATQAguaSZ22tpFhA1aM3unAR7i9YoGSQAaNhrNU7eZdu7s8P8SbFXCgGgQuL++ZM9JKKJF0QHpP2aDQQVycE5l0JmSspPR0tTUjY2TElJOSv27G7YCzACgIkoh1L30/9qw/Lly1cP2HkJAFp2XY4dfYAwDJ02bdrCjcJxg8ybjrUCYrBhaz9AgQDlLQcEu0W9AgDo0jHdspuXpj1KAslyF7crY1s/286Ke9svi61j2Q7rtvdrD4lWpIdENdIqgxcBusM5l8IHcXGTKi+Oi0tMjouLWyt0HE1aEdLZkA4Av3QNalsxFQDK/CgxO6I8ADxp/tiyLHy2t24wcG/wjwCQLN74hq+JMABN2ybmRgAKxNrdtuJJACjzofQHS2+tJoABZlz+XnuUBBIAoH/0O0Hlbc8PU6f3NiAfhRsB0/SlcUMyA0FUIV3LD5zWlWnWJvJ0X9veGnxOkvsbv0XyhQdLyeNhqWRcdDaZO6WsK9xLltu3kshWJpIsql7/MknOalNAXhsrHjkMR0hyAf5NEVJgvQFc0LqAPF35b2Wwqkw2k7zSJItkk3ncudPFKP9JXLqc5PTgX0k+XtNMXsT75LnooVlvsfTprTQPapvvAdGG9ITohKS52hukqcpskn/vSCk4zVK49zq5cpllO+3vDdF+/LNPdh94jCSNCwesmtv7MEnzu53eXfTiNdfT3/9QRM3FJA8MDgvrvYQkpw8jSZqW9Hh78dRs8djPmphJMqPSDRHydb+we0ZfJWl6q/+6ZS92qTjoukta07XD1h9I7PkfktzYbNWkUhej/CadeaJajcFn+evj5KahNcMeXUgubBa3dv7exuNOL2xdaQ7ZpnLrWW6JdqQHsTkjD8eENV90aFBY08klo+tUH3VKBk6rFLK6kXz0pIt5mdKz7Vs3vPt9RH6OfaTDRIyHLV8POUFsDWmXmJGZa3Z57hMs3vPhzpuWneuppa5G+U/SUGHKRbLQpN0ZEKJr+YzTenXlo9MzUdr+qEH/ve3//fsKO3oD+zrrlaCXArq2BUJG67n5P5P+bqMu/d1GXXop6NJLQZdeCrr0UtDluYJjNZuz9zZWdszfbj9VPUzs+OFAaDCNRqPB60pyGvnT1l+iwxwpeK9RiP+xObF+TEqrbz3v6a2pDSuqm/yTNWOqKMQZmPdsvxRdyTm5UmSLxpZMY1L9isj6rhm8/ZtJp1dAEuc2fkTZzZ688+qu2u+KHdb6CUr1+mNRh5HmOZuyTwxfp6KQ/MGQLuEjWEfW+GmZP3b7jiSLxs2/mNYvV2zyS/bEqKIQZ3B1bOKh2BobnJIrQ/Zo7MnMQXC92pU3encarT+JMYaO3yHsLjr2WsiDA8Y/UkfpSJ8SVQk4ctfdXtevw8j1zQag6get+kUKFKBwtpRPOxxY6w4dRIOFj50MBwe2/icSPj96n9Lkn+yJEaNQzeCVV2ugffHwP7VwSK4E2aNRkokHmt+45x914feqQLKrULivR10mpyNZ6HiOJG+MNHpfwQ4jR64iyYcOqSnz/g0Zq4ID6/6XSJpDN5Kfh+SS194rEpr8liUxqlwJMzBFjiV5DK86JleC7NEoycyZ6cN5PCgFmkh2b1gsdKSR5MSzPvAcRr4cvonMalWoouxZv1tKKahZN8u9TZJ1niI79HNqklQKqlwJMzC3G00yHeMDUAq2aIRk+lQK5Tx7yvjkpy8qCC3NAGxrX9+H1cxh5IS1jz065d31lURKwaa4b6WsnGrWWWMYAET8ghvJYz46GXr95cpKk8wnMiFX9hkYkgEgGZ3l3/jboxGTWbz0JmuOMci+QJDrn679idmho7C3yccqVo281gZYaFJRXr1AOauCmrUXH5Jkh9ZMRefvybc7lipN0lYFMVeOMzB16GyUvyrYoxGSeePPmTQPHOXdiTx6Ghy2OnXhYKO6bdXdvn4kIY4snPpMUr0Xh5YIlD2N6sj7kRFYRBEAGAkDgu8Dhh1cozTJk2OuhBnMrvBFsPxVwR6NkMyIQw1gGLtmTyA+YoqcvmGlumVNc1/nLo58qsvYR1PHJKxQKAWbhknMk8CKQB4A5FVBA7QCUDV8t9IkU+pcKTPYcPjLyAB8MmSPRkxmOQCNsFNyKXDBLgAt8IWqNfV4TR+nLo68smMUEL5y0pcKJeHcxAkTliB2QrGENImsOsgFgNxGqBxtCT1HaZL1XOeUK2UGu7ZsDkWK/FKwRyMks/OLACJxDTKsNmyxGZA54689gBuooerYD19LwT6SBhSGBAPA33IUysiRANZvjW0qI00iK6rlVQCmG72A+zMAGAtaC01yJOaKBtUMDn22NhgXE+6RXwu2aJRk8nxdAOfRTsaqYLS8KH+i1jI0aP8OgO8Mo8QOnESUjzO3jTxRaxka1PwGAHYOd6AUoUhKlkQWJu8wA7si+wEvH7wO7K04XmjyW0YTIEZxotYycQY/j4icO2fqiLvFHEqSLRolmUHDhgNIuOsJv58gfhs7ILL64+NyydTafcnjYzf9vL7BWlUHp+C8j3e8tpGptfuSZ/u+lbJ/xmqKFJ4adXd4lzEy7q5VLPPEZzK/77CHJFcOOPrlfTtUTX7JnhglitTafYUZmJtYsn1EzKEs2aJRkpk96bOU1zr/4t1p3L3QNn8mYDqQUbu9wx3PhcP9fXwPVhk5fybAIz9X6xQFaFP8lZqFMwcq9LT8qujcvsh7I6BqkiMhivkz/cqTF7JFoyTTfCj97j8HS323sfSNWQELIJDnvp2s24cM6PsKKwcFDh3Ic99O1u1DBrIUjK3uChg5kOe+nazbh9Rfftelv9CmC7rzuy7ozu+69AuErj9EKXji/i7Jt92D09wGh/hbjPQNF3Dnd8AD93dXvu1eS/M0aqNzKSRv7N8DhtR0f/cDF2jnd8/c30Xfdq3P+I95BnJl/y4YnUsieWP/XjbSY6ITUtv93WdcoJ3fAY/c30Xfdg2prNx9sH+vJ53kjf172UiPiU5Ibfd3n3GBdn4HPHJ/F33bNZQkz/5dEskb+/eykUk+27+7cH/3FRdg53e1+7to/652f1d8222+8IKtuWDl7kZa9u8ORueSSHBp/+7s/i4grVErSK+IaqTx/NHz0LB/9xUXYOd3QHR/F+zfHdzfFd92my+8YGtut3J3K2f7dyejc0kkuLR/d3Z/V5C2qBWkV0Q18szkNonQsH/3GRdY53cH93fF/t3R/V3xbbf1CLbmNit393Kyf3c2OpdEcm3/7uz+bkfao1aQnhOdkOaqb1LL/t1HnPOqEJ+cnDx1TnLydy2PJCe/J/YUIzx91QAAkQ+8hCtLRgYDCUvcVnJxSTgwPH8zACBy+CrCXNT1eNJDAB7YbD+q8/6rG0ZmPnAd9p7gyq2rAI0ueXVbYj+NVTdmjq4EhDbXOsQ/EoDRVzcDmXdY98b/qzIq35kMBK2uMWJX9REaSCVqKUhDZUDrVD7itD9X+GquG+f3f9uc393PftuFCUBJxXWWd9rHrfim+1e9kYx9mUCW3f09KxpRgwY91S1hjNDjZGvuVspp4Mr9XRIJABr2Wh0j2L9z/uWoK4UAKiS2v75LC1leiFoGUtv+3Veccylk5oJHS1Ow8c8pQEQDqJzfN3nn/G5xfy8PIHfDpVoA0LLr8u47FiMMQ9uLB73yDmDxbRd6VG+C5V280z1LOY11jLPRuSSS5S5uYMa2SbadFf/8pJdhh9X9ffPXPTWQHYWoDT4R1Uht+3dfcQF1fgec3N+t9u+O7u+Kb7uGL7zNyt2DO1RH+3dno3NJJAAu7d8d3d/tSP+Jnti/+4pzLoVX4+Pvi42Pj+0aHx8f/zKAQksmDzy+tA8aT1tpBH5KXozI+e/mAHlr3c19USsAQK/IeDMAYGDVwSOABjNWFALZH9iOOjONAK4eH6j0mIsKARQSQHCTczh9hweJUk5jHVP1lfhC4MzBm5BMQkYGUGHkWw8DKCoicKOwBnA+qzDvknHcQ7WeffCxAmekErWC9JzohGRRCcRT+YsLnPO7tvu71f7dwf1d8G239qhsza1W7m7lbP/uZHQuh1SG/ft4J/d3BWmLWkR6GpsT8nBMWPNFdLZ/9xWn9b7ClYHfAgMWuHg3z3yqWlXbVnSE9589FpREWkfXUD4sS/1TyaFTtTuFO/dYnwQuNvPkz/9Vp7GOMWdE1DxVLirUIJWkoaKMqNooqhBU5qykEt0nwRuc7vyuS3d+16U7v+vSX37Xpb/bqEsvBV16KejSS0GXXgq68Ptyflf08+en61RSmZSrXdMh0Y9dkcVSXr7LPADFgN2SNbvVu2D6bve5ly0hIl8ZWt/L7E/ndXhb2c2Zaoyp0XaF2PF1TKOIkJCQ0DSfp14y5s2edR89DuQNrFq/TqN8FUdWHajjAIC9Y7IdSZYmCSoe/3HP9kPzrHvXns1qf/jOjaotxv76eOsX1gegEpSI/GBo/F7i8vZrouHojO4F5HhcEDqGTZm7aNGiIbG++wfNbmkiP25vZs4DYwbPO6/mSJJDHCQLeiDdgWRt8l/mfjPJT/C9dXdiFskZFVPErQ/XkyxpnkPpUiLynaG1KkT3Ub1DH3EsFwjHb0JH9SWzpk17ppLvDjOmxd2DgO7JO4H7Vn48s66aI0kOcQBYMgqOJGuT//pi10tAz/c6WBfsdbMBDC7eJGxhdz6A8k1+lb8qKBH5zvDgYv/SlWjgYMOWQtOzADDrFd9tajPyIwBEB+0rmyNZPzSs40iyNfmvV3qEAVFPWe8VDE1KAVTGJWEL9aYlAVfOBcC7UYnId8atdn63qjpMAEzIFE3KnTlypVjKy3eZFwzYLaVgt3oXTN8Fa/YAPAlaIvKDceud30mSbQaTPIIeKpNyR9d0eX7spNpSXr7LvGDATkerd/uW4HMvW/bc+czwqBTInLYxpeqOpZP9mvm+Whdomh/2KFlK8mvb3ZadE4BS+H4dhe97TtuYUocmv/Qz7iOZrfqL1pldc1RbBf9YkVQPTxQHpBasufOd4WEpMAFvqztarfRv4inPvTbjOKZbdjIw24EjvxTyp5hV3/cEvO3Y5I/yLf8BJnywkLM+eeqtoavJm2OwNDClYMmd7wz39wp8o0MPoAW+mAApzu9WtVgKnEArdrl/ocWkXJsjTwnnJgJnERu5uIKNFGpvqghpBuz2ll1bNldEyj3K1pUdHwLhKyt++Zzs2JTc+cG49c7vFh09ORjY32igYlKuzZGVKwMUS/kzAXGZtxmw20zfFat325ZizS5bSu78YNx653eL4sYRJQsWVFBMytUcSRLiAGC1lA+My7zdTt5CU6ze7VuKNbtsKRH5w7jlzu8WbRuZsqdPosqkXPB+lyOHOBRL+YC4zNsN2Jlau69g9a5sCdbssqVE5Dvj1ju/29a0H6p3DVWblAfC+12MQ9OfHfIN2B1pws+d3ZpdtpSIfGbcRud3/GFt2H+fWQv6g9mX/0/E8fvMWtAfzL78fyGO32nW9JffdekvtOnSS0GX/k8AdEH/JwC69AuELr0UdHmr/wLs6DlGpD/ccwAAAABJRU5ErkJggg==',
    '1803.03670v1.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAMEAAACeCAAAAACu8TFiAAARBElEQVR42u2ceVyU1RqAnwFEVBAjQEtFjFwywMQlzZIyy1LTq5ZeS9vMNTPTsixNbbXF7ZZ21bxhLi2a5r5wTXEJ0tzXqwGSGwgqiwIzDPPeP2b7ZnDmm5mG1N9v3j/m58w538y8zHe+853H5z0a4SYPP3wZXPcQkYk37ZefKCIa3zi47hFg/0Lusbzb23rwRhmZefdG3xAZZH77zSueZJC2eN0KTzMwzDsTVDwqwjo25x8O9h9bA2BAy9bV/0x9tCMUzfzzfNyrta85ku0i/lXxJC6xwua5YaarRxr6vSeyp/E583Ndl2fK5Yf4XBGR2kCVCQaR3H+mS8GToakVjr5GBi09y6DILoOcIa4e+XOoVkQGP2l+/l6VQhFpP1RE5JFZ4+ZliogMyxSRorCGevujK28kr3C555L2gUDi6mLT87nxIUDbRcVA5PCPXooGWHPfZQhOzDzu2rXoapbBcpadTC0BKMu+SP4V0OfmoM8ssTtAe9Z8Vf7zl3QDwKG3LY263UfKnSRQnhwKcJt2q/F5/plggIgr+5W96mt1QDAXVEcyoP04KHxT1MRAgI1fDoh4I3RSlbWvH/845NL0Ff7DDg/psS9m8a3/0VgPSH8rJj6zGYBuXLP4FWvnNuKHhaWb+9N5AMi8lWOzX/q8vcMMci8HA4RyogsA1fzF+Lc9fh9w4Bd9wIuhsF0fCBwMiHVhJLe8/axI+VPdDSKystElEcOwp0UMjXvtOF/ne5H7OiaJnGSz9YBjISkiMoUVIl9HHRMZfU+5iDQxjYNZdQtENta67HAYnGCUiMh+JpleaH63iMhwpopI7BKDLG98ytSyi9EujeR+IiK7WSZSXG+8iMgB/ivSsrmxuXNEuYi+iuJK07aDiEgmK0SW3fK7yBpOWDPIDx0pIuVhsxxmcJCxxg950/TCSs15EW07PhGRAyIiCb2MDaVxnUrElZEcCNCyylrYfyYWIFazBogzNTfzA//Aq5Cfl5eXV0B22kOW87H3xZaZK7dTYH2zvQWBO3bs+DX8hMOzKIRyAD0hphe6fz7odMbHXQkH4gFaLc8F4O2In4Ncv6vQhJyE/xEE4Bd4DAgztRjfpBy6RURERPTmOKHWuWluu0W32pzyWUQGBQUFLX7TYQa1KAEotb7N6C9TtryWQ3NI22bM8QjA/HPrargwJ1vGR2ETiOEKQJn2TkBj32WpFgjiDq5aXho7d18MO8Cg0QDMH8hd3NrK+cU0tE4OQDZtLC81aABZ0c2he0FRIOi4FVhzYLEfh/2aufAb6ABS9V0gISIVYDePXeujb4uOjo6uQ/2mvwGUAEUznouBC7BpJ0FlkAHNb08G0K5ymIGmayZARlhLuFQKbO9ZAAXbJgRA868CgWOhd0HqrzP9YGVVV86i3Tlg+LxXT6gx58cMKJ/SpxvoCk35lQFSZr3Ca+YmHwaZTQlVqumBQ1VKzodxzx8gEDRvzUFgZn3HP8Looxkgy0f7k1f/fmDNqtMwo+EA4Mm7gFMp0wM43v/isKFDBnwd7cLVtFPW2LmLek/QiojIpoc+mtN9ok42dAmJeCxJZOcTtW7pnpb0UHD0U9br4/YO05eO+YGIJ2VZ/Snr3t87vN1beklvMGnyVhGRX9q9veCNtc5uK75PPHJpTH+dyNXY/iJyuMu2va93zBER0U+Ysf/7u6caRBKMXze+wsHXXuFkXWliGSDn8hsHqN4YnM1rZjgaHl6N8uNlzQKlMBQoOxRtGvxnrzTWOD38wrqr7VpYu1zckN+inenp0bSwdrWdHetbo/ky8GXgy8CXgS8DXwY3RvhPcpXJ9QiK83ZP66piZerG+BoK4vXtjpQ2gQCGpNUbkhuGAQP+LL+4K8mvoQt3dg6gDi+619MrxMvwykGRcw+lKtmX+jrZwcf8fsW9nl4hXquniYgc6KRkX3YR4OovrWnpZk+3ideLxdVtiNecqdX5rRggJgOIHO7qSC7POwvosi9ig7fK887bL+VMIMvMwshPTdFlmXsqiJdKOCNedafNMMDqLu5ci3YmRgyEuY1v+wp2tIicuP7TXX1fEI61jngFvmob1XQa9KsXmyRzexZn3L8T1sbfNm/WF1EpTF0QWu3DlqaeP4wr3dy//8I590U1nQb9692d5ALxwo54wfMNX3vwf6sWfQhwYPpn0wtwhV237iwi2oD3RWzwVmJvEckPf0lEDLE5CpBlYmHHE0XE0N7c08SLCuq9ICLlcTniGfHKuo/AB3RSgX055UW1AAKNK+qQQwOgYZXDgPEP9eLSYsh6LrLg7d41oZPfEtCEpLevc75vzv79oHne3NMUNYcsK4L0YZGOT4NSI6DSYIax7x3NBt0+9IAu5nW/7d2ygcX9NPQMHu3KjGbzkgVvmWJwwY/w7bO2ICsOoE10i9hRm1+yf7eBJYth8TNOTmRnxOvQs9M+O/zwph4GO/blxpxswVumiOn8b8oKIm1BVhhAUMq71Wd1etZgj2R6zpaSsppOPsIZ8Ro0JZyY5Em7NtiyL1eIl+gdNAzteSDjCWxBlgbgZLXJky8veGNgoqLz/IEwrOPO9L4eEq/CIw8AmombT3RRsi+136CqADlaB5/Xre6cbYnXAllHvodbRvU+bPn1jMSLB5vOOhTvdP5wTLyqaYoAiIpVsi/VDJpfApbXLLHDW2XGwRAw6NtojS3IMrGw2cWArq25p4l4oRn6Y4LzM9Ux8arSdwbA2UsdFOxL/Wqae//kdVPW1g1+5LICb23qHhbW9YKIyJngPFGCLDMLW/HYO0vWvjVTdpl6WojX5ehSldsKx8Sr9LnBWw/MfT5Lyb5cIV4XTzWqeTAkosa1MdVpM0C0BVmF1QJO6Bop6LiZeF2aPV5tWnZCvE7uLo5vrXHCviqZeBWkJ/Bpn+ibeIXzSavCotLov/f/9L0bnfekJr9buR9R2dz0wtnYKjd3Br6Vvi8DXwZcD0PqBjO4vJbBXzG43He8DAtOlxgG3Wmre7nheLlkcHkz7ImXAnNZ4Zc7jpdLBpc3w554KTCXFX5dB8cLjx2v385YMJdV93LT8cKRwWXVtcozC9H+vNPKu6xtKk6XGvFSYC4F/HLoeLmQQfpTE1M+SgawUC74olvyiMGfRvVYZeZdljZFJ9fCnngpMJcCfm0/XxtXHS+7UBhcVsq1tGq+6BuPkXSL+2VpU3W6VImXAnMp4ZfrjpddWA0uha71ZJyI9K0nYna/LG3qTpeq4yUnB7wexKPn7XQvB46X6nyQnTbRPG/sLQjcAYSfgFqXgAAj4IlD2abo5GJUIF6HhqwKHzpkU49UP7p/Pmh22cKuqeGAA8dLNQOFwZVFZBCwuC4MX3g4tmjLx1h4l6Vto7WTi1GBeA36NJyY5PcmbegCo3unaF8bT3OT41XVgzlZYXApKFfkiJmR2bP+YeVdljZ1p0uNeCkwl1L3csfxsgmFwaWgXFs6zPtw/j+UkMncpu50qREvBeZS6l6OHC/1kbyt6iERwyiWiKwNPiAin+wV2dZ4zuLlm/JFROJeEBFFm7WTq3EkKF3E0O4DkdzqLUXkpckiImce14qM9TskMqmFTuTYHYOHDBncP1rvwV2F1eCy6loF90TdUT8ooM8VM+9StKk7XSrEy4q5FPDLTccLBwaXmXKVtJnXFgx7nuurrOGxEDBVp0uFeCkwlxJ+eXOlv3zuBiMqXb/6Jl2jtTicBVC+8bHrf1/oIW3Z/VmzNtVPbu4wTHOzZgC5mSX1Gt4I9+Y+4uXLwJeBL4ObiHhVbjipaoQLI+eE4gXiVZnhxPHKHTO4NedFvEC8KjOcOF6lp/TTjRncVMRLUdVYtYG/6UW3iZec3JJjU9Vo9L0yLgLaU3oA/YULFGca7KoZr131SOVVNTrIYGvrXwxLJpfDxu77St94p8zse+1+4UXD11/tvHc2rEuoPfn98aseGasDdGOSQ1Y8fNJGC3PF71JzvKzhJvHa6r9HZEX1ZEVVo8X3enW/yDfVL4nIXbUPiFxJ6GawqWa0amEu+F2qjpeIaRw4JF7X/A1keKcECG5Ur+TlvreAZuiSzVbfK6s5NC3OBKo/Fg813l2zHGoVXYWO+9NttDAX/C5Vx8uW3w7s9KFrM9qZo3cDnfY3talqNPledwNBiiK6jqy1rWa0amHqfpdqVaNNuF7V+Ad1jP+wqWo0+V721ldIyEnbakZrB3W/S83xsgk3qhobcdr4D5uqRgdRWNSkQjWjOVT9LrWqRmW4Q7zqxu4D0K5wXtVoLH9MoYtNNaMyVP0utapGRbhT1YhmXuo64N/1lFWNZt+rDCgznkUp2VA8sVdPm2pGZdWjut+lUtUIaNECblU1ioikJrzz43vLFFWN1nLGqJ6X32wRcs8IkZb9xn29pOMErYi1mnGbbdWjut/l3PHSPv1InZBmPaZ7Qryy8xv7qVQ1topN+rPIVP6oqGZUhgt+l4rjVakr/VaxSdfZ7/qrK5wyPdfZ7wI3qlgqxKaRuzLTmkc4WTydq/+fN6vewLyoHH/0fn7X1e/yES9fBr4MfLyIv88HM/j9vRm474M5cbwAts7+8W8mXu76YE4cLxGRqzFd/27i5a4P5sTxEhH5IKar3LyOF5DWMNwj4mUTZprlsMEqeil8MC84XkDpT/08Il42a0kzzZrVKqp5UlK7+m038s96d35pbbAUNip8MLzheAH/GqnxhHjZhJVmZfrPEDkbsFZE2u9QNphFL4UPJl5xvGTvfJF7u7pNvOxoiIVmRT/6M0TWWgmG1u2VDabCRl5o0QHo695vUIF4KUoZyxY+7xHxsg0FzXom5TSbey3VsuMBu0274ow+2EMezDEVHS9rKeOXL/t5RLzsq+gtNKtHte/YMq18Peu62G3aFWbng7kRFR0vSynjUe2t+fn5+rL8Ig/28eJae3MF91w0omqNpxY/LkF2m3ZpwMYHwyuOV+6Z8cDRyPF3jHZIvNRHcqH/CBH5id3rt4usZ9zvsqVq0ha7BuNuloam3YxD070ZbWCCiMhnYXqRiyUiupACERF5JtnUXruriMiv4wwi8sEf7o9kJc2iU+TKBDpETu1g12AsbFTs6OVO2BEvRSmjcb64WuiMeKmv9AOafKEr+uEJzcKQPn74nbk3Ec2FmEdtGmqNOpjxS/k90ODBMQVnvmm7bNuuPm5kENFoQpuACYGf+sNPcb2g88L1NQtXLZhq3ENr2EcXcv+b3b7z0T179uw5GDXck3WykmblBwRDITUdYi6FD4YXHC/fSt+XgS8DXwb4dq7Ht3O9j3ipRsGhvJodTY+VGQY/MIh/JWRwfvG3j3c0PVaa42UiXtvHdYmoAfC0n1eJ16O9rY+V4XhZiddXpu/7oMe7YDm49VY8ehyr1iVBwoMjl5qeT0m+6Eeff00wfenpABybE1VDg7z7zY1oCjrZx8tKvPwGAywYGO2da5FDBOZJOHe8zMRrBEDGr8945WpqIV3eCeeOl5l4xQCGkVM0XrkWLVy2sWnr8332eGkuKaIGgL95r/WqsRcBTpEP7AtXbDP9Xdwt3pnRFEKXN8KZ42VDvHTjhnlpTq64Pf1fCmeOlw3xWq2J8lIGFben/0vhxPGyJV5JDb01JzsQujwNJ46XDfHSb+3kpQyKZgwzCV3B93sjA03XfVbHq3oQbJ+WFErBtqkBJCYCLIv9EuDolTAv3ZsqSVdZGWB69DicO14m4gXnbLb6+ytui5V0NXw57fTOJmeGp53e2eQ2zzOwJ163Z7a6NC3tZ5MjaSJecOy7hx/2Fi9yIHR5Hi45XrpFXWv7iJcvA18Gvgx8GeBjdviYnS+DSor/A9O8Z+ex6r3ZAAAAAElFTkSuQmCC',
    '1803.07835v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAocAAAA7CAAAAAAs6H8ZAAAQx0lEQVR42u1daVgURxp+h0vklkM0XihKgol4ACoRA0bMGjRm442oi0c00WiMxmTjsUQTj90k6m6iRowbjQaNGs16JAjxAIxHvFAEjUaQIIoCCijXwMy3P3qOnp7umZ4eBt19+n0eH7qqet7v/aq+7q6qrmoVBBkyHjvs5CqQIcehDBkMiChRrgUZjxGJRGSniUXbIfGJJXssNprCg/8xPQCgIOaf7dCo7IomGFb9D1XH/4ceBcn9QxmN07tLF3vmcf64dzDxk4cFzF+nti586Zoy/amt7QEAMxJ6m1bBpdRBCplZWmNSHWflH0BrHwC40syhnjpLM3Im9W6XcT57hqOKVbt2LqLJb1c5ob4+GEDVrWaKOu9iAIBLewezIqtSzj7oOKoTdo0SJM/LL+0TICKPx0WfVgp9SivHhBqaPUZsU3nMWKcQF4cPkipr6t96GijZkZnRaqorlRx9alkPGKXz1p3IahHviqqC9MqrTwPA/a9gJnS4lPoKkkCmlypAa0Sq5yz+4UqyW1YggK9/vhAdO9+8DR4jC+4ubZWT2G7XcHR6EOXvdCw/KLK2OD3gd9Hk5w5vehT28lIApfP3t4wb/0NmRuupTlXZFW8PtzMlkjYtHjyz8/XPO/kv58RhXsKbcZrDU9/+uJcbc3x5xs2jzLkcleiPkh0Gckx4taZtpNg4DOmzYoHAeNlwnFLyVgnRP5zSiIiyMZaISL3W6RviSZ/Gq0REdDskhYiIkuCrNO6FGia5lDpIIGNJFaDlkLI5s0ehfwMRUd0AtTgbXCP/CVURkSo2lFRe14joU6wlost+psi51fEhPtUchWcRUTbiiIjSnCerTIhUz/A4RUREWx26cyroB0zWHd/HXqMKNMrjb56y0LZ3jOUIeVUQqhI/KlH3+42nUY3jcLXzTqISPE9EdB3xTOYKu0yedDZGaLzfQEREo8chxVwccilZNWAxGUuqAC2HlM2Zvf5tTQzMFGmDa2TkfCbWQ6l0GhHRaqwnIoqvNkHOrY6b6MY06f1JbAvj8J0JkauxRXM0ghuH6rOPdMcPeeLwoZk41ArYgXhjOUJeTfzSkuHxzuE8VWI8TmlFAFxx3yBzjvcMMpEeUAAAd3wnYIfori2XUgIZn1QB2gEFxpzLgxbmmpVpojrymP5TeADusfsYIfdEkwMdXsw+DwDYHsfO7ohjwiJLFgeM1xzOMhp7hro2ysCjEzL45PB69eDAeEuoXztaLOZ9ytiHo4BfMNQg0zkmO1Movf93ePQFgJ2jBvrsrRMrR0uhPJOj0udaRMYnlauUTcrhdNlS/5d6czJNVEfQ7p8AQDELjn1Zhf3FkwOYhK8BAIdfZOcWobuwyORHQ7XNFuEBQFVaBEBZXAZAVXqHKakr0l6O+grW55lFJYL45PB6te9ZTezT9ZM14AiqLy5D+SMA5SfTlQUA4BC+V9R7PUdA+VnYIsPMdjgilD5bC7wCABn9HUdVpIi+LtrhCEBJr1XnRf4CaWR8UrlKWaRczr7vn11hVqZwdbyD2IF/P6lEFDr3ZJVFdBBPDgx3T64FkN3VnpV5feewKcIif0aw9tDp38AvUX5TgKSg1uuBK+F+swDgxqjE9OVpALuC9XkisMv+fV45fF5l9GH+Hhp2oXb+wnoDQQdDWm9c+3n7dHy2xbP5slAAQL8McfM2p/cc7bWbc3tviQK+dP5X9TfWa4Zs+e3tEffl9lfFxmFLFADrl+d6oNnQfC9pZDxSOUpZpMaciQc+GtoLlttgjPROfuPIEfgvekvgdZUYcriM3bhvNLB5hiZdsJtKc6+um6gQ5imCn+64M9DveG8A0xJcAQSfjwaAq70PvAD8HWBVMCvPJGqKocz9IXXvIH45PF79PoG5Lb57ugWiZyZ8yxY0JPaZM3MD1xX/tv8YEH6Y6YvsE7fOoc/KbQVxpZxhNdR8abfOnQPdNFk7xgKRbfdXiZ77hBoVC0Z4ADF2yRLJeKRylLJIjTmbfYOJZjsSwtUxJn/79C53Z33G/ztR5EACvgbqCwM1SZ+w8OfV5V0UJngc0GBI4QUATs0YfwFgUs8XAIwBWBWszzONKzt27Lk1/uorAnJ4vLrnBQA1M8e0ABRvJB82EKRwv9Gv1Z0xd7OyAEUCAMD7nsh5bEXQtjYvn7I3vAY7gCftF41BFdpeWKsrQMdb+8eKjMMidMD5CqfjAHyvSSUzllqEDihvAODoySHl4eyRuDhxJSy2oXXfa+xYSh+9ZIoX7+9EkSMiKLWozcFY3XUTAKyNH3iuqzGP1quAM0xDrs9wdFApFz1rdDMpPpWobVtdBbPyzIxQ5hhcxkZyjL265wEAWbeeA4DnFAcGGgrqBgC9A3o+G/PKVME4FHiv5xN+9pBBRiEGCaX7uAMAcp9u17Zt2yniR8yFGIQCtHR2dnb+9n3pZFyphRiEoX5+fn4jYEjKy/nXsE9OWG6Dcf87AFBEb3h4QeBnosgVk9TfYNdIg7wRtRt4eLReDQYzxH5z49Svf/y8qzHlVXhqD3UVzMqzGIZyjLxybgCA3+AMAHZOVzi/9gYA5/S/uayNmagGgAZnEfdDZWTDKSfAG9fZuaUpoZFC6QHMn+1zuwGgFT+Ve4lyrjQlNBLB8AkznF+xgIxXamlKaCR21QFw5pDycjp80/MvL8BCGxr3dzMPuT+hRuDH5sgZTFi4eZKbG+dBe4mHR+vVmIUHa50BwK032vjqugoNrEkXXd9DV8H1qJIch4ZyjLxqWQkAgXgEAPV1nTmCFABwvfmSJQ+2zJ8SBaC8pYj7Ye3ZS/cAFOomDgAAyxqSFKbSAOV0AwBFnHKvOOeWNSQp0P2pNACo2yeFjFfqsoYkBVoHBAQEtDLNyaw3QvDy34sstaFxP/cG81LWsMwCcgZtXro2Y7xRwxOqGzg8Wq9c15Z9wpxXremzEYC7+l5bu2dOA0ANoK9gVp6UOCRUNwh55V8MAL38TgLAGQw2FgTk7ABazBlxGQBKxMShR+yRtsAfOVH9AVRACQAP3960rxeM05okM6n1gPk7FJvUpnwypHDeeOASgH+2k0LGlspVamiNjzNHMyE7p7/J1/ImqkM1rhwA1k9qo4lZ6GpeHLluCjE7Uqe3AgDa29/Pw9ZSIZ7hny9NUgPATuZu2P0+gD0eNQBQXwUoktIuA7QONfoKZuWZbp4K45RWjoBXvc4DgOuGnXmAauXooYaClJUAgHXVAJR9AeB8qKj3yxM//eV4nyF3iLJeDYDbn+PH/anXO7eJjNJnhge5efUfc5qIKDPMo3mPAiKaG+Lq1m+W4IsjLiXRkYgFW+YflESml2pMS2SokMv5a4ynWwTzduxGgsl32ILVQd3Spn6ccmTO8HIiogNjo/3d/aPH/kxkipx4Fh3XtPiYObg4ItjdKyaRiNa7LMifZ4LnWFj3jcdTZi8uiCciKolc8uPKg23cBj34dZi395B7RJkvrN417zv4jdRVsEGekJ6LI55x8xjwLvHKEVSTFsz8TR2wfMOwRKWBoJRYd7/Bm4n2Dl6YfPCv/yQiorD/GFcJ3zrYrIvUM0QBNNFCy6JHQQqpZI0r1UIb50Ip+6wqtKfC2upI7eXLzbqbSsNM9rOvnq0M7tFCmyq72cXjkrufq15KUWlXda6vb3N2BbPyLFwHa1KOqlOKZmL9dnmQA7+gyuYO15RdnAHgVp+bjkZVIq/Hhrwe21qs+mONBWcvdFkIOQ7lOGx8Pcqo758SffL9wenN5X0BMmwAp3XTVWLPVU//V3OBy0G+H8r3QytxOGe2yDM3tokVikE5DuU4tBZVYpc9VrvwSrD/EEC0TT2NfmLJHouN6CcrDhtJj5PYEx0FJBBRlNzDkfEYESUwfyg/l+XnsryPXgbk733xDbRVUoqshbrxiWyoVoa1ML8uMvODWD9XABhnhwmh4S5/nHzpRZ4iS6Hbns4lBQAcW7dTokMcNhaRNWoNcG/2Bk/NdvbLbvbv6UeK6o23nKvn+Nnoyvw+t95ull8TWxWCkYgthTXq1ztb9d0509/7Wq8d1BCRPwDHxWq+IuGvOfHmsranc0iJiKoCh1hCxoYhG5tIlFqzNkrmTQvHHSIiUsbGq+i7kBLd3uG4pUTngm5bwS6Mh4MWK2nrMIlWG1uPkYhZl4huDzgpWYL5OJy94aeMzMyM6HwiGrT2g435/EUWOsrans4hJSL6WHocGrKxiUSpNWuj9mbDak0cLnWsJKJ+b+i+peBZR0TTRtoiDtUj44mos59Eq42thyti/yoioosxkiWYfy7bTQOALVMCALScIVhkIVjb0zmkAE519JV8fzdgMyCyQi0LzfT7dJJC3AH03fCZZmo2uZ8TgKjJPFO1VuPI7nMAttc3rVVBcEWcrgaAwDwbjlPeAoC8E/GWFUHK9nQNar+Pa5zaMiSyQi0vym+5AYDfoywmrUrzBIDWdcds0PBfevYAEBbRtFbFuo42q9aogf2xNhynBAJQz97KLCS7eKTBYbInb5GFYG1PNyQF/jXbmhWFLDZDImvU8qG5PTFX8tXnAQAlD9wAwBPXYhu94elox8Ikt4rJQU1qVbTrSPjHO3s2/rYt2YZxCADbuzErLi/mzlHs7Z3agafIYui2p3NJL/i2syYM9Ww8RJLV8j2hnysDgJsoh2afiisA2BusrG8kVJY9s+sju/z+WwY2pVXRrsP5WFxmSJ/Djrb9/wKUH7zJHHwbp8BrbnP5iiyGbns6h7R+a4I1VaRn4yGSrpYPS3OLAeUF7a72WuYlq0LaXiRzn5s5NdoOHQdNq21Kq6JdB5SB79plDi22bRzuV7RnDkIAIGxPCU+R5VAEbUt9WWVE+sVMq6b39Gw8RFao5cGwT18vzFsxBJqxkDtUANAA98Zvd1e0bw+ge96JprQq2nVkT1z1yeWBqa+qbRqHmztqhp8ZjOc5xkWS4BN+9hCXNLfOp7y8vKG+/KEkSj0bH5FVao0x94v0o+/c1W4b9WJuSbVWbFgXhIeTNwA0R05TWhXtOl5f6YvAtA9/TYEN+4cNx2I0V0HFQydACR/jIgvB2p7OIS25tQhAbstFneZKulB1bDxEUtUKokMHoCBA0xiere4CQDF6N36zO3SrAgAVfJvSqmjXK3P6A1AkHpY+WBIxlXlR+63bmE1ERDGe9URlNYZFFk6UVijsC4moJ47ykBKRv9R5bBYbi0i8WjE2tPPYGX8uJyr32qQ1MKUXEdEn3g02mMde4lhHRH/DNbZVKqsRabWx9XBcV7pXEBFRfJpUCWKey7eh+erFyGAAN9NXO6C0XaRhkaXPGf32dGNSQFVVKfGy0rOxiKxVy0GdZr/8gX2FwJqOE7QG5ubmAbRnrr0Nbj9vuB4C6NDULiyrjFlbWhUEx3XHMWsAoOj+C7Z8Ltdrm2/qkl+jr360MgFw6RRsWGQpNs87E0HzBn1lz0OKN7NwKeal96Tw6tlYRFarZfcoJpVku8d2iZ4DTLz84ELy+RRHrfKumyeva70s8D1bNHzL1BnKgK8CV7GtMmZtaVUQXNe/mD59XIvTJ750ghVrEM0thVRuG+KvOcw95R3hz19k6UJL/fZ0Lql1qzZNsIlQa9HK0LKU8p4R7Hnxez9WRZjZVi953Wl5eml4iFSrja7HSMT1M9Uh4QqrYlBejw15Pba8HluGDDkOZchxKEOGHIcy5DiUIcMA8vccIH/P4UmQQESJ8uUo4zEiUfM9Bxky5P6hDBlyHMp4MvBfPAhS0rmp7N0AAAAASUVORK5CYII=',
    '1806.04450v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAU8AAAC3CAAAAAC3EIPuAAAbjUlEQVR42u2dd0BUx9rGn6VJB7FLEfSqsaGIEDsGiD2a2LCkaDSxokYTzTVRoibRxCRqiiaWqFdjv9HYEvWzELECdko0QrAhRUGKwLK77/fH2d0zZ/csLLiFq+f9I5mZM2fm7OuemdmZH88rI0hmQrORXCD5syYbEcVIXjCJxRCR7HkYPy34IaX33bRmp1tQdM3ezlapULR2T8uxs4dCYR/0bH3ie3ft7UihaOwL4NYjB8gb+WivqU4WFTV66WnHT4HdjIiwASIirtK3EW2ARhFjycqWlfW0LQg+5I6IAKBRxGYioq/by9xCtvLXyl6shTGm60ptLhovbwPGk9VteoxJ/Ul0HmhSziWLa/8pvDb6Kf35PzB+Jpq6wdBuyNjLJTc17y685vbMz0fnT5u8yfeA5dx4uWKWzMzzkYhlPnCAvL437uXalXUoSSkqavGv9BTfF2pxV8tSk9xaNwOKkwsLu7udrt3+iU6Nsgu3bPwCvZgaMqYQystFRf5tChNKW/kD6UkuQZ5ss0+OTsDdc2jsJ9pVNZ3xqv8/Z86/COBQyVD2AdWmvFJU7NIT2elFxUG+bL/VmY9Exs+toUDTjURrnWsFll5sCvRr+ebUhm7fK4hItb1+x+8We72eTxebAofDhyA0XlgjNaDBqlVhzvNUfA0FU0gFLWwQs7n3Z30R/WTcuC/qex5nm13RrRZ8u3VbJd6VopqTxAogiojopWWCBySiiRhDBS1sEEG0rgGwS9Bv9ecjb29vb2/v2ur5SBWK/kRE4RuJSNUMAWlEd+ricyLajHbFRAfxFpGqKXqc3AfcEdZYBhSTPAA/MTXYQqIBCJ+qpASgdxzRVLwobLYZ1PORWFfV9GeBO2wziC665gkfkPMn0QBEENF9zp9Mv9X2Z2ZmZmZm5mrN/L4HuEx0wV9ORBSMaCKi+aj1gEq88AURqbztHhMFI4xKP/tZWINuvbKAiMZgIDE12EIaA6cSoifAO0S0Ch4kaFbjT/GuqrmImQ28T/TGdCKdZ+H8OQYRRFQC7NLpt1IzMH42BAB4arKDWqV8sRVfvG/PTGIdUZYwIPkRyhMANLh3KwhAN9Sap1MDTfcpUq5m/oU8gK8hLEQ7R8AeCAFgj1JAp1kAemXarqpl0ctVa2MKdl2H/rPomtizPN18BNjMHbtjsfL0ZrbMC7iLTCDZFsDwKH8AqAO9GshZ8IvPm53PJHBl6hrCQs0qRbtY0WlWrKzOU03DTYbtfLzhQf9mIs+iNxuLPMvT+hOj599Zpop2YovuAG3RDggfb2j1dQdoi+LuN8avcsB2QQ2dQj3TaRYlUzZU3FXVl0w7sfzxbxU/i0L0WUyy/rR/Hxv2TtHkSgHgJHyC4NsZsQBAkzME9fkacTfwgQNwT3BZtJAxtllHlOPJYRjsqnrWuQvSm3Yz9CyOIAB/6T2L6dbz4+vI39YOp7uvA7Eb7bc4Q7alzpbfgPJFyT6C6nwNH+AscDkDWbn8ZdFCdoONaTYUSTjTCQa7qqbNAri1vNizhCAXKPnaCRmqqvarO0FddnJ3dnN2d46jmS6urm4uru258kW1MtU1gjHw1QGRDYbcICKi3Kl+QeEd5xRRnIuzm6tzrF6NwyF2fV4Z+6CfK4byNfjCb/Kcnd1cnFOXuzi7uvSkYBdXZ9c1TLOU20/WsV2qoa6qvUlR3ixAzqWYZylxdnFzcc6h8mi3rsMi/mwAdGf7rWZXorZwEmn9OYNKclT8pSLdla5OjeK7pfrtiRYKTNtsUZbKYFfV9iflPazoWcqzHhHdyy5UVLFfY+aj86Omzs7+9oJggHFkMi5iNzE1nJ1FrosWCkzbrItLxV1VzzwrfBa7+gAaV71fY8bP4+m/K2dMbiptvpvoaCWrt2t5/wVqz5/pXWoDh6IKqlde4xk+PzKqK9X9+g7atMpWRgr7impXWuN596d0vimdb0LiQ8xmMRIfIvEh0vgpjZ8wPx9SnK7+feNf2U+nvKRcz156pfFHspqPrvPrEMmfnD3ccfqEzzhn5B5rvCSwwlvvbdk0WM+f87IWNUyK8d01BGljJ4+qmZ85b01BSfm0lgBQuPJ2ZrsZDUw7H+kSNxhLRFQS6RQvvKBaqVMzPEr33t+ClUSk7B9MtBdvUw0x4YfMmZZD9KXDUSLKGXmLHg/zOGuuroiI6I76EC4OETog0USdmn30/DnsA45oCSZSJRTVTH8ud9xJlIOuRDQ5nYgKvQIUJuuqgvnIV71BrbU9lX/b07g97BB/QBbsUjNf94YEwAWPABzomge4hqWnWmJ+P4F+AIDbx2+pAOAaf55IN09kqX+s/1PC3tNi9+8AIIsGlLmZwvbk8UnKmuDPkYXDgdMYCMC3TA7AFdkW8Gfih32+BiCffdRtT8RNYMe/S4+9/vpmADgZcly1daESwNEvzkeNI5YM6h/xxVk5wpASUi8aQJPOc5d9FVY3A7TmtSdp3U/XBIfaA/KvO30M4FRmAwBX7dqacz66g5Bly5ZENVitIiJa55dCNKuDkohaqsfPk7aJRHucj1Kf0O1EN3GMuXm7J4AG3xERhQ0lUrSXEyU6LCb6wfsx0WHPPOuPn0Tn5oRM5Af3C5hlifkoo3t4GhHtrp1AdAA3eH+qWvchoqPtU6hPPSWRwl4w7edtm9gc+IqIBgwleriA6HGzcAXle0wnIqXXDzXBn6T6q+8rOep0abvIEtN1ZXjR7vdbk74Jbhg6RJZ+9Swe8xfuJvcFEHkZQGsbwNahmL3Nc+RIih2xcDx3nOAQBZpUsMUWFx87xAGoe6Nm/ABtscW73zlbbsVcb6+jRX5vevW+cQBQremypU43tvxvNY0DgHsQZpbZAQCyXj8VXuLyrq2xftvmRkAG6js6Ojr+MreGTPJ1QhIOAwDW3z/kYiH+0xVZwJw1l5ohDlDJZACwfjya447BW3ZHAQD6QDvrX4+e2wdx5a1Qp1PN8KS8u+KcA+CFmwBw4MovNrhu09oC38/Sk7JwFK54qxmQDRw5DcdyIA3wbnsJAMrElqPJtwAAhWiv2Q6I6rAYuObevvFRACjbZ3V/liZczQZwB+0BnD2z0gb4rZY53/fHkANA6eTbiwJh76QAcM2+JNMLHf4GCJCtPXsIwI8+kJcDoHLmfVeOzgeA1eO8gfJiANPvb7NHybqGjmsPXAWw0tfq/nTvf9wHuJ0U1gNIff3h5EkT31jnb76twZR5V9LdI52Qf6/BBxEA/vve1MDEAesuhX1qmxY+ThYWBuDc1H7tU1s3WnpK1mNe6qb4uiFrNIfZgd/s8O/ksO/2zx7xn8ahy4br4d3DKPvELbk9Tnz0Usvrvfpbf/8zd3ZgF5pdd11DIPgi99RXLLjVqkwtb+1ABR4Ayq/5qynzB/ktREeKxGC6lqAMDhIF2+8VtZChJuwnX75CQYEy6XxT2p+HdL4pmeRPyZ+SPyWT/Km2MEg8mLRekgwGeAa3JprcP0VAQIX7WY+v5bqHG9ubOALxTPvz0Y6zp+gfNTmeHZoT3vXdCv2Z+ct/+hntT1EEwgqWPf0nDwCAau1dxycz6wlSMPH5UXoUPlcnV72Bm5Vu8fceasQ5gIaF0EcgLM8zzH43BNwf/6hGLSJKbHGfTZn8/N2xbejP3ABORe5wNOa4sHLLSa5CZfOaW/Sq0erkvkNzgY69prMpM8xHE/7+EwBwppvJPsSemjPE1Wpiq0lu7eYAIGz/EyZlBn9GOa/n/NlFU6JhGgybhlZQ5GRBka4+7lCmF6Bs72kBC6GLQFjRlEc9AKBR2Uk+ZQ5/uo/Y/RhAgZt6j1DLNODbnn4Nf8Sxxn5dzwlHYQ2tEBdUP+b3Ly9wkMN3A49Oe/dLv8FKhoXQQyCsaDl5rgDggRt8yvTzUeZiOoXVRLQhi6bijpBpUI5xziFV4K9M/QFDBbRC1/CNashhV618UrSYTbcYFkIEgbDC+ftybj66gZlERJfxCZ8yCw/WrcV6ADn11VnPwmIg/PItADY/B76u+mnJazqL0HlD3YFIm60A3K69AQTYXwd2tPCAbdAOCP6yLn24+moNsFI4AIAMJXzKLOfFsglzrgZebafJskyDw66gMaG6p0ACWkELOXg+AmDnIawqgkBYb6bn0AEF3PiUeX5vvmm3Dkde1s4gLNPgs3J7e93qAlpBCzlMyb2OwhNzdFZjugiEFc2T+zqWwoNPmYdnaDBwy2J77bJCwDSUnvho5CVvYXVxWqH+tJX1H/zwqja/fjxqlnk0zAKABwjlU6b3JxGA8Xvf/lpTULhispppcO0OivnYN3HECQfBLe0bH30bQNnhQWzpiZ58Vs1C1LB9pwGXACDNK5hPmf59T0ohoG+jQn8Aj1EAAdOgmOfexOY/STOZ1Wh5OVhagYccmnywZuueoxxJpmYhRBAIa1gZygAAs5LTAPp1li2TMvF6KaWPl2un74iW7ib6oncdtzbDUmm379JDiy9O6fKh4qOWzh2I1jm5dlmqoScHedUZlEjHu8zb9MFBotOveNYedG7jS67+w/Med/Br6utoN6KIiG41+WThScFVq62Xyka/3NCt9eDlRETbw5IezX5dLkg9lRm31cowDYb3jnRphZLQtZ0BVeJbUTECFqJG7SdnHyruwtEXfKqGbl3/uuYPAMCq3/dL+/MmsKDrGQCgPNwXkt6FKSx+WetQ55vHek6WSfoMJtpwSC/xCbCR9C6kl1A634TEM0DiGaT3Xfp+wprn7wCA4j8S8vxHNKWdUaYnEEoe8una+kf7OSm5jTsbuDUtPfdFf70kalr8I72z8jUNx8XnnZ+xenMwXZvoaOIT8+tTOqD2tLlzZw7zw3L9y+fH28wwdOsv/bFHmLzVY2t1zjuyotR6iQWL3xn47wdERMqfF34456Y5/n5TNdnjPBERbbALrhaBoKfjoOsyDOYUu8eKei7QoD/pkdaf6qRxIhAGeQZen0EVfZXo/ktnzcAzrFz9Hber+lZ/oDoEgoZdMGTO6lHG8avsKvIR9rrJQQnf4il4hgVLmsJ9g+1oJQ4GtAMarZhv+vkoZ37AGPW0ON2c7EK5HHU8n3rmroYIBMMz8PoM5+8CQLM00/tza9EATWF3V+gQCBquofzBQ+QX6ZAOatUGnl1QMw58ZdZOHQH6qPmHtIcAyv5RaOfDDJW+qEPZPe2yR5tU5mbqIBQ8Q2GE8foM3t+sUAH7+5ven8fQSvsvuRYAQyBouYaDgY3W/vCdXyxDOmhVG7TsgoZx0FbW8SeAwWr+IX7c26p1q0+/uIrz1pI1f771kZzFJIBbw2NiPz8KYTIlpF60EKHgGQpjPj2vzzA24L1ef+3b8pnp5/cg7BSK2PAEAs81qFoMictsuJ0lHbSqDRp2Qcs4aCpr7Bp8x4zojP1cTs0/zLhMtMH5EREFN75HpBw+SMViEilusUS0FHsESU4EgkEoBAxFpTwDq8+Q0RUOPeRmmN9DsE3oT16EgdFqCOZkv/kSXrVB7U9GkUFdmffnUFJm9N4vaP9VIjqLRCIKHkVEFI/dbBOdexIRpWOPIMnBKewjDmtHRFE+xvEhrD7DzTfed0TvTNPrM/jH87NuXm2wBALLNbTTIR0Y1QboMQ7t9McZv+kEKJMCNe23AeCIYgAcqhFsf3Ao38SDczGaXx9MEvqQhChDUaFx+gzXJu6rO2nikcFnbUw9fvbFJW16qZBAYLkGLx3SgVFtgB7jIHZyFNQGSL2gbV+Xc5C53WSaSOVBg1QR5oC/WZShqMjU+gzvLK2LZkc/ufCHyX9vRs07WKb++/p8nass1yDTKdFRbVg/nmUcxDboGwM4Z1gXgQpaMphEU2gJHSaJyhmKykytz+CT1AOALObYjf6m/n66fJ+zQuOUNwVXWK0GvRJGtYFjF4xQZKAfW4kVywHgrKI/04TvC+cBoARgkyJ2oufaz9Yb706NPoOTrBAA4NfW9PtLw1Yu2EwAcEjWUkAgsFyDvAAQlPCqDWp2gWEcuMrQEYBAyax7tbXtlwMo5973+CxA9dWQ15gmZGuOXgdoFUrYJNQiEAwkwTIUqJRn0Ooz2EetAIB7j3qaYT+ETga/uClu67tbScgn8FzDwf5u9fpuJGJIB6KzHT/auWi3ll0gDePwh6YyZ/GD/OD86pgxw1/0RDjfvt9reXOD3DpMI4rMmLNmy9D5ZcQ3QUSnei7fNXsH6g1jkxcGeXkN2Ms+IstQVM4zdOR8EEhU+ta7J6+sGZthJp4hJSHPJ9IdlXMNghKNaoOWXaiuIkNGUUs7PUziXm5rVXLduk7CJCpgKKq0n3wz/klgiEzan0flDIW0P4//VYbiWTs/EmUopPN3mJShkPwpnW9KJvEM0vsuve94znmGgvv29qQob+ShFz+imPlXtnGuio4Dp+FQJSUHS5rKxoz+fLA59rTrkICRHrrxI9A0L6yBw8n0Ft1LH8T6/10VHQdOw6FKSg7mdeB/k8ttojVaDCdX7YQ5+ZB4vCYWP0LpeYOIvsIPRHS9XhV1HDgNB6OUHMz/97CFL8+X0+ZBmuCbzQaYsCsxfQZmUN0cbgPAJgZA3ojmAGxhA6BN75Kq6TjYo4aIMwCgcfUX2WPhWXV2uSXnIyZ+RHYHpjwwGzCLjoMF7PjuWQC2/cblzgXUtaA/mfgR9iz11gOAmI6DbjwKsRgT+Wdj5RlW9OePHh0AdOKeufS/oyz5/eTjR+BfQUx5lyaAvo6DxjTxKERjTHy9ycPps2Arvu4nAu58tORDtRDDt9NlFuA/tRa6ddLx42jw8TTxy+M3bpsE6Abjio280BGFY7pFrv482R21BqazmNJf+08CIces58+Chy/sWmyT3mNTBIBLdX0tu56PSt82sXlW9NcwRsdB/QWYEtkRcG3uw6o2aC3r8mVANtaK/sS5ETYIePndUqB881gL/j5KkgPwHPnjXyfqLcw3oOOQcBVXdWiFu8ltAERefuHiY4e4uLgzwhgTof5BbWcem2A9f7rAzw9A+7QzwPdTbSzozw12evEjUKGOA4TxKERjTDjGLnD+IfJNldX86e7gBQBOSEJyWZ38/HxFeX6hRcbP+zYi8SNQgY4DZ1qyQVS14abTwoV5mz4YH2Ytf9q1KwYAJeoi5+7HAJLrf9x0lgX8eeM2kHyrGcDGj6hIx0FtGrLhUD8x1Yakv99H7ZnnrlvNnxj0qdwByEInhIUBwO6238MC8SYyRnoL40eoJXbKDOo4aAZVDdnAEA3l5YD6v6ueAJB3tt4AOsnlMECHJzTnssriAnOulxIWX8ax11zK751T9gIcPvuAix8BADi45UGK25K9DSdEAKkz4+Whb02zm/Ev4MtjiW4jWn3aUt1E57ipZ9qntg5B/32TXmp5vVdQ/KcJssExyk8TZINj0PLzNh6nellxAVr/yBS5/7pm33C5yZdxNbL3HFhmq7XC+BEVmDYehR7RUOBkd0Pe3BHW3E/Oj80NCZT256X9eUjncZJJ/pT8KflTMsmfkHgGab0kWZV4BslMyjMYaWljJ4/6n/nMuqElBHltKAqYg2cw1ozTSaiqkIN5eAbd0BJMnpFuMJ3exTVUHeNQJRRVo/Osidbw516PMiJ6d5hIvvQfxfKn86eJ5qPq6CRYKwiFbmgJJs9IN5hrfueJBF1tBlYWoXKdBBFFBz0hB1g2yIR43qz+5IkEPW0GVhahcp0EEUUHPSEHWDjIhHgepubr2PGTF0jQ12ZgZREq10kQUXTQE3KwyPipG1pCJ2/W8ZMhEviYEzK3W90aZkaxkSUAV0BQoBdrQnMXE7tCJFgFLBlkQjxvvvPipOaMxoK+NoNI7IgKdRLa6cWugH6wClgwyIR43pw8A0Mk6GsziMSOqFAnwUs/dgUAA9gDLBBkQjxvTp6BIRL0tRkqNBGdBJl+7AroCjnAkkEmxPNm+37euM0IJOhrM1RshnQSBO0YLeRg2iAT6XxoiUelwryZeQaeSNDXZhDGjqhcJ0FP0UFEyMEixoSWyPXtLsgz0g2mWi+p5RNGdLPFe6xAgp42A6vcULlOwh9iig46Qg6W+nsEPrREcdvXBXk2FIW54k1oiARjYk6gYp0EPUWHpxZyqN5+sm5oCdOEmjDn1nXNiTXxbOzPP5exJsz5T1djYk08M3oCNSTWhKTPII2fkkk8g/S+S+87pPN39Q+GvX+T36u+z/IHnzI21EAoC2NN/BaR36CKT1ocKCk7GTijxNCvVANhHoyL/mB1PQEioof2kw2FsjDWRG8R6UoxJDCPiOjJyz2fGCAQDOAL1aMarOLPNagrNxTKwlh7ZJw/FyGRS9y1n2iAQDCAL1SParCKP0eMxh9CEYyq+1PsFv35KGtJV7XMqPeQNcniBIIBfKF6VIM1LLPuG9humfl9a4lWhCacNgBQZGfjSboKDIGgzM2ESKgIrlgfhhAPOGFN2zk8os4efuOYD2XBhM/QABmisTXY6BcV+/MEf8bbDCeAQx0bLFz88b6X58i1BEJKSL1o6IeK4IrRpPPcZV+F1c2oLOCENe3PHvbDH2u00fn4FWz4DA2QsU8stgYT/aLyeBO/apKJqE9E1KrBFaKijgNVWgKBwxf0Q0WEDSVStJcTJTosriDghNXHz7SZRLFQB3Zi4lcwsAUPZIiQGIKQF5WMnzb8wX4J9/V17hsIuCw48CtfiQs8o+YXMtoDLzxJVxc/HmyPghHd/81wCloCoqbY9pFAd5/9HDkwLqgngCiAhTYYIEOExGBuqXQ975uYo91ygx9fHo6DQ3Xr6oWKAOAQBZpUsMW2koATVrX9DVOAgLv7R0IYv4KBLVggQ5fEEAt5YXD8jMRFTTIRkXy5m9tNiMd5EGINrq2xftvmRpUHnLCeJbf09fHxGc/N8Gz8Cga2YIEMXRIj1TD7oO/j0fP+kDtwQ+sBu3f48oLClppQEpU87vXouX0QV15ZwAnr2bZZ7QDQkt/zPYXxKxjYggUydEmMCkJe6H8/ay/N3sSljlye688HgIhFfw2BULEVR3VYDFxztzCnYLxRUjsAkI2S7xHGr2BhCz0gg7lYQcgLkf2lSbPf/xMAkiaMWsiVxD4AnsQMeU1DIKjxBf1QEeXFAKbf32aPknUNDQecsLJtzeP+PxDrVWDjV7CwBQNk6MfWEIS8qDzexJaA+eevfNXkOyWXDR7173Vbw+eXaUNJXBjk5TUgWy9UBFd8HN0/mjehGeSGAk5Yeb10qpO7U4cMIpoV6OLaLVoQyoKBLbRAhiiJwUS/MIZnUJ5OVTXvqZFD7NR24+1CdQAILYFgnJmXUzDZfrI2foUWthADMgREh4GQF8ZsXXdqu/G525+vLpBhzP58ueL522evLpBR+T/dkZX/59RrSavn7fyomkBG5V0pYQuFjc3zdx5XLSBDOt+UzjerbDGWjf8eA8lM8s9mFE8rmfS+S/58Vuz/AT8AAXeiI2N/AAAAAElFTkSuQmCC',
    '1806.04450v1.3.png': '...',
    '1807.06535v1.2.png': 'iVBORw0KGgoAAAANSUhEUgAAAngAAABsCAAAAADcJJBCAAAeKklEQVR42u2deWANVxvGn5tEhIgQjUhtQUutQYRamqRKi6gWtUb7tdXSamlrLV0U1U8pqv1QVFe7lFaFlrYSsdUSa1CaBEVWsgm56/P9Mffemblb5mZRrXn/yZn3nvOeM+d958ycM3N+0RCqqHL7xUPtAlXUwFPl7hGSkWovqHI7JZKkhoCmIp7zNHfww6Pm7nqwveNOV0P1VquK+oynihp4qqiiBp4qauCpoooaeH+TMOGObt6ef8iM3ctWcT27kpdJz6YAgNS0nE4h5VBL4UXhb606GlfZyqs6d+XQjsz7h9faNEBR3I0b4viHlNSsrhXVeH1SVtHQknv5eGbN7tXHLNGUgyuQfzKnevcKXkAGJXL4zQj4j1ooHKzug80snciMprwVgTpv/3fG4BYvZ7goU4bqSt82curzlw3HX5kTRjLloTUllF0wx8kPX/VCUkU1OGv6veEl5zo6BZPJL2e74wqnJ3zmpaoD3W5nyd1nbRDs/XAM/a3p6+UTeORJDCXJa2H10l0UclGdaZGrw7K07YcwI0ljnzCS3+N510Uvhhmd/fRRDUPFXSv3T1SQKQPbSFPXP9xwhYsTftT9wCux+8QGOXjG85Q8+FUqr4HVB54AEDDp8kQXuVxUl33a1WFZ5NvuHgA8pgNAv8OfuM78zotOn4oTIjwr7L6UeV7Je814j66A5rWpbrjCxQmXwvcldt/fN7lojN2lK7jZ5WFZJFV46gkPAaAJ83WZN3frCGc/GXdHidnsEm6Io8IJmq4KSu4Kqw6g/64M5a4o6YTdEzesuQo87ZXynyEVoCmgz7iGvBsAwPP7b9lWZ8y5AkCXcU14Bj2/KxPAyWkyMzaHAGDIysLNNJP5SHco2aiwSU1jtwOAZiwAY066/EcbO1taWnu2MEVrTqQZAeBYvhh4MRuFv4ndDVbd1RvALUtImMtoL1l7OM9l4V2hNXOKhKTpkk40yPRsiOpdUQDgFb5ZuSvsTthJ1+XtT9BdtOnkS7+lWLrb7CXBmiE7E4Y0s1+NaQXQfr/XjcBLGTQ94YOd5R14Gz2nIK5N8IrFnzZIAH7ud7R40lt6WXV7IwNHAsubBi8FgPjw30xrZhjXTy3+dcSIby1WLIfLujR4YAEwol7Lr7a1D5ox6+0tPSfrAHB5/5up3fYqa9Ib6PPIh/t1iATOhAeOBdDwwSnzPoq856K9nd2dzIkrMfOOzNwE4Orw2QeGrQAQX6ONNdvaResBIHHsFuuqwTdbHjr99ccrn4GkzJ+zlz4D4LvewOJaF10URvwD036cNAsA9vbeNG/70xnAlz88dGDvuztmPE2Lev054X7cdbdiVwgn/ElEgzqf4dd7G3Q54KTr5n/tX2V2GKSdrJuw02/zI+ch8ZJgbU+72tO3zz045DkC+LTvzldHzW3whLHEWS15EgNJnvFLIDmnvCYX5zEgPf3i9tGNtpA0NR2wJ73OOv5w/3XS9PJwm+rCHyOp9ZpFMt7zCLm56k6y2WiZPfNhfr3nSBpbZ5JsHnScvNG+r4lcXDef/LlGrrK2rasBIOhTkmTkQNIQqiOPeM9yYCdihfA34d6dNA3tQiY0PEgW+O0mo/tJTOZ1WUvuDr1kVRjfZHSjH9gt2CiWMU0yrfUykP36kvs945wX5lU8UkCtz14y9t40sg+u0TCB0WHLyEz8TrO6t0eeML8OU+4KRg4kaYypmk1Tm02UnnK0OLk4G0nS1FXWyZ83OEOOb2uUeSlyIEl26f4VeR6/khsr59HQdAJTlMxqhcB7MIIk08ov8JovXLhwRaKWJBkWSpI3671Nksfxi7y6no+RpO8s0tTiMZI7Q884CzzO8isgzy0hybD/CBOrWOb5jyNpDFisrG3MXTv6fuAjUujta++S+U26GxzYeWAjSfJa0PukPnSxkCBbj6G++gKpybzOq2WhszeODYaTB85SLJMUy+c7kgb/uSR7pzgvzLWaK6Sp5ju86reS5IuhZGIcG/QykcewnxZ1ByH3lvpuuEIIL+2DjxmXxpHSU5YEXoL/UZIrZJ0cW/MwuRXnZF4SyjwWaCQNlRaRT7UmOaSeg/73cjIOZxyY7nB9uQzTitclB60B4NjlVgDQSrP1EVl14u3/8uleAHocc2515IzVL2H1eFHRHXEDk/K99wC455zCptUYOpQJg2eMrCEceg8BXypY5Ql7O1nVAQBTs58HvI4BU3OeB8BLl3CsIEpq0n/bw0W/1hePm/tduTQM6ARJmZBW+u8nAEfzowA0DnFeGPFt7gWyc4vxtnYggN29gZZ+Vy59ogF+r9QWFnVfIXdAljuuEE55Y7uYjn0AOO66jiHtWvZ4/AVZJw8coEk7sR/5jrzUwgPw9C4CalwH4OXvxuTiLPwrcnIbAAB/wAcAPLzPOKvuT9QpwVBw/yW8pa8uKvz8zuMiavv4+PisnqKoLesBQBO1rPCoWVGtBVau/TYYDuz4CM/7cQ8HC8dxUcEALuY3wi7JIx4AJGlqSd+t1fSyTkytZWpW+vX6IOA3v3YAAjycF8aRhwCcQHtsf9gfSP8jCqjpleARAWB3uA8kagAw+LjvkHqL1oUCcNJ1PgnvVl3c4xmTtJNNyzuvqtXVsZeEBhiBMTmnULhrspJXZta5dlFFBp4GAJrgBgDotfc5qI4GAPfjL6lu5UjYHb7cfW+K9CVWQWEzNEetDsrbEisUfwzW+TVOjZ3yGPbo7e3ULgCA9CvDzPeFK4MBYC2GIj7CM19y8eyaFFelD0bIRq2atmWwo+X9wP7OXsD55nBemGdeAPBj5YiM9GcB7NJEGLS+iG9bE7j5w2zk3xLUHt0MWl8AebXdd0jxrreGHq0LJ113vsqMGblfTxoZKenkycuPNsEewKSx8ZKsv15dVDtj8ZNujHj1H/gdgMQXFSDtA/cDwCH0kldXmQAytQDqtjoKANrNgI8eSJVcU5bDqAcWnzSPNToASEAfhN67EwC0WxQ143SKsMKBUIumaEjbWcDJ6vZ2gjIAoLqmAQDggL8mBIDpqxe7mBIjsUASZRO3BtfYtmSVNPAeNt9HrWWAlBYAUlsAiI12Wfg+oDh2yr0eaAFge4uaceeA+EgA228MMs4zq0NrxJ0DgGz3A4/T354ZNlgHJ12XvA6o+frAU5JOLvz4P02ALGDHXomX7FYWI1bMXvmkwnU8HXSAZvnOUwCXlFfo5SNfWkUBAPgu25AKGOcM7iuvLvQ6gE3VbwGaFfu3AfisHtD2T0CyrGg91Ly0ob1llTUDuDl9QH/4rNh6AsCi+oqaZhyeBwBLn6sLQF8EYNzVtZVw6/M69nbaJwGA7xPnAHBBWpUn/gD4SuBHuFnY4ob44iJ+wtZgQBY8V86bA08sA9yXC5y+aALOBVV3UVjTLQeYeP8UBLbMBrb+0gp72+LK+UgAh1sHb4kwq1tgb1sASApzwxX6IgCGadUbenyT/LpJ2nV6vZhryU0AugclnVypigHAyUq30gMkXtIXAYBOD4B6I9Bw0vI1m3fmK/lIYEf/Zn41ur9MJkYs3DhhPQKfKodZ7fGBD1Sr/rDldeNPffwCe31Fkjse/mBZv+k6UlZddrcZ2+bE1a3WM5fc3/6tDTNjSaY0fG9GvOR1tPUwN6RYUIUNm/r5mu7vaEnyt87Tvp4Up2zG3XrnC+//9NvrA/LIg/0CAqKzfkO3t6a90AQ6ezs7m5Mk03vPj1s0+TCZ3vvjNYPf1pN8esrEv6zZxlw1T5eH6CyquKqWRRmxDLM6ffDtK0daf7tuuuQ1r4PCyX2+f+HFYpK/d9s8e3l82OcrLQZ3ha2ZJ1OT7PCDYlcIJ/xWs6ptyc+rVOs8x3rKB/sF1Op3xJxtc6+31sS9uUjWybH152yblTSm85sGq5cEa3sfr1Gz34GvHq4WMig3v22DxvV9vAbfULKcYpXLx3TFSZdulstyijO5kqx3UF3O4XweTy00kWT6GeG1vO7INWlB6+G1WWZN2H948ZRo7axJYdsO03R85fIjTrLL7RganDavrB2/JXys8FeS0KmmkzdcnmjxBfEDB0sZ0vTHSQP1py+YSuim4qOZ5sTxQvLqRdHglbNyNfnXvbpSuYLOuy5fZzp74pZNJxtOHdXSlGe+ms44+njiZqv9JI0Hm79n16B/9vbG/JT2mDvYvBDRodVXFd+2BZc+xh0ub1V9q+JcobyTAWDT8p+EO/X2H/9d2xs/7FBQWGxZANMbbkONr/5+9Q7vk+s7x1egdfc6ud2piwBg/LkXFLwyq5CPLStG4h/9aUK+kPy5j7f/E6crvm1JfQ28k8X41P6Kc4U7nSw8Qw6aHrdr+ZDFpn/ZrRZZV1qZPxszwhMGD4+Kb9uvyePu5AFvRd0+FecKdzrZLNlpt+o18rBvkIqwcFuKfO/kwLtZ9Z+BsFADDyo7RWWnqAJ1X60qqlSgeL4HIKoiLEfdwWcddXc5OepObJAKZlQFKphRnVyokwtVVFEnF6qogaeKKmrgqaIGniqqqIGnCv5pYEYAyMqrTG1gAMA/KmtMjYvTKnsZtCFV0gze0PoFQ/snAWjq1rDkN+3co+0T9XM333KCHwIw0dMFT7DosreXQVezNi4YKlHboIr79lW5EwPv6Ddrqo8eGgAUL1zeNOadzO++Od+j24tVforfUHfoo8HI2rzqj2fv0589Nvl5AEDmqF7jjfN/Wp/qop5pmTPrJE+vv1FpYCRO7RPoCwDDhTE5e13i7jov+OqST0VOD8L1VYkJ6DyzBza+XxAeMbGK+/bvaskat0yyE/PpsPCql/Y/2h23/V9K2X8oWIxe5s/v2+SRZI8qRSS5Dp8L6r7+epJbsIwk9R23kDS1ekoh/FAReXGpuXlRTtCOhl74giRHTTEptK/8y0hjuX+caShTDcZy/SY3e8KocEjpmEEAKr1jUmSqTDxMBQiLyrXMPK0Fm/0B6PZ1rQoACegBADAmRngBeBSfjQKw3PNxAJqO7VEy/HAMFJIXzyxr4KsB3/0SdjzBoRNXAZ7LW03oFYwLuZ9pFNp3LoWLLqW3fi3Ichi/ZIPtuGBJmlZc9rn5eqBLa7nLC27pX20mTdkO30INSoxJ21NO4je23qeHpIo2T15u3CNEWeFy5GE6HgBaBQkbsN4XiDMQ/jZvJPx6BPNJ8hy6kGTfF0mSz55ycZm1HyxcMC7opp/JqDyvC9yjb6WsmRjha2oITJoVeNxkHHBZsX3nQ8DQFOY/5W/5YryoSbR8XBCTpmEzySNNr7qynf1qNjnXe6c0ZTN8CzUoMSbmLtddCAtlI16MG6bkTipLg5wEXk+NnqQxRtjD9wH2CIjd54Rf5+MISc7CepJs27qYJDebXJztUI9twiYJUp+ZyaJUy+3DdG6fsH/yRIDsnP4kyZRRJvvA+wWPCCV74ptPV9nbd9sTL6eRLAxoZL4fvt8kmmTxBYPFPWLye38tyVEudxov9NlAZqOLNDVu2fbdiYm7o9KkNSgxJua+jYGnz8qgPiWHZHGaZbeo9uApg8xJZoUuPYe5hSRz98VrL7jTICfLKcHMAvDVUIG+El8lXLjTRpkHf/9QAPvmzR0MAJ1PRiy9AjypUQQ/lAEURTKjDXkRTQCYxs3ROOYJAoBmRbVxB4fbwxXdlq1dcoFqkWlnAQAHGt0DAJUbWmfVYnJNV28AkT/edEGNrUMAvrguTXmM6vVQt26pI0OkNSgxJuauSDm+cN5CcbO/wFU89Nzzps+X7u20BBLOpdVJFoUVsWkGN5b9VjsFh8nrZoS3zjcgJiYmJqYFhJg2+LfauH7eyFFHzROQlgAsV7MC+KEEoCghM9oC8EiuetM5T1CYgDxpcgRXdHMI6BKQQfJp/EaStyaaOkXbjQtC0lBzGEn+BitaoPcG4e/utnpxpznJnZgoTcmGb3MNyoxJ21NRI16rNSZuaioZrsxcxdeOkV9WvS6DNZqdZFWYEZsiuLGst9qP8SM5wcwK3I+ZJMmW1ke899JSfwlfbe3qX99oiG4mpfBDke0nJTPaB562/kXnPEGSnNsYax3BFd30hFFLkqFeWST54SU6Dbx0vCic/kIrQbHrOtKG3klS26tDnm3KGH1dSJhrUGhM0p6KCrzjJNl+gKgwcxWfJLkfR2SwRsFJEoWA2BTBjWWc1SIY6UjybywcJAh3sOzk58x3WvQLQaOxIzreJxxX6t597rDYnEB34IfojriBEjKjJVueAUAlfwD40cxkghOeYHLyr63Gdq/txL4bL2+8ARw6Pj4QwNF7XIB+CuELAJ4i9MY/rg+HInHsj9JSv2/a1T7WV54C1rYWKGWWGpQZc9mecpI2ANBheXagDVexJQAfFDmANUoVrQF7cGPpX5ndi3TTfMt/pIiv3BEAdlsf8aq3AaDjWQD4fT8AeMXA0y34Ifz8zkvJjBbpGxgYGDgQAPBVI5ct10+ZHzIv51Vn9t0T7cgeswHov33WRaZieAtsv1sSducna2ziDp3mrLo4LEeegm7qy0KrLTUoMua6PeUjB3YDgB+SYcNVtNIV7WCNUkWAI3Bjad9cAMFI/3pIFSFt2POgjzDQRZr/ocNDngAO4B4AiBXoZ9eaBMAt+GFBYTMpmRFm1OJGreWUDfE9XLb8gxdrYfSGjd8NdAJXdEumBX7vA+B/r7h6d+0HIwAY4Afn1FgAmqar6vY+4ClLWYZvaw2KjLluT/lIv/xCb0CHWk5z2MAaV46UKjSAPbix9CNeMM4kPm4ZWG8I9hIaNRQmQfmRAHAUtXEdiC8AAGx5GYrhh1aAooTMaEEtBoeEhITUAYDTN1yGctKlJwCPlVXHXHMMV3RLVl7d5gvgtLZWXl6eQZ9X6PhpQQjrYhk215YaK7xRDj/8szxlHr7FGpQYK6E9ZZbrxQBCl3oDOOPf3Gk2CaxRcJIdvVECbizjiFe1+oGVlqWMbYgAgEsnn5Ed56CWaezqgqTVvTTAhquvwTX88OcasMAPEzLqCABF32WjxzcWyIx25EVcRTU45wn+2X8FADSe/N7odV429t1fUDm+2gOnPFpkX34bwOnabzd2yL3xr5MJABnoCKfUWF03wwFvIADnxZRk+BZreEOBsRLaU2rRQgsAOQ2bHwaeag7gQsIyMRJ0GgDU6wHoYQR8Vgw50QZY1NPiJIlCQGxiyZiqZnBjGZdT2Owdc2JZdKBvxFT+MaiFX+MhKeQXfYN8o6aTnIlDs39kXP85w7b+8sagAsXwQxlAUUJmtCEvcgumOeUJbuhes+o9sSTn1K7q12HIDZl9t6d5+6aaSL7/p+XdpbNZLUe2J8l5AeKr113tr5K5ncUXLPkaz79ItsMuMUXyuOy/ywVFKzMmb085zWq1w3vW8WvxxEKyqNUIkoZ3Pj62ruV866KElavYoH/ulHZ+bV+Vci4tTjIrLIhNEdxY1uUU/nCrxPfFe5ecIJNv8GLsF+dNbsAP5QBFkcxoQ17UrsxQ3reu4Yqu23am8ajRo0eNCDFHgKHaQ0JiDqxrW+Zksk8Kaer8vjxU5MESnUDyonekQZIit2OchO5Y7SGFxmTtqTBwV/LKzRlKYY1WJ8nojSK4scyBV4GYMmEd7+8Um/fI5lUF4eilB6v5P/KhZFyQJtdFJl+fMELnihqb/cxHe/d0ik6XpuTDt7kGJcYkue90Ypy7Dfob9tWGtl6Ff+pG06xtRZ3buf7X1jh2nO3aaOQp3arooFIZ+9fuq73tgbdj0S9Vov7bXN3hjLt7Q/dtD7xSsP3UwPsXBp7X7a7UE39DpapA3WWmiipq4KmiBp4qauCpoooaeKr8awOvQoigdzJm9C5DoEbeiQ1SiaBQ1/FUIqgq6jOeKqqogaeKGniqqKIGnipq4KmiCsq82adY/D/M3t5QhvwsQVwQO4sk032PqqpPcLes8dit81wOqdW1Zt6muj1xOf6JWCBzVK+hxvlcn6op9eLRtMyZdZKX19942A5HBwTlRgZ5x6c17VackRDyJ1KffXlYWU6p5PKau3wdzx3c3239Hu+7J9ZURtymqa8A7+oAQ7+3Hwc+aNOh9J9pb9lx0ANtPnkcQM7Y2Y0LRjb7yboXzqRLvh+YP/G1MUDyw8DJxPvLFHhulxdRioDpu9N6j7GBcjCjROlEJDhXiTet5ZR4WGKCK09V85zsCwcXadklZ07h0UMf+wPgiOazkNQtPlhJMX46ruJRtI9eJzkJp0l+9g25uDNJ8vn/lX6HyVOTSJK/h9nh6MicUcIOwqUkGXOTpsM3yrSVpOTycIJSJAt7vqPjt/3kwFar0rmIOFcJb9FaThGDUTSh6xNj5Po22XbMyHLZ7OMe7s8qmaPLd7OPo8C7MpIkw2ubSL57xBny072dXCKxs37wdZL9YbV2eokk8D68cNt3mYkARZqeiiF5X6AMzCgqnUvPxVNXCKA20ZtiOUUeFk3MrFRAsutLDi7ScqVFDe5DkqsrF90+FqjrwEs6STLfczBJrr/lDPnp1tlKiJ0yHB1Jnk+SBN6+CzRkX3WJpbSCySwQShs6pSH7qiy7PaxS3ra1lTeQN/EAyV9whOShfbIN3TIlyet2CSlVU/SmWM7Ow65N1AsjyQnViuwv0vIMPHtCn6Ned8YCLSUM1CURtF0rAHuNUQAw2EcZ8rMEkRA7E9ODAJzwamX57b52koydG54JDxzrCktpEQuE0pZOeSY8cKw0e4mwyqGFg4C96AvgM/+2ADp0lv1uq4zZKPxN7G6wN2bc6Q8Awdp4sZxEp8RE3uVqABB44xjqa3UAqiGrIh7xs3OrAYA/zjmHgTplgZYJBuoKYcEpSLbuIneI/HTvMrMhdh7EeLurcKklGTnQNZaSJGmFUNrSKc3lLWpHsEr7ExYAiqZabS9M+2DKH7IbklzpBKAYc2zB3AV5MnijWE4Zg9FqotgzgiTnYqWcGVneI945vE6Sx/CecxiocxZoKWGgJZMEOgWKd1aHyE83z1ZG7Cxu3eOW88CLHugaS0kphNKOTimUt6odwSpt23ZgcvjoGyTz0HWekal1f5EGnlxJknld1tqCO604V9GbYjkHHnZlgqEtSXKMgNZ3dJGWT+CdwGSBxzrFKQzUFQu0dDDQkoiguHF4gHhnVYT8LEFkxE4zjs6llICltEAo7eiUkEMrn1cAq+zU8fxrw764BwU4sMYDjXqOSpa0zl7pv623qb6cxri6DdD//fHfSXiLYjmHDEbnJjDzyYw60B2FQcqMLH9xROiz6XWXLNBSw0BdvjIzP+JBKfKzBJETO804OtdSApbSAqG0o1NCDq1UBKvUNF21o7cRvmjQAEBo6j7Jbw6U/tvmjZRTQAWc66ZsiTfFco4ZjE5NoN9HL/6V+t9oAXyp6CIt3VDgiNAn73WXLNBSw0BdBl68hTuLWJMS5GcJEgsJsXPr8dWVccrN/xHTHLU6dOjQoYNltfN84YyDWQvXJkr1GgfZLflKMF8r/PDPqO4dAABVpFhWh0pbGqOIcxW9KZZTBHSUEmHH/y9h1xuZAmRS0UVaKnGE+3Pd6SvlCo3UD+UVeAmBFryJMuRnCSIldu7ft8gD+KGyexZsKZQWCKUdndIme4mwSl3H9joIAEWv1kXClS791xIOlLsmxW1fIiUP9eupg4BzFb0plnMCdHRmAkDDESOrXwwJLe1FqvDFVXQaAKQGhJWBBVoaGKirwMs91M08fBQkrSZKRn6WIMbheYBA7Dw74trLL41++vMQW7K11pLUFwHQ6W2xlFtPAFhkvT0tuQlA96BEb6ZTQl8kzW7O57RpxYdPZAH4C6FAvxQdgEx0kEGCbZXxE7cG19gmDRsR5yrxprWcAw+7MoHE/vlA/u53vEp7kUIZEXT86VSAm8ZLnqFsel3SuRYWqOgFCwy0hP51Yznl1rOPN/OrO+iZIpLOkZ9uTaUkxE45jk6QrUOjgvyCoob+QvJgv4CA6O9dYSlJSiGUNnTKg/0CAqKzrGpHsEp52yQAxcwaW0hTpxdkYEaZ0jFA8bN9JNMqfSHjLYrllDAYJSYme5wk32uns2NGljMR1I7Q5wAG6owFWloYqHIwo1Pkp1tn6x6xsyQspS2EUkantMnuCFZp865WAlA82CH28EvDC2TuEZVOAYpSnKvoTbGcAgajxMSpPruTJnbPpOOLtDzBjJlf/q9En7hmgboLA/07wYx33ndCIkAReQk54W3s3iU4Uto8xB4I6Bxky1sUyylhMIomrv2U166zRuXjqftq1X216r5aVaDuuVBFFTXwVFEDTxVV1MBTRQ08VVRRA08VNfBUwV1JEoiskH9rpLmDz1pzdzlZc4cSQVVRRb3VqqIGniqqVJj8H80FT7z+MoDqAAAAAElFTkSuQmCC',
    '1808.00179v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAUQAAABuCAAAAAB6wTX+AAANxUlEQVR42u2deVhU5R7HvzMsojCAIKK5AHa13PAiiqkkpJQGiuWGAlqGmnXd0pYnrkZ5u4YtokmaW2ohrmm5xZUbKrigQoks8miCGoYCCsg+wPzuH7OdgZkzcw4zw1yf8/7De855z/e88/Ms7/t+P89PEUEobS1iIQRCEC2jEFGAEAX+JYCIRASIzPFe5HcRo3fNJILC42yEYt2O1y756Q/q/UovxdaVUw/6hrsenmLo2XUP1fXO9lV35DXXbiJ+fSm3tiFpRye5TKfejLjUJGWUe87oQwfC2O7G9nmcmz9NXDdOfHHJC7F2ABD9YHW33K29DmYY+vTlbrpwtXOEPRqKLt+NW1bwXVpqt3n20tycgBh3Ho9z6n9vH6yPeGXod2mp3efZ1mRXLp0if0hp+0cvv/23G4nPOqzP0ClIRCAzlJYXaZriXU5EVPvimFoi+tm3mYiag33Zz9IolzCZiIjqXl9KRNmYSUT00LdnMYduMEogKokoG7OIiJLt3mgmIpK95XSJiIh2WvvqFGy3IK5GprxSZPMmEU17Tx4XLkHMxlR5pWwWEd1EBBER7VP85RzEcahiyIRjPxFRHL6XH5WF6g5ie31YHnw2aqi81mPK1jygQP4yGu7JWalRCldnxnYfpBqjg144A6B0lVeE4rFdYnmD7cS6scrqWNoJ9Dv0CwCIFnNWSjsFjGdsP0Y/Y3TwHoYASKwOUUbI38HigngafZTVp3EaeAfB49ZelIL7wD8NwGTG9kGrD4zQv5sHQqMA/Ir+yj0dtlncEKcIqkewM/4E/BIXpqTAfeUiTiqXIxvvph9TDXruQ5r306kjL7ata3cOUVle/qY5IgBFcFPt72txQRSjTvXbIQYQNj7pTMrNxQ0ruKj4JciK5qu2ru8DHCK/tm1j11yHoTKnoq9IHp5mCx5s98osVVZL0RsAnGfOpLMzPoly5vaP0XsJAc253gD6LDNK1xw8gW8ixmUOAOB5pUS1v7yzpb0Tg/CbspqJIGA/AIgCt1T9zlXJZyCQf9nY/ZtavwUAJkDdn1iL+7CEOyZJFctIx63nA4fkG+PVT7mh5SkvIH1A23rz42EAGnMZZ1wDgDD3Ew2KPRXWFhfEzrElu+W1U1c/8ATybgEAqjCEz3Let/3b1JniadPKgDK7DhpBJNQ22ceXrlfs2TGHdT2xXWYsshWOZ4mIcnrOaiKiZ/zKiYhi5nKYsZzDJCIiql3WnYgy4M+jG0RE1NjldRkV20YREWVgIhFRqRX+oG+LiTbYfi8jIjrxFVnetI8owWvVpawvPTY2ExENTp73aVLKsikVBgfxSmhvdHolImL6CGeMpaypzzo4vvAuvyDSsei7RRMmVhNlTe0vcQ6KIaLNnaILVxARnfEdsftc4oJEFsH2XJRtPp8v6zvGRv518aXsjGZfH1H7LMqWpFQN9W0xMT1FofKRwvWM8p5BjnqWwYSVbWFlG4LbJwRRKEIQhSBaULH6GECgOa4UaMazzC0oEBAQCAhhnIgnmoAoKCwb4WmO69cUAhIP5dbtasDLvv0EZWTFR1BHENP3nDxi0iCWb31c17joGTzafzGNbj+lmMH6lY4dtYD1N5cs2eLEWILakeNg9b49AMz2Hd7p7sWXxnITlG0rsqtdpvJR0j4MdrMHgHCx6lKGCOpY2XiEI6ZcxSldVEr0uW0yERWGYY1i76bZuMl61ooFw8HgG6TBEc2037uUiMgdgM0qGTdB2azVRJn9/lJub1Z+cTUupVdQ1zvRxrRPccL208BcaQwAu0F+38nf9lTtCDu2sySLN4Uzt2OTN4sxQ7IKALy/+XDbjdUiboJHT34ADA1U+fLXt/ySmpaWGrhT41J6BTl/WFIyFZU9RW0IYjcCYI9HAIB5f8iZhQuj9ZzVwUPzlbXVWwLguYRaAF3fXjNP+QIyWDBxtC2AgGO1ymgsmPC8v39BlGeLS+kR1BrEhnvKgYD0Sm4Lz3BtJ0WltKWZrWjbVPoATYV6rJKZVdOB85gIAAjrtEPexZHc/iUqihwAwK36aosDhgo2JzsBQPeGM4odiwCg4EIEuAlqCeKt6TFn1yQDAG19tbbA/zzz4HUbpZ3x2oEmjXerou05n64xv3x+OWwu+4DMBpB+NWwlAMBxxqFKAI8lHNHCjlYk/wn5AJAV90VcJTgJlpY7AIATbkAJYwCyJbGtTtMj2PrrnO93fAywFgA2r8lzRIeJhWoruH7hWmW1c9i76xmnKdv6Z4++WRIN376zx7L+gEuHTw89pPjMRe3auxAwnO9UPt2DHgLAbVQAyMpbJjrid8qDi2AV7AHACpWMnXsHa/GX2QVb34lzfcYACANQGT3VEQgSJ6oHGJNee061sao4ulG1wWgryZ4NeNnksP+AEbEJd2aVyeuj++0AUNqV65t1dd59QPo7mgDsmSXCqw7LOQnWwxYAREyfVvrhW1pasgu2uhPvp8co9/9WaXsOQBflzY4t73W9cEHd1H79z1v8FXVm2wFiwMq2Rt98qV9Cj5fTrQBANO/9a97XBnP+PIV+OX9T4w8hF7sA8AaAYVtL3TgISuSQSBMk6n3HRL21dZZVsFUQ86EazN5BVzsAe3ood4w/mxnWUd30zokZPtraygcB+iEW1+Hn/hMMAJgTvf3rU+9w/8gvn3q24Z2VGAKkS8cAkCA3kIOgs/wWrAdj/L7LS2tTVsFWQewD1S3UH67DNI55Jn677pjqjKKlSaoYtm7LWqT+Tem2gAtuyrfdJyb8y8aKx1DJwwO44zkECK2ssgWkcOUi6NTtAQDch59qV9OZIK1NWQVbvRN7PXsJAOoADHkqGQAajqqPLhy8UVVftE4dQ21t2V5GGddKAPyJIQARgKjyNyZzit6jegBpr1YClamrrIEhm20BXHfqz0VQFFIIAAUuvgpBIK/aBa0ndXoEWwVRtDU5B6BNqIPdtuPXAGzoxTj88S7lxyTbhrkOyWgrbQRAjWyPs2NwSk/gbm7A80DudQImdK/yBFCJx3p+dwMaAKCslz+A40f/BNZ7zQYwrT+A22fjrDkJLs8rAOjwciuFIPAXHFpcyhDB1nPntDFxB1fsh9s0opSR0bvfO6FxdH6SorKixeRa0fb8JOfOoem7XnDwnF7OMgue8+X5cyNCiun6eBeHYRuJYg8RrX3JVTJwWr7usxrCX+wmGTA5jqhmUCQR5QSn/vbu2AdERE2r1l/dN/ArGTdB2heQ+2hFpFQpSHQU0S0uZYCg1kXZe2UDZHldunQEcK+6n+YAs0ysuN9v92g5v27Vlm019GoW+XiL2riG+jCpwmekQiQv3WWkO+dF2ZKTNSOZ2IU0IcQdXBdlhZVtYWUbgmUqBFEoQhCFIMLy3L4AkTmuJDLjWWYUDFDciYHmwI1jzHiWOQUDIYwThXHik5kDQrq7rGekqP0ICJnYkggIfqV+zCL/AJcQmIyAqNpwt3jwUnddAMOZTQdgSQQErxI/SFYYWWBCAmLmLaqc5nRRB8BQ83QIWRQBwatcHijy/MELJiMgPvqsDxx3WoU3awcY4gwEFtDuBARYIIgaO5iUgDg+qhxwCCjMhzaAId2rCyyIgNBDM+iGIAAAjfcfoqIaJiAgejVIATigRBvAUP/jLFgOAaGPZtAFQWREXv41MrLmhHf3bd9s7H0Wxicg0ordAVyzHqQNYPh6ichAYAHGJiC0BnHz6r0Bkz6ZWOGfPepmSXTYul2nmZ/ghSuhhiAYB4Yl+PkmJNiH5PW78vf5He7rCcilD/w9ziisyqi6vTCEgBDbAriStcRNC8DwexeGFWSgIAcCglVQWxBZaQbdEIRqDC+5NbpbcRhMQ0A0RAX9WwvA0PjD6zAUWDA9AQF2moEFglAXg1gGngREtNtPdloAhvh/iA0GFmByAgLsNAMLBKEuLoZ9ovkQEDv+OtlBC8CQ1+BaATQ1VlhJLIGAADvNwAJBcFhu4k1AHM/aI0aOeEBLgKG0aCWAvK4r+yy3BAIC+mgGnRAEl/khTwLi4oUNYuDnDnJggQEwBMTHx8fHOw6KX24RBIR+mkEXBAFIHzP/wOgERH7kw7cWvjl7u6cCWFADDPKXTs1jwEIICL00gw4IIjPUxTnoC0oKlrhN2KVv7syLgFDkufNWAQsqgIGIaOFzDk7j1loMAQF2moEFgoB5CYhWAINAQAgr2xAsU6EIQRSCKARRKIppn4dAQPAvHoJ5Lwxx/g/Mez4ZDNrbvDeOoEzMtPHbZN7zyGBgcvOeJX2BcQQBOQLAsPHbZt5zz2BgcvNed/oC4wiqEQC1jd82897G4sx7lvQFxhEElAiA2sZ/0sx7lvQFxhFUIwBqG5+/ec8zgwEAU5r30J2+wCiCDARA08bnY97zy2BgevMeLOkLjCEINQKgaePzMe/zfRbHhv/zBqD28S3CvGdLX2AMwRYIAMPG527e88xgYB7zHjrSFxhFUAMBYNr43M17nhkMYBbzHjrSFxhFUAMBYNr43M17nhkMYA7zHjrSFxhFUBMB0LDxOZv3PDMYmMO815W+wDiCGgiApo3P2bznmcHADOY9dKUvMI4gEwFg2Pi8zHueGQxMb97rTl9gJEFAhQAwbHye5j2vDAamN+9Z0hcYR1CFADBs/LaY9/wyGKD90hcYhwbgbOML5r2wsg3B7ROCKBQhiEIQIfwHDnhS/wOHGOFe4l9iFINtoQjvRCGIT0T5HwVCxbsBvkBbAAAAAElFTkSuQmCC',
    '1808.00179v1.11.png': 'iVBORw0KGgoAAAANSUhEUgAAAUQAAABSCAAAAAAJefIoAAALbUlEQVR42u1ce1iUVR5+B4aLMAqIJCYiaJKoSYriaioI4oWKvJA3yK1ATZ+1MFvdLNdybavtpm2ZeX1KI28ZlhnrBRFKlEtJXHQ1QY1CQIURUBhh3v3jY8YZGJz5mIGG7Xufh2fO+c18v++cd+Zcfr/3fMgICebCRqJAItE6QDJYYqH1CCYpIyAza16U/bFnVRml4WwRyDtkq5VJ2a7BDxUqh0gLS6vxn8ddX1mUsWKyoxUN6Y42J56am+ECYNEXV2TWQWAHHM6cu9wFAMLKZZCGcytRdG4AAKBbiLTZbj2J2EcA6BkJaU5sLa563wqZMslPZl0EdrSFZUesCrh/0xiJRHNw5UDyQWWXsz0A1OuvjPVyKWIxEZ5xCcVxNw4C4IMZegu3flVaWFrCfgCA4u+QAbh4bqDue02qEokt4PZB4bUBf8K1I5t90i8BqEs5cLv6TtVKSFTuWvbP71H4o/WRmJ0jvH4V6o96pI6CLXBuou2AiVHa6u+STwSbIik8qabi3Rf9CmgcYLviNflJkszzuUhS3fMwyVv++8kZL2mq7Qvo/OnhpF8lSS68R219JE7+6okXjxxdNeESSZ63qya5457bAn9Ctf1JlHeI4FQHa4Y+mpdbO3uVDABSgpx53f10kBwXykZpqtaRT7S+4FQHQ4FBgzSVrNFYP8m970Xw/RFOmqp1LCzWF5y2hNHFmzz74im3dW9+P05btY7YWVxw+rtqLDV29sIM5LU9VKdqBRFLt432KfH9/dM6wK7R2R64MIVI7DxGU7WaLI5OcGr9al/ukijlpZd6WmMConrJ5s2xkmRqWvdJBuvvfBKFl8vYYtJGqfcfmkP0btwnHm8SnD6mDU6BX9M7j3FCSeoE+VFFqA2gPFU9tpve5y+xzb7g9vohmeXA4BZHNzjF0fcf8pwPrrMJ+9fI1/cA2+d5DZokHSQzutlOzj41AkD+eyky4K2p3a+PQX5oWcXfnGeG4ZuXz9tnjBQdx5Ql/kzvKb0aa5mHSvvNcd83zQKtv3XtTtnNuaoxg+Pu2fpIq6xCbkdVJxfBlZO3Dj81SVkVPjP6cPdM46tzxMI9XmGy1PRN3gDWLu80da0LsMhuHQAM/PMy8QOiYU3Cu2E26c+Oe8MRAFaUrvbM39hrT5YFhln++hOn3aKdUVeccfm9+MKtaamecc6q/LzgVd1bOZxTj1zcUxs9ZejWtNQecfY1ucrnpgmDlZv/PnnRfecS+ivWZhlaWPRTCNlkbsLWs2qSLKlW7hz4Lsn+X5JkNTLFJyDqpw2uIMmb4WNvktwf2ECyISLQMmmNU3iMJHnryedI5mIWSV4L9CoRkT9oghAoSeZiNkkedny6gSTVC11OkSS3yQNNy+Jo0Xc/uSqRLLG5TpLqnmkkN1eJ6u5qZAuFYrsFJKP+KnTeQiTmYrpQuDqb5HlEkyR3Nr62isQwVOm4moNdJPkePhXeVUc2J/Hume15yoPrO0cCmcPdAEC2/aPEba/1U4iZZEpfHzVUKPWctrEAKBRmm+E+Fk13q+DuqlPvg1RLufZFCoDylb7RjcP3WbGnwparyyfLAExuzOeMCylxdxDXiIRboZpi6K5tb8Fv97eTAcgWW5LEtJuPYKJO/Qb8LOX6VwQASKh+UvNzG60Qq7HYdJcBgLyzZhq9VySHOIY+mmJfHAOWICLszXQVLHo8Nw3AYzr1PbbLLeT5/O7IWABH4a+xOGxq//OJxdCOMzf8AgQlPJOcjO4v/8VSN8iIuX355NfaTc8VqAoSD30Zbr7jS3t5teDs+rkyAMXw0Nr7tT+JNril7SBsAMycmJSSfH5x3VIL3SBoh7p4nrZ2ZiegiHnfEtkc92FQ5lX2E4YiGn7Pk7K9sss1xXJ4A4DrrFk8PuPVWFeLfU/ezxJoyB8MoE+8xVqu8AE+jA7LHgDAJ7NMa69wa3fdeTx+0AaUGA/sAgBZyMdVlpRjhwwEzrbJ4YfptR8DwCTcae4b7S/ez+mSpBJKPCCfB+wVKhPvjHIL4F5f4OQA8/18sQ+AXgzjip8AYGb3b+oaLZXy9ifR7Y2yT4TSodPLfYCCCwCAKgRY9j7c4G+2j5KoqKvAVUcHPRKJm/XOH5SvbbRsmWuieC9WeL0b1Eu7HCfJPK/Z9STvD6ogyVVPWSZi+Q6PkiRvxvcgmYXR5rT+drcn1SyxjyXJLDxCkuW2+JkbSsh19p+qSfKbdygy7KNFxPsdvitP5bzd+98NJPnA4bg1Scnx0yotQWJmpDecpkRHPz7CFaHMmd5f0WXcC2a0/usVl4snPVJN5kz37+w6fhXJj5xWFC0lyZTAEZ98lzA/wYCDdjmf2PD9WXW/sXbC6hLI3KyGwCEyq0zKliVXDQ1sErceYqSwkTiTVeE1vksHOuTZcTLb0mNp0qO6EokSiRIkEtEGal+wzGzltQ2WzXYiwNz7BDf+EkPM2myvapMDqG3jtQ3uEwJpn2iRfaK8nRV3y8vtBqR3rypR2nubPSBpuuLe5Gr1pmLHm/F3kulPBA53upw+IVSk3N7Ea9W6yyUPPCd8RP1FwW2bxR76RuhK71GF4rR3Q0mhLXkK22XOBnrQwqkwmKW461+tnr2azPb7TWvoDsBupVqs3K7vtXzWBSqjXNJJsip8pYrbI/WNTaV3Udq7Aagiohu4a3C5wR6YmsURobjrX53oUkdyfpTWEP7hi5uKxMvt+l4XFpGs6upbT6qjokne56FnbCa9i9LeDZ04sLtB8qFnDPbARBKvdBqlKc6U5ZNDZwgtmG6UxBkRJPmZQ43GEG34zIKqjlyo09kM9LqL1149rpOcijzyCLJJZp7QM7ZM4ktYSLJM4dvQ+P4Rk0j0CiTJpYoaQz0w8QSEruLObYDf3m9hkuLecNgFAHrUpRj5YNohiJDbe9WpAChQBmxweRDAsJF6RhjV3h++i/beHJXFCgDwqD5txv/Faaa4744InTA20N6o4l5eoQAAF5yL0JhykuvlT7s0IzFQjNyeVm8P4Cf5IPCY7y8bFcqn/XSNME97b45OthTCubOjWuyBURJbrbhXwRkAbKHUclgQL/sy6FBvs+R2G3sAmTnPe+DGtf57/mFTNOaTMB0jzNPem8Nh0DUAuIhKwz0wKXZurrgXfb6gX+nid4zdvRb2Qiylvf6z2TJMVTwPPbn9810TcEdu37mvOObso8Zc18WOfw24gZMzbOAbPr9WxwjD2vvwUWqTtHcDWF1wBVD9iHrDPTCJxF5orrhv+O8xj1crjdy8s9DaemgO72AwAAzbV95UbgfQ8BMA9ImPj48bbfzIwgqPREfAGd7eAAIKT+gYYVh79wn40C+sAAB8dCbOClNIjHx73i+Frz+Mbi31wAQSW624uwo/wVpoppCTqQK3+TBTbt/y20FnAF3suwJAJ8FhoxHmae+G8PwHx48tKUVAyz0wSmKrFXcXz1IAuIIgzVcargKggruZcvuBnM8ckFcA+QM1wqMN3XSMaEl6N017N4jeMbFdLvkEtNwDoyS2WnGXPVwEAIVdA4HrtQACPrIHcMbF3zy5Pf3EOhtgvwMQeUEFoBTDdI0ADErvJmrvBrYDU5WAMnWl/K49MCLei1Dc9a/Od7xAqkeuIcudAkluOEGyyG6rWLld3+uZPvMXLJgf41NPlrp+RapHxOkZm0nvorR3A1hmk0u+MkRlsAemi/cmK+5Nrt4ZnH99aYyKrBkUQ7J+5drTOwe+oxYrt+t7bTywPJgkM4btzXpmzg19o770LlJ7N4C8iNQfXggtpYEeiBLvTVXcm15ddrBmpO4HC052HdkdsFyer/L41eGDDWfCmkvvpmjvhnAtqXKI5oGdu/ZAEu8l8V5S+yQSJUgkSiRaI2xfARBilouQNmlYSDsREGIBBzLpCXBpOEsk/r/gfz0QOVoT835hAAAAAElFTkSuQmCC',
    '1808.00179v1.14.png': 'iVBORw0KGgoAAAANSUhEUgAAAbUAAACHCAAAAAC0BJgTAAAbWElEQVR42u2dd3gU1frHv5tGSCGhg0BIaFICCqEXiTRpAtIhiIUm/BRpFxRRwMu9okgVQZoNDE2QosClhkTpoAGCSAkEg0BCSEJ6SPb7+2N2dspOsrNAEpbseR4e9swpb955Z885c+Yz3zUQjmR3yclxChxRc6TCSSRnOs6CHaWZJF0A4FHntlmzHCez0NIsAAYK/x4pGRxLmsJLBjrmNftMLsXN4eQ9p33bt4lObuxYQ9pP+t8A31njTkzv5m73o2QxmteODz/hA2DcltsGe45Y8RohOXyaDwB0jDfAMULaS7p2qT4AoFywY2/EjqKGrQSAKr3gmNfsZl6765cR3KdrHcPTELFitBpZNyIbeHZVO0fU7Gpv5PbPB3cll7pYGUCOciWW4+LYG3lSU6WRobEj7+8CwOdPKJaXyqxjNfLEpO0AAK+PYABw/VIDeZkq64jak5Ie7BL+z0VLJOxf7X80BkBW2M8PUqWsnUcteePU//6G6N+foqidjhT+39GhHnIQ3hrOwKWXnOu/1N+ctaOnoqBl2tN5T1rigvfrXKCOBNpF+o/LMZI873+dpLHKPpIZ9baTAz8Qs/aQQDKPNaRt+3V2sobsPnZz1Y6G8KOr/ABcqZ/oCfww6aYLq33byZS1733Ip2W/TpnmNHn5/LnMITMNABDW3JP3yv7R3AVX41qLWdj187WnZb9OmZoAgYFi5lRbLOtatuZ1cEkLDzFr36uRp2W/Lp/UNnZVpZp4o/TiT3970Zy1731IG/fr7JIbSXN1E2aDqms7yLJ2vDdSbqVb2IS69SKe5rs3Tzfgah9im3c7MWv3e/6y/bqnmNE6N7F/cswHVfAU7R6nTly9eoSDrHsSowaS7dU3ctuE/25gjb4bv+qOc1l4qbqJPT4M9X5dbwDCfh1w86h3Ow/cCu/icsCrgxOQfDz1hXLKBjEs0guveFk25LHyl+/X4cCSNpVGg4udOn7W6pPNwNpRVQO7OkbEJ+8u++Dp4y0ARC0MMwDzXql4rx2iOsQlvuc5qCN+mXHZ7USrx7FjErftCv36VDPlTu69U3to2a19C9bbjATpc2nPFNMmf9lKBbwDFJfo4srsklVNBj38pLOetudUov/AGtw06FHXkIr9ukXTSr6yyAcY57oYABq8NvWxjBa5c0IXdHQ6Ov7Fue4AMP3Ox5WiVlbbfKpgx6moZUf+KB3iiazYEzcWToj+OiK80kjP7Kjz7WdWLMgRMnz/9c2ZIX36R38dEV55pFvaueR3+zoBAFd/1G1crUuhdb0WnbJ1NaLesj9Nngv9+qKRJG+lJm9osIBk3Z9IMhUnH8uef07fRokkmd75hXSS24NySeZ2D3qYDXCb0nH0JklmvP4uyXMYTJIJQVVvFbDlYCST5DkMIcl97m/mkjSO9TlOkvzGJcgmw7DyB9TcTs7cRt5yukeSxioRJFenPGrUPsZp4UOs6xiS/f8lnNKCj9o59BM+3B1C8jJCSJIbTP8XnOWOSKHM4FBsJLkQ3wulxl62Rc3as+xRybuWefcCTjYrDQCGtcu3ffOf2l6PONDf+aR1E+FTlb4rLwDRwoDfzL+QZvMH2SjrK8vXQHjhLicCEAbEfxgQYhr1xj/ed2qmGeO7GQB0CxbyLwbfKlvikf/o0IwO4scOG7+ZhzqbdncDYHinkE5aRHpPvCTL30edwo3aTTwHhKa+Ln5p2no9Xm7EqaIBAFy8xcnwmUcPGg6hhvixJg4BE9G946dHs9G+sKIGoLcsv9l5WqEG7fKmXiOAA6gnHiixyg7eX4uFeXwqjb+B5qFvHTyIijPeLgTbJ4Y9uHFsp/le4DayL2zb+1PnwnI95kfevXBx2XADEIvy5sO17SBqTsgwnzY4ARj00p6wg5ffyZpc8LabrzPGjjLn/twAeA1bUngb/mWbIvl8Um0DABfk2tW7otVOx4sf4+EHAL6DB/PwwNkjfAvjmvEbTyA3qhGAGhMK2XUvf+DLkI6n6wP+J+PMhxNLP/k8ZCecMW+foROwEQAMwStSCgnka9wAuFiErHG/zBUAukJyd64dUKxDS+3JFj7xZ5dRwI9C5iVp4CzY9EwAcKx+ITm7ZauFOIgvzgIYVPGXLNOBJBc7iFrpuXHfCZ/2/jHNH7hwFQCQgucK7U/gV/UKx9Ct/v3vAnfdSyijRqSXWBq/yHRgzXB7IMbfmjwlHACiRg6ZDSB3aBIALH+jwB8rJ0P4kmdMulkaSEZygbtavtxrZXH7YoirYD4ZAPyc70Vj7d3+iz9aSwDYZXj2sbDHBbuvRHJdwIfHIz+v/kUuSTbcN3LOnoMT+iYVsOWTvfzg0SckZEALX3RgZL+6XqVenFLQPu+cfiO2a89UkpH96nn7dppJcrnH9GuTSYYFtfju19DRoY+HPS6EJ4S5v1001n7BVViSBPHcqdygxoan8qlo3MGUJkHqPb297OULAH+eSqzaqVTxeuuwOD7LdigywaGB4EiOqDmSI2qOqDkSngBGq7rhcTB6RUcHFi/L1eFY+TtW/g+f9hbV/vv2JNij5fy+a7o500e97nL8J0x5fFe8kbrFDIyBpzzwJFrOp6f89SFz54QuGO50tKecM51ZbbMeOvj+jovZ/v0qAsDqCs0rON07lDBaVSXx42mVzJmdccKWd9rBqIRGg4Vt1h8vudTtJ3csfvdfCOyrA1qJeL97eU8AGOoE46pY9/QJ0oN+Vf5wMw8ArwY187hxtEsHi2Ikrryf8eBt3Tu7MstSn5qGBcuySipLch9gw+6xLZypqoODz8/btWOM21qSbAO4ugPr1L13Q4SU6zyEJPlt6ZbrDgwMziG5u1THXdsCK5yQ6iwr1Wr9xoDqF63v4S43eRZMGod8TJ6u849YpM6/sY8kKwJw/dBoWRz/djz5mds+vbvHMsvmPrUNC5alSmpLsp5solht4UyVHaRUWk+SY1zThKgBlTeqW7wPWdQuIZwkl6JrNmNL4ggZU9IzlYwxlE8065/AcIcMQ60sq+du/Ird4RER4cHXyG0+WSRH9ze/4qXMp9fNIcnOX76/6ppGMRe6byLj0Vpv1GSWzX1qGjZZliqpLcl6siVqt0uaexhkiCKbDBSumX7WPTiMCUaSX+MkyTYvj3xvfaq6wY/15FGbHGgkec0Nx8nrBlwhpwsXmT+WSOxuAMlMJ4RaPXcThC/uWpIDu5PkDyXSTEWq/AbhKY2ZO1ZXX19iE5mOunqjJrOsYpnVPQuWpUpqS7KebInaAswQP67AFHKw0y6SZJh1D34FOkWSr3kmkmyzXgvbrn9cFrX0Ml+S5FQ45ZC8colkW7wshMp8cZZBIEn6YLTVc3eFJK+ONpI5pYeQ5EH8YhqXVfkekYqoqYvJbJL7MEVv1CTLqqhZ9CxYllVSWZL1pGXYRS9nuql7hy4vBLnp4EyblU3Y//wb1Td+5wsAjDjj3qCZYhGROOR7ubLHpqxhALAP5ROWxNZ+pT7Ak/AAAHccFyt53AMAZuGYVfs1ARjHrzUA8YleAOCDS92FBY0yH3+rkdAi8mCOy5s+FtUBVyB7ftMZelcjkmVzn9AybLYsVVJZkvdkw46WJWfqe/C91n5Ldfztbht9wK9njhsAAJi2o3Jmj+ePyhenIVMUzwiXv1oKAGLg8WanyZGNZgJZWULUXGC+qQnCfQBXMnFP1/lb37A0gBR4AoCzyBmo8hsGm4J2YcK//JvHWFQHcHxa2+phPrbcEQiWzX1qGhYtyytZWjL1ZMsaMgg/SCNeJZJMXD+mNvC5jtHiXPPKAPByJslFZ0l+gnJxspXIO+Q1aYQ8A2GccgF2k3eA7UwGRpBkb7iJjc674yr5jgue0cMBZFWLIcmzmEqSkZgmHFblW8YKhyNJsklfi+okafyr68vxNhAIJsvmPjUNi5YVldSWxJ5seaemGiw506/+OlR+tvVb+hMtK16JaA7s/BTAuw0BNMHdr8zFW47MV1Rf3rqR6Svt3BmoUArfoiSQBgDpMCuSNfhfYJdpff28ZJB1PmmnwQ8AvAW4NwemlxSU+b+8TGxRIwBoujVeXR0ADHXW7e1mAyJssmzuU8uw2bKiktqS2JMtI+QjcKaTc77xaHtsbWnsACIGbAbgCZwVS/8aO/H0sWO/A1HH7gNA8g/jhILyqOwMoAQi4eorRq2CudcXIvd3/WJSMgL1nLtvAyAAbBkAkAkfaOTXDROOHgsXTmyUurqQyjY79T/9UTNZNvepZVi0rKqksmTqySZifOj0Pdlucs50kG7O9EzNsoBhWPO6qcCHh38bAOQA5QAk3wg04O868wDEAUt9vmgMYK1Hf6FZhwtG4XqsALT9OQ0AMtDO1ArpH1d5xx/XiJ56dsjCOgEAfCrdAYDbaA7LvPEn02TbKznFDchGWXX17LY5x9yAMrisf2/OZNncp4Zhs2WpkoYlsSfbyLplWCne335Asv4V4TYOsVbH+Malskgy2/NVcpjTVJKrgU1kXEWME+usFec1Y4Op5ndvDffJDGdMIr9AQ5K5FbBZbLUFiCUX49kHOmaXSLwpfBjRhCTnlckhEzKUeYYPNdXutIYkO/k8UFdPNjj/TbIxDume10TLUp9CV3LDZstSJbmlhAylDzbtjRgnlzpMkuerDskh+WzzRJKc+Yb1mXkNRqeTxv/4xpC/+50mU57DACO5HwiwiNphw1XRYF/MI9fBL5FMr4cj5Ga0yxVbncRL2TzpUea8nnO3G+OFD1HuV0ljqzlkvEeQIk+O3mWq/dURktdcv7as3uMwyRi39jm6oyZalvoUupIbNluWVZIsCZZlPtj4XrZ+zlTVwQ+B5Tv39O/9F0mGNWzaxafK0mySGZ1Lmb6+A7y8vT29PA6THNzN3Cxtgkeb1u5DbpDk7QFeA17zHnFfajWvUo8XvAbF6Dp3OzBd3INoH3Vv8rBsMi1wmCLPzLritzbnw0V/bGgw32hZPX7457/92qLHLf1rSNGyrE+hK8mwZFlWSbJk+kNlPthIsermTNUdMP6mZ4Cr+Lwn1r9MnuPznWpbXpZy6Zfcaolvkt2/7uyv0LRNv1oywFnfM6LsdT1EJYq4XWmt5H+0Ob/l14XmgxeOlWlVUbP6H5Fs3Mig/6moZFnqU/2HyCzLKqktyXx44ijW/66Idi4ay6/MCIJdWi76Z9m5K8cUkRx7wvUmsF/LRRu13f+MKCLLmwYa7Nhy0Ubtl/4Vi8jy1hDYseWindfuu3gUkeW75WCnlh3v1MBOyTrnWQCCH7Wn4KI6d8XRcrBJQReOZD+pfZG+K+oYIe2cPS46ArjIoOcCYY9tk5p9xOvuMRPAKAzouVB8NjrZxh5HK6Rm4zc8jNSs8bu/M4yjakHB2AKWeYEAhnHLhQdO75QHgLjxK2RPJm0ggGXUM2K3R3sF9FFIPMngZgA6oWdLVlqHyymLb9xq+K5iI1Lhk5o95przXs5TlT8mFrZsE2x8vmaL1Kz2o553zpL/vHiUFiCuOi8QwCmdP8zm2l5k/OTRzSAzYwMBLKeev6qzeN8nzp5ygFYGN+uHnjVY6bx8llyOH3yVyf19jpp9UPmkZo+zu4fkcmMjBaGSVrOHzU9qbJGa1exg5wKSjOxECxBXlRc4XGP/EJK1ypOZ13MWyj3UTwDLqefrTmVOkh3gmmwulsHN+qFnDVY6T58ll8deI5lSJkC8QNQ+qdnjj13vk2zzlry3OXlHzYp+0wO6PazU7PF0AKgZDaDCOGWRMr+jpzOAgz+eBrD+AVBC9VMclQjAUw9Sd+b28UEGoMWKC00Rbbx3oCkq4EH086bS65Mw2xU5mRKMsiKwrer4ioxgT8Cv+vW1ClnYLdtsdPnnbVGl4dX+p4umHwZW+yT4LJ2HlY28AbRcMV8amo8FlHvYfciIvXhYqdkqCxYZgZ3drVZc+yoAfOXzPICmrSzLB6cMAH7TA4w4Y1GXs8Bhz1pAuzfbDwGiUM4scrY82ykIqH75kvg7axnfjDWojocLFFVN5cV5/qNvbXS5WlY2AC/E5euzOSXFegFA+dQ/zIcytwzBw+pDRgQ9tNTs659N3Lrqr3WhgArEVeUFDpeHAv5e6ZX8ptZFoZsAllPPLmsAbD9X8nuzYKcMboZ+6NmSldbjckSOG4CzLnkQZWr2uKQzha/QxdZilSXjDQ9B+5xDtZCBLbFTyF1G31u3YnaPCdih/2l8TGu4tcsmycBQI7fWuW4uUeSXfEaSSWgzL5fRVfYLU5li5uaxqc3GpOpRs9rvAwCThHWOcWIX58Z/SoVlENDj0NkBzh+JB1q8pT6eAbwh4LPe8pXIt3LqNj+fJZdJ8gQmycrkPgk+y87Dcw1IchzmizXOrCFbPNxqJDemizlq9RYuXLgqIssGJbDLr05xR5dbFiCuKi9wuDfgHEPy9RoZGlHTTQDLqWcy+vR811pbzIUyuNkG6NmClc7PZ8llkpkNO2XkETUL9ni74RaZ1Qqfii9rTMx96KiRP+8kcyLJy+hhq37b2VbxvNIRzXNN+dGIU5Sb8hc7kSQTBH5rIQ5oRY28W6KpdVLquOfLaRHNAcwWj7wHmMNWHs45JEvhFSE/qrXF8WwIdzidUVp6a6t9tt6oKV2e1EExPsh8MvksPw/ze964OmuO+afTFlxhflFzKjCp2VFzy6Hmvlkn9lgwtoq8icMt5VYGAEqaK6mSLgJYTj3/s/QUgJbAp2KpDG62AXpWs9L6XAaw5p9def0CtwZ7PGnp4UMT74iqpheyyiYlJeU8SEp5qNXIM3hYqdn7Ue0AGGYeuNRdBeIqwFyRw3VpmAYAubBY7tpAAMup5xcvedz0RQkgQaSe5XCzHuhZm5XW5zLwc+QPTjjvpHXuLNljANWrAzH+pqjFx84AcKHCjBqTbF2NmO6yjU3vPcwIme0t3N6G7JODuAkZKjDXTADPds0i+REuKUaThAxbCGA59VwFbknkamCYSD3L4WYd0LM2K52fzzKXeeR9I8k5V2iGiaUR0pI9ZnifJDLJd41UnWRF2+e1X4XXNZk+oTLJU2hr67w2cjZJxnbLkjG2AlgrY24lAviO7w7S2GIkSXIuhAVmvEeQLQSwnHqeg8/IrBYo87dIPcvhZh3QszYrna/Pkst/1hg9ZszoYf45NMPEok+a7PFUp3PkrMbZUnUyx6udjVGzTWpWM2qZr40Oi1z5eoycsRXAWhlzKxHAPNH0x1NvDb1PZg3tXMm7fu+FIrdrAwEso56NX9Zo3LOie8gNiXqWw83WoWdtVjpfnyWXTfBcI9EHmU+a7PH57uFnpnS4Q4k95lstvXw6flr4CrqXT6Y3ambQAnHNeTkBnHT4brNGmh3pJ4Dl1LMxLrZCVeVqS4KbbYCebfFZcjnvpMkeJ+xJaqzvJySfANqnyAjgR4WeizN7XGQEcNFBz08Be1xkBHDRQc9PAXtcZARw0UHPTwF7XGQE8CNDzw722GHZVsMuANo7FHTtyHJ707wW/Ki/UzOTRZWKoeVgh+6xgz1GcVMfRpFRz1rftbhEF1dml6xqQo49/KTHOWl7TiX6D6zBTYMKmsOVuNv81IStWLYmRFww1HNmCUMBSz1rPV+7uP/65syQPv3jN0SEVx7plnYu+d2+TgDA1R91G1fr0pK6XosGWe9ewmnVor8q0NaSPVZwt9Y0gPMhgC2aWrFsjRjWRT1neAwP8PYuaQDqN8lTtFmDegYSl/dX0k55aj1rb18HI1l4yDaEJPe5v5lL0jjW5zhJ8huXIOt7/hJOqxb9VYO2KvZYzd3mowFshQC2aGrFslViWBf1LD2N/z5v0WYL6jlx56pRvlBqoGpqPefzpKYjUihDjodiI8mF4p9h7KUjahJOqxb9VYG2avZYzd3mowFshQC2aGrFsjViWB/1vEMM2qvGPEWbLanncz3e+gTKqGlrPeuP2gcYS8Z5BYjkzn4dUasaRJKTvdIsRH+lEpkG8H6cJnlSoLmP/iAjXfLRANa0PGMSSabW0GhqxXK1yvdIvoLz2mSVTq3nhUGb94ZHhLdqn5m3aLOl1DPJa6qoaWs966B9TOkmngNCU3uItdt6WW0i4bS5+3wAoHJWmDZoa8keK7nbtwEg+kjIQ0DPqqbWLFsjhqGLer46qH/ndm1P3ttqno9+hyeAEt4Iy5t61kgWzWDTb9Nf3tRrBHAA5t8qLrHKahsJp62hEv1VgbYa7LGSu7WiAYx8oGdVU2uWrRDD0Ec9jy0P4NdZZyQdKgvRZkvqWSvlrfVsLWoxP/LuhYvLhhuAWJkOam2rfpQITACA60hSi/5KJQBEDeD7CXU3/9vpWrvvOgK/l6sGbTVhXck9bEhEoxYHXC2bWrPs5AbgZOSk8jZHzW1jv2R+jUkDANQHkDJ4Qi2pNChWJdosk3qeUvbfMz+YrdmpRTNra0jzvNb72rU/xjX9jSSbYb0t3IgZp7WQE1aCthbssQZ3m5cGsA4CWNHUimUrxLB+6pkf4qqsUC3arEE9a81r2lrPeuY1L3//576s0/ECAH/ZeJ9o/fLr9fmov6M/6YFyFnLC5hLArAHsCT8/AM9FH8HS/3PKS01YVzo3fMG88x339jZaNrViGQCml9/mbvutr0LrGUj8/Bm5/q1atFlD6lkr5a31rGs10i9zBYCukDSP5+poJeK0lnLCctDWgj3W4m7z0wBGvgSwqmm+loF8iWHopJ4BbM2ooZiGlaLNWtSzZspT69lyXtti6Asot2t8cRbAoOm/ZJmWRUm6FjEmnNZZJSesAG0t2WMN7jZ/DeD8CGCLpvlZRr7EMPRSzwC2mjnr5BuBBrVosyb1rEhWtZ7VQ/Q/MMSTjdyzZfdrp1HGyLQHmzHXVOnzi9ZnFwmnVasPy0DbPNhjM3crgLh5agBbI4BlTXVZlohhkQDWOa/JqWeStWFSGhb4ZaVoszb1LJ/X8tV61rzLflDudSNvuY0gSZ5CT5KMd8YVfnWLi92+N5LkL/N1rAkknFYtJyyV5MUei9ytCcTNUwPYCgEsa6rLskQMmwlgiRjWTT2TLAnT9ovALytFm7WpZ3nU8tV61t4b2Tn9RmzXnqkkI/vV8/btNJPkco/p1yaTDAtq8d2voaND9azkJJxWLScsK9Fij2XcrQnEzVMD2AoBLGuqy7JEDAvV5cSwDVrPZBBWCx9M/LJCtFmben7g7uXj5V3Ku+RhK1rPebDHcQdTmqg5yzt72csXAP48lVi1UyldTy1kOK1KTlgq0cce56kBbJ0AVjW11bL+JzUKrefUm3WUmwIy0ea8qWfLpKX1XJzZY/ulnosxe2zn1HMxZY/tnHoupuyxnVPPhuLJHtsx9exgj+Fgjx3sMQqLPS66i9WR7HU14kiOqBWf9P+j3FsKM6AwmQAAAABJRU5ErkJggg==',
    '1808.00179v1.3.png': 'iVBORw0KGgoAAAANSUhEUgAAAVoAAACHCAAAAABy7c7vAAAS3klEQVR42u2deVxU5ffHPzMMi+y4AKYiZJJbmpKauIC4L7lvaFrfXEp/pabfX6ZllD9Ly0wtc0lTS3NBM81c0kSEVFRICSFzQ4xCQUFEQUaYz++POwPDMOC9d2ZkyDmvFy+eZ56HM8yZe5/lPO97joKwiWVEaTOBzbTVT0hG2KxgbokgqQIAU8fb99+3GdPAIgAUFH5MEoVtLjRiEdtYazFRVf+PkHsgwTOk45Xc1rYVgpnl5+Ge7085NaePkzUOCtV6rD057pQHgCnfX1dYmVmr+4DAcbM8AKBblgK2AcGsknqhGQCgdqhtN2Zu02InAaDeANjGWvPKTb+C0EG9AxVWatZqPY1tGq8Gnl7T2WZa88v1n6L25bqfrwugqOykXKSy7cZMEt8Jm9Mn3NkHgM+eKrN4KFu1TWPSZDcAwPU9KABcvdBcv82gajOtJHmwT/hdjOdx65e1/ifSABRG//TgbmnV6kybu+2tj47hyhnrNm1CovD7x7CmKEJMMOyAC73smvUaVlKtWlc4WE4O9DhwL+ez2YEpFCNgFcmHqjiSPOd/laSm3iGSBU13kyPe0VWrSKD3U1biAm+T5GRvjXWbts+PY2f/cjiiZxpJXrS/S3KT9wPBqkK1Ck2rqmYb87Iyv80L55Luh0coACC6nQuza51tp8LlzGBdFVbmr7XijXlZaQO0aKGrxHfCit61Gl0FP2/vrKta2zRmxRvzSqRT+hrfRviP17KPj3UtqVqbD0HqxtxKzsbu2TsI41n9jWF6VavajdX+yiF6epOmsdVsleviAFweROxy66yrWqHnS29jXr1OdJPeHJab9k4963bP3H1z7drxtsNyExxWJEMMFmW7hF/X8LXIVVxDmy0NpKF2XXvUcGM+sGRjDuDvE26dnZER01N12DVMCeSevNuldtm/SGPVXh5WpwoK44sv/Y05cPjzjr6TwGXKbp90WLAd2DixfovetgFA3pYhKuFkewDJS6IVABYN9snujOSwzJy3XUZ2w953Lzqc6mCOTVrmrkv0G9RAWzt98Ebj0bV2DjHX5yq4VVr2csnTusBq+Ur9zzNzVPZU16iv1eDsp2ewewfic/xHPMnIkWJXCH0nb6/fTRFzYo0fACydVWPwUg9giv0yAGj+0ltmuZWK52/+rJvyxNSuC50AYM6Neb7JXzXYHm+uuzh5xfGzXmNcUJh+6tqS6VfWxcb4TnBRJ58LifCRpCrml6vb748ZNOzKutiYuhMc7iXlThsi3Olc+16fKU9d2NzEdWl8hdOYgXclgUzavO684JnJuJu7tflnJJv8QJJ3cdos7pmiIS1zSDK/R5d8kruDikkW9w2S5QcxLicxkCRZ8PI0kkkYRZK3gupnSFUVilySTEI4SR5yeqWYJDWTPU6SJNergqR4vkql0W4yYheZocwmSU29WJJr80w17TwkCIV0+1dJDvtfwRrmNG0ShgqFm+EkL2IMSXKr9rcEVd2QRz0No7GNJJfgW6FZM6AC0z7slGFi7r4VbgOA0229AECxceWu9R82djVxJLyxILiNUKo35KsU4IowkrX1N/85hBq1PPXqTyLGRI0BiAaQNTdgjPbenyqTVJylyeqjANBH6wTrGppRy9HkD7y5IExXDNu2fhECI/f3AaB4w+ymjc3vj1569TsINFHj32gFYPPdl3VXZSdXmWdjSh8FAKjcdAP0E6ZbFkfwpK7YCEeAN9G328cn1Agxv2kBDNSrb7ebZZrCi5EDxgM4jKa6VxzXWBNfm46Sm9QLfwHtNr8WFQWfd18367ucevHBtbg9Jcux61Cn7Dr4Qw/Z+tJ28GbK+RXjFADSUafk9cbWZFolCko+MZQARvY6EB118Y3CmeZ8l3abNOkTS2p/bAVcX/zcBHdYreeQe+52Y+EuRrF1UuENErJ0xSz4AYDnqFE8OuKD8Z7m/Qr9phIoTm4J4Mnppmpz9Qe+HNMtoRkA/9OZJa/neFkRh9Adv5Vsq9Ed2AYAitDVeWY/nG/dHDhvTopm6P3VANAbpf/qQmtCPEa7H1ALJf6kmgjsECq9SscJc8kTAUBcM7l//f3Oco9+eeJ3ABjps7dQ+8ptlTWZ1mth5jdC6eDZWf5AymUAQB5aWeDNuKqpzL/MGDbsJnDTybGsaYn8IpflWUu1r3w9ThLiYa4tUUWimel+lCTP1Q8vIvl0uxySjPiPGd/6V7xAksyfXpdkPDpJV/Wg9ssaZjiMJ0nGoz9JZtnhEldlkMscvtWQ5N7FlLfRtRjisSlg7snETxt+UUySzxyaMP9A1PQht8321qcH+MF50Jgxw9t7IoyJQ5u4unf9r2RVe+ZcS+/d/y7JxKFN3Ty7R5Bc6TwndSZJRge1/+bXzZM2V6Sr6vja4mPnNY272AtzWRCT4ouDWiuszRWeGZXXJshwl36QA4SFzB/xOfW7u/9b0WVrPWWwPUhq42ttprWJzbQ206Lan+g2VJjj2L2qlghWqaohbIuvf/niK3e9uH57r/9ronhYFsIodYKEzRPXsXGH9aGoXkid8TtBAoRhTEHOvFm+AIB7Ucm3Wo6yB4Cs/X+ixRCDs7UHXZ//FADSd19xDRjkCQBrvdt5K7OP3JoEAGePXPNu0s8BALa8GttK4l2cOXW1BwBo1qQ75U+vY8qAoFNVtlg5qQhTIQwjCor6IJYkucHr+U2HR4QWkVzh3mHLtoCG58v2nFq/kCRXBS47tMDOZRtJdgTsnYBNJAuHt5j3URAa7CVJDm9WIMUzlDVzUltkkKQmfB6ZEPiPbCdTqSr9ooyHmyRCGEYUzIZg2uXorWZ6DRwnD0Bxg4zGU4X6HePxGUleVdY8TYbBPlcwLVB3G0nOxWsaqsNQ4yxJnsBCKfa4f7VoiWCEXR6FJCcNk23aUlX6RVmmvV4jWFccqUgm24wQ/KxDxf1TO5oKpk11wEnyqgKXyG4IIHlfiTJuuHBVLklGAQvJUcAZkh1fmPD2FuGpry5AGrkXeIskNa0DiqX5M7VGGNGXJL9zvGeCa1TPnqJMqxQBYXA9ELhjP8RDGOfe2yAUVqqVQUDDixcaAWfgAsDRDdF6Pa9HtnEHgM6vhIQDyajdDABGr1kwygUAMNA+pC5QD7gIAIqeqb/ImVGKD3kAQN3CaCtYIZSDMCL7hvXsEuQgCsLICf9W+8DWIdS59Xl648HNADhnAwALEafXNaZYCBKh+hrA7qQa3zoAAGN/c2re1hHAjP9xBJACCFG82iKyp4xPmZXjCgAeuNC36je65SEMz6i3g/2Wi7pGxvxX5z5Og/Mr3WcmtowAEIQ7AC7dR7Ze3+N4SlvijF5DW//WBwAw68e69/s9ewIAHAFoVsB/GgCgBRLkfMo8uACAHXJR5Y8/MwjflR4y+ZJkzpZXGwOfihilZr9BpgpjrQrYT94AdpPnnHCZfEOFJ/T6DsRWXfFKwmL7p74nyaW/k1yA2pnalv9De+3cngkvOQPk78JQnYhZVjDWNkB5CGPVn0fqfHD7oV/W98cXo/SKt+sBeLtjA9D85xY9Zw3xc4X+6jIXJXxEQJsZMy8N3Qlg2jMA2uDmKu2oEjEuWvuYlQduy9mOugmsSxHcrMDzJR/C+HPymwlxcWeA5Lg7qIO6dgAckQigS+Ivvb+YkYsWer0dIAAJ/yyPB/A88DEQO3w7ABcIJ/6IHbZog9NZYSgqhIMcJ4qnADjch4cVTGOj5xxQO+hDGCPFQhh/BS4CkAks9/iidViKRrhcvIH8efXe8Ecq0V+vd03tyNv1gvPfnnAEbgFzjx4bDhQBtQHg+Es/hAE/3QAAZKOmnE/p4XsDAK6jnRWY1mvhlG8maiGMd/yBlMuNREIY3bsDwKaxWNkJGLs8I88N9+8gGDjwMYbUwx48PUyvd3MI7NQ9FCmAv4EOQAPlWACXgDAAJ3sHR0aqc4/OBQDclAeBKPqdAYArNYOsYBqTBGEYUbBRmMY0Q7CI3AS/HPI0eql52rnmOf1+URhJkpyPT8jC9qj5F3nGL4HMa4XhGjJedwsLO92vME/a3LMQV0ky2ekyqekw3xSaQqvKoChjoysFwiinYLirm5uLq/NR8t50547BTuHXSHKRb78uriPTyvQsblRf+Ca/fLJ1fx+nMddIMvqZ53p61FuuJtm1ZAgnSY5FkgR7FI7u4evWbOASkltDkrNnvqiWbVo9VXrFSnVV5goXDWFU6jPKv+DwlBZqzb9cI8Aw0s6itxJbAgA0mene9XWzama6v7FBtdD/2f0y/deZ++51qBwhMbcrvMpPGW4GjlohuvO68dEh1eaUoeoPcLa+nNRYZNeCZ0d9AJtpxc+jb/162Flcz/H/7LWznY1J+C8+GTxJXM8NWZF2wL/gAOcRft+3xT3BkO2lqE4nurbDcouZ1u59AKGmagqtutsu1CpVIVQbIAU2MbOEVCkVbqNn/h1iCOg8ChBHZdn4IuJFQwsurMoBOo8CxKlgQJAWX0TqrZTz1Z2CB68/DQB5y65lPDPNBzg6u28dFwAYrTTlLs7dcUHVZKizcUAHyFk5THhoXwaII47BEUHPSIkvIvHhpqzXs8hPHA6RzBp1mbnDPE6QK3VTq0kPN+1377ZvVwvvU8YAnZw9ayZ6YovOOScNxKFYBkeEU1FKfBGJpl3iFElmIZjk5FSSeTUDijh19f6Y2NiY0FRTTJtWw+Uumaaok2ME0Enq99oClJhWIohDsQyOaNOqC8nJeqY9hQamm3aLYySZjyYkG9TNJjkY5zhdIMQ2mvRI3hzhovfH50YAHZKppaaVDuKIY3BEx56JPQgzxxcBMCpvOHAM/QE0KFQDcEUmXgeAK8fHmKQ5RjiybVQmwIwO0DEYDaWDOBIZHOUjji8CALAH1IufexdAbIYPgN9VLdAIgGbqQpOWIDwNZwBwwkkjgI6BtEUk5DI4pqLL5o4voicndx5ps8MFgNIBwOnEGcKku+UZL5PUFhYKplXhtlFAp4xIB3EkMjiqRxdfRE/at7s4LXydNuhl4fjuHwIA1LN/NU2tuuQuLNR79SqMfmE1ITUhxn04CE+TFJieNtO88UXKjHSBm+r1iRM2CXPq7BLyMu5R+JmmtQZwDwDyhcurPKCjLx64TYUlGRzlI44vUiK12sb/DAD4+p99WkNsCDB1DPfUmdYb5QEdw9FDMogjkcFRWjK+SAW3bbs2agA1BWT2p8TvHHEuBUBRtJepqjsJpi1AZyOAjoFIB3EkMjhKC8YXqWjIiv89E8BfaAXgxPFlSmC3I4CUuzVNVd0LWQA06WUWjDpAx0CkgziKfqlSGJwKTZurvY0KZvztBeSakUx17xtVH7iWHNIZOP/ircmvvTp2rT+Af2BqqEaMb5p0AtiZ2Vn/EayOMBq69TcES1U/I+UKwJ0z7EwBk6TFF5HqQxj36bFf2/fLIKkNWtmSJH/EHJNjs1wf7jr8Jbfxd4wBOg+cXD1c3dzdahyVAeIIIoLBqeoAKWcT2bql4SSi3tTPx/S3vnPVzt8FxgEd00GchzM4j9exo1FAx6IgzuNzomsE0LEsiPP4mLY8oGNhEOfxORsrD+g8ChBH8Zic6BoAOhYGcWz0jGVNqwIQYguQYm5VIdrdWChNlIiqy08ZYZWqGGqLPWNbIaA6gjiqilPqeEjPqPMYiHgQx9iAoE2p00Z0Rh15t1IpQzM2qK3ztRM9wyxzF4tFXiSBOHqNRkCcyugZIaWO2Iw6smIL6zE0PgDs52pokajEopEXCSBO2cbyIE5liIeQUkdsRh1ZptVjaHp8OXtNKi0U8Fk08iIBxCnbWB7EEZVeCICEjDpSxJcAXJANwHvKRxP8LTU4bu7oACBkT760P1td0NYF8GuYtVH/1cWq8eUb27derYHMcJW6jDr9HpZRB/IYGkuK3LAzlYI4ZRsrAHFUZs2oA3kMDRKjilSvWCZWgcywM5WCOIaNxiPiqMyaUQfyGJrElOmKH9odtEjOWJlhZyoFcQwbjYM4yodl1GkbrBGfUUeatF+4KS38JoDvwhUY7DrDIletROQFYkAcw0bjII7yYRl1/Ft9GdgtBQD89Q6dc8zF0BzsUwy0BIDndmZZwrQyw85UCuIYNhqPiKM0a0YdyGJo4mIEGyRbwrQyw85UCuIYNhoHcZQiUuqIz6gjj6EZ0EMNQI1aljCt3LAzlYI4Bo3GQRyliJQ6EjLqyGJoWq10APCHR1NLmFYi8gJRII5BYwUgjuE2RD+ljtiMOrJ2Y/2OkkxzCCniquMkU+3XWWY3JibsjBFV+U1xnNyOzsVGIuUYNJaLiFPBRrckpY74jDryfAglDE3R3KVntzZfbCEfgmjkRQKIY9BYDsSpiJ4xklLnIRl1ZDqRSxmalLiaHXws5r8WibxIA3H0GsuDOLZjR4uBODbTWgzEsZnWYiCO7WzMsiCO7aq1BIhjGxBs9Ew1pWeq8IKDLd+YTWymtTL5fyIwOJMNXip+AAAAAElFTkSuQmCC',
    '1808.00179v1.6.png': 'iVBORw0KGgoAAAANSUhEUgAAAbUAAACHCAAAAAC0BJgTAAAa5ElEQVR42u2dd3hUxdfHv5tGSCGBUKUlIEgJIIQiCII06SJFShClI68FgRcUC+DL7yeKUiwg1QJGuhQFpJMoHSVAECmBYBBICMmS3vb7/nF397bd5N6EJCzZeZ48mblzzpyZe/bOnZ37uWcNhDM5XHJxngKn15ypeBLJWc6z4EBpFkk3ACjsvW32bOfJLLY0G4CBwl+hksG5pCm+ZKDzvuaYya20Ddi4+7R/x6ejjc2da0jHSb8O9p896cTMnp4OP0uWovva8ZEn/ABM2nzb4MgeK10zJEfO8AOALvEGOGdIR0nXLjUCAFTs5NwbcSCvYQsBoHo/OO9rDnNfu1srvVP/HvUNj4LHStFqZO2YLOCJFR2cXnOovZHbPx/YaSx3sRqAHPlKLMfNuTfysKaqY8Nix97fCYBPnpAtL+VF52rkoUnbAAA+H8AA4PqlxtI6RdHptYclZe8U/ufiKSTsWxl4NAZA5qGfs1PEooN7zbh++n9/R/Sfj5DXTkcK/7d3bogchLeDK3DpOddGzw2yFh3oqSioTru77U5NXPBO/QvUkECHSP9xO0aS5wOvkzRV30syveE28sV3LUVHSCBpZw2pb7/OQdaQvV7dWKOLIfzoiloArjRK9AZ+mHLTjTW/7WouOvY+5KOyXydPc1v0PX8uY9gsAwAcau3NewFnWrvhalw7SxEO/XztUdmvk6cWQHCwpXCqPZb0CKh7Hfy8jZel6NirkUdlvy6P1D52RdW6GFV+8ce/P2stOvY+pM79OofkRlLdPYS7QY01nSVFB94bqbjc49DkBg0jHuVvb94ewNX+xFbfDpaiw+/5S/brHmFG69xbg4wx71bHI7R7nPLWypVjnGTdw+g1kOyo/CK3Vfh3A6u0ffGr7TyXxZdqm9njw1Du1z0PQNivA24e9e3ghVvh3d32+3R2AYzHU56pKFeIYYl+8EqXZYOdlb90vw77P3+66nhwsUuXT9p+tBFYM65GcA/njPjwfcs+cPp4GwBRCw8ZgPkvVLnXAVGd4xLf9h7SBb+8d9njRNsHsWMSt/UKa/WvaS6d3HOn3vCALQOKdrTpCWK+vHeyeZM/oGoR7wDFJbq5M6tsDbNBr1riWU/dfSox8MU63DCksGtI2X7dohllX1jkB0xyXwwAjV+e/kBmi9y5YQu6uBx949l5ngAw886HVaOW19x4qmjnqaglR86UD/VGZuyJGwsnR6+OCK861jsr6nzHWVWKcoYM33d9Y0Zo/0HRqyPCq431SD1nfHOACwBw5Qc9Jz1+KayBz6JTelcjyi370+S5sNUXTSR5K8W4rvECkg1+IskUnHwge/45A5omkmRat2fSSG4LySWZ2yukIBvgutJxPE+STH/lTZLnMJQkE0Jq3Cpiy51gJMlzGEaSez1H55I0vep3nCT5jVuILsPIpwN1t5GztpK3XO6RpKl6BMmVyYX12oc4LWRi3SeQHPS/wikteq+dw0Ahc3cYycsIJUmuM/8vOstdkEyJweFYT3IhvhdqTf30eS2/Z9njjDuX+PYDTrYqDwCGNUu3fvOfej6FnOjvfNSuhZCrPmD5BSBamPBbBRbT3Tw7CwH+knIdhBfvciIIh4D494NCzbPeGw/2nZoZpvieBgA9OwnlZzvdCihT6E6HpXe2ZDuv/2Y+6m/Y1ROA4fViOmkRaX3wnKR8H/WL12s30QwIS3nFctG093mw3IhLFQMAuPlaboaPFd5pOIg6lmxdHATeQq8uHx/NQsfi8hqA5yXlja4zitVplzf0GwPsR0PLgTIrHOD9tVhY56fy+AdoHTbxwAFUee+1YrB9YkT2jWM7rN8FbiPrwtY9P3UrrqHHbOLdCxeXjDQAsahkPVzPAbzmgnTraYMLgCHP7T504PLrmVOL3nbrtabYcdbSX+sAnxGfF9+Gf0BLGM8n1TMAcEOuQ70rWvN0vCUbj1oA4D90KA+/OGeMf3F8Zmq9QSA3qimAOpOLeeg+gcBXoV1ONwICT8ZZDyeWf/h5yK74w7p9hq7AegAwdFqWXEwgX/PGwMUSZI0HZiwD0APicOc5AMU6vNzuLCHHn93GAZuEwnPixFm06bEg4FijYhrs5i2q4CD+OAtgSJVfMs0HktwcwGvl58V9J+T2nJkRCFy4CgBIRrNi6wK/blg8hm4NGnQXuOtZRu41Iq3Ml/GLzAdWjXQEYnzi1GnhABA1dtgcALnDkwBg6agif6xshHCRp0+5WR4wwljkQ61U8eUA3L4Y6i6YNwJALdd70Vhzd9DiD9YQAHYanngg7HHR7iuRXBv0/vHIT2t/kUuSTfaOnbv7wOQBSUVs+WS/WvDqHxo6uI0/OjNyYAOfcs9OK+ox75h5I7ZHnxSSkQMb+vp3nUVyqdfMa1NJHgpp891vYePDHgx7XAxPCHN/v2iq94y7sCQJ4blTuSHNDY/kU9G4A8ktQpR7envYzx8A/jqVWKNrudL11mFpfJbtjMgEZwwEZ3J6zZmcXnN6zZnwEDBatQ0PgtErOTqwdFmuDefK37nyL3jaU1L779uS4IiW87rWNHOmhf3c5QROnlYin3hT8CkvaZG2wiCYXApq2eSi1fIDiw+ZOzdswUiXo32knOmsmhu10MGx26J9gvr7q/LWlLh0kBSv2RE3Uilq3HTJrcFAycBWVm5d2eXewYTx+ZtPXnzjVpM3BS7VtPlCtsvrlWxVATjcykt6MOKdXpW8AWC45HQfWrJB8xmVNy/TVHTkcCuvPDtq+u6fdNO4x6Fz91gPZ6po4Ov6i/d+5Oq9XpkXUuKOFeP88aNUodswpeiucl12bg2ufEKUeRpw9wTW5r+HGz/0Ko2D/I6SZHK397O4pp+tKpIctVd2cKn5nHSStJZat7fm3WN58zJNRUc4am+eHTW9fpb899mjeilWPZypvIHrLhVOkp3hbpTnLRxp74kfQea1SwhXqMWU9U4hYwyVEqVeA6qt13DuXr1GMrlCUA5pGhRK8vFKNqpIMq1BjuzgG8t2hUdEhHe6Jmltrg6vyZuXaio7ktYgJ8+O7lhAkpFddXrtdtl2luwQQxTZ4kXB+sD8R3AAmEcOBf6U58V0Te61qcEmhdpM4eMeiM9Fr/Ud+/aPKVrOXc1q90i+gPPkPpwmefKIjSqSXDdNfnAySfLbNZLGjv7QRrvXZM3LNJUdWTct746+N4UkU+roZI+lnCm/Aepv2gWNnGmH0R2HAVGo2Eiet5fSv3nVoFALhy8A1JUiwcNXfDRUUyCXmplZAHwQB3zt9ySAlm1tVAHAmpfkB18DgOgjoWJbGZuH6VgoSJuXayo7sualvDtafcEiE7Cjl877Wl+sFj8nIeRxF3SedyRT8xPCrSi701bexrX2rbdRoWYqgyEk2Rs1xWstLHzR1xEZGj7xuZkk2cwtjqaAJ6/P/O+Mv21UkWRcCxsHc3vfk7T18Q3quNakLck0lR0RLOfR0fQgdLi4rbdR5wzZHFusr9igMsl1/gCqfKFpBKa3urs2/0udt+21NhOVaunAKJJ8Hr6i12pOW7+oXIMjGp8on8AUMglPz89ldPV96iqS/PwTGwfXvi0R/WMV9XhN0pJcU9kRiWU7HY1pB48OWXrfqQnBD5bsb6hKkok/TqgHfKppBNGnP3N/fLM6b8trfyBSqWYExghe87CKLTpL8iNUjNN07jKadE0nb8A1huQrddJVVST5VKz6YGbNGFE0661c3V4TWlJoKjsiWrbX0csvTfNE91s6vdYfiyzZn9DacjEcrOSbqHEEbwObbeVVXhvXTqWWBeG9sm4oL2/0V+BDTeduSucUkgkIIsmF2K+qInmxq42Dm2pJRBdcoW6vCS0pNBUdkVi209GzbeN5pQta5+pbjRSCM/33y1MAngI+ludtJ+MPk1Rq7v5IBYA0VLaIRQzeCMAbOKtlWbDq353eAMp5VACAsohSVQFYO8LGwW+DRNELmQFJSUk52UnJ2hckQktKTUVHRMv2OjpuXkXU3Tv7xG59xPjwmbuzPKSc6RDNnOmzl7xu+qMMkCDPw3gjWL1RvsZrkFqt/c+pwjsAHSxa7x/+fTCQA1TUcOp+jvzBBeddGrk1SRViOVRUVQGmn46qD+Yc6io2Ex/7HoALld+rM0Wr08wtKTXlHREt2+vo/agOAAyz9l/qpW9vZAmWm+N74l2Sja4IX+MQm+9sUR0eSeRKYIQ8H1cFk1QzpKnxdBtqX6AJydzK2GjRGuEyneRKYEP+89SRd0wk514h57hnkvwAl8iEdHkVw4er5RmJ0eY5zXqHqaJjhpS0JGompMs6IrFst6NZvsLqMXSvzr0R09Ryh0nyfI1hOSSfaJ1IkrNG5T+CufiEzGyDCv/I8/sgzN4yrx02XLWhltYQR8iN6JBr0fqz1mkyuRkGm/I9d3/VGT9hwvgRgTnkHf/tpKnNWDLeK0RexfE71fLchTeEvSkvyyZQjk8HzV6TtmTVjPcKkXZEYjmPjo6dQ5KxPTN1v5etnTOVN2D6qk7zPlU8Q28o8undyi0nme3p4+fjW8637GGSHNrTlhpvD/YZ/LLvmPtWLR5q0rK7X/Uvs/I/d+Z3h5uS5ImWm05NHH6fTA0eIa/KaJBtQ347Zgp7iMEjhMqJT/n4dflYo9ekLVk1haasHZFYzqOjGS+PPxS5/JWYAlCsmjlTZQOmuNjKNVzUeXW6U3NzX5tquH/dNVC+FRIXG1hB9/OSpMN3WzW1VbH5t4U2jmat7V2laJ4RiR2xbVnZ0csn05q2MjycFOt/l0W7lozlF94LgUNaLvln2bnLJ5RQOPaE6y3guJZL1mu7/h1TQpY3vGhwYMsl67VfBlUpIctbQuHAlkv2vnbfzauELN+tCAe17HynBg5K1rnOBtCpsC11KqlzVxotdzJH0IUzOU7qWKLvijpnSAdnj0uOAC4x6LlI2GN9oWYL9LkTwdyCc7iF/cSroeeMMoYHYVlJMauoZk3Us172OFoWajZ+XUFCzSYuv5+e/doTAF4KaeV142j3ztJaCZgrEMD2xbnqvI/rdE18lhRfvr/9YlbgQFl/Uw9EJTQd6g670HO618ggX9+yBqCRfAcj8cMZVXUMWUkxq6jmw628pENW1ZtWxHqmTa4EneyxnlCzNhuIfy2e/MRjL8kqANzflz1jkYK5AgFsVzyrV2gu1zeN10k9H3hy/s7tEzykcOO35Z9au//FTjn2oWfxoff3chK7JyI0sMfWMSgpZhXVPGqvbMjKetOwD8nT9f/Vyx7rCTVrs4GFnhvIeLQj2e2rd1Zck9dKwFyBALYv/qH7fZJPT9RHPSdX/ZEkJ7inWqu/RI8sxpbFEfvQ83aL016SP8d7B1q8Jo5BSTEry2kNcmRDVtZv9cskOX6QnSHnE78pmx4FDTVblQC8cQ9A5UnKymNB4g7B9j6ueYovb+oL4Klln+V784s23dvfEpWRHf0k/rh9fIgBaLPsQktz7fUpmOOOnAwRRlkW3F6hdTXkbT9PA2d4rJDd3jZv1Tlkl/EA8N2YQMvGoaK8vY+rbMjK+rCnPQB0HJ3mpW+GFK61/TuE3z+xXGv70EUzr5RFci+mkVRdn+nTTCLA1DsyT/FEdCTJefg9X8vZozvGkE1QMZP8DegaSb7sbYXKpsMlh+SVS9aLvMJXSq3XPiHJz55IkJ+LRsc1XWviGK6Q5NXx1gtWWRbGLA5ZUZ9TfpjA3v9SoBly9g4znmf22gTXPXqIzsweLZNIhp5Z8MmCJDtIr5kAtiue4foMSX5i80dz7FPPmQGAYfQcz/USMLfKnXdfnhuVB/TMqDiSEb6XZS3eCz51TZvXxDGoKGZZWRyzKC6tv4VxAj+8UPcMWehQs8e3HGyxyRtA5IXJhp9a77H+CNGfFWuKUuuG5i1eJjgBAK5DyxccTo3a3zysAQCP9QONXI0pg611MfAaPS3g/2a9O8d8YOlL5ZRajQAkD50se28sN3RayHW9QwbwYxN5pE5J2TJmqbi0PhneAOBqNzhbXtdabkx367XWcOHChSsiMnURnaa/e/SNJxlJki0G2EZ6RQ7Xjvg2wy0ysy0+1kc9n2tdDQD6Wl8NcAN2kXeAbXahZ5Lk+7gqX4m8Tmq+1ixjUFDMirJ1zKK4rP4sppNkJGYUaIb8eQeZE0leRu8CxW+7W6alZZ09HnG2kF4pAWxbnJ/1uXF19lytM6SZdD7u3Tc1ojWAOZbjleCaQ7IcXrALPZPkvbKPydaPmzpm6fCaOAYZxSwvS8dsHbJU/hqmCjPk3AL9CkNhQ80GtDr1K46FA4CvBayVg7kSAti2ODDly8MH37qjIeKnlHSemvONV/tja8rDupivhGquAMog0i70DABb0utI149/v/rW6WPH/gSijt3XPmTIKWZFWTpmi7hM3l+AhTPgh4JEq34MBQ01m9U+55gHUAGX0c+Y7AFkIcAG0mvhcO2LA6hdG4gJbKaLev6jbgBgGNG6QYqFeu58wQQAOeaVv03oGcAWq2HjjWAD/qk/H0Ac8KXfF801D1lBMcvL5jHLxOXyflXvAMBttEZBVv4kTS3vFWSGNBpc/yHZHAfZdRVJdvXLtoH0Wjhc2+JMSCfD+yeRSf6rqIt6bl4ukySzvF+yUM/HYbhPprsK7zzZhp5J1oOZ1BVZaXKNlhlSMgYJxWwes1i2jFkmrqCex7QgyfkVcnTe135DX+FLzeRqJE+hvd77Wu/DJGM8Oubw6yMkr7mvtoX0WglgW+KC/HSXc+Ts5ln6qOdVGJ9Gmv7jH2Ohnk0DMJ9ci1qJ9qFnkmUx0fK2pYWV1ug1yRhEitkyZmtZHLNUXEE9R3leJU1t5+rb0dIXatb2PuTIT3//rU3vW2TO+4vOrGv8mckG0isSwLbEBfnzvcL/mNb5DvVRz/whuFK3PoHP/y1Sz6mTvZ5u5znsBvOAnskQrDRvBZipZ3Kwj6+vt4/X4Xz3Ia1jEClmy5itZXHMUnEl9byuY9S9qSOySiCC7plINm9qAIALxyq0rZIfAWxXPGF3UvO22p4RSfFlxt/0DnKXVadd8njcIx/oOeWmpl93tz1mcQxKilksS8Ysiqvk43amtm3+kLLHJUYAFxZ6Ls3scYkRwCUHPT8C7HGJEcAlBz0/AuxxiRHAJQc9PwLscYkRwIWGnp3ssdNyQSINdnRG0HUgyx3N97VOLGSaxZJKpdByJ2fcYyd7jNIWfRglRj3butbiEt3cmVW2hhk59qolPs5J3X0qMfDFOtwwpGiiD5tciuETb5N6lhx0KRHqubDs8cV91zdmhPYfFL8uIrzaWI/Uc8Y3B7gAAFd+0HPS45c+b+CzaEj+zYsIsSLasDLIr8AeAxYgWSmugz1WGNZGPcsPSqMVqzpuN8nCMqup5/hdfyN4QBlVqGfAijXbCuysUsv7+VonGIWHbMNIcq/n6FySplf9jpMkv3EL0fA03ooQK6MNK4P8jtorA5KV4jrYY4VhbdSz/KC0WtVx+5alYZnV1POScm1/XB9U+yIV1LMUa7YR2FmtlveTmi5IlgJ1w7Ge5EILRm3qp8VrVoRYEW1YGeRXYI9FIFkprp09VhrWRj3LD0qjFSt7krfXLGGZ1dTzbhjukIfweKacepZhzerAzmo1fV57F6+ScT5BlsB3+7R4LdROtGFlkN910+QxhpXiNUJIcqpPqlavhdrM2ghkrLSsjFas7EleXhPDMh/GZBPJ1Tgpns0gkhkuCJOHeibJTQ0tXlMFdlaraaR9zOkmmgFhKb0t0u19UMBwwIAqyK8QfViMFKwQT4r1AYBKKWcexNJNFo5YaVkZrVjZ8bySGJbZFYu6nwUOe1upyj/hDaCMLw5BFuoZwPkPvrUb2FmlBl2/TX95Q78xwH5Yf6u4zAotapEHctxG+wEROR4AzroFm++kB4P+We5jHG3+HYb4W+ZARJ+/IYxEIV7WlcI298V2Wl1jMSzPyozYtCyvVvUkz8SIPzwbtyoDoFVAwr4nR9Ve/531BQmvewDATByz7PtnmimtxGHfB6hbsKem2Wsxm3j3wsUlIw1ALMTXqeppOXcWhNjFA8DJyClm9fsJDTb+n8u1Dt91ASByuFYgWSGuiz2WG9ZIPYsHZdWqnuSZZgxpdav3Y6vb2qKeQ2LvA7iSgXuQU88yrFlswZ5afmtI633t+WvXzkxq+TtJtpL/4EW+9zUZQixGG1YG+TVzuHIgWSKujz1WGNZCPYsHbcU5lvYkD8vSsMwq6vm8J66Sr7vhMQX1LMWa1YGdlWra72s+gYHNvqrf5QKAQMn8nqjh09cUAFpuiQcAzKy01dN83Bu1agFoFn0EAP72EX7Z/Mv/kXZFIo5+n477J/qj3tD8hENiWNYHhRG1ZXkf1D3JI73ZBEAL3P0aOPFUlSsRrYEd1qjBjX8N7j5jQC0fy2S1tJ05et2Rz2y2YEdN51PRgRnLAPSAGPN4Xv5KMoRYDCysDPJr5nDlQLJUXAd7rDKshXoWD9qKcyzvid0kDcuspp7xTOS+Hl9MMSJYRj3LsGZbgZ3lanne1zYbBgDy7Rp/nAUwZOYvmeZ7ZZKGRYwUIRYDCyuDAFvYYxmQLBOHdvZYZVgL9SweDFHHOVb2xF6ShmVWU89pH1Z/PRDXiD4y6lmGNcsCOwuxnhVqed3X/oUhnmzqmSX5vnYaFUxMzd6IeWahTy/mf3eRIMRifN+EdMqDAIvRh0UgWRI9WB97rDKshXpWRjcWcqowyXlbloZlVlPPm4FYcjGeyJZTz1JAVtqCmXqWq+X5LTu74ism3vIYQ5I8hT4kGe+KK/z6Fhd7fG8iyV8+07AmEBFiMb6vANZKgwCPl/4WigAkS8IB62SPlYa1Uc/y6MZCTh0mOW/L0rDMaur5JJ7L4kmvCucVoZ6lXpO2YKae5Wp5743smHkjtkefFJKRAxv6+nedRXKp18xrU0keCmnz3W9h48O0rOREhFiM72sGa8UgwCJ7LALJknDAOtljpWFt1LMsurE5pwqTnJ9laVhmNfU8v2rvZ3yGmN9PE6lnGdYsacFCPcvU8mGP4w4kt1Bylnf2sJ8/APx1KrFG13LanlrYJY7FIL+2YwArkg72WGVYE/X8QJ6KSsIyq6nntKtlg1xthHrOJ7CzRM3JHsORqedSzB47OPVcStljB6eeSyl77ODUs6F0sscOTD072WM42WMne4ziYo9L7sPqTI66GnEmp9dKT/p/04MzbDYPCmQAAAAASUVORK5CYII=',
    '1808.00179v1.8.png': 'iVBORw0KGgoAAAANSUhEUgAAAeUAAACHCAAAAAANEB5bAAAZGElEQVR42u1dd3gU1fp+N42QTglFICT0EoMQinSkKE0QiFKCWKhyBRH4wRVFkId7LShNLyhFUTBSBEEUI82QKDUoIQSRTgwGEkgPKST7/v7Y3WRmts3sLslm2e958mS+mTnfefd8M+ecOed7z1ERTnF4cXEWgdPLTnEMIbnIWQoOLItIugGAtW3z4sXOwrRbWQxARc2fVaJyduHsV1R0tssPh7g5i0Aj2dGnA3p3v5rd3tnHdlz5+dmAxdNPLhjk6bC1trNdxokJJ/0BTN95S+WIHnbW2ADACfP9AaBfugrOGttR5drFNgCA2n3g9LLjehm7CAANhsHZLjtsu3wnqKDPMwNbqBzZw04vY8vEYqDl+p5OLzv02NetHw7vy/a7UB9AibhHWuLmHPtyFKk3KSplUs4+AHzspKj7LVadva8qK3sAAD5vQwXg+sW2wmsS1enlqir392n+l+Jx3D24IfjYDQBFMT/czytXHdTL2dvm/fc3XP3jIfDy6QTN/+/7tkYJYrvBFbj4lGubpyLKVFT9KAJQX6IHROdnLn+jxXnKELBKy3/cjpPkueDrJNUNDpAsaL2HfO5NnVqVBSSN9LGVjetW8T724Fd2NOynij22PgjA5TaZ3sDXs2+6sdGm/lrVMcexHX1cVyxLOzx9LrFw7CIVAMR09mZGrTOd3XAlrZtOhUPOLzv6uK5YOgChoTolvgfWDKzV9Dq4uouXTnXM3pejj+uakB4p6+s1xUs1Vr3/2xNlqmOOYysc13WouK98dw9Nq9Vwc1+B6oBjX7XXecTMatU67mH8evb2AK48Q+z27alTHXZOSjCu+xDGcCa+HpF9480GeAhmK/Je37BhojNS1xG8DJK9pR/SuzX/krFR3od3Y2dZ2q801nIrjkA6rjscgGZcF7h5zLenF1Jjn3Q75NPXBcg+kdertjjBDdrVg+tEIgJi5EtKOK6LQ6u715sCrnLp90HXd3cAmyc3DB3orKGr/qjI4dMnugBIWhGjApaNqJvRE0l90zL/7T26H35865LHya62HBFL232ZQc800mqn9t9uPq7WrpEVWwoFd8uPa3jnaiehatVTVSyMtEw3dxZX99fk7xUkcE5+dHxm8HNNuH20rfrYonHdlfOrj1jpD0x3XwUAbV+YZ9PaqXRp1PJ+LsdmPvGeJwAsuL2kXtK6RjviK7aeTFpz9EyNSG8UpZxMXjHr6udxsfUmeRcnneu9qG5F1tixB6/vKIx8psPncbH1J3nkJ2a/NlJT13LD24OmN7sY1cpnZbylvS/plNJpMjHq8wtqkkzNy97adjnJVt+RZB5O2XROqmRkWCZJ3hvQ6x7JPeGlJEsHh1szAWORnMBwkmTBi6+RTMQYkrwb3jC1gpH0QTbJRIwlyQOeL5eSpPoV/xMkyS/cwi0CAjOAmu4hF+0mU10ySFLdII7khlxbeXkJTmsOUtynkoz4P02RV7yXEzFKc3BnLMlLiCRJbtX+rzgk/ZAryH8ctpHkCnyluaoeZpmXzcWKTM7et8Z3GHCqUw0AUG1eu/uL/zT3sVFDdPvdbh00Rw1GrjsPXNW0SJ2CK6mXcr8YtQIEehPEVmqvKQQxANIXhkRqa9+ZD4bzOF+dPkgFYFAfjf5En9Ra1Wz2K6IK+uoO+277YhlabP9pEADVjEoq1bh7Q/GUQM9Bi0r18k20AxCV96LuZezh82DivlzqqgDAzVfXmD9iOyfjFzTRHTbFL8DrGNzv/WPF6F1ZXgYwXKDvcJ1fmU6+tH3YRACH0Fp3ptr6KshfTkFZ/VgDfwOdo6YdPoy6b71aCVhOjr+ffHxv2bfVLRSf373/uwGVVDI3vuWd8xfWTFABSEFg2fnmVdDLLigoK1a4ABj9VHTM4UsziuZUPJbOW9Qpk8u0P7cCPuNXV9qEVK2OyD6X1VxTj6K0Sq9F0Oh0uu4wHUEAEDBmDI88987EgMp45oJmEihNCgPQZFbljlb5BAP/i+x3ug2A4FNpZecza1S9eOz++L1sWBX9gW0AoOrzWW4lBQi3bwtcsB8uxajCzwBgIMqL470qGHU/zi+6WHPEH9wmA99qlKfKK/KKlUdCgONtKqkwdu6SLsoVgLMAMLruj0XaM1luVdDLNd5L+1JztP/M/GDg/BUAQC7aVRokftq6cjJOjYi4A9zxrCbyMnGvxPuT9JXaMxsnVEUGzbQ5c2MBIGnS2HcAlI7LAoC1L1V4mEY2NJVKweybNYBsZFd4UQTWfqEWbl2IdEdZ/kGuGVex+Q4iVr29mQCwT9XSptyKihlXJLklZOGJhA8bf1xKko8emLQ0+vCskVkVjOTUsCB4PRMZ+WyXAPRlwqhWPn5PzK3oMtm7IDll4NA8MmFUa9+A/otIrvVacG0OScaEd/ny16gpUbblVqDiZsxLf7ugbt7LXdMFC2difGl4e9VDGUWQdji3Q7j41O39HKb53PgzPrNhfz8nS92JxMlSh3N9bKc4vewUp5ed4vSyU2AnMZyNVbaI+bWf6GMnEqE0hvNLyvklVfGy3w7mg/Zk2VWR2ASOqXdZHA9vIiDeVs9tSfCsuYavFFZTVdAbpA6N9zJ6zeVBv8t6OZiCY4v1sUuXRi2f4HJsqDYeXhMQv6jRDhm0h/Sf/kLoSO3kSv7hpLthY9wlt2QumV9PL93etAmGzRR4TQjx9a2uAtp0MGvFaAl++XeBenIz4WM88zN/6V1HOpWXau6q5NRHXysLvI9Zs70853U5BfdfbWl54Rs0IMxBH87z4Z28ko892deYDuWzFZJ4eJMB8RIDa/y6frMtpPEFkuSmGo9vOfRcnxKJ9UGI07czYKwRM0llcL8ybcXEHIF6xlnynyeO6fT0OVM6QT+o/qXylZ/Sx1xhdoS/LkV+0yHll15NJz/wOGDxbIVBA8Ic9OGwLgD3hWqjugVR95J4eJMB8WID0VDdJmPQrIjkJxhYzJTqOCpO8QYMePkiYo2Y+b7s8VWbtmKibPcuJ8mE/jq98HrJCn0v32tV/jy+co1kbs0Q7ZmlAh+s8NxOpqObxV42aGCpnpeFcDjgf2+sv0bjunEgbnLi4We2Aa66yA2IX4bgOsDjLpd3jsX12XjHHSWFqCO6ZeduQwk/C+1hxMyV8H/7e6o432O9yqwVY3LiHgA0varTqxlkXX8/tHylvh92J9WAT+/vLrQFgOMhAj5vPQLwRobFFbYhA6Ic9OGgznTxRamufFREGA/PLwC0+PYnyAuI/wPeAKr5IgZYW+wSDjS+dFG0zs65tzcZSFfwxSsqI2aujI4Y0LPHqYxd1cxaMSoNlq9UA3sHm75r8/Plx42KigH4IA0ACneOFdw2JvdZ4DcMtdjLBgyIc9CHA9uv0Pg0PtcdHkQ4yRMu6Pve0SIZtVNDhJJUeyKMbI+6t998YWmS6IaM0PhrBmrsTd7ZxswkpZGM871k1oqJerIgBD0v7BkizEO/xk7rIFBKi0iynVsaSb6fzC7i+rRoYMcsq6IIJAb0cxDDYeSZ5R8szzKuK2+X22NXGQUSdUhyawCAuh+b/0XDEUTyItCQrImQIb+cfdb1bVGfaRMNebnLNKNmSDKnwUKatWKqbG90g0fPYpr08uoPpKlOYjZJ/r6RYh8cn9dpap41sSJSA3o56MEJjVJzV4vrRnXlXg7H17rDX1GPJJn5zdTmwIdmf9E5T1whZ7jhEdIN+Im8DewR9Jlm0JCXf0eCUTMkuRBXRD0vg1ZMle2l5+d64slUk15+PEVyovDR/gUki18vlfpA/dfAp9OteZfFBgzkIIWTQJIdRhrVlXv5GazUHX6HzmXAfgn0zTT7i46ENp03YlkA2pGBcC0h6YcRZVe/7V1s0MuTpR1WgRmSGdUfEfavjVgxUbZnu6bzcj90LjXh5Qv9palm980jyeWXqe+DO9U6llgV9yU0YCgHfTjkFKSZ1BUxW6Xx8EoC4nslHBz48exshAKBqO8KoBoSdBf/euX108eP/wEkHc8RJsr+erpxMwB2FTQR9M2MWTEhk9+rjaYHFp+MNnHPlvGSExv/2ecN4HxRraysrJL7WbnCi7U6xf9sVZ9IYMBgDmI4x2MBwBdJRnQLGDTjFkQXewji4fHtaLkB8feWNJgRjGvEUKDveTUAlKAOgOzkUBX+brEMQBrwif/Hwp0zN3tFGDcDYBd0a55mJ4eqjFkxLjlJPQGoFh26aLyXrf7umPjEDwlfu+CcS5v0lLcAnK/zVpPZAIDiHiXHPYCauGShf/UMSHMwAGdYdq4HUFxWDlLdkrGvNVinG514kyTbXCZJ3kKKudppJ5BCrkLL++QJqHLIAlfMJtPqYrruns3Sulbddp7EqNAMyeZ4SdvvLDezWUGNXeyr6V1HHiDvFhiusWPHidMcfUNNcull3VjTEFKTOFvl+jfJ9vjFwhpbZEAHR5eDETj9N5Jkf//72hQC3eKxL/UcvyMkea7h2BKSbNk5kyQXvWT2F53CU8U85VXzHEn1SCwjtyAokzwIhBj18hHVFWmMtMAMyeqYpvuwKzOjxMuc9A5JpgwqYrqXdgDvPYg7qFP2idQ/m0yZOnXK+GBt21ni05PUJh5yhOQNj94Wt8sCA2VwtDkYg/PpUZLX3D/XpSjXrVlXRBQPbzIgXmJgWb0hvXxG39AMzc7y6t7Nc2wyyYIBftrq4VkfX19vH68jgjRjBulZFZohw7FB+92rM2PAiqmyLXxhSkzCuhdvkPmh40kWjRtQz7fN8BWCO1qJXwvt4F+YRpv2uI9/v/e1idMnfPjbr12GpFo+jl1uQGOxPAdjcEoWrjyzte1Hal2Kct2qqHthPLzJgHipgXtXqoeUjczdu+jRzCwN+HajnU/rN/BCM3k35SzkbHK+79Kpe2GdTBjZ+esK+S3rmQS2D1NZMfNozoABOOeP1+xa14Ru71H3//3sqmulR2iMeCvcnmJFbATHfmJFStdNrfztfO5e72BPQSK2hGMfXv7pn4mVD2L7c3a1FYst4diHl3+MqFv5IHZF2pOTbQrHPtrlHDevym8N79SGPcVw2gqOk/OIhyRS13UxgD7WWupjL2XrRGIYCMnecIrjSm+7WIvAWWM/PNwKO2M07D/5MHArlC3xb/a5lVIGFFEIHjyjAfJJHg7FrbgqWuI/fat1S/xLKQOmKQQQkx6EN6vXp3jemxUoP2cLWR5GSR5GrGSujVC6xLJBYoTpgpGSQ2zArVCyxL+Z+RcpZcAMhUBMehDerB67hDzd4h/ZM0GWsjyMkjwMWMncu35yAL5RGhFkiBhhumCk5BBbcCuULPFv5hdJKQNmKARi0oPw5t3+RSSnRMj1sqUsD+MkDwNWEodMexfKvWyIGGG6YKTkEBtwK7Rynx5WL/EvpQyYoxCISA+im6O6ewDo/fI9mU2VpSwP4yQPA1ZCf8D1N5S3YgaIEWYKRkoOsQG3Qitx+2HtEv9SyoAiCoHo5tID/gBQvyhGZtYWsjxMkDwMW7GNmCsYeeQQWLA+dly41Uv8r56pMqkDSE8Nk5M4PdMHAPxxUeYv9coAABbhOHAAgXdXpzQfIVoxN3PsVwZi47YXjVdkxTJJOFzi9rK/goJ58YPXd63/a0uUCRMWvMsnx4/uurj8Ab91Kzl6mvIl/v+o3cikDgBbx8hKnAtvAHCVvRRqOHIAXC5EBnADXi/3n5MQtkhwvTRyrqGJ+rXP+ymxYqGTz8/6v+DONxQUjGdMt7iwD7/zM25CeR87EaNYeuPJvVpeAlqvWLFifVyR0hgnKWVABoWgPLhScvNZzCPJBMyX2fuyjOVhiuRhyAp5zYLel5QYIaNgpOQQ+dwKtwe8xP8n/3IxqQPAXz4NZCX21ezfUAJfmZm3/flfT4661M0nKxCoke46AKjjl7NpWFnP6+gBQ6nWdgtTYsVCCQOAjuvSA2UXTOLU72tPm7p/+DEXIyYsbpfbFwEXToZZ/FvOF9XKAkruZ7n6GtQBGGA0GEkcoIn4L4S/3Ox7JSRfm1l/HkKBwHR9lsf600AqkOTWxk9I8linxIqFcry4l4YY0QdyC2byB7XR9MCSxdGDDZuw3MuPwLol/qWUARkUAuOJ/evdBoBb6Cwzd4tYHiZJHvpWLBQpMcJ8weiRQ2zBrSgbFVF3zCAvYYgVLM5yUoIMCoGU9CBIPLEDSS6rKTfW3RKWh2mSh2ErFrTLQmKEvILRI4co4Fa4VMAS/6X5OQDuNOoh0k1V2EUoMpR49vmrAHfNlhvvGYSn6iD+jZo73YBOI/kZsLM0aCFw9jZ+MvrteH6qDayYlYjWAK4fWeEmu2DcR68EgJsZvbQpBCYsfJeVLfFv+rkVkRLMUwjEpAdx4q29kzLmjC+WXatYwPIwQ/LQt3Lf08ffx9fPt/oRJe+ygBght2Ck5BAbcSsqZr5PEaMhbV9+1/YKGA2KWR7mSR6yuCIyysQsMUK/YKTkkCrErbAvRoOtSB5ObgXsl9FgFyQPB+RW2BejwS5IHg7IrbAvRoNdkDwckFthX4wGm5E84ORWwBmpW7FA3AD0du5o4MBIemvb5T60UhbRXsSJRE/6OPetcHIr8HAyLByR5WHoXU7LdHNncfWGWkqFV5BgPDw/Oj4z+Lkm3D76ATy3tiESVDYGO2N5GONWXDh4fUdh5DMR6VvjYutP8shPzH5tpOad54a3B01vdnF1K5+Vo5XlJd4ighvP+bjO84b8PSMM7Dqh3IE7z993mRFoGoSY5VEG22o4JvkZG+p0ruOS8cvdKbCK5ZHz/YXi4FF1Zc9J9UG2ZpJ5LEke8Hy5lCTVr/ifIEl+4RauZH5ZukVE8eDIUm4LS5e/Z4Q+sUD57FjugIXF3DzMDAghhnLYyuEoYnl0B9w9gS20iuVx+LFl+76f6rFZNreiH3IpoFSMwzbN5L52cxD1MEVelm4RscQ9h2T3afL3jNAnFij2sjoikmSzQNMghBgEsJXDUcTy6A4A9beR1rA8cut9Q5JT3fMt9PKbeIVkmk+Ibtnhg+EKVwkWeblhOEnO8RHD4VbBDHaj+hkkR0C7PiPfmk2SeU0s9/JBnCZ56qhpEFvnGoStHA4k5RlCstAFUeQ1D5wgr6twudzLT0/69zeG1lSfE6qWbeUIZqlJfo5T8mNFRHIT7QBE5Q3R3d7Dx4r2MSvFBwAC885A3p4RsIpYoJVP/R8D0LGraRBGWB7WwjHHzxi3/t0x3rCO5eGKlU+eBY54N1Ma3aeRS9uHTQRwCK11Z6qtt8LL1V2pmSi50M0okSCuxAPAWbdQGCMWKBT+EvL3Op/sl1uYBGGM5WEtHHP8DMb97tm2UzVYw/LoVOvuwcdearztywC5va+yGrvHju1rXu2/SU2S7bHd4nXdRTV2u7YkOR0fUdaeEaTBXSeUIclC92WlvNrgoEkQUgwi2IrgKNrLo3ujudtW+rU6SvlbeRiwctAfAGarlfe+hl+7dmZ6x99Ikp30IxUt8/IeVSpZ1BXvy9ozwtiuE8qQJMP1BskXmxSYAiHFIIStDI4ilsfKsyTfRe00q1geiZ3rA8DThZb1vsZ5JpHks1hVdkOGNV7mR0OTryxeio2Us2eEsV0nlHn5rmZV7RU4ZAKEHgYhbGVwFO3lQZL8GVhC2Vt56Fs54f10flxnAO9Y5uWdmEmSG/Fi2Q3zrPIyr2/ekD0D8cJTb22SJNkwRvBQdjlCUr0YP1rq5fse4ST5KVabAKGHQQBbIRwpktJrh1NKVYgk22j2TgpEWQ89NmI7yV8BMf8+y0vvA9qUlR7V7pDqzTUQLsPL3+4k2Vfk5UOaaau8uoG6H5q5wDovk+SwYGEkamnbHPHlvTNKyUTtfmPZPprWpucKi7+kwluR5P8QZRyEHgYBbKVwxEjy568meRX4hnxVs3dSDTxOMuusmuyN+iRjAHEz/HHtQiqw4tWGJPmXqqX5L6nUiIg7wB1PYX8vAGeJeyXen6Sv1J7ZOMHSvm5GIRA3IhvIjl0o7OD/1k7McTt2dJULsKeaJkV1lWZvlqBQizvZw64UA7iNjhqLhkBIMdgOTvT7M28Ce9EyAngeqblAYQ66Aektw14FGrk8D+AyIFoIhp++XE2JlZYpxQAQ4tXZfB/7fu0X1Uz1mEiSjMdQkkx3xWV+mkqu8vhKTZI/fqS096XbIiLdK5yc55JILm5fLGvPCM0mDeW7Tlj6Lt8O+J5Ud5mk2/bBAAgJBgFs5XCU7OXxR9BpMrcdnlXL38pD38pGTLlHqv8TcENGjb13QXLKwKF5JBNGtfYN6L+I5FqvBdfmkGRMeJcvf42aEqWsjy1gS+SHjifPDY79fW7f26L+tNE9IzREgnJigcXj2Cc7fhs/bVyOzqI+CCkGAWzlcBSxPGIe7fikf4NPxF9milkeX4cGDhgaPPwvWdyKtMO5HaTx3rf3c5jmY/vP+MyG/f2snGW7G53VvqtKGcPC3K4T5pFkHbnTKcwECEUsD3NwFLI80lKCa1rN8mD6Te8QdzuO7rMhkcDJ8rDXWBF7YFg4MMvDxfGIBHCyPOzUy/bAsHBgloedtMu2JBI4WR5ObgWc3AonowEOya2wn4ffKQ9OXJxF4PSyUxxC/h8J9x1KyLt1xgAAAABJRU5ErkJggg==',
    '1808.00179v1.9.png': 'iVBORw0KGgoAAAANSUhEUgAAAeUAAACHCAAAAAANEB5bAAAaUUlEQVR42u2dd3gUVffHv5tOCgklBARCQpMSQQi9CITeEUJHFKnyKiLwA0UQ9OV9LagUFaTZgNCRoojUkCg18BKaSA8GgYSQhPQl2e/vj93Nzky2zU5MlmXP8+TJ3J2ZM+feMzP3zr3nc6+KcIrDi4uzCJxedopjCMl5zlJwYJlH0g0AlNbN8+c7C9NuZT4AFbV/ikTlbMLZr6jorJefDnFzFoFW0veeDujQ9kZ6E2cb23Hl18EB8yefnN3Ty2Hf2s56GSdGn/QHMHnbPZUjetj5xgYAjp7lDwCdk1VwvrEdVW5eaQAAqNgRTi87rpexnQBQtR+c9bLD1ssPgnM6DuhRV+XIHnZ6GevGqoFnV7V3etmh+77u/XRoT3rZy1UA5ItbpPluzr4vR5HK46ISxz3aA4DPnxQ1v8VJZ+vriZWdAADf96ACcOtKQ+E+SdLp5SdVHu/R/i9AK6QcWB1yLAFAXvRPjzMNSQf1cvqmmf/9HTf+9xR4+XS89v+uiPrIR0wbuAJXurs26B5ZmMSTH0UAFpW9XfdmpX7+Tt1LtELAJ1r+43acJC+E3CKpqbqfZE79neSQd/XJJ1lA0kQbW16/7hPexu712pZqnVUxx1YFA7jWINUHWD/tjhurf9dFl3TMfmxH79cVy4KmfS+czx0+TwUA0S18+LDC2RZuuJ7URp+EQ44vO3q/rliaAmFh+kRcOyzrUaHWLXBpS2990jFbX47er2tG2iWuqlwLY8ot+fj3ToVJx+zHltmv61BxX1nuHtpaq9raCEHSAfu+Kq70iJ5ar37s0/j17OMBXB9A7PBrr0867JiUoF/3KYzhPP9WZHrCu1XxFIxWZL61evVYZ6SuI3gZJDtIP6R3aP/dxhrrPrxrOMvSfqWGjq04Amm/bn8A2n5d4M4xv/beuBvTze2gb4QLkH4i84WK4hMSaFc3rtMSkSEmvqSE/bo4uLRt5QngEpfOn7T+cAuwdny1sB7ON/ST3yty6PSJlgAuLopWAQtfDHrYHhcjklLf9hnaGT/PuepxsnVx9ogl7bjG4AHVdalT++7XGVFh+8CSLYWcFMN2OZ8M3SBUhcqqkjUjKdXNneoy/trrewcLnJO1Ny41ZEhNbh5aXG1sUb/u4lllXlzsD0x2XwIADV+eWaxvp4IFUZ93djk2pdNHXgAw+/4HlS+urL4lrmTfkxeXHT1bbqQP8hJP3l409cY3sTGVx/moL17oMC+oJN/YMQdubckdOaDpN7ExVcZ5ZJ1Pf3Og9l3L1e/1nFz7SlQ938Vxtra+pENKp8nzUd9c1pDk3cz0jQ0/J1nvR5LMxKliHZPKH9golSSzu76QTXJneAHJgl7hSgZgbJIT6E+SzHnlTZLnMYwkU8Kr3S1hSzoineR5DCfJ/V6vFpCk5jX/EyTJb93CbTIEFgyqtZOct4O86/KQJDVVY0muziguL3+A09qNRPeJJCP/T1vkJe/l8xik3XgwnORVjCRJbtT9LzlLOiNDcP0R2ESSi/CDdq+mn21ethQrMj59zzK/fsCp5uUAQLV2+Y5v/1PHt5gqovsftmmq3ao6cOUl4Ia2RmoeUkqtlMdqVAgQpGsiplRbTaGIBpA8N3Sk7u075Z9hHmdpknuqAPTsqE136ni3gmex5SIqJ0K/GbHp24Wou/mXngBUb5RSqcZm90F3QfoR6paql++gMYCozFf0D2M7338m7sslSAUAbn76yvyZ4nMyDqOmfrMWDgNvoVfnj4+p0aG0vAygvyC9xXVWaTr56uZ+YwEcRH39L56rnkB+ORGF78dy+AtoETXp0CEEzXm9FGw5Oerx7eO7C7+t7kF9ace+H7uWUskkbOWDS5eXjVYBSERg4e91nkAvuyCnsFjhAmBo973Rh66+kTe95G1psU6TOL4w9cdGwHfU0lIbkKrQDOkX0upo36MoeKLnIqh+Olm/mYxgAAgYNoxHhrw/NqA07rngKQQKLjYCUHNq6fZW+YYAX43sfLoBgJBTSYW/p5Z78uKxu+BMYbcqugCbAEDVcUVGKQUIN2kIXLYflmJQ7goA6AFDcXz0BEbdjyi7V63d4k9u44Gt2kR3w4u8ZOWZUOB4g1IqjG3bpZNyBeAcAAwN+jlP90ua2xPo5XIfJX2v3dp3dlYIcOk6ACADjUvNJH5dv3QufDcy8gHwwMtT5GUiO9/ny+TFul/WjH4SCZpJ02fEAMDFccPfB1AwIg0Alo8p8TCNdGhfKjnT7pQD0pFe4kURWPHlCrh3eaQ7Cq8f7PrwBtY+QOSS99YSAPaoni1WtqJk+hVJrgudeyL+0xpfFJDkc/vHLdh7aOrAtBK25FS/YHgPGDlycMsARDB+UD3fsp1mlHSZ7J59O7FHn0wyflB9v4Au80gu9559czpJRoe3/P63qAlRxctWoORGzAt+v6yp84K7tgkWzvNxBeFNVE9lFEHSoYym4eKf7u9jP+3nxh9xqdW6lHVS6k5LnJQ6nPNjO8XpZac4vewUp5edAjuJ4ayhKo6YX/uJPnZaIpQacH5JOb+kSl722cF40M40uyqSYjHH3LMsjoc3ExBfXPdtfsjUGSX2BGlodO4fTVict9UqXP7xZ9mMOSYyIG9+7IIFUZ+PdjnWRxcPrw2In1d9ixXYQ/IvfyJsoG5wJevQxZRGw9wFu9O3XnGrN8iI8buTRlutZnWlFpVcHh5OmWB9gX3/V45mfG19MvadXoE+ADBC5KsjzQ2GZSy5ffe5N4MA4KXw5t63j3WLEB4avWyz7e5LXfko5/Hrzxq7lilzoNl26bHLG4HmMwA5oxWSeHizAfESBcvKtt6wKbTGZZLkd+VarTs4pGO+YfcvZTvv2RFW6WRRPV2HW6+mLeDuBayzeoxA88Y58u9Ox/Tp5boC6Cg+bIxh5qfkYdeZHul/jCSDALjP1QiPzKrV2/bRiuTXk8lPPPYbu5YJc5jRda6aa/tZyICsqHtJPLzZgHixgr1Q3SejUTuP5JfooWZiGRwt3J1QxieTTFAFpkrVXEGM9WraAkCVTdaX7e7PSTK+iz49ZcUvMbGxMR1vio7Krme4kV67STKjfGg+ya5fvbNKfCQXKPHyIq/NZDLaGLuWCXM0kSNJ1g40nwFZXr5XptCAoaqLJJsO0V5qkMUcdUYoyVwXRJE3PXCCvKXCtcLds7U3XwiWStVMD9NYr6Zt33Fvb8iUUbZzppFkZk19eqr2HbFWfNRGwYBj9SoPSb6ICySLMhbH1rdU4OUNnpvJbNQzdi0T5hzAaZKnjprPgCwvf445+s0VmEFymMsekmS0xRyVRxhJ+mMCORMu+SSvXTHsboe+WidGSrRkl/9Khpq2G2SW7XIsKiA3vK5PXyPJ6xM04qN6xxu225S/R/IlHDLm5ZwZGiVepprkfswwdi0T5kT6F4j2Gc+AUUPcrIyHx1ube0V0eyHcw3JAvPdDAGAejgP7EZiyNLHOi4ZYKp6CNwB44YTkvM15o6xXAzD2jFfD5tYjAK988tb2VX+uizJkC9BMWSvuvEi+28iQiM33AHDOLQwA4g/lu73qb9i5dIqybg93QP1ZszlGr2XUHB4O/Wulb/qrdc1mQF7rqwm2FyKQqESSGwMABH1h+b7tj2CSV4BqZHmE9j58brDre4ZnABijPcpPoqXlJBlq2Lb6jE2Ly9Y7av0TlNAGHu3V4kCVtyXHLP1EetZJTCPJsCgNt9e9VfjzmTWkomeZx2c2n5hp9FrGzUlD24UFvFH1gNkMyHtjh2O9fvM3VCZJpm6YWAf41GKOLnjhOvmGG54h3YBfyPvATv3edGCs1okeYiVnEC9DDRefI/khKiZZXbZXX5rhhW5CVDWveoLkmFaJkh9yn+uSQ5LxJNl0YOH79q0CpV6m5s8efZONXcu4ObfhmkDylZo55jIgz8sDsFi/+SNaFBp2ONAv1WKOjoTVmvniwgA0JgPhmk+yLF4sLB9o2eCuKCc+a3wbylCjlV+BD6wt23Otk3mtM1oIKretwZJjLneRnjUtQvC8TYD+nvr8GhV7mXzg2Szf1LWKmJOCUC3letBMBmSSrdJ4eDkB8S/EH+jxxbR0hAGBqOIKwBPxhfVRALIAIBuVRCelr58sR03s4C0AfKCNWLZGxn9UEbX2zz+51/DTd6GSY9aNkvyw5u89PgBwPAYA/HBR+/OlvAppaWn5j9MyFNXNFZrH/Vr0WibMKetRHgDK6G0wmgGZI4/SeHgZAfHZb38V0qlqAtEHiIAGAPJRCUD6eQJop/VyDsQLvqz1jpSjZu7WNwHkAxWtLNFHF9sDUM1rf8XQoRotwVE0P0o69n6KX++JC5eAfl3VANTQzbuanDhnzpw5l/6cs8pG/6pbNFUDKI+rRa9lwhy357K0EzdVNJ0B2X1fy7BS3zvxLkk20H6q3kOipbfTNiCRXIJnH5MnoHpE5rhiGpkUhMkkv8BzJAsqYYuokmo4U6LUvJpRLjNJrgY2W/meVPulkyRH7idTtFVbPF4VHxMzQpw++o6G5IJrZJc1JNnF/3HhySSDbH9jp6tc/yLZBIep02i4lilz3nfPI/kerpjOgOy+L830skdI8kK14fkk+WyLVJKcN8Zijk6hu5qnvMtfIKkZiIXkOgSnkgeAUJLZ9XGU3IL2oq+/I6rr0hhps2r+F3yazGiMwRpry3bc+ySZ2DOPyd7aDrxfMEV8yIQ9ouQfNSdMnDhhVEg++fVRkjfdv2HhyWS+b3vb6+XeR0gmeHTI12kUXMuUOfcDdpGaluNoOgM2zCsiioc3GxAvUbCwcu8XfIdqW39ZU73btvEafptkTteyK0ny3mDfwS/7jX0kOmdYzyJazauJfq5ZN/+qX6qtLtvclydEx698JYHMChtFktyF2eIj6j0WpXWTYTQimT938dmNDT/TGE7mpFa+/p0/trkfe/Snv//Wsvdd6jQKrmXKHJ5stjVu0ohHpjNgU9S9MB7ebEC8VEH29TKhhaNi2Vc8aksw4Ee3XEPEDY371bf1LVrBm1eTlBhSXtZ439VT2Y2aC+xXr+stGgPa9tsi0ydfOl6+dVBxjoGejWeTRua6NIqak3bkQfNGMJ0Be4+6/++KG66lHqHx4pxwe4oVKSZz7CdWpGDlxNJfziflVlN7ChIpTnPsw8u//D229I3YPMSulmIpTnPsw8s/RwaVvhHbR9pVwFdxmmMf9fIjN+/Srw0fVIQ9xXAWlzlO5hFPSaSu63wAHZVq6mgvZeu0xLghJDvAKY4rHexiLgLnG/vpYSvsjGjYd/JpYCvkTfGv/L41RzRoXEr8CVIOeVhnieWsWUQ9LKoww1bcEE3xn7zR9in+pQwB11zwdZ3pYw4hAJKmrPA3TjAI90AW5WEExkjcecM3dECAMsgDOHv4dqV6vW2YslOYNWPwhnm2Qg7fYWr4RM4U/7CeIVD3GlnATY2SzSAEydMnNMddYwSDeA/lUR5FYYyv6y7Z/6GrzyZlkEfe4LAP/huO6j/LjggSwRlG4A0LbIVlvsPyyKOcKf5hPUPwgfsjkm0nmUYImHsrf5HBl0KCQbyH8iiPIjDGLZfyp8gIuKcrgjzmYpKG6giUOSvXyyI4wwi8YYGtsMx3WIzH1sljeiib4r8yAfjgoT69spEfgFYrPhO9oXf1EQxWeAqXmjseWtHEHitkIUIqAa1crm0bDqBvUMXGfQVVxQ3Nw4PNUAmPbzwvOW9FWDsTWm5Nw/vuyM8VBK0dwdfvBLtPP5QTJXNaSVHWUGmysWOEBXNo62kAGx6bVAFb+7Fj90HZFP/DMgYDv6OPLpmW6AsAgZlnRUetfcnE6bnbhitoAf0PPgA8/RANACNWfThM2B5o/2qH4cBFVJROr5rz7WsqE1qWq13CgRpXrxjWZO7v3qEKUBWCEC6rxKqsCQvma//nATRrbUPpWHiWY8OVTvEvZgjKuFJ7b11uY4poEIkygkGIZxSFMdzWANh5vswPHoogj2n/8gRwCWgizzhp1qTwhkW2QkbpuJib4n9o6/mGG/zevdt7J9kwxf+JWe1qROut9wxLAYBbEH0Fbhxm6mGsWF3J10w4HgG4lqutMGbtqpLb+/ljwpbntO6DmpzpKT1t+UtlTWlJgPerXabHN5on2O8JQLMMIW/Ke89IshZ/aer/hbRIEB8kLJhHKT5b/v3OxIiDtpSOudZXQUK33TouAfUXLVq0KjbPhhgnEUOwU3WXzGuNj80SDbo2lhGCQVbrS4hnGIUxbpz+zL32NiqCPEiS/0bLv2W1sYtkTQJvWMFWWOY7rG1j/7SbzI8nr0IJRyBkCD7rc/v6/AWiNX+LEg06XxohGGR5WYhnmIAx+DawTTHksc9ldI68MjEOZxjgDSvYCst8h7WrhhXPFP9ChmDal0cOv3VfNM15EaIBxUUwCPCMojDG31/GAWgFfAwFkAcAxEYu/M7r7JcyDCuSNQm8YQVbIad0LLS+noGyKf7V7fKPe4gZgho1gISQxkKE4JiJjqvEOQAuVZpTc5ptV8/+oOobIbhJ9AHmHvl9cCGMkX47TIVOV7zvBMATSLEAeQi0RFwSQh63w1QAcPTlHyOAn+7L6ZOTZq1feoaHAN4wUjBStkJW6VjsFdE0e2j7G1vIEKTkkIwZkEamBawxQzRI3stagkFPNMh6YwvxDCGMocUzqsIjjVwNjFICeZDH/bpPnDhmYIXFsmk4HZyRkiOCN6xnKyzxHZbr5d+0UwYwe2oVknFoZ1u9bGAItEDATJfz5PwmatMIAUl+hEJSWEswFBINgj2URXkIYQwtnrEAn5B5LVH+LyWQB+P03w8/y/WyDs5I9g4Xwhsy2ApLfIclL8ub4h/WMARaIOBCr5gzMyLum0MI8kZ0rezXoP8iIcGgPVm0xxpLhHiGAMbQ4hmar2o26RPkNfK2Msijk/6t+KdML+vhjKywUUJ4QwZbYYnvKMEVDSQMQcretCaitdjNEg1KLRHhGUVgDE1SYqVqLoohj+IZAy0Cb1hmK56g6D77IhqKC/JwshWwX6LBLiAPB2Qr7ItosAvIwwHZCvsiGuwC8nBAtsK+iIZigzzgZCvgjNQtWUPcAHRwrmjgwJZ00NXLHalQ5tFexGlJEenoXLfCyVbg6SQsHJHyMPYsJ6W6uVNdppoOqfAONgxPZu2NSw0ZUpObh/4T923xLNKgTIOdrVshh/LI9VTJYCsuH7i1JXfkgMjkjbExVcZ5ZJ1Pf3OgCwBw9Xs9J9e+srSe7+KhMq0VMxHGAQsRSCBhMuQs0mBcimjQrEr0yp4aaJbyKDTb2LISstatMM9nPNp1WR0yKEgR5ZHjPTrUz6+MCmjQ1Lrx5Y5I1w4yDyfJ/V6vFpDUvOZ/giT5rVu4rPEXKRNhArAQrRkhYTJkLNJAU7OgizVohn9Anq77txkbDGYbW1ZC1roV5vmMQ88v3LNrosdaRZSHIczkBytHHjsjgwKkYgQ2kVykP13TT56XpUyEccBCBBJImQwZizSYEKmGHf55JCdEmrZBYLaxZSXkrFthns/IqLyBJCe6ZymhPHbpnfySxjYvv4vXyCTfUP2kigfC5Y6Yi7xcLZwkp/tmmV6kQbqug4xFGkyIVMOQXiS53jPLpA0Cs40sKyFr3Qrzi3AcwVQNyW9wigqW8lgUvmVfTGxM6w65cqP7dHIHjYGozN76o9v5KmlRmAAsRCCBhMnA6wBw46iCvl2JhoL9/gBQJS/aGsqjep4agC+SbKQ+zPMZrljc7RxwxKc2FFAe14dGdm3f7tTD7Z5yo/u0cnVzv7HAQdTX/+C5SomXjQMWEsJCsq6DnEUajItEQ3KqLwD440ovKyiPostKyKM+zPMZzSukHHh+TI1N3wcooTxeCwTw2/wz5SF7TCph65blb0xetsMNSIShPVpHiZeNAxZSwkLEZAAANjxXDspEoCEDPtrHKN0aysPFA8Cp+CmBNlIf5vkMj03+4DfzJg+GEsqjQSCQMWxqbVgdw1lYL/e/efPs5Ga/k2RzbLB99n5RvWwUsChCWEjXdbB2kQbTItRwDjO1U0zPMmeD0GzhshJy162wwGecb1EFAPrmKqU85uK6DVH38A0JafxV3c6XAIQIaqVURY9Uv0/H/3Xjw97ieer/9K0q+Zyvu25fzwLBp6MqWOGjLNTghwJtZLWfORuEMjtwh5d++8t/yftsb/hrWLdZA4N9EQiUg2tXoFJZfKffe7JV0LXYFsDujyWPcptGMrQASP30mVCbowgG5a4A0AOGJSs+UlbcRgALY4SFeF0HaxdpMC1CDQHapRly4W8F5QGIl5WQT32Y5TOm53/r3e742nLYBWWUx/acmirr2IptqoGAuHsuAOcADJ39c56u+ZbmprC8iwAWEpDACJORH91FaW+hUIN/5fsAcA8trKA8APwUv94FF1wa2EZ9mOIztHjGmVoVANWoFvUyFVIe20Vshpl6+W+okslGXmrB9/JplNcw6/EWfKQ76NPLttbLpgALCUhQhMmQsUiDSSnUkJJDcmxTklxYPt8c5VFYLxuWlbBp3QoTfIYOz2hSNo8k1T4vKaM8WAdjrIu6f1zxFQ3veowlScahD0kmu+Iav77LJR4/aEjy589kt750TIRJwEIKEkiZDBmLNJgUvQatxote10lN6wVmKQ89ymFYVsK2dStM8Bk6PGMNJmSTmv8EJCiiPMgymGQlW7F79u3EHn0yScYPqu8X0GUeyeXes29OJxkd3vL736ImRMlsYxuYCFOARRGQQMpkyFikwaToNeg0buxw8eH0UWrTNghQDsOyEjauW2Gcz9Av5bE+LLBrn5D+fyqjPMhwrLaWrUg6lNFUGu99fx/7BQDAH3Gp1bqUVTrKVgSwMAISSNd1sHqRBpMi1ZC0J6u1cB2OkqM8ivAZTL7jE+qumPLIvFNXZc/RfcUIEjgpD3uNFbEHwsKBKQ8XxwMJ4KQ87NTL9kBYODDlYSf1cnGCBE7Kw8lWwMlWOIkGOCRbYT83v1P+OXFxFoHTy05xCPl/0dPZWBqUjsYAAAAASUVORK5CYII=',
    '1808.03399v1.1.png': 'iVBORw0KGgoAAAANSUhEUgAAAlcAAAB+CAAAAAAsbGuPAAAdT0lEQVR42u1daUBUVRt+ZthEQEEUhVTURHPBDRWtr0SjEDXX3JJS88syLNeScm2zLDUrc/lccm/TUsktScgNcUsrNxQUUMAEZHcEZt7vx11m7mzcuXNHwO77g5lzuOeZ9znnzNw79zzzHBVBCSVkD7XSBUoo80qJmhJE1EvpBSVkjF5EpCJAZeM1FpV6yJ1KXnYbVaUHqR7Ga8GHkJSKAGdxh2q6Nglyi/cILT/vuj9tXKJG5kw2HKqvWVFFvXDnw19cBzkX3Gv7prOQ567Vlx6f2fngsoshU57ElNjWS9pW25HMWvIrPTvTf15CRr+XQn/5cYMK1eE8CKo0bgwsJ+r5MlFRN6IrbiRvVNQpvPhr5YeBHBPPjyAi3cc9c4x43sBOIsp1XklEFHXfMS8uE6kBA4iIPvEmou8itXx1DFVBgIhEXrffjnIG1CrAs/89uMr+kQGvNs9U3VvLSQVAFdN6qhHPwF7bAGTX3QYgq7NrtT73ODnxf0fu5QdV+2tV5SPyPOjTiHvWv9wdoISM0NYArh1Vhwdwn8WHS4LcuwKZ8Zoe7VBwJC9Kfep6UOfco+rwuDYtgdvxWa0i1YIm7LHZhyviPHtU+dBMeHJ+SyHPqDcK6+CnL8ekBeKHETXl6qb0UO5YdjTqzb8ah1Avrp+BaycCQrbXe4odEm3SJffBtVFwuGREQnboo3kJbhHOeLD3GYKacs+61gFopV/4M6eB1fOG9RnzJ1OfMWvI+Mu/ADsWRI6Oexe6I2N1KIrZA+2h+euyhwP7R7V/de0iQRPuWKC0OgxIC5w24jlM9xN0JcN9twGZjav5dEpdvnz58qMA6Ow4bjTKugj7efPs3imdR/7NDcm8HeMbdLsD3YnodQ3Deq88GH5i3IO+viIioicmMI/XcY5owCJKdckgWhvFVO4NTSrT7KdbdW8TUcdYuoJyoqEfEJ1UpZffpsL6R4j2nxQ04Y/NwoO8FDGJkSOJiCgba4U8iYY+TYdj6fW2usvrHXkxIkMMejYjIyNjti8RZYIfjSO+gn5uH0va2me4IaGdO4kiviBKxTmiAaOJjjZ40NdXRtEGcCvDYde4DRvyApmq8Md6+ES1QYKrH4BWu5mLMBUAlW8TZz+cygkBIroJmvDHVo9IQVfjqqhDtw5EIOri+Z+GVvfTn3vjxo0be3Odzo6GUT+3u4aCWk25IUFkwdwldwoBFdoAbo8BbmUP+vrK7NmzPr3oxFdlLl+WuPWlBA8NqQCNJ1NZDgBuAOCNQnfjJsJjqzx29Qo2rupXd6PaBT1abHarW7NuIbGjAUATH8n3c8T5T3L21OeGhCJ6fOB8HEUezHjKu/JiC5pOx546QQARejeNBcpXMZWJS70j1xegT4NjQMnJl+CFUpT/WQEQAUCHrtuB0hWCJvyxhCq9OagjAPh+5wa1kCfgNvzDSEA1ZkXP6j6RmJx1OnZ42NHwKdFlpOv7+cyEmYt7gBuS9ITJzpSqW6PhxpOtf8CfV5+lXc2e2mkckD7bM/qz786mdRi0b+aFhmmvsf//e1XAqY/gdWD2aY+ETZ3gO3lh6JWQzcHBC4unTG0O551Ts5tdnVTbsAl3bOYsz4mTO6LK7ovGu8U4F5XWPe0l5AkAUb/1ADBmRUT1nlVZi89hxkz/eXvKo19sOsvzjeXsaDwWPrt4jn5M+nV0v99g4LIMZkiavPjWwH9ilvXN4caz9ULdm0tc5LvnbsdSQul9H+6OvNu9u/5qACjR+DI1+Q3zVHWdjI/VNzE4troueVByawC40rqGreNwo0HZvq58P59+d6N/xZ159T/mjqoo8EWZq0Mo2TmvoCyl1RxS+zZtUwGr0hc+AErKvPoXkfrhlL/qruc0F2VeKfOqhlJSdH1KwEHrgzVF11dX9TCOwMNHKrDy74Oi9HYP3SmjnFwfOlIPkFPluj4H6O1MVWcPSIdmKNmzfuS8T9dG1YxPhmrMydrSp3m9nXitWIyZoqnqzLDGgUu0AsmedUZPbhZLN6Zq152rJ6dK153N6u3Ea8WMjmSKpqozgxpHhkCyZ52Rq1i6Vaebq+6crJ0HWb0dq8RjVWCpjFbMRLdnKvFjj2TbsQ2dTFRnjA6Nlf1x4kAHhECyx2QryJP9Z0ninacAY7pckVMvMu2FLasgZOXEkJKJk/WPilLoxXusCozVihnr9sxI/Ngj2XZs0VR1RmfHgZP9ceJAR4ShZI/JVpgnE4lDvPruvwBjulyRbcKyFbSsipCTE0tKLk7WTvFZIL14j1OBMVoxI92eOYkfeyTXji2aqM4oE8TJ/riWDpPAMZI9NlthnkREVOL/KxEFbzahyxaZJjxbg5ZVp+uThxM/0PZzAhFVqmc47BoH5AUCkd/N9b5TKDxfs7o9+G3kjgKA8O971I5cwh1o1E5lcoAKYGV/KBf8w2HBcgo3fbVTWWEsN6O02SLTJN6QbTUJeznpB/rB6GQ4JZ6BCgya+Egj3Z45iR8AaOL7cu3U+oYC1RkAXvZn0NKRwWabZpgnk4cHNMzqmRFdrpizfFni1pfeMmDL90UVh72cEgyGUAZOVq+vCHrxHq8CY7RiRro9cxI/RlXGt2NFZiaqMxCIk/1xLeFY+RubrSBP5v+dusQBxf9UGNPlivFLvSPXF/BsDVqiiiV99nLih1AeTk4LgPcWmP9f5qzU64FNBiy+mrx7TJ061w+Ux/fd0vQZ35N/H5noidq5SUWxONDYef6V7Md8XNijAAB//56b+e3YIPie/PvIRH+uHVMsnJWa2Y89gEHKnJWa2U/df9Vf139+4Sbb0nxYylN0fPbt4azr2Z3AZptumCejhlb3W3bz6p6//2gbLKTLsff/PTfz27FtObYGLaWGvaTk4hTEDaH9nN5bIE7PwCrxOBUYpxUz0u2ZkfgxR/LqMa6hieqMb68XBzp6yUPwaoI8iuGZXauO2pguU+SbsGwFLat6HcdeTiwpuzkpOhmFlKKTUQKKr5oSyrxSQgn8m3V9gYqu72HR9dWsS1xNSgt33PWRgEXZuS1r5XujkkpNSvPaxi9gZ+/JSEoMpwdBqkZftxdNa9ssBwCQ1jJ4hhbA8Wfnn10769zz+H5U89A4AJdDui3QtI98862uvd6e1qevFbCsV/clf7TkVWTN7BA8IwtFr7cadoyvTJnRocOMFACJz8w9sy7mvDWrhqy3OoTkAXcnBQ07JshDgCyBlCEWRJHi0peXlGhODvRpcbD1yuRnPyEiomWRM4mIlna9TUT3wzoSafs2KyQi3bhbFpVvQqxByUT0RX/e947Gl5lWft4yi4ju92lnPauOg3RENLqMBHkIkKWQMsSyQMoSJ1lJieAk3U+mWoTTGyu0AO45uzoBSIrZ4gfAdS0A9fqiqQA2jAuwonwzjKNeAF7z5H3vEOBiUpk064dGAFzXW78gcvsh7isWwDAPAbIUUoZYokjp05eVlEhONfn7YKTTLwBYC6GlXZjfuj/aB4D//9bvRMrNXoBPKHjlm2WkDq/fAFznW638vEtn5qq0EsfKVqtmnuGe6/Own5QBlihSZjnJQEocp5o8r5yivwLotj8AIKkVW7sUAIaOe+XW4rdhbDRoKVZfaB40/vc2ViuTWhu+gJWIenEkr0Xh85CBlB5LFCmznOQgJYpTjb5/9XLiJSSxxqRaIZMvPB+PdhMNFPTXodEpvZdbraxQAQBjBGQ9vnSbSNLysE5KBk6ykBKTR42eVz5jvsZx1poqNJWt3AkAqDO0cXvxQOmuvd8/vPOdMsBJyyqEDCsBAN1TAGDfmy29ZleC5vHD7jXcc30eBsgSSUnlJDcpMZxq9v32yRuvNmKvOCclXmHegexXXpUt9xtnA8DABvlA+xRw72B9JfNaiRcB9P+yX5NK/VjaLZ9ykb+Zw+VhgCyVlEROspMSwakGz6u8MnQIGT+EFbY9HTO8EACWvgIj0z3hc7OxLwVArp8fMPb2eQAJ7QWVOh2A3u88nw8A2da77DYBGD/0pMlrGyBLJSXgURkpffqykxLBybmmzqqiWfuHT+8184x71uLz11ULnfB+t74DOuf/MbQVANyfuz932utBpgZ85qPJ9+qORTtWA2gRO3NI81Me0w0qU74+h+nRj+L97hH9QvD709ZufWcu2PPC252hWjWEKRvkYYAsjZQhJxGk9JxkJiWK00O1jpN9ObCZSgpWln/OZfdgRs2mvXo7uJ5xJfe+vejT2l2ygIpHdhApy5wcR8ocp3+zrk8+LEevD9Y8KEXXpwQU/ZUSNSicFgAIqyHJhlVLrLBqQ6q6QIVB2YdXCVSXfXghm7pOuW5/WK/b7ZxXRfMOlJ6uDwBpT7s/+6kTcHxB5/b5N0fP+O37n5P8PgoHLo9R948R7SsniezpfR7tuB0haO/x0qcGqYEl+YFuKngMsQNLXzh0xuXJEFFIZtufPeDuO8pFn1vliRTuSvGO4JeHf9jSy91TDYx0KdmZ4vJ4L3v6Z/f1/BajXCvpH1Mosxj8E9PG9ur67FLXyWO9sm9ImW7uR2xhcZz2nydH6YgGejcJDPSeaQ8WV9C+vqziRotyMUhm2+/bqKXfXjHIrdJELjSaeOIbryVckb3x2KUiLWTbmS2dXrWD06IvSTt8BFXSPxCHwT8xaWz3vJoS27SCiEqXD5pFRCdcLxMR0bWORJTp+zIRrU8gSvqB89NZUCr/vOp0kOhenQIiIipreZLoBM4RTSGi/FfL7cDiC+sHE2W8pRODZLb9wBtE1FOnz63SRCJqlRH9V53G7d7cZeRLYwM65ND07veIdrhqpXMa8wrR0lq6SvoH4jD4Jw6YVxXNdxLRlsxBs4hoRA+2ehoR0Q78TNfeJ6LkNG5enSqQfV7lIZWImvxARES6saeJrmE/UTKR7o3b9mDxhabfFqfpRCGZbz+p3XG6MNIgt8oSKXTxJ6J3sYItd8kg2tj+H6KpaL7u/oQI6ZxIqyUaOpwq6R+Iw+CfOECHLJ+6TmIkozaA2tnMuX1DCLDP73EgCNj6lJ89WFwhL/3WprMjEyS3x3sNHx+4abNBbpVFWrk3gLq4ypbPNEbyzN0NgMl+1yf4Jf4onRPU93e9WW89bOwfCxj8E0fcF5VNXScx7kMLQKe3N7q29BcvACheM9QuLK6QgZJJgz8Zeltqe7j3nHri692C3KzHPbgAcDHY91r38uDmAOr1bOlScDGq3I7+cW438MYe2Ng/ljD0YPLPK9nUdRKjDvIBFPDDlT5xVzfmW1RbtV1YXCEAjwMtyg5KbU9Dnv88efzIK4a5VQZTCKAY+l1a444FA9D+p+CvlNecdm+xo39cWoa/9cJ5G/vHEoYezAHrOHKp6yRGG7e7AOVzfqtp72wLvnQOAHY2sw+LK9SveweAWiu1/TWnTvD+cvTvhrlZjxY+hQCK0A3Q/pYPADvQFEDaxTG1mqyMR5pkToUDLgKtdcds7B/zGIZgcs8rGdV1EsMt8gqQWq8bcHlaBW5MHn8t7jN3ALhSRzrW5WkVfEE16RyQhqeltm90swxAwy6GuVkPlxfuJoNON4gA1oRPAICr8AfQ0CsRQKBLX8n9U3pOB1xWP2Fj/5jHMASDrL9LLZzUYGACxS6gzOlN279dQUS7e360d9tbiUREpHmrjd/UZObIT6PrN5/yjUOsgzOe+enIkDgi2tU4t7wZALiUExF5bZKOtatxrr6QP3jNb0N3i0Iy2377mP3HF60mw9wqSaR4XMDEsA6niOhowGdERIOdbxIRnejUbeJzXb+T3j+07rP4/d3XUSX9A3EYBmBGjR2wjmODuk6m++0Vf9zrZOa998dj7nZicQXdxYKOnuKQzLa/d74iuK5tiRRn1fEz6EVNIfvVLTengY/KHk6Ff6nbe1XWP6ZQ5jH0YIquT1kfVHR9SkDR9SmhhDKvlHD0vKopur5e1RKrV7UhVV2gesnyOy87rO6U63ZF12cpsub38P7D++T3WUt+pWdn+hfNigue/gRXmbLiIJ55/VEgcV6n4IJbo6fGO4SsQEuHrPWzAeDSzz1rXfcYZBuWUE6X+/NN3ye6AMBf8bmPPP2oDUiWgKSSAr3xsZfNYjwzqQB3Dl31G+0F2CYRNExH37WXdrh1iIAjdH32WN05Qtd3/N0XA4mI6GBt1JlYZBuWUE53ut6ac9MxjYjm+Wza1E09VzySJSCJpIhoNe7YLMYzJxGkrxt+fmZgJx3ZJhEUpMN37eZ+Gt30rxyi6/PNIqL7I4lo0CAiIpptWnnC9SyzHXF7h8wrgeaM6Gd2XsUX62zFEsrpXsfndAm1tKTzRAoV1MVp0UgWgKST+msJ7pCtYjxzEsE9WErUxqXURomgIB2ua8u9fiVK9Sl0hP5KTqs7aXH33KNArboHTP7hft9GpKJDPi6An45VfkQ9/yyO43E1VLNeaYY6gbwcSjKQZFL3vh0PAIgGaO77ztI5IQajSku/j3OH7mTb9WV7e6sl9THTtclFfkBg/ik4wPdj9YDmLf8zrpfVyqReIl3hYLeuzyA2/qlKmtfKFiQjOV3PH3dvWDV+CYA5ADIu+fS2G0gqqWVTmbUb28WKRqloLqpWpN75ZzEwedv1CdMfOSEpHbZrm6hLgBy64Yj7V/Ja3UEOXR8Tbee9+caLz9n0kWUip/MPjYz9hsm55PkmcQ3lAJJCam/3Btw/bBUrGqWSoaXmWw+4Dbxim0RQmA7XtV4ztwF7at1zxLyS1+oOMuj62KjbHOicfMo2JIGc7mZ6t2Fbi2fsBoD8AY+edU2TAUgKqayLeo2OrWJFo1R8gOFQtbq3yzaJoLCP+a79pPOSr9to2jhiXslsdQe7dX3cO8w3AVBDawuSXk6n/S0fBUFBOXAOwFkAGQP+u8196iX7gSSRupAzZ867WLgVsF2saCQR9A1CCaCzVSIoSMegayfMiK7t1Vl+/RWR7zUiygklopR654gofoug8rnniIjmtrlLRDTcMfcZBq8jutboHtGlqeVEtKMJEZFueCnRjqYlNmFFq66QrneDQlqJoVTqHlBMd5yckojOPwInFZAiGskSkGRSdAHMb2darbCxf/hUaCWGEi3GT6QLqXWp2OtlIkpzSRQHZZiOvmuHLCF6bZGk/egrC9ms7qTHVy/7NFi6pRaQvH1uvYsbjxe8FjQDqjcXPpmzdldtm5AWlfQekJy31wvBAT3hHhszqmF860+6Q/d0DrSAa6C9QFJJAet3e00ZOhxAlq1b5PKpIDigJzD1dvT2zNJdj+Hga907Z2Vt7iEhHb5rO+lO/vLITGv33KXfb7fD6s6xur6Sc15tnW3FEsjpKC8vwENiVmKBZBQripQI3rvl62O7RFCQDt+1l7M711X24VXWBxVdnxJQ9FdKKKHor6Dor1D99VfK9ZVyfWXaWJ59AbKWHnA5WA93343rMP2mgZ/aAoEq619zEjDr88Z6q9XMsGTzBkpIHwtH3BetZEdNS9uEynlftGqxIM7njfNWcyApx0FZsnn7aXAXhJBD9FdszLji8QURzSCBn5pQlfVvmVdmfd5Yb7WaOa8s2byl37hlYV7J9n3Q5h01ZYyC2O90hzZd4Yra4+u2lVYdlECr5PyEGqiPbDzSPxEXm6pqJCljDVfmru829qkX54smgWqH32ewdUdNGUN3Inpdw/BnTrPFeTvGN+h2p8qgzPu8sd5qNZKURZu3B3L/ysYdNWUMn//mdW8X0PEQW+z+lPqZJt9WGZR5nzcDb7UaR8qizdsDmVdmd9QUvU2ofV+L0QZw4+RekQVzl9wprDIosz5vBt5qNY+UJZs3KyHn/oPtlkf3Nt5RE+1/FLlNqGzrBhTR4wPn4yjyUFcNlLHP27vbGl267+HUCd5f3v29dU0kZajhSgjxBmfz9gA+ryztqClym1A7g0AAEeOslp4w2ZlSdWs0l6dVVAWUwFuN81LjvdVqIilLNm9WX1KO76e3XvEZdZaICp828VOjY+Ff731vsc6R36PTRnu+kr+qWchO2tU4l7Qvjv72i+96LGC8zWzEsh0KVr3ZeC811lvNkaQcB2XJ5m1HlxBPv55dNA/EVw1itgl16OJCRYEvylxlwaocygySWemUBW81GUk5EsqSzZvDfkcPZX1QWR9U9FdKQNFfKaHMKyWUMPo+qOyXqgRk3i9VDSCMakbMr5ZY86sNqeoCFWZRLyqDuZ7yfVDx6zMCEW2u98uPG2TcAKBSNHF+dAJvvFuLYgLk8P4zZ0tnGckchiWnPVGJ6A3yjFz87ICyYx9eAytEcxsUW/DrE2+u912kVpZP3RhxaKL86ATeeMVdkCiD9595WzpxelEOw5LTnqhEeIM8Yxc/O6Ds2IeXxzC/QbF5vahM5no2REUXe9ZxjLSMht542mFBts4rc95/FmzpROlFef9AS057ohLhvQeNoO2BsmMfXh7D/AbF5v0ZhOZ6146qwwMKDpeMSLgZ2porI/eoOjyuTcCh3LEAbsdntYpk71doky65D64NZB0uCXLvyjwEHcmLUp+6HtS54HDJiITs0EfzEtwinJF7TNP/cPETAUidfzUOoV6lh3LHApnxmh7tAINXRGVaxvqMlnESU476R++NN6dTo6uQpPUcbljJ2dL1sQPDVqc9YxD3+7UsQkuCsi0hCxgLFpXkNvlU3P2rpNYG5nqr5w3rM+ZPvd6QKUN7aP667OF0dhyA/aPav7p2EQRSxIxZQ8Zf/oV90B0Zq0NRzB5GthjWe+XB8BPjAO3Rlz8MaDbsG5QxF0N0dhywY0Hk6Lh3TRSOEK9l7Pnjtbenjt8OAJvTZ8vi/WfFlk68f6CNTnvGBnnL10QlW7YmlABlzz68HIa1DYpNPvQajyMi0hKRjlJdMojWRlEqzhENWMSX6aQqvfw2ZYKosP4Rov0n2cY7dxJFfEF7Q5PKNPu5hysoJxr6AbEwo4mONiCiU+5aor9dkumILxERZYJu1b1NRB1jiX/FSj/nT6IDES2F3tj35PYR9ZfoiI6G3aNoG8+Dv+MWEQV9IDyHvR1NtL7WcpFI5jGIip7SSk3kVirR/lYaE2g7oCwmZBbKPMY5vEeU4pMt6ncThuZ6h13jNmzIC+T1hlwZKt8mzn5QATiVEwJEdBNKEcMf6+ET1YZ7cGUki5xs8TFOuqhWA+3q/aYXNSLB1Q9Aq93GCkeI1zLy3njXo9dVFJehtNR+7z/LtnQ2+Afa5rRn3iDPErQUrz079uHlMKxsUGx6fp387MW2QP/+iFuI+vSik+H5Ul/m1eveKHQ3kSLmLF+WuPWlhEzmAQBQzsMIyFCJDwBo4iMBwENDKkDjacMKk5GWsSBId6u+c0Dq2UFnPV8C/sTULhtgr/cfJgB/mrelE4+BnU9ITeS+7/4wqKG1CC0ByraEzGNY2aDYdPR6v/N8PgBkq4HeTWOB8lW83pArg4iTIXbouh0oXcG05aSI8Uu9I9cXIJF58EIpyv+sMJAtMs3L8oFdvn3hU6LLSAcI1KfBMaDk5EsGh8I2LaOrqr47ctKc+mPY0aNHj3bA/zZI3ocXnO512FJg5RwfuzBs3BZYsPGt68BQ4HDTbga1dkPZsQ8vj2Ftg2IzJ9PY7gtiY2eumkhE14d9uGbOTb3ekCnTlcF13kylW1Ge0UQ3n5+zdtYNpiUnRfx2+Mpdc/YQ+0CTZ/308bAW23mYS4M93yijU24f71g68DJRxYCYydkM2tURn//vhQOGCkdbt6yN6zpgQou2zKa5H3dr4Nl+pPR9eC+8/Z86ry4meu+zpLkfaCXpRXkMS9veito898icA1t7/2FYaz+UHfvw8hjmNyi2rBc1MNcrvS98nxqXjaoYKaLG7d5dfzXYB0CT3zBPVdfJ6A5uWHExPJldmnx59WKJxtc+PzpRJns22uSZtaWzUS9qwWlPXCK8QZ4FFz8pUHbsw8tjmN2guIr1oom9S9XK+qCiF5U5kld0nnIfSjycoVL07crnlQM+r5wWAAirIW+CsGqJFVZtSFUXqDDm++B85WNbCRljPvt9UAkloPxuQgllXinxL43/A81OHJ07X3TpAAAAAElFTkSuQmCC',
}

**Module 3/4:** `src/table_transformer_structure_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Structure-recognition metrics for the adaptation contract, implemented here with no external scorer.

Boxes are ``[x_min, y_min, x_max, y_max]`` in pixels of the table image they belong to. A prediction is a
list of ``{"box", "label", "score"}`` entries per image — every (query, class) pair the heads emit, not just
those above a threshold, because average precision ranks them all; the reference is the list of
``{"label", "box"}`` structure objects per image.

- **AP@t per class** — average precision at IoU threshold *t* for one label: its predictions over all images
  are ranked by score, each is a true positive when it overlaps a not-yet-matched reference object of the
  same label on its image with IoU ≥ *t* (greedy in score order), the precision-recall curve is made
  monotone from the right and the area under it is summed over the recall steps (the VOC 2010+ / COCO
  "all-points" convention). A label absent from the references of a split is reported as ``None`` and left
  out of the means.
- **mAP@0.5 / mAP** — the mean over the scored labels of AP@0.5, and of the mean of AP@t over
  t = 0.50, 0.55, …, 0.95 (the COCO primary metric averaged over classes).
- **operating point** — at the pipeline's `RECOGNITION_THRESHOLD`, per-label recall / precision of the
  surviving predictions (greedy IoU ≥ 0.5 matching), and **grid agreement**: the fraction of tables whose
  surviving `table row` and `table column` counts both equal the reference counts — the structure-level
  question a caller actually asks.
- **mean best IoU** — for every reference object, the IoU of its best-overlapping prediction of the same
  label, averaged.

The **grid prior** frames the numbers: the training split's mean row and column counts, rounded, laid out as
a uniform grid over the whole image (table = the image), with a constant score — what "tables are usually
about this shape" alone buys.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .pipeline import box_iou` removed — names are kernel globals defined by the carried modules

IOU_THRESHOLDS = tuple(round(0.5 + 0.05 * i, 2) for i in range(10))
MATCH_IOU = 0.5

METRIC_DEFINITIONS = {
    "ap50": "per label: average precision at IoU 0.50 over all ranked predictions of that label",
    "ap75": "per label: average precision at IoU 0.75",
    "ap": "per label: mean of the average precision at IoU 0.50, 0.55, ..., 0.95",
    "map50": "mean over the scored labels of ap50",
    "map": "mean over the scored labels of ap (the COCO primary metric averaged over classes)",
    "recall_at_threshold": (
        "per label: fraction of reference objects matched (IoU >= 0.5) by a prediction of the same label at "
        "or above the operating threshold"
    ),
    "precision_at_threshold": (
        "per label: fraction of predictions of that label at or above the operating threshold that match a "
        "reference object"
    ),
    "grid_exact_at_threshold": (
        "fraction of tables whose surviving `table row` and `table column` counts both equal the reference "
        "counts"
    ),
    "mean_best_iou": (
        "mean over reference objects of the IoU of their best-overlapping prediction of the same label, "
        "any score"
    ),
}


def _check_pairs(
    predictions: Sequence[Sequence[Mapping[str, Any]]], references: Sequence[Sequence[Mapping[str, Any]]]
) -> None:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists for {len(references)} reference lists")
    if not references:
        raise ValueError("at least one image is required")
    for preds in predictions:
        for p in preds:
            if "box" not in p or "score" not in p or "label" not in p or len(p["box"]) != 4:
                raise ValueError("each prediction needs a 4-value 'box', a 'label' and a 'score'")
    for refs in references:
        for r in refs:
            if "box" not in r or "label" not in r or len(r["box"]) != 4:
                raise ValueError("each reference needs a 4-value 'box' and a 'label'")


def _of_label(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Mapping[str, Any]]],
    label: str,
) -> tuple[list[list[Mapping[str, Any]]], list[list[Sequence[float]]]]:
    return (
        [[p for p in preds if p["label"] == label] for preds in predictions],
        [[r["box"] for r in refs if r["label"] == label] for refs in references],
    )


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Mapping[str, Any]]],
    label: str,
    iou_threshold: float = MATCH_IOU,
) -> float | None:
    """All-points average precision for one label over a list of images (None when it has no references)."""
    _check_pairs(predictions, references)
    preds_l, refs_l = _of_label(predictions, references, label)
    n_ref = sum(len(r) for r in refs_l)
    if n_ref == 0:
        return None
    ranked = sorted(
        (
            (float(p["score"]), i, [float(v) for v in p["box"]])
            for i, preds in enumerate(preds_l)
            for p in preds
        ),
        key=lambda t: -t[0],
    )
    if not ranked:
        return 0.0
    matched = [np.zeros(len(r), dtype=bool) for r in refs_l]
    tp = np.zeros(len(ranked))
    for k, (_score, i, box) in enumerate(ranked):
        best, best_j = 0.0, -1
        for j, ref in enumerate(refs_l[i]):
            if matched[i][j]:
                continue
            iou = box_iou(box, ref)
            if iou > best:
                best, best_j = iou, j
        if best >= iou_threshold and best_j >= 0:
            matched[i][best_j] = True
            tp[k] = 1.0
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(1.0 - tp)
    recall = cum_tp / n_ref
    precision = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
    precision = np.maximum.accumulate(precision[::-1])[::-1]  # monotone envelope from the right
    recall = np.concatenate([[0.0], recall])
    return float(np.sum((recall[1:] - recall[:-1]) * precision))


def _operating_point(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Mapping[str, Any]]],
    labels: Sequence[str],
    threshold: float,
) -> dict[str, Any]:
    """Per-label recall / precision at or above `threshold` (greedy IoU >= MATCH_IOU) and grid agreement."""
    per_label = {}
    per_image: list[dict[str, Any]] = [{} for _ in references]
    for label in labels:
        preds_l, refs_l = _of_label(predictions, references, label)
        n_ref = sum(len(r) for r in refs_l)
        found = kept = true_kept = 0
        for i, (preds, refs) in enumerate(zip(preds_l, refs_l, strict=True)):
            surviving = [p for p in preds if float(p["score"]) >= threshold]
            kept += len(surviving)
            used: set[int] = set()
            image_found = 0
            for p in sorted(surviving, key=lambda q: -float(q["score"])):
                best, best_j = 0.0, -1
                for j, ref in enumerate(refs):
                    if j in used:
                        continue
                    iou = box_iou(p["box"], ref)
                    if iou > best:
                        best, best_j = iou, j
                if best >= MATCH_IOU and best_j >= 0:
                    used.add(best_j)
                    image_found += 1
                    true_kept += 1
            found += image_found
            per_image[i][label] = {
                "n_reference": len(refs),
                "n_at_threshold": len(surviving),
                "found": image_found,
            }
        per_label[label] = {
            "recall": found / n_ref if n_ref else None,
            "precision": true_kept / kept if kept else None,
            "n_reference": n_ref,
            "n_at_threshold": kept,
        }
    grid_hits = 0
    for entry in per_image:
        row, col = entry.get("table row"), entry.get("table column")
        if (
            row
            and col
            and row["n_at_threshold"] == row["n_reference"]
            and col["n_at_threshold"] == col["n_reference"]
        ):
            grid_hits += 1
            entry["grid_exact"] = True
        else:
            entry["grid_exact"] = False
    return {
        "threshold": float(threshold),
        "per_label": per_label,
        "grid_exact": grid_hits / len(references),
        "per_image": per_image,
    }


def structure_metrics(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Mapping[str, Any]]],
    *,
    labels: Sequence[str],
    threshold: float,
) -> dict[str, Any]:
    """Per-label AP@0.5 / AP@0.75 / AP, their class means, the operating point at `threshold` and at 0.5, grid
    agreement and mean best IoU."""
    _check_pairs(predictions, references)
    per_label: dict[str, dict[str, Any]] = {}
    for label in labels:
        aps = {t: average_precision(predictions, references, label, t) for t in IOU_THRESHOLDS}
        n_ref = sum(1 for refs in references for r in refs if r["label"] == label)
        if aps[0.5] is None:
            per_label[label] = {"n_reference": 0, "ap50": None, "ap75": None, "ap": None}
        else:
            per_label[label] = {
                "n_reference": n_ref,
                "ap50": aps[0.5],
                "ap75": aps[0.75],
                "ap": float(np.mean([v for v in aps.values()])),
                "ap_by_iou": {str(t): v for t, v in aps.items()},
            }
    scored = [label for label in labels if per_label[label]["ap50"] is not None]
    points = {
        t_: _operating_point(predictions, references, labels, t_) for t_ in sorted({0.5, float(threshold)})
    }
    primary = points[float(threshold)]
    best_ious = []
    per_image = []
    for i, (preds, refs) in enumerate(zip(predictions, references, strict=True)):
        image_best = []
        for ref in refs:
            same = [p for p in preds if p["label"] == ref["label"]]
            image_best.append(max((box_iou(p["box"], ref["box"]) for p in same), default=0.0))
        best_ious.extend(image_best)
        per_image.append(
            {
                "n_reference": len(refs),
                "grid_exact": primary["per_image"][i]["grid_exact"],
                "labels": {k: v for k, v in primary["per_image"][i].items() if k != "grid_exact"},
                "mean_best_iou": float(np.mean(image_best)) if image_best else 0.0,
            }
        )
    return {
        "n_images": len(references),
        "n_reference_objects": sum(len(r) for r in references),
        "labels": list(labels),
        "scored_labels": scored,
        "per_label": per_label,
        "map50": float(np.mean([per_label[label]["ap50"] for label in scored])) if scored else 0.0,
        "map75": float(np.mean([per_label[label]["ap75"] for label in scored])) if scored else 0.0,
        "map": float(np.mean([per_label[label]["ap"] for label in scored])) if scored else 0.0,
        "threshold": float(threshold),
        "recall_at_threshold": {label: primary["per_label"][label]["recall"] for label in labels},
        "precision_at_threshold": {label: primary["per_label"][label]["precision"] for label in labels},
        "grid_exact_at_threshold": primary["grid_exact"],
        "operating_points": {
            str(t_): {"per_label": point["per_label"], "grid_exact": point["grid_exact"]}
            for t_, point in points.items()
        },
        "mean_best_iou": float(np.mean(best_ious)) if best_ious else 0.0,
        "per_image": per_image,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def grid_prior(train_records: Sequence[Mapping[str, Any]]) -> dict[str, int]:
    """The training split's mean `table row` and `table column` counts, rounded to at least 1."""
    rows = [sum(1 for o in r["objects"] if o["label"] == "table row") for r in train_records]
    cols = [sum(1 for o in r["objects"] if o["label"] == "table column") for r in train_records]
    if not rows:
        raise ValueError("the training split has no records")
    return {"rows": max(1, round(float(np.mean(rows)))), "columns": max(1, round(float(np.mean(cols))))}


def grid_prior_prediction(width: int, height: int, grid: Mapping[str, int]) -> list[dict[str, Any]]:
    """A uniform grid over the whole image: one `table`, `rows` bands and `columns` bands, scored 1.0."""
    out = [{"box": [0.0, 0.0, float(width), float(height)], "label": "table", "score": 1.0}]
    n_rows, n_cols = int(grid["rows"]), int(grid["columns"])
    for i in range(n_rows):
        out.append(
            {
                "box": [0.0, height * i / n_rows, float(width), height * (i + 1) / n_rows],
                "label": "table row",
                "score": 1.0,
            }
        )
    for j in range(n_cols):
        out.append(
            {
                "box": [width * j / n_cols, 0.0, width * (j + 1) / n_cols, float(height)],
                "label": "table column",
                "score": 1.0,
            }
        )
    return out


def prior_baseline(
    train_records: Sequence[Mapping[str, Any]],
    test_records: Sequence[Mapping[str, Any]],
    *,
    labels: Sequence[str],
    threshold: float,
) -> dict[str, Any]:
    """The grid prior of the training split scored on the test records."""
    grid = grid_prior(train_records)
    predictions = [grid_prior_prediction(*record["image"].size, grid) for record in test_records]
    out = structure_metrics(
        predictions, [r["objects"] for r in test_records], labels=labels, threshold=threshold
    )
    out["baseline"] = (
        "a uniform grid of the training split's mean row and column counts over the whole image (grid prior)"
    )
    out["prior_grid"] = grid
    return out

**Module 4/4:** `src/table_transformer_structure_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Structure-labelled table dataset contract for adapting the recogniser: the embedded SciTSR-PD sample,
validation, seeded splitting by paper, BYOD loaders and CSV export.

The default dataset is **real** and public domain: 94 scientific tables from SciTSR-PD (arXiv LaTeX
tables rendered at 150 DPI whose source papers carry a CC0 or public-domain dedication), embedded in
`sample_data.py` with structure boxes derived once from the dataset's text chunks and logical cells (see
that module's docstring for the derivation and the drop rules). Column headers and projected row headers
are not annotated in SciTSR, so the sample carries four of the checkpoint's six labels — `table`,
`table column`, `table row`, `table spanning cell` — and the adaptation contract restricts itself to
them. Several tables come from the same paper, so the sample is split **by paper**, never by table.

A record is ``{id, image, objects}``: a PIL image (or a path to one) and a list of ``{label, box}``
structure objects with ``box`` as ``[x_min, y_min, x_max, y_max]`` pixels and ``label`` in
`ADAPT_LABELS`.
"""

from __future__ import annotations

import base64
import csv
import hashlib
import io
import json
import random
import re
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import ADAPT_LABELS, MAX_DETECTIONS, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .sample_data import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTSR-PD scientific tables"
CORPUS_RELEASE = SAMPLE_SOURCE
CORPUS_LICENSE = SAMPLE_LICENSE
CORPUS_DPI = SAMPLE_DPI
CORPUS_SOURCE_FILES = SOURCE_FILES
if tuple(SAMPLE_LABELS) != tuple(ADAPT_LABELS):
    raise RuntimeError("sample_data labels differ from the pipeline's adaptation labels")
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"test": 10, "validation": 7, "train": 29}  # papers (46 in the sample), drawn in this order
MIN_RECORDS = 8
MAX_RECORDS = 2_000
MAX_OBJECTS = (
    MAX_DETECTIONS  # the decoder emits at most this many boxes, so a table cannot carry more targets
)
MIN_BOX_SIDE = 2.0  # pixels: a row of a dense table at 150 DPI is about 20 px; 2 px refuses degenerate boxes
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def load_corpus() -> list[dict[str, Any]]:
    """Decode the embedded tables into `{id, image, objects}` records with provenance, verifying each PNG's
    byte size and SHA-256 against `SAMPLE_RECORDS` first."""
    out = []
    for entry in SAMPLE_RECORDS:
        data = base64.b64decode(SAMPLE_IMAGES_B64[entry["file"]])
        if len(data) != entry["png_bytes"] or _sha256_bytes(data) != entry["png_sha256"]:
            raise ValueError(f"{entry['table_id']}: embedded PNG does not match its recorded size / digest")
        image = Image.open(io.BytesIO(data))
        image.load()
        if image.size != (entry["width"], entry["height"]):
            raise ValueError(
                f"{entry['table_id']}: embedded PNG is {image.size}, "
                f"recorded {(entry['width'], entry['height'])}"
            )
        out.append(
            {
                "id": entry["table_id"],
                "image": image.convert("RGB"),
                "objects": [
                    {"label": o["label"], "box": [float(v) for v in o["box"]]} for o in entry["objects"]
                ],
                "paper_id": entry["paper_id"],
                "paper_title": entry["paper_title"],
                "paper_license": entry["paper_license"],
                "scitsr_split": entry["scitsr_split"],
                "grid": [entry["n_rows"], entry["n_cols"]],
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draw of whole papers into test / validation / train (in that order): `sizes` counts papers."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_paper: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_paper.setdefault(_split_unit(record), []).append(dict(record))
    papers = sorted(by_paper)
    rng.shuffle(papers)
    needed = sum(sizes.values())
    if len(papers) < needed:
        raise ValueError(f"only {len(papers)} papers available, need {needed}")
    out: dict[str, list[dict[str, Any]]] = {}
    cursor = 0
    for name, count in sizes.items():
        part = [r for paper in papers[cursor : cursor + count] for r in by_paper[paper]]
        cursor += count
        rng.shuffle(part)
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(part)]
    return out


def load_sample_dataset(
    *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the embedded corpus."""
    return build_sample_dataset(load_corpus(), seed=seed, sizes=sizes)


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/objects")
    for key in ("id", "image", "objects"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image, objects = record["id"], record["image"], record["objects"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: "
            f"{image.size}"
        )
    if (
        isinstance(objects, str | bytes)
        or not isinstance(objects, Sequence)
        or not 1 <= len(objects) <= MAX_OBJECTS
    ):
        raise ValueError(f"{label_name}: objects must be a list of 1..{MAX_OBJECTS} {{label, box}} entries")
    checked = []
    for b, obj in enumerate(objects):
        if not isinstance(obj, Mapping) or "label" not in obj or "box" not in obj:
            raise ValueError(f"{label_name}: objects[{b}] must be a mapping with label and box")
        label, box = obj["label"], obj["box"]
        if label not in ADAPT_LABELS:
            raise ValueError(f"{label_name}: objects[{b}] label {label!r} is not one of {ADAPT_LABELS}")
        if isinstance(box, str | bytes) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"{label_name}: objects[{b}] box must have four values")
        x0, y0, x1, y1 = (float(v) for v in box)
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(
                f"{label_name}: objects[{b}] box {[x0, y0, x1, y1]} must lie inside the {image.size} image "
                "with x0 < x1 and y0 < y1"
            )
        if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
            raise ValueError(f"{label_name}: objects[{b}] box is smaller than {MIN_BOX_SIDE} px on a side")
        checked.append({"label": label, "box": [x0, y0, x1, y1]})
    labels = [o["label"] for o in checked]
    if labels.count("table") > 1:
        raise ValueError(f"{label_name}: a record carries at most one `table` box")
    if "table row" not in labels or "table column" not in labels:
        raise ValueError(f"{label_name}: a record needs at least one `table row` and one `table column`")
    item = {"id": rid, "image": image.convert("RGB"), "objects": checked}
    for key in ("source_id", "paper_id", "paper_title", "paper_license", "scitsr_split", "group", "grid"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a structure-labelled table dataset; raises ValueError before any model
    import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, objects} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = [_check_record(record, index) for index, record in enumerate(records)]
    ids = [r["id"] for r in checked]
    if len(set(ids)) != len(ids):
        duplicate = next(i for i in ids if ids.count(i) > 1)
        raise ValueError(f"duplicate id {duplicate!r}")
    sides = [max(r["image"].size) for r in checked]
    n_objects = [len(r["objects"]) for r in checked]
    per_label = {label: 0 for label in ADAPT_LABELS}
    for r in checked:
        for o in r["objects"]:
            per_label[o["label"]] += 1
    return {
        "records": checked,
        "n_records": len(checked),
        "n_objects": sum(n_objects),
        "objects_per_label": per_label,
        "objects_per_image": {"min": min(n_objects), "max": max(n_objects)},
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same table matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [
            r["id"],
            image_digest(r["image"]),
            [[o["label"], [round(float(v), 2) for v in o["box"]]] for o in r["objects"]],
        ]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def _split_unit(record: Mapping[str, Any]) -> str:
    return str(record.get("group") or record.get("paper_id") or record.get("source_id") or record["id"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no table (by decoded-pixel digest) and no paper appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    units: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            unit = _split_unit(record)
            if unit in units and units[unit] != name:
                raise ValueError(f"paper {unit!r} has tables in both {units[unit]} and {name}")
            units[unit] = name
    return {name: len(records) for name, records in splits.items()}


def split_summary(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Tables, objects per label and papers per split (an observation of what the split unit was)."""
    out = {}
    for name, records in splits.items():
        per_label = {label: 0 for label in ADAPT_LABELS}
        for r in records:
            for o in r["objects"]:
                per_label[o["label"]] += 1
        out[name] = {
            "tables": len(records),
            "objects": per_label,
            "papers": len({_split_unit(r) for r in records}),
        }
    return out


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test by split unit (`group` / `paper_id`, else
    the table itself) after de-duplicating tables."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_unit: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_unit.setdefault(_split_unit(record), []).append(record)
    rng = random.Random(seed)
    units = sorted(by_unit)
    rng.shuffle(units)
    n_test = max(1, round(len(units) * test_fraction))
    n_val = round(len(units) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for name, chosen in (
        ("test", units[:n_test]),
        ("validation", units[n_test : n_test + n_val]),
        ("train", units[n_test + n_val :]),
    ):
        for unit in chosen:
            splits[name].extend(by_unit[unit])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, objects}` records from a directory or a zip holding `structure.csv` (columns `id`,
    `file`, `label`, `x_min`, `y_min`, `x_max`, `y_max`, optional `group`; one row per structure box, pixel
    coordinates) beside the image files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "structure.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "structure.csv" not in members:
            raise ValueError("BYOD zip must contain structure.csv")
        table = archive.read(members["structure.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError(
            "BYOD datasets must be a directory or a .zip holding structure.csv and the image files"
        )
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "label", "x_min", "y_min", "x_max", "y_max"} - set(
        rows[0].keys() if rows else set()
    )
    if missing:
        raise ValueError(f"structure.csv is missing columns {sorted(missing)}")
    grouped: dict[str, dict[str, Any]] = {}
    for row in rows:
        item = grouped.get(row["id"])
        if item is None:
            image = loader(row["file"])
            image.load()
            item = {"id": row["id"], "image": image.convert("RGB"), "objects": []}
            if row.get("group"):
                item["group"] = row["group"]
            grouped[row["id"]] = item
        item["objects"].append(
            {
                "label": row["label"],
                "box": [float(row["x_min"]), float(row["y_min"]), float(row["x_max"]), float(row["y_max"])],
            }
        )
    return list(grouped.values())


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the structure table of a split (one row per box, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["id", "file", "label", "x_min", "y_min", "x_max", "y_max", "group", "paper_id"],
        )
        writer.writeheader()
        for record in records:
            for obj in record["objects"]:
                box = obj["box"]
                writer.writerow(
                    {
                        "id": record["id"],
                        "file": f"{record.get('source_id') or record['id']}.png",
                        "label": obj["label"],
                        "x_min": round(box[0], 2),
                        "y_min": round(box[1], 2),
                        "x_max": round(box[2], 2),
                        "y_max": round(box[3], 2),
                        "group": _split_unit(record),
                        "paper_id": record.get("paper_id", ""),
                    }
                )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `7587a7ef111d…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TableTransformerStructurePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "table-transformer-structure-v1.1-all",
  "modelId": "microsoft/table-transformer-structure-recognition-v1.1-all",
  "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c",
  "files": [
    {
      "path": "README.md",
      "bytes": 1056,
      "sha256": "01eed360e0f54ad297ab347cfef208d68d8ac2c108b808147bb7af9f8d9132c8"
    },
    {
      "path": "config.json",
      "bytes": 76761,
      "sha256": "17a8a6edfb9e394263fa6ba9b82176ebccdfcc5d6cd29121ec91572c7d6be22c"
    },
    {
      "path": "model.safetensors",
      "bytes": 115437156,
      "sha256": "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 374,
      "sha256": "eead409bb80e36ae85b8377642c54550f0504f65688ba3a4967950cafe461df2"
    }
  ],
  "totalBytes": 115515347
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TableTransformerStructurePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Embedded corpus, validation and paper-level split

`load_corpus` decodes the 94 embedded PNGs after checking each against its recorded byte size and SHA-256, and turns each into a `{id, image, objects}` record with its derived structure boxes and provenance (paper id, title, licence, SciTSR split, grid size). The boxes were derived once by the build script recorded in the repository: the dataset's text chunks were placed on the trimmed image by an ink-coverage offset search, matched to the logical cells by text, and turned into a PubTables-style structure — an ink-bounded `table` box, `table row` / `table column` boxes tiling it at the mid-gaps, and a `table spanning cell` over the grid area of every multi-row or multi-column cell; 14 of the 108 SciTSR-PD tables were dropped by stated rules (unmatchable cells, overlapping rows or columns, a row or column without a text cell). `build_sample_dataset` draws 10 / 7 / 29 whole **papers** by a seeded shuffle into 21 / 23 / 50 test, validation and training tables; `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no table (by decoded-pixel digest) and no paper appears in two splits, `split_summary` reports tables, objects per label and papers per split, and the training structure table is written to `outputs/table_transformer_structure_train.csv` in the shape BYOD expects.

Look for: 94 tables from 46 papers, splits 50 / 23 / 21, three digests, and four refusal probes — a duplicate id, a box outside its image, a label outside the four, and an oversized image — each rejected before `torch` does anything. A few seconds.

In [ ]:
import hashlib
import io
import json
import time

from PIL import ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
t0 = time.perf_counter()
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_count = {'byod': len(records)}
else:
    corpus = load_corpus()
    raw_count = {'tables': len(corpus), 'papers': len({r['paper_id'] for r in corpus}), 'objects': sum(len(r['objects']) for r in corpus), 'spanning_cells': sum(1 for r in corpus for o in r['objects'] if o['label'] == 'table spanning cell')}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
load_seconds = round(time.perf_counter() - t0, 1)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
summary = split_summary(splits)
write_dataset_csv(train_records, 'outputs/table_transformer_structure_train.csv')
print({'data_source': data_source, 'raw': raw_count, 'splits': disjoint, 'split_summary': summary, 'load_seconds': load_seconds, 'source_files': [f['path'] for f in CORPUS_SOURCE_FILES]})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'objects': manifest['objects_per_label'], 'objects_per_image': manifest['objects_per_image'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {k: example[k] for k in ('id', 'source_id', 'paper_id', 'paper_title', 'paper_license', 'grid') if k in example}, 'size': example['image'].size, 'objects': [(o['label'], [round(v, 1) for v in o['box']]) for o in example['objects'][:4]]})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'box outside image': [{**train_records[0], 'objects': [{'label': 'table row', 'box': [0, 0, train_records[0]['image'].width + 5, 20]}, *train_records[0]['objects']]}, *train_records[1:8]],
    'label outside the four': [{**train_records[0], 'objects': [{'label': 'table column header', 'box': [1, 1, 40, 20]}, *train_records[0]['objects']]}, *train_records[1:8]],
    'oversized image': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 16))}, *train_records[1:8]],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Recognise through the inference contract, with reference boxes

Before any adaptation, the inference contract is exercised as it always was, on one test table. `validate_inputs` applies exactly the checks `recognize` applies — image type, sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, a threshold in `[0, 1]` — and returns an input manifest; a deliberately invalid threshold is validated too and its rejection recorded as a finding. `recognize` returns the queries at or above `RECOGNITION_THRESHOLD` with their six-class labels, and `structure_summary` counts them into the grid they imply. Because this table carries reference boxes, `evaluation_report` can for the first time return **`sample-sanity`**: one `box_iou` entry per reference, matched only against detections of the same label, with the reference and detected counts per label (the build record's probe table: every row, column and the table matched at IoU 0.8 or better). A rendered preview (reference boxes in green, detections coloured by label) is displayed. Read the header rows: the checkpoint may label the first row a `table column header` — a label the reference set does not carry, which the adaptation vocabulary leaves out rather than penalises.

In [ ]:
COLOURS = {'table': (60, 60, 220), 'table row': (0, 160, 0), 'table column': (230, 30, 30), 'table spanning cell': (200, 0, 200), 'table column header': (240, 140, 0), 'table projected row header': (0, 170, 170)}

def draw_objects(image, reference, detections, width=2):
    canvas = image.convert('RGB').copy()
    pen = ImageDraw.Draw(canvas)
    for obj in reference:
        pen.rectangle([round(v) for v in obj['box']], outline=(0, 200, 0), width=1)
    for det in detections:
        pen.rectangle([round(v) for v in det['box']], outline=COLOURS.get(det['label'], (120, 120, 120)), width=width)
    return canvas

def references_by_label(record):
    out = {}
    for obj in record['objects']:
        out.setdefault(obj['label'], []).append(obj['box'])
    return out

probe_record = test_records[0]
image = probe_record['image']
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS, 'LABELS': list(LABELS), 'ADAPT_LABELS': list(ADAPT_LABELS), 'RECOGNITION_THRESHOLD': RECOGNITION_THRESHOLD}, 'contract': {'NUM_QUERIES': NUM_QUERIES, 'D_MODEL': D_MODEL, 'DECODER_LAYERS': DECODER_LAYERS, 'PARAMETER_COUNT': PARAMETER_COUNT}})
input_manifest = validate_inputs(image, threshold=RECOGNITION_THRESHOLD, names=[probe_record['id']])
try:
    validate_inputs(image, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'threshold-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/table_transformer_structure_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.recognize(image, threshold=RECOGNITION_THRESHOLD)
recognize_seconds = round(time.perf_counter() - started, 3)
checks = {
    'at_most_num_queries': len(result['detections']) <= MAX_DETECTIONS,
    'labels_in_vocabulary': all(d['label'] in LABELS for d in result['detections']),
    'scores_descending': all(a['score'] >= b['score'] for a, b in zip(result['detections'], result['detections'][1:])),
    'boxes_inside_image': all(0 <= d['box'][0] <= d['box'][2] <= image.width + 1 and 0 <= d['box'][1] <= d['box'][3] <= image.height + 1 for d in result['detections']),
}
if not all(checks.values()):
    raise RuntimeError(f'recognize output failed a sanity check: {checks}')
grid = structure_summary(result)
report = evaluation_report(result, references_by_label(probe_record), sample_kind='one SciTSR-PD test table' if not USE_BYOD else 'one BYOD test table')
print({'probe_id': probe_record['id'], 'source_id': probe_record.get('source_id'), 'reference_grid': probe_record.get('grid'), 'reference_objects': len(probe_record['objects']), 'detections_at_threshold': len(result['detections']), 'implied_grid': [grid['n_rows'], grid['n_columns']], 'counts': grid['counts'], 'seconds': recognize_seconds, 'device': pipe.device, 'checks': checks, 'findings': len(input_manifest['findings'])})
print({'verdict': report['verdict'], 'reason': report.get('reason')})
for metric in report['metrics']:
    print({'reference': metric['reference'], 'box_iou': round(metric['value'], 3), 'n_reference': metric['n_reference'], 'n_detected': metric['n_detected']})
assert report['verdict'] == 'sample-sanity'
try:
    from IPython.display import display
    display(draw_objects(image, probe_record['objects'], result['detections']))
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in Section 9'})

## 6. The grid prior, the zero-shot checkpoint and the frozen policy

Three rows frame the adaptation, all on the 21 test tables and all threshold-free: per-label AP@0.5, AP@0.75 and AP rank every (query, class) pair of every image by score, so the recogniser is judged on its ordering, and the per-label recall at the operating threshold (0.5, the pipeline's default) and the **grid agreement** — the fraction of tables whose surviving row and column counts both match — are printed beside them. The **grid prior** lays the training split's mean row and column counts out uniformly over each image — what "tables are usually about this shape" alone buys. The **zero-shot checkpoint** (`evaluate_zero_shot`) scores every query by its own softmax probability for each of the four labels, headers neither scored nor penalised. The **frozen policy** is `adapt` with `trainable_layers=0`: it first copies the checkpoint's own rows for the four labels and no-object into a restricted head and copies the box head, scores them untouched on validation (epoch 0, the **zero-shot policy**), then trains both on the cached decoder features of the 50 training tables under the DETR set loss for `HEAD_STEPS` full-batch steps (epoch 1) and keeps whichever has the lower validation loss, scored on the test split by `evaluate`. The build record: prior mAP@0.5 38.7 % (grid agreement 4.8 %), zero-shot 88.7 % / AP@0.75 73.4 % / mAP 63.5 % (columns 100 %, rows 86.5 %, spanning cells 74.8 %; grid agreement 85.7 %), and the trained heads 85.2 % / 76.1 % / 71.5 % (validation loss 0.295 against 0.771 for the untouched rows, so the frozen policy was kept over the zero-shot one) — the checkpoint already recognises the structure, and training the heads mostly moves the boxes onto the derived convention (mean best IoU 0.81 → 0.91) at a cost on spanning cells. About a minute on CPU.

In [ ]:
HEAD_STEPS = 300  # @param {type:"integer"}
HEAD_LR = 1e-3  # @param {type:"number"}

def brief(m):
    return {'map50': round(m['map50'], 4), 'map75': round(m['map75'], 4), 'map': round(m['map'], 4), 'ap50_per_label': {label: (None if v['ap50'] is None else round(v['ap50'], 4)) for label, v in m['per_label'].items()}, 'grid_exact_at_0.5': round(m['grid_exact_at_threshold'], 4), 'recall_at_0.5': {label: (None if v is None else round(v, 4)) for label, v in m['recall_at_threshold'].items()}, 'mean_best_iou': round(m['mean_best_iou'], 4), 'n': m['n_images']}

prior = prior_baseline(train_records, test_records, labels=ADAPT_LABELS, threshold=RECOGNITION_THRESHOLD)
print({'grid_prior': brief(prior), 'baseline': prior['baseline'], 'prior_grid': prior['prior_grid']})
t0 = time.perf_counter()
zero_shot_test = pipe.evaluate_zero_shot(test_records)
print({'zero_shot': brief(zero_shot_test), 'policy': zero_shot_test['policy'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
probe_result = pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=0)
frozen_test = pipe.evaluate(test_records)
print({'frozen_policy_call': probe_result['policy'], 'best_epoch': probe_result['best_epoch'], 'head_final_loss': round(probe_result['head_final_loss'], 4), 'validation': {entry['epoch']: entry['val'] for entry in probe_result['history']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_policy_test': brief(frozen_test), 'loss': round(frozen_test['loss'], 4), 'verdict': frozen_test['verdict']})
print({'definitions': frozen_test['definitions']})
assert frozen_test['map50'] > prior['map50'] and probe_result['policy'] in (POLICY_ZERO_SHOT, POLICY_FROZEN)

## 7. The unfrozen policy: a bounded decoder unfreeze selected against the heads and the checkpoint

`adapt` with `TRAINABLE_LAYERS` > 0 repeats epochs 0 and 1 of the ladder (the untouched rows, then the restricted heads on the frozen features), then unfreezes the last `TRAINABLE_LAYERS` decoder layers — two by default, 3,157,504 of 28,828,619 parameters; the backbone, the input projection, the encoder, the query embeddings, the earlier decoder layers and the checkpoint's own heads stay frozen — and trains them with both heads end to end, one table per step, for `EPOCHS` epochs (AdamW at `LEARNING_RATE`, weight decay 0.01, gradient clipping 0.1, seeded order, no augmentation) under the same DETR set loss: exact Hungarian matching of the 125 queries to the reference objects (the O(n²m) shortest-augmenting-path algorithm in the carried module, no external solver), cross-entropy over the five logits with a 0.1 no-object weight, L1 and GIoU box terms. Every epoch is scored on validation, and the epoch with the **lowest validation loss** is kept — the checkpoint's own rows (epoch 0) and the heads alone (epoch 1) compete on equal terms, so the selected policy can be any of the three. mAP@0.5, mAP and grid agreement are printed beside the loss at every epoch.

Watch the validation loss: in the build record it fell from 0.771 (untouched rows) to 0.295 (heads), then 0.310 → 0.282 → 0.286 over three unfreeze epochs at 1e-4, so the second unfreeze epoch was selected; at 3e-4 the unfreeze never beat the heads (0.390 → 0.356 → 0.347) and the frozen policy was kept; with all six decoder layers unfrozen it reached 0.288 at the second unfreeze epoch (selected) for 85.6 % mAP@0.5 / 73.5 % mAP on the test split and a 38 MB adapter — no better than two layers. Note that DETR's train-mode dropout makes the per-table training loss sit above the full-batch head loss.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

def report_epoch(entry):
    row = {'epoch': entry['epoch'], 'stage': entry['stage'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_loss'] = round(entry['val']['loss'], 4)
        row['val_map50'] = round(entry['val']['map50'], 4)
        row['val_map'] = round(entry['val']['map'], 4)
        row['val_grid_exact'] = round(entry['val']['grid_exact'], 4)
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=TRAINABLE_LAYERS, epochs=EPOCHS, lr=LEARNING_RATE, progress=report_epoch)
adapt_seconds = round(time.perf_counter() - t0, 1)
report_epoch(adapt_result['history'][0])
print({'selected_policy': adapt_result['policy'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'trainable_heads': adapt_result['n_trainable_head'], 'trainable_layers': adapt_result['n_trainable_layers'], 'total_parameters': adapt_result['n_total'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or policy selection, and no paper in it appears in the training or validation splits. The selected model is scored exactly as the frozen policy was in Section 6, and the four rows are put side by side: grid prior, zero-shot checkpoint, frozen policy, selected policy — per label and as class means, with grid agreement and the set loss. Read the policy first: if validation kept the heads, the last two rows are the same model; if it kept the untouched rows, the selected row is the checkpoint under a restricted softmax; if it chose the unfreeze, the delta is what the unfreeze bought on 21 tables — the build record: 86.8 % mAP@0.5 / 77.5 % AP@0.75 / 72.2 % mAP against 85.2 % / 76.1 % / 71.5 % for the heads, and against 88.7 % / 73.4 % / 63.5 % for the untouched checkpoint: adaptation tightened the boxes onto the derived convention (mAP +8.7 points over the checkpoint) and lost AP@0.5 on spanning cells (74.8 % → 62.6 %, from 56 training instances) and grid agreement at 0.5 (85.7 % → 76.2 %). The cell asserts the selected model beats the grid prior on mAP@0.5; it does **not** assert a gain over the heads or the checkpoint, because that is the question, not the answer. 21 tables with 300 objects from one seeded split of one corpus give no dispersion estimate — one table is about five points of grid agreement.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
rows = {'grid_prior': prior, 'zero_shot': zero_shot_test, 'frozen_policy': frozen_test, 'selected_policy': adapted_test}
comparison = {metric: {name: round(m[metric], 4) for name, m in rows.items()} for metric in ('map50', 'map75', 'map', 'mean_best_iou', 'grid_exact_at_threshold')}
for label in ADAPT_LABELS:
    comparison[f'ap50[{label}]'] = {name: (None if m['per_label'][label]['ap50'] is None else round(m['per_label'][label]['ap50'], 4)) for name, m in rows.items()}
    comparison[f'recall_at_0.5[{label}]'] = {name: (None if m['recall_at_threshold'][label] is None else round(m['recall_at_threshold'][label], 4)) for name, m in rows.items()}
comparison['loss'] = {'frozen_policy': round(frozen_test['loss'], 4), 'selected_policy': round(adapted_test['loss'], 4)}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('map50', 'map75', 'map')}
comparison['delta_vs_zero_shot'] = {metric: round(adapted_test[metric] - zero_shot_test[metric], 4) for metric in ('map50', 'map75', 'map')}
comparison['selected_policy'] = adapt_result['policy']
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'split_summary': summary,
    'single_image_report': report,
    'baselines': {'grid_prior': prior, 'zero_shot': zero_shot_test},
    'frozen_policy': {'adaptation': {k: v for k, v in probe_result.items() if k not in ('history', 'trainable_names')}, 'history': probe_result['history'], 'test': frozen_test},
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/table_transformer_structure_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['map50'] > prior['map50']
print({'report': 'outputs/table_transformer_structure_evaluation_report.json'})

## 9. Recognise before and after, export the adapter and reload it

Three test tables are run through `recognize_adapted` with the selected model at the 0.5 threshold — each query's arg-max label among the four adapted classes when its probability reaches the threshold — and rendered beside the frozen policy's objects (from a fresh pipeline with heads trained the same way — after an unfreeze the decoder inside `pipe` has moved, so the frozen column needs its own decoder) and the reference boxes: reference in thin green, objects coloured by label (rows green, columns red, table blue, spanning cells magenta); `outputs/table_transformer_structure_preview.png` holds the sheet. `recognize` — the checkpoint's own six-class heads — still answers in its own label space on the same tables, headers included.

`save_artifact` writes the restricted class head and box head and, when the unfrozen policy was selected, the trained decoder-layer tensors — about 0.5 MB for the heads alone, 13.2 MB with two decoder layers — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the classes, the selected policy, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `TableTransformerStructurePipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, rebuilds the heads from the manifest, refuses any tensor that is not a decoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical (query, class) scores and boxes on three tables and an identical test mAP@0.5 (VER4).

In [ ]:
import shutil

show = test_records[:3]
frozen_pipe = TableTransformerStructurePipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
frozen_pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=0)
before_after = []
panels = []
for record in show:
    after = pipe.recognize_adapted(record['image'], threshold=RECOGNITION_THRESHOLD)
    before = frozen_pipe.recognize_adapted(record['image'], threshold=RECOGNITION_THRESHOLD)
    base = pipe.recognize(record['image'], threshold=RECOGNITION_THRESHOLD)
    reference_counts = {label: sum(1 for o in record['objects'] if o['label'] == label) for label in ADAPT_LABELS}
    def counts(detections):
        return {label: sum(1 for d in detections if d['label'] == label) for label in ADAPT_LABELS}
    before_after.append({'id': record['id'], 'source_id': record.get('source_id'), 'reference': reference_counts, 'frozen': counts(before['detections']), 'selected': counts(after['detections']), 'checkpoint': structure_summary(base)['counts']})
    print(before_after[-1])
    left = draw_objects(record['image'], record['objects'], before['detections'])
    right = draw_objects(record['image'], record['objects'], after['detections'])
    panel = Image.new('RGB', (left.width * 2 + 8, left.height), (255, 255, 255))
    panel.paste(left, (0, 0))
    panel.paste(right, (left.width + 8, 0))
    panels.append(panel)
sheet = Image.new('RGB', (max(p.width for p in panels), sum(p.height for p in panels) + 8 * (len(panels) - 1)), (255, 255, 255))
y = 0
for panel in panels:
    sheet.paste(panel, (0, y))
    y += panel.height + 8
sheet.save('outputs/table_transformer_structure_preview.png')
try:
    from IPython.display import display
    display(sheet)
except ImportError:
    pass

artifact_dir = Path('outputs/table_transformer_structure_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'table_transformer_structure', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'policy': artifact_manifest['adapter']['policy'], 'classes': artifact_manifest['adapter']['classes'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = TableTransformerStructurePipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
parity = {'queries_identical': reloaded.predict_objects(show) == pipe.predict_objects(show), 'map50_in_memory': round(adapted_test['map50'], 6), 'map50_reloaded': round(reloaded_test['map50'], 6), 'classes_identical': reloaded.classes == pipe.classes}
print({'reload_parity': parity, 'reloaded_policy': reloaded.adapter['policy'], 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['queries_identical'] and parity['classes_identical'] and abs(adapted_test['map50'] - reloaded_test['map50']) < 1e-9

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHTS_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes'], 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'source_files': list(CORPUS_SOURCE_FILES), 'tables': len(SAMPLE_RECORDS), 'license': CORPUS_LICENSE, 'labels': list(ADAPT_LABELS), 'dpi': CORPUS_DPI},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_id': probe_record['id'], 'implied_grid': grid, 'single_image_report': report, 'seconds': recognize_seconds},
    'comparison': comparison,
    'before_after': before_after,
    'preview_file': 'outputs/table_transformer_structure_preview.png',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'policy': artifact_manifest['adapter']['policy']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/table_transformer_structure_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

On 21 held-out scientific tables the grid prior scores 38.7 % mAP@0.5, the untouched checkpoint 88.7 % (mAP 63.5 %, grid agreement 85.7 %), the restricted heads trained on its frozen features 85.2 % (mAP 71.5 %), and the last two decoder layers unfrozen with them 86.8 % (mAP 72.2 %, AP@0.75 77.5 %), selected at the second of three unfreeze epochs by validation loss; the adapter reloads to identical query scores. That is the claim and the finding: on a corpus close to the model's training renders, adaptation does not buy recognition the checkpoint lacks — it buys **agreement with a box convention** (mean best IoU 0.81 → 0.90, mAP +8.7 points over the checkpoint) and pays for it on the rarest label (spanning cells, 56 training instances: AP@0.5 74.8 % → 62.6 %) and on the structure-level question a caller asks (grid agreement at 0.5: 85.7 % → 76.2 %). The ladder ran all three policies end to end on a real structure-labelled corpus with the set loss and an exact Hungarian matcher implemented in the open, chose among them on validation rather than by assumption — and the validation loss, which is a set loss under the derived convention, preferred the trained heads to the checkpoint's untouched rows by a wide margin (0.295 vs 0.771) while the checkpoint kept the higher mAP@0.5 on the test split: the selection criterion and the headline metric disagree, and the notebook reports both rather than hiding one.

The test split is 21 tables with 300 structure objects from one seeded split of one small corpus with no dispersion estimate — one table is about five points of grid agreement, so a few points of AP is noise. The reference boxes are a derivation from SciTSR's cells (rows tiling an ink-bounded table at the mid-gaps), stated in full in the carried `sample_data` module; a metric against them measures agreement with that convention as much as recognition, which is exactly why the untouched checkpoint's AP@0.75 rises after adaptation while its AP@0.5 does not. AP ranks every (query, class) pair by the head's score; the pipeline's operating threshold of 0.5 is the upstream script's value, not a calibration, and the per-label recall at it is reported beside the AP — a deployment must choose its own threshold on its own labelled tables. When the unfrozen policy is selected it changes the last decoder layers, which every query shares, so `recognize` — which keeps the checkpoint's heads — reads a moved decoder afterwards; the artifact records which policy won.

Three things to carry to real data. **Baselines first:** the grid prior and the zero-shot checkpoint on *your* tables are the numbers to read before any trained head's — if the checkpoint already agrees with your boxes, adaptation is a convention change, not a capability change. **Leakage:** split by paper, document or source (the contract splits by `group` / `paper_id`, never by table). **Selection vs headline:** a loss-based selection and an AP-based headline can disagree; report both and say which one you optimised.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, decode and digest-verify an embedded structure-labelled corpus, validate the demonstrated dataset contract without leakage, execute the inference contract with a per-label `sample-sanity` report against reference boxes, run a three-policy adaptation ladder with validation-based selection, evaluate by per-label AP and grid agreement against a prior and the zero-shot checkpoint on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, PubTables-1M or FinTabNet accuracy, GriTS, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** raise `TRAINABLE_LAYERS` to 6 (in the build record all six layers were selected at their second epoch for 85.6 % mAP@0.5 / 73.5 % mAP — within the grain of two layers, with a 38 MB adapter); raise `LEARNING_RATE` to 3e-4 (the unfreeze never beat the heads on validation and the frozen policy was kept, 85.2 %); set `EPOCHS = 0` to keep the frozen policy and read the before/after sheet as a heads-only result; raise `HEAD_STEPS`; or bring your own structure-labelled tables through BYOD and read the prior and the zero-shot row before any policy.

## References

- Repository README: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/table-transformer-structure-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/table-transformer-structure-recognition-v1.1-all
- Upstream code: https://github.com/microsoft/table-transformer
- Aligning benchmark datasets for table structure recognition (Smock, Pesala, Abraham, 2023): https://arxiv.org/abs/2303.00716
- PubTables-1M (Smock, Pesala, Abraham, 2021): https://arxiv.org/abs/2110.00061
- End-to-End Object Detection with Transformers (DETR; the set loss and Hungarian matching, Carion et al., 2020): https://arxiv.org/abs/2005.12872
- SciTSR: Complicated Table Structure Recognition (Chi et al., 2019): https://arxiv.org/abs/1908.04729 — the public-domain subset SciTSR-PD: https://huggingface.co/datasets/bevaya/SciTSR-pd
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)